# A 動能持股：公告資訊決定非等權配置
使用者修正後主策略是 A：每五交易日選 25 檔並依決策前資料更新不同權重。下方先前 AH 段落僅保留為舊對照，請優先閱讀第 12–14 cells 的 A 版本。沿用既有 kernel，不重啟。

In [ ]:
# Locate this competition repository; do not restart an existing kernel.
import sys as _repo_sys, importlib.util as _repo_importlib
from pathlib import Path as _PlainPath
_candidates = [_PlainPath.cwd(), *_PlainPath.cwd().parents]
_repo_root = next((p for p in _candidates if (p / '99_維護與備份' / '工具' / '研究路徑.py').exists()), None)
if _repo_root is None:
    raise RuntimeError('Open Jupyter from this repository root or its 04_Notebook directory.')
if 'research_layout' not in _repo_sys.modules:
    _spec = _repo_importlib.spec_from_file_location('research_layout', _repo_root / '99_維護與備份' / '工具' / '研究路徑.py')
    _mod = _repo_importlib.module_from_spec(_spec)
    _repo_sys.modules['research_layout'] = _mod
    _spec.loader.exec_module(_mod)
from research_layout import ResearchPath as Path
%whos dict
import sys,json,datetime
print('Environment:',sys.executable)
print('Cases:',len(historical_a25_documents))
prestart_information_inventory=[]
for (year,sid),doc in sorted(historical_a25_documents.items()):
    anchor=datetime.date(year,10,29) if year==2024 else datetime.date(year,10,28)
    cutoff=datetime.datetime.combine(anchor,datetime.time(8,55))
    finance_ann=[]
    for ann in doc['announcements']:
        if any(x in (ann['subject'] or '') for x in ['財務報','財報','自結']):
            try:stamp=datetime.datetime.fromisoformat(str(ann['announce_date'])+'T'+str(ann['announce_time']))
            except (ValueError,TypeError):continue
            if stamp<cutoff:finance_ann.append({k:str(ann.get(k,'')) for k in ['announce_date','announce_time','subject','ann_id']})
    periods=sorted(set(str(x['period']) for x in doc['financials']))
    inventory={'year':year,'stock_id':sid,'name':doc.get('name'),'cutoff':str(cutoff),'financial_periods':periods,'prestart_financial_announcements':finance_ann,'revenue_has_release_timestamp':any('announce_date' in x or 'release_date' in x for x in doc['revenues'])}
    prestart_information_inventory.append(inventory)
print('Inventoried',len(prestart_information_inventory),'cases; full metadata retained in kernel')
print('Financial item schema, sample only:',sorted(set(x['statement']+':'+x['item'] for x in historical_a25_documents[(2024,'3293')]['financials'])))


In [ ]:
import re
prestart_financial_candidates=[]
for (year,sid),doc in sorted(historical_a25_documents.items()):
    anchor=datetime.date(year,10,29) if year==2024 else datetime.date(year,10,28)
    candidates=[]
    for ann in doc['announcements']:
        try:stamp=datetime.datetime.fromisoformat(str(ann['announce_date'])+'T'+str(ann['announce_time']))
        except (ValueError,TypeError):continue
        subject=ann['subject'] or ''
        if stamp>=datetime.datetime.combine(anchor,datetime.time(8,55)):continue
        if any(x in subject for x in ['預計','召開日期','更正','代子','代重要','自結','法說']):continue
        if not any(x in subject for x in ['財務報','財報']):continue
        if not any(x in subject for x in ['通過','提報','申報','核閱']):continue
        q=3 if any(x in subject for x in ['第三季','第3季','第３季']) else 2 if any(x in subject for x in ['第二季','第2季','第２季','上半年度']) else None
        if q is None:continue
        # Require the current reporting year in subject; ROC and Gregorian are accepted.
        if not any(str(x) in subject for x in [year,year-1911]):continue
        candidates.append((q,stamp,ann))
    chosen=max(candidates,key=lambda x:(x[0],x[1])) if candidates else None
    prestart_financial_candidates.append({'year':year,'stock_id':sid,'quarter':chosen[0] if chosen else None,'stamp':str(chosen[1]) if chosen else None,'announcement':chosen[2] if chosen else None})
from collections import Counter
print('Latest explicitly published report candidate:',dict(Counter((x['year'],x['quarter']) for x in prestart_financial_candidates)))
print('Q3 / unknown candidates:',[{k:x[k] for k in ['year','stock_id','quarter','stamp']} for x in prestart_financial_candidates if x['quarter']!=2])
for sid in ['3293','5536']:
    item=next(x for x in prestart_financial_candidates if x['year']==2024 and x['stock_id']==sid)
    print(sid,'Q2 original excerpt:',(item['announcement'] or {}).get('content','')[:1700])


## 第一版：有時間戳的公告財務數字原型
這是完整報告判讀版的分項實驗，尚未加入法說敘事、新聞、營收 YoY／MoM 或市場預期。只使用開賽前公告原文中的累計營收、營業利益與母公司淨利，逐季相減；不相減 EPS，不從目前 DB 快照拿未發布 Q3。保留公告原文、時間、引用 ID。

固定規則：最新可完整核對單季的營業利益率比前季增加至少 1 個百分點，且母公司淨利率未下降，列 +1；營業利益率下降至少 1 個百分點且母公司淨利率未上升，列 -1；其餘或缺資料為 0（未核對者另標未知）。權重倍率分別 1.25／0.75／1，25 檔正規化到 90%，不改選股。參數在看本次報酬前固定，非最佳化所得；缺資料不是利空。

Q3 自結／法說可能更早揭露，但本原型尚未整合，不能聲稱最新全部公開資訊。兩個同季起點均只買一次持有 24 日，與 AH 等權／起始動能加權隔離比較；原 A 每五日調整另列參考，不把頻率差異歸因公告。

In [ ]:
import unicodedata
def prestart_stamp(ann):
    try:return datetime.datetime.fromisoformat(str(ann['announce_date'])+'T'+str(ann['announce_time']))
    except (ValueError,TypeError):return None
def prestart_quarter(ann,year):
    s=unicodedata.normalize('NFKC',ann['subject'] or '')
    if any(x in s for x in ['預計','召開日期','更正','代子','代重要','自結','法說']):return None
    if not any(x in s for x in ['財務報','財報']) or not any(x in s for x in ['通過','提報','申報','核閱']):return None
    if not any(str(x) in s for x in [year,year-1911]):return None
    for q,words in [(3,['第三季','第3季']),(2,['第二季','第2季','上半年度']),(1,['第一季','第1季'])]:
        if any(x in s for x in words):return q
    return None
def prestart_original_amounts(ann):
    body=unicodedata.normalize('NFKC',ann.get('content','')).replace('\r','')
    vals={}
    # Only explicitly labelled cumulative amounts from original announcements.
    for field,number,label in [('revenue',4,'營業收入'),('operating',6,'營業利益'),('parent_net',9,'歸屬於母公司')]:
        match=re.search(r'(?m)^\s*'+str(number)+r'[.、]\s*([^\n]+)',body)
        if not match or label not in match.group(1):return None
        line=match.group(1)
        if '累計' not in line or ':' not in line:return None
        value=line.rsplit(':',1)[1].strip().replace(',','').replace(' ','')
        if re.fullmatch(r'\(?-?\d+(?:\.\d+)?\)?',value) is None:return None
        vals[field]=float(value.strip('()'))*(-1 if value.startswith('(') else 1)
    if vals['revenue']<=0:return None
    return vals
prestart_financial_evidence=[]
for (year,sid),doc in sorted(historical_a25_documents.items()):
    cutoff=datetime.datetime(year,10,29 if year==2024 else 28,8,55)
    quarters={}
    for ann in doc['announcements']:
        stamp=prestart_stamp(ann);quarter=prestart_quarter(ann,year)
        if stamp is None or stamp>=cutoff or quarter is None:continue
        vals=prestart_original_amounts(ann)
        if vals is None:continue
        if quarter not in quarters or stamp>quarters[quarter]['stamp']:
            quarters[quarter]={'stamp':stamp,'values':vals,'ann_id':ann['ann_id'],'subject':ann['subject']}
    usable=[q for q in [2,3] if all(x in quarters for x in range(1,q+1))]
    q=max(usable) if usable else None
    record={'year':year,'stock_id':sid,'score':0,'quarter':q,'status':'未知：尚缺可解析原公告','source_ids':[]}
    if q:
        def single(quarter):
            return {k:quarters[quarter]['values'][k]-(quarters[quarter-1]['values'][k] if quarter>1 else 0) for k in ['revenue','operating','parent_net']}
        previous,current=single(q-1),single(q)
        if previous['revenue']>0 and current['revenue']>0:
            delta_op=100*(current['operating']/current['revenue']-previous['operating']/previous['revenue'])
            delta_parent=100*(current['parent_net']/current['revenue']-previous['parent_net']/previous['revenue'])
            score=1 if delta_op>=1 and delta_parent>=0 else -1 if delta_op<=-1 and delta_parent<=0 else 0
            record.update({'score':score,'status':'可核對','previous_quarter':previous,'current_quarter':current,'op_margin_change_pp':delta_op,'parent_net_margin_change_pp':delta_parent,'source_ids':[quarters[x]['ann_id'] for x in range(1,q+1)],'source_stamps':[str(quarters[x]['stamp']) for x in range(1,q+1)]})
            assert all(quarters[x]['stamp']<cutoff for x in range(1,q+1))
    prestart_financial_evidence.append(record)
prestart_evidence_lookup={(x['year'],x['stock_id']):x for x in prestart_financial_evidence}
print('Evidence coverage:',dict(Counter((x['year'],x['status'],x['score']) for x in prestart_financial_evidence)))
print('Weights can be formed for both years; unknown evidence stays neutral, not negative.')


In [ ]:
import inspect
prestart_targets={}
def prestart_allocation(start,t,selected,mode):
    year=dates[start].year
    if mode=='equal':mult=np.ones(COUNT)
    elif mode=='momentum':mult=np.linspace(1.25,.75,COUNT)
    elif mode=='financial':mult=np.array([1+.25*prestart_evidence_lookup[(year,stock_ids[j])]['score'] for j in selected])
    else:raise ValueError('Unknown prestart mode')
    w=INVESTED*mult/mult.sum()
    assert len(selected)==25 and abs(w.sum()-.9)<1e-12 and np.max(w)<.10
    prestart_targets[(year,mode)]=[(stock_ids[j],float(weight)) for j,weight in zip(selected,w)]
    return w
prestart_simulator_source=inspect.getsource(simulate)
prestart_simulator_source=prestart_simulator_source.replace('def simulate(start,strategy,horizon=HORIZON):','def simulate_prestart(start,mode,horizon=HORIZON):\n    strategy="AH"')
old_allocation='target[selected]=(np.floor((INVESTED/COUNT)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)'
assert prestart_simulator_source.count(old_allocation)==1
prestart_simulator_source=prestart_simulator_source.replace(old_allocation,'target[selected]=(np.floor(prestart_allocation(start,t,selected,mode)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)')
prestart_simulator_source=prestart_simulator_source.replace('violations=0; min_cash=','strict_cap_days=0; min_held=COUNT; max_held=0; max_actual_weight=0.; violations=0; min_cash=')
prestart_simulator_source=prestart_simulator_source.replace('cap_age=np.where(weights>caps+1e-12,cap_age+1,0)','strict_cap_days+=int(np.any(weights>caps+1e-12))\n        min_held=min(min_held,int(np.count_nonzero(shares)))\n        max_held=max(max_held,int(np.count_nonzero(shares)))\n        max_actual_weight=max(max_actual_weight,float(weights.max()))\n        cap_age=np.where(weights>caps+1e-12,cap_age+1,0)')
prestart_simulator_source=prestart_simulator_source.replace("'path':vals}","'path':vals,'strict_cap_days':strict_cap_days,'min_held':min_held,'max_held':max_held,'max_actual_weight':max_actual_weight}")
exec(prestart_simulator_source)
prestart_results={}
prestart_comparison_lines=['|起點|方案|有效|24 日報酬|相對等權持有|最大回撤|成本／本金|','|---|---|---|---:|---:|---:|---:|']
prestart_labels={'equal':'等權持有 AH','momentum':'起始動能加權持有','financial':'起始公告財務加權持有','A':'原 A 每五日調整（參考）'}
for year,anchor in [(2024,datetime.date(2024,10,29)),(2025,datetime.date(2025,10,28))]:
    i=int(np.flatnonzero(np.array([dates[s]==anchor for s in window_starts]))[0]); start=int(window_starts[i])
    results={mode:simulate_prestart(start,mode) for mode in ['equal','momentum','financial']}
    original=backtest_results['AH'][i]
    assert results['equal']['invalid']==original['invalid']
    for field in ['return','mdd','cost','turnover']:assert np.isclose(results['equal'][field],original[field],atol=1e-12)
    assert np.allclose(results['equal']['path'],original['path'],atol=1e-6)
    ids=[sid for sid,w in prestart_targets[(year,'equal')]]
    assert set(ids)=={x['stock_id'] for x in historical_a25_cases if x['year']==year}
    assert all([sid for sid,w in prestart_targets[(year,m)]]==ids for m in ['momentum','financial'])
    results['A']=backtest_results['A'][i]
    prestart_results[year]=results
    for mode,result in results.items():
        vals=[result['return'],result['return']-results['equal']['return'],result['mdd'],result['cost']]
        if result['invalid']:cells=['—']*4
        else:cells=[f'{v:.2%}' for v in vals]
        prestart_comparison_lines.append('|'+str(anchor)+'|'+prestart_labels[mode]+'|'+('排除' if result['invalid'] else '可判定')+'|'+'|'.join(cells)+'|')
prestart_comparison_table='\n'.join(prestart_comparison_lines)
from IPython.display import display,Markdown
display(Markdown(prestart_comparison_table))
print('Held stock / strict-cap audit:',{year:{mode:(x['min_held'],x['max_held'],x['strict_cap_days'],x['violations']) for mode,x in results.items() if mode!='A'} for year,results in prestart_results.items()})
print('Coverage:',dict(Counter((x['year'],x['status']) for x in prestart_financial_evidence)))
print('Equal AH accounting reproduced; no historical price was read to set feature thresholds.')


In [ ]:
prestart_report_directory='C:\\Users\\weiha\\Desktop\\文件\\03_課程資料\\115-1\\資料探勘與應用\\B_Final Project (0920~1221)\\競賽資料'
from research_layout import ResearchPath as Path
prestart_report='# 開賽前公告資訊：起始非等權配置比較\n\n2026-10-14。研究方向由使用者確認：A 先選 25 檔，開賽前資訊決定第一天權重。這份是公告財務數字的第一個分項原型，尚未完成完整財報／法說／新聞／营收判讀版本。\n\n'
prestart_report+='## 公平比較方式\n\n同一 25 檔、90% 投資、整千股與原費用：等權 AH、起始動能加權、起始公告財務加權都只買一次持有 24 日。原 A 每五日調整另列參考，選股或調整頻率不同，不能把其差異全歸因權重。期末不清倉。沒有比較只靠新聞選股的策略。\n\n'
prestart_report+='## 公告財務數字原型\n\n只用開賽日 08:55 前已有時間戳的母公司財務結果公告原文，將 1–6／1–9 月累計營收、營業利益及歸屬母公司淨利相減成單季；不相減 EPS。已提報／通過與預告董事會分開；自結與法說尚未整合。没有用事後的 Q3 正式結果補前期資料。\n\n'
prestart_report+='固定規則：營業利益率比前季提高至少 1 個百分點且母公司淨利率未下降，倍率 1.25；利益率下降至少 1 個百分點且母公司淨利率未上升，倍率 0.75；其他為 1。未核對也為 1，但另標未知，不能當成中性營運。25 檔正規化到 90%，因此未調整倍率者權重也會隨總額正規化變化。規則與倍率在本次報酬計算前固定，沒有搜尋最佳值；它衡量營運變化，不是市場預期差。\n\n'
prestart_report+='九月營收資料沒有首次發布時間，未以月份推定可得，也未使用十月營收；MoM／YoY、正式報告附註、現金流、法說敘事、新聞、估值及市場預期尚未納入。這不等於使用者要求的完整 LLM 證據判讀版已完成。\n\n'
prestart_report+='## 兩年精確同季起點結果\n\n'+prestart_comparison_table+'\n\n'
prestart_report+='三個只買一次的方案各年均可判定，股數均為 25 檔，當日單股超限及原限制違規計數均為零；完整 Active Share、流動性／市場衝擊、提交及作廢流程仍未驗證。2024 原 A 每五日調整在 11/6 缺成交價，排除，不用暫算值比較。兩個年份不足判定穩定優勢，沒有把兩點做箱形圖或顯著性檢定。\n\n'
prestart_report+='## 開賽前原公告證據與第一天目標權重\n\n每列原公告 ID 可回查原 25 檔公告查閱文件。未知不自行補值；以下为目標，實際權重受價格與整張影響。\n\n'
for year in [2024,2025]:
    prestart_report+='### '+str(year)+'\n\n|股票|採用季別|資料狀態|營業利益率季差|母公司淨利率季差|倍率|起始目標權重|原公告 ID／時間|\n|---|---|---|---:|---:|---:|---:|---|\n'
    for sid,w in prestart_targets[(year,'financial')]:
        x=prestart_evidence_lookup[(year,sid)];case=next(c for c in historical_a25_cases if c['year']==year and c['stock_id']==sid)
        delta=lambda key: f"{x[key]:+.2f} 個百分點" if key in x else '未知'
        refs='<br>'.join(a+'／'+t for a,t in zip(x.get('source_ids',[]),x.get('source_stamps',[]))) or '待補'
        prestart_report+='|'+sid+' '+case['name']+'|'+('Q'+str(x['quarter']) if x['quarter'] else '未知')+'|'+x['status']+'|'+delta('op_margin_change_pp')+'|'+delta('parent_net_margin_change_pp')+'|'+str(1+.25*x['score'])+'|'+f'{w:.2%}'+'|'+refs+'|\n'
prestart_report+='\n## 完整判讀版仍需補齊\n\n1. 營收首次公開時間與原始數值、MoM／YoY、近期趨勢及認列節奏。\n2. 開賽前法說／自結：最新結果可能先於正式財報，必須排除重複計分。\n3. 財報現金流、應收、存貨／合約資產、業外與股數口徑。\n4. 每檔報告敘事相對上次公開資訊的更新、反證、引用位置、資訊缺漏。\n5. 固定的完整證據→部位規則、產業共同曝險；與相同股票及頻率的對照比較。\n6. 擴充更多歷史起點；現有 50 檔個案資料不能推成 650 個公告窗口。\n\n'
prestart_report+='判讀规则還需外部驗證；不因兩年結果好看而調倍率，亦不因原型無效便否定完整報告分析。現有等權 A／AH與溫和動能保留作基準，未提交或採用新實盤配置。\n\n'
prestart_report+='## 計算保存\n\n沿用 aicup-backtest-20261013 kernel，未重啟或重查 DB。原 AH 等權的帳務、逐日路徑、費用及報酬核對一致；輸入選股與歷史 A25 名單一致。Notebook 保存執行輸出，但依賴既有 kernel，不宣稱可獨立 Restart and Run All。原資料與既有 Notebook 未修改。\n\n關鍵物件：prestart_information_inventory、prestart_financial_evidence、prestart_evidence_lookup、prestart_targets、prestart_results。\n\n[計算 Notebook](動能持股_開賽前公告配置研究_2026-10-14.ipynb)；[原 25 檔公告查閱](A策略_歷史25檔持股與公告查閱_2026-10-14.md)。\n'
prestart_report_path=Path(prestart_report_directory)/'動能持股_開賽前公告配置研究_2026-10-14.md'
prestart_report_path.write_text(prestart_report,encoding='utf-8')
print('Saved prestart comparison report and 50 target-weight evidence rows')


## 使用者修正：主策略必須為 A，不能改成 AH
此前三個持有方案只列對照；正式比較保留 A 每五交易日重新選 25 檔並依當時可得資料更新權重。選股仍用 A；公告證據用來調權重，不更改持股數或改成持有不動。只買整千股，初始金額不等權。完整財報／法說／新聞仍待補齊，現階段先執行有原公告引用的財務證據版本。

In [ ]:
%whos dict
import inspect,json,datetime
print('Existing document fetch signature:',inspect.signature(fetch_historical_case_documents))
a_announcement_selection_inventory=[]
for year,anchor in [(2024,datetime.date(2024,10,29)),(2025,datetime.date(2025,10,28))]:
    start=di[anchor]
    for offset in [0,5,10,15,20]:
        t=start+offset
        rank=scores['A'][t-1]
        eligible=np.flatnonzero(np.isfinite(rank));picked=eligible[np.argsort(-rank[eligible],kind='stable')[:COUNT]]
        a_announcement_selection_inventory.append({'year':year,'decision_date':str(dates[t]),'t':int(t),'stock_ids':[stock_ids[j] for j in picked]})
print('Rebalance / coverage inventory:',[{k:x[k] for k in ['year','decision_date','stock_ids']} for x in a_announcement_selection_inventory])
print('Missing document cases:',sorted({(x['year'],sid) for x in a_announcement_selection_inventory for sid in x['stock_ids']} - set(historical_a25_documents)))


In [ ]:
fetch_src=inspect.getsource(fetch_historical_case_documents)
print('Document case contract:',[line.strip() for line in fetch_src.splitlines() if 'case[' in line or 'for case in' in line or 'out[' in line or 'return out' in line])
print('Safe case metadata sample:',historical_a25_cases[0])


In [ ]:
import ast
fetch_ast=ast.parse(fetch_src)
print('Loop variables:',[ast.unparse(x.target) for x in ast.walk(fetch_ast) if isinstance(x,ast.For)])
print('Dictionary key contract:',sorted({(x.value.id,str(x.slice.value)) for x in ast.walk(fetch_ast) if isinstance(x,ast.Subscript) and isinstance(x.value,ast.Name) and isinstance(x.slice,ast.Constant) and isinstance(x.slice.value,str)}))
print('Return expression shape:',[type(x.value).__name__ for x in ast.walk(fetch_ast) if isinstance(x,ast.Return)])


In [ ]:
assert 'READ ONLY' in fetch_src.upper() and 'rollback' in fetch_src.lower()
a_announcement_missing=sorted({(x['year'],sid) for x in a_announcement_selection_inventory for sid in x['stock_ids']} - set(historical_a25_documents))
a_announcement_fetch_cases=[{'year':year,'stock_id':sid,'window_end':'2024-12-02' if year==2024 else '2025-11-28'} for year,sid in a_announcement_missing]
a_announcement_extra_docs=fetch_historical_case_documents(a_announcement_fetch_cases)
print('Read-only query returned:',type(a_announcement_extra_docs).__name__,len(a_announcement_extra_docs))
if isinstance(a_announcement_extra_docs,dict):
    print('Safe return keys:',list(a_announcement_extra_docs)[:2]);sample=next(iter(a_announcement_extra_docs.values()))
else:sample=a_announcement_extra_docs[0]
print('Document shape:',list(sample) if isinstance(sample,dict) else type(sample).__name__)


## A 版本：選股與五日調整頻率不變，只改每檔投入比例
本段取代 AH 為主要實驗。兩年各 5 次決策、每次 25 檔，使用該交易日 08:55 前公告。增加 41 個年度／股票文件索引，原 50 份不覆寫。

每次倍率仍固定為財務趨勢 +1→1.25、0→1、-1→0.75，正規化至投資 90%；不是每檔相同額度。若已有比解析季報更新的同年度季度自結而尚未整合，標「更新結果待整合」並不使用舊財務趨勢加減碼。未核對不等於利空。完整法說、營收 MoM／YoY 與新聞仍未加入，不宣稱完成 LLM 報告版。

比較等權 A、每五日的溫和動能加權 A、每五日公告財務加權 A。公告資料僅涵蓋兩個歷史窗口，不推成 650 窗口。已知 2024 缺成交價，仍記錄完整診斷路徑但不採用暫算報酬。

In [ ]:
a_announcement_documents=dict(historical_a25_documents)
a_announcement_documents.update(a_announcement_extra_docs)
assert len(a_announcement_documents)==91
def a_financial_evidence_asof(year,sid,cutoff):
    doc=a_announcement_documents[(year,sid)]
    quarters={}; unintegrated=[]
    for ann in doc['announcements']:
        stamp=prestart_stamp(ann)
        if stamp is None or stamp>=cutoff:continue
        subject=unicodedata.normalize('NFKC',ann['subject'] or '')
        if '代' in subject:continue
        q=prestart_quarter(ann,year)
        if q is not None:
            vals=prestart_original_amounts(ann)
            if vals is not None and (q not in quarters or stamp>quarters[q]['stamp']):
                quarters[q]={'values':vals,'stamp':stamp,'ann_id':ann['ann_id']}
        if '自結' in subject and any(str(x) in subject for x in [year,year-1911]):
            for qq,words in [(3,['第三季','第3季','3Q','Q3']),(2,['第二季','第2季','2Q','Q2','上半年'])]:
                if any(word in subject for word in words):unintegrated.append((qq,stamp,ann['ann_id'],subject));break
    usable=[q for q in [2,3] if all(x in quarters for x in range(1,q+1))]
    q=max(usable) if usable else None
    record={'year':year,'stock_id':sid,'cutoff':str(cutoff),'score':0,'quarter':q,'status':'未知：缺可解析公告／口徑','source_ids':[]}
    if q:
        def single(quarter):return {key:quarters[quarter]['values'][key]-(quarters[quarter-1]['values'][key] if quarter>1 else 0) for key in ['revenue','operating','parent_net']}
        previous,current=single(q-1),single(q)
        if previous['revenue']>0 and current['revenue']>0:
            delta_op=100*(current['operating']/current['revenue']-previous['operating']/previous['revenue'])
            delta_parent=100*(current['parent_net']/current['revenue']-previous['parent_net']/previous['revenue'])
            score=1 if delta_op>=1 and delta_parent>=0 else -1 if delta_op<=-1 and delta_parent<=0 else 0
            record.update({'score':score,'status':'可核對','op_margin_change_pp':delta_op,'parent_net_margin_change_pp':delta_parent,'source_ids':[quarters[x]['ann_id'] for x in range(1,q+1)],'source_stamps':[str(quarters[x]['stamp']) for x in range(1,q+1)]})
    newer=[item for item in unintegrated if q is None or item[0]>q]
    if newer:
        record.update({'score':0,'status':'更新季度自結待整合','unintegrated_sources':[(str(x[1]),x[2],x[3]) for x in newer]})
    return record
a_announcement_evidence=[];a_announcement_lookup={}
for selection in a_announcement_selection_inventory:
    cutoff=datetime.datetime.combine(dates[selection['t']],datetime.time(8,55))
    for sid in selection['stock_ids']:
        record=a_financial_evidence_asof(selection['year'],sid,cutoff)
        a_announcement_evidence.append(record)
        a_announcement_lookup[(selection['t'],sid)]=record
print('250 decision-stock evidence records:',dict(Counter((x['year'],x['status']) for x in a_announcement_evidence)))
a_announcement_plans={}
def a_announcement_allocation(t,selected,mode):
    if mode=='equal':mult=np.ones(COUNT)
    elif mode=='momentum':mult=np.linspace(1.25,.75,COUNT)
    elif mode=='financial':mult=np.array([1+.25*a_announcement_lookup[(t,stock_ids[j])]['score'] for j in selected])
    else:raise ValueError('Unknown A allocation')
    w=INVESTED*mult/mult.sum()
    assert len(selected)==25 and abs(w.sum()-.9)<1e-12 and np.max(w)<.1
    a_announcement_plans[(t,mode)]=[(stock_ids[j],float(weight)) for j,weight in zip(selected,w)]
    return w
a_announcement_simulator_source=inspect.getsource(simulate)
a_announcement_simulator_source=a_announcement_simulator_source.replace('def simulate(start,strategy,horizon=HORIZON):','def simulate_a_announcement(start,mode,horizon=HORIZON):\n    strategy="A"')
assert a_announcement_simulator_source.count(old_allocation)==1
a_announcement_simulator_source=a_announcement_simulator_source.replace(old_allocation,'target[selected]=(np.floor(a_announcement_allocation(t,selected,mode)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)')
a_announcement_simulator_source=a_announcement_simulator_source.replace('violations=0; min_cash=','strict_cap_days=0; min_held=COUNT; max_held=0; max_actual_weight=0.; violations=0; min_cash=')
a_announcement_simulator_source=a_announcement_simulator_source.replace('cap_age=np.where(weights>caps+1e-12,cap_age+1,0)','strict_cap_days+=int(np.any(weights>caps+1e-12))\n        min_held=min(min_held,int(np.count_nonzero(shares)))\n        max_held=max(max_held,int(np.count_nonzero(shares)))\n        max_actual_weight=max(max_actual_weight,float(weights.max()))\n        cap_age=np.where(weights>caps+1e-12,cap_age+1,0)')
a_announcement_simulator_source=a_announcement_simulator_source.replace("'path':vals}","'path':vals,'strict_cap_days':strict_cap_days,'min_held':min_held,'max_held':max_held,'max_actual_weight':max_actual_weight}")
exec(a_announcement_simulator_source)
a_announcement_results={}
a_announcement_labels={'equal':'A 等權','momentum':'A 溫和動能加權','financial':'A 公告財務加權'}
a_announcement_lines=['|起點|方案|有效|24 日報酬|相對等權 A|最大回撤|成本／本金|','|---|---|---|---:|---:|---:|---:|']
for year,anchor in [(2024,datetime.date(2024,10,29)),(2025,datetime.date(2025,10,28))]:
    start=di[anchor];i=int(np.flatnonzero(window_starts==start)[0])
    results={mode:simulate_a_announcement(start,mode) for mode in a_announcement_labels}
    old=backtest_results['A'][i];new=results['equal']
    assert old['invalid']==new['invalid']
    for field in ['return','mdd','cost','turnover']:assert np.isclose(old[field],new[field],atol=1e-12)
    assert np.allclose(old['path'],new['path'],atol=1e-6)
    a_announcement_results[year]=results
    for mode,result in results.items():
        vals=[result['return'],result['return']-new['return'],result['mdd'],result['cost']]
        cells=['—']*4 if result['invalid'] else [f'{v:.2%}' for v in vals]
        a_announcement_lines.append('|'+str(anchor)+'|'+a_announcement_labels[mode]+'|'+('排除' if result['invalid'] else '可判定')+'|'+'|'.join(cells)+'|')
    for offset in [0,5,10,15,20]:
        t=start+offset
        assert all([sid for sid,w in a_announcement_plans[(t,mode)]]==[sid for sid,w in a_announcement_plans[(t,'equal')]] for mode in ['momentum','financial'])
a_announcement_table='\n'.join(a_announcement_lines)
display(Markdown(a_announcement_table))
print('A frequency preserved; identical 25 stocks on every rebalance; original A paths reproduced.')
print('Daily count / strict cap / violations:',{year:{mode:(x['min_held'],x['max_held'],x['strict_cap_days'],x['violations']) for mode,x in results.items()} for year,results in a_announcement_results.items()})
print('2025 initial increased candidates:',[(sid,w,a_announcement_lookup[(di[datetime.date(2025,10,28)],sid)].get('op_margin_change_pp'),a_announcement_lookup[(di[datetime.date(2025,10,28)],sid)].get('parent_net_margin_change_pp')) for sid,w in a_announcement_plans[(di[datetime.date(2025,10,28)],'financial')] if a_announcement_lookup[(di[datetime.date(2025,10,28)],sid)]['score']>0])


In [ ]:
a_announcement_report_directory='C:\\Users\\weiha\\Desktop\\文件\\03_課程資料\\115-1\\資料探勘與應用\\B_Final Project (0920~1221)\\競賽資料'
from research_layout import ResearchPath as Path
a_announcement_report='# A：公告資料決定非等權配置\n\n更新：2026-10-14。主策略已依使用者修正回 A，每五交易日重選 25 檔並重新配置；先前 AH 僅保留舊對照，不當成 A。\n\n'
a_announcement_report+='## 零股與配置規則\n\n已下載的官方 D-Plan v4.3「orders」段落要求股數為 1,000 倍數，所以不能買零股。先由證據決定不同目標權重，再用前日淨值、前日收盤價換算整張，不能手動指定任意股數。官方網站此次重查遭 403，未確認是否另有更新；以本機已保存 v4.3 原指南核對，不聲稱此次成功讀取官網。\n\n'
a_announcement_report+='[本機官方 D-Plan 指南 v4.3](官方文件/D-Plan_撰寫指南_v4.3.md)；[官方下載入口](https://esun-ai-challenge.tw/api/events/fundaiagent/attachments/28/download/)。\n\n'
a_announcement_report+='目標股數＝floor(目標權重 × 前日可交易淨值 ÷ 前日收盤價 ÷ 1000) × 1000；委託股數＝目標股數−原持股。一般單股 10% 上限、台積電 25% 上限，均不要求等權。\n\n'
a_announcement_report+='## 此次實驗確實保留 A\n\nA 20／60 日動能選股不變，兩年各 5 次調整，每次目標 25 檔、投資 90%、同一選股及原費用。等權、溫和動能加權、公告財務加權三者都每五日調整。每次用當日 08:55 前公告，不把新買股票沿用起點評分，也不用未公開的季度資料。\n\n'
a_announcement_report+='原 50 個年度股票案例新增唯讀查詢 41 個，現有 91 份文件索引，建立 250 筆「決策日×股票」證據。正式公告單季利益率改善至少 1 個百分點且母公司淨利率未下降→倍率 1.25；本業利益率下降至少 1 個百分點且母公司淨利率未上升→0.75；其他→1，正規化至 90%。不相減累計 EPS。若存在較新季度自結卻未解析，先標待整合，不用較舊季度結果推加減碼。缺資料不代表利空。\n\n'
a_announcement_report+='这是有引用的財務數字原型，沒有完成完整 LLM 法說／新聞／營收 MoM、YoY／市場預期判讀。原型倍率及 1 個百分點閾值是研究設定，不是已證明最佳值；沒有在看結果後調整參數追求高報酬。\n\n'
a_announcement_report+='## 同季窗口費用後結果\n\n'+a_announcement_table+'\n\n'
a_announcement_report+='2024 在 11/6 缺成交價，三方案都排除；诊斷路徑因未成交曾持有 26 檔，不能當作有效的固定 25 檔績效。2025 每日實際均為 25 檔，嚴格當日單股超限及原現金／檔數／漂移違規計數为零。完整 Active Share、成交量、市場衝擊與官方整批作廢未驗證。只有一个有效同季 A 窗口，不能主張公告加權優於等權或挑出最適權重。\n\n'
a_announcement_report+='## 哪些資料支持買多一點：2025/10/28 歷史起點\n\n此節不是 2026 現時買單。目標資金以每個歷史窗口的初始 10 億元計算；實際成交金額會受當日均價與費用影響。以下的股數遵守整千股，不含事後加碼。\n\n'
a_announcement_report+='|股票|起始配置標記|財務採用季度|本業利益率季差|母公司淨利率季差|目標權重|目標金額（百萬元）|前收（元）|目標張數|資料狀態|\n|---|---|---|---:|---:|---:|---:|---:|---:|---|\n'
a_t0=di[datetime.date(2025,10,28)]
for sid,w in a_announcement_plans[(a_t0,'financial')]:
    e=a_announcement_lookup[(a_t0,sid)];j=si[sid]
    name=next(c['name'] for c in historical_a25_cases if c['year']==2025 and c['stock_id']==sid)
    tag='增權候選' if e['score']>0 else '減權候選' if e['score']<0 else '未調倍率'
    delta=lambda field:f"{e[field]:+.2f} 個百分點" if field in e else '未知'
    lots=int(np.floor(w*INITIAL/close[a_t0-1,j]/LOT))
    a_announcement_report+='|'+sid+' '+name+'|'+tag+'|'+('Q'+str(e['quarter']) if e['quarter'] else '未知')+'|'+delta('op_margin_change_pp')+'|'+delta('parent_net_margin_change_pp')+'|'+f'{w:.2%}'+'|'+f'{w*INITIAL/1e6:.3f}'+'|'+f'{close[a_t0-1,j]:.2f}'+'|'+str(lots)+'|'+e['status']+'|\n'
a_announcement_report+='\n台達電本業利益率季差約 +3.25 個百分點、母公司淨利率 +2.64；創見分別 +8.03、+0.51，符合增權規則，目標各 4.6875%。這是資料支持的增權候選，不等於證明公告未被股價反映或更值得作為現時投資。\n\n'
a_announcement_report+='## 各次 A 決策的 25 檔目標與引用\n\n每個時點重算；未知股不強行評為差。原公告 ID 可回查 DB／既有查閱文件，較新自結的待整合來源一併列出。\n\n'
a_names={c['stock_id']:c['name'] for c in historical_a25_cases}
for selection in a_announcement_selection_inventory:
    t=selection['t'];year=selection['year']
    a_announcement_report+='### '+selection['decision_date']+'\n\n|股票|季度|目標權重|判讀|狀態|原公告引用 ID|\n|---|---|---:|---|---|---|\n'
    for sid,w in a_announcement_plans[(t,'financial')]:
        e=a_announcement_lookup[(t,sid)];metadata=a_announcement_documents[(year,sid)].get('metadata')
        name=a_names.get(sid,'')
        if not name and isinstance(metadata,dict):name=metadata.get('name','')
        reason='本業及母公司淨利率支持增權' if e['score']>0 else '本業與母公司淨利率支持減權審查' if e['score']<0 else '未調倍率／證據待補'
        refs='<br>'.join(e['source_ids'])
        if e.get('unintegrated_sources'):refs+='<br>待整合：'+'<br>'.join(x[1]+'／'+x[0] for x in e['unintegrated_sources'])
        a_announcement_report+='|'+sid+' '+str(name)+'|'+('Q'+str(e['quarter']) if e['quarter'] else '未知')+'|'+f'{w:.2%}'+'|'+reason+'|'+e['status']+'|'+(refs or '待補')+'|\n'
a_announcement_report+='\n## 保存與後續邊界\n\n所有計算在既有 kernel，沒有重啟；團隊 DB 僅唯讀、函式完成 rollback／close。等權 A 報酬／成本／逐日帳戶精確重現，所有配置每次選股相同。原 150 檔名單回推偏誤、公司行動缺漏及缺漏樣本選擇仍存在。完整資訊版本要繼續補自結、法說、營收發布時間與趨勢、現金流、新聞及共同產業曝險；不得將本原型包裝成完整 LLM 判讀成果。\n\n'
a_announcement_report+='原先 AH 比較下移為歷史對照；主研究後續都以 A 为準。Notebook 依賴既有 kernel，未做清除狀態後 Run All。\n\n[計算 Notebook](動能持股_開賽前公告配置研究_2026-10-14.ipynb)。\n'
a_path=Path(a_announcement_report_directory)/'動能持股_開賽前公告配置研究_2026-10-14.md'
a_old=a_path.read_text(encoding='utf-8')
if a_old.startswith('# A：'):a_old=a_old.split('<summary>舊 AH 對照（非主策略）</summary>\n\n',1)[-1].rsplit('\n</details>',1)[0]
a_path.write_text(a_announcement_report+'\n<details>\n<summary>舊 AH 對照（非主策略）</summary>\n\n'+a_old+'\n</details>\n',encoding='utf-8')
print('Saved A-primary report, 250 rebalance weight rows and initial lot examples')
print('2025 increased-candidate lot examples:',[(sid,w,int(np.floor(w*INITIAL/close[a_t0-1,si[sid]]/LOT)),float(close[a_t0-1,si[sid]])) for sid,w in a_announcement_plans[(a_t0,'financial')] if a_announcement_lookup[(a_t0,sid)]['score']>0])


In [ ]:
# Unattended process: initial data audit
import inspect
print('Kernel retained objects:', {k: type(globals()[k]).__name__ for k in ['dates','execution','bad_action','a_announcement_results','a_announcement_selection_inventory'] if k in globals()})
print('Result fields:', {y: {m: list(r.keys()) for m,r in v.items()} for y,v in a_announcement_results.items()})
print('Execution shape:', execution.shape)
process_audit_day = next(i for i,d in enumerate(dates) if str(d) == '2024-11-06')
process_audit_inventory = [r for r in a_announcement_selection_inventory if str(r['decision_date']) == '2024-11-06']
process_audit_missing = [sid for r in process_audit_inventory for sid in r['stock_ids'] if not good_exec[process_audit_day, si[sid]]]
print('Missing execution among selected on 2024-11-06:', process_audit_missing)
print('Simulator missing-execution checks:')
print('\n'.join(line for line in inspect.getsource(simulate).splitlines() if any(s in line for s in ['good_exec','missing','invalid','bad_action'])))


In [ ]:
# Unattended process: missing sell-side execution
process_audit_previous = next(r for r in a_announcement_selection_inventory if str(r['decision_date']) == '2024-10-29')
process_audit_current = process_audit_inventory[0]
process_audit_sells = sorted(set(process_audit_previous['stock_ids']) - set(process_audit_current['stock_ids']))
process_audit_missing_sells = [sid for sid in process_audit_sells if not good_exec[process_audit_day, si[sid]]]
print('Missing execution on sell candidates:', process_audit_missing_sells)
for sid in process_audit_missing_sells:
    j = si[sid]
    print('Price audit:', sid, [(str(dates[t]), float(execution[t,j]), bool(good_exec[t,j]), bool(bad_action[t,j])) for t in range(process_audit_day-2,process_audit_day+3)])
print('Result validity:', {y:{m: {'invalid':bool(r['invalid']),'min_held':r['min_held'],'max_held':r['max_held']} for m,r in v.items()} for y,v in a_announcement_results.items()})


In [ ]:
# Immediate resume: VWAP validity and evidence inventory
%whos ndarray
process_j = si['3036']
print('3036 price bands:', [(str(dates[t]), float(low[t,process_j]), float(high[t,process_j]), float(close[t,process_j]), float(execution[t,process_j])) for t in range(process_audit_day-2,process_audit_day+3)])
print('Financial parser:', inspect.getsource(prestart_original_amounts))
print('Evidence fields:', list(a_announcement_evidence[0].keys()))
process_pending = [r for r in a_announcement_evidence if r.get('unintegrated_sources')]
print('Pending self-report records:', process_pending)
for record in process_pending:
    sid = record.get('stock_id',record.get('sid'))
    yr = record.get('year',2025)
    for ann in a_announcement_documents[(yr,sid)]['announcements']:
        if ann['ann_id'] in [x[1] for x in record['unintegrated_sources']]:
            print('Original self-report', sid, ann['ann_id'], ann['subject'], ann['content'][:12000])


In [ ]:
# Immediate resume: inspect contracts for fixed financial ablation
print(inspect.getsource(a_financial_evidence_asof))
print(inspect.getsource(a_announcement_allocation))
print(a_announcement_simulator_source)
print('Source quarter contract:', inspect.getsource(prestart_quarter))
print('Raw price metadata:', {'volume':float(volume[process_audit_day,process_j]),'turnover':float(turnover[process_audit_day,process_j]),'stored_vwap':float(vwap[process_audit_day,process_j])})


In [ ]:
# Immediate resume: operating-only ablation with original self-reports
def a_operating_evidence(record):
    result = dict(record)
    result['score'] = 0
    pending = record.get('unintegrated_sources', [])
    delta = record.get('op_margin_change_pp')
    if pending:
        yr, sid = record['year'], record['stock_id']
        cutoff = datetime.datetime.fromisoformat(record['cutoff'])
        anns = a_announcement_documents[(yr,sid)]['announcements']
        latest_id = max(pending,key=lambda x:x[0])[1]
        latest = next(a for a in anns if a['ann_id']==latest_id)
        body = unicodedata.normalize('NFKC',latest['content'])
        quarter_header = ('第3季' in body or re.search(r'7\s*-\s*9月',body))
        def self_pair(label):
            line = next((l for l in body.splitlines() if re.match(label,l.strip())), '')
            tail = re.sub(label,'',line.strip(),count=1).lstrip(':： ')
            numbers = re.findall(r'\(?-?\d[\d,]*\)?',tail)
            if len(numbers)!=2:return None
            return [float(x.strip('()').replace(',',''))*(-1 if x.startswith('(') else 1) for x in numbers]
        revenue = self_pair(r'合併營業收入(?:淨額)?')
        operating = self_pair(r'合併營業淨利(?:\(損\))?')
        qs = {}
        for a in anns:
            stamp = prestart_stamp(a)
            q = prestart_quarter(a,yr)
            vals = prestart_original_amounts(a) if q in [1,2] else None
            if stamp is not None and stamp<cutoff and vals and (q not in qs or stamp>qs[q]['stamp']):
                qs[q] = {'values':vals,'stamp':stamp,'ann_id':a['ann_id']}
        if not quarter_header or not revenue or not operating or not all(q in qs for q in [1,2]):
            result.update({'status':'自結口徑不足：中性','op_margin_change_pp':None})
            return result
        prev_rev = qs[2]['values']['revenue']-qs[1]['values']['revenue']
        prev_op = qs[2]['values']['operating']-qs[1]['values']['operating']
        # Verify table column interpretation against cumulative Q2 figures.
        consistent = abs((revenue[1]-revenue[0])-qs[2]['values']['revenue'])<=max(2,abs(qs[2]['values']['revenue'])*1e-6)
        if revenue[0]<=0 or prev_rev<=0 or not consistent:
            result.update({'status':'自結累計口徑不一致：中性','op_margin_change_pp':None})
            return result
        delta = 100*(operating[0]/revenue[0]-prev_op/prev_rev)
        result.update({'quarter':3,'op_margin_change_pp':delta,'status':'自結單季本業可核對；非母公司淨利','source_ids':[qs[1]['ann_id'],qs[2]['ann_id'],latest_id],'source_stamps':[str(qs[1]['stamp']),str(qs[2]['stamp']),str(prestart_stamp(latest))],'self_quarter_revenue':revenue[0],'self_quarter_operating':operating[0]})
    if delta is not None and result['status'] in ['可核對','自結單季本業可核對；非母公司淨利']:
        result['score'] = 1 if delta>=1 else -1 if delta<=-1 else 0
    return result

a_operating_evidence_records = [a_operating_evidence(r) for r in a_announcement_evidence]
a_operating_lookup = {(di[datetime.datetime.fromisoformat(r['cutoff']).date()],r['stock_id']):r for r in a_operating_evidence_records}
def a_operating_allocation(t,selected,mode):
    if mode!='operating_only':return a_announcement_allocation(t,selected,mode)
    multipliers = np.array([1+.25*a_operating_lookup[(t,stock_ids[j])]['score'] for j in selected])
    weights = INVESTED*multipliers/multipliers.sum()
    assert len(selected)==25 and abs(weights.sum()-.9)<1e-12 and max(weights)<.1
    return weights
a_operating_simulator_source = a_announcement_simulator_source.replace('def simulate_a_announcement(', 'def simulate_a_operating(').replace('a_announcement_allocation(', 'a_operating_allocation(')
exec(a_operating_simulator_source,globals())
a_operating_results = {}
for yr, modes in a_announcement_results.items():
    start = modes['equal']['start']
    control = simulate_a_operating(start,'equal')
    assert np.array_equal(control['path'],modes['equal']['path']) and control['cost']==modes['equal']['cost']
    a_operating_results[yr] = simulate_a_operating(start,'operating_only')
print('Self-reports integrated:', [{k:r.get(k) for k in ['stock_id','cutoff','score','op_margin_change_pp','status','source_ids','self_quarter_revenue','self_quarter_operating']} for r in a_operating_evidence_records if r.get('unintegrated_sources')])
print('Coverage:', dict(Counter(r['status'] for r in a_operating_evidence_records)))
for yr,r in a_operating_results.items():
    print('A operating-only:',yr,{k:r[k] for k in ['invalid','return','mdd','cost','min_held','max_held','strict_cap_days','violations']},'paired difference pp:',100*(r['return']-a_announcement_results[yr]['equal']['return']) if not r['invalid'] else 'excluded')


In [ ]:
# Immediate resume: monthly revenue publication-time coverage
process_revenue_fields = next(iter(a_announcement_documents.values()))['revenues']
print('Revenue schema sample:', process_revenue_fields[:1])
process_revenue_publication = []
for (yr,sid),doc in a_announcement_documents.items():
    relevant = [a for a in doc['announcements'] if re.search(r'營收|營業收入',a['subject'] or '') and not re.search(r'財報|財務報|自結',a['subject'] or '')]
    process_revenue_publication.append({'year':yr,'stock_id':sid,'rows':len(doc['revenues']),'timestamped_announcements':len(relevant)})
print('Revenue publication coverage:', {'documents':len(process_revenue_publication),'with_related_announcements':sum(r['timestamped_announcements']>0 for r in process_revenue_publication),'with_revenue_rows':sum(r['rows']>0 for r in process_revenue_publication)})
print('Related announcement examples:', [(yr,sid,str(prestart_stamp(a)),a['subject']) for (yr,sid),doc in a_announcement_documents.items() for a in doc['announcements'] if re.search(r'營收|營業收入',a['subject'] or '') and not re.search(r'財報|財務報|自結',a['subject'] or '')][:8])
assert all(r['source_stamps'][-1]<r['cutoff'] for r in a_operating_evidence_records if r.get('source_stamps'))
print('As-of timestamps: all used source stamps precede decision cutoff.')


In [ ]:
# Revenue resume: publication candidates and read-only schema contract
for (yr,sid),doc in a_announcement_documents.items():
    related = [a for a in doc['announcements'] if re.search(r'營收|營業收入',a['subject'] or '') and not re.search(r'財報|財務報|自結',a['subject'] or '')]
    if related:
        print('Revenue announcement inventory:', yr,sid,[(str(prestart_stamp(a)),a['subject']) for a in related if str(prestart_stamp(a))[:7] in [str(yr)+'-10',str(yr)+'-11']])
        for a in related:
            if str(prestart_stamp(a))[:7] in [str(yr)+'-10',str(yr)+'-11']:
                print('Revenue original:',a['ann_id'],a['content'][:5000])
process_fetch_ast = ast.parse(inspect.getsource(fetch_historical_case_documents))
print('Read-only fetch queries:', [n.value for n in ast.walk(process_fetch_ast) if isinstance(n,ast.Constant) and isinstance(n.value,str) and n.value.strip().upper().startswith('SELECT')])
print('Fetch top-level statement contract:',[(type(n).__name__,getattr(n,'name','')) for n in process_fetch_ast.body[0].body])


In [ ]:
# Revenue resume: connection structure without configuration values
print('Fetch assignment metadata:',[(ast.unparse(n.targets[0]),type(n.value).__name__,ast.unparse(n.value.func) if isinstance(n.value,ast.Call) else '') for n in process_fetch_ast.body[0].body if isinstance(n,ast.Assign)])
process_fetch_try = next(n for n in process_fetch_ast.body[0].body if isinstance(n,ast.Try))
print('Read-only try structure:',[(type(n).__name__,ast.unparse(n.items[0].context_expr) if isinstance(n,ast.With) else '') for n in process_fetch_try.body])
print('Cleanup structure:',ast.unparse(ast.Module(body=process_fetch_try.finalbody,type_ignores=[])))


In [ ]:
# Revenue resume: read-only database publication column audit
import copy
process_schema_ast = copy.deepcopy(process_fetch_ast)
process_schema_func = process_schema_ast.body[0]
process_schema_func.name = 'fetch_revenue_schema_readonly'
process_schema_func.args = ast.arguments(posonlyargs=[],args=[],kwonlyargs=[],kw_defaults=[],defaults=[])
process_schema_try = next(n for n in process_schema_func.body if isinstance(n,ast.Try))
process_schema_query = ast.parse("""
with dbc.cursor() as c:
    c.execute("SELECT TABLE_NAME,COLUMN_NAME,DATA_TYPE FROM information_schema.COLUMNS WHERE TABLE_SCHEMA=DATABASE() AND (TABLE_NAME LIKE '%revenue%' OR COLUMN_NAME IN ('announce_date','publish_date','release_date','updated_at','created_at')) ORDER BY TABLE_NAME,ORDINAL_POSITION")
    result = c.fetchall()
return result
""").body
process_schema_try.body = process_schema_try.body[:2]+process_schema_query
ast.fix_missing_locations(process_schema_ast)
exec(compile(process_schema_ast,'<readonly_revenue_schema>','exec'),globals())
process_revenue_schema = fetch_revenue_schema_readonly()
print('Revenue/publication schema:', process_revenue_schema)
print('Schema fetch preserves read-only transaction, rollback and close.')


In [ ]:
# Revenue resume: historical news schema for timestamped supporting evidence
process_news_schema_ast = copy.deepcopy(process_schema_ast)
process_news_schema_ast.body[0].name = 'fetch_news_schema_readonly'
process_news_schema_try = next(n for n in process_news_schema_ast.body[0].body if isinstance(n,ast.Try))
process_news_schema_try.body = process_news_schema_try.body[:2]+ast.parse("""
with dbc.cursor() as c:
    c.execute("SELECT TABLE_NAME,COLUMN_NAME,DATA_TYPE FROM information_schema.COLUMNS WHERE TABLE_SCHEMA=DATABASE() AND TABLE_NAME IN ('news','news_stock','fetch_log') ORDER BY TABLE_NAME,ORDINAL_POSITION")
    result = c.fetchall()
return result
""").body
ast.fix_missing_locations(process_news_schema_ast)
exec(compile(process_news_schema_ast,'<readonly_news_schema>','exec'),globals())
process_news_schema = fetch_news_schema_readonly()
print('News schema:',process_news_schema)


In [ ]:
# Revenue resume: bounded historical news and ingestion-time audit
process_news_coverage_ast = copy.deepcopy(process_schema_ast)
process_news_coverage_ast.body[0].name = 'fetch_news_coverage_readonly'
process_news_coverage_try = next(n for n in process_news_coverage_ast.body[0].body if isinstance(n,ast.Try))
process_news_coverage_try.body = process_news_coverage_try.body[:2]+ast.parse("""
with dbc.cursor() as c:
    c.execute("SELECT COUNT(*) AS total,MIN(published_at) AS first_published,MAX(published_at) AS last_published FROM news")
    overall = c.fetchall()
    c.execute("SELECT YEAR(published_at) AS yr,MONTH(published_at) AS mo,COUNT(*) AS records FROM news WHERE ((published_at>='2024-09-01' AND published_at<'2024-12-03') OR (published_at>='2025-09-01' AND published_at<'2025-11-29')) GROUP BY YEAR(published_at),MONTH(published_at)")
    historical = c.fetchall()
    c.execute("SELECT stock_id,`year_month`,source,updated_at FROM monthly_revenue WHERE stock_id='2330' AND `year_month` IN ('2024-09','2024-10','2025-09','2025-10') ORDER BY `year_month`")
    ingestion = c.fetchall()
return {'overall':overall,'historical':historical,'monthly_ingestion_examples':ingestion}
""").body
ast.fix_missing_locations(process_news_coverage_ast)
exec(compile(process_news_coverage_ast,'<readonly_news_coverage>','exec'),globals())
process_news_coverage = fetch_news_coverage_readonly()
print('Historical availability audit:',process_news_coverage)


In [ ]:
# Revenue resume: historical timestamped revenue news retrieval
process_revenue_news_ast = copy.deepcopy(process_schema_ast)
process_revenue_news_ast.body[0].name = 'fetch_revenue_news_readonly'
process_revenue_news_try = next(n for n in process_revenue_news_ast.body[0].body if isinstance(n,ast.Try))
process_revenue_news_try.body = process_revenue_news_try.body[:2]+ast.parse("""
with dbc.cursor() as c:
    ids = sorted({sid for yr,sid in a_announcement_documents})
    placeholders = ','.join(['%s']*len(ids))
    query = "SELECT n.news_id,n.published_at,n.title,n.summary,n.content,n.url,n.source,ns.stock_id,ns.match_type FROM news n JOIN news_stock ns ON n.news_id=ns.news_id WHERE ns.stock_id IN ("+placeholders+") AND ((n.published_at>='2024-09-01' AND n.published_at<'2024-12-03') OR (n.published_at>='2025-09-01' AND n.published_at<'2025-11-29')) AND n.title LIKE '%%營收%%' ORDER BY n.published_at,n.news_id,ns.stock_id LIMIT 1000"
    c.execute(query,tuple(ids))
    result = c.fetchall()
return result
""").body
ast.fix_missing_locations(process_revenue_news_ast)
exec(compile(process_revenue_news_ast,'<readonly_revenue_news>','exec'),globals())
a_revenue_news_candidates = fetch_revenue_news_readonly()
print('Revenue-news candidates:',{'joined_rows':len(a_revenue_news_candidates),'unique_news':len({n['news_id'] for n in a_revenue_news_candidates}),'year_stock_coverage':len({(n['published_at'].year,n['stock_id']) for n in a_revenue_news_candidates}),'limit_reached':len(a_revenue_news_candidates)>=1000})
print('Evidence samples:', [{k:n[k] for k in ['stock_id','published_at','title','url','source','match_type']} for n in a_revenue_news_candidates[:8]])
print('Coverage by year:',dict(Counter(n['published_at'].year for n in a_revenue_news_candidates)))


In [ ]:
# Revenue resume: single-company bulletin schema inspection
process_revenue_bulletins = [n for n in a_revenue_news_candidates if re.search(r'\('+re.escape(n['stock_id'])+r'\).*?\d{1,2}月營收',unicodedata.normalize('NFKC',n['title']))]
print('Single-company bulletin rows:',len(process_revenue_bulletins))
for n in [n for n in process_revenue_bulletins if n['published_at'].year==2025 and n['published_at'].month==10][:3]:
    print('Bulletin original sample:',{k:n[k] for k in ['news_id','stock_id','published_at','title','url']},'body:',(n['content'] or n['summary'] or '')[:1600])


In [ ]:
# Revenue resume: fixed two-sign rule from contemporaneous text
a_revenue_parsed = []
a_revenue_rejected = Counter()
for n in process_revenue_bulletins:
    yr,sid = n['published_at'].year,n['stock_id']
    if (yr,sid) not in a_announcement_documents:continue
    body = unicodedata.normalize('NFKC',n['content'] or n['summary'] or '')
    first = body.split('今年',1)[0]
    if sid+'-TW' not in first:
        a_revenue_rejected['stock_not_in_opening']+=1;continue
    month = re.search(r'公告(\d{4})年(\d{1,2})月營收',first)
    yoy = re.search(r'年增率\s*([+-]?\d+(?:\.\d+)?)%',first)
    mom = re.search(r'月增率\s*([+-]?\d+(?:\.\d+)?)%',first)
    if not (month and yoy and mom):
        a_revenue_rejected['opening_format']+=1;continue
    ym = f'{int(month[1]):04d}-{int(month[2]):02d}'
    rows = [r for r in a_announcement_documents[(yr,sid)]['revenues'] if r['year_month']==ym]
    yy,mm = float(yoy[1]),float(mom[1])
    if not rows or rows[0]['mom_pct'] is None or rows[0]['yoy_pct'] is None:
        a_revenue_rejected['no_db_crosscheck']+=1;continue
    if abs(float(rows[0]['mom_pct'])-mm)>.05 or abs(float(rows[0]['yoy_pct'])-yy)>.05:
        a_revenue_rejected['db_percentage_mismatch']+=1;continue
    a_revenue_parsed.append({'year':yr,'stock_id':sid,'year_month':ym,'published_at':n['published_at'],'yoy':yy,'mom':mm,'news_id':n['news_id'],'url':n['url'],'status':'原文雙率可核對；報導可用時間'})
a_revenue_lookup = {}
a_revenue_evidence_records = []
for e in a_announcement_evidence:
    cutoff = datetime.datetime.fromisoformat(e['cutoff'])
    matches = [r for r in a_revenue_parsed if r['year']==e['year'] and r['stock_id']==e['stock_id'] and r['published_at']<cutoff]
    record = {'year':e['year'],'stock_id':e['stock_id'],'cutoff':e['cutoff'],'score':0,'status':'未知：無核對來源'}
    if matches:
        latest = max(matches,key=lambda r:(r['year_month'],r['published_at']))
        age = (cutoff-latest['published_at']).total_seconds()/86400
        year,month = map(int,latest['year_month'].split('-'))
        lag = (cutoff.year-year)*12+cutoff.month-month
        record.update(latest)
        record['age_days']=age
        if age>45 or lag<1 or lag>2:record['status']='過舊或月份不適用：中性'
        else:record['score']=1 if latest['mom']>0 and latest['yoy']>0 else -1 if latest['mom']<0 and latest['yoy']<0 else 0
    t = di[cutoff.date()]
    a_revenue_lookup[(t,e['stock_id'])]=record
    a_revenue_evidence_records.append(record)
def a_revenue_allocation(t,selected,mode):
    if mode!='revenue_news':return a_announcement_allocation(t,selected,mode)
    multipliers=np.array([1+.25*a_revenue_lookup[(t,stock_ids[j])]['score'] for j in selected])
    weights=INVESTED*multipliers/multipliers.sum()
    assert len(selected)==25 and abs(weights.sum()-.9)<1e-12 and max(weights)<.1
    return weights
a_revenue_simulator_source=a_announcement_simulator_source.replace('def simulate_a_announcement(','def simulate_a_revenue(').replace('a_announcement_allocation(','a_revenue_allocation(')
exec(a_revenue_simulator_source,globals())
a_revenue_results={}
for yr,modes in a_announcement_results.items():
    control=simulate_a_revenue(modes['equal']['start'],'equal')
    assert np.array_equal(control['path'],modes['equal']['path']) and control['cost']==modes['equal']['cost']
    a_revenue_results[yr]=simulate_a_revenue(modes['equal']['start'],'revenue_news')
print('Original revenue extraction:',{'accepted':len(a_revenue_parsed),'rejected':dict(a_revenue_rejected)})
print('Decision evidence coverage:',dict(Counter(r['status'] for r in a_revenue_evidence_records)))
print('Initial 2025 evidence:',[{k:r.get(k) for k in ['stock_id','score','year_month','mom','yoy','published_at','status']} for r in a_revenue_evidence_records if r['cutoff'].startswith('2025-10-28')])
for yr,r in a_revenue_results.items():
    print('A revenue-news:',yr,{k:r[k] for k in ['invalid','return','mdd','cost','min_held','max_held','strict_cap_days','violations']},'paired difference pp:',100*(r['return']-a_announcement_results[yr]['equal']['return']) if not r['invalid'] else 'excluded')


In [ ]:
# Revenue resume: distinguish operating rule from self-report increment
def a_operating_ablation_allocation(t,selected,mode):
    if mode!='operating_no_self':return a_operating_allocation(t,selected,mode)
    multipliers=[]
    for j in selected:
        original=a_announcement_lookup[(t,stock_ids[j])]
        delta=original.get('op_margin_change_pp')
        score=0 if original['status']!='可核對' or delta is None else 1 if delta>=1 else -1 if delta<=-1 else 0
        multipliers.append(1+.25*score)
    weights=INVESTED*np.array(multipliers)/sum(multipliers)
    assert len(selected)==25 and abs(weights.sum()-.9)<1e-12 and max(weights)<.1
    return weights
a_operating_ablation_source=a_announcement_simulator_source.replace('def simulate_a_announcement(','def simulate_a_operating_ablation(').replace('a_announcement_allocation(','a_operating_ablation_allocation(')
exec(a_operating_ablation_source,globals())
a_operating_no_self_results={}
for yr,modes in a_announcement_results.items():
    a_operating_no_self_results[yr]=simulate_a_operating_ablation(modes['equal']['start'],'operating_no_self')
    r=a_operating_no_self_results[yr]
    print('A operating, no self-report:',yr,{k:r[k] for k in ['invalid','return','mdd','cost','min_held','max_held','strict_cap_days','violations']})
    if not r['invalid']:
        print('Same-window increments pp:',{'operating_rule_vs_original_financial':100*(r['return']-modes['financial']['return']),'self_reports_increment':100*(a_operating_results[yr]['return']-r['return'])})
assert all(r.get('published_at') is None or r['published_at']<datetime.datetime.fromisoformat(r['cutoff']) for r in a_revenue_evidence_records)
print('Revenue source cutoff verified for every decision.')


In [ ]:
# Text evidence: fixed two-company historical news retrieval
process_text_ast = copy.deepcopy(process_schema_ast)
process_text_ast.body[0].name = 'fetch_text_cases_readonly'
process_text_try = next(n for n in process_text_ast.body[0].body if isinstance(n,ast.Try))
process_text_try.body = process_text_try.body[:2]+ast.parse("""
with dbc.cursor() as c:
    c.execute("SELECT n.news_id,n.published_at,n.title,n.summary,n.content,n.url,n.source,ns.stock_id,ns.match_type FROM news n JOIN news_stock ns ON n.news_id=ns.news_id WHERE ns.stock_id IN ('6770','3105') AND n.published_at>='2025-10-01' AND n.published_at<'2025-11-29' ORDER BY n.published_at,n.news_id,ns.stock_id LIMIT 500")
    result=c.fetchall()
return result
""").body
ast.fix_missing_locations(process_text_ast)
exec(compile(process_text_ast,'<readonly_text_cases>','exec'),globals())
a_text_news_cases = fetch_text_cases_readonly()
a_text_focused_cases = [n for n in a_text_news_cases if re.search(r'法說|稼動|獲利|展望|毛利|營運|季報|財報',n['title']) and (('力積電' if n['stock_id']=='6770' else '穩懋') in n['title'])]
print('Fixed-case coverage:',{'joined_news':len(a_text_news_cases),'focused_company_news':len(a_text_focused_cases),'limit_reached':len(a_text_news_cases)>=500})
print('Focused source inventory:',[{k:n[k] for k in ['news_id','stock_id','published_at','title','url']} for n in a_text_focused_cases])


In [ ]:
# Text evidence: read fixed source bodies without return-based selection
for n in a_text_focused_cases:
    print('Case source reviewed:',n['news_id'],n['stock_id'],str(n['published_at']),n['title'],'body characters:',len(n['content'] or n['summary'] or ''))
print('Existing case holdings:',[(r['year'],str(r['decision_date']),[sid for sid in r['stock_ids'] if sid in ['6770','3105']]) for r in a_announcement_selection_inventory])


In [ ]:
# Text evidence: quarterly accounting consistency check
for sid in ['6770','3105']:
    for ann in a_announcement_documents[(2025,sid)]['announcements']:
        q=prestart_quarter(ann,2025)
        if q in [1,2]:
            vals=prestart_original_amounts(ann)
            if vals is not None:print('Formal prior quarter:',sid,q,str(prestart_stamp(ann)),ann['ann_id'],vals)


In [ ]:
# Text evidence: official WIN release in kernel memory
import importlib.util
import io
import requests
print('PDF parsers:',{name:bool(importlib.util.find_spec(name)) for name in ['pypdf','fitz']})
a_text_official_win = {}
for label,url in [('release','https://www.winfoundry.com/en-US/Base/DownLoadFile/516?TargetTable=QuarterlyAttachment&filename=3Q+2025+Press+Release.pdf')]:
    try:
        response=requests.get(url,timeout=25)
        response.raise_for_status()
        if not response.content.startswith(b'%PDF'):raise ValueError('Not a PDF')
        if importlib.util.find_spec('pypdf'):
            import pypdf
            pdf=pypdf.PdfReader(io.BytesIO(response.content))
            pages=[page.extract_text() or '' for page in pdf.pages]
        else:
            import fitz
            pdf=fitz.open(stream=response.content,filetype='pdf')
            pages=[page.get_text() for page in pdf]
        a_text_official_win[label]={'url':url,'bytes':response.content,'pages':pages}
        print('Official PDF retrieved:',label,len(pages),'pages',len(response.content),'bytes')
        print('Official release text:', '\n'.join(pages)[:14000])
    except Exception as error:
        print('Official PDF retrieval failed:',label,type(error).__name__)


In [ ]:
# Text evidence: final verified TLS attempt for official release
import urllib.request
print('Additional PDF parsers:',{name:bool(importlib.util.find_spec(name)) for name in ['pdfplumber','PyPDF2','pypdfium2']})
try:
    official_url='https://www.winfoundry.com/en-US/Base/DownLoadFile/516?TargetTable=QuarterlyAttachment&filename=3Q+2025+Press+Release.pdf'
    request=urllib.request.Request(official_url,headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(request,context=ssl.create_default_context(),timeout=20) as response:
        official_bytes=response.read()
    assert official_bytes.startswith(b'%PDF')
    a_text_official_win['release_bytes']={'url':official_url,'bytes':official_bytes}
    print('Verified TLS download succeeded:',len(official_bytes),'bytes')
except Exception as error:
    print('Final verified TLS attempt failed:',type(error).__name__)


In [ ]:
# Turnover branch: inspect retained broad-window contracts
print('Retained weight simulator:',weight_simulator_source)
print('Weight allocation:',inspect.getsource(weight_experiment_allocation))
print('Result mode names:',list(weight_experiment_results))
print('Window fields:',list(weight_experiment_results[next(iter(weight_experiment_results))][0]))
print('Block CI contract:',inspect.getsource(block_ci))


In [ ]:
# Turnover branch: fixed small-adjustment tolerance experiment
def a_turnover_allocation(t,selected,allocation):
    if allocation=='equal':return np.full(COUNT,INVESTED/COUNT)
    assert allocation=='mild'
    multipliers=np.linspace(1.25,.75,COUNT)
    return INVESTED*multipliers/multipliers.sum()
a_turnover_source=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_a_turnover(start,allocation="mild",horizon=HORIZON,tolerance=.005):').replace('weight_experiment_allocation(','a_turnover_allocation(')
a_turnover_source=a_turnover_source.replace('    strict_cap_days=0;','    skipped_orders=0\n    strict_cap_days=0;',1)
settle_line='            cash,shares,traded,cost,missing=settle(cash,shares,target,execution[t],good_exec[t])'
skip_lines="""            if offset>0 and tolerance>0:
                kept=(shares>0)&(target>0)
                small=kept&(np.abs(target-shares)*np.nan_to_num(close[t-1])<=tolerance*previous_nav)
                skipped_orders+=int(np.count_nonzero(small&(target!=shares)))
                target[small]=shares[small]
"""
assert a_turnover_source.count(settle_line)==1
a_turnover_source=a_turnover_source.replace(settle_line,skip_lines+settle_line)
a_turnover_source=a_turnover_source.replace("'max_held':max_held}","'max_held':max_held,'skipped_orders':skipped_orders}")
exec(a_turnover_source,globals())
for pos in [0,325,649]:
    control=simulate_a_turnover(int(window_starts[pos]),tolerance=0)
    assert np.array_equal(control['path'],weight_experiment_results['mild'][pos]['path']) and control['cost']==weight_experiment_results['mild'][pos]['cost']
a_turnover_results=[simulate_a_turnover(int(start)) for start in window_starts]
a_turnover_valid=np.array([not r['invalid'] and r['violations']==0 and r['strict_cap_days']==0 and r['min_held']==r['max_held']==25 for r in a_turnover_results])
a_turnover_common=weight_experiment_valid&a_turnover_valid
a_turnover_metrics={k:np.array([r[k] for r in a_turnover_results]) for k in ['return','mdd','cost','turnover','skipped_orders']}
a_turnover_difference=np.where(a_turnover_common,a_turnover_metrics['return']-np.array([r['return'] for r in weight_experiment_results['equal']]),np.nan)
a_turnover_vs_mild=np.where(a_turnover_common,a_turnover_metrics['return']-np.array([r['return'] for r in weight_experiment_results['mild']]),np.nan)
print('Tolerance eligibility:',{'original_common':int(weight_experiment_valid.sum()),'new_common':int(a_turnover_common.sum()),'lost_original_windows':int((weight_experiment_valid&~a_turnover_valid).sum())})
for mode,results in [('equal',weight_experiment_results['equal']),('mild',weight_experiment_results['mild']),('mild_tolerance',a_turnover_results)]:
    rr=np.array([r['return'] for r in results])[a_turnover_common]
    print('Common-window metrics:',mode,{'median_return':float(np.median(rr)),'P10':float(np.quantile(rr,.1)),'median_mdd':float(np.median([r['mdd'] for r,m in zip(results,a_turnover_common) if m])),'mean_cost':float(np.mean([r['cost'] for r,m in zip(results,a_turnover_common) if m])),'mean_turnover':float(np.mean([r['turnover'] for r,m in zip(results,a_turnover_common) if m]))})
a_turnover_ci={}
for base,series in [('equal',a_turnover_difference),('mild',a_turnover_vs_mild)]:
    a_turnover_ci[base]={block:block_ci(series,block) for block in [24,48]}
    print('Paired comparison:',base,'mean pp',100*float(np.nanmean(series)),'win rate',float(np.mean(series[a_turnover_common]>0)),'block CI pp',{b:(100*v).tolist() for b,v in a_turnover_ci[base].items()})
print('Turnover yearly paired means pp:',{yr:100*float(np.nanmean(np.where(np.array([dates[int(s)].year==yr for s in window_starts]),a_turnover_difference,np.nan))) for yr in [2024,2025,2026]})


In [ ]:
# Sector heartbeat: persistent state and read-only classification schema
print('Retained branch objects:',{k:type(globals()[k]).__name__ for k in ['process_schema_ast','a_turnover_results','weight_experiment_results','weight_experiment_valid'] if k in globals()})
process_sector_ast=copy.deepcopy(process_schema_ast)
process_sector_ast.body[0].name='fetch_sector_metadata_readonly'
process_sector_try=next(n for n in process_sector_ast.body[0].body if isinstance(n,ast.Try))
process_sector_try.body=process_sector_try.body[:2]+ast.parse("""
with dbc.cursor() as c:
    c.execute("SELECT TABLE_NAME,COLUMN_NAME,DATA_TYPE FROM information_schema.COLUMNS WHERE TABLE_SCHEMA=DATABASE() AND (TABLE_NAME='stocks' OR TABLE_NAME LIKE '%industry%' OR TABLE_NAME LIKE '%sector%' OR TABLE_NAME LIKE '%history%') ORDER BY TABLE_NAME,ORDINAL_POSITION")
    columns=c.fetchall()
    c.execute("SELECT stock_id,name,industry_name,updated_at FROM stocks ORDER BY stock_id")
    stocks=c.fetchall()
return {'columns':columns,'stocks':stocks}
""").body
ast.fix_missing_locations(process_sector_ast)
exec(compile(process_sector_ast,'<readonly_sector_metadata>','exec'),globals())
a_sector_metadata=fetch_sector_metadata_readonly()
a_sector_map={r['stock_id']:r['industry_name'] or '未分類' for r in a_sector_metadata['stocks']}
print('Sector schema:',a_sector_metadata['columns'])
print('Classification coverage:',{'pool':len(stock_ids),'known':sum(s in a_sector_map for s in stock_ids),'unknown':sum(s not in a_sector_map for s in stock_ids),'snapshot_update_range':(str(min(r['updated_at'] for r in a_sector_metadata['stocks'] if r['updated_at'])),str(max(r['updated_at'] for r in a_sector_metadata['stocks'] if r['updated_at'])))})


In [ ]:
# Sector heartbeat: diagnostic exposures using present-day classification
a_sector_labels=sorted({a_sector_map[s] for s in stock_ids})
a_sector_matrix=np.array([[a_sector_map[s]==label for s in stock_ids] for label in a_sector_labels],dtype=float)
a_sector_diag_source=a_turnover_source.replace('def simulate_a_turnover(','def simulate_a_sector_diag(')
a_sector_diag_source=a_sector_diag_source.replace('        caps=np.array(',"""        sector_weights=a_sector_matrix@weights
        a_sector_trace.append((float(sector_weights.max()),float(sector_weights[a_sector_labels.index('半導體業')]) if '半導體業' in a_sector_labels else np.nan,float(np.sum(weights**2))))
        caps=np.array(""",1)
exec(a_sector_diag_source,globals())
a_sector_diagnostics={mode:[] for mode in ['equal','mild','mild_tolerance']}
for pos in np.flatnonzero(a_turnover_common):
    for mode in a_sector_diagnostics:
        a_sector_trace=[]
        allocation='equal' if mode=='equal' else 'mild'
        tolerance=.005 if mode=='mild_tolerance' else 0
        result=simulate_a_sector_diag(int(window_starts[pos]),allocation=allocation,tolerance=tolerance)
        original=a_turnover_results[pos] if mode=='mild_tolerance' else weight_experiment_results[mode][pos]
        assert np.array_equal(result['path'],original['path']) and result['cost']==original['cost']
        trace=np.array(a_sector_trace)
        a_sector_diagnostics[mode].append({'position':int(pos),'start_date':str(dates[int(window_starts[pos])]),'max_sector_peak':float(trace[:,0].max()),'max_sector_mean':float(trace[:,0].mean()),'semiconductor_peak':float(trace[:,1].max()),'stock_hhi_mean':float(trace[:,2].mean())})
for mode,rows in a_sector_diagnostics.items():
    peak=np.array([r['max_sector_peak'] for r in rows])
    print('Current-classification diagnostic:',mode,{'windows':len(rows),'median_max_sector_peak':float(np.median(peak)),'P90_max_sector_peak':float(np.quantile(peak,.9)),'max_sector_peak':float(peak.max()),'peak_gt_50pct_share':float(np.mean(peak>.5)),'median_stock_HHI':float(np.median([r['stock_hhi_mean'] for r in rows]))})
print('2025 exact-season initial target exposure:')
selection=next(r for r in a_announcement_selection_inventory if str(r['decision_date'])=='2025-10-28')
selected=np.array([si[s] for s in selection['stock_ids']])
for mode in ['equal','mild']:
    weights=a_turnover_allocation(selection['t'],selected,mode)
    grouped={label:float(sum(w for sid,w in zip(selection['stock_ids'],weights) if a_sector_map[sid]==label)) for label in a_sector_labels}
    print(mode,{k:round(v,6) for k,v in grouped.items() if v>0})
print('This is a 2026-classification retrospective diagnostic, not a point-in-time trading signal.')


In [ ]:
# Sector heartbeat: fixed numeric combination and cost ablation
def a_combined_allocation(t,selected,allocation):
    assert allocation=='combined'
    momentum=a_turnover_allocation(t,selected,'mild')
    financial=a_operating_allocation(t,selected,'operating_only')
    revenue=a_revenue_allocation(t,selected,'revenue_news')
    weights=.5*momentum+.25*financial+.25*revenue
    assert len(selected)==25 and abs(weights.sum()-.9)<1e-12 and np.all(weights>0) and np.max(weights)<.1
    return weights
a_combined_source=a_turnover_source.replace('def simulate_a_turnover(','def simulate_a_combined(').replace('a_turnover_allocation(','a_combined_allocation(')
exec(a_combined_source,globals())
a_combined_results={}
for yr,modes in a_announcement_results.items():
    a_combined_results[yr]={}
    for label,tolerance in [('combined',0),('combined_tolerance',.005)]:
        r=simulate_a_combined(modes['equal']['start'],allocation='combined',tolerance=tolerance)
        a_combined_results[yr][label]=r
        print('Fixed combination:',yr,label,{k:r[k] for k in ['invalid','return','mdd','cost','min_held','max_held','strict_cap_days','violations','skipped_orders']},'paired vs A pp',100*(r['return']-modes['equal']['return']) if not r['invalid'] else 'excluded')
print('Case risk exposure is diagnostic only; no historical sector cap has been applied.')


In [ ]:
# Sector heartbeat: fixed independent short-reversal control
a_reversal_scores=np.full_like(scores['A'],np.nan)
for t in range(4,len(dates)):
    eligible=np.isfinite(scores['A'][t])&np.all(np.isfinite(signal_returns[t-4:t+1]),axis=0)
    rr=np.prod(1+signal_returns[t-4:t+1],axis=0)-1
    a_reversal_scores[t,eligible]=-rr[eligible]
a_reversal_source=weight_simulator_source.replace('def simulate_weights(','def simulate_reversal_control(').replace('strategy="A"','strategy="REV5"').replace('rank=scores[strategy][t-1]','rank=a_reversal_scores[t-1]').replace('weight_experiment_allocation(','a_turnover_allocation(')
exec(a_reversal_source,globals())
a_reversal_results=[simulate_reversal_control(int(s),'mild') for s in window_starts]
a_reversal_valid=np.array([not r['invalid'] and r['violations']==0 and r['strict_cap_days']==0 and r['min_held']==r['max_held']==25 for r in a_reversal_results])
a_reversal_common=weight_experiment_valid&a_reversal_valid
a_reversal_series=np.where(a_reversal_common,np.array([r['return'] for r in a_reversal_results])-np.array([r['return'] for r in weight_experiment_results['equal']]),np.nan)
a_reversal_ci={b:block_ci(a_reversal_series,b) for b in [24,48]}
print('Reversal shared eligibility:',{'original_A_common':int(weight_experiment_valid.sum()),'reversal_valid':int(a_reversal_valid.sum()),'paired_common':int(a_reversal_common.sum()),'lost_original_A_windows':int((weight_experiment_valid&~a_reversal_valid).sum())})
for label,results in [('A_equal',weight_experiment_results['equal']),('REV5_non_equal',a_reversal_results)]:
    rr=np.array([r['return'] for r in results])[a_reversal_common]
    print('Reversal shared metrics:',label,{'median_return':float(np.median(rr)),'P10':float(np.quantile(rr,.1)),'median_mdd':float(np.median([r['mdd'] for r,m in zip(results,a_reversal_common) if m])),'mean_cost':float(np.mean([r['cost'] for r,m in zip(results,a_reversal_common) if m]))})
print('Reversal paired vs A:',{'mean_pp':100*float(np.nanmean(a_reversal_series)),'win_rate':float(np.mean(a_reversal_series[a_reversal_common]>0)),'CI_pp':{b:(100*v).tolist() for b,v in a_reversal_ci.items()}})


In [ ]:
# Final validation: finite family correction and purged time sensitivity
print('Retained validation objects:',{k:type(globals()[k]).__name__ for k in ['a_turnover_results','a_reversal_results','weight_experiment_results'] if k in globals()})
a_validation_candidates={'mild':weight_experiment_results['mild'],'invvol':weight_experiment_results['invvol'],'mild_tolerance':a_turnover_results,'reversal':a_reversal_results}
a_validation_masks={'mild':weight_experiment_valid.copy(),'invvol':weight_experiment_valid.copy(),'mild_tolerance':a_turnover_common.copy(),'reversal':a_reversal_common.copy()}
a_validation_base=np.array([r['return'] for r in weight_experiment_results['equal']])
a_validation_differences={name:np.where(a_validation_masks[name],np.array([r['return'] for r in results])-a_validation_base,np.nan) for name,results in a_validation_candidates.items()}
def a_family_block_intervals(series_by_name,block,n_boot=5000):
    rng=np.random.default_rng(SEED+1000+block)
    n=len(window_starts)
    draws={name:[] for name in series_by_name}
    for _ in range(n_boot):
        starts=rng.integers(0,n-block+1,size=int(np.ceil(n/block)))
        indices=np.concatenate([np.arange(s,s+block) for s in starts])[:n]
        for name,series in series_by_name.items():
            sample=series[indices]
            if np.any(np.isfinite(sample)):draws[name].append(float(np.nanmean(sample)))
    alpha=.05/len(series_by_name)
    return {name:np.quantile(values,[alpha/2,1-alpha/2]) for name,values in draws.items()}
a_validation_family_ci={block:a_family_block_intervals(a_validation_differences,block) for block in [24,48]}
cut=datetime.date(2026,1,1)
first_2026=next(i for i,d in enumerate(dates) if d>=cut)
late_start=dates[first_2026+24]
a_validation_early=np.array([dates[r['end']]<cut for r in weight_experiment_results['equal']])
a_validation_late=np.array([dates[int(start)]>=late_start for start in window_starts])
a_validation_summary={}
for name,results in a_validation_candidates.items():
    mask=a_validation_masks[name]
    differences=a_validation_differences[name]
    candidate_returns=np.array([r['return'] for r in results])[mask]
    base_returns=a_validation_base[mask]
    mdds=np.array([r['mdd'] for r in results])[mask]
    base_mdds=np.array([r['mdd'] for r in weight_experiment_results['equal']])[mask]
    disjoint=0;last_end=-1
    for pos in np.flatnonzero(mask):
        if int(window_starts[pos])>last_end:
            disjoint+=1;last_end=int(results[pos]['end'])
    segments={}
    for label,segment in [('early',a_validation_early),('late_seen',a_validation_late)]:
        selected=mask&segment
        segments[label]={'n':int(selected.sum()),'mean_pp':100*float(np.nanmean(differences[selected])) if selected.any() else None}
    p10_delta=float(np.quantile(candidate_returns,.1)-np.quantile(base_returns,.1))
    mdd_delta=float(np.median(mdds)-np.median(base_mdds))
    statistical_pass=all(a_validation_family_ci[b][name][0]>0 for b in [24,48])
    risk_pass=p10_delta>=-.01 and mdd_delta>=-.01
    record={'paired_n':int(mask.sum()),'greedy_nonoverlapping_windows':disjoint,'mean_pp':100*float(np.nanmean(differences)),'family_CI_pp':{b:(100*a_validation_family_ci[b][name]).tolist() for b in [24,48]},'P10_delta_pp':100*p10_delta,'median_MDD_delta_pp':100*mdd_delta,'statistical_pass':bool(statistical_pass),'risk_pass':bool(risk_pass),'segments':segments,'complete_official_rules_verified':False}
    a_validation_summary[name]=record
    print('Final validation:',name,record)
print('Sensitivity date boundary:',str(late_start),'after 24-session gap; no unseen holdout claim.')


In [ ]:
# Final selection: freeze provisional non-equal A configuration
a_research_selection={
    'validated_improvement':None,
    'provisional_non_equal_method':'A_mild_tolerance_005',
    'status':'暫定研究配置；尚未證實優於等權 A，未完成官方全部限制查核',
    'selection_signal':'平均 20／60 日調整後報酬百分位，原 A 資料可用性條件',
    'stocks':25,'rebalance_sessions':5,'horizon_sessions':24,'invested_target':.9,
    'rank_multipliers':np.linspace(1.25,.75,25).tolist(),
    'initial_target_weights':(.9*np.linspace(1.25,.75,25)/25).tolist(),
    'skip_retained_adjustment_NAV_threshold':.005,
    'lot_size':1000,'buy_fee':FEE,'sell_fee':FEE,'sell_tax':TAX,
    'execution':'same-day VWAP; prior close only for target shares',
    'financial_news_and_sector_extra_weight':False,
    'reason':'保留使用者 A 與非等權需求；447 共同窗口資料較完整，較原溫和加權省成本且規則簡單，風險門檻在樣本內通過；統計家族門檻未通過，所以只列暫定。',
    'declined_for_now':['inverse_volatility_mix','five_day_reversal','financial_or_revenue_only_as_confirmed_winner','fixed_announcement_combination_as_confirmed_winner'],
    'next_required_before_live_use':['完整官方限制與整批作廢驗證','修復歷史成交與公司行動資料並重建歷史股票池','新而未見資料的事先固定規則驗證','補公告時序、修訂版本、法說原件與產業歷史分類']
}
assert sum(a_research_selection['initial_target_weights'])==.9 or abs(sum(a_research_selection['initial_target_weights'])-.9)<1e-12
assert not any(record['statistical_pass'] for record in a_validation_summary.values())
print('Research selection:',{k:v for k,v in a_research_selection.items() if k not in ['rank_multipliers','initial_target_weights']})
print('Initial rank weight range:',{'highest_rank':a_research_selection['initial_target_weights'][0],'lowest_rank':a_research_selection['initial_target_weights'][-1],'total':sum(a_research_selection['initial_target_weights'])})


In [ ]:
# Queue revision: retained kernel inventory; no experiment rerun
import sys
print('Kernel executable:',sys.executable)
queue_inventory_names=['dates','stock_ids','scores','weight_experiment_results','a_turnover_results','a_reversal_results','a_combined_results','a_announcement_documents']
print('Retained metadata:',{k:{'type':type(globals()[k]).__name__,'length':len(globals()[k]) if hasattr(globals()[k],'__len__') else None} for k in queue_inventory_names if k in globals()})
print('Score keys:',list(scores))
print('Weight result keys:',list(weight_experiment_results))
print('Recent safe input markers:',[(i,s.splitlines()[0][:120]) for _,i,s in get_ipython().history_manager.get_range(output=False) if s.startswith('# Final') or s.startswith('# Queue revision')][-6:])


## 純法人選股：第一個新增對照

前20交易日（外資＋投信）淨買股數／成交股數，由高到低選25檔；不混入動能分數，等權投資90%，每五日調整。保持原A的資料完整性／公司行動门檻、24日窗口、整千股與費用；這些資格門檻不是動能排名。原A輸出直接沿用。只做單一固定候選，不搜尋法人種類或窗長。缺法人值不可補零，0代表有效的淨買賣合計為零。原股票池與官方執行限制仍未完全排除。

In [ ]:
# Pure flow v1: retained kernel and source contracts
%whos ndarray
import sys,inspect,hashlib,json
print('Kernel executable:',sys.executable)
print('Required shapes:',{k:globals()[k].shape for k in ['flow','volume','signal_returns','bad_action','close']})
print('Retained A result windows:',len(weight_experiment_results['equal']))
print('Flow missing / genuine zero:',int(np.isnan(flow).sum()),int(np.sum(flow==0)))
print('Previous pure flow cached:', 'pure_flow_results_v1' in globals())
print('Safe recent markers:',[(i,s.splitlines()[0][:100]) for _,i,s in get_ipython().history_manager.get_range(output=False) if s.startswith('# Queue revision') or s.startswith('# Final')][-4:])
assert sys.executable.lower()==r'C:\Python314\python.exe'.lower()
assert COUNT==25 and HORIZON==24 and REBALANCE==5 and LOT==1000
assert np.isclose(FEE,.001425) and np.isclose(TAX,.003) and np.isclose(INVESTED,.9)


In [ ]:
# Pure flow v1: single fixed signal and exact retained accounting engine
pure_flow_definition_v1={'experiment_key':'institutional_20d_select25_equal','sessions':20,'stock_count':COUNT,'invested':INVESTED,'rebalance':REBALANCE,'horizon':HORIZON,'fee':FEE,'tax':TAX,'lot':LOT,'eligibility':'same raw 60-session quality gates as A; no momentum ranking','data_snapshot':snapshot_sha256,'engine_sha256':hashlib.sha256(weight_simulator_source.encode()).hexdigest()}
if 'pure_flow_results_v1' not in globals():
    pure_flow_scores_v1=np.full(shape,np.nan)
    for t in range(60,len(dates)):
        rr=signal_returns[t-59:t+1]
        ff=flow[t-19:t+1]; vv=volume[t-19:t+1]
        eligible=np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(ff)&np.isfinite(vv)&(vv>0),axis=0)&(close[t]>0)
        eligible &= ~np.any(bad_action[t-59:t+1],axis=0)
        assert np.array_equal(eligible,np.isfinite(scores['A'][t]))
        ratio=np.sum(ff,axis=0)/np.sum(vv,axis=0)
        pure_flow_scores_v1[t]=percentile(ratio,eligible)
    scores['PURE_FLOW']=pure_flow_scores_v1
    pure_flow_engine_v1=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_pure_flow_v1(start,allocation="equal",horizon=HORIZON):').replace('strategy="A"','strategy="PURE_FLOW"')
    pure_flow_allocation_line='weight_experiment_allocation(t,selected,allocation)'
    assert pure_flow_engine_v1.count(pure_flow_allocation_line)==1
    pure_flow_engine_v1=pure_flow_engine_v1.replace(pure_flow_allocation_line,'np.full(COUNT,INVESTED/COUNT)')
    exec(pure_flow_engine_v1)
    pure_flow_results_v1=[simulate_pure_flow_v1(int(s)) for s in window_starts]
    pure_flow_cache_definition_v1=dict(pure_flow_definition_v1)
else:
    assert pure_flow_cache_definition_v1==pure_flow_definition_v1
pure_flow_metrics_v1={f:np.array([r[f] for r in pure_flow_results_v1]) for f in ['return','mdd','cost','turnover','violations','strict_cap_days','max_actual_weight','min_held','max_held']}
pure_flow_valid_v1=np.array([not r['invalid'] for r in pure_flow_results_v1])
pure_flow_a_valid_v1=np.array([not r['invalid'] for r in weight_experiment_results['equal']])
pure_flow_pair_v1=pure_flow_valid_v1 & pure_flow_a_valid_v1
pure_flow_excess_v1=pure_flow_metrics_v1['return']-weight_experiment_metrics['equal']['return']
pure_flow_summary_v1=[]
for period,period_mask in [('all',np.ones(len(window_starts),dtype=bool))]+[(str(y),np.array([dates[s].year==y and dates[s+HORIZON-1].year==y for s in window_starts])) for y in sorted({dates[s].year for s in window_starts})]:
    mask=pure_flow_pair_v1 & period_mask
    count=int(mask.sum())
    if not count:continue
    pure_flow_summary_v1.append({'period':period,'paired_windows':count,'flow_median_pct':float(np.median(pure_flow_metrics_v1['return'][mask])*100),'A_median_pct':float(np.median(weight_experiment_metrics['equal']['return'][mask])*100),'paired_mean_excess_pp':float(np.mean(pure_flow_excess_v1[mask])*100),'beat_A_pct':float(np.mean(pure_flow_excess_v1[mask]>1e-12)*100),'flow_cost_pct':float(np.mean(pure_flow_metrics_v1['cost'][mask])*100),'A_cost_pct':float(np.mean(weight_experiment_metrics['equal']['cost'][mask])*100),'flow_P10_pct':float(np.quantile(pure_flow_metrics_v1['return'][mask],.1)*100),'flow_mdd_median_pct':float(np.median(pure_flow_metrics_v1['mdd'][mask])*100)})
assert len(pure_flow_results_v1)==len(window_starts)
print('Window audit:',{'total':len(window_starts),'flow_valid':int(pure_flow_valid_v1.sum()),'A_valid':int(pure_flow_a_valid_v1.sum()),'paired':int(pure_flow_pair_v1.sum()),'excluded_A_valid':int(np.sum(pure_flow_a_valid_v1&~pure_flow_valid_v1))})
print('Paired results:',json.dumps(pure_flow_summary_v1,ensure_ascii=False))
print('Valid flow rule diagnostics:',{'violation_days':int(pure_flow_metrics_v1['violations'][pure_flow_valid_v1].sum()),'max_weight':float(pure_flow_metrics_v1['max_actual_weight'][pure_flow_valid_v1].max()) if pure_flow_valid_v1.any() else None})
print('Exploratory only: overlapping windows, historical universe bias and incomplete official constraints; expanded family inference pending.')
print('PURE_FLOW_SAVED_RESULT_READY')


## 純低波動選股

前20交易日調整後日報酬標準差由低到高選25檔，不混動能；等權投入90%。沿用原交易與資料門檻，揭露暖機／完整性條件。

單一事先定義候選；保留原資料品質門檻、費用與成交引擎，不重跑A控制。

In [ ]:
# Numeric control pure_lowvol: retained audit
import sys
print('Kernel executable:',sys.executable)
print('Retained metadata:',{k:(type(globals()[k]).__name__,getattr(globals()[k],'shape',None)) for k in ['flow','signal_returns','scores','weight_experiment_results']})
print('Recent execution markers:',[(i,s.splitlines()[0][:90]) for _,i,s in get_ipython().history_manager.get_range(output=False) if s.startswith('# Pure flow') or s.startswith('# Numeric control')][-5:])
assert COUNT==25 and HORIZON==24 and REBALANCE==5 and LOT==1000
assert np.isclose(FEE,.001425) and np.isclose(TAX,.003)


In [ ]:
# Numeric control pure_lowvol: fixed experiment
import hashlib,json
numeric_control_summaries_v1=globals().get('numeric_control_summaries_v1',{})
def summarize_numeric_control_v1(name,results):
    metrics={f:np.array([r[f] for r in results]) for f in ['return','mdd','cost','turnover','violations','strict_cap_days','max_actual_weight']}
    valid=np.array([not r['invalid'] for r in results])
    base_valid=np.array([not r['invalid'] for r in weight_experiment_results['equal']])
    pair=valid&base_valid
    excess=metrics['return']-weight_experiment_metrics['equal']['return']
    rows=[]
    periods=[('all',np.ones(len(window_starts),dtype=bool))]+[(str(y),np.array([dates[s].year==y and dates[s+HORIZON-1].year==y for s in window_starts])) for y in sorted({dates[s].year for s in window_starts})]
    for title,pm in periods:
        mask=pair&pm
        if not mask.any():continue
        rows.append({'period':title,'paired_windows':int(mask.sum()),'candidate_median_pct':float(np.median(metrics['return'][mask])*100),'A_median_pct':float(np.median(weight_experiment_metrics['equal']['return'][mask])*100),'paired_mean_excess_pp':float(np.mean(excess[mask])*100),'beat_A_pct':float(np.mean(excess[mask]>1e-12)*100),'candidate_cost_pct':float(np.mean(metrics['cost'][mask])*100),'A_cost_pct':float(np.mean(weight_experiment_metrics['equal']['cost'][mask])*100),'candidate_P10_pct':float(np.quantile(metrics['return'][mask],.1)*100),'A_P10_pct':float(np.quantile(weight_experiment_metrics['equal']['return'][mask],.1)*100),'candidate_mdd_median_pct':float(np.median(metrics['mdd'][mask])*100),'A_mdd_median_pct':float(np.median(weight_experiment_metrics['equal']['mdd'][mask])*100)})
    summary={'name':name,'total':len(results),'candidate_valid':int(valid.sum()),'paired':int(pair.sum()),'excluded_A_valid':int(np.sum(base_valid&~valid)),'rows':rows,'valid_violation_days':int(metrics['violations'][valid].sum())}
    numeric_control_summaries_v1[name]=summary
    print(json.dumps(summary,ensure_ascii=False))
    return metrics,valid,pair,excess
pure_lowvol_definition_v1={'key':'lowvol_20d_select25_equal','data':snapshot_sha256,'engine':hashlib.sha256(weight_simulator_source.encode()).hexdigest(),'lookback':20,'eligibility':'same raw A quality gates, no momentum rank'}
if 'pure_lowvol_results_v1' not in globals():
    pure_lowvol_scores_v1=np.full(shape,np.nan)
    for t in range(60,len(dates)):
        rr=signal_returns[t-59:t+1]; ff=flow[t-19:t+1]; vv=volume[t-19:t+1]
        eligible=np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(ff)&np.isfinite(vv)&(vv>0),axis=0)&(close[t]>0)
        eligible &= ~np.any(bad_action[t-59:t+1],axis=0)
        assert np.array_equal(eligible,np.isfinite(scores['A'][t]))
        pure_lowvol_scores_v1[t]=percentile(-np.std(rr[-20:],axis=0),eligible)
    scores['PURE_LOWVOL']=pure_lowvol_scores_v1
    src=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_pure_lowvol_v1(start,allocation="equal",horizon=HORIZON):').replace('strategy="A"','strategy="PURE_LOWVOL"')
    line='weight_experiment_allocation(t,selected,allocation)'
    assert src.count(line)==1
    src=src.replace(line,'np.full(COUNT,INVESTED/COUNT)')
    exec(src)
    pure_lowvol_results_v1=[simulate_pure_lowvol_v1(int(s)) for s in window_starts]
    pure_lowvol_cached_definition_v1=dict(pure_lowvol_definition_v1)
else:assert pure_lowvol_cached_definition_v1==pure_lowvol_definition_v1
pure_lowvol_metrics_v1,pure_lowvol_valid_v1,pure_lowvol_pair_v1,pure_lowvol_excess_v1=summarize_numeric_control_v1('pure_lowvol',pure_lowvol_results_v1)
print('No family inference yet; overlapping windows and original data limitations remain.')
print('NUMERIC_CONTROL_READY pure_lowvol')


## 使用者同意後的kernel恢復

原kernel已結束。修正啟動工具為獨立kernel後，唯讀取回原截止2026-10-12資料，先比對原快照SHA256。沒有保存原始行情或逐日帳戶的外部checkpoint，因此必須重建必要矩陣與A控制帳戶；這是恢復計算狀態，不是新增實驗。與已保存HTML逐窗口A報酬／成本／有效性精確核對，資料快照不同則停止，不混用舊結果。恢復cells須先於待跑的純低波動cells48–49執行；原Notebook輸出保留。

In [ ]:
# Authorized kernel recovery v1: environment and read-only original snapshot
%whos
import sys,numpy as np,pymysql,json,re,hashlib,datetime,inspect
from research_layout import ResearchPath as Path
print('Authorized new kernel environment:',sys.executable,np.__version__)
assert sys.executable.lower()==r'C:\Python314\python.exe'.lower()
recovery_root=Path(r'C:\Users\weiha\Desktop\文件\03_課程資料\115-1\資料探勘與應用\B_Final Project (0920~1221)\競賽資料')
recovery_base_nb=json.loads((recovery_root/'動能基準研究_24日回測_2026-10-13.ipynb').read_text(encoding='utf-8'))
recovery_weight_nb=json.loads((recovery_root/'動能基準研究_25檔權重比較_2026-10-14.ipynb').read_text(encoding='utf-8'))
def recovery_source(nb,index):return ''.join(nb['cells'][index]['source'])
exec(recovery_source(recovery_base_nb,3))
snapshot_sha256=hashlib.sha256(json.dumps(raw_data,default=str,ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
assert snapshot_sha256=='1b826aa999a02664f43ff9c48a77259881901d8b8399c6ee85ebc275af7ea0e2','Data changed; cannot mix with preserved old outputs'
print('Original data snapshot exactly matched:',snapshot_sha256)


In [ ]:
# Authorized kernel recovery v1: required matrices, functions and verified A control
assert snapshot_sha256=='1b826aa999a02664f43ff9c48a77259881901d8b8399c6ee85ebc275af7ea0e2'
exec(recovery_source(recovery_base_nb,5))
exec(recovery_source(recovery_base_nb,8))
recovery_engine=recovery_source(recovery_base_nb,10).split('# Independent accounting checks:')[0]
recovery_engine_result=get_ipython().run_cell(recovery_engine,store_history=True)
assert recovery_engine_result.success
recovery_weights=recovery_source(recovery_weight_nb,3).split('weight_experiment_results=')[0]
recovery_weights_result=get_ipython().run_cell(recovery_weights,store_history=True)
assert recovery_weights_result.success
window_starts=np.arange(61,len(dates)-HORIZON+1)
weight_experiment_results={'equal':[simulate_weights(int(s),'equal') for s in window_starts]}
weight_experiment_metrics={'equal':{f:np.array([r[f] for r in weight_experiment_results['equal']]) for f in ['return','mdd','cost','turnover','violations','strict_cap_days','max_actual_weight','min_held','max_held']}}
weight_experiment_valid=np.array([not r['invalid'] for r in weight_experiment_results['equal']])
recovery_html=(recovery_root/'動能基準研究_互動箱形圖_2026-10-13.html').read_text(encoding='utf-8')
recovery_payload=json.loads(re.search(r'<script id="payload" type="application/json">(.*?)</script>',recovery_html,re.S).group(1))
recovery_A_index=next(i for i,x in enumerate(recovery_payload['strategies']) if x['key']=='A')
assert len(recovery_payload['windows'])==len(window_starts)==650
for record,result in zip(recovery_payload['windows'],weight_experiment_results['equal']):
    old=record['results'][recovery_A_index]
    assert record['start']==str(dates[result['start']])
    assert old['invalid']==result['invalid']
    assert np.isclose(old['return'],result['return'],rtol=1e-12,atol=1e-12)
    assert np.isclose(old['cost'],result['cost'],rtol=1e-12,atol=1e-12)
backtest_results={'A':weight_experiment_results['equal']}
print('RECOVERY_VERIFIED: original hash; 650 saved A returns/costs/validity exactly reproduced; A valid=',int(weight_experiment_valid.sum()))
print('Only required baseline state rebuilt; saved other experiment outputs untouched.')


## A持股100%逆波動配置

仍由A挑25檔；原前20日波動逆數100%決定原始配置，不摻等權。沿用既有權重正規化1.8%至5.4%界限、總投入90%，界限是共同風險約束而非50%等權混合。

單一事先定義候選；保留原資料品質門檻、費用與成交引擎，不重跑A控制。

In [ ]:
# Numeric control a_inversevol_100: retained audit
import sys
print('Kernel executable:',sys.executable)
print('Retained metadata:',{k:(type(globals()[k]).__name__,getattr(globals()[k],'shape',None)) for k in ['flow','signal_returns','scores','weight_experiment_results']})
print('Recent execution markers:',[(i,s.splitlines()[0][:90]) for _,i,s in get_ipython().history_manager.get_range(output=False) if s.startswith('# Pure flow') or s.startswith('# Numeric control')][-5:])
assert COUNT==25 and HORIZON==24 and REBALANCE==5 and LOT==1000
assert np.isclose(FEE,.001425) and np.isclose(TAX,.003)


In [ ]:
# Numeric control a_inversevol_100: fixed experiment
a_invvol100_definition_v1={'key':'A_select25_inversevol100_bounds018_054','data':snapshot_sha256,'engine':hashlib.sha256(weight_simulator_source.encode()).hexdigest(),'lookback':20,'lower':.018,'upper':.054,'total':.9}
if 'a_invvol100_results_v1' not in globals():
    a_invvol100_audit_v1=[]
    def a_invvol100_allocation_v1(t,selected,allocation):
        assert allocation=='invvol100'
        vol=np.std(signal_returns[t-20:t,selected],axis=0)
        if np.any(~np.isfinite(vol)) or np.any(vol<=0):raise ValueError('Invalid inverse-volatility signal')
        raw=1/vol
        weights=weight_bounded_normalize(raw,total=INVESTED,lower=.018,upper=.054)
        assert np.isclose(weights.sum(),INVESTED) and np.all(weights>=.018-1e-10) and np.all(weights<=.054+1e-10)
        a_invvol100_audit_v1.append((int(t),tuple(int(j) for j in selected)))
        return weights
    src=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_a_invvol100_v1(start,allocation="invvol100",horizon=HORIZON):')
    assert src.count('weight_experiment_allocation(t,selected,allocation)')==1
    src=src.replace('weight_experiment_allocation(t,selected,allocation)','a_invvol100_allocation_v1(t,selected,allocation)')
    exec(src)
    a_invvol100_results_v1=[simulate_a_invvol100_v1(int(s)) for s in window_starts]
    assert a_invvol100_audit_v1==[x[:2] for x in weight_experiment_audit['equal']]
    a_invvol100_cached_definition_v1=dict(a_invvol100_definition_v1)
else:assert a_invvol100_cached_definition_v1==a_invvol100_definition_v1
a_invvol100_metrics_v1,a_invvol100_valid_v1,a_invvol100_pair_v1,a_invvol100_excess_v1=summarize_numeric_control_v1('a_inversevol_100',a_invvol100_results_v1)
print('A selections match retained equal-weight A exactly; no equal-weight mixture added.')
print('No family inference yet; overlapping windows and original data limitations remain.')
print('NUMERIC_CONTROL_READY a_inversevol_100')


## 純財務選股：全股票池公開資料覆蓋

全150檔可選股票池，唯讀查原公告，保留歷史公布日期與時間。按原可解析Q1／Q2／Q3累計本業金額相減，得到Q2對Q1或Q3對Q2利潤率季變化；這是有明確口徑的部分年度原型，尚不含年度Q4／跨年Q1比較。以數值由高到低挑25檔，等權90%，不加動能。較新的自結或更正公告若未核對口徑，該股票暫不排名；不足25檔直接不可判定，不補中性。先保存覆蓋查核，再執行績效。

In [ ]:
# Pure financial v1: retained inventory and full-universe original announcement query
import sys,datetime,re,json,unicodedata,ast
from collections import Counter,defaultdict
from research_layout import ResearchPath as Path
print('Retained kernel:',sys.executable,'stocks:',len(stock_ids),'sessions:',len(dates))
print('Recent safe markers:',[(i,v.splitlines()[0][:80]) for _,i,v in get_ipython().history_manager.get_range(output=False) if v.startswith('# Numeric control')][-4:])
def fetch_full_pool_announcements_v1():
    text=next((Path.home()/'Downloads').glob('AI_CUP_DB*.md')).read_text(encoding='utf-8-sig')
    cfg={k:v.strip().strip('`').strip('"').strip("'") for k,v in re.findall(r'(?m)^(DB_[A-Z_]+)\s*=\s*(.*)$',text)}
    del text
    dbc=None
    try:
        dbc=pymysql.connect(host=cfg['DB_HOST'],port=int(cfg['DB_PORT']),user=cfg['DB_USER'],password=cfg['DB_PASSWORD'],database=cfg['DB_NAME'],charset='utf8mb4',connect_timeout=15,read_timeout=90,autocommit=False,cursorclass=pymysql.cursors.DictCursor)
        cfg.clear()
        with dbc.cursor() as cur:
            cur.execute('SET SESSION TRANSACTION READ ONLY')
            cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
            cur.execute('SELECT a.stock_id,a.ann_id,a.announce_date,a.announce_time,a.subject,a.event_date,a.content,a.source FROM announcements a JOIN universe u ON a.stock_id=u.stock_id AND u.active=1 WHERE a.announce_date BETWEEN %s AND %s AND (a.subject LIKE %s OR a.subject LIKE %s OR a.subject LIKE %s) ORDER BY a.stock_id,a.announce_date,a.announce_time',(f'{dates[0].year}-01-01',str(dates[-1]),'%財務報%','%財報%','%自結%'))
            rows=cur.fetchall()
            cur.execute("SELECT TABLE_NAME,COLUMN_NAME,DATA_TYPE FROM information_schema.columns WHERE table_schema=DATABASE() AND TABLE_NAME IN ('financials','monthly_revenue','announcements','news') ORDER BY TABLE_NAME,ORDINAL_POSITION")
            schema=cur.fetchall()
        return rows,schema
    except Exception as exc:
        print('Safe full-pool query failure:',type(exc).__name__)
        raise RuntimeError('Read-only extraction failed; connection details suppressed') from None
    finally:
        cfg.clear()
        if dbc is not None:dbc.rollback();dbc.close()
if 'full_pool_financial_announcements_v1' not in globals():
    full_pool_financial_announcements_v1,full_pool_schema_v1=fetch_full_pool_announcements_v1()
full_pool_ann_by_stock_v1=defaultdict(list)
for ann in full_pool_financial_announcements_v1:
    assert ann['stock_id'] in si
    full_pool_ann_by_stock_v1[ann['stock_id']].append(ann)
print('Full pool query coverage:',{'queried_stocks':len(stock_ids),'stocks_with_rows':len(full_pool_ann_by_stock_v1),'rows':len(full_pool_financial_announcements_v1),'years':dict(Counter(a['announce_date'].year for a in full_pool_financial_announcements_v1))})
print('Schema column names:',{t:[x['COLUMN_NAME'] for x in full_pool_schema_v1 if x['TABLE_NAME']==t] for t in ['financials','monthly_revenue']})


In [ ]:
# Pure financial v1: timestamped original amounts and decision coverage
# Restore only existing parser functions from saved notebooks, not previous experiments.
for restore_path in recovery_root.glob('*.ipynb'):
    restore_nb=json.loads(restore_path.read_text(encoding='utf-8'))
    for cell in restore_nb['cells']:
        if cell['cell_type']!='code':continue
        source=''.join(cell['source'])
        if 'def prestart_quarter(' not in source:continue
        tree=ast.parse(source)
        for part in tree.body:
            if isinstance(part,ast.FunctionDef) and part.name in ['prestart_stamp','prestart_quarter']:
                exec(compile(ast.Module(body=[part],type_ignores=[]),'<restored-parser>','exec'))
assert 'prestart_quarter' in globals()
def pure_financial_amounts_v1(a):
    body=unicodedata.normalize('NFKC',a.get('content') or '').replace('\r','')
    vals={}
    for field,number,label in [('revenue',4,'營業收入'),('operating',6,'營業利益')]:
        m=re.search(r'(?m)^\s*'+str(number)+r'[.、]\s*([^\n]+)',body)
        if not m or label not in m.group(1) or '累計' not in m.group(1) or ':' not in m.group(1):return None
        v=m.group(1).rsplit(':',1)[1].strip().replace(',','').replace(' ','')
        if re.fullmatch(r'\(?-?\d+(?:\.\d+)?\)?',v) is None:return None
        vals[field]=float(v.strip('()'))*(-1 if v.startswith('(') else 1)
    return vals if vals['revenue']>0 else None
def pure_financial_asof_v1(sid,cutoff):
    year=cutoff.year; quarters={}; pending=[]
    for ann in full_pool_ann_by_stock_v1[sid]:
        stamp=prestart_stamp(ann)
        if stamp is None or stamp>=cutoff:continue
        subject=unicodedata.normalize('NFKC',ann.get('subject') or '')
        if '代' in subject:continue
        q=prestart_quarter(ann,year)
        if q:
            vals=pure_financial_amounts_v1(ann)
            if q not in quarters or stamp>quarters[q]['stamp']:
                quarters[q]={'stamp':stamp,'values':vals,'ann_id':ann['ann_id']}
        if any(str(v) in subject for v in [year,year-1911]) and ('自結' in subject or '更正' in subject):
            for qq,words in [(3,['第三季','第3季','Q3','3Q']),(2,['第二季','第2季','Q2','2Q','上半年']),(1,['第一季','第1季','Q1','1Q'])]:
                if any(w in subject for w in words):pending.append((qq,stamp,ann['ann_id']));break
    usable=[q for q in [2,3] if all(x in quarters and quarters[x]['values'] is not None for x in range(1,q+1))]
    if not usable:return {'status':'unknown_originals'}
    q=max(usable)
    if any(qq>q or (qq==q and stamp>quarters[q]['stamp']) for qq,stamp,_ in pending):return {'status':'newer_self_or_correction_unverified'}
    def single(qq):
        return {k:quarters[qq]['values'][k]-(quarters[qq-1]['values'][k] if qq>1 else 0) for k in ['revenue','operating']}
    previous,current=single(q-1),single(q)
    if previous['revenue']<=0 or current['revenue']<=0:return {'status':'invalid_revenue_basis'}
    return {'status':'verified_formal','delta_pp':100*(current['operating']/current['revenue']-previous['operating']/previous['revenue']),'quarter':q,'source_ids':[quarters[x]['ann_id'] for x in range(1,q+1)],'source_stamps':[str(quarters[x]['stamp']) for x in range(1,q+1)]}
pure_financial_scores_v1=np.full(shape,np.nan)
pure_financial_evidence_v1={}; pure_financial_coverage_v1=[]
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55))
    status=Counter(); raw=np.full(len(stock_ids),np.nan)
    for sid in stock_ids:
        record=pure_financial_asof_v1(sid,cutoff); status[record['status']]+=1
        if record['status']=='verified_formal':raw[si[sid]]=record['delta_pp']
        pure_financial_evidence_v1[(t,sid)]=record
    eligible=np.isfinite(scores['A'][t-1])&np.isfinite(raw)
    pure_financial_scores_v1[t-1]=percentile(raw,eligible)
    pure_financial_coverage_v1.append({'date':str(dates[t]),'eligible':int(eligible.sum()),'status':dict(status)})
print('Usable decisions:',sum(x['eligible']>=COUNT for x in pure_financial_coverage_v1),'of',len(pure_financial_coverage_v1))
print('Coverage range:',min(x['eligible'] for x in pure_financial_coverage_v1),max(x['eligible'] for x in pure_financial_coverage_v1))
print('Competition anchors:',[x for x in pure_financial_coverage_v1 if x['date'] in ['2024-10-29','2025-10-28']])
print('Unknown is excluded, never neutral filled; unparsed self/correction blocks stale formal signal.')
print('PURE_FINANCIAL_COVERAGE_READY')


In [ ]:
# Pure financial v1: integrate explicitly labelled self-reports before performance
def pure_financial_self_asof_v1(sid,cutoff):
    eligible=[];qs={}
    for a in full_pool_ann_by_stock_v1[sid]:
        stamp=prestart_stamp(a)
        if stamp is None or stamp>=cutoff:continue
        subject=unicodedata.normalize('NFKC',a.get('subject') or '')
        if '代' in subject:continue
        q=prestart_quarter(a,cutoff.year)
        vals=pure_financial_amounts_v1(a) if q in [1,2] else None
        if vals and (q not in qs or stamp>qs[q]['stamp']):qs[q]={'values':vals,'stamp':stamp,'id':a['ann_id']}
        if any(str(v) in subject for v in [cutoff.year,cutoff.year-1911]) and any(w in subject for w in ['第三季','第3季','Q3','3Q']) and ('自結' in subject or '更正' in subject):eligible.append((stamp,a))
    if not eligible or not all(q in qs for q in [1,2]):return None
    stamp,a=max(eligible,key=lambda x:x[0])
    subject=unicodedata.normalize('NFKC',a.get('subject') or '')
    if '更正' in subject or '自結' not in subject:return None
    body=unicodedata.normalize('NFKC',a.get('content') or '')
    if not ('第3季' in body or re.search(r'7\s*-\s*9月',body)):return None
    def pair(label):
        line=next((v for v in body.splitlines() if re.match(label,v.strip())),'')
        tail=re.sub(label,'',line.strip(),count=1).lstrip(':： ')
        nums=re.findall(r'\(?-?\d[\d,]*\)?',tail)
        if len(nums)!=2:return None
        return [float(x.strip('()').replace(',',''))*(-1 if x.startswith('(') else 1) for x in nums]
    rev=pair(r'合併營業收入(?:淨額)?');op=pair(r'合併營業淨利(?:\(損\))?')
    if rev is None or op is None:return None
    previous={k:qs[2]['values'][k]-qs[1]['values'][k] for k in ['revenue','operating']}
    if rev[0]<=0 or previous['revenue']<=0:return None
    if abs(rev[1]-rev[0]-qs[2]['values']['revenue'])>max(2,abs(qs[2]['values']['revenue'])*1e-6):return None
    return {'status':'verified_self','quarter':3,'delta_pp':100*(op[0]/rev[0]-previous['operating']/previous['revenue']),'source_ids':[qs[1]['id'],qs[2]['id'],a['ann_id']],'source_stamps':[str(qs[1]['stamp']),str(qs[2]['stamp']),str(stamp)]}
pure_financial_self_updates_v1=Counter()
for (t,sid),record in list(pure_financial_evidence_v1.items()):
    if record['status']!='newer_self_or_correction_unverified':continue
    new=pure_financial_self_asof_v1(sid,datetime.datetime.combine(dates[t],datetime.time(8,55)))
    if new:
        pure_financial_evidence_v1[(t,sid)]=new
        pure_financial_self_updates_v1[sid]+=1
for t in range(61,len(dates)):
    raw=np.array([pure_financial_evidence_v1[(t,sid)].get('delta_pp',np.nan) for sid in stock_ids])
    eligible=np.isfinite(scores['A'][t-1])&np.isfinite(raw)
    pure_financial_scores_v1[t-1]=percentile(raw,eligible)
print('Verified self-report decision updates:',dict(pure_financial_self_updates_v1))
print('Final decisions with >=25 stocks:',int(np.sum(np.sum(np.isfinite(pure_financial_scores_v1),axis=1)>=COUNT)))
print('Unverified self/correction remains excluded. No performance used to change ranking.')


In [ ]:
# Pure financial v1: single fixed original-information backtest
pure_financial_definition_v1={'key':'operating_margin_qoq_asof_select25_equal','ranking':'single-quarter operating-margin change, descending','available_before':'decision 08:55','supported_quarters':[2,3],'self':'only explicitly labelled and cumulative-consistent Q3','missing':'excluded, never momentum filled','data':snapshot_sha256}
scores['PURE_FINANCIAL']=pure_financial_scores_v1
if 'pure_financial_results_v1' not in globals():
    src=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_pure_financial_v1(start,allocation="equal",horizon=HORIZON):').replace('strategy="A"','strategy="PURE_FINANCIAL"')
    assert src.count('weight_experiment_allocation(t,selected,allocation)')==1
    src=src.replace('weight_experiment_allocation(t,selected,allocation)','np.full(COUNT,INVESTED/COUNT)')
    exec(src)
    pure_financial_results_v1=[simulate_pure_financial_v1(int(s)) for s in window_starts]
    pure_financial_cached_definition_v1=dict(pure_financial_definition_v1)
else:assert pure_financial_cached_definition_v1==pure_financial_definition_v1
pure_financial_metrics_v1,pure_financial_valid_v1,pure_financial_pair_v1,pure_financial_excess_v1=summarize_numeric_control_v1('pure_financial',pure_financial_results_v1)
print('Same-season results:',[{**{k:r[k] for k in ['start','return','mdd','cost','invalid','violations']},'date':str(dates[r['start']])} for r in pure_financial_results_v1 if str(dates[r['start']]) in ['2024-10-29','2025-10-28']])
print('Coverage excludes unsupported Q4/Q1 transitions, missing originals and unverified revisions; family comparison pending.')
print('PURE_FINANCIAL_RESULT_READY')


## 純營收MoM／YoY選股

全150檔股票池，以歷史單公司營收速報原文第一段確認股票代號、營收年月及MoM／YoY，再和DB同月雙率核對（誤差不超過0.05百分點）。新聞published_at只代表可證明的已知可用時間，不冒稱首次公告；DB updated_at不是歷史公開時間。決策日08:55前可用、來源不超過45日、報告月落後1–2月才排名。MoM／YoY百分位各半平均，選25檔，等權90%；未知與過舊排除，不用動能補選。資料品質資格與費用仍沿用原引擎。單一設定，未用績效挑月份、調比例或閾值。

In [ ]:
# Pure revenue v1: retained objects and full-universe historical source query
import inspect,ast,copy,json,datetime,re,unicodedata
from collections import defaultdict,Counter
print('Kernel inventory:',{'stock_count':len(stock_ids),'sessions':len(dates),'financial_source_rows':len(full_pool_financial_announcements_v1),'A_windows':len(weight_experiment_results['equal'])})
print('Monthly revenue columns (already audited):',[r['COLUMN_NAME'] for r in full_pool_schema_v1 if r['TABLE_NAME']=='monthly_revenue'])
revenue_fetch_tree=ast.parse(inspect.getsource(fetch_full_pool_announcements_v1))
revenue_fetch_tree.body[0].name='fetch_full_pool_revenue_v1'
revenue_try=next(x for x in revenue_fetch_tree.body[0].body if isinstance(x,ast.Try))
revenue_with=next(x for x in revenue_try.body if isinstance(x,ast.With))
revenue_with.body=ast.parse("""
cur.execute('SET SESSION TRANSACTION READ ONLY')
cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
cur.execute('SELECT r.stock_id,r.year_month,r.mom_pct,r.yoy_pct,r.source FROM monthly_revenue r JOIN universe u ON r.stock_id=u.stock_id AND u.active=1 WHERE r.year_month BETWEEN %s AND %s ORDER BY r.stock_id,r.year_month',(f'{dates[0].year-1}-11',f'{dates[-1].year}-{dates[-1].month:02d}'))
rows=cur.fetchall()
cur.execute('SELECT n.news_id,n.published_at,n.title,n.summary,n.content,n.url,n.source,ns.stock_id,ns.match_type FROM news n JOIN news_stock ns ON n.news_id=ns.news_id JOIN universe u ON ns.stock_id=u.stock_id AND u.active=1 WHERE n.published_at BETWEEN %s AND %s AND n.title LIKE %s ORDER BY n.published_at,n.news_id,ns.stock_id LIMIT 20000',(f'{dates[0].year}-01-01',str(dates[-1])+' 23:59:59','%月營收%'))
schema=cur.fetchall()
""").body
ast.fix_missing_locations(revenue_fetch_tree)
exec(compile(revenue_fetch_tree,'<readonly-revenue-fullpool>','exec'))
if 'full_pool_revenue_rows_v1' not in globals():
    full_pool_revenue_rows_v1,full_pool_revenue_news_v1=fetch_full_pool_revenue_v1()
assert len(full_pool_revenue_news_v1)<20000,'Source cap reached: must page before treating coverage as complete'
print('Source query coverage:',{'monthly_rows':len(full_pool_revenue_rows_v1),'monthly_stocks':len({r['stock_id'] for r in full_pool_revenue_rows_v1}),'news_joined_rows':len(full_pool_revenue_news_v1),'news_stocks':len({r['stock_id'] for r in full_pool_revenue_news_v1}),'by_year':dict(Counter(r['published_at'].year for r in full_pool_revenue_news_v1))})


In [ ]:
# Pure revenue v1: original numeric extraction and point-in-time ranking coverage
full_pool_revenue_db_v1={(r['stock_id'],r['year_month']):r for r in full_pool_revenue_rows_v1}
pure_revenue_records_v1=[];pure_revenue_rejected_v1=Counter(); seen=set()
for row in full_pool_revenue_news_v1:
    sid=row['stock_id'];nid=row['news_id']
    if (nid,sid) in seen:continue
    seen.add((nid,sid))
    title=unicodedata.normalize('NFKC',row['title'] or '')
    if re.search(r'\('+re.escape(sid)+r'\).*?\d{1,2}月營收',title) is None:
        pure_revenue_rejected_v1['not_single_company_bulletin']+=1;continue
    first=unicodedata.normalize('NFKC',row['content'] or row['summary'] or '').split('今年',1)[0]
    if sid+'-TW' not in first:
        pure_revenue_rejected_v1['stock_not_in_opening']+=1;continue
    month=re.search(r'公告(\d{4})年(\d{1,2})月營收',first)
    yoy=re.search(r'年增率\s*([+-]?\d+(?:\.\d+)?)%',first)
    mom=re.search(r'月增率\s*([+-]?\d+(?:\.\d+)?)%',first)
    if not (month and yoy and mom):
        pure_revenue_rejected_v1['missing_original_dual_rates']+=1;continue
    ym=f'{int(month[1]):04d}-{int(month[2]):02d}'
    db=full_pool_revenue_db_v1.get((sid,ym))
    if db is None or db['mom_pct'] is None or db['yoy_pct'] is None:
        pure_revenue_rejected_v1['missing_db_crosscheck']+=1;continue
    mm,yy=float(mom[1]),float(yoy[1])
    if abs(float(db['mom_pct'])-mm)>.05 or abs(float(db['yoy_pct'])-yy)>.05:
        pure_revenue_rejected_v1['db_original_mismatch']+=1;continue
    if row['published_at'] is None:
        pure_revenue_rejected_v1['missing_known_available_time']+=1;continue
    pure_revenue_records_v1.append({'stock_id':sid,'year_month':ym,'mom':mm,'yoy':yy,'published_at':row['published_at'],'news_id':nid,'url':row['url'],'source':row['source'],'status':'verified_original_known_available_time'})
pure_revenue_by_stock_v1=defaultdict(list)
for r in pure_revenue_records_v1:pure_revenue_by_stock_v1[r['stock_id']].append(r)
pure_revenue_scores_v1=np.full(shape,np.nan)
pure_revenue_evidence_v1={}; pure_revenue_coverage_v1=[]
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55))
    mm=np.full(len(stock_ids),np.nan); yy=np.full(len(stock_ids),np.nan); statuses=Counter()
    for sid in stock_ids:
        candidates=[r for r in pure_revenue_by_stock_v1[sid] if r['published_at']<cutoff]
        if not candidates:record={'status':'unknown_source'}
        else:
            latest_month=max(r['year_month'] for r in candidates)
            # Earliest verified source for newest month: do not reset age by reprints.
            latest=min((r for r in candidates if r['year_month']==latest_month),key=lambda r:r['published_at'])
            age=(cutoff-latest['published_at']).total_seconds()/86400
            y,m=map(int,latest['year_month'].split('-'));lag=(cutoff.year-y)*12+cutoff.month-m
            record=dict(latest,age_days=age)
            if age>45 or lag<1 or lag>2:record['status']='stale_or_future_reporting_month'
            else:mm[si[sid]]=latest['mom'];yy[si[sid]]=latest['yoy']
        statuses[record['status']]+=1;pure_revenue_evidence_v1[(t,sid)]=record
    eligible=np.isfinite(scores['A'][t-1])&np.isfinite(mm)&np.isfinite(yy)
    pure_revenue_scores_v1[t-1]=.5*percentile(mm,eligible)+.5*percentile(yy,eligible)
    pure_revenue_coverage_v1.append({'date':str(dates[t]),'eligible':int(eligible.sum()),'status':dict(statuses)})
print('Original extraction:',{'accepted':len(pure_revenue_records_v1),'stock_count':len(pure_revenue_by_stock_v1),'rejected':dict(pure_revenue_rejected_v1)})
print('Usable decisions:',sum(x['eligible']>=COUNT for x in pure_revenue_coverage_v1),'of',len(pure_revenue_coverage_v1),'maximum_eligible:',max(x['eligible'] for x in pure_revenue_coverage_v1))
print('Competition anchors:',[x for x in pure_revenue_coverage_v1 if x['date'] in ['2024-10-29','2025-10-28']])
print('PURE_REVENUE_COVERAGE_READY')


In [ ]:
# Pure revenue v1: fixed experiment after original-information coverage audit
print('Stocks with accepted original sources:',len({r['stock_id'] for r in pure_revenue_records_v1}),'of',len(stock_ids))
pure_revenue_definition_v1={'key':'revenue_mom_yoy_equalrank_asof_select25_equal','ranking':'0.5 MoM percentile + 0.5 YoY percentile','cutoff':'decision 08:55','max_source_age_days':45,'report_month_lag':[1,2],'missing':'exclude, no neutral fill','available_time':'first verified bulletin in retained corpus, not first announcement','data':snapshot_sha256}
scores['PURE_REVENUE']=pure_revenue_scores_v1
if 'pure_revenue_results_v1' not in globals():
    src=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_pure_revenue_v1(start,allocation="equal",horizon=HORIZON):').replace('strategy="A"','strategy="PURE_REVENUE"')
    assert src.count('weight_experiment_allocation(t,selected,allocation)')==1
    src=src.replace('weight_experiment_allocation(t,selected,allocation)','np.full(COUNT,INVESTED/COUNT)')
    exec(src)
    pure_revenue_results_v1=[simulate_pure_revenue_v1(int(s)) for s in window_starts]
    pure_revenue_cached_definition_v1=dict(pure_revenue_definition_v1)
else:assert pure_revenue_cached_definition_v1==pure_revenue_definition_v1
pure_revenue_metrics_v1,pure_revenue_valid_v1,pure_revenue_pair_v1,pure_revenue_excess_v1=summarize_numeric_control_v1('pure_revenue',pure_revenue_results_v1)
print('Same-season results:',[{**{k:r[k] for k in ['start','return','mdd','cost','invalid','violations']},'date':str(dates[r['start']])} for r in pure_revenue_results_v1 if str(dates[r['start']]) in ['2024-10-29','2025-10-28']])
print('Do not treat missing sources as zero or fewer than 25 as a valid portfolio; provenance is known availability, not earliest release.')
print('Family-corrected comparison pending; historical universe and official execution limitations persist.')
print('PURE_REVENUE_RESULT_READY')


## 成交與公司行動：定向官方來源補查

不重做已確認的DB異常查核。以TWSE原始月資料定向核對3036的2024-11-06成交股數、成交金額與高低價；網頁工具已不能取得此JSON，本輪kernel最多再用requests／urllib兩種傳輸尝试，保留TLS驗證。只有來源可驗證才修正資料或引擎。先列公司行動缺口摘要，無法核對者維持排除，不造配息／配股。

In [ ]:
# Execution recheck v1: retained metadata and bounded official daily report
import json,requests,urllib.request,datetime
print('Retained audit objects:',{'shape':execution.shape,'bad_action_count':int(bad_action.sum()),'dividend_rows':len(raw_data['dividend'])})
execution_recheck_url_v1='https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date=20241101&stockNo=3036'
execution_recheck_attempts_v1=[]
execution_recheck_official_v1=None
for method in ['requests','urllib']:
    try:
        if method=='requests':
            response=requests.get(execution_recheck_url_v1,timeout=20)
            response.raise_for_status();report=response.json()
        else:
            with urllib.request.urlopen(execution_recheck_url_v1,timeout=20) as response:report=json.loads(response.read().decode('utf-8'))
        matches=[row for row in report.get('data',[]) if row[0]=='113/11/06']
        if len(matches)!=1:raise ValueError('Requested historical row absent')
        execution_recheck_official_v1={'url':execution_recheck_url_v1,'stat':report.get('stat'),'fields':report.get('fields'),'row':matches[0]}
        execution_recheck_attempts_v1.append({'method':method,'result':'verified_row_returned'})
        break
    except Exception as exc:execution_recheck_attempts_v1.append({'method':method,'result':type(exc).__name__})
print('Official source attempts:',execution_recheck_attempts_v1)
print('Official row:',json.dumps(execution_recheck_official_v1,ensure_ascii=False))
execution_recheck_corporate_v1=[{'stock_id':sid,'date':str(date),'cash':float(c or 0),'stock':float(st or 0),'before':float(before or 0),'reference':float(ref or 0)} for sid,date,c,st,before,ref,status in raw_data['dividend'] if sid in si and date in di and bad_action[di[date],si[sid]]]
print('Unsupported action rows:',len(execution_recheck_corporate_v1),'samples:',execution_recheck_corporate_v1[:6])
print('EXECUTION_RECHECK_SOURCE_READY')


## 成交引擎v2：統計口徑修正

TWSE已核對3036的原始成交股數與金額。日成交統計含鉅額，但鉅額價格不列一般高低價，不能把均價落在高低價內作通用有效條件。v2只改成交資料的有限正數門檻，不變策略參數、不補造價格，保留原公司行動排除；v1結果保留。官賽實際均價範圍尚需DataPack核對，因此仍是研究引擎。

來源：[TWSE統計說明](https://www.twse.com.tw/exchangeReport/STOCK_DAY?date=20230725&response=html&stockNo=2382)；[鉅額價格不納入高低價](https://investoredu.twse.com.tw/pages/TWSE_InvestmentQA.aspx?ID=3)。原始3036列見cell66。因引擎變更重驗證受影響候選，是有理由的版本更新，不是新增重複實驗。

In [ ]:
# Engine v2: fixed candidate revalidation after verified source-scope correction
import json
row=execution_recheck_official_v1['row'];t,j=di[datetime.date(2024,11,6)],si['3036']
assert float(row[1].replace(',',''))==volume[t,j]
assert float(row[2].replace(',',''))==turnover[t,j]
good_exec_v2=np.isfinite(execution)&(execution>0)&np.isfinite(volume)&(volume>0)&np.isfinite(turnover)&(turnover>0)&np.isfinite(close)&(close>0)
assert np.all(~good_exec|good_exec_v2)
engine_v2_meta={'version':'execution-v2-total-value-volume-positive','data_hash':snapshot_sha256,'restored_execution_rows':int(np.sum(good_exec_v2&~good_exec)),'unresolved_action_rows':len(execution_recheck_corporate_v1),'official_competition_vwap_scope_confirmed':False}
print('Correction metadata:',engine_v2_meta)
if 'PURE_FLOW' not in scores:
    recovered_flow=np.full(shape,np.nan)
    for q in range(60,len(dates)):
        ratio=np.sum(flow[q-19:q+1],axis=0)/np.sum(volume[q-19:q+1],axis=0)
        recovered_flow[q]=percentile(ratio,np.isfinite(scores['A'][q]))
    scores['PURE_FLOW']=recovered_flow
scores['AH']=scores['A'].copy();scores['BH']=scores['B'].copy()
scores['REV5']=np.full(shape,np.nan)
for q in range(4,len(dates)):
    eligible=np.isfinite(scores['A'][q])&np.all(np.isfinite(signal_returns[q-4:q+1]),axis=0)
    rr=np.prod(1+signal_returns[q-4:q+1],axis=0)-1
    scores['REV5'][q,eligible]=-rr[eligible]
def engine_v2_allocation(t,selected,allocation):
    if allocation=='equal':return np.full(COUNT,INVESTED/COUNT)
    if allocation=='mild':
        raw=np.linspace(1.25,.75,COUNT);return INVESTED*raw/raw.sum()
    assert allocation in ['invvol','invvol100']
    vol=np.std(signal_returns[t-20:t,selected],axis=0)
    assert np.all(np.isfinite(vol)&(vol>0))
    inverse=1/vol;raw=INVESTED*inverse/inverse.sum()
    if allocation=='invvol':raw=.5*np.full(COUNT,INVESTED/COUNT)+.5*raw
    return weight_bounded_normalize(raw,total=INVESTED,lower=.018,upper=.054)
src=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_engine_v2(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):').replace('    strategy="A"\n','')
assert src.count('good_exec[t]')==1 and src.count('weight_experiment_allocation(')==1
src=src.replace('good_exec[t]','good_exec_v2[t]').replace('weight_experiment_allocation(','engine_v2_allocation(')
line='            cash,shares,traded,cost,missing=settle(cash,shares,target,execution[t],good_exec_v2[t])'
skip='            if offset>0 and tolerance>0:\n                kept=(shares>0)&(target>0)\n                small=kept&(np.abs(target-shares)*np.nan_to_num(close[t-1])<=tolerance*previous_nav)\n                target[small]=shares[small]\n'
assert src.count(line)==1
src=src.replace(line,skip+line);exec(src)
engine_v2_fixed_methods={'REF':('REF','equal',0),'A_equal':('A','equal',0),'B':('B','equal',0),'C':('C','equal',0),'AH':('AH','equal',0),'BH':('BH','equal',0),'A_mild':('A','mild',0),'A_invvol50':('A','invvol',0),'A_mild_tolerance':('A','mild',.005),'REV5':('REV5','mild',0),'pure_flow':('PURE_FLOW','equal',0),'pure_lowvol':('PURE_LOWVOL','equal',0),'A_invvol100':('A','invvol100',0),'pure_financial':('PURE_FINANCIAL','equal',0),'pure_revenue':('PURE_REVENUE','equal',0)}
if 'engine_v2_results' not in globals():
    engine_v2_results={}
    for name,(strategy,allocation,tolerance) in engine_v2_fixed_methods.items():
        engine_v2_results[name]=[simulate_engine_v2(int(start),strategy,allocation,tolerance=tolerance) for start in window_starts]
        print('Revalidated:',name,'valid:',sum(not x['invalid'] for x in engine_v2_results[name]),flush=True)
engine_v2_metrics={name:{f:np.array([r[f] for r in rows]) for f in ['return','mdd','cost','turnover','violations','strict_cap_days','min_held','max_held']} for name,rows in engine_v2_results.items()}
engine_v2_valid={name:np.array([not x['invalid'] and x['violations']==0 for x in rows]) for name,rows in engine_v2_results.items()}
engine_v2_summary=[]
for name in engine_v2_results:
    pair=engine_v2_valid[name]&engine_v2_valid['A_equal']
    if not pair.any():continue
    candidate=engine_v2_metrics[name];base=engine_v2_metrics['A_equal']
    engine_v2_summary.append({'method':name,'paired_windows':int(pair.sum()),'median_pct':float(np.median(candidate['return'][pair])*100),'A_same_dates_median_pct':float(np.median(base['return'][pair])*100),'mean_difference_pp':float(np.mean(candidate['return'][pair]-base['return'][pair])*100),'mean_cost_pct':float(np.mean(candidate['cost'][pair])*100)})
print('v2 paired summaries:',json.dumps(engine_v2_summary,ensure_ascii=False))
print('v2 seasonal validity:',{name:[{'date':str(dates[x['start']]),'invalid':x['invalid'],'return_pct':float(x['return']*100) if not x['invalid'] else None} for x in rows if str(dates[x['start']]) in ['2024-10-29','2025-10-28']] for name,rows in engine_v2_results.items() if name in ['A_equal','pure_financial','pure_revenue']})
print('ENGINE_V2_REVALIDATION_READY; inference pending, action gaps unchanged.')


In [ ]:
# Corporate action source v1: original announcement coverage, no inferred ratios
import ast,inspect
action_tree=ast.parse(inspect.getsource(fetch_full_pool_announcements_v1));action_tree.body[0].name='fetch_action_originals_v1'
action_try=next(x for x in action_tree.body[0].body if isinstance(x,ast.Try))
action_with=next(x for x in action_try.body if isinstance(x,ast.With))
action_with.body=ast.parse("""
cur.execute('SET SESSION TRANSACTION READ ONLY')
cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
cur.execute('SELECT a.stock_id,a.ann_id,a.announce_date,a.announce_time,a.subject,a.event_date,a.content,a.source FROM announcements a JOIN universe u ON a.stock_id=u.stock_id AND u.active=1 WHERE a.announce_date BETWEEN %s AND %s AND (a.subject LIKE %s OR a.subject LIKE %s OR a.subject LIKE %s OR a.subject LIKE %s) ORDER BY a.stock_id,a.announce_date,a.announce_time',(f'{dates[0].year}-01-01',str(dates[-1]),'%股利%','%配息%','%除權%','%除息%'))
rows=cur.fetchall()
schema=[]
""").body
ast.fix_missing_locations(action_tree);exec(compile(action_tree,'<readonly-action-originals>','exec'))
action_originals_v1,_=fetch_action_originals_v1()
action_source_candidates_v1={}
for record in execution_recheck_corporate_v1:
    date=datetime.date.fromisoformat(record['date']);sid=record['stock_id']
    # Only source candidates; matching calendar text is not sufficient to repair amounts.
    selected=[a for a in action_originals_v1 if a['stock_id']==sid and a['announce_date']<=date and (date-a['announce_date']).days<=180]
    action_source_candidates_v1[(sid,record['date'])]=selected
print('Corporate source coverage:',{'original_rows':len(action_originals_v1),'unresolved_actions':len(execution_recheck_corporate_v1),'actions_with_pre_event_candidate':sum(bool(v) for v in action_source_candidates_v1.values())})
print('Candidate metadata samples:',[{'stock_id':key[0],'ex_date':key[1],'source_ids':[a['ann_id'] for a in values[-3:]],'subjects':[a['subject'] for a in values[-3:]]} for key,values in list(action_source_candidates_v1.items())[:4]])
print('No action amounts repaired: exact event date, rate, share unit and pre-event source must be verified, not inferred from price difference.')
print('ACTION_SOURCE_AUDIT_READY')


In [ ]:
# Corporate action exact-date v1: reuse pre-event sources and isolate date-matched evidence
import re,unicodedata,json,datetime
print('Retained objects:',{k:(type(globals()[k]).__name__,len(globals()[k])) for k in ['action_originals_v1','action_source_candidates_v1','engine_v2_results']})
print('Recent safe markers:',[(i,v.splitlines()[0][:80]) for _,i,v in get_ipython().history_manager.get_range(output=False) if v.startswith('# Corporate action') or v.startswith('# Engine v2')][-4:])
action_exact_date_v1=[]
for (sid,date),sources in action_source_candidates_v1.items():
    d=datetime.date.fromisoformat(date)
    pattern=rf'(?:{d.year}|{d.year-1911})\s*[/年.\-]\s*0?{d.month}\s*[/月.\-]\s*0?{d.day}(?:日|\b)'
    matches=[]
    for a in sources:
        body=unicodedata.normalize('NFKC',a.get('content') or '')
        for m in re.finditer(pattern,body):
            snippet=body[max(0,m.start()-100):m.end()+100]
            if not re.search(r'除息|除權|基準日|停止過戶',snippet):continue
            rate_lines=[line.strip()[:240] for line in body.splitlines() if re.search(r'每股|每.*股配|配股率|除權|除息',line)][:12]
            matches.append({'ann_id':a['ann_id'],'announce_date':str(a['announce_date']),'subject':a['subject'],'date_context':snippet,'rate_lines':rate_lines})
            break
    action_exact_date_v1.append({'stock_id':sid,'ex_date':date,'matches':matches,'status':'date_candidate_requires_manual_basis_check' if matches else 'no_exact_date_context'})
print('Exact-date source counts:',{'actions':len(action_exact_date_v1),'with_context':sum(bool(r['matches']) for r in action_exact_date_v1),'without_context':sum(not r['matches'] for r in action_exact_date_v1)})
print('Matched contexts sample:',json.dumps([r for r in action_exact_date_v1 if r['matches']][:4],ensure_ascii=False))
print('Original action arrays unchanged. Matching date text alone does not establish an ex-date or a rate.')
print('ACTION_EXACT_DATE_AUDIT_READY')


In [ ]:
# Corporate action rates v1: exact ex-trading date, explicit units and reference cross-check
print('Retained source metadata:',len(action_originals_v1),len(action_exact_date_v1))
action_rate_verified_v1=[];action_rate_unresolved_v1=[]
action_raw_lookup_v1={(r['stock_id'],r['date']):r for r in execution_recheck_corporate_v1}
for record in action_exact_date_v1:
    sid,date=record['stock_id'],record['ex_date'];d=datetime.date.fromisoformat(date)
    date_pattern=rf'(?:{d.year}|{d.year-1911})\s*[/年.\-]\s*0?{d.month}\s*[/月.\-]\s*0?{d.day}(?:日|\b)'
    exact=[]
    for a in action_source_candidates_v1[(sid,date)]:
        if (a.get('subject') or '').startswith('代'):continue
        stamp=prestart_stamp(a)
        if stamp is None or stamp>=datetime.datetime.combine(d,datetime.time(8,55)):continue
        body=unicodedata.normalize('NFKC',a.get('content') or '')
        if re.search(r'除權(?:\(息\)|、息|息)?交易日\s*[:：]\s*'+date_pattern,body) or re.search(r'除息交易日\s*[:：]\s*'+date_pattern,body):
            exact.append((stamp,a,body))
    if not exact:
        action_rate_unresolved_v1.append({'stock_id':sid,'ex_date':date,'reason':'no_exact_ex_trading_date_and_known_time'});continue
    stamp,a,body=max(exact,key=lambda x:x[0])
    kind_match=re.search(r'除權、息類別[^\n]*[:：]\s*(除權息|除息|除權)',body)
    kind=kind_match[1] if kind_match else None
    lines=body.splitlines();cash_values=[];stock_values=[]
    for line in lines:
        if any(w in line for w in ['特別股','甲種','乙種','約']):continue
        for pattern in [r'每股(?:分派|配發|發放)?(?:現金股利|現金股息)\s*(?:新[臺台]幣)?\s*(\d+(?:\.\d+)?)\s*元',r'(?:現金股利|現金股息)[^\n]{0,80}?每股[^0-9\n]{0,15}(\d+(?:\.\d+)?)\s*元']:
            m=re.search(pattern,line)
            if m:cash_values.append(float(m[1]))
        m=re.search(r'每(?:壹仟|一仟|仟|千|1,?000)股[^0-9\n]{0,20}(\d+(?:\.\d+)?)股',line)
        if m:stock_values.append(float(m[1])/1000)
    cash_values=set(cash_values);stock_values=set(stock_values)
    if kind=='除息':stock_values={0.}
    if kind=='除權':cash_values={0.}
    if len(cash_values)!=1 or len(stock_values)!=1:
        action_rate_unresolved_v1.append({'stock_id':sid,'ex_date':date,'reason':'explicit_cash_or_share_unit_missing_or_ambiguous','ann_id':a['ann_id'],'kind':kind,'rate_lines':[v.strip()[:200] for v in lines if re.search(r'每股|每.*股配|股利種類|除權、息',v)][:6]});continue
    c,st=next(iter(cash_values)),next(iter(stock_values));raw=action_raw_lookup_v1[(sid,date)]
    expected=(raw['before']-c)/(1+st)
    if abs(expected-raw['reference'])>max(.03,raw['before']*.0002):
        action_rate_unresolved_v1.append({'stock_id':sid,'ex_date':date,'reason':'source_rate_reference_mismatch','ann_id':a['ann_id'],'cash':c,'stock_ratio':st,'expected':expected,'reference':raw['reference']});continue
    action_rate_verified_v1.append({'stock_id':sid,'ex_date':date,'ann_id':a['ann_id'],'published_at':str(stamp),'cash':c,'stock_ratio':st,'reference':raw['reference'],'expected':expected,'source':a.get('source'),'basis':'original explicit cash/share units + exact ex-trading date + reference cross-check'})
print('Verified rate count:',len(action_rate_verified_v1),'unresolved:',len(action_rate_unresolved_v1))
print('Verified originals:',json.dumps(action_rate_verified_v1,ensure_ascii=False))
print('Unresolved reasons:',dict(Counter(x['reason'] for x in action_rate_unresolved_v1)))
print('Unresolved source samples:',json.dumps(action_rate_unresolved_v1[:6],ensure_ascii=False))
print('No cash/share arrays changed; corrections staged with original-source IDs.')
print('ACTION_RATE_VERIFICATION_READY')


In [ ]:
# Corporate action basis v1: classify remaining unit gaps without fabricating rates
print('Unresolved kinds:',dict(Counter(r.get('kind') for r in action_rate_unresolved_v1 if r['reason']=='explicit_cash_or_share_unit_missing_or_ambiguous')))
print('Rate basis examples:',json.dumps([r for r in action_rate_unresolved_v1 if r['reason']!='no_exact_ex_trading_date_and_known_time'][:10],ensure_ascii=False))
action_face_basis_v1=[]
for rec in action_rate_unresolved_v1:
    if not rec.get('ann_id'):continue
    cutoff=datetime.datetime.combine(datetime.date.fromisoformat(rec['ex_date']),datetime.time(8,55))
    matches=[]
    for a in full_pool_ann_by_stock_v1[rec['stock_id']]:
        stamp=prestart_stamp(a)
        if stamp is None or stamp>=cutoff:continue
        body=unicodedata.normalize('NFKC',a.get('content') or '')
        for pattern in [r'每股(?:之)?面額[^\d\n]{0,12}(\d+(?:\.\d+)?)元',r'每股票面金額[^\d\n]{0,12}(\d+(?:\.\d+)?)元']:
            m=re.search(pattern,body)
            if m:matches.append({'ann_id':a['ann_id'],'stamp':str(stamp),'face':float(m[1]),'context':body[max(0,m.start()-35):m.end()+35]});break
    if matches:action_face_basis_v1.append({'stock_id':rec['stock_id'],'ex_date':rec['ex_date'],'matches':matches})
print('Pre-event explicit face-value candidates:',json.dumps(action_face_basis_v1[:6],ensure_ascii=False),'actions_with_candidates:',len(action_face_basis_v1))
print('FACE_BASIS_AUDIT_READY')


In [ ]:
# Corporate cash-only basis v1: ordinary dividend section and explicit allocation shares
action_additional_verified_v1=[];action_basis_review_v1=[]
for rec in action_rate_unresolved_v1:
    if rec.get('reason')!='explicit_cash_or_share_unit_missing_or_ambiguous':continue
    a=next(v for v in action_originals_v1 if v['ann_id']==rec['ann_id'])
    body=unicodedata.normalize('NFKC',a.get('content') or '')
    match=re.search(r'(?ms)^\s*3[.、][^\n]*[:：](.*?)(?=^\s*4[.、])',body)
    block=match[1] if match else ''
    ordinary_lines=[line for line in block.splitlines() if not any(x in line for x in ['特別股','甲種','乙種','約'])]
    cash=None;ratio=None;basis=None
    if rec.get('kind')=='除息':
        vals=set()
        for line in ordinary_lines:
            for m in re.finditer(r'每股[^0-9\n]{0,20}(\d+(?:\.\d+)?)\s*元',line):vals.add(float(m[1]))
        if len(vals)==1:cash=next(iter(vals));ratio=0.;basis='explicit cash-only event kind + common section per-share amount'
    # A monetary stock dividend is usable only if the notice also explicitly specifies shares per 1000 shares.
    if rec.get('kind') in ['除權','除權息']:
        hints=[line.strip() for line in body.splitlines() if re.search(r'每.{0,15}(?:仟|千|1000|1,000).{0,25}股|每股面額|票面金額',line)]
        action_basis_review_v1.append({'stock_id':rec['stock_id'],'ex_date':rec['ex_date'],'ann_id':rec['ann_id'],'share_unit_lines':hints[:6]})
    if cash is None or ratio is None:continue
    raw=action_raw_lookup_v1[(rec['stock_id'],rec['ex_date'])]
    expected=(raw['before']-cash)/(1+ratio)
    if abs(expected-raw['reference'])>max(.03,raw['before']*.0002):continue
    action_additional_verified_v1.append({'stock_id':rec['stock_id'],'ex_date':rec['ex_date'],'ann_id':a['ann_id'],'published_at':str(prestart_stamp(a)),'cash':cash,'stock_ratio':ratio,'reference':raw['reference'],'expected':expected,'source':a.get('source'),'basis':basis})
action_corrections_staged_v1=action_rate_verified_v1+action_additional_verified_v1
print('Additional cash-only verified:',json.dumps(action_additional_verified_v1,ensure_ascii=False))
print('Explicit share-unit lines found:',json.dumps([r for r in action_basis_review_v1 if r['share_unit_lines']],ensure_ascii=False))
print('Total staged verified corrections:',len(action_corrections_staged_v1),'still unresolved:',54-len(action_corrections_staged_v1))
print('Cash/share/signal arrays unchanged; only original-information correction candidates staged.')
print('ACTION_STAGED_CORRECTIONS_READY')


## v3：可驗證公司行動覆蓋，不造未確認股利

僅套用5876／2026-07-28原公告已核對的現金1.8元及配股率0.01，原始矩陣與v2結果保留。修正同步重建資料資格、選股訊號及帳戶，不能只補資產值而漏掉選股資格變化。其他53筆仍排除；目前公告不足以安全轉換者列不可判定，有限研究不要求一定修復全資料。既有15個固定版本因資料覆蓋變更再驗證，不新增候選。

In [ ]:
# Engine v3: apply original-source overlay and rebuild affected signal eligibility
import json,hashlib
assert len(action_corrections_staged_v1)==1
cash_div_v3=cash_div.copy();stock_div_v3=stock_div.copy();bad_action_v3=bad_action.copy();signal_returns_v3=signal_returns.copy()
for a in action_corrections_staged_v1:
    t,j=di[datetime.date.fromisoformat(a['ex_date'])],si[a['stock_id']]
    assert np.isclose(a['cash'],1.8) and np.isclose(a['stock_ratio'],.01)
    cash_div_v3[t,j]=a['cash'];stock_div_v3[t,j]=a['stock_ratio']
    adjusted=(close[t,j]*(1+a['stock_ratio'])+a['cash'])/close[t-1,j]-1
    bad_action_v3[t,j]=adjusted<-.25 or abs(a['expected']-a['reference'])>max(.03,close[t-1,j]*.0002)
    signal_returns_v3[t,j]=close[t,j]/a['reference']-1
overlay_sha_v3=hashlib.sha256(json.dumps(action_corrections_staged_v1,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
scores_v3={k:np.full(shape,np.nan) for k in ['REF','A','B','C','PURE_FLOW','PURE_LOWVOL','PURE_FINANCIAL','PURE_REVENUE','REV5']}
for q in range(60,len(dates)):
    rr=signal_returns_v3[q-59:q+1];ff=flow[q-19:q+1];vv=volume[q-19:q+1]
    eligible=np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(ff)&np.isfinite(vv)&(vv>0),axis=0)&(close[q]>0)&~np.any(bad_action_v3[q-59:q+1],axis=0)
    momentum=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,eligible)+.5*percentile(np.prod(1+rr,axis=0)-1,eligible)
    fscore=np.sum(ff,axis=0)/np.sum(vv,axis=0);vol=np.std(rr[-20:],axis=0)
    scores_v3['A'][q]=momentum;scores_v3['B'][q]=.75*momentum+.25*percentile(fscore,eligible)
    scores_v3['C'][q]=.75*momentum+.25*percentile(-vol,eligible)
    scores_v3['PURE_FLOW'][q]=percentile(fscore,eligible);scores_v3['PURE_LOWVOL'][q]=percentile(-vol,eligible)
    scores_v3['REF'][q]=percentile(np.divide(np.nansum(turnover[q-19:q+1],axis=0),np.sum(np.isfinite(turnover[q-19:q+1]),axis=0),out=np.full(shape[1],np.nan),where=np.sum(np.isfinite(turnover[q-19:q+1]),axis=0)>0),eligible)
    scores_v3['REV5'][q,eligible]=-(np.prod(1+rr[-5:],axis=0)-1)[eligible]
scores_v3['AH']=scores_v3['A'].copy();scores_v3['BH']=scores_v3['B'].copy()
for t in range(61,len(dates)):
    eligible=np.isfinite(scores_v3['A'][t-1])
    raw=np.array([pure_financial_evidence_v1[(t,sid)].get('delta_pp',np.nan) for sid in stock_ids])
    scores_v3['PURE_FINANCIAL'][t-1]=percentile(raw,eligible&np.isfinite(raw))
    records=[pure_revenue_evidence_v1[(t,sid)] for sid in stock_ids]
    mm=np.array([r.get('mom',np.nan) if r['status']=='verified_original_known_available_time' else np.nan for r in records])
    yy=np.array([r.get('yoy',np.nan) if r['status']=='verified_original_known_available_time' else np.nan for r in records])
    keep=eligible&np.isfinite(mm)&np.isfinite(yy)
    scores_v3['PURE_REVENUE'][t-1]=.5*percentile(mm,keep)+.5*percentile(yy,keep)
def engine_v3_allocation(t,selected,allocation):
    if allocation in ['equal','mild']:return engine_v2_allocation(t,selected,allocation)
    vol=np.std(signal_returns_v3[t-20:t,selected],axis=0);assert np.all(np.isfinite(vol)&(vol>0))
    raw=INVESTED/vol/(1/vol).sum()
    if allocation=='invvol':raw=.5*np.full(COUNT,INVESTED/COUNT)+.5*raw
    return weight_bounded_normalize(raw,total=INVESTED,lower=.018,upper=.054)
src_v3=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def simulate_engine_v3(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):').replace('    strategy="A"\n','')
for old,new in [('good_exec[t]','good_exec_v2[t]'),('bad_action[t]','bad_action_v3[t]'),('cash_div[t]','cash_div_v3[t]'),('stock_div[t]','stock_div_v3[t]'),('scores[strategy]','scores_v3[strategy]'),('weight_experiment_allocation(','engine_v3_allocation(')]:src_v3=src_v3.replace(old,new)
line='            cash,shares,traded,cost,missing=settle(cash,shares,target,execution[t],good_exec_v2[t])'
skip='            if offset>0 and tolerance>0:\n                kept=(shares>0)&(target>0)\n                small=kept&(np.abs(target-shares)*np.nan_to_num(close[t-1])<=tolerance*previous_nav)\n                target[small]=shares[small]\n'
assert src_v3.count(line)==1;src_v3=src_v3.replace(line,skip+line);exec(src_v3)
if 'engine_v3_results' not in globals():
    engine_v3_results={}
    for name,(strategy,allocation,tolerance) in engine_v2_fixed_methods.items():
        engine_v3_results[name]=[simulate_engine_v3(int(start),strategy,allocation,tolerance=tolerance) for start in window_starts]
        print('v3 verified:',name,'valid:',sum(not r['invalid'] for r in engine_v3_results[name]),flush=True)
engine_v3_metrics={name:{f:np.array([r[f] for r in rows]) for f in ['return','mdd','cost','turnover','violations','strict_cap_days','min_held','max_held']} for name,rows in engine_v3_results.items()}
engine_v3_valid={name:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for name,rows in engine_v3_results.items()}
print('Source overlay hash:',overlay_sha_v3,'remaining bad-action matrix rows:',int(bad_action_v3.sum()))
print('v3 vs v2 changed A windows:',int(np.sum(~np.isclose(engine_v3_metrics['A_equal']['return'],engine_v2_metrics['A_equal']['return']))))
print('ENGINE_V3_READY')


In [ ]:
# Remaining schema v1: source coverage and historical index/classification availability
import ast,inspect
gap_tree=ast.parse(inspect.getsource(fetch_full_pool_announcements_v1));gap_tree.body[0].name='fetch_remaining_source_schema_v1'
gap_try=next(x for x in gap_tree.body[0].body if isinstance(x,ast.Try));gap_with=next(x for x in gap_try.body if isinstance(x,ast.With))
gap_with.body=ast.parse("""
cur.execute('SET SESSION TRANSACTION READ ONLY')
cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
cur.execute('SELECT TABLE_NAME,COLUMN_NAME,DATA_TYPE FROM information_schema.columns WHERE table_schema=DATABASE() ORDER BY TABLE_NAME,ORDINAL_POSITION')
rows=cur.fetchall()
cur.execute('SELECT a.stock_id,a.ann_id,a.announce_date,a.announce_time,a.subject,a.content,a.source FROM announcements a JOIN universe u ON a.stock_id=u.stock_id AND u.active=1 WHERE a.announce_date BETWEEN %s AND %s AND (a.subject LIKE %s OR a.subject LIKE %s) ORDER BY a.announce_date,a.announce_time',(f'{dates[0].year}-01-01',str(dates[-1]),'%營收%','%營業收入%'))
schema=cur.fetchall()
""").body
ast.fix_missing_locations(gap_tree);exec(compile(gap_tree,'<readonly-remaining-gaps>','exec'))
remaining_full_schema_v1,revenue_announcement_candidates_v1=fetch_remaining_source_schema_v1()
gap_tables_v1=sorted({r['TABLE_NAME'] for r in remaining_full_schema_v1})
print('Potential history/index tables:',[t for t in gap_tables_v1 if re.search(r'industry|sector|index|history|classification',t,re.I)])
print('Stock classification column metadata:',[r for r in remaining_full_schema_v1 if r['TABLE_NAME']=='stocks'])
print('Original revenue-announcement coverage:',{'rows':len(revenue_announcement_candidates_v1),'stocks':len({r['stock_id'] for r in revenue_announcement_candidates_v1})})
print('Revenue announcement samples:',[{k:a[k] for k in ['stock_id','ann_id','announce_date','announce_time','subject']} for a in revenue_announcement_candidates_v1[:5]])
print('REMAINING_SCHEMA_READY')


In [ ]:
# Index inventory v1: schema-directed read-only DB and official page metadata
print('Market index schema:',[r for r in remaining_full_schema_v1 if r['TABLE_NAME']=='market_index'])
idx_tree=ast.parse(inspect.getsource(fetch_full_pool_announcements_v1));idx_tree.body[0].name='fetch_market_index_inventory_v1'
idx_try=next(x for x in idx_tree.body[0].body if isinstance(x,ast.Try));idx_with=next(x for x in idx_try.body if isinstance(x,ast.With))
idx_with.body=ast.parse("""
cur.execute('SET SESSION TRANSACTION READ ONLY')
cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
cur.execute('SELECT * FROM market_index LIMIT 3')
rows=cur.fetchall()
schema=[]
""").body
ast.fix_missing_locations(idx_tree);exec(compile(idx_tree,'<readonly-index-inventory>','exec'))
market_index_sample_v1,_=fetch_market_index_inventory_v1()
print('Index samples:',json.dumps(market_index_sample_v1,default=str,ensure_ascii=False))
official_index_discovery_v1={}
for path in ['/zh/indices/taiex/eftri-hist.html','/zh/indices/taiex/eftri.html','/zh/indices/taiex/mi-5min-hist.html']:
    url='https://www.twse.com.tw'+path
    try:
        response=requests.get(url,timeout=20);response.raise_for_status()
        html=response.text
        api=re.findall(r'data-api\s*=\s*["\']([^"\']+)',html)
        inputs=re.findall(r'<(?:input|select)[^>]+(?:name|id)=["\']([^"\']+)',html)
        official_index_discovery_v1[path]={'url':url,'apis':api,'inputs':inputs,'html':html}
        print('Official index metadata:',path,'api:',api,'inputs:',inputs[:15])
    except Exception as exc:
        official_index_discovery_v1[path]={'url':url,'error':type(exc).__name__}
        print('Official index source error:',path,type(exc).__name__)
print('INDEX_INVENTORY_READY')


In [ ]:
# Official index probe v1: observed monthly endpoints and published daily sectors
official_index_probe_v1={}
for path,params in [('/rwd/zh/TAIEX/EFTRI_HIST',{'date':'20241001','response':'json'}),('/rwd/zh/TAIEX/MI_5MINS_HIST',{'date':'20241001','response':'json'}),('/rwd/zh/afterTrading/MI_INDEX',{'date':'20241028','type':'IND','response':'json'})]:
    try:
        response=requests.get('https://www.twse.com.tw'+path,params=params,timeout=20);response.raise_for_status()
        data=response.json()
        official_index_probe_v1[path]={'url':response.url,'data':data}
        print('Official data schema:',path,{k:v for k,v in data.items() if k in ['stat','date','title','fields']})
        if 'data' in data:print('Monthly rows/sample:',len(data['data']),data['data'][:2])
        print('Daily tables:',[(i,t.get('title'),t.get('fields'),len(t.get('data',[])),t.get('data',[])[:2]) for i,t in enumerate(data.get('tables',[]))][:5])
    except Exception as exc:
        official_index_probe_v1[path]={'error':type(exc).__name__}
        print('Probe failure:',path,type(exc).__name__)
print('OFFICIAL_INDEX_PROBE_READY')


## 歷年競賽產業指數：固定十年與同年其他月份對照

資料取得前固定2016–2025，逐年各月最後一個星期二起、遇非交易日順延，含起點24交易日，以前一交易日指數收盤到第24交易日收盤量測。使用官方交易日與當時公布的分類指數，避免用2026個股分類回推；這是上市分類指數觀察，不是150檔等權可交易策略，也不能提供產業內個股上漲比例。優先全期間同口徑報酬指數；若某分類的報酬指數資料不全則全期使用價格指數，不能逐年混口徑。只比較同口徑大盤。分類存在年限不足／缺資料保持缺值。

十月為研究季，其他11月份皆保留。年度為主要證據單位；完整同年12個窗口才計算十月相對其餘月份的季節差異，窗口可重疊，不視為120個獨立年份。公開來源快取保存本機以免重抓，來源錯誤最多三次有意義嘗試；沒有值不補造。

In [ ]:
# Historical sector fetch v1: fixed-year calendar and endpoint index levels
import time,concurrent.futures,json,calendar,bisect
sector_history_years_v1=list(range(2016,2026))
sector_cache_path_v1=recovery_root/'產業競賽期間_官方指數來源_2026-10-14.json'
sector_month_cache_v1=globals().get('sector_month_cache_v1',{})
sector_day_cache_v1=globals().get('sector_day_cache_v1',{})
if sector_cache_path_v1.exists():
    loaded=json.loads(sector_cache_path_v1.read_text(encoding='utf-8'))
    sector_month_cache_v1.update(loaded.get('months',{}));sector_day_cache_v1.update(loaded.get('days',{}))
def sector_save_cache_v1():
    sector_cache_path_v1.write_text(json.dumps({'definition_version':1,'years':sector_history_years_v1,'months':sector_month_cache_v1,'days':sector_day_cache_v1},ensure_ascii=False,separators=(',',':')),encoding='utf-8')
def sector_request_v1(path,params,kind):
    errors=[]
    for attempt in range(3):
        try:
            r=requests.get('https://www.twse.com.tw'+path,params=params,timeout=20);r.raise_for_status();data=r.json()
            if data.get('stat')!='OK':raise ValueError('Official report not OK')
            if kind=='month':
                rows=data.get('data',[]);assert rows
                return {'url':r.url,'fields':data['fields'],'rows':rows,'date':data.get('date')}
            tables=[t for t in data.get('tables',[]) if '臺灣證券交易所' in (t.get('title') or '') and ('價格指數' in t['title'] or '報酬指數' in t['title'])]
            assert tables
            return {'url':r.url,'date':data.get('date'),'tables':tables}
        except Exception as exc:
            errors.append(type(exc).__name__)
            if attempt<2:time.sleep(attempt+1)
    return {'error_types':errors,'params':params,'path':path}
month_keys=[f'{y}{m:02d}01' for y in range(2016,2027) for m in range(1,13) if (y,m)<=(2026,2)]
missing=[k for k in month_keys if k not in sector_month_cache_v1]
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    futures={pool.submit(sector_request_v1,'/rwd/zh/TAIEX/MI_5MINS_HIST',{'date':k,'response':'json'},'month'):k for k in missing}
    completed=0
    for f in concurrent.futures.as_completed(futures):
        key=futures[f];sector_month_cache_v1[key]=f.result();completed+=1
        if completed%20==0 or completed==len(missing):
            sector_save_cache_v1();print('Official calendar progress:',completed,'/',len(missing),flush=True)
def sector_parse_date_v1(s):
    y,m,d=map(int,s.split('/'));return datetime.date(y+1911 if y<1911 else y,m,d)
sector_calendar_v1=sorted({sector_parse_date_v1(row[0]) for v in sector_month_cache_v1.values() if 'rows' in v for row in v['rows']})
sector_windows_v1=[]
for year in sector_history_years_v1:
    for month in range(1,13):
        last=datetime.date(year,month,calendar.monthrange(year,month)[1])
        anchor=last-datetime.timedelta(days=(last.weekday()-1)%7)
        i=bisect.bisect_left(sector_calendar_v1,anchor)
        if i==0 or i+23>=len(sector_calendar_v1):
            sector_windows_v1.append({'year':year,'month':month,'anchor':str(anchor),'status':'calendar_missing'});continue
        # All intervening official calendar months must have successfully loaded.
        start,end=sector_calendar_v1[i],sector_calendar_v1[i+23]
        needed=[k for k in month_keys if start.replace(day=1)<=datetime.date(int(k[:4]),int(k[4:6]),1)<=end.replace(day=1)]
        if any('error_types' in sector_month_cache_v1[k] for k in needed):
            sector_windows_v1.append({'year':year,'month':month,'anchor':str(anchor),'status':'calendar_gap'});continue
        sector_windows_v1.append({'year':year,'month':month,'anchor':str(anchor),'start':str(start),'end':str(end),'previous':str(sector_calendar_v1[i-1]),'status':'calendar_verified'})
day_keys=sorted({w[k].replace('-','') for w in sector_windows_v1 if w['status']=='calendar_verified' for k in ['previous','end']})
missing_days=[k for k in day_keys if k not in sector_day_cache_v1]
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    futures={pool.submit(sector_request_v1,'/rwd/zh/afterTrading/MI_INDEX',{'date':k,'type':'IND','response':'json'},'day'):k for k in missing_days}
    completed=0
    for f in concurrent.futures.as_completed(futures):
        key=futures[f];sector_day_cache_v1[key]=f.result();completed+=1
        if completed%20==0 or completed==len(missing_days):
            sector_save_cache_v1();print('Official sector endpoint progress:',completed,'/',len(missing_days),flush=True)
sector_save_cache_v1()
print('Sector source final coverage:',{'calendar_months':len(sector_month_cache_v1),'calendar_failures':sum('error_types' in v for v in sector_month_cache_v1.values()),'window_definitions':len(sector_windows_v1),'endpoint_days':len(sector_day_cache_v1),'endpoint_failures':sum('error_types' in v for v in sector_day_cache_v1.values())})
print('HISTORICAL_SECTOR_SOURCE_READY')


## 最終統計設定

策略v3的14個非A控制方法一併以家族5% Bonferroni校正，24／48個窗口位置移動區塊、各5,000次，缺值保留時間位置，不把重疊650窗口當650次獨立實驗。未知資料排除而非用零補。年份及隔24交易日的前後段仍是已見資料敏感度，不是正式樣本外。下檔與回撤沿事先相對A不惡化逾1個百分點門檻；官賽Active Share等未確認不能列可上場。

產業指數採官方同口徑含息報酬指數與大盤含息基準，分類名稱依原報告分列，不臆測新舊名稱等價；只將有完整年度12個月份窗口的年份計算季節差異，至少5個完整年份才估探索性校正區間。年度重抽樣5,000次，不把120個月或個股當独立年度。指數未扣交易費、不等同25檔策略；個股上漲廣度不可由指數還原。

In [ ]:
# Final sector analysis v1: all years, original industry return indexes and figures
import numpy as np,matplotlib.pyplot as plt,html
from matplotlib import font_manager
sector_levels_v1={}
for key,source in sector_day_cache_v1.items():
    if 'tables' not in source:continue
    levels={}
    for table in source['tables']:
        if '報酬指數' not in table['title']:continue
        for row in table['data']:
            try:
                value=float(str(row[1]).replace(',',''))
                if value>0:levels[row[0]]=value
            except (ValueError,TypeError):continue
    sector_levels_v1[key]=levels
sector_names_v1=sorted({name for levels in sector_levels_v1.values() for name in levels if name.endswith('類報酬指數')})
sector_returns_v1=[];sector_window_gaps_v1=[]
benchmark_name='發行量加權股價報酬指數'
for window in sector_windows_v1:
    if window['status']!='calendar_verified':sector_window_gaps_v1.append(window);continue
    before=sector_levels_v1.get(window['previous'].replace('-',''),{})
    after=sector_levels_v1.get(window['end'].replace('-',''),{})
    if benchmark_name not in before or benchmark_name not in after:
        sector_window_gaps_v1.append(dict(window,status='benchmark_missing'));continue
    base=after[benchmark_name]/before[benchmark_name]-1
    for name in sector_names_v1:
        if name not in before or name not in after:continue
        value=after[name]/before[name]-1
        sector_returns_v1.append(dict(window,sector=name,return_value=value,benchmark_return=base,excess=value-base,basis='TWSE sector total-return index; no costs'))
sector_record_lookup_v1={(r['sector'],r['year'],r['month']):r for r in sector_returns_v1}
sector_statistics_v1=[];sector_matrix_v1=np.full((len(sector_names_v1),len(sector_history_years_v1)),np.nan)
seasonality_arrays_v1={}
for i,name in enumerate(sector_names_v1):
    seasonal=[];annual_differences=[]
    for j,y in enumerate(sector_history_years_v1):
        oct_record=sector_record_lookup_v1.get((name,y,10))
        if oct_record:seasonal.append(oct_record);sector_matrix_v1[i,j]=oct_record['return_value']*100
        other=[sector_record_lookup_v1.get((name,y,m)) for m in range(1,13) if m!=10]
        if oct_record and all(other):
            annual_differences.append(oct_record['excess']-np.median([r['excess'] for r in other]))
    seasonality_arrays_v1[name]=np.array(annual_differences)
    if not seasonal:continue
    rr=np.array([r['return_value'] for r in seasonal]);ee=np.array([r['excess'] for r in seasonal])
    stat={'sector':name,'years':len(rr),'positive_years':int(np.sum(rr>0)),'positive_year_fraction':float(np.mean(rr>0)),'median_return_pct':float(np.median(rr)*100),'median_excess_pp':float(np.median(ee)*100),'mean_excess_pp':float(np.mean(ee)*100),'worst_return_pct':float(rr.min()*100),'full_control_years':len(annual_differences),'seasonal_mean_excess_difference_pp':float(np.mean(annual_differences)*100) if annual_differences else None,'ci_family_pp':None,'seasonal_advantage_supported':False}
    if len(annual_differences)>=5:
        rng=np.random.default_rng(SEED+20261014+i);a=np.array(annual_differences)
        means=np.mean(a[rng.integers(0,len(a),size=(5000,len(a)))],axis=1)
        alpha=.05/max(1,len(sector_names_v1));ci=np.quantile(means,[alpha/2,1-alpha/2])*100
        stat['ci_family_pp']=ci.tolist();stat['seasonal_advantage_supported']=bool(ci[0]>0)
    sector_statistics_v1.append(stat)
print('Sector analysis coverage:',{'classification_rows':len(sector_names_v1),'sector_month_windows':len(sector_returns_v1),'annual_seasonal_points':int(np.isfinite(sector_matrix_v1).sum()),'window_gaps':len(sector_window_gaps_v1),'complete_calendar_source':all('error_types' not in v for v in sector_month_cache_v1.values()),'endpoint_source_failures':sum('error_types' in v for v in sector_day_cache_v1.values())})
print('Sector descriptions (full-year coverage first):',json.dumps(sorted(sector_statistics_v1,key=lambda r:(-r['years'],-r['median_return_pct'])),ensure_ascii=False))
plt.rcParams['font.family']='Microsoft JhengHei';plt.rcParams['axes.unicode_minus']=False
fig,ax=plt.subplots(figsize=(15,max(7,len(sector_names_v1)*.38)))
limit=max(5,float(np.nanpercentile(np.abs(sector_matrix_v1),95)))
cmap=plt.get_cmap('RdBu_r').copy();cmap.set_bad('#e5e7eb')
im=ax.imshow(sector_matrix_v1,aspect='auto',cmap=cmap,vmin=-limit,vmax=limit)
ax.set_xticks(range(len(sector_history_years_v1)),sector_history_years_v1)
ax.set_yticks(range(len(sector_names_v1)),[s.replace('類報酬指數','') for s in sector_names_v1])
for i in range(len(sector_names_v1)):
    for j in range(len(sector_history_years_v1)):
        v=sector_matrix_v1[i,j]
        ax.text(j,i,'—' if not np.isfinite(v) else f'{v:.1f}%',ha='center',va='center',fontsize=8,color='white' if np.isfinite(v) and abs(v)>.75*limit else '#111827')
ax.set_title('歷年競賽時段：十月最後星期二起 24 交易日\n官方產業含息指數；每年每產業全部點保留（非可交易策略、未扣交易費）',fontsize=13)
fig.colorbar(im,ax=ax,label='24交易日指數報酬（%）；顏色截尾，格內顯示實際值',fraction=.025)
fig.tight_layout();sector_heatmap_path_v1=recovery_root/'產業競賽期間_產業含息熱圖_2026-10-14.png';fig.savefig(sector_heatmap_path_v1,dpi=150,bbox_inches='tight');plt.close(fig)
fig,ax=plt.subplots(figsize=(13,max(7,len(sector_names_v1)*.36)))
valid_rows=[i for i in range(len(sector_names_v1)) if np.isfinite(sector_matrix_v1[i]).any()]
data=[sector_matrix_v1[i,np.isfinite(sector_matrix_v1[i])] for i in valid_rows]
ax.boxplot(data,vert=False,showfliers=False,medianprops={'color':'#db2777','linewidth':2},tick_labels=[sector_names_v1[i].replace('類報酬指數','') for i in valid_rows])
rng=np.random.default_rng(SEED)
for position,values in enumerate(data,1):ax.scatter(values,position+rng.uniform(-.15,.15,len(values)),s=18,alpha=.8,color='#2563eb',zorder=3)
ax.axvline(0,color='#111827',linestyle='--',linewidth=1.2);ax.set_xlabel('24交易日指數報酬（%）')
ax.set_title('每個歷年競賽時段全部點；桃紅為中位數、黑色虛線為0%');fig.tight_layout()
sector_box_path_v1=recovery_root/'產業競賽期間_產業含息箱形圖_2026-10-14.png';fig.savefig(sector_box_path_v1,dpi=150,bbox_inches='tight');plt.close(fig)
rows_html=[]
for name in sector_names_v1:
    cells=[]
    for y in sector_history_years_v1:
        rec=sector_record_lookup_v1.get((name,y,10))
        if not rec:cells.append('<td title="該分類名稱／原始指數資料不足">—</td>');continue
        v=rec['return_value']*100
        title=f"{rec['start']}～{rec['end']}｜基準 {rec['benchmark_return']*100:.2f}%｜超額 {rec['excess']*100:+.2f}pp｜起算前收盤 {rec['previous']}｜官方含息指數、未扣交易費"
        color=f'hsl({5 if v>=0 else 215},70%,{max(70,97-min(abs(v),30)*.8):.1f}%)'
        cells.append(f'<td style="background:{color}" title="{html.escape(title)}">{v:+.2f}%</td>')
    rows_html.append('<tr><th>'+html.escape(name.replace('類報酬指數',''))+'</th>'+''.join(cells)+'</tr>')
sector_html_path_v1=recovery_root/'產業競賽期間_產業含息互動表_2026-10-14.html'
sector_html_path_v1.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>歷年競賽期間產業觀察</title><style>body{font-family:Microsoft JhengHei,sans-serif;margin:24px;color:#172033}table{border-collapse:collapse;width:100%}td,th{padding:9px;border:1px solid #d7dce3;text-align:right}th:first-child{text-align:left;white-space:nowrap}thead{position:sticky;top:0;background:white}p{line-height:1.7}</style><h1>歷年競賽期間：產業含息指數</h1><p>十月最後一個星期二起24交易日。游標停在每一格可看日期、大盤與超額。所有年份保留；缺值不是0%。此為上市分類指數觀察，非150檔等權策略、未扣費用；沒有推算產業內個股上漲比例。</p><table><thead><tr><th>產業</th>'+''.join(f'<th>{y}</th>' for y in sector_history_years_v1)+'</tr></thead><tbody>'+''.join(rows_html)+'</tbody></table><p>同年其他11月份及家族校正另存Notebook；分類名稱改變不臆測等價合併，部分分類年份較少。資料：證交所每日收盤行情含息分類指數。</p></html>',encoding='utf-8')
sector_stats_path_v1=recovery_root/'產業競賽期間_含息統計與全部窗口_2026-10-14.json'
sector_stats_path_v1.write_text(json.dumps({'definition':'last Tuesday each month,24 sessions, prior close to final close; official total return indexes; no costs','years':sector_history_years_v1,'statistics':sector_statistics_v1,'windows':sector_returns_v1,'gaps':sector_window_gaps_v1},ensure_ascii=False,indent=2),encoding='utf-8')
print('Sector artifacts saved:',sector_heatmap_path_v1.name,sector_box_path_v1.name,sector_html_path_v1.name)
print('FINAL_SECTOR_ANALYSIS_READY')

sector_tr_statistics_v1=sector_statistics_v1
sector_tr_returns_v1=sector_returns_v1


In [ ]:
# Final strategy validation v3: paired costs, fixed family intervals and sensitivity
final_candidate_names_v3=[name for name in engine_v3_results if name!='A_equal']
final_family_alpha_v3=.05/len(final_candidate_names_v3)
def final_block_family_ci_v3(series,block,seed):
    series=np.asarray(series,dtype=float);length=len(series);rng=np.random.default_rng(seed);estimates=[]
    for begin in range(0,5000,100):
        starts=rng.integers(0,length-block+1,size=(min(100,5000-begin),int(np.ceil(length/block))))
        ix=(starts[:,:,None]+np.arange(block)).reshape(len(starts),-1)[:,:length]
        values=series[ix];count=np.sum(np.isfinite(values),axis=1);sums=np.nansum(values,axis=1)
        estimates.extend(np.divide(sums,count,out=np.full(len(sums),np.nan),where=count>0).tolist())
    estimates=np.array(estimates);estimates=estimates[np.isfinite(estimates)]
    return np.quantile(estimates,[final_family_alpha_v3/2,1-final_family_alpha_v3/2]).tolist()
final_strategy_stats_v3=[]
base=engine_v3_metrics['A_equal'];base_mask=engine_v3_valid['A_equal']
early=np.array([dates[s+HORIZON-1]<datetime.date(2026,1,1) for s in window_starts])
first2026=next(i for i,d in enumerate(dates) if d.year==2026);late=np.array([int(s)>=first2026+HORIZON for s in window_starts])
for i,name in enumerate(final_candidate_names_v3):
    candidate=engine_v3_metrics[name];pair=base_mask&engine_v3_valid[name]
    if not pair.any():continue
    diff=candidate['return']-base['return'];series=np.where(pair,diff,np.nan)
    cis={b:final_block_family_ci_v3(series,b,SEED+1000*i+b) for b in [24,48]}
    risk_p10=float(np.quantile(candidate['return'][pair],.1)-np.quantile(base['return'][pair],.1))
    risk_mdd=float(np.median(candidate['mdd'][pair])-np.median(base['mdd'][pair]))
    risk_pass=risk_p10>=-.01 and risk_mdd>=-.01
    yearly=[]
    for y in sorted({dates[s].year for s in window_starts}):
        mask=pair&np.array([dates[s].year==y and dates[s+HORIZON-1].year==y for s in window_starts])
        if mask.any():yearly.append({'year':y,'windows':int(mask.sum()),'mean_difference_pp':float(np.mean(diff[mask])*100)})
    stat={'method':name,'paired_windows':int(pair.sum()),'A_valid_excluded':int(np.sum(base_mask&~pair)),'median_return_pct':float(np.median(candidate['return'][pair])*100),'A_same_dates_median_pct':float(np.median(base['return'][pair])*100),'mean_difference_pp':float(np.mean(diff[pair])*100),'beat_A_pct':float(np.mean(diff[pair]>1e-12)*100),'mean_cost_pct':float(np.mean(candidate['cost'][pair])*100),'A_mean_cost_pct':float(np.mean(base['cost'][pair])*100),'P10_difference_pp':risk_p10*100,'mdd_difference_pp':risk_mdd*100,'risk_pass':risk_pass,'family_CI_pp':{str(b):(np.array(ci)*100).tolist() for b,ci in cis.items()},'both_intervals_positive':all(ci[0]>0 for ci in cis.values()),'yearly':yearly,'early_mean_pp':float(np.mean(diff[pair&early])*100) if (pair&early).any() else None,'late_mean_pp':float(np.mean(diff[pair&late])*100) if (pair&late).any() else None,'official_compliance_fully_verified':False}
    stat['supported_research_improvement']=stat['both_intervals_positive'] and risk_pass
    final_strategy_stats_v3.append(stat);print('Family validation:',json.dumps(stat,ensure_ascii=False),flush=True)
final_supported_v3=[r['method'] for r in final_strategy_stats_v3 if r['supported_research_improvement'] and r['method'] not in ['REF','AH','BH']]
print('Final supported fixed research improvements:',final_supported_v3)
print('All results use same v3 engine; no method chosen solely from highest median.')
print('FINAL_STRATEGY_VALIDATION_READY')


In [ ]:
sector_statistics_v1=sector_tr_statistics_v1
snapshot_sha256='1b826aa999a02664f43ff9c48a77259881901d8b8399c6ee85ebc275af7ea0e2'
overlay_sha_v3=saved_state_v4['company_action_overlay']['overlay_sha256']
# Final rules and selection v3: evidence gates, unresolved sources and deliverables
official_rule_audit_v3={'source':'AI CUP 2026玉山人工智慧公開挑戰賽_比賽辦法_1005更新.pdf pp8-9; D-Plan/DataPack v4.3','lot':1000,'fee':FEE,'tax':TAX,'hold_count_range':[20,30],'regular_cap':.1,'tsmc_cap':.25,'cash_max':.25,'top_active_share_count':15,'active_share_min':.15,'active_share_benchmark_history_verified':False,'batch_void_semantics_fully_reproduced':False,'daily_submission_and_reference_chain_verified':False,'explicit_trading_volume_cap_in_reviewed_rule':False,'market_impact_modelled':False}
print('Official rule audit:',official_rule_audit_v3)
print('Account checks: reuse already verified v3 Notebook outputs; do not recompute strategies during minimal recovery.')
final_selection_v3={'validated_for_competition':None,'supported_research_improvements':final_supported_v3,'provisional_non_equal_method':'A_mild_tolerance','initial_stock_count':25,'initial_invested':.9,'initial_weights':[.045,.027],'adjustment_skip_NAV':.005,'reason':'user non-equal constraint, same A selection, small fixed rank tilt and lower turnover; research support/compliance gates determine evidence status, no forced winner','company_action_unknown_events':53,'classification_individual_stock_history':'unavailable; official index observations are separate','news_first_publication':'not established; known available time only','full_conference_text_strategy':'not tested; retain fixed-case evidence, no new weights'}
if final_supported_v3:
    final_selection_v3['reason']='Some fixed variants pass exploratory return/risk gates; full competition compliance still not verified. Retain provisional configuration pending review of stability and evidence limitations.'
report_path_v3=recovery_root/'策略研究_統一比較與產業季節性_2026-10-14.md'
lines=['# 策略統一比較與競賽期間產業觀察','', '全部策略以同一v3引擎核對。原始資料快照不變，僅用原公告修正5876／2026-07-28的現金1.8元及配股率0.01；53筆未知公司行動維持排除。v1／v2舊輸出保留追溯，不混排。','', '## 選擇結論','', '已證實可上場的方法：無。完整Active Share歷史基準、D-Plan引用鏈、官賽批次作廢與成交均價範圍仍未全部核對。研究門檻通過候選：'+('、'.join(final_supported_v3) if final_supported_v3 else '無；未證實改良。'),'','依使用者非等權需求，保留A溫和動能＋0.5%NAV微調略過為暫定研究配置：A挑25檔、每5日調整，初始目標4.5%至2.7%、總投入90%。它是暫定配置，不是已證實最佳／可上場策略。','', '## 固定方法的費用後配對比較','', '|方法|共同窗口|報酬中位數|同日期A中位數|平均配對差pp|平均成本|24日校正區間pp|48日校正區間pp|風險門檻|','|---|---:|---:|---:|---:|---:|---|---|---|']
for r in final_strategy_stats_v3:
    ci=r['family_CI_pp'];lines.append(f"|{r['method']}|{r['paired_windows']}|{r['median_return_pct']:.2f}%|{r['A_same_dates_median_pct']:.2f}%|{r['mean_difference_pp']:+.3f}|{r['mean_cost_pct']:.3f}%|{ci['24'][0]:+.3f}～{ci['24'][1]:+.3f}|{ci['48'][0]:+.3f}～{ci['48'][1]:+.3f}|{'通過' if r['risk_pass'] else '未通過'}|")
lines.extend(['','14個非A方法一起做Bonferroni家族5%校正、每設定5000次24／48位置區塊抽樣，保留缺值時間位置。方法的有效日期不同，表內A都是各方法相同日期；不可按各列中位數直接排序。窗口重疊、不等於獨立實驗，較晚段也是已看過資料，非正式樣本外。早晚段／年度明細與全部窗口留在Notebook與JSON。','', '## 產業研究','', '2016–2025逐月最後星期二起24交易日，官方含息產業指數對官方大盤含息基準。這是上市指數觀察，不是150檔可交易等權組合、不扣費用，也無法提供群內個股上漲比例。產業名稱改變分列，不假定新舊名稱等價；資料不足保持缺值。','', '|產業|同季年份|上漲年份|報酬中位數|相對大盤中位數pp|最差同季年|完整月對照年份|季節差區間pp|','|---|---:|---:|---:|---:|---:|---:|---|'])
for r in sorted(sector_statistics_v1,key=lambda r:(-r['years'],-r['median_return_pct'])):
    ci=r['ci_family_pp'];ci_text='年份不足' if ci is None else f'{ci[0]:+.2f}～{ci[1]:+.2f}'
    lines.append(f"|{r['sector'].replace('類報酬指數','')}|{r['years']}|{r['positive_years']}|{r['median_return_pct']:.2f}%|{r['median_excess_pp']:+.2f}|{r['worst_return_pct']:.2f}%|{r['full_control_years']}|{ci_text}|")
lines.extend(['','季節差＝十月相對大盤超額−同年其他11月份超額中位數；至少5個完整年份才估年度5000次重抽樣與產業家族校正。十年仍是小樣本；同年月份窗口可重疊、產業與年份也有共同市場因素，探索區間不保證未來。分類存在年份較少者不可和十年系列直接比勝率。','', '熱圖：![全部同季年度點](產業競賽期間_產業含息熱圖_2026-10-14.png)','', '箱形圖：![全部年度點與0%線](產業競賽期間_產業含息箱形圖_2026-10-14.png)','', '[互動年度表](產業競賽期間_產業含息互動表_2026-10-14.html)','', '官方122個月交易日、122份價格月報與225日端點已完成來源恢復；JSON與官方CP950 CSV原始來源皆保留。含息分析仍有一個2019年4月窗口的大盤名稱差異，未擅自合併，故全月對照多為9年；新設／改名分類另保留缺格。十年同季資料仍是小樣本。', '', '## 尚未能驗證的分支','', '- 公司行動：僅一筆原公告已核對，53筆仍排除；股利元與配股率不能在面額／股數未核對時換算。','- 公告：全池原營收公告336筆只覆盖19檔；126檔有可核對速報。新聞已知可用時間不等於首次公告；DB updated_at不作發布時間。','- 法說：6770／3105原先四則固定新聞正反證已保存；完整影音未轉錄，穩懋原件來源嘗試已達有限重試上限，未產生或回測新文字權重。','- 歷史分類：DB沒有point-in-time個股產業分類；以官方歷史指數完成獨立觀察，不能回填個股分類或改歷史A權重。','- 規則：整千股、現金／檔數／漂移、費率可核對；DB有ETF資料表，但持股僅2026-10-07至10-12、各ETF多為1至2個日期；Active Share缺回測期間每日前15大權重、公開時間與口徑，整批作廢／引用鏈／真正交件未全驗證。每日成交量上限不是所讀辦法的明文限制，市場衝擊是研究限制而非自行增列官賽規則。','', '## 來源與執行記錄','', '- 原Notebook：動能持股_開賽前公告配置研究_2026-10-14.ipynb；策略計算在原研究kernel；該程序結束後經使用者同意啟動替代kernel，僅恢復快取、已保存驗證摘要及產業分析，不重跑策略／重抓來源。沒有寫團隊DB或提交交易。未做Restart and Run All，沒有宣稱整本已通過清空狀態重跑。','- 官方產業原始查詢URL／全部窗口／來源快取：產業競賽期間_官方指數來源_2026-10-14.json、產業競賽期間_含息統計與全部窗口_2026-10-14.json。','- 官方來源：https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX（type=IND）；交易日：https://www.twse.com.tw/rwd/zh/TAIEX/MI_5MINS_HIST。','- 原公司行動與成交證據來源ID／URL保留Notebook對應cells；不從價格倒造股利。'])
lines.extend(['','## 方法名稱對照與實驗範圍','','共21個已測版本：15個廣泛窗口方法（含A控制）＋6個公告配置原型；後6個僅2024／2025同季各一個窗口，不能說21個方法都測650次。廣泛方法按可用來源與公司行動排除後，各有不同有效窗口。',''])
method_labels_v4={'REF':'流動性等權靜態參考','A_equal':'A：20／60日動能選25檔、等權','B':'B：動能＋法人選股','C':'C：動能＋低波動選股','AH':'AH：A持有24日不換股','BH':'BH：B持有24日不換股','A_mild':'A：溫和動能排序配置','A_invvol50':'A：等權／逆波動各半配置','A_mild_tolerance':'A：溫和動能＋0.5%NAV小額調整略過','REV5':'五日短期反轉選股','pure_flow':'純法人20日選股','pure_lowvol':'純20日低波動選股','A_invvol100':'A：100%逆波動原始配置（仍受1.8%–5.4%界限）','pure_financial':'純財務：單季本業利益率改善選股原型','pure_revenue':'純營收：MoM／YoY百分位各半選股原型','financial_dual':'A：原財務雙條件配置','operating_no_self':'A：本業改善配置、不含自結','operating_self':'A：本業改善配置、納入核對自結','revenue_weight':'A：營收MoM／YoY雙率配置','combined':'A：50%動能／25%本業／25%營收配置','combined_tolerance':'A：上述組合＋0.5%NAV小額調整略過'}
lines.extend(['|代號|方法|','|---|---|']+[f'|{k}|{v}|' for k,v in method_labels_v4.items()])
lines.extend(['','## 六個公告配置原型：同季全部兩年點','','|方法|年份|費用後報酬|同期A|差pp|費用|','|---|---:|---:|---:|---:|---:|'])
for row in legacy_seasonal_rows_v3:
    lines.append(f"|{method_labels_v4[row['method']]}|{row['year']}|{row['return_pct']:.3f}%|{row['A_return_pct']:.3f}%|{row['difference_pp']:+.3f}|{row['cost_pct']:.3f}%|")
lines.extend(['','組合與組合加略過兩年均正向，僅屬趨勢線索；兩個年度點不足以給可靠多重比較區間，不能據此取代廣泛窗口結論。未知公告維持原型定義的中性或排除規則；沒有用LLM臆測法說權重。','','## 產業觀察的解讀','','航運含息指數同季10年中8年正報酬，中位數約5.62%；目前按9個完整月對照年份計算的家族校正探索區間為正。這支持把航運列為後續觀察候選，沒有證明可交易25檔方法有效。金融保險同季10年中9年上漲，中位數約3.59%，但相對同年其他月份的超額季節性區間跨零；高上漲比例不等於特殊季節優勢。區間尾端由5000次小年度樣本重抽樣估計，精度有限，分類家族彼此重疊、無真正樣本外驗證。','','電子／金融價格指數作獨立附錄，未含股息，不和上述含息表混排：','[價格指數10年逐年表](產業競賽期間_歷年比較_2026-10-14.html)','','主辦競賽即時頁本次403，僅能依已保存1005更新PDF與v4.3文件核對，不宣稱已確認沒有較新規則。'])
for index,line in enumerate(lines):
    for key,label in method_labels_v4.items():
        if line.startswith('|'+key+'|'):lines[index]=line.replace('|'+key+'|','|'+label+'（'+key+'）|',1)
report_path_v3.write_text('\n'.join(lines)+'\n',encoding='utf-8')
strategy_output_path_v3=recovery_root/'策略研究_統一比較結果_v3_2026-10-14.json'
strategy_output_path_v3.write_text(json.dumps({'engine':'v3 verified action overlay; existing saved summaries reused','raw_snapshot_sha256':snapshot_sha256,'overlay_sha256':overlay_sha_v3,'selection':final_selection_v3,'methods':final_strategy_stats_v3,'seasonal_announcement_prototypes':legacy_seasonal_rows_v3,'official_rules':official_rule_audit_v3,'sector_total_return_statistics':sector_tr_statistics_v1,'sector_price_statistics':sector_price_statistics_v1,'raw_curve_record':'Original Notebook executed v3 cells75/83 and prior embedded chart; not recomputed or recreated in minimal kernel recovery','original_notebook':str(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb')},ensure_ascii=False,indent=2),encoding='utf-8')
print('Final saved report:',report_path_v3.name,'result:',strategy_output_path_v3.name)
print('Final selection:',json.dumps(final_selection_v3,ensure_ascii=False))
print('FINAL_REPORT_SAVED')


In [ ]:
# Sector monthly alternative v1: independent official monthly industry price series
sector_price_months_v1=globals().get('sector_price_months_v1',{})
sector_price_cache_path_v1=recovery_root/'產業競賽期間_官方月報來源_2026-10-14.json'
if sector_price_cache_path_v1.exists():sector_price_months_v1.update(json.loads(sector_price_cache_path_v1.read_text(encoding='utf-8')))
missing=[k for k in month_keys if k not in sector_price_months_v1]
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    futures={pool.submit(sector_request_v1,'/rwd/zh/TAIEX/EFTRI_HIST',{'date':k,'response':'json'},'month'):k for k in missing}
    completed=0
    for f in concurrent.futures.as_completed(futures):
        key=futures[f];sector_price_months_v1[key]=f.result();completed+=1
        if completed%20==0 or completed==len(missing):
            sector_price_cache_path_v1.write_text(json.dumps(sector_price_months_v1,ensure_ascii=False,separators=(',',':')),encoding='utf-8')
            print('Official monthly industry progress:',completed,'/',len(missing),flush=True)
print('Monthly alternative coverage:',len(sector_price_months_v1),'errors:',sum('error_types' in v for v in sector_price_months_v1.values()))
print('SECTOR_PRICE_ALTERNATIVE_READY')


In [ ]:
# Source format diagnostic v1: inspect a single public response, not blind retries
diagnostic_month_v1=next(k for k,v in sector_price_months_v1.items() if 'error_types' in v)
diagnostic_response_v1=requests.get('https://www.twse.com.tw/rwd/zh/TAIEX/EFTRI_HIST',params={'date':diagnostic_month_v1,'response':'json'},timeout=20)
print('One-response format diagnosis:',{'month':diagnostic_month_v1,'status':diagnostic_response_v1.status_code,'content_type':diagnostic_response_v1.headers.get('Content-Type'),'encoding':diagnostic_response_v1.encoding,'first_bytes':repr(diagnostic_response_v1.content[:200])})
try:
    diagnostic_decoded_v1=json.loads(diagnostic_response_v1.content.decode('utf-8-sig'))
    print('Binary UTF8-sig decode:',diagnostic_decoded_v1.get('stat'),'rows:',len(diagnostic_decoded_v1.get('data',[])))
except Exception as exc:print('Binary decode result:',type(exc).__name__)
print('SOURCE_FORMAT_DIAGNOSTIC_READY')


## 官方產業月報替代觀察（價格指數，未含息）

逐日含息資料225端點中136個未能解析JSON、月報122份有部分同類問題。單筆格式診斷恢復可讀的2019-11月原報告並快取；不盲目重抓全部已達重試上限端點。以實際取得的電子／金融價格指數做獨立觀察，與同口徑大盤價格比較；不同含息／未含息資料分開，不混排名。十年目標未取全，缺格保持缺格，完整控制年份不足則不能證實季節性。

In [ ]:
# Price sector analysis v1: all years, original industry return indexes and figures
import numpy as np,matplotlib.pyplot as plt,html
from matplotlib import font_manager

if 'data' in globals().get('diagnostic_decoded_v1',{}):
    sector_price_months_v1[diagnostic_month_v1]={'url':diagnostic_response_v1.url,'fields':diagnostic_decoded_v1['fields'],'rows':diagnostic_decoded_v1['data'],'date':diagnostic_decoded_v1.get('date')}
    sector_price_cache_path_v1.write_text(json.dumps(sector_price_months_v1,ensure_ascii=False,separators=(',',':')),encoding='utf-8')
sector_levels_v1={}
for key,source in sector_price_months_v1.items():
    if 'rows' not in source:continue
    for row in source['rows']:
        date=sector_parse_date_v1(row[0]).strftime('%Y%m%d')
        levels=sector_levels_v1.setdefault(date,{})
        for name,value in zip(source['fields'][1:],row[1:]):
            try:
                v=float(str(value).replace(',',''))
                if v>0:levels[name]=v
            except (TypeError,ValueError):pass
for source in sector_month_cache_v1.values():
    if 'rows' not in source:continue
    for row in source['rows']:
        date=sector_parse_date_v1(row[0]).strftime('%Y%m%d')
        sector_levels_v1.setdefault(date,{})['發行量加權股價指數']=float(str(row[4]).replace(',',''))
sector_names_v1=sorted({name for levels in sector_levels_v1.values() for name in levels if name.endswith('類指數')})
sector_returns_v1=[];sector_window_gaps_v1=[]
benchmark_name='發行量加權股價指數'
for window in sector_windows_v1:
    if window['status']!='calendar_verified':sector_window_gaps_v1.append(window);continue
    before=sector_levels_v1.get(window['previous'].replace('-',''),{})
    after=sector_levels_v1.get(window['end'].replace('-',''),{})
    if benchmark_name not in before or benchmark_name not in after:
        sector_window_gaps_v1.append(dict(window,status='benchmark_missing'));continue
    base=after[benchmark_name]/before[benchmark_name]-1
    for name in sector_names_v1:
        if name not in before or name not in after:continue
        value=after[name]/before[name]-1
        sector_returns_v1.append(dict(window,sector=name,return_value=value,benchmark_return=base,excess=value-base,basis='TWSE sector price index; no dividends or costs'))
sector_record_lookup_v1={(r['sector'],r['year'],r['month']):r for r in sector_returns_v1}
sector_statistics_v1=[];sector_matrix_v1=np.full((len(sector_names_v1),len(sector_history_years_v1)),np.nan)
seasonality_arrays_v1={}
for i,name in enumerate(sector_names_v1):
    seasonal=[];annual_differences=[]
    for j,y in enumerate(sector_history_years_v1):
        oct_record=sector_record_lookup_v1.get((name,y,10))
        if oct_record:seasonal.append(oct_record);sector_matrix_v1[i,j]=oct_record['return_value']*100
        other=[sector_record_lookup_v1.get((name,y,m)) for m in range(1,13) if m!=10]
        if oct_record and all(other):
            annual_differences.append(oct_record['excess']-np.median([r['excess'] for r in other]))
    seasonality_arrays_v1[name]=np.array(annual_differences)
    if not seasonal:continue
    rr=np.array([r['return_value'] for r in seasonal]);ee=np.array([r['excess'] for r in seasonal])
    stat={'sector':name,'years':len(rr),'positive_years':int(np.sum(rr>0)),'positive_year_fraction':float(np.mean(rr>0)),'median_return_pct':float(np.median(rr)*100),'median_excess_pp':float(np.median(ee)*100),'mean_excess_pp':float(np.mean(ee)*100),'worst_return_pct':float(rr.min()*100),'full_control_years':len(annual_differences),'seasonal_mean_excess_difference_pp':float(np.mean(annual_differences)*100) if annual_differences else None,'ci_family_pp':None,'seasonal_advantage_supported':False}
    if len(annual_differences)>=5:
        rng=np.random.default_rng(SEED+20261014+i);a=np.array(annual_differences)
        means=np.mean(a[rng.integers(0,len(a),size=(5000,len(a)))],axis=1)
        alpha=.05/max(1,len(sector_names_v1));ci=np.quantile(means,[alpha/2,1-alpha/2])*100
        stat['ci_family_pp']=ci.tolist();stat['seasonal_advantage_supported']=bool(ci[0]>0)
    sector_statistics_v1.append(stat)
print('Sector analysis coverage:',{'classification_rows':len(sector_names_v1),'sector_month_windows':len(sector_returns_v1),'annual_seasonal_points':int(np.isfinite(sector_matrix_v1).sum()),'window_gaps':len(sector_window_gaps_v1),'complete_calendar_source':all('error_types' not in v for v in sector_month_cache_v1.values()),'endpoint_source_failures':sum('error_types' in v for v in sector_price_months_v1.values())})
print('Sector descriptions (full-year coverage first):',json.dumps(sorted(sector_statistics_v1,key=lambda r:(-r['years'],-r['median_return_pct'])),ensure_ascii=False))
plt.rcParams['font.family']='Microsoft JhengHei';plt.rcParams['axes.unicode_minus']=False
fig,ax=plt.subplots(figsize=(15,max(7,len(sector_names_v1)*.38)))
limit=max(5,float(np.nanpercentile(np.abs(sector_matrix_v1),95)))
cmap=plt.get_cmap('RdBu_r').copy();cmap.set_bad('#e5e7eb')
im=ax.imshow(sector_matrix_v1,aspect='auto',cmap=cmap,vmin=-limit,vmax=limit)
ax.set_xticks(range(len(sector_history_years_v1)),sector_history_years_v1)
ax.set_yticks(range(len(sector_names_v1)),[s.replace('類指數','') for s in sector_names_v1])
for i in range(len(sector_names_v1)):
    for j in range(len(sector_history_years_v1)):
        v=sector_matrix_v1[i,j]
        ax.text(j,i,'—' if not np.isfinite(v) else f'{v:.1f}%',ha='center',va='center',fontsize=8,color='white' if np.isfinite(v) and abs(v)>.75*limit else '#111827')
ax.set_title('歷年競賽時段：十月最後星期二起 24 交易日\n官方產業價格（未含息）指數；每年每產業全部點保留（非可交易策略、未扣交易費）',fontsize=13)
fig.colorbar(im,ax=ax,label='24交易日指數報酬（%）；顏色截尾，格內顯示實際值',fraction=.025)
fig.tight_layout();sector_heatmap_path_v1=recovery_root/'產業競賽期間_歷年指數熱圖_2026-10-14.png';fig.savefig(sector_heatmap_path_v1,dpi=150,bbox_inches='tight');plt.close(fig)
fig,ax=plt.subplots(figsize=(13,max(7,len(sector_names_v1)*.36)))
valid_rows=[i for i in range(len(sector_names_v1)) if np.isfinite(sector_matrix_v1[i]).any()]
data=[sector_matrix_v1[i,np.isfinite(sector_matrix_v1[i])] for i in valid_rows]
ax.boxplot(data,vert=False,showfliers=False,medianprops={'color':'#db2777','linewidth':2},tick_labels=[sector_names_v1[i].replace('類指數','') for i in valid_rows])
rng=np.random.default_rng(SEED)
for position,values in enumerate(data,1):ax.scatter(values,position+rng.uniform(-.15,.15,len(values)),s=18,alpha=.8,color='#2563eb',zorder=3)
ax.axvline(0,color='#111827',linestyle='--',linewidth=1.2);ax.set_xlabel('24交易日指數報酬（%）')
ax.set_title('每個歷年競賽時段全部點；桃紅為中位數、黑色虛線為0%');fig.tight_layout()
sector_box_path_v1=recovery_root/'產業競賽期間_全部年度箱形圖_2026-10-14.png';fig.savefig(sector_box_path_v1,dpi=150,bbox_inches='tight');plt.close(fig)
rows_html=[]
for name in sector_names_v1:
    cells=[]
    for y in sector_history_years_v1:
        rec=sector_record_lookup_v1.get((name,y,10))
        if not rec:cells.append('<td title="該分類名稱／原始指數資料不足">—</td>');continue
        v=rec['return_value']*100
        title=f"{rec['start']}～{rec['end']}｜基準 {rec['benchmark_return']*100:.2f}%｜超額 {rec['excess']*100:+.2f}pp｜起算前收盤 {rec['previous']}｜官方價格（未含息）指數、未扣交易費"
        color=f'hsl({5 if v>=0 else 215},70%,{max(70,97-min(abs(v),30)*.8):.1f}%)'
        cells.append(f'<td style="background:{color}" title="{html.escape(title)}">{v:+.2f}%</td>')
    rows_html.append('<tr><th>'+html.escape(name.replace('類指數',''))+'</th>'+''.join(cells)+'</tr>')
sector_html_path_v1=recovery_root/'產業競賽期間_歷年比較_2026-10-14.html'
sector_html_path_v1.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>歷年競賽期間產業觀察</title><style>body{font-family:Microsoft JhengHei,sans-serif;margin:24px;color:#172033}table{border-collapse:collapse;width:100%}td,th{padding:9px;border:1px solid #d7dce3;text-align:right}th:first-child{text-align:left;white-space:nowrap}thead{position:sticky;top:0;background:white}p{line-height:1.7}</style><h1>歷年競賽期間：產業價格（未含息）指數</h1><p>十月最後一個星期二起24交易日。游標停在每一格可看日期、大盤與超額。所有年份保留；缺值不是0%。此為上市分類指數觀察，非150檔等權策略、未扣費用；沒有推算產業內個股上漲比例。</p><table><thead><tr><th>產業</th>'+''.join(f'<th>{y}</th>' for y in sector_history_years_v1)+'</tr></thead><tbody>'+''.join(rows_html)+'</tbody></table><p>同年其他11月份及家族校正另存Notebook；分類名稱改變不臆測等價合併，部分分類年份較少。資料：證交所每日收盤行情價格（未含息）分類指數。</p></html>',encoding='utf-8')
sector_stats_path_v1=recovery_root/'產業競賽期間_統計與全部窗口_2026-10-14.json'
sector_stats_path_v1.write_text(json.dumps({'definition':'last Tuesday each month,24 sessions, prior close to final close; official total return indexes; no costs','years':sector_history_years_v1,'statistics':sector_statistics_v1,'windows':sector_returns_v1,'gaps':sector_window_gaps_v1},ensure_ascii=False,indent=2),encoding='utf-8')
print('Sector artifacts saved:',sector_heatmap_path_v1.name,sector_box_path_v1.name,sector_html_path_v1.name)
print('FINAL_SECTOR_ANALYSIS_READY')

print("PRICE_SECTOR_ANALYSIS_READY; missing official responses kept missing, not zeros.")

sector_price_statistics_v1=sector_statistics_v1
sector_price_returns_v1=sector_returns_v1


In [ ]:
# Legacy seasonal v3: same fixed announcement weighting prototypes, only original two anchors
current_nb=json.loads((recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').read_text(encoding='utf-8'))
def restore_selected_functions_v3(index,names):
    source=''.join(current_nb['cells'][index]['source'])
    tree=ast.parse(source)
    for node in tree.body:
        if isinstance(node,ast.FunctionDef) and node.name in names:exec(compile(ast.Module(body=[node],type_ignores=[]),'<legacy-fixed-function>','exec'),globals())
def prestart_original_amounts(ann):
    body=unicodedata.normalize('NFKC',ann.get('content') or '').replace('\r','');vals={}
    for field,number,label in [('revenue',4,'營業收入'),('operating',6,'營業利益'),('parent_net',9,'歸屬於母公司')]:
        match=re.search(r'(?m)^\s*'+str(number)+r'[.、]\s*([^\n]+)',body)
        if not match or label not in match[1] or '累計' not in match[1] or ':' not in match[1]:return None
        v=match[1].rsplit(':',1)[1].strip().replace(',','').replace(' ','')
        if re.fullmatch(r'\(?-?\d+(?:\.\d+)?\)?',v) is None:return None
        vals[field]=float(v.strip('()'))*(-1 if v.startswith('(') else 1)
    return vals if vals['revenue']>0 else None
a_announcement_documents={(y,sid):{'announcements':full_pool_ann_by_stock_v1[sid]} for y in [2024,2025] for sid in stock_ids}
restore_selected_functions_v3(13,{'a_financial_evidence_asof'})
restore_selected_functions_v3(19,{'a_operating_evidence'})
legacy_evidence_cache_v3={}
def legacy_announcement_allocation_v3(t,selected,mode):
    if mode=='equal':return np.full(COUNT,INVESTED/COUNT)
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));mult_dual=[];mult_op=[];mult_no_self=[];mult_rev=[]
    for j in selected:
        sid=stock_ids[j];key=(t,sid)
        if key not in legacy_evidence_cache_v3:
            raw=a_financial_evidence_asof(cutoff.year,sid,cutoff)
            op=a_operating_evidence(raw);legacy_evidence_cache_v3[key]=(raw,op)
        raw,op=legacy_evidence_cache_v3[key]
        delta=raw.get('op_margin_change_pp')
        no_self=0 if raw.get('unintegrated_sources') or raw['status']!='可核對' or delta is None else 1 if delta>=1 else -1 if delta<=-1 else 0
        mult_dual.append(1+.25*raw['score']);mult_op.append(1+.25*op['score']);mult_no_self.append(1+.25*no_self)
        rv=pure_revenue_evidence_v1[(t,sid)]
        score=0
        if rv['status']=='verified_original_known_available_time':score=1 if rv['mom']>0 and rv['yoy']>0 else -1 if rv['mom']<0 and rv['yoy']<0 else 0
        mult_rev.append(1+.25*score)
    def weights(x):x=np.array(x,dtype=float);return INVESTED*x/x.sum()
    if mode=='financial_dual':return weights(mult_dual)
    if mode=='operating_self':return weights(mult_op)
    if mode=='operating_no_self':return weights(mult_no_self)
    if mode=='revenue_weight':return weights(mult_rev)
    assert mode=='combined'
    return .5*engine_v3_allocation(t,selected,'mild')+.25*weights(mult_op)+.25*weights(mult_rev)
legacy_src_v3=src_v3.replace('def simulate_engine_v3(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):','def simulate_legacy_announcement_v3(start,allocation,horizon=HORIZON,tolerance=0):\n    strategy="A"').replace('engine_v3_allocation(','legacy_announcement_allocation_v3(')
exec(legacy_src_v3)
legacy_seasonal_results_v3={};legacy_seasonal_rows_v3=[]
for year,date in [(2024,datetime.date(2024,10,29)),(2025,datetime.date(2025,10,28))]:
    start=di[date];pos=int(np.flatnonzero(window_starts==start)[0])
    control=simulate_legacy_announcement_v3(start,'equal')
    assert np.allclose(control['path'],engine_v3_results['A_equal'][pos]['path'],rtol=1e-12,atol=1e-6)
    for name,mode,tolerance in [('financial_dual','financial_dual',0),('operating_no_self','operating_no_self',0),('operating_self','operating_self',0),('revenue_weight','revenue_weight',0),('combined','combined',0),('combined_tolerance','combined',.005)]:
        result=simulate_legacy_announcement_v3(start,mode,tolerance=tolerance);legacy_seasonal_results_v3[(year,name)]=result
        legacy_seasonal_rows_v3.append({'year':year,'method':name,'valid':not result['invalid'] and result['violations']==0,'return_pct':float(result['return']*100) if not result['invalid'] else None,'A_return_pct':float(control['return']*100) if not control['invalid'] else None,'cost_pct':float(result['cost']*100),'difference_pp':float((result['return']-control['return'])*100) if not result['invalid'] and not control['invalid'] else None})
print('Legacy two-anchor revalidation:',json.dumps(legacy_seasonal_rows_v3,ensure_ascii=False))
print('Only 2 seasonal anchors; no reliable family interval or winner declaration. Expanded original-document coverage noted; raw v1 outputs preserved.')
benchmark_tables_v3=sorted({r['TABLE_NAME'] for r in remaining_full_schema_v1 if re.search(r'etf|benchmark|holding',r['TABLE_NAME'],re.I)})
print('Potential ETF/benchmark schema:',{t:[r['COLUMN_NAME'] for r in remaining_full_schema_v1 if r['TABLE_NAME']==t] for t in benchmark_tables_v3})
print('LEGACY_SEASONAL_READY')


In [ ]:
# Safe legacy exception metadata
last_error=get_ipython().last_execution_result.error_in_exec
print(type(last_error).__name__,str(last_error) if isinstance(last_error,NameError) else 'details suppressed')


In [ ]:
# Active Share source audit v3: actual ETF-history coverage, no invented benchmarks
as_tree=ast.parse(inspect.getsource(fetch_full_pool_announcements_v1));as_tree.body[0].name='fetch_active_etf_audit_v3'
as_try=next(x for x in as_tree.body[0].body if isinstance(x,ast.Try));as_with=next(x for x in as_try.body if isinstance(x,ast.With))
as_with.body=ast.parse("""
cur.execute('SET SESSION TRANSACTION READ ONLY')
cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
cur.execute('SELECT etf_id,name,invests_tw,listed_date FROM active_etfs ORDER BY etf_id')
rows=cur.fetchall()
cur.execute('SELECT etf_id,MIN(data_date) AS first_date,MAX(data_date) AS last_date,COUNT(*) AS rows_count,COUNT(DISTINCT data_date) AS date_count,MIN(weight) AS min_weight,MAX(weight) AS max_weight FROM active_etf_holdings GROUP BY etf_id ORDER BY etf_id')
schema=cur.fetchall()
""").body
ast.fix_missing_locations(as_tree);exec(compile(as_tree,'<readonly-active-etf-audit>','exec'))
active_etf_inventory_v3,active_etf_coverage_v3=fetch_active_etf_audit_v3()
print('Active ETF listings:',json.dumps(active_etf_inventory_v3,default=str,ensure_ascii=False))
print('Active ETF history coverage:',json.dumps(active_etf_coverage_v3,default=str,ensure_ascii=False))
print('Data_date is holdings as-of date, not proven publication time; coverage alone does not establish full historical compliance.')
print('ACTIVE_ETF_SOURCE_AUDIT_READY')


## 官方CSV下載替代

不重試已達上限的JSON來源；只補缺失日期，使用同頁官網明列的CSV下載格式作獨立合法替代。低頻單工、每次至少間隔1.2秒，各下載至多3次且保留原文格式診斷；連續3份替代下載失敗就停止該來源批次。成功快取不重抓。不更改代理、身份或TLS驗證，不繞過網站驗證。

In [ ]:
# Official CSV alternative v1: probe normal public export format with explicit date checks
import csv,io,time
csv_alternative_attempts_v1=globals().get('csv_alternative_attempts_v1',{})
def official_csv_alternative_v1(kind,key):
    path='/rwd/zh/TAIEX/EFTRI_HIST' if kind=='month' else '/rwd/zh/afterTrading/MI_INDEX'
    params={'date':key,'response':'csv'}
    if kind=='day':params['type']='IND'
    errors=[]
    for attempt in range(3):
        try:
            response=requests.get('https://www.twse.com.tw'+path,params=params,timeout=20);response.raise_for_status()
            try:body=response.content.decode('utf-8-sig');source_encoding='utf-8-sig'
            except UnicodeDecodeError:body=response.content.decode('cp950');source_encoding='cp950'
            raw=[[c.lstrip('=').strip().strip('"') for c in row] for row in csv.reader(io.StringIO(body))]
            if '<html' in body.lower() or '<!doctype' in body.lower():raise ValueError('CSV export returned HTML, not data')
            if kind=='month':
                header=next((row for row in raw if row and row[0].replace(' ','').replace('\u3000','')=='日期' and any('電子類指數' in s for s in row)),None)
                if header is None:raise ValueError('Industry CSV header absent')
                rows=[row[:len(header)] for row in raw if row and re.fullmatch(r'\d{2,4}/\d{2}/\d{2}',row[0].strip()) and len(row)>=len(header)]
                assert rows and all(sector_parse_date_v1(row[0]).strftime('%Y%m')==key[:6] for row in rows)
                result={'url':response.url,'fields':header,'rows':rows,'date':key,'representation':'official CSV alternative','source_encoding':source_encoding}
            else:
                target_date=datetime.date(int(key[:4]),int(key[4:6]),int(key[6:]))
                patterns=[f'{target_date.year-1911}年{target_date.month:02d}月{target_date.day:02d}日',f'{target_date.year-1911}/{target_date.month:02d}/{target_date.day:02d}']
                if not any(p in body for p in patterns):raise ValueError('Daily CSV header date not verified')
                rows=[]
                for row in raw:
                    if len(row)<2:continue
                    name=row[0].strip()
                    if name.endswith('類報酬指數') or name=='發行量加權股價報酬指數':
                        value=float(row[1].replace(',',''));assert value>0;rows.append([name,str(value)])
                assert rows and any(row[0]=='發行量加權股價報酬指數' for row in rows)
                result={'url':response.url,'date':key,'tables':[{'title':'報酬指數(臺灣證券交易所) CSV','fields':['報酬指數','收盤指數'],'data':rows}],'representation':'official CSV alternative','source_encoding':source_encoding}
            csv_alternative_attempts_v1[(kind,key)]={'attempts':attempt+1,'status':'verified'}
            return result
        except Exception as exc:
            errors.append(type(exc).__name__+': '+str(exc)[:90] if isinstance(exc,(AssertionError,ValueError)) else type(exc).__name__)
            if attempt<2:time.sleep(4*(attempt+1))
    csv_alternative_attempts_v1[(kind,key)]={'attempts':3,'status':'failed','errors':errors}
    return {'error_types':errors,'representation':'official CSV alternative','source_encoding':source_encoding,'params':params}
csv_probe_month_key_v1=next(k for k,v in sector_price_months_v1.items() if 'error_types' in v)
csv_probe_month_v1=official_csv_alternative_v1('month',csv_probe_month_key_v1)
print('CSV alternative month probe:',csv_probe_month_key_v1,{'status':'verified' if 'rows' in csv_probe_month_v1 else 'failed','fields':csv_probe_month_v1.get('fields'),'rows':len(csv_probe_month_v1.get('rows',[])),'errors':csv_probe_month_v1.get('error_types')})
if 'rows' in csv_probe_month_v1:
    sector_price_months_v1[csv_probe_month_key_v1]=csv_probe_month_v1
    sector_price_cache_path_v1.write_text(json.dumps(sector_price_months_v1,ensure_ascii=False,separators=(',',':')),encoding='utf-8')
csv_probe_day_key_v1=next(k for k,v in sector_day_cache_v1.items() if 'error_types' in v)
csv_probe_day_v1=official_csv_alternative_v1('day',csv_probe_day_key_v1)
print('CSV alternative day probe:',csv_probe_day_key_v1,{'status':'verified' if 'tables' in csv_probe_day_v1 else 'failed','rows':len(csv_probe_day_v1.get('tables',[{}])[0].get('data',[])),'errors':csv_probe_day_v1.get('error_types')})
if 'tables' in csv_probe_day_v1:
    sector_day_cache_v1[csv_probe_day_key_v1]=csv_probe_day_v1;sector_save_cache_v1()
print('CSV_ALTERNATIVE_PROBE_READY')


In [ ]:
# Official CSV recovery v1: bounded low-frequency completion from verified export format
csv_recovery_summary_v1=[]
for kind,cache in [('month',sector_price_months_v1),('day',sector_day_cache_v1)]:
    missing=sorted(k for k,v in cache.items() if 'error_types' in v)
    consecutive_failures=0;successes=0
    for i,key in enumerate(missing,1):
        time.sleep(1.2)
        result=official_csv_alternative_v1(kind,key)
        if 'error_types' in result:
            consecutive_failures+=1
            cache[key]['csv_alternative_failure']=result
        else:
            result['prior_json_error_types']=cache[key].get('error_types',[])
            cache[key]=result;consecutive_failures=0;successes+=1
        if i%10==0 or i==len(missing) or consecutive_failures>=3:
            sector_price_cache_path_v1.write_text(json.dumps(sector_price_months_v1,ensure_ascii=False,separators=(',',':')),encoding='utf-8')
            sector_save_cache_v1()
            print('CSV verified progress:',kind,i,'/',len(missing),'successful:',successes,'consecutive failures:',consecutive_failures,flush=True)
        if consecutive_failures>=3:
            print('Stop exhausted CSV alternative batch for:',kind,flush=True);break
    csv_recovery_summary_v1.append({'kind':kind,'initial_missing':len(missing),'recovered':successes,'still_missing':sum('error_types' in v for v in cache.values())})
print('Final official source completion:',csv_recovery_summary_v1)
print('OFFICIAL_CSV_RECOVERY_READY')


In [ ]:
# Final source QA and minimal kernel metadata; do not resubmit source requests
print('Kernel metadata:', {'dates':len(dates),'stock_ids':len(stock_ids),'engine_methods':len(engine_v3_results),'comparison_rows':len(final_strategy_stats_v3),'legacy_rows':len(legacy_seasonal_rows_v3)})
for name in ['sector_month_cache_v1','sector_price_months_v1','sector_day_cache_v1']:
    cache=globals()[name]
    failures=[key for key,value in cache.items() if 'error_types' in value]
    assert not failures,(name,failures)
    for key,value in cache.items():
        if value.get('date'):assert str(value['date']).replace('-','')[:8]==key[:8],(name,key,value['date'])
    print('Source QA:',name,len(cache),'failed',len(failures))
assert len(sector_windows_v1)==120
assert all(w['status']=='calendar_verified' for w in sector_windows_v1)
print('Source window sample:',sector_windows_v1[0],sector_windows_v1[-1])
print('FINAL_SOURCE_QA_READY')


In [ ]:
# Authorized minimal recovery: saved verified sources/results, no DB or strategy reruns
from research_layout import ResearchPath as Path
import json,datetime,numpy as np,nbformat
recovery_root=Path(r'C:\Users\weiha\Desktop\文件\03_課程資料\115-1\資料探勘與應用\B_Final Project (0920~1221)\競賽資料')
saved_nb_v4=nbformat.read(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
saved_state_v4=json.loads((recovery_root/'A策略_無人值守研究狀態.json').read_text(encoding='utf-8'))
saved_sector_v4=json.loads((recovery_root/'產業競賽期間_官方指數來源_2026-10-14.json').read_text(encoding='utf-8'))
print('Saved source schema:',list(saved_sector_v4))
final_strategy_stats_v3=[]
for out in saved_nb_v4.cells[83].outputs:
    for line in out.get('text','').splitlines():
        if line.startswith('Family validation: '):final_strategy_stats_v3.append(json.loads(line[len('Family validation: '):]))
assert len(final_strategy_stats_v3)==14
final_supported_v3=[x['method'] for x in final_strategy_stats_v3 if x['supported_research_improvement'] and x['method'] not in ['REF','AH','BH']]
legacy_seasonal_rows_v3=[]
for out in saved_nb_v4.cells[89].outputs:
    for line in out.get('text','').splitlines():
        if line.startswith('Legacy two-anchor revalidation: '):legacy_seasonal_rows_v3=json.loads(line[len('Legacy two-anchor revalidation: '):])
assert len(legacy_seasonal_rows_v3)==12
print('Restored verified output summaries:',len(final_strategy_stats_v3),len(legacy_seasonal_rows_v3),'unproved',final_supported_v3)
print('MINIMAL_RECOVERY_READY')

import bisect,calendar
SEED=20261013
FEE=.001425
TAX=.003
sector_history_years_v1=saved_sector_v4["years"]
sector_month_cache_v1=saved_sector_v4["months"]
sector_day_cache_v1=saved_sector_v4["days"]
sector_price_months_v1=json.loads((recovery_root/"產業競賽期間_官方月報來源_2026-10-14.json").read_text(encoding="utf-8"))
def sector_parse_date_v1(s):
    y,m,d=map(int,s.split('/'));return datetime.date(y+1911 if y<1911 else y,m,d)
month_keys=sorted(sector_month_cache_v1)
sector_calendar_v1=sorted({sector_parse_date_v1(row[0]) for v in sector_month_cache_v1.values() if 'rows' in v for row in v['rows']})
sector_windows_v1=[]
for year in sector_history_years_v1:
    for month in range(1,13):
        last=datetime.date(year,month,calendar.monthrange(year,month)[1])
        anchor=last-datetime.timedelta(days=(last.weekday()-1)%7)
        i=bisect.bisect_left(sector_calendar_v1,anchor)
        if i==0 or i+23>=len(sector_calendar_v1):
            sector_windows_v1.append({'year':year,'month':month,'anchor':str(anchor),'status':'calendar_missing'});continue
        # All intervening official calendar months must have successfully loaded.
        start,end=sector_calendar_v1[i],sector_calendar_v1[i+23]
        needed=[k for k in month_keys if start.replace(day=1)<=datetime.date(int(k[:4]),int(k[4:6]),1)<=end.replace(day=1)]
        if any('error_types' in sector_month_cache_v1[k] for k in needed):
            sector_windows_v1.append({'year':year,'month':month,'anchor':str(anchor),'status':'calendar_gap'});continue
        sector_windows_v1.append({'year':year,'month':month,'anchor':str(anchor),'start':str(start),'end':str(end),'previous':str(sector_calendar_v1[i-1]),'status':'calendar_verified'})

assert len(sector_windows_v1)==120
assert all(x["status"]=="calendar_verified" for x in sector_windows_v1)
for cache in [sector_month_cache_v1,sector_price_months_v1,sector_day_cache_v1]:
    assert not any("error_types" in x for x in cache.values())
print("Source QA:",len(sector_month_cache_v1),len(sector_price_months_v1),len(sector_day_cache_v1),"calendar windows",len(sector_windows_v1))
print("Restored metadata", {k:type(globals()[k]).__name__ for k in ["sector_windows_v1","final_strategy_stats_v3","legacy_seasonal_rows_v3"]})


In [ ]:
e=get_ipython().last_execution_result.error_in_exec
print(type(e).__name__,str(e) if isinstance(e,NameError) else "safe summary only")

In [ ]:
# Inspect the single retained benchmark gap; no new requests
tr_saved_v4=json.loads((recovery_root/'產業競賽期間_含息統計與全部窗口_2026-10-14.json').read_text(encoding='utf-8'))
print('TR gaps:',tr_saved_v4['gaps'])
for w in tr_saved_v4['gaps']:
    for k in ['previous','end']:
        source=sector_day_cache_v1[w[k].replace('-','')]
        print(w[k],[(t['title'],len(t.get('data',[])),t.get('data',[])[:2]) for t in source.get('tables',[])],source.get('url'))
print('State hash metadata:',saved_state_v4.get('company_action_overlay'),saved_state_v4.get('engine_revalidation'))


In [ ]:
# Curve persistence recovery: read-only current kernel metadata
import sys
print('Kernel:',sys.executable)
print('Available recovery objects:',{k:type(globals()[k]).__name__ for k in ['raw_data','dates','scores','engine_v3_results','final_strategy_stats_v3','saved_nb_v4','sector_month_cache_v1'] if k in globals()})
print('Curve reconstruction is existing experiment recovery, no new candidates.')


## 逐窗口結果保存修復

2026-10-14使用者要求繼續，補足原kernel結束前未匯出v3逐窗口結果的缺口。沿用目前kernel與原Notebook程式，唯讀重取同截止資料；原行情快照SHA256必須完全一致，公告資訊覆蓋與最終14個候選的既有配對摘要須逐項一致，不一致即停止，不混用。這是恢復既有實驗／持久化，不新增候選，不重新抽區間、不依新績效調參。每個方法保存650窗口，未知窗口保留invalid並排除績效。

In [ ]:
# Curve recovery stage 1: exact raw snapshot and original engine definitions
curve_saved_nb_v4=nbformat.read(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
def curve_run_saved_v4(source):
    response=get_ipython().run_cell(source,store_history=True)
    if not response.success:raise RuntimeError('Nested recovery cell failed; stop before reconstruction')
curve_run_saved_v4(curve_saved_nb_v4.cells[51].source.replace('%whos','print("Recovering exact original data snapshot, no large object output")'))
src=curve_saved_nb_v4.cells[52].source.split("weight_experiment_results={'equal':")[0]
curve_run_saved_v4(src)
assert len(window_starts)==650 and COUNT==25 and LOT==1000 and HORIZON==24 and REBALANCE==5
print('Exact raw snapshot restored; matrices',shape,'windows',len(window_starts))


In [ ]:
# Curve recovery stage 2: original point-in-time parsers and sources
for index in [57,58,59,62,63]:
    source=curve_saved_nb_v4.cells[index].source
    if index==62:source=source.replace("len(weight_experiment_results['equal'])",'len(window_starts)')
    curve_run_saved_v4(source)
assert len(full_pool_financial_announcements_v1)==4442
assert len(full_pool_revenue_rows_v1)==5320 and len(full_pool_revenue_news_v1)==6704
assert len(pure_revenue_records_v1)==1280
print('Prior parser/source coverage matched; per-method final summary still must match.')


In [ ]:
# Curve recovery stage 3: unchanged v3 source; persist each matched method immediately
import ast,hashlib,os
good_exec_v2=np.isfinite(execution)&(execution>0)&np.isfinite(volume)&(volume>0)&np.isfinite(turnover)&(turnover>0)&np.isfinite(close)&(close>0)
v2_tree=ast.parse(curve_saved_nb_v4.cells[68].source)
for node in v2_tree.body:
    if isinstance(node,ast.FunctionDef) and node.name=='engine_v2_allocation':exec(compile(ast.Module(body=[node],type_ignores=[]),'<retained-v2-allocation>','exec'))
    if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='engine_v2_fixed_methods' for t in node.targets):exec(compile(ast.Module(body=[node],type_ignores=[]),'<retained-method-definitions>','exec'))
action_corrections_staged_v1=[]
for out in curve_saved_nb_v4.cells[71].outputs:
    for line in out.get('text','').splitlines():
        if line.startswith('Verified originals: '):action_corrections_staged_v1=json.loads(line[len('Verified originals: '):])
assert len(action_corrections_staged_v1)==1
source=curve_saved_nb_v4.cells[75].source
source=source[:source.index("if 'engine_v3_results' not in globals():")]
curve_run_saved_v4(source)
assert overlay_sha_v3==saved_state_v4['company_action_overlay']['overlay_sha256']
curve_engine_digest_v4=hashlib.sha256((source+curve_saved_nb_v4.cells[68].source).encode('utf-8')).hexdigest()
curve_output_path_v4=recovery_root/'策略研究_逐窗口結果_v3_2026-10-14.json'
def curve_json_value_v4(value):
    if isinstance(value,np.ndarray):return [curve_json_value_v4(x) for x in value.tolist()]
    if isinstance(value,np.generic):return curve_json_value_v4(value.item())
    if isinstance(value,float) and not np.isfinite(value):return None
    if isinstance(value,dict):return {str(k):curve_json_value_v4(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [curve_json_value_v4(v) for v in value]
    if isinstance(value,(datetime.date,datetime.datetime)):return str(value)
    return value
curve_export_v4={'schema_version':1,'status':'in_progress','purpose':'restore and persist existing experiments; no new method or inference','data_snapshot_sha256':snapshot_sha256,'company_action_overlay_sha256':overlay_sha_v3,'engine_definition_sha256':curve_engine_digest_v4,'definition':{'stock_count':COUNT,'horizon':HORIZON,'rebalance':REBALANCE,'lot':LOT,'invested':INVESTED,'buy_sell_fee':FEE,'sell_tax':TAX},'window_starts':[str(dates[int(s)]) for s in window_starts],'fixed_methods':engine_v2_fixed_methods,'verified_methods':{},'windows':{}}
engine_v3_results={};engine_v3_metrics={};engine_v3_valid={}
for name in ['A_equal']+[x for x in engine_v2_fixed_methods if x!='A_equal']:
    strategy,allocation,tolerance=engine_v2_fixed_methods[name]
    rows=[simulate_engine_v3(int(start),strategy,allocation,tolerance=tolerance) for start in window_starts]
    engine_v3_results[name]=rows
    engine_v3_metrics[name]={f:np.array([r[f] for r in rows]) for f in ['return','mdd','cost','turnover','violations','strict_cap_days','min_held','max_held']}
    engine_v3_valid[name]=np.array([not r['invalid'] and r['violations']==0 for r in rows])
    assert len(rows)==650
    if name=='A_equal':
        assert int(engine_v3_valid[name].sum())==456
        for day,expected in [('2024-10-29',.06804579304691871),('2025-10-28',.04703062307847983)]:
            result=next(r for r in rows if str(dates[r['start']])==day)
            assert not result['invalid'] and np.isclose(result['return'],expected,rtol=1e-11,atol=1e-12)
        check={'A_valid_windows':456,'seasonal_controls_match':True}
    else:
        expected=next(r for r in final_strategy_stats_v3 if r['method']==name)
        pair=engine_v3_valid[name]&engine_v3_valid['A_equal'];candidate=engine_v3_metrics[name];base=engine_v3_metrics['A_equal']
        check={'paired_windows':int(pair.sum()),'median_return_pct':float(np.median(candidate['return'][pair])*100),'A_same_dates_median_pct':float(np.median(base['return'][pair])*100),'mean_difference_pp':float(np.mean(candidate['return'][pair]-base['return'][pair])*100),'mean_cost_pct':float(np.mean(candidate['cost'][pair])*100),'A_mean_cost_pct':float(np.mean(base['cost'][pair])*100),'P10_difference_pp':float((np.quantile(candidate['return'][pair],.1)-np.quantile(base['return'][pair],.1))*100),'mdd_difference_pp':float((np.median(candidate['mdd'][pair])-np.median(base['mdd'][pair]))*100)}
        for key,value in check.items():assert np.isclose(value,expected[key],rtol=1e-10,atol=1e-10),(name,key,'prior verified summary mismatch')
    curve_export_v4['verified_methods'][name]=check
    curve_export_v4['windows'][name]=[dict(curve_json_value_v4(r),start_date=str(dates[r['start']]),end_date=str(dates[r['start']+HORIZON-1])) for r in rows]
    checkpoint=curve_output_path_v4.with_suffix('.json.new')
    checkpoint.write_text(json.dumps(curve_export_v4,ensure_ascii=False,separators=(',',':'),allow_nan=False),encoding='utf-8')
    os.replace(checkpoint,curve_output_path_v4)
    print('Matched and persisted:',name,'650 windows; valid',int(engine_v3_valid[name].sum()),flush=True)
curve_export_v4['status']='all_methods_matched_saved_v3_statistics'
checkpoint=curve_output_path_v4.with_suffix('.json.new');checkpoint.write_text(json.dumps(curve_export_v4,ensure_ascii=False,separators=(',',':'),allow_nan=False),encoding='utf-8');os.replace(checkpoint,curve_output_path_v4)
print('Saved curve keys:',list(engine_v3_results['A_equal'][0]))
print('Full result saved:',curve_output_path_v4.name,'bytes',curve_output_path_v4.stat().st_size)
print('ALL_V3_CURVES_RESTORED_MATCHED_AND_SAVED')


In [ ]:
# Persist source snapshot and render all matched v3 points with fixed comparison axes
import gzip,html,math
source_cache_path_v4=recovery_root/'策略研究_原資料快照_v3_2026-10-14.json.gz'
canonical_snapshot_v4=json.dumps(raw_data,default=str,ensure_ascii=False,separators=(',',':')).encode('utf-8')
assert hashlib.sha256(canonical_snapshot_v4).hexdigest()==snapshot_sha256
with gzip.open(source_cache_path_v4,'wb') as file:file.write(canonical_snapshot_v4)
recovered_curve_v4=json.loads(curve_output_path_v4.read_text(encoding='utf-8'))
assert recovered_curve_v4['status']=='all_methods_matched_saved_v3_statistics'
assert len(recovered_curve_v4['windows'])==15 and all(len(x)==650 for x in recovered_curve_v4['windows'].values())
curve_order_v4=list(engine_v2_fixed_methods)
valid_values_v4=[r['return']*100 for name in curve_order_v4 for r in engine_v3_results[name] if not r['invalid'] and r['violations']==0]
xlo_v4=5*math.floor(min(0,min(valid_values_v4))/5)-5;xhi_v4=5*math.ceil(max(0,max(valid_values_v4))/5)+5
chart_width_v4=1400;chart_left_v4=420;chart_right_v4=1300;chart_height_v4=70+60*len(curve_order_v4)
def chart_x_v4(v):return chart_left_v4+(v-xlo_v4)/(xhi_v4-xlo_v4)*(chart_right_v4-chart_left_v4)
curve_chart_counts_v4={}
def render_curve_chart_v4(scope):
    svg=[f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {chart_width_v4} {chart_height_v4}" role="img" aria-label="v3全部窗口箱形圖">']
    for tick in np.linspace(xlo_v4,xhi_v4,9):
        x=chart_x_v4(tick);svg.append(f'<line x1="{x}" x2="{x}" y1="25" y2="{chart_height_v4-30}" stroke="#e5e7eb"/><text x="{x}" y="{chart_height_v4-8}" text-anchor="middle" font-size="13">{tick:.0f}%</text>')
    x=chart_x_v4(0);svg.append(f'<line x1="{x}" x2="{x}" y1="25" y2="{chart_height_v4-30}" stroke="#111827" stroke-dasharray="6 4" stroke-width="2"/>')
    total=0
    for i,name in enumerate(curve_order_v4):
        y=45+i*60;label=method_labels_v4[name]
        if name=='A_invvol100':label='A：100%逆波動配置（有權重上下限）'
        rows=[r for r in engine_v3_results[name] if not r['invalid'] and r['violations']==0 and (scope=='all' or str(dates[r['start']]) in ['2024-10-29','2025-10-28'])]
        total+=len(rows)
        svg.append(f'<text x="12" y="{y+5}" font-size="15">{html.escape(label)}（n={len(rows)}）</text>')
        if not rows:continue
        values=np.array([r['return']*100 for r in rows]);q1,median,q3=np.quantile(values,[.25,.5,.75]);iqr=q3-q1
        lo=float(values[values>=q1-1.5*iqr].min());hi=float(values[values<=q3+1.5*iqr].max())
        svg.append(f'<line x1="{chart_x_v4(lo)}" x2="{chart_x_v4(hi)}" y1="{y}" y2="{y}" stroke="#64748b"/><rect x="{chart_x_v4(q1)}" y="{y-13}" width="{max(.3,chart_x_v4(q3)-chart_x_v4(q1))}" height="26" fill="#eff6ff" stroke="#334155"/><line x1="{chart_x_v4(median)}" x2="{chart_x_v4(median)}" y1="{y-17}" y2="{y+17}" stroke="#db2777" stroke-width="3"><title>中位數 {median:.3f}%</title></line>')
        for j,r in enumerate(rows):
            jitter=10*math.sin((int(r['start'])+1)*2.399963229728653)
            title=f"{label}｜{dates[r['start']]}～{dates[r['end']]}｜費用後報酬 {r['return']*100:+.3f}%｜費用 {r['cost']*100:.3f}%｜最大回撤 {r['mdd']*100:.3f}%"
            svg.append(f'<circle class="dot" cx="{chart_x_v4(r["return"]*100):.3f}" cy="{y+jitter:.3f}" r="3.5" fill="#2563eb" fill-opacity=".38"><title>{html.escape(title)}</title></circle>')
    svg.append('</svg>');curve_chart_counts_v4[scope]=total;return ''.join(svg)
all_svg_v4=render_curve_chart_v4('all');season_svg_v4=render_curve_chart_v4('season')
curve_html_path_v4=recovery_root/'策略研究_v3互動箱形圖_2026-10-14.html'
curve_html_path_v4.write_text('''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>v3 策略比較</title><style>body{font-family:Microsoft JhengHei,sans-serif;color:#172033;margin:24px}p{line-height:1.7}button{padding:9px 15px;margin-right:8px;border:1px solid #94a3b8;border-radius:6px;background:white;cursor:pointer}button[aria-pressed=true]{background:#dbeafe;border-color:#2563eb}svg{width:100%;min-width:1000px}.chart{overflow:auto}[hidden]{display:none}.dot:hover{fill:#f97316;fill-opacity:1;r:5}</style><h1>v3：15 個既有方法比較</h1><p>預設顯示全部有效窗口。每個有效窗口都是一個藍點；桃紅線是中位數，黑色虛線是0%。停在點上可看日期、費用後報酬與費用。兩個快捷鍵共用固定方法順序與橫軸範圍；只有一種原生提示，不另開第二個窗格。</p><button id="all-button" aria-pressed="true" onclick="showScope('all')">全部窗口</button><button id="season-button" aria-pressed="false" onclick="showScope('season')">歷年競賽時段</button><p>競賽時段為2024-10-29、2025-10-28起24交易日，只有兩年；無效窗口不畫成0%。各方法有效日期不同，不能只按中位數排名，正式結論請看同日期A配對與家族校正報告。公告配置六個原型僅有兩年點，另見報告，未混入這15方法。</p><div class="chart" id="all-chart">'''+all_svg_v4+'</div><div class="chart" id="season-chart" hidden>'+season_svg_v4+'''</div><p>既有策略定義未改；費率0.1425%、賣出稅0.3%、整千股、25檔、投入90%、24交易日、每5日調整（AH／BH為靜態對照）。未知公司行動等排除限制仍適用，並非官賽合規認證。</p><script>function showScope(scope){for(const name of ['all','season']){document.getElementById(name+'-chart').hidden=name!==scope;document.getElementById(name+'-button').setAttribute('aria-pressed',String(name===scope));}}</script></html>''',encoding='utf-8')
assert curve_html_path_v4.read_text(encoding='utf-8').count('class="dot"')==sum(curve_chart_counts_v4.values())
print('All points saved, fixed x range:',curve_chart_counts_v4,[xlo_v4,xhi_v4])
print('Snapshot cache matches original SHA256; no credentials stored.')
print('CURVE_HTML_AND_SNAPSHOT_SAVED')


In [ ]:
# Offline SVG and persisted result verification; browser file URL policy disallows UI check
import xml.etree.ElementTree as ET
text=curve_html_path_v4.read_text(encoding='utf-8')
svg_blocks=re.findall(r'<svg.*?</svg>',text,re.S)
assert len(svg_blocks)==2
for svg in svg_blocks:
    root=ET.fromstring(svg)
    dots=[el for el in root.iter() if el.tag.endswith('circle') and el.get('class')=='dot']
    assert all(chart_left_v4<=float(el.get('cx'))<=chart_right_v4 for el in dots)
    assert all(0<float(el.get('cy'))<chart_height_v4 for el in dots)
    assert all(len(el)>0 and '費用後報酬' in el[0].text for el in dots)
print('Offline verification: SVGs parse; every dot has date, return, cost tooltip; fixed axes and buttons present.')
verified_export_v4=json.loads(curve_output_path_v4.read_text(encoding='utf-8'))
assert verified_export_v4['status']=='all_methods_matched_saved_v3_statistics'
assert len(verified_export_v4['verified_methods'])==15
assert all(len(rows)==650 for rows in verified_export_v4['windows'].values())
with gzip.open(source_cache_path_v4,'rb') as source:assert hashlib.sha256(source.read()).hexdigest()==snapshot_sha256
print('Final durable artifacts verified: 9750 records including invalid; 6753 valid plotted; exact original raw snapshot.')
print('Browser UI not verified: local file URL blocked by browser security policy. No workaround attempted.')


In [ ]:
# Source follow-up: retained metadata and official page API discovery only
import sys,urllib.request,re,json,datetime
print('Kernel:',sys.executable)
print('Retained:',{k:(type(globals()[k]).__name__,getattr(globals()[k],'shape',None)) for k in ['raw_data','bad_action','engine_v3_results','action_originals_v1','action_corrections_staged_v1','full_pool_financial_announcements_v1'] if k in globals()})
assert 'raw_data' in globals() and 'engine_v3_results' in globals()
print('Dividend schema sample:',list(raw_data['dividend'][0]),raw_data['dividend'][0])
official_action_pages_v4={}
for url in ['https://www.twse.com.tw/zh/announcement/ex-right/twt48u.html','https://www.twse.com.tw/zh/announcement/ex-right/twt49u.html','https://www.tpex.org.tw/zh-tw/announce/market/ex/cal.html']:
    try:
        with urllib.request.urlopen(url,timeout=30) as response:body=response.read().decode('utf-8')
        official_action_pages_v4[url]=body
        print('Official API declarations:',url,re.findall(r'data-api=["\']([^"\']+)',body)[:8],re.findall(r'(?:/www/|/rwd/)[^"\'<> ]+',body)[:8])
    except Exception as error:print('Source page unavailable:',url,type(error).__name__)


In [ ]:
# Official action alternative source: normal HTTPS; no TLS bypass or price inference
import requests,time
official_action_attempts_v4=[]
for url in ['https://www.twse.com.tw/zh/announcement/ex-right/twt48u.html','https://www.twse.com.tw/zh/announcement/ex-right/twt49u.html']:
    try:
        response=requests.get(url,timeout=30);response.raise_for_status();body=response.text
        official_action_pages_v4[url]=body
        print('API declarations:',url,re.findall(r'data-api=["\']([^"\']+)',body)[:8])
        official_action_attempts_v4.append({'url':url,'attempt':2,'status':response.status_code,'result':'page_retrieved'})
    except Exception as error:
        official_action_attempts_v4.append({'url':url,'attempt':2,'result':type(error).__name__,'status':getattr(getattr(error,'response',None),'status_code',None)})
    time.sleep(1)
print('Source attempts:',official_action_attempts_v4)
corporate_remaining_v4=[{'stock_id':sid,'date':str(date),'before':float(before or 0),'reference':float(ref or 0)} for sid,date,c,st,before,ref,status in raw_data['dividend'] if sid in si and date in di and bad_action_v3[di[date],si[sid]]]
print('Remaining action inventory:',len(corporate_remaining_v4),'all dates',sorted({r['date'] for r in corporate_remaining_v4}))


In [ ]:
# Official action API schema probe from actual page declarations
for url,body in official_action_pages_v4.items():
    print('Declared form:',url,re.findall(r'<(?:input|select)[^>]+>',body)[:20])
    print('API context:',body[max(0,body.find('data-api')-150):body.find('data-api')+500])
official_action_probe_v4={}
for kind in ['TWT48U','TWT49U']:
    url='https://www.twse.com.tw/rwd/zh/exRight/'+kind
    try:
        response=requests.get(url,params={'startDate':'20240806','endDate':'20240806','response':'json'},timeout=30)
        response.raise_for_status();doc=response.json()
        official_action_probe_v4[kind]={'url':response.url,'document':doc}
        print('Report schema:',kind,list(doc),'stat',doc.get('stat'),'fields',doc.get('fields'),'rows',len(doc.get('data',[])),'first',doc.get('data',[])[:1])
    except Exception as error:print('Official API failure:',kind,type(error).__name__)
    time.sleep(1)


In [ ]:
# Historical results and actual page script declarations, with explicit date verification
body=official_action_pages_v4['https://www.twse.com.tw/zh/announcement/ex-right/twt49u.html']
print('Page script URLs:',re.findall(r'<script[^>]*src=["\']([^"\']+)',body))
url='https://www.twse.com.tw/rwd/zh/exRight/TWT49U'
response=requests.get(url,params={'startDate':'20231002','endDate':'20261012','response':'json'},timeout=45);response.raise_for_status()
official_action_full_v4={'url':response.url,'retrieved_at':datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=8))).isoformat(),'document':response.json()}
document=official_action_full_v4['document']
assert document.get('strDate')=='20231002' and document.get('endDate')=='20261012'
def official_roc_date_v4(value):
    match=re.fullmatch(r'(\d+)年(\d+)月(\d+)日',value);assert match,value
    return datetime.date(int(match[1])+1911,int(match[2]),int(match[3]))
official_action_by_key_v4={(row[1],str(official_roc_date_v4(row[0]))):row for row in document.get('data',[])}
matches=[(rec,official_action_by_key_v4.get((rec['stock_id'],rec['date']))) for rec in corporate_remaining_v4]
print('Historical results:',len(document.get('data',[])),'matched unknown events',sum(row is not None for rec,row in matches))
print('Matched event kinds:',[(rec['stock_id'],rec['date'],row[6],row[5]) for rec,row in matches if row is not None])
print('Cash-only records:',[(rec,row[:12]) for rec,row in matches if row is not None and row[6]=='息'])


In [ ]:
# Targeted official daily sources: avoid timed-out full-range query; save successful sources
action_history_cache_path_v4=recovery_root/'A策略_公司行動官方核對_2026-10-14.json'
action_history_cache_v4={'definition':'official exact-date results; latest financial columns not historical data; no rate inference','pages':official_action_attempts_v4,'bulk_attempt':'20231002–20261012 ReadTimeout; replaced by only needed dates','reports':{},'failures':{},'scripts':{}}
for kind,source in official_action_probe_v4.items():
    if kind=='TWT49U' and source['document'].get('strDate')=='20240806' and source['document'].get('endDate')=='20240806':
        action_history_cache_v4['reports']['2024-08-06']=source
def save_action_history_v4():
    action_history_cache_path_v4.write_text(json.dumps(action_history_cache_v4,ensure_ascii=False,indent=2),encoding='utf-8')
for script_name in ['web-report.js','web.js']:
    url='https://www.twse.com.tw/res/js/'+script_name
    try:
        response=requests.get(url,timeout=20);response.raise_for_status()
        script=response.content.decode('utf-8');action_history_cache_v4['scripts'][script_name]={'url':url,'sha256':hashlib.sha256(response.content).hexdigest()}
        print('Official script API hints:',script_name,[script[max(0,m.start()-100):m.end()+220] for m in re.finditer(r'TWT49U|TWT48U|exRight',script)][:12])
    except Exception as error:print('Script unavailable:',script_name,type(error).__name__)
save_action_history_v4()
targets=sorted({r['date'] for r in corporate_remaining_v4})
consecutive=0
for number,day in enumerate(targets,1):
    if day in action_history_cache_v4['reports']:continue
    date=day.replace('-','')
    try:
        response=requests.get('https://www.twse.com.tw/rwd/zh/exRight/TWT49U',params={'startDate':date,'endDate':date,'response':'json'},timeout=20);response.raise_for_status()
        document=response.json()
        assert document.get('strDate')==date and document.get('endDate')==date,'Requested date mismatch'
        assert document.get('stat')=='OK' or not document.get('data'),document.get('stat')
        action_history_cache_v4['reports'][day]={'url':response.url,'document':document}
        consecutive=0
    except Exception as error:
        action_history_cache_v4['failures'][day]={'attempts':1,'result':type(error).__name__}
        consecutive+=1
    save_action_history_v4()
    if number%5==0:print('Official exact-date progress',number,'/',len(targets),'retrieved',len(action_history_cache_v4['reports']),'failed',len(action_history_cache_v4['failures']),flush=True)
    if consecutive>=3:
        print('Three consecutive source failures; stop this source batch, retain successes.');break
    time.sleep(1.2)
print('OFFICIAL_TARGETED_ACTION_SOURCES_SAVED',len(action_history_cache_v4['reports']),len(action_history_cache_v4['failures']))


In [ ]:
# Audit official exact-date fields; retain unresolved units and dates explicitly
print('Fetch failures:',action_history_cache_v4['failures'])
def official_roc_date_v4(value):
    match=re.fullmatch(r'(\d+)年(\d+)月(\d+)日',value)
    if match is None:return None
    return str(datetime.date(int(match[1])+1911,int(match[2]),int(match[3])))
official_action_by_key_v4={}
for day,source in action_history_cache_v4['reports'].items():
    document=source['document']
    assert document.get('strDate')==day.replace('-','') and document.get('endDate')==day.replace('-','')
    for row in document.get('data',[]):
        parsed=official_roc_date_v4(row[0])
        if parsed==day:official_action_by_key_v4[(row[1],day)]={'row':row,'url':source['url'],'fields':document['fields']}
action_official_matches_v4=[];action_official_cash_candidates_v4=[]
for rec in corporate_remaining_v4:
    hit=official_action_by_key_v4.get((rec['stock_id'],rec['date']))
    if hit is None:
        action_official_matches_v4.append(dict(rec,status='no_official_exact_date_match'));continue
    row=hit['row'];number=lambda value:float(str(value).replace(',',''))
    before_match=np.isclose(number(row[3]),rec['before'],rtol=0,atol=.03)
    reference_match=np.isclose(number(row[4]),rec['reference'],rtol=0,atol=.03)
    record=dict(rec,status='official_exact_date_verified',kind=row[6],official_before=number(row[3]),official_reference=number(row[4]),combined_rights_dividend_value=number(row[5]),before_match=bool(before_match),reference_match=bool(reference_match),url=hit['url'])
    action_official_matches_v4.append(record)
    if row[6]=='息' and before_match and reference_match:
        action_official_cash_candidates_v4.append(dict(record,cash=number(row[5]),stock_ratio=0.,status='cash_only_official_value_candidate'))
print('Official match audit:',{'events':len(action_official_matches_v4),'matched_dates':sum(r['status']=='official_exact_date_verified' for r in action_official_matches_v4),'cash_only_candidates':len(action_official_cash_candidates_v4)})
print('Cash-only candidates:',action_official_cash_candidates_v4)
print('Price mismatch records:',[r for r in action_official_matches_v4 if r.get('status')=='official_exact_date_verified' and not(r['before_match'] and r['reference_match'])])
print('Simulator selection definition:',[line.strip() for line in weight_simulator_source.splitlines() if 'selected=' in line or 'order=' in line])
action_history_cache_v4['event_audit']=action_official_matches_v4
action_history_cache_v4['cash_candidates']=action_official_cash_candidates_v4
save_action_history_v4()
print('ACTION_OFFICIAL_MATCH_AUDIT_SAVED')


In [ ]:
# Retry only three missing action dates using official CSV representation
import csv,io
print('Matched official kind distribution:',dict(Counter(r.get('kind','unmatched') for r in action_official_matches_v4)))
for day in list(action_history_cache_v4['failures']):
    date=day.replace('-','')
    try:
        response=requests.get('https://www.twse.com.tw/rwd/zh/exRight/TWT49U',params={'startDate':date,'endDate':date,'response':'csv'},timeout=25);response.raise_for_status()
        decoded=response.content.decode('cp950')
        rows=list(csv.reader(io.StringIO(decoded)))
        expected=next(source['document']['fields'] for source in action_history_cache_v4['reports'].values())
        records=[]
        for row in rows:
            clean=[value.lstrip('=').strip().strip('"') for value in row]
            if len(clean)>=len(expected) and official_roc_date_v4(clean[0])==day:records.append(clean[:len(expected)])
        assert records,'No exact-date CSV rows found; not treated as complete'
        document={'stat':'OK','strDate':date,'endDate':date,'fields':expected,'data':records,'source_encoding':'cp950','source_representation':'official_csv'}
        action_history_cache_v4['reports'][day]={'url':response.url,'document':document}
        action_history_cache_v4['failures'].pop(day)
        print('CSV recovered:',day,len(records))
    except Exception as error:
        record=action_history_cache_v4['failures'][day];record['attempts']+=1;record['csv_result']=type(error).__name__
        print('CSV source still unavailable:',day,type(error).__name__)
    save_action_history_v4();time.sleep(1.2)
print('Targeted dates retained:',len(action_history_cache_v4['reports']),'unresolved fetches',action_history_cache_v4['failures'])


In [ ]:
# Historical A25 original-source inventory; no new LLM score or weighting rule
curve_run_saved_v4(curve_saved_nb_v4.cells[76].source)
prestart_a25_source_inventory_v4=[]
for anchor in ['2024-10-29','2025-10-28']:
    start=di[datetime.date.fromisoformat(anchor)];cutoff=datetime.datetime.combine(dates[start],datetime.time(8,55))
    rank=scores_v3['A'][start-1];eligible=np.flatnonzero(np.isfinite(rank)&good_exec_v2[start])
    selected=eligible[np.argsort(-rank[eligible],kind='stable')[:COUNT]]
    assert len(selected)==25
    weights=engine_v3_allocation(start,selected,'mild')
    for position,(j,weight) in enumerate(zip(selected,weights),1):
        sid=stock_ids[j];financial=pure_financial_evidence_v1[(start,sid)];revenue=pure_revenue_evidence_v1[(start,sid)]
        observed=[]
        for a in revenue_announcement_candidates_v1:
            if a['stock_id']!=sid:continue
            stamp=prestart_stamp(a)
            if stamp and stamp<cutoff:
                observed.append({'ann_id':a['ann_id'],'stamp':str(stamp),'subject':a['subject'],'source':a['source']})
        record={'anchor':anchor,'stock_id':sid,'momentum_rank':position,'mild_initial_target_weight':float(weight),'financial':curve_json_value_v4(financial),'revenue':curve_json_value_v4(revenue),'original_revenue_announcements_observed_before_start':sorted(observed,key=lambda a:a['stamp']),'first_publication_verified':False,'limitations':'known retained-source availability only; first official release/revision history not proven; no sentiment or full conference-text judgment'}
        prestart_a25_source_inventory_v4.append(record)
    subset=[r for r in prestart_a25_source_inventory_v4 if r['anchor']==anchor]
    print('A25 availability:',anchor,{'stocks':len(subset),'financial_status':dict(Counter(r['financial']['status'] for r in subset)),'revenue_status':dict(Counter(r['revenue']['status'] for r in subset)),'with_observed_official_revenue_notice':sum(bool(r['original_revenue_announcements_observed_before_start']) for r in subset)})
inventory_path_v4=recovery_root/'A策略_歷年開賽25檔公告來源盤點_2026-10-14.json'
inventory_path_v4.write_text(json.dumps(prestart_a25_source_inventory_v4,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('A25 source inventory saved:',inventory_path_v4.name)


In [ ]:
# Actual detail source observed via official historical-result link, not a guessed endpoint
detail_page_url_v4='https://www.twse.com.tw/zh/announcement/ex-right/twt49u-detail.html?2887,20240806'
response=requests.get(detail_page_url_v4,timeout=25);response.raise_for_status()
detail_page_body_v4=response.content.decode('utf-8')
print('Detail data API:',re.findall(r'data-api=["\']([^"\']+)',detail_page_body_v4))
print('Detail script declarations:',re.findall(r'<script[^>]*src=["\']([^"\']+)',detail_page_body_v4))
print('Relevant HTML declarations:',[line for line in detail_page_body_v4.splitlines() if any(k in line for k in ['data-api','data-field','data-query','stockNo','2887','0.599999'])][:20])


In [ ]:
# Official detail endpoint declared by verified page, one focused parameter query
response=requests.get('https://www.twse.com.tw/rwd/zh/exRight/TWT49UDetail',params={'stockNo':'2887','date':'20240806','response':'json'},timeout=25);response.raise_for_status()
detail_probe_v4={'url':response.url,'document':response.json()}
print('Detail schema:',detail_probe_v4)


In [ ]:
# Read actual public query serialization, not guessed parameter variants
official_script_source_v4={}
for name in ['web-report.js','web.js']:
    response=requests.get('https://www.twse.com.tw/res/js/'+name,timeout=20);response.raise_for_status()
    source=response.content.decode('utf-8');official_script_source_v4[name]=source
    hits=[]
    for pattern in [r'location.search',r'URLSearchParams',r'query',r'params',r'Detail',r'detail']:
        for m in list(re.finditer(pattern,source))[:4]:hits.append(source[max(0,m.start()-140):m.end()+180])
    print('Actual query contract:',name,hits)


In [ ]:
# Follow the loaded official main script to locate documented page query handling
response=requests.get('https://www.twse.com.tw/res/js/main.js',timeout=20);response.raise_for_status()
official_main_source_v4=response.content.decode('utf-8')
print('Main script context:',[official_main_source_v4[max(0,m.start()-150):m.end()+240] for m in re.finditer(r'location.search|URLSearchParams|TWT49U|exRight|data-api',official_main_source_v4)][:12])
print('Detail page inline scripts:',re.findall(r'<script(?:\s[^>]*)?>(.*?)</script>',detail_page_body_v4,re.S)[-4:])


In [ ]:
# Use exact STK_NO/T1 parameters from official pageInit declaration
response=requests.get('https://www.twse.com.tw/rwd/zh/exRight/TWT49UDetail',params={'STK_NO':'2887','T1':'20240806','response':'json'},timeout=25);response.raise_for_status()
detail_exact_probe_v4={'url':response.url,'document':response.json(),'parameters_source':detail_page_url_v4}
print('Exact detail response:',detail_exact_probe_v4)


In [ ]:
# Exact official share/cash units; serial targeted extraction with per-record checkpoints
action_history_cache_v4['detail_reports']={'2887|2024-08-06':detail_exact_probe_v4}
action_history_cache_v4['detail_failures']={}
consecutive=0
for number,event in enumerate(corporate_remaining_v4,1):
    sid,day=event['stock_id'],event['date'];key=sid+'|'+day
    if key in action_history_cache_v4['detail_reports']:continue
    try:
        response=requests.get('https://www.twse.com.tw/rwd/zh/exRight/TWT49UDetail',params={'STK_NO':sid,'T1':day.replace('-',''),'response':'json'},timeout=20);response.raise_for_status()
        document=response.json();assert document.get('stat','').lower()=='ok'
        assert len(document.get('data',[]))==1 and document['data'][0][0].strip()==sid,'Stock mismatch'
        action_history_cache_v4['detail_reports'][key]={'url':response.url,'document':document,'page_link':'https://www.twse.com.tw/zh/announcement/ex-right/twt49u-detail.html?'+sid+','+day.replace('-','')}
        consecutive=0
    except Exception as error:
        action_history_cache_v4['detail_failures'][key]={'attempts':1,'result':type(error).__name__};consecutive+=1
    save_action_history_v4()
    if number%5==0:print('Exact unit source progress',number,'/53 recovered',len(action_history_cache_v4['detail_reports']),'failed',len(action_history_cache_v4['detail_failures']),flush=True)
    if consecutive>=3:print('Three consecutive failures; stop source batch.');break
    time.sleep(1.2)
print('EXACT_ACTION_DETAIL_SOURCES_SAVED',len(action_history_cache_v4['detail_reports']),len(action_history_cache_v4['detail_failures']))


In [ ]:
# Validate explicitly published units, capital increases and reference price before applying
action_unit_verified_v4=[];action_unit_unresolved_v4=[]
def explicit_value_v4(text,unit):
    match=re.fullmatch(r'\s*([+-]?\d[\d,]*(?:\.\d+)?)\s*'+re.escape(unit)+r'\s*',text)
    if match is None:raise ValueError('Original explicit unit not matched')
    value=float(match[1].replace(',',''));assert np.isfinite(value) and value>=0
    return value
for event in corporate_remaining_v4:
    sid,day=event['stock_id'],event['date'];key=sid+'|'+day;source=action_history_cache_v4['detail_reports'].get(key)
    if not source:action_unit_unresolved_v4.append(dict(event,reason='missing_original_detail'));continue
    try:
        doc=source['document'];row=doc['data'][0]
        assert row[0].strip()==sid
        cash=explicit_value_v4(row[2],'元／股')
        free=explicit_value_v4(row[4],'股')/1000
        paid_total=explicit_value_v4(row[6],'股');paid_price=explicit_value_v4(row[7],'元／股');paid_ratio=explicit_value_v4(row[11],'股')/1000
        official=official_action_by_key_v4[(sid,day)]['row']
        before=float(official[3].replace(',',''));reference=float(official[4].replace(',',''))
        expected=(before-cash+paid_price*paid_ratio)/(1+free+paid_ratio)
        assert abs(expected-reference)<=max(.03,before*.0002),'Explicit units/reference mismatch'
        if paid_ratio>0 or paid_total>0:
            action_unit_unresolved_v4.append(dict(event,reason='paid_capital_or_rights_requires_accounting_model',cash=cash,stock_ratio=free,paid_ratio=paid_ratio,paid_price=paid_price,url=source['url']));continue
        action_unit_verified_v4.append({'stock_id':sid,'ex_date':day,'cash':cash,'stock_ratio':free,'reference':reference,'expected':expected,'source':'TWSE historical TWT49U and TWT49UDetail','detail_url':source['url'],'date_price_url':official_action_by_key_v4[(sid,day)]['url'],'basis':'explicit per-share cash and free shares per 1000; paid capital zero; reference cross-check','first_published_at':None,'usage':'historical corporate-action accounting and price adjustment only; not an announcement trading signal'})
    except Exception as error:action_unit_unresolved_v4.append(dict(event,reason=type(error).__name__,url=source['url']))
action_history_cache_v4['verified_explicit_units']=action_unit_verified_v4
action_history_cache_v4['unresolved_units']=action_unit_unresolved_v4
save_action_history_v4()
print('Explicit unit verification:',len(action_unit_verified_v4),'unresolved',len(action_unit_unresolved_v4))
print('Unresolved detail:',action_unit_unresolved_v4)
print('Verified examples:',action_unit_verified_v4[:3])
print('Daily accounting source semantics:',[line.strip() for line in weight_simulator_source.splitlines() if any(k in line for k in ['stock_div','cash_div','floor','shares*='])][:16])
print('ACTION_UNIT_VALIDATION_COMPLETE')


## v4：官方明列單位補查後的受影響重驗證

新增41筆官方每股現金／每千股無償配股，與歷史日期、前價及參考價核對；12筆未安全確認或涉及現金增資／認購／員工配發，維持排除，不臆測認購資金與權利處理。原v3資料、曲線、区間及圖表完整保留。只改公司行動資料覆蓋，不改選股／權重參數或費稅；同樣14非A家族、24／48窗口位置區塊5000次。原已驗證MOPS1筆＋此次41筆共42筆覆蓋。官方歷史記錄僅用於帳務與價格調整，不把當前查詢或最新財報欄位當事前公告訊號；真正首次公開時間仍不可判定。現金／股票股利沿原引擎記應收，未建完整實際發放日。

In [ ]:
# v4 company-action data overlay; preserve all v3 objects and fixed strategy definitions
company_detail_review_v4=[]
for event in action_unit_unresolved_v4:
    row=action_history_cache_v4['detail_reports'][event['stock_id']+'|'+event['date']]['document']['data'][0]
    company_detail_review_v4.append({'stock_id':event['stock_id'],'date':event['date'],'cash_field':row[2],'free_field':row[4],'employee_field':row[5],'paid_total_field':row[6],'paid_price_field':row[7],'shareholder_subscription_field':row[11],'reason':event['reason']})
print('Unmodelled exact fields:',company_detail_review_v4)
assert len(action_unit_verified_v4)==41 and len(action_unit_unresolved_v4)==12
action_corrections_applied_v4=action_corrections_staged_v1+action_unit_verified_v4
assert len({(a['stock_id'],a['ex_date']) for a in action_corrections_applied_v4})==42
source=curve_saved_nb_v4.cells[75].source
source=source[:source.index("if 'engine_v3_results' not in globals():")]
source=source.replace('_v3','_v4').replace('action_corrections_staged_v1','action_corrections_applied_v4').replace('assert len(action_corrections_applied_v4)==1','assert len(action_corrections_applied_v4)==42').replace("assert np.isclose(a['cash'],1.8) and np.isclose(a['stock_ratio'],.01)","assert a['cash']>=0 and a['stock_ratio']>=0")
curve_run_saved_v4(source)
print('v4 action hash:',overlay_sha_v4,'remaining bad-action matrix flags',int(bad_action_v4.sum()),'remaining dividend events',12)
engine_v4_results={};engine_v4_metrics={};engine_v4_valid={}
v4_export_path=recovery_root/'策略研究_逐窗口結果_v4_2026-10-14.json'
v4_export={'engine':'v4 explicit historical action-unit source correction','status':'in_progress','raw_snapshot_sha256':snapshot_sha256,'company_action_overlay_sha256':overlay_sha_v4,'action_corrections':action_corrections_applied_v4,'unresolved_events':action_unit_unresolved_v4,'fixed_methods':engine_v2_fixed_methods,'parameters':curve_export_v4['definition'],'windows':{},'v3_comparison':{}}
for name,(strategy,allocation,tolerance) in engine_v2_fixed_methods.items():
    rows=[simulate_engine_v4(int(start),strategy,allocation,tolerance=tolerance) for start in window_starts]
    engine_v4_results[name]=rows
    engine_v4_metrics[name]={f:np.array([r[f] for r in rows]) for f in ['return','mdd','cost','turnover','violations','strict_cap_days','min_held','max_held']}
    engine_v4_valid[name]=np.array([not r['invalid'] and r['violations']==0 for r in rows])
    old=engine_v3_metrics[name];old_valid=engine_v3_valid[name]
    v4_export['v3_comparison'][name]={'v3_valid':int(old_valid.sum()),'v4_valid':int(engine_v4_valid[name].sum()),'new_valid':int((engine_v4_valid[name]&~old_valid).sum()),'lost_valid':int((old_valid&~engine_v4_valid[name]).sum()),'changed_returns':int(np.sum(~np.isclose(engine_v4_metrics[name]['return'],old['return'],rtol=1e-12,atol=1e-12)))}
    v4_export['windows'][name]=[dict(curve_json_value_v4(row),start_date=str(dates[row['start']]),end_date=str(dates[row['start']+HORIZON-1])) for row in rows]
    checkpoint=v4_export_path.with_suffix('.json.new');checkpoint.write_text(json.dumps(v4_export,ensure_ascii=False,separators=(',',':'),allow_nan=False),encoding='utf-8');os.replace(checkpoint,v4_export_path)
    print('v4 persisted',name,v4_export['v3_comparison'][name],flush=True)
v4_export['status']='all_methods_saved_inference_pending'
v4_export_path.write_text(json.dumps(v4_export,ensure_ascii=False,separators=(',',':'),allow_nan=False),encoding='utf-8')
print('ENGINE_V4_REVALIDATION_SAVED')


In [ ]:
# Final strategy validation v3: paired costs, fixed family intervals and sensitivity
final_candidate_names_v4=[name for name in engine_v4_results if name!='A_equal']
final_family_alpha_v4=.05/len(final_candidate_names_v4)
def final_block_family_ci_v4(series,block,seed):
    series=np.asarray(series,dtype=float);length=len(series);rng=np.random.default_rng(seed);estimates=[]
    for begin in range(0,5000,100):
        starts=rng.integers(0,length-block+1,size=(min(100,5000-begin),int(np.ceil(length/block))))
        ix=(starts[:,:,None]+np.arange(block)).reshape(len(starts),-1)[:,:length]
        values=series[ix];count=np.sum(np.isfinite(values),axis=1);sums=np.nansum(values,axis=1)
        estimates.extend(np.divide(sums,count,out=np.full(len(sums),np.nan),where=count>0).tolist())
    estimates=np.array(estimates);estimates=estimates[np.isfinite(estimates)]
    return np.quantile(estimates,[final_family_alpha_v4/2,1-final_family_alpha_v4/2]).tolist()
final_strategy_stats_v4=[]
base=engine_v4_metrics['A_equal'];base_mask=engine_v4_valid['A_equal']
early=np.array([dates[s+HORIZON-1]<datetime.date(2026,1,1) for s in window_starts])
first2026=next(i for i,d in enumerate(dates) if d.year==2026);late=np.array([int(s)>=first2026+HORIZON for s in window_starts])
for i,name in enumerate(final_candidate_names_v4):
    candidate=engine_v4_metrics[name];pair=base_mask&engine_v4_valid[name]
    if not pair.any():continue
    diff=candidate['return']-base['return'];series=np.where(pair,diff,np.nan)
    cis={b:final_block_family_ci_v4(series,b,SEED+1000*i+b) for b in [24,48]}
    risk_p10=float(np.quantile(candidate['return'][pair],.1)-np.quantile(base['return'][pair],.1))
    risk_mdd=float(np.median(candidate['mdd'][pair])-np.median(base['mdd'][pair]))
    risk_pass=risk_p10>=-.01 and risk_mdd>=-.01
    yearly=[]
    for y in sorted({dates[s].year for s in window_starts}):
        mask=pair&np.array([dates[s].year==y and dates[s+HORIZON-1].year==y for s in window_starts])
        if mask.any():yearly.append({'year':y,'windows':int(mask.sum()),'mean_difference_pp':float(np.mean(diff[mask])*100)})
    stat={'method':name,'paired_windows':int(pair.sum()),'A_valid_excluded':int(np.sum(base_mask&~pair)),'median_return_pct':float(np.median(candidate['return'][pair])*100),'A_same_dates_median_pct':float(np.median(base['return'][pair])*100),'mean_difference_pp':float(np.mean(diff[pair])*100),'beat_A_pct':float(np.mean(diff[pair]>1e-12)*100),'mean_cost_pct':float(np.mean(candidate['cost'][pair])*100),'A_mean_cost_pct':float(np.mean(base['cost'][pair])*100),'P10_difference_pp':risk_p10*100,'mdd_difference_pp':risk_mdd*100,'risk_pass':risk_pass,'family_CI_pp':{str(b):(np.array(ci)*100).tolist() for b,ci in cis.items()},'both_intervals_positive':all(ci[0]>0 for ci in cis.values()),'yearly':yearly,'early_mean_pp':float(np.mean(diff[pair&early])*100) if (pair&early).any() else None,'late_mean_pp':float(np.mean(diff[pair&late])*100) if (pair&late).any() else None,'official_compliance_fully_verified':False}
    stat['supported_research_improvement']=stat['both_intervals_positive'] and risk_pass
    final_strategy_stats_v4.append(stat);print('Family validation:',json.dumps(stat,ensure_ascii=False),flush=True)
final_supported_v4=[r['method'] for r in final_strategy_stats_v4 if r['supported_research_improvement'] and r['method'] not in ['REF','AH','BH']]
print('Final supported fixed research improvements:',final_supported_v4)
print('All results use same v3 engine; no method chosen solely from highest median.')
print('FINAL_STRATEGY_VALIDATION_READY')


In [ ]:
# Legacy seasonal v3: same fixed announcement weighting prototypes, only original two anchors
current_nb=json.loads((recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').read_text(encoding='utf-8'))
def restore_selected_functions_v4(index,names):
    source=''.join(current_nb['cells'][index]['source'])
    tree=ast.parse(source)
    for node in tree.body:
        if isinstance(node,ast.FunctionDef) and node.name in names:exec(compile(ast.Module(body=[node],type_ignores=[]),'<legacy-fixed-function>','exec'),globals())
def prestart_original_amounts(ann):
    body=unicodedata.normalize('NFKC',ann.get('content') or '').replace('\r','');vals={}
    for field,number,label in [('revenue',4,'營業收入'),('operating',6,'營業利益'),('parent_net',9,'歸屬於母公司')]:
        match=re.search(r'(?m)^\s*'+str(number)+r'[.、]\s*([^\n]+)',body)
        if not match or label not in match[1] or '累計' not in match[1] or ':' not in match[1]:return None
        v=match[1].rsplit(':',1)[1].strip().replace(',','').replace(' ','')
        if re.fullmatch(r'\(?-?\d+(?:\.\d+)?\)?',v) is None:return None
        vals[field]=float(v.strip('()'))*(-1 if v.startswith('(') else 1)
    return vals if vals['revenue']>0 else None
a_announcement_documents={(y,sid):{'announcements':full_pool_ann_by_stock_v1[sid]} for y in [2024,2025] for sid in stock_ids}
restore_selected_functions_v4(13,{'a_financial_evidence_asof'})
restore_selected_functions_v4(19,{'a_operating_evidence'})
legacy_evidence_cache_v4={}
def legacy_announcement_allocation_v4(t,selected,mode):
    if mode=='equal':return np.full(COUNT,INVESTED/COUNT)
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));mult_dual=[];mult_op=[];mult_no_self=[];mult_rev=[]
    for j in selected:
        sid=stock_ids[j];key=(t,sid)
        if key not in legacy_evidence_cache_v4:
            raw=a_financial_evidence_asof(cutoff.year,sid,cutoff)
            op=a_operating_evidence(raw);legacy_evidence_cache_v4[key]=(raw,op)
        raw,op=legacy_evidence_cache_v4[key]
        delta=raw.get('op_margin_change_pp')
        no_self=0 if raw.get('unintegrated_sources') or raw['status']!='可核對' or delta is None else 1 if delta>=1 else -1 if delta<=-1 else 0
        mult_dual.append(1+.25*raw['score']);mult_op.append(1+.25*op['score']);mult_no_self.append(1+.25*no_self)
        rv=pure_revenue_evidence_v1[(t,sid)]
        score=0
        if rv['status']=='verified_original_known_available_time':score=1 if rv['mom']>0 and rv['yoy']>0 else -1 if rv['mom']<0 and rv['yoy']<0 else 0
        mult_rev.append(1+.25*score)
    def weights(x):x=np.array(x,dtype=float);return INVESTED*x/x.sum()
    if mode=='financial_dual':return weights(mult_dual)
    if mode=='operating_self':return weights(mult_op)
    if mode=='operating_no_self':return weights(mult_no_self)
    if mode=='revenue_weight':return weights(mult_rev)
    assert mode=='combined'
    return .5*engine_v4_allocation(t,selected,'mild')+.25*weights(mult_op)+.25*weights(mult_rev)
legacy_src_v4=src_v4.replace('def simulate_engine_v4(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):','def simulate_legacy_announcement_v4(start,allocation,horizon=HORIZON,tolerance=0):\n    strategy="A"').replace('engine_v4_allocation(','legacy_announcement_allocation_v4(')
exec(legacy_src_v4)
legacy_seasonal_results_v4={};legacy_seasonal_rows_v4=[]
for year,date in [(2024,datetime.date(2024,10,29)),(2025,datetime.date(2025,10,28))]:
    start=di[date];pos=int(np.flatnonzero(window_starts==start)[0])
    control=simulate_legacy_announcement_v4(start,'equal')
    assert np.allclose(control['path'],engine_v4_results['A_equal'][pos]['path'],rtol=1e-12,atol=1e-6)
    for name,mode,tolerance in [('financial_dual','financial_dual',0),('operating_no_self','operating_no_self',0),('operating_self','operating_self',0),('revenue_weight','revenue_weight',0),('combined','combined',0),('combined_tolerance','combined',.005)]:
        result=simulate_legacy_announcement_v4(start,mode,tolerance=tolerance);legacy_seasonal_results_v4[(year,name)]=result
        legacy_seasonal_rows_v4.append({'year':year,'method':name,'valid':not result['invalid'] and result['violations']==0,'return_pct':float(result['return']*100) if not result['invalid'] else None,'A_return_pct':float(control['return']*100) if not control['invalid'] else None,'cost_pct':float(result['cost']*100),'difference_pp':float((result['return']-control['return'])*100) if not result['invalid'] and not control['invalid'] else None})
print('Legacy two-anchor revalidation:',json.dumps(legacy_seasonal_rows_v4,ensure_ascii=False))
print('Only 2 seasonal anchors; no reliable family interval or winner declaration. Expanded original-document coverage noted; raw v1 outputs preserved.')


In [ ]:
# Historical A25 original-source inventory; no new LLM score or weighting rule
assert 'revenue_announcement_candidates_v1' in globals()
prestart_a25_source_inventory_v5=[]
for anchor in ['2024-10-29','2025-10-28']:
    start=di[datetime.date.fromisoformat(anchor)];cutoff=datetime.datetime.combine(dates[start],datetime.time(8,55))
    rank=scores_v4['A'][start-1];eligible=np.flatnonzero(np.isfinite(rank)&good_exec_v2[start])
    selected=eligible[np.argsort(-rank[eligible],kind='stable')[:COUNT]]
    assert len(selected)==25
    weights=engine_v4_allocation(start,selected,'mild')
    for position,(j,weight) in enumerate(zip(selected,weights),1):
        sid=stock_ids[j];financial=pure_financial_evidence_v1[(start,sid)];revenue=pure_revenue_evidence_v1[(start,sid)]
        observed=[]
        for a in revenue_announcement_candidates_v1:
            if a['stock_id']!=sid:continue
            stamp=prestart_stamp(a)
            if stamp and stamp<cutoff:
                observed.append({'ann_id':a['ann_id'],'stamp':str(stamp),'subject':a['subject'],'source':a['source']})
        record={'anchor':anchor,'stock_id':sid,'momentum_rank':position,'mild_initial_target_weight':float(weight),'financial':curve_json_value_v4(financial),'revenue':curve_json_value_v4(revenue),'original_revenue_announcements_observed_before_start':sorted(observed,key=lambda a:a['stamp']),'first_publication_verified':False,'limitations':'known retained-source availability only; first official release/revision history not proven; no sentiment or full conference-text judgment'}
        prestart_a25_source_inventory_v5.append(record)
    subset=[r for r in prestart_a25_source_inventory_v5 if r['anchor']==anchor]
    print('A25 availability:',anchor,{'stocks':len(subset),'financial_status':dict(Counter(r['financial']['status'] for r in subset)),'revenue_status':dict(Counter(r['revenue']['status'] for r in subset)),'with_observed_official_revenue_notice':sum(bool(r['original_revenue_announcements_observed_before_start']) for r in subset)})
inventory_path_v4=recovery_root/'A策略_歷年開賽25檔公告來源盤點_v4_2026-10-14.json'
inventory_path_v4.write_text(json.dumps(prestart_a25_source_inventory_v5,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('A25 source inventory saved:',inventory_path_v4.name)


In [ ]:
print('v4 provisional method summary:',next(r for r in final_strategy_stats_v4 if r['method']=='A_mild_tolerance'))
print('v4 seasonal A controls:',[{k:r[k] for k in ['start','return','invalid','violations']} for r in engine_v4_results['A_equal'] if str(dates[r['start']]) in ['2024-10-29','2025-10-28']])

In [ ]:
# Final v4 inference and announcement outputs into full-result checkpoint
assert not final_supported_v4
for event in action_unit_unresolved_v4:
    source=action_history_cache_v4['detail_reports'][event['stock_id']+'|'+event['date']]
    row=source['document']['data'][0]
    assert explicit_value_v4(row[6],'股')>0
    event['reason']='paid_capital_increase_or_subscription_accounting_not_modelled'
v4_export['status']='all_fixed_methods_revalidated_and_saved'
v4_export['comparison']=final_strategy_stats_v4
v4_export['legacy_two_anchor_prototypes']=legacy_seasonal_rows_v4
v4_export['unresolved_events']=action_unit_unresolved_v4
v4_export_path.write_text(json.dumps(curve_json_value_v4(v4_export),ensure_ascii=False,separators=(',',':'),allow_nan=False),encoding='utf-8')
print('v4 result checkpoint finalized:',v4_export_path.name)


In [ ]:
recovered_curve_v4=json.loads(v4_export_path.read_text(encoding='utf-8'))
assert recovered_curve_v4['status']=='all_fixed_methods_revalidated_and_saved'
assert len(recovered_curve_v4['windows'])==15 and all(len(x)==650 for x in recovered_curve_v4['windows'].values())
curve_order_v4=list(engine_v2_fixed_methods)
valid_values_v4=[r['return']*100 for name in curve_order_v4 for r in engine_v4_results[name] if not r['invalid'] and r['violations']==0]
xlo_v4=5*math.floor(min(0,min(valid_values_v4))/5)-5;xhi_v4=5*math.ceil(max(0,max(valid_values_v4))/5)+5
chart_width_v4=1400;chart_left_v4=420;chart_right_v4=1300;chart_height_v4=70+60*len(curve_order_v4)
def chart_x_v4(v):return chart_left_v4+(v-xlo_v4)/(xhi_v4-xlo_v4)*(chart_right_v4-chart_left_v4)
curve_chart_counts_v4={}
def render_curve_chart_v4(scope):
    svg=[f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {chart_width_v4} {chart_height_v4}" role="img" aria-label="v4全部窗口箱形圖">']
    for tick in np.linspace(xlo_v4,xhi_v4,9):
        x=chart_x_v4(tick);svg.append(f'<line x1="{x}" x2="{x}" y1="25" y2="{chart_height_v4-30}" stroke="#e5e7eb"/><text x="{x}" y="{chart_height_v4-8}" text-anchor="middle" font-size="13">{tick:.0f}%</text>')
    x=chart_x_v4(0);svg.append(f'<line x1="{x}" x2="{x}" y1="25" y2="{chart_height_v4-30}" stroke="#111827" stroke-dasharray="6 4" stroke-width="2"/>')
    total=0
    for i,name in enumerate(curve_order_v4):
        y=45+i*60;label=method_labels_v4[name]
        if name=='A_invvol100':label='A：100%逆波動配置（有權重上下限）'
        rows=[r for r in engine_v4_results[name] if not r['invalid'] and r['violations']==0 and (scope=='all' or str(dates[r['start']]) in ['2024-10-29','2025-10-28'])]
        total+=len(rows)
        svg.append(f'<text x="12" y="{y+5}" font-size="15">{html.escape(label)}（n={len(rows)}）</text>')
        if not rows:continue
        values=np.array([r['return']*100 for r in rows]);q1,median,q3=np.quantile(values,[.25,.5,.75]);iqr=q3-q1
        lo=float(values[values>=q1-1.5*iqr].min());hi=float(values[values<=q3+1.5*iqr].max())
        svg.append(f'<line x1="{chart_x_v4(lo)}" x2="{chart_x_v4(hi)}" y1="{y}" y2="{y}" stroke="#64748b"/><rect x="{chart_x_v4(q1)}" y="{y-13}" width="{max(.3,chart_x_v4(q3)-chart_x_v4(q1))}" height="26" fill="#eff6ff" stroke="#334155"/><line x1="{chart_x_v4(median)}" x2="{chart_x_v4(median)}" y1="{y-17}" y2="{y+17}" stroke="#db2777" stroke-width="3"><title>中位數 {median:.3f}%</title></line>')
        for j,r in enumerate(rows):
            jitter=10*math.sin((int(r['start'])+1)*2.399963229728653)
            title=f"{label}｜{dates[r['start']]}～{dates[r['end']]}｜費用後報酬 {r['return']*100:+.3f}%｜費用 {r['cost']*100:.3f}%｜最大回撤 {r['mdd']*100:.3f}%"
            svg.append(f'<circle class="dot" cx="{chart_x_v4(r["return"]*100):.3f}" cy="{y+jitter:.3f}" r="3.5" fill="#2563eb" fill-opacity=".38"><title>{html.escape(title)}</title></circle>')
    svg.append('</svg>');curve_chart_counts_v4[scope]=total;return ''.join(svg)
all_svg_v4=render_curve_chart_v4('all');season_svg_v4=render_curve_chart_v4('season')
curve_html_path_v4=recovery_root/'策略研究_v4互動箱形圖_2026-10-14.html'
curve_html_path_v4.write_text('''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>v4 策略比較</title><style>body{font-family:Microsoft JhengHei,sans-serif;color:#172033;margin:24px}p{line-height:1.7}button{padding:9px 15px;margin-right:8px;border:1px solid #94a3b8;border-radius:6px;background:white;cursor:pointer}button[aria-pressed=true]{background:#dbeafe;border-color:#2563eb}svg{width:100%;min-width:1000px}.chart{overflow:auto}[hidden]{display:none}.dot:hover{fill:#f97316;fill-opacity:1;r:5}</style><h1>v4：15 個既有方法比較</h1><p>預設顯示全部有效窗口。每個有效窗口都是一個藍點；桃紅線是中位數，黑色虛線是0%。停在點上可看日期、費用後報酬與費用。兩個快捷鍵共用固定方法順序與橫軸範圍；只有一種原生提示，不另開第二個窗格。</p><button id="all-button" aria-pressed="true" onclick="showScope('all')">全部窗口</button><button id="season-button" aria-pressed="false" onclick="showScope('season')">歷年競賽時段</button><p>競賽時段為2024-10-29、2025-10-28起24交易日，只有兩年；無效窗口不畫成0%。各方法有效日期不同，不能只按中位數排名，正式結論請看同日期A配對與家族校正報告。公告配置六個原型僅有兩年點，另見報告，未混入這15方法。</p><div class="chart" id="all-chart">'''+all_svg_v4+'</div><div class="chart" id="season-chart" hidden>'+season_svg_v4+'''</div><p>既有策略定義未改；費率0.1425%、賣出稅0.3%、整千股、25檔、投入90%、24交易日、每5日調整（AH／BH為靜態對照）。未知公司行動等排除限制仍適用，並非官賽合規認證。</p><script>function showScope(scope){for(const name of ['all','season']){document.getElementById(name+'-chart').hidden=name!==scope;document.getElementById(name+'-button').setAttribute('aria-pressed',String(name===scope));}}</script></html>''',encoding='utf-8')
assert curve_html_path_v4.read_text(encoding='utf-8').count('class="dot"')==sum(curve_chart_counts_v4.values())
print('All points saved, fixed x range:',curve_chart_counts_v4,[xlo_v4,xhi_v4])
print('Raw snapshot unchanged; company action overlay v4 separately saved.')
print('CURVE_HTML_AND_SNAPSHOT_SAVED')


In [ ]:
# Final v4 report: distinguish corrected accounting, source availability and strategy evidence
report_path_v4=recovery_root/'策略研究_統一比較與公告來源_v4_2026-10-14.md'
lines=['# v4 策略比較與開賽前公告來源盤點','','本報告取代v3作現行比較；v3資料／曲線／報告保留追溯。原150檔行情快照SHA256不變，只有公司行動覆蓋更新，沒有新增策略、調選股參數或追求較高回測值。','','## 本次新增證據','','- 53筆未確認公司行動已全部對上官方歷史事件日期、前價與參考價，並取得53份官方詳細資料。','- 新增41筆每股現金／每千股無償配股，現金增資為零、參考價可核對；加原MOPS1筆，共42筆核對修正。','- 其餘12筆有現金增資／認購，維持排除；不是12筆股利率仍缺來源，而是資金／權利／認購時程尚無帳務模型。另有1個非原股利事件的價格異常旗標留排除。','- 不把權值＋息值當現金股利，不從價差倒推配股，不用每股股票股利元數除以假設面額10。','- 歷史官方結果僅用於公司行動帳務與價格調整，不作事前公告訊號；首次發布／更正時間仍未證實。股利沿既有引擎記應收，真正發放日未全建模。','','## 選擇結論','','沒有方法通過既定改良門檻，未證實改良，也沒有已完成官賽合規認證的方法。A溫和動能＋0.5%NAV小額調整略過仍是符合非等權需求的暫定研究配置，非已證實最佳。A有效窗口由456增至535，並非535次獨立實驗。','','|方法|與A共同窗口|費用後中位數|同日期A中位數|平均配對差pp|平均成本|24位置家族區間pp|48位置家族區間pp|','|---|---:|---:|---:|---:|---:|---|---|']
for r in final_strategy_stats_v4:
    ci=r['family_CI_pp'];lines.append(f"|{method_labels_v4[r['method']]}|{r['paired_windows']}|{r['median_return_pct']:.3f}%|{r['A_same_dates_median_pct']:.3f}%|{r['mean_difference_pp']:+.3f}|{r['mean_cost_pct']:.3f}%|{ci['24'][0]:+.3f}～{ci['24'][1]:+.3f}|{ci['48'][0]:+.3f}～{ci['48'][1]:+.3f}|")
lines.extend(['','14非A方法同一Bonferroni家族5%，每個24／48位置區塊各5000次；風險沿同日期A之P10及中位回撤不惡化超過1pp。不同有效日期不能按跨列中位數排名。年份、早晚段與完整區間存在JSON；沒有真正未看過的樣本外，也不能因增加有效窗口宣稱新的證據獨立。','','## 六個公告調權原型：仍只有兩年','','|方法|年份|費用後報酬|同期A|平均差pp|','|---|---:|---:|---:|---:|'])
for r in legacy_seasonal_rows_v4:
    val=lambda x:'不可判定' if x is None else f'{x:.3f}'
    lines.append(f"|{method_labels_v4[r['method']]}|{r['year']}|{val(r['return_pct'])}|{val(r['A_return_pct'])}|{val(r['difference_pp'])}|")
lines.extend(['','兩個年度點僅作描述，不能稱穩定提升。沒有新增LLM新聞／完整法說權重。','','## 歷年A25：開賽前可核對資料','','下列為每年A動能排序的25檔；初始溫和動能目標是既有4.5%至2.7%排序配置，尚未用以下表格另造權重。財務欄是既有解析器支持的單季本業利益率變化，並非讀完每份財報後的完整質性判斷；營收雙率只使用當時可用、45日內且報告月落後1至2月的來源。未知／較新更正未核對不能當作利空。新聞公開時間只證明當時可用，不是首次官方公告。'])
status_labels={'verified_formal':'可核對正式公告','verified_self':'可核對自結','unknown_originals':'原文不足','newer_self_or_correction_unverified':'較新自結／更正未核對','unknown_source':'來源不足','stale_or_future_reporting_month':'過舊或報告月份不符','verified_original_known_available_time':'當時可用雙率'}
for anchor in ['2024-10-29','2025-10-28']:
    lines.extend(['',f'### {anchor} 起24交易日','','|排序|股票|初始目標|財務狀態／本業變化pp|營收狀態|MoM|YoY|營收來源已知時間|','|---:|---|---:|---|---|---:|---:|---|'])
    for rec in prestart_a25_source_inventory_v5:
        if rec['anchor']!=anchor:continue
        f=rec['financial'];rv=rec['revenue'];delta=f.get('delta_pp')
        fs=status_labels.get(f['status'],f['status'])+('' if delta is None else f'／{delta:+.2f}')
        usable=rv['status']=='verified_original_known_available_time'
        mm=f"{rv['mom']:+.2f}%" if usable else '—';yy=f"{rv['yoy']:+.2f}%" if usable else '—'
        stamp=str(rv.get('published_at','—')) if usable else '—'
        lines.append(f"|{rec['momentum_rank']}|{rec['stock_id']}|{rec['mild_initial_target_weight']*100:.2f}%|{fs}|{status_labels.get(rv['status'],rv['status'])}|{mm}|{yy}|{stamp}|")
lines.extend(['','2024有21／25可核對財務、6／25有效營收雙率；2025有20／25可核對財務（含1自結）、12／25有效營收雙率。這是特定A持股資料覆蓋，非全池覆蓋，不能據此宣稱首次公告缺口已解。來源ID／時間／原公告觀察清單在A25盤點JSON。','','## 已保存成果','','[v4互動箱形圖](策略研究_v4互動箱形圖_2026-10-14.html)保留所有有效點、桃紅中位數、0%線及固定座標快捷鍵；[v4全部窗口JSON](策略研究_逐窗口結果_v4_2026-10-14.json)15方法各650筆，invalid不補零。','[A25逐檔來源清單](A策略_歷年開賽25檔公告來源盤點_v4_2026-10-14.json)、[官方公司行動來源及核對](A策略_公司行動官方核對_2026-10-14.json)。','','產業2016–2025官方指數研究未改：[原產業觀察與圖表](策略研究_統一比較與產業季節性_2026-10-14.md)。這次沒有新增產業交易權重或把今日分類回填歷史。','','## 尚未完成的驗證','','首次公告／更正時間、完整法說文字、個股歷史分類與產業廣度、全歷史Active Share、12筆認購帳務、實際交件／作廢／引用鏈及官賽均價範圍仍未全驗證。未Restart and Run All。瀏覽器對本機file://政策阻擋，圖表完成離線SVG核對但未實際點擊驗證。','','## 原始依據','','[TWSE歷史除權息計算結果](https://www.twse.com.tw/zh/announcement/ex-right/twt49u.html)、[2887／2024-08-06精確單位示例](https://www.twse.com.tw/zh/announcement/ex-right/twt49u-detail.html?2887,20240806)。使用官方頁面宣告的API與STK_NO／T1參數；一般requests HTTPS保留驗證，不绕過網站安全限制，不輸出DB憑證。所有分析於保留IPython kernel，程式及輸出保存原Notebook。'])
report_path_v4.write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('v4 report saved',report_path_v4.name)
print('v4 provisional statistics',next(r for r in final_strategy_stats_v4 if r['method']=='A_mild_tolerance'))


In [ ]:
# Final v4 integrity checks and saved source status; no new experiment
import xml.etree.ElementTree as ET
parsed=json.loads(v4_export_path.read_text(encoding='utf-8'))
assert parsed['status']=='all_fixed_methods_revalidated_and_saved'
assert parsed['raw_snapshot_sha256']==snapshot_sha256
assert len(parsed['windows'])==15 and all(len(rows)==650 for rows in parsed['windows'].values())
assert len(final_strategy_stats_v4)==14 and len(legacy_seasonal_rows_v4)==12 and not final_supported_v4
assert int(engine_v4_valid['A_equal'].sum())==535
assert len(action_corrections_applied_v4)==42 and len(action_unit_unresolved_v4)==12
assert all(explicit_value_v4(action_history_cache_v4['detail_reports'][event['stock_id']+'|'+event['date']]['document']['data'][0][6],'股')>0 for event in action_unit_unresolved_v4)
svg_blocks=re.findall(r'<svg.*?</svg>',curve_html_path_v4.read_text(encoding='utf-8'),re.S)
assert len(svg_blocks)==2
for block in svg_blocks:
    root=ET.fromstring(block);dots=[node for node in root.iter() if node.tag.endswith('circle') and node.get('class')=='dot']
    assert all(chart_left_v4<=float(node.get('cx'))<=chart_right_v4 and 0<float(node.get('cy'))<chart_height_v4 for node in dots)
    assert all('費用後報酬' in node[0].text for node in dots)
assert sum(len([node for node in ET.fromstring(block).iter() if node.tag.endswith('circle') and node.get('class')=='dot']) for block in svg_blocks)==7956+27
action_history_cache_v4['unresolved_units']=action_unit_unresolved_v4
action_history_cache_v4['applied_engine']='v4'
action_history_cache_v4['new_verified_and_applied_events']=41
save_action_history_v4()
assert len(prestart_a25_source_inventory_v5)==50
print('V4_FINAL_QA_PASSED: 9750 records, 7956 valid plotted, A535, 41 extra verified actions; 12 paid actions excluded.')


In [ ]:
# Case review starts from existing A25, known availability and pre-start information only
import sys
print('Kernel:',sys.executable)
print('Retained source objects:',{k:type(globals()[k]).__name__ for k in ['prestart_a25_source_inventory_v5','full_pool_ann_by_stock_v1','full_pool_revenue_news_v1','signal_returns_v4','dates','raw_data'] if k in globals()})
assert all(k in globals() for k in ['prestart_a25_source_inventory_v5','full_pool_ann_by_stock_v1','full_pool_revenue_news_v1'])
print('Relevant safe history markers:',[(i,s.splitlines()[0][:85]) for _,i,s in get_ipython().history_manager.get_range(output=False) if s.startswith('# Historical A25') or s.startswith('# Final v4 report')][-4:])
print('Current name metadata sample:',raw_data['stocks'][:1])
for anchor in ['2024-10-29','2025-10-28']:
    available=[r for r in prestart_a25_source_inventory_v5 if r['anchor']==anchor and r['financial']['status'] in ['verified_formal','verified_self'] and r['revenue']['status']=='verified_original_known_available_time']
    print('Fully numeric-source cases:',anchor,[{'id':r['stock_id'],'rank':r['momentum_rank'],'financial_quarter':r['financial']['quarter'],'op_delta_pp':r['financial']['delta_pp'],'mom':r['revenue']['mom'],'yoy':r['revenue']['yoy'],'month':r['revenue']['year_month'],'financial_source_ids':r['financial']['source_ids'],'revenue_url':r['revenue']['url']} for r in available])


## 2025 A25公告原文案例判讀（尚未產生LLM權重）

以開賽前資訊選案例：2408為資料完整名單中動能排名最高且營收雙率分歧；8299為營收雙率正、本業利益率改善的最高動能排名；3189為最新可核對季度為Q3、營收雙率正但本業利益率下降者。案例為刻意對照樣本，不能代表25檔或證明績效。先盲於開賽後報酬閱讀；不建立LLM權重、不新增門檻。官方重大訊息摘要與完整財務報表／法說全文不同，必須明列文件層級及尚缺細節。

In [ ]:
# Read original MOPS source bodies and bulletin for contrastive pre-start cases, no future outcomes
case_ids_v1=['2408','8299','3189']
case_year_v1=2025;case_anchor_v1='2025-10-28';case_start_v1=di[datetime.date.fromisoformat(case_anchor_v1)]
case_cutoff_v1=datetime.datetime.combine(dates[case_start_v1],datetime.time(8,55))
case_originals_v1=[]
for sid in case_ids_v1:
    rec=next(r for r in prestart_a25_source_inventory_v5 if r['anchor']==case_anchor_v1 and r['stock_id']==sid)
    ann={a['ann_id']:a for a in full_pool_ann_by_stock_v1[sid]}
    source_docs=[ann[identifier] for identifier in rec['financial']['source_ids']]
    news=next(r for r in full_pool_revenue_news_v1 if r['stock_id']==sid and r['news_id']==rec['revenue']['news_id'])
    assert all(prestart_stamp(a)<case_cutoff_v1 for a in source_docs) and news['published_at']<case_cutoff_v1
    pre_returns={'20_sessions_pct':float((np.prod(1+signal_returns_v4[case_start_v1-20:case_start_v1,si[sid]])-1)*100),'60_sessions_pct':float((np.prod(1+signal_returns_v4[case_start_v1-60:case_start_v1,si[sid]])-1)*100)}
    case_originals_v1.append({'stock_id':sid,'anchor':case_anchor_v1,'inventory':rec,'financial_source_documents':curve_json_value_v4(source_docs),'revenue_source_document':curve_json_value_v4(news),'pre_start_price_context':pre_returns,'post_start_return_read_for_selection':False,'document_level':'MOPS earnings-summary announcements and revenue bulletin; not full financial statements or conference transcript'})
    print('CASE',sid,'rank',rec['momentum_rank'],'pre-start prices',pre_returns)
    print('All retained current-year financial subjects before cutoff:',[{'id':a['ann_id'],'stamp':str(prestart_stamp(a)),'subject':a['subject']} for a in full_pool_ann_by_stock_v1[sid] if prestart_stamp(a) and prestart_stamp(a).year==2025 and prestart_stamp(a)<case_cutoff_v1])
    for a in source_docs:
        print('ORIGINAL FINANCIAL',a['ann_id'],str(prestart_stamp(a)),a['subject'],'\\n'+a['content'])
    print('ORIGINAL REVENUE',news['news_id'],str(news['published_at']),news['url'],'\\n'+str(news['content'] or news['summary']))
case_originals_path_v1=recovery_root/'A策略_公告原文對照案例_2026-10-14.json'
case_originals_path_v1.write_text(json.dumps(case_originals_v1,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('PRESTART_THREE_CASE_ORIGINALS_SAVED')


In [ ]:
# Save already-read originals; single-quarter absolute levels and profit-quality contrasts
case_originals_path_v1.write_text(json.dumps(case_originals_v1,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
def case_summary_amounts_v1(document):
    body=unicodedata.normalize('NFKC',document['content']).replace('\r','');values={}
    for field,number,label in [('revenue',4,'營業收入'),('gross',5,'營業毛利'),('operating',6,'營業利益'),('pretax',7,'稅前'),('net',8,'本期淨利'),('parent_net',9,'歸屬於母公司')]:
        m=re.search(r'(?m)^\s*'+str(number)+r'[.、]\s*([^\n]+)',body)
        assert m and label in m[1] and '(仟元)' in m[1]
        value=m[1].rsplit(':',1)[1].strip().replace(',','')
        assert re.fullmatch(r'\(?-?\d+(?:\.\d+)?\)?',value)
        values[field]=float(value.strip('()'))*(-1 if value.startswith('(') else 1)
    return values
case_accounting_details_v1=[]
for case in case_originals_v1:
    cumulative=[case_summary_amounts_v1(d) for d in case['financial_source_documents']]
    quarters=[]
    for i,total in enumerate(cumulative):
        q={k:v-(cumulative[i-1][k] if i else 0) for k,v in total.items()};assert q['revenue']>0
        q.update(quarter=i+1,operating_margin_pct=q['operating']/q['revenue']*100,gross_margin_pct=q['gross']/q['revenue']*100,pretax_minus_operating=q['pretax']-q['operating'])
        quarters.append(q)
    previous,current=quarters[-2:]
    detail={'stock_id':case['stock_id'],'unit':'TWD thousands','quarters':quarters,'comparison':{'revenue_qoq_pct':(current['revenue']/previous['revenue']-1)*100,'operating_margin_change_pp':current['operating_margin_pct']-previous['operating_margin_pct'],'gross_margin_change_pp':current['gross_margin_pct']-previous['gross_margin_pct'],'operating_amount_change_thousand':current['operating']-previous['operating'],'parent_net_change_thousand':current['parent_net']-previous['parent_net'],'parent_net_qoq_pct':(current['parent_net']/previous['parent_net']-1)*100 if current['parent_net']>0 and previous['parent_net']>0 else None}}
    assert np.isclose(detail['comparison']['operating_margin_change_pp'],case['inventory']['financial']['delta_pp'])
    case_accounting_details_v1.append(detail)
    print('Single-quarter accounting contrast:',json.dumps(detail,ensure_ascii=False))
print('CASE_ACCOUNTING_CONTRASTS_VERIFIED; first-publication and full-statement coverage still separate gates.')


In [ ]:
# Pre-start disclosed event calendar; do not read post-start realized earnings into judgment
case_calendar_v1=[]
for sid in ['8299','3189']:
    for ann in full_pool_ann_by_stock_v1[sid]:
        stamp=prestart_stamp(ann)
        if not stamp or not(datetime.datetime(2025,10,1)<=stamp<case_cutoff_v1):continue
        if '預計' in ann['subject'] or '召開日期' in ann['subject'] or '法說' in ann['subject']:
            item={'stock_id':sid,'source_id':ann['ann_id'],'stamp':str(stamp),'subject':ann['subject'],'content':ann['content']}
            case_calendar_v1.append(item);print('Known calendar original:',sid,item)
print('NO_POSTSTART_EARNINGS_READ_FOR_CASE_JUDGMENT')


In [ ]:
# Structured LLM-assisted reading: evidence, contradictions and research-only direction, no weight assigned
import hashlib
nanya_primary_v1={'url':'https://www.nanya.com/en/IR/16?IRId=10046','calendar_url':'https://www.nanya.com/en/IR/14/Investor%20Calendar?IRId=3270','release_date':'2025-10-13','exact_release_time':None,'quarter':'2025Q3','basis':'issuer published unaudited self-reported release','revenue_TWD_million':18779,'gross_TWD_million':3465,'operating_TWD_million':1119,'net_TWD_million':1563,'gross_margin_pct':18.5,'operating_margin_pct':6.0,'EPS_TWD':.50,'available_before_case_start_by_date':True,'included_in_v4_model':False}
assert datetime.date.fromisoformat(nanya_primary_v1['release_date'])<datetime.date.fromisoformat(case_anchor_v1)
primary_cache_v1={'source':nanya_primary_v1,'raw_archived':False}
try:
    response=requests.get(nanya_primary_v1['url'],timeout=25);response.raise_for_status()
    page=response.content.decode('utf-8')
    assert '2025/10/13' in page and ('18,779' in page or '18779' in page)
    primary_cache_v1.update(raw_archived=True,html=page,sha256=hashlib.sha256(response.content).hexdigest())
except Exception as error:primary_cache_v1['archive_error']=type(error).__name__
case_review_v1={
 '2408':{'company':'南亞科','direction':'維持；列為有條件加碼研究候選','support':['補查的公司Q3自結已由虧轉盈，並在开賽前公開','9月營收YoY157.81%'], 'contradictions':['9月MoM−1.45%','開賽前60日調整後報酬約184.36%，不能由好消息推論仍未反映','DB只保留Q2，舊訊號不能當現實最新；Q3自結未經核閱'], 'missing':['市場事前盈利預期／估值／持續性與完整Q3財報核閱細節'], 'numeric_weight':None,'confidence':'資訊改善方向可核對；加碼報酬效果未測'},
 '8299':{'company':'群聯','direction':'維持並查核；暫不直接因雙率正就加碼','support':['Q2本業利益率由8.39%升至13.11%','9月MoM9.80%、YoY47.04%'], 'contradictions':['Q2毛利率下降約1.87pp','Q2歸母淨利季減約34.72%','稅前與本業利益差額轉負，尚未拆解為匯兌或其他項目'], 'missing':['業外／稅負／一次性項目明細、研發及費用變化的持續性','開賽期間11/7法說內容；其結果在開賽時尚未知'], 'numeric_weight':None,'confidence':'本業改善可核對；完整盈利品質與事件後果不可判定'},
 '3189':{'company':'景碩','direction':'降低加碼信心；列為減碼研究候選','support':['9月MoM4.82%、YoY35.51%','Q3單季營收較Q2增加8.29%'], 'contradictions':['Q3毛利率由約21.08%降至18.96%','Q3本業利益率由約6.48%降至6.06%','Q3歸母淨利僅季增約0.34%，成長轉成獲利的幅度有限'], 'missing':['产品組合／產能利用率／成本／折舊原因、同比与同季基準'], 'numeric_weight':None,'confidence':'成長與利潤率分歧可核對；減碼能否改善24日報酬未測'}}
case_review_output_v1={'decision_cutoff':'2025-10-28 08:55 Asia/Taipei','selection':'purposive A25 contrast cases, no stock post-start return accessed for case selection','cases':case_originals_v1,'quarter_accounting':case_accounting_details_v1,'issuer_primary_update':nanya_primary_v1,'prestart_disclosed_calendar':case_calendar_v1,'review':case_review_v1,'status':'case_reading_complete; no LLM weighting backtest or trades','generalization':'three selected cases not evidence for whole A25 or strategy superiority','v4_prototype_caveat':'financial/revenue source corpus incomplete; known missed prestart 2408 Q3 issuer release. Existing v4 outputs are as-retained-corpus prototypes, not full-information announcement backtests.'}
case_review_output_path_v1=recovery_root/'A策略_公告原文判讀結果_2026-10-14.json'
case_review_output_path_v1.write_text(json.dumps(case_review_output_v1,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
primary_cache_path_v1=recovery_root/'A策略_公告案例官方來源補查_2026-10-14.json'
primary_cache_path_v1.write_text(json.dumps(primary_cache_v1,ensure_ascii=False,indent=2),encoding='utf-8')
case_report_path_v1=recovery_root/'動能持股公告個案研究_2026-10-14.md'
old=case_report_path_v1.read_text(encoding='utf-8')
section=['## 最新：2025 A25公告原文對照判讀','','本節為2025-10-28開賽前的歷史研究，不是2026當前股票推薦。保留25檔A主線，挑3檔對照原文，不新增LLM倍率、不更動持股或權重。所讀文件是重大訊息財務摘要、公司自結新聞稿與營收速報；不是已讀完整財務報表所有附註或完整法說問答。案例依資訊與訊號分歧選取，未查看個股開賽後報酬來挑案例；資料與總體策略績效已看過，不能冒稱真正樣本外。','','### 一個會改變判讀的來源缺口','','南亞科在[2025-10-13公司公告](https://www.nanya.com/en/IR/16?IRId=10046)已公開Q3自結：本業轉盈、營業利益率6.0%；早於10/28開賽。DB財務摘要卻只保留到Q2。因此「DB有可核對資料」不等於「已取得當時最新公開資料」。此公司公告是未核閱自結；精確首發時分未知，但日期早於決策可確認。沒有把現在抓取時間當歷史時間。','','原v4純財務／公告調權仍是保留來源範圍內的原型，不能稱完整公告策略。新Q3資訊只用於本次案例閱讀，未只替一檔補資料就重算全池選股，避免不對稱覆蓋；來源ID／URL／時間与原文另存JSON。','','### 三個案例的可核對對照','','|股票|已讀季度|財務重點|9月MoM／YoY|研究方向，尚非權重規則|','|---|---|---|---|---|','|南亞科2408|DB Q1/Q2＋補查公司Q3自結|Q2本業仍虧損；已公開Q3本業轉盈，不能用舊Q2判定最新現況|−1.45%／＋157.81%|維持，有條件加碼研究候選；需看預期與已反映程度|','|群聯8299|Q1/Q2財務摘要|本業利益率改善，毛利率下降、歸母淨利季減34.72%|＋9.80%／＋47.04%|維持並查核，不直接因雙率正加碼|','|景碩3189|Q1/Q2/Q3財務摘要|Q3營收增8.29%，毛利率降2.12pp、本業利益率降0.42pp|＋4.82%／＋35.51%|降低加碼信心，減碼研究候選；原因尚待附註核對|','','群聯最新Q2公告在2025-08-14；Q3法說日期11/7已於10/15公告，落在比賽24交易日內。這是開賽前可知事件日，不代表可預知其數字。現在找到的11/7業績或10月營收不可放回10/28判讀。景碩Q3摘要10/27 18:03:42已公開，可在10/28 08:55截止前讀到。','','### 為何不能只用正負號','','南亞科DB的Q2本業利益率改善1.14pp，但仍為−42.76%，絕對本業虧損還擴大。若漏讀已公開Q3自結，會把舊狀況當新狀況。群聯本業增加、淨利卻下降，必須拆解差額與費用來源；在未讀到明細前不能擅自命名為匯損。景碩營收成長但毛利率下降，需要查產品組合與成本；同比及季節性不足時不能推論公司變差。','','南亞科／群聯／景碩在開賽前60日的調整後價格報酬約184.36%／85.09%／40.09%，只是既有動能背景，不能由漲幅估出消息已反映多少、或宣稱公告是上漲原因。沒有分析師事前預期與估值，不能把成長當成超預期。','','### 後續判讀需逐项留下的欄位','','1. 文件完整度與發布時間；是否已有更近的自結、公司新聞稿或更正。','2. 季度累計數轉單季；獲利正負與絕對額，避免改善率掩蓋虧損。','3. MoM／YoY、同比基期與連續月份；數字成長與本業轉換是否一致。','4. 毛利、本業、業外／稅負與一次性項目；未核對原因保持未知。','5. 管理層前瞻陳述與已實現數字分列，不能把法說預期當實際成果。','6. 開賽前股價背景與可知事件日；事前預期差、估值和利空是否反映，不足就不可判定。','7. 加碼／減碼只先形成研究標籤；倍率、期限、單股上限與績效驗證另行定義，不按這3例結果造規則。','','### 原始來源與保存','','DB重大訊息原文IDs、時間與季度算式在原Notebook cells133–135；原文歸檔[A策略_公告原文對照案例_2026-10-14.json](A策略_公告原文對照案例_2026-10-14.json)，[判讀與算式](A策略_公告原文判讀結果_2026-10-14.json)。','[南亞科Q3公司原公告](https://www.nanya.com/en/IR/16?IRId=10046)及[法說行事曆](https://www.nanya.com/en/IR/14/Investor%20Calendar?IRId=3270)核對日期，原網頁補查歸檔[A策略_公告案例官方來源補查_2026-10-14.json](A策略_公告案例官方來源補查_2026-10-14.json)。','营收来源：[南亞科9月](https://news.cnyes.com/news/id/6177441)、[群聯9月](https://news.cnyes.com/news/id/6183976)、[景碩9月](https://news.cnyes.com/news/id/6177700)，在保留DB原文核對雙率。群聯Q2公司PDF搜尋摘要可核對部分數字，但連結目前404，沒有宣稱已讀其完整原件、現金流與附註。','','這輪完成的是可追溯的原文案例判讀，不是LLM已通過加減碼回測。三例皆未配置新數字權重，也未提交交易。']
case_report_path_v1.write_text(old.split('\n',1)[0]+'\n\n'+'\n'.join(section)+'\n\n'+old.split('\n',1)[1],encoding='utf-8')
print('Case report and structured review saved:',case_report_path_v1.name,case_review_output_path_v1.name,'primary raw archive',primary_cache_v1['raw_archived'])
print('ANNOUNCEMENT_THREE_CASE_REVIEW_COMPLETE; no LLM weight or performance inferred.')


In [ ]:
# A25 latest-source audit: fixed complete list, pre-start scope, no new weights
import sys
print('Kernel',sys.executable)
assert 'prestart_a25_source_inventory_v5' in globals() and 'full_pool_ann_by_stock_v1' in globals()
a25_freshness_scope_v1=[r for r in prestart_a25_source_inventory_v5 if r['anchor']=='2025-10-28']
assert len(a25_freshness_scope_v1)==25
for r in a25_freshness_scope_v1:
    print({'id':r['stock_id'],'rank':r['momentum_rank'],'status':r['financial']['status'],'quarter':r['financial'].get('quarter'),'latest_retained_stamp':max(r['financial'].get('source_stamps',['unknown'])),'mom_yoy_status':r['revenue']['status']})
print('Safe recent markers:',[(i,s.splitlines()[0][:80]) for _,i,s in get_ipython().history_manager.get_range(output=False) if s.startswith('# Structured LLM')][-2:])


## 2025 A25最新資訊覆蓋查核

固定2025-10-28 08:55截止與原25檔，不用賽後數字。擴到全重大訊息，避免原財務關鍵字漏掉公告；新聞只作補查線索，實際數字／預估／事件預告分列。查無新文件不能證明當時不存在，需公司IR或官方原件。新增來源不不對稱回填原全池純財務回測、不造LLM權重。

In [ ]:
# A25 all original announcements and retained financial news, read-only and before cutoff
import ast,inspect
a25_freshness_ids_v1=[r['stock_id'] for r in a25_freshness_scope_v1]
assert len(set(a25_freshness_ids_v1))==25 and all(re.fullmatch(r'\d{4}',s) for s in a25_freshness_ids_v1)
tree=ast.parse(inspect.getsource(fetch_full_pool_announcements_v1));tree.body[0].name='fetch_a25_latest_corpus_v1'
stmt=next(x for x in tree.body[0].body if isinstance(x,ast.Try))
with_stmt=next(x for x in stmt.body if isinstance(x,ast.With))
with_stmt.body=ast.parse("""
cur.execute('SET SESSION TRANSACTION READ ONLY')
cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
slots=','.join(['%s']*len(a25_freshness_ids_v1))
cur.execute('SELECT stock_id,ann_id,announce_date,announce_time,subject,event_date,content,source FROM announcements WHERE stock_id IN ('+slots+') AND announce_date BETWEEN %s AND %s ORDER BY stock_id,announce_date,announce_time',tuple(a25_freshness_ids_v1)+('2025-07-01','2025-10-28'))
rows=cur.fetchall()
cur.execute('SELECT n.news_id,n.published_at,n.title,n.summary,n.content,n.url,n.source,ns.stock_id,ns.match_type FROM news n JOIN news_stock ns ON n.news_id=ns.news_id WHERE ns.stock_id IN ('+slots+') AND n.published_at>=%s AND n.published_at<%s AND n.title REGEXP %s ORDER BY ns.stock_id,n.published_at,n.news_id LIMIT 20000',tuple(a25_freshness_ids_v1)+('2025-07-01',case_cutoff_v1,'財報|財務|自結|法說|獲利|每股|淨利|季報|第.?季|3Q|Q3|前三季'))
schema=cur.fetchall()
cur.execute('SELECT stock_id,name,market FROM stocks WHERE stock_id IN ('+slots+')',tuple(a25_freshness_ids_v1))
names=cur.fetchall()
""").body
ret=next(x for x in stmt.body if isinstance(x,ast.Return));ret.value=ast.Tuple(elts=[ast.Name(id='rows',ctx=ast.Load()),ast.Name(id='schema',ctx=ast.Load()),ast.Name(id='names',ctx=ast.Load())],ctx=ast.Load())
ast.fix_missing_locations(tree);exec(compile(tree,'<readonly-A25-latest-corpus>','exec'))
a25_all_announcements_v1,a25_financial_news_v1,a25_names_v1=fetch_a25_latest_corpus_v1()
a25_all_announcements_v1=[a for a in a25_all_announcements_v1 if prestart_stamp(a) and prestart_stamp(a)<case_cutoff_v1]
assert len(a25_financial_news_v1)<20000,'Source cap reached; not complete'
a25_latest_corpus_path_v1=recovery_root/'A策略_2025開賽25檔最新來源查核_2026-10-14.json'
a25_latest_corpus_v1={'scope':'fixed A25,2025-07-01 through2025-10-28 08:55; all retained original announcements; finance-news leads, not first-publication certification','cutoff':str(case_cutoff_v1),'stock_ids':a25_freshness_ids_v1,'names_snapshot':a25_names_v1,'announcements':curve_json_value_v4(a25_all_announcements_v1),'news_leads':curve_json_value_v4(a25_financial_news_v1),'news_stock_link_is_not_source_verification':True}
a25_latest_corpus_path_v1.write_text(json.dumps(a25_latest_corpus_v1,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Latest-source query coverage:',len(a25_all_announcements_v1),'original notices;',len(a25_financial_news_v1),'news stock links')
for record in a25_freshness_scope_v1:
    sid=record['stock_id'];name=next((x['name'] for x in a25_names_v1 if x['stock_id']==sid),sid)
    ann=[a for a in a25_all_announcements_v1 if a['stock_id']==sid and re.search(r'財務|財報|自結|營運績效|法說|法人說明',a['subject'])]
    news=[a for a in a25_financial_news_v1 if a['stock_id']==sid and a['published_at']>=datetime.datetime(2025,10,1)]
    print('STOCK',sid,name,'ANN',[(str(prestart_stamp(a)),a['ann_id'],a['subject']) for a in ann[-6:]],'NEWS',[(str(a['published_at']),a['news_id'],a['title'],a['url']) for a in news[-5:]])
print('A25_FULL_SOURCE_CORPUS_SAVED')


In [ ]:
# Compact all-stock source triage; original bodies only for newer-results/corrections/missing parsers
a25_triage_v1=[]
old_fin_ids={a['ann_id'] for a in full_pool_financial_announcements_v1}
for record in a25_freshness_scope_v1:
    sid=record['stock_id'];name=next(r['name'] for r in a25_names_v1 if r['stock_id']==sid)
    stock_ann=[a for a in a25_all_announcements_v1 if a['stock_id']==sid]
    retained=record['financial'].get('source_stamps',[])
    old_stamp=datetime.datetime.fromisoformat(max(retained)) if retained else None
    candidates=[]
    for a in stock_ann:
        title=unicodedata.normalize('NFKC',a['subject']);stamp=prestart_stamp(a)
        kind='other'
        if re.search(r'更正|補充',title) and re.search(r'財報|財務|自結|法說|公告|營收',title):kind='correction_or_supplement_needs_read'
        elif re.search(r'注意交易|注意.*資訊|財務業務.*重大訊息',title):kind='attention_disclosure_period_unknown'
        elif re.search(r'董事會.*(?:預計|召開日期)|財務報告.*(?:預計|召開日期)|將於.*董事會|受邀|召開.*法[說人]|舉行.*法人',title):kind='event_schedule_not_actual_result'
        elif re.search(r'財務報|財報|自結.*(?:損益|獲利)|(?:法人說明會|法說).*摘要|發布.*財務|財務.*發布',title):kind='actual_results_or_summary_needs_read'
        if kind!='other' and (old_stamp is None or stamp>old_stamp):
            candidates.append({'ann_id':a['ann_id'],'stamp':str(stamp),'subject':a['subject'],'kind':kind,'excluded_from_old_financial_query':a['ann_id'] not in old_fin_ids})
    stock_news=[a for a in a25_financial_news_v1 if a['stock_id']==sid and a['published_at']>=datetime.datetime(2025,10,1) and name in a['title']]
    relevant=[{'id':a['news_id'],'stamp':str(a['published_at']),'title':a['title'],'url':a['url']} for a in stock_news]
    a25_triage_v1.append({'stock_id':sid,'name':name,'old_status':record['financial']['status'],'old_quarter':record['financial'].get('quarter'),'old_latest':str(old_stamp) if old_stamp else None,'original_candidates':candidates,'single_name_news_leads':relevant})
    print('TRIAGE',sid,name,'new original',[(c['ann_id'],c['kind'],c['subject']) for c in candidates],'focused news',relevant)
print('New actual-summary originals outside old query:',[(r['stock_id'],c['ann_id'],c['subject']) for r in a25_triage_v1 for c in r['original_candidates'] if c['kind']=='actual_results_or_summary_needs_read' and c['excluded_from_old_financial_query']])
for sid in ['2408','2337','6669','3665','6223','3443']:
    docs=[a for a in a25_all_announcements_v1 if a['stock_id']==sid and (re.search(r'財務報|財報|自結|摘要|注意|更正',a['subject']))]
    for a in docs[-3:]:print('TARGET ORIGINAL',sid,a['ann_id'],str(prestart_stamp(a)),a['subject'],'\\n'+a['content'])


In [ ]:
# All-original semantic audit and declared event calendar; a future meeting is not reported earnings
def a25_source_kind_v1(a):
    title=unicodedata.normalize('NFKC',a['subject']);body=unicodedata.normalize('NFKC',a['content'] or '')
    if re.search(r'更正',title) and re.search(r'财|財|費用|會計|淨利|損益|營收',title):return 'financial_correction_needs_original_review'
    if re.search(r'注意交易|注意.*資訊|財務業務.*重大訊息',title):return 'attention_monthly_or_quarterly_numeric_disclosure'
    if '董事會預計召開日期' in body or re.search(r'董事會.*(?:預計|召開日期)|將提.*董事會|將於.*董事會',title):return 'future_board_plan_not_earnings'
    if '累計至本期止營業收入' in body and '累計至本期止營業利益' in body:return 'published_formal_financial_summary'
    if re.search(r'(?:第.?季|Q[1234]).*(?:自結|損益)|自結.*(?:第.?季|Q[1234])',title):return 'published_quarter_self_report'
    if re.search(r'法人說明會.*摘要|法說.*摘要',title):return 'published_conference_summary'
    if re.search(r'召開.*法人|召開.*法說|受邀|舉行.*法人',title):return 'meeting_plan_not_new_earnings'
    return 'other_not_financial_update'
def a25_periods_v1(a,kind):
    body=unicodedata.normalize('NFKC',a['content'] or '')
    if kind=='published_formal_financial_summary':
        m=re.search(r'(?:114|2025)/0?1/0?1\s*[-~至]\s*(?:114|2025)/(\d{1,2})/(\d{1,2})',body)
        if m:return ['2025Q'+str(int(m[1])//3)+' cumulative']
    if kind=='attention_monthly_or_quarterly_numeric_disclosure':
        periods=[]
        for m in re.finditer(r'(?:114|2025)年?\s*(\d{1,2})\s*月',body):
            if 1<=int(m[1])<=12:periods.append('2025-'+f'{int(m[1]):02d}')
        for m in re.finditer(r'(?:114|2025)年?\s*第?\s*([1234一二三四])\s*季',body):
            digit={'一':'1','二':'2','三':'3','四':'4'}.get(m[1],m[1]);periods.append('2025Q'+digit)
        return sorted(set(periods))
    if kind in ['published_quarter_self_report','published_conference_summary']:
        text=unicodedata.normalize('NFKC',a['subject']+' '+body)
        return sorted(set('2025Q'+{'一':'1','二':'2','三':'3','四':'4'}.get(m[1],m[1]) for m in re.finditer(r'(?:114|2025)年?\s*第?\s*([1234一二三四])\s*季',text)))
    return []
def a25_calendar_dates_v1(a):
    body=unicodedata.normalize('NFKC',a['content'] or '')
    dates_found=[]
    for kind,pattern in [('earnings_board',r'董事會預計召開日期\s*[:：]\s*(?:114|2025)[/.](\d{1,2})[/.](\d{1,2})'),('investor_meeting',r'(?:召開法人說明會之日期|法人說明會之日期)\s*[:：]\s*(?:114|2025)[/.](\d{1,2})[/.](\d{1,2})')]:
        for m in re.finditer(pattern,body):
            day=datetime.date(2025,int(m[1]),int(m[2]))
            event_type=kind
            if kind=='investor_meeting' and re.search(r'3Q25|第三季|第3季|Q3|財報',a['subject']+' '+body):event_type='earnings_meeting'
            dates_found.append((event_type,day))
    return dates_found
a25_calendar_end_v1=dates[case_start_v1+23]
a25_freshness_audit_v1=[];a25_declared_events_v1=[]
for record in a25_freshness_scope_v1:
    sid=record['stock_id'];name=next(x['name'] for x in a25_names_v1 if x['stock_id']==sid)
    sources=[a for a in a25_all_announcements_v1 if a['stock_id']==sid]
    typed=[]
    oldstamp=max(record['financial'].get('source_stamps',[]),default=None)
    for a in sources:
        kind=a25_source_kind_v1(a)
        if kind!='other_not_financial_update':
            typed.append({'ann_id':a['ann_id'],'stamp':str(prestart_stamp(a)),'subject':a['subject'],'kind':kind,'periods':a25_periods_v1(a,kind),'newer_than_model_source':oldstamp is None or str(prestart_stamp(a))>oldstamp,'outside_original_query':a['ann_id'] not in old_fin_ids})
        for event_type,day in a25_calendar_dates_v1(a):
            if dates[case_start_v1]<=day<=a25_calendar_end_v1:
                a25_declared_events_v1.append({'stock_id':sid,'name':name,'kind':event_type,'event_date':str(day),'ann_id':a['ann_id'],'known_at':str(prestart_stamp(a)),'subject':a['subject'],'scheduled_not_actual_release_certification':True})
    actual=[x for x in typed if x['kind'] in ['published_formal_financial_summary','published_quarter_self_report','published_conference_summary']]
    newest=max(actual,key=lambda x:x['stamp']) if actual else None
    attention=[x for x in typed if x['kind']=='attention_monthly_or_quarterly_numeric_disclosure' and x['newer_than_model_source']]
    corrections=[x for x in typed if x['kind']=='financial_correction_needs_original_review']
    audit={'stock_id':sid,'name_snapshot':name,'momentum_rank':record['momentum_rank'],'original_model':record['financial'],'latest_actual_in_retained_originals':newest,'newer_attention_disclosures':attention,'financial_corrections':corrections,'typed_originals':typed,'issuer_complete_history_verified':False,'first_publication_certified':False,'source_status':'retained original corpus reviewed; missing outside-source documents still possible'}
    a25_freshness_audit_v1.append(audit)
print('Semantic audit stocks:',len(a25_freshness_audit_v1),'window',str(dates[case_start_v1]),str(a25_calendar_end_v1))
for a in a25_freshness_audit_v1:
    latest=a['latest_actual_in_retained_originals']
    print('SEMANTIC',a['stock_id'],a['name_snapshot'],'latest',None if latest is None else (latest['stamp'],latest['kind'],latest['periods']),'new attention',len(a['newer_attention_disclosures']),'corrections',[(c['ann_id'],c['subject']) for c in a['financial_corrections']])
print('Declared in-window events:',a25_declared_events_v1)
print('Old-parser failure example IDs:',[(r['stock_id'],r['latest_actual_in_retained_originals']['ann_id']) for r in a25_freshness_audit_v1 if r['original_model']['status']=='unknown_originals' and r['latest_actual_in_retained_originals'] is not None])
print('A25_SEMANTIC_AND_EVENT_AUDIT_READY')


In [ ]:
# Read all newer attention disclosure periods and numeric blocks; publication recency is not period recency
attention_original_review_v1=[]
for audit in a25_freshness_audit_v1:
    for ref in audit['newer_attention_disclosures']:
        ann=next(a for a in a25_all_announcements_v1 if a['ann_id']==ref['ann_id'])
        body=unicodedata.normalize('NFKC',ann['content'] or '')
        begin=body.find('3.財務')
        if begin<0:begin=body.find('3.財務業務')
        end=body.find('4.有無',max(0,begin))
        block=body[max(0,begin):end if end>=0 else len(body)]
        attention_original_review_v1.append({'stock_id':audit['stock_id'],'name':audit['name_snapshot'],'ann_id':ann['ann_id'],'stamp':str(prestart_stamp(ann)),'reported_periods':ref['periods'],'numeric_block':block,'not_a_full_new_quarter_report':True})
        print('ATTENTION ORIGINAL',audit['stock_id'],audit['name_snapshot'],ann['ann_id'],str(prestart_stamp(ann)),'\\n'+block)
print('PERIOD_SPECIFIC_ATTENTION_ORIGINALS_REVIEWED',len(attention_original_review_v1))
for sid in ['6669','3665','3443']:
    docs=full_pool_ann_by_stock_v1[sid]
    printed=False
    for ann in docs:
        if prestart_stamp(ann) and prestart_stamp(ann).year==2025 and prestart_stamp(ann)<case_cutoff_v1 and '累計至本期止營業收入' in (ann['content'] or ''):
            quarter=prestart_quarter(ann,2025)
            print('PARSER DIAGNOSTIC',sid,ann['ann_id'],ann['subject'],'old_quarter',quarter,'amounts_present',pure_financial_amounts_v1(ann) is not None)


In [ ]:
# Inspect why source-present statements failed extraction; do not neutral-fill missing fundamentals
for sid in ['3665','3443']:
    for a in full_pool_ann_by_stock_v1[sid]:
        if a['ann_id'] in ['d3e0e1adf7537b4b79e1f6166546b8eac067187c','33586cd6178cec2d297e53411873d608ffc0d7dd','798198da285ff6a30f34520b89371251daafff5c']:
            print('PARSER ORIGINAL',sid,a['ann_id'],a['subject'],'\\n'+a['content'])


In [ ]:
# Source-derived extractor staged for audit only; no change to scores_v4 or portfolio weights
def a25_formal_summary_extractor_v1(ann):
    body=unicodedata.normalize('NFKC',ann.get('content') or '').replace('\r','')
    period=re.search(r'起訖日期[^:]*:\s*(\d{3,4})/(\d{1,2})/(\d{1,2})\s*[-~至]\s*(\d{3,4})/(\d{1,2})/(\d{1,2})',body)
    if not period:return None
    y1,m1,d1,y2,m2,d2=map(int,period.groups());y1=y1+1911 if y1<1911 else y1;y2=y2+1911 if y2<1911 else y2
    if y1!=y2 or (m1,d1)!=(1,1) or (m2,d2) not in [(3,31),(6,30),(9,30),(12,31)]:return None
    cutoff=prestart_stamp(ann)
    if cutoff is None or cutoff.date()<datetime.date(y2,m2,d2):return None
    vals={}
    for field,number,label in [('revenue',4,'營業收入'),('operating',6,'營業利益')]:
        line=re.search(r'(?m)^\s*'+str(number)+r'[.、]\s*([^\n]+)',body)
        if not line or label not in line[1] or '累計' not in line[1] or '(仟元)' not in line[1] or ':' not in line[1]:return None
        value=line[1].rsplit(':',1)[1].strip()
        match=re.fullmatch(r'(?:新台幣\s*)?(\(?[+-]?\d[\d,]*(?:\.\d+)?\)?)(?:\s*仟元)?',value)
        if not match:return None
        numeric=match[1].replace(',','');vals[field]=float(numeric.strip('()'))*(-1 if numeric.startswith('(') else 1)
    if vals['revenue']<=0:return None
    return {'year':y2,'quarter':m2//3,'period_start':str(datetime.date(y1,m1,d1)),'period_end':str(datetime.date(y2,m2,d2)),'unit':'thousand; explicit source label, numeric TWD prefix accepted','amounts':vals,'ann_id':ann['ann_id'],'known_at':str(cutoff),'not_applied_to_v4_scores':True}
a25_staged_parser_results_v1=[];preserved_values_count=0
for sid in a25_freshness_ids_v1:
    for ann in full_pool_ann_by_stock_v1[sid]:
        if not prestart_stamp(ann) or prestart_stamp(ann)>=case_cutoff_v1:continue
        extracted=a25_formal_summary_extractor_v1(ann)
        if extracted:
            old=pure_financial_amounts_v1(ann)
            if old:
                assert all(np.isclose(old[k],extracted['amounts'][k],rtol=0,atol=.001) for k in ['revenue','operating'])
                preserved_values_count+=1
            a25_staged_parser_results_v1.append(dict(extracted,stock_id=sid))
parser_repaired_a25_v1=[]
for sid in ['6669','3665','3443']:
    records=[a for a in a25_staged_parser_results_v1 if a['stock_id']==sid and a['year']==2025 and a['quarter'] in [1,2]]
    latest={q:max((a for a in records if a['quarter']==q),key=lambda a:a['known_at']) for q in [1,2]}
    first=latest[1]['amounts'];second={k:latest[2]['amounts'][k]-first[k] for k in first}
    assert first['revenue']>0 and second['revenue']>0
    repaired={'stock_id':sid,'source_ids':[latest[q]['ann_id'] for q in [1,2]],'source_stamps':[latest[q]['known_at'] for q in [1,2]],'Q1_revenue_thousand':first['revenue'],'Q1_operating_thousand':first['operating'],'Q2_revenue_thousand':second['revenue'],'Q2_operating_thousand':second['operating'],'margin_change_pp':100*(second['operating']/second['revenue']-first['operating']/first['revenue']),'stage_only_no_v4_update':True}
    parser_repaired_a25_v1.append(repaired);print('Source-present parser repair verified:',repaired)
print('Existing parseable amounts unchanged:',preserved_values_count)
print('No financial score/weight changed; whole-universe evaluation would require a separate controlled source/extractor version.')


In [ ]:
# Persist complete fixed-25 audit, distinctions, verification limits and declared event dates
for audit in a25_freshness_audit_v1:
    keep=[]
    for ref in audit['newer_attention_disclosures']:
        ann=next(a for a in a25_all_announcements_v1 if a['ann_id']==ref['ann_id'])
        if '公司債相關資訊' in ann['content'] and '營業收入' not in ann['content']:
            ref['kind']='security_disclosure_not_earnings'
        else:keep.append(ref)
    audit['newer_attention_disclosures']=keep
    audit['staged_parser_repair']=next((r for r in parser_repaired_a25_v1 if r['stock_id']==audit['stock_id']),None)
    audit['declared_in_window_events']=[e for e in a25_declared_events_v1 if e['stock_id']==audit['stock_id']]
    if audit['stock_id']=='2408':
        audit['missed_by_old_title_filter']={'ann_id':'9d4f234378a373ac94b74ced64d48e4bcaf299f3','known_at':'2025-10-13 16:32:51','period':'2025Q3 self-reported','cause':'old subject filter excluded conference summary; DB does contain disclosure','issuer_crosscheck':'https://www.nanya.com/en/IR/16?IRId=10046','first_publication_not_certified':True}
actual_count=sum(r['latest_actual_in_retained_originals'] is not None for r in a25_freshness_audit_v1)
attention_stocks=sum(bool(r['newer_attention_disclosures']) for r in a25_freshness_audit_v1)
attention_count=sum(len(r['newer_attention_disclosures']) for r in a25_freshness_audit_v1)
earnings_events=[e for e in a25_declared_events_v1 if e['kind'] in ['earnings_board','earnings_meeting']]
earnings_issuer_count=len({e['stock_id'] for e in earnings_events})
audit_summary_v1={'stocks':25,'original_notices':len(a25_all_announcements_v1),'news_stock_links':len(a25_financial_news_v1),'stocks_with_actual_summary_retained':actual_count,'stocks_with_newer_published_attention_earnings':attention_stocks,'newer_attention_earnings_notices':attention_count,'parser_fix_verified_stocks':3,'prior_parseable_values_preserved':preserved_values_count,'earnings_calendar_issuers':earnings_issuer_count,'earnings_calendar_notice_count':len(earnings_events),'individual_first_publication_certified':False,'complete_issuer_IR_history_certified':False,'model_or_weights_changed':False,'audit_window':[str(dates[case_start_v1]),str(a25_calendar_end_v1)]}
a25_audit_result_path_v1=recovery_root/'A策略_2025開賽25檔資訊新鮮度結果_2026-10-14.json'
a25_audit_result_v1={'summary':audit_summary_v1,'audit':a25_freshness_audit_v1,'declared_events':a25_declared_events_v1,'attention_original_reading':attention_original_review_v1,'staged_extractor_results':parser_repaired_a25_v1,'next_work':'uniform extractor and source-type correction for all decision timestamps and supported universe, source-coverage checks first; no three-stock asymmetric score patch'}
a25_audit_result_path_v1.write_text(json.dumps(a25_audit_result_v1,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
report_path_a25_v1=recovery_root/'A策略_2025開賽25檔資訊新鮮度查核_2026-10-14.md'
lines=['# 2025歷史開賽A25資訊新鮮度查核','','範圍固定：2025-10-28 08:55前的A25，競賽窗口10/28–11/28共24交易日。不是2026當前投資推薦。核對保留來源全部379則原公告、262筆財務新聞／股票關聯，不把新聞關聯當公司來源證明，不用賽後數字。','','## 必須修正先前的說法','','南亞科DB有2025-10-13 16:32:51的法說摘要（ID 9d4f234378a373ac94b74ced64d48e4bcaf299f3），明列Q3自結收入與損益，與[公司原公告](https://www.nanya.com/en/IR/16?IRId=10046)一致。先前「DB只保留Q2」指原財務抓取／解析結果，不能概括整個DB。此次確認是財務標題關鍵字漏掉法說摘要；此時間是保留MOPS公告可用時點，不冒稱最早公開時刻。','','## 查核結果','','- 25檔全數完成保留原公告的分類與時間核對；24檔有正式財務摘要／季度自結／法說業績摘要。旺宏另有Q2與單月數字的注意交易揭露，但未在保留資料找到完整正式財報摘要。','- 3檔原有財務資料被解析器漏掉：緯穎中文年份、貿聯新台幣與仟元字樣、創意公告標題。已從報導起訖日期與明列金額核對暫存解析，192筆原可解析金額保持一致。未改scores_v4或權重。',f'- {attention_stocks}檔有較新發布的注意交易損益，共{attention_count}則；較新發布不一定較新經濟期間。单月、自結、核閱季度與四季累計分列，不能湊成新Q3或把單月EPS乘3。',f'- {earnings_issuer_count}檔有開賽前已公告、落在競賽期間的財報董事會或季報法說，合計{len(earnings_events)}則預告；預定日期不代表已取得實際數字。投資論壇另列。','- 公司IR及首發／更正歷史沒有全覆蓋認證。保留原文查無新結果，只能說本資料範圍未見；不能保證當時沒有其他新公告。','','## 全部25檔','','|排序|股票|原模型季度／狀態|保留原文最新實際摘要|較新發布的單月／注意損益|財務更正|查核結論|','|---:|---|---|---|---|---|---|']
kind_label={'published_formal_financial_summary':'正式摘要','published_quarter_self_report':'季度自結','published_conference_summary':'法說業績摘要'}
for r in a25_freshness_audit_v1:
    latest=r['latest_actual_in_retained_originals'];old=r['original_model']
    before='Q'+str(old['quarter']) if old.get('quarter') else old['status']
    after='未見完整實際摘要' if latest is None else latest['stamp'][:10]+'／'+kind_label.get(latest['kind'],latest['kind'])+'／'+','.join(latest['periods'])
    monthly='；'.join(c['stamp'][:10]+'：'+','.join(c['periods']) for c in r['newer_attention_disclosures']) or '保留資料未見'
    correction='；'.join(c['subject'] for c in r['financial_corrections']) or '保留資料未見'
    conclusion='已有來源，IR完整度未證明'
    if r['stock_id']=='2408':conclusion='原查詢漏法說Q3，自結已公開'
    elif r['staged_parser_repair']:conclusion='原文存在，解析修正已暫存'
    elif r['financial_corrections']:conclusion='更正範圍需核對，不能把未知當利空'
    elif latest is None:conclusion='有部分損益；完整季度來源仍不足'
    lines.append(f"|{r['momentum_rank']}|{r['stock_id']} {r['name_snapshot']}|{before}|{after}|{monthly}|{correction}|{conclusion}|")
lines.extend(['','公司名稱是DB目前快照便於辨識，歷史身份以股票代號為主。「保留資料未見」不是證實不存在。上表列出的是文件與期間，不能直接轉成加碼強弱。','','## 三個暫存解析修正','','|股票|Q2本業利益率相對Q1變化pp|修正原因|原始公告來源IDs|','|---|---:|---|---|'])
reasons={'6669':'從114/01/01～114/06/30確認年季，避免中文一一四被漏掉','3665':'接受明列新台幣／仟元格式，保持數值與單位','3443':'從實際報導起訖及財務欄位辨識，不強依賴標題用詞'}
for r in parser_repaired_a25_v1:lines.append(f"|{r['stock_id']}|{r['margin_change_pp']:+.4f}|{reasons[r['stock_id']]}|{'、'.join(r['source_ids'])}|")
lines.extend(['','只核對這三檔資料擷取的正確性，沒有把它們單獨插回全池純財務選股，沒有重跑或更新v4績效。要比較改善效果，必須相同修正套在全可選池／決策日期，且先處理更正與較新自結。','','## 開賽前已知的賽期事件','','|股票|預定日期|事件|何時已公告|來源ID|','|---|---|---|---|---|'])
event_labels={'earnings_board':'財報董事會預定','earnings_meeting':'季度財報法說預定','investor_meeting':'投資人會議／論壇預定'}
for e in sorted(earnings_events,key=lambda e:(e['event_date'],e['stock_id'],e['kind'])):lines.append(f"|{e['stock_id']} {e['name']}|{e['event_date']}|{event_labels[e['kind']]}|{e['known_at']}|{e['ann_id']}|")
lines.extend(['','投資論壇預告也存JSON，與財報公告事件分列。數字實際何時公開、是否改期、公告是否超預期仍需逐事件核對，不能把11月的實際成果放到10月選股。','','## 特別需要保留的區別','','- 華邦電原注意交易公告有8／9月單月獲利，與Q2虧損不同期間；不把它稱完整Q3。','- 群聯9月單月歸母淨利843百萬元、EPS4.06，與前輪看到Q2淨利下降是不同時間範圍，原案例判讀需納入此更新；仍不能從單月數字推算Q3全部或保證股價續漲。','- 旺矽更正明列財報附註員工福利／折舊與攤銷，不能一看到「更正」就當整份財報利空；是否影響本業訊號須核對修訂表。','- 力積電更正的是注意交易資料單位；錯誤版本不能和更正版本混用。','- 勤誠有一則公司債資料公告，已剔除損益更新統計；奇鋐與環球晶部分新公告重列6月數字，發布日較新不代表近期營運改善。','','## 保存與下一個實作入口','','[逐檔結果與所有預定事件JSON](A策略_2025開賽25檔資訊新鮮度結果_2026-10-14.json)、[原公告及新聞線索快取](A策略_2025開賽25檔最新來源查核_2026-10-14.json)，原Notebook cells137–144與本節輸出。','下一個實作先統一修正資料擷取與文件分類，把原值、期間、已知可用時間、更正範圍、是否自結保存清楚，再定義公告加減碼的有限候選。全池純財務回測與A25資訊調權的資料範圍不同，不能混稱同一實驗。沒有新LLM倍率、沒有改25檔主線、沒有交易提交。'])
report_path_a25_v1.write_text('\n'.join(lines)+'\n',encoding='utf-8')
assert len(a25_audit_result_v1['audit'])==25 and len({r['stock_id'] for r in a25_audit_result_v1['audit']})==25
assert all(datetime.datetime.fromisoformat(c['known_at'])<case_cutoff_v1 for c in a25_declared_events_v1)
print('Complete A25 audit saved:',audit_summary_v1)
print('Report:',report_path_a25_v1.name)
print('A25_LATEST_SOURCE_AUDIT_COMPLETE_NO_MODEL_CHANGE')


In [ ]:
import sys, json, inspect
probe_names=['raw_data','dates','stock_ids','full_pool_financial_announcements_v1','a25_all_announcements_v1','a25_formal_summary_extractor_v1','scores_v4','good_exec_v2','engine_v4_results']
print('Environment',sys.executable)
for name in probe_names:
    obj=globals().get(name)
    print(name,type(obj).__name__,getattr(obj,'shape',None),len(obj) if hasattr(obj,'__len__') else None)
print('Recent history execution numbers',get_ipython().history_manager.get_tail(3,include_latest=False,output=False).__class__.__name__)
print('Engine functions',[n for n in globals() if callable(globals()[n]) and any(x in n for x in ['engine','simulate','extractor']) and not n.startswith('_')])


In [ ]:
# Six-stage continuation: recover cached source state, inspect execution timing before reruns
import sys,json,re,ast,datetime,unicodedata,gzip,hashlib,inspect
import numpy as np, pymysql
from research_layout import ResearchPath as Path
from collections import Counter,defaultdict
print('Kernel environment',sys.executable)
assert sys.executable.lower()==r'C:\Python314\python.exe'.lower()
recovery_root=Path(r'C:\Users\weiha\Desktop\文件\03_課程資料\115-1\資料探勘與應用\B_Final Project (0920~1221)\競賽資料')
current_nb=json.loads((recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').read_text(encoding='utf-8'))
recovery_base_nb=json.loads((recovery_root/'動能基準研究_24日回測_2026-10-13.ipynb').read_text(encoding='utf-8'))
recovery_weight_nb=json.loads((recovery_root/'動能基準研究_25檔權重比較_2026-10-14.ipynb').read_text(encoding='utf-8'))
def cell_source(nb,i):return ''.join(nb['cells'][i]['source'])
for source_nb in [current_nb,recovery_base_nb,recovery_weight_nb]:
    for cell in source_nb['cells']:
        text=''.join(cell.get('source',[]))
        if not any('def '+name+'(' in text for name in ['prestart_stamp','prestart_quarter','pure_financial_amounts_v1','a25_formal_summary_extractor_v1','fetch_full_pool_announcements_v1']):continue
        try:tree=ast.parse(text)
        except SyntaxError:continue
        for part in tree.body:
            if isinstance(part,ast.FunctionDef) and part.name in ['prestart_stamp','prestart_quarter','pure_financial_amounts_v1','a25_formal_summary_extractor_v1','fetch_full_pool_announcements_v1']:
                exec(compile(ast.Module(body=[part],type_ignores=[]),'<restore-source-functions>','exec'),globals())
cached=json.loads(gzip.decompress((recovery_root/'策略研究_原資料快照_v3_2026-10-14.json.gz').read_bytes()).decode('utf-8'))
print('Snapshot schema',list(cached) if isinstance(cached,dict) else type(cached).__name__)
latest_a25_cache=json.loads((recovery_root/'A策略_2025開賽25檔最新來源查核_2026-10-14.json').read_text(encoding='utf-8'))
print('Cached original A25 notices',len(latest_a25_cache['announcements']))
engine_source_review_v5=[]
for name,book in [('base',recovery_base_nb),('weight',recovery_weight_nb),('announcement',current_nb)]:
    for i,cell in enumerate(book['cells']):
        text=''.join(cell.get('source',[]))
        for line in text.splitlines():
            if (name=='base' and i==10 or name=='weight' and i==3) and any(s in line for s in ['good_exec[t]','previous_nav','receivable','cash_div','stock_div','target[selected]','eligible=']):
                engine_source_review_v5.append({'notebook':name,'cell':i,'line':line.strip()})
print('Execution source audit',json.dumps(engine_source_review_v5,ensure_ascii=False))
print('RECOVERED_CACHED_SOURCE_AND_ENGINE_AUDIT')


In [ ]:
# Uniform source v5: all 150 stocks, retained announcements without financial-title filter
raw_data=cached
stock_ids=[str(r[0]) for r in raw_data['universe']]
dates=sorted({datetime.date.fromisoformat(r[1]) for r in raw_data['price']})
si={s:i for i,s in enumerate(stock_ids)};di={d:i for i,d in enumerate(dates)}
tree=ast.parse(inspect.getsource(fetch_full_pool_announcements_v1)) if False else ast.parse(cell_source(current_nb,57))
original_fn=next(n for n in tree.body if isinstance(n,ast.FunctionDef))
original_fn.name='fetch_uniform_announcements_v5'
old_sql='SELECT a.stock_id,a.ann_id,a.announce_date,a.announce_time,a.subject,a.event_date,a.content,a.source FROM announcements a JOIN universe u ON a.stock_id=u.stock_id AND u.active=1 WHERE a.announce_date BETWEEN %s AND %s AND (a.subject LIKE %s OR a.subject LIKE %s OR a.subject LIKE %s) ORDER BY a.stock_id,a.announce_date,a.announce_time'
new_sql='SELECT a.stock_id,a.ann_id,a.announce_date,a.announce_time,a.subject,a.event_date,a.content,a.source FROM announcements a JOIN universe u ON a.stock_id=u.stock_id AND u.active=1 WHERE a.announce_date BETWEEN %s AND %s ORDER BY a.stock_id,a.announce_date,a.announce_time'
for node in ast.walk(original_fn):
    if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute) and node.func.attr=='execute' and node.args and isinstance(node.args[0],ast.Constant) and node.args[0].value==old_sql:
        node.args[0]=ast.Constant(new_sql)
        node.args[1]=ast.parse("(f'{dates[0].year}-01-01',str(dates[-1]))",mode='eval').body
ast.fix_missing_locations(original_fn)
exec(compile(ast.Module(body=[original_fn],type_ignores=[]),'<readonly-uniform-source>','exec'),globals())
uniform_source_cache_v5=recovery_root/'A策略_全池公告原文快取_v5_2026-10-14.json.gz'
def restore_notice_v5(a):
    a=dict(a)
    for k in ['announce_date','event_date']:
        if isinstance(a.get(k),str) and re.fullmatch(r'\d{4}-\d{2}-\d{2}',a[k]):a[k]=datetime.date.fromisoformat(a[k])
    if isinstance(a.get('announce_time'),str):
        m=re.fullmatch(r'(\d+):(\d+):(\d+)(?:\.\d+)?',a['announce_time'])
        if m:a['announce_time']=datetime.timedelta(hours=int(m[1]),minutes=int(m[2]),seconds=int(m[3]))
    return a
if uniform_source_cache_v5.exists():
    notice_cache_v5=json.loads(gzip.decompress(uniform_source_cache_v5.read_bytes()).decode('utf-8'))
    full_pool_announcements_v5=[restore_notice_v5(a) for a in notice_cache_v5['announcements']]
else:
    full_pool_announcements_v5,uniform_schema_v5=fetch_uniform_announcements_v5()
    notice_cache_v5={'scope':'active official150 retained DB originals, no subject filter; known availability, not certified first public time','from':f'{dates[0].year}-01-01','to':str(dates[-1]),'announcements':full_pool_announcements_v5}
    uniform_source_cache_v5.write_bytes(gzip.compress(json.dumps(notice_cache_v5,ensure_ascii=False,default=str).encode('utf-8')))
full_pool_ann_by_stock_v5=defaultdict(list)
formal_records_v5=[];old_formal_parse_count_v5=0;preserved_old_v5=0
for ann in full_pool_announcements_v5:
    full_pool_ann_by_stock_v5[ann['stock_id']].append(ann)
    old=pure_financial_amounts_v1(ann)
    old_formal_parse_count_v5+=bool(old)
    new=a25_formal_summary_extractor_v1(ann)
    if new:
        if old:
            assert all(np.isclose(old[k],new['amounts'][k],atol=.001,rtol=0) for k in ['revenue','operating']);preserved_old_v5+=1
        formal_records_v5.append(dict(new,stock_id=ann['stock_id']))
print('Uniform cache coverage',{'stocks':len(full_pool_ann_by_stock_v5),'announcements':len(full_pool_announcements_v5),'formal_records':len(formal_records_v5),'old_numeric_parses':old_formal_parse_count_v5,'overlap_values_unchanged':preserved_old_v5,'years':dict(Counter(r['year'] for r in formal_records_v5))})
print('Original examples recovered',[(a['stock_id'],str(prestart_stamp(a)),a['subject']) for a in full_pool_announcements_v5 if a['ann_id']=='9d4f234378a373ac94b74ced64d48e4bcaf299f3'])
print('FULL_UNIVERSE_SOURCE_V5_SAVED_NO_SCORE_CHANGE')


In [ ]:
# v5 source coverage: period-aware numerical eligibility, conservative newer-source gates
for key in ['price','institution','dividend']:
    raw_data[key]=[list(row) for row in raw_data[key]]
    for row in raw_data[key]:
        if isinstance(row[1],str):row[1]=datetime.date.fromisoformat(row[1])
exec(cell_source(recovery_base_nb,5))
exec(cell_source(recovery_base_nb,8))
v4_saved=json.loads((recovery_root/'策略研究_逐窗口結果_v4_2026-10-14.json').read_text(encoding='utf-8'))
signal_returns_v5=signal_returns.copy();bad_action_v5=bad_action.copy()
for a in v4_saved['action_corrections']:
    t,j=di[datetime.date.fromisoformat(a['ex_date'])],si[a['stock_id']]
    bad_action_v5[t,j]=((close[t,j]*(1+a['stock_ratio'])+a['cash'])/close[t-1,j]-1<-.25 or abs(a['expected']-a['reference'])>max(.03,close[t-1,j]*.0002))
    signal_returns_v5[t,j]=close[t,j]/a['reference']-1
scores_a_v5=np.full(shape,np.nan)
for q in range(60,len(dates)):
    rr=signal_returns_v5[q-59:q+1];ff=flow[q-19:q+1];vv=volume[q-19:q+1]
    eligible=np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(ff)&np.isfinite(vv)&(vv>0),axis=0)&(close[q]>0)&~np.any(bad_action_v5[q-59:q+1],axis=0)
    scores_a_v5[q]=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,eligible)+.5*percentile(np.prod(1+rr,axis=0)-1,eligible)
formal_by_stock_v5=defaultdict(list)
for r in formal_records_v5:formal_by_stock_v5[r['stock_id']].append(r)
def source_class_v5(a):
    s=unicodedata.normalize('NFKC',a.get('subject') or '')
    if re.search(r'更正|補充',s) and re.search(r'財務|財報|損益|自結',s):return 'financial_correction_requires_scope_review'
    if a25_formal_summary_extractor_v1(a):return 'formal_period_amounts'
    if re.search(r'預計|預定|召開|舉行|召集',s) and re.search(r'財務報|財報|法說|法人說明|董事會',s):return 'scheduled_not_actual_result'
    if re.search(r'自結|營運績效|營運結果|法人說明會摘要|財務業務|相關財務|損益情形|財務報告|財務報表',s):return 'actual_or_attention_financial_unparsed'
    return 'other_notice'
notice_class_v5={a['ann_id']:source_class_v5(a) for a in full_pool_announcements_v5}
def financial_asof_v5(sid,cutoff):
    quarters={}
    for r in formal_by_stock_v5[sid]:
        if r['year']==cutoff.year and datetime.datetime.fromisoformat(r['known_at'])<cutoff:
            q=r['quarter']
            if q not in quarters or r['known_at']>quarters[q]['known_at']:quarters[q]=r
    usable=[q for q in [2,3] if all(x in quarters for x in range(1,q+1))]
    if not usable:return {'status':'unknown_period_or_q1_basis','source_ids':[]}
    q=max(usable);latest=datetime.datetime.fromisoformat(quarters[q]['known_at'])
    unresolved=[a['ann_id'] for a in full_pool_ann_by_stock_v5[sid] if prestart_stamp(a) and latest<prestart_stamp(a)<cutoff and notice_class_v5[a['ann_id']] in ['financial_correction_requires_scope_review','actual_or_attention_financial_unparsed']]
    if unresolved:return {'status':'newer_source_requires_period_or_correction_review','quarter':q,'source_ids':[quarters[x]['ann_id'] for x in range(1,q+1)],'pending_ids':unresolved}
    def single(k):return {f:quarters[k]['amounts'][f]-(quarters[k-1]['amounts'][f] if k>1 else 0) for f in ['revenue','operating']}
    p,c=single(q-1),single(q)
    if p['revenue']<=0 or c['revenue']<=0:return {'status':'invalid_single_quarter_revenue','source_ids':[]}
    return {'status':'verified_formal_period','quarter':q,'delta_pp':100*(c['operating']/c['revenue']-p['operating']/p['revenue']),'source_ids':[quarters[x]['ann_id'] for x in range(1,q+1)],'known_at':[quarters[x]['known_at'] for x in range(1,q+1)]}
coverage_a_decisions_v5=[];coverage_pool_v5=[];evidence_v5={}
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));rank=scores_a_v5[t-1]
    selected=np.flatnonzero(np.isfinite(rank));selected=selected[np.argsort(-rank[selected],kind='stable')[:25]]
    status=Counter();a_status=Counter()
    for sid in stock_ids:
        r=financial_asof_v5(sid,cutoff);evidence_v5[(t,sid)]=r;status[r['status']]+=1
    for j in selected:a_status[evidence_v5[(t,stock_ids[j])]['status']]+=1
    coverage_pool_v5.append({'date':str(dates[t]),'status':dict(status)})
    coverage_a_decisions_v5.append({'date':str(dates[t]),'stocks':[stock_ids[j] for j in selected],'status':dict(a_status)})
old_only_examples_v5=[]
for a in full_pool_announcements_v5:
    if pure_financial_amounts_v1(a) and not a25_formal_summary_extractor_v1(a):
        body=unicodedata.normalize('NFKC',a.get('content') or '')
        old_only_examples_v5.append({'stock_id':a['stock_id'],'ann_id':a['ann_id'],'subject':a['subject'],'period_lines':[line for line in body.splitlines() if '起訖' in line][:2],'known_at':str(prestart_stamp(a))})
stage1_summary_v5={'all_stocks':len(stock_ids),'original_notices':len(full_pool_announcements_v5),'formal_records':len(formal_records_v5),'overlap_amounts_unchanged':preserved_old_v5,'old_amounts_without_new_period_confirmation':len(old_only_examples_v5),'classes':dict(Counter(notice_class_v5.values())),'decisions':len(coverage_pool_v5),'A25_decision_status':dict(sum((Counter(x['status']) for x in coverage_a_decisions_v5),Counter())),'first_publication_certified':False,'scores_or_weights_changed':False,'financial_gate':'strict conservative newer actual/correction review; does not imply every attention item is bad news'}
out={'summary':stage1_summary_v5,'all_pool_daily_coverage':coverage_pool_v5,'A25_daily_coverage':coverage_a_decisions_v5,'old_only_review':old_only_examples_v5,'execution_audit':{'selection_uses_previous_day_signal':True,'orders_use_tradable_nav_excluding_receivables':True,'current_day_price_used_for_settlement_not_selection':True,'actual_dividend_payment_dates_modelled':False,'official_batch_void_semantics_certified':False}}
(recovery_root/'A策略_來源與時間統一查核_v5_2026-10-14.json').write_text(json.dumps(out,ensure_ascii=False,indent=2),encoding='utf-8')
print('Stage1 coverage summary',json.dumps(stage1_summary_v5,ensure_ascii=False))
print('Seasonal A25',[r for r in coverage_a_decisions_v5 if r['date'] in ['2024-10-29','2025-10-28']])
print('Old-only period restrictions examples',old_only_examples_v5[:8])
print('UNIFORM_COVERAGE_V5_SAVED; pending source period and correction review retained')


In [ ]:
# Uniform extractor refinement: explicit parenthesized period; delegated subsidiary excluded
# Source-derived extractor staged for audit only; no change to scores_v4 or portfolio weights
def formal_summary_extractor_v5(ann):
    subject=unicodedata.normalize('NFKC',ann.get('subject') or '')
    if re.search(r'^代|公告代|代重要子公司|代子公司',subject):return None
    body=unicodedata.normalize('NFKC',ann.get('content') or '').replace('\r','')
    period=re.search(r'起訖日期[^:]*:\s*\(?\s*(\d{3,4})/(\d{1,2})/(\d{1,2})\s*[-~至]\s*(\d{3,4})/(\d{1,2})/(\d{1,2})',body)
    if not period:return None
    y1,m1,d1,y2,m2,d2=map(int,period.groups());y1=y1+1911 if y1<1911 else y1;y2=y2+1911 if y2<1911 else y2
    if y1!=y2 or (m1,d1)!=(1,1) or (m2,d2) not in [(3,31),(6,30),(9,30),(12,31)]:return None
    cutoff=prestart_stamp(ann)
    if cutoff is None or cutoff.date()<datetime.date(y2,m2,d2):return None
    vals={}
    for field,number,label in [('revenue',4,'營業收入'),('operating',6,'營業利益')]:
        line=re.search(r'(?m)^\s*'+str(number)+r'[.、]\s*([^\n]+)',body)
        if not line or label not in line[1] or '累計' not in line[1] or '(仟元)' not in line[1] or ':' not in line[1]:return None
        value=line[1].rsplit(':',1)[1].strip()
        match=re.fullmatch(r'(?:新台幣\s*)?(\(?[+-]?\d[\d,]*(?:\.\d+)?\)?)(?:\s*仟元)?',value)
        if not match:return None
        numeric=match[1].replace(',','');vals[field]=float(numeric.strip('()'))*(-1 if numeric.startswith('(') else 1)
    if vals['revenue']<=0:return None
    return {'year':y2,'quarter':m2//3,'period_start':str(datetime.date(y1,m1,d1)),'period_end':str(datetime.date(y2,m2,d2)),'unit':'thousand; explicit source label, numeric TWD prefix accepted','amounts':vals,'ann_id':ann['ann_id'],'known_at':str(cutoff),'not_applied_to_v4_scores':True}

formal_records_v5=[];preserved_old_v5=0
for a in full_pool_announcements_v5:
    r=formal_summary_extractor_v5(a)
    if r:
        old=pure_financial_amounts_v1(a)
        if old:assert all(np.isclose(old[k],r['amounts'][k],rtol=0,atol=.001) for k in ['revenue','operating']);preserved_old_v5+=1
        formal_records_v5.append(dict(r,stock_id=a['stock_id']))
old_only_remaining_v5=[a for a in full_pool_announcements_v5 if pure_financial_amounts_v1(a) and not formal_summary_extractor_v5(a)]
print('Refined period coverage',len(formal_records_v5),'old overlap unchanged',preserved_old_v5,'old-only unresolved',len(old_only_remaining_v5))
print('Remaining format checks',[{'stock_id':a['stock_id'],'known_at':str(prestart_stamp(a)),'subject':a['subject'],'period':[l for l in unicodedata.normalize('NFKC',a.get('content') or '').splitlines() if '起訖' in l][:1]} for a in old_only_remaining_v5[:15]])
# Keep v1 intact; subsequent source classifier uses uniformly refined extractor.
a25_formal_summary_extractor_v1=formal_summary_extractor_v5
print('FORMAL_EXTRACTOR_V5_VALIDATED; missing publication times remain excluded')


In [ ]:
# Recompute only affected source coverage after period-wrapper refinement
formal_by_stock_v5=defaultdict(list)
for r in formal_records_v5:formal_by_stock_v5[r['stock_id']].append(r)
def source_class_v5(a):
    s=unicodedata.normalize('NFKC',a.get('subject') or '')
    if re.search(r'更正|補充',s) and re.search(r'財務|財報|損益|自結',s):return 'financial_correction_requires_scope_review'
    if a25_formal_summary_extractor_v1(a):return 'formal_period_amounts'
    if re.search(r'預計|預定|召開|舉行|召集',s) and re.search(r'財務報|財報|法說|法人說明|董事會',s):return 'scheduled_not_actual_result'
    if re.search(r'自結|營運績效|營運結果|法人說明會摘要|財務業務|相關財務|損益情形|財務報告|財務報表',s):return 'actual_or_attention_financial_unparsed'
    return 'other_notice'
notice_class_v5={a['ann_id']:source_class_v5(a) for a in full_pool_announcements_v5}
def financial_asof_v5(sid,cutoff):
    quarters={}
    for r in formal_by_stock_v5[sid]:
        if r['year']==cutoff.year and datetime.datetime.fromisoformat(r['known_at'])<cutoff:
            q=r['quarter']
            if q not in quarters or r['known_at']>quarters[q]['known_at']:quarters[q]=r
    usable=[q for q in [2,3] if all(x in quarters for x in range(1,q+1))]
    if not usable:return {'status':'unknown_period_or_q1_basis','source_ids':[]}
    q=max(usable);latest=datetime.datetime.fromisoformat(quarters[q]['known_at'])
    unresolved=[a['ann_id'] for a in full_pool_ann_by_stock_v5[sid] if prestart_stamp(a) and latest<prestart_stamp(a)<cutoff and notice_class_v5[a['ann_id']] in ['financial_correction_requires_scope_review','actual_or_attention_financial_unparsed']]
    if unresolved:return {'status':'newer_source_requires_period_or_correction_review','quarter':q,'source_ids':[quarters[x]['ann_id'] for x in range(1,q+1)],'pending_ids':unresolved}
    def single(k):return {f:quarters[k]['amounts'][f]-(quarters[k-1]['amounts'][f] if k>1 else 0) for f in ['revenue','operating']}
    p,c=single(q-1),single(q)
    if p['revenue']<=0 or c['revenue']<=0:return {'status':'invalid_single_quarter_revenue','source_ids':[]}
    return {'status':'verified_formal_period','quarter':q,'delta_pp':100*(c['operating']/c['revenue']-p['operating']/p['revenue']),'source_ids':[quarters[x]['ann_id'] for x in range(1,q+1)],'known_at':[quarters[x]['known_at'] for x in range(1,q+1)]}
coverage_a_decisions_v5=[];coverage_pool_v5=[];evidence_v5={}
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));rank=scores_a_v5[t-1]
    selected=np.flatnonzero(np.isfinite(rank));selected=selected[np.argsort(-rank[selected],kind='stable')[:25]]
    status=Counter();a_status=Counter()
    for sid in stock_ids:
        r=financial_asof_v5(sid,cutoff);evidence_v5[(t,sid)]=r;status[r['status']]+=1
    for j in selected:a_status[evidence_v5[(t,stock_ids[j])]['status']]+=1
    coverage_pool_v5.append({'date':str(dates[t]),'status':dict(status)})
    coverage_a_decisions_v5.append({'date':str(dates[t]),'stocks':[stock_ids[j] for j in selected],'status':dict(a_status)})
old_only_examples_v5=[]
for a in full_pool_announcements_v5:
    if pure_financial_amounts_v1(a) and not a25_formal_summary_extractor_v1(a):
        body=unicodedata.normalize('NFKC',a.get('content') or '')
        old_only_examples_v5.append({'stock_id':a['stock_id'],'ann_id':a['ann_id'],'subject':a['subject'],'period_lines':[line for line in body.splitlines() if '起訖' in line][:2],'known_at':str(prestart_stamp(a))})
stage1_summary_v5={'all_stocks':len(stock_ids),'original_notices':len(full_pool_announcements_v5),'formal_records':len(formal_records_v5),'overlap_amounts_unchanged':preserved_old_v5,'old_amounts_without_new_period_confirmation':len(old_only_examples_v5),'classes':dict(Counter(notice_class_v5.values())),'decisions':len(coverage_pool_v5),'A25_decision_status':dict(sum((Counter(x['status']) for x in coverage_a_decisions_v5),Counter())),'first_publication_certified':False,'scores_or_weights_changed':False,'financial_gate':'strict conservative newer actual/correction review; does not imply every attention item is bad news'}
out={'summary':stage1_summary_v5,'all_pool_daily_coverage':coverage_pool_v5,'A25_daily_coverage':coverage_a_decisions_v5,'old_only_review':old_only_examples_v5,'execution_audit':{'selection_uses_previous_day_signal':True,'orders_use_tradable_nav_excluding_receivables':True,'current_day_price_used_for_settlement_not_selection':True,'actual_dividend_payment_dates_modelled':False,'official_batch_void_semantics_certified':False}}
(recovery_root/'A策略_來源與時間統一查核_v5_2026-10-14.json').write_text(json.dumps(out,ensure_ascii=False,indent=2),encoding='utf-8')
print('Stage1 coverage summary',json.dumps(stage1_summary_v5,ensure_ascii=False))
print('Seasonal A25',[r for r in coverage_a_decisions_v5 if r['date'] in ['2024-10-29','2025-10-28']])
print('Old-only period restrictions examples',old_only_examples_v5[:8])
print('UNIFORM_COVERAGE_V5_SAVED; pending source period and correction review retained')


In [ ]:
# Freeze narrow numerical prestart experiment before inspecting its returns
stage2_definition_v5={'experiment_key':'A_formal_operating_prestart_v5','stock_selection':'same A previous-session 20/60 momentum,25 stocks,every5 sessions','signal':'latest verified cumulative financial period at window start08:55, derive single Q2/Q1 or Q3/Q2 operating margins','threshold_pp':1.0,'multiplier':'1+0.25*sign; positive if delta>=1,negative if<=-1,otherwise0','unknown':'multiplier1; not bearish; record coverage separately','newer_unparsed_or_correction':'multiplier1 pending period/scope verification','freeze':'all150 numerical evidence fixed at window start; subsequent selected stocks use that prestart evidence, no new announcement update','investment':.9,'horizon':24,'rebalance':5,'lot':1000,'fee':.001425,'sell_tax':.003,'engine':'v4 exact corporate action overlay,tradable nav orders,AUM return,unpaid receivables not tradeable','controls':['A_equal_v4','A_mild_v4','A_mild_tolerance_v4'],'family':'new standalone candidate and prior14 nonA; at least15 comparisons, not isolated unadjusted significance','same_season':'only2024-10-29 and2025-10-28; insufficient for stable seasonal inference','scope_limit':'formal operating-only subexperiment; revenue/self/conference numeric and LLM remain pending, not all-announcement model','no_tuning':True}
stage2_definition_hash_v5=hashlib.sha256(json.dumps(stage2_definition_v5,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
(recovery_root/'A策略_開賽前資訊調權實驗定義_v5_2026-10-14.json').write_text(json.dumps({'definition':stage2_definition_v5,'sha256':stage2_definition_hash_v5},ensure_ascii=False,indent=2),encoding='utf-8')
print('Definition frozen before new performance',stage2_definition_hash_v5)
INITIAL=1e9;COUNT=25;INVESTED=.9;HORIZON=24;REBALANCE=5;LOT=1000;FEE=.001425;TAX=.003
cash_div_v5=cash_div.copy();stock_div_v5=stock_div.copy()
for a in v4_saved['action_corrections']:
    t,j=di[datetime.date.fromisoformat(a['ex_date'])],si[a['stock_id']]
    cash_div_v5[t,j]=a['cash'];stock_div_v5[t,j]=a['stock_ratio']
good_exec_v5=np.isfinite(execution)&(execution>0)&np.isfinite(volume)&(volume>0)&np.isfinite(turnover)&(turnover>0)&np.isfinite(close)&(close>0)
base_tree=ast.parse(cell_source(recovery_base_nb,10))
settle_node=next(n for n in base_tree.body if isinstance(n,ast.FunctionDef) and n.name=='settle')
exec(compile(ast.Module(body=[settle_node],type_ignores=[]),'<exact-settlement>','exec'),globals())
base_node=next(n for n in base_tree.body if isinstance(n,ast.FunctionDef) and n.name=='simulate')
base_src=ast.get_source_segment(cell_source(recovery_base_nb,10),base_node)
def numerical_allocation_v5(start,t,selected,mode):
    if mode=='equal':return np.full(COUNT,INVESTED/COUNT)
    if mode=='mild':return INVESTED*np.linspace(1.25,.75,COUNT)/COUNT
    assert mode=='formal_prestart'
    multipliers=[]
    for j in selected:
        r=evidence_v5[(start,stock_ids[j])];sign=0
        if r['status']=='verified_formal_period':sign=1 if r['delta_pp']>=1 else -1 if r['delta_pp']<=-1 else 0
        multipliers.append(1+.25*sign)
    w=INVESTED*np.array(multipliers)/sum(multipliers)
    assert len(w)==25 and np.isclose(w.sum(),.9) and w.max()<.1
    return w
src=base_src.replace('def simulate(start,strategy,horizon=HORIZON):','def simulate_numeric_prestart_v5(start,mode,horizon=HORIZON):\n    strategy="A"')
for old,new in [('bad_action[t]','bad_action_v5[t]'),('cash_div[t]','cash_div_v5[t]'),('stock_div[t]','stock_div_v5[t]'),('scores[strategy][t-1]','scores_a_v5[t-1]'),('good_exec[t]','good_exec_v5[t]')]:src=src.replace(old,new)
old='target[selected]=(np.floor((INVESTED/COUNT)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)'
assert src.count(old)==1
src=src.replace(old,'target[selected]=(np.floor(numerical_allocation_v5(start,t,selected,mode)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)')
exec(src,globals())
window_starts=np.arange(61,len(dates)-HORIZON+1)
numeric_prestart_results_v5={}
for mode,key in [('equal','A_equal'),('mild','A_mild')]:
    # Changed source inputs cannot alter the controls: verify exact v4 reproduction.
    control=[simulate_numeric_prestart_v5(int(s),mode) for s in window_starts]
    old_rows=v4_saved['windows'][key]
    for old,new in zip(old_rows,control):
        assert old['invalid']==new['invalid'] and old['violations']==new['violations']
        for f in ['return','mdd','cost','turnover']:assert np.isclose(old[f],new[f],rtol=1e-12,atol=1e-12),(key,new['start'],f)
    print('Exact control reproduction',key,len(control))
numeric_prestart_results_v5['A_formal_operating_prestart']=[simulate_numeric_prestart_v5(int(s),'formal_prestart') for s in window_starts]
def clean_v5(obj):
    if isinstance(obj,np.ndarray):return [clean_v5(x) for x in obj]
    if isinstance(obj,(np.integer,)):return int(obj)
    if isinstance(obj,(np.floating,float)):return float(obj) if np.isfinite(obj) else None
    if isinstance(obj,dict):return {k:clean_v5(v) for k,v in obj.items()}
    if isinstance(obj,(list,tuple)):return [clean_v5(v) for v in obj]
    return obj
numerical_export_v5={'definition':stage2_definition_v5,'definition_sha256':stage2_definition_hash_v5,'raw_snapshot_sha256':v4_saved['raw_snapshot_sha256'],'company_action_overlay_sha256':v4_saved['company_action_overlay_sha256'],'results':clean_v5(numeric_prestart_results_v5),'status':'saved_inference_pending'}
(recovery_root/'A策略_開賽前財務調權逐窗口_v5_2026-10-14.json').write_text(json.dumps(numerical_export_v5,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('Numerical results saved',{'windows':len(window_starts),'valid':sum(not r['invalid'] and r['violations']==0 for r in numeric_prestart_results_v5['A_formal_operating_prestart'])})
print('NUMERICAL_PRESTART_V5_SAVED_INFERENCE_PENDING')


In [ ]:
# Fixed numerical subexperiment inference: paired after-fee paths, dependent windows, conservative family
new_rows=numeric_prestart_results_v5['A_formal_operating_prestart'];base_rows=v4_saved['windows']['A_equal']
valid=np.array([not n['invalid'] and n['violations']==0 and not b['invalid'] and b['violations']==0 for n,b in zip(new_rows,base_rows)])
new_return=np.array([n['return'] for n in new_rows]);base_return=np.array([b['return'] for b in base_rows])
delta=np.where(valid,new_return-base_return,np.nan)
def paired_block_interval_v5(diff,length,seed,family=21,reps=5000):
    rng=np.random.default_rng(seed);n=len(diff);means=np.full(reps,np.nan);blocks=int(np.ceil(n/length))
    for i in range(reps):
        starts=rng.integers(0,n-length+1,size=blocks)
        ix=(starts[:,None]+np.arange(length)).ravel()[:n]
        finite=diff[ix][np.isfinite(diff[ix])]
        if len(finite):means[i]=finite.mean()*100
    alpha=.05/family
    return np.nanquantile(means,[alpha/2,1-alpha/2]).tolist()
numeric_stats_v5={'paired_windows':int(valid.sum()),'mean_difference_pp':float(np.mean(delta[valid])*100),'candidate_median_pct':float(np.median(new_return[valid])*100),'A_median_same_dates_pct':float(np.median(base_return[valid])*100),'beat_A_pct':float(np.mean(delta[valid]>0)*100),'candidate_cost_pct':float(np.mean([new_rows[i]['cost'] for i in np.flatnonzero(valid)])*100),'A_cost_pct':float(np.mean([base_rows[i]['cost'] for i in np.flatnonzero(valid)])*100),'family':21,'family_reason':'14 prior broad nonA candidates plus6 seasonal prototypes plus1 new candidate; conservative count, not joint proof for seasonal prototypes','block24_interval_pp':paired_block_interval_v5(delta,24,240514),'block48_interval_pp':paired_block_interval_v5(delta,48,480514),'p10_difference_pp':float((np.quantile(new_return[valid],.1)-np.quantile(base_return[valid],.1))*100),'median_mdd_difference_pp':float((np.median([new_rows[i]['mdd'] for i in np.flatnonzero(valid)])-np.median([base_rows[i]['mdd'] for i in np.flatnonzero(valid)]))*100),'years':{},'halves':{},'seasonal':[]}
for year in sorted({dates[s].year for s in window_starts}):
    m=valid&np.array([dates[s].year==year for s in window_starts]);numeric_stats_v5['years'][str(year)]={'pairs':int(m.sum()),'mean_difference_pp':float(np.mean(delta[m])*100) if m.any() else None}
for label,m in [('early',np.arange(len(valid))<len(valid)//2),('late',np.arange(len(valid))>=len(valid)//2)]:
    m=m&valid;numeric_stats_v5['halves'][label]={'pairs':int(m.sum()),'mean_difference_pp':float(np.mean(delta[m])*100)}
for year,day in [(2024,'2024-10-29'),(2025,'2025-10-28')]:
    pos=next(i for i,s in enumerate(window_starts) if str(dates[s])==day)
    numeric_stats_v5['seasonal'].append({'date':day,'valid':bool(valid[pos]),'candidate_return_pct':float(new_return[pos]*100),'A_return_pct':float(base_return[pos]*100),'difference_pp':float(delta[pos]*100) if valid[pos] else None,'source_status':next(r['status'] for r in coverage_a_decisions_v5 if r['date']==day)})
numeric_stats_v5['risk_pass']=numeric_stats_v5['p10_difference_pp']>=-1 and numeric_stats_v5['median_mdd_difference_pp']>=-1
numeric_stats_v5['supported']=all(numeric_stats_v5[k][0]>0 for k in ['block24_interval_pp','block48_interval_pp']) and numeric_stats_v5['risk_pass'] and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in numeric_stats_v5['years'].values()) and all(v['mean_difference_pp']>0 for v in numeric_stats_v5['halves'].values())
numeric_stats_v5['conclusion']='supported within retained formal-only scope' if numeric_stats_v5['supported'] else 'unproven improvement; formal-only conservative prototype, not all announcements'
(recovery_root/'A策略_開賽前財務調權統計_v5_2026-10-14.json').write_text(json.dumps(numeric_stats_v5,ensure_ascii=False,indent=2),encoding='utf-8')
print('Numerical comparison',json.dumps(numeric_stats_v5,ensure_ascii=False))
# Compact static audit table plus full-window standalone hover chart; no dependency download.
plot_rows=[]
for name,rows in [('A等權',base_rows),('A溫和動能',v4_saved['windows']['A_mild']),('A財務開賽前調權',new_rows)]:
    for i,r in enumerate(rows):
        if not r['invalid'] and r['violations']==0:plot_rows.append({'method':name,'start':str(dates[int(window_starts[i])]),'end':str(dates[int(window_starts[i])+23]),'return_pct':r['return']*100,'cost_pct':r['cost']*100,'mdd_pct':r['mdd']*100})
chart_html_v5='''<!doctype html><meta charset="utf-8"><title>A開賽前財務調權比較</title><style>body{font:16px sans-serif;margin:28px;background:#fafafa}svg{background:white;width:100%;max-width:1050px}button{margin:6px;padding:8px}#tip{position:fixed;background:#fff;border:1px solid #777;padding:10px;pointer-events:none;display:none}p{max-width:1000px}</style><h1>A開賽前財務調權比較</h1><p>同25檔A選股、24交易日、每5日調整、90%投入、整千股與費稅。財務訊號固定開賽前；未知保持中性。桃紅線是中位數，虛線是0%。每個有效窗口都畫點。僅正式財報本業子實驗，未含完整自結、營收與LLM。</p><button onclick="draw(false)">全部窗口</button><button onclick="draw(true)">歷年比賽起點</button><div id="counts"></div><svg id="plot" viewBox="0 0 1050 620"></svg><div id="tip"></div><script>const data=PAYLOAD;const names=['A等權','A溫和動能','A財務開賽前調權'];const lo=Math.min(0,...data.map(d=>d.return_pct))-3,hi=Math.max(0,...data.map(d=>d.return_pct))+3;const y=v=>550-(v-lo)/(hi-lo)*490;function quant(a,p){let k=(a.length-1)*p,i=Math.floor(k);return a[i]+(a[Math.min(i+1,a.length-1)]-a[i])*(k-i)}function draw(season){let rows=data.filter(d=>!season||['2024-10-29','2025-10-28'].includes(d.start));let html='';for(let v=Math.ceil(lo/10)*10;v<=hi;v+=10)html+=`<line x1="80" x2="1000" y1="${y(v)}" y2="${y(v)}" stroke="#eee"/><text x="20" y="${y(v)+4}">${v}%</text>`;html+=`<line x1="80" x2="1000" y1="${y(0)}" y2="${y(0)}" stroke="#333" stroke-dasharray="6 4"/><text x="20" y="${y(0)-6}">0%</text>`;names.forEach((name,i)=>{let r=rows.filter(d=>d.method===name),a=r.map(d=>d.return_pct).sort((a,b)=>a-b),x=240+320*i;if(!a.length)return;let q1=quant(a,.25),m=quant(a,.5),q3=quant(a,.75);html+=`<rect x="${x-48}" y="${y(q3)}" width="96" height="${y(q1)-y(q3)}" fill="#aac9e4" fill-opacity=".45" stroke="#466b8a"/><line x1="${x-48}" x2="${x+48}" y1="${y(m)}" y2="${y(m)}" stroke="#df2873" stroke-width="4"/><text x="${x}" y="595" text-anchor="middle">${name} (${r.length})</text>`;r.forEach((d,j)=>{let jitter=season?(j-.5)*20:((j*0.61803398875)%1-.5)*170;html+=`<circle cx="${x+jitter}" cy="${y(d.return_pct)}" r="${season?6:2.6}" fill="#2d6996" opacity="${season?.9:.45}" data-idx="${data.indexOf(d)}"/>`})});document.getElementById('plot').innerHTML=html;document.getElementById('counts').textContent='有效點數：'+rows.length+'；兩種快捷鍵固定座標與方法順序。';document.querySelectorAll('circle').forEach(c=>{c.onmousemove=e=>{let d=data[+c.dataset.idx],t=document.getElementById('tip');t.textContent=`${d.method}｜${d.start}～${d.end}｜報酬${d.return_pct.toFixed(2)}%｜費用${d.cost_pct.toFixed(2)}%｜回撤${d.mdd_pct.toFixed(2)}%`;t.style.display='block';t.style.left=Math.min(e.clientX+12,innerWidth-420)+'px';t.style.top=e.clientY+14+'px'};c.onmouseleave=()=>document.getElementById('tip').style.display='none'})}draw(false)</script>'''.replace('PAYLOAD',json.dumps(plot_rows,ensure_ascii=False,allow_nan=False))
(recovery_root/'A策略_開賽前財務調權比較_v5_2026-10-14.html').write_text(chart_html_v5,encoding='utf-8')
print('NUMERICAL_INFERENCE_AND_ALL_POINTS_SAVED',len(plot_rows))


In [ ]:
# Recover retained-source revenue protocol, freeze two legacy-formula prestart variants
extra_definitions_v5={'A_revenue_prestart_v5':{'signal':'MoM and YoY same sign, known original bulletin before start08:55,45day age and1-2month lag','weights':'1+.25 sign normalized to90%; missingneutral; all150 evidence frozen at start','formula_origin':'existing revenue_weight, timing now prestart freeze'},'A_combined_prestart_v5':{'weights':'50% mild rank weights +25% formal operating prestart weights +25% revenue prestart weights','formula_origin':'existing combined blend; no parameter search','unparsed_self':'neutral, never bearish; full self/conference integration not claimed'}}
(recovery_root/'A策略_營收與組合開賽前實驗定義_v5_2026-10-14.json').write_text(json.dumps(extra_definitions_v5,ensure_ascii=False,indent=2),encoding='utf-8')
revenue_cache_path_v5=recovery_root/'A策略_營收原文快取_v5_2026-10-14.json.gz'
if revenue_cache_path_v5.exists():
    rev_cache=json.loads(gzip.decompress(revenue_cache_path_v5.read_bytes()).decode('utf-8'))
    full_pool_revenue_rows_v1=rev_cache['monthly_rows'];full_pool_revenue_news_v1=rev_cache['news_originals']
    for row in full_pool_revenue_news_v1:
        if row.get('published_at'):row['published_at']=datetime.datetime.fromisoformat(row['published_at'])
else:
    recover_revenue_src=cell_source(current_nb,62)
    recover_revenue_src=recover_revenue_src[recover_revenue_src.index('revenue_fetch_tree='):]
    recover_revenue_src=recover_revenue_src.replace('ast.parse(inspect.getsource(fetch_full_pool_announcements_v1))',"ast.Module(body=[next(n for n in ast.parse(cell_source(current_nb,57)).body if isinstance(n,ast.FunctionDef))],type_ignores=[])")
    exec(recover_revenue_src,globals())
    rev_cache={'scope':'legacy fullpool known bulletin-time recovery, not first publication certification','monthly_rows':full_pool_revenue_rows_v1,'news_originals':full_pool_revenue_news_v1}
    revenue_cache_path_v5.write_bytes(gzip.compress(json.dumps(rev_cache,ensure_ascii=False,default=str).encode('utf-8')))
recover_revenue_parse=cell_source(current_nb,63)
exec(recover_revenue_parse,globals())
prior_numerical_allocation_v5=numerical_allocation_v5
def numerical_allocation_v5(start,t,selected,mode):
    if mode not in ['revenue_prestart','combined_prestart']:return prior_numerical_allocation_v5(start,t,selected,mode)
    multipliers=[]
    for j in selected:
        r=pure_revenue_evidence_v1[(start,stock_ids[j])];sign=0
        if r['status']=='verified_original_known_available_time':sign=1 if r['mom']>0 and r['yoy']>0 else -1 if r['mom']<0 and r['yoy']<0 else 0
        multipliers.append(1+.25*sign)
    w=INVESTED*np.array(multipliers)/sum(multipliers)
    if mode=='combined_prestart':w=.5*prior_numerical_allocation_v5(start,t,selected,'mild')+.25*prior_numerical_allocation_v5(start,t,selected,'formal_prestart')+.25*w
    assert np.isclose(w.sum(),.9) and w.max()<.1 and len(w)==25
    return w
for mode,name in [('revenue_prestart','A_revenue_prestart'),('combined_prestart','A_combined_prestart')]:
    numeric_prestart_results_v5[name]=[simulate_numeric_prestart_v5(int(s),mode) for s in window_starts]
    print('Saved new prestart variant',name,'650 windows,valid',sum(not r['invalid'] and r['violations']==0 for r in numeric_prestart_results_v5[name]),flush=True)
numerical_export_v5['extra_definitions']=extra_definitions_v5;numerical_export_v5['results']=clean_v5(numeric_prestart_results_v5)
(recovery_root/'A策略_開賽前財務調權逐窗口_v5_2026-10-14.json').write_text(json.dumps(numerical_export_v5,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('REVENUE_AND_COMBINED_PRESTART_SAVED_INFERENCE_PENDING')


In [ ]:
# Unified numerical inference: all three frozen timing variants; 23-comparison conservative family
numeric_family_results_v5={}
def numerical_summary_v5(rows,base,family=23):
    valid=np.array([not r['invalid'] and r['violations']==0 and not b['invalid'] and b['violations']==0 for r,b in zip(rows,base)])
    rv=np.array([r['return'] for r in rows]);bv=np.array([b['return'] for b in base]);diff=np.where(valid,rv-bv,np.nan)
    changed=valid&~np.isclose(diff,0,rtol=0,atol=1e-12)
    out={'pairs':int(valid.sum()),'mean_difference_pp':float(np.mean(diff[valid])*100),'candidate_median_pct':float(np.median(rv[valid])*100),'A_median_same_dates_pct':float(np.median(bv[valid])*100),'ties':int((valid&~changed).sum()),'beat_pct_all_pairs':float(np.mean(diff[valid]>1e-12)*100),'beat_pct_changed_pairs':float(np.mean(diff[changed]>0)*100) if changed.any() else None,'mean_cost_pct':float(np.mean([rows[i]['cost'] for i in np.flatnonzero(valid)])*100),'block24_interval_pp':paired_block_interval_v5(diff,24,240514,family),'block48_interval_pp':paired_block_interval_v5(diff,48,480514,family),'family':family,'p10_difference_pp':float((np.quantile(rv[valid],.1)-np.quantile(bv[valid],.1))*100),'median_mdd_difference_pp':float((np.median([rows[i]['mdd'] for i in np.flatnonzero(valid)])-np.median([base[i]['mdd'] for i in np.flatnonzero(valid)]))*100),'years':{},'halves':{},'seasonal':[],'other_controls':{}}
    for y in sorted({dates[s].year for s in window_starts}):
        mask=valid&np.array([dates[s].year==y for s in window_starts])
        if mask.any():out['years'][str(y)]={'pairs':int(mask.sum()),'mean_difference_pp':float(np.mean(diff[mask])*100)}
    for label,mask in [('early',np.arange(len(valid))<len(valid)//2),('late',np.arange(len(valid))>=len(valid)//2)]:
        mask=mask&valid;out['halves'][label]={'pairs':int(mask.sum()),'mean_difference_pp':float(np.mean(diff[mask])*100)}
    for day in ['2024-10-29','2025-10-28']:
        i=next(i for i,s in enumerate(window_starts) if str(dates[s])==day)
        out['seasonal'].append({'date':day,'valid':bool(valid[i]),'return_pct':float(rv[i]*100),'A_return_pct':float(bv[i]*100),'difference_pp':float(diff[i]*100) if valid[i] else None})
    for name in ['A_mild','A_mild_tolerance']:
        cr=v4_saved['windows'][name];mask=valid&np.array([not r['invalid'] and r['violations']==0 for r in cr])
        cd=np.array([r['return'] for r in cr]);out['other_controls'][name]={'pairs':int(mask.sum()),'mean_difference_pp':float(np.mean(rv[mask]-cd[mask])*100)}
    out['risk_pass']=out['p10_difference_pp']>=-1 and out['median_mdd_difference_pp']>=-1
    out['supported']=all(out[k][0]>0 for k in ['block24_interval_pp','block48_interval_pp']) and out['risk_pass'] and all(v['mean_difference_pp']>0 for v in out['years'].values()) and all(v['mean_difference_pp']>0 for v in out['halves'].values())
    return out
for name,rows in numeric_prestart_results_v5.items():
    numeric_family_results_v5[name]=numerical_summary_v5(rows,v4_saved['windows']['A_equal'])
    print('UNIFIED_NUMERICAL',name,json.dumps(numeric_family_results_v5[name],ensure_ascii=False),flush=True)
numeric_family_export_v5={'status':'numeric_subexperiments_complete_no_LLM_or_full_self_integration','family':23,'family_members':'14 earlier broad nonA+6 earlier seasonal+3 new frozen-prestart timing variants; no data-driven parameter grid','results':numeric_family_results_v5,'selected_supported':[name for name,r in numeric_family_results_v5.items() if r['supported']],'limitations':['retained original known-time sources do not certify first publication','formal financial onlyQ2/Q1,Q3/Q2; newer unparsed earnings/corrections neutral','no untouched out-of-sample evaluation','only2 competition-season years']}
(recovery_root/'A策略_開賽前資訊調權統一統計_v5_2026-10-14.json').write_text(json.dumps(numeric_family_export_v5,ensure_ascii=False,indent=2),encoding='utf-8')
# Reuse chart layout, include all methods and all points with fixed axes across shortcuts.
plot_rows=[]
plot_methods=[('A等權',v4_saved['windows']['A_equal']),('A溫和動能',v4_saved['windows']['A_mild'])]+[(n,r) for n,r in numeric_prestart_results_v5.items()]
for name,rows in plot_methods:
    for i,r in enumerate(rows):
        if not r['invalid'] and r['violations']==0:plot_rows.append({'method':name,'start':str(dates[int(window_starts[i])]),'end':str(dates[int(window_starts[i])+23]),'return_pct':r['return']*100,'cost_pct':r['cost']*100,'mdd_pct':r['mdd']*100})
chart=chart_html_v5
start=chart.index('const data=');stop=chart.index(';const names=',start)
chart=chart[:start]+'const data='+json.dumps(plot_rows,ensure_ascii=False,allow_nan=False)+chart[stop:]
names=[n for n,_ in plot_methods];start=chart.index('const names=');stop=chart.index(';const lo=',start)
chart=chart[:start]+'const names='+json.dumps(names,ensure_ascii=False)+chart[stop:]
labels={'A_formal_operating_prestart':'A財務調權','A_revenue_prestart':'A營收調權','A_combined_prestart':'A組合調權'}
chart=chart.replace('x=240+320*i','x=170+185*i').replace('${name} (${r.length})','${labels[name]||name} (${r.length})').replace('const names=', 'const labels='+json.dumps(labels,ensure_ascii=False)+';const names=')
chart=chart.replace('let q1=quant(a,.25),m=quant(a,.5),q3=quant(a,.75);','let q1=quant(a,.25),m=quant(a,.5),q3=quant(a,.75),iqr=q3-q1,wlo=a.find(v=>v>=q1-1.5*iqr),whi=[...a].reverse().find(v=>v<=q3+1.5*iqr);html+=`<line x1="${x}" x2="${x}" y1="${y(wlo)}" y2="${y(whi)}" stroke="#466b8a"/><line x1="${x-16}" x2="${x+16}" y1="${y(wlo)}" y2="${y(wlo)}" stroke="#466b8a"/><line x1="${x-16}" x2="${x+16}" y1="${y(whi)}" y2="${y(whi)}" stroke="#466b8a"/>`;')
chart=chart.replace('A開賽前財務調權比較','A開賽前資訊調權比較').replace('財務訊號固定開賽前','資訊訊號固定開賽前').replace('僅正式財報本業子實驗，未含完整自結、營收與LLM。','包含正式財報本業、營收雙率與既有組合公式的開賽前固定訊號；未含完整自結、法說與LLM判讀。')
(recovery_root/'A策略_開賽前資訊調權比較_v5_2026-10-14.html').write_text(chart,encoding='utf-8')
print('ALL_NUMERIC_UNIFIED_SAVED',len(plot_rows),'supported',numeric_family_export_v5['selected_supported'])


In [ ]:
# Prepare point-in-time LLM input packets and historical order checks; never submit trades
llm_packets_v5=[];opening_order_audits_v5=[]
for day in ['2024-10-29','2025-10-28']:
    t=di[datetime.date.fromisoformat(day)];cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55))
    selected=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day)
    for sid in selected:
        formal=[r for r in formal_by_stock_v5[sid] if r['year']==cutoff.year and datetime.datetime.fromisoformat(r['known_at'])<cutoff]
        latest_q={}
        for r in formal:
            q=r['quarter']
            if q not in latest_q or r['known_at']>latest_q[q]['known_at']:latest_q[q]=r
        ids={r['ann_id'] for r in latest_q.values()}
        cutoff_date=datetime.datetime.combine(datetime.date(cutoff.year,7,1),datetime.time())
        extra=[a for a in full_pool_ann_by_stock_v5[sid] if prestart_stamp(a) and cutoff_date<=prestart_stamp(a)<cutoff and notice_class_v5[a['ann_id']] in ['financial_correction_requires_scope_review','actual_or_attention_financial_unparsed']]
        ids.update(a['ann_id'] for a in extra)
        source_rows=[{'source_id':a['ann_id'],'known_at':str(prestart_stamp(a)),'subject':a['subject'],'content':a['content'],'source':a['source'],'classification':notice_class_v5[a['ann_id']]} for a in full_pool_ann_by_stock_v5[sid] if a['ann_id'] in ids]
        assert all(datetime.datetime.fromisoformat(a['known_at'])<cutoff for a in source_rows)
        rv=pure_revenue_evidence_v1[(t,sid)]
        rev_source=[]
        if rv['status']=='verified_original_known_available_time':
            news=next(a for a in full_pool_revenue_news_v1 if a['news_id']==rv['news_id'] and a['stock_id']==sid)
            rev_source=[{'source_id':rv['news_id'],'known_at':str(rv['published_at']),'url':rv['url'],'title':news['title'],'content':news['content'] or news['summary'],'role':'known original bulletin rates; not certified first public time'}]
        llm_packets_v5.append({'year':cutoff.year,'stock_id':sid,'cutoff':str(cutoff),'formal_evidence':evidence_v5[(t,sid)],'original_sources':source_rows,'revenue_sources':rev_source,'coverage':'retained selected formal periods plus all classified actual/correction notices since July1; not complete issuer IR or annual notes','required_output':['source_references','economic_period','unit','verified_facts','supporting_evidence','contradictions','unresolved_questions','confidence','no_weight_until_fixed_quality_protocol'],'llm_output':None})
    j=np.array([si[s] for s in selected])
    for mode in ['equal','mild','formal_prestart','revenue_prestart','combined_prestart']:
        w=numerical_allocation_v5(t,t,j,mode);qty=(np.floor(w*INITIAL/close[t-1,j]/1000)*1000).astype(np.int64)
        estimated_buy=float(np.dot(qty,close[t-1,j]));cash_estimate=INITIAL-estimated_buy*(1+FEE)
        post_weights=qty*close[t-1,j]/(INITIAL-estimated_buy*FEE)
        checks={'selected25':len(j)==25,'held20to30':20<=int(np.count_nonzero(qty))<=30,'lot1000':bool(np.all(qty%1000==0)),'positive_previous_close':bool(np.all(np.isfinite(close[t-1,j])&(close[t-1,j]>0))),'weights90pct':bool(np.isclose(w.sum(),.9)),'estimated_cash_positive':cash_estimate>0,'estimated_cash_below25pct':cash_estimate/(INITIAL-estimated_buy*FEE)<.25,'estimated_cap_under10pct':bool(np.all(post_weights<.1))}
        opening_order_audits_v5.append({'trade_date':day,'as_of_date':str(dates[t-1]),'mode':mode,'account_kind':'historical research initial1b cash, not actual official DataPack','target_weights':dict(zip(selected,w.tolist())),'order_quantities':dict(zip(selected,qty.tolist())),'previous_close_estimated_cash':cash_estimate,'checks':checks,'official_active_share':'not_verifiable_without_corresponding_benchmark_weights','actual_cash_feasibility':'needs official settlement and void semantics; previous-close check is estimate','valid_official_DPlan':False,'submitted':False})
llm_input_path_v5=recovery_root/'A策略_LLM歷史原文輸入包_v5_2026-10-14.json.gz'
llm_input_path_v5.write_bytes(gzip.compress(json.dumps(llm_packets_v5,ensure_ascii=False,default=str).encode('utf-8')))
(recovery_root/'A策略_歷史開賽股數與資金查核_v5_2026-10-14.json').write_text(json.dumps(opening_order_audits_v5,ensure_ascii=False,indent=2),encoding='utf-8')
print('LLM point-in-time packets',len(llm_packets_v5),'original documents',sum(len(r['original_sources']) for r in llm_packets_v5),'revenue original packets',sum(len(r['revenue_sources']) for r in llm_packets_v5),'LLM numerical weights evaluated',False)
print('Historical opening checks',len(opening_order_audits_v5),'all mechanical checks pass',all(all(r['checks'].values()) for r in opening_order_audits_v5))
print('LLM batch runtime and official DataPack/team are required inputs, not fabricated by this Notebook.')
print('LLM_INPUTS_AND_HISTORICAL_ORDER_CHECKS_SAVED_NO_TRADES')


In [ ]:
# Mechanically matched diagnostic control; no search or new candidate strategy
matched_control_definition_v5={'role':'diagnostic, not candidate selected by return','weights':'.5*mild+.5*equal, matching combined momentum coefficient with information component neutral','reason':'combined vs full-mild alone confounds momentum strength with information','params':'same frozen legacy combined .5 coefficient; no threshold/multiplier tuning','family_adjustment':24,'LLM_branch':'deferred at explicit user request','team_id':'TEAM_11693'}
(recovery_root/'A策略_資訊增量對照定義_v5_2026-10-14.json').write_text(json.dumps(matched_control_definition_v5,ensure_ascii=False,indent=2),encoding='utf-8')
numerical_allocation_before_diagnostic_v5=numerical_allocation_v5
def numerical_allocation_v5(start,t,selected,mode):
    if mode=='matched_neutral':return .5*prior_numerical_allocation_v5(start,t,selected,'mild')+.5*prior_numerical_allocation_v5(start,t,selected,'equal')
    return numerical_allocation_before_diagnostic_v5(start,t,selected,mode)
matched_neutral_results_v5=[simulate_numeric_prestart_v5(int(s),'matched_neutral') for s in window_starts]
matched_info_stats_v5=numerical_summary_v5(numeric_prestart_results_v5['A_combined_prestart'],matched_neutral_results_v5,family=24)
# Correct inherited labels from generic summary: reference is matched neutral, not A.
matched_info_stats_v5['reference']='.5mild+.5equal same25 A selection'
matched_info_stats_v5['reference_median_same_dates_pct']=matched_info_stats_v5.pop('A_median_same_dates_pct')
for r in matched_info_stats_v5['seasonal']:r['reference_return_pct']=r.pop('A_return_pct')
matched_info_stats_v5.pop('other_controls')
for name,rows in numeric_prestart_results_v5.items():numeric_family_results_v5[name]=numerical_summary_v5(rows,v4_saved['windows']['A_equal'],family=24)
numeric_family_export_v5['family']=24;numeric_family_export_v5['results']=numeric_family_results_v5
numeric_family_export_v5['matched_information_diagnostic']=matched_info_stats_v5
numeric_family_export_v5['family_members']='14 prior broad nonA+6 prior seasonal+3 new frozen-prestart variants+1 matched information diagnostic, conservatively counted'
numeric_family_export_v5['LLM_branch']='deferred by user; not backtested'
numeric_family_export_v5['selected_supported']=[n for n,r in numeric_family_results_v5.items() if r['supported']]
(recovery_root/'A策略_開賽前資訊調權統一統計_v5_2026-10-14.json').write_text(json.dumps(numeric_family_export_v5,ensure_ascii=False,indent=2),encoding='utf-8')
(recovery_root/'A策略_資訊增量對照結果_v5_2026-10-14.json').write_text(json.dumps({'definition':matched_control_definition_v5,'stats':matched_info_stats_v5,'control_windows':clean_v5(matched_neutral_results_v5)},ensure_ascii=False),encoding='utf-8')
print('Matched information diagnostic',json.dumps(matched_info_stats_v5,ensure_ascii=False))
print('NO_SUPPORTED_NUMERICAL_IMPROVEMENT',numeric_family_export_v5['selected_supported'])


In [ ]:
# Persistent report and finite continuation state: LLM explicitly deferred, official account pending
actual_revenue_stock_count_v5=len({r['stock_id'] for r in pure_revenue_records_v1})
report_lines_v5=['# A開賽前資訊調權研究 v5','',
'固定A選25檔、24交易日、每五日調整、90%投入、整千股與費稅。開賽前訊號固定至窗口結束；新公告不在這個實驗中調權。主線A，不是AH靜態持有。',
'',
'本轮完成正式財報本業、營收雙率、既有組合公式的三個開賽前固定訊號原型，各保存650候選窗口；與A共有535有效窗口。A等權與A溫和動能的650筆報酬／費用／有效性與v4逐筆一致。資料快照與42筆公司行動修正沿用，沒有依新結果調參。',
'',
'## 數值比較與選擇',
'',
'|方法|共同窗口|平均相對A（百分點）|費用後報酬中位數|平均費用／本金|平均相對A溫和動能（百分點）|判斷|',
'|---|---:|---:|---:|---:|---:|---|']
labels_v5={'A_formal_operating_prestart':'A正式財報本業開賽前調權','A_revenue_prestart':'A營收MoM／YoY開賽前調權','A_combined_prestart':'A動能＋財務＋營收開賽前組合'}
for name,r in numeric_family_results_v5.items():
    report_lines_v5.append(f"|{labels_v5[name]}|{r['pairs']}|{r['mean_difference_pp']:+.4f}|{r['candidate_median_pct']:.4f}%|{r['mean_cost_pct']:.4f}%|{r['other_controls']['A_mild']['mean_difference_pp']:+.4f}|未證實改良|")
report_lines_v5+=['',
'24／48窗口位置區塊各5000次，保留無效窗口的位置；保守計入24個已比較候選／診斷，控制家族5%。三個原型的校正區間均跨零；財報與組合2026負向，營收原型2024／2025負向。未知中性造成部分窗口與A相同，因此總勝率與非相同窗口勝率在JSON分列。',
'',
f"組合策略的動能係數為50%，不能只與100%溫和動能比較就推論資訊價值。另使用相同50%動能＋50%等權的機械對照，資訊增量平均只有 {matched_info_stats_v5['mean_difference_pp']:+.4f} 百分點，24區塊區間 {matched_info_stats_v5['block24_interval_pp']}、48區塊區間 {matched_info_stats_v5['block48_interval_pp']} 均跨零；2025／2026及晚段為負。沒有證據證實這個簡化資訊規則改善動能配置。這不代表所有財報／法說資訊都無用。",
'',
'目前不更換A基準，也不將新組合宣稱為勝者。若必須符合第一天非等權的需求，既有A溫和動能＋小額略過仍只作暫定研究配置，不代表通過上場門檻。',
'',
'## 比賽同期案例',
'',
'|方法|2024開賽24日|2025開賽24日|',
'|---|---:|---:|']
for name,r in numeric_family_results_v5.items():report_lines_v5.append('|'+labels_v5[name]+'|'+'|'.join(f"{x['return_pct']:.4f}%（相對A{x['difference_pp']:+.4f}pp）" for x in r['seasonal'])+'|')
report_lines_v5+=['','只有兩個同季年份，不把兩點都正向當穩定證據。廣泛窗口與同季案例分開判斷。',
'','## 來源與模型邊界','',
f'150檔、34,379筆原公告。移除舊財報標題篩選；正式報告期間擷取1535筆，1513筆與舊可解析數值一致；另外21筆舊數值紀錄因缺可用時間／代理子公司等仍不直接套用。起訖日期括號、中文年份、貨幣／單位與標題漏取已統一處理。',
f'月營收數值5320筆、6704新聞股票關聯，1280筆單公司雙率原文可核對，實際來源覆蓋{actual_revenue_stock_count_v5}檔。defaultdict的空項不能算有來源。使用報導可證明的已知時間，不冒稱首次公告；入庫時間不用作決策時間。',
'673個日決策股票池已做一致查核。正式財報只支持Q2對Q1、Q3對Q2；缺本年Q1／可解析期間保持中性。新自結、法說摘要、注意損益與更正未完成期間／範圍核對時保持中性，不能假稱全部公告已整合。2025起點25檔中12檔有可用正式比較、12檔有較新來源待核對、1檔缺正式期間。',
'選股使用前一日訊號，股數使用可交易NAV，不含未到帳應收；當日價格只用於歷史成交結算。12筆現金增資與1筆價格異常維持排除；股利實際發放日及官方整批作廢語義仍未全建模。未做Restart and Run All，保留kernel。',
'',
'## 尚未完成與使用者最新決定','',
'LLM分支依使用者明確要求先不測。已保存50個歷史股票／年份資料包、159份原公告與18份營收原文，未產生LLM數值權重或宣稱LLM回測完成。',
'團隊代號為TEAM_11693。DataPack是主辦方結算後的帳戶資料JSON，在參賽後台「文件繳交」頁下載，包含account、holdings、as_of_date、prev_day_orders與compliance。不能以研究假設帳戶替代它；尚無官方DataPack，正式D-Plan與Active Share演練仍待資料。',
'歷史兩年、五種配置共10組開賽機械檢查通過：25檔、千股倍數、90%目標、前日收盤與預估資金／10%個股上限。它們不是正式DataPack、不是保證當日可成交或Active Share合規，沒有提交交易。',
'可執行數值原型與相應統計已完成。完整較新自結／更正語義尚未整合，維持來源限制；LLM暫緩；實際操作分支等待官方DataPack。排程保持PAUSED，不是因20分鐘期限停止，也不終止kernel。',
'','## 成果入口','',
'[五方法全部有效點互動箱形圖](A策略_開賽前資訊調權比較_v5_2026-10-14.html)：2676點、固定座標與方法順序、桃紅中位數、0%線、單一懸停日期／報酬／費用／回撤。',
'[統一統計JSON](A策略_開賽前資訊調權統一統計_v5_2026-10-14.json)、[逐窗口結果](A策略_開賽前財務調權逐窗口_v5_2026-10-14.json)、[來源與時間查核](A策略_來源與時間統一查核_v5_2026-10-14.json)、[歷史股數與資金查核](A策略_歷史開賽股數與資金查核_v5_2026-10-14.json)。']
report_path_v5=recovery_root/'A策略_開賽前資訊調權研究_v5_2026-10-14.md'
report_path_v5.write_text('\n'.join(report_lines_v5)+'\n',encoding='utf-8')
state_path=recovery_root/'A策略_無人值守研究狀態.json';state=json.loads(state_path.read_text(encoding='utf-8'))
state['team_id']='TEAM_11693';state['status']='numerical_prestart_prototypes_complete_LLM_deferred_official_datapack_pending';state['current_node']=None
state['continuation_plan_v5']={'scope':'six-stage plan accepted; finite numeric work completed within stated source limits','stages':[
{'id':'v5_source','status':'completed_uniform_conservative_coverage_with_unparsed_gates','cells':[147,148,149,150,151],'remaining':'full newer self/conference/correction interpretation not integrated; timestamp gaps retained'},
{'id':'v5_definitions','status':'numeric_frozen_LLM_deferred','cells':[152,154,157]},
{'id':'v5_numeric','status':'three_frozen_prestart_prototypes_completed','cells':[152,154]},
{'id':'v5_LLM','status':'deferred_by_user','inputs_cell':156,'not_backtested':True},
{'id':'v5_inference','status':'completed_numeric_only_unproven_improvement','cells':[153,155,157]},
{'id':'v5_official','status':'historical_mechanical_checks_completed_awaiting_official_DataPack','cell':156,'team_id':'TEAM_11693','not_submitted':True}],
'supported':[],'report':report_path_v5.name,'chart':'A策略_開賽前資訊調權比較_v5_2026-10-14.html','kernel_connection':'kernel-aicup-backtest-20261013.json','running_work':False,'LLM_user_request':'目前先不要測這個','next_required_user_input':'official DataPack when available; do not fabricate actual account or restart LLM testing'}
state['automation_status']='PAUSED';state['last_completed_action']='v5 uniform150 source audit,3 frozen numerical prestart prototypes650 each,535 pairs,24-family block inference,matched information diagnostic,50 LLM input packets deferred by user,10 historical opening checks; official DataPack pending'
state['updated_at']=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=8))).isoformat()
state.setdefault('final_artifacts',[])
for name in [report_path_v5.name,'A策略_開賽前資訊調權統一統計_v5_2026-10-14.json','A策略_開賽前資訊調權比較_v5_2026-10-14.html','A策略_來源與時間統一查核_v5_2026-10-14.json','A策略_歷史開賽股數與資金查核_v5_2026-10-14.json']:
    if name not in state['final_artifacts']:state['final_artifacts'].append(name)
state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
workflow=recovery_root/'A策略_無人值守研究流程.md';text=workflow.read_text(encoding='utf-8')
update='## 本輪執行結果 v5：數值原型完成，LLM依使用者要求暫緩\n\n[完整結果與剩餘限制](A策略_開賽前資訊調權研究_v5_2026-10-14.md)。全150檔34,379原公告一致查核，正式財報／營收／組合三個開賽前固定訊號原型各650窗口、535與A配對；控制組精確重現v4。24家族校正未證實任何改良。匹配50%動能的對照後，資訊增量平均僅+0.0042pp、區間跨零。較新自結／更正仍保留未解析限制，不稱全部公告完成。\n\n使用者本輪要求LLM先不測；50原文資料包保存但未產生LLM權重。團隊TEAM_11693，10組歷史開賽股數／資金機械檢查完成，正式帳戶／Active Share／D-Plan待官方DataPack。現行狀態以此節與state.continuation_plan_v5為準；排程保持PAUSED，無執行中工作，kernel保留。下列六階段為計畫與歷史定義，不把尚未通過的官方门檻偽稱完成。\n\n'
if '## 本輪執行結果 v5：' not in text:workflow.write_text(text.split('\n',1)[0]+'\n\n'+update+text.split('\n',1)[1],encoding='utf-8')
print('V5_REPORT_STATE_AND_WORKFLOW_SAVED',report_path_v5.name,'verified revenue-source stocks',actual_revenue_stock_count_v5)


In [ ]:
# v6 original-period review: reconnect metadata and pending original notices only
import sys,json,datetime,re,unicodedata
from collections import Counter,defaultdict
print('Retained kernel',sys.executable)
for name in ['full_pool_announcements_v5','formal_by_stock_v5','evidence_v5','numeric_prestart_results_v5','numerical_allocation_v5']:
    obj=globals().get(name);print(name,type(obj).__name__,len(obj) if hasattr(obj,'__len__') else None)
print('Safe history markers',[(i,s.splitlines()[0][:90]) for _,i,s in get_ipython().history_manager.get_tail(8,include_latest=False) if s.startswith('#')])
v6_review_cutoff=datetime.datetime(2025,10,28,8,55)
v6_review_t=di[v6_review_cutoff.date()]
v6_initial_ids=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']=='2025-10-28')
v6_pending_ids=sorted({aid for sid in v6_initial_ids for aid in evidence_v5[(v6_review_t,sid)].get('pending_ids',[])})
v6_pending_originals=[a for a in full_pool_announcements_v5 if a['ann_id'] in v6_pending_ids]
print('Pending count',len(v6_pending_originals),'stocks',len({a['stock_id'] for a in v6_pending_originals}))
for a in v6_pending_originals:
    body=unicodedata.normalize('NFKC',a.get('content') or '')
    print('ORIGINAL',a['stock_id'],a['ann_id'],str(prestart_stamp(a)),a['subject'])
    if a['stock_id'] in ['2408','6770','8299','6223'] or '更正' in a['subject']:
        print(body[:6500])
    else:print('\n'.join(l for l in body.splitlines() if re.search(r'單位|仟元|千元|百萬|期間|月|季|年|營業|營收|淨利|每股|更正',l))[:2300])
print('V6_PENDING_ORIGINAL_REVIEW_READY')


In [ ]:
# v6 uniform type and explicit quarter extraction, no portfolio changes
def numeric_token_v6(token):
    s=token.replace(',','').strip();return float(s.strip('()'))*(-1 if s.startswith('(') else 1)
def original_kind_v6(a):
    s=unicodedata.normalize('NFKC',a.get('subject') or '');b=unicodedata.normalize('NFKC',a.get('content') or '')
    if re.search(r'^代|公告代|代重要子公司|代子公司',s):return 'delegated_subsidiary'
    if re.search(r'注意交易|處置交易|注意資訊|公布.*財務業務|公佈.*財務業務',s):return 'attention_period_disclosure'
    if re.search(r'更正|補充',s) and re.search(r'財務|財報|損益|自結',s):return 'financial_correction_unresolved'
    if re.search(r'背書保證|資金貸與|公司債|資金貸款',s):return 'financing_not_income_result'
    if re.search(r'\d+月.*(?:自結.*)?(?:合併)?營收',s) and not re.search(r'損益|淨利',s):return 'monthly_revenue_only'
    if formal_summary_extractor_v5(a):return 'formal_cumulative_quarter'
    if re.search(r'預計|預定|召開|舉行|召集',s) and re.search(r'財務報|財報|法說|法人說明|董事會',s):return 'scheduled_not_actual'
    if re.search(r'自結|營運績效|營運結果|法人說明會摘要|財務業務|相關財務|損益情形|財務報告|財務報表',s):return 'quarter_result_or_unresolved'
    return 'other_notice'
def explicit_quarter_v6(a):
    kind=original_kind_v6(a);stamp=prestart_stamp(a)
    if stamp is None or kind not in ['formal_cumulative_quarter','quarter_result_or_unresolved']:return None
    f=formal_summary_extractor_v5(a)
    if f:return dict(f,stock_id=a['stock_id'],basis='cumulative',verification='explicit formal Jan1-to-quarterend period')
    b=unicodedata.normalize('NFKC',a.get('content') or '').replace('\r','')
    # Explicit single-quarter plus Jan-to-quarter cumulative self report table.
    header=re.search(r'(\d{3,4})年\s*(1|4|7|10)\s*[-~至]\s*(3|6|9|12)月\s+(\d{3,4})年\s*1\s*[-~至]\s*(3|6|9|12)月',b)
    if header and re.search(r'單位\s*:\s*新(?:台|臺)幣(?:仟|千)元',b):
        yr,m1,m2,yr2,m22=map(int,header.groups());yr=yr+1911 if yr<1911 else yr;yr2=yr2+1911 if yr2<1911 else yr2
        if yr!=yr2 or m2!=m22 or m1!=m2-2 or stamp.date()<datetime.date(yr,m2,[31,30,30,31][m2//3-1]):return None
        amounts={};cumul={}
        for field,label in [('revenue',r'合併營業收入(?:淨額)?'),('operating',r'合併營業淨利(?:\(損\))?')]:
            line=next((l.strip() for l in b.splitlines() if re.match(label+r'\s*:',l.strip())),None)
            if line is None:return None
            tail=re.sub(label+r'\s*:\s*','',line,count=1)
            match=re.fullmatch(r'(\(?-?\d[\d,]*\)?)\s+(\(?-?\d[\d,]*\)?)',tail)
            if match is None:return None
            amounts[field]=numeric_token_v6(match[1]);cumul[field]=numeric_token_v6(match[2])
        if amounts['revenue']<=0:return None
        return {'stock_id':a['stock_id'],'year':yr,'quarter':m2//3,'basis':'single_with_cumulative','amounts':amounts,'cumulative_amounts':cumul,'known_at':str(stamp),'ann_id':a['ann_id'],'unit':'TWD thousand','verification':'explicit quarter and cumulative headers; previous-cumulative consistency required at asof'}
    # Narrative requires exactly one single-quarter heading and one consolidated field value.
    joined=re.sub(r'\s+','',b)
    quarter=re.search(r'本公司(\d{3,4})年第([1-4])季自結合併',joined)
    if quarter and not re.search(r'前三季|累計|1[-~至]9月|1[-~至]6月',joined):
        yr,q=map(int,quarter.groups());yr=yr+1911 if yr<1911 else yr
        if stamp.date()<datetime.date(yr,q*3,[31,30,30,31][q-1]):return None
        amounts={};factors={'億元':100000.,'百萬元':1000.,'佰萬元':1000.,'仟元':1.,'千元':1.}
        for field,label in [('revenue',r'自結合併(?:營收|營業收入淨額)'),('operating',r'自結合併營業淨利(?:\(損\))?')]:
            matches=list(re.finditer(label+r'新(?:台|臺)幣(\(?-?\d+(?:\.\d+)?\)?)(億元|百萬元|佰萬元|仟元|千元)',joined))
            if len(matches)!=1:return None
            amounts[field]=numeric_token_v6(matches[0][1])*factors[matches[0][2]]
        if amounts['revenue']<=0:return None
        return {'stock_id':a['stock_id'],'year':yr,'quarter':q,'basis':'single','amounts':amounts,'known_at':str(stamp),'ann_id':a['ann_id'],'unit':'TWD thousand converted from explicit text','verification':'single-quarter consolidated narrative; no cumulative ambiguity'}
    return None
quarter_records_v6=[];notice_kinds_v6={};unparsed_quarter_sources_v6=[]
for a in full_pool_announcements_v5:
    kind=original_kind_v6(a);notice_kinds_v6[a['ann_id']]=kind
    r=explicit_quarter_v6(a)
    if r:quarter_records_v6.append(r)
    elif kind=='quarter_result_or_unresolved':unparsed_quarter_sources_v6.append(a)
quarter_by_stock_v6=defaultdict(list)
for r in quarter_records_v6:quarter_by_stock_v6[r['stock_id']].append(r)
print('Uniform notice kinds',dict(Counter(notice_kinds_v6.values())))
print('Quarter parsed',dict(Counter(r['basis'] for r in quarter_records_v6)))
print('New self-quarter source examples',[{k:r.get(k) for k in ['stock_id','year','quarter','basis','amounts','cumulative_amounts','ann_id','known_at']} for r in quarter_records_v6 if r['basis']!='cumulative'][:18])
print('Residual actual-result subject examples',dict(Counter(a['subject'][:45] for a in unparsed_quarter_sources_v6).most_common(12)))
assert next(r for r in quarter_records_v6 if r['ann_id']=='9d4f234378a373ac94b74ced64d48e4bcaf299f3')['amounts']=={'revenue':18779000.,'operating':1119000.}
assert next(r for r in quarter_records_v6 if r['ann_id']=='e674523c9664ad96a8bd31342129b65b5ff55451')['amounts']=={'revenue':11840427.,'operating':-2686163.}
print('V6_UNIFORM_QUARTER_EXTRACTION_VERIFIED')


In [ ]:
# Inspect residual format families before extending source-derived parser
format_seen_v6=set()
for a in unparsed_quarter_sources_v6:
    if not prestart_stamp(a) or prestart_stamp(a)>=v6_review_cutoff:continue
    if not re.search(r'自結.*損益|法人說明會摘要',a['subject']):continue
    b=unicodedata.normalize('NFKC',a.get('content') or '')
    if not re.search(r'營業.*(?:淨利|利益|損失)',b):continue
    signature=(a['stock_id'],'self' if '自結' in a['subject'] else 'conference')
    if signature in format_seen_v6:continue
    format_seen_v6.add(signature)
    print('FORMAT',a['stock_id'],a['ann_id'],a['subject'],str(prestart_stamp(a)),'\n',b[:4200])
    if len(format_seen_v6)>=9:break
for a in full_pool_ann_by_stock_v5['8299']:
    if prestart_stamp(a) and datetime.datetime(2025,9,1)<=prestart_stamp(a)<v6_review_cutoff and '注意' in a['subject']:
        print('ATTENTION_EXTRA',a['ann_id'],original_kind_v6(a),'\n',unicodedata.normalize('NFKC',a.get('content') or '')[:2800])
print('V6_FORMAT_FAMILY_REVIEW_READY')


In [ ]:
# v6 global explicit-quarter extensions; metadata check before further calculation
import ast,sys
print('Kernel',sys.executable,'retained originals',len(full_pool_announcements_v5))
for a in full_pool_ann_by_stock_v5['3105']:
    if prestart_stamp(a) and datetime.datetime(2025,7,1)<prestart_stamp(a)<datetime.datetime(2025,11,1) and '自結' in a['subject']:
        print('3105 table header',a['ann_id'],unicodedata.normalize('NFKC',a.get('content') or '')[:1700])
old_explicit_quarter_v6=explicit_quarter_v6
def explicit_quarter_extended_v6(a):
    result=old_explicit_quarter_v6(a)
    if result:return result
    if original_kind_v6(a)!='quarter_result_or_unresolved':return None
    body=unicodedata.normalize('NFKC',a.get('content') or '')
    if not re.search(r'自結合併|合併自結',body):return None
    body=re.sub(r'(\d{3,4})年\s*(1|4|7|10)/1\s*[-~至]\s*(3|6|9|12)/(31|30)',lambda m:f'{m[1]}年{m[2]}-{m[3]}月' if (int(m[2]),int(m[3]),int(m[4])) in [(1,3,31),(4,6,30),(7,9,30),(10,12,31),(1,6,30),(1,9,30),(1,12,31)] else m[0],body)
    # Entire short table-header line explicitly contains single-quarter and cumulative columns.
    def quarter_header(m):
        y,q,y2,tail=m.groups();q=int(q)
        if y!=y2:return m[0]
        if (q==4 and tail=='度') or tail in [f'前{q}季',f'1-{q*3}月']:
            return f'{y}年{q*3-2}-{q*3}月 {y2}年1-{q*3}月'
        return m[0]
    body=re.sub(r'(?m)^\s*(\d{3,4})年第([1-4])季\s+(\d{3,4})年(度|前[1-4]季|1-\d+月)\s*$',quarter_header,body)
    # Normalize income-row labels only at the line start; no inference of currencies or units.
    body=re.sub(r'(?m)^(\s*)(?:合併)?營業收入(?:淨額)?\s*:?',r'\1合併營業收入淨額:',body)
    body=re.sub(r'(?m)^(\s*)(?:合併)?營業(?:淨利|利益)(\(損\))?\s*:?',r'\1合併營業淨利(損):',body)
    modified=dict(a,content=body)
    result=old_explicit_quarter_v6(modified)
    if result:result['verification']+='; explicit slash-date/quarter-column header and equivalent line-label normalization'
    return result
quarter_records_v6=[];quarter_by_stock_v6=defaultdict(list)
for a in full_pool_announcements_v5:
    r=explicit_quarter_extended_v6(a)
    if r:quarter_records_v6.append(r);quarter_by_stock_v6[r['stock_id']].append(r)
print('Global quarter types',dict(Counter(r['basis'] for r in quarter_records_v6)))
print('New single/cumulative source stocks',sorted({r['stock_id'] for r in quarter_records_v6 if r['basis']!='cumulative'}))
print('V6_EXTENDED_EXPLICIT_QUARTER_READY')


In [ ]:
# Authorized dead-kernel recovery from persisted source snapshots; no external data refetch
import json,ast,datetime
from research_layout import ResearchPath as Path
recovery_root=Path(r'C:\Users\weiha\Desktop\文件\03_課程資料\115-1\資料探勘與應用\B_Final Project (0920~1221)\競賽資料')
saved_nb=json.loads((recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').read_text(encoding='utf-8'))
def recovered_cell(i):return ''.join(saved_nb['cells'][i]['source'])
for i in [147,148,149,150,151]:
    status=get_ipython().run_cell(recovered_cell(i),store_history=True)
    if not status.success:raise RuntimeError('Recovery failed at saved cell '+str(i))
status=get_ipython().run_cell(recovered_cell(152).split('numeric_prestart_results_v5={}')[0],store_history=True)
assert status.success
status=get_ipython().run_cell(recovered_cell(154).split("for mode,name in [('revenue_prestart'")[0],store_history=True)
assert status.success
numerical_export_v5=json.loads((recovery_root/'A策略_開賽前財務調權逐窗口_v5_2026-10-14.json').read_text(encoding='utf-8'))
numeric_prestart_results_v5=numerical_export_v5['results']
window_starts=np.arange(61,len(dates)-HORIZON+1)
for i,names in [(153,['paired_block_interval_v5']),(155,['numerical_summary_v5'])]:
    tree=ast.parse(recovered_cell(i))
    for node in tree.body:
        if isinstance(node,ast.FunctionDef) and node.name in names:exec(compile(ast.Module(body=[node],type_ignores=[]),'<restored-stat-function>','exec'),globals())
for i in [160,162]:
    status=get_ipython().run_cell(recovered_cell(i),store_history=True)
    assert status.success
print('V6_RECOVERY_COMPLETE_CACHED_ORIGINALS_AND_SAVED_V5_RESULTS')


In [ ]:
# v6 as-of integration; fixed financial rule, explicit unknowns, no monthly-to-quarter extrapolation
def is_monthly_only_result_v6(a):
    s=unicodedata.normalize('NFKC',a.get('subject') or '')
    return bool(re.search(r'\d{3,4}年\s*\d{1,2}月(?:份)?.*(?:自結|損益)',s)) and not re.search(r'季|年度|全年',s)
def quarter_asof_v6(sid,cutoff):
    known=[r for r in quarter_by_stock_v6[sid] if r['year']==cutoff.year and datetime.datetime.fromisoformat(r['known_at'])<cutoff]
    cumulative={};single={};failures=[]
    for r in sorted(known,key=lambda r:r['known_at']):
        if r['basis']=='cumulative':cumulative[r['quarter']]=r
    for q,r in cumulative.items():
        if q==1:amounts=r['amounts'];sources=[r['ann_id']];stamps=[r['known_at']]
        elif q-1 in cumulative:
            amounts={f:r['amounts'][f]-cumulative[q-1]['amounts'][f] for f in ['revenue','operating']};sources=[cumulative[q-1]['ann_id'],r['ann_id']];stamps=[cumulative[q-1]['known_at'],r['known_at']]
        else:continue
        single[q]={'amounts':amounts,'source_ids':sources,'known_at':stamps,'latest':max(stamps),'basis':'formal_cumulative_difference'}
    for r in sorted(known,key=lambda r:r['known_at']):
        if r['basis']=='cumulative':continue
        q=r['quarter'];source_ids=[r['ann_id']];stamps=[r['known_at']]
        if r['basis']=='single_with_cumulative':
            previous=cumulative.get(q-1)
            if q>1 and previous is None:failures.append(r['ann_id']);continue
            expected={f:r['cumulative_amounts'][f]-r['amounts'][f] for f in ['revenue','operating']}
            consistent=all(abs(expected[f]-(previous['amounts'][f] if previous else 0))<=max(2.,abs(expected[f])*1e-6) for f in expected)
            if not consistent:failures.append(r['ann_id']);continue
            if previous:source_ids.insert(0,previous['ann_id']);stamps.insert(0,previous['known_at'])
        if q not in single or r['known_at']>single[q]['latest']:single[q]={'amounts':r['amounts'],'source_ids':source_ids,'known_at':stamps,'latest':max(stamps),'basis':r['basis']}
    qs=[q for q in [2,3] if q in single and q-1 in single]
    if not qs:return {'status':'unknown_quarter_or_prior_basis','source_ids':[],'consistency_failures':failures}
    q=max(qs);latest=single[q]['latest'];pending=[]
    parsed_ids={r['ann_id'] for r in known}
    for a in full_pool_ann_by_stock_v5[sid]:
        stamp=prestart_stamp(a)
        if stamp is None or not (datetime.datetime.fromisoformat(latest)<stamp<cutoff):continue
        kind=notice_kinds_v6[a['ann_id']]
        if kind=='financial_correction_unresolved' or (kind=='quarter_result_or_unresolved' and a['ann_id'] not in parsed_ids and not is_monthly_only_result_v6(a)):
            pending.append(a['ann_id'])
    if pending:return {'status':'newer_quarter_or_correction_unresolved','quarter':q,'source_ids':single[q]['source_ids'],'pending_ids':pending,'consistency_failures':failures}
    prior,current=single[q-1]['amounts'],single[q]['amounts']
    if prior['revenue']<=0 or current['revenue']<=0:return {'status':'invalid_quarter_revenue','source_ids':[]}
    ids=list(dict.fromkeys(single[q-1]['source_ids']+single[q]['source_ids']));stamps=list(dict.fromkeys(single[q-1]['known_at']+single[q]['known_at']))
    return {'status':'verified_quarter','quarter':q,'delta_pp':100*(current['operating']/current['revenue']-prior['operating']/prior['revenue']),'source_ids':ids,'known_at':stamps,'basis':single[q]['basis'],'prior_amounts':prior,'current_amounts':current,'consistency_failures':failures,'monthly_supplement_not_used_for_quarter_weights':True}
evidence_v6={};daily_coverage_v6=[];changed_evidence_v6=[]
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));counts=Counter()
    for sid in stock_ids:
        r=quarter_asof_v6(sid,cutoff);evidence_v6[(t,sid)]=r;counts[r['status']]+=1
        old=evidence_v5[(t,sid)]
        old_sign=0 if old['status']!='verified_formal_period' else 1 if old['delta_pp']>=1 else -1 if old['delta_pp']<=-1 else 0
        new_sign=0 if r['status']!='verified_quarter' else 1 if r['delta_pp']>=1 else -1 if r['delta_pp']<=-1 else 0
        if old_sign!=new_sign:changed_evidence_v6.append({'date':str(dates[t]),'stock_id':sid,'old_sign':old_sign,'new_sign':new_sign,'old_status':old['status'],'new_status':r['status'],'source_ids':r.get('source_ids',[])})
        if r['status']=='verified_quarter':assert all(datetime.datetime.fromisoformat(s)<cutoff for s in r['known_at'])
    daily_coverage_v6.append({'date':str(dates[t]),'status':dict(counts)})
initial_a25_comparison_v6=[]
for day in ['2024-10-29','2025-10-28']:
    t=di[datetime.date.fromisoformat(day)];selected=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day)
    for sid in selected:
        old=evidence_v5[(t,sid)];new=evidence_v6[(t,sid)]
        initial_a25_comparison_v6.append({'date':day,'stock_id':sid,'old_status':old['status'],'new_status':new['status'],'quarter':new.get('quarter'),'delta_pp':new.get('delta_pp'),'basis':new.get('basis'),'source_ids':new.get('source_ids',[]),'pending_ids':new.get('pending_ids',[])})
print('A25 source-status comparison',dict(Counter((r['date'],r['new_status']) for r in initial_a25_comparison_v6)))
print('2025 resolved former12',[r for r in initial_a25_comparison_v6 if r['date']=='2025-10-28' and r['old_status']=='newer_source_requires_period_or_correction_review'])
print('Changed sign decision-stock records',len(changed_evidence_v6),'stocks',len({r['stock_id'] for r in changed_evidence_v6}))
v6_source_export={'scope':'all150,673 decision dates, fixed Q2/Q1 Q3/Q2 protocol, monthly-only and non-income notices do not replace operating margin','quarter_records':quarter_records_v6,'daily_coverage':daily_coverage_v6,'initial_A25':initial_a25_comparison_v6,'changed_signal_records':changed_evidence_v6,'known_limits':['unresolved corrections still neutral','monthly net/pre-tax earnings not substituted for quarterly operating income','unparsed true quarterly results still neutral','not certified first-publication time']}
(recovery_root/'A策略_自結與公告期間整合_v6_2026-10-15.json').write_text(json.dumps(v6_source_export,ensure_ascii=False,indent=2),encoding='utf-8')
print('V6_ASOF_INTEGRATION_SAVED')


In [ ]:
# v6 global explicit-quarter extensions; metadata check before further calculation
import ast,sys
print('Kernel',sys.executable,'retained originals',len(full_pool_announcements_v5))
for a in full_pool_ann_by_stock_v5['3105']:
    if prestart_stamp(a) and datetime.datetime(2025,7,1)<prestart_stamp(a)<datetime.datetime(2025,11,1) and '自結' in a['subject']:
        print('3105 table header',a['ann_id'],unicodedata.normalize('NFKC',a.get('content') or '')[:1700])
old_explicit_quarter_v6=explicit_quarter_v6
def explicit_quarter_extended_v6(a):
    result=old_explicit_quarter_v6(a)
    if result:return result
    if original_kind_v6(a)!='quarter_result_or_unresolved':return None
    body=unicodedata.normalize('NFKC',a.get('content') or '')
    if not re.search(r'自結合併|合併自結',body):return None
    body=re.sub(r'(\d{3,4})年\s*(1|4|7|10)/1\s*[-~至]\s*(3|6|9|12)/(31|30)',lambda m:f'{m[1]}年{m[2]}-{m[3]}月' if (int(m[2]),int(m[3]),int(m[4])) in [(1,3,31),(4,6,30),(7,9,30),(10,12,31),(1,6,30),(1,9,30),(1,12,31)] else m[0],body)
    # Entire short table-header line explicitly contains single-quarter and cumulative columns.
    def quarter_header(m):
        y,q,y2,tail=m.groups();q=int(q)
        if y!=y2:return m[0]
        if (q==4 and tail=='度') or (q==2 and tail=='上半年度') or (q==3 and tail=='前三季') or tail in [f'前{q}季',f'1-{q*3}月']:
            return f'{y}年{q*3-2}-{q*3}月 {y2}年1-{q*3}月'
        return m[0]
    body=re.sub(r'(?m)^\s*(\d{3,4})年第([1-4])季\s+(\d{3,4})年(度|上半年度|前三季|前[1-4]季|1-\d+月)\s*$',quarter_header,body)
    # Normalize income-row labels only at the line start; no inference of currencies or units.
    body=re.sub(r'(?m)^(\s*)(?:合併)?營業收入(?:淨額)?\s*:?',r'\1合併營業收入淨額:',body)
    if re.search(r'(?m)^\s*(?:合併)?營業淨損\s*:?\s*\d',body):return None
    body=re.sub(r'(?m)^(\s*)(?:合併)?營業(?:淨利|淨損|利益)(\(損\))?\s*:?',r'\1合併營業淨利(損):',body)
    modified=dict(a,content=body)
    result=old_explicit_quarter_v6(modified)
    if result:result['verification']+='; explicit slash-date/quarter-column header and equivalent line-label normalization'
    return result
quarter_records_v6=[];quarter_by_stock_v6=defaultdict(list)
for a in full_pool_announcements_v5:
    r=explicit_quarter_extended_v6(a)
    if r:quarter_records_v6.append(r);quarter_by_stock_v6[r['stock_id']].append(r)
print('Global quarter types',dict(Counter(r['basis'] for r in quarter_records_v6)))
print('New single/cumulative source stocks',sorted({r['stock_id'] for r in quarter_records_v6 if r['basis']!='cumulative'}))
print('V6_EXTENDED_EXPLICIT_QUARTER_READY')


In [ ]:
# Supplementary monthly attention records: source units and latest correction, never annualize EPS
attention_monthly_v6=[]
for a in full_pool_announcements_v5:
    if notice_kinds_v6[a['ann_id']]!='attention_period_disclosure' or prestart_stamp(a) is None:continue
    b=unicodedata.normalize('NFKC',a.get('content') or '')
    pos=b.find('最近一月')
    if pos<0:continue
    piece=b[pos:]
    header=piece.split('營業收入',1)[0]
    month=re.search(r'(\d{3,4})年\s*(\d{1,2})月',header)
    if month is None:continue
    y,m=map(int,month.groups());y=y+1911 if y<1911 else y
    if not 1<=m<=12:continue
    # First income row after the month header always starts with current-month value.
    fields={}
    for name,label in [('revenue',r'營業收入'),('pretax',r'稅前淨利'),('parent_net',r'(?:歸屬(?:於)?母公司)?(?:業主)?(?:本期)?淨利|業主本期淨利|司業主淨利'),('eps',r'每股盈餘')]:
        for line in piece.splitlines():
            line=line.strip()
            if re.match(r'(?:'+label+r')',line) is None:continue
            tail=re.sub(r'^(?:'+label+r')','',line,count=1)
            tail=re.sub(r'^\s*\((?:百萬元?|仟元|元)\)','',tail).lstrip(' :')
            match=re.match(r'(\(?[+-]?\d[\d,]*(?:\.\d+)?\)?)',tail)
            if match:fields[name]=numeric_token_v6(match[1]);break
    if 'revenue' not in fields:continue
    unit=re.search(r'單位\s*:\s*(新(?:台|臺)幣)?\s*(佰萬元|百萬元|仟元|千元)',b)
    source_unit=unit[0] if unit else 'million; currency not explicit' if re.search(r'\(百萬(?:元)?\)',b) else 'unit_unverified'
    attention_monthly_v6.append({'stock_id':a['stock_id'],'year_month':f'{y:04d}-{m:02d}','known_at':str(prestart_stamp(a)),'ann_id':a['ann_id'],'subject':a['subject'],'values_as_printed':fields,'source_unit':source_unit,'is_correction':bool(re.search('更正',a['subject'])),'role':'monthly supplementary record only; not quarterly operating-margin score'})
attention_asof_initial_v6=[]
for day in ['2024-10-29','2025-10-28']:
    cutoff=datetime.datetime.combine(datetime.date.fromisoformat(day),datetime.time(8,55));ids=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day)
    for sid in ids:
        known=[r for r in attention_monthly_v6 if r['stock_id']==sid and datetime.datetime.fromisoformat(r['known_at'])<cutoff]
        if not known:continue
        month=max(r['year_month'] for r in known);candidates=[r for r in known if r['year_month']==month]
        latest=max(candidates,key=lambda r:r['known_at'])
        attention_asof_initial_v6.append(dict(latest,trade_date=day,superseded_source_ids=[r['ann_id'] for r in candidates if r['ann_id']!=latest['ann_id']]))
(recovery_root/'A策略_單月注意損益與更正查核_v6_2026-10-15.json').write_text(json.dumps({'all_monthly':attention_monthly_v6,'initial_A25_latest_month':attention_asof_initial_v6,'not_used_to_replace_quarter_operating_income':True},ensure_ascii=False,indent=2),encoding='utf-8')
print('Monthly attention parsed',len(attention_monthly_v6),'stocks',len({r['stock_id'] for r in attention_monthly_v6}))
print('Unit correction verified',[r for r in attention_monthly_v6 if r['ann_id']=='ea4e1a26474bc0932fd7d7720ab8112859026616'])
print('2025 initial latest monthly',[{k:r[k] for k in ['stock_id','year_month','source_unit','values_as_printed']} for r in attention_asof_initial_v6 if r['trade_date']=='2025-10-28'])
print('V6_MONTHLY_SUPPLEMENT_AND_CORRECTION_SAVED')


In [ ]:
def clean_v5(obj):
    if isinstance(obj,np.ndarray):return [clean_v5(x) for x in obj]
    if isinstance(obj,(np.integer,)):return int(obj)
    if isinstance(obj,(np.floating,float)):return float(obj) if np.isfinite(obj) else None
    if isinstance(obj,dict):return {k:clean_v5(v) for k,v in obj.items()}
    if isinstance(obj,(list,tuple)):return [clean_v5(v) for v in obj]
    return obj

# Freeze v6 data repair definition; retain v5 returns and rules, re-evaluate affected allocations only
definition_v6={'change':'source typing and explicit single/cumulative quarter integration; not new weights','selection':'same A25 every5days,24days,90pct,1000share,fees unchanged','weights':'existing1+.25sign with +/-1pp operating-margin threshold; existing combined .5mild+.25financial+.25revenue','timing':'all information frozen start08:55; no subsequent announcements used','unknown':'neutral multiplier1; unresolved corrections and quarterly originals remain neutral','monthly':'supplement only; never substitute net income or EPS for operating profit','quarters':'Q2/Q1 orQ3/Q2 in same fiscal year, unchanged','family':26,'family_reason':'v5 conservative24 plus2 updated source versions; no tuning to returns','LLM':'deferred by user','raw_snapshot_sha256':v4_saved['raw_snapshot_sha256'],'action_overlay_sha256':v4_saved['company_action_overlay_sha256']}
(recovery_root/'A策略_公告整合重驗證定義_v6_2026-10-15.json').write_text(json.dumps(definition_v6,ensure_ascii=False,indent=2),encoding='utf-8')
allocator_v5_retained=numerical_allocation_v5
def numerical_allocation_v6(start,t,selected,mode):
    if mode in ['equal','mild','revenue_prestart']:return allocator_v5_retained(start,t,selected,mode)
    if mode=='matched_neutral':return .5*prior_numerical_allocation_v5(start,t,selected,'mild')+.5*prior_numerical_allocation_v5(start,t,selected,'equal')
    assert mode in ['formal_prestart','combined_prestart']
    multipliers=[]
    for j in selected:
        r=evidence_v6[(start,stock_ids[j])];sign=0
        if r['status']=='verified_quarter':sign=1 if r['delta_pp']>=1 else -1 if r['delta_pp']<=-1 else 0
        multipliers.append(1+.25*sign)
    w=INVESTED*np.array(multipliers)/sum(multipliers)
    if mode=='combined_prestart':w=.5*allocator_v5_retained(start,t,selected,'mild')+.25*w+.25*allocator_v5_retained(start,t,selected,'revenue_prestart')
    assert np.isclose(w.sum(),.9) and len(w)==25 and max(w)<.1
    return w
simulator_src_v6=src.replace('def simulate_numeric_prestart_v5(', 'def simulate_numeric_prestart_v6(').replace('numerical_allocation_v5(', 'numerical_allocation_v6(')
exec(simulator_src_v6,globals())
results_v6={};stats_v6={};repair_change_v6={}
for name,mode in [('A_formal_operating_prestart','formal_prestart'),('A_combined_prestart','combined_prestart')]:
    rows=[simulate_numeric_prestart_v6(int(s),mode) for s in window_starts]
    old=numeric_prestart_results_v5[name]
    assert all(a['invalid']==b['invalid'] and a['violations']==b['violations'] for a,b in zip(rows,old))
    results_v6[name]=rows;stats_v6[name]=numerical_summary_v5(rows,v4_saved['windows']['A_equal'],family=26)
    paired=np.array([not r['invalid'] and r['violations']==0 for r in rows]);difference=np.array([a['return']-b['return'] for a,b in zip(rows,old)])
    repair_change_v6[name]={'changed_windows':int(np.sum(~np.isclose(difference,0,atol=1e-12,rtol=0))),'mean_change_vs_v5_pp':float(difference[paired].mean()*100)}
    print('V6_METHOD',name,json.dumps(stats_v6[name],ensure_ascii=False),'DATA_REPAIR_EFFECT',repair_change_v6[name],flush=True)
matched_v5_path=recovery_root/'A策略_資訊增量對照結果_v5_2026-10-14.json'
matched_saved_v5=json.loads(matched_v5_path.read_text(encoding='utf-8'))
matched_stats_v6=numerical_summary_v5(results_v6['A_combined_prestart'],matched_saved_v5['control_windows'],family=26)
matched_stats_v6['reference']='50pct mild+50pct equal, informational components neutral'
matched_stats_v6['reference_median_same_dates_pct']=matched_stats_v6.pop('A_median_same_dates_pct')
for r in matched_stats_v6['seasonal']:r['reference_return_pct']=r.pop('A_return_pct')
print('V6_MATCHED_INFORMATION_INCREMENT',json.dumps(matched_stats_v6,ensure_ascii=False))
export_v6={'definition':definition_v6,'results':clean_v5(results_v6),'stats':stats_v6,'data_repair_effect':repair_change_v6,'matched_information_diagnostic':matched_stats_v6,'supported':[n for n,r in stats_v6.items() if r['supported']]}
(recovery_root/'A策略_公告整合重驗證結果_v6_2026-10-15.json').write_text(json.dumps(export_v6,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('V6_RESULTS_SAVED',export_v6['supported'])


In [ ]:
# v6 as-of integration; fixed financial rule, explicit unknowns, no monthly-to-quarter extrapolation
def is_monthly_only_result_v6(a):
    s=unicodedata.normalize('NFKC',a.get('subject') or '')
    return bool(re.search(r'\d{3,4}年\s*\d{1,2}月(?:份)?.*(?:自結|損益)',s)) and not re.search(r'季|年度|全年',s)
def quarter_asof_v6(sid,cutoff):
    known=[r for r in quarter_by_stock_v6[sid] if r['year']==cutoff.year and datetime.datetime.fromisoformat(r['known_at'])<cutoff]
    cumulative={};single={};failures=[]
    for r in sorted(known,key=lambda r:r['known_at']):
        if r['basis']=='cumulative':cumulative[r['quarter']]=r
    for q,r in cumulative.items():
        if q==1:amounts=r['amounts'];sources=[r['ann_id']];stamps=[r['known_at']]
        elif q-1 in cumulative:
            amounts={f:r['amounts'][f]-cumulative[q-1]['amounts'][f] for f in ['revenue','operating']};sources=[cumulative[q-1]['ann_id'],r['ann_id']];stamps=[cumulative[q-1]['known_at'],r['known_at']]
        else:continue
        single[q]={'amounts':amounts,'source_ids':sources,'known_at':stamps,'latest':max(stamps),'basis':'formal_cumulative_difference'}
    for r in sorted(known,key=lambda r:r['known_at']):
        if r['basis']=='cumulative':continue
        q=r['quarter'];source_ids=[r['ann_id']];stamps=[r['known_at']]
        if r['basis']=='single_with_cumulative':
            previous=cumulative.get(q-1)
            if q>1 and previous is None:failures.append(r['ann_id']);continue
            expected={f:r['cumulative_amounts'][f]-r['amounts'][f] for f in ['revenue','operating']}
            consistent=all(abs(expected[f]-(previous['amounts'][f] if previous else 0))<=max(2.,abs(expected[f])*1e-6) for f in expected)
            if not consistent:failures.append(r['ann_id']);continue
            if previous:source_ids.insert(0,previous['ann_id']);stamps.insert(0,previous['known_at'])
        if q not in single or r['known_at']>single[q]['latest']:single[q]={'amounts':r['amounts'],'source_ids':source_ids,'known_at':stamps,'latest':max(stamps),'basis':r['basis']}
    qs=[q for q in [2,3] if q in single and q-1 in single]
    if not qs:return {'status':'unknown_quarter_or_prior_basis','source_ids':[],'consistency_failures':failures}
    q=max(qs);latest=single[q]['latest'];pending=[]
    parsed_ids={r['ann_id'] for r in known if r['ann_id'] not in failures}
    for a in full_pool_ann_by_stock_v5[sid]:
        stamp=prestart_stamp(a)
        if stamp is None or not (datetime.datetime.fromisoformat(latest)<stamp<cutoff):continue
        kind=notice_kinds_v6[a['ann_id']]
        if kind=='financial_correction_unresolved' or (kind=='quarter_result_or_unresolved' and a['ann_id'] not in parsed_ids and not is_monthly_only_result_v6(a)):
            pending.append(a['ann_id'])
    if pending:return {'status':'newer_quarter_or_correction_unresolved','quarter':q,'source_ids':single[q]['source_ids'],'pending_ids':pending,'consistency_failures':failures}
    prior,current=single[q-1]['amounts'],single[q]['amounts']
    if prior['revenue']<=0 or current['revenue']<=0:return {'status':'invalid_quarter_revenue','source_ids':[]}
    ids=list(dict.fromkeys(single[q-1]['source_ids']+single[q]['source_ids']));stamps=list(dict.fromkeys(single[q-1]['known_at']+single[q]['known_at']))
    return {'status':'verified_quarter','quarter':q,'delta_pp':100*(current['operating']/current['revenue']-prior['operating']/prior['revenue']),'source_ids':ids,'known_at':stamps,'basis':single[q]['basis'],'prior_amounts':prior,'current_amounts':current,'consistency_failures':failures,'monthly_supplement_not_used_for_quarter_weights':True}
evidence_v6={};daily_coverage_v6=[];changed_evidence_v6=[]
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));counts=Counter()
    for sid in stock_ids:
        r=quarter_asof_v6(sid,cutoff);evidence_v6[(t,sid)]=r;counts[r['status']]+=1
        old=evidence_v5[(t,sid)]
        old_sign=0 if old['status']!='verified_formal_period' else 1 if old['delta_pp']>=1 else -1 if old['delta_pp']<=-1 else 0
        new_sign=0 if r['status']!='verified_quarter' else 1 if r['delta_pp']>=1 else -1 if r['delta_pp']<=-1 else 0
        if old_sign!=new_sign:changed_evidence_v6.append({'date':str(dates[t]),'stock_id':sid,'old_sign':old_sign,'new_sign':new_sign,'old_status':old['status'],'new_status':r['status'],'source_ids':r.get('source_ids',[])})
        if r['status']=='verified_quarter':assert all(datetime.datetime.fromisoformat(s)<cutoff for s in r['known_at'])
    daily_coverage_v6.append({'date':str(dates[t]),'status':dict(counts)})
initial_a25_comparison_v6=[]
for day in ['2024-10-29','2025-10-28']:
    t=di[datetime.date.fromisoformat(day)];selected=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day)
    for sid in selected:
        old=evidence_v5[(t,sid)];new=evidence_v6[(t,sid)]
        initial_a25_comparison_v6.append({'date':day,'stock_id':sid,'old_status':old['status'],'new_status':new['status'],'quarter':new.get('quarter'),'delta_pp':new.get('delta_pp'),'basis':new.get('basis'),'source_ids':new.get('source_ids',[]),'pending_ids':new.get('pending_ids',[])})
print('A25 source-status comparison',dict(Counter((r['date'],r['new_status']) for r in initial_a25_comparison_v6)))
print('2025 resolved former12',[r for r in initial_a25_comparison_v6 if r['date']=='2025-10-28' and r['old_status']=='newer_source_requires_period_or_correction_review'])
print('Changed sign decision-stock records',len(changed_evidence_v6),'stocks',len({r['stock_id'] for r in changed_evidence_v6}))
v6_source_export={'scope':'all150,673 decision dates, fixed Q2/Q1 Q3/Q2 protocol, monthly-only and non-income notices do not replace operating margin','quarter_records':quarter_records_v6,'daily_coverage':daily_coverage_v6,'initial_A25':initial_a25_comparison_v6,'changed_signal_records':changed_evidence_v6,'known_limits':['unresolved corrections still neutral','monthly net/pre-tax earnings not substituted for quarterly operating income','unparsed true quarterly results still neutral','not certified first-publication time']}
(recovery_root/'A策略_自結與公告期間整合_v6_2026-10-15.json').write_text(json.dumps(v6_source_export,ensure_ascii=False,indent=2),encoding='utf-8')
print('V6_ASOF_INTEGRATION_SAVED')


In [ ]:
# v6 as-of integration; fixed financial rule, explicit unknowns, no monthly-to-quarter extrapolation
def is_monthly_only_result_v6(a):
    s=unicodedata.normalize('NFKC',a.get('subject') or '')
    return bool(re.search(r'\d{3,4}年\s*\d{1,2}月(?:份)?.*(?:自結|損益)',s)) and not re.search(r'季|年度|全年',s)
def quarter_asof_v6(sid,cutoff):
    known=[r for r in quarter_by_stock_v6[sid] if r['year']==cutoff.year and datetime.datetime.fromisoformat(r['known_at'])<cutoff]
    cumulative={};single={};failures=[]
    for r in sorted(known,key=lambda r:r['known_at']):
        if r['basis']=='cumulative':cumulative[r['quarter']]=r
    for q,r in cumulative.items():
        if q==1:amounts=r['amounts'];sources=[r['ann_id']];stamps=[r['known_at']]
        elif q-1 in cumulative:
            amounts={f:r['amounts'][f]-cumulative[q-1]['amounts'][f] for f in ['revenue','operating']};sources=[cumulative[q-1]['ann_id'],r['ann_id']];stamps=[cumulative[q-1]['known_at'],r['known_at']]
        else:continue
        single[q]={'amounts':amounts,'source_ids':sources,'known_at':stamps,'latest':max(stamps),'basis':'formal_cumulative_difference'}
    for r in sorted(known,key=lambda r:r['known_at']):
        if r['basis']=='cumulative':continue
        q=r['quarter'];source_ids=[r['ann_id']];stamps=[r['known_at']]
        if r['basis']=='single_with_cumulative':
            previous=cumulative.get(q-1)
            if q>1 and previous is None:failures.append(r['ann_id']);continue
            expected={f:r['cumulative_amounts'][f]-r['amounts'][f] for f in ['revenue','operating']}
            consistent=all(abs(expected[f]-(previous['amounts'][f] if previous else 0))<=max(2.,abs(expected[f])*1e-6) for f in expected)
            if not consistent:failures.append(r['ann_id']);continue
            if previous:source_ids.insert(0,previous['ann_id']);stamps.insert(0,previous['known_at'])
        if q not in single or r['known_at']>single[q]['latest']:single[q]={'amounts':r['amounts'],'source_ids':source_ids,'known_at':stamps,'latest':max(stamps),'basis':r['basis']}
    qs=[q for q in [2,3] if q in single and q-1 in single]
    if not qs:return {'status':'unknown_quarter_or_prior_basis','source_ids':[],'consistency_failures':failures}
    q=max(qs);latest=single[q]['latest'];pending=[]
    parsed_ids={r['ann_id'] for r in known if r['ann_id'] not in failures}
    for a in full_pool_ann_by_stock_v5[sid]:
        stamp=prestart_stamp(a)
        if stamp is None or not (datetime.datetime.fromisoformat(latest)<stamp<cutoff):continue
        kind=notice_kinds_v6[a['ann_id']]
        attention_op_unparsed=kind=='attention_period_disclosure' and bool(re.search(r'(?m)^\s*(?:合併)?營業(?:利益|淨利|淨損)',unicodedata.normalize('NFKC',a.get('content') or '')))
        if kind=='financial_correction_unresolved' or attention_op_unparsed or (kind=='quarter_result_or_unresolved' and a['ann_id'] not in parsed_ids and not is_monthly_only_result_v6(a)):
            pending.append(a['ann_id'])
    if pending:return {'status':'newer_quarter_or_correction_unresolved','quarter':q,'source_ids':single[q]['source_ids'],'pending_ids':pending,'consistency_failures':failures}
    prior,current=single[q-1]['amounts'],single[q]['amounts']
    if prior['revenue']<=0 or current['revenue']<=0:return {'status':'invalid_quarter_revenue','source_ids':[]}
    ids=list(dict.fromkeys(single[q-1]['source_ids']+single[q]['source_ids']));stamps=list(dict.fromkeys(single[q-1]['known_at']+single[q]['known_at']))
    return {'status':'verified_quarter','quarter':q,'delta_pp':100*(current['operating']/current['revenue']-prior['operating']/prior['revenue']),'source_ids':ids,'known_at':stamps,'basis':single[q]['basis'],'prior_amounts':prior,'current_amounts':current,'consistency_failures':failures,'monthly_supplement_not_used_for_quarter_weights':True}
evidence_v6={};daily_coverage_v6=[];changed_evidence_v6=[]
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));counts=Counter()
    for sid in stock_ids:
        r=quarter_asof_v6(sid,cutoff);evidence_v6[(t,sid)]=r;counts[r['status']]+=1
        old=evidence_v5[(t,sid)]
        old_sign=0 if old['status']!='verified_formal_period' else 1 if old['delta_pp']>=1 else -1 if old['delta_pp']<=-1 else 0
        new_sign=0 if r['status']!='verified_quarter' else 1 if r['delta_pp']>=1 else -1 if r['delta_pp']<=-1 else 0
        if old_sign!=new_sign:changed_evidence_v6.append({'date':str(dates[t]),'stock_id':sid,'old_sign':old_sign,'new_sign':new_sign,'old_status':old['status'],'new_status':r['status'],'source_ids':r.get('source_ids',[])})
        if r['status']=='verified_quarter':assert all(datetime.datetime.fromisoformat(s)<cutoff for s in r['known_at'])
    daily_coverage_v6.append({'date':str(dates[t]),'status':dict(counts)})
initial_a25_comparison_v6=[]
for day in ['2024-10-29','2025-10-28']:
    t=di[datetime.date.fromisoformat(day)];selected=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day)
    for sid in selected:
        old=evidence_v5[(t,sid)];new=evidence_v6[(t,sid)]
        initial_a25_comparison_v6.append({'date':day,'stock_id':sid,'old_status':old['status'],'new_status':new['status'],'quarter':new.get('quarter'),'delta_pp':new.get('delta_pp'),'basis':new.get('basis'),'source_ids':new.get('source_ids',[]),'pending_ids':new.get('pending_ids',[])})
print('A25 source-status comparison',dict(Counter((r['date'],r['new_status']) for r in initial_a25_comparison_v6)))
print('2025 resolved former12',[r for r in initial_a25_comparison_v6 if r['date']=='2025-10-28' and r['old_status']=='newer_source_requires_period_or_correction_review'])
print('Changed sign decision-stock records',len(changed_evidence_v6),'stocks',len({r['stock_id'] for r in changed_evidence_v6}))
v6_source_export={'scope':'all150,673 decision dates, fixed Q2/Q1 Q3/Q2 protocol, monthly-only and non-income notices do not replace operating margin','quarter_records':quarter_records_v6,'daily_coverage':daily_coverage_v6,'initial_A25':initial_a25_comparison_v6,'changed_signal_records':changed_evidence_v6,'known_limits':['unresolved corrections still neutral','monthly net/pre-tax earnings not substituted for quarterly operating income','unparsed true quarterly results still neutral','not certified first-publication time']}
(recovery_root/'A策略_自結與公告期間整合_v6_2026-10-15.json').write_text(json.dumps(v6_source_export,ensure_ascii=False,indent=2),encoding='utf-8')
print('V6_ASOF_INTEGRATION_SAVED')


In [ ]:
# v6 persist already computed results, comparative chart, source references and 25-stock target audit
tree=ast.parse(''.join(saved_nb['cells'][152]['source']))
for n in tree.body:
    if isinstance(n,ast.FunctionDef) and n.name=='clean_v5':exec(compile(ast.Module(body=[n],type_ignores=[]),'<json-converter>','exec'),globals())
export_v6={'definition':definition_v6,'results':clean_v5(results_v6),'stats':stats_v6,'data_repair_effect':repair_change_v6,'matched_information_diagnostic':matched_stats_v6,'supported':[n for n,r in stats_v6.items() if r['supported']]}
(recovery_root/'A策略_公告整合重驗證結果_v6_2026-10-15.json').write_text(json.dumps(export_v6,ensure_ascii=False,allow_nan=False),encoding='utf-8')
stock_target_audit_v6=[]
for day in ['2024-10-29','2025-10-28']:
    t=di[datetime.date.fromisoformat(day)];ids=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day);selected=np.array([si[s] for s in ids])
    financial_weights=numerical_allocation_v6(t,t,selected,'formal_prestart');combined_weights=numerical_allocation_v6(t,t,selected,'combined_prestart')
    for rank,(sid,wf,wc) in enumerate(zip(ids,financial_weights,combined_weights),1):
        r=evidence_v6[(t,sid)]
        stock_target_audit_v6.append({'trade_date':day,'stock_id':sid,'A_rank':rank,'information_status':r['status'],'quarter':r.get('quarter'),'margin_change_pp':r.get('delta_pp'),'financial_weight':float(wf),'combined_weight':float(wc),'source_ids':r.get('source_ids',[]),'pending_ids':r.get('pending_ids',[]),'not_actual_trade':True})
(recovery_root/'A策略_歷年開賽25檔權重與證據_v6_2026-10-15.json').write_text(json.dumps(stock_target_audit_v6,ensure_ascii=False,indent=2),encoding='utf-8')
for r in attention_asof_initial_v6:
    r['available_age_days']=(datetime.datetime.combine(datetime.date.fromisoformat(r['trade_date']),datetime.time(8,55))-datetime.datetime.fromisoformat(r['known_at'])).total_seconds()/86400
    r['stale_reporting_year']=int(r['year_month'][:4])<int(r['trade_date'][:4])
(recovery_root/'A策略_單月注意損益與更正查核_v6_2026-10-15.json').write_text(json.dumps({'all_monthly':attention_monthly_v6,'initial_A25_latest_known_month':attention_asof_initial_v6,'not_used_to_replace_quarter_operating_income':True,'freshness':'latest known retained source may be stale; period and age explicitly retained'},ensure_ascii=False,indent=2),encoding='utf-8')
chart=(recovery_root/'A策略_開賽前資訊調權比較_v5_2026-10-14.html').read_text(encoding='utf-8')
chart_methods=[('A等權',v4_saved['windows']['A_equal']),('A溫和動能',v4_saved['windows']['A_mild']),('A_formal_operating_prestart',results_v6['A_formal_operating_prestart']),('A_revenue_prestart',numeric_prestart_results_v5['A_revenue_prestart']),('A_combined_prestart',results_v6['A_combined_prestart'])]
plot_rows=[]
for name,rows in chart_methods:
    for i,r in enumerate(rows):
        if not r['invalid'] and r['violations']==0:plot_rows.append({'method':name,'start':str(dates[int(window_starts[i])]),'end':str(dates[int(window_starts[i])+23]),'return_pct':r['return']*100,'cost_pct':r['cost']*100,'mdd_pct':r['mdd']*100})
begin=chart.index('const data=');end=chart.index(';const labels=',begin)
chart=chart[:begin]+'const data='+json.dumps(plot_rows,ensure_ascii=False)+chart[end:]
chart=chart.replace('A開賽前資訊調權比較','A開賽前資訊調權比較 v6').replace('未含完整自結、法說與LLM判讀。','已納入可核對的季度自結與公告分類修正；單月損益另列，部分更正及季度原文仍未解析，LLM未啟用。')
(recovery_root/'A策略_開賽前資訊調權比較_v6_2026-10-15.html').write_text(chart,encoding='utf-8')
print('V6_RESULTS_TARGETS_AND_ALL_POINTS_SAVED',len(plot_rows),'weights rows',len(stock_target_audit_v6))


In [ ]:
# Supplementary monthly attention records: source units and latest correction, never annualize EPS
attention_monthly_v6=[]
for a in full_pool_announcements_v5:
    if notice_kinds_v6[a['ann_id']]!='attention_period_disclosure' or prestart_stamp(a) is None:continue
    b=unicodedata.normalize('NFKC',a.get('content') or '')
    pos=b.find('最近一月')
    if pos<0:continue
    piece=b[pos:]
    header=piece.split('營業收入',1)[0]
    month=re.search(r'(\d{3,4})年\s*(\d{1,2})月',header)
    if month is None:continue
    y,m=map(int,month.groups());y=y+1911 if y<1911 else y
    if not 1<=m<=12:continue
    # First income row after the month header always starts with current-month value.
    fields={};revenue_value_end=None
    for name,label in [('revenue',r'營業收入'),('pretax',r'稅前淨利'),('parent_net',r'(?:歸屬(?:於)?母公司)?(?:業主)?(?:本期)?淨利|業主本期淨利|司業主淨利'),('eps',r'每股盈餘')]:
        for line in piece.splitlines():
            line=line.strip()
            if re.match(r'(?:'+label+r')',line) is None:continue
            tail=re.sub(r'^(?:'+label+r')','',line,count=1)
            tail=re.sub(r'^\s*\((?:百萬元?|仟元|元)\)','',tail).lstrip(' :')
            match=re.match(r'(\(?[+-]?\d[\d,]*(?:\.\d+)?\)?)',tail)
            if match:
                numeric_end=line.find(match[1])+len(match[1])
                if name!='revenue' and revenue_value_end is not None and numeric_end>revenue_value_end+8:continue
                fields[name]=numeric_token_v6(match[1])
                if name=='revenue':revenue_value_end=numeric_end
                break
    if 'revenue' not in fields:continue
    unit=re.search(r'單位\s*:\s*(新(?:台|臺)幣)?\s*(佰萬元|百萬元|仟元|千元)',b)
    source_unit=unit[0] if unit else 'million; currency not explicit' if re.search(r'\(百萬(?:元)?\)',b) else 'unit_unverified'
    attention_monthly_v6.append({'stock_id':a['stock_id'],'year_month':f'{y:04d}-{m:02d}','known_at':str(prestart_stamp(a)),'ann_id':a['ann_id'],'subject':a['subject'],'values_as_printed':fields,'source_unit':source_unit,'is_correction':bool(re.search('更正',a['subject'])),'role':'monthly supplementary record only; not quarterly operating-margin score'})
attention_asof_initial_v6=[]
for day in ['2024-10-29','2025-10-28']:
    cutoff=datetime.datetime.combine(datetime.date.fromisoformat(day),datetime.time(8,55));ids=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day)
    for sid in ids:
        known=[r for r in attention_monthly_v6 if r['stock_id']==sid and datetime.datetime.fromisoformat(r['known_at'])<cutoff]
        if not known:continue
        month=max(r['year_month'] for r in known);candidates=[r for r in known if r['year_month']==month]
        latest=max(candidates,key=lambda r:r['known_at'])
        attention_asof_initial_v6.append(dict(latest,trade_date=day,superseded_source_ids=[r['ann_id'] for r in candidates if r['ann_id']!=latest['ann_id']]))
(recovery_root/'A策略_單月注意損益與更正查核_v6_2026-10-15.json').write_text(json.dumps({'all_monthly':attention_monthly_v6,'initial_A25_latest_month':attention_asof_initial_v6,'not_used_to_replace_quarter_operating_income':True},ensure_ascii=False,indent=2),encoding='utf-8')
print('Monthly attention parsed',len(attention_monthly_v6),'stocks',len({r['stock_id'] for r in attention_monthly_v6}))
print('Unit correction verified',[r for r in attention_monthly_v6 if r['ann_id']=='ea4e1a26474bc0932fd7d7720ab8112859026616'])
print('2025 initial latest monthly',[{k:r[k] for k in ['stock_id','year_month','source_unit','values_as_printed']} for r in attention_asof_initial_v6 if r['trade_date']=='2025-10-28'])
print('V6_MONTHLY_SUPPLEMENT_AND_CORRECTION_SAVED')


In [ ]:
# v6 complete finite source-repair comparison and record unresolved gates
names={r['stock_id']:r['name'] for r in latest_a25_cache['names_snapshot']}
lines=['# A公告期間與自結整合重驗證 v6','',
'日期：2026-10-15。沿用150檔、2023-10-02～2026-10-12原價量快照與42筆公司行動修正。未重新查DB、未啟用LLM調權、未提交交易。','',
'## 結論','',
'已將較新季度自結、非業績公告分類與更正防護一致套用到150檔／673個歷史決策日，兩種受影響配置各保存650窗口，與A共同535個有效窗口。每種配置有219個窗口結果改變；原A與未受影響營收配置直接沿用。權重、選股、費用及24日／五日調整規則未變。',
'','|方法|相對A平均費用後差（百分點）|相對v5改變（百分點）|相對A溫和動能（百分點）|判斷|','|---|---:|---:|---:|---|']
labels={'A_formal_operating_prestart':'季度本業資訊調權','A_combined_prestart':'動能＋財務＋營收組合'}
for n,s in stats_v6.items():lines.append(f"|{labels[n]}|{s['mean_difference_pp']:+.4f}|{repair_change_v6[n]['mean_change_vs_v5_pp']:+.4f}|{s['other_controls']['A_mild']['mean_difference_pp']:+.4f}|未證實改良|")
lines+=['',f"相同50%動能比重的對照下，資訊增量平均為{matched_stats_v6['mean_difference_pp']:+.4f}百分點；24區塊家族校正區間{matched_stats_v6['block24_interval_pp']}、48區塊{matched_stats_v6['block48_interval_pp']}均跨零。兩個新版本與資訊增量2026均為負。不能因修正後有兩年同期正向就宣稱有效。",'',
'統計沿用費用後配對報酬、原窗口位置的24／48日區塊各5000次；保守計入先前24項加本次兩個資料修正版共26項，家族5%。已看過的歷史不是未見測試集。風險門檻通過不等於報酬改良通過。未強選勝者，也未用結果尋找新門檻或倍率。','',
'## 資料修正具體內容','',
'2025開賽25檔可核對季度本業訊號由12增至23：原12檔待核對中11檔恢復訊號；旺矽6223因財報附註薪資成本更正仍保留中性，旺宏2337缺有效正式季度基礎，仍未知中性。2024同期由16增至21，另4檔仍有較新季度／更正未解。這是季度本業覆蓋，不是全部最新財務資訊已讀完。',
'全池解析1530筆正式累計季度、2筆明列單季敘述及29筆單季／累計自結表；自結表必須和當時已公開前季累計數一致，不一致者不能繼續當作已整合來源。正式原文中的代理子公司、背書保證、資金貸與、純月營收與未來法說預告分開，不誤當成新季度本業資料。',
'南亞科2408：2025-10-13 16:32:51 Q3法說摘要明列單季自結營收187.79億元、營業淨利11.19億元，換算後才和Q2單季比較；營益率差約+48.72百分點。這是未經查核自結，不等同正式完整財報。',
'力積電6770：2025-10-21 Q3單季／累計營收與營業損益可和Q2累計交叉核對，營益率差約−7.64百分點；9/19注意損益單位更正由仟元改百萬元，保留更正前後ID與時間，不將錯單位延用。',
'單月注意損益另存199筆、60檔來源。群聯9月原文843百萬元淨利與EPS4.06等是單月資料，不拿它冒充Q3營業利益、不乘三／四估季度／全年。原文未明列幣別者不自行假定；無法可靠對齊欄位不硬填。最新保留來源也可能是前一年資料，附表保留月份與已知時間／年齡，不能誤稱新鮮。','',
'## 兩年比賽同期（各24交易日）','',
'|方法|2024-10-29起|2025-10-28起|','|---|---:|---:|']
for n,s in stats_v6.items():lines.append('|'+labels[n]+'|'+'|'.join(f"{r['return_pct']:.4f}%（相對A{r['difference_pp']:+.4f}pp）" for r in s['seasonal'])+'|')
lines+=['','只有兩年，不足稱穩定季節性或公告策略證據。','',
'## 2025開賽25檔目標權重與訊號','',
'此表是歷史研究配置、非2026實際建議或已下單；總投入90%，目標股數仍須以當時前日價格與可交易資金機械換算。訊號以開賽前08:55固定，季度本業權重使用既有±1pp門檻與1±0.25倍率，再正規化；組合為50%溫和動能＋25%本業＋25%營收。不因幅度大就無限加碼。',
'','|A排名|股票|季度|營益率差pp|本業目標權重|組合目標權重|資料狀態|','|---:|---|---:|---:|---:|---:|---|']
for r in stock_target_audit_v6:
    if r['trade_date']!='2025-10-28':continue
    d='—' if r['margin_change_pp'] is None else f"{r['margin_change_pp']:+.2f}"
    status={'verified_quarter':'可核對','newer_quarter_or_correction_unresolved':'更正／新季未解，中性','unknown_quarter_or_prior_basis':'缺季度基礎，中性'}.get(r['information_status'],r['information_status'])
    lines.append(f"|{r['A_rank']}|{r['stock_id']} {names.get(r['stock_id'],'')}|{r['quarter'] or '—'}|{d}|{r['financial_weight']:.3%}|{r['combined_weight']:.3%}|{status}|")
lines+=['','## 保留限制與停止位置','',
'原文格式仍有未解析季度，財報更正不因只改附註就假設不影響本業，缺首次公開時間也不造時間。沒有全池完整財報附註／公司IR歷史的完整性認證；Q4／跨年Q1未新增到原固定訊號。單月淨利與非經常項目僅作附加紀錄，未新設因子或風險權重。',
'12筆現金增資、1筆價格異常、股利實際到帳日、官方成交／整批作廢與Active Share等原上場限制仍存在。LLM依使用者要求暫緩；團隊TEAM_11693，目前主辦方尚無DataPack可下載，正式帳戶演練等待開放。沒有交易或官方交件。',
'本輪有限的來源修正、受影響重算、配對統計與全部點圖表已完成；留下明確未知，不把「未證實改良」改稱有效。排程維持暫停，保留kernel；停止原因不是20分鐘期限。','',
'## 成果','',
'[五方法全部有效點互動箱形圖](A策略_開賽前資訊調權比較_v6_2026-10-15.html)：2676點與同季10點、固定方法順序與座標、桃紅中位數、0%線、單一懸停日期／報酬／費用。',
'[逐窗口與統計](A策略_公告整合重驗證結果_v6_2026-10-15.json)、[來源期間與變更](A策略_自結與公告期間整合_v6_2026-10-15.json)、[兩年全部25檔權重及證據ID](A策略_歷年開賽25檔權重與證據_v6_2026-10-15.json)、[單月與更正附表](A策略_單月注意損益與更正查核_v6_2026-10-15.json)。原Notebook保存程式與輸出，未重啟執行全部。']
report_path_v6=recovery_root/'A策略_公告期間與自結整合研究_v6_2026-10-15.md'
report_path_v6.write_text('\n'.join(lines)+'\n',encoding='utf-8')
state_path=recovery_root/'A策略_無人值守研究狀態.json';state=json.loads(state_path.read_text(encoding='utf-8'))
state['announcement_integration_v6']={'status':'completed_finite_source_repair_with_unresolved_gates','date':'2026-10-15','A25_2025_verified':23,'A25_2025_neutral_unresolved':['6223','2337'],'full_scope':'150stocks,673decision dates','changed_sign_records':len(changed_evidence_v6),'updated_methods':list(stats_v6),'supported':export_v6['supported'],'report':report_path_v6.name,'chart':'A策略_開賽前資訊調權比較_v6_2026-10-15.html','LLM':'deferred_by_user','DataPack':'not_yet_downloadable_per_user','notebook_cells':[159,160,161,163,165,169,166,167,170],'running_work':False,'remaining':'unparsed true quarters/corrections stay neutral; official execution waits DataPack'}
state['current_node']=None;state['status']='v6_finite_announcement_repair_completed_unproven_improvement_LLM_deferred_DataPack_unavailable';state['automation_status']='PAUSED'
state['updated_at']=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=8))).isoformat()
state['last_completed_action']='Completed uniform v6 quarter self/report type repair and fixed-weight revalidation; two methods650 each535pairs, no supported improvement; A25 coverage23/25. Monthly and correction audit retained, noLLM/no trades.'
for path in [report_path_v6,recovery_root/'A策略_公告整合重驗證結果_v6_2026-10-15.json',recovery_root/'A策略_開賽前資訊調權比較_v6_2026-10-15.html']:
    if path.name not in state['final_artifacts']:state['final_artifacts'].append(path.name)
state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
workflow=recovery_root/'A策略_無人值守研究流程.md';text=workflow.read_text(encoding='utf-8')
note='## 最新v6：季度自結與公告分類修正完成（2026-10-15）\n\n[完整報告、兩年25檔權重與所有點](A策略_公告期間與自結整合研究_v6_2026-10-15.md)。全150檔／673歷史日一致處理，2025 A25可核對季度本業由12增至23；6223更正及2337基礎不足保持中性。兩種受影響配置各650窗口、535配對，均有219窗口改變；固定原權重／費用，26家族校正未證實改良。相同動能比重對照之資訊增量約+0.0049pp，區間跨零。\n\n單月注意損益及單位更正另存附表，不冒充季度本業資料。未解析季度／更正仍明列未知。LLM依使用者要求暫緩；TEAM_11693等待主辦方開放DataPack。此輪有限修正與可行統計／圖表完成，PAUSED，kernel保留；下文v5與早期「最新」保留作歷史。\n\n'
if '## 最新v6：' not in text:workflow.write_text(text.split('\n',1)[0]+'\n\n'+note+text.split('\n',1)[1],encoding='utf-8')
print('V6_REPORT_AND_STATE_SAVED',report_path_v6.name)


In [ ]:
# v6 independent boundary and unchanged-execution checks
before=quarter_asof_v6('2408',datetime.datetime(2025,10,13,8,55))
after=quarter_asof_v6('2408',datetime.datetime(2025,10,14,8,55))
assert before.get('quarter')==2 and after.get('quarter')==3
assert '9d4f234378a373ac94b74ced64d48e4bcaf299f3' not in before.get('source_ids',[])
assert '9d4f234378a373ac94b74ced64d48e4bcaf299f3' in after['source_ids']
assert after['current_amounts']['revenue']==18779000 and after['current_amounts']['operating']==1119000
assert evidence_v6[(di[datetime.date(2025,10,28)],'6223')]['status']=='newer_quarter_or_correction_unresolved'
assert evidence_v6[(di[datetime.date(2025,10,28)],'2337')]['status']=='unknown_quarter_or_prior_basis'
assert notice_kinds_v6['ad68a2386e7924ca45f9446078aae9180b0bbc24']=='financing_not_income_result'
pos_checks=sorted(set([0,len(window_starts)-1]+[int(np.flatnonzero(window_starts==di[datetime.date.fromisoformat(d)] )[0]) for d in ['2024-10-29','2025-10-28']]))
for i in pos_checks:
    result=simulate_numeric_prestart_v6(int(window_starts[i]),'equal');old=v4_saved['windows']['A_equal'][i]
    assert result['invalid']==old['invalid']
    for field in ['return','mdd','cost','turnover']:assert np.isclose(result[field],old[field],rtol=1e-12,atol=1e-12)
for day in ['2024-10-29','2025-10-28']:
    rows=[r for r in stock_target_audit_v6 if r['trade_date']==day]
    assert len(rows)==25 and len({r['stock_id'] for r in rows})==25
    for field in ['financial_weight','combined_weight']:assert np.isclose(sum(r[field] for r in rows),.9) and max(r[field] for r in rows)<.1
verification_v6={'nanya_release_boundary':'passed before08:55 Oct13 excludes Q3, Oct14 includes source quantities','unresolved6223_and2337':'retained_neutral','financing_misclassification':'repaired','unchanged_execution_controls_checked':len(pos_checks),'source_known_at_before_cutoff':'verified for every accepted decision record','weight25_sum90_cap10':'both anchors passed','LLM_evaluated':False,'submitted_trades':False,'restart_and_run_all':False}
(recovery_root/'A策略_公告整合驗證_v6_2026-10-15.json').write_text(json.dumps(verification_v6,ensure_ascii=False,indent=2),encoding='utf-8')
print('V6_VALIDATION_PASSED',json.dumps(verification_v6,ensure_ascii=False))


In [ ]:
# D-Plan offline gate: environment, schema, and officially marked demonstration examples
import sys,json,datetime,hashlib,copy,math,re,importlib.metadata
from research_layout import ResearchPath as Path
print('Kernel',sys.executable)
print('Existing useful state',[(n,type(globals().get(n)).__name__) for n in ['recovery_root','dates','stats_v6']])
print('jsonschema installed version',importlib.metadata.version('jsonschema'))
from jsonschema import Draft202012Validator,FormatChecker
dplan_root=Path(r'C:\Users\weiha\Desktop\文件\03_課程資料\115-1\資料探勘與應用\B_Final Project (0920~1221)\競賽資料')
dplan_schema_path=dplan_root/'官方文件/D-Plan.schema_v4.3.json'
dplan_schema=json.loads(dplan_schema_path.read_text(encoding='utf-8'))
Draft202012Validator.check_schema(dplan_schema)
dplan_example=json.loads((dplan_root/'官方文件/D-Plan_TEAM_042_2026-10-28.json').read_text(encoding='utf-8'))
print('Official example schema errors',[list(e.absolute_path) for e in Draft202012Validator(dplan_schema,format_checker=FormatChecker()).iter_errors(dplan_example)])
print('Example metadata warning',dplan_example.get('_NOTE'))
print('Prior holdings example count',len(dplan_example['no_trade_decisions']),'decisions',len(dplan_example['decisions']),'orders',len(dplan_example['orders']))
print('DPLAN_SCHEMA_ENVIRONMENT_READY')


In [ ]:
# D-Plan local validation prototype; no network calls, no submission, no invented live account
from decimal import Decimal,ROUND_FLOOR
def dnum(value):
    if isinstance(value,bool) or not isinstance(value,(int,float,Decimal)) or not math.isfinite(float(value)):raise ValueError('finite_number_required')
    return Decimal(str(value))
def dtime(value):
    t=datetime.datetime.fromisoformat(value)
    if t.utcoffset()!=datetime.timedelta(hours=8):raise ValueError('Taipei_offset_required')
    return t
def validate_dplan_local(plan,*,team_id='TEAM_11693',trade_date=None,cutoff=None,datapack=None,previous_trade_date=None,first_day=False,previous_closes=None,universe=None):
    errors=[];pending=['official_content_and_strategy_review','official_ActiveShare_and_grace_period_review','official_execution_mean_and_batch_result']
    def add(code,path=''):errors.append({'code':code,'path':path})
    try:json.dumps(plan,allow_nan=False)
    except (ValueError,TypeError):add('non_finite_or_non_JSON')
    schema_errors=list(Draft202012Validator(dplan_schema,format_checker=FormatChecker()).iter_errors(plan))
    for e in schema_errors[:50]:add('schema','/'.join(map(str,e.absolute_path)))
    if schema_errors or not isinstance(plan,dict):return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if plan['schema_version']!='4.3':add('unsupported_DPlan_version','schema_version')
    if plan['team_id']!=team_id:add('wrong_team','team_id')
    if trade_date is None:pending.append('official_trade_date')
    elif plan['trade_date']!=trade_date:add('wrong_trade_date','trade_date')
    if cutoff is None:pending.append('trusted_receipt_cutoff')
    else:
        try:
            deadline=dtime(cutoff)
            if deadline.date().isoformat()!=plan['trade_date']:add('cutoff_date_mismatch')
            for i,s in enumerate(plan['sources']):
                stamps={k:dtime(s[k]) for k in ['published_at','content_as_of','fetched_at'] if k in s}
                if any(t>deadline for t in stamps.values()):add('source_after_cutoff',f'sources/{i}')
                if 'published_at' in stamps and 'fetched_at' in stamps and stamps['published_at']>stamps['fetched_at']:add('source_fetch_before_publication',f'sources/{i}')
            meta=plan['agent_metadata'];start=dtime(meta['run_started_at']);end=dtime(meta['run_completed_at'])
            if not start<=end<=deadline:add('invalid_agent_run_time','agent_metadata')
        except (ValueError,TypeError):add('invalid_Taipei_timestamp')
    tables={};prefixes={'sources':('source_id','S'),'observations':('obs_id','O'),'inferences':('inf_id','I'),'decisions':('decision_id','D')}
    for section,(key,prefix) in prefixes.items():
        ids=[r[key] for r in plan[section]]
        if len(ids)!=len(set(ids)):add('duplicate_id',section)
        if set(ids)!={prefix+str(i+1) for i in range(len(ids))}:add('non_contiguous_ids',section)
        tables[section]={r[key]:r for r in plan[section]}
    def refs(rows,key,target,label):
        for i,row in enumerate(rows):
            for ref in row.get(key,[]):
                if ref not in tables[target]:add('broken_reference',f'{label}/{i}/{key}')
    refs(plan['observations'],'source_ref','sources','observations');refs(plan['inferences'],'premise_refs','observations','inferences')
    refs([plan['market_view']],'basis_refs','observations','market_view');refs(plan['decisions'],'inference_refs','inferences','decisions');refs(plan['no_trade_decisions'],'reason_refs','inferences','no_trade_decisions')
    decision_by_ticker={}
    for i,d in enumerate(plan['decisions']):
        if d['ticker'] in decision_by_ticker:add('duplicate_ticker_decision',d['ticker'])
        decision_by_ticker[d['ticker']]=d
        cap=Decimal('.25') if d['ticker']=='2330' else Decimal('.10')
        if dnum(d['target_weight'])>cap:add('target_cap_exceeded',d['ticker'])
        if d['action']=='SELL_ALL' and dnum(d['target_weight'])!=0:add('sell_all_target_not_zero',d['ticker'])
        if 'funding_for' in d:
            if d['action'] not in ['TRIM','SELL_ALL']:add('funding_must_be_sell',d['ticker'])
            for ref in d['funding_for']:
                if ref not in tables['decisions'] or tables['decisions'][ref]['action'] not in ['BUY','ADD']:add('invalid_funding_reference',d['ticker'])
    held_no_trade=[r['ticker'] for r in plan['no_trade_decisions']]
    if len(held_no_trade)!=len(set(held_no_trade)):add('duplicate_no_trade_ticker')
    if set(held_no_trade)&set(decision_by_ticker):add('trade_no_trade_overlap')
    seen_orders=set();orders_by_decision={}
    for i,o in enumerate(plan['orders']):
        if o['ticker'] in seen_orders:add('duplicate_order_ticker',o['ticker'])
        seen_orders.add(o['ticker']);ref=o['decision_ref']
        if ref not in tables['decisions']:add('broken_order_reference',str(i));continue
        d=tables['decisions'][ref]
        if o['ticker']!=d['ticker']:add('order_ticker_mismatch',str(i))
        if o['side']!=('BUY' if d['action'] in ['BUY','ADD'] else 'SELL'):add('order_side_mismatch',str(i))
        if ref in orders_by_decision:add('duplicate_order_decision',ref)
        orders_by_decision[ref]=o
    if universe is None:pending.append('official_universe')
    elif (set(decision_by_ticker)|set(held_no_trade)|seen_orders)-set(universe):add('ticker_outside_universe')
    cash_range=plan['market_view']['posture']['target_cash_pct_range']
    if cash_range[0]>cash_range[1]:add('cash_range_reversed')
    if datapack is None:
        pending.append('official_DataPack')
        return {'status':'REJECTED' if errors else 'WAITING_DATAPACK','errors':errors,'pending':pending,'ready_to_submit':False}
    if not isinstance(datapack,dict):add('invalid_DataPack_type');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if datapack.get('doc_type')!='DataPack':add('wrong_DataPack_doc_type')
    if datapack.get('team_id')!=team_id:add('DataPack_wrong_team')
    if datapack.get('trade_date')!=plan['trade_date']:add('DataPack_wrong_trade_date')
    # Archived document4.3 examples use DataPack3.0. Do not assume DataPack4.3.
    if datapack.get('schema_version')!='3.0':pending.append('verify_new_DataPack_schema_version')
    if cutoff is not None:
        try:
            if dtime(datapack['generated_at'])>deadline:add('DataPack_generated_after_cutoff')
        except (ValueError,KeyError,TypeError):add('DataPack_invalid_generated_at')
    missing=datapack.get('missing')
    if not isinstance(missing,list):add('DataPack_missing_list_invalid');missing=['unknown']
    if missing or datapack.get('account') is None or datapack.get('holdings') is None:
        pending.append('DataPack_settlement_or_prices_missing')
        return {'status':'REJECTED' if errors else 'WAITING_DATAPACK','errors':errors,'pending':pending,'ready_to_submit':False}
    account=datapack['account'];holdings=datapack['holdings']
    if not isinstance(account,dict) or not isinstance(holdings,list):add('DataPack_invalid_account_or_holdings');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if first_day:
        if datapack.get('as_of_date') is not None:add('first_day_unexpected_asof')
    elif previous_trade_date is None:pending.append('official_previous_trading_date')
    elif datapack.get('as_of_date')!=previous_trade_date:add('DataPack_stale_asof')
    try:
        nav=dnum(account['nav_tradable']);cash=dnum(account['cash']);holdings_count=dnum(account['holdings_count'])
        if nav<=0 or cash<0:raise ValueError('invalid_account_numbers')
        quantities={};prices={};book=cash
        for h in holdings:
            code=h['code'];q=dnum(h['shares']);rq=dnum(h['receivable_shares'])
            if code in quantities or q<0 or rq<0 or q%1!=0 or rq%1!=0:add('invalid_or_duplicate_holding',code)
            quantities[code]=q
            if q>0:
                price=dnum(h['close_price'])
                if price<=0:raise ValueError('missing_held_price')
                prices[code]=price;book+=q*price
        if abs(book-nav)>Decimal('.02'):add('DataPack_NAV_reconciliation')
        if holdings_count!=sum(q>0 for q in quantities.values()):add('DataPack_holdings_count_mismatch')
        if first_day and (any(q>0 for q in quantities.values()) or nav!=cash):add('first_day_account_not_all_cash')
    except (ValueError,KeyError,TypeError):
        add('DataPack_incomplete_or_invalid_numbers');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    actual_held={s for s,q in quantities.items() if q>0}
    if actual_held-(set(decision_by_ticker)|set(held_no_trade)):add('prior_holdings_missing_decisions')
    if set(held_no_trade)-actual_held:add('no_trade_for_unheld_stock')
    if previous_closes is None:pending.append('official_previous_closes');return {'status':'REJECTED' if errors else 'WAITING_PRICES','errors':errors,'pending':pending,'ready_to_submit':False}
    prices.update({k:dnum(v) for k,v in previous_closes.items() if isinstance(v,(int,float,Decimal)) and not isinstance(v,bool) and math.isfinite(float(v)) and v>0})
    projected=dict(quantities);flow=Decimal(0);fees=Decimal(0);sale_for_funding=Decimal(0);details=[]
    for ticker,d in decision_by_ticker.items():
        current=quantities.get(ticker,Decimal(0));price=prices.get(ticker)
        if price is None or price<=0:pending.append('missing_previous_close:'+ticker);continue
        target=(dnum(d['target_weight'])*nav/price/1000).to_integral_value(rounding=ROUND_FLOOR)*1000
        change=target-current;o=orders_by_decision.get(d['decision_id'])
        if d['action']=='BUY' and current!=0:add('BUY_requires_zero_tradable_holding',ticker)
        if d['action'] in ['ADD','TRIM','SELL_ALL'] and current<=0:add('adjust_requires_held_stock',ticker)
        if d['action'] in ['BUY','ADD'] and change<=0:add('buy_target_not_increase',ticker)
        if d['action'] in ['TRIM','SELL_ALL'] and change>=0:add('sell_target_not_decrease',ticker)
        if change!=0 and (o is None or dnum(o['shares'])!=abs(change)):add('order_quantity_formula_mismatch',ticker)
        if change==0 and o is not None:add('unneeded_zero_delta_order',ticker)
    for o in plan['orders']:
        ticker=o['ticker'];price=prices.get(ticker)
        if price is None:pending.append('missing_previous_close:'+ticker);continue
        shares=dnum(o['shares']);delta=shares if o['side']=='BUY' else -shares
        current=projected.get(ticker,Decimal(0))
        if current+delta<0:add('sell_exceeds_tradable_holding',ticker)
        projected[ticker]=current+delta;gross=shares*price
        fee=gross*(Decimal('.001425') if delta>0 else Decimal('.004425'))
        flow+=delta*price;fees+=fee
        ref=tables['decisions'].get(o['decision_ref'],{})
        if delta<0 and ref.get('funding_for'):sale_for_funding+=gross-fee
        details.append({'ticker':ticker,'shares':int(shares),'side':o['side']})
    estimated_cash=cash-flow-fees;estimated_nav=nav-fees
    if estimated_nav<=0:add('estimated_nav_nonpositive');ratio=Decimal(0)
    else:ratio=estimated_cash/estimated_nav
    if estimated_cash<0:add('estimated_cash_negative')
    if ratio>=Decimal('.25'):add('estimated_cash_limit')
    if not dnum(cash_range[0])<=ratio<=dnum(cash_range[1]):add('cash_outside_declared_range')
    if not 20<=sum(q>0 for q in projected.values())<=30:add('projected_holdings_count_limit')
    intent=plan['market_view']['posture']['net_exposure_intent']
    if intent=='increase' and flow<=0 or intent=='reduce' and flow>=0 or intent=='hold' and abs(flow)>Decimal('.02')*nav:add('net_flow_intent_mismatch')
    if sale_for_funding and (estimated_cash-sale_for_funding)/max(estimated_nav,Decimal('.01'))>=dnum(cash_range[0]):add('funding_gap_not_demonstrated')
    # Market-drift cap grace needs incident history; flag for review instead of declaring automatic disqualification.
    for ticker,q in projected.items():
        if q>0 and ticker in prices and estimated_nav>0 and q*prices[ticker]/estimated_nav>(Decimal('.25') if ticker=='2330' else Decimal('.10')):pending.append('projected_cap_grace_review:'+ticker)
    return {'status':'REJECTED' if errors else 'LOCAL_CHECKS_PASSED_PENDING_OFFICIAL_REVIEW','errors':errors,'pending':list(dict.fromkeys(pending)),'ready_to_submit':False,'previous_close_estimate':{'cash':float(estimated_cash),'fee_and_tax':float(fees),'cash_ratio':float(ratio),'net_buy_value':float(flow),'holdings_count':sum(q>0 for q in projected.values())}}
print('DPLAN_LOCAL_VALIDATOR_DEFINED; official review gates remain even with locally valid inputs')


In [ ]:
# D-Plan local validation prototype; no network calls, no submission, no invented live account
from decimal import Decimal,ROUND_FLOOR
def dnum(value):
    if isinstance(value,bool) or not isinstance(value,(int,float,Decimal)) or not math.isfinite(float(value)):raise ValueError('finite_number_required')
    return Decimal(str(value))
def dtime(value):
    t=datetime.datetime.fromisoformat(value)
    if t.utcoffset()!=datetime.timedelta(hours=8):raise ValueError('Taipei_offset_required')
    return t
def validate_dplan_local(plan,*,team_id='TEAM_11693',trade_date=None,cutoff=None,datapack=None,previous_trade_date=None,first_day=False,previous_closes=None,universe=None):
    errors=[];pending=['official_content_and_strategy_review','official_ActiveShare_and_grace_period_review','official_execution_mean_and_batch_result']
    def add(code,path=''):errors.append({'code':code,'path':path})
    try:json.dumps(plan,allow_nan=False)
    except (ValueError,TypeError):
        add('non_finite_or_non_JSON');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    schema_errors=list(Draft202012Validator(dplan_schema,format_checker=FormatChecker()).iter_errors(plan))
    for e in schema_errors[:50]:add('schema','/'.join(map(str,e.absolute_path)))
    if schema_errors or not isinstance(plan,dict):return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if plan['schema_version']!='4.3':add('unsupported_DPlan_version','schema_version')
    if plan['team_id']!=team_id:add('wrong_team','team_id')
    if trade_date is None:pending.append('official_trade_date')
    elif plan['trade_date']!=trade_date:add('wrong_trade_date','trade_date')
    if cutoff is None:pending.append('trusted_receipt_cutoff')
    else:
        try:
            deadline=dtime(cutoff)
            if deadline.date().isoformat()!=plan['trade_date']:add('cutoff_date_mismatch')
            for i,s in enumerate(plan['sources']):
                stamps={k:dtime(s[k]) for k in ['published_at','content_as_of','fetched_at'] if k in s}
                if any(t>deadline for t in stamps.values()):add('source_after_cutoff',f'sources/{i}')
                if 'published_at' in stamps and 'fetched_at' in stamps and stamps['published_at']>stamps['fetched_at']:add('source_fetch_before_publication',f'sources/{i}')
            meta=plan['agent_metadata'];start=dtime(meta['run_started_at']);end=dtime(meta['run_completed_at'])
            if not start<=end<=deadline:add('invalid_agent_run_time','agent_metadata')
        except (ValueError,TypeError):add('invalid_Taipei_timestamp')
    tables={};prefixes={'sources':('source_id','S'),'observations':('obs_id','O'),'inferences':('inf_id','I'),'decisions':('decision_id','D')}
    for section,(key,prefix) in prefixes.items():
        ids=[r[key] for r in plan[section]]
        if len(ids)!=len(set(ids)):add('duplicate_id',section)
        if set(ids)!={prefix+str(i+1) for i in range(len(ids))}:add('non_contiguous_ids',section)
        tables[section]={r[key]:r for r in plan[section]}
    def refs(rows,key,target,label):
        for i,row in enumerate(rows):
            for ref in row.get(key,[]):
                if ref not in tables[target]:add('broken_reference',f'{label}/{i}/{key}')
    refs(plan['observations'],'source_ref','sources','observations');refs(plan['inferences'],'premise_refs','observations','inferences')
    refs([plan['market_view']],'basis_refs','observations','market_view');refs(plan['decisions'],'inference_refs','inferences','decisions');refs(plan['no_trade_decisions'],'reason_refs','inferences','no_trade_decisions')
    decision_by_ticker={}
    for i,d in enumerate(plan['decisions']):
        if d['ticker'] in decision_by_ticker:add('duplicate_ticker_decision',d['ticker'])
        decision_by_ticker[d['ticker']]=d
        cap=Decimal('.25') if d['ticker']=='2330' else Decimal('.10')
        if dnum(d['target_weight'])>cap:add('target_cap_exceeded',d['ticker'])
        if d['action']=='SELL_ALL' and dnum(d['target_weight'])!=0:add('sell_all_target_not_zero',d['ticker'])
        if 'funding_for' in d:
            if d['action'] not in ['TRIM','SELL_ALL']:add('funding_must_be_sell',d['ticker'])
            for ref in d['funding_for']:
                if ref not in tables['decisions'] or tables['decisions'][ref]['action'] not in ['BUY','ADD']:add('invalid_funding_reference',d['ticker'])
    held_no_trade=[r['ticker'] for r in plan['no_trade_decisions']]
    if len(held_no_trade)!=len(set(held_no_trade)):add('duplicate_no_trade_ticker')
    if set(held_no_trade)&set(decision_by_ticker):add('trade_no_trade_overlap')
    seen_orders=set();orders_by_decision={}
    for i,o in enumerate(plan['orders']):
        if o['ticker'] in seen_orders:add('duplicate_order_ticker',o['ticker'])
        seen_orders.add(o['ticker']);ref=o['decision_ref']
        if ref not in tables['decisions']:add('broken_order_reference',str(i));continue
        d=tables['decisions'][ref]
        if o['ticker']!=d['ticker']:add('order_ticker_mismatch',str(i))
        if o['side']!=('BUY' if d['action'] in ['BUY','ADD'] else 'SELL'):add('order_side_mismatch',str(i))
        if ref in orders_by_decision:add('duplicate_order_decision',ref)
        orders_by_decision[ref]=o
    if universe is None:pending.append('official_universe')
    elif (set(decision_by_ticker)|set(held_no_trade)|seen_orders)-set(universe):add('ticker_outside_universe')
    cash_range=plan['market_view']['posture']['target_cash_pct_range']
    if cash_range[0]>cash_range[1]:add('cash_range_reversed')
    if datapack is None:
        pending.append('official_DataPack')
        return {'status':'REJECTED' if errors else 'WAITING_DATAPACK','errors':errors,'pending':pending,'ready_to_submit':False}
    if not isinstance(datapack,dict):add('invalid_DataPack_type');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if datapack.get('doc_type')!='DataPack':add('wrong_DataPack_doc_type')
    if datapack.get('team_id')!=team_id:add('DataPack_wrong_team')
    if datapack.get('trade_date')!=plan['trade_date']:add('DataPack_wrong_trade_date')
    # Archived document4.3 examples use DataPack3.0. Do not assume DataPack4.3.
    if datapack.get('schema_version')!='3.0':pending.append('verify_new_DataPack_schema_version')
    if cutoff is not None:
        try:
            if dtime(datapack['generated_at'])>deadline:add('DataPack_generated_after_cutoff')
        except (ValueError,KeyError,TypeError):add('DataPack_invalid_generated_at')
    missing=datapack.get('missing')
    if not isinstance(missing,list):add('DataPack_missing_list_invalid');missing=['unknown']
    if missing or datapack.get('account') is None or datapack.get('holdings') is None:
        pending.append('DataPack_settlement_or_prices_missing')
        return {'status':'REJECTED' if errors else 'WAITING_DATAPACK','errors':errors,'pending':pending,'ready_to_submit':False}
    account=datapack['account'];holdings=datapack['holdings']
    if not isinstance(account,dict) or not isinstance(holdings,list):add('DataPack_invalid_account_or_holdings');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if first_day:
        if datapack.get('as_of_date') is not None:add('first_day_unexpected_asof')
    elif previous_trade_date is None:pending.append('official_previous_trading_date')
    elif datapack.get('as_of_date')!=previous_trade_date:add('DataPack_stale_asof')
    try:
        nav=dnum(account['nav_tradable']);cash=dnum(account['cash']);holdings_count=dnum(account['holdings_count'])
        if nav<=0 or cash<0:raise ValueError('invalid_account_numbers')
        quantities={};prices={};book=cash
        for h in holdings:
            code=h['code'];q=dnum(h['shares']);rq=dnum(h['receivable_shares'])
            if code in quantities or q<0 or rq<0 or q%1!=0 or rq%1!=0:add('invalid_or_duplicate_holding',code)
            quantities[code]=q
            if q>0:
                price=dnum(h['close_price'])
                if price<=0:raise ValueError('missing_held_price')
                prices[code]=price;book+=q*price
        if abs(book-nav)>Decimal('.02'):add('DataPack_NAV_reconciliation')
        if holdings_count!=sum(q>0 for q in quantities.values()):add('DataPack_holdings_count_mismatch')
        if first_day and (any(q>0 for q in quantities.values()) or nav!=cash):add('first_day_account_not_all_cash')
    except (ValueError,KeyError,TypeError):
        add('DataPack_incomplete_or_invalid_numbers');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    actual_held={s for s,q in quantities.items() if q>0}
    if actual_held-(set(decision_by_ticker)|set(held_no_trade)):add('prior_holdings_missing_decisions')
    if set(held_no_trade)-actual_held:add('no_trade_for_unheld_stock')
    if previous_closes is None:pending.append('official_previous_closes');return {'status':'REJECTED' if errors else 'WAITING_PRICES','errors':errors,'pending':pending,'ready_to_submit':False}
    prices.update({k:dnum(v) for k,v in previous_closes.items() if isinstance(v,(int,float,Decimal)) and not isinstance(v,bool) and math.isfinite(float(v)) and v>0})
    projected=dict(quantities);flow=Decimal(0);fees=Decimal(0);sale_for_funding=Decimal(0);details=[]
    for ticker,d in decision_by_ticker.items():
        current=quantities.get(ticker,Decimal(0));price=prices.get(ticker)
        if price is None or price<=0:pending.append('missing_previous_close:'+ticker);continue
        target=(dnum(d['target_weight'])*nav/price/1000).to_integral_value(rounding=ROUND_FLOOR)*1000
        change=target-current;o=orders_by_decision.get(d['decision_id'])
        if d['action']=='BUY' and current!=0:add('BUY_requires_zero_tradable_holding',ticker)
        if d['action'] in ['ADD','TRIM','SELL_ALL'] and current<=0:add('adjust_requires_held_stock',ticker)
        if d['action'] in ['BUY','ADD'] and change<=0:add('buy_target_not_increase',ticker)
        if d['action'] in ['TRIM','SELL_ALL'] and change>=0:add('sell_target_not_decrease',ticker)
        if change!=0 and (o is None or dnum(o['shares'])!=abs(change)):add('order_quantity_formula_mismatch',ticker)
        if change==0 and o is not None:add('unneeded_zero_delta_order',ticker)
    for o in plan['orders']:
        ticker=o['ticker'];price=prices.get(ticker)
        if price is None:pending.append('missing_previous_close:'+ticker);continue
        shares=dnum(o['shares']);delta=shares if o['side']=='BUY' else -shares
        current=projected.get(ticker,Decimal(0))
        if current+delta<0:add('sell_exceeds_tradable_holding',ticker)
        projected[ticker]=current+delta;gross=shares*price
        fee=gross*(Decimal('.001425') if delta>0 else Decimal('.004425'))
        flow+=delta*price;fees+=fee
        ref=tables['decisions'].get(o['decision_ref'],{})
        if delta<0 and ref.get('funding_for'):sale_for_funding+=gross-fee
        details.append({'ticker':ticker,'shares':int(shares),'side':o['side']})
    estimated_cash=cash-flow-fees;estimated_nav=nav-fees
    if estimated_nav<=0:add('estimated_nav_nonpositive');ratio=Decimal(0)
    else:ratio=estimated_cash/estimated_nav
    if estimated_cash<0:add('estimated_cash_negative')
    if ratio>=Decimal('.25'):add('estimated_cash_limit')
    if not dnum(cash_range[0])<=ratio<=dnum(cash_range[1]):add('cash_outside_declared_range')
    if not 20<=sum(q>0 for q in projected.values())<=30:add('projected_holdings_count_limit')
    intent=plan['market_view']['posture']['net_exposure_intent']
    if intent=='increase' and flow<=0 or intent=='reduce' and flow>=0 or intent=='hold' and abs(flow)>Decimal('.02')*nav:add('net_flow_intent_mismatch')
    if sale_for_funding and (estimated_cash-sale_for_funding)/max(estimated_nav,Decimal('.01'))>=dnum(cash_range[0]):add('funding_gap_not_demonstrated')
    # Market-drift cap grace needs incident history; flag for review instead of declaring automatic disqualification.
    for ticker,q in projected.items():
        if q>0 and ticker in prices and estimated_nav>0 and q*prices[ticker]/estimated_nav>(Decimal('.25') if ticker=='2330' else Decimal('.10')):pending.append('projected_cap_grace_review:'+ticker)
    return {'status':'REJECTED' if errors else 'LOCAL_CHECKS_PASSED_PENDING_OFFICIAL_REVIEW','errors':errors,'pending':list(dict.fromkeys(pending)),'ready_to_submit':False,'previous_close_estimate':{'cash':float(estimated_cash),'fee_and_tax':float(fees),'cash_ratio':float(ratio),'net_buy_value':float(flow),'holdings_count':sum(q>0 for q in projected.values())}}
print('DPLAN_LOCAL_VALIDATOR_DEFINED; official review gates remain even with locally valid inputs')


In [ ]:
# Local software fixtures only: synthetic prices/accounts are not market data or strategy backtests
offline_fixture_tickers=list(stock_ids[:28])
offline_fixture_prices={s:100. for s in offline_fixture_tickers}
offline_fixture_plan=copy.deepcopy(dplan_example)
offline_fixture_plan.update(team_id='TEAM_11693',_NOTE='SOFTWARE TEST ONLY. Official example text is illustrative; prices/account below are synthetic unit-test fixtures. Not a live D-Plan or market evidence.',decisions=[],orders=[])
offline_fixture_plan['no_trade_decisions']=[{'ticker':s,'reason_refs':['I1'],'reason':'僅測試引用與持股覆蓋，不是實際投資推論。'} for s in offline_fixture_tickers[:25]]
offline_fixture_plan['market_view']['posture']={'net_exposure_intent':'hold','target_cash_pct_range':[.03,.10]}
offline_fixture_pack={'schema_version':'3.0','doc_type':'DataPack','team_id':'TEAM_11693','trade_date':'2026-10-28','as_of_date':'2026-10-27','generated_at':'2026-10-28T05:00:00+08:00','missing':[],'prev_day_orders':{'outcome':'executed','void_reasons':[],'excluded_codes':[]},'account':{'cash':250000.,'nav_tradable':2750000.,'aum':3750000.,'receivable_cash':1000000.,'holdings_count':25,'cash_ratio_tradable':250000/2750000},'holdings':[{'code':s,'shares':1000,'receivable_shares':0,'close_price':100.,'tradable_weight':100000/2750000} for s in offline_fixture_tickers[:25]]+[{'code':offline_fixture_tickers[25],'shares':0,'receivable_shares':3000,'close_price':None,'tradable_weight':None}],'compliance':{'violations':0,'notices':0,'incidents':[]}}
def fixture_context_v1():return {'team_id':'TEAM_11693','trade_date':'2026-10-28','cutoff':'2026-10-28T08:55:00+08:00','datapack':copy.deepcopy(offline_fixture_pack),'previous_trade_date':'2026-10-27','first_day':False,'previous_closes':dict(offline_fixture_prices),'universe':set(stock_ids)}
def fixture_case_v1(name,change=None,expected_code=None,expected_status=None,plan=None):
    p=copy.deepcopy(offline_fixture_plan if plan is None else plan);c=fixture_context_v1()
    if change:change(p,c)
    result=validate_dplan_local(p,**c)
    assert result['ready_to_submit'] is False,name
    if expected_code:assert expected_code in {e['code'] for e in result['errors']},(name,result)
    if expected_status:assert result['status']==expected_status,(name,result)
    dplan_test_results_v1.append({'name':name,'expected_code':expected_code,'status':result['status'],'codes':sorted({e['code'] for e in result['errors']}),'pass':True})
    return result
def with_add_v1():
    p=copy.deepcopy(offline_fixture_plan);ticker=offline_fixture_tickers[0]
    p['decisions']=[{'decision_id':'D1','ticker':ticker,'action':'ADD','target_weight':.08,'inference_refs':['I1']}]
    p['orders']=[{'ticker':ticker,'side':'BUY','shares':1000,'decision_ref':'D1'}]
    p['no_trade_decisions']=p['no_trade_decisions'][1:]
    p['market_view']['posture']['net_exposure_intent']='increase'
    return p
dplan_test_results_v1=[]
baseline_fixture_result=fixture_case_v1('positive_no_trade_with_receivables',expected_status='LOCAL_CHECKS_PASSED_PENDING_OFFICIAL_REVIEW')
fixture_case_v1('positive_add_uses_tradableNAV_not_AUM',plan=with_add_v1(),expected_status='LOCAL_CHECKS_PASSED_PENDING_OFFICIAL_REVIEW')
fixture_case_v1('no_DataPack_stops',lambda p,c:c.update(datapack=None),expected_status='WAITING_DATAPACK')
fixture_case_v1('null_account_not_zero',lambda p,c:c['datapack'].update(account=None,holdings=None,missing=['prev_settlement']),expected_status='WAITING_DATAPACK')
fixture_case_v1('null_holdings_not_empty',lambda p,c:c['datapack'].update(holdings=None),expected_status='WAITING_DATAPACK')
fixture_case_v1('missing_previous_close',lambda p,c:c['datapack'].update(missing=['prev_close']),expected_status='WAITING_DATAPACK')
fixture_case_v1('missing_market_prices',lambda p,c:c.update(previous_closes=None),expected_status='WAITING_PRICES')
fixture_case_v1('wrong_team',lambda p,c:p.update(team_id='TEAM_999'),expected_code='wrong_team')
fixture_case_v1('wrong_pack_team',lambda p,c:c['datapack'].update(team_id='TEAM_999'),expected_code='DataPack_wrong_team')
fixture_case_v1('stale_account_date',lambda p,c:c['datapack'].update(as_of_date='2026-10-26'),expected_code='DataPack_stale_asof')
fixture_case_v1('NAV_reconciliation',lambda p,c:c['datapack']['account'].update(nav_tradable=3750000),expected_code='DataPack_NAV_reconciliation')
fixture_case_v1('unknown_extra_field',lambda p,c:p.update(unsafe_extra=True),expected_code='schema')
fixture_case_v1('NaN_never_passes',lambda p,c:p['observations'][0]['values'].update(nan=float('nan')),expected_code='non_finite_or_non_JSON')
fixture_case_v1('source_after_deadline',lambda p,c:p['sources'][0].update(content_as_of='2026-10-28T09:00:00+08:00'),expected_code='source_after_cutoff')
fixture_case_v1('broken_source_reference',lambda p,c:p['observations'][0].update(source_ref=['S999']),expected_code='broken_reference')
fixture_case_v1('noncontiguous_source_ID',lambda p,c:p['sources'][0].update(source_id='S0'),expected_code='non_contiguous_ids')
fixture_case_v1('duplicate_ID',lambda p,c:p['sources'].append(copy.deepcopy(p['sources'][0])),expected_code='duplicate_id')
fixture_case_v1('missing_old_holding_reason',lambda p,c:p['no_trade_decisions'].pop(),expected_code='prior_holdings_missing_decisions')
fixture_case_v1('non_lot_order500',lambda p,c:p['orders'][0].update(shares=500),plan=with_add_v1(),expected_code='schema')
fixture_case_v1('wrong_order_quantity',lambda p,c:p['orders'][0].update(shares=2000),plan=with_add_v1(),expected_code='order_quantity_formula_mismatch')
fixture_case_v1('missing_required_order',lambda p,c:p.update(orders=[]),plan=with_add_v1(),expected_code='order_quantity_formula_mismatch')
fixture_case_v1('BUY_on_existing_holding',lambda p,c:p['decisions'][0].update(action='BUY'),plan=with_add_v1(),expected_code='BUY_requires_zero_tradable_holding')
fixture_case_v1('side_conflicts_decision',lambda p,c:p['orders'][0].update(side='SELL'),plan=with_add_v1(),expected_code='order_side_mismatch')
fixture_case_v1('cash_range_reversed',lambda p,c:p['market_view']['posture'].update(target_cash_pct_range=[.10,.03]),expected_code='cash_range_reversed')
fixture_case_v1('wrong_market_net_intent',lambda p,c:p['market_view']['posture'].update(net_exposure_intent='hold'),plan=with_add_v1(),expected_code='net_flow_intent_mismatch')
fixture_case_v1('funding_tag_on_buy',lambda p,c:p['decisions'][0].update(funding_for=['D1']),plan=with_add_v1(),expected_code='funding_must_be_sell')
def oversell_fixture(p,c):
    ticker=offline_fixture_tickers[25]
    p['decisions']=[{'decision_id':'D1','ticker':ticker,'action':'SELL_ALL','target_weight':0,'inference_refs':['I1']}]
    p['orders']=[{'ticker':ticker,'side':'SELL','shares':3000,'decision_ref':'D1'}]
    p['market_view']['posture']={'net_exposure_intent':'reduce','target_cash_pct_range':[0,.24]}
fixture_case_v1('receivable_shares_cannot_sell',oversell_fixture,expected_code='sell_exceeds_tradable_holding')
def cash_short_fixture(p,c):
    p['decisions']=[{'decision_id':f'D{i+1}','ticker':s,'action':'BUY','target_weight':.09,'inference_refs':['I1']} for i,s in enumerate(offline_fixture_tickers[25:27])]
    p['orders']=[{'ticker':s,'side':'BUY','shares':2000,'decision_ref':f'D{i+1}'} for i,s in enumerate(offline_fixture_tickers[25:27])]
    p['market_view']['posture']['net_exposure_intent']='increase'
fixture_case_v1('buying_power_short',cash_short_fixture,expected_code='estimated_cash_negative')
def first_day_fixture(p,c):
    c['first_day']=True;c['previous_trade_date']=None
    c['datapack'].update(as_of_date=None,holdings=[],prev_day_orders={'outcome':'none','void_reasons':[],'excluded_codes':[]})
    c['datapack']['account'].update(cash=100000000.,nav_tradable=100000000.,aum=100000000.,receivable_cash=0,holdings_count=0,cash_ratio_tradable=1.)
    p['decisions']=[{'decision_id':f'D{i+1}','ticker':s,'action':'BUY','target_weight':.036,'inference_refs':['I1']} for i,s in enumerate(offline_fixture_tickers[:25])]
    p['orders']=[{'ticker':s,'side':'BUY','shares':36000,'decision_ref':f'D{i+1}'} for i,s in enumerate(offline_fixture_tickers[:25])]
    p['no_trade_decisions']=[];p['market_view']['posture']={'net_exposure_intent':'increase','target_cash_pct_range':[.08,.12]}
fixture_case_v1('first_day_true_cash_account',first_day_fixture,expected_status='LOCAL_CHECKS_PASSED_PENDING_OFFICIAL_REVIEW')
print('DPLAN_SOFTWARE_TEST_RESULTS',json.dumps(dplan_test_results_v1,ensure_ascii=False))
print('All cases passed',len(dplan_test_results_v1),'official readiness',False,'real DataPack used',False)


In [ ]:
# D-Plan local validation prototype; no network calls, no submission, no invented live account
from decimal import Decimal,ROUND_FLOOR
def dnum(value):
    if isinstance(value,bool) or not isinstance(value,(int,float,Decimal)) or not math.isfinite(float(value)):raise ValueError('finite_number_required')
    return Decimal(str(value))
def dtime(value):
    t=datetime.datetime.fromisoformat(value)
    if t.utcoffset()!=datetime.timedelta(hours=8):raise ValueError('Taipei_offset_required')
    return t
def validate_dplan_local(plan,*,team_id='TEAM_11693',trade_date=None,cutoff=None,datapack=None,previous_trade_date=None,first_day=False,previous_closes=None,universe=None):
    errors=[];pending=['official_content_and_strategy_review','official_ActiveShare_and_grace_period_review','official_execution_mean_and_batch_result']
    def add(code,path=''):errors.append({'code':code,'path':path})
    try:json.dumps(plan,allow_nan=False)
    except (ValueError,TypeError):
        add('non_finite_or_non_JSON');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    schema_errors=list(Draft202012Validator(dplan_schema,format_checker=FormatChecker()).iter_errors(plan))
    for e in schema_errors[:50]:add('schema','/'.join(map(str,e.absolute_path)))
    if schema_errors or not isinstance(plan,dict):return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if plan['schema_version']!='4.3':add('unsupported_DPlan_version','schema_version')
    if plan['team_id']!=team_id:add('wrong_team','team_id')
    if trade_date is None:pending.append('official_trade_date')
    elif plan['trade_date']!=trade_date:add('wrong_trade_date','trade_date')
    if cutoff is None:pending.append('trusted_receipt_cutoff')
    else:
        try:
            deadline=dtime(cutoff)
            if deadline.date().isoformat()!=plan['trade_date']:add('cutoff_date_mismatch')
            for i,s in enumerate(plan['sources']):
                stamps={k:dtime(s[k]) for k in ['published_at','content_as_of','fetched_at'] if k in s}
                if any(t>deadline for t in stamps.values()):add('source_after_cutoff',f'sources/{i}')
                if 'published_at' in stamps and 'fetched_at' in stamps and stamps['published_at']>stamps['fetched_at']:add('source_fetch_before_publication',f'sources/{i}')
            meta=plan['agent_metadata'];start=dtime(meta['run_started_at']);end=dtime(meta['run_completed_at'])
            if not start<=end<=deadline:add('invalid_agent_run_time','agent_metadata')
        except (ValueError,TypeError):add('invalid_Taipei_timestamp')
    tables={};prefixes={'sources':('source_id','S'),'observations':('obs_id','O'),'inferences':('inf_id','I'),'decisions':('decision_id','D')}
    for section,(key,prefix) in prefixes.items():
        ids=[r[key] for r in plan[section]]
        if len(ids)!=len(set(ids)):add('duplicate_id',section)
        if set(ids)!={prefix+str(i+1) for i in range(len(ids))}:add('non_contiguous_ids',section)
        tables[section]={r[key]:r for r in plan[section]}
    def refs(rows,key,target,label):
        for i,row in enumerate(rows):
            for ref in row.get(key,[]):
                if ref not in tables[target]:add('broken_reference',f'{label}/{i}/{key}')
    refs(plan['observations'],'source_ref','sources','observations');refs(plan['inferences'],'premise_refs','observations','inferences')
    refs([plan['market_view']],'basis_refs','observations','market_view');refs(plan['decisions'],'inference_refs','inferences','decisions');refs(plan['no_trade_decisions'],'reason_refs','inferences','no_trade_decisions')
    decision_by_ticker={}
    for i,d in enumerate(plan['decisions']):
        if d['ticker'] in decision_by_ticker:add('duplicate_ticker_decision',d['ticker'])
        decision_by_ticker[d['ticker']]=d
        cap=Decimal('.25') if d['ticker']=='2330' else Decimal('.10')
        if dnum(d['target_weight'])>cap:add('target_cap_exceeded',d['ticker'])
        if d['action']=='SELL_ALL' and dnum(d['target_weight'])!=0:add('sell_all_target_not_zero',d['ticker'])
        if 'funding_for' in d:
            if d['action'] not in ['TRIM','SELL_ALL']:add('funding_must_be_sell',d['ticker'])
            for ref in d['funding_for']:
                if ref not in tables['decisions'] or tables['decisions'][ref]['action'] not in ['BUY','ADD']:add('invalid_funding_reference',d['ticker'])
    held_no_trade=[r['ticker'] for r in plan['no_trade_decisions']]
    if len(held_no_trade)!=len(set(held_no_trade)):add('duplicate_no_trade_ticker')
    if set(held_no_trade)&set(decision_by_ticker):add('trade_no_trade_overlap')
    seen_orders=set();orders_by_decision={}
    for i,o in enumerate(plan['orders']):
        if o['ticker'] in seen_orders:add('duplicate_order_ticker',o['ticker'])
        seen_orders.add(o['ticker']);ref=o['decision_ref']
        if ref not in tables['decisions']:add('broken_order_reference',str(i));continue
        d=tables['decisions'][ref]
        if o['ticker']!=d['ticker']:add('order_ticker_mismatch',str(i))
        if o['side']!=('BUY' if d['action'] in ['BUY','ADD'] else 'SELL'):add('order_side_mismatch',str(i))
        if ref in orders_by_decision:add('duplicate_order_decision',ref)
        orders_by_decision[ref]=o
    if universe is None:pending.append('official_universe')
    elif (set(decision_by_ticker)|set(held_no_trade)|seen_orders)-set(universe):add('ticker_outside_universe')
    cash_range=plan['market_view']['posture']['target_cash_pct_range']
    if cash_range[0]>cash_range[1]:add('cash_range_reversed')
    if errors:return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if datapack is None:
        pending.append('official_DataPack')
        return {'status':'REJECTED' if errors else 'WAITING_DATAPACK','errors':errors,'pending':pending,'ready_to_submit':False}
    if not isinstance(datapack,dict):add('invalid_DataPack_type');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if datapack.get('doc_type')!='DataPack':add('wrong_DataPack_doc_type')
    if datapack.get('team_id')!=team_id:add('DataPack_wrong_team')
    if datapack.get('trade_date')!=plan['trade_date']:add('DataPack_wrong_trade_date')
    # Archived document4.3 examples use DataPack3.0. Do not assume DataPack4.3.
    if datapack.get('schema_version')!='3.0':pending.append('verify_new_DataPack_schema_version')
    if cutoff is not None:
        try:
            if dtime(datapack['generated_at'])>deadline:add('DataPack_generated_after_cutoff')
        except (ValueError,KeyError,TypeError):add('DataPack_invalid_generated_at')
    missing=datapack.get('missing')
    if not isinstance(missing,list):add('DataPack_missing_list_invalid');missing=['unknown']
    if missing or datapack.get('account') is None or datapack.get('holdings') is None:
        pending.append('DataPack_settlement_or_prices_missing')
        return {'status':'REJECTED' if errors else 'WAITING_DATAPACK','errors':errors,'pending':pending,'ready_to_submit':False}
    account=datapack['account'];holdings=datapack['holdings']
    if not isinstance(account,dict) or not isinstance(holdings,list):add('DataPack_invalid_account_or_holdings');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    if first_day:
        if datapack.get('as_of_date') is not None:add('first_day_unexpected_asof')
    elif previous_trade_date is None:pending.append('official_previous_trading_date')
    elif datapack.get('as_of_date')!=previous_trade_date:add('DataPack_stale_asof')
    try:
        nav=dnum(account['nav_tradable']);cash=dnum(account['cash']);holdings_count=dnum(account['holdings_count'])
        if nav<=0 or cash<0:raise ValueError('invalid_account_numbers')
        quantities={};prices={};book=cash
        for h in holdings:
            code=h['code'];q=dnum(h['shares']);rq=dnum(h['receivable_shares'])
            if code in quantities or q<0 or rq<0 or q%1!=0 or rq%1!=0:add('invalid_or_duplicate_holding',code)
            quantities[code]=q
            if q>0:
                price=dnum(h['close_price'])
                if price<=0:raise ValueError('missing_held_price')
                prices[code]=price;book+=q*price
        if abs(book-nav)>Decimal('.02'):add('DataPack_NAV_reconciliation')
        if holdings_count!=sum(q>0 for q in quantities.values()):add('DataPack_holdings_count_mismatch')
        if first_day and (any(q>0 for q in quantities.values()) or nav!=cash):add('first_day_account_not_all_cash')
    except (ValueError,KeyError,TypeError):
        add('DataPack_incomplete_or_invalid_numbers');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    actual_held={s for s,q in quantities.items() if q>0}
    if actual_held-(set(decision_by_ticker)|set(held_no_trade)):add('prior_holdings_missing_decisions')
    if set(held_no_trade)-actual_held:add('no_trade_for_unheld_stock')
    if previous_closes is None:pending.append('official_previous_closes');return {'status':'REJECTED' if errors else 'WAITING_PRICES','errors':errors,'pending':pending,'ready_to_submit':False}
    if not isinstance(previous_closes,dict):
        add('invalid_previous_closes');return {'status':'REJECTED','errors':errors,'pending':pending,'ready_to_submit':False}
    for ticker,price in prices.items():
        if ticker in previous_closes:
            try:
                if abs(dnum(previous_closes[ticker])-price)>Decimal('.000001'):add('price_account_mismatch',ticker)
            except ValueError:add('invalid_previous_close',ticker)
    prices.update({k:dnum(v) for k,v in previous_closes.items() if isinstance(v,(int,float,Decimal)) and not isinstance(v,bool) and math.isfinite(float(v)) and v>0})
    missing_prices=(set(decision_by_ticker)|seen_orders)-set(prices)
    if missing_prices:
        pending.extend('missing_previous_close:'+s for s in sorted(missing_prices))
        return {'status':'REJECTED' if errors else 'WAITING_PRICES','errors':errors,'pending':pending,'ready_to_submit':False}
    projected=dict(quantities);flow=Decimal(0);fees=Decimal(0);sale_for_funding=Decimal(0);details=[]
    for ticker,d in decision_by_ticker.items():
        current=quantities.get(ticker,Decimal(0));price=prices.get(ticker)
        if price is None or price<=0:pending.append('missing_previous_close:'+ticker);continue
        target=(dnum(d['target_weight'])*nav/price/1000).to_integral_value(rounding=ROUND_FLOOR)*1000
        change=target-current;o=orders_by_decision.get(d['decision_id'])
        if d['action']=='BUY' and current!=0:add('BUY_requires_zero_tradable_holding',ticker)
        if d['action'] in ['ADD','TRIM','SELL_ALL'] and current<=0:add('adjust_requires_held_stock',ticker)
        if d['action'] in ['BUY','ADD'] and change<=0:add('buy_target_not_increase',ticker)
        if d['action'] in ['TRIM','SELL_ALL'] and change>=0:add('sell_target_not_decrease',ticker)
        if change!=0 and (o is None or dnum(o['shares'])!=abs(change)):add('order_quantity_formula_mismatch',ticker)
        if change==0 and o is not None:add('unneeded_zero_delta_order',ticker)
    for o in plan['orders']:
        ticker=o['ticker'];price=prices.get(ticker)
        if price is None:pending.append('missing_previous_close:'+ticker);continue
        shares=dnum(o['shares']);delta=shares if o['side']=='BUY' else -shares
        current=projected.get(ticker,Decimal(0))
        if current+delta<0:add('sell_exceeds_tradable_holding',ticker)
        projected[ticker]=current+delta;gross=shares*price
        fee=gross*(Decimal('.001425') if delta>0 else Decimal('.004425'))
        flow+=delta*price;fees+=fee
        ref=tables['decisions'].get(o['decision_ref'],{})
        if delta<0 and ref.get('funding_for'):sale_for_funding+=gross-fee
        details.append({'ticker':ticker,'shares':int(shares),'side':o['side']})
    estimated_cash=cash-flow-fees;estimated_nav=nav-fees
    if estimated_nav<=0:add('estimated_nav_nonpositive');ratio=Decimal(0)
    else:ratio=estimated_cash/estimated_nav
    if estimated_cash<0:add('estimated_cash_negative')
    if ratio>=Decimal('.25'):add('estimated_cash_limit')
    if not dnum(cash_range[0])<=ratio<=dnum(cash_range[1]):add('cash_outside_declared_range')
    if not 20<=sum(q>0 for q in projected.values())<=30:add('projected_holdings_count_limit')
    intent=plan['market_view']['posture']['net_exposure_intent']
    if intent=='increase' and flow<=0 or intent=='reduce' and flow>=0 or intent=='hold' and abs(flow)>Decimal('.02')*nav:add('net_flow_intent_mismatch')
    if sale_for_funding and (estimated_cash-sale_for_funding)/max(estimated_nav,Decimal('.01'))>=dnum(cash_range[0]):add('funding_gap_not_demonstrated')
    # Market-drift cap grace needs incident history; flag for review instead of declaring automatic disqualification.
    for ticker,q in projected.items():
        if q>0 and ticker in prices and estimated_nav>0 and q*prices[ticker]/estimated_nav>(Decimal('.25') if ticker=='2330' else Decimal('.10')):pending.append('projected_cap_grace_review:'+ticker)
    return {'status':'REJECTED' if errors else 'LOCAL_CHECKS_PASSED_PENDING_OFFICIAL_REVIEW','errors':errors,'pending':list(dict.fromkeys(pending)),'ready_to_submit':False,'previous_close_estimate':{'cash':float(estimated_cash),'fee_and_tax':float(fees),'cash_ratio':float(ratio),'net_buy_value':float(flow),'holdings_count':sum(q>0 for q in projected.values())}}
print('DPLAN_LOCAL_VALIDATOR_DEFINED; official review gates remain even with locally valid inputs')


In [ ]:
# Additional checks target malformed context and inconsistent account/price inputs
fixture_case_v1('invalid_cutoff_rejected',lambda p,c:c.update(cutoff='invalid'),expected_status='REJECTED')
fixture_case_v1('invalid_prices_context_type',lambda p,c:c.update(previous_closes=[]),expected_code='invalid_previous_closes')
fixture_case_v1('price_differs_from_official_holding',lambda p,c:c['previous_closes'].update({offline_fixture_tickers[0]:200.}),expected_code='price_account_mismatch',plan=with_add_v1())
fixture_case_v1('null_held_price_stops',lambda p,c:c['datapack']['holdings'][0].update(close_price=None),expected_code='DataPack_incomplete_or_invalid_numbers')
def new_buy_missing_quote(p,c):
    ticker=offline_fixture_tickers[26]
    p['decisions']=[{'decision_id':'D1','ticker':ticker,'action':'BUY','target_weight':.04,'inference_refs':['I1']}]
    p['orders']=[{'ticker':ticker,'side':'BUY','shares':1000,'decision_ref':'D1'}]
    p['market_view']['posture']['net_exposure_intent']='increase'
    c['previous_closes'].pop(ticker)
fixture_case_v1('new_stock_missing_quote',new_buy_missing_quote,expected_status='WAITING_PRICES')
def unnecessary_funding(p,c):
    sell,buy=offline_fixture_tickers[0],offline_fixture_tickers[26]
    p['decisions']=[{'decision_id':'D1','ticker':sell,'action':'SELL_ALL','target_weight':0,'inference_refs':['I1'],'funding_for':['D2']},{'decision_id':'D2','ticker':buy,'action':'BUY','target_weight':.08,'inference_refs':['I1']}]
    p['orders']=[{'ticker':sell,'side':'SELL','shares':1000,'decision_ref':'D1'},{'ticker':buy,'side':'BUY','shares':2000,'decision_ref':'D2'}]
    p['no_trade_decisions']=p['no_trade_decisions'][1:];p['market_view']['posture']={'net_exposure_intent':'increase','target_cash_pct_range':[.01,.20]}
fixture_case_v1('funding_when_cash_already_sufficient',unnecessary_funding,expected_code='funding_gap_not_demonstrated')
print('DPLAN_ALL_EDGE_CASES_PASSED',len(dplan_test_results_v1))


In [ ]:
# Persist technical validation prototype, tested boundaries and official-account dependency
assert len(dplan_test_results_v1)==35 and all(r['pass'] for r in dplan_test_results_v1)
validation_result_v1={'purpose':'offline software validation only; no strategy returns or live DataPack','team_id':'TEAM_11693','schema_file':str(dplan_schema_path),'schema_sha256':hashlib.sha256(dplan_schema_path.read_bytes()).hexdigest(),'jsonschema_version':importlib.metadata.version('jsonschema'),'official_example_warning':dplan_example.get('_NOTE'),'test_fixtures':'synthetic prices/account, official illustrative text; never real market observations','test_results':dplan_test_results_v1,'test_count':35,'ready_to_submit':False,'real_DataPack_received':False,'submitted':False,'live_unsupported_gates':['official DataPack and actual previous trading date','official current universe and matching previous close','official Active Share and cap grace history','source content truthfulness and strategy/prospectus consistency','official receipt cutoff, latest rule verification and mean-price/batch settlement'],'notebook_cells':[174,178,177,179],'call_signature':'validate_dplan_local(plan, team_id, trade_date, cutoff, datapack, previous_trade_date, first_day, previous_closes, universe)'}
result_path=dplan_root/'TEAM_11693_DPlan離線驗證結果_2026-10-15.json'
result_path.write_text(json.dumps(validation_result_v1,ensure_ascii=False,indent=2),encoding='utf-8')
lines=['# TEAM_11693：D-Plan離線驗證原型','',
'核對日期：2026-10-15。沿用本機官方D-Plan schema v4.3及DataPack規格書。尚未收到真正DataPack，沒有產生可直接提交的實際交易檔，也沒有向平台送出資料。','',
'## 已完成','',
'原研究Notebook內新增 validate_dplan_local 函式。官方JSON schema先檢查型別、欄位、格式與千股倍數，再檢查引用與機械交易關係。35個軟體案例均通過，包括正常續抱、加碼、初始現金帳戶，以及下列拒絕／等待情況。',
'',
'|檢查層|目前處理|',
'|---|---|',
'|檔案格式|D-Plan4.3、TEAM_11693、交易日期、型別、額外欄位、NaN與千股倍數|',
'|引用與覆蓋|S/O/I/D流水號不重複且連續；來源→事實→推論→決策→委託引用存在；舊持股每檔異動或續抱|',
'|時間|台北時區、來源與執行時間不晚於指定收件時點；公開時間不得晚於抓取時間|',
'|帳戶完整性|缺DataPack、null帳戶／持股、prev_settlement或prev_close缺漏時等待；不把null當0|',
'|帳戶與報價|團隊、交易日、前交易日一致；現金＋可交易持股核對nav_tradable；持股價格不能與外部前收盤價矛盾|',
'|機械股數|floor(目標權重×前日nav_tradable÷前收盤價÷1000)×1000，再扣現有可交易股數；BUY/ADD/TRIM/SELL_ALL與委託方向一致|',
'|資金與持股|應收現金不拿來買、應收配股不能賣；含費稅的前收盤預估現金、檔數、資金調度宣告及淨流向意圖|',
'|未完成官方層|來源內容真實性、策略與公開說明書一致性、Active Share、上限寬限、官方均價／作廢與實際成交|',
'',
'上限超限若可能涉及價格漂移，會保留待寬限歷史查核，不擅自宣告取消資格。前收盤預估不能保證當日均價成交後仍符合現金限制。官方語義與實際資料未接上前，ready_to_submit 固定為 false。','',
'## 測試與真實資料的界線','',
'官方正例本身註明行情是推估範例。測試使用其示範格式與另建的軟體測試價格／帳戶，包含應收資產與缺值等邊界；它們不是歷史行情，不併入任何策略績效，不拿來代替TEAM_11693帳戶。測試資料只保留在Notebook中，沒有另存成看似可上傳的正式D-Plan檔案。',
'',
'DataPack規格書內範例的schema_version為3.0，D-Plan為4.3。原型依本機已核對格式處理，不能猜正式DataPack也叫4.3；若實際版本不同，會標示需重新核對。參考官方格式通過也不代表該示範文本的市場論述已被驗證。','',
'## 取得真正DataPack後的接續方式','',
'1. 確認檔案屬TEAM_11693，以及trade_date、as_of_date、missing、account與holdings；初始真值現金帳戶與尚未結算的null帳戶分開。',
'2. 由可信官方交易日曆提供前交易日，配對官方股票池、前收盤價與收件時點；不以普通工作日減一天代替交易日曆。',
'3. 將凍結的研究配置及真正帳戶機械換算成決策／股數，送入本機驗證函式；錯誤拒絕、缺資料等待、通過仍保留官方檢查。',
'4. 再核對Active Share、公開說明書一致性、內容證據、官方最新規則與結算結果；沒有這些資料不宣稱正式就緒。','',
'主辦方目前尚無DataPack可下載，因此實際帳戶演練仍等待外部資料。這一步沒有新增研究候選、調整策略參數、啟用LLM或恢復排程。','',
'## 35個測試案例','',
'|案例|結果|偵測代碼|','|---|---|---|']
for row in dplan_test_results_v1:lines.append('|'+row['name']+'|'+row['status']+'|'+(', '.join(row['codes']) or '符合預期')+'|')
lines+=['','[可機讀驗證結果](TEAM_11693_DPlan離線驗證結果_2026-10-15.json)。函式與成功測試輸出保存於原研究Notebook；保留kernel，未執行Restart and Run All。']
report_path=dplan_root/'TEAM_11693_DPlan離線驗證說明_2026-10-15.md'
report_path.write_text('\n'.join(lines)+'\n',encoding='utf-8')
state_path=dplan_root/'A策略_無人值守研究狀態.json';state=json.loads(state_path.read_text(encoding='utf-8'))
state['offline_DPlan_validation']={'status':'technical_prototype_tested_actual_account_pending','date':'2026-10-15','team_id':'TEAM_11693','test_count':35,'report':report_path.name,'results':result_path.name,'notebook_cells':[174,178,177,179],'live_DataPack_used':False,'officially_ready':False,'submitted':False}
state['current_node']=None;state['status']='research_converged_offline_DPlan_checks_tested_waiting_official_DataPack';state['last_completed_action']='Notebook DPlan4.3 schema/reference/account/lot/quantity/cash prototype passed35 isolated software tests; no actual DataPack or order submission.'
state['updated_at']=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=8))).isoformat()
for filename in [report_path.name,result_path.name]:
    if filename not in state['final_artifacts']:state['final_artifacts'].append(filename)
state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
workflow=dplan_root/'A策略_無人值守研究流程.md';text=workflow.read_text(encoding='utf-8')
note='## 官方資料開放前準備：D-Plan離線驗證原型（2026-10-15）\n\n[驗證說明與35個案例](TEAM_11693_DPlan離線驗證說明_2026-10-15.md)。在原Notebook內完成官方schema4.3、引用、時點、帳戶缺漏、千股、公式股數及預估資金檢查。測試是明確標示的軟體案例，不是歷史行情、真正DataPack或正式D-Plan。所有結果ready_to_submit仍false，來源內容／Active Share／官方規則及成交層待核對。實際帳戶演練等待主辦方DataPack開放；研究參數與LLM暫緩決定不變，排程保持暫停。\n\n'
if '## 官方資料開放前準備：D-Plan離線驗證原型' not in text:workflow.write_text(text.split('\n',1)[0]+'\n\n'+note+text.split('\n',1)[1],encoding='utf-8')
print('DPLAN_PROTOTYPE_REPORT_AND_STATE_SAVED',report_path.name,result_path.name)


In [ ]:
# Strategy-only gap audit; operational tasks2/3/4 explicitly deferred by user
import sys,json,re,datetime
from research_layout import ResearchPath as Path
from collections import Counter
print('Retained kernel',sys.executable)
for n in ['full_pool_announcements_v5','evidence_v6','quarter_by_stock_v6','results_v6','v4_saved']:
    obj=globals().get(n);print(n,type(obj).__name__,len(obj) if hasattr(obj,'__len__') else None)
gap_cutoff=datetime.datetime(2025,10,28,8,55)
for sid in ['2337','6223']:
    print('GAP_STOCK',sid)
    for a in full_pool_ann_by_stock_v5[sid]:
        if not re.search(r'114|2025',a.get('subject') or ''):continue
        if not re.search(r'財務|財報|更正|自結',a.get('subject') or ''):continue
        if str(a.get('announce_date'))>'2025-10-28':continue
        print({k:str(a.get(k)) for k in ['stock_id','ann_id','announce_date','announce_time','subject','source']},'effective',str(prestart_stamp(a)))
        if sid=='2337' and '6cc7fac0e6fc0fe2ecc68ab196786f681f5ca79b'==a['ann_id']:print('2337 formal content',a['content'][:6500])
print('Unresolved company actions',v4_saved['unresolved_events'])
state_gap=json.loads((recovery_root/'A策略_無人值守研究狀態.json').read_text(encoding='utf-8'))
print('Retry-related state keys',[k for k in state_gap if any(s in k for s in ['attempt','retry','limit'])])
print('2024 residual initials',[r for r in initial_a25_comparison_v6 if r['date']=='2024-10-29' and r['new_status']!='verified_quarter'])
print('STRATEGY_GAP_PRIORITIES_READY')


In [ ]:
# Exact source parsing and cached capital-action evidence, no new requests
import ast
for book in [current_nb,recovery_base_nb,recovery_weight_nb]:
    for cell in book['cells']:
        source=''.join(cell.get('source',[]))
        if 'def prestart_stamp(' not in source:continue
        tree=ast.parse(source)
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name=='prestart_stamp':print('STAMP_PARSER',ast.get_source_segment(source,node))
for record in initial_a25_comparison_v6:
    if record['date']!='2024-10-29' or record['new_status']=='verified_quarter':continue
    for aid in record['pending_ids']:
        a=next(a for a in full_pool_announcements_v5 if a['ann_id']==aid)
        print('2024_PENDING_ORIGINAL',a['stock_id'],a['ann_id'],a['subject'],str(prestart_stamp(a)),'\n',a['content'][:5500])
missing_stamp_formals=[a for a in full_pool_announcements_v5 if pure_financial_amounts_v1(a) and prestart_stamp(a) is None]
print('Unparsed formal time samples',[{k:str(a.get(k)) for k in ['stock_id','announce_date','announce_time','ann_id']} for a in missing_stamp_formals[:25]])
company_cache=json.loads((recovery_root/'A策略_公司行動官方核對_2026-10-14.json').read_text(encoding='utf-8'))
print('Company cache keys',list(company_cache))
for k in ['detail_reports','unresolved_units']:
    if k in company_cache:
        print(k,'count',len(company_cache[k]))
        if k=='detail_reports':print('First action example',next(iter(company_cache[k].values())))
print('STRATEGY_GAP_SOURCE_DETAILS_READY')


In [ ]:
# Twelve capital events: existing-holder entitlement versus issuance totals; retain exact official fields
for event in v4_saved['unresolved_events']:
    key=event['stock_id']+'|'+event['date'];src=company_cache['detail_reports'][key];doc=src['document']
    print('CAPITAL_EVENT',key,dict(zip(doc['fields'],doc['data'][0])))
print('Remaining invalid price/action markers',[{'stock_id':stock_ids[j],'date':str(dates[t]),'before_close':float(close[t-1,j]),'close':float(close[t,j]),'raw_total_return':float(returns[t,j])} for t,j in np.argwhere(bad_action_v5)])
print('STRATEGY_CAPITAL_EVIDENCE_READY')


In [ ]:
# Public primary-source lookup for the sole remaining non-dividend price discontinuity
import requests,hashlib
gap_primary_cache_path=recovery_root/'A策略_策略資料缺口原始來源_v7_2026-10-15.json'
gap_primary_cache=json.loads(gap_primary_cache_path.read_text(encoding='utf-8')) if gap_primary_cache_path.exists() else {'requests':{},'failures':{}}
for name,url in [('igs_dividends','https://www.igs.com.tw/investors/shareholder/26/'),('tpex_cal_page','https://www.tpex.org.tw/zh-tw/announce/market/ex/cal.html')]:
    if name not in gap_primary_cache['requests']:
        try:
            response=requests.get(url,timeout=25);response.raise_for_status()
            gap_primary_cache['requests'][name]={'url':response.url,'body':response.content.decode('utf-8'),'sha256':hashlib.sha256(response.content).hexdigest(),'use':'historical action verification, not first-publication evidence'}
        except Exception as e:gap_primary_cache['failures'][name]={'attempts':1,'error':type(e).__name__}
        gap_primary_cache_path.write_text(json.dumps(gap_primary_cache,ensure_ascii=False,indent=2),encoding='utf-8')
    if name=='tpex_cal_page' and name in gap_primary_cache['requests']:
        body=gap_primary_cache['requests'][name]['body']
        print('TPEX page API declarations',re.findall(r'data-api=["\']([^"\']+)',body))
        print('TPEX query fields',[line.strip() for line in body.splitlines() if any(t in line for t in ['data-query','name=','data-api','data-field','startDate','endDate'])][:35])
for a in full_pool_ann_by_stock_v5['3293']:
    if str(a.get('announce_date'))[:7]=='2024-07' and re.search('除權|除息|增資',a['subject']):print('Retained original3293',a['ann_id'],str(prestart_stamp(a)),a['subject'],'\n',a['content'][:4800])
print('STRATEGY_PRIMARY_SOURCE_PAGE_CACHED')


In [ ]:
# Inspect actual official API pattern and form serialization; no guessed dates or prices
for name,url in [('tpex_main_script','https://www.tpex.org.tw/rsrc/js/main.js'),('tpex_table_script','https://www.tpex.org.tw/rsrc/js/tables.js')]:
    if name not in gap_primary_cache['requests']:
        try:
            response=requests.get(url,timeout=25);response.raise_for_status()
            gap_primary_cache['requests'][name]={'url':response.url,'body':response.content.decode('utf-8')}
        except Exception as e:gap_primary_cache['failures'][name]={'attempts':1,'error':type(e).__name__}
        gap_primary_cache_path.write_text(json.dumps(gap_primary_cache,ensure_ascii=False,indent=2),encoding='utf-8')
    if name in gap_primary_cache['requests']:
        body=gap_primary_cache['requests'][name]['body']
        contexts=[]
        for pattern in ['API_PATTERN','startDate','endDate','serialize','response','/www/']:
            contexts += [body[max(0,m.start()-90):m.end()+150] for m in list(re.finditer(re.escape(pattern),body))[:2]]
        print('API_CONTRACT',name,contexts)
print('TPEX_OFFICIAL_QUERY_CONTRACT_READY')


In [ ]:
# Exact endpoint from official page and main script; narrow historical day only
key='tpex_exDailyQ_20240726'
if key not in gap_primary_cache['requests']:
    try:
        response=requests.get('https://www.tpex.org.tw/www/zh-tw/bulletin/exDailyQ',params={'startDate':'2024/07/26','endDate':'2024/07/26','response':'json'},timeout=30);response.raise_for_status()
        doc=response.json();gap_primary_cache['requests'][key]={'url':response.url,'document':doc,'page_contract':'cal.html action bulletin/exDailyQ, main.js /www/{LANG}/{ACTION}'}
    except Exception as e:gap_primary_cache['failures'][key]={'attempts':1,'error':type(e).__name__}
    gap_primary_cache_path.write_text(json.dumps(gap_primary_cache,ensure_ascii=False,indent=2),encoding='utf-8')
doc=gap_primary_cache['requests'].get(key,{}).get('document',{})
print('Official result schema',list(doc))
if 'tables' in doc:
    for table in doc['tables']:
        print('Fields',table.get('fields'),'data count',len(table.get('data',[])))
        print('IGS exact result',[row for row in table.get('data',[]) if any(str(x).strip()=='3293' for x in row)])
else:print('Result excerpt',str(doc)[:1600])
print('TPEX_ONE_DAY_RESULT_CACHED')


In [ ]:
# Check returned query date before trusting any historical record
print('Historical query response',gap_primary_cache['requests']['tpex_exDailyQ_20240726']['document'])
key='tpex_exDailyQ_20240724_26'
if key not in gap_primary_cache['requests']:
    try:
        response=requests.get('https://www.tpex.org.tw/www/zh-tw/bulletin/exDailyQ',params={'startDate':'2024/07/24','endDate':'2024/07/26','response':'json'},timeout=30);response.raise_for_status()
        gap_primary_cache['requests'][key]={'url':response.url,'document':response.json()}
    except Exception as e:gap_primary_cache['failures'][key]={'attempts':1,'error':type(e).__name__}
    gap_primary_cache_path.write_text(json.dumps(gap_primary_cache,ensure_ascii=False,indent=2),encoding='utf-8')
doc=gap_primary_cache['requests'].get(key,{}).get('document',{})
print('Range response',doc)
print('Adjacent actual snapshot sessions',[str(d) for d in dates if datetime.date(2024,7,22)<=d<=datetime.date(2024,7,29)])
print('TPEX_DATE_RANGE_VERIFICATION_READY')


In [ ]:
# Source repair v7: parse recorded single-digit hours; explicitly bounded correction and scheduled-notice rules
prestart_stamp_before_v7=prestart_stamp
def prestart_stamp(ann):
    try:
        day=ann['announce_date'];day=datetime.date.fromisoformat(day) if isinstance(day,str) else day
        if isinstance(day,datetime.datetime):day=day.date()
        if not isinstance(day,datetime.date):return None
        value=ann.get('announce_time')
        if value is None:return None
        if isinstance(value,datetime.timedelta):
            seconds=value.total_seconds()
            if not 0<=seconds<86400:return None
            return datetime.datetime.combine(day,datetime.time())+value
        if isinstance(value,datetime.time):
            return datetime.datetime.combine(day,value) if value.tzinfo is None else None
        match=re.fullmatch(r'(\d{1,2}):([0-5]\d):([0-5]\d)(?:\.(\d{1,6}))?',str(value))
        if match is None or int(match[1])>23:return None
        return datetime.datetime.combine(day,datetime.time(int(match[1]),int(match[2]),int(match[3]),int((match[4] or '').ljust(6,'0') or 0)))
    except (ValueError,TypeError,KeyError,OverflowError):return None
time_repairs_v7=[];unchanged_valid_times_v7=0
for a in full_pool_announcements_v5:
    old,new=prestart_stamp_before_v7(a),prestart_stamp(a)
    if old is not None:assert new==old;unchanged_valid_times_v7+=1
    elif new is not None:time_repairs_v7.append({'stock_id':a['stock_id'],'ann_id':a['ann_id'],'recorded_date':str(a['announce_date']),'recorded_time':str(a['announce_time']),'parsed':str(new)})
def correction_scope_v7(a):
    s=unicodedata.normalize('NFKC',a.get('subject') or '');b=unicodedata.normalize('NFKC',a.get('content') or '')
    if not re.search(r'更正|補充',s):return None
    explicit=re.search(r'(?:此|本次)更正[^。；\n]{0,80}損益(?:表)?無影響',b)
    if explicit:return {'scope':'issuer_explicit_no_profit_loss_effect','evidence':explicit[0],'applies_to':'operating margin only; not complete financial-report endorsement'}
    before=re.search(r'7\.[^\n]*更正前[^\n]*\n(.*?)\n8\.',b,re.S);after=re.search(r'8\.[^\n]*更正後[^\n]*\n(.*?)\n9\.',b,re.S)
    if before and after:
        old=re.sub(r'\s+','',before[1]);new=re.sub(r'\s+','',after[1])
        if re.fullmatch(r'\d{3,4}年上半年每股盈餘之計算股數誤植為[\d,]+仟股',old) and re.fullmatch(r'\d{3,4}年上半年每股盈餘之計算股數應為[\d,]+仟股',new):
            return {'scope':'explicit_EPS_denominator_only','evidence':[old,new],'applies_to':'quarter operating profit/revenue rule only; EPS values not approved'}
    return None
def original_kind_v7(a):
    kind=original_kind_v6(a)
    if kind=='financial_correction_unresolved' and correction_scope_v7(a):return 'correction_scope_outside_operating_margin'
    body=unicodedata.normalize('NFKC',a.get('content') or '')
    if kind=='quarter_result_or_unresolved' and '董事會預計召開日期' in body and '預計提報董事會' in body:return 'scheduled_not_actual'
    return kind
notice_kinds_v7={a['ann_id']:original_kind_v7(a) for a in full_pool_announcements_v5}
quarter_records_v7=[];quarter_by_stock_v7=defaultdict(list)
for a in full_pool_announcements_v5:
    r=explicit_quarter_extended_v6(a)
    if r:quarter_records_v7.append(r);quarter_by_stock_v7[r['stock_id']].append(r)
scope_repairs_v7=[{'stock_id':a['stock_id'],'ann_id':a['ann_id'],'known_at':str(prestart_stamp(a)),**correction_scope_v7(a)} for a in full_pool_announcements_v5 if correction_scope_v7(a)]
print('Recorded-time repairs',len(time_repairs_v7),'existing valid times unchanged',unchanged_valid_times_v7,'formal repairs',sum(any(r['ann_id']==a['ann_id'] for r in quarter_records_v7) for a in time_repairs_v7))
print('Correction scope repairs',scope_repairs_v7)
print('Missing raw time remains',[a['ann_id'] for a in full_pool_announcements_v5 if a['stock_id']=='1785' and a['ann_id']=='1e7bedb1f6247aead99d5ee27ff436140393df68' and prestart_stamp(a) is None])
print('Holiday raw actions',[row[:6] for row in raw_data['dividend'] if datetime.date(2024,7,24)<=row[1]<=datetime.date(2024,7,26)])
print('TWSE cached reports keys',list(company_cache['reports'])[:8])
print('STRATEGY_V7_SOURCE_REPAIRS_READY')


In [ ]:
# Audit all affected official-universe stocks across closure dates, not only3293
t=di[datetime.date(2024,7,26)]
print('TPEX closure records in150',[{'source':dict(zip(table['fields'],row)),'matrix_cash':float(cash_div_v5[t,si[row[1]]]),'matrix_stock':float(stock_div_v5[t,si[row[1]]]),'prior_close':float(close[t-1,si[row[1]]]),'first_close':float(close[t,si[row[1]]])} for table in gap_primary_cache['requests']['tpex_exDailyQ_20240724_26']['document']['tables'] for row in table['data'] if row[1] in si])
sample=company_cache['reports']['2024-07-23']
print('TWSE cache document shape',list(sample))
print('TWSE cache source',sample.get('url'))
doc=sample.get('document',sample)
print('TWSE doc keys',list(doc))
if 'data' in doc:
    print('TWSE fields',doc.get('fields'),'rows',len(doc['data']))
    print('Holiday-date rows',[r for r in doc['data'] if any(str(v).strip() in ['113年07月24日','113年07月25日','113/07/24','113/07/25'] for v in r)])
print('HOLIDAY_ACTION_SCOPE_AUDIT_READY')


In [ ]:
# Official closure-day action coverage for TWSE150 as well as TPEx150
key='twse_holiday_20240724_25'
if key not in gap_primary_cache['requests']:
    try:
        response=requests.get('https://www.twse.com.tw/rwd/zh/exRight/TWT49U',params={'startDate':'20240724','endDate':'20240725','response':'json'},timeout=25);response.raise_for_status()
        gap_primary_cache['requests'][key]={'url':response.url,'document':response.json()}
    except Exception as e:gap_primary_cache['failures'][key]={'attempts':1,'error':type(e).__name__}
    gap_primary_cache_path.write_text(json.dumps(gap_primary_cache,ensure_ascii=False,indent=2),encoding='utf-8')
doc=gap_primary_cache['requests'].get(key,{}).get('document',{})
holiday_twse_rows_v7=[row for row in doc.get('data',[]) if str(row[1]).strip() in si]
print('TWSE closure query',doc.get('stat'),doc.get('strDate'),doc.get('endDate'),'pool records',len(holiday_twse_rows_v7))
for row in holiday_twse_rows_v7:
    sid=row[1].strip();y,m,d=map(int,re.findall(r'\d+',row[0]));y=y+1911 if y<1911 else y;date=datetime.date(y,m,d)
    assert date in [datetime.date(2024,7,24),datetime.date(2024,7,25)]
    k='twse_closure_detail_'+sid+'_'+date.isoformat()
    if k not in gap_primary_cache['requests']:
        try:
            response=requests.get('https://www.twse.com.tw/rwd/zh/exRight/TWT49UDetail',params={'STK_NO':sid,'T1':date.strftime('%Y%m%d'),'response':'json'},timeout=20);response.raise_for_status()
            gap_primary_cache['requests'][k]={'url':response.url,'document':response.json()}
        except Exception as e:gap_primary_cache['failures'][k]={'attempts':1,'error':type(e).__name__}
        gap_primary_cache_path.write_text(json.dumps(gap_primary_cache,ensure_ascii=False,indent=2),encoding='utf-8')
    result=gap_primary_cache['requests'].get(k,{}).get('document',{})
    print('TWSE closure action',sid,date,'reference row',row[:5],'exact units',result.get('data'))
print('TWSE_CLOSURE_ACTION_DETAILS_CACHED')


In [ ]:
# Engine v7 verified closure-date alignment, all8 in-universe actions, same receivable model
closure_sources_v7=['https://www.twse.com.tw/staticFiles/news/news/tsecnews/8a8216d6906d4d340190df83a6ac0170.pdf','https://www.twse.com.tw/staticFiles/news/news/tsecnews/8a8216d6906d4d340190e4a4ef640174.pdf']
holiday_actions_v7=[];effective_day=datetime.date(2024,7,26);t=di[effective_day]
assert dates[t-1]==datetime.date(2024,7,23) and datetime.date(2024,7,24) not in di and datetime.date(2024,7,25) not in di
for table in gap_primary_cache['requests']['tpex_exDailyQ_20240724_26']['document']['tables']:
    for row in table['data']:
        sid=row[1].strip()
        if sid not in si:continue
        y,m,d=map(int,row[0].split('/'));day=datetime.date(y+1911,m,d)
        if day==effective_day:continue
        cash=float(row[13]);stock_ratio=float(row[14])/1000;paid=float(row[15]);before=float(row[3]);reference=float(row[4])
        assert paid==0
        holiday_actions_v7.append({'stock_id':sid,'official_ex_date':str(day),'effective_price_session':str(effective_day),'cash':cash,'stock_ratio':stock_ratio,'before':before,'reference':reference,'source_url':gap_primary_cache['requests']['tpex_exDailyQ_20240724_26']['url'],'basis':'TPEx explicit cash and free shares per1000; official closure dates aligned to first actual session, not inferred from price drop'})
for row in holiday_twse_rows_v7:
    sid=row[1].strip();y,m,d=map(int,re.findall(r'\d+',row[0]));day=datetime.date(y+1911 if y<1911 else y,m,d)
    source=gap_primary_cache['requests']['twse_closure_detail_'+sid+'_'+day.isoformat()];detail=source['document']['data'][0]
    def val(text,unit):
        m=re.fullmatch(r'\s*([\d,.]+)\s*'+re.escape(unit)+r'\s*',text)
        if not m:raise ValueError('unverified_explicit_unit')
        return float(m[1].replace(',',''))
    cash=val(detail[2],'元／股');stock_ratio=val(detail[4],'股')/1000
    assert val(detail[6],'股')==0
    holiday_actions_v7.append({'stock_id':sid,'official_ex_date':str(day),'effective_price_session':str(effective_day),'cash':cash,'stock_ratio':stock_ratio,'before':float(row[3]),'reference':float(row[4]),'source_url':source['url'],'basis':'TWSE exact explicit per-share cash and shareholder free shares per1000; employee allotment not assigned to portfolio'})
assert len(holiday_actions_v7)==8 and len({a['stock_id'] for a in holiday_actions_v7})==8
cash_div_v7=cash_div_v5.copy();stock_div_v7=stock_div_v5.copy();signal_returns_v7=signal_returns_v5.copy();bad_action_v7=bad_action_v5.copy()
for a in holiday_actions_v7:
    j=si[a['stock_id']]
    assert np.isclose(close[t-1,j],a['before'],atol=.001)
    expected=(a['before']-a['cash'])/(1+a['stock_ratio'])
    assert abs(expected-a['reference'])<=max(.03,a['before']*.0002)
    assert cash_div_v7[t,j]==0 and stock_div_v7[t,j]==0,'Existing booked action must not be double-counted'
    cash_div_v7[t,j]=a['cash'];stock_div_v7[t,j]=a['stock_ratio']
    signal_returns_v7[t,j]=close[t,j]/a['reference']-1
    bad_action_v7[t,j]=False
    a['validation']={'before_price_matches':True,'explicit_amount_reference_check':True,'receivable_cash_and_shares_not_spendable':True}
assert bad_action_v7.sum()==12 and not bad_action_v7[t,si['3293']]
gap_action_export_v7={'actions_applied':holiday_actions_v7,'closure_evidence':closure_sources_v7,'cash_and_stock_timing':'same prior receivable model; no new capital subscription assumptions','still_unmodelled_paid_capital':v4_saved['unresolved_events'],'prior_anomaly3293':'verified ex-right35TWD cash +1000free shares per1000, reference715, not a46pct market crash'}
(recovery_root/'A策略_休市除權息對齊查核_v7_2026-10-15.json').write_text(json.dumps(gap_action_export_v7,ensure_ascii=False,indent=2),encoding='utf-8')
print('Eight verified actions',[{k:a[k] for k in ['stock_id','official_ex_date','effective_price_session','cash','stock_ratio','reference']} for a in holiday_actions_v7])
print('Remaining price action flags',int(bad_action_v7.sum()),'all12 are paid capital events, no unidentified crash remains')
print('STRATEGY_V7_EIGHT_ACTIONS_APPLIED')


In [ ]:
# v7 uniform evidence, keep v6 in memory for paired source comparison
source=''.join(json.loads((recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').read_text(encoding='utf-8'))['cells'][169]['source'])
tree=ast.parse(source);node=next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name=='quarter_asof_v6')
function=ast.get_source_segment(source,node).replace('quarter_asof_v6','quarter_asof_v7').replace('quarter_by_stock_v6','quarter_by_stock_v7').replace('notice_kinds_v6','notice_kinds_v7')
exec(function,globals())
evidence_v7={};gap_daily_coverage_v7=[];signal_repairs_v7=[]
for t in range(61,len(dates)):
    cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));counts=Counter()
    for sid in stock_ids:
        r=quarter_asof_v7(sid,cutoff);evidence_v7[(t,sid)]=r;counts[r['status']]+=1
        old=evidence_v6[(t,sid)]
        old_sign=0 if old['status']!='verified_quarter' else 1 if old['delta_pp']>=1 else -1 if old['delta_pp']<=-1 else 0
        new_sign=0 if r['status']!='verified_quarter' else 1 if r['delta_pp']>=1 else -1 if r['delta_pp']<=-1 else 0
        if old_sign!=new_sign:signal_repairs_v7.append({'date':str(dates[t]),'stock_id':sid,'old_sign':old_sign,'new_sign':new_sign,'source_ids':r.get('source_ids',[])})
        if r['status']=='verified_quarter':assert all(datetime.datetime.fromisoformat(s)<cutoff for s in r['known_at'])
    gap_daily_coverage_v7.append({'date':str(dates[t]),'status':dict(counts)})
gap_initial_comparison_v7=[]
for day in ['2024-10-29','2025-10-28']:
    t=di[datetime.date.fromisoformat(day)];ids=next(r['stocks'] for r in coverage_a_decisions_v5 if r['date']==day)
    for sid in ids:
        r=evidence_v7[(t,sid)]
        gap_initial_comparison_v7.append({'date':day,'stock_id':sid,'old_status':evidence_v6[(t,sid)]['status'],'new_status':r['status'],'quarter':r.get('quarter'),'delta_pp':r.get('delta_pp'),'source_ids':r.get('source_ids',[]),'pending_ids':r.get('pending_ids',[])})
gap_source_export_v7={'time_repairs':time_repairs_v7,'preserved_valid_timestamps':unchanged_valid_times_v7,'correction_scope_repairs':scope_repairs_v7,'initial_A25':gap_initial_comparison_v7,'daily_coverage':gap_daily_coverage_v7,'changed_sign_records':signal_repairs_v7,'limits':'no time invented for rawnull; only explicitly bounded correction scope clears operating-margin gate; remaining true quarter/corrections unknown'}
(recovery_root/'A策略_時間格式與更正範圍查核_v7_2026-10-15.json').write_text(json.dumps(gap_source_export_v7,ensure_ascii=False,indent=2),encoding='utf-8')
print('A25 coverage after source repair',dict(Counter((r['date'],r['new_status']) for r in gap_initial_comparison_v7)))
print('2025 former missing2337',next(r for r in gap_initial_comparison_v7 if r['date']=='2025-10-28' and r['stock_id']=='2337'))
print('Full-pool changed-sign rows',len(signal_repairs_v7),'stocks',len({r['stock_id'] for r in signal_repairs_v7}))
print('STRATEGY_V7_ASOF_SAVED')


In [ ]:
# v7 fixed-method engine revalidation: changes are source corrections, no parameter search
fixed_v7_definition={'raw_snapshot_sha256':v4_saved['raw_snapshot_sha256'],'changes':['8 verified closure-date action alignment','recorded single-digit hour parsing','explicitly bounded non-operating corrections and future-board notices'],'scope':'150same stocks,650start windows,24days,5day adjustment,90pct,1000shares,samefees','broad_methods':v4_saved['fixed_methods'],'financial_rule':'same Q2/Q1 orQ3/Q2 operating-margin change; uniform repaired quarter evidence','announcement_rule':'same prestart +/-1pp,1+/-.25 and .5mild+.25financial+.25revenue; frozen atstart','paid_capital':'12events still excluded, subscription/lapse policy not invented','family_count':43,'family_reason':'prior26 plus14 updated broad nonA plus2 updated announcement versions plus1 matched diagnostic; conservatively count all looks','not_tasks':'DataPack,DPlan,official rule clarification and operational rehearsal deferred by user; LLM deferred'}
(recovery_root/'A策略_關鍵資料修正重驗證定義_v7_2026-10-15.json').write_text(json.dumps(fixed_v7_definition,ensure_ascii=False,indent=2),encoding='utf-8')
scores_v7={k:np.full(shape,np.nan) for k in ['REF','A','B','C','PURE_FLOW','PURE_LOWVOL','PURE_FINANCIAL','PURE_REVENUE','REV5']}
for q in range(60,len(dates)):
    rr=signal_returns_v7[q-59:q+1];ff=flow[q-19:q+1];vv=volume[q-19:q+1]
    eligible=np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(ff)&np.isfinite(vv)&(vv>0),axis=0)&(close[q]>0)&~np.any(bad_action_v7[q-59:q+1],axis=0)
    momentum=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,eligible)+.5*percentile(np.prod(1+rr,axis=0)-1,eligible)
    fscore=np.sum(ff,axis=0)/np.sum(vv,axis=0);vol=np.std(rr[-20:],axis=0)
    scores_v7['A'][q]=momentum;scores_v7['B'][q]=.75*momentum+.25*percentile(fscore,eligible);scores_v7['C'][q]=.75*momentum+.25*percentile(-vol,eligible)
    scores_v7['PURE_FLOW'][q]=percentile(fscore,eligible);scores_v7['PURE_LOWVOL'][q]=percentile(-vol,eligible)
    scores_v7['REF'][q]=percentile(np.divide(np.nansum(turnover[q-19:q+1],axis=0),np.sum(np.isfinite(turnover[q-19:q+1]),axis=0),out=np.full(shape[1],np.nan),where=np.sum(np.isfinite(turnover[q-19:q+1]),axis=0)>0),eligible)
    scores_v7['REV5'][q,eligible]=-(np.prod(1+rr[-5:],axis=0)-1)[eligible]
scores_v7['AH']=scores_v7['A'].copy();scores_v7['BH']=scores_v7['B'].copy()
for t in range(61,len(dates)):
    eligible=np.isfinite(scores_v7['A'][t-1])
    raw=np.array([evidence_v7[(t,sid)].get('delta_pp',np.nan) if evidence_v7[(t,sid)]['status']=='verified_quarter' else np.nan for sid in stock_ids])
    scores_v7['PURE_FINANCIAL'][t-1]=percentile(raw,eligible&np.isfinite(raw))
    records=[pure_revenue_evidence_v1[(t,sid)] for sid in stock_ids]
    mm=np.array([r.get('mom',np.nan) if r['status']=='verified_original_known_available_time' else np.nan for r in records]);yy=np.array([r.get('yoy',np.nan) if r['status']=='verified_original_known_available_time' else np.nan for r in records]);keep=eligible&np.isfinite(mm)&np.isfinite(yy)
    scores_v7['PURE_REVENUE'][t-1]=.5*percentile(mm,keep)+.5*percentile(yy,keep)
weight_prefix=cell_source(recovery_weight_nb,3).split('weight_experiment_results=')[0].replace('inspect.getsource(simulate)','base_src')
exec(weight_prefix,globals())
def engine_v7_allocation(t,selected,allocation):
    if allocation=='equal':return np.full(COUNT,INVESTED/COUNT)
    if allocation=='mild':return INVESTED*np.linspace(1.25,.75,COUNT)/COUNT
    vol=np.std(signal_returns_v7[t-20:t,selected],axis=0);assert np.all(np.isfinite(vol)&(vol>0))
    raw=INVESTED/vol/(1/vol).sum()
    if allocation=='invvol':raw=.5*np.full(COUNT,INVESTED/COUNT)+.5*raw
    return weight_bounded_normalize(raw,total=INVESTED,lower=.018,upper=.054)
saved_current=json.loads((recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').read_text(encoding='utf-8'))
template=''.join(saved_current['cells'][75]['source']);engine_part=template[template.index('src_v3='):template.index("if 'engine_v3_results' not in globals():")]
engine_part=engine_part.replace('_v3','_v7');good_exec_v2=good_exec_v5
exec(engine_part,globals())
engine_v7_results={};v7_price_export={'definition':fixed_v7_definition,'status':'in_progress','windows':{}}
v7_price_export_path=recovery_root/'A策略_關鍵資料修正逐窗口_v7_2026-10-15.json'
for name,(strategy,allocation,tolerance) in v4_saved['fixed_methods'].items():
    rows=[simulate_engine_v7(int(s),strategy,allocation,tolerance=tolerance) for s in window_starts];engine_v7_results[name]=rows
    v7_price_export['windows'][name]=clean_v5(rows)
    v7_price_export_path.write_text(json.dumps(v7_price_export,ensure_ascii=False,allow_nan=False),encoding='utf-8')
    print('V7_METHOD_SAVED',name,'valid',sum(not r['invalid'] and r['violations']==0 for r in rows),flush=True)
v7_price_export['status']='all_fixed_broad_methods_saved_inference_pending'
v7_price_export_path.write_text(json.dumps(v7_price_export,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('V7_BROAD_REVALIDATION_COMPLETE')


In [ ]:
# Same v7 engine for fixed prestart information candidates and matched control
def announcement_v7_allocation(t,selected,mode):
    start=announcement_start_v7
    if mode in ['equal','mild']:return engine_v7_allocation(t,selected,mode)
    if mode=='matched_neutral':return .5*engine_v7_allocation(t,selected,'mild')+.5*engine_v7_allocation(t,selected,'equal')
    f=[];rv=[]
    for j in selected:
        r=evidence_v7[(start,stock_ids[j])];sign=0
        if r['status']=='verified_quarter':sign=1 if r['delta_pp']>=1 else -1 if r['delta_pp']<=-1 else 0
        f.append(1+.25*sign)
        r=pure_revenue_evidence_v1[(start,stock_ids[j])];sign=0
        if r['status']=='verified_original_known_available_time':sign=1 if r['mom']>0 and r['yoy']>0 else -1 if r['mom']<0 and r['yoy']<0 else 0
        rv.append(1+.25*sign)
    wf=INVESTED*np.array(f)/sum(f);wr=INVESTED*np.array(rv)/sum(rv)
    if mode=='formal_prestart':return wf
    if mode=='revenue_prestart':return wr
    if mode=='combined_prestart':return .5*engine_v7_allocation(t,selected,'mild')+.25*wf+.25*wr
    raise ValueError('Undefined fixed information candidate')
announcement_src_v7=src_v7.replace('def simulate_engine_v7(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):','def simulate_announcement_v7(start,allocation,horizon=HORIZON,tolerance=0):\n    global announcement_start_v7\n    announcement_start_v7=start\n    strategy="A"').replace('engine_v7_allocation(','announcement_v7_allocation(')
assert 'def simulate_announcement_v7(' in announcement_src_v7
exec(announcement_src_v7,globals())
announcement_results_v7={}
for mode,name in [('formal_prestart','A_formal_operating_prestart'),('revenue_prestart','A_revenue_prestart'),('combined_prestart','A_combined_prestart'),('matched_neutral','matched_neutral')]:
    rows=[simulate_announcement_v7(int(s),mode) for s in window_starts];announcement_results_v7[name]=rows
    print('V7_INFO_SAVED',name,'valid',sum(not r['invalid'] and r['violations']==0 for r in rows),flush=True)
    v7_price_export['windows'][name]=clean_v5(rows)
    v7_price_export_path.write_text(json.dumps(v7_price_export,ensure_ascii=False,allow_nan=False),encoding='utf-8')
for i in [0,200,400,649]:
    r=simulate_announcement_v7(int(window_starts[i]),'equal');base=engine_v7_results['A_equal'][i]
    for f in ['return','mdd','cost','turnover']:assert np.isclose(r[f],base[f],rtol=1e-12,atol=1e-12)
v7_price_export['status']='all_fixed_methods_and_information_saved_inference_pending'
v7_price_export_path.write_text(json.dumps(v7_price_export,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('V7_INFO_COMPLETE_IDENTICAL_ENGINE_CONTROLS_VERIFIED')


In [ ]:
# All updated methods on identical v7 accounting; family includes old and updated looks
stats_function_tree=ast.parse(''.join(json.loads((recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').read_text(encoding='utf-8'))['cells'][155]['source']))
node=next(n for n in stats_function_tree.body if isinstance(n,ast.FunctionDef) and n.name=='numerical_summary_v5')
stats_source=ast.unparse(node).replace('numerical_summary_v5','summary_v7').replace("v4_saved['windows'][name]","engine_v7_results[name]")
exec(stats_source,globals())
all_stats_v7={}
for name,rows in {**engine_v7_results,**{k:v for k,v in announcement_results_v7.items() if k!='matched_neutral'}}.items():
    if name=='A_equal':continue
    s=summary_v7(rows,engine_v7_results['A_equal'],family=44);all_stats_v7[name]=s
    print('V7_STAT',name,'pairs',s['pairs'],'meanpp',round(s['mean_difference_pp'],5),'CI24',np.round(s['block24_interval_pp'],5).tolist(),'CI48',np.round(s['block48_interval_pp'],5).tolist(),'support',s['supported'],flush=True)
matched_stats_v7=summary_v7(announcement_results_v7['A_combined_prestart'],announcement_results_v7['matched_neutral'],family=44)
matched_stats_v7['reference']='same50pctmild+50pctequal, same v7 engine, informational component neutral'
matched_stats_v7['reference_median_same_dates_pct']=matched_stats_v7.pop('A_median_same_dates_pct')
for r in matched_stats_v7['seasonal']:r['reference_return_pct']=r.pop('A_return_pct')
fixed_v7_definition['family_count']=44
fixed_v7_definition['family_reason']='prior26+14 updated broad nonA+3 updated prestart information+1 matched diagnostic, conservatively count every look'
print('V7_INFORMATION_INCREMENT',json.dumps(matched_stats_v7,ensure_ascii=False))
print('V7_PROVISIONAL_DETAIL',json.dumps(all_stats_v7['A_mild_tolerance'],ensure_ascii=False))
print('V7_INFO_DETAILS',json.dumps({n:all_stats_v7[n] for n in ['A_formal_operating_prestart','A_revenue_prestart','A_combined_prestart']},ensure_ascii=False))
supported_v7=[n for n,s in all_stats_v7.items() if s['supported']]
v7_stats_export={'definition':fixed_v7_definition,'stats':all_stats_v7,'matched_information_diagnostic':matched_stats_v7,'supported':supported_v7,'conclusion':'no supported improvement' if not supported_v7 else 'candidates meet local historical criteria; external validity still limited','sample_note':'overlapping windows, not independent trials; 2seasonal years, no untouched holdout','operational_work':'tasks2/3/4 deferred; no DataPack or DPlan work this round; LLM deferred'}
(recovery_root/'A策略_關鍵資料修正統一統計_v7_2026-10-15.json').write_text(json.dumps(v7_stats_export,ensure_ascii=False,indent=2),encoding='utf-8')
(recovery_root/'A策略_關鍵資料修正重驗證定義_v7_2026-10-15.json').write_text(json.dumps(fixed_v7_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v7_price_export['definition']=fixed_v7_definition;v7_price_export['status']='saved_with_completed_inference'
v7_price_export_path.write_text(json.dumps(v7_price_export,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('V7_UNIFIED_INFERENCE_SAVED',supported_v7)


In [ ]:
# Final strategy-only gap report, all-point comparison, boundary checks and scope persistence
assert len(holiday_actions_v7)==8 and int(bad_action_v7.sum())==12
assert len(time_repairs_v7)==574 and unchanged_valid_times_v7==33773
assert prestart_stamp({'announce_date':'2025-07-31','announce_time':'6:21:15'})==datetime.datetime(2025,7,31,6,21,15)
assert prestart_stamp({'announce_date':'2025-07-31','announce_time':None}) is None
assert prestart_stamp({'announce_date':'2025-07-31','announce_time':'25:21:15'}) is None
assert quarter_asof_v7('2337',datetime.datetime(2025,7,31,6,21,14)).get('quarter')!=2
assert quarter_asof_v7('2337',datetime.datetime(2025,7,31,6,21,16))['status']=='verified_quarter'
assert quarter_asof_v7('6223',datetime.datetime(2025,10,28,8,55))['status']=='newer_quarter_or_correction_unresolved'
remaining_initial_v7=[r for r in gap_initial_comparison_v7 if r['new_status']!='verified_quarter']
assert {(r['date'],r['stock_id']) for r in remaining_initial_v7}=={('2024-10-29','3529'),('2025-10-28','6223')}
common_old_A_v7=np.array([not r['invalid'] and r['violations']==0 and not b['invalid'] and b['violations']==0 for r,b in zip(engine_v7_results['A_equal'],v4_saved['windows']['A_equal'])])
price_repair_check_v7={'old_A_valid':sum(not r['invalid'] and r['violations']==0 for r in v4_saved['windows']['A_equal']),'new_A_valid':sum(not r['invalid'] and r['violations']==0 for r in engine_v7_results['A_equal']),'same_old_new_A_pairs':int(common_old_A_v7.sum()),'A_mean_return_change_same_dates_pp':float(np.mean([r['return']-b['return'] for i,(r,b) in enumerate(zip(engine_v7_results['A_equal'],v4_saved['windows']['A_equal'])) if common_old_A_v7[i]])*100)}
for day in ['2024-10-29','2025-10-28']:
    t=di[datetime.date.fromisoformat(day)]
    assert np.array_equal(scores_v7['A'][t-1],scores_a_v5[t-1],equal_nan=True)
main_chart_rows_v7=[]
chart_methods=[('A等權',engine_v7_results['A_equal']),('A溫和動能',engine_v7_results['A_mild']),('A_mild_tolerance',engine_v7_results['A_mild_tolerance']),('A_formal_operating_prestart',announcement_results_v7['A_formal_operating_prestart']),('A_revenue_prestart',announcement_results_v7['A_revenue_prestart']),('A_combined_prestart',announcement_results_v7['A_combined_prestart'])]
for name,rows in chart_methods:
    for i,r in enumerate(rows):
        if not r['invalid'] and r['violations']==0:main_chart_rows_v7.append({'method':name,'start':str(dates[int(window_starts[i])]),'end':str(dates[int(window_starts[i])+23]),'return_pct':r['return']*100,'cost_pct':r['cost']*100,'mdd_pct':r['mdd']*100})
chart=(recovery_root/'A策略_開賽前資訊調權比較_v6_2026-10-15.html').read_text(encoding='utf-8')
begin=chart.index('const data=');end=chart.index(';const labels=',begin);chart=chart[:begin]+'const data='+json.dumps(main_chart_rows_v7,ensure_ascii=False)+chart[end:]
begin=chart.index('const labels=');end=chart.index(';const lo=',begin)
chart_labels={'A_mild_tolerance':'A動能＋略過','A_formal_operating_prestart':'A季度資訊','A_revenue_prestart':'A營收資訊','A_combined_prestart':'A資訊組合'}
chart=chart[:begin]+'const labels='+json.dumps(chart_labels,ensure_ascii=False)+';const names='+json.dumps([n for n,_ in chart_methods],ensure_ascii=False)+chart[end:]
chart=chart.replace('x=170+185*i','x=145+165*i').replace('A開賽前資訊調權比較 v6','A關鍵資料修正比較 v7')
chart=chart.replace('已納入可核對的季度自結與公告分類修正；單月損益另列，部分更正及季度原文仍未解析，LLM未啟用。','已修正公告時間、有限更正範圍及8筆休市除權息；12筆有償認購仍保留排除。單月損益另列，LLM未啟用。')
(recovery_root/'A策略_關鍵資料修正比較_v7_2026-10-15.html').write_text(chart,encoding='utf-8')
lines=['# A策略關鍵資料缺口查核與策略比較 v7','',
'日期：2026-10-15。本輪僅執行使用者指定①補完關鍵資料缺口；②正式帳戶／D-Plan、③官賽規則、④流程演練全部延後，LLM繼續暫緩。沒有新增策略參數、交易或排程。','',
'## 查核結果','',
'1. 公告時間格式：574筆原本解析失敗的紀錄有真實日期和單位數小時，已一致解析；33,773筆原本有效時間完全不變。缺原始時間的紀錄仍未知，沒有補造時間。旺宏2337的Q2公告因此恢復，並驗證公布前一秒／後一秒的邊界。',
'2. 更正範圍：僅在發行人明列不影響損益，或更正前後僅EPS計算股數時，解除「季度營業利益率」訊號的阻擋；不宣稱完整財報無風險，也不把修正套到EPS因子。旺矽6223的薪資成本附註更正仍未知。台積電董事會預計日期按預告處理。',
'3. 原價格異常3293已查明：官方櫃買資料列現金35元／股、每仟股無償1000股、參考價715元；7/24、25休市，原7/24權息在行情首個交易日7/26反映。前收1465到786不是可直接當成約46%市場崩跌的原始報酬。',
'4. 同一休市日期對齊查核一致涵蓋全150檔，共8筆：3293、6290、5483、8069、1519、2002、2884、3702；都核對前價、明列現金／股數及參考價。只加入可驗證應收帳務與訊號調整，未用權息價差倒造股利。',
'5. 12筆現金增資仍有原股東認購權、認購價與認購比例。它們是有償權利，不是可直接補成免費配股的漏資料；資金／權利取得與放棄、發放時程沒有完整模型，因此繼續排除，不私自新增認購策略。','',
'官方來源：[TPEx歷史計算結果](https://www.tpex.org.tw/www/zh-tw/bulletin/exDailyQ?startDate=2024%2F07%2F24&endDate=2024%2F07%2F26&response=json)、[TWSE歷史計算結果](https://www.twse.com.tw/rwd/zh/exRight/TWT49U?startDate=20240724&endDate=20240725&response=json)、[7/24休市公告]('+closure_sources_v7[0]+')、[7/25休市公告]('+closure_sources_v7[1]+')。原公告／回應快取、精確單位與推導已保存；本次查詢時間不冒稱事件首次發布時間。','',
'## 公告覆蓋與剩餘未知','',
'2024、2025開賽25檔各有24檔可核對季度本業。2024力旺3529有較新Q3財報申報完成公告，但保留原文沒列新季度數字；2025旺矽6223有薪資成本附註更正。兩者仍保持未知中性。這是固定季度營益率訊號覆蓋，不是每份完整財報／法說都已判讀。','',
f"A有效窗口由{price_repair_check_v7['old_A_valid']}增至{price_repair_check_v7['new_A_valid']}。同舊有效日期{price_repair_check_v7['same_old_new_A_pairs']}個配對，A平均費用後報酬因資料修正改變{price_repair_check_v7['A_mean_return_change_same_dates_pp']:+.4f}pp；不能將不同有效樣本的整體中位數差都稱策略進步。",'',
'## 同引擎的全部策略比較','',
'15個既有廣泛窗口方法、3個既有開賽前資訊配置與匹配診斷沿固定定義比較，每個650候選窗口。持股25、24交易日、五日調整、90%投入、千股與費稅不變；AH／BH只作對照。舊六種僅兩年公告原型仍保留舊版本，不與新引擎混排名。',
'','|方法|與A同日有效窗口|平均相對A費用後差pp|24區塊校正區間pp|48區塊校正區間pp|判斷|','|---|---:|---:|---|---|---|']
method_labels={'REF':'流動性靜態參考','B':'B動能＋法人','C':'C動能＋低波動','AH':'AH靜態持有','BH':'BH靜態持有','A_mild':'A溫和動能','A_invvol50':'A逆波動各半','A_mild_tolerance':'A溫和動能＋小額略過','REV5':'五日反轉','pure_flow':'純法人','pure_lowvol':'純低波動','A_invvol100':'A100%逆波動原始權重','pure_financial':'純季度本業選股原型','pure_revenue':'純營收選股原型','A_formal_operating_prestart':'A開賽前季度資訊','A_revenue_prestart':'A開賽前營收資訊','A_combined_prestart':'A開賽前資訊組合'}
for name,s in all_stats_v7.items():
    ci24='～'.join(f'{x:+.3f}' for x in s['block24_interval_pp']);ci48='～'.join(f'{x:+.3f}' for x in s['block48_interval_pp'])
    lines.append(f"|{method_labels[name]}|{s['pairs']}|{s['mean_difference_pp']:+.4f}|{ci24}|{ci48}|未證實改良|")
lines+=['','原窗口位置24／48日區塊各5000次、保守44項比較家族5%，跨年／早晚段與風險檢查。不同方法配對樣本不同，不用跨列中位數直接排名；重疊窗口不是獨立試驗，已檢視歷史也不是未見樣本外。完整統計含年份、風險、成本與勝率。', '',
f"同50%動能比重的機械對照，資訊組合額外增量平均僅{matched_stats_v7['mean_difference_pp']:+.4f}pp，24／48校正區間仍跨零，2026負向；公告資料修正後，仍沒有證據支持目前這個簡化資訊加權改善動能。這不等於所有公告資訊都沒有價值。",'',
'## 回到策略：目前有意義的討論','',
'A等權仍是比較基準。若沿用第一天非等權需求，A溫和動能＋小額略過仍只作暫定研究候選：目標約4.5%至2.7%，略過保留股不超過0.5%前日可交易NAV的調整。固定公式未因本輪結果重調，尚未證實優於A。',
'下一個策略問題應先決定「我們要改善什麼」：追求24日費用後報酬、降低24日下檔損失，或降低不必要換手。現有資料較適合用同日配對拆出報酬／風險／費用的取捨，再預先定義單一機制；不因法人、低波動或財報名稱不同就當作已做好風控。',
'目前未開始新的策略實驗。可以討論的假說包含動能遇到什麼行情會失效、哪些交易只是產生成本，以及持股集中在相同產業時的共同風險。這些只是待討論問題，不是新增權重、停損或產業配置的授權定案。', '',
'## 交付與界線','',
'[六個主線配置全部有效點](A策略_關鍵資料修正比較_v7_2026-10-15.html)；其餘全部方法逐窗口在JSON中，沒有只保存好看的點。桃紅中位數、0%線、固定座標與方法順序沿用。',
'[統一統計](A策略_關鍵資料修正統一統計_v7_2026-10-15.json)、[全部逐窗口](A策略_關鍵資料修正逐窗口_v7_2026-10-15.json)、[時間及更正查核](A策略_時間格式與更正範圍查核_v7_2026-10-15.json)、[休市公司行動查核](A策略_休市除權息對齊查核_v7_2026-10-15.json)。Notebook保留程式／成功輸出與kernel，不做Restart and Run All。',
'本輪第①項可驗證修正及公平重算完成；尚未解析原文、12筆有償認購、首公開時間完整性與實際發放時程仍留限制。②帳戶與D-Plan、③官賽規則、④流程演練依使用者明確要求延後，不讓DataPack等待阻擋策略討論。LLM暫緩，排程維持暫停、無交易。']
report_path_v7=recovery_root/'A策略_關鍵資料缺口與策略比較_v7_2026-10-15.md'
report_path_v7.write_text('\n'.join(lines)+'\n',encoding='utf-8')
state_path=recovery_root/'A策略_無人值守研究狀態.json';state=json.loads(state_path.read_text(encoding='utf-8'))
state['user_scope_20261015']={'active':'item1 limited critical data gap audit, then strategy discussion','deferred':['item2 actual account and DPlan','item3 competition operating rules','item4 precompetition operating rehearsal','LLM weighting'],'number_reference':'the four-item remaining-work table, not six-stage roadmap'}
state['strategy_gap_v7']={'status':'completed_feasible_source_repairs_and_fixed_method_revalidation_with_limits','report':report_path_v7.name,'source_time_repairs':len(time_repairs_v7),'closure_actions':len(holiday_actions_v7),'A_valid':557,'A_initial_quarter_coverage':'24of25 in2024 and2025','supported':supported_v7,'remaining':['3529newquarter numbers missing in retained notice','6223salary-cost correction unresolved','12paid-capital subscriptions need entitlement policy/accounting','first public time and full report completeness not certified'],'running_work':False,'notebook_cells':[181,182,183,184,185,186,187,188,189,190,191,192,193,194,195],'next':'discuss desired24day objective and fixed testable strategy mechanism; operations2/3/4 remain deferred'}
state['status']='strategy_gap_v7_completed_strategy_discussion_active_operations_deferred';state['current_node']=None;state['automation_status']='PAUSED'
state['last_completed_action']='Item1 only: repaired574 recorded timestamps, scoped corrections and8 closure actions, same-engine fixed revalidation,557Avalid; no supported improvement. Items2/3/4 deferred.'
state['updated_at']=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=8))).isoformat()
for f in [report_path_v7.name,'A策略_關鍵資料修正統一統計_v7_2026-10-15.json','A策略_關鍵資料修正比較_v7_2026-10-15.html','A策略_關鍵資料修正逐窗口_v7_2026-10-15.json']:
    if f not in state['final_artifacts']:state['final_artifacts'].append(f)
state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
workflow=recovery_root/'A策略_無人值守研究流程.md';text=workflow.read_text(encoding='utf-8')
note='## 現行範圍：第①項查核完成，回到策略討論（2026-10-15）\n\n[最新v7資料缺口與策略比較](A策略_關鍵資料缺口與策略比較_v7_2026-10-15.md)。修正574筆原時間格式、有限更正範圍與8筆休市公司行動；A有效557窗口，兩年初始A25各24檔可核對。固定方法同引擎重驗證仍無已證實改良。12筆有償認購及未解原文仍保留限制。\n\n使用者所指1／2／3／4是四件待辦表：只做①補資料缺口，②帳戶與D-Plan、③官賽規則、④流程演練全部延後。DataPack不是目前策略討論的阻礙。LLM暫緩；未新增策略參數或交易。PAUSED，kernel保留；舊「下一步等DataPack」段落不是現行優先順序。\n\n'
if '## 現行範圍：第①項查核完成' not in text:workflow.write_text(text.split('\n',1)[0]+'\n\n'+note+text.split('\n',1)[1],encoding='utf-8')
print('V7_FINAL_REPORT_SAVED',report_path_v7.name,'chartpoints',len(main_chart_rows_v7),'source repair effect',price_repair_check_v7)


In [ ]:
# A failure study: existing computational state and source availability only
import sys,json,datetime,inspect,hashlib
from research_layout import ResearchPath as Path
from collections import Counter,defaultdict
print('Kernel',sys.executable)
for n in ['scores_v7','signal_returns_v7','engine_v7_results','simulate_engine_v7','src_v7','settle','all_stats_v7']:
    v=globals().get(n);print(n,type(v).__name__,getattr(v,'shape',None),len(v) if hasattr(v,'__len__') else None)
print('Raw stock snapshot schema',type(raw_data['stocks']).__name__,'row width',len(raw_data['stocks'][0]) if raw_data['stocks'] else None)
print('Existing sector or taxonomy metadata',[n for n in globals() if any(s in n.lower() for s in ['sector','industry','classification'])][:35])
failure_prior_state=json.loads((recovery_root/'A策略_無人值守研究狀態.json').read_text(encoding='utf-8'))
print('Saved history/classification limits',[v for v in failure_prior_state.get('known_limitations',[]) if any(s in str(v) for s in ['分類','classification','產業'])])
print('Engine definition',v4_saved['fixed_methods'])
print('Saved baseline count',len(engine_v7_results['A_equal']),'valid',sum(not r['invalid'] and r['violations']==0 for r in engine_v7_results['A_equal']))
print('A_FAILURE_EXISTING_STATE_READY')


In [ ]:
# Freeze minimal mechanism definitions before inspecting diagnostic relationships or candidate returns
failure_definition_v8={
'date':'2026-10-15','base':'v7 fixed snapshot and verified action overlay','scope':'A same25,24sessions,every5sessions,90pct,1000share,buy/sellfee .001425,selltax .003',
'common_reference':'A_mild_tolerance existing .005previous tradableNAV retained-stock skip; exact same selection and skip for new candidates',
'R_market_reversal':{'state':'previous20days cumulative return of daily equalweighted available150 proxy,negative triggers; proxy excludes flagged corporate-action returns','beta':'previous60 adjusted returns beta to same daily proxy; require>=40paired finite days; unknown multiplier1','allocation':'base mild weight divided by (1+max(beta,0)) in negative state; otherwise base mild; normalize total90pct with existing1.8-5.4pct bounds','cap_beta':3.,'cash_change':False,'meaning':'lower estimated sensitivity inside same25, not remove overall market exposure'},
'E_overextension':{'definition':'previous20days cumulative adjusted return minus mean of those20 path levels, divided by populationstd of20pathlevels; equivalent latest price relative20day mean in action-adjusted units','trigger':'last adjusted level >mean+2std, multiplier .75; else1; unknown1','allocation':'mild raw weight times multiplier, normalize90pct using same1.8-5.4pct bounds','meaning':'a simple fixed extension rule, not claim all big winners reverse'},
'S_common_risk':{'industry_data':'no verified point-in-time individual classifications in retained sources; industry cap backtest not feasible','diagnostic':'prior60day mean pairwise correlation of selected25 adjusted returns; pair requires>=40finite days; groups <.25,.25-.5,>=.5 fixed','not_industry':'correlation is statistical common risk, not a reconstructed industry classification'},
'T_turnover':{'candidate':'reuse existing A_mild_tolerance; no new allocation parameter or cycle search','attribution':'record exact replay settlement costs by initial entry,new later entry,full exit,retained adjust; reconcile return/fees to saved v7','causality':'fee difference alone not counterfactual PnL; replay only to obtain trade attribution, not a duplicate candidate'},
'diagnostics':{'initial_asof':'all stock features based on t-1,not future24day returns','proxy_scope':'150 current contest universe historical data; not official TAIEX nor survivor-free market index','realized_reversal':'prior20proxy>=0 and subsequent24proxy<0 descriptive only, never used in candidate weights','overlap':'all windows overlap; stratified summaries descriptive, not independent observations','year_tests':'retain all years and early/late; no untouched holdout'},
'new_candidates':['A_R_state_beta_tolerance','A_E_extension_tolerance'],'no_auto_combination':True,'family':46,'family_reason':'prior conservative44 plus2 fixed single mechanisms','risk_criteria':'pairedP10 and medianMDD deterioration<=1pp;44old+2new family5pct block24/48 each5000,years/halves must all positive','deferred':['official account/DPlan','competition operating rules','operational rehearsal','LLM weighting']}
failure_definition_sha_v8=hashlib.sha256(json.dumps(failure_definition_v8,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
(recovery_root/'A策略_失效診斷與機制定義_v8_2026-10-15.json').write_text(json.dumps({'definition':failure_definition_v8,'sha256':failure_definition_sha_v8},ensure_ascii=False,indent=2),encoding='utf-8')
print('A_FAILURE_DEFINITIONS_FROZEN_BEFORE_OUTCOMES',failure_definition_sha_v8)


In [ ]:
# Predecision features from retained adjusted series; outcome fields kept separate
safe_r_v8=signal_returns_v7.copy()
safe_r_v8[bad_action_v7|~np.isfinite(safe_r_v8)|(safe_r_v8<=-1)]=np.nan
safe_counts_v8=np.isfinite(safe_r_v8).sum(axis=1)
proxy_daily_v8=np.divide(np.nansum(safe_r_v8,axis=1),safe_counts_v8,out=np.full(len(dates),np.nan),where=safe_counts_v8>=75)
market_prior20_v8=np.full(len(dates),np.nan);beta_v8=np.full(shape,np.nan);extension_v8=np.full(shape,np.nan);hot_v8=np.zeros(shape,bool)
correlation_at_start_v8={};features_at_start_v8=[]
for t in range(61,len(dates)):
    q=t-1;market=proxy_daily_v8[t-20:t]
    if np.all(np.isfinite(market)):market_prior20_v8[t]=np.prod(1+market)-1
    rr=safe_r_v8[t-60:t];p=proxy_daily_v8[t-60:t]
    for j in range(len(stock_ids)):
        keep=np.isfinite(rr[:,j])&np.isfinite(p)
        if keep.sum()>=40 and np.var(p[keep])>1e-12:beta_v8[t,j]=np.mean((rr[keep,j]-rr[keep,j].mean())*(p[keep]-p[keep].mean()))/np.var(p[keep])
        last=safe_r_v8[t-20:t,j]
        if np.all(np.isfinite(last)):
            levels=np.cumprod(1+last);std=np.std(levels)
            extension_v8[t,j]=(levels[-1]-levels.mean())/std if std>1e-12 else 0.
            hot_v8[t,j]=extension_v8[t,j]>2.
for s in window_starts:
    t=int(s);rank=scores_v7['A'][t-1];selected=np.flatnonzero(np.isfinite(rank));selected=selected[np.argsort(-rank[selected],kind='stable')[:25]]
    rr=safe_r_v8[t-60:t,selected];corrs=[]
    for a in range(len(selected)):
        for b in range(a+1,len(selected)):
            keep=np.isfinite(rr[:,a])&np.isfinite(rr[:,b])
            if keep.sum()>=40 and np.std(rr[keep,a])>1e-12 and np.std(rr[keep,b])>1e-12:corrs.append(float(np.corrcoef(rr[keep,a],rr[keep,b])[0,1]))
    mean_corr=float(np.mean(corrs)) if corrs else None;correlation_at_start_v8[t]=mean_corr
    post=proxy_daily_v8[t:t+24];future=float(np.prod(1+post)-1) if len(post)==24 and np.all(np.isfinite(post)) else None
    record={'start':str(dates[t]),'t':t,'asof':str(dates[t-1]),'selected':[stock_ids[j] for j in selected],'prior20_proxy_return':float(market_prior20_v8[t]) if np.isfinite(market_prior20_v8[t]) else None,'initial_hot_count':int(hot_v8[t,selected].sum()),'mean_pairwise_corr60':mean_corr,'corr_pairs':len(corrs),'mean_beta60':float(np.nanmean(beta_v8[t,selected])),'realized_proxy_next24':future,'realized_reversal':bool(market_prior20_v8[t]>=0 and future is not None and future<0),'outcome_role':'diagnosis only; future24 fields never referenced by candidate allocations'}
    features_at_start_v8.append(record)
print('FEATURES_READY',len(features_at_start_v8),'negative prior states',sum(r['prior20_proxy_return'] is not None and r['prior20_proxy_return']<0 for r in features_at_start_v8),'hot windows',sum(r['initial_hot_count']>0 for r in features_at_start_v8),'corr coverage',sum(r['mean_pairwise_corr60'] is not None for r in features_at_start_v8))
print('Stock classifications unavailable: S industry limit not backtested; correlation is diagnosis only.')


In [ ]:
# Descriptive diagnostics conditional on v7-valid A windows; no causal or OOS claim
valid_A_v8=np.array([not r['invalid'] and r['violations']==0 for r in engine_v7_results['A_equal']])
def group_summary_v8(mask):
    ix=np.flatnonzero(valid_A_v8&np.array(mask,dtype=bool));rows=[engine_v7_results['A_equal'][i] for i in ix]
    if not rows:return {'n':0}
    values=np.array([r['return'] for r in rows]);out={'n':len(rows),'loss_pct':float(np.mean(values<0)*100),'mean_return_pct':float(values.mean()*100),'median_return_pct':float(np.median(values)*100),'p10_return_pct':float(np.quantile(values,.1)*100),'median_mdd_pct':float(np.median([r['mdd'] for r in rows])*100),'years':{}}
    for yr in sorted({dates[int(window_starts[i])].year for i in ix}):
        yy=[engine_v7_results['A_equal'][i]['return'] for i in ix if dates[int(window_starts[i])].year==yr];out['years'][str(yr)]={'n':len(yy),'mean_return_pct':float(np.mean(yy)*100),'loss_pct':float(np.mean(np.array(yy)<0)*100)}
    return out
groups_v8={
'R_prior_proxy_negative':group_summary_v8([r['prior20_proxy_return'] is not None and r['prior20_proxy_return']<0 for r in features_at_start_v8]),
'R_prior_proxy_nonnegative':group_summary_v8([r['prior20_proxy_return'] is not None and r['prior20_proxy_return']>=0 for r in features_at_start_v8]),
'R_realized_reversal_descriptive':group_summary_v8([r['realized_reversal'] for r in features_at_start_v8]),
'R_no_realized_reversal_descriptive':group_summary_v8([not r['realized_reversal'] for r in features_at_start_v8]),
'E_at_least_one_hot':group_summary_v8([r['initial_hot_count']>0 for r in features_at_start_v8]),
'E_no_hot':group_summary_v8([r['initial_hot_count']==0 for r in features_at_start_v8]),
'S_correlation_below25':group_summary_v8([r['mean_pairwise_corr60'] is not None and r['mean_pairwise_corr60']<.25 for r in features_at_start_v8]),
'S_correlation25to50':group_summary_v8([r['mean_pairwise_corr60'] is not None and .25<=r['mean_pairwise_corr60']<.5 for r in features_at_start_v8]),
'S_correlation_atleast50':group_summary_v8([r['mean_pairwise_corr60'] is not None and r['mean_pairwise_corr60']>=.5 for r in features_at_start_v8])}
stock_hot_outcomes_v8={'hot':[],'not_hot':[]};stock_hot_pairs_v8=[]
for i,f in enumerate(features_at_start_v8):
    if not valid_A_v8[i]:continue
    t=f['t'];hot_values=[];normal_values=[]
    for sid in f['selected']:
        j=si[sid];r=safe_r_v8[t:t+24,j]
        if len(r)!=24 or not np.all(np.isfinite(r)):continue
        # Diagnostic adjusted signal return only, no claim this is tradable position PnL.
        future=float(np.prod(1+r)-1)
        dest='hot' if hot_v8[t,j] else 'not_hot';stock_hot_outcomes_v8[dest].append(future)
        (hot_values if hot_v8[t,j] else normal_values).append(future)
    if hot_values and normal_values:stock_hot_pairs_v8.append(float(np.mean(hot_values)-np.mean(normal_values)))
hot_summary_v8={k:{'stock_window_observations':len(v),'mean_adjusted_signal_return_pct':float(np.mean(v)*100),'loss_pct':float(np.mean(np.array(v)<0)*100)} for k,v in stock_hot_outcomes_v8.items() if v}
hot_summary_v8['within_same_window_hot_minus_normal_pp']={'windows':len(stock_hot_pairs_v8),'mean':float(np.mean(stock_hot_pairs_v8)*100) if stock_hot_pairs_v8 else None}
print('DESCRIPTIVE_GROUPS',json.dumps(groups_v8,ensure_ascii=False))
print('HOT_STOCK_DIAGNOSTIC',json.dumps(hot_summary_v8,ensure_ascii=False))
(recovery_root/'A策略_失效原因診斷_v8_2026-10-15.json').write_text(json.dumps({'definition_sha256':failure_definition_sha_v8,'groups':groups_v8,'hot_stock_diagnostic':hot_summary_v8,'features':features_at_start_v8,'industry_status':'unknown historical individual classifications; no industry backtest','limits':['overlapping windows and repeated stocks','descriptive realized reversal is not a usable trading input','proxy current contest150 not official market index','hot stock outcomes are adjusted signal returns, not position-level fee-adjusted PnL']},ensure_ascii=False,indent=2),encoding='utf-8')
print('A_FAILURE_DIAGNOSTIC_SAVED')


In [ ]:
# Two predeclared candidates; same A selections and skip rule, no automatic mix
failure_weights_audit_v8=[]
def failure_allocation_v8(t,selected,allocation):
    base=engine_v7_allocation(t,selected,'mild');raw=base.copy()
    if allocation=='R':
        if np.isfinite(market_prior20_v8[t]) and market_prior20_v8[t]<0:
            beta=beta_v8[t,selected];mult=np.where(np.isfinite(beta),1/(1+np.clip(beta,0,3)),1.);raw=base*mult
    elif allocation=='E':raw=base*np.where(hot_v8[t,selected],.75,1.)
    elif allocation=='control':return base
    else:raise ValueError('Undefined frozen mechanism')
    weights=weight_bounded_normalize(raw,total=.9,lower=.018,upper=.054)
    assert len(selected)==25 and np.isclose(weights.sum(),.9) and weights.max()<=.054+1e-12 and weights.min()>=.018-1e-12
    failure_weights_audit_v8.append({'t':t,'mode':allocation,'selected':tuple(int(j) for j in selected),'sum':float(weights.sum()),'max':float(weights.max())})
    return weights
failure_src_v8=src_v7.replace('def simulate_engine_v7(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):','def simulate_failure_v8(start,allocation,horizon=HORIZON,tolerance=.005):\n    strategy="A"').replace('engine_v7_allocation(','failure_allocation_v8(')
assert 'def simulate_failure_v8(' in failure_src_v8
exec(failure_src_v8,globals())
# Control check verifies exact same engine before adding any mechanistic weighting.
for i in [0,200,400,649]:
    control=simulate_failure_v8(int(window_starts[i]),'control');old=engine_v7_results['A_mild_tolerance'][i]
    assert control['invalid']==old['invalid'] and control['violations']==old['violations']
    for f in ['return','mdd','cost','turnover']:assert np.isclose(control[f],old[f],rtol=1e-12,atol=1e-12)
failure_candidate_results_v8={}
for name,mode in [('A_R_state_beta_tolerance','R'),('A_E_extension_tolerance','E')]:
    failure_candidate_results_v8[name]=[simulate_failure_v8(int(s),mode) for s in window_starts]
    print('FAILURE_CANDIDATE_SAVED',name,'valid',sum(not r['invalid'] and r['violations']==0 for r in failure_candidate_results_v8[name]),flush=True)
    (recovery_root/'A策略_失效機制逐窗口_v8_2026-10-15.json').write_text(json.dumps({'definition':failure_definition_v8,'sha256':failure_definition_sha_v8,'results':clean_v5(failure_candidate_results_v8)},ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('A_FAILURE_TWO_CANDIDATES_SAVED')


In [ ]:
# Exact replay only to recover trade-cost attribution absent from saved paths
settle_before_attribution_v8=settle
turnover_context_v8={};turnover_events_v8=[]
def attributed_settle_v8(cash,shares,target,prices,available,fee=FEE,tax=TAX):
    outcome=settle_before_attribution_v8(cash,shares,target,prices,available,fee,tax)
    frame=inspect.currentframe().f_back;offset=frame.f_locals.get('offset');t=frame.f_locals.get('t');del frame
    delta=np.where(available,target-shares,0);safe_prices=np.where(available,prices,0)
    for kind,mask in [
      ('initial_entry',(shares==0)&(delta>0)&(offset==0)),
      ('new_later_entry',(shares==0)&(delta>0)&(offset!=0)),
      ('full_exit',(shares>0)&(target==0)&(delta<0)),
      ('retained_adjust',(shares>0)&(target>0)&(delta!=0))]:
        traded=float(np.sum(np.abs(delta[mask])*safe_prices[mask]));cost=float(np.sum(np.maximum(delta[mask],0)*safe_prices[mask]*fee)+np.sum(np.maximum(-delta[mask],0)*safe_prices[mask]*(fee+tax)))
        if traded:turnover_events_v8.append({'method':turnover_context_v8['method'],'window_start':turnover_context_v8['start'],'t':int(t),'offset':int(offset),'kind':kind,'traded_value':traded,'cost':cost,'trade_count':int(mask.sum())})
    return outcome
try:
    settle=attributed_settle_v8
    for name,tolerance in [('A_mild',0),('A_mild_tolerance',.005)]:
        for s in window_starts:
            turnover_context_v8={'method':name,'start':int(s)}
            r=simulate_engine_v7(int(s),'A','mild',tolerance=tolerance)
            old=engine_v7_results[name][int(s-window_starts[0])]
            assert r['invalid']==old['invalid'] and r['violations']==old['violations']
            for f in ['return','cost','turnover']:assert np.isclose(r[f],old[f],rtol=1e-12,atol=1e-12)
        print('ATTRIBUTION_REPLAY_EXACT',name,len(window_starts),flush=True)
finally:settle=settle_before_attribution_v8
turnover_summary_v8={}
for name in ['A_mild','A_mild_tolerance']:
    rows=engine_v7_results[name];valid_starts={int(window_starts[i]) for i,r in enumerate(rows) if not r['invalid'] and r['violations']==0}
    events=[r for r in turnover_events_v8 if r['method']==name and r['window_start'] in valid_starts]
    total_cost=sum(r['cost'] for r in events);by_kind={}
    for kind in ['initial_entry','new_later_entry','full_exit','retained_adjust']:
        subset=[r for r in events if r['kind']==kind];cost=sum(r['cost'] for r in subset)
        by_kind[kind]={'mean_cost_pct_initial':cost/len(valid_starts)/INITIAL*100,'share_of_all_cost_pct':cost/total_cost*100 if total_cost else None,'mean_trade_stock_count':sum(r['trade_count'] for r in subset)/len(valid_starts),'mean_gross_traded_pct_initial':sum(r['traded_value'] for r in subset)/len(valid_starts)/INITIAL*100}
    total_from_rows=sum(r['cost']*INITIAL for i,r in enumerate(rows) if int(window_starts[i]) in valid_starts)
    assert np.isclose(total_from_rows,total_cost,rtol=1e-12,atol=.001)
    turnover_summary_v8[name]={'valid_windows':len(valid_starts),'mean_total_cost_pct_initial':total_cost/len(valid_starts)/INITIAL*100,'by_trade_type':by_kind}
common_turnover_v8=np.array([not a['invalid'] and a['violations']==0 and not b['invalid'] and b['violations']==0 for a,b in zip(engine_v7_results['A_mild_tolerance'],engine_v7_results['A_mild'])])
turnover_pairs_v8={'pairs':int(common_turnover_v8.sum()),'mean_cost_saving_pp':float(np.mean([engine_v7_results['A_mild'][i]['cost']-engine_v7_results['A_mild_tolerance'][i]['cost'] for i in np.flatnonzero(common_turnover_v8)])*100),'mean_net_return_change_pp':float(np.mean([engine_v7_results['A_mild_tolerance'][i]['return']-engine_v7_results['A_mild'][i]['return'] for i in np.flatnonzero(common_turnover_v8)])*100),'counterfactual_note':'fewer fees and changed trades interact; this is paired observed result, not causal proof every skipped trade was useless'}
(recovery_root/'A策略_換手成本歸因_v8_2026-10-15.json').write_text(json.dumps({'definition_sha256':failure_definition_sha_v8,'summary':turnover_summary_v8,'paired':turnover_pairs_v8,'events':turnover_events_v8},ensure_ascii=False),encoding='utf-8')
print('TURNOVER_COST_ATTRIBUTION',json.dumps(turnover_summary_v8,ensure_ascii=False))
print('TURNOVER_SAME_DATE_COMPARISON',json.dumps(turnover_pairs_v8,ensure_ascii=False))
print('A_FAILURE_TURNOVER_ATTRIBUTION_SAVED_NO_NEW_CANDIDATE')


In [ ]:
# Both candidate comparisons versus equal-A and identical unequal-reference, corrected family
failure_stats_v8={}
for name,rows in failure_candidate_results_v8.items():
    vs_equal=summary_v7(rows,engine_v7_results['A_equal'],family=46)
    vs_reference=summary_v7(rows,engine_v7_results['A_mild_tolerance'],family=46)
    vs_reference['reference']='A_mild_tolerance v7 same25 and same .005skip'
    vs_reference['reference_median_same_dates_pct']=vs_reference.pop('A_median_same_dates_pct')
    for record in vs_reference['seasonal']:record['reference_return_pct']=record.pop('A_return_pct')
    failure_stats_v8[name]={'vs_A_equal':vs_equal,'vs_identical_unequal_reference':vs_reference,'supported_as_mechanism':bool(vs_reference['supported']),'supported_vs_both':bool(vs_equal['supported'] and vs_reference['supported'])}
    print('FAILURE_STRATEGY_COMPARISON',name,json.dumps(failure_stats_v8[name],ensure_ascii=False),flush=True)
turnover_inference_v8=summary_v7(engine_v7_results['A_mild_tolerance'],engine_v7_results['A_mild'],family=46)
turnover_inference_v8['reference']='A_mild without tolerance';turnover_inference_v8['reference_median_same_dates_pct']=turnover_inference_v8.pop('A_median_same_dates_pct')
for record in turnover_inference_v8['seasonal']:record['reference_return_pct']=record.pop('A_return_pct')
failure_final_v8={'definition':failure_definition_v8,'definition_sha256':failure_definition_sha_v8,'stats':failure_stats_v8,'turnover_inference_existing_candidate':turnover_inference_v8,'diagnostic_groups':groups_v8,'hot_stock_diagnostic':hot_summary_v8,'supported_candidates':[n for n,r in failure_stats_v8.items() if r['supported_vs_both']],'industry':'cannot determine industry effect without historical individual classification; correlation diagnosis not industry strategy','parameter_search':False,'no_auto_combination':True}
(recovery_root/'A策略_失效機制統一比較_v8_2026-10-15.json').write_text(json.dumps(failure_final_v8,ensure_ascii=False,indent=2),encoding='utf-8')
print('TOLERANCE_INCREMENT_INFERENCE',json.dumps(turnover_inference_v8,ensure_ascii=False))
print('A_FAILURE_FAMILY_INFERENCE_COMPLETE',failure_final_v8['supported_candidates'])


In [ ]:
# Persist interpretation and full-point comparison from verified kernel results
import re
assert not failure_final_v8['supported_candidates']
assert turnover_inference_v8['supported']
assert all(len(rows)==650 for rows in failure_candidate_results_v8.values())
for a in failure_weights_audit_v8:
    t=a['t'];expected=np.argsort(-scores_v7['A'][t-1],kind='stable')[:25]
    # Selection check uses membership; engine tie sorting is preserved in its saved source.
    assert set(a['selected'])==set(expected),('selection mismatch',t)
print('AUDIT same25, bounded weights, original engine control and full cost-replay checks passed')
chart_name='A策略_失效機制比較_v8_2026-10-15.html'
report_name='A策略_失效診斷與機制研究_v8_2026-10-15.md'
chart_methods={'A等權':engine_v7_results['A_equal'],'A非等權＋略過':engine_v7_results['A_mild_tolerance'],'R反轉降權':failure_candidate_results_v8['A_R_state_beta_tolerance'],'E過熱降權':failure_candidate_results_v8['A_E_extension_tolerance']}
chart_points=[]
for name,rows in chart_methods.items():
    for row in rows:
        if row['invalid'] or row['violations']:continue
        chart_points.append({'method':name,'start':str(dates[row['start']])[:10],'end':str(dates[row['end']])[:10],'return_pct':row['return']*100,'cost_pct':row['cost']*100,'mdd_pct':row['mdd']*100})
html=(recovery_root/'A策略_關鍵資料修正比較_v7_2026-10-15.html').read_text(encoding='utf-8')
html=re.sub(r'const data=.*?;const labels=.*?;const names=.*?;const lo=',lambda m:'const data='+json.dumps(chart_points,ensure_ascii=False)+';const labels={};const names='+json.dumps(list(chart_methods),ensure_ascii=False)+';const lo=',html,count=1,flags=re.S)
html=html.replace('A關鍵資料修正比較 v7','A失效機制比較 v8').replace('x=145+165*i','x=200+230*i')
intro='固定A選股25檔、24交易日、每5日調整、90%投入、整千股與費稅。R：前20日市場代理下跌時，依過去60日beta降權；E：20日價格位置z大於2時，原權重乘0.75後重新分配。兩者都保留原本略過小額調整機制。兩個新規則尚未證實改善；產業分類不足，未回測產業上限。桃紅線為中位數、虛線為0%，全部有效點皆呈現，懸停可看日期、報酬、費用及回撤。'
html=re.sub(r'<p>.*?</p>',lambda m:'<p>'+intro+'</p>',html,count=1,flags=re.S)
(recovery_root/chart_name).write_text(html,encoding='utf-8')
lines=['# A策略失效診斷與機制研究 v8','日期：2026-10-15。研究已完成可行分支；新規則不強選冠軍。','',
'## 結論','兩個新增機制都未通過預先保存的雙基準判準。略過小額調整相較相同非等權但不略過的配置，有樣本內改善證據；這不等於整套非等權配置已證實勝過等權A。','',
'## 實驗範圍','沿用v7資料與引擎；固定A挑選的25檔、24交易日、每5日再平衡、90%投入、整千股及原費稅。650個滑動窗口均保留，包括無效結果；圖表只呈現無資料排除、無違規的有效窗口。未啟用LLM、未加入公告新權重、未提交交易。','兩個機制以既有溫和動能權重加0.5%前期可交易NAV調整容忍為共同底座。兩者投入額度不同，並非每檔同額。原目標權重約4.5%至2.7%；新機制重配總和90%，單檔目標1.8%至5.4%。實際股數按整千股取整，目標權重不等於每天實際權重。','定義先保存於 A策略_失效診斷與機制定義_v8_2026-10-15.json，再執行候選，不依結果搜尋新門檻或自動組合。','',
'## 診斷與規則','### R：市場反轉','事後定義的「此前20日上漲、之後24日下跌」105個有效A窗口，虧損比例73.33%。這用了未來結果，只能解釋風險，不能當買賣訊號。可事先觀察的前20日下跌組，虧損比例26.77%，其他組27.91%，跨年關係也不一致，不能把它稱為可靠反轉預警。','實驗R：市場代理前20日報酬小於0時，以過去60日股票對市場代理beta調降高beta權重；乘數1/(1+clip(beta,0,3))，再按固定上下限正規化。至少40個共同有效日；未知beta維持原倍率。市場代理為現有150檔股票可用日報酬等權，不是加權指數，也不是完整無存活偏差市場。總投入仍90%，本實驗不測減倉。','### E：漲多／過熱','以此前20日調整後價格路徑的末值相對均值及標準差計算z，z>2為過熱；過熱股票的後24日調整後訊號報酬並未較差。同窗口比較，過熱組平均反而高約0.725個百分點。這是重疊股票窗口的描述統計，不是實際持倉費後損益，也不是漲多必漲的因果證據。','實驗E：過熱股票原始權重乘0.75，重新分配到同25檔，未知訊號維持原倍率；門檻及懲罰不調參。','### S：共同風險／產業集中','此前60日兩兩相關性診斷完成，但高相關組僅39個有效窗口且全部位於2025年，與年份行情混雜，不能推論高相關安全或危險。缺可驗證的歷史個股產業分類，因此產業曝險上限策略未回測，保持不可判定；沒有用今日分類或統計相關群冒充歷史產業。','',
'## 新機制的費用後配對結果','單位為每個24日窗口報酬差的百分點。配對日期相同；不能以各方法全部有效點的中位數直接判定優劣。','',
'|機制|有效窗口|相較等權A：配對平均差|相較共同非等權底座：配對平均差|對底座24日區塊校正區間|對底座48日區塊校正區間|判斷|','|---|---:|---:|---:|---|---|---|']
for name,label in [('A_R_state_beta_tolerance','R反轉降權'),('A_E_extension_tolerance','E過熱降權')]:
    st=failure_stats_v8[name];a=st['vs_A_equal'];b=st['vs_identical_unequal_reference'];valid=sum(not r['invalid'] and not r['violations'] for r in failure_candidate_results_v8[name]);fmt=lambda x:f'[{x[0]:+.3f}, {x[1]:+.3f}]'
    lines.append(f"|{label}|{valid}/650|{a['mean_difference_pp']:+.3f}（n={a['pairs']}）|{b['mean_difference_pp']:+.3f}（n={b['pairs']}）|{fmt(b['block24_interval_pp'])}|{fmt(b['block48_interval_pp'])}|未證實|")
lines+=['','R相較共同底座，2024、2026平均較差；E在2025約持平略差，而且兩個比賽同季起點都落後共同底座。兩個新規則的區間均跨0，不能因相較等權A的均值正向而誤認機制有效。',
'統計沿用原判準：24／48日區塊重抽樣各5000次、多重比較家族46、年份與前後半期、P10及中位最大回撤惡化不超過1個百分點。新候選須對等權A與相同非等權底座皆通過。區塊依原窗口位置處理，650窗口不是650次獨立比賽。','',
'## T：換手成本歸因與既有略過機制','逐筆歸因重播僅為補齊舊結果缺少的交易分類；兩種既有方法各650窗口的報酬、費用及換手與保存結果逐筆一致，未當成新候選重複計數。','',
'|非等權配置|初始買入|中途新進買入|整檔退出|既有持股微調|','|---|---:|---:|---:|---:|']
for n,label in [('A_mild','每次調整'),('A_mild_tolerance','略過小額調整')]:
    s=turnover_summary_v8[n]['by_trade_type'];lines.append('|'+label+'|'+'|'.join(f"{s[k]['share_of_all_cost_pct']:.2f}%" for k in ['initial_entry','new_later_entry','full_exit','retained_adjust'])+'|')
lines+=['','上表為各自有效窗口的費用結構比例，不是同日期比較。多數費用來自退出換股，而非微調；不應據此事後改成禁止退出或任意延長持有。',
f"同日期557組配對，略過小額調整平均少付費稅 {turnover_pairs_v8['mean_cost_saving_pp']:.5f} 個百分點，費後報酬平均增加 {turnover_pairs_v8['mean_net_return_change_pp']:.5f} 個百分點。24日區塊家族校正區間 [{turnover_inference_v8['block24_interval_pp'][0]:.5f}, {turnover_inference_v8['block24_interval_pp'][1]:.5f}]；48日 [{turnover_inference_v8['block48_interval_pp'][0]:.5f}, {turnover_inference_v8['block48_interval_pp'][1]:.5f}]。三年及前後半期平均增量均正，通過本次沿用判準。",
'此支持僅限「相同溫和非等權配置中，略過小額調整比每次都調整」的歷史樣本內增量；不代表每筆略過都有益。節費與持股路徑改變會交互作用，不能把報酬提升全歸因於省費用。重抽樣極端尾端受5000次模擬誤差影響，仍需要未使用資料驗證，不升格成已驗證實盤優勢。','',
'## 選擇理由與未解限制','保留等權A為基準，溫和非等權＋略過為暫定研究配置；優先保留有清楚成本邏輯且對自身底座有增量證據的略過機制。R與E不納入預設配置；產業限制等待歷史分類，不能宣稱已測完。沒有強選歷史最高報酬，也不新建R+E綜合策略。',
'資料沿用v7的公司行動與公告修正，仍有12筆有償認購事件未建模而排除、季度原文與更正缺口、實際付款日處理等限制。有效窗口依持股而異，除每組配對外不直接比較整體分布均值。只有2024、2025兩個同季起點，不能證明穩定季節性。沒有全新未見測試期；既有股票池存活偏差、公司行動與成交近似限制仍適用。',
'Notebook保留原kernel並保存程式與輸出，未做會清除即時狀態的Restart and Run All。官方帳戶／D-Plan、操作規則與演練仍按使用者指示延後；LLM不測。排程維持PAUSED。','',
'## 成果',f'- 互動箱形圖：[{chart_name}]({chart_name})；每個有效點皆保留，快捷鍵固定座標及順序、單一懸停資訊。',
'- 定義、診斷、兩個候選逐窗口、費用歸因、統一統計均以同名v8 JSON保存；原Notebook追加cells197–204。',
'- 驗證：同25檔與權重範圍檢查、共同底座4個起點引擎一致、兩個既有配置各650窗口費用重播精確一致；互動腳本另做NodeVM模擬DOM驗證，不宣稱瀏覽器目視驗證。']
(recovery_root/report_name).write_text('\n'.join(lines)+'\n',encoding='utf-8')
state_path=recovery_root/'A策略_無人值守研究狀態.json';state=json.loads(state_path.read_text(encoding='utf-8'))
branch=state['A_failure_research'];branch.update({'status':'completed_feasible_diagnostics_and_two_fixed_candidates_with_industry_unknown','report':report_name,'chart':chart_name,'definition_sha256':failure_definition_sha_v8,'notebook_cells':list(range(197,205)),'new_supported_candidates':[],'tolerance_increment_vs_same_unequal_supported_in_sample':True,'running_work':False,'selection':'A_equal benchmark; A_mild_tolerance provisional unequal reference; R/E not adopted; industry unknown','all_candidate_windows_saved':1300})
for node in branch['branches']:
    node['status']={'R_market_reversal':'completed_not_supported','E_overextension':'completed_not_supported','S_industry_concentration':'correlation_diagnosis_completed_industry_backtest_unavailable','T_turnover':'completed_exact_attribution_existing_candidate_reused_increment_supported_in_sample'}[node['id']]
    node['parameters']='see frozen v8 definition '+failure_definition_sha_v8
state['status']='A_failure_v8_feasible_research_completed_industry_unresolved';state['current_node']=None;state['automation_status']='PAUSED'
state_path.write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
workflow_path=recovery_root/'A策略_無人值守研究流程.md'
workflow=workflow_path.read_text(encoding='utf-8')
workflow+='\n\n## 2026-10-15 v8實際研究結果（取代先前未啟動狀態）\nR與E各650窗口及雙基準統計已完成，均未證實改善，不加入預設配置。T既有略過規則完成精確費用歸因，相較同非等權但不略過的配置有樣本內增量支持，非對等權A的整體勝出證據。S完成相關性描述，但歷史個股產業分類不足，產業策略仍不可判定。詳見 '+report_name+'。不自行組合新策略、不重跑既有候選。官方操作2/3/4與LLM延後，排程維持PAUSED。\n'
workflow_path.write_text(workflow,encoding='utf-8')
print('V8_REPORT_CHART_STATE_SAVED','points',len(chart_points),'methods',len(chart_methods))


In [ ]:
# v9 redesign: reconnect existing external computational memory; read metadata only
import sys,inspect,json
print('PYTHON',sys.executable)
for n in ['scores_v7','engine_v7_results','signal_returns_v7','failure_final_v8','raw_data','src_v7']:
    obj=globals().get(n);print(n,type(obj).__name__,getattr(obj,'shape',None),len(obj) if hasattr(obj,'__len__') else None)
print('ENGINE_SOURCE',src_v7)
print('ALLOCATION_SOURCE',inspect.getsource(engine_v7_allocation) if False else 'saved cell source available')
for n in ['dates','stock_ids','close','execution','good_exec_v2']:
    obj=globals().get(n);print(n,type(obj).__name__,getattr(obj,'shape',None))
print('SAFE_HISTORY',[(i, str(s)[:90]) for i,s in enumerate(get_ipython().history_manager.input_hist_raw) if s.lstrip().startswith('#') and ('v8' in s[:100] or 'v7' in s[:100])][-5:])


In [ ]:
# v9 preregistration: flexible membership, comparative signal weights and official caps
import json,hashlib,datetime,collections
redesign_definition_v9={
 'version':'v9','date':'2026-10-15','status':'frozen_before_new_results',
 'reason':'user replaces fixed25 requirement; compare signal weights and allow exclusion/inclusion',
 'rule_source':'saved official 1005 PDF and D-Plan v4.3; live website403 on2026-10-15, no assertion latest',
 'official_constraints':{'universe':150,'held_min':20,'held_max':30,'cap_2330':.25,'cap_other':.10,'cash':'0<=cash/nav<.25','lot':1000,'active_share':'15largest,15pct; historical benchmark unavailable: not certified','drift':'retained simplified five-day grace; official semantics not fully certified'},
 'unchanged':{'horizon':24,'rebalance':5,'invested':.9,'fee':.001425,'sell_tax':.003,'corporate_actions':'v7 repaired cache;12unmodeled subscription events excluded','LLM':False},
 'signal_components':['A','PURE_FLOW','PURE_LOWVOL','PURE_FINANCIAL','PURE_REVENUE'],
 'signal_weight_grid':'five pure; A/nonA mixtures A fraction .25,.5,.75 for each of4 nonA =>17; percentile component scales inherited; no post-result fine search',
 'missing_signal':'if any positively weighted component unknown exclude that stock; no missing-as-bad assumption or zero imputation; insufficient20 invalid',
 'count_grid':[20,25,30,'adaptive20to30'],
 'adaptive_rule':'count finite composite scores>=.8, clip20..30; select topcount; .8 frozen research threshold, not claimed optimal',
 'allocation_grid':['equal','conviction'],
 'conviction_rule':'raw exp(8*(composite_score-.5)); proportional cap redistribution at25pct2330/10pctothers; no entitlement to overweight2330; selected target can become0shares after lots =>20min compliance checked',
 'trade_rule':'all new cells retained positions delta<=.005 previous tradableNAV skipped; entries/exits not skipped; zero target for excluded names; final actual heldcount20..30 daily required',
 'total_candidates':136,'reference':'saved v7 A_equal and A_mild_tolerance, no rerun unchanged methods',
 'evaluation':'all650 retained; historical descriptive grid not winner selection; walkforward2025/2026 selects from enddate<Jan1 testyear with>=100 trainpaired and>=80pct baseline available training coverage; risk P10 and medianMDD no worse1pp; maximize paired train mean, then lower turnover then lexical ID; train/test price horizon purged',
 'selection_safety':'training objective uses same-date candidate-minus-A; test selection cannot use test outcomes; candidate gaps during test not silently switched; coverage and test exclusions separately reported',
 'inference':'24/48 position blocks,5000reps,family183=46 prior+136 new+1adaptive selection pipeline; retrospective heldout-year protocol, not pristine unseen data',
 'seasonality':'separate walkforward index diagnostic using prior years only; historical individual classifications unavailable so industry stock-selection component pending, not silently replaced by current labels',
 'no_other_changes':'no DPlan/submission, no operations rehearsal, no automation activation, no new project/repo; reuse retained caches'
}
redesign_sha_v9=hashlib.sha256(json.dumps(redesign_definition_v9,sort_keys=True).encode()).hexdigest()
(recovery_root/'策略重新比較_預先定義_v9_2026-10-15.json').write_text(json.dumps(redesign_definition_v9,ensure_ascii=False,indent=2),encoding='utf-8')
redesign_plan_v9="""# 策略重新比較 v9：不再固定25檔

本次正式變更：25檔僅保留為舊控制。候選權重以有限比較取得，允許不買／出清與加入其他股票；不能低於官方20檔或超過30檔。

## 官方限制
依保存的10/5辦法、v4.3指南：150檔股票池，20–30檔，2330上限25%，其他單股10%，現金非負且低於25%，整千股。官網2026-10-15查詢403，未確認是否有更新。特殊上限不等於必須持有2330或給它25%。Active Share仍缺歷史基準，不能宣稱完全符合官賽資格。

## 預先固定比較
五個純訊號：動能、法人、低波動、本業利益率改善、營收MoM/YoY。各非動能訊號與動能比較25/75、50/50、75/25，合計17組。訊號比重與買進股票的資金比重是兩個不同層次。

17組交叉20/25/30檔與機會門檻自動20–30檔；每組比較等權、依綜合分數集中配置兩種。共136個事先登記候選，全部用同費用、90%投入、24日、每5日調整、同0.5%NAV小額調整略過。分數集中採exp(8*(score-.5))及官方個股上限再分配，僅是待測固定假說。落選者權重0。

不能依全部回測最高值選冠軍。用2025、2026逐年向前驗證：僅用測試年之前且持有期已結束的窗口選參數；至少100配對窗口、覆蓋至少80%基準可用訓練日期、P10與中位回撤不惡化逾1pp。合格中按訓練費後配對增量、換手較低排序，測試年鎖定不調參。排除與不足另列，不藏缺值。已看過歷史，不稱全新樣本外。

## 產業季節性
既有2016–2025官方指數成果沿用，只用之前年度建立當年排名，另做前瞻式指數層級診斷。個股歷史分類不足時，產業＋個股選股支線保留待補，不冒充已完成，不用今日分類回填。純指數診斷不等於可交易股票組合。

## 保留與交付
舊v7/v8結果保留引用，不重新爬取成功資料。原Notebook追加程式與輸出；保存新定義、136候選全部窗口、逐年選擇與風險／費用、產業驗證及限制。LLM與官方操作2/3/4延後。定義固定後才執行，不再依結果追加細網格。
"""
(recovery_root/'策略重新比較_研究計畫_v9_2026-10-15.md').write_text(redesign_plan_v9,encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'))
st['strategy_redesign_v9']={'status':'defined_running','definition_sha256':redesign_sha_v9,'candidate_count':136,'old_fixed25_superseded':True,'industry_stock_mapping':'not available; pending source','running_work':True,'automation':'PAUSED unchanged'}
sp.write_text(json.dumps(st,ensure_ascii=False,indent=2),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## v9授權變更：解除固定25檔\n依2026-10-15使用者授權，採策略重新比較_研究計畫_v9_2026-10-15.md與預先定義。舊25檔限制僅用控制，不再約束新候選。訊號權重比較、20–30檔及個股不同上限納入；產業個股映射不足分支不可假稱已測。排程不自行開啟。\n',encoding='utf-8')
print('V9_FROZEN',redesign_sha_v9,'candidate_count',136)


In [ ]:
# v9 complete finite grid; original source engine unchanged except explicit count/rank/allocation inputs
mixtures_v9={k:{k:1.} for k in redesign_definition_v9['signal_components']}
for other in redesign_definition_v9['signal_components'][1:]:
    for alpha in [.25,.5,.75]:mixtures_v9[f'A{int(alpha*100)}_{other}']={'A':alpha,other:1-alpha}
mixed_scores_v9={}
for key,weights in mixtures_v9.items():
    mat=sum(scores_v7[k]*w for k,w in weights.items())
    mixed_scores_v9[key]=mat
assert len(mixed_scores_v9)==17
caps_v9=np.array([.25 if sid=='2330' else .10 for sid in stock_ids])
def allocation_v9(t,selected,allocation):
    if allocation=='equal':return np.full(len(selected),.9/len(selected))
    z=current_scores_v9[t-1,selected];raw=np.exp(8*(z-.5));caps=caps_v9[selected]
    out=np.zeros(len(selected));active=np.ones(len(selected),bool);remaining=.9
    for _ in range(len(selected)+1):
        w=raw[active]/raw[active].sum()*remaining;hit=w>caps[active]+1e-12
        idx=np.flatnonzero(active)
        if not hit.any():out[idx]=w;break
        out[idx[hit]]=caps[idx[hit]];remaining=.9-out.sum();active[idx[hit]]=False
    assert np.isclose(out.sum(),.9) and np.all(out<=caps+1e-10) and np.all(out>0)
    return out
v9_source=src_v7.replace('def simulate_engine_v7(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):','def simulate_v9(start,allocation,count_mode,horizon=HORIZON,tolerance=.005):\n    strategy="A"').replace('min_held=COUNT','min_held=30').replace("rank=scores_v7[strategy][t-1]","rank=current_scores_v9[t-1]\n            count=int(count_mode) if count_mode!='adaptive' else int(np.clip(np.count_nonzero(np.isfinite(rank)&(rank>=.8)),20,30))").replace('len(eligible)<COUNT','len(eligible)<count').replace('[:COUNT]','[:count]').replace('engine_v7_allocation(t,selected,allocation)','allocation_v9(t,selected,allocation)')
assert 'COUNT' not in v9_source
exec(v9_source,globals())
# Verify original control exactly before running altered settings.
current_scores_v9=scores_v7['A']
for i in [0,200,400,649]:
    new=simulate_v9(int(window_starts[i]),'equal',25,tolerance=0);old=engine_v7_results['A_equal'][i]
    assert new['invalid']==old['invalid'] and new['violations']==old['violations']
    for f in ['return','mdd','cost','turnover']:assert np.isclose(new[f],old[f],rtol=1e-12,atol=1e-12)
redesign_results_v9={};redesign_specs_v9={}
for signal_key in mixtures_v9:
    current_scores_v9=mixed_scores_v9[signal_key]
    for count in [20,25,30,'adaptive']:
        for allocation in ['equal','conviction']:
            key=f'{signal_key}|N{count}|{allocation}'
            redesign_specs_v9[key]={'signal':signal_key,'signal_weights':mixtures_v9[signal_key],'count':count,'allocation':allocation}
            redesign_results_v9[key]=[simulate_v9(int(start),allocation,count) for start in window_starts]
    # checkpoint for all results completed so far, not a reason to stop this cell
    (recovery_root/'策略重新比較_全部窗口_v9_2026-10-15.json').write_text(json.dumps({'definition_sha256':redesign_sha_v9,'specs':redesign_specs_v9,'results':clean_v5(redesign_results_v9)},ensure_ascii=False,allow_nan=False),encoding='utf-8')
    print('V9_GRID_PROGRESS',signal_key,len(redesign_results_v9),'/136',flush=True)
assert len(redesign_results_v9)==136
print('V9_GRID_COMPLETE',136*650)


In [ ]:
# v9 purged annual selection and full-grid descriptive audit; no full-period winner
def valid_mask_v9(rows):return np.array([not row['invalid'] and row['violations']==0 for row in rows])
reference_v9=engine_v7_results['A_equal'];ref_valid_v9=valid_mask_v9(reference_v9)
ref_return_v9=np.array([row['return'] for row in reference_v9]);ref_mdd_v9=np.array([row['mdd'] for row in reference_v9])
grid_summary_v9={}
for key,rows in redesign_results_v9.items():
    valid=valid_mask_v9(rows);pair=valid&ref_valid_v9;ret=np.array([row['return'] for row in rows]);mdd=np.array([row['mdd'] for row in rows]);ix=np.flatnonzero(pair)
    grid_summary_v9[key]={'spec':redesign_specs_v9[key],'valid_windows':int(valid.sum()),'invalid_data_windows':sum(row['invalid'] for row in rows),'violation_windows':sum(row['violations']>0 for row in rows),'pairs':int(pair.sum()),'mean_paired_pp':float(np.mean(ret[pair]-ref_return_v9[pair])*100) if pair.any() else None,'p10_paired_difference_pp':float((np.quantile(ret[pair],.1)-np.quantile(ref_return_v9[pair],.1))*100) if pair.any() else None,'median_mdd_difference_pp':float((np.median(mdd[pair])-np.median(ref_mdd_v9[pair]))*100) if pair.any() else None,'mean_cost_pct':float(np.mean([rows[i]['cost'] for i in ix])*100) if len(ix) else None,'mean_turnover':float(np.mean([rows[i]['turnover'] for i in ix])) if len(ix) else None,'valid_min_held':min((rows[i]['min_held'] for i in np.flatnonzero(valid)),default=None),'valid_max_held':max((rows[i]['max_held'] for i in np.flatnonzero(valid)),default=None),'max_actual_weight':max((rows[i]['max_actual_weight'] for i in np.flatnonzero(valid)),default=None)}
pipeline_rows_v9=[dict(row,invalid=True) for row in reference_v9];folds_v9=[]
for year in [2025,2026]:
    cutoff=datetime.date(year,1,1)
    train=np.array([dates[int(s)+23]<cutoff for s in window_starts]);test=np.array([dates[int(s)].year==year for s in window_starts]);train_base=int((train&ref_valid_v9).sum());ranked=[];rejections=collections.Counter()
    for key,rows in redesign_results_v9.items():
        pair=train&ref_valid_v9&valid_mask_v9(rows);count=int(pair.sum())
        if count<100 or count<.8*train_base:rejections['insufficient_training_coverage']+=1;continue
        rr=np.array([row['return'] for row in rows]);mm=np.array([row['mdd'] for row in rows]);p10=(np.quantile(rr[pair],.1)-np.quantile(ref_return_v9[pair],.1))*100;md=(np.median(mm[pair])-np.median(ref_mdd_v9[pair]))*100
        if p10 < -1 or md < -1:rejections['training_risk_gate']+=1;continue
        mean=float(np.mean(rr[pair]-ref_return_v9[pair]));turn=float(np.mean([rows[i]['turnover'] for i in np.flatnonzero(pair)]));ranked.append((mean,-turn,key,count))
    ranked.sort(key=lambda item:(-item[0],-item[1],item[2]));chosen=ranked[0] if ranked else None
    if chosen:
        rows=redesign_results_v9[chosen[2]]
        for i in np.flatnonzero(test):pipeline_rows_v9[i]=rows[i]
        pair=test&ref_valid_v9&valid_mask_v9(rows)
        fold={'year':year,'selected':chosen[2],'signal_weights':redesign_specs_v9[chosen[2]]['signal_weights'],'training_pairs':chosen[3],'training_mean_difference_pp':chosen[0]*100,'eligible_candidates':len(ranked),'rejections':dict(rejections),'test_pairs':int(pair.sum()),'test_base_valid':int((test&ref_valid_v9).sum()),'test_candidate_valid':int((test&valid_mask_v9(rows)).sum()),'test_mean_difference_pp':float(np.mean([rows[i]['return']-reference_v9[i]['return'] for i in np.flatnonzero(pair)])*100) if pair.any() else None,'train_last_window_end':str(max(dates[int(window_starts[i])+23] for i in np.flatnonzero(train))),'test_first_start':str(min(dates[int(window_starts[i])] for i in np.flatnonzero(test)))}
        assert fold['train_last_window_end']<fold['test_first_start']
    else:fold={'year':year,'selected':None,'rejections':dict(rejections),'test_pairs':0,'reason':'no eligible training candidate; no substitute silently chosen'}
    folds_v9.append(fold);print('V9_YEAR_SELECTION',json.dumps(fold,ensure_ascii=False),flush=True)
pipeline_stats_v9=summary_v7(pipeline_rows_v9,reference_v9,family=183)
# summary_v7 includes inactive2024 in year dictionary; evaluate only actual predeclared test years.
pipeline_stats_v9['years']={k:v for k,v in pipeline_stats_v9['years'].items() if k in ['2025','2026']}
pair_v9=valid_mask_v9(pipeline_rows_v9)&ref_valid_v9;ix=np.flatnonzero(pair_v9);half=len(ix)//2
pipeline_stats_v9['halves']={k:{'pairs':len(part),'mean_difference_pp':float(np.mean([pipeline_rows_v9[i]['return']-reference_v9[i]['return'] for i in part])*100) if len(part) else None} for k,part in [('early',ix[:half]),('late',ix[half:])]}
coverage_ok_v9=all(f.get('test_pairs',0)>=.8*f.get('test_base_valid',1) and f.get('selected') for f in folds_v9)
pipeline_stats_v9['supported']=bool(coverage_ok_v9 and pipeline_stats_v9['risk_pass'] and all(pipeline_stats_v9[k][0]>0 for k in ['block24_interval_pp','block48_interval_pp']) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in pipeline_stats_v9['years'].values()) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in pipeline_stats_v9['halves'].values()))
pipeline_stats_v9['coverage_pass']=bool(coverage_ok_v9)
redesign_eval_v9={'definition_sha256':redesign_sha_v9,'grid_summary':grid_summary_v9,'folds':folds_v9,'pipeline_stats':pipeline_stats_v9,'pipeline_rows':clean_v5(pipeline_rows_v9),'no_untouched_holdout':True,'historical_industry_stock_selection':'unavailable point-in-time individual classifications','official_active_share':'not certified'}
(recovery_root/'策略重新比較_年度選擇與統計_v9_2026-10-15.json').write_text(json.dumps(clean_v5(redesign_eval_v9),ensure_ascii=False,allow_nan=False,indent=2),encoding='utf-8')
print('V9_PIPELINE_INFERENCE',json.dumps(pipeline_stats_v9,ensure_ascii=False),flush=True)


In [ ]:
# v9 ex-ante industry-index seasonality diagnostic, reusing official cached series only
sector_saved_v9=json.loads((recovery_root/'產業競賽期間_含息統計與全部窗口_2026-10-14.json').read_text(encoding='utf-8'))
sector_month_v9={(r['sector'],r['year'],r['month']):r for r in sector_saved_v9['windows'] if r.get('return_value') is not None and r.get('excess') is not None}
sector_fold_v9=[]
for year in range(2021,2026):
    ranked=[]
    for sector in sorted({k[0] for k in sector_month_v9}):
        training=[]
        for prev in range(2016,year):
            if not all((sector,prev,m) in sector_month_v9 for m in range(1,13)):continue
            october=sector_month_v9[(sector,prev,10)]['excess'];other=[sector_month_v9[(sector,prev,m)]['excess'] for m in range(1,13) if m!=10]
            training.append(october-float(np.median(other)))
        if len(training)>=5:ranked.append({'sector':sector,'train_years':len(training),'train_seasonal_difference_pp':float(np.mean(training)*100)})
    ranked.sort(key=lambda r:(-r['train_seasonal_difference_pp'],r['sector']))
    if len(ranked)<3:sector_fold_v9.append({'year':year,'status':'insufficient_prior_complete_years','eligible_sectors':len(ranked)});continue
    # Selection fixed using train only, no dropping absent test returns and re-ranking.
    selected=ranked[:3];missing=[r['sector'] for r in ranked if (r['sector'],year,10) not in sector_month_v9]
    selected_missing=[r['sector'] for r in selected if (r['sector'],year,10) not in sector_month_v9]
    if selected_missing:sector_fold_v9.append({'year':year,'status':'missing_selected_test_return','selected':selected,'missing':selected_missing});continue
    values=[sector_month_v9[(r['sector'],year,10)] for r in selected];controls=[sector_month_v9[(r['sector'],year,10)] for r in ranked] if not missing else []
    row={'year':year,'status':'diagnostic_complete' if not missing else 'top3_complete_allsector_unknown','selected':selected,'qualified_sector_count':len(ranked),'missing_control_sectors':missing,'top3_return_pct':float(np.mean([r['return_value'] for r in values])*100),'qualified_allsector_return_pct':float(np.mean([r['return_value'] for r in controls])*100) if controls else None,'benchmark_return_pct':float(values[0]['benchmark_return']*100),'top3_excess_pp':float(np.mean([r['excess'] for r in values])*100),'top3_minus_allsector_pp':float((np.mean([r['return_value'] for r in values])-np.mean([r['return_value'] for r in controls]))*100) if controls else None}
    sector_fold_v9.append(row)
seasonality_eval_v9={'definition':'each testyear prior>=5 full12month years, score Octoberexcess minus sameyear other11month medianexcess, mean across prior years, choose top3, current-year test returns excluded from selection; exact index names not spliced','folds':sector_fold_v9,'limits':['index diagnostic not executable stockportfolio','no fees','overlapping hierarchical index groups','few testyears','full dataset already examined previously','no individual point-in-time mapping; cannot combine stock scores yet','not same as current-year industry momentum']}
(recovery_root/'產業季節性_逐年向前驗證_v9_2026-10-15.json').write_text(json.dumps(seasonality_eval_v9,ensure_ascii=False,indent=2),encoding='utf-8')
print('V9_SEASONALITY',json.dumps(sector_fold_v9,ensure_ascii=False))


In [ ]:
# v9 final saved report and all-point paired graph; scope and unresolved branches explicit
import re,html as html_module
report_name_v9='策略重新比較_結果與選擇邏輯_v9_2026-10-15.md'
chart_name_v9='策略重新比較_配對箱形圖與全部候選_v9_2026-10-15.html'
plot_methods_v9={'舊A等權':engine_v7_results['A_equal'],'舊A非等權＋略過':engine_v7_results['A_mild_tolerance'],'新A等權25＋略過':redesign_results_v9['A|N25|equal'],'逐年選參數策略':pipeline_rows_v9}
common_plot_v9=np.logical_and.reduce([valid_mask_v9(rows) for rows in plot_methods_v9.values()])
plot_points_v9=[]
for name,rows in plot_methods_v9.items():
    for i in np.flatnonzero(common_plot_v9):
        row=rows[i];plot_points_v9.append({'method':name,'start':str(dates[row['start']]),'end':str(dates[row['end']]),'return_pct':row['return']*100,'cost_pct':row['cost']*100,'mdd_pct':row['mdd']*100})
template=(recovery_root/'A策略_失效機制比較_v8_2026-10-15.html').read_text(encoding='utf-8')
template=re.sub(r'const data=.*?;const labels=.*?;const names=.*?;const lo=',lambda m:'const data='+json.dumps(plot_points_v9,ensure_ascii=False)+';const labels={};const names='+json.dumps(list(plot_methods_v9),ensure_ascii=False)+';const lo=',template,count=1,flags=re.S)
template=template.replace('A失效機制比較 v8','策略重新比較 v9').replace('x=200+230*i','x=190+235*i')
intro='只畫四種方法共同有效的逐年測試日期，所有點保留。訓練年選比重／檔數／配置，測試年鎖定。不是在全部回測挑冠軍，也不是未使用過的新資料。桃紅為中位數、虛線為0%；兩個快捷鍵固定座標與方法順序。同季只剩2025一個測試年，不得據此判定比賽勝率。產業個股選股仍缺歷史分類，尚未納入。'
template=re.sub(r'<p>.*?</p>',lambda m:'<p>'+intro+'</p>',template,count=1,flags=re.S)
table=['<details><summary>全部136個候選：描述性結果（展開）</summary><p>以下使用各自和舊A共同有效日期，列間日期不同，不能按平均差直接選冠軍。財報／營收資料缺漏與違規均另列。</p><table border="1" cellpadding="5"><tr><th>候選（訊號／檔數／資金配置）</th><th>有效／650</th><th>資料排除窗口</th><th>限制不符窗口</th><th>配對數</th><th>平均差pp</th><th>平均費稅%</th></tr>']
for key,s in grid_summary_v9.items():
    fmt=lambda v:'未知' if v is None else f'{v:.3f}'
    table.append(f"<tr><td>{html_module.escape(key)}</td><td>{s['valid_windows']}</td><td>{s['invalid_data_windows']}</td><td>{s['violation_windows']}</td><td>{s['pairs']}</td><td>{fmt(s['mean_paired_pp'])}</td><td>{fmt(s['mean_cost_pct'])}</td></tr>")
table.append('</table></details>')
(recovery_root/chart_name_v9).write_text(template+'\n'+''.join(table),encoding='utf-8')
lines=['# 策略重新比較：結果與選擇邏輯 v9','2026-10-15。解除固定25檔，保留舊A基準；本輪不是官方規則已變更的宣稱。','',
'## 完成範圍','136組×650窗口＝88,400組結果全部保存，包含無效旗標、持股檔數、費用及路徑。候選：17組訊號比重×4種檔數規則×2種資金配置。舊結果直接引用，不重跑原定義。','訊號組合：動能、純法人、純低波動、本業改善、營收MoM/YoY各自100%；動能與其餘四訊號各比較25/75、50/50、75/25。分數沿用v7歷史可用資訊；財務／營收缺值排除而非當作差股票，稀疏覆蓋會影響選股池，原文與首次公開時間限制仍適用。','持股數：20、25、30，或綜合分數>=0.8的股票數限制在20–30。機會少時仍須滿足官方20檔；資料不足20檔標無效，不能持10檔卻稱合法。落選者目標0，可新進與出清。','配置：等權或exp(8*(綜合分數−0.5))，再按台積電25%、其餘10%作比例上限重配。台積電不必入選、不預設25%；實際持倉仍受每日價格漂移與整千股影響。新候選均使用0.5%NAV小額調整略過，同費稅、90%投入、每5日調整、24日窗口。這些閾值為固定研究假說，沒有聲稱最佳。','',
'## 官方規則與研究限制','依本機保存10/5辦法及D-Plan v4.3：150檔池、20–30檔、2330上限25%、其他10%、現金非負且低於25%、千股倍數、費率0.1425%及賣出稅0.3%。官網本輪403，未確認有無更新。','維持原引擎的公司行動與漂移寬限處理，不把目標上限視為實際持倉永不超限。Active Share缺歷史前15大基準及精確口徑，未認證正式官賽合規；12筆有償認購仍排除。沒有交易提交。','',
'## 逐年向前選參數','只用測試年前、持有24日期間已結束的訓練窗口。至少100個配對，且覆蓋至少80%基準訓練日期；P10和中位回撤不得惡化逾1pp。合格者依費後配對平均增量、較低換手、固定代號排序。測試年中不依結果換參數；失敗窗口照實保留，不默默改買另一組。','',
'|測試年|依過去資料選中的設定|訊號比重|訓練配對|測試配對／基準可用|測試平均差pp|','|---|---|---|---:|---:|---:|']
for f in folds_v9:
    if f.get('selected'):lines.append(f"|{f['year']}|{f['selected']}|{json.dumps(f['signal_weights'],ensure_ascii=False)}|{f['training_pairs']}|{f['test_pairs']}/{f['test_base_valid']}|{f['test_mean_difference_pp']:+.3f}|")
    else:lines.append(f"|{f['year']}|無合格設定|—|—|0|不可判定|")
ps=pipeline_stats_v9
lines+=['',f"逐年選參數流程整體：配對 {ps['pairs']}，費後平均差 {ps['mean_difference_pp']:+.3f}pp；24日區塊家族校正區間 {ps['block24_interval_pp']}pp；48日 {ps['block48_interval_pp']}pp。風險門檻：{ps['risk_pass']}；測試覆蓋門檻：{ps['coverage_pass']}；全部預先判準：{ps['supported']}。",
'家族183為既有46次比較＋136新候選＋1選擇流程的保守校正；各區塊5000次重抽樣，極端尾端模擬精度有限。測試年度與前後半期須正增量、兩區塊下界均正、風險及覆蓋皆過，才稱歷史判準通過。650窗口高度重疊；先前已看過這些歷史資料，這是逐年向前協定驗證，不是乾淨的新樣本外。','',
'## 產業季節性：原成果往前推一步','重用2016–2025官方含息產業指數。每年只用之前至少5個完整12月對照年份，以10月超額減同年其他11月超額中位數、再跨年平均排名，選前三個產業。當年結果不進排名；不事後固定選已知較好的航運。','',
'|測試年|產業（依當時可知資料）|前三產業平均報酬%|相較大盤pp|相較全部合格產業pp|','|---|---|---:|---:|---:|']
for f in sector_fold_v9:
    if f['status'] in ['diagnostic_complete','top3_complete_allsector_unknown']:
        other='缺分類對照' if f['top3_minus_allsector_pp'] is None else f"{f['top3_minus_allsector_pp']:+.3f}"
        lines.append(f"|{f['year']}|{'、'.join(x['sector'] for x in f['selected'])}|{f['top3_return_pct']:.3f}|{f['top3_excess_pp']:+.3f}|{other}|")
    else:lines.append(f"|{f['year']}|{f['status']}|—|—|—|")
lines+=['','此處為少數年度、無費用的指數層級診斷；產業指數有大類／子類重疊，不是可執行的股票組合。不把4個測試年度等同4×650個獨立樣本。沒有個股歷史產業映射，尚不能計算「產業季節性＋個股訊號」的150檔排名，因此此選股分支仍未完成；沒有用今日分類補歷史。',
'','## 選擇與後續界線',
('流程通過本次歷史門檻，仍僅可列為待新資料驗證候選，不能稱已證實可上場。' if ps['supported'] else '逐年選參數流程未通過全部門檻，不能用136組中全期間最高值替代失敗結果；目前沒有證據支持因放開檔數及加大集中就能可靠改善。'),
'保留舊等權A與溫和非等權＋略過為控制；v9沒有自動取代正式配置。使用者授權的產業個股選股仍待歷史分類，Active Share待官方歷史基準，LLM與操作演練繼續延後。排程維持PAUSED，不自動擴增參數搜尋。','',
'## 交付與驗證',f'[配對箱形圖與全部136候選表]({chart_name_v9})，四方法只畫共同有效測試日，每個點保留，固定座標、桃紅中位數、0%線、單一提示。',
'原Notebook cells205–210保存連線摘要、定義、完整候選、逐年評估、產業診斷及報告。引擎對舊A等權4個起點的費後報酬、回撤、費用、換手與無效旗標一致；未Restart and Run All。互動腳本另作NodeVM模擬DOM驗證，不宣稱實際瀏覽器目視驗證。',
'定義SHA256：'+redesign_sha_v9]
(recovery_root/report_name_v9).write_text('\n'.join(lines)+'\n',encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));branch=st['strategy_redesign_v9'];branch.update({'status':'completed_feasible_grid_and_walkforward_industry_stock_branch_pending','running_work':False,'report':report_name_v9,'chart':chart_name_v9,'candidate_windows':88400,'notebook_cells':list(range(205,211)),'supported_pipeline':ps['supported'],'folds':folds_v9,'industry_index_walkforward_completed':True,'industry_stock_mapping':'historical individual mapping unavailable; stock-level seasonal selection not tested','graph_points':len(plot_points_v9),'graph_common_dates':int(common_plot_v9.sum())});st['status']='v9_feasible_research_completed_industry_stock_mapping_pending';st['current_node']=None;st['automation_status']='PAUSED';sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8').split('\n\n## v9執行結果')[0]+'\n\n## v9執行結果\n136候選×650窗口與2025/2026逐年選參數完成；產業指數逐年向前診斷完成，個股歷史產業映射仍缺，產業選股未完成。是否通過歷史判準：'+str(ps['supported'])+'。詳見'+report_name_v9+'。無執行中分析；不自行追最高值或新調門檻。\n',encoding='utf-8')
print('V9_DELIVERABLES_SAVED','points',len(plot_points_v9),'common_dates',int(common_plot_v9.sum()),'supported',ps['supported'])


In [ ]:
# v9 artifact and execution validation; existing kernel objects remain in memory
import ast,subprocess,nbformat
assert len(redesign_results_v9)==136 and all(len(rows)==650 for rows in redesign_results_v9.values())
assert sum(len(rows) for rows in redesign_results_v9.values())==88400
for s in grid_summary_v9.values():
    if s['valid_windows']:assert 20<=s['valid_min_held']<=s['valid_max_held']<=30
for fold in folds_v9:
    if fold.get('selected'):assert fold['train_last_window_end']<fold['test_first_start']
assert not pipeline_stats_v9['supported']
assert all(max(r['train_years'] for r in f['selected'])<=f['year']-2016 for f in sector_fold_v9 if f.get('selected'))
wp=Path('C:/Users/weiha/Documents/Codex/2026-10-09/data-mining-final-project/work')
tree=ast.parse((wp/'verify_v5_deliverables.py').read_text(encoding='utf-8'))
qa=next(ast.literal_eval(n.value) for n in tree.body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='qa' for t in n.targets))
expected_season=sum(pt['start'] in ['2024-10-29','2025-10-28'] for pt in plot_points_v9)
qa=qa.replace('circles(season)!==10','circles(season)!=='+str(expected_season))
out=subprocess.run(['C:/Program Files/nodejs/node.exe','-e',qa,str(recovery_root/chart_name_v9)],capture_output=True,text=True,encoding='utf-8')
assert out.returncode==0,out.stderr[:600]
qa_result_v9=json.loads(out.stdout);assert qa_result_v9['full_points']==1284 and qa_result_v9['methods']==4
book=nbformat.read(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
for i in range(205,211):
    assert book.cells[i].outputs and not any(o.output_type=='error' or 'Execution failed:' in o.get('text','') for o in book.cells[i].outputs)
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['strategy_redesign_v9']['notebook_cells']=list(range(205,212));st['strategy_redesign_v9']['verification']={'type':'NodeVM DOM simulation, not browser visual QA',**qa_result_v9,'grid_records':88400,'saved_cell_outputs':'205-210 verified;211 appended verification','train_test_horizons':'purged'};sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
rp=recovery_root/report_name_v9;report=rp.read_text(encoding='utf-8');report+='\n最終驗證：136組各650筆完整保存；所有有效窗口實際持股20–30檔；年度訓練持有期結束早於測試起點。配對圖1284點＝321共同日期×4方法，同季4點；固定0%座標、桃紅中位數、四方法與快捷鍵NodeVM驗證通過。原Notebook另存cell211驗證輸出。產業2023–2025因未選中的觀光分類改名，全部合格產業對照未知，但所選三產業與大盤的比較仍可計算，未刪除該年度。\n';rp.write_text(report,encoding='utf-8')
print('V9_VERIFIED',json.dumps(qa_result_v9),'88400 records, no new parameter tuning; kernel retained')


In [ ]:
# User reporting requirement: after-cost 24-session win rate and empirical expectancy
print('EXISTING_KERNEL',type(redesign_results_v9).__name__,len(redesign_results_v9),'candidates',len(window_starts),'windows; no simulations rerun')
def window_metrics_v9(rows,mask=None):
    good=valid_mask_v9(rows)
    if mask is not None:good=good&mask
    a=np.array([row['return'] for row in rows])[good];a=a[np.isfinite(a)];n=len(a)
    if not n:return {'n':0,'win_rate_pct':None,'loss_rate_pct':None,'flat_rate_pct':None,'avg_win_pct':None,'avg_loss_pct':None,'expected_return_pct':None}
    wins=a>0;losses=a<0;flats=a==0
    expected=float(a.mean()*100)
    win=float(a[wins].mean()*100) if wins.any() else None;loss=float(a[losses].mean()*100) if losses.any() else None
    decomposed=float(wins.mean()*(win or 0)+losses.mean()*(loss or 0))
    assert np.isclose(expected,decomposed,rtol=1e-12,atol=1e-10)
    return {'n':n,'wins':int(wins.sum()),'losses':int(losses.sum()),'flats':int(flats.sum()),'win_rate_pct':float(wins.mean()*100),'loss_rate_pct':float(losses.mean()*100),'flat_rate_pct':float(flats.mean()*100),'avg_win_pct':win,'avg_loss_pct':loss,'expected_return_pct':expected,'expectancy_decomposed_pct':decomposed}
metric_policy_v9={'unit':'24-trading-session portfolio window, net of simulated fees and taxes','win':'net return>0; zero separately flat; loss<0','expectancy':'sample arithmetic mean net window return = pwin*meanwin + ploss*meanloss; not future guaranteed return','comparison':'paired same-date sample; report beat-reference separately from positive-return win rate','invalid':'excluded with counts disclosed; no zero imputation','inference':'overlapping windows dependent; point estimates not independent trials; retain block intervals for improvements','future_report_columns':['valid sample n','positive-return win rate','expected net return','conditional average gain','conditional average loss','beat-reference rate on paired dates']}
metrics_candidates_v9={}
for key,rows in redesign_results_v9.items():
    full=window_metrics_v9(rows);pair=valid_mask_v9(rows)&ref_valid_v9;paired=window_metrics_v9(rows,pair)
    ix=np.flatnonzero(pair);paired['reference']=window_metrics_v9(reference_v9,pair)
    paired['beat_reference_pct']=float(np.mean([rows[i]['return']>reference_v9[i]['return'] for i in ix])*100) if len(ix) else None
    paired['tie_reference_pct']=float(np.mean([rows[i]['return']==reference_v9[i]['return'] for i in ix])*100) if len(ix) else None
    metrics_candidates_v9[key]={'all_valid':full,'paired_with_A':paired}
    grid_summary_v9[key]['win_rate_pct']=paired['win_rate_pct'];grid_summary_v9[key]['expected_return_pct']=paired['expected_return_pct'];grid_summary_v9[key]['beat_A_pct']=paired['beat_reference_pct']
main_metrics_v9={name:window_metrics_v9(rows,common_plot_v9) for name,rows in plot_methods_v9.items()}
for name,rows in plot_methods_v9.items():
    ix=np.flatnonzero(common_plot_v9);main_metrics_v9[name]['beat_A_pct']=float(np.mean([rows[i]['return']>reference_v9[i]['return'] for i in ix])*100)
metrics_export_v9={'policy':metric_policy_v9,'main_common_date_metrics':main_metrics_v9,'main_common_dates':int(common_plot_v9.sum()),'candidates':metrics_candidates_v9,'pipeline_by_year':{str(y):window_metrics_v9(pipeline_rows_v9,ref_valid_v9&np.array([dates[int(s)].year==y for s in window_starts])) for y in [2025,2026]}}
(recovery_root/'策略研究_勝率與期望值_v9_2026-10-15.json').write_text(json.dumps(clean_v5(metrics_export_v9),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
def metrics_md_v9():
    lines=['## 勝率與期望值（新增固定報告項目）','勝率＝費後正報酬窗口數÷有效窗口數；零報酬另外列。期望值＝每個24交易日窗口的費後報酬平均，也等於勝率×平均獲利＋虧損率×平均虧損（負值）。這是歷史樣本估計，非未來保證。打贏A的比例另列，不與正報酬勝率混淆。','四種配置均用相同321個有效測試起點；個別候選全樣本及與A配對的兩種口徑均另存JSON。','',
    '|配置|有效窗口|勝率|費後期望報酬／24日|平均獲利窗口報酬|平均虧損窗口報酬|打贏舊A比例|','|---|---:|---:|---:|---:|---:|---:|']
    for name,m in main_metrics_v9.items():
        fmt=lambda v:'無該類窗口' if v is None else f'{v:.3f}%'
        lines.append(f"|{name}|{m['n']}|{fmt(m['win_rate_pct'])}|{fmt(m['expected_return_pct'])}|{fmt(m['avg_win_pct'])}|{fmt(m['avg_loss_pct'])}|{fmt(m['beat_A_pct'])}|")
    lines+=['','A自己與自己的勝出率為0，表示全數相同，不是A正報酬勝率為0。高勝率可能被少數大虧損抵銷，須與期望值及風險一起看；不同資料覆蓋的方法不能直接按全樣本期望值排名。滑動窗口重疊，不能當321次獨立比賽。']
    return '\n'.join(lines)
rp=recovery_root/report_name_v9;report=rp.read_text(encoding='utf-8');report+='\n\n'+metrics_md_v9()+'\n';rp.write_text(report,encoding='utf-8')
chart_path=recovery_root/chart_name_v9;chart_text=chart_path.read_text(encoding='utf-8')
chart_text=chart_text.replace('<th>平均費稅%</th>','<th>平均費稅%</th><th>正報酬勝率%</th><th>費後期望報酬%／24日</th><th>打贏舊A%</th>')
for key,mm in metrics_candidates_v9.items():
    row_pattern=r'(<tr><td>'+re.escape(html_module.escape(key))+r'</td>.*?)</tr>'
    m=mm['paired_with_A'];fmt=lambda v:'未知' if v is None else f'{v:.3f}'
    suffix=''.join('<td>'+fmt(m[k])+'</td>' for k in ['win_rate_pct','expected_return_pct','beat_reference_pct'])
    chart_text,count=re.subn(row_pattern,lambda match:match.group(1)+suffix+'</tr>',chart_text,count=1,flags=re.S);assert count==1,key
main_table=['<h2>勝率與期望值：相同321個測試日期</h2><p>每次為24交易日、費後報酬。勝率指正報酬；期望值為歷史平均，非未來保證。窗口重疊，非獨立比賽。</p><table border="1" cellpadding="5"><tr><th>配置</th><th>窗口數</th><th>勝率%</th><th>期望報酬%／24日</th><th>平均獲利%</th><th>平均虧損%</th></tr>']
for name,m in main_metrics_v9.items():
    main_table.append('<tr><td>'+html_module.escape(name)+'</td><td>'+str(m['n'])+'</td>'+''.join('<td>'+('未知' if m[k] is None else f'{m[k]:.3f}')+'</td>' for k in ['win_rate_pct','expected_return_pct','avg_win_pct','avg_loss_pct'])+'</tr>')
main_table.append('</table>')
chart_text=chart_text.replace('<details>',''.join(main_table)+'<details>',1)
chart_path.write_text(chart_text,encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['reporting_requirements']=metric_policy_v9;st['strategy_redesign_v9']['win_expectancy_report']='策略研究_勝率與期望值_v9_2026-10-15.json';st['strategy_redesign_v9']['notebook_cells'].append(212);sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## 固定報告要求：勝率與期望值\n使用者2026-10-15要求：之後每次策略比較必列費後24日正報酬勝率、平均報酬（樣本期望值）、有效窗口數、平均獲利／平均虧損；零報酬單列。對基準比較須同日期，勝過基準比例另列，不與正報酬勝率混淆。區塊區間、重疊窗口、資料覆蓋與風險限制仍保留。v9已補齊136候選全有效與配對兩種口徑；不得因只看高勝率省略虧損幅度。\n',encoding='utf-8')
assert chart_text.count('<th>正報酬勝率%</th>')==1
assert all(m['n']==321 for m in main_metrics_v9.values())
print('WIN_EXPECTANCY_SAVED',json.dumps(main_metrics_v9,ensure_ascii=False))


In [ ]:
# v10 complete finite batch registration; no repeated approval checkpoints
import json,hashlib,datetime,collections
print('EXISTING_MEMORY',len(redesign_results_v9),'v9 candidates',signal_returns_v7.shape,'v8 feature shapes',beta_v8.shape,hot_v8.shape)
bulk_definition_v10={
 'version':'v10','date':'2026-10-15','human_request':'plan all remaining experiments once and run feasible branches continuously; no per-node approval waits',
 'data':'unchanged v7 snapshot and action corrections; existing v8 exante features; v9 result objects reused',
 'constraints':'20..30actual stocks,90pctinvested,24days,5dayrebalance,1000shares,existingfees,caps25pct2330/10pctothers;ActiveShare uncertified',
 'nodes':[
 {'id':'reuse','status':'reuse_only','definition':'v9 136 candidates and v8/v7; no duplicate simulation'},
 {'id':'multi_signal','count':32,'definition':'equal thirds A/FLOW/LOWVOL; A/FIN/REV; FLOW/FIN/REV; equal fifths all5;4count rules*2allocations'},
 {'id':'short_reversal','count':16,'definition':'REV5 percentile restandardized to0..1; pure or75pctA+25pctREV5;4count rules*2allocations'},
 {'id':'risk_overlays','count':16,'definition':'A selection; same v8 marketprior20<0 beta penalty or extensionz>2*.75;4count rules*2allocations, with new official caps'},
 {'id':'special_cap','count':8,'definition':'A exp32 allocation to actually test stronger concentration;4count rules*2330cap25pct versus uniform10pct; caps not minimums'},
 {'id':'entry_exit_buffer','count':8,'definition':'A selection; incumbent eligible if score rank<=N+5, retain up toN then fill best new;4count rules*2allocations; same .005retained-trade skip'},
 {'id':'industry_mapping','status':'source_gate','definition':'reuse DB schema audit(no historical validity); two new official MOPS historical monthly-report endpoints inaccessible via web2026-10-15. Need dated stock-level classification; no current-classification backfill; index walkforward v9 reused'},
 {'id':'stability','status':'planned','definition':'paired 24/48blocks, family264=183 prior+80new+1expanded selection pipeline;5000reps; pooled purged annual2025/26 selector; same v9 training gates; fixed nonoverlapping sensitivity offsets0..23'},
 {'id':'report','status':'planned','definition':'all80new and136existing; win/expectancy, avgwin/loss, fees, drawdown, coverage, paired benchmark, all points, conclusions and source gaps'}],
 'new_candidates':80,'total_reused_and_new':216,'new_window_records':52000,
 'count_rules':[20,25,30,'adaptive'],'adaptive':'count scores>=.8 clipped20..30 as v9',
 'allocations':['equal','conviction exp8'],'missing':'any positive-weight component unknown excludes stock; insufficient N invalid, never missing=bad',
 'selection':'same v9 training coverage80pct,min100pairs,P10/medianMDD within1pp; choose trainmeanincrement then turnover then ID; outcomes end before nextyear start; lock testyear',
 'special_cap_test':'gamma32 registered before outcomes; not search gamma grid; compare identical definition except cap2330',
 'reporting':'positive-return win rate,mean net24day return,avgwin,avgloss,n; paired same-date controls; return-minus-control and beat-control separate',
 'source_attempt_limit':'existing attempts retained; no repeated endless crawl; unavailable source node ends unavailable with evidence',
 'deferred':['LLM','formal account/DPlan/DataPack','operational rehearsal','new public website','new numbered project/repository','live trades'],
 'stop':'all feasible nodes saved and evaluated, conditional unavailable branch named; no adding experiments based on best return; automation remainsPAUSED'
}
bulk_sha_v10=hashlib.sha256(json.dumps(bulk_definition_v10,sort_keys=True).encode()).hexdigest()
(recovery_root/'策略批次研究_預先定義_v10_2026-10-15.json').write_text(json.dumps(bulk_definition_v10,ensure_ascii=False,indent=2),encoding='utf-8')
bulk_plan_v10="""# 一次規畫、連續執行：策略研究 v10

使用者要求：先一次登記全部剩餘實驗，連續完成可行分支，不因完成一個cell或checkpoint停下等待檢查。不新增子代理、不開新對話、不自行恢復20分鐘排程。舊結果去重引用。

|順序|實驗|新候選數|完成條件|
|---|---|---:|---|
|1|既有資料／引擎與136候選|0|沿用kernel與快取，確認維度|
|2|三種三訊號等比組合＋五訊號等比組合|32|4種檔數×2種配置，650窗口全保存|
|3|純五日反轉、75%動能＋25%反轉|16|反轉先轉百分位，與其他分數同尺度|
|4|市場狀態beta降權、過熱降權|16|保留v8事先訊號規則，套新檔數與官方權重上限|
|5|台積電特殊上限的影響|8|較集中exp32原始權重；同設定比較2330上限25%或10%，不強制持有|
|6|降低整檔進出換手|8|保留排名前N+5的既有股票，補滿N；對照v9同N同配置|
|7|歷史產業映射來源門檻|條件分支|舊DB無PIT分類；新官方月報兩端點不可存取；無驗證時間不回填，v9產業指數驗證直接引用|
|8|擴充逐年選參數與穩健性|1流程|訓練期結束早於測試起點，216候選一起選；24/48日區塊及不重疊敏感度|
|9|統一成果|0|全部表、勝率、期望值、風險、費稅、缺值、互動全部點與結論保存|

新增80×650＝52,000筆，合併舊136候選共216組。固定24日、每5日、90%投入、整千股、原費稅與公司行動處理。變動20/25/30或自適應20–30檔。未知分數排除，不當負面訊號。特殊股上限不是應配置比重。

所有新參數在結果產生前固定。重用v9逐年選擇規則，不看測試年改配方；家族264保守包含歷次與本批試驗。勝率是費後正報酬比例，期望值是費後24日平均；與打贏A比例分開。沒有未使用的新資料，不能稱乾淨樣本外。

未完成的產業個股策略若仍缺歷史分類，保留不可判定，繼續其他分支；不以已完成指數研究冒充個股研究。LLM及官方帳戶／交件／演練仍延後。所有可行工作完成後一次交付，不為取得顯著結果追加候選。
"""
(recovery_root/'策略批次研究_完整計畫_v10_2026-10-15.md').write_text(bulk_plan_v10,encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['bulk_v10']={'status':'registered_running','sha256':bulk_sha_v10,'new_candidates':80,'running_work':True,'queue':[r['id'] for r in bulk_definition_v10['nodes']],'source_gate':'historical classification unavailable, no current backfill','deferred':bulk_definition_v10['deferred']};sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## 現行入口v10：整批連續研究\n使用者要求一次規畫並執行所有當下可行分支，不停在checkpoint。唯一新批次定義為策略批次研究_預先定義_v10_2026-10-15.json；完整計畫同名v10.md。80新候選、136舊候選引用，條件資料分支明列。工作未完成持續下一節點；不因20分鐘而切批，不自行追加候選，不開排程。\n',encoding='utf-8')
print('BULK_V10_PREREGISTERED',bulk_sha_v10,'80new;216total;no approval checkpoints')


In [ ]:
# Full finite v10 grid, continuous execution; no data reload and no old simulation reruns
rev_percentile_v10=np.full_like(scores_v7['REV5'],np.nan)
for q in range(len(dates)):
    eligible=np.isfinite(scores_v7['REV5'][q]);rev_percentile_v10[q]=percentile(scores_v7['REV5'][q],eligible)
bulk_signal_weights_v10={'A_FLOW_LOWVOL':{'A':1/3,'PURE_FLOW':1/3,'PURE_LOWVOL':1/3},'A_FIN_REVENUE':{'A':1/3,'PURE_FINANCIAL':1/3,'PURE_REVENUE':1/3},'FLOW_FIN_REVENUE':{'PURE_FLOW':1/3,'PURE_FINANCIAL':1/3,'PURE_REVENUE':1/3},'ALL5':{k:.2 for k in redesign_definition_v9['signal_components']},'REV5':{'REV5_PERCENTILE':1.},'A75_REV25':{'A':.75,'REV5_PERCENTILE':.25}}
bulk_scores_v10={}
for name,weights in bulk_signal_weights_v10.items():bulk_scores_v10[name]=sum((rev_percentile_v10 if k=='REV5_PERCENTILE' else scores_v7[k])*w for k,w in weights.items())
bulk_scores_v10['A']=scores_v7['A']
bulk_specs_v10={}
for name in bulk_signal_weights_v10:
    for count in [20,25,30,'adaptive']:
        for alloc in ['equal','conviction']:bulk_specs_v10[f'{name}|N{count}|{alloc}']={'family':'multi_signal' if name not in ['REV5','A75_REV25'] else 'short_reversal','signal':name,'count':count,'allocation':alloc,'gamma':8,'overlay':None,'buffer':False,'cap2330':.25}
for overlay in ['R','E']:
    for count in [20,25,30,'adaptive']:
        for alloc in ['equal','conviction']:bulk_specs_v10[f'A_{overlay}|N{count}|{alloc}']={'family':'risk_overlays','signal':'A','count':count,'allocation':alloc,'gamma':8,'overlay':overlay,'buffer':False,'cap2330':.25}
for count in [20,25,30,'adaptive']:
    for cap in [.10,.25]:bulk_specs_v10[f'A_exp32|N{count}|cap2330_{cap}']={'family':'special_cap','signal':'A','count':count,'allocation':'conviction','gamma':32,'overlay':None,'buffer':False,'cap2330':cap}
    for alloc in ['equal','conviction']:bulk_specs_v10[f'A_buffer5|N{count}|{alloc}']={'family':'entry_exit_buffer','signal':'A','count':count,'allocation':alloc,'gamma':8,'overlay':None,'buffer':True,'cap2330':.25}
assert len(bulk_specs_v10)==80
def bulk_allocation_v10(t,selected,allocation):
    spec=current_spec_v10
    raw=np.ones(len(selected)) if allocation=='equal' else np.exp(spec['gamma']*(current_scores_v9[t-1,selected]-.5))
    if spec['overlay']=='R' and np.isfinite(market_prior20_v8[t]) and market_prior20_v8[t]<0:
        bb=beta_v8[t,selected];raw=raw*np.where(np.isfinite(bb),1/(1+np.clip(bb,0,3)),1.)
    if spec['overlay']=='E':raw=raw*np.where(hot_v8[t,selected],.75,1.)
    caps=np.array([spec['cap2330'] if stock_ids[j]=='2330' else .1 for j in selected]);out=np.zeros(len(selected));active=np.ones(len(selected),bool);remaining=.9
    for _ in range(len(selected)+1):
        w=raw[active]/raw[active].sum()*remaining;hit=w>caps[active]+1e-12;ii=np.flatnonzero(active)
        if not hit.any():out[ii]=w;break
        out[ii[hit]]=caps[ii[hit]];remaining=.9-out.sum();active[ii[hit]]=False
    assert np.isclose(out.sum(),.9) and np.all(out<=caps+1e-10) and np.all(out>0)
    return out
bulk_source_v10=v9_source.replace('def simulate_v9(','def simulate_bulk_v10(').replace('allocation_v9(t,selected,allocation)','bulk_allocation_v10(t,selected,allocation)')
old_selection="selected=eligible[np.argsort(-rank[eligible],kind='stable')[:count]]"
new_selection="""ordered=eligible[np.argsort(-rank[eligible],kind='stable')]
            selected=ordered[:count]
            if current_spec_v10['buffer'] and offset>0:
                kept=[int(j) for j in ordered[:min(count+5,len(ordered))] if shares[j]>0][:count]
                fill=[int(j) for j in ordered if j not in kept][:count-len(kept)]
                selected=np.array(kept+fill,dtype=int)
                # Preserve score order for identical allocation semantics.
                selected=selected[np.argsort(-rank[selected],kind='stable')]"""
assert old_selection in bulk_source_v10;bulk_source_v10=bulk_source_v10.replace(old_selection,new_selection)
exec(bulk_source_v10,globals())
current_spec_v10={'signal':'A','gamma':8,'overlay':None,'buffer':False,'cap2330':.25};current_scores_v9=scores_v7['A']
for i in [0,200,400,649]:
    new=simulate_bulk_v10(int(window_starts[i]),'conviction',25);old=redesign_results_v9['A|N25|conviction'][i]
    assert new['invalid']==old['invalid'] and new['violations']==old['violations']
    for f in ['return','mdd','turnover','cost']:assert np.isclose(new[f],old[f],rtol=1e-12,atol=1e-12)
bulk_results_v10={};last_family=None
for key,spec in bulk_specs_v10.items():
    current_spec_v10=spec;current_scores_v9=bulk_scores_v10[spec['signal']]
    bulk_results_v10[key]=[simulate_bulk_v10(int(start),spec['allocation'],spec['count']) for start in window_starts]
    if len(bulk_results_v10)%8==0:
        (recovery_root/'策略批次研究_全部新窗口_v10_2026-10-15.json').write_text(json.dumps({'definition_sha256':bulk_sha_v10,'specs':bulk_specs_v10,'results':clean_v5(bulk_results_v10)},ensure_ascii=False,allow_nan=False),encoding='utf-8')
        print('BULK_V10_PROGRESS',len(bulk_results_v10),'/80',key,flush=True)
assert len(bulk_results_v10)==80
print('BULK_V10_ALL_NEW_SAVED',52000)


In [ ]:
# Unified216 candidate inference; simultaneous resampling of shared time positions
all_results_v10={**redesign_results_v9,**bulk_results_v10};all_keys_v10=list(all_results_v10)
assert len(all_results_v10)==216
all_metrics_v10={};paired_diffs_v10=[]
for key,rows in all_results_v10.items():
    valid=valid_mask_v9(rows);pair=valid&ref_valid_v9;ix=np.flatnonzero(pair)
    m={'all_valid':window_metrics_v9(rows),'paired':window_metrics_v9(rows,pair),'valid_windows':int(valid.sum()),'invalid_data_windows':sum(row['invalid'] for row in rows),'violation_windows':sum(row['violations']>0 for row in rows),'years':{},'halves':{}}
    diff=np.where(pair,np.array([row['return'] for row in rows])-ref_return_v9,np.nan);paired_diffs_v10.append(diff)
    m['paired']['reference']=window_metrics_v9(reference_v9,pair);m['paired']['mean_difference_pp']=float(np.nanmean(diff)*100) if len(ix) else None
    m['paired']['beat_A_pct']=float(np.mean(diff[pair]>0)*100) if len(ix) else None
    m['paired']['cost_pct']=float(np.mean([rows[i]['cost'] for i in ix])*100) if len(ix) else None
    m['paired']['turnover']=float(np.mean([rows[i]['turnover'] for i in ix])) if len(ix) else None
    m['paired']['p10_difference_pp']=float((np.quantile([rows[i]['return'] for i in ix],.1)-np.quantile(ref_return_v9[pair],.1))*100) if len(ix) else None
    m['paired']['median_mdd_difference_pp']=float((np.median([rows[i]['mdd'] for i in ix])-np.median(ref_mdd_v9[pair]))*100) if len(ix) else None
    m['risk_pass']=bool(len(ix) and m['paired']['p10_difference_pp']>=-1 and m['paired']['median_mdd_difference_pp']>=-1)
    for year in [2024,2025,2026]:
        keep=pair&np.array([dates[int(s)].year==year for s in window_starts]);m['years'][str(year)]={'pairs':int(keep.sum()),'mean_difference_pp':float(np.nanmean(diff[keep])*100) if keep.any() else None}
    for label,mask in [('early',np.arange(len(pair))<len(pair)//2),('late',np.arange(len(pair))>=len(pair)//2)]:
        keep=pair&mask;m['halves'][label]={'pairs':int(keep.sum()),'mean_difference_pp':float(np.nanmean(diff[keep])*100) if keep.any() else None}
    all_metrics_v10[key]=m
diff_matrix_v10=np.asarray(paired_diffs_v10);n_windows_v10=diff_matrix_v10.shape[1]
def simultaneous_blocks_v10(matrix,length,seed,reps=5000,family=264):
    rng=np.random.default_rng(seed);n=matrix.shape[1];blocks=int(np.ceil(n/length))
    starts=rng.integers(0,n-length+1,size=(reps,blocks));sample=(starts[:,:,None]+np.arange(length)).reshape(reps,-1)[:,:n]
    codes=sample+np.arange(reps)[:,None]*n
    counts=np.bincount(codes.ravel(),minlength=reps*n).reshape(reps,n).astype(float)
    sums=counts@np.nan_to_num(matrix,nan=0).T;denom=counts@np.isfinite(matrix).astype(float).T
    means=np.divide(sums,denom,out=np.full_like(sums,np.nan),where=denom>0)*100
    alpha=.05/family
    return np.nanquantile(means,[alpha/2,1-alpha/2],axis=0).T
intervals_v10={}
for length in [24,48]:
    intervals_v10[length]=simultaneous_blocks_v10(diff_matrix_v10,length,seed=2641000+length)
    # Independent scalar implementation equivalence on two distinct missingness patterns.
    for j in [0,len(all_keys_v10)-1]:
        scalar=paired_block_interval_v5(diff_matrix_v10[j],length,2641000+length,family=264,reps=5000)
        assert np.allclose(intervals_v10[length][j],scalar,rtol=1e-10,atol=1e-10,equal_nan=True)
    print('BULK_V10_BLOCKS_COMPLETE',length,'all216',flush=True)
for j,key in enumerate(all_keys_v10):
    m=all_metrics_v10[key];m['block24_interval_pp']=intervals_v10[24][j].tolist();m['block48_interval_pp']=intervals_v10[48][j].tolist()
    m['supported_historical']=bool(m['risk_pass'] and m['paired']['n']>=100 and all(m[k][0]>0 for k in ['block24_interval_pp','block48_interval_pp']) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in m['years'].values()) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in m['halves'].values()))
supported_historical_v10=[k for k,m in all_metrics_v10.items() if m['supported_historical']]
pipeline_v10=[dict(row,invalid=True) for row in reference_v9];bulk_folds_v10=[]
for year in [2025,2026]:
    cutoff=datetime.date(year,1,1);train=np.array([dates[int(s)+23]<cutoff for s in window_starts]);test=np.array([dates[int(s)].year==year for s in window_starts]);train_base=int((train&ref_valid_v9).sum());ranked=[];rejected=collections.Counter()
    for key,rows in all_results_v10.items():
        pair=train&ref_valid_v9&valid_mask_v9(rows);count=int(pair.sum())
        if count<100 or count<.8*train_base:rejected['training_coverage']+=1;continue
        rr=np.array([row['return'] for row in rows]);mm=np.array([row['mdd'] for row in rows]);p10=(np.quantile(rr[pair],.1)-np.quantile(ref_return_v9[pair],.1))*100;mdd=(np.median(mm[pair])-np.median(ref_mdd_v9[pair]))*100
        if p10 < -1 or mdd < -1:rejected['training_risk']+=1;continue
        mean=float(np.mean(rr[pair]-ref_return_v9[pair]));turn=float(np.mean([rows[i]['turnover'] for i in np.flatnonzero(pair)]));ranked.append((mean,turn,key,count))
    ranked.sort(key=lambda row:(-row[0],row[1],row[2]));chosen=ranked[0] if ranked else None
    if chosen:
        rows=all_results_v10[chosen[2]]
        for i in np.flatnonzero(test):pipeline_v10[i]=rows[i]
        pair=test&ref_valid_v9&valid_mask_v9(rows);ix=np.flatnonzero(pair)
        f={'year':year,'selected':chosen[2],'training_pairs':chosen[3],'training_mean_difference_pp':chosen[0]*100,'eligible':len(ranked),'rejected':dict(rejected),'test_base_valid':int((test&ref_valid_v9).sum()),'test_pairs':len(ix),'test_win_expectancy':window_metrics_v9(rows,pair),'test_mean_difference_pp':float(np.mean([rows[i]['return']-reference_v9[i]['return'] for i in ix])*100) if len(ix) else None,'train_last_end':str(max(dates[int(window_starts[i])+23] for i in np.flatnonzero(train))),'test_first_start':str(min(dates[int(window_starts[i])] for i in np.flatnonzero(test)))}
        assert f['train_last_end']<f['test_first_start']
    else:f={'year':year,'selected':None,'test_pairs':0,'test_base_valid':int((test&ref_valid_v9).sum()),'rejected':dict(rejected)}
    bulk_folds_v10.append(f);print('BULK_V10_FOLD',json.dumps(f,ensure_ascii=False),flush=True)
pipeline_stats_v10=summary_v7(pipeline_v10,reference_v9,family=264)
pipeline_stats_v10['years']={k:v for k,v in pipeline_stats_v10['years'].items() if k in ['2025','2026']}
pipeline_pair_v10=valid_mask_v9(pipeline_v10)&ref_valid_v9;ii=np.flatnonzero(pipeline_pair_v10);half=len(ii)//2
pipeline_stats_v10['halves']={k:{'pairs':len(ix),'mean_difference_pp':float(np.mean([pipeline_v10[i]['return']-reference_v9[i]['return'] for i in ix])*100) if len(ix) else None} for k,ix in [('early',ii[:half]),('late',ii[half:])]}
coverage_v10=all(f['test_pairs']>=.8*f['test_base_valid'] and f['selected'] is not None for f in bulk_folds_v10)
pipeline_stats_v10['coverage_pass']=bool(coverage_v10);pipeline_stats_v10['win_expectancy']=window_metrics_v9(pipeline_v10,pipeline_pair_v10)
pipeline_stats_v10['supported']=bool(coverage_v10 and pipeline_stats_v10['risk_pass'] and all(pipeline_stats_v10[k][0]>0 for k in ['block24_interval_pp','block48_interval_pp']) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in pipeline_stats_v10['years'].values()) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in pipeline_stats_v10['halves'].values()))
nonoverlap_v10=[]
for offset in range(24):
    mask=pipeline_pair_v10&(np.arange(len(window_starts))%24==offset);ix=np.flatnonzero(mask)
    assert all(window_starts[b]-window_starts[a]>=24 for a,b in zip(ix,ix[1:]))
    nonoverlap_v10.append({'offset':offset,'n':len(ix),'mean_difference_pp':float(np.mean([pipeline_v10[i]['return']-reference_v9[i]['return'] for i in ix])*100) if len(ix) else None,'win_expectancy':window_metrics_v9(pipeline_v10,mask)})
# Isolated mechanism comparisons are descriptive secondary contrasts, no extra significance claim.
mechanism_contrasts_v10={}
for key,spec in bulk_specs_v10.items():
    if spec['family'] in ['risk_overlays','entry_exit_buffer']:control=f"A|N{spec['count']}|{spec['allocation']}"
    elif spec['family']=='special_cap' and spec['cap2330']==.25:control=key.replace('cap2330_0.25','cap2330_0.1')
    else:continue
    rows=bulk_results_v10[key];base=all_results_v10[control];mask=valid_mask_v9(rows)&valid_mask_v9(base);ix=np.flatnonzero(mask)
    mechanism_contrasts_v10[key]={'control':control,'pairs':len(ix),'mean_difference_pp':float(np.mean([rows[i]['return']-base[i]['return'] for i in ix])*100) if len(ix) else None,'mean_fee_saving_pp':float(np.mean([base[i]['cost']-rows[i]['cost'] for i in ix])*100) if len(ix) else None,'net_metrics':window_metrics_v9(rows,mask),'type':'descriptive secondary; no significance declaration'}
source_gate_v10={'status':'unavailable_verified_historical_individual_classification','prior_schema_evidence':'stocks snapshot updated2026-10-11 only; no valid_from/valid_to table found in prior authorized readonly DB audit','new_attempts':[{'url':'https://mops.twse.com.tw/nas/t21/sii/t21sc03_113_9.html','result':'web inaccessible2026-10-15'},{'url':'https://mopsov.twse.com.tw/nas/t21/sii/t21sc03_113_9.html','result':'web inaccessible2026-10-15'}],'industry_index':'v9 prior-year-only index validation reused; not stock portfolio','not_claimed':'did not merge current industry labels into historical rankings; did not claim all potential external archives exhausted'}
bulk_eval_v10={'definition_sha256':bulk_sha_v10,'candidate_metrics':all_metrics_v10,'candidate_count':216,'new_candidates':80,'family':264,'supported_historical':supported_historical_v10,'folds':bulk_folds_v10,'pipeline_stats':pipeline_stats_v10,'pipeline_rows':clean_v5(pipeline_v10),'nonoverlap_sensitivity':nonoverlap_v10,'nonoverlap_note':'24starting offsets overlap with each other, not24independent experiments','mechanism_contrasts':mechanism_contrasts_v10,'source_gate':source_gate_v10,'limits':['5000bootstrap tail quantiles atfamily264 have substantial MonteCarlo error; exploratory intervals','all dates already studied; no pristine holdout','missing financial/revenue coverage is not evidence those factors fail','official ActiveShare uncertified','historical current-universe survivorship bias','12unmodeled paidcapital events excluded']}
(recovery_root/'策略批次研究_統一統計與逐年驗證_v10_2026-10-15.json').write_text(json.dumps(clean_v5(bulk_eval_v10),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
(recovery_root/'策略批次研究_來源缺口_v10_2026-10-15.json').write_text(json.dumps(source_gate_v10,ensure_ascii=False,indent=2),encoding='utf-8')
print('BULK_V10_SUPPORTED_HISTORICAL',supported_historical_v10)
print('BULK_V10_PIPELINE',json.dumps(pipeline_stats_v10,ensure_ascii=False))
print('BULK_V10_NONOVERLAP',len(nonoverlap_v10),'positive_offsets',sum(x['mean_difference_pp'] is not None and x['mean_difference_pp']>0 for x in nonoverlap_v10))


In [ ]:
# One final delivery for entire feasible batch, not per-node approval
import re,html as html_mod
bulk_report_v10='策略批次研究_總結與選擇邏輯_v10_2026-10-15.md'
bulk_chart_v10='策略批次研究_全部比較與箱形圖_v10_2026-10-15.html'
bulk_plot_methods_v10={'舊A等權':reference_v9,'舊A非等權＋略過':engine_v7_results['A_mild_tolerance'],'v9逐年選參數':pipeline_rows_v9,'v10逐年選參數':pipeline_v10}
bulk_common_v10=np.logical_and.reduce([valid_mask_v9(rows) for rows in bulk_plot_methods_v10.values()])
bulk_main_metrics_v10={name:window_metrics_v9(rows,bulk_common_v10) for name,rows in bulk_plot_methods_v10.items()}
bulk_points_v10=[]
for name,rows in bulk_plot_methods_v10.items():
    for i in np.flatnonzero(bulk_common_v10):
        row=rows[i];bulk_points_v10.append({'method':name,'start':str(dates[row['start']]),'end':str(dates[row['end']]),'return_pct':row['return']*100,'cost_pct':row['cost']*100,'mdd_pct':row['mdd']*100})
template=(recovery_root/'策略重新比較_配對箱形圖與全部候選_v9_2026-10-15.html').read_text(encoding='utf-8').split('</script>',1)[0]+'</script>'
template=re.sub(r'const data=.*?;const labels=.*?;const names=.*?;const lo=',lambda m:'const data='+json.dumps(bulk_points_v10,ensure_ascii=False)+';const labels={};const names='+json.dumps(list(bulk_plot_methods_v10),ensure_ascii=False)+';const lo=',template,count=1,flags=re.S)
template=re.sub(r'<title>.*?</title>','<title>策略整批研究 v10</title>',template,count=1)
template=re.sub(r'<h1>.*?</h1>','<h1>策略整批研究 v10：216組統一比較</h1>',template,count=1)
intro='80組新實驗連續跑完，136組舊實驗引用。下圖僅畫四個策略共同有效測試日期，所有點保留；桃紅為中位數、虛線為0%，兩個快捷鍵固定座標。勝率是費後正報酬比例，期望值是費後24日平均。窗口重疊、不是獨立比賽；全部歷史已看過。產業個股策略仍缺歷史分類，LLM未測。'
template=re.sub(r'<p>.*?</p>',lambda m:'<p>'+intro+'</p>',template,count=1,flags=re.S)
fmt=lambda v:'未知' if v is None else f'{v:.3f}'
tables=['<h2>同日期勝率與期望值</h2><table border="1" cellpadding="5"><tr><th>策略</th><th>窗口</th><th>勝率%</th><th>期望值%／24日</th><th>平均獲利%</th><th>平均虧損%</th></tr>']
for name,m in bulk_main_metrics_v10.items():tables.append('<tr><td>'+html_mod.escape(name)+'</td><td>'+str(m['n'])+'</td>'+''.join('<td>'+fmt(m[k])+'</td>' for k in ['win_rate_pct','expected_return_pct','avg_win_pct','avg_loss_pct'])+'</tr>')
tables+=['</table>','<details><summary>全部216個候選與校正區間（展開）</summary><p>各列與A使用各自共同有效日期，不能按不同樣本的均值排名。所有費用後結果；區間為探索性區塊重抽樣，264家族、5000次尾端精度有限。</p><table border="1" cellpadding="4"><tr><th>候選</th><th>有效／650</th><th>配對n</th><th>資料排除</th><th>限制不符</th><th>勝率%</th><th>期望值%</th><th>平均獲利%</th><th>平均虧損%</th><th>費用%</th><th>對A差pp</th><th>打贏A%</th><th>24日區塊pp</th><th>48日區塊pp</th><th>歷史門檻</th></tr>']
for key,m in all_metrics_v10.items():
    a=m['paired'];vals=[str(m['valid_windows']),str(a['n']),str(m['invalid_data_windows']),str(m['violation_windows'])]+[fmt(a[k]) for k in ['win_rate_pct','expected_return_pct','avg_win_pct','avg_loss_pct','cost_pct','mean_difference_pp','beat_A_pct']]+[str([round(v,3) for v in m[k]]) for k in ['block24_interval_pp','block48_interval_pp']]+['通過（歷史）' if m['supported_historical'] else '未通過']
    tables.append('<tr><td>'+html_mod.escape(key)+'</td>'+''.join('<td>'+v+'</td>' for v in vals)+'</tr>')
tables.append('</table></details>')
(recovery_root/bulk_chart_v10).write_text(template+'\n'+''.join(tables),encoding='utf-8')
lines=['# 策略整批研究 v10：總結與選擇邏輯','日期：2026-10-15。本批已連續完成所有可行實驗，沒有停在小節等待使用者檢查。','',
'## 執行盤點','新增80候選×650窗口＝52,000筆，既有136候選×650＝88,400筆直接引用。合計216候選、140,400筆；完整新窗口獨立保存，統計合併比較。沒有重啟kernel、重抓成功資料、寫DB或提交交易。','',
'|家族|新增數|定義|','|---|---:|---|',
'|多訊號|32|動能＋法人＋低波動、動能＋財務＋營收、法人＋財務＋營收各等分；五訊號各20%|',
'|短期反轉|16|純五日反轉百分位，或75%動能＋25%反轉|',
'|風險降權|16|v8市場狀態beta規則或過熱懲罰，使用新檔數與上限|',
'|台積電上限|8|較集中exp32配置，同檔數比較2330上限10%或25%|',
'|進出緩衝|8|保留排名前N+5的既有股票，再補足N，減少整檔退出與新進|',
'所有家族皆事先固定，使用20/25/30或訊號門檻自適應20–30檔；依家族比較等權／exp8。總投入90%、24日、五日調整、整千股、原費稅、0.5%NAV小額調整略過。落選股票為0權重；特殊上限不是目標保證。','',
'## 全部候選與選擇流程','216候選全部與舊A等權作同日期費後比較，家族264保守涵蓋過去比較、80新候選與一個擴充選擇流程。以24／48原位置區塊重抽樣各5000次，候選間使用共同抽樣；兩個樣本以原逐次函式驗證數值一致。','全樣本歷史門檻通過名單：'+('、'.join(supported_historical_v10) if supported_historical_v10 else '無')+'。這不是未使用資料的新證據，也不能用來跳過逐年驗證。',
'逐年流程只用測試年前已結束的持有窗口；訓練須至少100配對且覆蓋80%基準可用日期，P10與中位回撤不惡化逾1pp，合格候選再依訓練配對平均增量、換手、固定代號選擇。測試年中不改參數，缺值不偷偷換策略。','',
'|測試年|當時訓練選中的設定|訓練配對|測試配對／A可用|測試勝率|測試期望值／24日|對A平均差pp|','|---|---|---:|---:|---:|---:|---:|']
for f in bulk_folds_v10:
    if f.get('selected'):
        m=f['test_win_expectancy'];lines.append(f"|{f['year']}|{f['selected']}|{f['training_pairs']}|{f['test_pairs']}/{f['test_base_valid']}|{fmt(m['win_rate_pct'])}%|{fmt(m['expected_return_pct'])}%|{f['test_mean_difference_pp']:+.3f}|")
    else:lines.append(f"|{f['year']}|無合格訓練設定|—|0|—|—|—|")
ps=pipeline_stats_v10
lines+=['',f"流程配對 {ps['pairs']}，平均增量 {ps['mean_difference_pp']:+.3f}pp；24日區塊區間 {ps['block24_interval_pp']}pp；48日 {ps['block48_interval_pp']}pp。風險門檻 {ps['risk_pass']}，覆蓋門檻 {ps['coverage_pass']}，全部判準 {ps['supported']}。",'5000次重抽樣在264家族的極端尾端只有很少模擬點，區間精度有限，不宣稱精確涵蓋率。已看過的歷史仍有設計偏誤；訓練／測試持有期隔開不會抹掉先前看過結果的事實。','',
'## 勝率與期望值：四種策略完全相同日期','勝率為費後正報酬窗口比例；期望值為費後24日平均，等於勝率×平均獲利＋虧損率×平均虧損，零報酬另外列。不是勝過A比例、不是實際比賽獲勝機率。','',
'|策略|配對窗口|勝率|期望值／24日|平均獲利|平均虧損|','|---|---:|---:|---:|---:|---:|']
for name,m in bulk_main_metrics_v10.items():lines.append(f"|{name}|{m['n']}|{fmt(m['win_rate_pct'])}%|{fmt(m['expected_return_pct'])}%|{fmt(m['avg_win_pct'])}%|{fmt(m['avg_loss_pct'])}%|")
offsets=[r for r in nonoverlap_v10 if r['mean_difference_pp'] is not None]
lines+=['',f"不重疊敏感度：24種起始相位中，對A平均增量正向 {sum(r['mean_difference_pp']>0 for r in offsets)}/{len(offsets)}；每相位有效窗口 {min(r['n'] for r in offsets)}–{max(r['n'] for r in offsets)}。相位之間互相重疊，不是24個獨立實驗；未用此診斷再選參數。",'',
'## 機制對照：避免把選股或底座效果當作機制增量','以下是次要描述比較，不作新增顯著性宣稱。風險／換股緩衝皆對相同A訊號、N、配置的v9底座；台積電對同exp32但上限10%的版本。配對日期仍各自相同。','',
'|新機制|對照|配對|費後平均差pp|平均節省費用pp|','|---|---|---:|---:|---:|']
for key,m in mechanism_contrasts_v10.items():lines.append(f"|{key}|{m['control']}|{m['pairs']}|{fmt(m['mean_difference_pp'])}|{fmt(m['mean_fee_saving_pp'])}|")
lines+=['',
'## 產業與其他資料分支','歷史個股分類：舊唯讀DB查核只有2026-10-11現況快照，無有效起訖分類表。本輪兩個官方MOPS歷史月報端點不可取得；不能證明歷史當時的150檔分類。這不是說世上沒有其他檔案，而是本批已有可驗證來源不足。保留產業＋個股選股不可判定，不用現況分類回填。','v9產業指數逐年驗證直接引用：2022–2025四年中三年勝過大盤，但只有少數年、無費用、分類大／子類重疊，不是可交易股票組合。所有個股產業權重及產業上限分支均未假稱完成。','金融／營收稀疏覆蓋會使候選未達訓練80%覆蓋而不入選；不等於該訊號無效。缺原文、首次公開時間、12筆有償認購排除、現有股票池存活偏誤、官方Active Share歷史基準仍未解。LLM與正式帳戶／DPlan／演練按使用者指示延後。','',
'## 統一選擇理由',
('擴充逐年流程通過本次歷史門檻；僅列歷史支持候選，仍須新資料與完整官賽規則驗證。' if ps['supported'] else '擴充逐年選擇流程未通過全部門檻，不能改拿216組全期間最高報酬當勝者。沒有新增方法可據此升格為已證實最佳策略。'),
'保留舊A等權為控制；舊溫和非等權＋小額略過仍僅是暫定研究配置，沒有自動換成較集中／較少股票。需要後續資訊的分支保持明確缺口，本批不繼續追參數或爬到顯著為止。所有當下可行節點已保存，沒有分析仍在執行，排程保持PAUSED。','',
'## 成果與驗證',f'[全部216組、勝率期望值與配對箱形圖]({bulk_chart_v10})。原Notebook cells213–216保存定義、全部80新實驗、統一評估與本總結，另存驗證cell217。',
'新窗口：策略批次研究_全部新窗口_v10_2026-10-15.json；統一統計、逐年流程與不重疊敏感度：策略批次研究_統一統計與逐年驗證_v10_2026-10-15.json；來源缺口獨立保存。','所有點保留、固定座標、桃紅中位數與0%線，圖表腳本另作NodeVM模擬DOM檢查，不宣稱瀏覽器目視檢查。未Restart and Run All；原kernel狀態保留。',
'預先定義SHA256：'+bulk_sha_v10]
(recovery_root/bulk_report_v10).write_text('\n'.join(lines)+'\n',encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['bulk_v10'].update({'status':'all_feasible_completed_industry_mapping_unavailable','running_work':False,'completed_nodes':['reuse','multi_signal','short_reversal','risk_overlays','special_cap','entry_exit_buffer','stability','report'],'unavailable_nodes':{'industry_mapping':source_gate_v10},'new_saved_records':52000,'total_candidate_records_referenced':140400,'total_candidates':216,'report':bulk_report_v10,'chart':bulk_chart_v10,'notebook_cells':list(range(213,217)),'supported_historical':supported_historical_v10,'supported_pipeline':ps['supported'],'folds':bulk_folds_v10,'graph_points':len(bulk_points_v10)});st['status']='bulk_v10_feasible_completed_with_explicit_source_gap';st['current_node']=None;st['automation_status']='PAUSED';sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## v10整批結案\n80新候選、136舊候選、逐年流程、24/48區塊與24種不重疊相位已完成；216候選140400筆引用。來源門檻仍無個股歷史分類，產業選股不可判定，不視為成功完成。全部可行工作一次交付；詳見'+bulk_report_v10+'。排程PAUSED，無執行中工作；不依結果擴增參數。\n',encoding='utf-8')
print('BULK_V10_REPORT_SAVED','new52000,total140400','points',len(bulk_points_v10),'pipeline_supported',ps['supported'])
print('BULK_V10_MAIN_METRICS',json.dumps(bulk_main_metrics_v10,ensure_ascii=False))


In [ ]:
# Verify full batch artefacts and retained computational state, without reset or new trials
import ast,subprocess,nbformat
assert len(bulk_results_v10)==80 and all(len(rows)==650 for rows in bulk_results_v10.values())
assert len(all_results_v10)==216 and sum(len(rows) for rows in all_results_v10.values())==140400
for rows in bulk_results_v10.values():
    for row in rows:
        if not row['invalid'] and row['violations']==0:
            assert 20<=row['min_held']<=row['max_held']<=30
            assert row['min_cash']>=0 and row['max_cash']<.25
for fold in bulk_folds_v10:
    if fold.get('selected'):assert fold['train_last_end']<fold['test_first_start']
for metrics in all_metrics_v10.values():
    m=metrics['paired']
    if m['n']:
        assert m['wins']+m['losses']+m['flats']==m['n']
        assert np.isclose(m['expected_return_pct'],m['expectancy_decomposed_pct'],atol=1e-10)
assert supported_historical_v10==[] and not pipeline_stats_v10['supported']
wp=Path('C:/Users/weiha/Documents/Codex/2026-10-09/data-mining-final-project/work')
tree=ast.parse((wp/'verify_v5_deliverables.py').read_text(encoding='utf-8'))
qa=next(ast.literal_eval(node.value) for node in tree.body if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='qa' for t in node.targets))
season_count=sum(pt['start'] in ['2024-10-29','2025-10-28'] for pt in bulk_points_v10)
qa=qa.replace('circles(season)!==10','circles(season)!=='+str(season_count))
out=subprocess.run(['C:/Program Files/nodejs/node.exe','-e',qa,str(recovery_root/bulk_chart_v10)],capture_output=True,text=True,encoding='utf-8')
assert out.returncode==0,out.stderr[:600]
q=json.loads(out.stdout);assert q['full_points']==1260 and q['methods']==4
saved=nbformat.read(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
for i in range(213,217):
    assert saved.cells[i].outputs and not any(o.output_type=='error' or 'Execution failed:' in o.get('text','') for o in saved.cells[i].outputs)
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['bulk_v10']['notebook_cells']=list(range(213,218));st['bulk_v10']['verification']={'graph':'NodeVM DOM simulation, not browser visual QA',**q,'table_candidates':216,'new_records':52000,'total_records':140400,'metrics_decomposition':'all216checked','scalar_vs_simultaneous_bootstrap':'exact anchors24and48','notebook_cells':'213-216 saved outputs checked;217 verification output','kernel':'retained, no reset'};sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
rp=recovery_root/bulk_report_v10;rp.write_text(rp.read_text(encoding='utf-8')+'\n最終驗證：80新候選各650筆、216合計140400筆；有效結果持股20–30與現金限制通過，勝率分母／期望值拆解一致。4方法315個共同日期共1260點，同季4點；NodeVM驗證全部點、固定0%座標、中位數顏色與快捷鍵通過。216列統計與全部新窗口已保存。cells213–216已檢查成功輸出，cell217保存驗證，kernel未重啟。\n',encoding='utf-8')
print('BULK_V10_FINAL_VERIFIED',json.dumps(q),'all216 candidates,140400 records; complete feasible work; no task left executing')


In [ ]:
# v11 recover retained metadata and structured revenue schema only
import inspect,ast,json,datetime,collections
for name in ['all_results_v10','pure_revenue_records_v1','pure_revenue_evidence_v1','revenue_announcement_candidates_v1','remaining_full_schema_v1','fetch_full_pool_announcements_v1','full_pool_announcements_v5','raw_data']:
    obj=globals().get(name);print('META',name,type(obj).__name__,len(obj) if hasattr(obj,'__len__') else None)
print('REVENUE_RECORD_KEYS',list(pure_revenue_records_v1[0]) if pure_revenue_records_v1 else [])
print('REVENUE_EVIDENCE_KEYS',list(next(iter(pure_revenue_evidence_v1.values()))))
for rec in pure_revenue_records_v1[:2]:print('REVENUE_SAFE_SAMPLE',{k:rec.get(k) for k in rec if k not in ['content','subject','text','source','url']})
prior_schema=globals().get('remaining_full_schema_v1') or []
print('TABLE_NAMES',sorted({row['TABLE_NAME'] for row in prior_schema}))
print('PRICE_COLUMNS',[r for r in prior_schema if r['TABLE_NAME'] in ['prices','daily_prices','universe','stocks','monthly_revenue','revenues']])
print('ASOF_FUNCTIONS',[k for k,v in list(globals().items()) if callable(v) and ('revenue' in k.lower() or 'notice' in k.lower())][:25])


In [ ]:
# v11 authorized readonly source availability audit, no credentials printed
import nbformat,copy
source_book_v11=nbformat.read(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
fetch_node_v11=None
for cc in source_book_v11.cells:
    if cc.cell_type=='code' and 'def fetch_full_pool_announcements_v1(' in cc.source:
        parsed=ast.parse(cc.source);fetch_node_v11=next(n for n in parsed.body if isinstance(n,ast.FunctionDef) and n.name=='fetch_full_pool_announcements_v1');break
assert fetch_node_v11 is not None,'Original readonly transport definition not found'
tree=ast.Module(body=[copy.deepcopy(fetch_node_v11)],type_ignores=[]);tree.body[0].name='fetch_schema_v11'
body_try=next(node for node in tree.body[0].body if isinstance(node,ast.Try));body_with=next(node for node in body_try.body if isinstance(node,ast.With))
body_with.body=ast.parse("cur.execute('SET SESSION TRANSACTION READ ONLY')\ncur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')\ncur.execute('SELECT TABLE_NAME,COLUMN_NAME,DATA_TYPE FROM information_schema.columns WHERE table_schema=DATABASE() ORDER BY TABLE_NAME,ORDINAL_POSITION')\nrows=cur.fetchall()\nschema=[]").body
ast.fix_missing_locations(tree);exec(compile(tree,'<readonly-v11-schema>','exec'),globals())
try:
    schema_v11,_=fetch_schema_v11();schema_status_v11='verified_readonly_schema'
except Exception as exc:
    schema_v11=[];schema_status_v11='query_unavailable_'+type(exc).__name__
print('SCHEMA_STATUS',schema_status_v11,'columns',len(schema_v11))
print('TABLE_NAMES',sorted({r['TABLE_NAME'] for r in schema_v11}))
print('RELEVANT_COLUMNS',[r for r in schema_v11 if r['TABLE_NAME'] in ['prices','daily_prices','universe','stocks','monthly_revenue','revenues','price_daily','institutional_flows']])
# Revenue original records stay in kernel; check consecutive-month availability without outcomes.
rev_count_by_month_v11=collections.Counter(r['year_month'] for r in pure_revenue_records_v1)
print('REVENUE_MONTH_COVERAGE',dict(sorted(rev_count_by_month_v11.items())))
print('REVENUE_RECORD_STATUS',dict(collections.Counter(r.get('status') for r in pure_revenue_records_v1)))
print('REVENUE_ASOF','decision08:55;earliest verified bulletin for newest available reportingmonth;45day maxage;monthlag1or2')


In [ ]:
# Readonly aggregate history audit; no full data reload or invented vintage labels
queries_v11=[]
for table in ['daily_price','institutional','dividend','monthly_revenue']:
    cols=[row for row in schema_v11 if row['TABLE_NAME']==table];names=[row['COLUMN_NAME'] for row in cols]
    date_fields=[r['COLUMN_NAME'] for r in cols if r['DATA_TYPE']=='date']
    field=next((x for x in ['trade_date','date','ex_date','year_month'] if x in names),date_fields[0] if date_fields else None)
    if field and 'stock_id' in names:
        assert re.fullmatch('[A-Za-z0-9_]+',table) and re.fullmatch('[A-Za-z0-9_]+',field)
        queries_v11.append((table,f'SELECT MIN(p.`{field}`) AS first_date,MAX(p.`{field}`) AS last_date,COUNT(*) AS rows_count,COUNT(DISTINCT p.stock_id) AS stocks_count FROM `{table}` p JOIN universe u ON p.stock_id=u.stock_id AND u.active=1'))
tree=ast.Module(body=[copy.deepcopy(fetch_node_v11)],type_ignores=[]);tree.body[0].name='fetch_history_ranges_v11'
tt=next(node for node in tree.body[0].body if isinstance(node,ast.Try));ww=next(node for node in tt.body if isinstance(node,ast.With))
ww.body=ast.parse("cur.execute('SET SESSION TRANSACTION READ ONLY')\ncur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')\nrows=[]\nfor label,query in queries_v11:\n    cur.execute(query)\n    rows.append({'table':label,**cur.fetchone()})\ncur.execute('SELECT MIN(basis_date) AS first_basis,MAX(basis_date) AS last_basis,COUNT(*) AS stocks_count FROM universe WHERE active=1')\nschema=cur.fetchall()").body
ast.fix_missing_locations(tree);exec(compile(tree,'<readonly-v11-ranges>','exec'),globals())
try:history_ranges_v11,universe_vintage_v11=fetch_history_ranges_v11();history_query_status_v11='readonly_success'
except Exception as exc:history_ranges_v11=[];universe_vintage_v11=[];history_query_status_v11='unavailable_'+type(exc).__name__
print('HISTORY_RANGES',json.dumps(history_ranges_v11,default=str,ensure_ascii=False))
print('UNIVERSE_VINTAGE',json.dumps(universe_vintage_v11,default=str,ensure_ascii=False))
print('RETAINED_PRICE_WINDOW',str(dates[0]),str(dates[-1]),len(dates),'current contest stocks',len(stock_ids))


In [ ]:
# Freeze all five follow-up directions before outcome inspection
import hashlib
follow_definition_v11={
 'version':'v11','date':'2026-10-15','request':'complete all five proposed research branches continuously in current session, no intermediate approval waits',
 'reuse':'same retained kernel,v7 action/price engine,v9/10 216 candidates and source evidence; no LLM,DB writes or trades',
 'branches':[
 {'id':'industry_seasonal_stock','gate':'verified historical individual classifications','candidates_if_available':4,'definition':'pure prior-year seasonality rank or50pctA+50pctseasonality,25stocks,equal/exp8; no present-day label backfill','known_gap':'fresh schema has no classification vintage; old official historical endpoints unavailable; reuse previous attempts'},
 {'id':'revenue_acceleration','candidates':4,'definition':'previous consecutive reportingmonth, both records published before decision08:55; score .5percentile(deltaMoM)+.5percentile(deltaYoY); pure or50pctA;25stocks,equal/exp8; unknown excludes'},
 {'id':'announcement_price','candidates':4,'definition':'same A25; latest verified monthly revenue bothMoM>0 andYoY>0; first usable decision withinlast5sessions; preannouncement20session adjusted return<=0 underreaction or>0 runup; raw weight*1.25 for each separate gate;equal/exp8;unknown neutral;no first-release claim'},
 {'id':'announcement_event_study','definition':'all deduplicated stock-month earliest retained verified bulletin, availability mapped to first08:55 after timestamp; past20>=20knownreturns; future24 signalreturns only for evaluation; positive dualrate versus other, crossed prior20<=0/>0; all groups reported, not fee-adjusted portfolio'},
 {'id':'long_history_universe','definition':'readonly aggregate earliest/latest price/flow/action, universe basis_date; extend only if price+flow+actions+PIT universe and sources support; otherwise state limitation; no synthetic or reclassified history'},
 {'id':'selector_logic','procedures':3,'definition':'same candidate set all216+feasible8+oldAequal/oldmildtol controls; same purged2025/26 and train100pairs/80pctcoverage/P10-MDD<=1pp; mean objective, stable=max worst calendar-quarter mean(pairdiff) withquarter>=10pairs, lowcost=minmeanfees among trainingmean>=0 andriskpass;fallbackbaseline allowed; no adaptive test-year tuning'}],
 'constraints':'24tradingdays,every5days,90pct,1000sharelots,fees/tax unchanged,actual20..30,caps2330_25pct/other10pct;25only isolates new information experiment',
 'source_policy':'earliest retained verified bulletin is known available time, not certified original firstrelease; revisions and missing originals remain limitations',
 'comparison':'each informational overlay versus identical A25equal/exp8+.005skip; all new versus oldA; selectors on identical valid dates; stock-event returns cannot be called tradable expectancy',
 'reporting':'net positive-return win rate,mean24day portfolio return,conditionalmeanwin/loss,n,fees,mdd,paired beatbaseline; all point estimates label dependent samples',
 'max_new_candidates':12,'immediately_feasible_candidates':8,'family':279,'family_reason':'264 prior+12maximum conditional candidates+3selection procedures; newmechanism contrasts secondary descriptive',
 'statistics':'24/48positionblocks5000reps,family279;shallow extreme tails exploratory,year andriskgates;no pristineholdout',
 'stop':'all feasible branches saved and unavailable branches recorded; do not keep adding hypotheses until positive; no schedule activation needed for one continuous batch',
 'deferred':['LLM','formal account/DataPack/DPlan','operational rehearsal','new repository/project/site']}
follow_sha_v11=hashlib.sha256(json.dumps(follow_definition_v11,sort_keys=True).encode()).hexdigest()
(recovery_root/'後續研究_五方向預先定義_v11_2026-10-15.json').write_text(json.dumps(follow_definition_v11,ensure_ascii=False,indent=2),encoding='utf-8')
plan_v11="""# 五方向一次完成：v11計畫

所有可行節點同回合連續執行，不在一個cell或checkpoint停下請人檢查。沿用原kernel與快取，不開新任務、不重新抓成功來源、不自動啟動排程。

1. 產業季節性接個股：新唯讀schema查核分類歷史有效期；只有驗證歷史分類後才測純產業或50%動能＋50%產業各等權／分數配置共4候選。無歷史分類就保持未能判定，不用今日分類。既有產業指數逐年結果直接引用。
2. 營收改善速度：同股票連續月份、兩份皆已可取得的MoM／YoY差，百分位各半；純加速度或50%動能各等權／exp8共4候選。新資料不足25檔就排除窗口，不能當低分。同行比較同樣需歷史分類，無資料分支記錄。
3. 公告前後反應：先做全部股票月份的四組事件診斷；兩個獨立A25調權規則（雙率正且公告前20日未漲／已漲）各等權／exp8共4候選。只在可知公告後5個交易日內乘1.25，其餘中性。事件統計後24日只作結果，不能餵入決策；速報可知時間不是首次公開認證。
4. 更長歷史與股票池：唯讀查價格／法人／公司行動最早日期和現有名單basis_date；只有必要資料齊全才延長，否則保存不能延長原因。不能把2026名單當往年全市場名單。
5. 選策略方法：同一216＋可行新候選＋舊A控制，逐年比較訓練平均增量、最差季度增量優先、合格中低費用優先三個固定流程。2025／2026測試年鎖定，未過門檻可保留A。

比較同日期費後勝率、期望值、平均獲利／虧損、費用、回撤、資料覆蓋與區塊區間；家族279保守包含最多12候選與3流程。窗口重疊、公告事件同股重複、歷史已看過等限制保留。全部可行工作完成再一次交付，不追到顯著才停。LLM和正式操作仍延後。
"""
(recovery_root/'後續研究_五方向完整計畫_v11_2026-10-15.md').write_text(plan_v11,encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['followup_v11']={'status':'defined_running','sha256':follow_sha_v11,'branches':[n['id'] for n in follow_definition_v11['branches']],'max_new':12,'running_work':True};sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## 現行v11：五方向全部連續執行\n依使用者2026-10-15授權，後續研究_五方向完整計畫_v11_2026-10-15.md及同版預先定義為本批入口。資料不足分支明列不可判定，其他分支持續，不停在小步等檢查。8可行資訊候選、最多4條件產業候選、3個固定選擇流程；不自行重啟排程。\n',encoding='utf-8')
print('V11_PREREGISTERED',follow_sha_v11,'8feasible+4conditional;3selectors')


In [ ]:
# Numeric-only prior-month acceleration and publication-time-safe event features
import bisect
decision_clocks_v11=[datetime.datetime.combine(day,datetime.time(8,55)) for day in dates]
close_clocks_v11=[datetime.datetime.combine(day,datetime.time(13,30)) for day in dates]
rev_first_v11={}
for rec in pure_revenue_records_v1:
    key=(rec['stock_id'],rec['year_month'])
    if key not in rev_first_v11 or rec['published_at']<rev_first_v11[key]['published_at']:rev_first_v11[key]=rec
event_info_v11={};event_rows_v11=[];event_exclusions_v11=collections.Counter()
for key,rec in rev_first_v11.items():
    j=si.get(rec['stock_id']);published=rec['published_at'];entry=bisect.bisect_right(decision_clocks_v11,published);pre_end=bisect.bisect_left(close_clocks_v11,published)-1
    if j is None or entry>=len(dates) or pre_end<19:event_exclusions_v11['no_supported_pre_or_entry']+=1;continue
    prior=safe_r_v8[pre_end-19:pre_end+1,j]
    if not np.all(np.isfinite(prior)):event_exclusions_v11['invalid_pre20']+=1;continue
    before=float(np.prod(1+prior)-1);good=rec['mom']>0 and rec['yoy']>0
    info={'stock_id':rec['stock_id'],'year_month':rec['year_month'],'news_id':rec['news_id'],'published_at':str(published),'first_usable_t':entry,'first_usable_date':str(dates[entry]),'prior_last_close_date':str(dates[pre_end]),'prior20_return':before,'positive_both':bool(good),'underreact_proxy':bool(before<=0)}
    assert close_clocks_v11[pre_end]<published<decision_clocks_v11[entry]
    event_info_v11[key]=info
    if entry+24>len(dates):event_exclusions_v11['right_censored_next24']+=1;continue
    after=safe_r_v8[entry:entry+24,j]
    if not np.all(np.isfinite(after)):event_exclusions_v11['invalid_post24']+=1;continue
    event_rows_v11.append({**info,'post24_signal_return_pct':float((np.prod(1+after)-1)*100),'outcome_role':'evaluation only, adjusted signal-close returns, not traded PnL and not fee-adjusted'})
accel_scores_v11=np.full_like(scores_v7['A'],np.nan);delta_mom_v11=np.full_like(accel_scores_v11,np.nan);delta_yoy_v11=np.full_like(accel_scores_v11,np.nan)
under_gate_v11=np.zeros_like(accel_scores_v11,dtype=bool);runup_gate_v11=np.zeros_like(accel_scores_v11,dtype=bool);accel_coverage_v11=[];asof_checks_v11=0
for t in range(61,len(dates)):
    cutoff=decision_clocks_v11[t]
    for j,sid in enumerate(stock_ids):
        rec=pure_revenue_evidence_v1.get((t,sid),{})
        if rec.get('status')!='verified_original_known_available_time':continue
        year,month=map(int,rec['year_month'].split('-'));prev=f'{year-1}-12' if month==1 else f'{year}-{month-1:02d}'
        old=rev_first_v11.get((sid,prev))
        assert rec['published_at']<cutoff
        if old is not None and old['published_at']<cutoff:
            delta_mom_v11[t-1,j]=rec['mom']-old['mom'];delta_yoy_v11[t-1,j]=rec['yoy']-old['yoy'];asof_checks_v11+=1
        event=event_info_v11.get((sid,rec['year_month']))
        if event is not None and 0<=t-event['first_usable_t']<5 and event['positive_both']:
            under_gate_v11[t,j]=event['underreact_proxy'];runup_gate_v11[t,j]=not event['underreact_proxy']
    eligible=np.isfinite(scores_v7['A'][t-1])&np.isfinite(delta_mom_v11[t-1])&np.isfinite(delta_yoy_v11[t-1])
    accel_scores_v11[t-1]=.5*percentile(delta_mom_v11[t-1],eligible)+.5*percentile(delta_yoy_v11[t-1],eligible)
    accel_coverage_v11.append({'date':str(dates[t]),'stocks':int(eligible.sum())})
event_groups_v11={}
for good in [False,True]:
    for under in [False,True]:
        rows=[r for r in event_rows_v11 if r['positive_both']==good and r['underreact_proxy']==under];a=np.array([r['post24_signal_return_pct'] for r in rows])
        event_groups_v11[f"{'positive_dualrate' if good else 'other_dualrate'}|{'prior_nonpositive' if under else 'prior_positive'}"]={'n':len(rows),'stocks':len({r['stock_id'] for r in rows}),'years':sorted({r['first_usable_date'][:4] for r in rows}),'positive_return_rate_pct':float(np.mean(a>0)*100) if len(a) else None,'mean_signal_return_pct':float(a.mean()) if len(a) else None,'median_signal_return_pct':float(np.median(a)) if len(a) else None,'mean_positive_pct':float(a[a>0].mean()) if np.any(a>0) else None,'mean_negative_pct':float(a[a<0].mean()) if np.any(a<0) else None}
event_export_v11={'definition_sha256':follow_sha_v11,'groups':event_groups_v11,'events':event_rows_v11,'exclusions':dict(event_exclusions_v11),'acceleration_coverage':accel_coverage_v11,'acceleration_asof_checks':asof_checks_v11,'announcement_time_limit':'known verified bulletin, not certified first announcement; actual earlier release may already have moved price','industry_relative_acceleration':'not available without historical individual industry mapping'}
(recovery_root/'後續研究_營收加速度與公告事件_v11_2026-10-15.json').write_text(json.dumps(clean_v5(event_export_v11),ensure_ascii=False,indent=2),encoding='utf-8')
print('V11_ACCELERATION_COVERAGE','decisions_ge25',sum(r['stocks']>=25 for r in accel_coverage_v11),'/',len(accel_coverage_v11),'max',max(r['stocks'] for r in accel_coverage_v11),'consecutive_known_pairs',asof_checks_v11)
print('V11_EVENT_GROUPS',json.dumps(event_groups_v11,ensure_ascii=False))
follow_scores_v11={'ACCEL':accel_scores_v11,'A50_ACCEL50':.5*scores_v7['A']+.5*accel_scores_v11,'A':scores_v7['A']}
follow_specs_v11={}
for signal in ['ACCEL','A50_ACCEL50']:
    for alloc in ['equal','conviction']:follow_specs_v11[f'{signal}|N25|{alloc}']={'signal':signal,'overlay':None,'allocation':alloc,'count':25}
for gate in ['under','runup']:
    for alloc in ['equal','conviction']:follow_specs_v11[f'A_revenue_{gate}|N25|{alloc}']={'signal':'A','overlay':gate,'allocation':alloc,'count':25}
def allocation_follow_v11(t,selected,allocation):
    spec=current_follow_spec_v11;raw=np.ones(len(selected)) if allocation=='equal' else np.exp(8*(current_scores_v9[t-1,selected]-.5))
    if spec['overlay'] is not None:
        gate=under_gate_v11 if spec['overlay']=='under' else runup_gate_v11;raw=raw*np.where(gate[t,selected],1.25,1.)
    caps=caps_v9[selected];out=np.zeros(len(selected));active=np.ones(len(selected),bool);remaining=.9
    for _ in range(len(selected)+1):
        w=raw[active]/raw[active].sum()*remaining;hit=w>caps[active]+1e-12;ix=np.flatnonzero(active)
        if not hit.any():out[ix]=w;break
        out[ix[hit]]=caps[ix[hit]];remaining=.9-out.sum();active[ix[hit]]=False
    assert np.isclose(out.sum(),.9) and np.all(out>0) and np.all(out<=caps+1e-10)
    return out
follow_source_v11=v9_source.replace('def simulate_v9(','def simulate_follow_v11(').replace('allocation_v9(t,selected,allocation)','allocation_follow_v11(t,selected,allocation)');exec(follow_source_v11,globals())
current_follow_spec_v11={'signal':'A','overlay':None};current_scores_v9=scores_v7['A']
for i in [0,200,400,649]:
    new=simulate_follow_v11(int(window_starts[i]),'equal',25);old=redesign_results_v9['A|N25|equal'][i]
    assert new['invalid']==old['invalid'] and new['violations']==old['violations']
    for f in ['return','mdd','cost','turnover']:assert np.isclose(new[f],old[f],rtol=1e-12,atol=1e-12)
follow_results_v11={}
for name,spec in follow_specs_v11.items():
    current_follow_spec_v11=spec;current_scores_v9=follow_scores_v11[spec['signal']]
    follow_results_v11[name]=[simulate_follow_v11(int(start),spec['allocation'],25) for start in window_starts]
    (recovery_root/'後續研究_全部新窗口_v11_2026-10-15.json').write_text(json.dumps({'definition_sha256':follow_sha_v11,'specs':follow_specs_v11,'results':clean_v5(follow_results_v11)},ensure_ascii=False,allow_nan=False),encoding='utf-8')
    print('V11_NEW_CANDIDATE',name,'valid',sum(not r['invalid'] and r['violations']==0 for r in follow_results_v11[name]),flush=True)
print('V11_ALL8_NEW_CANDIDATES_SAVED',5200)


In [ ]:
# Fixed selector comparisons, same pool and same training gates; no new simulation
follow_metrics_v11={};follow_diffs_v11=[];follow_names_v11=list(follow_results_v11)
for key,rows in follow_results_v11.items():
    mask=valid_mask_v9(rows)&ref_valid_v9;ix=np.flatnonzero(mask);diff=np.where(mask,np.array([r['return'] for r in rows])-ref_return_v9,np.nan);follow_diffs_v11.append(diff)
    m={'full_valid':window_metrics_v9(rows),'paired':window_metrics_v9(rows,mask),'reference':window_metrics_v9(reference_v9,mask),'invalid_data':sum(row['invalid'] for row in rows),'violation_windows':sum(row['violations']>0 for row in rows),'years':{},'halves':{}}
    m['mean_difference_pp']=float(np.nanmean(diff)*100) if len(ix) else None;m['beat_A_pct']=float(np.mean(diff[mask]>0)*100) if len(ix) else None
    m['cost_pct']=float(np.mean([rows[i]['cost'] for i in ix])*100) if len(ix) else None
    m['p10_difference_pp']=float((np.quantile([rows[i]['return'] for i in ix],.1)-np.quantile(ref_return_v9[mask],.1))*100) if len(ix) else None
    m['median_mdd_difference_pp']=float((np.median([rows[i]['mdd'] for i in ix])-np.median(ref_mdd_v9[mask]))*100) if len(ix) else None
    m['risk_pass']=bool(len(ix) and m['p10_difference_pp']>=-1 and m['median_mdd_difference_pp']>=-1)
    for year in [2024,2025,2026]:
        keep=mask&np.array([dates[int(s)].year==year for s in window_starts]);m['years'][str(year)]={'n':int(keep.sum()),'mean_difference_pp':float(np.nanmean(diff[keep])*100) if keep.any() else None}
    for label,part in [('early',np.arange(len(mask))<len(mask)//2),('late',np.arange(len(mask))>=len(mask)//2)]:
        keep=mask&part;m['halves'][label]={'n':int(keep.sum()),'mean_difference_pp':float(np.nanmean(diff[keep])*100) if keep.any() else None}
    spec=follow_specs_v11[key];control=f"A|N25|{spec['allocation']}";base=redesign_results_v9[control];same=valid_mask_v9(rows)&valid_mask_v9(base);xx=np.flatnonzero(same)
    m['same_allocation_control']={'control':control,'n':len(xx),'mean_difference_pp':float(np.mean([rows[i]['return']-base[i]['return'] for i in xx])*100) if len(xx) else None,'type':'secondary descriptive, not additional significance declaration'}
    follow_metrics_v11[key]=m
mat=np.asarray(follow_diffs_v11)
for length in [24,48]:
    intervals=simultaneous_blocks_v10(mat,length,2791000+length,family=279)
    for j,key in enumerate(follow_names_v11):follow_metrics_v11[key][f'block{length}_interval_pp']=intervals[j].tolist()
for key,m in follow_metrics_v11.items():
    m['supported_historical']=bool(m['paired']['n']>=100 and m['risk_pass'] and all(m[k][0]>0 for k in ['block24_interval_pp','block48_interval_pp']) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in m['years'].values()) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in m['halves'].values()))
    print('V11_NEW_STATS',key,'pairs',m['paired']['n'],'meanpp',m['mean_difference_pp'],'supported',m['supported_historical'],flush=True)
selector_pool_v11={**all_results_v10,**follow_results_v11,'BASE_A_equal':reference_v9,'BASE_A_mildtol':engine_v7_results['A_mild_tolerance']}
selector_rows_v11={name:[dict(r,invalid=True) for r in reference_v9] for name in ['mean','stable_quarter','low_cost']};selector_folds_v11=[]
for year in [2025,2026]:
    cutoff=datetime.date(year,1,1);train=np.array([dates[int(s)+23]<cutoff for s in window_starts]);test=np.array([dates[int(s)].year==year for s in window_starts]);base_count=int((train&ref_valid_v9).sum());eligible=[];reject=collections.Counter()
    quarters=[(dates[int(s)].year,(dates[int(s)].month-1)//3+1) for s in window_starts]
    for key,rows in selector_pool_v11.items():
        keep=train&ref_valid_v9&valid_mask_v9(rows);ix=np.flatnonzero(keep)
        if len(ix)<100 or len(ix)<.8*base_count:reject['coverage']+=1;continue
        rr=np.array([r['return'] for r in rows]);mm=np.array([r['mdd'] for r in rows]);diff=rr-ref_return_v9
        p10=(np.quantile(rr[keep],.1)-np.quantile(ref_return_v9[keep],.1))*100;md=(np.median(mm[keep])-np.median(ref_mdd_v9[keep]))*100
        if p10 < -1 or md < -1:reject['risk']+=1;continue
        qmeans=[]
        for quarter in sorted({quarters[i] for i in ix}):
            part=[i for i in ix if quarters[i]==quarter]
            if len(part)>=10:qmeans.append(float(np.mean(diff[part])))
        eligible.append({'key':key,'mean':float(np.mean(diff[keep])),'worst_quarter':min(qmeans) if qmeans else float('-inf'),'quarters':len(qmeans),'fee':float(np.mean([rows[i]['cost'] for i in ix])),'turnover':float(np.mean([rows[i]['turnover'] for i in ix])),'n':len(ix)})
    assert eligible,'Baseline must pass identical training gate'
    for mode in selector_rows_v11:
        if mode=='mean':ranked=sorted(eligible,key=lambda r:(-r['mean'],r['turnover'],r['key']))
        elif mode=='stable_quarter':ranked=sorted(eligible,key=lambda r:(-r['worst_quarter'],-r['mean'],r['turnover'],r['key']))
        else:ranked=sorted([r for r in eligible if r['mean']>=-1e-14],key=lambda r:(r['fee'],-r['mean'],r['turnover'],r['key']))
        chosen=ranked[0];rows=selector_pool_v11[chosen['key']]
        for i in np.flatnonzero(test):selector_rows_v11[mode][i]=rows[i]
        mask=test&ref_valid_v9&valid_mask_v9(rows);ix=np.flatnonzero(mask)
        f={'year':year,'procedure':mode,'selected':chosen['key'],'training':chosen,'eligible_count':len(eligible),'rejected':dict(reject),'test_base_valid':int((test&ref_valid_v9).sum()),'test_pairs':len(ix),'test_metrics':window_metrics_v9(rows,mask),'test_mean_difference_pp':float(np.mean([rows[i]['return']-reference_v9[i]['return'] for i in ix])*100) if len(ix) else None,'train_last_end':str(max(dates[int(window_starts[i])+23] for i in np.flatnonzero(train))),'test_first_start':str(min(dates[int(window_starts[i])] for i in np.flatnonzero(test)))}
        assert f['train_last_end']<f['test_first_start'];selector_folds_v11.append(f)
        print('V11_SELECTOR_FOLD',mode,year,chosen['key'],'testmeanpp',f['test_mean_difference_pp'],flush=True)
selector_stats_v11={}
for mode,rows in selector_rows_v11.items():
    s=summary_v7(rows,reference_v9,family=279);s['years']={k:v for k,v in s['years'].items() if k in ['2025','2026']};mask=valid_mask_v9(rows)&ref_valid_v9;ii=np.flatnonzero(mask);half=len(ii)//2
    s['halves']={k:{'n':len(ix),'mean_difference_pp':float(np.mean([rows[i]['return']-reference_v9[i]['return'] for i in ix])*100) if len(ix) else None} for k,ix in [('early',ii[:half]),('late',ii[half:])]}
    folds=[r for r in selector_folds_v11 if r['procedure']==mode];s['coverage_pass']=all(f['test_pairs']>=.8*f['test_base_valid'] for f in folds)
    s['win_expectancy']=window_metrics_v9(rows,mask);s['supported']=bool(s['coverage_pass'] and s['risk_pass'] and all(s[k][0]>0 for k in ['block24_interval_pp','block48_interval_pp']) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in s['years'].values()) and all(v['mean_difference_pp'] is not None and v['mean_difference_pp']>0 for v in s['halves'].values()))
    selector_stats_v11[mode]=s;print('V11_SELECTOR_STATS',mode,json.dumps(s,ensure_ascii=False),flush=True)
follow_source_limits_v11={'industry_stock':'fresh260column schema audit still no historicalclassification validfrom/to;4conditional candidates not executed','industry_relative_revenue':'unavailable same classification gate','history':history_ranges_v11,'universe_vintage':universe_vintage_v11,'history_status':'no earlier than2023-10-02 in teamDB;2026-10-13/14 updates are extra recent days, not extra regimes; frozen comparison snapshot kept','universe_status':'150active rows have nullbasis_date; cannot construct historical eligible universe','first_release':'bulletin timestamp is known available time, not verified first public release','LLM':'deferred; no model service used'}
follow_eval_v11={'definition_sha256':follow_sha_v11,'new_candidate_stats':follow_metrics_v11,'selector_pool_size':len(selector_pool_v11),'selector_folds':selector_folds_v11,'selector_stats':selector_stats_v11,'selector_rows':clean_v5(selector_rows_v11),'sources':follow_source_limits_v11,'supported_new':[k for k,m in follow_metrics_v11.items() if m['supported_historical']],'supported_selectors':[k for k,m in selector_stats_v11.items() if m['supported']],'limits':['same historical data already examined; not pristine holdout','279family5000reps extreme tail precision limited','financial/news missingness nonrandom; do not equate coverage failure with no factor effect','events are adjusted close signal returns, no fees, not portfolio expectancy','official ActiveShare and12paid-capital action gaps remain']}
(recovery_root/'後續研究_五方向統一結果_v11_2026-10-15.json').write_text(json.dumps(clean_v5(follow_eval_v11),ensure_ascii=False,indent=2,allow_nan=False,default=str),encoding='utf-8')
print('V11_UNIFIED_SAVED','new',follow_eval_v11['supported_new'],'selectors',follow_eval_v11['supported_selectors'])


In [ ]:
# v11 one completed batch delivery, with unavailable research explicitly separate
import html as html_module
follow_report_v11='後續研究_五方向總結_v11_2026-10-15.md'
follow_chart_v11='後續研究_五方向比較與箱形圖_v11_2026-10-15.html'
mode_labels_v11={'mean':'依平均報酬選擇','stable_quarter':'依最差季度選擇','low_cost':'合格中低費用優先'}
follow_plot_v11={'舊A等權':reference_v9,'舊A非等權＋略過':engine_v7_results['A_mild_tolerance'],**{mode_labels_v11[k]:v for k,v in selector_rows_v11.items()}}
follow_common_v11=np.logical_and.reduce([valid_mask_v9(rows) for rows in follow_plot_v11.values()]);follow_main_metrics_v11={label:window_metrics_v9(rows,follow_common_v11) for label,rows in follow_plot_v11.items()}
follow_points_v11=[]
for label,rows in follow_plot_v11.items():
    for i in np.flatnonzero(follow_common_v11):
        row=rows[i];follow_points_v11.append({'method':label,'start':str(dates[row['start']]),'end':str(dates[row['end']]),'return_pct':row['return']*100,'cost_pct':row['cost']*100,'mdd_pct':row['mdd']*100})
template=(recovery_root/'策略批次研究_全部比較與箱形圖_v10_2026-10-15.html').read_text(encoding='utf-8').split('</script>',1)[0]+'</script>'
template=re.sub(r'const data=.*?;const labels=.*?;const names=.*?;const lo=',lambda m:'const data='+json.dumps(follow_points_v11,ensure_ascii=False)+';const labels={};const names='+json.dumps(list(follow_plot_v11),ensure_ascii=False)+';const lo=',template,count=1,flags=re.S)
template=template.replace('x=190+235*i','x=140+185*i')
template=re.sub(r'<title>.*?</title>','<title>五方向研究 v11</title>',template,count=1);template=re.sub(r'<h1>.*?</h1>','<h1>五方向研究 v11：資訊增量與選擇方法</h1>',template,count=1)
intro='下圖保留五種配置共同有效測試日期的全部點。勝率為費後正報酬比例，期望值為費後24日平均；桃紅中位數、0%虛線與兩個快捷鍵固定座標。三種選擇程序只用測試年前已結束的持有期，測試年鎖定。資料是已看過的歷史，不能稱真正未見樣本外；產業個股及更長歷史仍缺資料。'
template=re.sub(r'<p>.*?</p>',lambda m:'<p>'+intro+'</p>',template,count=1,flags=re.S)
fmt=lambda v:'未知' if v is None else f'{v:.3f}'
extra=['<h2>費後勝率與期望值：完全相同日期</h2><table border="1" cellpadding="5"><tr><th>配置</th><th>n</th><th>勝率%</th><th>期望值%／24日</th><th>平均獲利%</th><th>平均虧損%</th></tr>']
for label,m in follow_main_metrics_v11.items():extra.append('<tr><td>'+html_module.escape(label)+'</td><td>'+str(m['n'])+'</td>'+''.join('<td>'+fmt(m[k])+'</td>' for k in ['win_rate_pct','expected_return_pct','avg_win_pct','avg_loss_pct'])+'</tr>')
extra+=['</table>','<details><summary>8個新資訊候選與對照（展開）</summary><p>每列與舊A使用同日期，但列間日期不同，不能按各列期望值排序。公告反應另對相同配置底座做描述比較，以分開小額略過與資訊本身效果。</p><table border="1" cellpadding="4"><tr><th>方法</th><th>全有效</th><th>對A配對n</th><th>勝率%</th><th>期望值%</th><th>平均獲利%</th><th>平均虧損%</th><th>費用%</th><th>對A差pp</th><th>對同配置底座差pp</th><th>24日區塊pp</th><th>48日區塊pp</th></tr>']
for key,m in follow_metrics_v11.items():
    vals=[str(m['full_valid']['n']),str(m['paired']['n'])]+[fmt(m['paired'][k]) for k in ['win_rate_pct','expected_return_pct','avg_win_pct','avg_loss_pct']]+[fmt(m['cost_pct']),fmt(m['mean_difference_pp']),fmt(m['same_allocation_control']['mean_difference_pp'])]+[str([round(x,3) for x in m[k]]) for k in ['block24_interval_pp','block48_interval_pp']]
    extra.append('<tr><td>'+html_module.escape(key)+'</td>'+''.join('<td>'+v+'</td>' for v in vals)+'</tr>')
extra+=['</table></details>','<details><summary>公告事件描述：不是費後投資組合（展開）</summary><p>以第一份保留且核對的速報時間為基準，未認證首次公告。股票月事件重複、未扣費、用調整後收盤訊號報酬，不能和上方策略期望值混排。</p><table border="1" cellpadding="5"><tr><th>事件組</th><th>n</th><th>正報酬比例%</th><th>24日訊號報酬均值%</th><th>中位數%</th></tr>']
for key,m in event_groups_v11.items():extra.append('<tr><td>'+html_module.escape(key)+'</td><td>'+str(m['n'])+'</td>'+''.join('<td>'+fmt(m[k])+'</td>' for k in ['positive_return_rate_pct','mean_signal_return_pct','median_signal_return_pct'])+'</tr>')
extra.append('</table></details>')
(recovery_root/follow_chart_v11).write_text(template+'\n'+''.join(extra),encoding='utf-8')
lines=['# 五方向研究 v11：一次連續執行總結','日期：2026-10-15。全部當下可行節點已完成；資料缺口沒有冒稱已回測。','',
'## 完成盤點','新增8候選×650窗口＝5,200筆；引用216舊候選，合計224個實驗候選、145,600筆窗口。另加舊等權A與溫和非等權控制，三種選擇程序使用完全相同的226設定池。沒有重跑舊模擬、重啟kernel、寫DB或送交易。','',
'|方向|實際完成|仍不能判定|','|---|---|---|',
'|產業季節性接個股|重新唯讀查260個欄位，確認無歷史分類有效期；沿用v9指數診斷|4個條件候選未跑，不能以今日分類代替|',
'|營收改善速度|連續月份MoM／YoY差排名，純加速度及50%動能各2配置，共4候選|同業相對改善需歷史分類，未做；首發時間未認證|',
'|公告前後反應|全部四組股票月事件＋4個A持股資訊調權候選|事件不是費後組合，不能當首次公告因果效果|',
'|更長歷史與股票池|團隊DB價格／法人／股利／營收日期範圍、名單日期查核|無更早價格／法人或歷史名單，未延長回測|',
'|如何選策略|平均增量、最差季度、合格低費用三程序，2025／2026向前驗證|歷史已看過，不能稱乾淨未見樣本外|','',
'## 一、來源與時間邊界','1,280份已核對營收速報，先按股票＋報告月份去重取最早已保留時間。加速度只比較連續報告月，兩份都必須早於當日08:55；當期沿用45日及落後1–2月限制。共有 '+str(asof_checks_v11)+' 個決策日股票對可計算，'+str(sum(r['stocks']>=25 for r in accel_coverage_v11))+'/673個決策日足25檔。未知排除，不能當利空。','公告前價格嚴格截至速報時間之前的最後一次13:30收盤；若速報在盤中，不能拿當日收盤當公告前價格。可用交易日取速報之後第一個08:55。後24日只作結果，不進決策。所有時間邊界均已assert檢查。速報可能晚於真實公告，因此事件只是相對保留速報的診斷，不能宣稱尚未反應的價格。','資訊候選固定A25是為隔離新訊號，不恢復所有策略必須25檔的限制。其他候選仍保留20–30檔。新候選同樣24日、每5日、90%、整千股與費稅，單股上限及小額略過不變。','',
'## 二、營收加速度與公告調權：費後對照','加速度＝MoM差百分位與YoY差百分位各半。公告調權：雙率都正、首次可用後5個交易日內，依公告前20日未漲或已漲分別乘1.25；其他維持原始權重，兩個規則分開測，未知中性。','',
'|候選|全有效|對A配對n|勝率|期望值／24日|對A平均差pp|對相同配置底座差pp|資料排除窗口|限制不符窗口|','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for key,m in follow_metrics_v11.items():lines.append(f"|{key}|{m['full_valid']['n']}|{m['paired']['n']}|{fmt(m['paired']['win_rate_pct'])}%|{fmt(m['paired']['expected_return_pct'])}%|{fmt(m['mean_difference_pp'])}|{fmt(m['same_allocation_control']['mean_difference_pp'])}|{m['invalid_data']}|{m['violation_windows']}|")
lines+=['','8個新候選均未通過家族279的24／48日區塊、年度與風險全部判準。純加速度集中版即使平均差正，只有30個與A配對，且大量限制不符，不能按最高平均差選它。不同列的配對日期不同，不可按期望值直接排序。所有區間、平均獲利／虧損及逐年結果存JSON與互動表。公告調權須看對同配置底座的增量，不能把小額略過或exp8配置的效果算給公告資訊。','',
'## 三、公告事件：只作描述，不當交易績效','',
'|速報雙率|公告前20日|事件數|股票數|後24日正報酬比例|後24日訊號報酬均值（未扣費）|','|---|---|---:|---:|---:|---:|']
for key,m in event_groups_v11.items():
    good,price=key.split('|');lines.append(f"|{'MoM與YoY皆正' if good=='positive_dualrate' else '其他'}|{'已上漲' if price=='prior_positive' else '未上漲'}|{m['n']}|{m['stocks']}|{fmt(m['positive_return_rate_pct'])}%|{fmt(m['mean_signal_return_pct'])}%|")
lines+=['','雙率正的事件平均較高，但未同時控制年份、產業、個股重複與事件選擇偏誤；不能歸因於營收公告。未漲組沒有呈現單純壓倒已漲組的關係，不支持直接採用「好消息且之前沒漲就必買」。非有效首次公布時間、未扣費及非實際成交是重要限制；不將此表與費後策略勝率／期望值混排。','',
'## 四、三種選策略程序','全部程序同226設定池、同訓練至少100配對及80%基準日期、P10及中位回撤惡化不逾1pp。只有測試年前已結束的持有期進訓練，測試年鎖定。平均程序最大化平均費後增量；穩定程序最大化最差日曆季度平均增量（季度至少10配對）；低費用程序在訓練平均不負的合格候選中最小化費用。各項排序與平手規則先固定，允許選回A。','',
'|程序|年|訓練選中的設定|測試配對|測試勝率|測試期望值／24日|對A平均差pp|','|---|---:|---|---:|---:|---:|---:|']
for f in selector_folds_v11:lines.append(f"|{mode_labels_v11[f['procedure']]}|{f['year']}|{f['selected']}|{f['test_pairs']}|{fmt(f['test_metrics']['win_rate_pct'])}%|{fmt(f['test_metrics']['expected_return_pct'])}%|{fmt(f['test_mean_difference_pp'])}|")
lines+=['','低費用程序兩年都選到A動能、30檔等權、前N+5名保留緩衝。2025／2026對A增量都小幅正、風險門檻通過，但區塊區間仍跨0，不能稱已證實改良，更不能推論30檔普遍最佳。穩定程序2026選回等權A，表示可接受不改動；以訓練最高平均為目標的流程仍在2026落後。','',
'|程序|與A配對n|平均差pp|24日區塊區間pp|48日區塊區間pp|風險門檻|全部判準|','|---|---:|---:|---|---|---|---|']
for mode,s in selector_stats_v11.items():lines.append(f"|{mode_labels_v11[mode]}|{s['pairs']}|{s['mean_difference_pp']:+.3f}|{[round(x,3) for x in s['block24_interval_pp']]}|{[round(x,3) for x in s['block48_interval_pp']]}|{s['risk_pass']}|{s['supported']}|")
lines+=['','## 五、完全相同日期的費後勝率與期望值','為公平比較，下表只用五種配置共同有效日期；與上表各程序各自配對日期數不同。勝率為費後正報酬比例，零報酬另列；期望值為樣本費後24日平均＝勝率×平均獲利＋虧損率×平均虧損。','',
'|配置|n|勝率|期望值／24日|平均獲利|平均虧損|','|---|---:|---:|---:|---:|---:|']
for label,m in follow_main_metrics_v11.items():lines.append(f"|{label}|{m['n']}|{fmt(m['win_rate_pct'])}%|{fmt(m['expected_return_pct'])}%|{fmt(m['avg_win_pct'])}%|{fmt(m['avg_loss_pct'])}%|")
lines+=['','## 六、歷史與未完成分支','團隊DB的價格及法人最早均2023-10-02，股利最早2023-10-16，營收2023-10；價格／法人最新2026-10-14，只比固定快照新增最近兩個交易日，並不是新增景氣循環，因此本批維持固定快照。150筆現行名單basis_date全部null，無法重建歷史可投股票池。','無法由現有資料完成更長年代或無股票池偏誤回測；不是宣稱外界不存在其他歷史來源。產業歷史分類與同業相對營收仍缺資料，4個條件產業候選未執行。既有產業指數2016–2025觀察不替代個股歷史映射。已完成可行分支，對無資料分支保持不可判定，不造時間、產業標籤或價格。','',
'## 統一結論','本批8個資訊候選與3個選擇流程均未通過全部預先門檻。保留A等權為基準與既有溫和非等權＋小額略過為暫定配置。低費用＋換股緩衝可列方向一致的待驗證候選，不能升格成已證實最佳。沒有依結果追加參數、混合所有好看的策略或強選勝者。','家族279、各區塊5000次極端尾端精度有限，區間屬探索性；窗口重疊、事件重複、已看過歷史、股票池與公告覆蓋偏誤仍在。官方Active Share、12筆有償認購及首發認證仍限制實盤解讀。LLM、正式帳戶／DataPack／DPlan、操作演練繼續按使用者指示延後。','',
'## 成果與保存',f'[互動箱形圖、8新候選與事件表]({follow_chart_v11})。所有共同有效點保留，固定座標、桃紅中位數與0%線。','原Notebook cells218–224保存查核、定義、特徵、8候選、三程序及本報告，另存225驗證。新窗口5,200筆、五方向統一結果及事件明細分別保存JSON。分析全在既有kernel；沒有Restart and Run All，不宣稱清空狀態重跑驗證。',
'預先定義SHA256：'+follow_sha_v11]
(recovery_root/follow_report_v11).write_text('\n'.join(lines)+'\n',encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['followup_v11'].update({'status':'all_feasible_completed_with_industry_and_history_gaps','running_work':False,'new_saved_records':5200,'new_candidates':8,'conditional_candidates_not_run':4,'total_candidates_excluding_controls':224,'selector_pool':226,'supported_new':follow_eval_v11['supported_new'],'supported_selectors':follow_eval_v11['supported_selectors'],'report':follow_report_v11,'chart':follow_chart_v11,'source_limits':follow_source_limits_v11,'branches_status':{'industry_seasonal_stock':'unavailable_pointintime_classification','revenue_acceleration':'completed4candidates;industryrelativeunavailable','announcement_price':'completed4candidates+4eventgroups','long_history_universe':'readonlyaudit_completed_extension_unavailable','selector_logic':'completed3procedures'},'notebook_cells':list(range(218,225)),'graph_points':len(follow_points_v11),'graph_common_dates':int(follow_common_v11.sum())});st['status']='followup_v11_all_feasible_completed_with_explicit_data_gaps';st['current_node']=None;st['automation_status']='PAUSED';sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2,default=str),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## v11五方向一次交付\n8新資訊候選5200窗口、4事件組、3選擇程序與資料來源查核已完成；目前可行工作無待執行。歷史產業分類／同業營收與更長歷史股票池無資料，保留不可判定。8候選與3流程皆未通過全部判準，低費用流程僅方向一致待驗證。詳見'+follow_report_v11+'。排程保持PAUSED，不再追加調參直到顯著。\n',encoding='utf-8')
print('V11_FINISH_SAVED','points',len(follow_points_v11),'common_dates',int(follow_common_v11.sum()))
print('V11_MAIN_METRICS',json.dumps(follow_main_metrics_v11,ensure_ascii=False))
print('V11_IDENTICAL_CONTROL_INCREMENTS',json.dumps({k:v['same_allocation_control'] for k,v in follow_metrics_v11.items()},ensure_ascii=False))


In [ ]:
# End-to-end saved branch verification; retain kernel and do not create further experiments
import ast,subprocess,nbformat
assert len(follow_results_v11)==8 and all(len(rows)==650 for rows in follow_results_v11.values())
assert len(selector_pool_v11)==226 and len(selector_folds_v11)==6
assert follow_eval_v11['supported_new']==[] and follow_eval_v11['supported_selectors']==[]
for event in event_rows_v11:
    stamp=datetime.datetime.fromisoformat(event['published_at']);entry=event['first_usable_t'];last=datetime.datetime.combine(datetime.date.fromisoformat(event['prior_last_close_date']),datetime.time(13,30))
    assert last<stamp<decision_clocks_v11[entry]
for fold in selector_folds_v11:assert fold['train_last_end']<fold['test_first_start']
for rows in follow_results_v11.values():
    for row in rows:
        if not row['invalid'] and row['violations']==0:assert 20<=row['min_held']<=row['max_held']<=30 and row['min_cash']>=0 and row['max_cash']<.25
for m in follow_main_metrics_v11.values():
    assert m['n']==317 and m['wins']+m['losses']+m['flats']==317 and np.isclose(m['expected_return_pct'],m['expectancy_decomposed_pct'],atol=1e-10)
wp=Path('C:/Users/weiha/Documents/Codex/2026-10-09/data-mining-final-project/work');tree=ast.parse((wp/'verify_v5_deliverables.py').read_text(encoding='utf-8'))
qa=next(ast.literal_eval(n.value) for n in tree.body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='qa' for t in n.targets))
season=sum(pt['start'] in ['2024-10-29','2025-10-28'] for pt in follow_points_v11);qa=qa.replace('circles(season)!==10','circles(season)!=='+str(season))
out=subprocess.run(['C:/Program Files/nodejs/node.exe','-e',qa,str(recovery_root/follow_chart_v11)],capture_output=True,text=True,encoding='utf-8')
assert out.returncode==0,out.stderr[:600];qa_meta_v11=json.loads(out.stdout);assert qa_meta_v11['full_points']==1585 and qa_meta_v11['methods']==5
book=nbformat.read(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
for i in range(218,225):
    assert book.cells[i].outputs and not any(o.output_type=='error' or 'Execution failed:' in o.get('text','') for o in book.cells[i].outputs)
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['followup_v11']['notebook_cells']=list(range(218,226));st['followup_v11']['verification']={'type':'NodeVM simulated DOM; no browser visual QA',**qa_meta_v11,'new_window_records':5200,'asof_consecutive_month_checks':asof_checks_v11,'prior_publication_close_checks':len(event_rows_v11),'purged_annual_folds':6,'saved_cells':'218-224 no errors;225 verification output','DB':'fresh readonly schema260cols and aggregate ranges; no writes','kernel':'retained, no reset'};sp.write_text(json.dumps(clean_v5(st),ensure_ascii=False,indent=2),encoding='utf-8')
rp=recovery_root/follow_report_v11;rp.write_text(rp.read_text(encoding='utf-8')+'\n最終驗證：8候選各650窗口、226設定池與6個年度程序選擇完整；'+str(asof_checks_v11)+'個連續月可知檢查、'+str(len(event_rows_v11))+'事件公告前收盤邊界通過，所有訓練結束早於測試起點。五配置共同317日期共1585點、同季5點；NodeVM驗證所有點、固定0%座標、中位數顏色及快捷鍵。cells218–224成功輸出已保存，cell225保存驗證。無執行中研究，僅明列資料不足與先前延後項目。\n',encoding='utf-8')
print('V11_FINAL_VERIFIED',json.dumps(qa_meta_v11),'all feasible work saved; no active research job; kernel retained')


In [ ]:
# v12 external historical source gates; new sources, not repeated DB limitation
import requests,concurrent.futures,json,hashlib,datetime,collections,re,time
print('RETAINED_STATE',len(selector_pool_v11),'selector settings',len(stock_ids),'stocks')
source_specs_v12=[
 ('twse_industry2024','https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX',{'date':'20241028','type':'01','response':'json'}),
 ('twse_market2016','https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX',{'date':'20161025','type':'ALLBUT0999','response':'json'}),
 ('twse_before_reclassification','https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX',{'date':'20230630','type':'20','response':'json'}),
 ('twse_after_reclassification','https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX',{'date':'20230703','type':'35','response':'json'}),
 ('tpex_history_probe','https://www.tpex.org.tw/www/zh-tw/afterTrading/dailyQuotes',{'date':'2016/10/25','response':'json'})]
def external_probe_v12(spec):
    key,url,params=spec
    try:
        response=requests.get(url,params=params,timeout=20);meta={'url':response.url,'http_status':response.status_code,'body_sha256':hashlib.sha256(response.content).hexdigest(),'bytes':len(response.content)}
        try:data=response.json()
        except ValueError:return key,{**meta,'status':'non_json','content_type':response.headers.get('Content-Type','')},None
        tables=data.get('tables') or [{'fields':data.get('fields',[]),'data':data.get('data',[]),'title':data.get('title','')}]
        meta.update(status=data.get('stat',data.get('status')),date=data.get('date'),tables=[{'title':t.get('title'),'fields':t.get('fields'),'rows':len(t.get('data') or [])} for t in tables])
        return key,meta,data
    except Exception as exc:return key,{'url':url,'params':params,'status':'error_'+type(exc).__name__},None
external_raw_v12={};external_meta_v12={}
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    for key,meta,data in pool.map(external_probe_v12,source_specs_v12):
        external_meta_v12[key]=meta
        if data is not None:external_raw_v12[key]=data
        print('V12_SOURCE_PROBE',key,json.dumps(meta,ensure_ascii=False),flush=True)
(recovery_root/'外部歷史研究_來源探測_v12_2026-10-15.json').write_text(json.dumps({'metadata':external_meta_v12,'raw':external_raw_v12},ensure_ascii=False),encoding='utf-8')
print('SOURCE_FUNCTION_AVAILABLE',type(globals().get('sector_request_v1')).__name__)
print('UNIVERSE_SNAPSHOT_SCHEMA',list(raw_data),'stock_row_type',type(raw_data['stocks'][0]).__name__,'width',len(raw_data['stocks'][0]))


In [ ]:
# Verify historical classification against published 2023-07-03 reclassification
def quote_table_v12(data):
    tables=data.get('tables') or [{'fields':data.get('fields',[]),'data':data.get('data',[]),'title':data.get('title','')}]
    return next((t for t in tables if t.get('fields') and '證券代號' in t['fields'] and '收盤價' in t['fields']),None)
before_table=quote_table_v12(external_raw_v12['twse_before_reclassification']);after_table=quote_table_v12(external_raw_v12['twse_after_reclassification'])
before_codes={str(row[0]).strip() for row in before_table['data']};after_codes={str(row[0]).strip() for row in after_table['data']}
change_witnesses_v12=sorted(set(['6581','6641','6806','8341','8422','8438','8473','9930','9955'])&before_codes&after_codes)
classification_endpoint_valid_v12=len(change_witnesses_v12)>=5
print('HISTORICAL_CLASSIFICATION_BOUNDARY',classification_endpoint_valid_v12,'witnesses',change_witnesses_v12)
print('BOUNDARY_DETAILS',{code:{'in_before_other':code in before_codes,'in_after_green':code in after_codes} for code in ['6581','8341','8422','8438','9930','9955']})
source_more_v12=[
 ('twse_marketcap2016','https://www.twse.com.tw/rwd/zh/fund/MI_QFIIS',{'date':'20161025','selectType':'ALLBUT0999','response':'json'}),
 ('twse_actions2016','https://www.twse.com.tw/rwd/zh/exRight/TWT48U',{'startDate':'20161001','endDate':'20161130','response':'json'}),
 ('twse_actioncalc2016','https://www.twse.com.tw/rwd/zh/exRight/TWT49U',{'startDate':'20161001','endDate':'20161130','response':'json'}),
 ('tpex_history_legacy','https://www.tpex.org.tw/web/stock/aftertrading/daily_close_quotes/stk_quote_result.php',{'l':'zh-tw','o':'json','d':'105/10/25','se':'EW'}),
 ('tpex_history_current2024','https://www.tpex.org.tw/www/zh-tw/afterTrading/dailyQuotes',{'date':'2024/10/28','response':'json'})]
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    for key,meta,data in pool.map(external_probe_v12,source_more_v12):
        external_meta_v12[key]=meta
        if data is not None:external_raw_v12[key]=data
        print('V12_SOURCE_DETAIL',key,json.dumps(meta,ensure_ascii=False),flush=True)
(recovery_root/'外部歷史研究_來源探測_v12_2026-10-15.json').write_text(json.dumps({'metadata':external_meta_v12,'raw':external_raw_v12,'reclassification_witnesses':change_witnesses_v12},ensure_ascii=False),encoding='utf-8')


In [ ]:
# Correct archived monthly-report filenames; validate before/after actual reclassification
from bs4 import BeautifulSoup
archive_specs_v12=[('mops_before','https://mopsov.twse.com.tw/nas/t21/sii/t21sc03_112_5_0.html'),('mops_after','https://mopsov.twse.com.tw/nas/t21/sii/t21sc03_112_7_0.html'),('mops_twse2024','https://mopsov.twse.com.tw/nas/t21/sii/t21sc03_113_9_0.html'),('mops_tpex2024','https://mopsov.twse.com.tw/nas/t21/otc/t21sc03_113_9_0.html')]
def archive_fetch_v12(spec):
    key,url=spec
    try:
        res=requests.get(url,timeout=25);res.encoding='cp950';txt=res.text;soup=BeautifulSoup(txt,'html.parser');text=soup.get_text(' ',strip=True)
        parsed={}
        # The company tables have a preceding industry caption and stock-code cells.
        for table in soup.find_all('table'):
            rows=table.find_all('tr',recursive=False)
            if not rows:
                body=table.find('tbody',recursive=False);rows=body.find_all('tr',recursive=False) if body else []
            whole=table.get_text(' ',strip=True);match=re.search(r'產業別[：:]\s*([^\s]+)',whole)
            if not match:continue
            industry=match.group(1)
            for row in table.find_all('tr'):
                cells=[x.get_text(' ',strip=True) for x in row.find_all(['td','th'],recursive=False)]
                if cells and re.fullmatch(r'\d{4}',cells[0]) and len(cells)>=3:parsed[cells[0]]={'industry':industry,'name':cells[1]}
        meta={'url':url,'http_status':res.status_code,'body_sha256':hashlib.sha256(res.content).hexdigest(),'bytes':len(res.content),'title':soup.title.get_text(' ',strip=True) if soup.title else None,'companies':len(parsed),'publication_text':re.findall(r'出表日期.{0,35}',text)[:1],'known_migration_examples':{code:parsed.get(code) for code in ['6581','8422','8438','9930','9955']}}
        return key,meta,txt,parsed
    except Exception as exc:return key,{'url':url,'error':type(exc).__name__},None,{}
archive_raw_v12={};archive_meta_v12={};archive_class_v12={}
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    for key,meta,txt,parsed in pool.map(archive_fetch_v12,archive_specs_v12):
        archive_meta_v12[key]=meta;archive_class_v12[key]=parsed
        if txt is not None:archive_raw_v12[key]=txt
        print('V12_ARCHIVE',key,json.dumps(meta,ensure_ascii=False),flush=True)
news_spec=('twse_reclass_news','https://www.twse.com.tw/rwd/zh/news/newsList',{'startDate':'20240101','endDate':'20261015','query':'產業類別','response':'json'})
key,meta,data=external_probe_v12(news_spec);external_meta_v12[key]=meta
if data is not None:external_raw_v12[key]=data
print('V12_NEWS_INDEX',json.dumps(meta,ensure_ascii=False))
(recovery_root/'外部歷史研究_月報存檔探測_v12_2026-10-15.json').write_text(json.dumps({'metadata':archive_meta_v12,'classifications':archive_class_v12,'html':archive_raw_v12},ensure_ascii=False),encoding='utf-8')


In [ ]:
# Pre-register seasonal ten-year historical-universe study before returns are computed
news_doc=external_raw_v12.get('twse_reclass_news',{});news_tables=news_doc.get('tables') or [news_doc]
news_hits_v12=[]
for table in news_tables:
    for row in table.get('data',[]):
        text=' '.join(str(x) for x in row)
        if '產業類別' in text:news_hits_v12.append(row)
print('DATED_RECLASSIFICATION_NEWS',json.dumps(news_hits_v12,ensure_ascii=False))
print('OLD_CAPITAL_SAMPLE',[row for row in external_raw_v12['twse_marketcap2016'].get('data',[]) if str(row[0]).strip() in ['1101','2330','2317']])
print('ACTION_SAMPLE',external_raw_v12['twse_actions2016'].get('data',[])[:2])
cal_source_v12=json.loads((recovery_root/'產業競賽期間_官方指數來源_2026-10-14.json').read_text(encoding='utf-8'))
print('CALENDAR_FILE_KEYS',[(k,type(v).__name__,len(v) if hasattr(v,'__len__') else None) for k,v in cal_source_v12.items()])
print('CALENDAR_GLOBALS',[name for name in globals() if any(x in name for x in ['calendar','month_cache','month_dates','all_days'])][:35])
external_definition_v12={'version':'v12','date':'2026-10-15','authorization':'user asks execute historical industry and longer-history/universe branches directly',
 'industry_gate':'MI_INDEX historical-date sector membership fails2023reclassification witness; regenerated MOPS monthly files show2026publication even for2023May and newlabels. Rejected, no current-label backfill. Continue dated news/immutable documents, do not bypass gate',
 'long_study':'2016..2025 lastTuesdayOctober24sessions;TWSE-only independent sub-study, not official150fullmarket strategy',
 'data':'officialdailyMI_INDEXquotes;MI_QFIISissuedshares for sameprior session;TWT48Ucash/free-stockandpaidcapital events plusTWT49Ureferencecheck;rawdailyreturns andfees model;no syntheticactionprices',
 'universes':['fixed current contest stocks currentlyTWSE listed, intersecthistorical trading universe','historicalTWSE ordinarycommon-stock topK marketcap(previousclose*then-issuedshares),K=numbercurrentcontestTWSEstocks;excludeETF/TDR/preferredbyformat and ISIN;publishedsameprior day'],
 'universe_limits':'historicaltopK proxy not reconstructionofpastcontest150;compositioneffect not pure survivorship effect; currentmarketclassifications not used',
 'signals':'20/60 price momentum percentile eachhalf,20day positivevolume and60valid adjustedrefreturns; noinstitutional filter because independent price-only variant, not exactoldA',
 'policies':['equal,no tolerance','equal,.005retained adjustment skip','mild rank1.25to.75,.005skip'],
 'constraints':'25held isolatesuniversechange;legal20..30daily,90pctinvested,rebalance5,24sessions,1000shares,fee.001425,taxsell.003,caps2330.25others.10,5daycapgrace;same receivables-notspendable accounting;ActiveShareuncertified',
 'actions':'free-stockrate percent/100,cashdivper-share;paidcapital and unexplainedreferencechanges invalid whenheld, excludeonlyfromeligibilityafterknownpastaction;never filter usingfutureactions',
 'source_bound':'tenyears,60past+24future daily marketrequests peryear from reused official calendar;max2IOworkers;each source up to3meaningful attempts;progress saved;no fixed timer deadline',
 'outputs':'6strategies*10seasonyears,eachpointinclinvalid saved;winrateandmean24dnetreturn conditionalgain/loss,n;paired sameyears and corporateaction/sourcecoverage',
 'interpretation':'10annualpoints not650independentwindows;no tuning bestperiod;ifaction/price gate fails keepyearinvalid;do not present price-only conditional study as full-market proven strategy',
 'deferred':['LLM','trades','DPlan','restartkernel','new project/repository','automations']}
external_definition_sha_v12=hashlib.sha256(json.dumps(external_definition_v12,sort_keys=True).encode()).hexdigest()
(recovery_root/'外部歷史研究_預先定義_v12_2026-10-15.json').write_text(json.dumps(external_definition_v12,ensure_ascii=False,indent=2),encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['external_history_v12']={'status':'sources_validated_fetching_and_classification_gate_rejected','sha256':external_definition_sha_v12,'running_work':True,'industry_source_problem':'historical query not historical classification','study':'TWSE-only10seasonyears;6fixedstrategycontrols'};sp.write_text(json.dumps(st,ensure_ascii=False,indent=2),encoding='utf-8')
print('V12_PREREGISTERED',external_definition_sha_v12)


In [ ]:
# Ten-year official quote/capital/action collection, bounded to competition seasons
import gzip,bisect,calendar
def roc_day_v12(value):
    nums=re.findall(r'\d+',str(value))
    if len(nums)<3:raise ValueError('Unparseable official date')
    y,m,d=map(int,nums[:3]);return datetime.date(y+1911 if y<1911 else y,m,d)
calendar_days_v12=sorted({roc_day_v12(row[0]) for doc in cal_source_v12['months'].values() if 'rows' in doc for row in doc['rows']})
long_cases_v12={}
for year in range(2016,2026):
    endmonth=datetime.date(year,10,31);anchor=endmonth-datetime.timedelta(days=(endmonth.weekday()-1)%7);ix=bisect.bisect_left(calendar_days_v12,anchor)
    span=calendar_days_v12[ix-61:ix+24];assert len(span)==85
    long_cases_v12[year]={'year':year,'anchor':str(anchor),'start':str(span[61]),'end':str(span[-1]),'previous':str(span[60]),'dates':span,'start_index':61}
external_definition_v12['data_gate_amendment']='TWT48U ignores historic range and returned2026forecast, rejected; use datedTWT49Ucalculation table. Pure cash rows kind息 have explicit權值+息值 cashamount; allrights/paidcapital/unexplained reference events invalid whenheld, not fabricated.'
external_definition_v12['actions']='cash-only TWT49U source; rights and unknown excluded whenheld and after known past event in lookback; no stock-rate inference from prices'
external_definition_sha_v12=hashlib.sha256(json.dumps(external_definition_v12,sort_keys=True).encode()).hexdigest()
(recovery_root/'外部歷史研究_預先定義_v12_2026-10-15.json').write_text(json.dumps(external_definition_v12,ensure_ascii=False,indent=2),encoding='utf-8')
long_raw_v12={'quotes':{},'capital':{},'actions':{},'errors':{},'source_metadata':{}}
long_cache_v12=recovery_root/'外部歷史研究_官方行情來源_v12_2026-10-15.json.gz'
if long_cache_v12.exists():
    with gzip.open(long_cache_v12,'rt',encoding='utf-8') as fh:long_raw_v12=json.load(fh)
long_raw_v12['quotes'].setdefault('20161025',external_raw_v12['twse_market2016'])
long_raw_v12['capital'].setdefault('20161025',external_raw_v12['twse_marketcap2016'])
def long_save_v12():
    with gzip.open(long_cache_v12,'wt',encoding='utf-8') as fh:json.dump(long_raw_v12,fh,ensure_ascii=False,separators=(',',':'))
def long_request_v12(item):
    kind,key,params=item;path={'quotes':'/rwd/zh/afterTrading/MI_INDEX','capital':'/rwd/zh/fund/MI_QFIIS','actions':'/rwd/zh/exRight/TWT49U'}[kind];errors=[]
    for attempt in range(3):
        try:
            res=requests.get('https://www.twse.com.tw'+path,params=params,timeout=20);res.raise_for_status();data=res.json()
            assert data.get('stat')=='OK'
            if kind!='actions':assert data.get('date')==params['date']
            if kind=='quotes':assert quote_table_v12(data) is not None
            if kind=='capital':assert '發行股數' in data.get('fields',[])
            if kind=='actions':assert '除權息計算結果表' in data.get('title','') and str(int(params['startDate'][:4])-1911) in data.get('title','')
            return kind,key,data,{'url':res.url,'sha256':hashlib.sha256(res.content).hexdigest(),'bytes':len(res.content),'attempts':attempt+1}
        except Exception as exc:
            errors.append(type(exc).__name__)
            if attempt<2:time.sleep(1+attempt)
    return kind,key,None,{'url':'https://www.twse.com.tw'+path,'params':params,'errors':errors}
pending_v12=[]
for day in sorted({day for case in long_cases_v12.values() for day in case['dates']}|{datetime.date(2026,10,12)}):
    key=day.strftime('%Y%m%d')
    if key not in long_raw_v12['quotes']:pending_v12.append(('quotes',key,{'date':key,'type':'ALLBUT0999','response':'json'}))
for year,case in long_cases_v12.items():
    key=case['previous'].replace('-','')
    if key not in long_raw_v12['capital']:pending_v12.append(('capital',key,{'date':key,'selectType':'ALLBUT0999','response':'json'}))
    if str(year) not in long_raw_v12['actions']:pending_v12.append(('actions',str(year),{'startDate':case['dates'][0].strftime('%Y%m%d'),'endDate':case['dates'][-1].strftime('%Y%m%d'),'response':'json'}))
print('V12_LONG_FETCH_PLAN',len(pending_v12),'requests,maxworkers2,10years85sessions;cache reused',flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
    for count,(kind,key,data,meta) in enumerate(pool.map(long_request_v12,pending_v12),1):
        long_raw_v12['source_metadata'][kind+':'+key]=meta
        if data is None:long_raw_v12['errors'][kind+':'+key]=meta
        else:long_raw_v12[kind][key]=data
        if count%40==0 or count==len(pending_v12):
            long_save_v12();print('V12_LONG_FETCH_PROGRESS',count,'/',len(pending_v12),'errors',len(long_raw_v12['errors']),flush=True)
long_save_v12()
print('V12_LONG_FETCH_COMPLETE',{k:len(v) for k,v in long_raw_v12.items()})


In [ ]:
# Confirmed security-denied source batch: cancel pending HTTP jobs, no kernel reset
pool.shutdown(wait=False,cancel_futures=True)
long_save_v12()
print('FAILED_FETCH_CANCELLED_KERNEL_RETAINED',{k:len(v) for k,v in long_raw_v12.items()})
external_source_failure_v12={'date':'2026-10-15','diagnostic_url':'https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX?date=20160930&type=ALLBUT0999&response=json','http_status':307,'message':'FOR SECURITY REASONS, THIS PAGE CAN NOT BE ACCESSED.','action':'cancel queued requests; preserve cache and kernel; no alternatehost/IP or other bypass; no further batch requests','failed_cell':230,'interruption':'only failed fetch cell interrupted, not kernel restart/shutdown; pending executor tasks cancelled','partial_counts':{k:len(v) for k,v in long_raw_v12.items()}}
(recovery_root/'外部歷史研究_來源停止原因_v12_2026-10-15.json').write_text(json.dumps(external_source_failure_v12,ensure_ascii=False,indent=2),encoding='utf-8')
print('SAFE_OBJECTS_PRESERVED',len(selector_pool_v11),len(follow_results_v11),len(stock_ids))
print('UNIVERSE_ROW_SCHEMA',raw_data['universe'][0])


In [ ]:
# Feasible source audits after stopping security-denied bulk fetch; no more TWSE API requests
import ast,copy,io
tree=ast.Module(body=[copy.deepcopy(fetch_node_v11)],type_ignores=[]);tree.body[0].name='fetch_current_market_labels_v12'
tt=next(n for n in tree.body[0].body if isinstance(n,ast.Try));ww=next(n for n in tt.body if isinstance(n,ast.With))
ww.body=ast.parse("cur.execute('SET SESSION TRANSACTION READ ONLY')\ncur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')\ncur.execute('SELECT s.stock_id,s.name,s.market,s.updated_at FROM stocks s JOIN universe u ON s.stock_id=u.stock_id AND u.active=1 ORDER BY s.stock_id')\nrows=cur.fetchall()\nschema=[]").body
ast.fix_missing_locations(tree);exec(compile(tree,'<readonly-currentmarkets-v12>','exec'),globals())
try:current_market_rows_v12,_=fetch_current_market_labels_v12();market_source_status_v12='readonly_currentlabels_success'
except Exception as exc:current_market_rows_v12=[];market_source_status_v12='unavailable_'+type(exc).__name__
print('CURRENT_MARKET_LABEL_COUNTS',dict(collections.Counter(r['market'] for r in current_market_rows_v12)))
def number_v12(value):
    s=re.sub('<[^>]*>','',str(value)).replace(',','').strip()
    try:return float(s)
    except ValueError:return np.nan
def ordinary_code_v12(code):return bool(re.fullmatch(r'[1-9]\d{3}',code)) and not code.startswith('91')
q2016=quote_table_v12(external_raw_v12['twse_market2016']);f=q2016['fields'];code_col=f.index('證券代號');close_col=f.index('收盤價')
prices2016_v12={str(row[code_col]).strip():number_v12(row[close_col]) for row in q2016['data'] if ordinary_code_v12(str(row[code_col]).strip())}
capital_doc=external_raw_v12['twse_marketcap2016'];fields=capital_doc['fields'];capital2016_v12={};last_report_date_violations_v12=[]
for row in capital_doc['data']:
    code=str(row[fields.index('證券代號')]).strip()
    if not ordinary_code_v12(code):continue
    shares=number_v12(row[fields.index('發行股數')]);price=prices2016_v12.get(code,np.nan)
    if np.isfinite(shares) and shares>0 and np.isfinite(price) and price>0:
        report=row[fields.index('最近一次上市公司申報外資及陸資持股異動日期')]
        if re.fullmatch(r'\d{2,3}/\d{2}/\d{2}',str(report)) and roc_day_v12(report)>datetime.date(2016,10,25):last_report_date_violations_v12.append(code)
        capital2016_v12[code]={'code':code,'name':row[fields.index('證券名稱')],'issued_shares':shares,'close':price,'market_cap':shares*price,'last_reported':str(report)}
assert not last_report_date_violations_v12
twse_labels={'TWSE','TSE','sii','上市','上市公司'}
fixed_twse_v12={r['stock_id'] for r in current_market_rows_v12 if str(r['market']) in twse_labels}
# If market label spelling differs, retain the source and do not invent the split.
if not fixed_twse_v12:print('MARKET_LABEL_REQUIRES_EXPLICIT_MAPPING')
ranked2016_v12=sorted(capital2016_v12.values(),key=lambda r:(-r['market_cap'],r['code']))
historical_top_v12=ranked2016_v12[:len(fixed_twse_v12)] if fixed_twse_v12 else []
top_codes={r['code'] for r in historical_top_v12}
snapshot_audit_v12={'asof':'2016-10-25 close, usable following session; not prior2016competitionstart','ordinary_stocks_with_close_and_shares':len(capital2016_v12),'current_contest_TWSE_count':len(fixed_twse_v12),'historical_topK_count':len(top_codes),'overlap_with_current_TWSE_contest':len(top_codes&fixed_twse_v12),'historical_topK_not_current_TWSE_contest':sorted(top_codes-fixed_twse_v12),'current_TWSE_contest_not_historical_topK':sorted(fixed_twse_v12-top_codes),'topK_records':historical_top_v12,'pointintime_share_date_checks':len(capital2016_v12),'future_lastreported_found':last_report_date_violations_v12,'not_claimed':'not officialpastcontest150, not all difference pure survivorship, currentlabels used only todefine presentTWSEsubset'}
archive_key,archive_meta,archive_text,archive_classes=archive_fetch_v12(('mops_preannouncement_april2023','https://mopsov.twse.com.tw/nas/t21/sii/t21sc03_112_4_0.html'))
archive_meta_v12[archive_key]=archive_meta;archive_class_v12[archive_key]=archive_classes
if archive_text is not None:archive_raw_v12[archive_key]=archive_text
april_new_labels=[code for code in ['6581','8422','8438','9930','9955'] if archive_classes.get(code,{}).get('industry')=='綠能環保']
classification_gate_evidence_v12={'MI_INDEX_effective_date_witness_pass':classification_endpoint_valid_v12,'MOPS_april2023_still_new_green_labels':april_new_labels,'april_report_publication_metadata':archive_meta.get('publication_text'),'official_change_published':'2023-05-22','official_change_effective':'2023-07-03','official_source':'https://www.twse.com.tw/rwd/staticFiles/product/publication/0005000075.pdf','decision':'reject regenerated monthly classifications as historical-vintage source; old reference months do not prove contemporaneous labels','monthly_body_sha256':archive_meta.get('body_sha256')}
# A fixed, dated official announcement is a different public document, not a retry of denied quote API.
pdf_meta_v12={};pdf_text_v12=''
pdf_url='https://www.twse.com.tw/staticFiles/news/news/tsecnews/8a8216d69dbea9fd019e0714a8d30169.pdf'
try:
    res=requests.get(pdf_url,timeout=20);pdf_meta_v12={'url':pdf_url,'http_status':res.status_code,'bytes':len(res.content),'sha256':hashlib.sha256(res.content).hexdigest()}
    if res.status_code==200 and res.content.startswith(b'%PDF'):
        from pypdf import PdfReader
        doc=PdfReader(io.BytesIO(res.content));pdf_text_v12='\n'.join(page.extract_text() or '' for page in doc.pages)
        (recovery_root/'官方文件'/'產業類別調整_2026-05-08_官方公告.pdf').write_bytes(res.content)
        pdf_meta_v12.update(pages=len(doc.pages),verified_2601_transition=all(s in pdf_text_v12 for s in ['2601','貿易百貨','航運']),relevant_text=' '.join(pdf_text_v12.split())[:1000])
except Exception as exc:pdf_meta_v12['error']=type(exc).__name__
source_completion_v12={'universe_snapshot':snapshot_audit_v12,'classification_gate':classification_gate_evidence_v12,'dated2026notice':pdf_meta_v12,'current_market_labels_source':market_source_status_v12,'fetch_stop':external_source_failure_v12,'full_study_status':'not executable: quote/security gate plus incomplete actions;6plannedmethods0newperformanceclaims'}
(recovery_root/'外部歷史研究_股票池與分類驗證_v12_2026-10-15.json').write_text(json.dumps(clean_v5(source_completion_v12),ensure_ascii=False,indent=2,default=str),encoding='utf-8')
(recovery_root/'外部歷史研究_月報存檔探測_v12_2026-10-15.json').write_text(json.dumps({'metadata':archive_meta_v12,'classifications':archive_class_v12,'html':archive_raw_v12},ensure_ascii=False),encoding='utf-8')
print('V12_HISTORICAL_UNIVERSE_SNAPSHOT',{k:v for k,v in snapshot_audit_v12.items() if k not in ['topK_records','historical_topK_not_current_TWSE_contest','current_TWSE_contest_not_historical_topK']})
print('V12_CLASSIFICATION_GATE',json.dumps(classification_gate_evidence_v12,ensure_ascii=False))
print('V12_DATED_NOTICE',json.dumps(pdf_meta_v12,ensure_ascii=False))


In [ ]:
# Preserve verified progress and precise external block; no incomplete performance claims
fixed_twse_v12={r['stock_id'] for r in current_market_rows_v12 if str(r['market']).lower() in {'twse','tse','sii','上市','上市公司'}}
assert len(fixed_twse_v12)==100
historical_top_v12=ranked2016_v12[:len(fixed_twse_v12)];top_codes={r['code'] for r in historical_top_v12}
snapshot_audit_v12.update(current_contest_TWSE_count=100,historical_topK_count=100,overlap_with_current_TWSE_contest=len(top_codes&fixed_twse_v12),historical_topK_not_current_TWSE_contest=sorted(top_codes-fixed_twse_v12),current_TWSE_contest_not_historical_topK=sorted(fixed_twse_v12-top_codes),topK_records=historical_top_v12)
pdf_meta_v12.update(is_pdf=False,usable_official_document=False,note='HTTP200 body752bytes was not a PDF; not accepted as dated official source')
case_coverage_v12=[]
for year,case in long_cases_v12.items():
    missing=[str(day) for day in case['dates'] if day.strftime('%Y%m%d') not in long_raw_v12['quotes']]
    case_coverage_v12.append({'year':year,'planned_sessions':85,'successful_quote_sessions':85-len(missing),'missing_quote_dates':missing,'capital_prior_session_available':case['previous'].replace('-','') in long_raw_v12['capital'],'actions_range_available':str(year) in long_raw_v12['actions'],'backtest_status':'not run because full source gates not met'})
source_completion_v12['universe_snapshot']=snapshot_audit_v12;source_completion_v12['dated2026notice']=pdf_meta_v12;source_completion_v12['case_coverage']=case_coverage_v12
source_completion_v12['fetch_stop']['partial_counts']={k:len(v) for k,v in long_raw_v12.items()}
source_completion_v12['industry_state']='source_validation_completed_strategy_still_unavailable';source_completion_v12['long_history_state']='2016officialuniverse_snapshot_rebuilt;10yearbulk_backtest_blocked_by_security_denial;0returnsinvented'
(recovery_root/'外部歷史研究_股票池與分類驗證_v12_2026-10-15.json').write_text(json.dumps(clean_v5(source_completion_v12),ensure_ascii=False,indent=2,default=str),encoding='utf-8')
v12_report_name='外部歷史研究_分類與股票池進度_v12_2026-10-15.md'
overlap=len(top_codes&fixed_twse_v12)
lines=['# 外部歷史來源：產業分類與歷史股票池 v12','2026-10-15。接續使用者授權的兩項分支，沒有只重查原DB，也沒有把來源取得當作回測完成。','',
'## 兩個選擇程序的精確意思','### 依最差季度選擇','只看訓練期，各候選與A在相同有效窗口的報酬差，按窗口起始日所屬日曆季度平均；季度至少10個有效配對。取各候選最差季度的平均差，選最差值較高者；平手按整體平均差、換手及固定代號排序。訓練至少100配對、80%覆蓋、P10與中位最大回撤惡化不超過1pp的門檻先通過。不是挑最賺的季度，也不是單看絕對損失最少。2026流程選回A，是因為當時這個保守排序沒挑出更好的改動。','例如（純示意，不是回測）：甲四季相對A為+2、+2、+2、−5pp，乙為+0.3、+0.2、−0.2、+0.1pp。最差季甲−5、乙−0.2，故此排序偏好乙。它不保證未來不虧，也不等於單一股票有季度季節性。',
'### 合格方案中低費用優先','同一訓練資料／同一候選池，先通過上面資料與風險門檻，再要求訓練期對A的平均費後增量不為負，最後選訓練窗口平均實際費稅最低者。平均增量、換手及代號作平手排序。不是先忽略報酬挑最便宜股票，也不是按股價或檔數多少選。沒有文字LLM判斷。','這次2025、2026都選到A動能30檔等權＋前N+5名保留緩衝。所謂便宜是模擬交易費稅較低，並非30檔必然比較省；同樣90%投入時，費用按交易金額與換手而非單純檔數計。兩年小幅正增量仍不足以通過區塊校正門檻。','',
'## 實際執行一：歷史分類來源驗證','新的公開來源已實際取得：指定日期MI_INDEX產業成員、MOPS歷史月份營收HTML、官方2023產業重分類公告與2024–2026新聞索引。','但資料品質門檻沒有通過：',
'- 官方2023-05-22公告，47家公司於2023-07-03調整產業。對照6家公司，查2023-06-30「其他」沒有把這些當時尚未生效移出的公司放回原類，7/3綠能資料卻有它們，未通過有效日期驗證。僅此不能斷言所有資料完全用今天分類，但不能把該篩選器當精確歷史分類。',
'- 再查2023年4月MOPS月報，仍把6581、8422、8438、9930、9955列綠能環保，而頁面出表日期為2026-10-13。4月營收報告參考月份早於5月22日公告，不代表抓到的分類也是當時版本。故拒絕用這種重新產製的歷史月份頁面作PIT分類。',
'- 2024年9月上市／上櫃月報也顯示2026重新出表。2026定期調整新聞僅找到1家，無法以「其他標題沒搜到」證明所有現況分類都可以回填2024／2025。固定公告PDF回應不是PDF，未當成功文件。',
'- 因此沒有啟動純產業或動能＋產業個股回測，不把來源缺漏當中性分數、不以現況標籤回填。既有官方產業指數觀察與此缺口是兩件事。',
'[官方2023年47家公司分類調整公告](https://www.twse.com.tw/rwd/staticFiles/product/publication/0005000075.pdf)。來源URL、SHA256、出表日期與反例另存JSON／原HTML。','',
'## 實際執行二：2016歷史行情與股票池','成功取得證交所2016-10-25全市場每日行情及同日外資持股統計中的發行股數。只用普通股代碼格式篩選，排除ETF、TDR、特別股等；收盤價×當時發行股數建立歷史市值，不使用2026股數。871檔有可配對的價格與股數；其中申報異動日期未發現晚於2016-10-25的資料。',
f'目前比賽150檔由100檔上市、50檔上櫃組成。以2016-10-25當時上市市值前100作研究代理，與目前比賽的上市100檔只重疊{overlap}檔。完整名單與市值均已保存。這顯示兩個股票池組成確實不同；差異同時包含市值排名、上市／下市／轉板及名單規則，不全是單一存活偏誤。',
'這是2016-10-25收盤快照，最早下一交易日可用；它不是當年比賽起點開盤前就可知的名單，也不是官方過去150檔比賽名單。已先登記各年比賽前一交易日股數與行情的取法，不以同日收盤回填早上決策。','',
'## 十年回測為何沒有硬算','原先已在結果產生前登記2016–2025各年十月最後星期二起24交易日、每年61日暖機＋24日、上市限定的現況／歷史前K兩股票池×等權、不等權等3配置，共6個固定方法。這是隔離股票池效應的價格動能子研究，未宣稱精確重現原A的法人資料資格條件，也不混入上櫃行情。',
'抓取在部分成功後連續回傳非JSON；獨立狀態診斷確認HTTP307及「FOR SECURITY REASONS, THIS PAGE CAN NOT BE ACCESSED.」。已取消待處理HTTP工作，不切換網域／IP或其他方式繞過。只中斷確認失敗的抓取cell，不是20分鐘到期，也沒有重啟、關閉或清除kernel。',
'已保存'+str(len(long_raw_v12['quotes']))+'份日行情、1份發行股數來源，以及失敗／缺漏清單。成功快取壓縮保存，之後若來源允許恢復可續用，不重抓。除權息預告TWT48U忽略歷史範圍返回2026資料，已拒絕；TWT49U可取得歷史計算表，但完整十年事件與行情尚未齊備，不由價格倒造股利／股票股利。','',
'|年度|規畫行情日數|成功取得|前一交易日股數|完整事件區間|績效|','|---|---:|---:|---|---|---|']
for row in case_coverage_v12:lines.append(f"|{row['year']}|85|{row['successful_quote_sessions']}|{row['capital_prior_session_available']}|{row['actions_range_available']}|未執行|")
lines+=['','## 勝率與期望值','本批是來源驗證與股票池重建，6個新回測方法尚未具備完整來源，因此勝率／期望值為「尚不能計算」，不是0。既有v11費後勝率與期望值不冒充本輪新結果。產業個股回測也保持未完成。','',
'## 保存與下一個可續跑條件','原Notebook cells226–233保存來源探測、分類反例、預先定義、失敗抓取與安全恢復、歷史股票池及本報告；234另存驗證。cell230的KeyboardInterrupt是有意停止確認失敗來源的紀錄，保留不清除。其他既有224候選與kernel記憶物件完整保留，沒有新增交易或排程。',
'原始快取：外部歷史研究_官方行情來源_v12_2026-10-15.json.gz；股票池與分類：外部歷史研究_股票池與分類驗證_v12_2026-10-15.json；停止原因、月報原HTML與來源探測另存同版JSON。','後續需可合法取得的當時分類存檔，以及可穩定存取、完成暖機及24日區間的價格／公司行動／前日股數。來源拒絕期間不自動反覆抓取、不宣稱已完成兩項策略，也不要求使用者逐小步核准。',
'預先定義SHA256：'+external_definition_sha_v12]
(recovery_root/v12_report_name).write_text('\n'.join(lines)+'\n',encoding='utf-8')
sp=recovery_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['external_history_v12'].update({'status':'external_source_security_denied_industry_gate_failed_snapshot_saved','running_work':False,'report':v12_report_name,'industry_status':'source-validation completed, historical individual strategy still unavailable','historical_snapshot_asof':'2016-10-25close','snapshot_ordinary_stocks':871,'snapshot_top100_overlap_current100':overlap,'quote_cache_count':len(long_raw_v12['quotes']),'planned_backtest_methods':6,'executed_backtest_methods':0,'source_stop':external_source_failure_v12,'notebook_cells':list(range(226,234)),'resume_gate':'do not retry security-denied batch; need authorized accessible historical source or genuine dated classification archive','kernel_reset':False});st['status']='v12_external_source_blocked_feasible_validation_saved';st['current_node']=None;st['automation_status']='PAUSED';sp.write_text(json.dumps(st,ensure_ascii=False,indent=2),encoding='utf-8')
wf=recovery_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## v12外部來源實作與停止條件\n依最新授權已進入DB以外來源，實際取得2016行情及股數，重建871檔市值與100檔代理股票池。歷史分類API／MOPS月報未通過版本門檻，不能用於回測。十年批次遇來源HTTP307安全拒絕已取消，不繞過、不再自動重試；保留快取與kernel，未執行6個不完整新策略。詳見'+v12_report_name+'。需來源恢復／合格歷史存檔才續跑，排程PAUSED。\n',encoding='utf-8')
old_report=recovery_root/'後續研究_五方向總結_v11_2026-10-15.md';old=old_report.read_text(encoding='utf-8');note='> 後續v12已取得DB以外2016官方行情與股數，重建歷史股票池快照；完整回測因來源安全拒絕未完成，產業分類來源未通過版本門檻。詳見 [v12來源實作]('+v12_report_name+')。\n\n';old_report.write_text(note+old,encoding='utf-8')
print('V12_FEASIBLE_RESULTS_SAVED','snapshot871,top100overlap',overlap,'quotes',len(long_raw_v12['quotes']),'newbacktests0,sourceblocked,kernelretained')


In [ ]:
# Legitimate independent provider probe, not an alternate route to denied TWSE API
provider_probe_v12=[];provider_raw_v12={}
for dataset in ['TaiwanStockPrice','TaiwanStockDividend']:
    params={'dataset':dataset,'data_id':'2330','start_date':'2016-07-01','end_date':'2016-11-30'}
    try:
        res=requests.get('https://api.finmindtrade.com/api/v4/data',params=params,timeout=20);doc=res.json();rows=doc.get('data') or []
        meta={'provider':'FinMind','dataset':dataset,'url':res.url,'http_status':res.status_code,'status':doc.get('status'),'message':str(doc.get('msg',''))[:250],'rows':len(rows),'fields':list(rows[0]) if rows else [],'sha256':hashlib.sha256(res.content).hexdigest(),'token_used':False}
        provider_probe_v12.append(meta)
        if doc.get('status')==200:provider_raw_v12[dataset]=rows
        print('V12_INDEPENDENT_PROVIDER',json.dumps(meta,ensure_ascii=False),flush=True)
        if res.status_code in [401,402,403] or doc.get('status') in [401,402,403]:break
    except Exception as exc:provider_probe_v12.append({'provider':'FinMind','dataset':dataset,'error':type(exc).__name__});print('V12_INDEPENDENT_PROVIDER_ERROR',type(exc).__name__)
(recovery_root/'外部歷史研究_獨立供應商探測_v12_2026-10-15.json').write_text(json.dumps({'documentation':'https://finmind.github.io/en/quickstart/','metadata':provider_probe_v12,'data':provider_raw_v12},ensure_ascii=False),encoding='utf-8')


In [ ]:
# Independent provider, public documented quota; never retry denied origin or bypass tiers
provider_definition_v12={'date':'2026-10-15','provider':'FinMind public unauthenticated API','docs':'https://finmind.github.io/en/quickstart/','quota':'300requests/hour withouttoken; planned<=281 including pilots, stop immediately on access/rate denial','source_amendment_reason':'official origin security-denied; independent provider succeeded on rawprice/dividend pilot; no returns used to change parameters','tickers':'union currentTWSE100 and2016historic top100 =138, no source IDs sent beyond public ticker codes','dates':'price2016-07-01..2025-12-31;dividend2015-01-01..2025-12-31;cash/source checking only','studies':'fixed currentTWSE100,2016..2025 October lastTuesday24days,3unchanged allocation rules;separate matched currentvsPIT top100 shifted2016-10-26 start because2016-10-25closecap notknownatOct25open','PIT_limit':'2017..2025annual historical cap universes still missing; do not call fixedcurrentpool survivorshipfree','actions':'cashper-shareexplicitpolicy;noncash/split/parvalue/reduction invalid ifheld, no stock-dividend-rate inference from prices;predecision eligibility excludes only known past unresolved actions','no_more_grid':True,'future_inference':'at most10annualpoints, pairedsameyears; onlyhistorical exploratory, no winner forced'}
provider_definition_sha_v12=hashlib.sha256(json.dumps(provider_definition_v12,sort_keys=True).encode()).hexdigest()
(recovery_root/'外部歷史研究_獨立供應商補充定義_v12_2026-10-15.json').write_text(json.dumps(provider_definition_v12,ensure_ascii=False,indent=2),encoding='utf-8')
provider_cache_v12=recovery_root/'外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz'
provider_history_v12={'prices':{},'dividends':{},'global_events':{},'metadata':{},'errors':{}}
if provider_cache_v12.exists():
    with gzip.open(provider_cache_v12,'rt',encoding='utf-8') as fh:provider_history_v12=json.load(fh)
provider_tickers_v12=sorted(fixed_twse_v12|top_codes);assert len(provider_tickers_v12)==138
provider_requests_v12=[('prices',sid,{'dataset':'TaiwanStockPrice','data_id':sid,'start_date':'2016-07-01','end_date':'2025-12-31'}) for sid in provider_tickers_v12 if sid not in provider_history_v12['prices']]
provider_requests_v12 += [('dividends',sid,{'dataset':'TaiwanStockDividend','data_id':sid,'start_date':'2015-01-01','end_date':'2025-12-31'}) for sid in provider_tickers_v12 if sid not in provider_history_v12['dividends']]
provider_requests_v12 += [('global_events',ds,{'dataset':ds,'start_date':'2015-01-01','end_date':'2025-12-31'}) for ds in ['TaiwanStockCapitalReductionReferencePrice','TaiwanStockSplitPrice','TaiwanStockParValueChange'] if ds not in provider_history_v12['global_events']]
assert len(provider_requests_v12)+len(provider_probe_v12)<=300
def provider_save_v12():
    with gzip.open(provider_cache_v12,'wt',encoding='utf-8') as fh:json.dump(provider_history_v12,fh,ensure_ascii=False,separators=(',',':'))
provider_stop_v12=__import__('threading').Event()
def provider_request_v12(item):
    kind,key,params=item
    if provider_stop_v12.is_set():return kind,key,None,{'status':'not_submitted_after_provider_stop'}
    time.sleep(.35)
    try:
        response=requests.get('https://api.finmindtrade.com/api/v4/data',params=params,timeout=30);doc=response.json();meta={'url':response.url,'http_status':response.status_code,'status':doc.get('status'),'message':str(doc.get('msg',''))[:150],'sha256':hashlib.sha256(response.content).hexdigest(),'bytes':len(response.content)}
        if response.status_code in [401,402,403,429] or doc.get('status') in [401,402,403,429]:provider_stop_v12.set();return kind,key,None,meta
        if response.status_code!=200 or doc.get('status')!=200:return kind,key,None,meta
        return kind,key,doc.get('data') or [],meta
    except Exception as exc:return kind,key,None,{'error':type(exc).__name__}
print('V12_PROVIDER_PLAN',len(provider_requests_v12),'requests;publicquota;stoponrefusal',flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as provider_pool_v12:
    for count,(kind,key,rows,meta) in enumerate(provider_pool_v12.map(provider_request_v12,provider_requests_v12),1):
        provider_history_v12['metadata'][kind+':'+key]=meta
        if rows is None:provider_history_v12['errors'][kind+':'+key]=meta
        else:provider_history_v12[kind][key]=rows
        if count%20==0 or count==len(provider_requests_v12):
            provider_save_v12();print('V12_PROVIDER_PROGRESS',count,'/',len(provider_requests_v12),'errors',len(provider_history_v12['errors']),'stop',provider_stop_v12.is_set(),flush=True)
provider_save_v12()
print('V12_PROVIDER_FETCH_DONE',{k:len(v) for k,v in provider_history_v12.items()})


In [ ]:
# Cross-source numerical validation, conservative cash-only action accounting, then fixed studies
provider_definition_v12['reference_tolerance']='max(.021NTD,.0001*previousclose); unexplainedreferencechanges flagged, never fabricate dividends'
provider_definition_sha_v12=hashlib.sha256(json.dumps(provider_definition_v12,sort_keys=True).encode()).hexdigest()
(recovery_root/'外部歷史研究_獨立供應商補充定義_v12_2026-10-15.json').write_text(json.dumps(provider_definition_v12,ensure_ascii=False,indent=2),encoding='utf-8')
price_lookup_v12={sid:{r['date']:r for r in rows} for sid,rows in provider_history_v12['prices'].items()}
cross_source_v12={'comparisons':0,'close_matches':0,'volume_matches':0,'mismatch_samples':[]}
for day,doc in long_raw_v12['quotes'].items():
    tab=quote_table_v12(doc);fields=tab['fields'];date=f'{day[:4]}-{day[4:6]}-{day[6:]}'
    for row in tab['data']:
        sid=str(row[fields.index('證券代號')]).strip();other=price_lookup_v12.get(sid,{}).get(date)
        if other is None:continue
        px=number_v12(row[fields.index('收盤價')]);vol=number_v12(row[fields.index('成交股數')])
        if not np.isfinite(px):continue
        cross_source_v12['comparisons']+=1;ok=abs(px-number_v12(other['close']))<=.011;cross_source_v12['close_matches']+=int(ok)
        cross_source_v12['volume_matches']+=int(abs(vol-number_v12(other['Trading_Volume']))<=1)
        if not ok and len(cross_source_v12['mismatch_samples'])<5:cross_source_v12['mismatch_samples'].append({'date':date,'sid':sid,'official':px,'provider':other['close']})
cross_source_v12['close_match_fraction']=cross_source_v12['close_matches']/cross_source_v12['comparisons'] if cross_source_v12['comparisons'] else 0
assert cross_source_v12['comparisons']>=100 and cross_source_v12['close_match_fraction']>=.995,'Provider raw-price validation failed; no backtest'
print('V12_PROVIDER_CROSS_CHECK',json.dumps(cross_source_v12),flush=True)
all_external_ids_v12=provider_tickers_v12;ext_si_v12={sid:i for i,sid in enumerate(all_external_ids_v12)}
cash_records_v12=collections.defaultdict(list);bad_action_dates_v12=set();cash_audit_v12=collections.Counter()
def valid_event_day_v12(value):
    try:
        day=datetime.date.fromisoformat(str(value)[:10])
        if day<calendar_days_v12[0] or day>calendar_days_v12[-1]:return None
        idx=bisect.bisect_left(calendar_days_v12,day)
        return calendar_days_v12[idx] if idx<len(calendar_days_v12) else None
    except (ValueError,TypeError):return None
for sid,rows in provider_history_v12['dividends'].items():
    for row in rows:
        cash_amt=sum(number_v12(row.get(k,0)) if np.isfinite(number_v12(row.get(k,0))) else 0 for k in ['CashEarningsDistribution','CashStatutorySurplus'])
        cashday=valid_event_day_v12(row.get('CashExDividendTradingDate'))
        if cash_amt>0:
            if cashday is not None:cash_records_v12[(cashday,sid)].append(float(cash_amt))
            else:cash_audit_v12['positive_cash_without_usable_exdate']+=1
        stock_amt=sum(number_v12(row.get(k,0)) if np.isfinite(number_v12(row.get(k,0))) else 0 for k in ['StockEarningsDistribution','StockStatutorySurplus'])
        stockday=valid_event_day_v12(row.get('StockExDividendTradingDate'))
        if stock_amt>0 and stockday is not None:bad_action_dates_v12.add((stockday,sid));cash_audit_v12['noncash_stock_events']+=1
cash_amount_v12={}
for key,amounts in cash_records_v12.items():
    if np.ptp(amounts)<=1e-6:cash_amount_v12[key]=amounts[0]
    else:bad_action_dates_v12.add(key);cash_audit_v12['ambiguous_same_day_cash_policy']+=1
for dataset,rows in provider_history_v12['global_events'].items():
    for row in rows:
        sid=str(row.get('stock_id',''));day=valid_event_day_v12(row.get('date'))
        if sid in ext_si_v12 and day is not None:bad_action_dates_v12.add((day,sid));cash_audit_v12['global_nonstandard_events']+=1
print('V12_CASH_SOURCE_AUDIT',len(cash_amount_v12),'knowncash events',dict(cash_audit_v12),flush=True)
case_arrays_v12={}
def build_case_v12(case):
    days=case['dates'];shape=(len(days),len(all_external_ids_v12));cl=np.full(shape,np.nan);vo=np.full(shape,np.nan);exe=np.full(shape,np.nan);spread=np.full(shape,np.nan);lo=np.full(shape,np.nan);hi=np.full(shape,np.nan);cashdiv=np.zeros(shape);bad=np.zeros(shape,bool)
    for t,day in enumerate(days):
        for j,sid in enumerate(all_external_ids_v12):
            row=price_lookup_v12.get(sid,{}).get(str(day))
            if row is not None:
                cl[t,j]=number_v12(row['close']);vo[t,j]=number_v12(row['Trading_Volume']);spread[t,j]=number_v12(row['spread']);lo[t,j]=number_v12(row['min']);hi[t,j]=number_v12(row['max'])
                if vo[t,j]>0:exe[t,j]=number_v12(row['Trading_money'])/vo[t,j]
            cashdiv[t,j]=cash_amount_v12.get((day,sid),0.);bad[t,j]=(day,sid) in bad_action_dates_v12
    returns=np.full(shape,np.nan);unknown_ref=[]
    for t in range(1,len(days)):
        ref=cl[t]-spread[t];expected=cl[t-1]-cashdiv[t]
        valid=np.isfinite(cl[t])&np.isfinite(cl[t-1])&(ref>0)&(cl[t-1]>0)
        mismatch=valid&(np.abs(ref-expected)>np.maximum(.021,.0001*cl[t-1]));bad[t]|=mismatch
        if mismatch.any():unknown_ref.extend({'date':str(days[t]),'sid':all_external_ids_v12[j],'previous':float(cl[t-1,j]),'reported_reference':float(ref[j]),'known_cash':float(cashdiv[t,j])} for j in np.flatnonzero(mismatch))
        returns[t,valid]=cl[t,valid]/ref[valid]-1
    available=np.isfinite(exe)&(exe>0)&np.isfinite(lo)&np.isfinite(hi)&(exe>=lo-.011)&(exe<=hi+.011)
    return {'dates':days,'close':cl,'volume':vo,'execution':exe,'cashdiv':cashdiv,'bad':bad,'returns':returns,'available':available,'unknown_reference_events':unknown_ref}
def simulate_external_v12(case,pool_ids,allocation,tolerance):
    arrays=case_arrays_v12[case['key']];cl=arrays['close'];cashdiv=arrays['cashdiv'];bad=arrays['bad'];r=arrays['returns'];vo=arrays['volume'];days=arrays['dates'];start=case['start_index'];shares=np.zeros(len(all_external_ids_v12),np.int64);cash=INITIAL;receivable=0.;values=[INITIAL];fees=0.;turnover_sum=0.;reasons=set();violations=0;cap_age=np.zeros(len(shares),int);minheld=999;maxheld=0;caps=np.array([.25 if sid=='2330' else .1 for sid in all_external_ids_v12]);traces=[]
    poolmask=np.array([sid in pool_ids for sid in all_external_ids_v12])
    for offset,t in enumerate(range(start,start+24)):
        before=shares.copy()
        if np.any(bad[t]&(before>0)):reasons.add('unmodeled_or_ambiguous_company_action_while_held')
        receivable+=float(np.dot(before,cashdiv[t]));prior_nav=cash+np.dot(shares,np.nan_to_num(cl[t-1]))
        if offset%5==0:
            rr=r[t-60:t];eligible=poolmask&np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(vo[t-20:t])&(vo[t-20:t]>0),axis=0)&~np.any(bad[t-60:t],axis=0)&np.isfinite(cl[t-1])&(cl[t-1]>0)
            if eligible.sum()<25:reasons.add('fewer_than25_past_data_eligible_stocks');break
            score=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,eligible)+.5*percentile(np.prod(1+rr,axis=0)-1,eligible)
            ordered=np.flatnonzero(eligible);selected=ordered[np.argsort(-score[ordered],kind='stable')[:25]]
            weights=np.full(25,.9/25) if allocation=='equal' else .9*np.linspace(1.25,.75,25)/25
            target=np.zeros_like(shares);target[selected]=(np.floor(weights*prior_nav/cl[t-1,selected]/1000)*1000).astype(np.int64)
            if offset>0 and tolerance:
                keep=(shares>0)&(target>0)&(np.abs(target-shares)*np.nan_to_num(cl[t-1])<=tolerance*prior_nav);target[keep]=shares[keep]
            cash,shares,traded,cost,missing=settle(cash,shares,target,arrays['execution'][t],arrays['available'][t]);fees+=cost;turnover_sum+=traded
            if missing:reasons.add('invalid_execution_price')
            traces.append({'date':str(days[t]),'selected':[all_external_ids_v12[j] for j in selected]})
        if np.any((shares>0)&~np.isfinite(cl[t])):reasons.add('missing_held_close')
        nav=cash+np.dot(shares,np.nan_to_num(cl[t]));aum=nav+receivable;values.append(aum)
        if nav<=0:reasons.add('nonpositive_tradable_nav');break
        weights=shares*np.nan_to_num(cl[t])/nav;cap_age=np.where(weights>caps+1e-12,cap_age+1,0);held=int(np.count_nonzero(shares));minheld=min(minheld,held);maxheld=max(maxheld,held)
        violations+=int(cash<0 or cash/nav>=.25 or not 20<=held<=30 or np.any(cap_age>5))
    invalid=bool(reasons or violations);vals=np.array(values);net=float(vals[-1]/INITIAL-1)
    return {'case':case['key'],'year':case['year'],'start':str(days[start]),'end':str(days[start+23]),'return':None if invalid else net,'mdd':None if invalid else float(np.min(vals/np.maximum.accumulate(vals)-1)),'cost':fees/INITIAL,'turnover':turnover_sum/INITIAL,'invalid':bool(reasons),'violations':violations,'reasons':sorted(reasons),'min_held':minheld,'max_held':maxheld,'trace':traces,'path':vals.tolist() if not invalid else None}
policies_v12={'等權完整調整':('equal',0.),'等權小額略過':('equal',.005),'溫和非等權略過':('mild',.005)}
external_fixed_results_v12={name:[] for name in policies_v12}
for year,basecase in long_cases_v12.items():
    case=dict(basecase,key=f'fixed_{year}');case_arrays_v12[case['key']]=build_case_v12(case)
    for name,(allocation,tolerance) in policies_v12.items():external_fixed_results_v12[name].append(simulate_external_v12(case,fixed_twse_v12,allocation,tolerance))
    print('V12_TEN_YEAR_CASE',year,{name:('invalid:'+','.join(rows[-1]['reasons']) if rows[-1]['invalid'] or rows[-1]['violations'] else round(rows[-1]['return']*100,3)) for name,rows in external_fixed_results_v12.items()},flush=True)
# The official2016 cap snapshot is only available after Oct25close: matched Oct26start only.
base=long_cases_v12[2016];start_ix=bisect.bisect_left(calendar_days_v12,datetime.date(2016,10,26));shift_days=calendar_days_v12[start_ix-61:start_ix+24]
shift_case_v12={'key':'snapshot2016_shifted','year':2016,'dates':shift_days,'start_index':61};case_arrays_v12[shift_case_v12['key']]=build_case_v12(shift_case_v12)
external_snapshot_results_v12={}
for pool_label,pool_ids in [('current100',fixed_twse_v12),('historical100',top_codes)]:
    missing_known_members=[sid for sid in pool_ids if sid in capital2016_v12 and '2016-10-25' not in price_lookup_v12.get(sid,{})]
    for name,(allocation,tolerance) in policies_v12.items():
        row=simulate_external_v12(shift_case_v12,pool_ids,allocation,tolerance)
        if missing_known_members:row.update(invalid=True,return_for_analysis=None);row['return']=None;row['mdd']=None;row['reasons'].append('known_historical_member_missing_provider_history')
        row['missing_known_members']=missing_known_members;external_snapshot_results_v12[pool_label+'|'+name]=row
print('V12_SNAPSHOT_MATCHED_RESULTS',{key:(row['return']*100 if row['return'] is not None else row['reasons']) for key,row in external_snapshot_results_v12.items()})
external_results_export_v12={'definition_sha256':provider_definition_sha_v12,'cross_source':cross_source_v12,'cash_audit':dict(cash_audit_v12),'known_cash_events':len(cash_amount_v12),'fixed_current_TWSE100':external_fixed_results_v12,'matched2016snapshot':external_snapshot_results_v12,'unknown_reference_events':{key:arrays['unknown_reference_events'] for key,arrays in case_arrays_v12.items()},'limits':['currentTWSE100 survivorshipandselectionbias remains','price-only momentum eligibility not old150A full model','cash-only model; stock/paidcapital unknown excluded on holding, not future selection','10annualseasonwindows not daily650trials','one2016snapshotcase is shifted1day, not actualcompetitionanchor','PITuniverse2017..2025not reconstructed','same simulated competition receivables convention,not ordinarycashsettlement']}
(recovery_root/'外部歷史研究_十年同期與股票池回測_v12_2026-10-15.json').write_text(json.dumps(external_results_export_v12,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('V12_BACKTESTS_SAVED','30annualcases+6matchedsnapshot;invalidkeptunknown')


In [ ]:
# Diagnose missing cases without changing selection or fetching again.
print('PROVIDER_ERRORS',provider_history_v12['errors'])
print('GLOBAL_SCHEMAS',{key:{'rows':len(rows),'first':rows[:1]} for key,rows in provider_history_v12['global_events'].items()})
failure_details_v12=[]
for name,rows in external_fixed_results_v12.items():
    if name!='等權完整調整':continue
    for result in rows:
        if result['return'] is not None:continue
        arrays=case_arrays_v12[result['case']];days=arrays['dates'];held=set()
        for trace in result['trace']:
            start=days.index(datetime.date.fromisoformat(trace['date']));held=set(trace['selected'])
            for t in range(start,min(start+5,len(days))):
                for sid in sorted(held):
                    j=external_id_index_v12[sid] if 'external_id_index_v12' in globals() else all_external_ids_v12.index(sid)
                    if arrays['bad'][t,j] or not np.isfinite(arrays['close'][t,j]) or not arrays['available'][t,j]:
                        failure_details_v12.append({'year':result['year'],'date':str(days[t]),'sid':sid,'bad_action':bool(arrays['bad'][t,j]),'close_present':bool(np.isfinite(arrays['close'][t,j])),'execution_available':bool(arrays['available'][t,j]),'price_record':price_lookup_v12.get(sid,{}).get(str(days[t])),'unknown_reference':[x for x in arrays['unknown_reference_events'] if x['sid']==sid and x['date']==str(days[t])]})
print('CASE_FAILURE_DETAILS',json.dumps(failure_details_v12,ensure_ascii=False,default=str))


In [ ]:
# Persist all annual points, paired exploratory statistics and explicit unresolved branches.
import html as html_v12
import plotly.graph_objects as go_v12
from plotly.offline import plot as plot_v12
def stats_external_v12(rows):
    valid=[r for r in rows if r['return'] is not None and not r['invalid'] and not r['violations']]
    x=np.array([r['return'] for r in valid]);cost=np.array([r['cost'] for r in valid]);draw=np.array([r['mdd'] for r in valid])
    return {'n':len(x),'years':[r['year'] for r in valid],'invalid_years':[r['year'] for r in rows if r not in valid],'win_rate':float(np.mean(x>0)) if len(x) else None,'flat_count':int(np.sum(x==0)),'expectancy':float(np.mean(x)) if len(x) else None,'median':float(np.median(x)) if len(x) else None,'average_win':float(np.mean(x[x>0])) if np.any(x>0) else None,'average_loss':float(np.mean(x[x<0])) if np.any(x<0) else None,'mean_cost':float(np.mean(cost)) if len(x) else None,'worst_return':float(np.min(x)) if len(x) else None,'median_mdd':float(np.median(draw)) if len(x) else None,'worst_mdd':float(np.min(draw)) if len(x) else None}
external_metrics_v12={name:stats_external_v12(rows) for name,rows in external_fixed_results_v12.items()}
base_external_v12={r['year']:r for r in external_fixed_results_v12['等權完整調整'] if r['return'] is not None}
external_pairs_v12={}
rng_external_v12=np.random.default_rng(20261015)
for name,rows in external_fixed_results_v12.items():
    if name=='等權完整調整':continue
    pairs=[(r,base_external_v12[r['year']]) for r in rows if r['return'] is not None and r['year'] in base_external_v12]
    d=np.array([a['return']-b['return'] for a,b in pairs]);boot=np.mean(d[rng_external_v12.integers(0,len(d),size=(20000,len(d)))],axis=1)
    external_pairs_v12[name]={'n':len(d),'years':[a['year'] for a,b in pairs],'mean_excess':float(np.mean(d)),'beat_base_rate':float(np.mean(d>0)),'mean_cost_change':float(np.mean([a['cost']-b['cost'] for a,b in pairs])),'paired_year_bootstrap_97_5pct':np.quantile(boot,[.0125,.9875]).tolist(),'note':'exploratory only; two comparisons Bonferroni marginal97.5% intervals; seven nonrandomly retained years, fixed current pool and reused development period prohibit confirmation','yearly_excess':[{'year':a['year'],'excess':float(a['return']-b['return'])} for a,b in pairs]}
provider_summary_v12={'metrics':external_metrics_v12,'paired_comparisons':external_pairs_v12,'failure_details':failure_details_v12,'selection_conclusion':'未證實改良；不由七個有效年度選出新贏家','industry_status':'blocked_no_verified_point_in_time_individual_classification','historical_pool_status':'partial_one2016TWSE100_snapshot_only','source_errors':provider_history_v12['errors']}
(recovery_root/'外部歷史研究_統計摘要_v12_2026-10-15.json').write_text(json.dumps(provider_summary_v12,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('V12_FINAL_METRICS',json.dumps(external_metrics_v12,ensure_ascii=False))
print('V12_PAIRED_EXCESS',json.dumps(external_pairs_v12,ensure_ascii=False))
def pct_external_v12(value):return '不可判定' if value is None else f'{value*100:.3f}%'
metric_header='|配置|有效／計畫年度|費後勝率|24日期望值|平均費稅／初始資金|最差報酬|最差最大回撤|\n|---|---:|---:|---:|---:|---:|---:|\n'
metric_md=metric_header+'\n'.join('|'+name+'|'+str(s['n'])+'/10|'+pct_external_v12(s['win_rate'])+'|'+pct_external_v12(s['expectancy'])+'|'+pct_external_v12(s['mean_cost'])+'|'+pct_external_v12(s['worst_return'])+'|'+pct_external_v12(s['worst_mdd'])+'|' for name,s in external_metrics_v12.items())
annual_md='|年度|起日～迄日|等權完整調整|等權小額略過|溫和非等權略過|\n|---|---|---:|---:|---:|\n'
for ix,year in enumerate(range(2016,2026)):
    first=next(iter(external_fixed_results_v12.values()))[ix]
    annual_md+='|'+str(year)+'|'+first['start']+'～'+first['end']+'|'+'|'.join(pct_external_v12(rows[ix]['return']) for rows in external_fixed_results_v12.values())+'|\n'
snapshot_md='|同一2016-10-26起24日配置|目前100檔名單|2016-10-25收盤市值前100|差值（歷史－目前）|\n|---|---:|---:|---:|\n'
for name in policies_v12:
    a=external_snapshot_results_v12['current100|'+name]['return'];b=external_snapshot_results_v12['historical100|'+name]['return']
    snapshot_md+='|'+name+'|'+pct_external_v12(a)+'|'+pct_external_v12(b)+'|'+pct_external_v12(None if a is None or b is None else b-a)+'|\n'
report_path_v12=recovery_root/'外部歷史研究_分類與股票池進度_v12_2026-10-15.md'
old_report_v12=report_path_v12.read_text(encoding='utf-8')
if '# 附錄：獨立供應商補充前的來源查核紀錄' in old_report_v12:old_report_v12=old_report_v12.split('# 附錄：獨立供應商補充前的來源查核紀錄',1)[1].lstrip()
report_new_v12='''# 外部歷史研究 v12：十年同期、股票池與產業分類
2026-10-15。已接續執行兩個資料分支；本摘要取代下方早期「尚未回測」的進度判斷。原始失敗與來源查核保留為附錄。

## 兩個排序的意思
「依最差季度選擇」：只用訓練期，先通過資料與風險門檻，再比較各方案最差日曆季度的平均『費後報酬減A』，偏好最差季度相對A較好的方案。不是保證不虧；2026的這個程序選回A。
「合格方案中低費用優先」：先通過相同門檻及訓練平均費後增量非負，再選模擬平均交易費稅最低的方案。費用取決於交易金額與換手，並非低股價。v11兩年都選A30等權＋N+5保留緩衝，但仍未證實改善。

## 本次實際取得與驗證
沿用官方快取，另以獨立FinMind公開API取得138檔普通股2016–2025價格與股利政策（目前上市100檔與2016歷史前100的聯集）。279筆請求中278筆成功；減資資料回覆升級權限，未購買或繞過。現金股利可查、非現金與未解參考價變動保留未知。與官方成功快取11,577筆價量交叉比對，收盤價及成交量全部相同。這驗證重疊樣本，不能證明所有其他日期完整無誤。
原TWSE批次遭安全拒絕後已停止，不以換網域／IP重試。FinMind是另行記錄的獨立資料供應商。

## 十年同期的費後結果
各年十月最後一個星期二起24交易日；25檔、90%投入、每5日調整、整千股，費率0.1425%、賣出稅0.3%。台積電上限25%、其他10%，沿用既有5日權重寬限檢查。用前20／60日價格動能排名；未含原A的法人資料資格，因此是A價格動能子研究，不可直接和舊150檔A混稱同一策略。
三個配置在產生結果前固定：等權完整調整；等權且同一持股目標變動≤NAV0.5%時略過；動能排名權重從1.25到0.75線性遞減且同樣略過。不以這次結果調參。
勝率＝費後24日報酬>0的有效年度比例；期望值＝有效年度費後報酬算術平均，含虧損，不年化。這裡是一年一個窗口，不是650個日滑動窗口。

'''+metric_md+'\n\n'+annual_md+'''
2017、2025持倉期間有未完整處理的公司行動；2018另有持倉缺報價／模擬VWAP超出日內高低範圍。三年都保留為不可判定，不以0補值、不先看未來事件剔除股票重選。因缺漏可能非隨機，7年勝率／期望值不能代表完整十年。
目前100檔是今天名單，因此十年回測仍有存活與選擇偏誤。7個年度、同季窗口及開發期重用，不足以把正增量當確認性證據。配對年度bootstrap區間只作探索性描述，已對2項比較採97.5%邊際區間；不是原650窗口的24／48日區塊驗證，也不能取代之。結論：未證實新配置改善，不強選贏家。

## 歷史股票池的同日起點比較
官方2016-10-25收盤價×當時股數，重建871檔普通股並取市值前100；與今天上市100檔只重疊62檔。快照收盤後才可用，故以下一律2016-10-26起24日，與上表2016起點不同，不混算。

'''+snapshot_md+'''
這證明這一個例子對名單組成敏感，不能稱歷史池一定較好或把全部差異歸因於存活偏誤；還包括市值排序、IPO、下市、轉板、名單規則。此快照是研究代理，非當年官方比賽名單。2017–2025逐年股票池、上櫃50檔均未完整重建，仍未完成完整歷史股票池修正。

## 產業個股／同業營收分支
已實際查官方歷史日期產業篩選與MOPS月份頁面，未通過歷史版本門檻：2023年4月營收頁已含2023-05-22公告、7月才生效的綠能分類，頁面出表日是2026年。不能把參考月份誤當分類當時版本。故產業個股選股、同業營收排名尚未產生可信回測；不拿今天分類補歷史，也不冒稱已完成。既有產業指數季節性觀察仍可參考，與個股分類是不同證據。

## 來源與持久紀錄
[FinMind公開API額度](https://finmind.github.io/en/quickstart/)；[FinMind資料集目錄](https://github.com/FinMind/FinMind-MCP/blob/master/knowledge/datasets.md)；[證交所2023分類調整公告](https://www.twse.com.tw/rwd/staticFiles/product/publication/0005000075.pdf)。請求URL、回應摘要與SHA256在快取中；沒有憑證或付費內容。
原Notebook cells226–238保存本輪來源、計算與輸出；kernel未重啟或清除。輸出：十年同期與股票池回測JSON、統計摘要JSON、互動圖HTML、官方與獨立供應商快取。附錄是較早的中間查核，遇到進度矛盾以本段及機器狀態為準。

# 附錄：獨立供應商補充前的來源查核紀錄
'''+old_report_v12
report_path_v12.write_text(report_new_v12,encoding='utf-8')
# One hover surface per annual dot; boxes and median lines do not hover.
fig_external_v12=go_v12.Figure();colors_v12=['#3974b4','#2a947a','#a77ac4'];yr_external_v12=go_v12.Figure()
for ix,(name,rows) in enumerate(external_fixed_results_v12.items()):
    valid=[r for r in rows if r['return'] is not None];ys=[r['return']*100 for r in valid]
    fig_external_v12.add_trace(go_v12.Box(x=[ix]*len(ys),y=ys,name=name,boxpoints=False,hoverinfo='skip',fillcolor=colors_v12[ix],opacity=.25,line_color=colors_v12[ix],showlegend=False))
    jitter=np.linspace(-.12,.12,len(valid))
    fig_external_v12.add_trace(go_v12.Scatter(x=ix+jitter,y=ys,mode='markers',name=name,marker={'size':9,'color':colors_v12[ix]},customdata=[[r['year'],r['start'],r['end'],r['cost']*100] for r in valid],hovertemplate='%{customdata[0]}年<br>%{customdata[1]}～%{customdata[2]}<br>費後報酬 %{y:.3f}%<br>費稅／初始資金 %{customdata[3]:.3f}%<extra>'+name+'</extra>',showlegend=False))
    median=float(np.median(ys));fig_external_v12.add_trace(go_v12.Scatter(x=[ix-.2,ix+.2],y=[median,median],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
    yr_external_v12.add_trace(go_v12.Scatter(x=[r['year'] for r in rows],y=[None if r['return'] is None else r['return']*100 for r in rows],mode='markers+lines',connectgaps=False,name=name,marker={'size':9,'color':colors_v12[ix]},hovertemplate='%{x}年：%{y:.3f}%<extra>'+name+'</extra>'))
for figure in [fig_external_v12,yr_external_v12]:
    figure.update_layout(template='plotly_white',height=440,margin={'l':65,'r':25,'t':35,'b':65},yaxis_title='24交易日費後報酬（%）',hovermode='closest')
    figure.add_hline(y=0,line_width=2,line_dash='dash',line_color='#333')
fig_external_v12.update_xaxes(tickmode='array',tickvals=[0,1,2],ticktext=list(policies_v12),range=[-.5,2.5]);yr_external_v12.update_xaxes(dtick=1)
returns_all_v12=[r['return']*100 for rows in external_fixed_results_v12.values() for r in rows if r['return'] is not None];axislo_v12=min(-1,min(returns_all_v12)-1);axishi_v12=max(1,max(returns_all_v12)+1)
for figure in [fig_external_v12,yr_external_v12]:figure.update_yaxes(range=[axislo_v12,axishi_v12],autorange=False)
def table_external_v12(header,body):return '<table><thead><tr>'+''.join('<th>'+html_v12.escape(str(x))+'</th>' for x in header)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+html_v12.escape(str(x))+'</td>' for x in row)+'</tr>' for row in body)+'</tbody></table>'
metric_html_v12=table_external_v12(['配置','有效年度','費後勝率','24日期望值','平均費稅'],[[name,str(s['n'])+'/10',pct_external_v12(s['win_rate']),pct_external_v12(s['expectancy']),pct_external_v12(s['mean_cost'])] for name,s in external_metrics_v12.items()])
annual_html_v12=table_external_v12(['年度']+list(policies_v12),[[year]+[pct_external_v12(rows[ix]['return']) for rows in external_fixed_results_v12.values()] for ix,year in enumerate(range(2016,2026))])
snapshot_html_v12=table_external_v12(['配置','目前100檔','2016歷史100檔'],[[name]+[pct_external_v12(external_snapshot_results_v12[pool+'|'+name]['return']) for pool in ['current100','historical100']] for name in policies_v12])
html_external_v12='''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>外部歷史研究 v12</title><style>body{font:16px/1.7 system-ui,sans-serif;color:#203043;background:#f4f7fa;margin:0}main{max-width:1120px;margin:24px auto;padding:24px;background:white;border-radius:14px}h1{font-size:26px}h2{font-size:21px;margin-top:32px}.note{background:#fff4dc;padding:14px;border-left:4px solid #d49423}table{border-collapse:collapse;width:100%;margin:18px 0}th,td{padding:10px;border-bottom:1px solid #dce3eb;text-align:right}th:first-child,td:first-child{text-align:left}a{color:#2369a5}</style><main><h1>十年同期 × 股票池：外部歷史研究 v12</h1><p class="note">這是目前上市100檔的價格動能子研究，保留存活與選擇偏誤；不是原150檔A的十年驗證。2017、2018、2025不可判定，不能將剩餘7年當完整十年證據。未證實改良，不由此強選贏家。</p><p>每年十月最後星期二起24交易日，25檔、90%投入、每5日調整、整千股，含手續費及賣出稅。勝率＝有效年度費後正報酬比例；期望值＝平均費後24日報酬。</p>'''+metric_html_v12+'<h2>全部有效年度點與箱形圖</h2><p>每個有效年度都是一個點；滑鼠可看日期、報酬與費稅。<b style="color:#dc2477">桃紅色為中位數</b>，黑色虛線為0%。</p>'+plot_v12(fig_external_v12,output_type='div',include_plotlyjs=True,config={'displaylogo':False,'responsive':True})+'<h2>逐年比較</h2>'+plot_v12(yr_external_v12,output_type='div',include_plotlyjs=False,config={'displaylogo':False,'responsive':True})+annual_html_v12+'<p>2017／2025：持倉遇未解公司行動；2018：缺持倉報價或模擬成交價未通過高低範圍檢查。保留缺值，不補零。</p><h2>同日起點的2016股票池比較</h2><p>兩池只重疊62／100檔。2016-10-25收盤快照只能於次日使用；下表皆2016-10-26起24日，與上圖2016起點不同，沒有混入箱形圖。</p>'+snapshot_html_v12+'<p>只有一個歷史快照；不能推論歷史池總是較好。2017–2025歷史池、上櫃50檔，以及可驗證的歷史個股產業分類仍缺，產業選股／同業營收策略尚無可信結果。</p><h2>資料來源與限制</h2><p>獨立FinMind價格與官方快取11,577筆交叉比對，收盤價、成交量全相符。279筆請求成功278筆；減資資料需更高權限，未繞過。缺漏可能非隨機；費用與滑價模型仍是模擬。股利應收沿用既有競賽模型，不視為可支用現金。</p><p><a href="https://finmind.github.io/en/quickstart/">FinMind公開API說明</a> · <a href="https://www.twse.com.tw/rwd/staticFiles/product/publication/0005000075.pdf">證交所產業分類公告</a></p></main></html>'
chart_path_v12=recovery_root/'外部歷史研究_十年同期與股票池比較_v12_2026-10-15.html';chart_path_v12.write_text(html_external_v12,encoding='utf-8')
statepath_external_v12=recovery_root/'A策略_無人值守研究狀態.json';state_external_v12=json.loads(statepath_external_v12.read_text(encoding='utf-8-sig'))
state_external_v12['external_history_v12']={'status':'partial_completed_feasible_work_remaining_external_gaps','running':False,'notebook_cells':[226,238],'industry':'blocked_PIT_classification_failed_validation','long_history':'10annual_cases_7valid_3unknown_currentTWSE100_price_only','historical_pool':'one2016_snapshot_matched_nextday_only_not_full_correction','fixed_methods':3,'planned_annual_cases':30,'valid_annual_cases':21,'matched_snapshot_cases':6,'selection':'未證實改良','provider_requests':279,'provider_errors':1,'cross_source_rows':11577,'report':report_path_v12.name,'chart':chart_path_v12.name,'statistics':'外部歷史研究_統計摘要_v12_2026-10-15.json','automation':'preserve_paused_no_new_runs','next_required_sources':['PIT individual sector classifications','2017to2025 historical universes and TPEx history','unambiguous corporate action records and missing held quotes'],'source_stop':'TWSE_security_denial_respected_independent_FinMind_used'}
statepath_external_v12.write_text(json.dumps(state_external_v12,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
workflowpath_external_v12=recovery_root/'A策略_無人值守研究流程.md'
with workflowpath_external_v12.open('a',encoding='utf-8') as f:f.write('\n\n## v12 外部歷史分支結果（2026-10-15）\n已完成可行部分：FinMind138檔歷史價量、股利查核，11,577筆官方重疊價量一致；3配置×10同期年度，其中7年有效、3年未知；另3配置×2池的2016-10-26配對案例。完整輸出見「'+report_path_v12.name+'」。目前名單回看有偏誤，歷史池僅單一2016快照，未證實改良。產業個股／同業營收仍缺可驗證當時分類，不可用今日標籤代入。原TWSE安全拒絕已尊重停止；FinMind減資資料升級限制未繞過。無執行中cell或排程，不反覆重試；要有新可驗證來源才續缺口。\n')
print('V12_FINAL_ARTIFACTS',report_path_v12.name,chart_path_v12.name,'state_updated_no_running_work')


In [ ]:
# Install the missing chart dependency in this retained kernel environment only.
import sys
print('KERNEL_PYTHON',sys.executable)
get_ipython().run_line_magic('pip','install plotly')
import plotly
print('PLOTLY_VERIFIED',plotly.__version__)


In [ ]:
# Verify artifacts against retained arrays; no restart or repeat simulation.
assert [external_metrics_v12[k]['n'] for k in policies_v12]==[7,7,7]
assert sum(len(t.y) for t in fig_external_v12.data if t.type=='scatter' and t.mode=='markers')==21
assert len(fig_external_v12.data)==9
assert len(yr_external_v12.data)==3 and all(len(t.y)==10 for t in yr_external_v12.data)
for figure in [fig_external_v12,yr_external_v12]:
    assert any(s.y0==0 and s.y1==0 for s in figure.layout.shapes)
    assert tuple(figure.layout.yaxis.range)==(axislo_v12,axishi_v12)
assert sum(t.mode=='lines' and t.line.color=='#dc2477' for t in fig_external_v12.data if t.type=='scatter')==3
for rows in external_fixed_results_v12.values():
    for row in rows:
        assert (row['return'] is None)==bool(row['invalid'] or row['violations'])
        if row['return'] is not None:assert np.isclose(row['return'],row['path'][-1]/INITIAL-1)
for name,m in external_metrics_v12.items():
    assert np.isclose(m['expectancy'],m['win_rate']*m['average_win']+(1-m['win_rate'])*m['average_loss'])
assert len(external_snapshot_results_v12)==6
assert chart_path_v12.stat().st_size>1000000
paired_md_v12='\n## 配對年度增量（探索性）\n|配置對完整等權|同年勝出比例|平均增量|97.5%配對年度bootstrap區間|\n|---|---:|---:|---|\n'
for name,s in external_pairs_v12.items():
    lo,hi=s['paired_year_bootstrap_97_5pct'];paired_md_v12+='|'+name+'|'+pct_external_v12(s['beat_base_rate'])+'|'+f"{s['mean_excess']*100:.4f}pp"+'|'+f'{lo*100:.4f}～{hi*100:.4f}pp'+'|\n'
paired_md_v12+='\n期望值與區間均只涵蓋7個有效年度；小額略過雖在這7年皆勝出，但非隨機缺漏、現況名單偏誤與開發期重用仍未解，不能宣稱已通過確認性驗證。非等權增量區間跨0。\n\n'
report_text_v12=report_path_v12.read_text(encoding='utf-8').replace('## 歷史股票池的同日起點比較',paired_md_v12+'## 歷史股票池的同日起點比較',1)
report_path_v12.write_text(report_text_v12,encoding='utf-8')
verification_v12={'annual_points':21,'invalid_cases_retained':9,'snapshot_cases':6,'zero_lines':2,'distinct_medians':3,'fixed_axes':True,'expectancy_identity':True,'path_return_identity':True,'verification':'programmatic arrays/layout checks, not browser visual QA','kernel_restarted':False}
state_external_v12['external_history_v12']['verification']=verification_v12
state_external_v12['external_history_v12']['notebook_cells']=[226,240]
statepath_external_v12.write_text(json.dumps(state_external_v12,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('V12_VERIFIED',json.dumps(verification_v12,ensure_ascii=False))


In [ ]:
# v13 restore minimal retained metadata and pre-register full150 extension.
import sys,os
print('V13_KERNEL',sys.executable,'pid',os.getpid())
assert 'provider_history_v12' in globals() and 'external_fixed_results_v12' in globals()
tpex_ids_v13=sorted(str(row['stock_id']) for row in current_market_rows_v12 if row['market'].lower()=='tpex')
full_ids_v13=sorted(set(fixed_twse_v12)|set(tpex_ids_v13))
assert len(tpex_ids_v13)==50 and len(full_ids_v13)==150
print('V13_POOL',{'twse':len(fixed_twse_v12),'tpex':len(tpex_ids_v13),'full':len(full_ids_v13),'cached_prices':sum(sid in provider_history_v12['prices'] for sid in tpex_ids_v13),'cached_dividends':sum(sid in provider_history_v12['dividends'] for sid in tpex_ids_v13)})
print('V13_DB_PRICE_SAMPLE_SCHEMA',{'rows':len(raw_data['price']),'first_row_columns':len(raw_data['price'][0]),'sample_first':raw_data['price'][0]})
print('V13_EXISTING_TPEX_SOURCES',[(key,meta.get('status',meta.get('stat')),meta.get('date')) for key,meta in external_meta_v12.items() if 'tpex' in key])
definition_v13={'version':'v13_full150_TWSE100_TPEX50_fixed_current_universe','purpose':'add50TPEX without changing v12 pricesignals or allocation parameters','current_universe':full_ids_v13,'tpex50':tpex_ids_v13,'years':list(range(2016,2026)),'window':'lastTuesdayOctober_start24sessions','policies':policies_v12,'holdings':25,'invested':.9,'rebalance':5,'lots':1000,'fee':FEE,'tax':TAX,'caps':'2330 .25 others .10, existing5daygrace','incomplete_actions':'unresolved held action or missingheldquote invalidates entire window; no futurefilter stockselection','metrics':['valid_count','winrate','mean24dnetreturn','meanwin','meanloss','fees','mdd','pairedsameyear100vs150'],'history_warning':'current150fixed throughhistory still biased; no claim historical150 reconstructed','unchanged_v12_control':'same100 rerun only to verify expandedarraydimension leaves resultsunchanged','source':'reuseexisting138price/dividendcaches; requestonlymissing50publictickers','refuse':'no newpaid requests, no retry officialsecuritydeniedorigin; stop onauth/quota errors','not_full_original_A':'priceonly20and60daymomentum without institutional eligibility'}
(recovery_root/'完整150檔研究_預先定義_v13_2026-10-15.json').write_text(json.dumps(definition_v13,ensure_ascii=False,indent=2),encoding='utf-8')
print('V13_DEFINITION_SAVED',hashlib.sha256(json.dumps(definition_v13,sort_keys=True).encode()).hexdigest())


In [ ]:
# Fetch only missing public50 TPEX price/dividend records, retaining all existing caches.
import gzip
tpex_cache_v13=recovery_root/'完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz'
if tpex_cache_v13.exists():
    with gzip.open(tpex_cache_v13,'rt',encoding='utf-8') as f:tpex_history_v13=json.load(f)
else:tpex_history_v13={'prices':{},'dividends':{},'metadata':{},'errors':{},'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
assert not provider_stop_v12.is_set(),'Prior provider stop needs inspection, do not silently reset'
tpex_requests_v13=[]
for sid in tpex_ids_v13:
    for kind,dataset,start in [('prices','TaiwanStockPrice','2016-07-01'),('dividends','TaiwanStockDividend','2015-01-01')]:
        if sid in tpex_history_v13[kind]:continue
        if kind+':'+sid in tpex_history_v13['errors']:continue
        tpex_requests_v13.append((kind,sid,{'dataset':dataset,'data_id':sid,'start_date':start,'end_date':'2025-12-31'}))
def save_tpex_v13():
    with gzip.open(tpex_cache_v13,'wt',encoding='utf-8') as f:json.dump(tpex_history_v13,f,ensure_ascii=False,separators=(',',':'))
print('V13_FETCH_PLAN',len(tpex_requests_v13),'requests;previous batch ended over1hourago;public300hourquota',flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as tpex_pool_v13:
    for count,(kind,sid,rows,meta) in enumerate(tpex_pool_v13.map(provider_request_v12,tpex_requests_v13),1):
        meta['retrieved_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat();key=kind+':'+sid;tpex_history_v13['metadata'][key]=meta
        if rows is None:tpex_history_v13['errors'][key]=meta
        else:tpex_history_v13[kind][sid]=rows
        if count%10==0 or count==len(tpex_requests_v13):save_tpex_v13();print('V13_FETCH_PROGRESS',count,'/',len(tpex_requests_v13),'errors',len(tpex_history_v13['errors']),flush=True)
tpex_history_v13['finished_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat();save_tpex_v13()
print('V13_FETCH_DONE',{k:len(tpex_history_v13[k]) for k in ['prices','dividends','errors']})


In [ ]:
# Full150 extension uses a namespaced copy of the verified v12 engine.
assert len(tpex_history_v13['prices'])==50 and len(tpex_history_v13['dividends'])==50,'Missing source: do not call full150 complete'
provider_history_v13={kind:{**provider_history_v12[kind],**tpex_history_v13[kind]} for kind in ['prices','dividends','metadata','errors']}
provider_history_v13['global_events']=provider_history_v12['global_events']
provider_tickers_v13=sorted(provider_history_v13['prices'])
provider_definition_v13={**provider_definition_v12,'version':'v13_current150_extension','added_TPEX50':tpex_ids_v13,'universe_definition':definition_v13}
long_raw_v13=long_raw_v12;quote_table_v13=quote_table_v12;number_v13=number_v12;calendar_days_v13=calendar_days_v12
# Cross-source numerical validation, conservative cash-only action accounting, then fixed studies
provider_definition_v13['reference_tolerance']='max(.021NTD,.0001*previousclose); unexplainedreferencechanges flagged, never fabricate dividends'
provider_definition_sha_v13=hashlib.sha256(json.dumps(provider_definition_v13,sort_keys=True).encode()).hexdigest()
(recovery_root/'外部歷史研究_獨立供應商補充定義_v13_2026-10-15.json').write_text(json.dumps(provider_definition_v13,ensure_ascii=False,indent=2),encoding='utf-8')
price_lookup_v13={sid:{r['date']:r for r in rows} for sid,rows in provider_history_v13['prices'].items()}
cross_source_v13={'comparisons':0,'close_matches':0,'volume_matches':0,'mismatch_samples':[]}
for day,doc in long_raw_v13['quotes'].items():
    tab=quote_table_v13(doc);fields=tab['fields'];date=f'{day[:4]}-{day[4:6]}-{day[6:]}'
    for row in tab['data']:
        sid=str(row[fields.index('證券代號')]).strip();other=price_lookup_v13.get(sid,{}).get(date)
        if other is None:continue
        px=number_v13(row[fields.index('收盤價')]);vol=number_v13(row[fields.index('成交股數')])
        if not np.isfinite(px):continue
        cross_source_v13['comparisons']+=1;ok=abs(px-number_v13(other['close']))<=.011;cross_source_v13['close_matches']+=int(ok)
        cross_source_v13['volume_matches']+=int(abs(vol-number_v13(other['Trading_Volume']))<=1)
        if not ok and len(cross_source_v13['mismatch_samples'])<5:cross_source_v13['mismatch_samples'].append({'date':date,'sid':sid,'official':px,'provider':other['close']})
cross_source_v13['close_match_fraction']=cross_source_v13['close_matches']/cross_source_v13['comparisons'] if cross_source_v13['comparisons'] else 0
assert cross_source_v13['comparisons']>=100 and cross_source_v13['close_match_fraction']>=.995,'Provider raw-price validation failed; no backtest'
print('V12_PROVIDER_CROSS_CHECK',json.dumps(cross_source_v13),flush=True)
all_external_ids_v13=provider_tickers_v13;ext_si_v13={sid:i for i,sid in enumerate(all_external_ids_v13)}
cash_records_v13=collections.defaultdict(list);bad_action_dates_v13=set();cash_audit_v13=collections.Counter()
def valid_event_day_v13(value):
    try:
        day=datetime.date.fromisoformat(str(value)[:10])
        if day<calendar_days_v13[0] or day>calendar_days_v13[-1]:return None
        idx=bisect.bisect_left(calendar_days_v13,day)
        return calendar_days_v13[idx] if idx<len(calendar_days_v13) else None
    except (ValueError,TypeError):return None
for sid,rows in provider_history_v13['dividends'].items():
    for row in rows:
        cash_amt=sum(number_v13(row.get(k,0)) if np.isfinite(number_v13(row.get(k,0))) else 0 for k in ['CashEarningsDistribution','CashStatutorySurplus'])
        cashday=valid_event_day_v13(row.get('CashExDividendTradingDate'))
        if cash_amt>0:
            if cashday is not None:cash_records_v13[(cashday,sid)].append(float(cash_amt))
            else:cash_audit_v13['positive_cash_without_usable_exdate']+=1
        stock_amt=sum(number_v13(row.get(k,0)) if np.isfinite(number_v13(row.get(k,0))) else 0 for k in ['StockEarningsDistribution','StockStatutorySurplus'])
        stockday=valid_event_day_v13(row.get('StockExDividendTradingDate'))
        if stock_amt>0 and stockday is not None:bad_action_dates_v13.add((stockday,sid));cash_audit_v13['noncash_stock_events']+=1
cash_amount_v13={}
for key,amounts in cash_records_v13.items():
    if np.ptp(amounts)<=1e-6:cash_amount_v13[key]=amounts[0]
    else:bad_action_dates_v13.add(key);cash_audit_v13['ambiguous_same_day_cash_policy']+=1
for dataset,rows in provider_history_v13['global_events'].items():
    for row in rows:
        sid=str(row.get('stock_id',''));day=valid_event_day_v13(row.get('date'))
        if sid in ext_si_v13 and day is not None:bad_action_dates_v13.add((day,sid));cash_audit_v13['global_nonstandard_events']+=1
print('V12_CASH_SOURCE_AUDIT',len(cash_amount_v13),'knowncash events',dict(cash_audit_v13),flush=True)
case_arrays_v13={}
def build_case_v13(case):
    days=case['dates'];shape=(len(days),len(all_external_ids_v13));cl=np.full(shape,np.nan);vo=np.full(shape,np.nan);exe=np.full(shape,np.nan);spread=np.full(shape,np.nan);lo=np.full(shape,np.nan);hi=np.full(shape,np.nan);cashdiv=np.zeros(shape);bad=np.zeros(shape,bool)
    for t,day in enumerate(days):
        for j,sid in enumerate(all_external_ids_v13):
            row=price_lookup_v13.get(sid,{}).get(str(day))
            if row is not None:
                cl[t,j]=number_v13(row['close']);vo[t,j]=number_v13(row['Trading_Volume']);spread[t,j]=number_v13(row['spread']);lo[t,j]=number_v13(row['min']);hi[t,j]=number_v13(row['max'])
                if vo[t,j]>0:exe[t,j]=number_v13(row['Trading_money'])/vo[t,j]
            cashdiv[t,j]=cash_amount_v13.get((day,sid),0.);bad[t,j]=(day,sid) in bad_action_dates_v13
    returns=np.full(shape,np.nan);unknown_ref=[]
    for t in range(1,len(days)):
        ref=cl[t]-spread[t];expected=cl[t-1]-cashdiv[t]
        valid=np.isfinite(cl[t])&np.isfinite(cl[t-1])&(ref>0)&(cl[t-1]>0)
        mismatch=valid&(np.abs(ref-expected)>np.maximum(.021,.0001*cl[t-1]));bad[t]|=mismatch
        if mismatch.any():unknown_ref.extend({'date':str(days[t]),'sid':all_external_ids_v13[j],'previous':float(cl[t-1,j]),'reported_reference':float(ref[j]),'known_cash':float(cashdiv[t,j])} for j in np.flatnonzero(mismatch))
        returns[t,valid]=cl[t,valid]/ref[valid]-1
    available=np.isfinite(exe)&(exe>0)&np.isfinite(lo)&np.isfinite(hi)&(exe>=lo-.011)&(exe<=hi+.011)
    return {'dates':days,'close':cl,'volume':vo,'execution':exe,'cashdiv':cashdiv,'bad':bad,'returns':returns,'available':available,'unknown_reference_events':unknown_ref}
def simulate_external_v13(case,pool_ids,allocation,tolerance):
    arrays=case_arrays_v13[case['key']];cl=arrays['close'];cashdiv=arrays['cashdiv'];bad=arrays['bad'];r=arrays['returns'];vo=arrays['volume'];days=arrays['dates'];start=case['start_index'];shares=np.zeros(len(all_external_ids_v13),np.int64);cash=INITIAL;receivable=0.;values=[INITIAL];fees=0.;turnover_sum=0.;reasons=set();violations=0;cap_age=np.zeros(len(shares),int);minheld=999;maxheld=0;caps=np.array([.25 if sid=='2330' else .1 for sid in all_external_ids_v13]);traces=[]
    poolmask=np.array([sid in pool_ids for sid in all_external_ids_v13])
    for offset,t in enumerate(range(start,start+24)):
        before=shares.copy()
        if np.any(bad[t]&(before>0)):reasons.add('unmodeled_or_ambiguous_company_action_while_held')
        receivable+=float(np.dot(before,cashdiv[t]));prior_nav=cash+np.dot(shares,np.nan_to_num(cl[t-1]))
        if offset%5==0:
            rr=r[t-60:t];eligible=poolmask&np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(vo[t-20:t])&(vo[t-20:t]>0),axis=0)&~np.any(bad[t-60:t],axis=0)&np.isfinite(cl[t-1])&(cl[t-1]>0)
            if eligible.sum()<25:reasons.add('fewer_than25_past_data_eligible_stocks');break
            score=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,eligible)+.5*percentile(np.prod(1+rr,axis=0)-1,eligible)
            ordered=np.flatnonzero(eligible);selected=ordered[np.argsort(-score[ordered],kind='stable')[:25]]
            weights=np.full(25,.9/25) if allocation=='equal' else .9*np.linspace(1.25,.75,25)/25
            target=np.zeros_like(shares);target[selected]=(np.floor(weights*prior_nav/cl[t-1,selected]/1000)*1000).astype(np.int64)
            if offset>0 and tolerance:
                keep=(shares>0)&(target>0)&(np.abs(target-shares)*np.nan_to_num(cl[t-1])<=tolerance*prior_nav);target[keep]=shares[keep]
            cash,shares,traded,cost,missing=settle(cash,shares,target,arrays['execution'][t],arrays['available'][t]);fees+=cost;turnover_sum+=traded
            if missing:reasons.add('invalid_execution_price')
            traces.append({'date':str(days[t]),'selected':[all_external_ids_v13[j] for j in selected]})
        if np.any((shares>0)&~np.isfinite(cl[t])):reasons.add('missing_held_close')
        nav=cash+np.dot(shares,np.nan_to_num(cl[t]));aum=nav+receivable;values.append(aum)
        if nav<=0:reasons.add('nonpositive_tradable_nav');break
        weights=shares*np.nan_to_num(cl[t])/nav;cap_age=np.where(weights>caps+1e-12,cap_age+1,0);held=int(np.count_nonzero(shares));minheld=min(minheld,held);maxheld=max(maxheld,held)
        violations+=int(cash<0 or cash/nav>=.25 or not 20<=held<=30 or np.any(cap_age>5))
    invalid=bool(reasons or violations);vals=np.array(values);net=float(vals[-1]/INITIAL-1)
    return {'case':case['key'],'year':case['year'],'start':str(days[start]),'end':str(days[start+23]),'return':None if invalid else net,'mdd':None if invalid else float(np.min(vals/np.maximum.accumulate(vals)-1)),'cost':fees/INITIAL,'turnover':turnover_sum/INITIAL,'invalid':bool(reasons),'violations':violations,'reasons':sorted(reasons),'min_held':minheld,'max_held':maxheld,'trace':traces,'path':vals.tolist() if not invalid else None}

# Cross-source validation specifically for TPEX, separately from prior TWSE matches.
tpex_cross_v13={'comparisons':0,'close_matches':0,'volume_matches':0,'mismatch_samples':[]}
for row in raw_data['price']:
    sid=str(row[0]);day=str(row[1])
    if sid not in tpex_ids_v13:continue
    other=price_lookup_v13.get(sid,{}).get(day)
    if other is None:continue
    db_close=number_v12(row[2]);db_volume=number_v12(row[3])
    if not np.isfinite(db_close) or not np.isfinite(db_volume):continue
    close_match=abs(db_close-number_v12(other['close']))<=.011
    volume_match=abs(db_volume-number_v12(other['Trading_Volume']))<=1
    tpex_cross_v13['comparisons']+=1;tpex_cross_v13['close_matches']+=int(close_match);tpex_cross_v13['volume_matches']+=int(volume_match)
    if not close_match and len(tpex_cross_v13['mismatch_samples'])<5:tpex_cross_v13['mismatch_samples'].append({'sid':sid,'date':day,'DB_close':float(row[2]),'provider_close':other['close']})
assert tpex_cross_v13['comparisons']>=1000 and tpex_cross_v13['close_matches']/tpex_cross_v13['comparisons']>=.995,'TPEX crosssource gate failed'
print('V13_TPEX_CROSS_CHECK',json.dumps(tpex_cross_v13),flush=True)
full_results_v13={name:[] for name in policies_v12};control_results_v13={name:[] for name in policies_v12};coverage_v13=[]
for year,basecase in long_cases_v12.items():
    case=dict(basecase,key=f'full150_{year}');arrays=build_case_v13(case);case_arrays_v13[case['key']]=arrays
    t=case['start_index'];rr=arrays['returns'][t-60:t];poolmask=np.array([sid in full_ids_v13 for sid in all_external_ids_v13])
    eligible=poolmask&np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(arrays['volume'][t-20:t])&(arrays['volume'][t-20:t]>0),axis=0)&~np.any(arrays['bad'][t-60:t],axis=0)&np.isfinite(arrays['close'][t-1])&(arrays['close'][t-1]>0)
    coverage_v13.append({'year':year,'current_pool150':150,'TPEX_price_on_prior_day':sum(sid in tpex_ids_v13 and np.isfinite(arrays['close'][t-1,j]) for j,sid in enumerate(all_external_ids_v13)),'TPEX_firstday_eligible':sum(sid in tpex_ids_v13 and eligible[j] for j,sid in enumerate(all_external_ids_v13)),'total_firstday_eligible':int(eligible.sum())})
    for name,(allocation,tolerance) in policies_v12.items():
        full_results_v13[name].append(simulate_external_v13(case,set(full_ids_v13),allocation,tolerance))
        control=simulate_external_v13(case,fixed_twse_v12,allocation,tolerance);control_results_v13[name].append(control)
        original=next(r for r in external_fixed_results_v12[name] if r['year']==year)
        assert control['reasons']==original['reasons'] and control['violations']==original['violations'],'Expanded dimensions changed control eligibility or actions'
        assert (control['return'] is None)==(original['return'] is None)
        if control['return'] is not None:assert np.isclose(control['return'],original['return'],rtol=0,atol=1e-12),'Control return changed'
    print('V13_FULL150_YEAR',year,{name:round(rows[-1]['return']*100,4) if rows[-1]['return'] is not None else 'unknown:'+','.join(rows[-1]['reasons']) for name,rows in full_results_v13.items()},flush=True)
full_export_v13={'definition':definition_v13,'full150':full_results_v13,'same100_control':control_results_v13,'TPEX_DB_crosscheck':tpex_cross_v13,'coverage':coverage_v13,'unknown_reference_events':{key:arrays['unknown_reference_events'] for key,arrays in case_arrays_v13.items()},'source_cache':tpex_cache_v13.name,'limits':['current150fixed historical bias persists','pricesignals only not full original A','cashonlyaction model invalidates unknown held events','unavailablehistoricalstocks not silently fabricated','TPEX DBcrosscheck2023to2025 only, earlier prices not independently verified']}
(recovery_root/'完整150檔研究_同期回測與100檔配對_v13_2026-10-15.json').write_text(json.dumps(full_export_v13,ensure_ascii=False,indent=2,default=lambda x:int(x) if isinstance(x,np.integer) else str(x),allow_nan=False),encoding='utf-8')
print('V13_BACKTEST_SAVED','30full150cases;30controlchecks;no changes to selection settings')


In [ ]:
# Investigate TPEX volume mismatch before accepting full150 data quality.
samples_v13=[]
for row in raw_data['price']:
    sid=str(row[0]);day=str(row[1]);other=price_lookup_v13.get(sid,{}).get(day)
    if sid not in tpex_ids_v13 or other is None or day!='2024-10-28':continue
    samples_v13.append({'sid':sid,'date':day,'DB_volume':str(row[3]),'provider_volume':other['Trading_Volume'],'DB_money':str(row[4]),'provider_money':other['Trading_money'],'DB_execution':str(row[5]),'provider_execution':other['Trading_money']/other['Trading_Volume'] if other['Trading_Volume'] else None,'provider_low':other['min'],'provider_high':other['max']})
print('V13_VOLUME_SAMPLES',json.dumps(samples_v13[:5],ensure_ascii=False))
tpexdoc_v13=external_raw_v12.get('tpex_history_current2024')
print('V13_TPEX_OFFICIAL_METADATA',external_meta_v12.get('tpex_history_current2024'))
print('V13_TPEX_OFFICIAL_STRUCTURE',{k:type(v).__name__ for k,v in (tpexdoc_v13 or {}).items()})
if tpexdoc_v13:
    tabs=tpexdoc_v13.get('tables') or [tpexdoc_v13]
    print('V13_TPEX_OFFICIAL_FIELDS',[(tab.get('fields'),len(tab.get('data') or []),(tab.get('data') or [])[:1]) for tab in tabs])


In [ ]:
# Match independently cached TPEx daily report, including volume and turnover.
official_match_v13={'date':'2024-10-28','comparisons':0,'close_matches':0,'volume_matches':0,'money_matches':0,'mismatch_samples':[]}
tab=tpexdoc_v13['tables'][0];fields=tab['fields']
for row in tab['data']:
    sid=str(row[fields.index('代號')]).strip()
    if sid not in tpex_ids_v13:continue
    provider=price_lookup_v13.get(sid,{}).get('2024-10-28')
    if provider is None:continue
    official_match_v13['comparisons']+=1
    okclose=abs(number_v12(row[fields.index('收盤')])-provider['close'])<=.011
    okvol=abs(number_v12(row[fields.index('成交股數')])-provider['Trading_Volume'])<=1
    okmoney=abs(number_v12(row[fields.index('成交金額(元)')])-provider['Trading_money'])<=1
    official_match_v13['close_matches']+=int(okclose);official_match_v13['volume_matches']+=int(okvol);official_match_v13['money_matches']+=int(okmoney)
    if not (okclose and okvol and okmoney):official_match_v13['mismatch_samples'].append({'sid':sid,'close_ok':okclose,'volume_ok':okvol,'money_ok':okmoney,'official_volume':row[fields.index('成交股數')],'provider_volume':provider['Trading_Volume']})
assert official_match_v13['comparisons']>=40
assert official_match_v13['close_matches']==official_match_v13['comparisons']
assert official_match_v13['volume_matches']==official_match_v13['comparisons'] and official_match_v13['money_matches']==official_match_v13['comparisons'],'Independent official volume/turnover mismatch'
print('V13_OFFICIAL_TPEX_MATCH',json.dumps(official_match_v13,ensure_ascii=False))
full_export_v13['official_TPEX_crosscheck']=official_match_v13
full_export_v13['TPEX_DB_crosscheck']['note']='DB closes match; DB and provider volume/money definitions differ. Provider matches independent official20241028 report; use provider consistently for this study, not claim DB VWAP identical. No adjustment inferred from mismatch.'
(recovery_root/'完整150檔研究_同期回測與100檔配對_v13_2026-10-15.json').write_text(json.dumps(full_export_v13,ensure_ascii=False,indent=2,default=lambda x:int(x) if isinstance(x,np.integer) else str(x),allow_nan=False),encoding='utf-8')


In [ ]:
# Full150 and100 controls: persist comparable metrics and every annual point.
metrics150_v13={name:stats_external_v12(rows) for name,rows in full_results_v13.items()}
metrics100_v13={name:stats_external_v12(rows) for name,rows in control_results_v13.items()}
paired150_v13={};rng_v13=np.random.default_rng(20261015)
for name,rows in full_results_v13.items():
    old={r['year']:r for r in control_results_v13[name] if r['return'] is not None}
    matched=[(r,old[r['year']]) for r in rows if r['return'] is not None and r['year'] in old]
    diff=np.array([a['return']-b['return'] for a,b in matched]);boot=np.mean(diff[rng_v13.integers(0,len(diff),(20000,len(diff)))],axis=1) if len(diff) else np.array([])
    paired150_v13[name]={'n':len(diff),'years':[a['year'] for a,b in matched],'mean_excess150minus100':float(np.mean(diff)) if len(diff) else None,'beat100_rate':float(np.mean(diff>0)) if len(diff) else None,'family3_exploratory_bootstrap98_333pct':np.quantile(boot,[.05/6,1-.05/6]).tolist() if len(diff) else None,'interpretation':'descriptive7nonrandomlyvalidyears,currentuniversebias,developmentperiodreuse;notconfirmation','yearly':[{'year':a['year'],'return150':a['return'],'return100':b['return'],'excess':a['return']-b['return']} for a,b in matched]}
summary150_v13={'full150':metrics150_v13,'control100':metrics100_v13,'paired':paired150_v13,'definition':definition_v13,'official_TPEX_check':official_match_v13,'DB_TPEX_check':tpex_cross_v13,'coverage':coverage_v13,'conclusion':'未證實150檔股票池或非等權可穩定改善；完成完整現況150檔的歷史同期子研究，非歷史股票池修正完成'}
(recovery_root/'完整150檔研究_統計摘要_v13_2026-10-15.json').write_text(json.dumps(summary150_v13,ensure_ascii=False,indent=2,default=lambda x:int(x) if isinstance(x,np.integer) else str(x),allow_nan=False),encoding='utf-8')
print('V13_FULL150_METRICS',json.dumps(metrics150_v13,ensure_ascii=False))
print('V13_150MINUS100',json.dumps(paired150_v13,ensure_ascii=False))
metricrows_v13=[]
for name in policies_v12:
    a=metrics100_v13[name];b=metrics150_v13[name];d=paired150_v13[name]
    metricrows_v13.append([name,str(b['n'])+'/10',pct_external_v12(a['win_rate']),pct_external_v12(a['expectancy']),pct_external_v12(b['win_rate']),pct_external_v12(b['expectancy']),f"{d['mean_excess150minus100']*100:+.3f}pp",pct_external_v12(b['mean_cost'])])
headers_v13=['配置','有效年度','100檔勝率','100檔期望值','150檔勝率','150檔期望值','同年增量','150檔平均費稅']
annualrows_v13=[]
for ix,year in enumerate(range(2016,2026)):
    row=[str(year)]
    for name in policies_v12:row.extend([pct_external_v12(control_results_v13[name][ix]['return']),pct_external_v12(full_results_v13[name][ix]['return'])])
    annualrows_v13.append(row)
annualheaders_v13=['年度']+[name+' '+pool for name in policies_v12 for pool in ['100檔','150檔']]
def mdtable_v13(headers,rows):return '|'+ '|'.join(headers)+'|\n|'+ '|'.join('---' for h in headers)+'|\n'+'\n'.join('|'+ '|'.join(str(v) for v in row)+'|' for row in rows)+'\n'
coverage_md_v13=mdtable_v13(['年度','上櫃前日有報價','上櫃起日通過60日條件','全池起日合格'],[[r['year'],r['TPEX_price_on_prior_day'],r['TPEX_firstday_eligible'],r['total_firstday_eligible']] for r in coverage_v13])
report150_v13='''# 完整150檔：上市100＋上櫃50同期比較 v13
2026-10-15。使用者要求補上50檔；重用上市資料，僅新增上櫃50檔的歷史價格與股利，不改選股／配置參數。

## 完成範圍
已取得目前比賽100上市＋50上櫃，共150檔的2016–2025歷史日價量與股利政策。新增100筆公開API請求全部成功。這是『目前150檔回看歷史』，不是已重建各年當時150檔，也不是當年所有股票都有上市或60日暖機資料。資料不存在就保持缺漏，不倒造上市前價格。
原138檔來源仍保留，其中含2016快照歷史名單；本輪可選名單嚴格只用現在150檔，其餘不參與排名。

## 固定設計與資料品質
2016–2025各年10月最後星期二起24交易日、原3配置；25檔持倉、投入90%、5日調整、1000股單位、手續費0.1425%、賣出稅0.3%，沿用既有現金股利應收與權重限制檢查。前20／60日價格動能各半排名。25檔是這次補資料比較的控制設定，不代表往後策略必須25檔。
本輪不含法人歷史資格條件，故不是完整原A；也不拿本輪數字和舊650窗口混排。
上櫃收盤價與團隊DB有24,942筆可比紀錄，全部相符；DB成交量僅10筆與供應商完全相同，成交量／金額口徑不同，沒有以推測比例修正。另用先前保存的官方2024-10-28上櫃行情核對，45檔價格、量、金額均吻合供應商；另外5檔未形成當日可比樣本，不補造。單日核對不能證明早年所有日資料正確。
本輪用同一供應商價量／成交金額，VWAP須落在當日高低價範圍才可用。未知持倉公司行動、缺持倉收盤或無效成交會使整個窗口不可判定，不依未來資訊換掉該股票後重選。原100檔在擴增矩陣後30個控制案例與v12全數一致。

## 費後勝率與期望值
勝率＝有效年度24日費後正報酬比例；期望值＝有效年度費後報酬平均，含虧損、不年化。同年增量只用雙方均有效的配對，單位pp為百分點。平均費稅以初始資金為分母。

'''+mdtable_v13(headers_v13,metricrows_v13)+'''
目前兩邊共同有效年度皆為2016、2019–2024，共7年；2017、2018、2025均不可判定，不能說是完整十年勝率。

## 每個年度的結果
'''+mdtable_v13(annualheaders_v13,annualrows_v13)+'''
2017／2025持倉遇未解公司行動；2018另有缺持倉收盤與模擬成交價異常。原始JSON保留每個案例原因、每日資產路徑及選股軌跡，不把缺值補零。圖中每個有效年度各一點；三個配置各7點，100與150合計42點。

## 當年可用資料覆蓋
股票池標示150不等於每年都有150檔可選；尚未上市、缺60日資料或過去事件未解者依原規則不合格。下列是各年起始日覆蓋，不是整段期間的保證。
'''+coverage_md_v13+'''
## 如何解讀
150檔納入上櫃後，有些年度更好、有些更差；本輪平均較高不等於穩定勝出。對3個配置的同年150減100做年度bootstrap探索性區間（每個98.333%邊際區間以反映3次比較），原始統計另存JSON。七個非隨機有效年度、現況名單偏誤與開發期重用仍在，不能把區間當確認性驗證或650窗口的24／48日區塊驗證。
結論：完成現況完整150檔子研究，未證實股票池擴充或非等權穩定改善，不強選新贏家。歷史分類、逐年歷史股票池與三年事件缺口依然未解。

## 來源與交付
重用FinMind歷史價量、股利、官方TWSE快取、官方TPEx2024-10-28快取及唯讀DB快照，沒有重新請求已拒絕來源或升級付費資料。
[FinMind公開API](https://finmind.github.io/en/quickstart/)；[官方上櫃行情來源](https://www.tpex.org.tw/www/zh-tw/afterTrading/dailyQuotes?date=2024%2F10%2F28&response=json)。新增請求URL、SHA256、抓取時間與回應均存上櫃快取；沒有憑證。
原Notebook cells241–246保存本輪定義、抓取、核對、回測、統計與圖表；kernel未重啟或清除。本輪沒有重啟排程、提交交易或新增LLM權重。
'''
report_path150_v13=recovery_root/'完整150檔研究_同期比較_v13_2026-10-15.md';report_path150_v13.write_text(report150_v13,encoding='utf-8')
fig150_v13=go_v12.Figure();figyears_v13=go_v12.Figure();palette_v13=['#3974b4','#2a947a','#a77ac4']
returns_v13=[r['return']*100 for pool in [full_results_v13,control_results_v13] for rows in pool.values() for r in rows if r['return'] is not None];range_v13=[min(-1,min(returns_v13)-1),max(1,max(returns_v13)+1)]
for poolidx,(poollabel,result) in enumerate([('150檔（上市＋上櫃）',full_results_v13),('100檔（僅上市）',control_results_v13)]):
    for ix,(name,rows) in enumerate(result.items()):
        valid=[r for r in rows if r['return'] is not None];ys=[r['return']*100 for r in valid];visible=poolidx==0
        fig150_v13.add_trace(go_v12.Box(x=[ix]*len(ys),y=ys,boxpoints=False,hoverinfo='skip',fillcolor=palette_v13[ix],opacity=.25,line_color=palette_v13[ix],showlegend=False,visible=visible))
        fig150_v13.add_trace(go_v12.Scatter(x=ix+np.linspace(-.12,.12,len(valid)),y=ys,mode='markers',name=poollabel+'／'+name,marker={'size':10,'color':palette_v13[ix]},customdata=[[r['year'],r['start'],r['end'],r['cost']*100] for r in valid],hovertemplate='%{customdata[0]}年<br>%{customdata[1]}～%{customdata[2]}<br>費後報酬 %{y:.3f}%<br>費稅／初始資金 %{customdata[3]:.3f}%<extra>'+poollabel+'／'+name+'</extra>',visible=visible,showlegend=False))
        m=float(np.median(ys));fig150_v13.add_trace(go_v12.Scatter(x=[ix-.2,ix+.2],y=[m,m],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False,visible=visible))
        figyears_v13.add_trace(go_v12.Scatter(x=[r['year'] for r in rows],y=[None if r['return'] is None else r['return']*100 for r in rows],mode='markers+lines',connectgaps=False,name=poollabel+'／'+name,line={'color':palette_v13[ix],'dash':'solid' if visible else 'dot'},marker={'size':9 if visible else 6},hovertemplate='%{x}年：%{y:.3f}%<extra>'+poollabel+'／'+name+'</extra>'))
for figure in [fig150_v13,figyears_v13]:
    figure.update_layout(template='plotly_white',height=490,margin={'l':65,'r':25,'t':85,'b':65},yaxis_title='24交易日費後報酬（%）',hovermode='closest')
    figure.add_hline(y=0,line_width=2,line_dash='dash',line_color='#333');figure.update_yaxes(range=range_v13,autorange=False)
fig150_v13.update_xaxes(tickmode='array',tickvals=[0,1,2],ticktext=list(policies_v12),range=[-.5,2.5]);figyears_v13.update_xaxes(dtick=1)
fig150_v13.update_layout(updatemenus=[{'type':'buttons','direction':'right','x':0,'y':1.18,'buttons':[{'label':'完整150檔（預設）','method':'update','args':[{'visible':[True]*9+[False]*9}]},{'label':'原100檔對照','method':'update','args':[{'visible':[False]*9+[True]*9}]}]}])
metric_html150_v13=table_external_v12(headers_v13,metricrows_v13);annual_html150_v13=table_external_v12(annualheaders_v13,annualrows_v13)
coverage_html150_v13=table_external_v12(['年度','上櫃前日有報價','上櫃起日合格','全池起日合格'],[[r['year'],r['TPEX_price_on_prior_day'],r['TPEX_firstday_eligible'],r['total_firstday_eligible']] for r in coverage_v13])
html150_v13='''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>完整150檔研究 v13</title><style>body{font:16px/1.7 system-ui,sans-serif;color:#203043;background:#f4f7fa;margin:0}main{max-width:1200px;margin:24px auto;padding:24px;background:white;border-radius:14px}h1{font-size:27px}h2{font-size:21px;margin-top:32px}.note{background:#fff4dc;padding:14px;border-left:4px solid #d49423}.scroll{overflow-x:auto}table{border-collapse:collapse;width:100%;margin:18px 0;font-size:14px}th,td{padding:9px;border-bottom:1px solid #dce3eb;text-align:right;white-space:nowrap}th:first-child,td:first-child{text-align:left}a{color:#2369a5}</style><main><h1>完整150檔：上市100＋上櫃50</h1><p>2016–2025各年比賽同期24交易日。三種配置與費率沿用v12；實際各選25檔，並非買入150檔。沒有新增LLM判斷。</p><p class="note">已補50檔歷史價量與股利，但仍是今天150檔名單回看歷史；不是逐年當時150檔。2017／2018／2025不可判定，表中勝率／期望值只含共同有效7年。未證實新配置或股票池穩定改善。</p><div class="scroll">'''+metric_html150_v13+'''</div><p>勝率＝有效年度費後正報酬比例；期望值＝平均24日費後報酬，含虧損、不年化。同年增量為150減100的百分點。平均費稅以初始資金計。</p><h2>箱形圖與全部有效年度點</h2><p>預設完整150檔；按鈕切換100檔對照，兩者座標固定。每個點可查年份、起訖日、費後報酬與費稅。<b style="color:#dc2477">桃紅線為中位數</b>，黑色虛線為0%。</p>'''+plot_v12(fig150_v13,output_type='div',include_plotlyjs=True,config={'displaylogo':False,'responsive':True})+'<h2>逐年比較：實線150、點線100</h2>'+plot_v12(figyears_v13,output_type='div',include_plotlyjs=False,config={'displaylogo':False,'responsive':True})+'<div class="scroll">'+annual_html150_v13+'</div><p>2017／2025持倉公司行動未解；2018另有持倉收盤缺漏或模擬成交價異常。保留不可判定，不補零、不以未來事件換股後重選。</p><h2>歷史資料覆蓋</h2>'+coverage_html150_v13+'<p>目前名單150檔不代表早年每檔均已上市。上表為開賽前可取得報價及起日通過原60日資料／事件條件的檔數，不倒造上市前資料。</p><h2>來源與比較限制</h2><p>新增50檔的100筆請求全成功。上櫃收盤價與團隊DB 24,942筆一致；成交量口徑不同，沒有宣稱價量全相同。另與官方2024-10-28可比45檔核對，價格、量、金額全相符。本輪統一供應商口徑，無法由單日驗證保證早年所有資料。原100檔30個控制案例均與v12一致。</p><p>前20／60日價格動能各半；投入90%、每5日調整、整千股、手續費0.1425%及賣出稅0.3%。不是含法人資格的原A全模型。25檔是此次控制設定，不是將來必須25檔。7個非隨機有效年度、現況名單偏誤與開發期重用仍在，不選新贏家。</p><p><a href="https://finmind.github.io/en/quickstart/">FinMind公開API</a> · <a href="https://www.tpex.org.tw/www/zh-tw/afterTrading/dailyQuotes?date=2024%2F10%2F28&response=json">官方上櫃行情核對來源</a></p></main></html>'
chart_path150_v13=recovery_root/'完整150檔研究_同期配對箱形圖_v13_2026-10-15.html';chart_path150_v13.write_text(html150_v13,encoding='utf-8')
state150_v13=json.loads(statepath_external_v12.read_text(encoding='utf-8'))
state150_v13['full150_extension_v13']={'status':'completed_feasible_full_current150_with_explicit_invalid_years','running':False,'universe':{'twse':100,'tpex':50},'added_price_series':50,'added_dividend_series':50,'source_requests':100,'source_errors':0,'annual_cases':30,'valid_cases':21,'unknown_cases':9,'control_checks':30,'DB_close_matches':24942,'DB_volume_matches':10,'official_TPEX_matches':official_match_v13,'report':report_path150_v13.name,'chart':chart_path150_v13.name,'statistics':'完整150檔研究_統計摘要_v13_2026-10-15.json','notebook_cells':[241,246],'limits':['currentuniversebias','3invalidyears','priceonlysignalsnotoriginalA','notPITuniversecorrection','pre2023TPEXhistorynotindependentlycrossverified'],'conclusion':'未證實穩定改善','automation':'preservepaused'}
statepath_external_v12.write_text(json.dumps(state150_v13,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
with workflowpath_external_v12.open('a',encoding='utf-8') as f:f.write('\n\n## v13 補上50檔上櫃（2026-10-15）\n已完成目前150檔子研究。新增50價格與50股利系列，100請求成功；24,942筆DB收盤相符、成交量口徑不同，官方單日45檔價量金額全相符；原100檔30案例控制不變。3配置×10年，21有效／9未知。比較必須列費後勝率、24日期望值及有效年度，不冒充歷史當時150檔，不混舊650窗口。詳見「'+report_path150_v13.name+'」。未證實改善；未重新啟動排程。\n')
print('V13_REPORT_SAVED',report_path150_v13.name,chart_path150_v13.name)


In [ ]:
# Checks on actual arrays and generated layouts, without restarting the retained kernel.
assert len(set(full_ids_v13))==150 and len(set(tpex_ids_v13))==50
assert len(tpex_history_v13['prices'])==50 and len(tpex_history_v13['dividends'])==50 and not tpex_history_v13['errors']
assert sum(len(t.y) for t in fig150_v13.data if t.type=='scatter' and t.mode=='markers')==42
assert sum(len(t.y) for t in fig150_v13.data if t.type=='scatter' and t.mode=='markers' and t.visible)==21
assert len(fig150_v13.data)==18
assert sum(t.mode=='lines' and t.line.color=='#dc2477' for t in fig150_v13.data if t.type=='scatter')==6
buttons=fig150_v13.layout.updatemenus[0].buttons
assert list(buttons[0].args[0]['visible'])==[True]*9+[False]*9
assert list(buttons[1].args[0]['visible'])==[False]*9+[True]*9
for figure in [fig150_v13,figyears_v13]:
    assert tuple(figure.layout.yaxis.range)==tuple(range_v13)
    assert any(s.y0==0 and s.y1==0 for s in figure.layout.shapes)
for name,rows in full_results_v13.items():
    m=metrics150_v13[name]
    assert m['n']==7 and m['invalid_years']==[2017,2018,2025]
    assert np.isclose(m['expectancy'],m['win_rate']*m['average_win']+(1-m['win_rate'])*m['average_loss'])
    for row in rows:
        assert (row['return'] is None)==bool(row['invalid'] or row['violations'])
        if row['return'] is not None:
            assert np.isclose(row['return'],row['path'][-1]/INITIAL-1)
            for trace in row['trace']:assert len(trace['selected'])==25 and set(trace['selected'])<=set(full_ids_v13)
assert chart_path150_v13.stat().st_size>1000000
verification150_v13={'data_series':{'added_price':50,'added_dividend':50,'current_pool':150},'annual_points':42,'default_full150_points':21,'unknown_annual_cases':9,'control_checks':30,'fixed_axes_switch':True,'median_lines_distinct':6,'zero_lines':2,'expectancy_identity':True,'path_return_identity':True,'verification':'programmatic arrays/layout and serialized artifact checks, not browser visual QA','kernel_restarted':False}
riskrows_v13=[]
for name in policies_v12:
    a=metrics100_v13[name];b=metrics150_v13[name];d=paired150_v13[name]
    riskrows_v13.append([name,f"{round(d['beat100_rate']*d['n'])}/{d['n']}",pct_external_v12(a['worst_return']),pct_external_v12(b['worst_return']),pct_external_v12(a['worst_mdd']),pct_external_v12(b['worst_mdd'])])
risk_md_v13='\n## 風險與跨年一致性\n'+mdtable_v13(['配置','150勝過100年度','100最差報酬','150最差報酬','100最差最大回撤','150最差最大回撤'],riskrows_v13)+'\n150檔平均期望值雖較高，但三種配置都只在7個有效年度中的3年勝過100檔；最差報酬與最大回撤也更差，不能只依平均值說納入上櫃較好。\n\n'
report150_v13=report_path150_v13.read_text(encoding='utf-8').replace('## 如何解讀',risk_md_v13+'## 如何解讀',1).replace('cells241–246','cells241–247')
report_path150_v13.write_text(report150_v13,encoding='utf-8')
html150_v13=chart_path150_v13.read_text(encoding='utf-8').replace('<h2>歷史資料覆蓋</h2>','<h2>風險與跨年一致性</h2><div class="scroll">'+table_external_v12(['配置','150勝過100年度','100最差報酬','150最差報酬','100最差回撤','150最差回撤'],riskrows_v13)+'</div><p>三種配置都只有3／7年勝過100檔，且150檔最差報酬與最大回撤更差；平均值較高不代表穩定改善。</p><h2>歷史資料覆蓋</h2>')
chart_path150_v13.write_text(html150_v13,encoding='utf-8')
old_v12_report=report_path_v12.read_text(encoding='utf-8')
if not old_v12_report.startswith('> 後續已補上櫃50檔'):
    report_path_v12.write_text('> 後續已補上櫃50檔，完整現況150檔比較見 ['+report_path150_v13.name+']('+report_path150_v13.name+')。本v12保留作100檔子研究紀錄。\n\n'+old_v12_report,encoding='utf-8')
state150_v13['full150_extension_v13']['verification']=verification150_v13
state150_v13['full150_extension_v13']['notebook_cells']=[241,247]
statepath_external_v12.write_text(json.dumps(state150_v13,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('V13_FINAL_VERIFIED',json.dumps(verification150_v13,ensure_ascii=False))


In [ ]:
# Descriptive market groups; no new selection strategy or data request.
print('GROUP_CONTEXT',{'current_TWSE':len(fixed_twse_v12),'current_TPEX':len(tpex_ids_v13),'cached_annual_cases':len(case_arrays_v13)})
market_group_rows_v13=[]
for year,base in long_cases_v12.items():
    arrays=case_arrays_v13[f'full150_{year}'];start=base['start_index'];end=start+23
    for group,ids in [('上市100',set(fixed_twse_v12)),('上櫃50',set(tpex_ids_v13))]:
        price_returns=[];cash_returns=[];excluded=[]
        for j,sid in enumerate(all_external_ids_v13):
            if sid not in ids:continue
            closes=arrays['close'][start-1:end+1,j]
            if not np.all(np.isfinite(closes)&(closes>0)):
                excluded.append({'sid':sid,'reason':'missing_prior_or_period_close'});continue
            if np.any(arrays['bad'][start:end+1,j]):
                excluded.append({'sid':sid,'reason':'unresolved_non_cash_or_reference_action'});continue
            before=closes[0];after=closes[-1];cash_received=float(np.sum(arrays['cashdiv'][start:end+1,j]));price_returns.append(after/before-1);cash_returns.append((after+cash_received)/before-1)
        price_ret=np.array(price_returns);cash_ret=np.array(cash_returns)
        result={'year':year,'group':group,'start':base['start'],'end':base['end'],'available':len(price_returns),'pool_count':len(ids),'excluded':excluded,'mean_price_return':float(np.mean(price_ret)) if len(price_ret) else None,'price_up':int(np.sum(price_ret>1e-12)),'price_down':int(np.sum(price_ret<-1e-12)),'price_flat':int(np.sum(np.abs(price_ret)<=1e-12)),'price_up_fraction':float(np.mean(price_ret>1e-12)) if len(price_ret) else None,'mean_cash_total_return':float(np.mean(cash_ret)) if len(cash_ret) else None,'cash_total_up_fraction':float(np.mean(cash_ret>1e-12)) if len(cash_ret) else None}
        market_group_rows_v13.append(result)
print('GROUP_ANNUAL_RESULTS',json.dumps([{k:v for k,v in row.items() if k!='excluded'} for row in market_group_rows_v13],ensure_ascii=False))
for group in ['上市100','上櫃50']:
    rows=[r for r in market_group_rows_v13 if r['group']==group]
    print('GROUP_YEAR_AVERAGE',group,json.dumps({'years':len(rows),'positive_mean_price_years':sum(r['mean_price_return']>0 for r in rows),'mean_annual_price_return':float(np.mean([r['mean_price_return'] for r in rows])),'mean_annual_cash_total_return':float(np.mean([r['mean_cash_total_return'] for r in rows])),'warning':'current fixed universe bias; equal weight available stocks, varying coverage; no fees or signal trading; not official TWSE/TPEx index'},ensure_ascii=False))
assert len(market_group_rows_v13)==20
for row in market_group_rows_v13:assert row['price_up']+row['price_down']+row['price_flat']==row['available'] and row['available']+len(row['excluded'])==row['pool_count']


In [ ]:
# Diagnose existing invalid cases with identical orders; never turn unknown into valid.
import inspect
source_diag_v13=inspect.getsource(simulate_external_v13) if False else None
# IPython inspect is unnecessary: get the saved engine source from its persisted cell.
import nbformat as nb_diag_v13
notebook_diag_v13=nb_diag_v13.read(recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
engine_diag_v13=notebook_diag_v13.cells[243].source.split('def simulate_external_v13(',1)[1].split('# Cross-source validation specifically',1)[0]
engine_diag_v13='def diagnose_external_v13('+engine_diag_v13
engine_diag_v13=engine_diag_v13.replace("if np.any(bad[t]&(before>0)):reasons.add('unmodeled_or_ambiguous_company_action_while_held')", "if np.any(bad[t]&(before>0)):\n            reasons.add('unmodeled_or_ambiguous_company_action_while_held')\n            diagnostic_events_v13.extend({'type':'company_action','date':str(days[t]),'sid':all_external_ids_v13[j],'previous_close':float(cl[t-1,j]),'close':float(cl[t,j]),'known_cash':float(cashdiv[t,j]),'unknown_reference':[e for e in arrays['unknown_reference_events'] if e['sid']==all_external_ids_v13[j] and e['date']==str(days[t])]} for j in np.flatnonzero(bad[t]&(before>0)))")
engine_diag_v13=engine_diag_v13.replace("if missing:reasons.add('invalid_execution_price')", "if missing:\n                reasons.add('invalid_execution_price')\n                diagnostic_events_v13.extend({'type':'invalid_execution_price','date':str(days[t]),'sid':all_external_ids_v13[j],'execution':float(arrays['execution'][t,j]) if np.isfinite(arrays['execution'][t,j]) else None,'source':price_lookup_v13.get(all_external_ids_v13[j],{}).get(str(days[t]))} for j in np.flatnonzero((target!=before)&~arrays['available'][t]))")
engine_diag_v13=engine_diag_v13.replace("if np.any((shares>0)&~np.isfinite(cl[t])):reasons.add('missing_held_close')", "if np.any((shares>0)&~np.isfinite(cl[t])):\n            reasons.add('missing_held_close')\n            diagnostic_events_v13.extend({'type':'missing_held_close','date':str(days[t]),'sid':all_external_ids_v13[j]} for j in np.flatnonzero((shares>0)&~np.isfinite(cl[t])))")
exec(engine_diag_v13)
invalid_explanations_v13={}
for year in [2017,2018,2025]:
    diagnostic_events_v13=[];case=dict(long_cases_v12[year],key=f'full150_{year}')
    checked=diagnose_external_v13(case,set(full_ids_v13),'equal',0.)
    original=next(row for row in full_results_v13['等權完整調整'] if row['year']==year)
    assert checked['reasons']==original['reasons'] and checked['violations']==original['violations']
    for event in diagnostic_events_v13:
        sid=event['sid'];day=event['date'];event['stock_name']=next((row.get('name') for row in current_market_rows_v12 if str(row['stock_id'])==sid),None)
        if event['type']=='company_action':
            event['provider_nonstandard_events']=[{'dataset':dataset,'record':record} for dataset,rows in provider_history_v13['global_events'].items() for record in rows if str(record.get('stock_id'))==sid and str(valid_event_day_v13(record.get('date')))==day]
            event['stock_dividend_records']=[record for record in provider_history_v13['dividends'].get(sid,[]) if str(valid_event_day_v13(record.get('StockExDividendTradingDate')))==day]
    invalid_explanations_v13[year]=diagnostic_events_v13
print('V13_INVALID_YEAR_CAUSES',json.dumps(invalid_explanations_v13,ensure_ascii=False,default=str))


In [ ]:
# Explain retained uncertainty in the existing report/chart without changing results.
detail_note_v13='''<h2>為什麼三年不可判定？</h2><p>以下是完整150檔「等權完整調整」的實際失敗紀錄；其餘配置同年也未通過檢查。不是整年沒有資料，也不是把虧損年刪掉。</p><ul><li>2017-11-15，6548長科*：持有期間遇到未解參考價變動；股利政策資料同日附有現金增資線索。現有現金股利模型未處理認購權及除權基準，無法可信計算損益。</li><li>2018-11-06，2886兆豐金：成交金額÷成交股數估出的25.847元低於日內最低26.05元，不能直接當一般盤中成交價；不代表原始行情必然錯誤。</li><li>2018-11-29、11-30，2404漢唐：供應商快取缺持有股票的收盤報價，無法完整評價。未確認原因前不宣稱停牌。</li><li>2025-11-27，1815富喬：持有期間遇到未解參考價變動，資料同日附有現金增資線索；認購權及除權基準尚未建模。</li></ul><p>參考價異常是由收盤價與供應商漲跌欄位推算的檢查訊號，不直接等同已確認的官方除權參考價。現金增資不能當現金股利補入。因現有模型或必要資料未齊，整個24日窗口保持未知，不填零、不把模型算出的不完整損益列入勝率。</p>'''
chart_current_v13=chart_path150_v13.read_text(encoding='utf-8')
if '<h2>為什麼三年不可判定？</h2>' not in chart_current_v13:chart_path150_v13.write_text(chart_current_v13.replace('<h2>風險與跨年一致性</h2>',detail_note_v13+'<h2>風險與跨年一致性</h2>',1),encoding='utf-8')
md_detail_v13='\n## 三年不可判定的具體原因（完整150檔等權案例）\n- 2017-11-15 長科*6548：持倉未解參考價變動，同日有現增線索；模型未處理認購權及除權基準。\n- 2018-11-06 兆豐金2886：估計VWAP25.847元低於當日最低26.05元，估計成交口徑未過關，不等於原始行情一定錯。\n- 2018-11-29、11-30 漢唐2404：快取缺持股收盤，無法完整評價，未確認缺漏原因，不宣稱停牌。\n- 2025-11-27 富喬1815：持倉未解參考價變動，同日有現增線索；模型未處理認購權及除權基準。\n參考價變動由收盤價及供應商漲跌欄位推算，不能直接當官方已確認除權基準。未造股利或認購權價值，維持整窗口未知。這是模型／資料缺口，不代表那一年虧損；缺漏可能非隨機，所以7年統計不能冒充完整十年。細節保存在原Notebook cell249，cell250更新說明，未改績效。\n\n'
report_current_v13=report_path150_v13.read_text(encoding='utf-8')
if '## 三年不可判定的具體原因' not in report_current_v13:report_path150_v13.write_text(report_current_v13.replace('## 如何解讀',md_detail_v13+'## 如何解讀',1),encoding='utf-8')
print('INVALID_DETAIL_ADDED','same performance values; no data refetch or changed eligibility')


In [ ]:
# v14 read-only source and retained-kernel audit; no reset or duplicated bulk fetch.
print('V14_CONTEXT',{'kernel':__import__('sys').executable,'full150_cases':sum(len(rows) for rows in full_results_v13.values()),'source_prices':len(provider_history_v13['prices'])})
import importlib.util
print('PDF_MODULES',{name:importlib.util.find_spec(name) is not None for name in ['pypdf','fitz','pdfplumber']})
repair_raw_v14={};repair_meta_v14={}
for key,url in [('uisco_investor','https://www.uisco.com.tw/left.htm'),('cwtc_financial','https://www.cwtcglobal.com/財務報告')]:
    try:
        response=requests.get(url,timeout=25);response.encoding=response.apparent_encoding;body=response.text
        repair_raw_v14[key]=body;repair_meta_v14[key]={'url':response.url,'status':response.status_code,'sha256':hashlib.sha256(response.content).hexdigest(),'bytes':len(response.content)}
        soup=__import__('bs4').BeautifulSoup(body,'html.parser')
        links=[{'text':a.get_text(' ',strip=True),'href':a.get('href')} for a in soup.find_all('a',href=True) if (key=='uisco_investor' and any(x in a.get_text(' ',strip=True)+a.get('href','') for x in ['107','2018','108','2019','year'])) or (key=='cwtc_financial' and ('2017' in a.get_text(' ',strip=True)+a.get('href','') or '106' in a.get('href','')))]
        print('V14_SOURCE_LINKS',key,json.dumps(links[:20],ensure_ascii=False))
    except Exception as exc:repair_meta_v14[key]={'url':url,'error':type(exc).__name__};print('V14_SOURCE_ERROR',key,type(exc).__name__)
for year in [2017,2018,2025]:
    for name,(allocation,tolerance) in policies_v12.items():
        diagnostic_events_v13=[];case=dict(long_cases_v12[year],key=f'full150_{year}');checked=diagnose_external_v13(case,set(full_ids_v13),allocation,tolerance)
        print('V14_ALL_POLICY_FAILURES',year,name,[{k:event[k] for k in ['type','date','sid']} for event in diagnostic_events_v13])


In [ ]:
# Authorized recovery after oldkernel ended; restore only saved inputs needed for v14.
from research_layout import ResearchPath as Path
import sys,json,gzip,datetime,bisect,re,hashlib,collections,ast,requests,io,calendar,concurrent.futures,time
import numpy as np
import nbformat
recovery_root=Path('C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料')
notebook_path_v14=recovery_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb'
saved_nb_v14=nbformat.read(notebook_path_v14,as_version=4)
INITIAL=1e9;FEE=.001425;TAX=.003
restored_functions_v14=[]
for wanted in ['settle','percentile','number_v12']:
    chosen=None
    for cell in saved_nb_v14.cells:
        if cell.cell_type!='code' or f'def {wanted}(' not in cell.source:continue
        try:tree=ast.parse(cell.source)
        except SyntaxError:continue
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name==wanted:chosen=node
    if chosen is None:
        author_saved_v14=Path('C:/Users/weiha/Documents/Codex/2026-10-09/data-mining-final-project/work/momentum_add_backtests.py').read_text(encoding='utf-8')
        for literal in ast.walk(ast.parse(author_saved_v14)):
            if not isinstance(literal,ast.Constant) or not isinstance(literal.value,str) or f'def {wanted}(' not in literal.value:continue
            for node in ast.walk(ast.parse(literal.value)):
                if isinstance(node,ast.FunctionDef) and node.name==wanted:chosen=node
    assert chosen is not None,wanted
    exec(compile(ast.fix_missing_locations(ast.Module(body=[chosen],type_ignores=[])),'<saved-notebook-function>','exec'),globals());restored_functions_v14.append(wanted)
def load_gzip_v14(name):
    with gzip.open(recovery_root/name,'rt',encoding='utf-8') as f:return json.load(f)
provider_old_v14=load_gzip_v14('外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz');tpex_old_v14=load_gzip_v14('完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz')
provider_history_v13={kind:{**provider_old_v14[kind],**tpex_old_v14[kind]} for kind in ['prices','dividends','metadata','errors']};provider_history_v13['global_events']=provider_old_v14['global_events']
provider_tickers_v13=sorted(provider_history_v13['prices']);price_lookup_v13={sid:{r['date']:r for r in rows} for sid,rows in provider_history_v13['prices'].items()}
definition_v13=json.loads((recovery_root/'完整150檔研究_預先定義_v13_2026-10-15.json').read_text(encoding='utf-8'))
tpex_ids_v13=definition_v13['tpex50'];full_ids_v13=definition_v13['current_universe'];fixed_twse_v12=set(full_ids_v13)-set(tpex_ids_v13)
full_export_v13=json.loads((recovery_root/'完整150檔研究_同期回測與100檔配對_v13_2026-10-15.json').read_text(encoding='utf-8'));full_results_v13=full_export_v13['full150'];control_results_v13=full_export_v13['same100_control']
policies_v12={name:tuple(value) for name,value in definition_v13['policies'].items()}
cal_source_v12=json.loads((recovery_root/'產業競賽期間_官方指數來源_2026-10-14.json').read_text(encoding='utf-8'))
def roc_day_v14(value):
    y,m,d=map(int,re.findall(r'\d+',str(value))[:3]);return datetime.date(y+1911 if y<1911 else y,m,d)
calendar_days_v13=sorted({roc_day_v14(row[0]) for doc in cal_source_v12['months'].values() if 'rows' in doc for row in doc['rows']})
long_cases_v12={}
for year in range(2016,2026):
    endmonth=datetime.date(year,10,31);anchor=endmonth-datetime.timedelta(days=(endmonth.weekday()-1)%7);ix=bisect.bisect_left(calendar_days_v13,anchor);span=calendar_days_v13[ix-61:ix+24];assert len(span)==85
    long_cases_v12[year]={'year':year,'anchor':str(anchor),'start':str(span[61]),'end':str(span[-1]),'previous':str(span[60]),'dates':span,'start_index':61}
number_v13=number_v12
saved_engine_v14=saved_nb_v14.cells[243].source
cash_code_v14=saved_engine_v14.split('all_external_ids_v13=',1)[1].split('def build_case_v13(',1)[0]
exec('all_external_ids_v13='+cash_code_v14,globals())
for node in ast.parse(saved_engine_v14).body:
    if isinstance(node,ast.FunctionDef) and node.name in ['build_case_v13','simulate_external_v13']:exec(compile(ast.fix_missing_locations(ast.Module(body=[node],type_ignores=[])),'<saved-v13-engine>','exec'),globals())
for year,base in long_cases_v12.items():
    case=dict(base,key=f'full150_{year}');case_arrays_v13[case['key']]=build_case_v13(case)
baseline_recovery_checks_v14=0
for name,(allocation,tolerance) in policies_v12.items():
    case=dict(long_cases_v12[2024],key='full150_2024');row=simulate_external_v13(case,set(full_ids_v13),allocation,tolerance);old=next(r for r in full_results_v13[name] if r['year']==2024)
    assert np.isclose(row['return'],old['return'],rtol=0,atol=1e-12);baseline_recovery_checks_v14+=1
print('V14_RESTORED',{'python':sys.executable,'pid':__import__('os').getpid(),'tickers':len(provider_tickers_v13),'current150':len(full_ids_v13),'cases':len(case_arrays_v13),'baseline2024checks':baseline_recovery_checks_v14,'restored_functions':restored_functions_v14,'requests_sent':0})


In [ ]:
import sys,traceback
error_v14=getattr(sys,'last_value',None)
print('REPAIR_ERROR',type(error_v14).__name__,str(error_v14)[:150])
tb_v14=getattr(sys,'last_traceback',None)
print('REPAIR_TRACE_LINES',[(frame.name,frame.lineno) for frame in traceback.extract_tb(tb_v14)[-3:]])
print('REPAIR_SAFE_STATE',{'functions':globals().get('restored_functions_v14',[]),'cash_arrays':'case_arrays_v13' in globals()})


In [ ]:
# Primary issuer documents and contest rules; sources only, no new return assumptions yet.
import importlib.util
print('V14_PDF_ENV',{name:importlib.util.find_spec(name) is not None for name in ['pypdf','fitz']})
repair_raw_v14={};repair_meta_v14={};repair_text_v14={}
repair_specs_v14=[('cwtc2017','https://www.cwtcglobal.com/archive/3-Investor/annual-report/投資人專區-股東專欄-年報(股東會後修訂本)-106年.pdf'),('uisco2018','https://www.uisco.com.tw/107year.pdf'),('tpex_mark_rule','https://www.tpex.org.tw/storage/stock_index/企業活動及企業事件處理原則.pdf')]
for key,url in repair_specs_v14:
    try:
        response=requests.get(url,timeout=30);valid=response.content.startswith(b'%PDF')
        repair_meta_v14[key]={'url':response.url,'status':response.status_code,'is_pdf':valid,'bytes':len(response.content),'sha256':hashlib.sha256(response.content).hexdigest()}
        if not valid:print('V14_PRIMARY_NOT_PDF',key,repair_meta_v14[key]);continue
        repair_raw_v14[key]=response.content
        if importlib.util.find_spec('pypdf'):
            from pypdf import PdfReader
            pages=[page.extract_text() or '' for page in PdfReader(io.BytesIO(response.content)).pages]
        else:
            import fitz
            pages=[page.get_text() for page in fitz.open(stream=response.content,filetype='pdf')]
        repair_text_v14[key]=pages
        targets=['11月29','11 月 29','11月15','11 月 15','現金增資發行普通股6,000','減資退','每股400','每股 400','保留市值']
        hits=[]
        for pidx,txt in enumerate(pages):
            for word in targets:
                hit=txt.find(word)
                if hit>=0:hits.append({'page':pidx+1,'keyword':word,'excerpt':txt[max(0,hit-100):hit+220]})
        print('V14_PRIMARY_PDF',key,'pages',len(pages),'hits',json.dumps(hits[:8],ensure_ascii=False))
    except Exception as exc:repair_meta_v14[key]={'url':url,'error':type(exc).__name__};print('V14_PRIMARY_ERROR',key,type(exc).__name__)
print('V14_2404_PRICE_BOUNDARY',[price_lookup_v13['2404'].get(day) for day in ['2018-11-28','2018-11-29','2018-11-30','2018-12-10']])
rules_text_v14='\n'.join((recovery_root/'官方文件'/name).read_text(encoding='utf-8') for name in ['D-Plan_撰寫指南_v4.3.md','DataPack_規格書_v4.3.md'])
print('V14_CONTEST_CORPORATE_ACTION_RULE_WORDS',{word:rules_text_v14.count(word) for word in ['現金增資','停牌','停止買賣','減資','認購']})


In [ ]:
# Read already downloaded issuer PDFs using bundled pure-Python pypdf; no refetch/install.
sys.path.append('C:/Users/weiha/.cache/codex-runtimes/codex-primary-runtime/dependencies/python/Lib/site-packages')
from pypdf import PdfReader
repair_hits_v14={}
for key,body in repair_raw_v14.items():
    pages=[page.extract_text() or '' for page in PdfReader(io.BytesIO(body)).pages];repair_text_v14[key]=pages
    hits=[]
    for pidx,txt in enumerate(pages):
        normalized=re.sub(r'\s+','',txt)
        targets={'uisco2018':['11月29','每仟股','每壹仟股','減資20','減資退還','476,466'],'cwtc2017':['每股400','2,400,000','現金增資發行普通股6,000','11月21'],'tpex_mark_rule':['保留市值','停止買賣']}.get(key,[])
        for word in targets:
            pos=normalized.find(word)
            if pos>=0:hits.append({'page':pidx+1,'word':word,'excerpt':normalized[max(0,pos-50):pos+190]})
    repair_hits_v14[key]=hits;repair_meta_v14[key]={'url':dict(repair_specs_v14)[key],'is_pdf':True,'bytes':len(body),'sha256':hashlib.sha256(body).hexdigest(),'pages':len(pages),'extractor':'bundled pypdf6.10.0 in retained Python314kernel'}
    print('V14_CACHED_PDF_EVIDENCE',key,json.dumps(hits[:6],ensure_ascii=False))
print('V14_PRIMARY_EXTRACTION_SUCCESS',list(repair_text_v14))


In [ ]:
# v14 predetermined corrections and one declared accounting sensitivity, before outcomes.
repair_definition_v14={'version':'v14_execution_and_suspension_corrections','universe':definition_v13['current_universe'],'selection':'unchangedv13price20/60momentum;25stocks,90pct,rebalance5,lot1000;threeexistingallocations','fee':FEE,'sell_tax':TAX,'execution_correction':'finitepositiveTrading_money/Trading_Volume;do_not_require_aggregate_VWAP_inside_regular_session_highlow;appliesallstocksallyears','execution_limit':'aggregateprovideraverageapproximatesofficialcontestformula,butexactofficialDataPacknotverified','suspension':{'sid':'2404','start':'2018-11-29','end':'2018-12-07','last_trade':'2018-11-28','marking':'preservelasttradedstockmarketvalue,sellbuyblocked;rawclosesremainmissing','no_early_action':'reduction80pctsharesand2NTDrefundNOTappliedbefore2018-12-10;outside24daywindowending2018-11-30','status':'historicalvaluationconventioninspiredbyofficialindexprinciple,notcertifiedcontestrule','announcement_reference':'issuerannouncementreprinted2018-11-16 https://news.cnyes.com/news/id/4238624;issuer2018annualreportPDFp44confirms2018-12-10newshares'} ,'primary':'unknown_paid_rights_still_invalid_whenheld','sensitivity':'waive_all_verifiedpaid_subscription_rights;no newshares,no cashoutflow,no rightsvalue;not officialcontestassumption','paid_event_gate':'positivecashcapital,zero cashdividend,zero free stock;announcement datebefore exdate;no otherknownnonstandardevent;appliesall qualifying eventrecords,not just two observed years','no_future_filter':'pastbadlookbackeligibilityunchanged;neverremove astockbecauseitsfutureeventknownnow','data_change':'no modified rawclose,spread,volume,money;separatemarkarray onlyforverifiedsuspension','no_forced_selection':'no rankingchanges or bestparametersearch','metrics':'annualvalidcount,winrate,mean24dnetreturn,fees,mdd;pairedsamevalidyears;preserveunknown','comparison':'v13 oldnumbersretained;correctionandconditional10yearscenarioreportedseparately'}
repair_definition_sha_v14=hashlib.sha256(json.dumps(repair_definition_v14,sort_keys=True).encode()).hexdigest()
(recovery_root/'公司行動與成交修正_預先定義_v14_2026-10-15.json').write_text(json.dumps(repair_definition_v14,ensure_ascii=False,indent=2),encoding='utf-8')
repair_source_urls_v14={'TWSE_aggregate_coverage':'https://eshop.twse.com.tw/zh/product/detail/cfec9a1470e448ec91bfde006db361e8','TPEX_exright_rule':'https://www.tpex.org.tw/zh-tw/announce/market/ex/cal.html','UISCO_suspension_issuer_announcement_reprint':'https://news.cnyes.com/news/id/4238624','FFG_issuer_announcement_reprint':'https://www.nstock.tw/news/article_m?id=389226','contest_source':'競賽規則與每日運作說明_2026-10-13.md'}
for key,body in repair_raw_v14.items():
    target=recovery_root/f'公司行動修正_來源_{key}_v14_2026-10-15.pdf';target.write_bytes(body);repair_meta_v14[key]['saved']=target.name
(recovery_root/'公司行動與成交修正_來源查核_v14_2026-10-15.json').write_text(json.dumps({'primary_PDFs':repair_meta_v14,'primary_excerpts':repair_hits_v14,'other_sources':repair_source_urls_v14,'source_first_release_warning':'annualreportsproveeventexpost;notaddedasdecisiontimefeatures','kernel_restored_from_cache':True,'old_kernel_dead_not_restarted':'newkernel-aicup-repair-20261015;otherprojectuntouched'},ensure_ascii=False,indent=2),encoding='utf-8')
print('V14_REGISTERED',repair_definition_sha_v14)


In [ ]:
# Apply only documented corrections; preserve raw prices and original signal eligibility.
paid_known_days_v14=set();paid_records_v14=[]
other_action_days_v14={(str(record.get('stock_id')),str(record.get('date'))) for rows in provider_history_v13['global_events'].values() for record in rows}
for sid,rows in provider_history_v13['dividends'].items():
    for record in rows:
        cap=number_v12(record.get('TotalNumberOfCashCapitalIncrease',0));stock=sum(number_v12(record.get(k,0)) for k in ['StockEarningsDistribution','StockStatutorySurplus']);cash=sum(number_v12(record.get(k,0)) for k in ['CashEarningsDistribution','CashStatutorySurplus']);day=valid_event_day_v13(record.get('StockExDividendTradingDate'))
        try:ann=datetime.date.fromisoformat(str(record.get('AnnouncementDate'))[:10])
        except (ValueError,TypeError):continue
        if not (cap>0 and stock==0 and cash==0 and day is not None and ann<day and (sid,str(day)) not in other_action_days_v14):continue
        paid_known_days_v14.add((sid,day));paid_records_v14.append({'sid':sid,'date':str(day),'announced':str(ann),'cash_capital_shares':cap,'rate_field':record.get('CashIncreaseSubscriptionRate'),'price':record.get('CashIncreaseSubscriptionpRrice')})
case_arrays_v14={};execution_changes_v14=[];stale_marks_v14=[]
for key,old in case_arrays_v13.items():
    arrays={k:(v.copy() if isinstance(v,np.ndarray) else v) for k,v in old.items()};arrays['mark']=arrays['close'].copy();arrays['paid_known']=np.zeros_like(arrays['bad'],dtype=bool)
    arrays['available']=np.isfinite(arrays['execution'])&(arrays['execution']>0)&np.isfinite(arrays['volume'])&(arrays['volume']>0)
    for t,j in zip(*np.where(arrays['available']&~old['available'])):
        execution_changes_v14.append({'case':key,'date':str(arrays['dates'][t]),'sid':all_external_ids_v13[j],'aggregate_vwap':float(arrays['execution'][t,j])})
    for t,day in enumerate(arrays['dates']):
        for j,sid in enumerate(all_external_ids_v13):
            arrays['paid_known'][t,j]=(sid,day) in paid_known_days_v14
        if datetime.date(2018,11,29)<=day<=datetime.date(2018,12,7):
            j=ext_si_v13['2404'];last=price_lookup_v13['2404'].get('2018-11-28');assert last and last['close']==63.9
            assert not np.isfinite(arrays['close'][t,j]),'Unexpected actualquote during documented suspension; inspect'
            arrays['mark'][t,j]=last['close'];arrays['available'][t,j]=False
            stale_marks_v14.append({'case':key,'date':str(day),'sid':'2404','valuation_mark':63.9,'raw_close':None,'tradeable':False})
    case_arrays_v14[key]=arrays
engine_node_v14=next(node for node in ast.parse(saved_engine_v14).body if isinstance(node,ast.FunctionDef) and node.name=='simulate_external_v13')
engine_source_v14=ast.unparse(engine_node_v14).replace('def simulate_external_v13(case, pool_ids, allocation, tolerance):',"def simulate_repaired_v14(case, pool_ids, allocation, tolerance, paid_policy='unresolved'):")
engine_source_v14=engine_source_v14.replace('case_arrays_v13[','case_arrays_v14[').replace("cl = arrays['close']","cl = arrays['close']\n    mark = arrays['mark']")
engine_source_v14=engine_source_v14.replace('bad[t] & (before > 0)',"bad[t] & (before > 0) & (~arrays['paid_known'][t] if paid_policy == 'waive' else True)")
engine_source_v14=engine_source_v14.replace('np.nan_to_num(cl[t - 1])','np.nan_to_num(mark[t - 1])').replace('np.nan_to_num(cl[t])','np.nan_to_num(mark[t])').replace('~np.isfinite(cl[t])','~np.isfinite(mark[t])')
engine_source_v14=engine_source_v14.replace('for offset, t in enumerate(range(start, start + 24)):',"assumption_events = []\n    for offset, t in enumerate(range(start, start + 24)):")
engine_source_v14=engine_source_v14.replace('before = shares.copy()',"before = shares.copy()\n        if paid_policy == 'waive':\n            assumption_events.extend({'sid': all_external_ids_v13[j], 'date': str(days[t]), 'assumption': 'waive_paid_subscription_no_rights_asset_no_free_shares'} for j in np.flatnonzero(arrays['paid_known'][t] & bad[t] & (before > 0)))")
engine_source_v14=engine_source_v14.replace("return {'case': case['key']", "return {'paid_policy':paid_policy, 'assumption_events':assumption_events, 'case': case['key']")
assert 'def simulate_repaired_v14' in engine_source_v14 and "~arrays['paid_known'][t]" in engine_source_v14
exec(engine_source_v14,globals())
strict_repair_results_v14={name:[] for name in policies_v12};conditional_repair_results_v14={name:[] for name in policies_v12};repair_control_reproduced_v14=0
for year,base in long_cases_v12.items():
    case=dict(base,key=f'full150_{year}')
    for name,(allocation,tolerance) in policies_v12.items():
        old=next(r for r in full_results_v13[name] if r['year']==year);control=simulate_external_v13(case,set(full_ids_v13),allocation,tolerance)
        assert control['reasons']==old['reasons'] and (control['return'] is None)==(old['return'] is None)
        if old['return'] is not None:assert np.isclose(control['return'],old['return'],rtol=0,atol=1e-12)
        repair_control_reproduced_v14+=1
        strict_repair_results_v14[name].append(simulate_repaired_v14(case,set(full_ids_v13),allocation,tolerance,'unresolved'))
        conditional_repair_results_v14[name].append(simulate_repaired_v14(case,set(full_ids_v13),allocation,tolerance,'waive'))
    print('V14_YEAR',year,{'main':[round(rows[-1]['return']*100,4) if rows[-1]['return'] is not None else rows[-1]['reasons'] for rows in strict_repair_results_v14.values()],'no_subscription_scenario':[round(rows[-1]['return']*100,4) if rows[-1]['return'] is not None else rows[-1]['reasons'] for rows in conditional_repair_results_v14.values()]},flush=True)
repair_export_v14={'definition':repair_definition_v14,'definition_sha256':repair_definition_sha_v14,'strict':strict_repair_results_v14,'conditional_no_subscription':conditional_repair_results_v14,'aggregate_execution_gate_changes':execution_changes_v14,'separate_stale_marks':stale_marks_v14,'verified_paid_policy_records':paid_records_v14,'source_metadata':repair_meta_v14,'control_reproduced':repair_control_reproduced_v14,'engine_source':engine_source_v14,'official_contest_certified':False}
(recovery_root/'公司行動與成交修正_回測結果_v14_2026-10-15.json').write_text(json.dumps(repair_export_v14,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('V14_CORRECTIONS_SAVED',{'controls':repair_control_reproduced_v14,'execution_gate_changes':len(execution_changes_v14),'explicit_suspension_marks':len(stale_marks_v14),'paid_event_records':len(paid_records_v14)})


In [ ]:
# Persist corrections separately from accounting assumptions and original v13 artifacts.
import html as html_v12
import plotly.graph_objects as go_v14
from plotly.offline import plot as plot_v14
for cellno,wanted in [(238,['stats_external_v12','pct_external_v12','table_external_v12']),(246,['mdtable_v13'])]:
    for node in ast.parse(saved_nb_v14.cells[cellno].source).body:
        if isinstance(node,ast.FunctionDef) and node.name in wanted:exec(compile(ast.fix_missing_locations(ast.Module(body=[node],type_ignores=[])),'<saved-report-functions>','exec'),globals())
strict_metrics_v14={name:stats_external_v12(rows) for name,rows in strict_repair_results_v14.items()};conditional_metrics_v14={name:stats_external_v12(rows) for name,rows in conditional_repair_results_v14.items()}
print('V14_MAIN_METRICS',json.dumps(strict_metrics_v14,ensure_ascii=False));print('V14_CONDITIONAL_METRICS',json.dumps(conditional_metrics_v14,ensure_ascii=False))
summary_v14={'corrected_with_suspension_valuation_convention':strict_metrics_v14,'conditional_waive_paid_rights':conditional_metrics_v14,'baseline_v13_cases_reproduced':30,'primary_still_unknown_years':[2017,2025],'restored_2018_note':'aggregateVWAPformula and lastmarketvalue suspension marking;notactualtradingquote','official_paid_subscription_rule':'notfoundinretainedofficialdocuments','conclusion':'2018 recoverable with explicit historical marking convention;2017/2025 computable only conditional on declared no-subscription/no-rights-value policy;notall10yearsofficiallycertified'}
(recovery_root/'公司行動與成交修正_統計摘要_v14_2026-10-15.json').write_text(json.dumps(summary_v14,ensure_ascii=False,indent=2),encoding='utf-8')
comparison_rows_v14=[]
for name in policies_v12:
    a=strict_metrics_v14[name];b=conditional_metrics_v14[name]
    comparison_rows_v14.append([name,f"{a['n']}/10",pct_external_v12(a['win_rate']),pct_external_v12(a['expectancy']),f"{b['n']}/10",pct_external_v12(b['win_rate']),pct_external_v12(b['expectancy'])])
comparison_headers_v14=['配置','修正口徑有效年','修正口徑勝率','修正口徑期望值','放棄認購情境年','情境勝率','情境期望值']
annual_rows_v14=[]
for ix,year in enumerate(range(2016,2026)):
    row=[str(year)]
    for name in policies_v12:row.extend([pct_external_v12(strict_repair_results_v14[name][ix]['return']),pct_external_v12(conditional_repair_results_v14[name][ix]['return'])])
    annual_rows_v14.append(row)
annual_headers_v14=['年度']+[name+'／'+mode for name in policies_v12 for mode in ['修正口徑','放棄認購']]
report_v14='''# 完整150檔：公司行動與成交修正 v14
2026-10-15。回應使用者要求查核2017／2018／2025是否能導正；保留v13，不追求把全部年份強制算成有效。

## 結論
2018可以在明示的歷史估值口徑下恢復計算：刪除不合本比賽均價定義的高低價邊界檢查；漢唐停止交易期間保留最後市值、不能交易，不填造收盤價。主表因此為8個有效年度，仍留2017／2025未知。
2017與2025是現金增資／認購權處理，不是漏了免費配股。另固定一個『不參與增資、不認列認購權價值』的情境，可得10年條件式結果；它不是已核實的主辦方結算方式，不與主表混算。

## 查核與修正
### 兆豐金2886，2018-11-06
先前以25.847元均價低於盤中最低26.05元，將窗口判無效。這個檢查過嚴：本比賽採成交金額÷成交股數，而官方收盤行情有一般、零股、盤後定價、鉅額等不同統計口徑，不能要求綜合成交均價必然落在一般盤中高低之間。已對全股票、全年度一致移除這個邊界，保留金額／量有限正值檢查，未把均價硬拉回26.05元。這是模擬口徑修正，不宣稱供應商或官方資料一定錯。
資料仍非主辦方正式DataPack，因此不能保證綜合價量就是主辦方最終指定的逐股資料版本；來源單位及前期官方重疊核對仍適用。
### 漢唐2404，2018-11-29起
公司公告轉載明列2018-11-29至12-07停止交易；公司107年年報PDF第44頁確認減資換發新股於12-10上市，第80頁確認減資20%。原資料沒有11-29、11-30收盤，是停牌，不應憑空補成交。
歷史估值採官方指數企業事件處理中的保留市值原則：把11-28最後收盤63.9元保留作『估值標記』，原始收盤仍為缺值，成交仍禁止。此次24日窗口11-30已結束，所以沒有提前把12-10後的股數乘0.8或加每股2元現金。這是研究採用的停牌估值約定，不是官賽規則已認證。
### 長科6548，2017-11-15；富喬1815，2025-11-27
原股利政策提供現增與認購線索，長科公司106年年報PDF第8頁確認當年另有每股400元的現增。富喬公司公告轉載列每千股可認購91.55303926股、認購66元，繳款期在12月；這些不等於無償配股。
供應商除權當日漲跌欄為0時，用收盤減漲跌推算的參考價，不等於已核實的官方除權參考價。不能用它倒造股利或把認購權直接當現金。已讀官賽DPlan與DataPack文件未找到現增、減資、認購或停牌的明確帳務條款，所以主表保留未知。
情境表採不參與現增、權利不計價，股票股數不增加、不扣認購款、不添可交易現金；對所有符合資料門檻的現增事件套用同一約定，非只為修補兩個年度。原來60日回看中的未知公司行動篩選不變，沒有用未來事件提前避開股票。

## 費後勝率與期望值
150檔現況名單、原3配置、25檔持倉、90%投入、5日調整、整千股、手續費0.1425%及賣出稅0.3%都未改。勝率為有效年度24日費後正報酬比例；期望值為有效年度費後報酬算術平均。主表與情境有效年度不同，不可直接以平均差歸因策略改善。

'''+mdtable_v13(comparison_headers_v14,comparison_rows_v14)+'''
## 逐年全部點
'''+mdtable_v13(annual_headers_v14,annual_rows_v14)+'''
這是歷史資料與帳務定義修正，不是新選股優勢證據。原v13的30個案例均先成功重現；舊7個有效年度的報酬不因本次修正改變，新增2018依估值口徑計算。2017／2025的數字只在放棄認購情境成立。

## 來源
- [官方收盤行情的不同交易統計口徑](https://eshop.twse.com.tw/zh/product/detail/cfec9a1470e448ec91bfde006db361e8)
- [櫃買中心企業活動及企業事件處理原則，第4頁停牌保留市值](https://www.tpex.org.tw/storage/stock_index/企業活動及企業事件處理原則.pdf)
- [漢唐公司減資換股公告轉載，2018-11-16](https://news.cnyes.com/news/id/4238624)
- [漢唐107年年報，第44、80頁](https://www.uisco.com.tw/107year.pdf)
- [長科106年年報，第8頁](https://www.cwtcglobal.com/archive/3-Investor/annual-report/投資人專區-股東專欄-年報(股東會後修訂本)-106年.pdf)
- [富喬公司現增認股公告轉載](https://www.nstock.tw/news/article_m?id=389226)
- [櫃買中心除權息計算，理論參考價和開始交易基準價須區分](https://www.tpex.org.tw/zh-tw/announce/market/ex/cal.html)

年報是事後驗證事件的來源，沒有拿來冒充開賽前已知的選股訊號；公告時間與權利判斷另保存。來源PDF、SHA256、查核與回測JSON均保留。官方已拒絕的批次來源未重送，也未購買權限資料。

## 限制與復原紀錄
現況150檔回看歷史的選擇／存活偏誤、價格動能子研究未含原A法人資格、早年行情交叉驗證不足、官賽ActiveShare等限制未消失。不能由8或10個同季窗口選出已證實贏家。
舊研究kernel已在本次前結束；依先前恢復授權，使用新kernel-aicup-repair-20261015，從既有壓縮快取與原Notebook只恢復必要物件，3個2024案例及全部30個原案例成功重現。另一個專案kernel未動。沒有重抓整批資料或重啟既有kernel。Notebook cells252–258保存復原、來源、定義、計算與本報告；不宣稱已完成Restart and Run All。
'''
report_path_v14=recovery_root/'公司行動與成交修正_比較報告_v14_2026-10-15.md';report_path_v14.write_text(report_v14,encoding='utf-8')
fig_v14=go_v14.Figure();colors_v14=['#3974b4','#2a947a','#a77ac4'];all_returns_v14=[]
for poolidx,(label,results) in enumerate([('修正口徑（8年）',strict_repair_results_v14),('放棄認購情境（10年）',conditional_repair_results_v14)]):
    for ix,(name,rows) in enumerate(results.items()):
        valid=[r for r in rows if r['return'] is not None];ys=[r['return']*100 for r in valid];all_returns_v14.extend(ys);visible=poolidx==0
        fig_v14.add_trace(go_v14.Box(x=[ix]*len(ys),y=ys,boxpoints=False,hoverinfo='skip',fillcolor=colors_v14[ix],opacity=.25,line_color=colors_v14[ix],showlegend=False,visible=visible))
        fig_v14.add_trace(go_v14.Scatter(x=ix+np.linspace(-.13,.13,len(valid)),y=ys,mode='markers',marker={'size':10,'color':colors_v14[ix]},name=label+'／'+name,customdata=[[r['year'],r['start'],r['end'],r['cost']*100,('停牌保留市值估值' if r['year']==2018 else '不參與現增、不計權利價值' if poolidx==1 and r['year'] in [2017,2025] else '原口徑有效年度')] for r in valid],hovertemplate='%{customdata[0]}年<br>%{customdata[1]}～%{customdata[2]}<br>費後報酬 %{y:.3f}%<br>費稅／初始資金 %{customdata[3]:.3f}%<br>%{customdata[4]}<extra>'+label+'／'+name+'</extra>',visible=visible,showlegend=False))
        med=float(np.median(ys));fig_v14.add_trace(go_v14.Scatter(x=[ix-.2,ix+.2],y=[med,med],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False,visible=visible))
fig_v14.update_layout(template='plotly_white',height=500,margin={'l':65,'r':25,'t':90,'b':65},yaxis_title='24交易日費後報酬（%）',hovermode='closest',updatemenus=[{'type':'buttons','direction':'right','x':0,'y':1.18,'buttons':[{'label':'修正口徑8年（預設）','method':'update','args':[{'visible':[True]*9+[False]*9}]},{'label':'放棄認購情境10年','method':'update','args':[{'visible':[False]*9+[True]*9}]}]}])
fig_v14.update_xaxes(tickmode='array',tickvals=[0,1,2],ticktext=list(policies_v12),range=[-.5,2.5]);fig_v14.update_yaxes(range=[min(all_returns_v14)-1,max(all_returns_v14)+1],autorange=False);fig_v14.add_hline(y=0,line_width=2,line_dash='dash',line_color='#333')
html_v14='''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>完整150檔：修正口徑與條件情境 v14</title><style>body{font:16px/1.7 system-ui,sans-serif;color:#203043;background:#f4f7fa;margin:0}main{max-width:1200px;margin:24px auto;padding:24px;background:white;border-radius:14px}h1{font-size:26px}h2{font-size:21px}.note{background:#fff4dc;padding:14px;border-left:4px solid #d49423}.scroll{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}th,td{padding:9px;border-bottom:1px solid #dce3eb;text-align:right;white-space:nowrap}th:first-child,td:first-child{text-align:left}a{color:#2369a5}</style><main><h1>完整150檔：2018修正與現增情境 v14</h1><p class="note">主表採停牌期間保留最後市值的歷史估值約定，有效年度由7增至8；2017／2025仍未知。另有明確放棄現增認購、不計權利價值的10年條件式情境。兩者都未獲官賽帳務認證，不能混稱10年全部驗證完成。</p><p>候選上市100＋上櫃50，原3配置、各選25檔、90%投入、每5日調整、整千股及費稅未改。勝率為有效年度正報酬比例；期望值為平均24日費後報酬，不年化。</p><div class="scroll">'''+table_external_v12(comparison_headers_v14,comparison_rows_v14)+'''</div><h2>每個年度點與箱形圖</h2><p>預設8年口徑；10年情境需明確切換。兩者座標固定，桃紅線為中位數，黑色虛線為0%。滑鼠可查各點的日期、費後報酬、費稅與估值／現增假設。</p>'''+plot_v14(fig_v14,output_type='div',include_plotlyjs=True,config={'displaylogo':False,'responsive':True})+'''<h2>哪些已能導正？</h2><ul><li><b>兆豐金2886：</b>不再把綜合成交金額÷成交股數強制限制在一般盤中的高低價；沒有把25.847元硬改成26.05元。相同修正套用全股票、全年度，仍需正成交量、金額與有限正均價。不是確認供應商資料出錯。</li><li><b>漢唐2404：</b>2018-11-29至12-07停止交易。11-29、11-30僅以11-28最後價63.9元保留估值，原始收盤仍缺、不能交易。12-10才恢復交易，未在此次11-30結束的窗口提前減股或加2元現金。此估值參考官方指數保留市值原則，非官賽已確認方式。</li><li><b>長科6548／富喬1815：</b>現金增資是有償認購，不能變免費配股。主結果仍未知；情境採不參與、不認列權利價值，無免費股票、無認購扣款、無虛構現金。供應商當日漲跌欄0推算的參考價，不等於已確認的官方除權參考價。</li></ul><h2>逐年對照</h2><div class="scroll">'''+table_external_v12(annual_headers_v14,annual_rows_v14)+'''</div><p>8年與10年樣本不同，期望值差異不能直接當策略優劣。原100／150股票池比較不混入新帳務情境。現況名單偏誤、未含原A法人資格、同季年度少等限制仍在，未證實新選股改良。</p><h2>來源與驗證</h2><p>原v13全部30案例先成功重現；沒有重抓整批資料，也沒有用未來事件提前換股。來源PDF與雜湊、原Notebook、回測及統計JSON均保存。官方DPlan／DataPack文件未找到認購、減資與停牌明確帳務條款，所以不宣稱官賽全數驗證。</p><p><a href="https://eshop.twse.com.tw/zh/product/detail/cfec9a1470e448ec91bfde006db361e8">官方行情統計口徑</a> · <a href="https://www.tpex.org.tw/storage/stock_index/企業活動及企業事件處理原則.pdf">官方企業事件處理原則</a> · <a href="https://www.uisco.com.tw/107year.pdf">漢唐107年年報</a> · <a href="https://news.cnyes.com/news/id/4238624">漢唐公司公告轉載</a> · <a href="https://www.nstock.tw/news/article_m?id=389226">富喬公司公告轉載</a></p></main></html>'''
chart_path_v14=recovery_root/'完整150檔研究_修正後同期比較_v14_2026-10-15.html';chart_path_v14.write_text(html_v14,encoding='utf-8')
old_chart_v14=recovery_root/'完整150檔研究_同期配對箱形圖_v13_2026-10-15.html';old_html_v14=old_chart_v14.read_text(encoding='utf-8')
if 'v14更新：' not in old_html_v14:
    banner_v14='<p class="note"><b>v14更新：</b>2018已能依停牌保留市值估值口徑計算；2017／2025另有放棄認購條件情境。<a href="'+chart_path_v14.name+'">查看修正與情境比較</a>。以下v13數字保留作修正前紀錄。</p>'
    old_chart_v14.write_text(old_html_v14.replace('<main>','<main>'+banner_v14,1),encoding='utf-8')
statepath_v14=recovery_root/'A策略_無人值守研究狀態.json';state_v14=json.loads(statepath_v14.read_text(encoding='utf-8'))
state_v14['correction_v14']={'status':'completed_feasible_corrections_and_explicit_sensitivity','running':False,'kernel_connection':'C:/Users/weiha/AppData/Roaming/jupyter/runtime/kernel-aicup-repair-20261015.json','recovery_authorization':'prior user consent to restore dead research kernel and finish remaining work','raw_cache_reloaded_not_refetched':True,'main_valid_years':8,'main_unknown_years':[2017,2025],'conditional_no_subscription_years':10,'conditional_is_official_certified':False,'original_controls_reproduced':30,'report':report_path_v14.name,'chart':chart_path_v14.name,'definition_sha256':repair_definition_sha_v14,'notebook_cells':[252,258],'automation':'preservepaused','unresolved':['official paid-subscription/right valuation','official suspensionmark/DataPack convention','currentuniversebias','exacthistoricalcontestaveragefeed']}
statepath_v14.write_text(json.dumps(state_v14,ensure_ascii=False,indent=2),encoding='utf-8')
with (recovery_root/'A策略_無人值守研究流程.md').open('a',encoding='utf-8') as f:f.write('\n\n## v14 三年缺口導正（2026-10-15）\n原30案例成功重現；移除綜合成交均價必須落在一般盤中高低價內的不適用檢查。漢唐減資停牌採保留最後市值但不成交的明示研究估值，主結果恢復2018，共8年，2017／2025現增權利仍未知。另依預先定義不參與現增、不計認購權的條件情境算10年，不當官方結算認證。勝率／期望值分表，不混樣本。詳見「'+report_path_v14.name+'」。舊kernel已結束，依既有恢復授權用kernel-aicup-repair-20261015，只恢復必要快取；另一專案kernel不動。\n')
print('V14_REPORT_SAVED',report_path_v14.name,chart_path_v14.name)


In [ ]:
# Numerical/accounting regression checks and figure serialization, not browser visual QA.
unchanged_paths_v14=0;scenario_paths_v14=0;assumption_events_v14=[]
for name in policies_v12:
    assert strict_metrics_v14[name]['n']==8 and strict_metrics_v14[name]['invalid_years']==[2017,2025]
    assert conditional_metrics_v14[name]['n']==10
    for old,strict,scenario in zip(full_results_v13[name],strict_repair_results_v14[name],conditional_repair_results_v14[name]):
        if old['return'] is not None:
            assert np.allclose(old['path'],strict['path'],rtol=0,atol=1e-5);assert np.isclose(old['cost'],strict['cost'],rtol=0,atol=1e-12);unchanged_paths_v14+=1
        if strict['return'] is not None:
            assert np.allclose(strict['path'],scenario['path'],rtol=0,atol=1e-5);scenario_paths_v14+=1
        assumption_events_v14.extend(scenario.get('assumption_events',[]))
        for row in [strict,scenario]:
            assert (row['return'] is None)==bool(row['invalid'] or row['violations'])
            if row['return'] is not None:assert np.isclose(row['return'],row['path'][-1]/INITIAL-1)
    for m in [strict_metrics_v14[name],conditional_metrics_v14[name]]:
        assert np.isclose(m['expectancy'],m['win_rate']*m['average_win']+(1-m['win_rate'])*m['average_loss'])
for key,arrays in case_arrays_v14.items():
    assert np.array_equal(arrays['close'],case_arrays_v13[key]['close'],equal_nan=True)
    assert np.array_equal(arrays['cashdiv'],case_arrays_v13[key]['cashdiv'])
    assert np.array_equal(arrays['returns'],case_arrays_v13[key]['returns'],equal_nan=True)
    for event in [e for e in stale_marks_v14 if e['case']==key]:
        t=arrays['dates'].index(datetime.date.fromisoformat(event['date']));j=ext_si_v13[event['sid']]
        assert np.isnan(arrays['close'][t,j]) and arrays['mark'][t,j]==63.9 and not arrays['available'][t,j]
assert {(event['sid'],event['date']) for event in assumption_events_v14}=={('6548','2017-11-15'),('1815','2025-11-27')}
assert len(assumption_events_v14)==6
assert sum(len(t.y) for t in fig_v14.data if t.type=='scatter' and t.mode=='markers')==54
assert sum(len(t.y) for t in fig_v14.data if t.type=='scatter' and t.mode=='markers' and t.visible)==24
assert sum(t.mode=='lines' and t.line.color=='#dc2477' for t in fig_v14.data if t.type=='scatter')==6
assert any(shape.y0==0 and shape.y1==0 for shape in fig_v14.layout.shapes)
assert fig_v14.layout.yaxis.autorange is False
assert list(fig_v14.layout.updatemenus[0].buttons[0].args[0]['visible'])==[True]*9+[False]*9
assert list(fig_v14.layout.updatemenus[0].buttons[1].args[0]['visible'])==[False]*9+[True]*9
assert chart_path_v14.stat().st_size>1000000
assert json.loads((recovery_root/'公司行動與成交修正_回測結果_v14_2026-10-15.json').read_text(encoding='utf-8'))['official_contest_certified'] is False
verification_v14={'old_cases_reproduced':30,'unchanged_original_valid_paths':unchanged_paths_v14,'main_equals_sensitivity_on_common_paths':scenario_paths_v14,'unchanged_raw_close_arrays':10,'cash_dividend_arrays_unchanged':True,'signal_return_arrays_unchanged':True,'suspension_raw_missing_preserved':2,'suspension_trades_blocked':True,'paid_assumption_instances':6,'main_points':24,'scenario_points':30,'total_plot_points':54,'pink_medians':6,'zero_line':True,'fixed_axis_switch':True,'expectancy_identity':True,'verification':'programmatic arrays/accounting/layout only; not browser visual QA','official_contest_certified':False}
state_v14['correction_v14']['verification']=verification_v14;state_v14['correction_v14']['notebook_cells']=[252,259]
statepath_v14.write_text(json.dumps(state_v14,ensure_ascii=False,indent=2),encoding='utf-8')
report_current_v14=report_path_v14.read_text(encoding='utf-8').replace('cells252–258','cells252–259')
report_path_v14.write_text(report_current_v14+'\n\n## 數值驗證\n原30案例重現；原7有效年共21條資產路徑與費用不變。主表與放棄認購情境共同8年共24條路徑一致。10個原始收盤矩陣、現金股利與訊號報酬矩陣未改；2筆停牌估值保持原始收盤缺漏與禁止成交。6次持倉現增假設只對應6548的2017-11-15與1815的2025-11-27。勝率／期望值條件損益恆等式、路徑終值報酬、54個點、桃紅中位數、0%線與固定座標切換皆通過程式檢查；未做瀏覽器視覺QA，未宣稱官賽認證。\n',encoding='utf-8')
print('V14_VERIFIED',json.dumps(verification_v14,ensure_ascii=False))


In [ ]:
# v15 retained-state and saved-input inventory; no restart, fetch or repeated strategy grid.
import sys,json,gzip,hashlib,datetime,collections,ast,nbformat,numpy as np
print('V15_KERNEL',sys.executable,'pid',__import__('os').getpid())
for name in ['case_arrays_v14','strict_repair_results_v14','flow','raw_data','pure_revenue_records_v1','scores_v7']:
    obj=globals().get(name);print('V15_META',name,type(obj).__name__,getattr(obj,'shape',len(obj) if hasattr(obj,'__len__') else None))
for filename in ['策略研究_原資料快照_v3_2026-10-14.json.gz','A策略_營收原文快取_v5_2026-10-14.json.gz']:
    with gzip.open(recovery_root/filename,'rt',encoding='utf-8') as f:obj=json.load(f)
    print('V15_CACHE_SCHEMA',filename,type(obj).__name__,list(obj)[:12] if isinstance(obj,dict) else len(obj))
    if isinstance(obj,dict):
        for key,value in list(obj.items())[:8]:
            print('V15_CACHE_FIELD',key,type(value).__name__,len(value) if hasattr(value,'__len__') else None,'record_fields',list(value[0]) if isinstance(value,list) and value and isinstance(value[0],dict) else ('row_width:'+str(len(value[0])) if isinstance(value,list) and value and isinstance(value[0],list) else None))
print('V15_RELATED_FILES',[p.name for p in recovery_root.glob('*v7*json')])


In [ ]:
# Recover local engine/source schemas without dumping news text or credentials.
base_book_v15=nbformat.read(recovery_root/'動能基準研究_24日回測_2026-10-13.ipynb',as_version=4)
print('V15_MATRIX_SOURCE',base_book_v15.cells[5].source)
with gzip.open(recovery_root/'策略研究_原資料快照_v3_2026-10-14.json.gz','rt',encoding='utf-8') as f:short_raw_v15=json.load(f)
print('V15_RAW_ROW_TYPES',{key:[type(x).__name__ for x in rows[0]] for key,rows in short_raw_v15.items()})
v7defs_v15=json.loads((recovery_root/'A策略_關鍵資料修正重驗證定義_v7_2026-10-15.json').read_text(encoding='utf-8'))
v7data_v15=json.loads((recovery_root/'A策略_關鍵資料修正逐窗口_v7_2026-10-15.json').read_text(encoding='utf-8'))
v7sources_v15=json.loads((recovery_root/'A策略_策略資料缺口原始來源_v7_2026-10-15.json').read_text(encoding='utf-8'))
print('V15_V7_DEFINITION_KEYS',list(v7defs_v15));print('V15_V7_RESULT_KEYS',list(v7data_v15));print('V15_V7_SOURCE_KEYS',list(v7sources_v15))
print('V15_OVERLAY_FILES',[p.name for p in recovery_root.glob('*v4*json')])
for idx,cell in enumerate(saved_nb_v14.cells):
    if cell.cell_type=='code' and 'src_v7=' in cell.source:print('V15_ENGINE_CELL',idx,cell.source[:650])


In [ ]:
# Restore original650-window numerical snapshot and verified action overlays only.
raw_data={key:[list(row) for row in rows] for key,rows in short_raw_v15.items()}
for key in ['price','institution','dividend']:
    for row in raw_data[key]:row[1]=datetime.date.fromisoformat(row[1])
COUNT=25;HORIZON=24;REBALANCE=5;LOT=1000;INVESTED=.9
exec(base_book_v15.cells[5].source,globals())
exec(base_book_v15.cells[8].source,globals())
v4_saved=json.loads((recovery_root/'策略研究_逐窗口結果_v4_2026-10-14.json').read_text(encoding='utf-8'))
print('V15_V4_KEYS',list(v4_saved));print('V15_V4_METHODS',v4_saved['fixed_methods']);print('V15_V7_METHODS',list(v7data_v15['windows']))
print('V15_V4_ACTION_RECORD_SCHEMA',v4_saved['action_corrections'][0]);print('V15_V4_UNRESOLVED_SCHEMA',v4_saved['unresolved_events'][0])
holiday_saved_v15=json.loads((recovery_root/'A策略_休市除權息對齊查核_v7_2026-10-15.json').read_text(encoding='utf-8'))
print('V15_HOLIDAY_ACTION_SCHEMA',holiday_saved_v15['actions_applied'][0])
current_book_v15=nbformat.read(notebook_path_v14,as_version=4)
print('V15_ENGINE_SOURCE_TEMPLATE',current_book_v15.cells[75].source[current_book_v15.cells[75].source.index('src_v3='):current_book_v15.cells[75].source.index("if 'engine_v3_results' not in globals():")])
print('V15_LEGACY_SIMULATOR',ast.get_source_segment(base_book_v15.cells[10].source,next(n for n in ast.parse(base_book_v15.cells[10].source).body if isinstance(n,ast.FunctionDef) and n.name=='simulate')))


In [ ]:
# v15 finite comparison preregistration and future seasonality dependency plan.
snapshot_sha_v15=hashlib.sha256(json.dumps(raw_data,default=str,ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
assert snapshot_sha_v15==v4_saved['raw_snapshot_sha256'],'Snapshotchanged: do not reuse old results'
definition_v15={'version':'v15_five_pure_families_same_engine','request':'fivefamiliesfaircomparisonplusfutureindustryseasonalityplan','universe':'samecurrentTWSE100plusTPEX50','raw_snapshot_sha256':snapshot_sha_v15,'period':'2023-10-02to2026-10-12;650startsafter61sessionwarmup;24sessionholding','families':{'MOM':'equalrankprevious20and60daymomentum','FLOW':'previous20dayforeignplusinvestmenttrustnetbuyshares/volume;nomomentumranking','REV':'negativeprevious5dayadjustedreturn;nomomentumranking','LOWVOL':'negativeprevious20dayadjustedreturndailySD;nomomentumranking','REVENU':'latestknownpublishedmonthlyrevenueMoM/YoYpercentiles50/50;notLLM;knownavailabletime notcertifiedearliestrelease'},'shared_eligibility':'samev7past60completepricedataandpast20flowvolumequality;notdifferentmomentumfilters;pastunknownactionflagsexclude;missingfactorvaluesexcludeonlyaffectedstock','revenue_eligibility':'publicationbeforedecision08:55,latestreportmonthlag1or2,earliestverifiedsourceofthatmonth,45daymaxage,originaldualratecrosscheckedDBwithin0.05pp','portfolio_count':25,'count_scope':'controlforsignalisolation;notmandatoryfuture25;previous20to30experimentsremainreference','allocations':{'equal':'90pct/25;full adjustment tolerance0','mild_skip':'90pct*linear1.25to.75inrankorder/25;retainedpositiondelta<=0.005previousNAVskip;entriesexitsexecute'},'allocation_comparison_limit':'mild_skipalsochangestolerance;itisaallocationplustradingbundle,notisolatedweightingeffect','holding':24,'rebalance':5,'lots':1000,'invested':.9,'fee':.001425,'tax':.003,'caps':'2330.25others.10;existing5daygrace;daily20to30actualholdings;cashnonnegativeand<.25','actions':'samev7verifiedfreecashandstockreceivablesnotspendable;12paidcapitalunknown;noautomaticwaiverinprimary','v14_applicability':'oldshortenginealreadyusesaggregateVWAPwithoutregularhighlowgate;suspension2018outsideperiod;no needchangeprimaryscope','max_candidates':10,'expected_reuse':['MOM_equal','MOM_mild_skip','FLOW_equal','LOWVOL_equal','REVENU_equal'],'new_only':['REV_equal','FLOW_mild_skip','REV_mild_skip','LOWVOL_mild_skip','REVENU_mild_skip'],'dedup':'experimentdefinition+snapshot+enginehash;20samplecontrolsorallfailedmethodrecalculation;preserve reason','comparisons':'allvaliddescriptive;all10commonvaliddatesprimary;familywithinallocationvsMOM;allocationbundleswithinfamily;pairedintersectioncoverage','statistics':'netwinrate,mean24dEV,meanwin/meanloss,flatcount,n,fees,mdd,P10;24/48positionmovingblocks5000repseed20261015;279priorfamily+10conservativefamily289','support_gate':'bothblocklowerbounds>0,positiveyearandearlylatepairedmeans,training/testcoverage>=.8,p10andmedianmddnotworse1pp;no untouchedholdout;do notforcewinner','no_data_fishing':'no newlookbacksweightsstockcountsafteroutcome;no newstrategiesuntilbatchreported','deferred':'LLM,DataPack,DPlan,trades,automations,newrepo/project;seasonalityexecutionawaitsPITclassificationgate'}
definition_sha_v15=hashlib.sha256(json.dumps(definition_v15,sort_keys=True).encode()).hexdigest()
(recovery_root/'五策略家族_統一比較定義_v15_2026-10-15.json').write_text(json.dumps(definition_v15,ensure_ascii=False,indent=2),encoding='utf-8')
seasonality_plan_v15='''# 產業季節性：後續研究計畫 v15
2026-10-15。只規畫後續執行，沒有因產業指數曾上漲就直接加碼個股。

## 要回答的問題
同一產業是否在十月最後星期二起24交易日，持續比同期市場與該產業平常月份更強？如果有，加入原策略後是否改善費後報酬與風險？把產業長期強勢和特定月份季節性分開。

## 已有成果直接引用
既有2016–2025官方產業指數、120個月別窗口、逐年全點與v9逐年向前觀察直接引用，不重抓、不把同一份資料包裝成新證據。既有指數結果不是可交易個股策略；產業指數無個股費稅，不能與投資組合期望值混列。

## 依賴樹與完成條件
```mermaid
flowchart TD
 A[既有產業指數／月別來源與品質查核] --> B[歷史個股分類與公開／生效日驗證]
 B --> C[只用歷年已知資料產生當年季節分數]
 C --> D[純產業與動能混合：固定四候選]
 D --> E[同資料同配置費後配對與風險檢查]
 E --> F[通過才列候選；未通過保留基準]
 B --> G[來源不足：個股分支不可判定]
```
1. **來源門檻。** 建立股票代號、產業碼、有效起日／迄日、公告時間、原始來源與雜湊。先核對2023-07-03重分類，以及其他可證明的分類改變；包含47家公司公告的已知反例。不得用目前分類回填；MOPS歷史營收月份頁在2026重新出表，不能當當時分類存檔。先做來源可得性，不反覆要求已被拒絕的API，也不突破權限或付費限制。
2. **股票池門檻。** 第一層在同一目前150檔池隔離產業增量，明寫存活與選擇偏誤；第二層只有逐年上市／上櫃歷史股票池可驗證才驗證外推，不能把2016單一上市100快照說成逐年官方比賽150。
3. **指數口徑門檻。** 產業新增／重分類前的歷史不能擅自拼接。優先使用可驗證的含息指數；只有價格指數時分表標示，不和含息成果混算；大類與子類重疊須明列。分類不齊不能默默從樣本剔除後稱全市場季節性。
4. **分數固定。** 至少5個完整過往年度，當年只用決策日前已結束窗口。每產業每過往年先算『10月同期相對市場報酬－同年其他11個月份窗口的平均相對市場報酬』，再跨過往年取平均，轉成產業間百分位。至少5年且每年12月別足夠完整才可比；不足保留未知，不填中性。10月及其他月份皆沿用既有最後星期二起24交易日定義。跨年一致性與正超額年份比例作診斷，不看到結果再改公式。已看過資料，仍屬回溯驗證，非全新未見樣本外。
5. **固定候選。** 在同一配置與規則下，純產業分數，以及動能75%＋產業25%、動能50%＋產業50%、動能25%＋產業75%，共4個選股候選。純產業同分股票用過去20日成交金額排名決定，不能偷加動能。先沿用本輪25檔控制與兩種配置，共最多8個新增定義；A對照直接引用。季節性股票不足25就該窗口不可判定，不拿其他訊號偷偷補足。產業集中風險必列，不默加新的產業上限後把效果歸給季節性。
6. **評估。** 完全相同有效日期列費後勝率、24日期望值、平均獲利／虧損、有效樣本／排除原因、費用、P10、最大回撤、產業曝險與對A增量。24／48日位置區塊、多重比較家族、逐年與不重疊相位作敏感度。只有兩個比賽同期起點不能稱穩定季節性；即使十年只有十個同期點，也不能當650個獨立樣本。
7. **選擇。** 混合比重只能用當時之前已結束的訓練窗口比較，測試年鎖定；至少100訓練配對、80%基準日期覆蓋與既有風險門檻。季節性年份／公告窗口不足就不選；不以全期最高平均值挑權重，不擴增參數直到顯著。

## 本次不做與後續啟動條件
目前歷史個股分類來源未通過門檻，因此本文件是可執行的預先計畫，不是已跑完的產業選股。來源門檻通過後才能啟動四候選；否則保留指數層級觀察與清楚的缺口，不造分類。產業相對營收是更後面的獨立研究，不混入本季節性公式。LLM與正式DataPack／DPlan／交易不在本輪。
'''
seasonality_plan_path_v15=recovery_root/'產業季節性_後續研究計畫_v15_2026-10-15.md';seasonality_plan_path_v15.write_text(seasonality_plan_v15,encoding='utf-8')
state_v15=json.loads((recovery_root/'A策略_無人值守研究狀態.json').read_text(encoding='utf-8'))
state_v15['five_family_v15']={'status':'defined_running','running':True,'max_candidates':10,'definition_sha256':definition_sha_v15,'source_scope':'retainedsnapshotnoIO','seasonality_plan':seasonality_plan_path_v15.name,'industry_execution':'notstartedPITclassificationgate','kernel_connection':'C:/Users/weiha/AppData/Roaming/jupyter/runtime/kernel-aicup-repair-20261015.json'}
(recovery_root/'A策略_無人值守研究狀態.json').write_text(json.dumps(state_v15,ensure_ascii=False,indent=2),encoding='utf-8')
print('V15_FROZEN',definition_sha_v15,'max10;reuse5expected;seasonalityplan saved')


In [ ]:
# Build exact prior numerical evidence from saved snapshots and source code.
cash_div_v15=cash_div.copy();stock_div_v15=stock_div.copy();signal_returns_v15=signal_returns.copy();bad_action_v15=bad_action.copy()
for action in v4_saved['action_corrections']:
    day=datetime.date.fromisoformat(action['ex_date']);sid=action['stock_id']
    if day not in di or sid not in si:continue
    t,j=di[day],si[sid];cash_div_v15[t,j]=float(action['cash']);stock_div_v15[t,j]=float(action['stock_ratio']);signal_returns_v15[t,j]=close[t,j]/float(action['reference'])-1;bad_action_v15[t,j]=False
for action in holiday_saved_v15['actions_applied']:
    t,j=di[datetime.date.fromisoformat(action['effective_price_session'])],si[action['stock_id']]
    assert np.isclose(close[t-1,j],action['before'],atol=.001)
    cash_div_v15[t,j]=action['cash'];stock_div_v15[t,j]=action['stock_ratio'];signal_returns_v15[t,j]=close[t,j]/action['reference']-1;bad_action_v15[t,j]=False
assert int(bad_action_v15.sum())==12,'Overlay failed to reproduce prior verified action flags'
good_exec_v15=np.isfinite(execution)&(execution>0)&np.isfinite(volume)&(volume>0)&np.isfinite(turnover)&(turnover>0)&np.isfinite(close)&(close>0)
v7_engine_cell_v15=next(cell.source for cell in current_book_v15.cells if cell.cell_type=='code' and cell.source.startswith('# v7 fixed-method engine revalidation:'))
score_code_v15=v7_engine_cell_v15[v7_engine_cell_v15.index('scores_v7='):v7_engine_cell_v15.index("scores_v7['AH']")].replace('_v7','_v15')
exec(score_code_v15,globals())
import unicodedata
from collections import defaultdict,Counter
with gzip.open(recovery_root/'A策略_營收原文快取_v5_2026-10-14.json.gz','rt',encoding='utf-8') as f:rev_cache_v15=json.load(f)
full_pool_revenue_rows_v1=rev_cache_v15['monthly_rows'];full_pool_revenue_news_v1=rev_cache_v15['news_originals']
for row in full_pool_revenue_news_v1:
    if isinstance(row['published_at'],str):row['published_at']=datetime.datetime.fromisoformat(row['published_at'])
revenue_code_v15=current_book_v15.cells[63].source
revenue_code_v15=revenue_code_v15[:revenue_code_v15.index("print('Original extraction:")].replace("scores['A']","scores_v15['A']")
exec(revenue_code_v15,globals())
scores_v15['PURE_REVENUE']=pure_revenue_scores_v1
family_signal_v15={'MOM':'A','FLOW':'PURE_FLOW','REV':'REV5','LOWVOL':'PURE_LOWVOL','REVENU':'PURE_REVENUE'}
family_labels_v15={'MOM':'動能','FLOW':'純法人買超','REV':'短期反轉','LOWVOL':'純低波動','REVENU':'營收MoM／YoY'}
window_starts_v15=np.arange(61,len(dates)-HORIZON+1)
assert len(window_starts_v15)==650 and len(stock_ids)==150
feature_hashes_v15={key:hashlib.sha256(scores_v15[value].tobytes()).hexdigest() for key,value in family_signal_v15.items()}
source_features_v15={'snapshot_sha256':snapshot_sha_v15,'shape':list(shape),'action_flags':int(bad_action_v15.sum()),'window_count':650,'revenue_accepted':len(pure_revenue_records_v1),'revenue_stocks':len({r['stock_id'] for r in pure_revenue_records_v1}),'revenue_rejected':dict(pure_revenue_rejected_v1),'feature_sha256':feature_hashes_v15,'revenue_coverage':pure_revenue_coverage_v1,'revenue_provenance':'original bulletin+DBnumericcrosscheck;knownavailable notcertifiedfirstrelease','data_gate_missing':'missingnotzero;sameprice/flowqualitygate,no momentumranking inotherfamilies'}
(recovery_root/'五策略家族_資料與覆蓋查核_v15_2026-10-15.json').write_text(json.dumps(source_features_v15,ensure_ascii=False,indent=2),encoding='utf-8')
print('V15_FEATURES_READY',{'shape':shape,'flags':int(bad_action_v15.sum()),'revenue_originals':len(pure_revenue_records_v1),'revenue_stocks':source_features_v15['revenue_stocks'],'revenue_decisions_ge25':sum(r['eligible']>=25 for r in pure_revenue_coverage_v1),'window_starts':len(window_starts_v15)})
def family_allocation_v15(selected,allocation):
    if allocation=='equal':return np.full(len(selected),.9/len(selected))
    assert allocation=='mild_skip';return .9*np.linspace(1.25,.75,len(selected))/len(selected)
base_node_v15=next(node for node in ast.parse(base_book_v15.cells[10].source).body if isinstance(node,ast.FunctionDef) and node.name=='simulate')
family_engine_source_v15=ast.get_source_segment(base_book_v15.cells[10].source,base_node_v15)
family_engine_source_v15=family_engine_source_v15.replace('def simulate(start,strategy,horizon=HORIZON):','def simulate_family_v15(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):')
for old,new in [('bad_action[t]','bad_action_v15[t]'),('cash_div[t]','cash_div_v15[t]'),('stock_div[t]','stock_div_v15[t]'),('good_exec[t]','good_exec_v15[t]'),('scores[strategy]','scores_v15[strategy]')]:family_engine_source_v15=family_engine_source_v15.replace(old,new)
targetline='target[selected]=(np.floor((INVESTED/COUNT)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)'
assert family_engine_source_v15.count(targetline)==1
family_engine_source_v15=family_engine_source_v15.replace(targetline,'target[selected]=(np.floor(family_allocation_v15(selected,allocation)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)')
line='            cash,shares,traded,cost,missing=settle(cash,shares,target,execution[t],good_exec_v15[t])'
skip='            if offset>0 and tolerance>0:\n                kept=(shares>0)&(target>0)\n                small=kept&(np.abs(target-shares)*np.nan_to_num(close[t-1])<=tolerance*previous_nav)\n                target[small]=shares[small]\n'
assert family_engine_source_v15.count(line)==1;family_engine_source_v15=family_engine_source_v15.replace(line,skip+line)
exec(family_engine_source_v15,globals())
family_engine_hash_v15=hashlib.sha256(family_engine_source_v15.encode()).hexdigest()
print('V15_ENGINE_READY',family_engine_hash_v15,'samepriceengineandreceivables;newnamespacedfunction')


In [ ]:
# Run only non-duplicate definitions; calibrate retained rows before referencing old outputs.
family_specs_v15={};family_results_v15={};family_reuse_v15={};calibration_checks_v15=[]
old_map_v15={'MOM_equal':'A_equal','MOM_mild_skip':'A_mild_tolerance','FLOW_equal':'pure_flow','LOWVOL_equal':'pure_lowvol','REVENU_equal':'pure_revenue'}
def serial_family_v15(value):
    if isinstance(value,np.ndarray):return value.tolist()
    if isinstance(value,np.integer):return int(value)
    if isinstance(value,np.floating):return float(value)
    if isinstance(value,np.bool_):return bool(value)
    if isinstance(value,(datetime.date,datetime.datetime)):return value.isoformat()
    raise TypeError(type(value).__name__)
family_path_v15=recovery_root/'五策略家族_全部逐窗口_v15_2026-10-15.json'
for family,signal in family_signal_v15.items():
    for allocation in ['equal','mild_skip']:
        key=family+'_'+allocation;tol=0 if allocation=='equal' else .005
        spec={'family':family,'signal':signal,'allocation':allocation,'tolerance':tol,'count':25,'snapshot':snapshot_sha_v15,'feature_hash':feature_hashes_v15[family],'engine_hash':family_engine_hash_v15,'definition_hash':definition_sha_v15}
        spec['experiment_key']=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest();family_specs_v15[key]=spec
        oldkey=old_map_v15.get(key);reuse=False
        if oldkey is not None:
            oldrows=v7data_v15['windows'][oldkey];checks=[]
            for index in [0,100,200,400,649]:
                actual=simulate_family_v15(int(window_starts_v15[index]),signal,allocation,tolerance=tol);old=oldrows[index]
                ok=bool(actual['invalid']==old['invalid'] and actual['violations']==old['violations'] and all(np.isclose(actual[field],old[field],rtol=0,atol=1e-12) for field in ['return','mdd','cost','turnover']))
                checks.append(ok);calibration_checks_v15.append({'key':key,'index':index,'matched':ok})
            reuse=all(checks)
        if reuse:
            family_results_v15[key]=oldrows;family_reuse_v15[key]={'status':'reused_verified','source':'A策略_關鍵資料修正逐窗口_v7_2026-10-15.json','method':oldkey,'matched_sample_controls':5}
        else:
            family_results_v15[key]=[simulate_family_v15(int(start),signal,allocation,tolerance=tol) for start in window_starts_v15]
            family_reuse_v15[key]={'status':'computed','reason':'new common allocation definition' if oldkey is None else 'saved source/definition sample doesnotmatch;recalculatedall650','prior_candidate':oldkey}
        payload={'definition':definition_v15,'specs':family_specs_v15,'reuse':family_reuse_v15,'calibration_checks':calibration_checks_v15,'results':family_results_v15}
        family_path_v15.write_text(json.dumps(payload,ensure_ascii=False,default=serial_family_v15,allow_nan=False),encoding='utf-8')
        print('V15_FAMILY_DONE',key,family_reuse_v15[key]['status'],'valid',sum(not r['invalid'] and r['violations']==0 for r in family_results_v15[key]),'/',650,flush=True)
assert len(family_results_v15)==10
print('V15_FIVE_FAMILIES_FINISHED',{'definitions':10,'retained_rows':6500,'reused':sum(r['status']=='reused_verified' for r in family_reuse_v15.values()),'new':sum(r['status']=='computed' for r in family_reuse_v15.values()),'calibration_checks':len(calibration_checks_v15)})


In [ ]:
# Compare all10 on identical dates, plus explicitly separate pairwise coverage/inference.
keys_v15=list(family_results_v15)
valid_v15={k:np.array([not r['invalid'] and r['violations']==0 and np.isfinite(r['return']) for r in rows]) for k,rows in family_results_v15.items()}
returns_v15={k:np.array([r['return'] for r in rows],float) for k,rows in family_results_v15.items()};mdd_v15={k:np.array([r['mdd'] for r in rows],float) for k,rows in family_results_v15.items()}
cost_v15={k:np.array([r['cost'] for r in rows],float) for k,rows in family_results_v15.items()};turn_v15={k:np.array([r['turnover'] for r in rows],float) for k,rows in family_results_v15.items()}
common_v15=np.logical_and.reduce(list(valid_v15.values()))
def metrics_family_v15(key,mask):
    x=returns_v15[key][mask];m=mdd_v15[key][mask]
    return {'n':int(mask.sum()),'win_rate':float(np.mean(x>0)) if len(x) else None,'flat_count':int(np.sum(x==0)),'expectancy':float(x.mean()) if len(x) else None,'median':float(np.median(x)) if len(x) else None,'mean_win':float(np.mean(x[x>0])) if np.any(x>0) else None,'mean_loss':float(np.mean(x[x<0])) if np.any(x<0) else None,'P10':float(np.quantile(x,.1)) if len(x) else None,'worst_return':float(x.min()) if len(x) else None,'median_mdd':float(np.median(m)) if len(x) else None,'worst_mdd':float(m.min()) if len(x) else None,'mean_cost':float(cost_v15[key][mask].mean()) if len(x) else None,'mean_turnover':float(turn_v15[key][mask].mean()) if len(x) else None}
all_metrics_v15={key:metrics_family_v15(key,valid_v15[key]) for key in keys_v15};common_metrics_v15={key:metrics_family_v15(key,common_v15) for key in keys_v15}
year_masks_v15={str(year):np.array([dates[int(s)].year==year and dates[int(s)+23].year==year for s in window_starts_v15]) for year in [2024,2025,2026]}
year_common_metrics_v15={year:{key:metrics_family_v15(key,common_v15&ymask) for key in keys_v15} for year,ymask in year_masks_v15.items()}
rng_family_v15=np.random.default_rng(20261015);block_indices_v15={}
for block in [24,48]:
    n=len(window_starts_v15);count=int(np.ceil(n/block));starts=rng_family_v15.integers(0,n-block+1,size=(5000,count));block_indices_v15[block]=(starts[:,:,None]+np.arange(block)).reshape(5000,-1)[:,:n]
alpha_v15=.05/289;contrasts_v15={}
def contrast_family_v15(key,base):
    pair=valid_v15[key]&valid_v15[base];ix=np.flatnonzero(pair);d=returns_v15[key]-returns_v15[base];common_pair=pair&common_v15
    out={'candidate':key,'base':base,'paired_n':int(pair.sum()),'base_valid_n':int(valid_v15[base].sum()),'coverage':float(pair.sum()/valid_v15[base].sum()) if valid_v15[base].any() else None,'mean_excess':float(d[pair].mean()) if pair.any() else None,'beat_base_rate':float(np.mean(d[pair]>1e-12)) if pair.any() else None,'same_all10_common_n':int(common_pair.sum()),'common_mean_excess':float(d[common_pair].mean()) if common_pair.any() else None,'candidate_stats':metrics_family_v15(key,pair),'base_stats':metrics_family_v15(base,pair),'family_size':289,'bootstrap_reps':5000,'interval_precision_warning':'Bonferronitail0.43draws/reps;extremetailnotreliableconfirmation;allretrospective'}
    masked=np.where(pair,d,np.nan)
    for block,indices in block_indices_v15.items():
        sampled=masked[indices];denom=np.sum(np.isfinite(sampled),axis=1);assert np.all(denom>0)
        means=np.nansum(sampled,axis=1)/denom
        out[f'block{block}_interval']=np.quantile(means,[alpha_v15/2,1-alpha_v15/2]).tolist()
    candidate=out['candidate_stats'];baseline=out['base_stats'];out['P10_change']=candidate['P10']-baseline['P10'];out['median_mdd_change']=candidate['median_mdd']-baseline['median_mdd'];out['risk_pass']=bool(out['P10_change']>=-.01 and out['median_mdd_change']>=-.01)
    out['years']={}
    for year,ymask in year_masks_v15.items():
        keep=pair&ymask;out['years'][year]={'pairs':int(keep.sum()),'mean_excess':float(d[keep].mean()) if keep.any() else None,'base_valid':int((valid_v15[base]&ymask).sum())}
    half=len(ix)//2;out['halves']={name:{'n':len(part),'mean_excess':float(np.mean(d[part])) if len(part) else None} for name,part in [('early',ix[:half]),('late',ix[half:])]}
    out['training_coverage_checks']={}
    for year in [2025,2026]:
        train=np.array([dates[int(s)+23]<datetime.date(year,1,1) for s in window_starts_v15]);p=train&pair;b=train&valid_v15[base];pcount=int(p.sum());bcount=int(b.sum());risk=False
        if pcount:
            aa=metrics_family_v15(key,p);bb=metrics_family_v15(base,p);risk=aa['P10']-bb['P10']>=-.01 and aa['median_mdd']-bb['median_mdd']>=-.01
        test=out['years'][str(year)];out['training_coverage_checks'][str(year)]={'training_pairs':pcount,'training_base':bcount,'training_risk_pass':bool(risk),'pass':bool(pcount>=100 and pcount>=.8*bcount and risk and test['pairs']>=.8*test['base_valid'])}
    numerical_gate=bool(out['risk_pass'] and out['coverage']>=.8 and all(out[f'block{block}_interval'][0]>0 for block in [24,48]) and all(out['years'][year]['mean_excess'] is not None and out['years'][year]['mean_excess']>0 for year in ['2025','2026']) and all(r['mean_excess'] is not None and r['mean_excess']>0 for r in out['halves'].values()) and all(r['pass'] for r in out['training_coverage_checks'].values()))
    out['passes_predefined_numerical_gates']=numerical_gate;out['confirmed_improvement']=False;out['confirmation_limit']='past data already observed, extreme bootstrap tails coarse and universe/action/source/official-rule limitations persist'
    return out
for allocation in ['equal','mild_skip']:
    for family in ['FLOW','REV','LOWVOL','REVENU']:
        key=family+'_'+allocation;base='MOM_'+allocation;name=key+'__vs__'+base;contrasts_v15[name]=contrast_family_v15(key,base)
for family in family_signal_v15:
    key=family+'_mild_skip';base=family+'_equal';contrasts_v15[key+'__vs__'+base]=contrast_family_v15(key,base)
family_summary_v15={'definition':definition_v15,'definition_sha256':definition_sha_v15,'all_valid':all_metrics_v15,'all10_common':common_metrics_v15,'all10_common_count':int(common_v15.sum()),'common_window_indices':np.flatnonzero(common_v15).tolist(),'common_by_year':year_common_metrics_v15,'paired_contrasts':contrasts_v15,'reuse':family_reuse_v15,'feature_coverage':source_features_v15,'numerical_gate_passes':[key for key,row in contrasts_v15.items() if row['passes_predefined_numerical_gates']],'confirmed_improvements':[],'no_untouched_holdout':True,'official_compliance_certified':False,'common_dates_warning':'all10intersectionforprimaryranking;pairwiseextrasuseotherdenominatorsandnotmixed','allocation_bundle_warning':'equalfulltrade vs mildrankplus.005skip;notpureweightingeffect'}
(recovery_root/'五策略家族_統計與配對比較_v15_2026-10-15.json').write_text(json.dumps(family_summary_v15,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V15_COMMON_METRICS',json.dumps(common_metrics_v15,ensure_ascii=False),flush=True)
print('V15_PAIRWISE_SUMMARY',json.dumps({key:{field:row[field] for field in ['paired_n','mean_excess','beat_base_rate','risk_pass','passes_predefined_numerical_gates']} for key,row in contrasts_v15.items()},ensure_ascii=False),flush=True)
print('V15_EVALUATION_DONE','all10common',int(common_v15.sum()),'numericalgatepasses',family_summary_v15['numerical_gate_passes'],'no confirmed winner')


In [ ]:
# Report every family, same-date primary comparison and separate source limitations.
main_rows_v15=[];risk_rows_v15=[];coverage_rows_v15=[];paired_rows_v15=[]
for family in family_signal_v15:
    a=common_metrics_v15[family+'_equal'];b=common_metrics_v15[family+'_mild_skip'];label=family_labels_v15[family]
    main_rows_v15.append([label,str(b['n']),pct_external_v12(a['win_rate']),pct_external_v12(a['expectancy']),pct_external_v12(b['win_rate']),pct_external_v12(b['expectancy'])])
    risk_rows_v15.append([label,pct_external_v12(b['mean_win']),pct_external_v12(b['mean_loss']),pct_external_v12(b['mean_cost']),pct_external_v12(b['P10']),pct_external_v12(b['median_mdd']),pct_external_v12(b['worst_mdd'])])
    for allocation in ['equal','mild_skip']:
        key=family+'_'+allocation;m=all_metrics_v15[key]
        coverage_rows_v15.append([label+'／'+('等權' if allocation=='equal' else '非等權＋略過'),str(m['n'])+'/650',str(650-m['n']),pct_external_v12(m['win_rate']),pct_external_v12(m['expectancy']),family_reuse_v15[key]['status']])
        if family!='MOM':
            c=contrasts_v15[key+'__vs__MOM_'+allocation];paired_rows_v15.append([label+'／'+('等權' if allocation=='equal' else '非等權＋略過'),str(c['paired_n']),f"{c['mean_excess']*100:+.3f}pp",pct_external_v12(c['beat_base_rate']),pct_external_v12(c['coverage']),'通過' if c['passes_predefined_numerical_gates'] else '未通過'])
main_headers_v15=['家族','共同窗口','等權勝率','等權24日期望值','非等權＋略過勝率','非等權＋略過24日期望值']
risk_headers_v15=['家族（非等權＋略過）','平均獲利','平均虧損','平均費稅','報酬P10','中位最大回撤','最差最大回撤']
coverage_headers_v15=['配置','各自有效窗口','排除窗口','各自勝率','各自期望值','計算來源']
paired_headers_v15=['與同配置動能配對','配對窗口','平均增量','勝過動能比例','動能日期覆蓋','預定數值判準']
annual_family_rows_v15=[]
for year,table in year_common_metrics_v15.items():
    for family in family_signal_v15:
        m=table[family+'_mild_skip'];annual_family_rows_v15.append([year,family_labels_v15[family],str(m['n']),pct_external_v12(m['win_rate']),pct_external_v12(m['expectancy']),pct_external_v12(m['median_mdd'])])
report_v15='''# 五策略家族：相同150檔股票池與費後比較 v15
2026-10-15。使用者同意先比較五個選股家族，同時規畫產業季節性；本批不擴增訊號窗長或參數直到顯著。

## 這次已完成
動能、純法人、短期反轉、純低波動、營收MoM／YoY，五個家族各比較等權完整調整與溫和非等權＋小額略過，共10定義×650窗口。5個定義引用已保存v7，5個新增，共3250新窗口；25個抽查控制全相符。原始快照SHA與已保存來源一致，不重爬、不寫DB、不呼叫LLM。
目前150檔為上市100＋上櫃50，日期2023-10-02至2026-10-12，暖機後650個24交易日滑動窗口。這不是v14的8／10個年度同期研究，不能把勝率差異當策略變好變差。

## 選股邏輯與共同規則
- 動能：過去20／60日調整後報酬百分位各半。
- 純法人：過去20日外資＋投信淨買股數／成交股數，由高到低；沒有混入動能排名。
- 短期反轉：過去5日調整後報酬由低到高。
- 純低波動：過去20日日報酬標準差由低到高。
- 營收：決策08:55之前已可取得的最新報告月MoM／YoY百分位各半。雙率須原文與DB一致，來源不超45日、報告月落後1或2月；未知排除，不補零、不用動能補選。共1280份可接受原文、126檔涵蓋，不是150檔全歷史完整公告，也未認證首次公開時間。
五家族採相同過去60日價格與過去20日法人／成交量品質資格門檻；這是資料完整性條件，不是把其他家族也按動能挑股。未知公司行動仍按過去資料與持有期間檢查。
本輪固定25檔、投入90%、24交易日、每5日調整、千股單位、買賣手續費0.1425%、賣出稅0.3%、2330上限25%／其他10%、既有5日漂移寬限與現金／實際20–30持股檢查。25檔只為隔離選股效果；先前20／25／30及自適應研究保留，不改成正式必須25檔。
非等權原始權重按排名從1.25遞減至0.75，正規化至90%；相同續抱股票目標變動≤前日NAV0.5%略過，進出場不略過。它同時包含權重與交易緩衝兩個改變，因此不能把等權／非等權全部差異都歸給權重。

## 為何不用重新跑原來全部實驗
原650窗口的v7引擎已不要求綜合均價落在一般盤中高低價內；2018漢唐停牌不在此資料區間。v14對長歷史的修正沒有改變本輪原始資料與既有近年引擎。重建相同現金／免費配股應收、8筆休市除權息對齊及12筆仍未建模現增旗標後，先核對舊結果，再引用相同定義。未把v14放棄認購情境混入主表。

## 主比較：完全相同313個窗口
勝率＝費後24日報酬>0比例；期望值＝費後報酬算術平均，含虧損不年化。以下各格同日期、同分母，避免拿不同期間平均值排名。

'''+mdtable_v13(main_headers_v15,main_rows_v15)+'''
純法人勝率較高但單次平均報酬較低，低波動回撤較小但報酬較低；動能與營收非等權均值接近，但營收最差回撤較大。反轉換手與費用明顯較高。這是取捨，不能只按勝率或均值挑贏家。

## 主比較的風險與獲利／虧損
'''+mdtable_v13(risk_headers_v15,risk_rows_v15)+'''
平均獲利／虧損以各自正／負報酬窗口為分母，平均費稅以初始資金為分母。以上仍是同313窗口。最差最大回撤必列，不能只用中位數說風險調整完成。

## 同家族配置效果與對動能配對
下表是每個候選與相同配置動能的各自配對，日期數不同於主表，不能拿跨列均值當同日期排行。
'''+mdtable_v13(paired_headers_v15,paired_rows_v15)+'''
13項對比（8個家族對動能＋5個家族內配置包對比）均未通過全部事先數值判準。無新方法可宣稱已證實改善；動能仍是比較基準，非等權配置仍待驗證，不自動換成純營收或法人。
每項採24／48個起始位置區塊各5000次，共同抽樣索引、289家族保守Bonferroni探索區間，另查2025／2026逐年、早晚分段、訓練100配對／80%覆蓋與P10及中位回撤不惡化逾1pp。289家族尾端在5000次只有約0.43個抽樣點，精度不足，不能稱嚴格確認性推論。資料與既有成果已看過，沒有未見holdout。

## 逐年共同窗口（非等權＋略過）
'''+mdtable_v13(['年','家族','共同窗口','勝率','24日期望值','中位最大回撤'],annual_family_rows_v15)+'''
## 各自有效窗口與覆蓋
以下用於檢查來源與排除，不同分母不供跨家族排名。原650個點及invalid／violations皆保留JSON；不能可靠計算的點不填零進統計。
'''+mdtable_v13(coverage_headers_v15,coverage_rows_v15)+'''
營收有126檔可接受原文，673個可決策日中548日有至少25檔營收訊號；仍有資料不足、過期及持倉未知公司行動等排除。報酬窗口重疊、稀疏來源與排除可能非隨機，313不是313個獨立樣本。

## 產業季節性後續計畫
已保存「產業季節性_後續研究計畫_v15_2026-10-15.md」。先引用既有十年官方產業指數與月別窗口；個股分支須先取得可驗證的歷史分類與生效／公告時間，不用今日分類或2026重產製月份頁回填。分數只用至少5個過往完整年度，比較十月同期相對市場的表現是否高於該產業其他月份；純產業與動能75／50／25%混合共4選股候選、兩配置，最多8新定義，來源門檻未過不啟動。依年份向前驗證，不能從同一批歷史最高值倒選權重。
目前本批只完成計畫，未宣稱產業個股回測完成。歷史股票池偏誤、認購權、首次公開時點、官賽ActiveShare及正式帳戶等限制仍在。LLM與DataPack／DPlan／交易演練仍延後，未建立新排程或交易。

## 保存與驗證
原Notebook cells260–267保存盤點、資料快取恢復、定義、特徵、有限新增計算、統計與報告；原kernel2524持續沿用，未重啟／清除。定義、全部6500窗口、資料覆蓋與配對統計另存v15 JSON，原v7/v14不覆寫。新增圖表以共同313窗口為預設，可切換等權／非等權固定座標；另有各自有效全部點的覆蓋圖。中位數桃紅、0%虛線、單一點提示。
'''
report_path_v15=recovery_root/'五策略家族_統一比較報告_v15_2026-10-15.md';report_path_v15.write_text(report_v15,encoding='utf-8')
import plotly.graph_objects as go_v15
from plotly.offline import plot as plot_v15
palette_v15=['#3974b4','#269878','#d98628','#795bb0','#c85673'];fig_v15=go_v15.Figure();fig_all_v15=go_v15.Figure()
common_values_v15=[];all_values_v15=[]
def add_family_traces_v15(figure,key,ix,mask,visible,label):
    indices=np.flatnonzero(mask);ys=returns_v15[key][mask]*100;family=family_specs_v15[key]['family'];color=palette_v15[ix]
    figure.add_trace(go_v15.Box(x=[ix]*len(ys),y=ys,boxpoints=False,hoverinfo='skip',fillcolor=color,opacity=.25,line_color=color,visible=visible,showlegend=False))
    figure.add_trace(go_v15.Scatter(x=ix+np.linspace(-.16,.16,len(indices)),y=ys,mode='markers',name=family_labels_v15[family]+'／'+label,marker={'size':5,'color':color,'opacity':.55},customdata=[[str(dates[int(window_starts_v15[i])]),str(dates[int(window_starts_v15[i])+23]),float(cost_v15[key][i]*100),float(mdd_v15[key][i]*100),int(i)] for i in indices],hovertemplate='%{customdata[0]}～%{customdata[1]}<br>費後24日報酬 %{y:.3f}%<br>費稅／初始資金 %{customdata[2]:.3f}%<br>最大回撤 %{customdata[3]:.3f}%<br>窗口索引 %{customdata[4]}<extra>'+family_labels_v15[family]+'／'+label+'</extra>',visible=visible,showlegend=False))
    med=float(np.median(ys));figure.add_trace(go_v15.Scatter(x=[ix-.23,ix+.23],y=[med,med],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',visible=visible,showlegend=False))
for mode in ['mild_skip','equal']:
    visible=mode=='mild_skip';label='溫和非等權＋小額略過' if visible else '等權完整調整'
    for ix,family in enumerate(family_signal_v15):
        key=family+'_'+mode;add_family_traces_v15(fig_v15,key,ix,common_v15,visible,label);common_values_v15.extend((returns_v15[key][common_v15]*100).tolist())
        add_family_traces_v15(fig_all_v15,key,ix,valid_v15[key],visible,label);all_values_v15.extend((returns_v15[key][valid_v15[key]]*100).tolist())
for figure,values in [(fig_v15,common_values_v15),(fig_all_v15,all_values_v15)]:
    figure.update_layout(template='plotly_white',height=540,margin={'l':65,'r':25,'t':85,'b':65},yaxis_title='費後24交易日報酬（%）',hovermode='closest',updatemenus=[{'type':'buttons','direction':'right','x':0,'y':1.16,'buttons':[{'label':'非等權＋略過（預設）','method':'update','args':[{'visible':[True]*15+[False]*15}]},{'label':'等權對照','method':'update','args':[{'visible':[False]*15+[True]*15}]}]}])
    figure.update_xaxes(tickmode='array',tickvals=list(range(5)),ticktext=list(family_labels_v15.values()),range=[-.5,4.5]);figure.update_yaxes(range=[min(values)-2,max(values)+2],autorange=False);figure.add_hline(y=0,line_width=2,line_dash='dash',line_color='#333')
main_html_v15=table_external_v12(main_headers_v15,main_rows_v15);risk_html_v15=table_external_v12(risk_headers_v15,risk_rows_v15);coverage_html_v15=table_external_v12(coverage_headers_v15,coverage_rows_v15)
html_v15='''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>五策略家族：同日期比較 v15</title><style>body{font:16px/1.7 system-ui,sans-serif;color:#203043;background:#f4f7fa;margin:0}main{max-width:1240px;margin:24px auto;padding:24px;background:white;border-radius:14px}h1{font-size:27px}h2{font-size:21px}.note{background:#fff4dc;padding:14px;border-left:4px solid #d49423}.info{background:#edf5ff;padding:14px}.scroll{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}th,td{padding:9px;border-bottom:1px solid #dce3eb;text-align:right;white-space:nowrap}th:first-child,td:first-child{text-align:left}a{color:#2369a5}summary{cursor:pointer;font-weight:bold;padding:12px;background:#edf5ff}</style><main><h1>五個策略家族 × 相同313個窗口</h1><p>候選上市100＋上櫃50；固定25檔隔離選股效果，並非以後必須25檔。無LLM。勝率＝費後正報酬窗口比例；期望值＝平均費後24日報酬，含虧損、不年化。</p><p class="info">本頁是2023–2026資料的650個滑動起點中，十種配置皆有效的共同313個窗口。不是v14的8／10個年度比賽同期，不能直接比較兩頁勝率。313個窗口互相重疊，也不是313個獨立樣本。</p><p class="note">本輪13項對比均未通過全部事先數值判準，沒有已證實新贏家。法人較高勝率不等於較高期望值；低波動減少回撤同時降低報酬。現況名單偏誤、公告覆蓋及公司行動排除仍在。</p><div class="scroll">'''+main_html_v15+'''</div><h2>主比較：每個有效點都保留</h2><p>預設溫和非等權＋小額略過，按鈕可切等權；兩者座標固定。桃紅線是中位數，黑色虛線為0%。每個點可查起訖日、費後報酬、費稅與回撤。等權對照同時改變配置與小額交易緩衝，不能把全部差異都解釋成權重效果。</p>'''+plot_v15(fig_v15,output_type='div',include_plotlyjs=True,config={'displaylogo':False,'responsive':True})+'''<h2>共同313窗口：風險與平均獲利／虧損</h2><div class="scroll">'''+risk_html_v15+'''</div><h2>五個家族怎麼選？</h2><ul><li><b>動能：</b>過去20／60日調整報酬百分位各半。</li><li><b>純法人：</b>過去20日外資＋投信淨買股數／成交股數，沒有混動能排名。</li><li><b>短期反轉：</b>過去5日調整報酬由低到高。</li><li><b>純低波動：</b>過去20日日報酬標準差由低到高。</li><li><b>營收：</b>08:55前已知最新報告月MoM／YoY各半；原文與DB雙率核對，最長45日、報告月落後1或2月。不是LLM判讀，來源未知或過期就排除。</li></ul><p>五者共用資料完整性門檻，固定90%投入、24日、每5日調整、整千股、費率0.1425%、賣出稅0.3%；非等權依名次1.25遞減至0.75，續抱股票目標變動≤NAV0.5%略過。上限2330為25%、其他10%，實際限制與既有5日漂移檢查沿用；未認證完整官賽合規。</p><h2>候選與動能的各自配對（分母不同，勿跨列排名）</h2><div class="scroll">'''+table_external_v12(paired_headers_v15,paired_rows_v15)+'''</div><h2>逐年共同窗口（非等權＋略過）</h2><div class="scroll">'''+table_external_v12(['年份','家族','共同窗口','勝率','24日期望值','中位最大回撤'],annual_family_rows_v15)+'''</div><details><summary>各自有效窗口、全部點與覆蓋限制</summary><p>下表及下圖使用各自有效日期，與主表不同，不適合直接排名。原650個窗口的invalid與violations另存JSON，無法可靠計算者不補零。資料快取的營收原文涵蓋126／150檔、1280份可接受原文；日期是可驗證已知時間，不保證首次公開。</p><div class="scroll">'''+coverage_html_v15+'</div>'+plot_v15(fig_all_v15,output_type='div',include_plotlyjs=False,config={'displaylogo':False,'responsive':True})+'''</details><h2>後續：產業季節性</h2><p>已規畫：沿用十年官方產業指數 → 歷史個股分類／生效日驗證 → 至少5個過往年度比較十月相對其他月份 → 純產業與動能混合四個有限候選 → 同日期費後、回撤及年份檢查。來源門檻未過不啟動個股回測，不以今日分類回填。</p><p>本輪只完成後續計畫；未宣稱產業選股已跑完。詳見同資料夾的「產業季節性_後續研究計畫_v15_2026-10-15.md」。</p><h2>來源與檢查</h2><p>5個定義引用舊結果、5個新增3250窗口；25個抽查控制相符，原快照SHA一致，未重爬／寫DB／重啟kernel。原近年引擎已採綜合均價，2018停牌不在此期間，v14長歷史情境未混入。24／48位置區塊各5000次、家族289探索性校正，極端尾端精度有限；已看過的歷史不是未見holdout。LLM與正式DataPack／DPlan／交易演練仍延後。</p></main></html>'''
chart_path_v15=recovery_root/'五策略家族_共同窗口箱形圖_v15_2026-10-15.html';chart_path_v15.write_text(html_v15,encoding='utf-8')
state_v15['five_family_v15'].update(status='completed_finite_family_comparison_and_seasonality_plan',running=False,definition_count=10,reused_definitions=5,new_definitions=5,new_windows=3250,all_rows=6500,common_windows=313,numerical_gate_passes=[],confirmed_improvements=[],report=report_path_v15.name,chart=chart_path_v15.name,statistics='五策略家族_統計與配對比較_v15_2026-10-15.json',industry_execution='notstartedPITclassificationgate',notebook_cells=[260,267],automation='preservepaused')
(recovery_root/'A策略_無人值守研究狀態.json').write_text(json.dumps(state_v15,ensure_ascii=False,indent=2),encoding='utf-8')
with (recovery_root/'A策略_無人值守研究流程.md').open('a',encoding='utf-8') as f:f.write('\n\n## v15 五個純策略家族同日期比較與產業計畫（2026-10-15）\n五家族×等權／非等權＋略過，10定義×650；5引用舊、5新增3250窗口，25抽查全符。主比較共同313日期，另列各自有效與配對分母，費後勝率／24日期望值／條件損益／費用／回撤均保存。13對比皆未過全部預定門檻，無已證實新贏家。產業季節性後續計畫已保存，歷史分類門檻未過，不宣稱個股產業回測已做。詳見「'+report_path_v15.name+'」及「'+seasonality_plan_path_v15.name+'」。原650窗口引擎已無盤中高低邊界，2018不在期間，未重跑不受影響的舊網格；LLM、官賽操作與排程仍延後。\n')
print('V15_REPORT_SAVED',report_path_v15.name,chart_path_v15.name,seasonality_plan_path_v15.name)


In [ ]:
# Source timing, accounting identities, dedup and full-point chart regression checks.
assert len(family_results_v15)==10 and all(len(rows)==650 for rows in family_results_v15.values())
assert len({spec['experiment_key'] for spec in family_specs_v15.values()})==10
assert len(calibration_checks_v15)==25 and all(row['matched'] for row in calibration_checks_v15)
assert sum(row['status']=='reused_verified' for row in family_reuse_v15.values())==5
assert int(common_v15.sum())==313
for key,rows in family_results_v15.items():
    assert np.all(valid_v15[key][common_v15])
    for index,row in enumerate(rows):
        assert int(row['start'])==int(window_starts_v15[index])
        if valid_v15[key][index]:
            assert row['violations']==0 and len(row['path'])==25
            assert np.isclose(row['return'],row['path'][-1]/INITIAL-1)
            assert row['cost']>=0 and -1<=row['mdd']<=1e-12
    for metrics in [all_metrics_v15[key],common_metrics_v15[key]]:
        positive=metrics['win_rate'];negative=1-positive-metrics['flat_count']/metrics['n']
        reconstructed=positive*(metrics['mean_win'] or 0)+negative*(metrics['mean_loss'] or 0)
        assert np.isclose(reconstructed,metrics['expectancy'])
revenue_asof_checks_v15=0
for q,j in zip(*np.where(np.isfinite(scores_v15['PURE_REVENUE']))):
    t=q+1
    if t>=len(dates):continue
    record=pure_revenue_evidence_v1[(t,stock_ids[j])];cutoff=datetime.datetime.combine(dates[t],datetime.time(8,55));y,m=map(int,record['year_month'].split('-'));lag=(cutoff.year-y)*12+cutoff.month-m
    assert record['status']=='verified_original_known_available_time' and record['published_at']<cutoff and record['age_days']<=45 and lag in [1,2]
    revenue_asof_checks_v15+=1
assert sum(len(t.y) for t in fig_v15.data if t.type=='scatter' and t.mode=='markers')==3130
assert sum(len(t.y) for t in fig_v15.data if t.type=='scatter' and t.mode=='markers' and t.visible)==1565
assert sum(len(t.y) for t in fig_all_v15.data if t.type=='scatter' and t.mode=='markers')==sum(int(m.sum()) for m in valid_v15.values())==5506
for figure in [fig_v15,fig_all_v15]:
    assert len(figure.data)==30
    assert sum(t.mode=='lines' and t.line.color=='#dc2477' for t in figure.data if t.type=='scatter')==10
    assert figure.layout.yaxis.autorange is False
    assert any(shape.y0==0 and shape.y1==0 for shape in figure.layout.shapes)
    assert list(figure.layout.updatemenus[0].buttons[0].args[0]['visible'])==[True]*15+[False]*15
    assert list(figure.layout.updatemenus[0].buttons[1].args[0]['visible'])==[False]*15+[True]*15
assert chart_path_v15.stat().st_size>1000000 and seasonality_plan_path_v15.exists()
assert not family_summary_v15['numerical_gate_passes'] and not family_summary_v15['confirmed_improvements']
assert all(np.isnan(block) if False else row['family_size']==289 for row in contrasts_v15.values())
current_report_v15=report_path_v15.read_text(encoding='utf-8').replace('## 逐年共同窗口（非等權＋略過）','## 逐年共同窗口（非等權＋略過、起迄均在該年）').replace('## 各自有效窗口與覆蓋','跨年窗口仍保留於313總表，但不歸入起迄均在單一年份的上表，因此逐年筆數加總可能較少。\n\n## 各自有效窗口與覆蓋').replace('cells260–267','cells260–268')
verification_v15={'definitions':10,'all_window_rows':6500,'reused_definitions':5,'new_definitions':5,'new_windows':3250,'calibration_controls':25,'common_windows':313,'source_publication_checks':revenue_asof_checks_v15,'common_plot_points':3130,'default_plot_points':1565,'all_valid_plot_points':5506,'pink_medians_per_chart':10,'zero_line_per_chart':True,'fixed_axis_switch':True,'expectancy_identity':True,'path_return_identity':True,'industry_execution_not_started':True,'verification':'programmaticsource/accounting/layoutandserializedoutputchecks,notbrowservisualQA','kernel_restarted':False}
current_report_v15+='\n\n## 最終驗證\n10個唯一實驗鍵、6500窗口、25校準控制、'+str(revenue_asof_checks_v15)+'筆營收可知時間／時效／報告月檢查、所有有效路徑終值與期望值條件損益恆等式通過。主圖兩配置3130點（預設1565）、各自有效圖5506點、中位數桃紅、0%線及固定座標切換通過程式檢查。沒有重啟kernel，也不宣稱瀏覽器目視QA。產業個股回測仍未啟動。\n'
report_path_v15.write_text(current_report_v15,encoding='utf-8')
current_html_v15=chart_path_v15.read_text(encoding='utf-8').replace('逐年共同窗口（非等權＋略過）','逐年共同窗口（非等權＋略過、起迄均在該年）').replace('<details><summary>','<p>跨年窗口保留於313總表，但不歸入起迄均在單一年份的上表。</p><details><summary>',1)
chart_path_v15.write_text(current_html_v15,encoding='utf-8')
state_v15['five_family_v15']['verification']=verification_v15;state_v15['five_family_v15']['notebook_cells']=[260,268]
(recovery_root/'A策略_無人值守研究狀態.json').write_text(json.dumps(state_v15,ensure_ascii=False,indent=2),encoding='utf-8')
print('V15_FINAL_VERIFIED',json.dumps(verification_v15,ensure_ascii=False))
print('V15_COMMON_DATE_RANGE',str(dates[int(window_starts_v15[np.flatnonzero(common_v15)[0]])]),str(dates[int(window_starts_v15[np.flatnonzero(common_v15)[-1]])]),'yearcounts',{year:int((common_v15&mask).sum()) for year,mask in year_masks_v15.items()})


In [ ]:
# v16 separate weight changes, trade-skip effects and existing ranking exits.
print('V16_CONTEXT',{'kernel':__import__('sys').executable,'pid':__import__('os').getpid(),'family_methods':len(family_results_v15),'windows':len(window_starts_v15)})
definition_v16={'version':'v16_weight_vs_skip_factorial_statistics','user_request':'pairedweightstatisticsandassessstockselection/earlyexitinterpretation','scope':'same5families;2weightmodes*2skiprules=20fixedcells;reuseallv15plusv7Amild;no new signalparameters','factors':{'weight':['equal','mildrank1.25to.75'],'retained_adjustment_skip':[0,.005]},'primary':'mild_fullminus_equal_fullisolatesweight;also mild_skipminus_equal_skip;notdiagonalonly','population':'same150fixedcurrentpool;650overlapping24daywindows;25hold,90pct,5dayrebalance,lotsandfeesunchanged','sampling':'all20commonprimary plus eachpaircoverage diagnostics;no mixingindependentdenominators','metrics':'netwinrate,mean24dEV,conditionalmeanwin/loss,n,cost,mdd,P10;pairedmean/medianexcess,p10/p90diff,beatbase,samepositive_negativeoutcomeagreement','intervals':'24/48positionmovingblocks5000sameindices;ordinary95pctdescriptive and conservativefamily289+11=300 exploratory;90pct sensitivityequivalence bands ±.10/.25/.50pp, allreported not pickbestmargin','equivalence_warning':'lackofsignificanceisnotequivalence;bandsarepostplotresearchreportingscalesnotuserutilityorprospectiveconfirmatoryTOST;mustboth90pctblockintervalsinsideband','exit_reference':'reusev7AHequalinitial25buyholdversusAequal5dayranking;pairedmatchedcoverage;not daily stoploss;includesnewselectionandweightretargeting','do_not_claim':'cannotknowwrongstockexante;no automatic-5or-10stopthreshold;no futurefilter;stockselectionfamilyeffectnotcausalattribution','decision':'ifweightincrementunprovenkeepallocationascontrol;selection/exitareprioritiesforfurtherhypothesesnotprovenmaincause','no_additional_execution':'no daily stoploss experiment thisbatch, no LLMortradesorschedules'}
definition_sha_v16=hashlib.sha256(json.dumps(definition_v16,sort_keys=True).encode()).hexdigest()
(recovery_root/'等權非等權_配對統計定義_v16_2026-10-15.json').write_text(json.dumps(definition_v16,ensure_ascii=False,indent=2),encoding='utf-8')
print('V16_FROZEN',definition_sha_v16,'20factorialcellsmax;0newstoplossrules')


In [ ]:
# Four cells per family: weight and small-trade skip independently controlled.
factor_results_v16={};factor_specs_v16={};factor_reuse_v16={};factor_calibration_v16=[]
for family,signal in family_signal_v15.items():
    for tag,allocation,tol in [('E0','equal',0.),('E5','equal',.005),('M0','mild_skip',0.),('M5','mild_skip',.005)]:
        key=family+'_'+tag;factor_specs_v16[key]={'family':family,'weight':'equal' if tag.startswith('E') else 'mild','tolerance':tol,'signal':signal}
        if tag in ['E0','M5']:
            sourcekey=family+('_equal' if tag=='E0' else '_mild_skip');factor_results_v16[key]=family_results_v15[sourcekey];factor_reuse_v16[key]={'status':'reused_v15','sourcekey':sourcekey}
        elif family=='MOM' and tag=='M0':
            old=v7data_v15['windows']['A_mild']
            for ix in [0,100,200,400,649]:
                r=simulate_family_v15(int(window_starts_v15[ix]),signal,allocation,tolerance=tol);o=old[ix]
                assert r['invalid']==o['invalid'] and r['violations']==o['violations'] and all(np.isclose(r[f],o[f],rtol=0,atol=1e-12) for f in ['return','mdd','cost','turnover'])
                factor_calibration_v16.append({'method':key,'index':ix,'matched':True})
            factor_results_v16[key]=old;factor_reuse_v16[key]={'status':'reused_v7','sourcekey':'A_mild','verified_controls':5}
        else:
            factor_results_v16[key]=[simulate_family_v15(int(start),signal,allocation,tolerance=tol) for start in window_starts_v15];factor_reuse_v16[key]={'status':'computed_new_factor_cell'}
        print('V16_FACTOR_DONE',key,factor_reuse_v16[key]['status'],'valid',sum(not r['invalid'] and r['violations']==0 for r in factor_results_v16[key]),flush=True)
scores_v15['AH']=scores_v15['A']
hold_results_v16=v7data_v15['windows']['AH']
for ix in [0,100,200,400,649]:
    r=simulate_family_v15(int(window_starts_v15[ix]),'AH','equal',tolerance=0);o=hold_results_v16[ix]
    assert r['invalid']==o['invalid'] and r['violations']==o['violations'] and all(np.isclose(r[f],o[f],rtol=0,atol=1e-12) for f in ['return','mdd','cost','turnover'])
    factor_calibration_v16.append({'method':'AH_initial25_buyhold_reference','index':ix,'matched':True})
factor_export_v16={'definition':definition_v16,'definition_sha256':definition_sha_v16,'specs':factor_specs_v16,'reuse':factor_reuse_v16,'calibration_checks':factor_calibration_v16,'results':factor_results_v16,'A_initial25_buyhold_reference':hold_results_v16,'hold_reference_warning':'diagnosticonlyAHnotmainrequestedstrategy;5dayrankingexitnotdailystoploss'}
(recovery_root/'等權非等權_四格控制逐窗口_v16_2026-10-15.json').write_text(json.dumps(factor_export_v16,ensure_ascii=False,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V16_FACTORIAL_READY',{'cells':20,'new':sum(v['status']=='computed_new_factor_cell' for v in factor_reuse_v16.values()),'reused':sum(v['status']!='computed_new_factor_cell' for v in factor_reuse_v16.values()),'controls':len(factor_calibration_v16),'holdreference':'existingAHonly'})


In [ ]:
# Paired statistics, practical-size sensitivity and a named-reference accounting decomposition.
mechanism_definition_v16={'status':'fixed_before_newcontraststatistics_butpastresultsalreadyseen','nodes':['REF_liquidity25_equal_buyhold','AH_momentum25_equal_buyhold','A_momentum25_equal5day','A_momentum25_mild5day','A_momentum25_mild5day_skip005'],'contrasts':['AHminusREFinitialrankingcomparison','AequalminusAH5dayreselectionandretargeting','AmildminusAequalweightonly','AmildskipminusAmildsmalltradeskiponly'],'identity':'sum4pairedincrements=AmildskipminusREFonexactcommon5nodewindows','limits':'reference-specificdescriptivenotcausal;rankingexitincludesentriesandretargeting;no daily loss-stop rule or futureknowledge'}
(recovery_root/'等權非等權_機制比較補充定義_v16_2026-10-15.json').write_text(json.dumps(mechanism_definition_v16,ensure_ascii=False,indent=2),encoding='utf-8')
ref_results_v16=v7data_v15['windows']['REF']
for ix in [0,100,200,400,649]:
    r=simulate_family_v15(int(window_starts_v15[ix]),'REF','equal',tolerance=0);o=ref_results_v16[ix]
    assert r['invalid']==o['invalid'] and r['violations']==o['violations'] and all(np.isclose(r[f],o[f],rtol=0,atol=1e-12) for f in ['return','mdd','cost','turnover'])
factor_rows_v16={**factor_results_v16,'AH':hold_results_v16,'REF':ref_results_v16}
valid_factor_v16={key:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for key,rows in factor_rows_v16.items()}
return_factor_v16={key:np.array([r['return'] for r in rows]) for key,rows in factor_rows_v16.items()};cost_factor_v16={key:np.array([r['cost'] for r in rows]) for key,rows in factor_rows_v16.items()}
common_factor_v16=np.logical_and.reduce([valid_factor_v16[key] for key in factor_results_v16])
def metric_factor_v16(key,mask):
    rows=factor_rows_v16[key];x=return_factor_v16[key][mask];m=np.array([r['mdd'] for r in rows])[mask]
    return {'n':int(mask.sum()),'win_rate':float(np.mean(x>0)),'expectancy':float(np.mean(x)),'mean_win':float(np.mean(x[x>0])) if np.any(x>0) else None,'mean_loss':float(np.mean(x[x<0])) if np.any(x<0) else None,'flat_count':int(np.sum(x==0)),'mean_cost':float(cost_factor_v16[key][mask].mean()),'median_mdd':float(np.median(m)),'worst_mdd':float(m.min()),'P10':float(np.quantile(x,.1))}
def paired_factor_v16(candidate,base,scope=None):
    pair=valid_factor_v16[candidate]&valid_factor_v16[base]
    if scope is not None:pair &= scope
    x=return_factor_v16[candidate][pair];y=return_factor_v16[base][pair];d=return_factor_v16[candidate]-return_factor_v16[base];diff=d[pair]
    result={'candidate':candidate,'base':base,'n':int(pair.sum()),'mean_excess':float(np.mean(diff)),'median_excess':float(np.median(diff)),'P10_diff':float(np.quantile(diff,.1)),'P90_diff':float(np.quantile(diff,.9)),'beat_base_rate':float(np.mean(diff>1e-12)),'net_sign_agreement':float(np.mean(np.sign(x)==np.sign(y))),'return_correlation':float(np.corrcoef(x,y)[0,1]),'mean_cost_change':float(np.mean((cost_factor_v16[candidate]-cost_factor_v16[base])[pair])),'candidate_stats':metric_factor_v16(candidate,pair),'base_stats':metric_factor_v16(base,pair),'years':{year:{'n':int((pair&ymask).sum()),'mean_excess':float(np.mean(d[pair&ymask])) if (pair&ymask).any() else None} for year,ymask in year_masks_v15.items()},'equivalence_warning':'predefinedafterplotreportingscales,notprospectiveclinicalequivalencetestoruserutility'}
    masked=np.where(pair,d,np.nan)
    for block,indices in block_indices_v15.items():
        sampled=masked[indices];counts=np.sum(np.isfinite(sampled),axis=1);assert np.all(counts>0);means=np.nansum(sampled,axis=1)/counts
        result[f'block{block}_95']=np.quantile(means,[.025,.975]).tolist();result[f'block{block}_90']=np.quantile(means,[.05,.95]).tolist();result[f'block{block}_family300']=np.quantile(means,[.05/300/2,1-.05/300/2]).tolist()
    result['both95includezero']=bool(all(result[f'block{block}_95'][0]<=0<=result[f'block{block}_95'][1] for block in [24,48]))
    result['band90_contains_both_blockintervals']={f'{margin*100:.2f}pp':bool(all(result[f'block{block}_90'][0]>-margin and result[f'block{block}_90'][1]<margin for block in [24,48])) for margin in [.001,.0025,.005]}
    return result
weight_stats_v16={};skip_stats_v16={};factorial_means_v16={}
for family in family_signal_v15:
    weight_stats_v16[family+'_full']=paired_factor_v16(family+'_M0',family+'_E0',common_factor_v16)
    weight_stats_v16[family+'_skip']=paired_factor_v16(family+'_M5',family+'_E5',common_factor_v16)
    skip_stats_v16[family+'_equal']=paired_factor_v16(family+'_E5',family+'_E0',common_factor_v16)
    skip_stats_v16[family+'_mild']=paired_factor_v16(family+'_M5',family+'_M0',common_factor_v16)
    E0,E5,M0,M5=[return_factor_v16[family+'_'+tag][common_factor_v16] for tag in ['E0','E5','M0','M5']]
    factorial_means_v16[family]={'weight_main_effect':float(np.mean(((M0-E0)+(M5-E5))/2)),'skip_main_effect':float(np.mean(((E5-E0)+(M5-M0))/2)),'interaction':float(np.mean((M5-M0)-(E5-E0))),'cells':{tag:metric_factor_v16(family+'_'+tag,common_factor_v16) for tag in ['E0','E5','M0','M5']}}
mechanism_nodes_v16=['REF','AH','MOM_E0','MOM_M0','MOM_M5'];mechanism_mask_v16=np.logical_and.reduce([valid_factor_v16[key] for key in mechanism_nodes_v16])
mechanism_pairs_v16={name:paired_factor_v16(candidate,base,mechanism_mask_v16) for name,candidate,base in [('initial_selection','AH','REF'),('five_day_reselection','MOM_E0','AH'),('weight_only','MOM_M0','MOM_E0'),('skip_only','MOM_M5','MOM_M0')]}
mechanism_node_metrics_v16={key:metric_factor_v16(key,mechanism_mask_v16) for key in mechanism_nodes_v16}
mechanism_sum_v16=sum(row['mean_excess'] for row in mechanism_pairs_v16.values());mechanism_total_v16=float(np.mean((return_factor_v16['MOM_M5']-return_factor_v16['REF'])[mechanism_mask_v16]));assert np.isclose(mechanism_sum_v16,mechanism_total_v16,atol=1e-12)
weight_ownpair_v16={family:paired_factor_v16(family+'_M0',family+'_E0') for family in family_signal_v15}
summary_v16={'definition':definition_v16,'definition_sha256':definition_sha_v16,'common20_n':int(common_factor_v16.sum()),'common20_indices':np.flatnonzero(common_factor_v16).tolist(),'weight_only':weight_stats_v16,'skip_only':skip_stats_v16,'factorial_means':factorial_means_v16,'weight_ownpair_diagnostic':weight_ownpair_v16,'mechanism_definition':mechanism_definition_v16,'mechanism_common_n':int(mechanism_mask_v16.sum()),'mechanism_pairs':mechanism_pairs_v16,'mechanism_node_metrics':mechanism_node_metrics_v16,'mechanism_sum':mechanism_sum_v16,'mechanism_total':mechanism_total_v16,'limits':['overlappingwindows','pastdataalreadyseen','postplotpracticalbands','currentuniversebias','unknownpaidcapitalexcluded','notdailystoploss','allocationrangeonlymild1.25to.75']}
(recovery_root/'等權非等權_配對與機制統計_v16_2026-10-15.json').write_text(json.dumps(summary_v16,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('V16_PURE_WEIGHT_COMMON',int(common_factor_v16.sum()),json.dumps({key:{field:row[field] for field in ['mean_excess','beat_base_rate','net_sign_agreement','return_correlation','block24_95','block48_95','band90_contains_both_blockintervals']} for key,row in weight_stats_v16.items() if key.endswith('_full')},ensure_ascii=False))
print('V16_MECHANISM_DECOMPOSITION',int(mechanism_mask_v16.sum()),json.dumps({key:{field:row[field] for field in ['mean_excess','mean_cost_change','beat_base_rate','block24_95','block48_95']} for key,row in mechanism_pairs_v16.items()},ensure_ascii=False))
print('V16_MECHANISM_NODE_METRICS',json.dumps(mechanism_node_metrics_v16,ensure_ascii=False))
print('V16_FACTORIAL_ANALYSIS_FINISHED','no daily stoploss rule added')


In [ ]:
# Save statistical explanation without equating nonsignificance with equality or rebalancing with stoploss.
def pp_v16(value):return f'{value*100:+.3f}pp'
def ci_v16(interval):return '['+f'{interval[0]*100:+.3f}, {interval[1]*100:+.3f}'+']pp'
weight_rows_v16=[];payoff_rows_v16=[];skip_rows_v16=[];equiv_rows_v16=[]
for family in family_signal_v15:
    label=family_labels_v15[family];row=weight_stats_v16[family+'_full'];a=row['base_stats'];b=row['candidate_stats']
    weight_rows_v16.append([label,str(row['n']),pp_v16(row['mean_excess']),ci_v16(row['block24_95']),ci_v16(row['block48_95']),pct_external_v12(row['net_sign_agreement']),f"{row['return_correlation']:.4f}"])
    payoff_rows_v16.append([label,pct_external_v12(a['win_rate']),pct_external_v12(a['expectancy']),pct_external_v12(b['win_rate']),pct_external_v12(b['expectancy']),pct_external_v12(b['mean_win']),pct_external_v12(b['mean_loss']),pct_external_v12(b['mean_cost']),pct_external_v12(b['worst_mdd'])])
    equiv_rows_v16.append([label]+['區間落在帶內' if row['band90_contains_both_blockintervals'][key] else '未落在帶內' for key in ['0.10pp','0.25pp','0.50pp']])
    for mode in ['equal','mild']:
        s=skip_stats_v16[family+'_'+mode];skip_rows_v16.append([label+'／'+('等權' if mode=='equal' else '溫和非等權'),pp_v16(s['mean_excess']),pp_v16(s['mean_cost_change']),ci_v16(s['block48_95'])])
weight_headers_v16=['家族（相同交易規則）','窗口','非等權－等權平均差','24位置區塊95%區間','48位置區塊95%區間','同漲同跌比例','報酬相關係數']
payoff_headers_v16=['家族','等權勝率','等權24日期望值','非等權勝率','非等權24日期望值','非等權平均獲利','非等權平均虧損','非等權費稅','非等權最差回撤']
mechanism_labels_v16={'initial_selection':'初始選股：動能持有－成交金額排名持有','five_day_reselection':'五日重選＋再平衡－初始動能持有','weight_only':'只改溫和權重－等權','skip_only':'只加小額略過－每次調整'}
mechanism_rows_v16=[[mechanism_labels_v16[key],str(row['n']),pp_v16(row['mean_excess']),pp_v16(row['mean_cost_change']),ci_v16(row['block48_95'])] for key,row in mechanism_pairs_v16.items()]
node_labels_v16={'REF':'成交金額排名25檔、等權持有','AH':'動能初始25檔、等權持有','MOM_E0':'動能、等權、每5日重選調整','MOM_M0':'動能、溫和非等權、每5日重選調整','MOM_M5':'動能、溫和非等權＋小額略過'}
node_rows_v16=[[node_labels_v16[key],str(row['n']),pct_external_v12(row['win_rate']),pct_external_v12(row['expectancy']),pct_external_v12(row['mean_cost']),pct_external_v12(row['worst_mdd'])] for key,row in mechanism_node_metrics_v16.items()]
report_v16='''# 等權、非等權、初始選股與退出：配對統計 v16
2026-10-15。使用者觀察等權／非等權差異不大，要求統計並討論初始選股及選錯後退出。這次做統計隔離，沒有擅自加入固定百分比停損。

## 先說結論
在本輪溫和權重（排名1.25至0.75）與相同交易規則下，五個家族的費後平均差約−0.12至＋0.19pp／24日，98%–99%窗口同漲同跌，權重對方向的影響有限。但小平均差不代表數學相等，或極端集中等所有非等權都沒有差；部分家族的普通95%區間不跨0，多重比較與實務幅度仍須分開看。
可以降低繼續微調此權重範圍的研究優先度。『初始選股、退出條件值得優先』是合理研究方向，但目前未證實『越快砍越好』；五日重選與再平衡對比持有不動的費後平均反而較低，且區間跨0。

## 為什麼新增四格控制
v15比較的是等權完整調整與溫和非等權＋小額略過，同時改了權重與交易規則。v16對每家族補齊等權不略過E0、等權略過E5、非等權不略過M0、非等權略過M5。M0−E0、M5−E5隔離權重；E5−E0、M5−M0隔離小額略過。20格中11格引用已核對結果，9格新增5850窗口；原資料、選股、25檔、90%投入、費稅與規則未變。

## 純權重比較：相同313窗口
以下雙方都不略過小額調整，所以不是權重與省費包的混合效果。正增量表示溫和非等權較好，單位pp是百分點。普通95%區間為描述，未校正多重比較，不直接稱確認性優勢。

'''+mdtable_v13(weight_headers_v16,weight_rows_v16)+'''
動能在這313窗口的純權重平均差為−0.027pp；v15的非等權＋略過對等權平均差約＋0.040pp，兩者不矛盾，因後者加了交易緩衝。法人偏向等權，反轉與營收非等權點估計較高，不能把五家族全稱完全無差。

## 勝率、期望值與風險：同313窗口、雙方不略過
勝率＝費後24日正報酬比例；期望值＝平均費後報酬、含虧損不年化。條件平均獲利／虧損仍列；同漲同跌比例不是勝率。
'''+mdtable_v13(payoff_headers_v16,payoff_rows_v16)+'''
## 是否可以說實務上接近？
沒有統計顯著不等於證明等價。另列24／48位置區塊90%區間是否都落在±0.10／±0.25／±0.50pp。三個容忍帶是看過圖後固定的研究報告尺度，不是使用者指定可接受損益，也不是前瞻正式等價試驗，不挑最有利門檻。
'''+mdtable_v13(['家族','±0.10pp','±0.25pp','±0.50pp'],equiv_rows_v16)+'''
以±0.25pp尺度，四個家族符合區間帶內的描述，營收未符合；±0.10pp尺度五者都不符合。是否值得付出配置複雜度，仍取決於實務效益與風險，不可只用p值。

## 小額交易略過是另外的效果
以下同313窗口，兩邊權重相同；費用變化為略過減不略過，負值是少付費用。
'''+mdtable_v13(['家族／相同權重','略過的費後增量','費稅變化','48位置區塊普通95%區間'],skip_rows_v16)+'''
## 初始選股與中途動作：另外518個共同窗口
引用已完成的成交金額排名持有REF與動能初始持有AH作機制對照，沒有重跑或改用AH當本輪主策略。這518日期與上方313不同，不能跨兩張表直接比效應大小。
REF、AH都初始選25檔、等權持有24日；AH−REF只比較這兩種已定義的初始排名。A等權−AH從同初始動能名單開始，但同時包含後續換股與續抱權重重設，不能稱純粹停損效果。

'''+mdtable_v13(['機制','共同窗口','費後平均增量','費稅變化','48位置區塊普通95%區間'],mechanism_rows_v16)+'''
初始動能排名相對成交金額排名點估計＋0.833pp，區間跨0，不能聲稱已確認選股優勢。五日重選／再平衡平均費後−0.395pp，同時多付費稅＋0.666pp，區間也跨0；這不證明永遠不能換股，但不支持越快換越好。費用與持股路徑會互相影響，不把淨增量完全歸因於費用，也不把加回費用當真正無費用重跑。
四段增量相加，精確等於同日期M5相對REF的總增量。這是依指定對照路徑的報酬差分解，不是因果或普遍的貢獻百分比。

'''+mdtable_v13(['既有對照節點','窗口','費後勝率','24日期望值','平均費稅','最差最大回撤'],node_rows_v16)+'''
## 對下一階段研究的含義
1. 先保留簡單配置作對照，把資源移向有明確機制的選股／退出假說，不再只對溫和權重做細網格。
2. 『選錯』必須由當時可觀察的訊號定義，例如前日已知排名惡化或價格條件，不能用後來跌很多才回頭貼標籤。
3. 『快砍』尚未測：需獨立比較五日換股、以前日訊號觸發的次日均價退出，以及再進場／替換安排；不能用觸發當日最低價假設成交。固定百分比停損、排名退出、過熱降權是不同假說，不能混成一個。
4. 退出仍要符合20–30檔、現金低於25%、整千股與費稅；跌停／停牌不一定成交，需分清官賽均價模型和真實市場。多次賣出再買回可能增加成本或錯失反彈，現有結果不保證快速停損改善。
5. 本輪只做權重與既有機制統計，沒有啟動新的每日停損、LLM或交易。產業季節性後續計畫仍保留，歷史分類門檻未變。

## 統計限制與保存
24／48個起始位置區塊各5000次，普通95%用於描述；另保存家族300保守探索區間，但尾端精度有限。窗口重疊、歷史已看過、現況150名單偏誤、未知認購事件排除與官方帳務未全驗證仍在。不是650次獨立比賽，不泛化到所有權重方式或所有市場。
原Notebook cells269–272保存定義、9格新增計算、配對／四格統計與本報告；沿用原kernel，不重啟、不清除。原13000個四格窗口、既有持有對照及所有統計另存v16 JSON。所有有效點保留在差值圖與等權對非等權散點圖；0線、同值對角線與固定座標輔助比較。
'''
report_path_v16=recovery_root/'等權非等權_統計與選股退出解讀_v16_2026-10-15.md';report_path_v16.write_text(report_v16,encoding='utf-8')
import plotly.graph_objects as go_v16
from plotly.offline import plot as plot_v16
fig_diff_v16=go_v16.Figure();fig_pair_v16=go_v16.Figure();fig_forest_v16=go_v16.Figure();plot_values_v16=[];diff_values_v16=[]
for modeindex,(suffix,tagA,tagB,label) in enumerate([('full','M0','E0','相同完整調整'),('skip','M5','E5','相同小額略過')]):
    for ix,family in enumerate(family_signal_v15):
        candidate=family+'_'+tagA;base=family+'_'+tagB;data=weight_stats_v16[family+'_'+suffix];indices=np.flatnonzero(common_factor_v16);x=return_factor_v16[base][common_factor_v16]*100;y=return_factor_v16[candidate][common_factor_v16]*100;diff=y-x;visible=modeindex==0;color=palette_v15[ix]
        cd=[[str(dates[int(window_starts_v15[i])]),str(dates[int(window_starts_v15[i])+23]),float(return_factor_v16[base][i]*100),float(return_factor_v16[candidate][i]*100)] for i in indices]
        fig_diff_v16.add_trace(go_v16.Box(x=[ix]*len(diff),y=diff,boxpoints=False,hoverinfo='skip',fillcolor=color,opacity=.2,line_color=color,visible=visible,showlegend=False))
        fig_diff_v16.add_trace(go_v16.Scatter(x=ix+np.linspace(-.16,.16,len(diff)),y=diff,mode='markers',marker={'size':5,'opacity':.55,'color':color},customdata=cd,hovertemplate='%{customdata[0]}～%{customdata[1]}<br>等權 %{customdata[2]:.3f}%<br>非等權 %{customdata[3]:.3f}%<br>差值 %{y:.3f}pp<extra>'+family_labels_v15[family]+'／'+label+'</extra>',visible=visible,showlegend=False))
        median=float(np.median(diff));fig_diff_v16.add_trace(go_v16.Scatter(x=[ix-.22,ix+.22],y=[median,median],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',visible=visible,showlegend=False))
        fig_pair_v16.add_trace(go_v16.Scatter(x=x,y=y,mode='markers',name=family_labels_v15[family],marker={'size':5,'opacity':.45,'color':color},customdata=cd,hovertemplate='%{customdata[0]}～%{customdata[1]}<br>等權 %{x:.3f}%<br>非等權 %{y:.3f}%<extra>'+family_labels_v15[family]+'／'+label+'</extra>',visible=visible))
        plot_values_v16.extend(x.tolist()+y.tolist());diff_values_v16.extend(diff.tolist())
for figure,firstcount in [(fig_diff_v16,15),(fig_pair_v16,5)]:
    figure.update_layout(template='plotly_white',height=490,margin={'l':75,'r':25,'t':85,'b':70},hovermode='closest',updatemenus=[{'type':'buttons','direction':'right','x':0,'y':1.16,'buttons':[{'label':'同樣完整調整（預設）','method':'update','args':[{'visible':[True]*firstcount+[False]*firstcount}]},{'label':'同樣略過小額調整','method':'update','args':[{'visible':[False]*firstcount+[True]*firstcount}]}]}])
fig_diff_v16.update_xaxes(tickmode='array',tickvals=list(range(5)),ticktext=list(family_labels_v15.values()),range=[-.5,4.5]);fig_diff_v16.update_yaxes(title='非等權－等權的費後報酬差（百分點）',range=[min(diff_values_v16)-.2,max(diff_values_v16)+.2],autorange=False);fig_diff_v16.add_hline(y=0,line_width=2,line_dash='dash',line_color='#333')
lo,hi=min(plot_values_v16)-2,max(plot_values_v16)+2
fig_pair_v16.update_xaxes(title='等權24日費後報酬（%）',range=[lo,hi],autorange=False);fig_pair_v16.update_yaxes(title='溫和非等權24日費後報酬（%）',range=[lo,hi],autorange=False,scaleanchor='x',scaleratio=1);fig_pair_v16.add_shape(type='line',x0=lo,x1=hi,y0=lo,y1=hi,line={'color':'#333','dash':'dash','width':2});fig_pair_v16.add_hline(y=0,line_color='#aab1bb');fig_pair_v16.add_vline(x=0,line_color='#aab1bb')
for block,color,offset in [(24,'#3974b4',-.12),(48,'#d98628',.12)]:
    means=[weight_stats_v16[family+'_full']['mean_excess']*100 for family in family_signal_v15];bounds=[weight_stats_v16[family+'_full'][f'block{block}_95'] for family in family_signal_v15]
    fig_forest_v16.add_trace(go_v16.Scatter(x=means,y=np.arange(5)+offset,mode='markers',name=f'{block}位置區塊普通95%區間',marker={'size':9,'color':color},error_x={'type':'data','symmetric':False,'array':[(ci[1]*100)-mean for ci,mean in zip(bounds,means)],'arrayminus':[mean-ci[0]*100 for ci,mean in zip(bounds,means)]},hovertemplate='平均差 %{x:.3f}pp<extra>'+str(block)+'位置區塊</extra>'))
fig_forest_v16.update_layout(template='plotly_white',height=400,margin={'l':110,'r':25,'t':25,'b':70},xaxis_title='純權重費後平均差（非等權－等權，百分點）');fig_forest_v16.update_yaxes(tickvals=list(range(5)),ticktext=list(family_labels_v15.values()));fig_forest_v16.add_vline(x=0,line_width=2,line_dash='dash',line_color='#333')
html_v16='''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>權重、選股與退出配對統計 v16</title><style>body{font:16px/1.7 system-ui,sans-serif;color:#203043;background:#f4f7fa;margin:0}main{max-width:1240px;margin:24px auto;padding:24px;background:white;border-radius:14px}h1{font-size:27px}h2{font-size:21px}.note{background:#fff4dc;padding:14px;border-left:4px solid #d49423}.scroll{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}th,td{padding:9px;border-bottom:1px solid #dce3eb;text-align:right;white-space:nowrap}th:first-child,td:first-child{text-align:left}</style><main><h1>等權與非等權：差多少？選股與退出呢？</h1><p class="note">本輪只對溫和權重1.25至0.75做結論：同樣完整調整時，五家族平均差約−0.12～＋0.19pp／24日，98%～99%窗口同漲同跌。這支持降低微調權重的優先度，不等於證明任何非等權都無差。選股與退出值得研究，但尚未證實越快砍越好。</p><p>為排除省費規則影響，補齊各家族等權／非等權×完整調整／小額略過四格。主權重比較同313窗口；13,000四格結果中11格引用、9格新算。不是新增20種選股策略。</p><h2>純權重配對統計：雙方完整調整</h2><div class="scroll">'''+table_external_v12(weight_headers_v16,weight_rows_v16)+'</div>'+plot_v16(fig_forest_v16,output_type='div',include_plotlyjs=True,config={'displaylogo':False,'responsive':True})+'''<p>95%區間是未校正多重比較的描述；有些不跨0，不可說所有配置統計上完全無差。窗口重疊，已用24／48位置區塊，資料已看過仍不是未見驗證。</p><h2>勝率與期望值：相同313窗口</h2><div class="scroll">'''+table_external_v12(payoff_headers_v16,payoff_rows_v16)+'''</div><p>勝率是費後正報酬比例，期望值為平均24日費後報酬。不是勝過另一配置的比例。非等權均值不能和v15含小額略過的配置混稱純權重效果。</p><h2>每個窗口的差值</h2><p>每個點是同日期非等權減等權，黑色虛線0表示沒有差異，桃紅為中位數。切換保持相同規則的另一組配對，座標固定；正差非等權較好。</p>'''+plot_v16(fig_diff_v16,output_type='div',include_plotlyjs=False,config={'displaylogo':False,'responsive':True})+'''<h2>等權對非等權：同值對角線</h2><p>點在對角線上表示兩者相同，上方非等權較好、下方等權較好。兩軸同尺度，所有313個窗口皆保留。</p>'''+plot_v16(fig_pair_v16,output_type='div',include_plotlyjs=False,config={'displaylogo':False,'responsive':True})+'''<h2>實務幅度敏感度</h2><div class="scroll">'''+table_external_v12(['家族','90%區間都在±0.10pp內','±0.25pp內','±0.50pp內'],equiv_rows_v16)+'''</div><p>沒有顯著差不等於等價。這三個幅度是看圖後固定的報告尺度，不是使用者的效益門檻，也不是前瞻等價試驗。普通95%有差與實務上小可以同時成立。</p><h2>初始選股與中途動作：另外518個共同窗口</h2><div class="scroll">'''+table_external_v12(['機制','共同窗口','費後平均增量','費稅變化','48位置區塊普通95%區間'],mechanism_rows_v16)+'''</div><p>初始排名＋0.833pp、五日重選／再平衡−0.395pp都只是對指定對照的點估計，區間跨0。後者平均多付0.666pp費稅，不能據此推論快速停損有益，也不能推論永不換股最好。這518窗口與上方313不同，不能跨表比較效應大小。</p><div class="scroll">'''+table_external_v12(['節點','窗口','費後勝率','24日期望值','平均費稅','最差回撤'],node_rows_v16)+'''</div><h2>如何研究「選錯快砍」？</h2><p>先把選錯定義為當時可觀察的排名或價格條件，再以次日均價成交獨立比較退出、替換與再進場。不能用後來跌多少回頭貼標籤，或用當日最低價假設成交。仍須滿足20～30檔、現金比例與費稅；多次賣出買回可能增加成本、錯失反彈。這輪沒有新增每日停損規則。</p><h2>小額略過與限制</h2><div class="scroll">'''+table_external_v12(['家族／相同權重','略過費後增量','費稅變化','48位置區塊普通95%區間'],skip_rows_v16)+'''</div><p>固定150現況名單的偏誤、未知認購事件與來源覆蓋限制仍在。24日、5日調整、25檔、90%、整千股與費用未變；不能泛化到極端集中配置。產業季節性計畫保留，歷史分類門檻尚未通過。沒有新交易、LLM或排程。</p></main></html>'''
chart_path_v16=recovery_root/'等權非等權_配對差值與選股退出_v16_2026-10-15.html';chart_path_v16.write_text(html_v16,encoding='utf-8')
state_v16=json.loads((recovery_root/'A策略_無人值守研究狀態.json').read_text(encoding='utf-8'))
state_v16['weight_selection_v16']={'status':'completed_factorial_paired_statistics_no_new_stoploss','running':False,'definition_sha256':definition_sha_v16,'factor_cells':20,'new_cells':9,'reused_cells':11,'common20_windows':313,'mechanism_common_windows':518,'hold_references':'existingREFandAHdiagnosticonly','weight_mean_range_pp':[-.1203936640988727,.18796529721854724],'report':report_path_v16.name,'chart':chart_path_v16.name,'statistics':'等權非等權_配對與機制統計_v16_2026-10-15.json','notebook_cells':[269,272],'stoploss':'notexperimented;hypothesisonly','industry_plan':'unchangedv15PITgate','automation':'preservepaused'}
(recovery_root/'A策略_無人值守研究狀態.json').write_text(json.dumps(state_v16,ensure_ascii=False,indent=2),encoding='utf-8')
with (recovery_root/'A策略_無人值守研究流程.md').open('a',encoding='utf-8') as f:f.write('\n\n## v16 權重與交易略過拆解（2026-10-15）\n5家族四格20定義，11引用9新增5850窗口。純權重同313日期平均差−0.12～＋0.19pp／24日，同漲跌約98～99%；不等於所有非等權等價。另518共同日期初始排名／五日重選調整／權重／略過分解，前兩區間跨0，五日重選多付費稅，不能說越快砍越好。詳見「'+report_path_v16.name+'」。每日停損尚未測，產業季節性v15計畫保留，未改排程。\n')
print('V16_REPORT_READY',report_path_v16.name,chart_path_v16.name)


In [ ]:
# Verify independent factor settings, matched dates, accounting and every plotted paired point.
assert len(factor_results_v16)==20 and all(len(rows)==650 for rows in factor_results_v16.values())
assert sum(row['status']=='computed_new_factor_cell' for row in factor_reuse_v16.values())==9
assert int(common_factor_v16.sum())==313 and int(mechanism_mask_v16.sum())==518
for family in family_signal_v15:
    assert factor_results_v16[family+'_E0'] is family_results_v15[family+'_equal']
    assert factor_results_v16[family+'_M5'] is family_results_v15[family+'_mild_skip']
    for tag in ['E0','E5','M0','M5']:
        assert factor_specs_v16[family+'_'+tag]['tolerance']==(0 if tag.endswith('0') else .005)
        assert factor_specs_v16[family+'_'+tag]['weight']==('equal' if tag.startswith('E') else 'mild')
    for suffix in ['full','skip']:
        row=weight_stats_v16[family+'_'+suffix];assert row['n']==313
        assert np.isclose(row['mean_excess'],row['candidate_stats']['expectancy']-row['base_stats']['expectancy'])
        for block in [24,48]:assert row[f'block{block}_95'][0]<=row[f'block{block}_95'][1] and row[f'block{block}_90'][0]<=row[f'block{block}_90'][1]
        for m in [row['candidate_stats'],row['base_stats']]:
            negative=1-m['win_rate']-m['flat_count']/m['n'];assert np.isclose(m['expectancy'],m['win_rate']*(m['mean_win'] or 0)+negative*(m['mean_loss'] or 0))
for index in range(650):
    assert np.isclose(factor_results_v16['MOM_E0'][index]['path'][1],hold_results_v16[index]['path'][1],rtol=0,atol=1e-5),'A/AHinitialday mustmatch'
per_window_sum_v16=np.zeros(650)
for candidate,base in [('AH','REF'),('MOM_E0','AH'),('MOM_M0','MOM_E0'),('MOM_M5','MOM_M0')]:per_window_sum_v16+=return_factor_v16[candidate]-return_factor_v16[base]
assert np.allclose(per_window_sum_v16[mechanism_mask_v16],(return_factor_v16['MOM_M5']-return_factor_v16['REF'])[mechanism_mask_v16],rtol=0,atol=1e-12)
assert sum(len(t.y) for t in fig_diff_v16.data if t.type=='scatter' and t.mode=='markers')==3130
assert sum(len(t.y) for t in fig_pair_v16.data if t.type=='scatter' and t.mode=='markers')==3130
assert sum(len(t.y) for t in fig_diff_v16.data if t.type=='scatter' and t.mode=='markers' and t.visible)==1565
assert sum(t.mode=='lines' and t.line.color=='#dc2477' for t in fig_diff_v16.data if t.type=='scatter')==10
assert fig_diff_v16.layout.yaxis.autorange is False and fig_pair_v16.layout.xaxis.autorange is False and fig_pair_v16.layout.yaxis.autorange is False
assert tuple(fig_pair_v16.layout.xaxis.range)==tuple(fig_pair_v16.layout.yaxis.range)
assert any(shape.x0==shape.y0 and shape.x1==shape.y1 and shape.x0!=shape.x1 for shape in fig_pair_v16.layout.shapes)
assert chart_path_v16.stat().st_size>1000000
verification_v16={'factor_cells':20,'new_cells':9,'new_windows':5850,'referenced_cells':11,'common20_n':313,'mechanism_common_n':518,'A_AH_initial_day_path_checks':650,'factorial_same_source':True,'paired_EV_identity':True,'conditional_payoff_identity':True,'per_window_mechanism_additivity':True,'difference_plot_points':3130,'paired_scatter_points':3130,'default_points_each':1565,'same_scale_diagonal':True,'fixed_axis_buttons':True,'daily_stoploss_not_tested':True,'verification':'programmaticaccounting/source/layoutchecks,notbrowservisualQA','kernel_restarted':False}
state_v16['weight_selection_v16']['verification']=verification_v16;state_v16['weight_selection_v16']['notebook_cells']=[269,273]
(recovery_root/'A策略_無人值守研究狀態.json').write_text(json.dumps(state_v16,ensure_ascii=False,indent=2),encoding='utf-8')
report_verified_v16=report_path_v16.read_text(encoding='utf-8').replace('cells269–272','cells269–273')+'\n\n## 數值驗證\n四格獨立因子規則、313與518同日期分母、650個動能重選／初始持有的首日資產完全相同、每個窗口四段差分加總、配對期望值差與條件損益恆等式皆通過。差值圖3130點、配對散點3130點、預設各1565點、桃紅中位數、同尺度對角線及固定座標切換通過程式檢查，未做瀏覽器目視QA。沒有新增每日停損規則，也沒有重啟kernel。\n'
report_path_v16.write_text(report_verified_v16,encoding='utf-8')
print('V16_VERIFIED',json.dumps(verification_v16,ensure_ascii=False))


In [ ]:
# v17 read-only retained kernel context; no server/kernel changes.
print('V17_CONTEXT',{'python':__import__('sys').executable,'pid':__import__('os').getpid(),'shape':shape,'n_dates':len(dates),'first':str(dates[0]),'last':str(dates[-1]),'factor_keys':list(factor_results_v16),'row_keys':list(factor_results_v16['MOM_E0'][0])})
print('V17_ENGINE_SOURCE',family_engine_source_v15)
print('V17_SCORE_SOURCE',score_code_v15)
print('V17_SETTLE_SOURCE',ast.get_source_segment(base_book_v15.cells[10].source,next(n for n in ast.parse(base_book_v15.cells[10].source).body if isinstance(n,ast.FunctionDef) and n.name=='settle')))
print('V17_ROW_SAMPLE',json.dumps({k:v for k,v in factor_results_v16['MOM_E0'][100].items() if k!='path'},default=serial_family_v15))
print('V17_SELECTED_METADATA',[(name,type(globals()[name]).__name__,getattr(globals()[name],'shape',None)) for name in ['close','flow','scores_v15','block_indices_v15','factor_results_v16']])


In [ ]:
# v17 fixed descriptive diagnostics: two equal-weight families, no new trading rule.
definition_v17={'families':['MOM_E0','FLOW_E0'],'common_dates_only':True,'allocation':'equal,90%,25 names,5day,24sessions,existingfees','diagnostics':['unique_initial_decision_set_overlap','paired_return_and_loss_overlap','fixed_trade_per_stock_PnLattribution','market_down_capture','past5return_and_flow_change_atdecision'],'not_tests':['daily_stoploss','optimizedblend','causalnews','seasonality'],'bias':'current150universe;overlapwindows;previouslyseenresults;descriptiveonly'}
(recovery_root/'動能法人_重疊與虧損研究定義_v17_2026-10-15.json').write_text(json.dumps(definition_v17,ensure_ascii=False,indent=2),encoding='utf-8')
mask_v17=valid_factor_v16['MOM_E0']&valid_factor_v16['FLOW_E0'];ix_v17=np.flatnonzero(mask_v17)
print('V17_START',len(ix_v17),flush=True)
# Instrument only the retained engine, preserving its orders and accounting.
instrument_v17=family_engine_source_v15.replace('def simulate_family_v15(', 'def trace_family_v17(')
instrument_v17=instrument_v17.replace('    for offset,t in enumerate(range(start,start+horizon)):', '    attribution=np.zeros(len(stock_ids)); daily=[]\n    for offset,t in enumerate(range(start,start+horizon)):')
instrument_v17=instrument_v17.replace('        before=shares.copy()', '        before=shares.copy(); old_receivable=receivable_shares.copy(); old_mark=np.nan_to_num(mark[t-1]); trade_pnl=np.zeros(len(stock_ids)); daycost=np.zeros(len(stock_ids))')
instrument_v17=instrument_v17.replace('            cash,shares,traded,cost,missing=settle(', '            delta=target-shares; executed=np.where(good_exec_v15[t],delta,0); safe=np.where(good_exec_v15[t],execution[t],0); daycost=np.maximum(executed,0)*safe*FEE+np.maximum(-executed,0)*safe*(FEE+TAX); trade_pnl=-executed*safe-daycost\n            cash,shares,traded,cost,missing=settle(')
instrument_v17=instrument_v17.replace('        values.append(aum)', '        values.append(aum)\n        pnl=shares*np.nan_to_num(mark[t])-before*old_mark+trade_pnl+before*cash_div_v15[t]+before*stock_div_v15[t]*np.nan_to_num(mark[t])+old_receivable*(np.nan_to_num(mark[t])-old_mark)\n        attribution+=pnl\n        daily.append({"t":t,"held":np.flatnonzero(shares>0).tolist(),"before":np.flatnonzero(before>0).tolist(),"weights":weights.copy(),"pnl":pnl.copy(),"fee":daycost.copy(),"rebalanced":rebalance})')
instrument_v17=instrument_v17.replace("'path':vals}","'path':vals,'attribution':attribution/INITIAL,'daily':daily}")
exec(instrument_v17,globals())
traces_v17={key:{} for key in ['MOM_E0','FLOW_E0']}
for key,signal in [('MOM_E0','A'),('FLOW_E0','PURE_FLOW')]:
    for idx in ix_v17:
        tr=trace_family_v17(int(window_starts_v15[idx]),signal);saved=factor_results_v16[key][idx]
        assert not tr['invalid'] and tr['violations']==0
        for field in ['return','cost','turnover','mdd']:assert np.isclose(tr[field],saved[field],rtol=0,atol=1e-12)
        assert np.isclose(tr['attribution'].sum(),tr['return'],rtol=0,atol=1e-10)
        traces_v17[key][int(idx)]=tr
    print('V17_TRACE_DONE',key,len(traces_v17[key]),flush=True)
def blockmean_v17(series,mask):
    x=np.where(mask,series,np.nan);out={}
    for b,inds in block_indices_v15.items():out[str(b)]=np.nanquantile(np.nanmean(x[inds],axis=1),[.025,.975]).tolist()
    return out
x_v17=return_factor_v16['MOM_E0'][mask_v17];y_v17=return_factor_v16['FLOW_E0'][mask_v17]
blend_v17=(x_v17+y_v17)/2
overlap_v17=[];decision_v17={};regimes_v17={};lossdetails_v17={}
market_daily_v17=np.nanmean(signal_returns_v15,axis=1)
for idx in ix_v17:
    a=traces_v17['MOM_E0'][int(idx)];b=traces_v17['FLOW_E0'][int(idx)];sa=set(a['daily'][0]['held']);sb=set(b['daily'][0]['held'])
    decision_v17[a['start']]={'date':str(dates[a['start']]),'intersection':len(sa&sb),'jaccard':len(sa&sb)/len(sa|sb),'momentum':[stock_ids[j] for j in sorted(sa)],'flow':[stock_ids[j] for j in sorted(sb)],'common':[stock_ids[j] for j in sorted(sa&sb)]}
    ov=[len(set(da['held'])&set(db['held'])) for da,db in zip(a['daily'],b['daily'])]
    overlap_v17.append({'index':int(idx),'start':str(dates[a['start']]),'end':str(dates[a['end']]),'initial_common':len(sa&sb),'mean_daily_common':float(np.mean(ov)),'MOM':a['return'],'FLOW':b['return'],'market':float(np.prod(1+market_daily_v17[a['start']:a['end']+1])-1)})
market_window_v17=np.array([r['market'] for r in overlap_v17])
for key,signal in [('MOM_E0','A'),('FLOW_E0','PURE_FLOW')]:
    returns=return_factor_v16[key][mask_v17];costs=cost_factor_v16[key][mask_v17];loss=returns<0
    regimes_v17[key]={'all':metric_factor_v16(key,mask_v17),'mean_CI':blockmean_v17(return_factor_v16[key],mask_v17),'loss_count':int(loss.sum()),'fee_sign_flip_fixed_trades':int(np.sum(loss&(returns+costs>=0))),'loss_mean_cost':float(costs[loss].mean()),'loss_no_fee_fixed_trade_mean':float((returns+costs)[loss].mean()),'market_negative_loss_windows':int(np.sum(loss&(market_window_v17<0))),'regimes':{}}
    for name,cond in [('market_down',market_window_v17<0),('market_up_or_flat',market_window_v17>=0)]:
        z=returns[cond];regimes_v17[key]['regimes'][name]={'n':len(z),'win_rate':float(np.mean(z>0)),'expectancy':float(z.mean()),'market_mean':float(market_window_v17[cond].mean())}
    details=[]
    for idx in ix_v17:
        tr=traces_v17[key][int(idx)];att=tr['attribution'];worst=np.argsort(att)[:5]
        neg=-att[att<0].sum();topnegative=-att[worst][att[worst]<0].sum()
        details.append({'index':int(idx),'start':str(dates[tr['start']]),'end':str(dates[tr['end']]),'return':tr['return'],'cost':tr['cost'],'mdd':tr['mdd'],'top5_negative_share':float(topnegative/neg) if neg>0 else None,'top5':[{'stock':stock_ids[j],'contribution':float(att[j])} for j in worst],'stock_attribution':{stock_ids[j]:float(att[j]) for j in np.flatnonzero(np.abs(att)>1e-12)}})
    lossdetails_v17[key]=details
    regimes_v17[key]['worst5']=sorted(details,key=lambda r:r['return'])[:5]
    regimes_v17[key]['loss_top5_negative_share_mean']=float(np.mean([r['top5_negative_share'] for r in details if r['return']<0 and r['top5_negative_share'] is not None]))
# Decision-time observables across unique rebalance dates: retrospective association, not causal or predictive validation.
decision_features_v17={}
for key in traces_v17:
    unique={}
    for tr in traces_v17[key].values():
        for d in tr['daily']:
            if not d['rebalanced']:continue
            t=d['t'];held=d['held'];j=np.array(held);end=min(t+5,len(dates))
            fwd=float(np.mean(np.prod(1+signal_returns_v15[t:end,j],axis=0)-1))
            prior=float(np.mean(np.prod(1+signal_returns_v15[t-5:t,j],axis=0)-1))
            oldflow=np.sum(flow[t-20:t-5,j],axis=0)/np.sum(volume[t-20:t-5,j],axis=0);newflow=np.sum(flow[t-5:t,j],axis=0)/np.sum(volume[t-5:t,j],axis=0)
            unique[t]={'date':str(dates[t]),'past5_return':prior,'flow_change':float(np.mean(newflow-oldflow)),'forward5_gross_selected_mean':fwd,'forward5_sessions':end-t}
    vals=list(unique.values());full=[r for r in vals if r['forward5_sessions']==5];cuts=np.quantile([r['past5_return'] for r in full],[1/3,2/3]);groups={}
    for name,pred in [('past5_low',lambda r:r['past5_return']<=cuts[0]),('past5_middle',lambda r:cuts[0]<r['past5_return']<=cuts[1]),('past5_high',lambda r:r['past5_return']>cuts[1]),('flow_weakening',lambda r:r['flow_change']<0),('flow_strengthening',lambda r:r['flow_change']>=0)]:
        z=[r['forward5_gross_selected_mean'] for r in full if pred(r)];groups[name]={'n':len(z),'mean':float(np.mean(z)),'positive_rate':float(np.mean(np.array(z)>0))}
    decision_features_v17[key]={'unique_n':len(full),'groups':groups,'rows':vals,'warning':'fullsampletertilesposthoc;grossfixedselectedbasket5daynotnetstrategy;overlapping;noactionrulevalidated'}
pair_v17={'n':len(x_v17),'correlation':float(np.corrcoef(x_v17,y_v17)[0,1]),'both_loss':int(np.sum((x_v17<0)&(y_v17<0))),'mom_only_loss':int(np.sum((x_v17<0)&(y_v17>=0))),'flow_only_loss':int(np.sum((x_v17>=0)&(y_v17<0))),'neither_loss':int(np.sum((x_v17>=0)&(y_v17>=0))),'flow_minus_mom':paired_factor_v16('FLOW_E0','MOM_E0'),'blend_diagnostic':{'win_rate':float(np.mean(blend_v17>0)),'expectancy':float(blend_v17.mean()),'P10':float(np.quantile(blend_v17,.1)),'worst':float(blend_v17.min()),'warning':'50/50independentfundedsleevesreturnidentityonly;combinedholdings/limits/feesnotvalidated'}}
summary_v17={'definition':definition_v17,'common_indices':ix_v17.tolist(),'pair':pair_v17,'initial_overlap':{'n':len(decision_v17),'mean':float(np.mean([r['intersection'] for r in decision_v17.values()])),'median':float(np.median([r['intersection'] for r in decision_v17.values()])),'min':min(r['intersection'] for r in decision_v17.values()),'max':max(r['intersection'] for r in decision_v17.values())},'strategies':regimes_v17,'decisions':list(decision_v17.values()),'window_pairs':overlap_v17,'loss_attribution':lossdetails_v17,'decision_features':decision_features_v17,'market_warning':'samecurrent150equalweightadjustedsignalreturnproxy,notTWSEindex,notPITuniverse','CI_warning':'unadjusted24/48positionblockbootstrap;meanuncertaintynotpredictioninterval;notindependent650trials'}
(recovery_root/'動能法人_重疊與虧損統計_v17_2026-10-15.json').write_text(json.dumps(summary_v17,ensure_ascii=False,indent=2,default=serial_family_v15),encoding='utf-8')
print('V17_RESULTS',json.dumps({'pair':{k:v for k,v in pair_v17.items() if k!='flow_minus_mom'},'overlap':summary_v17['initial_overlap'],'strategies':regimes_v17,'features':{k:{'unique_n':v['unique_n'],'groups':v['groups']} for k,v in decision_features_v17.items()}},ensure_ascii=False,default=serial_family_v15),flush=True)


In [ ]:
# Enforce complete forward-basket data; missing is not a negative outcome.
for key in decision_features_v17:
    rows=decision_features_v17[key]['rows'];validrows=[];excluded=Counter()
    for r in rows:
        if r['forward5_sessions']!=5:excluded['short_forward_window']+=1;continue
        if not all(np.isfinite(r[f]) for f in ['past5_return','flow_change','forward5_gross_selected_mean']):excluded['missing_full_basket_return_or_feature']+=1;continue
        validrows.append(r)
    cuts=np.quantile([r['past5_return'] for r in validrows],[1/3,2/3]);groups={}
    for name,pred in [('past5_low',lambda r:r['past5_return']<=cuts[0]),('past5_middle',lambda r:cuts[0]<r['past5_return']<=cuts[1]),('past5_high',lambda r:r['past5_return']>cuts[1]),('flow_weakening',lambda r:r['flow_change']<0),('flow_strengthening',lambda r:r['flow_change']>=0)]:
        z=np.array([r['forward5_gross_selected_mean'] for r in validrows if pred(r)]);groups[name]={'n':len(z),'mean':float(z.mean()),'positive_rate':float(np.mean(z>0))}
    decision_features_v17[key].update({'unique_n':len(validrows),'excluded':dict(excluded),'groups':groups,'rows':[{k:(None if isinstance(v,float) and not np.isfinite(v) else v) for k,v in r.items()} for r in rows]})
    regimes_v17[key]['years']={year:metric_factor_v16(key,mask_v17&ymask) for year,ymask in year_masks_v15.items() if np.any(mask_v17&ymask)}
    selected=[]
    for row in sorted(lossdetails_v17[key],key=lambda r:r['return']):
        if row['return']>=0:break
        if all(abs(int(window_starts_v15[row['index']])-int(window_starts_v15[old['index']]))>=24 for old in selected):selected.append(row)
        if len(selected)==3:break
    regimes_v17[key]['worst3_nonoverlap']=selected
summary_v17['decision_features']=decision_features_v17
json_path_v17=recovery_root/'動能法人_重疊與虧損統計_v17_2026-10-15.json'
json_path_v17.write_text(json.dumps(summary_v17,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
import plotly.graph_objects as go
import html
fig_pair_v17=go.Figure(go.Scatter(x=x_v17*100,y=y_v17*100,mode='markers',marker={'size':7,'color':[r['initial_common'] for r in overlap_v17],'colorscale':'Viridis','colorbar':{'title':'重疊檔數'}},text=[r['start']+'～'+r['end'] for r in overlap_v17],hovertemplate='%{text}<br>動能 %{x:.2f}%<br>純法人 %{y:.2f}%<br>重疊 %{marker.color} 檔<extra></extra>'))
lim=[float(min(x_v17.min(),y_v17.min())*100-2),float(max(x_v17.max(),y_v17.max())*100+2)]
fig_pair_v17.add_shape(type='line',x0=lim[0],x1=lim[1],y0=lim[0],y1=lim[1],line={'color':'#888','dash':'dot'})
fig_pair_v17.add_hline(y=0,line_color='#222');fig_pair_v17.add_vline(x=0,line_color='#222')
fig_pair_v17.update_layout(title='全部511個配對窗口：同漲同跌與持股重疊',xaxis={'title':'動能費後24日報酬 %','range':lim},yaxis={'title':'純法人費後24日報酬 %','range':lim,'scaleanchor':'x','scaleratio':1},height=660,template='plotly_white')
fig_overlap_v17=go.Figure(go.Scatter(x=[r['date'] for r in decision_v17.values()],y=[r['intersection'] for r in decision_v17.values()],mode='markers',marker={'size':5},text=['、'.join(r['common']) for r in decision_v17.values()],hovertemplate='%{x}<br>重疊 %{y} 檔<br>%{text}<extra></extra>'))
fig_overlap_v17.update_layout(title='每個有效起始日的實際重疊股票（全點）',yaxis={'title':'兩組25檔的交集檔數','range':[-.5,25.5]},height=350,template='plotly_white')
def pct17(x):return f'{x*100:.2f}%'
lines17=['# 動能與純法人：持股重疊、共同虧損與風險診斷 v17','', '## 口徑與信賴區間','只引用既有兩個等權方法，25檔、90%投入、五日調整、24交易日、整千股及既有費稅。只取兩者共同有效511窗口，與v16五家族共同313窗口不同，數字不能直接混比。重播1022個窗口的報酬、費用、換手及回撤皆與原結果一致；每股損益合計也逐窗口核對。','v16差值圖的區間是「非等權減等權的平均效果」不確定性，不是動能本身的虧損範圍。本次另列各策略平均報酬的24／48位置區塊95%區間；平均值區間也不是下一場報酬的預測區間。','', '## 相同窗口的勝率與期望值','|策略|窗口數|淨勝率|24日期望值|平均獲利|平均虧損|平均費稅|P10|最差窗口回撤|','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for key,label in [('MOM_E0','動能'),('FLOW_E0','純法人買超')]:
    s=regimes_v17[key];m=s['all'];lines17.append('|'+label+'|'+str(m['n'])+'|'+'|'.join(pct17(m[f]) for f in ['win_rate','expectancy','mean_win','mean_loss','mean_cost','P10','worst_mdd'])+'|')
    ci=s['mean_CI']['48'];lines17.append(f'\n{label}平均淨報酬的48日區塊95%區間：{pct17(ci[0])}～{pct17(ci[1])}。\n')
lines17+=['','## 是否重疊？',f"平均交集 {summary_v17['initial_overlap']['mean']:.2f}/25檔（約28.5%），中位數7檔、最少0檔、最多16檔。這是511個不同起始日的名單；同一段行情可能重複出現在滑動窗口。報酬相關 {pair_v17['correlation']:.3f}。",'|窗口結果|個數|','|---|---:|',f"|兩者都虧|{pair_v17['both_loss']}|",f"|只有動能虧|{pair_v17['mom_only_loss']}|",f"|只有法人虧|{pair_v17['flow_only_loss']}|",f"|兩者皆不虧|{pair_v17['neither_loss']}|",'兩者能提供部分互補，不能當成兩個獨立的避險策略。各半資金袖套的算術診斷勝率72.99%、期望值4.11%，並未同時優於兩者；這不是已驗證的可交易混合策略，合併檔數、規則、整張及費用尚未檢查。','', '## 為什麼會賠？本次能證明到哪裡','市場診斷使用同一目前150檔的等權調整報酬代理，非官方加權指數、也不是歷史全市場。窗口是否下跌是事後分類，不能直接拿來交易。']
for key,label in [('MOM_E0','動能'),('FLOW_E0','純法人買超')]:
    s=regimes_v17[key];down=s['regimes']['market_down'];up=s['regimes']['market_up_or_flat'];lines17 +=[f'\n### {label}',f"虧損{s['loss_count']}窗口中，{s['market_negative_loss_windows']}個與150檔市場代理同步下跌。市場代理下跌的{down['n']}窗口，策略勝率{pct17(down['win_rate'])}、期望值{pct17(down['expectancy'])}；代理未跌的{up['n']}窗口則為{pct17(up['win_rate'])}、{pct17(up['expectancy'])}。",f"虧損窗口平均費稅{pct17(s['loss_mean_cost'])}；固定原交易、只加回費稅後，平均仍虧{pct17(s['loss_no_fee_fixed_trade_mean'])}。只有{s['fee_sign_flip_fixed_trades']}個虧損窗口會因此轉為不虧；這是成本分解，不是重新模擬零費率。",f"虧損窗口中，最拖累五檔平均只占全部負損益的{pct17(s['loss_top5_negative_share_mean'])}；負損益通常還分散在其他股票，不能預設砍一兩檔就解決。這是全窗口累積歸因，可能包含超過初始25檔的換股。",'最差三個互不重疊窗口（避免把同次事件連列成多個獨立案例）：','|起訖|淨報酬|費稅|拖累前五檔及對初始資金貢獻|','|---|---:|---:|---|']
    for r in s['worst3_nonoverlap']:lines17.append('|'+r['start']+'～'+r['end']+'|'+pct17(r['return'])+'|'+pct17(r['cost'])+'|'+'；'.join(z['stock']+' '+pct17(z['contribution']) for z in r['top5'])+'|')
lines17+=['','## 訊號失效的探索','動能依過去20／60日漲幅排名；純法人依過去20日淨買超／成交量排名，不是法人保證未來上漲，也沒有要求每一檔淨買超必須大於零。兩者都持有約90%股票，多頭曝險仍在。過去強勢是否反轉、買盤是否衰減，是待驗證機制，不能憑排名或這份歸因斷言原因。','以下獨立決策日分組僅為找線索：看固定當日股票未來5日的等權調整毛報酬，不是完整策略淨報酬。完整籃子資料缺失就整日排除，不把缺值當跌；全樣本三分位為事後分組，沒有多重校正或樣本外證據。','|方法|分組|完整決策日數|未來5日籃子平均毛報酬|正報酬率|','|---|---|---:|---:|---:|']
for key,label in [('MOM_E0','動能'),('FLOW_E0','法人')]:
    for name,m in decision_features_v17[key]['groups'].items():lines17.append(f"|{label}|{name}|{m['n']}|{pct17(m['mean'])}|{pct17(m['positive_rate'])}|")
    lines17.append(f"\n{label}排除：{decision_features_v17[key]['excluded']}。\n")
lines17+=['','## 如何使用這次結果','可以把動能與純法人保留為下一階段的主要候選，但目前不能宣稱已選出未來有效策略。動能在這份樣本的平均獲利較大，虧損幅度也更大；純法人勝率較高、平均虧損較小，仍有約19%的窗口內最大回撤。高勝率與正均值不能取代風險檢查。','下一輪先固定少量風控假說：選股訊號失效後剔除、股票整體風險降曝險、換股加緩衝避免反覆交易。每一項都要用決策當時可見訊號，單獨比較費後勝率、期望值、下行、漏掉反彈及費用；在訓練期間選條件、之後期間鎖定驗證，不能用本次事後市場下跌標籤當進出場。此次沒有新增或調優這些交易規則。','限制：目前150檔股票池有存活／選擇偏誤；未知公司行動與不合法窗口排除；非每年10月單獨樣本；511個窗口互相重疊；未做新資料的樣本外確認，不能稱511次獨立勝負。沒有使用LLM新聞判讀，也沒有爬新資料。','', '## 保存與驗證','原Notebook cells274–276保存程式與輸出；沿用PID2524，未重啟。1022個重播窗口與原引擎一致，1022個每股損益加總核對；只讀取既有價格、法人與公司行動狀態。圖包含511配對點與511起始名單點，零線及相同尺度；程式檢查完成，未做瀏覽器目視QA。']
report_text_v17='\n'.join(lines17)
report_path_v17=recovery_root/'動能法人_持股重疊與虧損歸因_v17_2026-10-15.md';report_path_v17.write_text(report_text_v17,encoding='utf-8')
chart_path_v17=recovery_root/'動能法人_持股重疊與虧損歸因_v17_2026-10-15.html'
chart_path_v17.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>動能與純法人：重疊及虧損診斷 v17</title><style>body{font:16px/1.7 system-ui;margin:28px auto;max-width:1150px;padding:0 20px;color:#172539}pre{white-space:pre-wrap;font:15px/1.8 system-ui;background:#f5f7fb;padding:20px}</style><h1>動能與純法人：重疊及虧損診斷</h1><p>511個相同有效窗口；費用後報酬。事後歸因尚未證明可交易的風控規則。</p>'+fig_pair_v17.to_html(full_html=False,include_plotlyjs=True)+fig_overlap_v17.to_html(full_html=False,include_plotlyjs=False)+'<h2>完整統計、虧損案例與限制</h2><pre>'+html.escape(report_text_v17)+'</pre></html>',encoding='utf-8')
assert len(fig_pair_v17.data[0].x)==511 and len(fig_overlap_v17.data[0].x)==511
assert sum(pair_v17[k] for k in ['both_loss','mom_only_loss','flow_only_loss','neither_loss'])==511
assert all(np.isfinite(m['mean']) for f in decision_features_v17.values() for m in f['groups'].values())
state_path_v17=recovery_root/'A策略_無人值守研究狀態.json';state_final_v17=json.loads(state_path_v17.read_text(encoding='utf-8'));state_final_v17['overlap_loss_v17']={'status':'completed','running':False,'n_common':511,'traces_verified':1022,'definition':'動能法人_重疊與虧損研究定義_v17_2026-10-15.json','outputs':[report_path_v17.name,chart_path_v17.name,json_path_v17.name],'notebook_cells':[274,276],'new_risk_rules_tested':False};state_path_v17.write_text(json.dumps(state_final_v17,ensure_ascii=False,indent=2),encoding='utf-8')
workflow_path_v17=recovery_root/'A策略_無人值守研究流程.md';workflow_text_v17=workflow_path_v17.read_text(encoding='utf-8');workflow_text_v17+='\n\n## v17 動能法人重疊與虧損診斷（完成）\n同有效511窗口，重播1022次核對原引擎與逐股會計歸因；平均起始重疊7.12檔、報酬相關0.762、共同虧損94窗口。主要報告：'+report_path_v17.name+'；未測新停損／混合配置，沒有自動重開排程。\n';workflow_path_v17.write_text(workflow_text_v17,encoding='utf-8')
print('V17_VERIFIED',{'n':511,'trace_checks':1022,'attribution_checks':1022,'plot_points':1022,'reports':[report_path_v17.name,chart_path_v17.name],'features':{k:{'n':v['unique_n'],'excluded':v['excluded'],'groups':v['groups']} for k,v in decision_features_v17.items()}})


In [ ]:
# Conservative future company-action exclusion and readable HTML tables.
for key in decision_features_v17:
    held_by_t={d['t']:d['held'] for tr in traces_v17[key].values() for d in tr['daily'] if d['rebalanced']}
    f=decision_features_v17[key];kept=[];unknown=0
    for r in f['rows']:
        t=di[datetime.date.fromisoformat(r['date'])]
        if r['forward5_sessions']!=5 or any(r[z] is None for z in ['past5_return','flow_change','forward5_gross_selected_mean']):continue
        if np.any(bad_action_v15[t:t+5,held_by_t[t]]):unknown+=1;continue
        kept.append(r)
    cuts=np.quantile([r['past5_return'] for r in kept],[1/3,2/3]);g={}
    for name,pred in [('past5_low',lambda r:r['past5_return']<=cuts[0]),('past5_middle',lambda r:cuts[0]<r['past5_return']<=cuts[1]),('past5_high',lambda r:r['past5_return']>cuts[1]),('flow_weakening',lambda r:r['flow_change']<0),('flow_strengthening',lambda r:r['flow_change']>=0)]:
        z=np.array([r['forward5_gross_selected_mean'] for r in kept if pred(r)]);g[name]={'n':len(z),'mean':float(z.mean()),'positive_rate':float(np.mean(z>0))}
    f['groups']=g;f['unique_n']=len(kept);f['excluded']['unknown_forward_company_action']=unknown
text17=report_path_v17.read_text(encoding='utf-8');start17=text17.index('|方法|分組|');end17=text17.index('## 如何使用',start17)
table17=['|方法|分組|完整決策日數|未來5日籃子平均毛報酬|正報酬率|','|---|---|---:|---:|---:|']
for key,label in [('MOM_E0','動能'),('FLOW_E0','法人')]:
    for name,m in decision_features_v17[key]['groups'].items():table17.append(f"|{label}|{name}|{m['n']}|{pct17(m['mean'])}|{pct17(m['positive_rate'])}|")
    table17.append(f"\n{label}排除：{decision_features_v17[key]['excluded']}。\n")
text17=text17[:start17]+'\n'.join(table17)+'\n\n'+text17[end17:]
text17=text17.replace('原Notebook cells274–276','原Notebook cells274–277').replace('完整籃子資料缺失就整日排除','完整籃子資料缺失或後續未知公司行動就整日排除')
text17+='\n市場代理只作描述：其中未知公司行動可能影響代理報酬，尚不是乾淨的官方市場因子；同步下跌統計不作因果歸因或交易門檻。未來五日籃子診斷已額外排除未知公司行動。\n'
report_path_v17.write_text(text17,encoding='utf-8');summary_v17['forward_action_filter']=True;json_path_v17.write_text(json.dumps(summary_v17,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
def render_report17(text):
    out=[];table=False
    for line in text.splitlines():
        if line.startswith('|'):
            if not table:out.append('<table>');table=True
            cells=line.strip('|').split('|')
            if all(set(c.strip())<=set('-: ') for c in cells):continue
            out.append('<tr>'+''.join('<td>'+html.escape(c)+'</td>' for c in cells)+'</tr>');continue
        if table:out.append('</table>');table=False
        if line.startswith('#'):
            level=len(line)-len(line.lstrip('#'));out.append(f'<h{level}>'+html.escape(line[level:].strip())+f'</h{level}>')
        elif line.strip():out.append('<p>'+html.escape(line)+'</p>')
    if table:out.append('</table>')
    return ''.join(out)
chart_path_v17.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>動能與純法人：重疊及虧損診斷 v17</title><style>body{font:16px/1.7 system-ui;margin:28px auto;max-width:1150px;padding:0 20px;color:#172539}table{border-collapse:collapse;width:100%;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#edf2f8;font-weight:bold}h2{margin-top:34px}p{overflow-wrap:anywhere}</style><h1>動能與純法人：重疊及虧損診斷</h1><p>511個相同有效窗口；費用後報酬。事後歸因尚未證明可交易的風控規則。</p>'+fig_pair_v17.to_html(full_html=False,include_plotlyjs=True)+fig_overlap_v17.to_html(full_html=False,include_plotlyjs=False)+render_report17(text17)+'</html>',encoding='utf-8')
state_final_v17['overlap_loss_v17']['notebook_cells']=[274,277];state_final_v17['overlap_loss_v17']['forward_action_filter']=True;state_path_v17.write_text(json.dumps(state_final_v17,ensure_ascii=False,indent=2),encoding='utf-8')
assert '<table>' in chart_path_v17.read_text(encoding='utf-8')
print('V17_FINAL',{'forward_features':{k:{'n':v['unique_n'],'groups':v['groups'],'excluded':v['excluded']} for k,v in decision_features_v17.items()},'files_verified':all(p.exists() for p in [report_path_v17,chart_path_v17,json_path_v17])})


In [ ]:
# v18 freeze six finite risk overlays before computing results; retained kernel only.
print('V18_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'shape':shape,'base_windows':len(window_starts_v15),'engine_hash':family_engine_hash_v15})
definition_v18={'families':['MOM','FLOW'],'overlays':['signal','exposure','both'],'baseline':'v16 E0 equal25 invested90 rebalance5 horizon24 lots1000 fees unchanged','signal':'daily preopen using t-1: held rank percentile<0.5 AND adjusted past5return<0; replace failed with highest same-family eligible not currently held nor failed; maintain25; retarget equal25 on event; regular5daybaseline overrides; no intraday stop','exposure':'daily t-1 same150 current-universe proxy; exclude unknown action daily; require>=120 validnames perday for20days; risk if trailing20 compounded mean daily return<0 AND fraction eligible names positive20return<0.5; invested82% risk else90%; adjust on statechange or regular5day; unknown leaves90, countunknown','both':'union signal/exposure event; explicit interaction not isolatedstoploss','candidates':6,'thresholds':'fixedmechanisticroundnumbersnotoptimized;historicaldataalreadyseen','split':'train window end<=2024-12-31; test window start>=2025-01-01; crossingpurged;common dates across8methods;notgenuineunseenOOS','selection':'trainingonly: adjusted24&48block lowerbound excess>0 AND P10>=base AND worstMDD>=base AND coverage>=80% AND train>=100; if none no forced choice; lock chosen then evaluate test','secondary_risktradeoff':'P10andworstMDDimprove while adjustedlowerbound excess>-.005; sensitivityresearchnotuserutilityorconfirmedimprovement','multiplicity':'sixoverlaycomparisons Bonferroni withinblock, bothblock required;5000bootstrap extreme-tail resolutionlimit','scope':'samecurrent150biasedpool,notfullPIT,unknowncompanyactions excluded,ActiveShare not certified,notformalcompetitionexecution','data_snapshot':snapshot_sha_v15}
definition_hash_v18=hashlib.sha256(json.dumps(definition_v18,sort_keys=True).encode()).hexdigest()
(recovery_root/'動能法人_失效剔除與降曝險定義_v18_2026-10-15.json').write_text(json.dumps(definition_v18,ensure_ascii=False,indent=2),encoding='utf-8')
source_v18=family_engine_source_v15.replace('def simulate_family_v15(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):','def simulate_risk_v18(start,strategy,overlay="none",allocation="equal",horizon=HORIZON,tolerance=0):')
source_v18=source_v18.replace('    for offset,t in enumerate(range(start,start+horizon)):', '    last_exposure=.9;events=[];unknown_risk=0\n    for offset,t in enumerate(range(start,start+horizon)):')
old="        rebalance=offset==0 or (strategy!='REF' and not strategy.endswith('H') and offset%REBALANCE==0)"
new="""        rank=scores_v15[strategy][t-1];eligible=np.flatnonzero(np.isfinite(rank))
        exposure=.9
        if overlay in ('exposure','both'):
            if np.isfinite(risk_state_v18[t-1]):exposure=.82 if risk_state_v18[t-1]>0 else .9
            else:unknown_risk+=1
        failed=np.array([],dtype=int)
        if offset>0 and overlay in ('signal','both'):
            r5=np.prod(1+signal_returns_v15[t-5:t],axis=0)-1
            failed=np.flatnonzero((shares>0)&np.isfinite(rank)&(rank<.5)&np.isfinite(r5)&(r5<0))
        regular=offset==0 or offset%REBALANCE==0
        exposure_change=abs(exposure-last_exposure)>1e-12
        rebalance=regular or len(failed)>0 or exposure_change"""
assert old in source_v18;source_v18=source_v18.replace(old,new)
oldselect="            selected=eligible[np.argsort(-rank[eligible],kind='stable')[:COUNT]]"
newselect="""            ranked=eligible[np.argsort(-rank[eligible],kind='stable')]
            if regular:selected=ranked[:COUNT]
            else:
                retained=[j for j in np.flatnonzero(shares>0) if j not in failed]
                add=[j for j in ranked if j not in retained and j not in failed][:COUNT-len(retained)]
                selected=np.array(retained+add,dtype=int)
                if len(selected)!=COUNT:invalid=True;break
            events.append({'t':t,'regular':regular,'failed':[stock_ids[j] for j in failed], 'exposure':exposure,'exposure_change':exposure_change})
            last_exposure=exposure"""
assert oldselect in source_v18;source_v18=source_v18.replace(oldselect,newselect)
source_v18=source_v18.replace('family_allocation_v15(selected,allocation)*previous_nav','(exposure/COUNT)*previous_nav')
source_v18=source_v18.replace("'path':vals}","'path':vals,'events':events,'unknown_risk_days':unknown_risk}")
# Exclude unknown actions from observable market proxy. No future information.
proxy_valid_v18=np.isfinite(signal_returns_v15)&~bad_action_v15
proxy_count_v18=proxy_valid_v18.sum(axis=1)
proxy_daily_v18=np.divide(np.where(proxy_valid_v18,signal_returns_v15,0).sum(axis=1),proxy_count_v18,out=np.full(len(dates),np.nan),where=proxy_count_v18>0)
risk_state_v18=np.full(len(dates),np.nan)
for q in range(60,len(dates)):
    rr=signal_returns_v15[q-19:q+1];ok=np.all(np.isfinite(rr)&~bad_action_v15[q-19:q+1],axis=0)
    if ok.sum()<120 or np.any(proxy_count_v18[q-19:q+1]<120):continue
    market20=np.prod(1+proxy_daily_v18[q-19:q+1])-1;breadth=np.mean(np.prod(1+rr[:,ok],axis=0)>1)
    risk_state_v18[q]=float(market20<0 and breadth<.5)
exec(source_v18,globals())
for signal,key in [('A','MOM_E0'),('PURE_FLOW','FLOW_E0')]:
    for idx in [0,100,200,400,649]:
        got=simulate_risk_v18(int(window_starts_v15[idx]),signal);old=factor_results_v16[key][idx]
        assert got['invalid']==old['invalid'] and got['violations']==old['violations']
        assert all(np.isclose(got[f],old[f],atol=1e-12,rtol=0) for f in ['return','cost','mdd','turnover'])
state_path_v18=recovery_root/'A策略_無人值守研究狀態.json';state_v18=json.loads(state_path_v18.read_text(encoding='utf-8'));state_v18['risk_overlay_v18']={'status':'defined_running','running':True,'definition_hash':definition_hash_v18,'candidates':6};state_path_v18.write_text(json.dumps(state_v18,ensure_ascii=False,indent=2),encoding='utf-8')
print('V18_DEFINED',definition_hash_v18,'10baselinecontrols matched; no kernel restart')


In [ ]:
# Execute only six new definitions, reuse two exact baselines.
results_v18={'MOM_base':factor_results_v16['MOM_E0'],'FLOW_base':factor_results_v16['FLOW_E0']}
for family,signal in [('MOM','A'),('FLOW','PURE_FLOW')]:
    for overlay in ['signal','exposure','both']:
        key=family+'_'+overlay
        results_v18[key]=[simulate_risk_v18(int(start),signal,overlay) for start in window_starts_v15]
        print('V18_DONE',key,'valid',sum(not r['invalid'] and r['violations']==0 for r in results_v18[key]),flush=True)
(recovery_root/'動能法人_失效剔除與降曝險逐窗口_v18_2026-10-15.json').write_text(json.dumps({'definition':definition_v18,'hash':definition_hash_v18,'results':results_v18},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
valid_v18={key:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for key,rows in results_v18.items()}
ret_v18={key:np.array([r['return'] for r in rows]) for key,rows in results_v18.items()};common_v18=np.logical_and.reduce(list(valid_v18.values()))
train_v18=np.array([dates[int(s)+23]<=datetime.date(2024,12,31) for s in window_starts_v15]);test_v18=np.array([dates[int(s)]>=datetime.date(2025,1,1) for s in window_starts_v15]);purged_v18=~(train_v18|test_v18)
def metrics18(key,mask):
    x=ret_v18[key][mask];rows=results_v18[key];n=int(mask.sum())
    if not n:return {'n':0}
    return {'n':n,'win_rate':float(np.mean(x>0)),'expectancy':float(x.mean()),'mean_win':float(x[x>0].mean()) if np.any(x>0) else None,'mean_loss':float(x[x<0].mean()) if np.any(x<0) else None,'P10':float(np.quantile(x,.1)),'mean_cost':float(np.mean([rows[i]['cost'] for i in np.flatnonzero(mask)])),'worst_mdd':float(min(rows[i]['mdd'] for i in np.flatnonzero(mask))),'median_mdd':float(np.median([rows[i]['mdd'] for i in np.flatnonzero(mask)])),'mean_turnover':float(np.mean([rows[i]['turnover'] for i in np.flatnonzero(mask)]))}
def contrast18(key,mask):
    base=key.split('_')[0]+'_base';x=ret_v18[key];b=ret_v18[base];d=x-b;pair=mask&valid_v18[key]&valid_v18[base];ids=np.flatnonzero(pair);out={'n':len(ids),'mean_excess':float(d[pair].mean()),'beat_base_rate':float(np.mean(d[pair]>0)),'candidate':metrics18(key,pair),'base':metrics18(base,pair),'base_up_underperformance':float(np.mean(d[pair&(b>0)]<0)) if np.any(pair&(b>0)) else None,'base_up_mean_excess':float(d[pair&(b>0)].mean()) if np.any(pair&(b>0)) else None,'base_down_mean_excess':float(d[pair&(b<0)].mean()) if np.any(pair&(b<0)) else None,'mean_cost_change':float(np.mean([results_v18[key][i]['cost']-results_v18[base][i]['cost'] for i in ids]))}
    for block,inds in block_indices_v15.items():
        z=np.where(pair,d,np.nan)[inds];cnt=np.isfinite(z).sum(axis=1);means=np.nansum(z,axis=1)[cnt>0]/cnt[cnt>0]
        out[f'CI{block}']=np.quantile(means,[.025,.975]).tolist();out[f'familyCI{block}']=np.quantile(means,[.05/12,1-.05/12]).tolist()
    return out
stats_v18={};train_candidates_v18={};selected_v18={}
for key in results_v18:
    if key.endswith('_base'):continue
    base=key.split('_')[0]+'_base';pair=valid_v18[key]&valid_v18[base];base_n=valid_v18[base].sum()
    stats_v18[key]={'own_pair':contrast18(key,pair),'common_all':contrast18(key,common_v18),'train':contrast18(key,common_v18&train_v18),'test':contrast18(key,common_v18&test_v18),'coverage_vs_base':float(pair.sum()/base_n),'invalid_count':sum(r['invalid'] for r in results_v18[key]),'violation_windows':sum(r['violations']>0 for r in results_v18[key]),'years':{year:contrast18(key,common_v18&ymask) for year,ymask in year_masks_v15.items() if np.any(common_v18&ymask)}}
    tr=stats_v18[key]['train'];coverage=float((pair&train_v18).sum()/max(1,(valid_v18[base]&train_v18).sum()));riskok=tr['candidate']['P10']>=tr['base']['P10'] and tr['candidate']['worst_mdd']>=tr['base']['worst_mdd'];lower=min(tr['familyCI24'][0],tr['familyCI48'][0]);tr['pass_improvement']=bool(tr['n']>=100 and coverage>=.8 and lower>0 and riskok);tr['risk_tradeoff_diagnostic']=bool(tr['n']>=100 and coverage>=.8 and lower>-.005 and tr['candidate']['P10']>tr['base']['P10'] and tr['candidate']['worst_mdd']>tr['base']['worst_mdd']);tr['coverage']=coverage
    train_candidates_v18[key]=tr['pass_improvement']
    events=[e for i in np.flatnonzero(common_v18) for e in results_v18[key][i]['events']]
    stats_v18[key]['event_counts_overlapping_windows']={'signal_nonregular':sum(bool(e['failed']) and not e['regular'] for e in events),'exposure_state_changes':sum(e['exposure_change'] for e in events),'warning':'repeatedwindowsnoteventindependence'}
for family in ['MOM','FLOW']:
    qualified=[key for key,yes in train_candidates_v18.items() if yes and key.startswith(family+'_')]
    selected_v18[family]=min(qualified,key=lambda k:stats_v18[k]['train']['candidate']['mean_cost']) if qualified else family+'_base'
summary_v18={'definition':definition_v18,'hash':definition_hash_v18,'common_n':int(common_v18.sum()),'train_n':int((common_v18&train_v18).sum()),'test_n':int((common_v18&test_v18).sum()),'purged_crossyear_n':int((common_v18&purged_v18).sum()),'common_indices':np.flatnonzero(common_v18).tolist(),'metrics':{key:metrics18(key,common_v18) for key in results_v18},'contrasts':stats_v18,'training_selected':selected_v18,'selected_test':{family:metrics18(key,common_v18&test_v18) for family,key in selected_v18.items()},'warning':'retrospectivefixedthreshold,notgenuineunseenOOS;baselinefallbacknotproofofoptimum;ActiveShareunverified'}
(recovery_root/'動能法人_失效剔除與降曝險統計_v18_2026-10-15.json').write_text(json.dumps(summary_v18,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V18_STATS',json.dumps({'n':summary_v18['common_n'],'train':summary_v18['train_n'],'test':summary_v18['test_n'],'metrics':summary_v18['metrics'],'selected':selected_v18,'contrasts':{k:{'delta':s['common_all']['mean_excess'],'CI48':s['common_all']['familyCI48'],'train_pass':s['train']['pass_improvement'],'tradeoff':s['train']['risk_tradeoff_diagnostic'],'test_delta':s['test']['mean_excess'],'feechange':s['common_all']['mean_cost_change'],'up_delta':s['common_all']['base_up_mean_excess'],'down_delta':s['common_all']['base_down_mean_excess'],'violations':s['violation_windows']} for k,s in stats_v18.items()}},default=serial_family_v15),flush=True)


In [ ]:
# Guard missing prior-close before integer sizing. Recompute affected invalid/violation rows only.
target18='            target=np.zeros_like(shares)'
guard18='            if np.any(~np.isfinite(close[t-1,selected]) | (close[t-1,selected]<=0)):\n                invalid=True;break\n'
assert source_v18.count(target18)==1
source_v18=source_v18.replace(target18,guard18+target18);exec(source_v18,globals())
old_valid18={k:v.copy() for k,v in valid_v18.items()};repaired18=0
for family,signal in [('MOM','A'),('FLOW','PURE_FLOW')]:
    for overlay in ['signal','exposure','both']:
        key=family+'_'+overlay
        for idx in np.flatnonzero(~old_valid18[key]):
            results_v18[key][int(idx)]=simulate_risk_v18(int(window_starts_v15[idx]),signal,overlay);repaired18+=1
        for idx in np.flatnonzero(old_valid18[key])[::100]:
            now=simulate_risk_v18(int(window_starts_v15[idx]),signal,overlay);old=results_v18[key][int(idx)]
            assert not now['invalid'] and now['violations']==0
            assert all(np.isclose(now[f],old[f],rtol=0,atol=1e-12) for f in ['return','cost','mdd','turnover'])
for key in results_v18:assert np.array_equal(old_valid18[key],np.array([not r['invalid'] and r['violations']==0 for r in results_v18[key]]))
(recovery_root/'動能法人_失效剔除與降曝險逐窗口_v18_2026-10-15.json').write_text(json.dumps({'definition':definition_v18,'hash':definition_hash_v18,'engine_source_sha256':hashlib.sha256(source_v18.encode()).hexdigest(),'results':results_v18},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
quality_eval18=nbformat.read(notebook_path_v14,as_version=4).cells[279].source
exec(quality_eval18[quality_eval18.index('valid_v18='):],globals())
print('V18_QUALITY',{'invalid_rows_guarded':repaired18,'valid_masks_unchanged':True,'no_kernel_restart':True})


In [ ]:
# Rebound diagnostic of actually triggered nonregular failed-name events, deduplicated by date/name.
rebounds18={}
for key in stats_v18:
    unique=set();vals=[];excluded=0
    for idx in np.flatnonzero(common_v18):
        for e in results_v18[key][idx]['events']:
            if e['regular']:continue
            for sid in e['failed']:unique.add((e['t'],sid))
    for t,sid in sorted(unique):
        j=si[sid];rr=signal_returns_v15[t:t+5,j]
        if len(rr)<5 or not np.all(np.isfinite(rr)) or np.any(bad_action_v15[t:t+5,j]):excluded+=1;continue
        vals.append(float(np.prod(1+rr)-1))
    rebounds18[key]={'unique_stock_dates':len(unique),'valid_n':len(vals),'excluded':excluded,'positive5day_rate':float(np.mean(np.array(vals)>0)) if vals else None,'mean5day_gross':float(np.mean(vals)) if vals else None,'warning':'fixedsoldnamescounterfactualpriceonlynotnetstrategygain;overlapstockdates;positivefuturedoesnotprovesellwrong'}
labels18={'MOM_base':'動能原策略','FLOW_base':'純法人原策略','MOM_signal':'動能＋失效換股','MOM_exposure':'動能＋降曝險','MOM_both':'動能＋兩者合併','FLOW_signal':'法人＋失效換股','FLOW_exposure':'法人＋降曝險','FLOW_both':'法人＋兩者合併'}
fmt18=lambda v:'—' if v is None else f'{v*100:.2f}%'
text18=['# 動能／純法人：失效換股與降曝險 v18','', '## 結論','六個新增候選沒有通過預先設定的訓練改善門檻，兩家族均保留原策略對照，不能稱原策略是最佳或已證實有效。降低曝險在這份資料改善最差回撤，但犧牲部分平均報酬；不能把風險／報酬取捨稱為全面改善。','', '## 預先固定規則','1. 原策略：每五日按原家族重新選25檔，股票90%，等權。','2. 失效換股：每日用前一日資訊，持股原家族排名百分位低於0.5且過去五日報酬負值時換掉；以原家族最高分、非現有持股且非當日失效股票補回，維持25檔。觸發時所有持股重新等權，因此效果包括其他持股重新配置與費用，不是純粹賣掉失效股票的因果效果。每五日原重選照常。沒有盤中停損。','3. 降曝險：同目前150檔市場代理過去20日累積負報酬，且完整資料股票20日上漲比例低於50%時，目標股票82%，否則90%。每日可見資訊只截至前一日；狀態改變或正常五日調整才重配。每日代理排除未知公司行動，20日每天及個股完整樣本均須至少120檔，資訊不足回90%並記錄。非官方加權指數，不是歷史全市場；82%保留現金上限緩衝。','4. 合併：同時執行上述兩規則。','25檔僅為這輪隔離風控差異的控制，不宣稱應固定持有25檔。整千股、均價成交、前收盤換算、應收股息不動用、買賣0.1425%及賣出0.3%稅沿用原模型；這輪沒有驗證Active Share或正式DataPack，不能視為完整官賽合規。','', '## 同窗口勝率與期望值',f"八方法共同{summary_v18['common_n']}窗口（v17是兩方法共同511窗口，不能跨表直接比較），其中訓練{summary_v18['train_n']}、後續期間{summary_v18['test_n']}、跨年淨空{summary_v18['purged_crossyear_n']}。",'|方法|費後勝率|費後24日期望值|平均獲利|平均虧損|P10|最差窗口內回撤|平均費稅|','|---|---:|---:|---:|---:|---:|---:|---:|']
for key,m in summary_v18['metrics'].items():text18.append('|'+labels18[key]+'|'+'|'.join(fmt18(m[f]) for f in ['win_rate','expectancy','mean_win','mean_loss','P10','worst_mdd','mean_cost'])+'|')
text18+=['','## 相對原策略：全部共同窗口','以下差值都是百分點；區間是平均差的不確定性，不是下次報酬區間。24／48位置區塊bootstrap各5000次，六比較Bonferroni；重疊樣本不當作獨立試驗。','|候選|平均報酬差pp|48區塊家族95%區間pp|費稅增加pp|原策略上漲窗口差pp|原策略虧損窗口差pp|','|---|---:|---|---:|---:|---:|']
for key,s in stats_v18.items():
    a=s['common_all'];ci=a['familyCI48'];text18.append(f"|{labels18[key]}|{a['mean_excess']*100:+.3f}|[{ci[0]*100:+.3f},{ci[1]*100:+.3f}]|{a['mean_cost_change']*100:+.3f}|{a['base_up_mean_excess']*100:+.3f}|{a['base_down_mean_excess']*100:+.3f}|")
text18+=['','## 時間分段與選擇','訓練只用窗口結束日<=2024-12-31；後續只用起始日>=2025-01-01，跨界窗口不拿來訓練或測試。門檻為訓練至少100窗口、覆蓋原策略80%以上、24和48區塊六比較校正的平均差下界>0、P10與最差回撤均不劣。合格者才依訓練費用較低優先；無合格者回原策略。門檻與資料已被研究過，因此只是按時間隔離的回溯檢查，不能宣稱全新未見樣本外。','次要風險取捨診斷：兩個下行指標均改善，且校正後平均差下界>-0.5pp；0.5pp只是敏感度門檻，不是使用者已接受的報酬代價，也不是改良通過。','|候選|訓練平均差pp|後續平均差pp|訓練改善通過|訓練風險取捨診斷|','|---|---:|---:|---|---|']
for key,s in stats_v18.items():text18.append(f"|{labels18[key]}|{s['train']['mean_excess']*100:+.3f}|{s['test']['mean_excess']*100:+.3f}|{s['train']['pass_improvement']}|{s['train']['risk_tradeoff_diagnostic']}|")
text18+=['','## 換股後是否反彈？','只觀察非例行調整日觸發失效的股票，在賣出當日起後續五日的固定個股調整毛報酬；依日期／股票去重並排除缺價、未知公司行動。不含費用、替補股、權重，因此不是策略收益比較；賣後上漲也不必然代表賣錯。','|候選|有效股票日期數|缺漏排除|之後5日上漲比例|之後5日平均毛報酬|','|---|---:|---:|---:|---:|']
for key,r in rebounds18.items():
    if r['unique_stock_dates']:text18.append(f"|{labels18[key]}|{r['valid_n']}|{r['excluded']}|{fmt18(r['positive5day_rate'])}|{fmt18(r['mean5day_gross'])}|")
text18+=['','## 資料與可行性覆蓋','|方法|原策略有效日期覆蓋|資料／成交不可判定窗口|限制違反窗口|','|---|---:|---:|---:|']
for key,s in stats_v18.items():text18.append(f"|{labels18[key]}|{fmt18(s['coverage_vs_base'])}|{s['invalid_count']}|{s['violation_windows']}|")
text18+=['同一窗口可能同時資料不足及違規，兩欄不可相加；僅在共同有效日期比較。無效窗口不是零報酬，也沒有默默換股票修復後算成功。新增非例行調整較容易遇到缺價，這本身就是實作限制。市場及股票訊號均用既有快照，沒有造價格、造公告時間或用事後大盤漲跌決定交易。','', '## 判斷與後續','本輪不支持「只要更快砍掉就改善」；既定失效換股對動能的期望值與最差回撤均沒有改善，法人有微小平均增量但區間跨零。降曝險在原策略虧損窗口有緩衝，原策略上漲窗口則有機會成本，並有調整費用。這不排除其他失效條件可能有效，但不能繼續改門檻直到顯著。','若要下一輪，優先明確決定要優化的是平均報酬或下行風險，再區分「持股換掉」與「未失效持股是否一起重配」的影響；此為後續建議，沒有自動擴充新規則或排程。','限制：目前150檔的選擇與存活偏誤、未解公司行動、短歷史、重疊窗口、門檻回溯、日資料均價成交非盤中停損、官方Active Share未驗證。沒有使用LLM判讀或提交交易。','', '## 驗證','原Notebook cells278–281保存定義、程式、輸出；原kernel PID2524保留。10個無風控控制與舊引擎一致；缺前收盤時在整數換算前標不可判定，修正590個原本無效／違規窗口且所有有效mask不變，抽樣原有效結果一致。費後期望值與配對差、條件損益恆等式、時間分段與六比較區間皆程式核對；圖保留每個點，未做瀏覽器目視QA。']
report18='\n'.join(text18);report_path18=recovery_root/'動能法人_失效剔除與降曝險研究_v18_2026-10-15.md';report_path18.write_text(report18,encoding='utf-8')
import plotly.graph_objects as go
rng18=np.random.default_rng(18015);fig18=go.Figure();figdiff18=go.Figure()
for pos,(key,m) in enumerate(summary_v18['metrics'].items()):
    z=ret_v18[key][common_v18]*100;fig18.add_trace(go.Box(x=np.full(len(z),pos),y=z,name=labels18[key],boxpoints=False,showlegend=False,line={'color':'#4e77a8'},fillcolor='rgba(78,119,168,.15)'))
    fig18.add_trace(go.Scatter(x=pos+rng18.uniform(-.17,.17,len(z)),y=z,mode='markers',marker={'size':4,'opacity':.4},text=[str(dates[int(window_starts_v15[i])]) for i in np.flatnonzero(common_v18)],hovertemplate=labels18[key]+'<br>%{text}<br>費後報酬 %{y:.2f}%<extra></extra>',showlegend=False))
    med=float(np.median(z));fig18.add_trace(go.Scatter(x=[pos-.2,pos+.2],y=[med,med],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
for pos,(key,s) in enumerate(stats_v18.items()):
    base=key.split('_')[0]+'_base';z=(ret_v18[key]-ret_v18[base])[common_v18]*100
    figdiff18.add_trace(go.Scatter(x=pos+rng18.uniform(-.17,.17,len(z)),y=z,mode='markers',name=labels18[key],text=[str(dates[int(window_starts_v15[i])]) for i in np.flatnonzero(common_v18)],marker={'size':4,'opacity':.4},hovertemplate=labels18[key]+'<br>%{text}<br>相對原策略 %{y:.3f} pp<extra></extra>',showlegend=False))
    a=s['common_all'];ci=a['familyCI48'];mean=a['mean_excess']*100;figdiff18.add_trace(go.Scatter(x=[pos],y=[mean],mode='markers',marker={'color':'#dc2477','size':10},error_y={'type':'data','array':[ci[1]*100-mean],'arrayminus':[mean-ci[0]*100]},showlegend=False,hovertemplate='平均差與48區塊家族區間 %{y:.3f} pp<extra></extra>'))
fig18.add_hline(y=0,line_color='#222');figdiff18.add_hline(y=0,line_color='#222')
fig18.update_layout(title='481個共同窗口：每一點、箱形圖與桃紅中位數',template='plotly_white',height=550,xaxis={'tickmode':'array','tickvals':list(range(8)),'ticktext':list(labels18.values())},yaxis_title='費後24日報酬 %')
# insertion-order labels match metric order; explicitly derive to avoid mapping order mismatch.
fig18.update_xaxes(ticktext=[labels18[k] for k in summary_v18['metrics']])
figdiff18.update_layout(title='相對原策略的全部配對差：桃紅為均值及家族校正區間',template='plotly_white',height=500,xaxis={'tickmode':'array','tickvals':list(range(6)),'ticktext':[labels18[k] for k in stats_v18]},yaxis_title='費後報酬差 百分點')
chart_path18=recovery_root/'動能法人_風控比較與全窗口_v18_2026-10-15.html'
chart_path18.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>動能法人風控比較 v18</title><style>body{font:16px/1.7 system-ui;max-width:1250px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#eef3fa;font-weight:bold}p{overflow-wrap:anywhere}</style><h1>動能／法人：失效換股與降曝險</h1><p>六項新增候選未通過預先改善門檻；降曝險呈現風險與報酬取捨。</p>'+fig18.to_html(full_html=False,include_plotlyjs=True)+figdiff18.to_html(full_html=False,include_plotlyjs=False)+render_report17(report18)+'</html>',encoding='utf-8')
for key,m in summary_v18['metrics'].items():
    x=ret_v18[key][common_v18];assert np.isclose(x.mean(),np.mean(x>0)*(m['mean_win'] or 0)+np.mean(x<0)*(m['mean_loss'] or 0),atol=1e-12)
for key,s in stats_v18.items():assert np.isclose(s['common_all']['mean_excess'],s['common_all']['candidate']['expectancy']-s['common_all']['base']['expectancy'])
assert not np.any(train_v18&test_v18);assert sum(len(t.y) for t in fig18.data if t.type=='scatter' and t.mode=='markers')==481*8
summary_v18['rebound_diagnostic']=rebounds18;summary_v18['verification']={'baseline_controls':10,'invalid_guard_valid_masks_unchanged':True,'new_candidate_windows':3900,'paired_points':2886,'return_points':3848,'no_genuine_unseen_test':True,'kernel_restarted':False}
(recovery_root/'動能法人_失效剔除與降曝險統計_v18_2026-10-15.json').write_text(json.dumps(summary_v18,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
state_v18=json.loads(state_path_v18.read_text(encoding='utf-8'));state_v18['risk_overlay_v18'].update({'status':'completed','running':False,'common_n':481,'train_n':165,'test_n':294,'selection':selected_v18,'confirmed_improvement':False,'outputs':[report_path18.name,chart_path18.name,'動能法人_失效剔除與降曝險統計_v18_2026-10-15.json'],'notebook_cells':[278,281]});state_path_v18.write_text(json.dumps(state_v18,ensure_ascii=False,indent=2),encoding='utf-8')
wf18=recovery_root/'A策略_無人值守研究流程.md';wf18.write_text(wf18.read_text(encoding='utf-8')+'\n\n## v18 失效換股／降曝險（完成）\n兩選股家族各三風控，共六新增定義，3900窗口；八方法共同481、訓練165、後續294、跨界22。沒有訓練候選通過校正改善門檻，保留原策略對照。報告：'+report_path18.name+'。未新增排程或提交交易。\n',encoding='utf-8')
print('V18_COMPLETE',{'report':report_path18.name,'html':chart_path18.name,'selection':selected_v18,'rebounds':rebounds18,'verification':summary_v18['verification']})


In [ ]:
# v19 finite momentum signal experiment, no threshold search; use retained memory.
print('V19_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'shape':shape,'dates':len(dates),'baseline_methods':len(factor_results_v16)})
definition_v19={'baseline':'A=.5rank(R20)+.5rank(R60);sameeligibility;25equal;90%stocks;rebalance5;horizon24;lot1000;fee.001425;tax.003','new_candidates':{'short10_20':'0.5rankR10+0.5rankR20;rebalance5','skip5':'0.5rankR20ending5sessionsago+0.5rankR60ending5sessionsago;65sessionhistoryrequired;rebalance5','risk_normalized':'0.5rank(R20/std20)+0.5rank(R60/std60);notinversevolweight;rebalance5','consistency':'0.75baseline+0.25rank(fractionpositiveadjusteddailyreturns20);rebalance5','volume_confirmation':'0.75baseline+0.25rank(meanvolume5/meanvolumeprior15);notinstitutionalflow;rebalance5','adaptive5':'rankR20weight0.25ifmarketproxy20negativeORstockstd20/std60>1.25else0.75;remainingweightR60;rebalance5','base_daily':'unchangedbaseline;rebalance1','adaptive_daily':'sameadaptive5score;rebalance1'},'candidates':8,'adaptive_caveat':'perstockweightregime,notestimatedoptimalglobalweight;marketproxycurrent150dailyknownonly;minimum120valid/day20sessionselseusevolruleonly','timing':'scoresobservedqclose,earliestordersq+1VWAP;nointradayreactivity','controls':'reusebaseline;originalandadaptivefrequencyfactorial;othercandidates5day;equalstockweightsunchanged;noLLM','comparison':'common9datesprimary;ownpairscoverage;trainend<=2024-12-31,teststart>=2025-01-01,crossyearpurged;24/48positionblocks5000;Bonferroni8','gate':'training>=100,coverage>=80%,bothadjustedlowerexcess>0,P10andworstMDD>=base;lowesttrainingfeeamongqualified;otherwisebasefallback','warning':'pastresultsseen,notgenuineunseenOOS;fixedhandchosenroundnumbersnotuniversallyoptimal;current150bias;noActiveSharecertification','sources':[{'title':'Momentum Crashes','authors':'Daniel and Moskowitz','url':'https://www.nber.org/papers/w20439','scope':'paper motivates regime and variance sensitivity,doesnotvalidateourhandchosenrulesorTaiwanlongonly24day'},{'title':'Momentum Trading, Return Chasing, and Predictable Crashes','url':'https://www.nber.org/papers/w20660','scope':'longhistoricalsamplecrashrisk,notspecific20/60parameters'}],'snapshot':snapshot_sha_v15}
definition_hash_v19=hashlib.sha256(json.dumps(definition_v19,sort_keys=True).encode()).hexdigest()
(recovery_root/'動能訊號_多週期與動態調整定義_v19_2026-10-15.json').write_text(json.dumps(definition_v19,ensure_ascii=False,indent=2),encoding='utf-8')
signal_names_v19={'short10_20':'M19_SHORT','skip5':'M19_SKIP','risk_normalized':'M19_RISK','consistency':'M19_CONSIST','volume_confirmation':'M19_VOLUME','adaptive5':'M19_ADAPT','base_daily':'A','adaptive_daily':'M19_ADAPT'}
for name in ['M19_SHORT','M19_SKIP','M19_RISK','M19_CONSIST','M19_VOLUME','M19_ADAPT']:scores_v15[name]=np.full(shape,np.nan)
for q in range(60,len(dates)):
    eligible=np.isfinite(scores_v15['A'][q]);rr=signal_returns_v15[q-59:q+1];r20=np.prod(1+rr[-20:],axis=0)-1;r60=np.prod(1+rr,axis=0)-1;r10=np.prod(1+rr[-10:],axis=0)-1
    p20=percentile(r20,eligible);p60=percentile(r60,eligible);s20=np.std(rr[-20:],axis=0);s60=np.std(rr,axis=0)
    scores_v15['M19_SHORT'][q]=.5*percentile(r10,eligible)+.5*p20
    if q>=64:
        skiprr=signal_returns_v15[q-64:q-4];ok=eligible&np.all(np.isfinite(skiprr)&(skiprr>-1)&~bad_action_v15[q-64:q-4],axis=0)
        scores_v15['M19_SKIP'][q]=.5*percentile(np.prod(1+skiprr[-20:],axis=0)-1,ok)+.5*percentile(np.prod(1+skiprr,axis=0)-1,ok)
    ok=eligible&(s20>0)&(s60>0)
    a=np.divide(r20,s20,out=np.full(shape[1],np.nan),where=s20>0);b=np.divide(r60,s60,out=np.full(shape[1],np.nan),where=s60>0)
    scores_v15['M19_RISK'][q]=.5*percentile(a,ok)+.5*percentile(b,ok)
    scores_v15['M19_CONSIST'][q]=.75*scores_v15['A'][q]+.25*percentile(np.mean(rr[-20:]>0,axis=0),eligible)
    vr=np.mean(volume[q-4:q+1],axis=0)/np.mean(volume[q-19:q-4],axis=0)
    scores_v15['M19_VOLUME'][q]=.75*scores_v15['A'][q]+.25*percentile(vr,eligible)
    marketdown=bool(np.all(proxy_count_v18[q-19:q+1]>=120) and np.prod(1+proxy_daily_v18[q-19:q+1])<1)
    ratio=np.divide(s20,s60,out=np.full(shape[1],np.nan),where=s60>0);shortweight=np.where(marketdown|(ratio>1.25),.25,.75)
    scores_v15['M19_ADAPT'][q]=shortweight*p20+(1-shortweight)*p60
source_v19=family_engine_source_v15.replace('def simulate_family_v15(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0):','def simulate_momentum_v19(start,strategy,allocation="equal",horizon=HORIZON,tolerance=0,rebalance_every=5):').replace('offset%REBALANCE==0','offset%rebalance_every==0')
exec(source_v19,globals())
for idx in [0,100,200,400,649]:
    got=simulate_momentum_v19(int(window_starts_v15[idx]),'A');old=factor_results_v16['MOM_E0'][idx]
    assert got['invalid']==old['invalid'] and got['violations']==old['violations'] and all(np.isclose(got[f],old[f],atol=1e-12,rtol=0) for f in ['return','cost','mdd','turnover'])
state_path19=recovery_root/'A策略_無人值守研究狀態.json';state19=json.loads(state_path19.read_text(encoding='utf-8'));state19['momentum_signal_v19']={'status':'defined_running','running':True,'definition_hash':definition_hash_v19,'candidates':8};state_path19.write_text(json.dumps(state19,ensure_ascii=False,indent=2),encoding='utf-8')
print('V19_FROZEN',definition_hash_v19,'5baselinecontrolsmatched',flush=True)


In [ ]:
# Eight new candidates once; baseline directly referenced.
results_v19={'base':factor_results_v16['MOM_E0']}
for name,signal in signal_names_v19.items():
    freq=1 if name.endswith('daily') else 5
    results_v19[name]=[simulate_momentum_v19(int(start),signal,rebalance_every=freq) for start in window_starts_v15]
    print('V19_RUN',name,'valid',sum(not r['invalid'] and r['violations']==0 for r in results_v19[name]),flush=True)
valid_v19={k:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results_v19.items()};ret_v19={k:np.array([r['return'] for r in rows]) for k,rows in results_v19.items()};common_v19=np.logical_and.reduce(list(valid_v19.values()))
def metrics19(key,mask):
    rows=results_v19[key];ids=np.flatnonzero(mask);z=ret_v19[key][mask]
    if len(ids)==0:return {'n':0}
    return {'n':len(ids),'win_rate':float(np.mean(z>0)),'expectancy':float(z.mean()),'mean_win':float(z[z>0].mean()) if np.any(z>0) else None,'mean_loss':float(z[z<0].mean()) if np.any(z<0) else None,'P10':float(np.quantile(z,.1)),'worst_mdd':float(min(rows[i]['mdd'] for i in ids)),'median_mdd':float(np.median([rows[i]['mdd'] for i in ids])),'mean_cost':float(np.mean([rows[i]['cost'] for i in ids])),'mean_turnover':float(np.mean([rows[i]['turnover'] for i in ids]))}
def contrast19(key,mask,base='base'):
    pair=mask&valid_v19[key]&valid_v19[base];d=ret_v19[key]-ret_v19[base];ids=np.flatnonzero(pair)
    out={'n':len(ids),'mean_excess':float(d[pair].mean()),'beat_base_rate':float(np.mean(d[pair]>0)),'candidate':metrics19(key,pair),'base':metrics19(base,pair),'mean_cost_change':float(np.mean([results_v19[key][i]['cost']-results_v19[base][i]['cost'] for i in ids]))}
    for block,inds in block_indices_v15.items():
        x=np.where(pair,d,np.nan)[inds];n=np.isfinite(x).sum(axis=1);z=np.nansum(x,axis=1)[n>0]/n[n>0]
        out[f'CI{block}']=np.quantile(z,[.025,.975]).tolist();out[f'familyCI{block}']=np.quantile(z,[.05/16,1-.05/16]).tolist()
    return out
stats_v19={}
for key in signal_names_v19:
    pair=valid_v19[key]&valid_v19['base'];trmask=common_v19&train_v18;temask=common_v19&test_v18
    row={'all':contrast19(key,common_v19),'own_pair':contrast19(key,pair),'train':contrast19(key,trmask),'test':contrast19(key,temask),'coverage_vs_base':float(pair.sum()/valid_v19['base'].sum()),'invalid_windows':sum(r['invalid'] for r in results_v19[key]),'violation_windows':sum(r['violations']>0 for r in results_v19[key]),'years':{year:contrast19(key,common_v19&mask) for year,mask in year_masks_v15.items() if np.any(common_v19&mask)}}
    tr=row['train'];coverage=float((pair&train_v18).sum()/max(1,(valid_v19['base']&train_v18).sum()));tr['coverage_vs_base']=coverage
    tr['pass']=bool(tr['n']>=100 and coverage>=.8 and tr['familyCI24'][0]>0 and tr['familyCI48'][0]>0 and tr['candidate']['P10']>=tr['base']['P10'] and tr['candidate']['worst_mdd']>=tr['base']['worst_mdd'])
    stats_v19[key]=row
qualified19=[k for k,s in stats_v19.items() if s['train']['pass']];selected19=min(qualified19,key=lambda k:stats_v19[k]['train']['candidate']['mean_cost']) if qualified19 else 'base'
frequency19={'unchanged_score_daily_minus5':contrast19('base_daily',common_v19),'adaptive_score_daily_minus5':contrast19('adaptive_daily',common_v19,'adaptive5'),'adaptive_minusoriginal_5day':contrast19('adaptive5',common_v19),'adaptive_minusoriginal_daily':contrast19('adaptive_daily',common_v19,'base_daily')}
summary_v19={'definition':definition_v19,'hash':definition_hash_v19,'common_n':int(common_v19.sum()),'train_n':int((common_v19&train_v18).sum()),'test_n':int((common_v19&test_v18).sum()),'purged_n':int((common_v19&purged_v18).sum()),'common_indices':np.flatnonzero(common_v19).tolist(),'metrics':{k:metrics19(k,common_v19) for k in results_v19},'comparisons':stats_v19,'frequency_factorial_diagnostic':frequency19,'training_selected':selected19,'selected_test':metrics19(selected19,common_v19&test_v18),'frequency_warning':'fourcontrastsexploratorynotoverall8testgates;nounseenOOS'}
(recovery_root/'動能訊號_多週期與動態調整逐窗口_v19_2026-10-15.json').write_text(json.dumps({'definition':definition_v19,'results':results_v19},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
(recovery_root/'動能訊號_多週期與動態調整統計_v19_2026-10-15.json').write_text(json.dumps(summary_v19,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V19_STATS',json.dumps({'n':summary_v19['common_n'],'train':summary_v19['train_n'],'test':summary_v19['test_n'],'selected':selected19,'metrics':summary_v19['metrics'],'differences':{k:{'delta':s['all']['mean_excess'],'CI48':s['all']['familyCI48'],'train_delta':s['train']['mean_excess'],'test_delta':s['test']['mean_excess'],'pass':s['train']['pass']} for k,s in stats_v19.items()},'frequency':{k:{'delta':s['mean_excess'],'costchange':s['mean_cost_change']} for k,s in frequency19.items()}},default=serial_family_v15),flush=True)


In [ ]:
# Final v19 evidence, all dots, no forced selection or hidden threshold expansion.
labels19={'base':'原20／60日','short10_20':'短週期10／20日','skip5':'略過最近5日','risk_normalized':'波動調整動能','consistency':'上漲連續性','volume_confirmation':'成交量確認','adaptive5':'動態比重／5日交易','base_daily':'原公式／每日交易','adaptive_daily':'動態比重／每日交易'}
fmt19=lambda v:'—' if v is None else f'{v*100:.2f}%'
lines19=['# 動能訊號：多週期與動態調整 v19','', '## 結論','可以加入其他參數，每日更新訊號，但不能把每日更新直接等同每日交易。本輪八候選均未通過預先設定的訓練改善門檻，保留原20／60日作對照，不宣稱原式最佳。','波動調整有減少平均虧損及尾端損失的線索；上漲連續性平均報酬較高但最差回撤惡化；動態比重五日交易有小幅平均增量但風險未一致改善。原公式每日交易的平均費後報酬明顯較低，費用上升。因此先保留風險／報酬取捨，不把增加參數稱成改良。','', '## 既有方法與本輪差別','原動能=.5×20日調整報酬百分位+.5×60日調整報酬百分位，每五日選25檔。既有B混入法人、C混入低波動、逆波動配置與v18市場降曝險都已測過。本輪波動調整是「報酬除波動後再排名」，不同於單純混入低波動分數或依逆波動決定買入額度。股數與權重仍等權、90%投入；25檔只是隔離訊號變化的實驗控制。','', '## 八個固定候選','|候選|定義|交易頻率|研究目的|','|---|---|---|---|','|短週期|10日與20日報酬百分位各半|五日|較快反應是否值得額外換手|','|略過最近5日|20日及60日報酬都截止於五日前，各半；需要65日歷史|五日|降低最近短期價格跳動的影響|','|波動調整|20日報酬／20日標準差、60日報酬／60日標準差，各自排名後各半|五日|區分漲得多與承受較高波動才漲得多|','|上漲連續性|原分數75%＋近20日上漲天數比例排名25%|五日|區分連續上行與少數日大漲|','|成交量確認|原分數75%＋最近5日平均量／此前15日平均量排名25%|五日|檢查近期量能增加是否提供額外資訊|','|動態比重|市場代理20日累積負報酬，或個股20日波動／60日波動>1.25時，20／60日權重25／75%；否則75／25%|五日|狀態不同時改變短長週期比重；手工規則，非已估計的最佳權重|','|原公式每日交易|原20／60日訊號不變|每日|隔離頻率影響|','|動態比重每日交易|同上動態分數|每日|與五日版本比較頻率、與原每日版本比較訊號|','分數每天用當日收盤前已形成的日資料更新，最早下一交易日均價交易，不是盤中即時。動態比重是每檔股票的短長週期分數權重，不是對持股金額作非等權配置。','市場代理來自目前150檔，排除未知公司行動；近20日每天至少120檔有效資料才判定市場狀態，否則僅用個股波動條件。非官方指數或歷史全市場。所有方法繼承同一原始資料品質及可選門檻，法人資料品質門檻沒有偷變成法人排名。','', '## 同窗口費後結果',f"九方法共同{summary_v19['common_n']}窗口；訓練{summary_v19['train_n']}、後續{summary_v19['test_n']}、跨年淨空{summary_v19['purged_n']}。這與v17的511、v18的481窗口不同，請只在本表內配對比較。",'|方法|勝率|24日期望值|平均獲利|平均虧損|P10|最差窗口內回撤|平均費稅|','|---|---:|---:|---:|---:|---:|---:|---:|']
for key,m in summary_v19['metrics'].items():lines19.append('|'+labels19[key]+'|'+'|'.join(fmt19(m[f]) for f in ['win_rate','expectancy','mean_win','mean_loss','P10','worst_mdd','mean_cost'])+'|')
lines19+=['','## 相對原公式的配對差','以下為百分點，24／48日位置區塊bootstrap各5000次、八候選Bonferroni校正。區間代表平均增量的不確定性，不是下一場報酬範圍。','|候選|全期平均差pp|48區塊家族95%區間pp|訓練平均差pp|後續平均差pp|訓練通過|','|---|---:|---|---:|---:|---|']
for key,s in stats_v19.items():
    a=s['all'];ci=a['familyCI48'];lines19.append(f"|{labels19[key]}|{a['mean_excess']*100:+.3f}|[{ci[0]*100:+.3f},{ci[1]*100:+.3f}]|{s['train']['mean_excess']*100:+.3f}|{s['test']['mean_excess']*100:+.3f}|{s['train']['pass']}|")
lines19+=['','## 每日觀察與每日交易','|同公式比較：每日交易減五日交易|期望值差pp|費稅增加pp|','|---|---:|---:|']
for name,label in [('unchanged_score_daily_minus5','原20／60日'),('adaptive_score_daily_minus5','動態比重')]:
    s=frequency19[name];lines19.append(f"|{label}|{s['mean_excess']*100:+.3f}|{s['mean_cost_change']*100:+.3f}|")
lines19+=['每日版本每天重新選股與將保留持股重配至等權，故頻率差同時包含換股、重新配置及成本，不能把全部落差都歸因費用。固定原交易只加回費稅可作會計分解，並非實際零費率重跑。此四格分解為探索性診斷，不額外宣稱已通過新的多重比較家族。','', '## 選擇方式與證據強度','先以2024-12-31前已結束窗口訓練，後續窗口起始須>=2025-01-01，跨界排除。門檻：共同訓練至少100窗口、候選相對原策略訓練有效日期覆蓋>=80%、24和48區塊八比較校正的平均差下界均>0、P10及最差回撤不劣；合格者才按訓練費用低優先。本輪沒有合格者，不強選。後續期間全部候選仍列出供透明比較，但未用後續最高值回頭挑門檻。','這些資料在前輪已被看過，屬時間分段回溯研究，不能宣稱真正未見的樣本外。5000次bootstrap的校正尾端解析度有限，且滑動窗口不獨立。沒有用最高全期平均值重新找20／60比重或1.25門檻。','', '## 覆蓋與限制','|候選|對原策略有效日期覆蓋|資料／成交不可判定|限制違反窗口|','|---|---:|---:|---:|']
for key,s in stats_v19.items():lines19.append(f"|{labels19[key]}|{fmt19(s['coverage_vs_base'])}|{s['invalid_windows']}|{s['violation_windows']}|")
lines19+=['無效窗口不是零報酬；兩種排除可能重疊，不能相加。每個候選另保存各自配對日期統計與年度摘要，以檢查共同九方法日期的選擇影響。略過五日候選額外要求65日歷史，起始不足不能偷用未來資料。','股票池是目前150檔，含存活及選擇偏誤；仍有未知公司行動；只觀察這段歷史，不代表每年10月同期都成立。延用整千股、90%投入、相同均價及費稅模型，期末按持股估值、未強制清倉；Active Share與正式DataPack規則未完整驗證，不代表可以直接官賽提交。','', '## 文獻動機與適用範圍','Daniel與Moskowitz的Momentum Crashes研究提示動能風險會隨市場狀態而變，包含市場下跌後高波動及反彈狀態；其多空動能與我們只買股票的短賽期不能直接等同，本文手工參數也非該論文已驗證規則。來源：https://www.nber.org/papers/w20439 。','Chabot等人的歷史研究也記錄動能的崩跌風險及條件性，不能據此推斷本輪短期參數必然有效。來源：https://www.nber.org/papers/w20660 。','若擴大為系統性文獻比較，可用Elicit Web協助篩選與擷取，但重要結論仍回查原始論文、樣本與限制；本輪沒有聲稱已取得Elicit付費內容或完成系統性回顧。','', '## 後續值得保留的問題','先保留波動調整作風險方向、上漲連續性作報酬方向、動態五日比重作狀態方向的待驗證線索。三者沒有通過改善門檻，不直接混合為最終策略；下一輪需要先固定主要目標及風險代價，再設計獨立驗證。每日觀察可以保留，每日全面換股目前缺乏支持。','', '## 驗證與保存','原Notebook cells282–284。沿用PID2524，未重啟；只重用已保存價量、法人品質與公司行動快照。原公式五個抽樣控制與v16一致；八候選5200窗口。條件損益／期望值、配對差恆等式、時間分段互斥、圖全點數、分數範圍及決策日前資料切片核對。圖包含459×9報酬點與459×8配對差點，中位數桃紅、零線；只做程式檢查，未做瀏覽器目視QA。']
report19='\n'.join(lines19);report_path19=recovery_root/'動能訊號_多週期與動態調整研究_v19_2026-10-15.md';report_path19.write_text(report19,encoding='utf-8')
import plotly.graph_objects as go
rng19=np.random.default_rng(19015);figret19=go.Figure();figdelta19=go.Figure()
ids19=np.flatnonzero(common_v19);dates19=[str(dates[int(window_starts_v15[i])]) for i in ids19]
for pos,key in enumerate(results_v19):
    z=ret_v19[key][common_v19]*100
    figret19.add_trace(go.Box(x=np.full(len(z),pos),y=z,name=labels19[key],boxpoints=False,showlegend=False,line={'color':'#4e77a8'},fillcolor='rgba(78,119,168,.15)',hoverinfo='skip'))
    figret19.add_trace(go.Scatter(x=pos+rng19.uniform(-.17,.17,len(z)),y=z,mode='markers',marker={'size':4,'opacity':.4},text=dates19,hovertemplate=labels19[key]+'<br>%{text}<br>費後24日報酬 %{y:.2f}%<extra></extra>',showlegend=False))
    med=float(np.median(z));figret19.add_trace(go.Scatter(x=[pos-.2,pos+.2],y=[med,med],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
for pos,(key,s) in enumerate(stats_v19.items()):
    z=(ret_v19[key]-ret_v19['base'])[common_v19]*100
    figdelta19.add_trace(go.Scatter(x=pos+rng19.uniform(-.17,.17,len(z)),y=z,mode='markers',marker={'size':4,'opacity':.4},text=dates19,hovertemplate=labels19[key]+'<br>%{text}<br>配對差 %{y:.3f} pp<extra></extra>',showlegend=False))
    a=s['all'];mean=a['mean_excess']*100;ci=a['familyCI48'];figdelta19.add_trace(go.Scatter(x=[pos],y=[mean],mode='markers',marker={'color':'#dc2477','size':10},error_y={'type':'data','array':[ci[1]*100-mean],'arrayminus':[mean-ci[0]*100]},showlegend=False,hovertemplate='平均差及48區塊八比較校正區間 %{y:.3f} pp<extra></extra>'))
figret19.add_hline(y=0,line_color='#222');figdelta19.add_hline(y=0,line_color='#222')
figret19.update_layout(template='plotly_white',height=570,title='全部459共同窗口：費後報酬、箱形圖與桃紅中位數',xaxis={'tickmode':'array','tickvals':list(range(9)),'ticktext':[labels19[k] for k in results_v19],'tickangle':-20},yaxis_title='24日報酬 %')
figdelta19.update_layout(template='plotly_white',height=510,title='全部配對差；桃紅為平均差及家族校正區間',xaxis={'tickmode':'array','tickvals':list(range(8)),'ticktext':[labels19[k] for k in stats_v19],'tickangle':-20},yaxis_title='相對原策略 百分點')
chart_path19=recovery_root/'動能訊號_多週期動態比較與全點_v19_2026-10-15.html'
chart_path19.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>動能參數比較 v19</title><style>body{font:16px/1.7 system-ui;max-width:1350px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#eef3fa;font-weight:bold}p{overflow-wrap:anywhere}</style><h1>動能：多週期與動態調整</h1><p>八個固定候選；九方法共同459窗口。全部候選未通過訓練改善門檻，每日更新不等於每日交易。</p>'+figret19.to_html(full_html=False,include_plotlyjs=True)+figdelta19.to_html(full_html=False,include_plotlyjs=False)+render_report17(report19)+'<p>原研究：<a href="https://www.nber.org/papers/w20439">Momentum Crashes</a>；<a href="https://www.nber.org/papers/w20660">Momentum Trading, Return Chasing, and Predictable Crashes</a></p></html>',encoding='utf-8')
for key,m in summary_v19['metrics'].items():
    z=ret_v19[key][common_v19];assert np.isclose(z.mean(),np.mean(z>0)*(m['mean_win'] or 0)+np.mean(z<0)*(m['mean_loss'] or 0))
for key,s in stats_v19.items():assert np.isclose(s['all']['mean_excess'],s['all']['candidate']['expectancy']-s['all']['base']['expectancy'])
for name in set(signal_names_v19.values()):
    score=scores_v15[name];finite=score[np.isfinite(score)];assert finite.min()>=-1e-12 and finite.max()<=1+1e-12
assert not np.any(train_v18&test_v18)
assert sum(len(t.y) for t in figret19.data if t.type=='scatter' and t.mode=='markers')==459*9
assert sum(len(t.y) for t in figdelta19.data if t.type=='scatter' and len(t.y)>1)==459*8
summary_v19['verification']={'baseline_checks':5,'new_windows':5200,'return_plot_points':4131,'paired_plot_points':3672,'accounting_expectancy_checked':True,'data_time_q_to_order_qplus1':True,'kernel_restarted':False,'no_unseenOOS':True}
(recovery_root/'動能訊號_多週期與動態調整統計_v19_2026-10-15.json').write_text(json.dumps(summary_v19,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
state19=json.loads(state_path19.read_text(encoding='utf-8'));state19['momentum_signal_v19'].update({'status':'completed','running':False,'common_n':459,'train_n':127,'test_n':310,'training_selected':selected19,'confirmed_improvement':False,'outputs':[report_path19.name,chart_path19.name,'動能訊號_多週期與動態調整統計_v19_2026-10-15.json'],'notebook_cells':[282,284]});state_path19.write_text(json.dumps(state19,ensure_ascii=False,indent=2),encoding='utf-8')
wf19=recovery_root/'A策略_無人值守研究流程.md';wf19.write_text(wf19.read_text(encoding='utf-8')+'\n\n## v19 動能多週期與動態調整（完成）\n八候選5200窗口，九方法共同459（訓練127／後續310／跨界22）。未通過訓練改善門檻，保留原策略；每日全面換股費後結果較差，波動調整／連續性僅保留取捨線索。報告：'+report_path19.name+'。未自動擴增參數或啟動排程。\n',encoding='utf-8')
print('V19_COMPLETE',{'report':report_path19.name,'html':chart_path19.name,'selected':selected19,'verification':summary_v19['verification']})


In [ ]:
# v20 isolate 20 vs60 lookback only; fixed 5day trades, identical original eligibility.
print('V20_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'shape':shape,'windows':len(window_starts_v15)})
definition20={'signals':['100%rankR20','100%rankR60','50%rankR20+50%rankR60 reference'],'interpretation':'100%referstosignalweightnot100%stockexposure','fixed':'rebalance5,25equal,90%stocks,24sessions,1000sharelots,existingexecutionandcosts','eligibility':'original60dayprice/actionand20dayflow/volumequalitygateheldfixedforall;20dayscorestillrequires60daysqualityhistory','primary':'all3exactcommondates;no other v19methods restrictdenominator','statistics':'netwinrate,EV,conditionalpayoffs,fees,P10,MDD,24/48block5000,Bonferroni3paircontrasts','split':'end<=2024-12-31 training,start>=2025-01-01 later,crossyearpurged;pastdataalreadyseen','new_candidates':2,'no_frequency_or_weighting_change':True,'snapshot':snapshot_sha_v15}
hash20=hashlib.sha256(json.dumps(definition20,sort_keys=True).encode()).hexdigest()
(recovery_root/'動能20日60日_單獨訊號比較定義_v20_2026-10-15.json').write_text(json.dumps(definition20,ensure_ascii=False,indent=2),encoding='utf-8')
for name in ['M20_ONLY20','M20_ONLY60']:scores_v15[name]=np.full(shape,np.nan)
for q in range(60,len(dates)):
    ok=np.isfinite(scores_v15['A'][q]);rr=signal_returns_v15[q-59:q+1]
    scores_v15['M20_ONLY20'][q]=percentile(np.prod(1+rr[-20:],axis=0)-1,ok)
    scores_v15['M20_ONLY60'][q]=percentile(np.prod(1+rr,axis=0)-1,ok)
    assert np.allclose((scores_v15['M20_ONLY20'][q,ok]+scores_v15['M20_ONLY60'][q,ok])/2,scores_v15['A'][q,ok])
results20={'mix':factor_results_v16['MOM_E0']}
for key,signal in [('only20','M20_ONLY20'),('only60','M20_ONLY60')]:
    results20[key]=[simulate_momentum_v19(int(s),signal,rebalance_every=5) for s in window_starts_v15]
    print('V20_DONE',key,sum(not r['invalid'] and r['violations']==0 for r in results20[key]),flush=True)
valid20={k:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results20.items()};ret20={k:np.array([r['return'] for r in rows]) for k,rows in results20.items()};common20=np.logical_and.reduce(list(valid20.values()))
def metric20(key,mask):
    ids=np.flatnonzero(mask);z=ret20[key][mask];rows=results20[key]
    return {'n':len(ids),'win_rate':float(np.mean(z>0)),'expectancy':float(z.mean()),'mean_win':float(z[z>0].mean()) if np.any(z>0) else None,'mean_loss':float(z[z<0].mean()) if np.any(z<0) else None,'P10':float(np.quantile(z,.1)),'worst_return':float(z.min()),'worst_mdd':float(min(rows[i]['mdd'] for i in ids)),'mean_cost':float(np.mean([rows[i]['cost'] for i in ids])),'turnover':float(np.mean([rows[i]['turnover'] for i in ids]))}
def pair20(a,b,mask):
    pair=mask&valid20[a]&valid20[b];d=ret20[a]-ret20[b];r={'n':int(pair.sum()),'mean_excess':float(d[pair].mean()),'beat_base_rate':float(np.mean(d[pair]>0)),'candidate':metric20(a,pair),'base':metric20(b,pair)}
    for block,idxs in block_indices_v15.items():
        x=np.where(pair,d,np.nan)[idxs];cnt=np.isfinite(x).sum(axis=1);means=np.nansum(x,axis=1)[cnt>0]/cnt[cnt>0]
        r[f'CI{block}']=np.quantile(means,[.025,.975]).tolist();r[f'familyCI{block}']=np.quantile(means,[.05/6,1-.05/6]).tolist()
    return r
pairs20={}
for a,b in [('only20','mix'),('only60','mix'),('only20','only60')]:
    pairs20[a+'_minus_'+b]={'all':pair20(a,b,common20),'train':pair20(a,b,common20&train_v18),'later':pair20(a,b,common20&test_v18),'own_pair':pair20(a,b,valid20[a]&valid20[b]),'years':{y:pair20(a,b,common20&m) for y,m in year_masks_v15.items() if np.any(common20&m)}}
summary20={'definition':definition20,'hash':hash20,'n':int(common20.sum()),'train_n':int((common20&train_v18).sum()),'later_n':int((common20&test_v18).sum()),'purged_n':int((common20&purged_v18).sum()),'common_indices':np.flatnonzero(common20).tolist(),'metrics':{k:metric20(k,common20) for k in results20},'pairs':pairs20,'valid_counts':{k:int(m.sum()) for k,m in valid20.items()},'exclusions':{k:{'invalid':sum(r['invalid'] for r in rows),'violations':sum(r['violations']>0 for r in rows)} for k,rows in results20.items()},'not_independent_or_unseen':True}
(recovery_root/'動能20日60日_單獨訊號逐窗口_v20_2026-10-15.json').write_text(json.dumps({'definition':definition20,'results':results20},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
(recovery_root/'動能20日60日_單獨訊號統計_v20_2026-10-15.json').write_text(json.dumps(summary20,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V20_STATS',json.dumps(summary20,ensure_ascii=False,default=serial_family_v15)[:16000],flush=True)


In [ ]:
# v20 persist fair paired comparison and every point.
labels20={'mix':'20日50%＋60日50%','only20':'全20日訊號','only60':'全60日訊號'}
fmt20=lambda x:'—' if x is None else f'{x*100:.2f}%'
lines20=['# 全20日與全60日動能比較 v20','', '## 比較方式','三個方法只改選股訊號：全20日漲幅排名、全60日漲幅排名、原20／60日排名各半。全部每5個交易日調整，固定25檔等權、90%投入、24日窗口、整千股、原執行均價與買賣手續費0.1425%、賣出另0.3%稅；沒有每日交易、沒有新增風控或改持股金額比重。「全20日／全60日」指訊號100%權重，不是股票投入100%。','同一可選池及原60日價量／公司行動品質門檻固定，因此20日方法仍要求60日品質歷史；目的是隔離選股週期，而非擴大短歷史股票池。原各半結果直接引用v16，不重跑。','', '## 費後勝率與期望值',f"三方法共同{summary20['n']}個24交易日窗口；訓練{summary20['train_n']}、後續{summary20['later_n']}、跨年淨空{summary20['purged_n']}。此分母不同於v19的459，不要跨表比較。",'|方法|勝率|24日期望值|平均獲利|平均虧損|P10|最差窗口內回撤|平均費稅|','|---|---:|---:|---:|---:|---:|---:|---:|']
for key,m in summary20['metrics'].items():lines20.append('|'+labels20[key]+'|'+'|'.join(fmt20(m[f]) for f in ['win_rate','expectancy','mean_win','mean_loss','P10','worst_mdd','mean_cost'])+'|')
lines20+=['','## 三組配對平均差','所有差值為百分點。使用24／48日位置區塊bootstrap各5000次，三個配對Bonferroni校正；滑動窗口不是552次獨立勝負，區間是平均差不確定性，不是未來報酬範圍。','|配對|全期平均差pp|24區塊家族95%區間pp|48區塊家族95%區間pp|訓練平均差pp|後續平均差pp|','|---|---:|---|---|---:|---:|']
for name,row in pairs20.items():
    a,b=name.split('_minus_');x=row['all'];c=x['familyCI24'];d=x['familyCI48'];lines20.append(f"|{labels20[a]}減{labels20[b]}|{x['mean_excess']*100:+.3f}|[{c[0]*100:+.3f},{c[1]*100:+.3f}]|[{d[0]*100:+.3f},{d[1]*100:+.3f}]|{row['train']['mean_excess']*100:+.3f}|{row['later']['mean_excess']*100:+.3f}|")
lines20+=['','## 如何解讀','全20日的全期平均報酬最高，但勝率略低、交易費用較高，P10也略差於各半；不能只按全期最高均值改成全20日。全60日勝率最高、換手與費用最低，但平均虧損更深、最差回撤更大，顯示高勝率不代表較安全。全60日訊號在後續期間相對各半表現轉弱；相反，前段曾較強，不能說較長週期必然穩定。','三個全期配對的校正區間是否跨零，及兩個區塊是否一致，皆列在表內；本輪沒有證明哪個週期可穩定取代原各半。後續期間與逐年資料都曾被研究過，是時間分段的回溯描述，不是真正未見樣本外。','', '## 各年度共同窗口','年度僅列起訖同年的窗口；跨年點仍保留於全期。','|年度|共同窗口數|各半：勝率／期望值|全20日：勝率／期望值|全60日：勝率／期望值|','|---|---:|---|---|---|']
for year,ym in year_masks_v15.items():
    mask=common20&ym
    if not mask.any():continue
    vals=[metric20(k,mask) for k in ['mix','only20','only60']]
    lines20.append('|'+year+'|'+str(int(mask.sum()))+'|'+'|'.join(fmt20(m['win_rate'])+'／'+fmt20(m['expectancy']) for m in vals)+'|')
lines20+=['','## 覆蓋與限制','|方法|原650窗口中有效數|資料或成交不可判定|限制違反|','|---|---:|---:|---:|']
for k in results20:lines20.append(f"|{labels20[k]}|{summary20['valid_counts'][k]}|{summary20['exclusions'][k]['invalid']}|{summary20['exclusions'][k]['violations']}|")
lines20+=['兩排除可能重疊，不可相加。另存每組自己的有效配對，避免只看三方法共同日期而忽略覆蓋。期末持股按資產估值，沒有假設全清倉；既有公司行動修正與應收股息沿用，未知事件仍保留不可判定。','目前150檔股票池有存活與選擇偏誤，且僅這段歷史可用；不代表每年比賽同期都有效。沒有驗證正式Active Share、DataPack或提交交易。原kernel PID2524保留，未重啟，沒有重爬資料。','', '## 保存及驗證','原Notebook cells285–286保存定義、程式與輸出。兩新定義共1300窗口，原各半直接引用；每個有效評分日驗證全20與全60分數各半正好等於原A。條件損益與期望值、配對平均差、時間分段、552×3報酬點及552×3差值點核對。桃紅中位數／均值及0線保留，只有程式版面檢查，未做瀏覽器目視QA。']
report20='\n'.join(lines20);rp20=recovery_root/'動能20日60日_五日調整比較_v20_2026-10-15.md';rp20.write_text(report20,encoding='utf-8')
import plotly.graph_objects as go
rng20=np.random.default_rng(20015);fr20=go.Figure();fd20=go.Figure();ids20=np.flatnonzero(common20);days20=[str(dates[int(window_starts_v15[i])]) for i in ids20]
for pos,key in enumerate(['only20','mix','only60']):
    z=ret20[key][common20]*100
    fr20.add_trace(go.Box(x=np.full(len(z),pos),y=z,name=labels20[key],boxpoints=False,line={'color':'#4e77a8'},fillcolor='rgba(78,119,168,.15)',hoverinfo='skip',showlegend=False))
    fr20.add_trace(go.Scatter(x=pos+rng20.uniform(-.18,.18,len(z)),y=z,mode='markers',marker={'size':5,'opacity':.45},text=days20,hovertemplate=labels20[key]+'<br>%{text}<br>費後24日報酬 %{y:.2f}%<extra></extra>',showlegend=False))
    median=float(np.median(z));fr20.add_trace(go.Scatter(x=[pos-.2,pos+.2],y=[median,median],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
for pos,(name,row) in enumerate(pairs20.items()):
    a,b=name.split('_minus_');z=(ret20[a]-ret20[b])[common20]*100
    fd20.add_trace(go.Scatter(x=pos+rng20.uniform(-.18,.18,len(z)),y=z,mode='markers',marker={'size':5,'opacity':.45},text=days20,hovertemplate=labels20[a]+'減'+labels20[b]+'<br>%{text}<br>配對差 %{y:.3f} pp<extra></extra>',showlegend=False))
    x=row['all'];ci=x['familyCI48'];mean=x['mean_excess']*100;fd20.add_trace(go.Scatter(x=[pos],y=[mean],mode='markers',marker={'color':'#dc2477','size':11},error_y={'type':'data','array':[ci[1]*100-mean],'arrayminus':[mean-ci[0]*100]},hovertemplate='平均差及48區塊家族區間 %{y:.3f} pp<extra></extra>',showlegend=False))
fr20.add_hline(y=0,line_color='#222');fd20.add_hline(y=0,line_color='#222')
fr20.update_layout(title='固定每5日調整：552共同窗口全部點與桃紅中位數',height=560,template='plotly_white',xaxis={'tickmode':'array','tickvals':[0,1,2],'ticktext':[labels20[k] for k in ['only20','mix','only60']]},yaxis_title='費後24日報酬 %')
fd20.update_layout(title='三組配對：全部差值點與均值／48區塊家族校正區間',height=500,template='plotly_white',xaxis={'tickmode':'array','tickvals':[0,1,2],'ticktext':[labels20[a]+'減'+labels20[b] for a,b in [n.split('_minus_') for n in pairs20]]},yaxis_title='費後差值 百分點')
hp20=recovery_root/'動能20日60日_五日調整全點比較_v20_2026-10-15.html';hp20.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>全20日／全60日動能 v20</title><style>body{font:16px/1.7 system-ui;max-width:1250px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#eef3fa;font-weight:bold}p{overflow-wrap:anywhere}</style><h1>全20日 vs 各半 vs 全60日</h1><p>固定每5日調整；相同552個24交易日窗口；等權、90%投入與原費稅。</p>'+fr20.to_html(full_html=False,include_plotlyjs=True)+fd20.to_html(full_html=False,include_plotlyjs=False)+render_report17(report20)+'</html>',encoding='utf-8')
for k,m in summary20['metrics'].items():
    z=ret20[k][common20];assert np.isclose(z.mean(),np.mean(z>0)*(m['mean_win'] or 0)+np.mean(z<0)*(m['mean_loss'] or 0))
for name,row in pairs20.items():assert np.isclose(row['all']['mean_excess'],row['all']['candidate']['expectancy']-row['all']['base']['expectancy'])
assert sum(len(t.y) for t in fr20.data if t.type=='scatter' and t.mode=='markers')==552*3
assert sum(len(t.y) for t in fd20.data if t.type=='scatter' and len(t.y)>1)==552*3
summary20['verification']={'new_windows':1300,'baseline_reused':True,'score_mix_identity_checked':True,'return_points':1656,'paired_difference_points':1656,'rebalance_every':5,'kernel_restarted':False}
(recovery_root/'動能20日60日_單獨訊號統計_v20_2026-10-15.json').write_text(json.dumps(summary20,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
sp20=recovery_root/'A策略_無人值守研究狀態.json';st20=json.loads(sp20.read_text(encoding='utf-8'));st20['pure_horizons_v20']={'status':'completed','running':False,'definition_hash':hash20,'n_common':552,'rebalance_every':5,'outputs':[rp20.name,hp20.name,'動能20日60日_單獨訊號統計_v20_2026-10-15.json'],'notebook_cells':[285,286]};sp20.write_text(json.dumps(st20,ensure_ascii=False,indent=2),encoding='utf-8')
wf20=recovery_root/'A策略_無人值守研究流程.md';wf20.write_text(wf20.read_text(encoding='utf-8')+'\n\n## v20 全20日與全60日（完成）\n固定五日調整，兩新定義1300窗口；三方法共同552窗口（訓練174／後續356／跨界22）。只改訊號週期、不改配置與頻率。報告：'+rp20.name+'。沒有自動擴增網格。\n',encoding='utf-8')
print('V20_COMPLETE',{'report':rp20.name,'html':hp20.name,'pairs':{k:{'mean_excess':v['all']['mean_excess'],'CI24':v['all']['familyCI24'],'CI48':v['all']['familyCI48'],'train':v['train']['mean_excess'],'later':v['later']['mean_excess']} for k,v in pairs20.items()},'verification':summary20['verification']})


In [ ]:
# v21 retained-kernel metadata only; names lookup without private raw values.
print('V21_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'shape':shape,'lastdate':str(dates[-1]),'raw_type':type(raw_data).__name__,'raw_keys':list(raw_data)[:20] if isinstance(raw_data,dict) else None})
print('V21_NAME_OBJECTS',[(k,type(v).__name__,getattr(v,'shape',None)) for k,v in list(globals().items()) if any(w in k.lower() for w in ['stock_name','name_map','universe','stock_list']) and not k.startswith('_')][:35])


In [ ]:
# Lookup only stock-master schemas and small public-name samples.
for key in ['universe','stocks']:
    v=raw_data[key];print('V21_MASTER',key,type(v).__name__, 'keys',list(v)[:6] if isinstance(v,dict) else None, 'sample',str(v[:2])[:1800] if isinstance(v,list) else '')


In [ ]:
# v21 observational latest25 charts: no new trading/stop rule and no new data fetch.
import re,importlib.util
q21=len(dates)-1;eligible21=np.flatnonzero(np.isfinite(scores_v15['A'][q21]));selected21=eligible21[np.argsort(-scores_v15['A'][q21,eligible21],kind='stable')[:25]]
assert len(selected21)==25
names21={str(r['stock_id']):r['name'] for r in globals().get('current_market_rows_v12',[]) if isinstance(r,dict) and r.get('name')}
name_source21='retainedpublicmaster' if names21 else 'officiallocal150PDF'
pdf_text21=''
pdf_path21=recovery_root/'官方文件'/'玉山挑戰賽_投資組合及交易標的_150檔清單_v1.pdf'
if not names21 and importlib.util.find_spec('pypdf') and pdf_path21.exists():
    from pypdf import PdfReader
    pdf_text21='\n'.join(page.extract_text() or '' for page in PdfReader(str(pdf_path21)).pages)
    # Parse public code-name pairs only. Keep unknown rather than invent names.
    for sid in stock_ids:
        match=re.search(r'(?<!\d)'+re.escape(sid)+r'\s+([^\s\d|]{2,12})',pdf_text21)
        if match:names21[sid]=match.group(1)
curve_returns21={h:np.full(shape,np.nan) for h in [3,5,20,60]}
for h,arr in curve_returns21.items():
    for q in range(h,len(dates)):
        rr=signal_returns_v15[q-h+1:q+1];ok=np.all(np.isfinite(rr)&(rr>-1)&~bad_action_v15[q-h+1:q+1],axis=0)
        arr[q,ok]=np.prod(1+rr[:,ok],axis=0)-1
display_start21=q21-59;xx21=[str(d) for d in dates[display_start21:q21+1]]
members21=[];curves21={}
for rank,j in enumerate(selected21,1):
    sid=stock_ids[j];rr=signal_returns_v15[display_start21+1:q21+1,j]
    assert np.all(np.isfinite(rr)) and not np.any(bad_action_v15[display_start21:q21+1,j])
    index=np.r_[100.,100*np.cumprod(1+rr)]
    ma={h:np.array([np.mean(index[i-h+1:i+1]) if i>=h-1 else np.nan for i in range(len(index))]) for h in [3,5]}
    r3=float(curve_returns21[3][q21,j]);r5=float(curve_returns21[5][q21,j]);below=bool(index[-1]<ma[5][-1])
    watch='3／5日皆負且低於5日均線' if r3<0 and r5<0 and below else ('3或5日轉負' if r3<0 or r5<0 else '3／5日皆非負')
    m={'rank':rank,'stock_id':sid,'name':names21.get(sid,''),'score':float(scores_v15['A'][q21,j]),'close_raw':float(close[q21,j]),'R3':r3,'R5':r5,'R20':float(curve_returns21[20][q21,j]),'R60':float(curve_returns21[60][q21,j]),'below_MA5':below,'short_observation':watch}
    members21.append(m);curves21[sid]={'dates':xx21,'signal_price_index':index,'MA3':ma[3],'MA5':ma[5],**{f'R{h}':a[display_start21:q21+1,j] for h,a in curve_returns21.items()}}
print('V21_MEMBERS',json.dumps(members21,ensure_ascii=False),flush=True)
print('V21_NAMECHECK',{'names_found':sum(bool(m['name']) for m in members21),'source':name_source21,'pdf_matching_lines':[line for line in pdf_text21.splitlines() if any(re.search(r'(?<!\d)'+re.escape(stock_ids[j])+r'(?!\d)',line) for j in selected21)][:30]})


In [ ]:
# v21 all25 individual curves, consistent axes; observation only.
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import html
definition21={'asof':str(dates[q21]),'selection':'originalA=.5rankR20+.5rankR60top25stableties','members':'latestcandidatesnotactualaccount,nothistoricalmembership,noorderissued','display_last_sessions':60,'curves':['signaladjustedpriceindexfirst100','MA3','MA5','rollingR20','rollingR60','rollingR3','rollingR5'],'return_formula':'product(1+adjusted_daily_signal_return)-1','MA_formula':'arithmeticmovingmeanofadjustedpriceindex;not3or5dayreturn','companyactions':'unknownrollingeventsorNaN=>gap;notzero;nointerpolation','shortflag':'R3<0&R5<0&index<MA5descriptiveonly,noexecutionorvalidatedthreshold','source':'retainedsnapshot'+snapshot_sha_v15,'names':'localofficial150PDF','bias':'currenttop25selectionhindsightforpastcurves;notpasttradingperformance','trading_frequency_unchanged':5}
def clean21(x):
    if isinstance(x,np.ndarray):return clean21(x.tolist())
    if isinstance(x,dict):return {k:clean21(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)):return [clean21(v) for v in x]
    if isinstance(x,(float,np.floating)):return float(x) if np.isfinite(x) else None
    if isinstance(x,np.integer):return int(x)
    if isinstance(x,np.bool_):return bool(x)
    return x
jp21=recovery_root/'動能前25檔_短長期曲線資料_v21_2026-10-15.json';jp21.write_text(json.dumps(clean21({'definition':definition21,'members':members21,'curves':curves21}),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
fmt21=lambda v:f'{v*100:+.2f}%'
lines21=['# 動能前25檔：20／60日動能與3／5日風險觀察 v21','', '## 名單與口徑',f"既有價格快照截至{dates[q21]}，以當日原A各半動能分數前25名列出；不是市值前25大、不是實際帳戶持股，也不是某個回測窗口當日持股。原本等權及每5日調整未改動，這次沒有新增交易規則或績效回測。",'目前選出的25檔，其過去曲線僅供觀察：不能當成當時已知道這25檔，也不能推論用這些股票回看過去的績效。名稱來自本機官方150檔清單；價量及公司行動沿用既有快照，沒有重新下載或宣稱為最新即時行情。','', '## 如何閱讀每檔三個面板','1. 公司行動調整價格指數：顯示期首設100；MA3／MA5為該指數最近3／5個交易日平均，保留使用者所說的短期均線。首2／4點不足資料留空。不是實際持股淨值，也不是把每日報酬平均當均線。','2. 20／60日動能：各自滾動累乘公司行動調整日報酬，減1；這是累積漲跌幅，與MA20／MA60不同。','3. 3／5日漲跌幅：短期滾動累積報酬，提供轉弱觀察。3日線／5日線若指均線，請看第一面板；若指近幾日漲跌，請看第三面板。','25張圖使用相同各面板縱軸範圍，保留每天所有點與0%線。未知公司行動或缺資料的報酬留斷線，不補0、不跨缺口連接；所有25檔均直接展開，點表格股票可跳到曲線，不需要設定filter。','', '## 前25名完整表','|動能順位|股票|20日漲跌|60日漲跌|3日漲跌|5日漲跌|短期觀察|','|---|---|---:|---:|---:|---:|---|']
for m in members21:lines21.append(f"|{m['rank']}|{m['stock_id']} {m['name']}|{fmt21(m['R20'])}|{fmt21(m['R60'])}|{fmt21(m['R3'])}|{fmt21(m['R5'])}|{m['short_observation']}|")
both21=[m for m in members21 if m['R3']<0 and m['R5']<0 and m['below_MA5']]
lines21+=['','## 這次能看出的現象',f"25檔中，{len(both21)}檔同時近3／5日累積負報酬且低於5日均線："+'、'.join(m['stock_id']+' '+m['name'] for m in both21)+'。其20／60日仍為正，符合『慢訊號仍強、短期已弱』的觀察。','這不等於三檔都屬急遽下跌，也不等於應立即賣出：不同跌幅與波動、後續反彈、次日均價成交及替補股結果仍需驗證。R3、R5、MA3與MA5都由相同價格構成，彼此相關，不能把三個條件當三份獨立證據。','友達2409是這份快照的明顯示例：20日約+16.56%、60日約+27.78%，近3日約-5.07%、近5日約-12.11%。可以在圖中檢視短期轉弱發生在哪一天，但此處沒有驗證當日是否可預測後續損失。','若要檢驗提前退出，下一個階段應使用每個歷史決策日當時選出的持股，觀察急跌前後及後續反彈，不能只拿今天這25檔回看調參。比賽最早用前日訊號在次日均價成交，不能假設在急跌觸發價格即時停損。','', '## 驗證與保存','原Notebook cells287–290。PID2524及大型資料保留，未重啟；名單依原排序及相同資格門檻，25個官方名稱查得並比對來源。最後一日R20／R60排名各半核對原A；25張圖共175條線，每條60個時間位置，未知值顯示斷線。每檔三面板及相同座標範圍程式核對；未做瀏覽器目視QA。']
report21='\n'.join(lines21);rp21=recovery_root/'動能前25檔_成員與短期風險觀察_v21_2026-10-15.md';rp21.write_text(report21,encoding='utf-8')
allprice21=np.concatenate([c['signal_price_index'] for c in curves21.values()]);alllong21=np.concatenate([c[k]*100 for c in curves21.values() for k in ['R20','R60']]);allshort21=np.concatenate([c[k]*100 for c in curves21.values() for k in ['R3','R5']])
def bounds21(a):
    a=a[np.isfinite(a)];lo=float(a.min());hi=float(a.max());pad=max((hi-lo)*.06,1.);return [min(0,lo-pad),hi+pad]
pricebounds21=[float(allprice21.min()*.94),float(allprice21.max()*1.06)];longbounds21=bounds21(alllong21);shortbounds21=bounds21(allshort21)
cards21=[];figures21=[]
for m in members21:
    sid=m['stock_id'];c=curves21[sid];f=make_subplots(rows=3,cols=1,shared_xaxes=True,vertical_spacing=.07,subplot_titles=['調整價格指數與MA3／MA5','20日／60日累積動能','3日／5日累積漲跌'])
    for name,key,color,dash in [('調整價格指數','signal_price_index','#26344a','solid'),('MA3','MA3','#e5a11c','solid'),('MA5','MA5','#a846a3','solid')]:f.add_trace(go.Scatter(x=c['dates'],y=c[key],name=name,line={'color':color,'dash':dash},connectgaps=False,hovertemplate=name+' %{y:.2f}<extra></extra>'),row=1,col=1)
    for name,key,color,row in [('20日動能','R20','#2563eb',2),('60日動能','R60','#e28021',2),('3日漲跌','R3','#db2777',3),('5日漲跌','R5','#08916f',3)]:f.add_trace(go.Scatter(x=c['dates'],y=c[key]*100,name=name,line={'color':color},connectgaps=False,hovertemplate=name+' %{y:.2f}%<extra></extra>'),row=row,col=1)
    for row,ran,title in [(1,pricebounds21,'期首100'),(2,longbounds21,'累積報酬 %'),(3,shortbounds21,'累積報酬 %')]:f.update_yaxes(range=ran,autorange=False,title_text=title,row=row,col=1)
    f.add_hline(y=0,line_color='#333',row=2,col=1);f.add_hline(y=0,line_color='#333',row=3,col=1)
    f.update_xaxes(type='category',tickmode='array',tickvals=xx21[::12]+[xx21[-1]])
    f.update_layout(height=650,template='plotly_white',hovermode='x unified',margin={'l':60,'r':25,'t':45,'b':45},legend={'orientation':'h','y':1.13,'x':0},uirevision=sid)
    figures21.append(f);cards21.append('<section id="s'+sid+'"><h2>'+str(m['rank'])+'．'+sid+' '+html.escape(m['name'])+'</h2><p>'+html.escape(m['short_observation'])+'；20日 '+fmt21(m['R20'])+'，60日 '+fmt21(m['R60'])+'，3日 '+fmt21(m['R3'])+'，5日 '+fmt21(m['R5'])+'。<a href="#members">回名單</a></p>'+f.to_html(full_html=False,include_plotlyjs=len(cards21)==0,config={'responsive':True,'displaylogo':False})+'</section>')
table21=['<table id="members"><thead><tr><th>順位</th><th>股票（點擊看曲線）</th><th>20日</th><th>60日</th><th>3日</th><th>5日</th><th>觀察</th></tr></thead><tbody>']
for m in members21:
    table21.append('<tr><td>'+str(m['rank'])+'</td><td><a href="#s'+m['stock_id']+'">'+m['stock_id']+' '+html.escape(m['name'])+'</a></td>'+''.join('<td class="'+('negative' if m[k]<0 else 'positive')+'">'+fmt21(m[k])+'</td>' for k in ['R20','R60','R3','R5'])+'<td>'+html.escape(m['short_observation'])+'</td></tr>')
table21.append('</tbody></table>')
hp21=recovery_root/'動能前25檔_20與60日動能及3與5日曲線_v21_2026-10-15.html'
hp21.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>動能前25檔與短期曲線 v21</title><style>html{scroll-behavior:smooth}body{font:16px/1.7 system-ui;max-width:1250px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:9px;border-bottom:1px solid #ddd;text-align:left}thead{background:#eef3fa}.negative{color:#187447}.positive{color:#b42f41}section{border-top:2px solid #dae1eb;margin-top:36px;padding-top:15px}a{color:#2563eb}.note{background:#f2f5fa;padding:18px;border-radius:8px}</style><h1>動能前25檔：長期強勢與近期轉弱</h1><p class="note">資料截至 '+str(dates[q21])+'。原20／60日各半訊號的最新前25名，非帳戶實際持股；五日調整未改動。每檔三張面板直接展開：調整價格與3／5日均線、20／60日動能、3／5日累積漲跌。這是觀察圖，短期轉弱標示不是已驗證的賣出訊號。</p>'+''.join(table21)+''.join(cards21)+'<h2>資料定義與限制</h2>'+render_report17(report21)+'</html>',encoding='utf-8')
assert len(members21)==len(figures21)==25 and len({m['stock_id'] for m in members21})==25
assert all(len(f.data)==7 and all(len(t.x)==60 for t in f.data) for f in figures21)
assert all(list(f.layout.yaxis2.range)==longbounds21 and list(f.layout.yaxis3.range)==shortbounds21 for f in figures21)
assert np.allclose((scores_v15['M20_ONLY20'][q21,selected21]+scores_v15['M20_ONLY60'][q21,selected21])/2,scores_v15['A'][q21,selected21])
assert all(m['name'] for m in members21);assert len(re.findall(r'<section id="s\d+">',hp21.read_text(encoding='utf-8')))==25
sp21=recovery_root/'A策略_無人值守研究狀態.json';state21=json.loads(sp21.read_text(encoding='utf-8'));state21['momentum_curves_v21']={'status':'completed','running':False,'asof':str(dates[q21]),'selected':25,'short3and5negative_belowMA5':[m['stock_id'] for m in both21],'outputs':[hp21.name,rp21.name,jp21.name],'notebook_cells':[287,290],'new_trade_rule':False};sp21.write_text(json.dumps(state21,ensure_ascii=False,indent=2),encoding='utf-8')
wf21=recovery_root/'A策略_無人值守研究流程.md';wf21.write_text(wf21.read_text(encoding='utf-8')+'\n\n## v21 前25名個股動能曲線（完成）\n截至2026-10-12的原A最新候選，非實際帳戶持股。每檔20／60日動能、3／5日累積漲跌與調整價格MA3／MA5，共25張三面板圖；不新增停損規則。成果：'+hp21.name+'。\n',encoding='utf-8')
print('V21_COMPLETE',{'html':hp21.name,'report':rp21.name,'stocks':25,'figures':25,'traces':175,'bothshortnegative':[m['stock_id'] for m in both21],'asof':str(dates[q21]),'kernel_restarted':False})


In [ ]:
# v22 forward24 outcome, not future predictor; extend v21 in place with archival v22 copy.
print('V22_CONTEXT',{'pid':__import__('os').getpid(),'lastdate':str(dates[-1]),'members':len(members21),'asof':str(dates[q21])})
forward24_v22=np.full(shape,np.nan);forward_status_v22={}
for q in range(len(dates)-24):
    rr=signal_returns_v15[q+1:q+25];ok=np.all(np.isfinite(rr)&(rr>-1)&~bad_action_v15[q+1:q+25],axis=0)&np.isfinite(close[q])&(close[q]>0)
    forward24_v22[q,ok]=np.prod(1+rr[:,ok],axis=0)-1
last_start22=len(dates)-25
reference22={'start':str(dates[last_start22]),'end':str(dates[last_start22+24]),'meaning':'same latest25fixedmembers retrospective24dayinterval,not point-in-timehistoricalportfolio'}
for m in members21:
    j=si[m['stock_id']];val=forward24_v22[q21,j]
    m['forward24_from_latest']=None if not np.isfinite(val) else float(val);m['forward24_status']='尚無後續24交易日資料' if q21+24>=len(dates) else ('資料／公司行動不可判定' if not np.isfinite(val) else '已實現')
    val=forward24_v22[last_start22,j];m['latest_complete24_reference']=float(val) if np.isfinite(val) else None
    curves21[m['stock_id']]['forward_R24']=forward24_v22[display_start21:q21+1,j]
definition22={**definition21,'new_panel':'forward24 realisedsignalreturn','formula':'product(1+r[q+1:q+25])-1,24futuretradingreturnsfromqclose','signal_time':'resultknownq+24,notq;neverusedselection','fee_scope':'individualadjustedsignalpricegross,notsimulatedVWAPordersorportfolio_netreturn','latest_unavailable':True,'reference24':reference22,'unknown_outcomes':'NaNbecomesnull;insufficientfuture≠0;unknownactionexcluded'}
rp22=recovery_root/'動能前25檔_24日後結果說明_v22_2026-10-15.md'
lines22=['# 動能前25檔：補上24日後實現漲跌 v22','',f"最新名單的訊號日為{dates[q21]}，資料只到{dates[-1]}，所以最新25檔的『此刻選出後24日報酬』全部尚無資料，不能推估或填0。",'每檔第四面板改為歷史日期q往後24個交易日的實現調整漲跌，使用q+1至q+24日的日報酬累乘。結果直到q+24才知道，圖上日期是訊號參考日，不是收益已可知日期；不可用此線判斷當時買賣。','這是個股公司行動調整訊號口徑毛報酬，不含策略換股、整張、現金、均價成交及費稅，不等於24日投資組合淨期望值。缺資料或未知公司行動留空。','表中額外列出最近有完整資料的24日區間 '+reference22['start']+'→'+reference22['end']+'，只為看這批股票最近的已實現表現；成員仍是最新日期選出的25檔，不能當成當年該起點就選出的歷史組合或樣本外績效。','', '|順位|股票|最新訊號日起24日後|最近完整區間的24日漲跌（事後固定名單）|','|---|---|---|---:|']
for m in members21:lines22.append(f"|{m['rank']}|{m['stock_id']} {m['name']}|{m['forward24_status']}|"+('不可判定' if m['latest_complete24_reference'] is None else fmt21(m['latest_complete24_reference']))+'|')
lines22+=['','原20／60日動能、3／5日漲跌及MA3／MA5保留；每檔第四面板增加向前24日結果，最近24個訊號日期因尚無完整後續資料而留空。全部25檔展開，相同縱軸，0%線保留。','Notebook cells291；沿用PID2524，未重啟。原v21 HTML同步更新，並保存v22副本及JSON；最新日期25檔無結果、最後可計算起點與24個交易日距離、公司行動／缺值遮罩、25×8條曲線均程式核對，未做瀏覽器目視QA。']
rp22.write_text('\n'.join(lines22),encoding='utf-8')
allfwd22=np.concatenate([c['forward_R24']*100 for c in curves21.values()]);fwbounds22=bounds21(allfwd22)
cards22=[];figures22=[]
for m in members21:
    sid=m['stock_id'];c=curves21[sid]
    f=make_subplots(rows=4,cols=1,shared_xaxes=True,vertical_spacing=.055,subplot_titles=['調整價格指數與MA3／MA5','20日／60日累積動能（當時可見）','3日／5日累積漲跌（當時可見）','該日之後24交易日實現漲跌（事後才知道）'])
    for name,key,color,row,scale in [('調整價格指數','signal_price_index','#26344a',1,1),('MA3','MA3','#e5a11c',1,1),('MA5','MA5','#a846a3',1,1),('20日動能','R20','#2563eb',2,100),('60日動能','R60','#e28021',2,100),('3日漲跌','R3','#db2777',3,100),('5日漲跌','R5','#08916f',3,100),('往後24日結果','forward_R24','#7046b0',4,100)]:
        enddates=[str(dates[display_start21+i+24]) if display_start21+i+24<len(dates) else '尚無資料' for i in range(60)]
        f.add_trace(go.Scatter(x=c['dates'],y=c[key]*scale,name=name,line={'color':color},connectgaps=False,customdata=enddates,hovertemplate=name+' %{y:.2f}'+('%' if scale==100 else '')+('<br>結果截至 %{customdata}' if row==4 else '')+'<extra></extra>'),row=row,col=1)
    for row,ran,title in [(1,pricebounds21,'期首100'),(2,longbounds21,'累積報酬 %'),(3,shortbounds21,'累積報酬 %'),(4,fwbounds22,'後24日毛報酬 %')]:
        f.update_yaxes(range=ran,autorange=False,title_text=title,row=row,col=1)
        if row>1:f.add_hline(y=0,line_color='#333',row=row,col=1)
    f.update_xaxes(type='category',tickmode='array',tickvals=xx21[::12]+[xx21[-1]])
    f.update_layout(height=820,template='plotly_white',hovermode='x unified',margin={'l':70,'r':25,'t':50,'b':45},legend={'orientation':'h','y':1.12,'x':0},uirevision=sid)
    figures22.append(f);cards22.append('<section id="s'+sid+'"><h2>'+str(m['rank'])+'．'+sid+' '+html.escape(m['name'])+'</h2><p>最新訊號日起24日後：<b>'+m['forward24_status']+'</b>。<a href="#members">回名單</a></p>'+f.to_html(full_html=False,include_plotlyjs=len(cards22)==0,config={'responsive':True,'displaylogo':False})+'</section>')
table22=['<table id="members"><thead><tr><th>順位</th><th>股票／曲線</th><th>20日</th><th>60日</th><th>3日</th><th>5日</th><th>最新訊號後24日</th><th>最近完整24日區間<br>'+reference22['start']+'→'+reference22['end']+'<br>（事後固定名單參考）</th></tr></thead><tbody>']
for m in members21:
    table22.append('<tr><td>'+str(m['rank'])+'</td><td><a href="#s'+m['stock_id']+'">'+m['stock_id']+' '+html.escape(m['name'])+'</a></td>'+''.join('<td>'+fmt21(m[k])+'</td>' for k in ['R20','R60','R3','R5'])+'<td>'+m['forward24_status']+'</td><td>'+('不可判定' if m['latest_complete24_reference'] is None else fmt21(m['latest_complete24_reference']))+'</td></tr>')
table22.append('</tbody></table>')
hp22=recovery_root/'動能前25檔_短長期曲線與24日後結果_v22_2026-10-15.html'
page22='<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>動能前25檔＋24日後結果 v22</title><style>body{font:16px/1.7 system-ui;max-width:1350px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:8px;border-bottom:1px solid #ddd;text-align:left}thead{background:#eef3fa}section{border-top:2px solid #dae1eb;margin-top:36px;padding-top:15px}a{color:#2563eb}.note{background:#fff4db;padding:18px;border-radius:8px}</style><h1>動能前25檔：補上24日後結果</h1><p class="note">最新訊號日 '+str(dates[q21])+'，資料只到 '+str(dates[-1])+'，最新名單尚無後續24交易日資料。第四面板可查看較早各日期之後的24日實現漲跌，這條線是事後結果，不能用於當日決策。表格最後一欄是較早完整區間的參考，並非最新訊號後報酬。</p>'+''.join(table22)+''.join(cards22)+render_report17('\n'.join(lines22))+'</html>'
hp22.write_text(page22,encoding='utf-8');hp21.write_text(page22,encoding='utf-8')
jp22=recovery_root/'動能前25檔_含24日後結果資料_v22_2026-10-15.json';jp22.write_text(json.dumps(clean21({'definition':definition22,'members':members21,'curves':curves21}),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
assert np.all(np.isnan(forward24_v22[-24:])) and all(m['forward24_from_latest'] is None for m in members21)
assert len(figures22)==25 and all(len(f.data)==8 for f in figures22)
for j in selected21:
    q=last_start22;rr=signal_returns_v15[q+1:q+25,j]
    if np.isfinite(forward24_v22[q,j]):assert len(rr)==24 and np.isclose(forward24_v22[q,j],np.prod(1+rr)-1)
sp22=recovery_root/'A策略_無人值守研究狀態.json';st22=json.loads(sp22.read_text(encoding='utf-8'));st22['forward_curves_v22']={'status':'completed','running':False,'latest_outcomes':'notyetavailable','reference24':reference22,'outputs':[hp22.name,rp22.name,jp22.name],'v21_html_also_updated':True,'notebook_cells':[291]};sp22.write_text(json.dumps(st22,ensure_ascii=False,indent=2),encoding='utf-8')
wf22=recovery_root/'A策略_無人值守研究流程.md';wf22.write_text(wf22.read_text(encoding='utf-8')+'\n\n## v22 個股曲線新增24日後結果（完成）\n最新2026-10-12名單無後續資料；新增各歷史日期向前24交易日的實現漲跌面板，標記事後資訊且不作選股。v21原HTML同步更新，v22保留獨立副本。成果：'+hp22.name+'。\n',encoding='utf-8')
print('V22_COMPLETE',{'html':hp22.name,'v21_updated':hp21.name,'reference24':reference22,'latest_forward_all_missing':True,'figures':25,'traces':200,'kernel_restarted':False})


In [ ]:
# v23 freeze exit x add factorial, retain original25 selection and5day schedule.
print('V23_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'shape':shape,'baseline_windows':len(window_starts_v15)})
definition23={'base':'originalA20/60percentilerankhalf;25equal90%;rebalance5;24session;lot1000fees.001425tax.003','factorial':'exit none/MA5below/2consecutiveMA5below/closebelowprevious5closesmin x addoff/on;8cells7new','indicators':'companyactionadjustedsignalprice,allknownqclose,earliestq+1VWAP;nointradaystop','exit':'qclosebelowownMA5 orqandq-1belowownMA5 orqclosebelowminq-5..q-1;statesnotonlycrossevents;knownqualityrequired','initialcontrol':'allcellsidenticalinitial25and90%;exceptionmonitoring/addstartsnextsession;initialbuydoesnotfiltercurrentweak','replacement':'fromsession2,normal5daynewselectionandreplacementexcludeownweakflag;nonregularonlyfailedheldsharesreplacedoneforonebyhighestrankedhealthynewnamesatpriorvaluetarget,otherholdingsnotretargeted;retain25;insufficienthealthy=>infeasiblenotfabricated','add':'ifprior20daycurrent150qualityproxypositiveANDpositive20daybreadth>=60%ANDselectedholdingR3>0R5>0aboveMA5;market>=120valid/dayandnames;onfirstconditionactivationraisegrossstocktargetup to95%byaddingonlyqualifiednames equalextrabudget;initial90%;regular5day90%baseequalplus5%equaloverqualified;signaloffafteraddontrimtotalto90%;noeverydayfullretarget','add_caps':'newbuyscappedat10%each,2330 25%;notfilledextraifcapsbind;cashdriftsnotfixed5or10;previousNAVandcloseonly','scope':'stoponlystill90targettoisolatestop;addcellsvariable90/95,allcellscash>=0cash<25%,20..30heldand5daycapgracechecked;notActiveSharecertified','data_sufficiency':'cannotreplaceenoughhealthy25orbadprices=>invalid;reportlostbaselineworstwindowsandcoverage;commondateconditionalrisknotunconditionalrisk','analysis':'7newvsbaseline+3addonmarginalwithstops+3stopmarginalwithaddon=13contrastsBonferroni;24/48blocks5000;common8primaryplusownpairs;train<=2024end,laterstart>=2025;crossyearpurged','selection':'commontrain>=100,traincoverage>=80%,bothfamilylowerexcess>0,P10andworstMDDnotworse;qualifythenlowtrainfee;none=>noforcedchoice','knownlimits':'previouslyobservedhistory/current150bias/unknownactions/insufficienthealthy/nextdaygaps/noncertifiedofficialexecution','snapshot':snapshot_sha_v15,'new_windows':4550}
hash23=hashlib.sha256(json.dumps(definition23,sort_keys=True).encode()).hexdigest();(recovery_root/'動能_三種跌破退出與加碼定義_v23_2026-10-15.json').write_text(json.dumps(definition23,ensure_ascii=False,indent=2),encoding='utf-8')
weak23={k:np.zeros(shape,dtype=bool) for k in ['ma5','ma5two','low5']};known23=np.zeros(shape,dtype=bool);above_ma23=np.zeros(shape,dtype=bool)
for q in range(5,len(dates)):
    rr=signal_returns_v15[q-4:q+1];ok=np.all(np.isfinite(rr)&(rr>-1)&~bad_action_v15[q-4:q+1],axis=0);known23[q]=ok
    # Six signal-price levels q-5..q; means q-4..q and q-5..q-1.
    levels=np.vstack([np.ones(shape[1]),np.cumprod(1+rr,axis=0)])
    below=levels[-1]<np.mean(levels[-5:],axis=0);prevbelow=levels[-2]<np.mean(levels[:5],axis=0)
    weak23['ma5'][q]=ok&below;weak23['ma5two'][q]=ok&below&prevbelow;weak23['low5'][q]=ok&(levels[-1]<np.min(levels[:5],axis=0));above_ma23[q]=ok&(levels[-1]>np.mean(levels[-5:],axis=0))
strong23=np.zeros(len(dates),dtype=bool);strong_known23=np.zeros(len(dates),dtype=bool)
for q in range(60,len(dates)):
    rr=signal_returns_v15[q-19:q+1];ok=np.all(np.isfinite(rr)&~bad_action_v15[q-19:q+1],axis=0)
    if ok.sum()<120 or np.any(proxy_count_v18[q-19:q+1]<120):continue
    strong_known23[q]=True;strong23[q]=(np.prod(1+proxy_daily_v18[q-19:q+1])>1 and np.mean(np.prod(1+rr[:,ok],axis=0)>1)>=.6)
def simulate_stopadd23(start,exit_rule='none',add=False):
    shares=np.zeros(len(stock_ids),dtype=np.int64);cash=INITIAL;rc=0.;rs=np.zeros(len(stock_ids));values=[INITIAL];gross=0.;fees=0.;invalid=False;violations=0;mincash=1.;maxcash=0.;cap_age=np.zeros(len(stock_ids),dtype=int);active=False;events=[];exposure=[];reason=None;unknown=0
    caps=np.array([.25 if sid=='2330' else .10 for sid in stock_ids])
    for offset,t in enumerate(range(start,start+24)):
        q=t-1;before=shares.copy();invalid|=bool(np.any(bad_action_v15[t]&((shares+rs)>0)));rc+=float(np.dot(before,cash_div_v15[t]));rs+=before*stock_div_v15[t]
        navprev=cash+np.dot(shares,np.nan_to_num(mark[q]));rank=scores_v15['A'][q];eligible=np.flatnonzero(np.isfinite(rank));ranked=eligible[np.argsort(-rank[eligible],kind='stable')]
        regular=offset%5==0;weak=weak23[exit_rule][q] if exit_rule!='none' else np.zeros(len(stock_ids),dtype=bool)
        failed=np.flatnonzero((shares>0)&weak) if offset>0 else np.array([],dtype=int)
        healthy=[j for j in ranked if exit_rule=='none' or (known23[q,j] and not weak[j])]
        if regular:
            selected=np.array(ranked[:25] if offset==0 or exit_rule=='none' else healthy[:25],dtype=int)
        elif len(failed):
            retained=[j for j in np.flatnonzero(shares>0) if j not in failed];new=[j for j in healthy if shares[j]==0][:len(failed)];selected=np.array(retained+new,dtype=int)
        else:selected=np.flatnonzero(shares>0)
        if len(selected)!=25:invalid=True;reason='insufficient_healthy25';break
        if np.any(~np.isfinite(close[q,selected])|(close[q,selected]<=0)):invalid=True;reason='missing_prior_close';break
        qualified=selected[(curve_returns21[3][q,selected]>0)&(curve_returns21[5][q,selected]>0)&above_ma23[q,selected]]
        wanted=bool(add and offset>0 and strong23[q] and len(qualified)>0)
        if add and not strong_known23[q]:unknown+=1
        changed=wanted!=active;rebalance=regular or len(failed)>0 or changed
        if rebalance:
            target=shares.copy();newids=[]
            if regular:
                target=np.zeros_like(shares);weights=np.full(25,.9/25)
                if wanted:
                    for pos,j in enumerate(selected):
                        if j in qualified:weights[pos]+=.05/len(qualified)
                target[selected]=np.floor(weights*navprev/close[q,selected]/LOT).astype(np.int64)*LOT
            elif len(failed):
                newids=[j for j in selected if shares[j]==0]
                for old,new in zip(failed,newids):
                    target[old]=0;target[new]=int(np.floor(shares[old]*mark[q,old]/close[q,new]/LOT))*LOT
            if not regular and wanted and not active:
                current=float(np.dot(target,np.nan_to_num(close[q])));budget=max(0.,.95*navprev-current)
                target[qualified]=np.floor((target[qualified]*close[q,qualified]+budget/len(qualified))/close[q,qualified]/LOT).astype(np.int64)*LOT
            elif not regular and active and not wanted:
                current=float(np.dot(target,np.nan_to_num(close[q])));scale=min(1.,.9*navprev/current) if current>0 else 1.
                target=np.floor(target*scale/LOT).astype(np.int64)*LOT
            if add:
                # New target exposure caps at known previous prices; no foresight of actual execution.
                maxshares=np.floor(caps*navprev/np.where(np.isfinite(close[q])&(close[q]>0),close[q],np.inf)/LOT).astype(np.int64)*LOT
                buys=target>shares;target=np.where(buys,np.minimum(target,maxshares),target)
            cash,shares,traded,cost,missing=settle(cash,shares,target,execution[t],good_exec_v15[t]);gross+=traded;fees+=cost;invalid|=missing
            events.append({'t':t,'regular':regular,'failed':[stock_ids[j] for j in failed],'new':[stock_ids[j] for j in newids],'add_on':bool(wanted and not active),'add_off':bool(active and not wanted),'active':wanted,'traded':traded/INITIAL,'cost':cost/INITIAL});active=wanted
        nav=cash+np.dot(shares,np.nan_to_num(mark[t]));aum=nav+rc+np.dot(rs,np.nan_to_num(mark[t]));invalid|=bool(np.any((shares>0)&~np.isfinite(mark[t])));weights=shares*np.nan_to_num(mark[t])/nav
        cap_age=np.where(weights>caps+1e-12,cap_age+1,0);violations+=int(cash<0 or cash/nav>=.25 or not 20<=np.count_nonzero(shares)<=30 or np.any(cap_age>5));mincash=min(mincash,cash/nav);maxcash=max(maxcash,cash/nav);values.append(aum);exposure.append(float(1-cash/nav))
    vals=np.array(values)
    return {'start':int(start),'end':int(start+23),'return':float(vals[-1]/INITIAL-1),'mdd':float(np.min(vals/np.maximum.accumulate(vals)-1)),'turnover':gross/INITIAL,'cost':fees/INITIAL,'invalid':bool(invalid),'violations':violations,'min_cash':mincash,'max_cash':maxcash,'path':vals,'events':events,'mean_exposure':float(np.mean(exposure)) if exposure else None,'max_exposure':max(exposure) if exposure else None,'failure_reason':reason,'unknown_add_days':unknown}
for idx in [0,100,200,400,649]:
    idx=int(np.flatnonzero(valid_factor_v16['MOM_E0'])[[0,100,200,400,550]][[0,100,200,400,649].index(idx)])
    now=simulate_stopadd23(int(window_starts_v15[idx]));old=factor_results_v16['MOM_E0'][idx]
    assert now['invalid']==old['invalid'] and now['violations']==old['violations'] and all(np.isclose(now[k],old[k],rtol=0,atol=1e-12) for k in ['return','cost','turnover','mdd'])
sp23=recovery_root/'A策略_無人值守研究狀態.json';st23=json.loads(sp23.read_text(encoding='utf-8'));st23['stop_add_v23']={'status':'defined_running','running':True,'definition_hash':hash23,'new_candidates':7};sp23.write_text(json.dumps(st23,ensure_ascii=False,indent=2),encoding='utf-8')
print('V23_FROZEN',hash23,'5baselinecontrolsmatched',flush=True)


In [ ]:
# Seven new cells once, with original baseline referenced.
results23={'none_base':factor_results_v16['MOM_E0']};specs23={}
for rule in ['none','ma5','ma5two','low5']:
    for add in [False,True]:
        key=rule+('_add' if add else '_base');specs23[key]={'rule':rule,'add':add}
        if key=='none_base':continue
        results23[key]=[simulate_stopadd23(int(start),rule,add) for start in window_starts_v15]
        print('V23_DONE',key,'valid',sum(not r['invalid'] and r['violations']==0 for r in results23[key]),'infeasible',sum(r['failure_reason']=='insufficient_healthy25' for r in results23[key]),flush=True)
(recovery_root/'動能_三種跌破退出與加碼逐窗口_v23_2026-10-15.json').write_text(json.dumps({'definition':definition23,'specs':specs23,'results':results23},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
valid23={k:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results23.items()};ret23={k:np.array([r['return'] for r in rows]) for k,rows in results23.items()};common23=np.logical_and.reduce(list(valid23.values()))
def metrics23(key,mask):
    ids=np.flatnonzero(mask);z=ret23[key][mask];rows=results23[key]
    if not len(ids):return {'n':0}
    return {'n':len(ids),'win_rate':float(np.mean(z>0)),'expectancy':float(z.mean()),'mean_win':float(z[z>0].mean()) if np.any(z>0) else None,'mean_loss':float(z[z<0].mean()) if np.any(z<0) else None,'P10':float(np.quantile(z,.1)),'worst_return':float(z.min()),'worst_mdd':float(min(rows[i]['mdd'] for i in ids)),'mean_cost':float(np.mean([rows[i]['cost'] for i in ids])),'mean_turnover':float(np.mean([rows[i]['turnover'] for i in ids])),'min_actual_cash':float(min(rows[i]['min_cash'] for i in ids)),'max_actual_cash':float(max(rows[i]['max_cash'] for i in ids)),'mean_exposure':float(np.mean([rows[i]['mean_exposure'] for i in ids])) if key!='none_base' else None}
def contrast23(a,b,scope):
    mask=scope&valid23[a]&valid23[b];ids=np.flatnonzero(mask);d=ret23[a]-ret23[b]
    if not len(ids):return {'n':0}
    out={'n':len(ids),'mean_excess':float(d[mask].mean()),'beat_base_rate':float(np.mean(d[mask]>0)),'candidate':metrics23(a,mask),'base':metrics23(b,mask),'mean_cost_change':float(np.mean([results23[a][i]['cost']-results23[b][i]['cost'] for i in ids]))}
    for block,inds in block_indices_v15.items():
        z=np.where(mask,d,np.nan)[inds];n=np.isfinite(z).sum(axis=1);means=np.nansum(z,axis=1)[n>0]/n[n>0]
        out[f'CI{block}']=np.quantile(means,[.025,.975]).tolist();out[f'familyCI{block}']=np.quantile(means,[.05/26,1-.05/26]).tolist();out[f'bootstrap_finite{block}']=len(means)
    return out
comparisons23={};all23=np.ones(len(window_starts_v15),dtype=bool)
definitions_pairs23=[(k,'none_base') for k in results23 if k!='none_base']+[(k+'_add',k+'_base') for k in ['ma5','ma5two','low5']]+[(k+'_add','none_add') for k in ['ma5','ma5two','low5']]
assert len(definitions_pairs23)==13
for a,b in definitions_pairs23:
    key=a+'_minus_'+b;comparisons23[key]={'candidate':a,'base':b,'all':contrast23(a,b,common23),'own_pair':contrast23(a,b,all23),'train':contrast23(a,b,common23&train_v18),'later':contrast23(a,b,common23&test_v18),'years':{year:contrast23(a,b,common23&m) for year,m in year_masks_v15.items() if np.any(common23&m)}}
coverage23={}
for k,rows in results23.items():
    lost=valid23['none_base']&~valid23[k];worst=sorted(np.flatnonzero(lost),key=lambda i:ret23['none_base'][i])[:5]
    coverage23[k]={'valid':int(valid23[k].sum()),'valid_pair_base':int((valid23[k]&valid23['none_base']).sum()),'coverage':float((valid23[k]&valid23['none_base']).sum()/valid23['none_base'].sum()),'invalid':sum(r['invalid'] for r in rows),'violations':sum(r['violations']>0 for r in rows),'failure_reasons':dict(Counter(r.get('failure_reason') for r in rows if r.get('failure_reason'))),'lost_base_worst':[{'start':str(dates[int(window_starts_v15[i])]),'base_return':float(ret23['none_base'][i]),'new_reason':rows[i].get('failure_reason'),'new_invalid':bool(rows[i]['invalid']),'new_violations':rows[i]['violations']} for i in worst]}
qualify23=[]
for k in results23:
    if k=='none_base':continue
    tr=comparisons23[k+'_minus_none_base']['train'];coverage=float((valid23[k]&valid23['none_base']&train_v18).sum()/max(1,(valid23['none_base']&train_v18).sum()));passed=bool(tr.get('n',0)>=100 and coverage>=.8 and tr['familyCI24'][0]>0 and tr['familyCI48'][0]>0 and tr['candidate']['P10']>=tr['base']['P10'] and tr['candidate']['worst_mdd']>=tr['base']['worst_mdd']);tr['pass']=passed;tr['coverage']=coverage
    if passed:qualify23.append(k)
selected23=min(qualify23,key=lambda k:comparisons23[k+'_minus_none_base']['train']['candidate']['mean_cost']) if qualify23 else 'none_base'
events23={}
for k,rows in results23.items():
    if k=='none_base':continue
    ee=[e for i in np.flatnonzero(common23) for e in rows[i]['events']];events23[k]={'add_on':sum(e['add_on'] for e in ee),'add_off':sum(e['add_off'] for e in ee),'nonregular_exit':sum(bool(e['failed']) and not e['regular'] for e in ee),'failed_stock_count':sum(len(e['failed']) for e in ee),'warning':'countsrepeatacrossoverlappingwindows'}
summary23={'definition':definition23,'hash':hash23,'n_common':int(common23.sum()),'train_n':int((common23&train_v18).sum()),'later_n':int((common23&test_v18).sum()),'purged_n':int((common23&purged_v18).sum()),'common_indices':np.flatnonzero(common23).tolist(),'metrics':{k:metrics23(k,common23) for k in results23},'comparisons':comparisons23,'coverage':coverage23,'events':events23,'training_selected':selected23,'warning':'common8conditionalonallfeasible;cannotclaimunconditionalriskreduction;current150bias;nounseenOOS'}
(recovery_root/'動能_三種跌破退出與加碼統計_v23_2026-10-15.json').write_text(json.dumps(summary23,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V23_STATS',json.dumps({'common':summary23['n_common'],'train':summary23['train_n'],'later':summary23['later_n'],'metrics':summary23['metrics'],'selected':selected23,'coverage':{k:{f:v[f] for f in ['valid','coverage','invalid','violations','failure_reasons']} for k,v in coverage23.items()},'pairs':{k:{'delta':v['all'].get('mean_excess'),'CI48':v['all'].get('familyCI48'),'trainpass':v['train'].get('pass')} for k,v in comparisons23.items()},'events':events23},ensure_ascii=False,default=serial_family_v15),flush=True)


In [ ]:
# Price sizing guard: missing close must not invalidate a no-trade carry position.
book23=nbformat.read(notebook_path_v14,as_version=4);context23=book23.cells[292].source
node23=next(n for n in ast.parse(context23).body if isinstance(n,ast.FunctionDef) and n.name=='simulate_stopadd23');engine23=ast.get_source_segment(context23,node23)
oldguard23="        if np.any(~np.isfinite(close[q,selected])|(close[q,selected]<=0)):invalid=True;reason='missing_prior_close';break\n"
assert engine23.count(oldguard23)==1
engine23=engine23.replace(oldguard23,'')
engine23=engine23.replace('            target=shares.copy();newids=[]',"            if regular and np.any(~np.isfinite(close[q,selected])|(close[q,selected]<=0)):invalid=True;reason='missing_prior_close';break\n            if wanted and not active and np.any(~np.isfinite(close[q,qualified])|(close[q,qualified]<=0)):invalid=True;reason='missing_add_close';break\n            target=shares.copy();newids=[]")
engine23=engine23.replace('np.dot(target,np.nan_to_num(close[q]))','np.dot(target,np.nan_to_num(mark[q]))')
exec(engine23,globals());oldvalid23={k:m.copy() for k,m in valid23.items()};recomputed23=0;baseline_controls23={}
for i in np.flatnonzero(valid23['none_base']):
    now=simulate_stopadd23(int(window_starts_v15[i]));old=results23['none_base'][i]
    assert not now['invalid'] and now['violations']==0 and all(np.isclose(now[f],old[f],rtol=0,atol=1e-12) for f in ['return','cost','mdd','turnover'])
    baseline_controls23[int(i)]=now
print('V23_FULL_BASELINE_MATCHED',len(baseline_controls23),flush=True)
for key,spec in specs23.items():
    if key=='none_base':continue
    # Valuation guard also affects additive budgeting while a held close is missing;
    # recalculate all7 once under one corrected, versioned engine to keep accounting uniform.
    results23[key]=[simulate_stopadd23(int(start),spec['rule'],spec['add']) for start in window_starts_v15];recomputed23+=650
    print('V23_QUALITY_DONE',key,sum(not r['invalid'] and r['violations']==0 for r in results23[key]),flush=True)
definition23['engine_quality_revision']='missingcloseguardonlywhenactuallysizing;carryusespreviousmarknotzerofilledmissingclose;all4550recomputed;initialdefinitionsunchanged'
(recovery_root/'動能_三種跌破退出與加碼逐窗口_v23_2026-10-15.json').write_text(json.dumps({'definition':definition23,'specs':specs23,'engine_source_sha256':hashlib.sha256(engine23.encode()).hexdigest(),'results':results23},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
exec(book23.cells[293].source[book23.cells[293].source.index('valid23='):],globals())
summary23['verification']={'baseline_all_valid_controls':len(baseline_controls23),'quality_revision_all_new_rows_recomputed':recomputed23,'not_independent_new_experiments':True,'quality_reason':'missingpriorcloseforunchangedholdmustnotdiscardwindow;noorderusesfutureprice'}
print('V23_QUALITY_COMPLETE',summary23['verification'],flush=True)


In [ ]:
# v23 final accounting, feasibility and all paired points; no parameter expansion.
labels23={'none_base':'原策略／不加碼','none_add':'原策略＋條件加碼','ma5_base':'低於MA5即換股','ma5_add':'低於MA5＋加碼','ma5two_base':'連兩日低於MA5換股','ma5two_add':'連兩日低於MA5＋加碼','low5_base':'破前5日收盤低點換股','low5_add':'破前5日低點＋加碼'}
ids23=np.flatnonzero(common23);summary23['metrics']['none_base']['mean_exposure']=float(np.mean([baseline_controls23[int(i)]['mean_exposure'] for i in ids23]))
fmt23=lambda x:'—' if x is None else f'{x*100:.2f}%'
lines23=['# 動能：三種跌破退出與條件加碼 v23','', '## 本輪結論與分母限制','八組實驗已完成，七個新增定義各650個窗口。三種退出在八組共同可執行日期中，均低於原策略的費後平均報酬；加碼帶來小幅平均增量，但校正區間跨零，未證實改良。不能由這輪推論所有停損都無效。',f"重要：八組共同只有{summary23['n_common']}窗口，訓練{summary23['train_n']}、後續{summary23['later_n']}、跨界{summary23['purged_n']}。原策略全體有效557窗口，部分差行情因找不到足夠替補或整張後難維持本輪目標25檔而不在共同表。不得把共同表較小回撤當成已躲過完整歷史的崩跌。共同訓練只有79，未達預先100窗口門檻，也不能強選新方法。",'原策略在本表的高勝率與高均值是共同日期改變的結果，不是原策略本身突然改良；請只在本表內比較，另看每方法自己與原策略的配對及覆蓋。','', '## 固定交易與三種退出','主線維持原20／60日排名各半、目標25檔、每五日例行評估、24日賽期。第一次建倉所有組完全相同且只投入90%，例外監測與加碼最早次日啟動。三退出用前一日公司行動調整價格判斷，次日均價執行，不是盤中停損。','1. 單日：收盤價低於該日MA5。這是持續低於的狀態條件，不只限第一次穿越。','2. 兩日：昨日與前日，各自收盤都低於各自MA5。','3. 前低：昨日收盤低於此前五個交易日的最低收盤價；比較區間不含昨日。這不是含盤中高低的K棒最低價。','觸發時換出持股，依原動能排名找沒有觸發同一條件的新股票一對一替補。非例行退出時，未退出的股票不重新等權，替補目標沿用退出股的前日估值。後續五日例行選股也排除觸發同條件股票，因此效果同時包含提前退出與避免重新買入弱股，不是只改賣出的純因果效果。','本輪維持目標25以隔離差異。候選不足、整張後持股不足以維持本輪目標或資料不齊就標不可執行；沒有硬買已跌破的股票湊數。官方最低是20，不是25，本輪不足25不等於官方違規，後續若改20～25須另有定義，不在這輪事後放寬以改善結果。','', '## 加碼規則：現金不再固定10%','市場代理過去20日累積正報酬，而且至少60%的完整資料股票20日上漲時，才允許加碼。代理來自目前150檔、排除未知公司行動，近20日每天與個股完整樣本均至少120檔；資訊不足不加碼。','加碼對象限定已選持股中：近3日與5日報酬皆正、收盤高於MA5。首次條件成立，把整體股票目標從90%提高到最高95%，額外預算均分給符合者，並檢查台積電25%／其他10%上限，碰上限或整張取整則不強行用完。首次建倉仍90%，並非用事後資料決定一開始加滿。','條件撤回時降低至90%目標；下一個五日例行調整才重新分配基礎等權及加碼額度。同一狀態維持時不每日全面重配。95%是研究用固定水準，不是已證實最佳比例；實際現金會隨價格、費用及整張漂移，不會永遠剛好5%或10%。','當日均價與前收不同，所有現金／權重／持股數的日末檢查仍執行。應收股息不動用。所有買賣手續費0.1425%、賣出0.3%稅及整千股沿用原模型；期末按資產估值，沒有假設最後全部清倉。Active Share與正式DataPack未驗證，不能稱完整官賽合規。','', '## 同219窗口：勝率、期望值與風險','|方法|費後勝率|24日期望值|平均獲利|平均虧損|P10|最差窗口內回撤|平均費稅|平均實際股票曝險|','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for k,m in summary23['metrics'].items():lines23.append('|'+labels23[k]+'|'+'|'.join(fmt23(m[f]) for f in ['win_rate','expectancy','mean_win','mean_loss','P10','worst_mdd','mean_cost','mean_exposure'])+'|')
lines23+=['','## 提前退出與加碼的配對效果','13個預先對照：七新增相對原策略、三個加碼對同退出不加碼、三個退出加碼對只加碼。24／48位置區塊bootstrap各5000次，13比較Bonferroni；以下區間為平均差不確定性，不是下一次結果範圍，重疊窗口不當作獨立試驗。','|候選減對照|平均差pp|48區塊家族95%區間pp|增加費稅pp|','|---|---:|---|---:|']
for name,s in comparisons23.items():
    a=s['all'];ci=a['familyCI48'];lines23.append(f"|{labels23[s['candidate']]}減{labels23[s['base']]}|{a['mean_excess']*100:+.3f}|[{ci[0]*100:+.3f},{ci[1]*100:+.3f}]|{a['mean_cost_change']*100:+.3f}|")
lines23+=['','## 各自配對的覆蓋與增量：不可跨列直接排名','以下每列使用該方法與原策略自己的相同有效日期，因此分母不同，用來檢查共同219日期的偏差。','|候選|自己配對n|原策略有效日期覆蓋|自己配對平均差pp|48區塊家族95%區間pp|','|---|---:|---:|---:|---|']
for k in results23:
    if k=='none_base':continue
    s=comparisons23[k+'_minus_none_base']['own_pair'];ci=s['familyCI48'];lines23.append(f"|{labels23[k]}|{s['n']}|{fmt23(coverage23[k]['coverage'])}|{s['mean_excess']*100:+.3f}|[{ci[0]*100:+.3f},{ci[1]*100:+.3f}]|")
lines23+=['','## 為何不能宣稱已防住崩跌？','|候選|原650窗口中有效數|不可判定數|限制違反窗口|排除的原策略最差窗口之一及其原報酬|','|---|---:|---:|---:|---|']
for k,c in coverage23.items():
    if k=='none_base':continue
    lost=c['lost_base_worst'];sample=(lost[0]['start']+'，原策略'+fmt23(lost[0]['base_return'])+'；'+str(lost[0]['new_reason'])) if lost else '沒有排除原有效窗口'
    lines23.append(f"|{labels23[k]}|{c['valid']}|{c['invalid']}|{c['violations']}|{sample}|")
lines23+=['日誌insufficient_healthy25包含候選數量不足，以及整張／加減碼後持股數未達本輪目標25的情況，不能全歸因市場同步跌破。資料／公司行動／成交不可判定與限制違反可能重疊，不能把排除數相加；不把無效窗口填零或拿部分窗口報酬當完成結果。','尤其單日MA5退出覆蓋原策略有效日期不到一半；此策略在該配置下無法支援完整研究期間。若要評估放寬成20～25檔、部分減碼或持有更多現金，必須預先訂規則並遵守官方現金上限，不能用這份被篩選的共同樣本斷言已有效止損。','', '## 分段選擇與結論限制','訓練僅用2024年底前已結束窗口，後續僅用2025年起始窗口，跨界淨空；共同訓練79未達100。其他預先門檻為覆蓋80%、兩個區塊校正下界>0、P10與最差回撤不劣，合格才按訓練費用低優先，本輪無合格者，不強選。','只加碼在共同樣本的淨平均增量約+0.144pp，但48區塊校正區間約[-0.164,+0.536]pp，且尾端虧損與最差回撤較深。加碼不會免費提高獲利，它增加股票曝險與交易費。三個退出合併加碼相較各自只退出有小幅正均值，但區間皆跨零，仍不足稱穩定改良。','三種快速退出的費用顯著升高；單日MA5平均費稅約3.33%，原策略約0.80%。不能僅從扣費結果斷言全差額都是費用，也不能把所有技術停損一概判無效；此處只否定本輪固定規則已證實改善。','這些過往資料在前輪已看過，是時間分段回溯研究，不是真正未見樣本外；目前150檔有存活及選擇偏誤、未解公司行動、較短歷史。13比較校正只是本輪家族，不消除前面多輪研究選擇偏誤。','', '## 驗證','原Notebook cells292–295。原kernel PID2524保留，未重啟。557個原策略有效窗口全部重播，報酬、費用、換手及回撤逐項一致。缺前收盤但未下單的持股用原mark估值，不以缺值當0或無條件排除；修正後4550列依同一引擎重算，屬品質修正而非新增實驗。有效樣本條件損益恆等式、配對期望差、首日相同、無未來價格下單、13對照及每日限制程式核對。圖保留每個點、中位數及0線；未做瀏覽器目視QA。']
report23='\n'.join(lines23);rp23=recovery_root/'動能_三種跌破退出與加碼研究_v23_2026-10-15.md';rp23.write_text(report23,encoding='utf-8')
import plotly.graph_objects as go
rng23=np.random.default_rng(23015);fig23=go.Figure();diff23=go.Figure();dates23=[str(dates[int(window_starts_v15[i])]) for i in ids23]
for pos,(key,m) in enumerate(summary23['metrics'].items()):
    z=ret23[key][common23]*100
    fig23.add_trace(go.Box(x=np.full(len(z),pos),y=z,name=labels23[key],boxpoints=False,line={'color':'#4e77a8'},fillcolor='rgba(78,119,168,.15)',hoverinfo='skip',showlegend=False))
    fig23.add_trace(go.Scatter(x=pos+rng23.uniform(-.18,.18,len(z)),y=z,mode='markers',marker={'size':5,'opacity':.45},text=dates23,hovertemplate=labels23[key]+'<br>%{text}<br>費後24日報酬 %{y:.2f}%<extra></extra>',showlegend=False))
    median=float(np.median(z));fig23.add_trace(go.Scatter(x=[pos-.2,pos+.2],y=[median,median],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
for pos,key in enumerate([k for k in results23 if k!='none_base']):
    z=(ret23[key]-ret23['none_base'])[common23]*100;s=comparisons23[key+'_minus_none_base']['all'];ci=s['familyCI48'];mean=s['mean_excess']*100
    diff23.add_trace(go.Scatter(x=pos+rng23.uniform(-.18,.18,len(z)),y=z,mode='markers',marker={'size':5,'opacity':.45},text=dates23,hovertemplate=labels23[key]+'<br>%{text}<br>相對原策略 %{y:.3f} pp<extra></extra>',showlegend=False))
    diff23.add_trace(go.Scatter(x=[pos],y=[mean],mode='markers',marker={'color':'#dc2477','size':11},error_y={'type':'data','array':[ci[1]*100-mean],'arrayminus':[mean-ci[0]*100]},hovertemplate='平均差及48區塊家族校正區間 %{y:.3f} pp<extra></extra>',showlegend=False))
fig23.add_hline(y=0,line_color='#222');diff23.add_hline(y=0,line_color='#222')
fig23.update_layout(template='plotly_white',height=580,title='219共同可執行窗口：全部點與桃紅中位數（不是完整歷史風險）',xaxis={'tickmode':'array','tickvals':list(range(8)),'ticktext':[labels23[k] for k in results23],'tickangle':-20},yaxis_title='費後24日報酬 %')
diff23.update_layout(template='plotly_white',height=520,title='七候選相對原策略：全差值點及均值／家族校正區間',xaxis={'tickmode':'array','tickvals':list(range(7)),'ticktext':[labels23[k] for k in results23 if k!='none_base'],'tickangle':-20},yaxis_title='配對差 百分點')
hp23=recovery_root/'動能_跌破退出加碼八組比較_v23_2026-10-15.html'
hp23.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>跌破退出與加碼八組 v23</title><style>body{font:16px/1.7 system-ui;max-width:1400px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#eef3fa;font-weight:bold}p{overflow-wrap:anywhere}.note{background:#fff4db;padding:18px;border-radius:8px}</style><h1>動能：三種跌破退出 × 條件加碼</h1><p class="note">完成八組，但共同只有219可執行窗口、訓練79不足門檻。單日MA5退出覆蓋不到原有效日期一半，不得把排除差行情後的回撤當成完整風險。加碼目標90%／95%隨條件改變，現金不再固定10%；未證實有穩定改善。</p>'+fig23.to_html(full_html=False,include_plotlyjs=True)+diff23.to_html(full_html=False,include_plotlyjs=False)+render_report17(report23)+'</html>',encoding='utf-8')
first_day23=0
for k,rows in results23.items():
    if k=='none_base':continue
    for i,row in enumerate(rows):
        old=results23['none_base'][i]
        if len(row['path'])>1 and len(old['path'])>1:assert np.isclose(row['path'][1],old['path'][1],rtol=0,atol=1e-5);first_day23+=1
for key,m in summary23['metrics'].items():
    x=ret23[key][common23];assert np.isclose(m['expectancy'],np.mean(x>0)*(m['mean_win'] or 0)+np.mean(x<0)*(m['mean_loss'] or 0))
for s in comparisons23.values():assert np.isclose(s['all']['mean_excess'],s['all']['candidate']['expectancy']-s['all']['base']['expectancy'])
for key,rows in results23.items():
    for i in np.flatnonzero(valid23[key]):assert rows[i]['min_cash']>=0 and rows[i]['max_cash']<.25 and rows[i]['violations']==0
assert sum(len(t.y) for t in fig23.data if t.type=='scatter' and t.mode=='markers')==219*8
assert sum(len(t.y) for t in diff23.data if t.type=='scatter' and len(t.y)>1)==219*7
summary23['verification'].update({'initial_day_checks':first_day23,'return_plot_points':1752,'paired_plot_points':1533,'conditional_payoffs_checked':True,'valid_daily_cash_caps_checked':True,'no_kernel_restart':True})
(recovery_root/'動能_三種跌破退出與加碼統計_v23_2026-10-15.json').write_text(json.dumps(summary23,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
st23=json.loads(sp23.read_text(encoding='utf-8'));st23['stop_add_v23'].update({'status':'completed','running':False,'new_candidates':7,'new_windows':4550,'common_n':219,'train_n':79,'later_n':133,'selection':selected23,'confirmed_improvement':False,'coverage_warning':'singleMA5lesshalfvaliddates;commonriskconditional','outputs':[rp23.name,hp23.name,'動能_三種跌破退出與加碼統計_v23_2026-10-15.json'],'notebook_cells':[292,295]});sp23.write_text(json.dumps(st23,ensure_ascii=False,indent=2),encoding='utf-8')
wf23=recovery_root/'A策略_無人值守研究流程.md';wf23.write_text(wf23.read_text(encoding='utf-8')+'\n\n## v23 三跌破退出與條件加碼（完成）\n三退出×有無加碼，加原策略／只加碼，共八組，七新增4550窗口。557原有效控制全數一致；共同219（訓練79不足100），單日MA5覆蓋不到一半。加碼90%／95%條件變化，不固定10%現金，增量尚未證實；三退出在共同窗口費後較差。不可把無法替補的差行情排除後稱有效避險。成果：'+hp23.name+'。沒有自動擴增新實驗或排程。\n',encoding='utf-8')
print('V23_COMPLETE',{'report':rp23.name,'html':hp23.name,'verification':summary23['verification'],'ownpairs':{k:{'n':v['own_pair']['n'],'mean_excess':v['own_pair']['mean_excess'],'CI48':v['own_pair']['familyCI48']} for k,v in comparisons23.items() if v['base']=='none_base'}})


In [ ]:
# v24 exposure / cash reserve / conditional add factorial, no exit rules.
print('V24_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'shape':shape,'v23_done':len(results23),'snapshot':snapshot_sha_v15})
specs24={'F80':(.8,.8,False),'F85':(.85,.85,False),'F90':(.9,.9,False),'F95':(.95,.95,False),'F100':(1.,1.,False),'D80_85':(.8,.85,True),'D80_95':(.8,.95,True),'D90_95':(.9,.95,True),'D80_100':(.8,1.,True),'D90_100':(.9,1.,True)}
core24=['F80','F85','F90','F95','D80_85','D80_95','D90_95'];full24=['F100','D80_100','D90_100']
pair_defs24=[('F80','F90'),('F95','F90'),('F100','F90'),('D80_85','F80'),('D80_95','F80'),('D90_95','F90'),('D80_100','F80'),('D90_100','F90'),('D80_85','F85'),('D80_95','F95'),('D90_95','F95'),('D80_100','F100'),('D90_100','F100')]
definition24={'fixed':'originalA20/60half,25target,5dayregular,24sessions,lots1000,samefeesandpriceengine,exitnone','methods':specs24,'reuse':['F90fromv16','D90_95fromv23'],'new_methods':8,'core_shared':core24,'literal100_diagnostics':full24,'denominator':'100cellsnotallowedtodropfeasiblecorewindowstoalowcount;100compareonlyownvalidpairs+showdata-quality-onlyincludingviolationsasmodeledunsafe diagnostic','add_rule':'exactv23market20positivebreadth>=60%,>=120names/day;qualifyheldR3R5positiveaboveMA5;extraonlyqualifiednames;offreturntobase;initialbaseexposureallthenactivationq+1;firstdaynotforcedceiling','cash_100':'literal100%previousNAVstocksbeforefees;integerlotresidualnotsparecashpolicy;noforesightresizingorautomatic99.xfallback;negativecash/otherlimitsreportednotcertified;100+add100identicalfixed100,noadditionalcashwithoutsellingorleverage','outcomes':'netstrategypositivewinrate;conditionalincrementpositivevsnoaddwinrate;allwindowandtriggeredwindowseparate;tiesexplicit;meanincrement,costincrement,risk','mechanism':'same+5pointadd80to85vs90to95;dynamicvsstaticceiling;notallbenefittocashitself,stockselection/addallocation/timing/costremain','contrasts':'13pairs+1differenceofincrements=14Bonferroni;24/48positionblocks5000;trainandlaterasprior,no genuineunseenOOS','fees':'buysell.001425+selltax.003;cashzerointerest;terminalmarknotforcedliquidation','limits':'current150survivorship;unknownactions;25targetinfeasibility/lotshortfall;ActiveShareunverified;familyonlycurrentexperiment','snapshot':snapshot_sha_v15}
hash24=hashlib.sha256(json.dumps(definition24,sort_keys=True).encode()).hexdigest();(recovery_root/'動能_現金比例與加碼歸因定義_v24_2026-10-15.json').write_text(json.dumps(definition24,ensure_ascii=False,indent=2),encoding='utf-8')
cash_engine24=engine23.replace("def simulate_stopadd23(start,exit_rule='none',add=False):", "def simulate_cash24(start,base_invest=.9,add_ceiling=.95,add=False):\n    exit_rule='none'")
for old,new in [('.95*navprev','add_ceiling*navprev'),('.9/25','base_invest/25'),('.05/len(qualified)','(add_ceiling-base_invest)/len(qualified)'),('.9*navprev','base_invest*navprev')]:cash_engine24=cash_engine24.replace(old,new)
exec(cash_engine24,globals())
for key,source in [('F90',results23['none_base']),('D90_95',results23['none_add'])]:
    mask=np.array([not r['invalid'] and r['violations']==0 for r in source]);indices=np.flatnonzero(mask)[[0,100,200,400,500]]
    for i in indices:
        got=simulate_cash24(int(window_starts_v15[i]),*specs24[key]);old=source[int(i)]
        assert got['invalid']==old['invalid'] and got['violations']==old['violations'] and all(np.isclose(got[f],old[f],rtol=0,atol=1e-12) for f in ['return','mdd','turnover','cost'])
sp24=recovery_root/'A策略_無人值守研究狀態.json';state24=json.loads(sp24.read_text(encoding='utf-8'));state24['cash_add_v24']={'status':'defined_running','running':True,'hash':hash24,'new_methods':8,'reuse':2};sp24.write_text(json.dumps(state24,ensure_ascii=False,indent=2),encoding='utf-8')
print('V24_FROZEN',hash24,'10reusecontrolsverified',flush=True)


In [ ]:
# v24 finite runs, retaining complete cost/feasibility records.
results24={'F90':results23['none_base'],'D90_95':results23['none_add']}
for key,spec in specs24.items():
    if key in results24:continue
    results24[key]=[simulate_cash24(int(start),*spec) for start in window_starts_v15]
    print('V24_DONE',key,'valid',sum(not r['invalid'] and r['violations']==0 for r in results24[key]),'negativecash',sum(r['min_cash']<0 for r in results24[key]),flush=True)
# Preserve definition order, independent of reuse insertion order.
results24={k:results24[k] for k in specs24};valid24={k:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results24.items()};data24={k:np.array([not r['invalid'] for r in rows]) for k,rows in results24.items()};ret24={k:np.array([r['return'] for r in rows]) for k,rows in results24.items()}
common24=np.logical_and.reduce([valid24[k] for k in core24]);allcommon24=np.logical_and.reduce(list(valid24.values()));alltrue24=np.ones(650,dtype=bool)
def metrics24(key,mask):
    ids=np.flatnonzero(mask);x=ret24[key][mask];rows=results24[key]
    if not len(ids):return {'n':0}
    return {'n':len(ids),'win_rate':float(np.mean(x>0)),'expectancy':float(x.mean()),'mean_win':float(x[x>0].mean()) if np.any(x>0) else None,'mean_loss':float(x[x<0].mean()) if np.any(x<0) else None,'P10':float(np.quantile(x,.1)),'worst_mdd':float(min(rows[i]['mdd'] for i in ids)),'mean_cost':float(np.mean([rows[i]['cost'] for i in ids])),'min_cash':float(min(rows[i]['min_cash'] for i in ids)),'max_cash':float(max(rows[i]['max_cash'] for i in ids)),'mean_exposure':float(np.mean([rows[i]['mean_exposure'] if key!='F90' else baseline_controls23[int(i)]['mean_exposure'] for i in ids]))}
def interval24(d,mask):
    out={}
    if not np.any(mask):return out
    for block,inds in block_indices_v15.items():
        x=np.where(mask,d,np.nan)[inds];cnt=np.isfinite(x).sum(axis=1);means=np.nansum(x,axis=1)[cnt>0]/cnt[cnt>0]
        out[f'CI{block}']=np.quantile(means,[.025,.975]).tolist();out[f'familyCI{block}']=np.quantile(means,[.05/28,1-.05/28]).tolist();out[f'bootstrap_finite{block}']=len(means)
    return out
def pair24(a,b,scope):
    mask=scope&valid24[a]&valid24[b];ids=np.flatnonzero(mask);d=ret24[a]-ret24[b]
    if not len(ids):return {'n':0}
    has_add=specs24[a][2];trigger=np.array([any(e['add_on'] for e in r.get('events',[])) for r in results24[a]]) if has_add else np.zeros(650,dtype=bool);tol=1e-12
    result={'n':len(ids),'mean_excess':float(d[mask].mean()),'increment_positive_win_rate':float(np.mean(d[mask]>tol)),'increment_negative_rate':float(np.mean(d[mask]<-tol)),'increment_flat_rate':float(np.mean(np.abs(d[mask])<=tol)),'candidate':metrics24(a,mask),'base':metrics24(b,mask),'mean_cost_change':float(np.mean([results24[a][i]['cost']-results24[b][i]['cost'] for i in ids])),'triggered_window_n':int((mask&trigger).sum()),'triggered_increment_win_rate':float(np.mean(d[mask&trigger]>tol)) if np.any(mask&trigger) else None,'triggered_mean_increment':float(d[mask&trigger].mean()) if np.any(mask&trigger) else None,'triggered_warning':'windowhas>=1actualaddactivation,notperindependentaddtrade'}
    result.update(interval24(d,mask));return result
comparisons24={}
for a,b in pair_defs24:
    core=a in core24 and b in core24;scope=common24 if core else alltrue24
    comparisons24[a+'_minus_'+b]={'candidate':a,'base':b,'primary_scope':'core7commondates' if core else 'literal100ownfeasiblepair_notcomparabletoothers','main':pair24(a,b,scope),'own_pair':pair24(a,b,alltrue24),'train':pair24(a,b,scope&train_v18),'later':pair24(a,b,scope&test_v18),'years':{y:pair24(a,b,scope&m) for y,m in year_masks_v15.items() if np.any(scope&m&valid24[a]&valid24[b])}}
dd24=(ret24['D80_85']-ret24['F80'])-(ret24['D90_95']-ret24['F90']);dd_summary24={'n':int(common24.sum()),'mean_difference_of_increments':float(dd24[common24].mean()),'fraction_positive':float(np.mean(dd24[common24]>1e-12)),'formula':'(D80to85-F80)-(D90to95-F90);same5pointincrement',**interval24(dd24,common24)}
coverage24={}
for key,rows in results24.items():
    coverage24[key]={'valid_n':int(valid24[key].sum()),'data_complete_n':int(data24[key].sum()),'negative_cash_windows':sum(r['min_cash']<0 for r in rows),'cash25_or_more_windows':sum(r['max_cash']>=.25 for r in rows),'violation_windows':sum(r['violations']>0 for r in rows),'invalid_n':sum(r['invalid'] for r in rows),'coverage_vs_F90':float((valid24[key]&valid24['F90']).sum()/valid24['F90'].sum()),'failure_reasons':dict(Counter(r.get('failure_reason') for r in rows if r.get('failure_reason')))}
summary24={'definition':definition24,'hash':hash24,'core_common_n':int(common24.sum()),'all10_common_n':int(allcommon24.sum()),'train_n':int((common24&train_v18).sum()),'later_n':int((common24&test_v18).sum()),'purged_n':int((common24&purged_v18).sum()),'core_common_indices':np.flatnonzero(common24).tolist(),'core_metrics':{k:metrics24(k,common24) for k in core24},'literal100_own_valid_metrics':{k:metrics24(k,valid24[k]) for k in full24},'literal100_data_complete_but_may_violate_UNSAFE':{k:metrics24(k,data24[k]) for k in full24},'comparisons':comparisons24,'same5increment_difference':dd_summary24,'coverage':coverage24,'interpretation_limits':['cashitselfearnszero','addedstockexposure/timing/allocation/costnotcausallyseparatedcompletely','100validselectionbias','overlappingwindows','pastseenhistory','current150bias','ActiveShareunverified']}
(recovery_root/'動能_現金比例與加碼歸因逐窗口_v24_2026-10-15.json').write_text(json.dumps({'definition':definition24,'specs':specs24,'engine_hash':hashlib.sha256(cash_engine24.encode()).hexdigest(),'results':results24},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
(recovery_root/'動能_現金比例與加碼歸因統計_v24_2026-10-15.json').write_text(json.dumps(summary24,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V24_STATS',json.dumps({'core_n':summary24['core_common_n'],'all10_n':summary24['all10_common_n'],'train':summary24['train_n'],'later':summary24['later_n'],'metrics':summary24['core_metrics'],'literal100_ownvalid':summary24['literal100_own_valid_metrics'],'coverage':coverage24,'pairs':{k:{'n':v['main'].get('n'),'delta':v['main'].get('mean_excess'),'increment_win':v['main'].get('increment_positive_win_rate'),'triggered_n':v['main'].get('triggered_window_n'),'triggered_win':v['main'].get('triggered_increment_win_rate'),'CI48':v['main'].get('familyCI48')} for k,v in comparisons24.items()},'same5':dd_summary24},ensure_ascii=False,default=serial_family_v15),flush=True)


In [ ]:
# v24 final exposure attribution with separate increment win rates and infeasible100diagnostics.
labels24={'F80':'固定80%／現金目標20%','F85':'固定85%／現金目標15%','F90':'固定90%／現金目標10%','F95':'固定95%／現金目標5%','F100':'名目固定100%／0%','D80_85':'80%→條件加碼85%','D80_95':'80%→條件加碼95%','D90_95':'90%→條件加碼95%','D80_100':'80%→條件加碼100%','D90_100':'90%→條件加碼100%'}
fmt24=lambda x:'—' if x is None else f'{x*100:.2f}%'
# Clarify an interpretation in the frozen plan, without changing any tested rule.
definition24['cash_100_interpretation_clarification']='nominal100hasnoplannedreservebutlot/pricecashresidualcanexist;reinvestmentofthatresidualisadistinctuntestedrule,notanadditional20or10pointbudget;didnotrunD100to100'
for k in full24:coverage24[k]['negative_cash_in_data_complete']=sum(bool(data24[k][i]) and r['min_cash']<0 for i,r in enumerate(results24[k]))
lines24=['# 原五日動能策略：現金比例與加碼來源 v24','', '## 回答本輪問題','這份樣本支持「增加投入額度主要放大獲利與虧損幅度」，沒有證實預留20%比預留10%讓同一個加碼訊號更準。現金在模型中利息為0；加碼增益來自資金投入股票後的曝險、選股額度、時點與成本，不是閒置現金本身產生報酬。','固定80%、85%、90%、95%在同499窗口的費後勝率恰好都70.94%。等幅加5pp時，80→85與90→95的實際觸發窗口增量勝率也恰同51.61%；增加15pp的80→95為52.52%，不是大幅提高判斷準確度。這是樣本事實，不是數學上所有資金比例都必然同勝率。','在這份資料，80→85比固定85平均少約0.173pp；90→95比固定95少約0.168pp；80→95比固定95少約0.503pp。但本輪家族校正區間仍跨零，不能稱已證明固定上限一定優於等待訊號。沒有證實加碼時機提供額外優勢。','', '## 固定交易與對照','全部維持原20／60日動能各半、目標25檔、每5日例行調整、24交易日、整千股、同均價與費稅，不加v23的任何跌破退出。買賣手續費各0.1425%、賣出另0.3%，期末估值而非強制全賣。','固定80／85／90／95／100%是每次五日調整的股票目標，不保證每日實際現金等於20／15／10／5／0%。價格、整張、費用會改變現金比例。','條件加碼完整沿用v23：市場20日正報酬且上漲家數>=60%，資料須足120檔；額外加給已持股中近3／5日皆正且高於MA5者。條件撤回退至各自基礎投入；維持同狀態不每日全面重配；初始建倉只用基礎投入、最早次日啟動加碼。單股新買入上限檢查，未能用完預算則留現金。','同5pp比較80→85與90→95，隔離加碼幅度；相同上限比較動態與固定85／95，檢查等訊號的效果。選股與加碼分配仍有差別，因此不是完整純因果時點效果。','本輪10方法，其中F90與90→95各5個有效窗口控制核對後直接引用；8新增定義共5200窗口，沒有重算兩個既有方法。','', '## 勝率有兩種不同意思','1. 策略勝率＝24日費後總報酬>0的窗口比例。','2. 加碼增量勝率＝同日期「加碼淨報酬−相同基礎不加碼淨報酬」>0的比例。另只在實際觸發過加碼的窗口計算，避免把沒有加碼的窗口混成輸贏。平手另外保存；這不是每筆獨立加碼交易勝率。','', '## 七個保留緩衝的方法：同499窗口',f"共同{summary24['core_common_n']}窗口，訓練{summary24['train_n']}、後續{summary24['later_n']}、跨年淨空{summary24['purged_n']}。名目100%三組另列，不用它們的低覆蓋把本表縮成118窗口。",'|方法|策略勝率|24日期望值|平均獲利|平均虧損|P10|最差窗口內回撤|平均費稅|平均實際股票曝險|','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for k,m in summary24['core_metrics'].items():lines24.append('|'+labels24[k]+'|'+'|'.join(fmt24(m[f]) for f in ['win_rate','expectancy','mean_win','mean_loss','P10','worst_mdd','mean_cost','mean_exposure'])+'|')
lines24+=['','## 加碼是否勝過相同基礎投入？','以下前三列同499窗口，其中436窗口實際觸發過加碼；不要把51%與策略勝率70%混在一起。增量收益單位是百分點，不是新增資金的投資報酬率。','|加碼方式|增加額度pp|相對不加碼平均增量pp|全部窗口增量勝率|觸發窗口數|觸發窗口增量勝率|未觸發／平手窗口比例|48區塊家族95%區間pp|','|---|---:|---:|---:|---:|---:|---:|---|']
for a,b in [('D80_85','F80'),('D80_95','F80'),('D90_95','F90')]:
    x=comparisons24[a+'_minus_'+b]['main'];ci=x['familyCI48'];extra=(specs24[a][1]-specs24[a][0])*100
    lines24.append(f"|{labels24[a]}|{extra:.0f}|{x['mean_excess']*100:+.3f}|{fmt24(x['increment_positive_win_rate'])}|{x['triggered_window_n']}|{fmt24(x['triggered_increment_win_rate'])}|{fmt24(x['increment_flat_rate'])}|[{ci[0]*100:+.3f},{ci[1]*100:+.3f}]|")
dd=dd_summary24;ci=dd['familyCI48'];lines24+=['',f"同增加5pp的增量差：(80→85−固定80)−(90→95−固定90)＝{dd['mean_difference_of_increments']*100:+.4f}pp，48區塊14比較校正區間[{ci[0]*100:+.4f},{ci[1]*100:+.4f}]pp。差異很小且跨零，未證實20%現金底子比10%更有利於此5pp加碼。這不是正式等效性證明。",'80→95的平均增量約為80→85的三倍左右，符合增加資金幅度也約三倍的方向；不能把它解讀成訊號準確度提高三倍。','', '## 等訊號加碼，是否優於直接投入同一上限？','|動態方法減固定上限|平均差pp|勝過固定上限比例|48區塊家族95%區間pp|','|---|---:|---:|---|']
for a,b in [('D80_85','F85'),('D80_95','F95'),('D90_95','F95')]:
    x=comparisons24[a+'_minus_'+b]['main'];ci=x['familyCI48'];lines24.append(f"|{labels24[a]}減{labels24[b]}|{x['mean_excess']*100:+.3f}|{fmt24(x['increment_positive_win_rate'])}|[{ci[0]*100:+.3f},{ci[1]*100:+.3f}]|")
lines24+=['','## 100%／0%與用盡現金的實驗','名目100%目標是以全部前日淨值換成股票，再加費用，沒有使用當日未知均價事後縮量。整張取整可能留下餘額，因此它不是每日現金精確為0%。負現金是不可接受的執行結果，不會當成借款後合法賺錢。','100%基礎沒有刻意預留的10／20pp加碼資金；若日後出現整張或價格漂移餘額，再投入那種餘額是另一規則，本輪沒有把100→100說成已測的新增加碼策略。','|名目目標|650窗口中資料完整數|其中合法有效數|所有窗口曾負現金數|資料完整中曾負現金數|原F90有效日期覆蓋|','|---|---:|---:|---:|---:|---:|']
for k in full24:
    c=coverage24[k];lines24.append(f"|{labels24[k]}|{c['data_complete_n']}|{c['valid_n']}|{c['negative_cash_windows']}|{c['negative_cash_in_data_complete']}|{fmt24(c['coverage_vs_F90'])}|")
lines24+=['100%三方法已跑，不是漏掉。但有效窗口被現金可行性篩選，不可把其個別有效樣本的勝率直接與499窗口表比較。特別是固定100%在211可行窗口的79.62%勝率／8.29%均值，只是被選出的子樣本，不能拿來宣稱優於80／90%。這兩數只為揭示偏誤，不作策略推薦。','資料完整但違規的數值只在JSON以UNSAFE欄保留供模型診斷；那是假想未阻擋負現金的會計結果，非正式可行策略，沒有提交交易。固定100%與加碼至100%的合法自己配對、分母及費用另存，不能隱藏不可行窗口。','', '## 覆蓋與現金上限','|方法|有效n|不可判定n|違規窗口|曾現金>=25%窗口|原F90有效日期覆蓋|','|---|---:|---:|---:|---:|---:|']
for k,c in coverage24.items():lines24.append(f"|{labels24[k]}|{c['valid_n']}|{c['invalid_n']}|{c['violation_windows']}|{c['cash25_or_more_windows']}|{fmt24(c['coverage_vs_F90'])}|")
lines24+=['固定80%也有現金上限問題：股價下跌時現金占比被動增加，650窗口中29個曾達25%以上。本輪沒有額外每日強迫買回以維持比例；資料不齊與違規可重疊，不能相加。這說明保留20%目標現金並不自動保證所有日子合規。','目標25檔與整張後持股不足的缺口沿用v23記錄，不把缺漏填0、不換資料源補造價格。Active Share與正式DataPack尚未完整驗證，本文「有效」只指現有模擬檢核通過，不等於官賽完整認證。','', '## 統計限制與可用結論','13配對加同5pp增量差，共14比較，採24／48位置區塊bootstrap各5000次及Bonferroni。本輪主要加碼增量區間皆跨零；不因全期均值高就選策略。全部499窗口互相重疊，且過往已看過，訓練／後續分段僅為回溯敏感度，不是真正未見樣本外。','可以說：此加碼規則沒有顯示更高整體勝率，保留20%也沒有讓同5pp加碼更準。增加曝險在這份正報酬樣本放大均值，同時放大虧損；實際要投入多少仍是風險預算問題。不能說：現金比例與訊號時機永遠無關、固定95一定最好、100%勝率最高就直接採用。','目前150檔存在選擇與存活偏誤、未知公司行動與較短歷史；各方法實際曝險、加碼分配、成本及取整仍不同，對照可檢查但不是完整因果識別。無LLM判讀、無新爬取、無交易提交，排程未啟動。','', '## 保存與驗證','Notebook cells296–298。沿用kernel PID2524，未重啟。F90及D90→95共10個有效抽樣控制核對後引用；8新增共5200窗口。所有有效窗口費後期望值與條件損益、增量恆等式、平手／正負差分比例及現金限制核對。圖保留499×7全部報酬點、三組499×3加碼增量點、中位數及0線；只做程式版面檢查，未做瀏覽器目視QA。']
report24='\n'.join(lines24);rp24=recovery_root/'動能_現金比例與加碼來源研究_v24_2026-10-15.md';rp24.write_text(report24,encoding='utf-8')
import plotly.graph_objects as go
rng24=np.random.default_rng(24015);figret24=go.Figure();figinc24=go.Figure();ids24=np.flatnonzero(common24);days24=[str(dates[int(window_starts_v15[i])]) for i in ids24]
for pos,key in enumerate(core24):
    z=ret24[key][common24]*100
    figret24.add_trace(go.Box(x=np.full(len(z),pos),y=z,name=labels24[key],boxpoints=False,line={'color':'#4e77a8'},fillcolor='rgba(78,119,168,.15)',hoverinfo='skip',showlegend=False))
    figret24.add_trace(go.Scatter(x=pos+rng24.uniform(-.18,.18,len(z)),y=z,mode='markers',marker={'size':4,'opacity':.4},text=days24,hovertemplate=labels24[key]+'<br>%{text}<br>費後24日報酬 %{y:.2f}%<extra></extra>',showlegend=False))
    med=float(np.median(z));figret24.add_trace(go.Scatter(x=[pos-.2,pos+.2],y=[med,med],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
for pos,(a,b) in enumerate([('D80_85','F80'),('D80_95','F80'),('D90_95','F90')]):
    z=(ret24[a]-ret24[b])[common24]*100;x=comparisons24[a+'_minus_'+b]['main'];ci=x['familyCI48'];mean=x['mean_excess']*100
    figinc24.add_trace(go.Scatter(x=pos+rng24.uniform(-.18,.18,len(z)),y=z,mode='markers',marker={'size':5,'opacity':.4},text=days24,hovertemplate=labels24[a]+'減'+labels24[b]+'<br>%{text}<br>費後增量 %{y:.3f} pp<extra></extra>',showlegend=False))
    figinc24.add_trace(go.Scatter(x=[pos],y=[mean],mode='markers',marker={'size':11,'color':'#dc2477'},error_y={'type':'data','array':[ci[1]*100-mean],'arrayminus':[mean-ci[0]*100]},hovertemplate='平均增量及48區塊家族區間 %{y:.3f} pp<extra></extra>',showlegend=False))
figret24.add_hline(y=0,line_color='#222');figinc24.add_hline(y=0,line_color='#222')
figret24.update_layout(template='plotly_white',height=570,title='499共同窗口：固定投入與條件加碼，全部報酬點及桃紅中位數',xaxis={'tickmode':'array','tickvals':list(range(7)),'ticktext':[labels24[k] for k in core24],'tickangle':-20},yaxis_title='費後24日報酬 %')
figinc24.update_layout(template='plotly_white',height=500,title='加碼相對不加碼的增量：全部點及均值／家族校正區間',xaxis={'tickmode':'array','tickvals':[0,1,2],'ticktext':['80→85 減固定80','80→95 減固定80','90→95 減固定90']},yaxis_title='加碼增量 百分點')
hp24=recovery_root/'動能_80與90與100投入及加碼比較_v24_2026-10-15.html';hp24.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>現金比例與加碼歸因 v24</title><style>body{font:16px/1.7 system-ui;max-width:1400px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#eef3fa;font-weight:bold}p{overflow-wrap:anywhere}.note{background:#fff4db;padding:18px;border-radius:8px}</style><h1>原五日策略：80%／90%／100%與加碼歸因</h1><p class="note">固定80／85／90／95在同499窗口的勝率恰好相同。加碼增量勝率與策略勝率要分開：同加5pp的80→85、90→95，觸發窗口增量勝率同51.61%。100%三組另列，固定100%有411／650窗口曾負現金，不能只取合法子樣本宣稱最好。</p>'+figret24.to_html(full_html=False,include_plotlyjs=True)+figinc24.to_html(full_html=False,include_plotlyjs=False)+render_report17(report24)+'</html>',encoding='utf-8')
for key,m in summary24['core_metrics'].items():
    x=ret24[key][common24];assert np.isclose(x.mean(),np.mean(x>0)*(m['mean_win'] or 0)+np.mean(x<0)*(m['mean_loss'] or 0))
for s in comparisons24.values():
    x=s['main']
    if x['n']:
        assert np.isclose(x['mean_excess'],x['candidate']['expectancy']-x['base']['expectancy'])
        assert np.isclose(x['increment_positive_win_rate']+x['increment_negative_rate']+x['increment_flat_rate'],1.)
for key,rows in results24.items():
    for i in np.flatnonzero(valid24[key]):assert rows[i]['min_cash']>=0 and rows[i]['max_cash']<.25 and rows[i]['violations']==0
assert np.isclose(dd_summary24['mean_difference_of_increments'],comparisons24['D80_85_minus_F80']['main']['mean_excess']-comparisons24['D90_95_minus_F90']['main']['mean_excess'])
assert not np.any(train_v18&test_v18);assert sum(len(t.y) for t in figret24.data if t.type=='scatter' and t.mode=='markers')==499*7;assert sum(len(t.y) for t in figinc24.data if t.type=='scatter' and len(t.y)>1)==499*3
summary24['verification']={'reuse_controls':10,'new_windows':5200,'core499_not100selected118':True,'return_plot_points':3493,'increment_plot_points':1497,'increment_accounting_and_sign_rates':True,'valid_cash_limits_checked':True,'literal100violations_retained':True,'kernel_restarted':False}
(recovery_root/'動能_現金比例與加碼歸因統計_v24_2026-10-15.json').write_text(json.dumps(summary24,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
state24=json.loads(sp24.read_text(encoding='utf-8'));state24['cash_add_v24'].update({'status':'completed','running':False,'core_common_n':499,'all10_common_n':118,'new_methods':8,'new_windows':5200,'timing_advantage_confirmed':False,'100percent_exposure_not_general_feasible':True,'outputs':[rp24.name,hp24.name,'動能_現金比例與加碼歸因統計_v24_2026-10-15.json'],'notebook_cells':[296,298]});sp24.write_text(json.dumps(state24,ensure_ascii=False,indent=2),encoding='utf-8')
wf24=recovery_root/'A策略_無人值守研究流程.md';wf24.write_text(wf24.read_text(encoding='utf-8')+'\n\n## v24 現金比例與加碼歸因（完成）\n固定80／85／90／95／100、80→85／95／100、90→95／100，共10方法；8新5200窗口，2方法直接引用。七保留緩衝方法同499窗口，100%另列避免118子樣本偏差。同+5pp觸發增量勝率同51.61%，未證實多預留現金讓訊號更準；固定100%有411／650負現金。成果：'+hp24.name+'。未擴增新參數或排程。\n',encoding='utf-8')
print('V24_COMPLETE',{'report':rp24.name,'html':hp24.name,'core_common':499,'100negative_windows':411,'same5_triggered_win_rate':comparisons24['D80_85_minus_F80']['main']['triggered_increment_win_rate'],'verification':summary24['verification']})


In [ ]:
# v25 test whether add gain is primarily exposure rather than timing; freeze before new matched controls.
print('V25_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'shape':shape,'v24_core_n':int(common24.sum())})
dynamics25=['D80_85','D80_95','D90_95'];base25={'D80_85':'F80','D80_95':'F80','D90_95':'F90'}
def exposure_array25(key,rows):
    return np.array([baseline_controls23.get(i,{}).get('mean_exposure',np.nan) if key=='F90' else (r.get('mean_exposure') if r.get('mean_exposure') is not None else np.nan) for i,r in enumerate(rows)],dtype=float)
exposure_old25={k:exposure_array25(k,results24[k]) for k in core24};train_calibration25=common24&train_v18
e80_train25=float(exposure_old25['F80'][train_calibration25].mean());e95_train25=float(exposure_old25['F95'][train_calibration25].mean())
matched_specs25={}
for key in dynamics25:
    desired=float(exposure_old25[key][train_calibration25].mean());target=.8+(desired-e80_train25)/(e95_train25-e80_train25)*.15;target=round(target,4)
    assert .8<=target<=.95
    matched_specs25[key]={'fixed_key':'MATCH_'+key,'fixed_target':target,'dynamic_train_actual_exposure':desired,'train_n':int(train_calibration25.sum()),'calibration_only_exposure_not_return':True}
definition25={'hypothesis':'addedEVmaybeidlecashdecreaseandmorestockexposure,notbettertiming','strategy':'originalA,25target,5dayregular,24sessions,lotsfeesallv24unchanged;noexit','new_controls':matched_specs25,'fixed_target_calibration':'useonlyv24commontrainingwindowsend<=2024-12-31;linearactualexposuremapF80/F95;matchdynamictrainingmeanexposure;roundtarget1bp,freeze;noadjustafterlaterresults','primary':'3dynamicminusmatchedfixed,24/48block5000Bonferroni3;common10coreincl3new+old7;laterstart>=2025-01-01;oldhistoryalreadyseen','secondary':'all-common-windowexpostaccounting3parts: exposureeffect via staticgross-return/exposure interpolation + timing/allocation/modelresidual - feeincrease;identityperwindow;descriptive95intervalsnotcausalpercentages','accounting_warning':'usesactualfutureaverageexposure&realizedwindowreturns;nottradablesignalorforecast;feeaddbackfixedordersnotzerofeecounterfactual;residualincludesallocation,timing,drift,rounding,financingzero','not_added':['newcashgrid','newstops','newaddsignal','100percentleverage'], 'new_method_count':3,'new_windows':1950,'snapshot':snapshot_sha_v15}
hash25=hashlib.sha256(json.dumps(definition25,sort_keys=True).encode()).hexdigest();(recovery_root/'動能_加碼增益曝險匹配實驗定義_v25_2026-10-15.json').write_text(json.dumps(definition25,ensure_ascii=False,indent=2),encoding='utf-8')
for key in ['F80','F95','D90_95']:
    spec=specs24[key];ix=np.flatnonzero(valid24[key])[100];r=simulate_cash24(int(window_starts_v15[ix]),*spec);old=results24[key][int(ix)]
    assert all(np.isclose(r[f],old[f],rtol=0,atol=1e-12) for f in ['return','cost','turnover','mdd'])
sp25=recovery_root/'A策略_無人值守研究狀態.json';state25=json.loads(sp25.read_text(encoding='utf-8'));state25['matched_exposure_v25']={'status':'defined_running','running':True,'hash':hash25,'new_methods':3,'matched_specs':matched_specs25};sp25.write_text(json.dumps(state25,ensure_ascii=False,indent=2),encoding='utf-8')
print('V25_FROZEN',json.dumps(matched_specs25,ensure_ascii=False),hash25,flush=True)


In [ ]:
# v25 run only new matched-exposure controls; reuse existing7 core methods.
results25={k:results24[k] for k in core24};reuse25={k:'v24_exact_reference' for k in core24}
for dyn,spec in matched_specs25.items():
    key=spec['fixed_key'];p=spec['fixed_target'];results25[key]=[simulate_cash24(int(start),p,p,False) for start in window_starts_v15]
    print('V25_DONE',key,p,'valid',sum(not r['invalid'] and r['violations']==0 for r in results25[key]),flush=True)
valid25={k:np.array([not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results25.items()};ret25={k:np.array([r['return'] for r in rows]) for k,rows in results25.items()};cost25={k:np.array([r['cost'] for r in rows]) for k,rows in results25.items()};exp25={k:exposure_array25(k,rows) for k,rows in results25.items()};common25=np.logical_and.reduce(list(valid25.values()))
def metrics25(key,mask):
    ids=np.flatnonzero(mask);x=ret25[key][mask];rows=results25[key]
    return {'n':len(ids),'win_rate':float(np.mean(x>0)),'expectancy':float(x.mean()),'mean_win':float(x[x>0].mean()) if np.any(x>0) else None,'mean_loss':float(x[x<0].mean()) if np.any(x<0) else None,'mean_cost':float(cost25[key][mask].mean()),'mean_exposure':float(exp25[key][mask].mean()),'P10':float(np.quantile(x,.1)),'worst_mdd':float(min(rows[i]['mdd'] for i in ids))}
def intervals25(x,mask,adjust=1):
    out={}
    for b,inds in block_indices_v15.items():
        z=np.where(mask,x,np.nan)[inds];n=np.isfinite(z).sum(axis=1);v=np.nansum(z,axis=1)[n>0]/n[n>0]
        out[f'CI{b}']=np.quantile(v,[.025,.975]).tolist();out[f'adjustedCI{b}']=np.quantile(v,[.05/(adjust*2),1-.05/(adjust*2)]).tolist()
    return out
matched_stats25={}
for dyn,spec in matched_specs25.items():
    fixed=spec['fixed_key'];d=ret25[dyn]-ret25[fixed];pair=valid25[dyn]&valid25[fixed];row={'fixed_target':spec['fixed_target'],'own_pair_n':int(pair.sum()),'coverage_vs_dynamic':float(pair.sum()/valid25[dyn].sum())}
    for scope,mask in [('all',common25),('train',common25&train_v18),('later',common25&test_v18)]:
        row[scope]={'n':int(mask.sum()),'dynamic':metrics25(dyn,mask),'matched_fixed':metrics25(fixed,mask),'mean_excess':float(d[mask].mean()),'beat_matched_rate':float(np.mean(d[mask]>1e-12)),'mean_exposure_gap':float((exp25[dyn]-exp25[fixed])[mask].mean()),**intervals25(d,mask,3)}
    matched_stats25[dyn]=row
gross80_25=ret25['F80']+cost25['F80'];gross95_25=ret25['F95']+cost25['F95'];span25=exp25['F95']-exp25['F80'];slope25=np.divide(gross95_25-gross80_25,span25,out=np.full(650,np.nan),where=np.isfinite(span25)&(span25>0))
decomp25={};part_arrays25={}
for dyn in dynamics25:
    base=base25[dyn];total=ret25[dyn]-ret25[base]
    exposure_effect=(exp25[dyn]-exp25[base])*slope25
    fee_effect=-(cost25[dyn]-cost25[base])
    residual=total-exposure_effect-fee_effect
    assert np.allclose(total[common25],(exposure_effect+fee_effect+residual)[common25],atol=1e-12,rtol=0)
    part_arrays25[dyn]={'total':total,'exposure':exposure_effect,'fee':fee_effect,'residual':residual}
    row={'n':int(common25.sum()),'base':base,'extrapolated_windows':int(np.sum(common25&((exp25[dyn]<exp25['F80'])|(exp25[dyn]>exp25['F95'])))),'warning':'expostmodelaccountingnotcausalalpha;allcomponentCIsdescriptiveunadjusted'}
    for scope,mask in [('all',common25),('train',common25&train_v18),('later',common25&test_v18)]:row[scope]={part:{'mean':float(arr[mask].mean()),**intervals25(arr,mask,1)} for part,arr in part_arrays25[dyn].items()}
    decomp25[dyn]=row
static_errors25={}
for key in ['F85','F90']:
    pred=gross80_25+(exp25[key]-exp25['F80'])*slope25;error=(ret25[key]+cost25[key])-pred
    static_errors25[key]={'mean_error':float(error[common25].mean()),'mean_absolute_error':float(np.abs(error[common25]).mean()),'max_abs_error':float(np.max(np.abs(error[common25])))}
summary25={'definition':definition25,'hash':hash25,'matched_specs':matched_specs25,'common_n':int(common25.sum()),'train_n':int((common25&train_v18).sum()),'later_n':int((common25&test_v18).sum()),'purged_n':int((common25&purged_v18).sum()),'common_indices':np.flatnonzero(common25).tolist(),'matched_controls':matched_stats25,'accounting_decomposition':decomp25,'static_interpolation_check':static_errors25,'metrics':{k:metrics25(k,common25) for k in results25},'valid_counts':{k:int(mask.sum()) for k,mask in valid25.items()},'limits':['exposurematchedintrainingnotexacteachtestday','expostdecompnottradable','residualnottimingalone','previouslyseenhistory','overlapwindows','current150selectionbias','notcompleteofficialrules']}
(recovery_root/'動能_加碼增益曝險匹配逐窗口_v25_2026-10-15.json').write_text(json.dumps({'definition':definition25,'new_fixed_results':{spec['fixed_key']:results25[spec['fixed_key']] for spec in matched_specs25.values()},'reused_methods':reuse25,'decomposition_arrays':part_arrays25},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
(recovery_root/'動能_加碼增益曝險匹配統計_v25_2026-10-15.json').write_text(json.dumps(summary25,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V25_STATS',json.dumps({'common':summary25['common_n'],'train':summary25['train_n'],'later':summary25['later_n'],'matched':matched_stats25,'parts':{k:{p:v['mean'] for p,v in s['all'].items()} for k,s in decomp25.items()},'linearity':static_errors25},ensure_ascii=False,default=serial_family_v15),flush=True)


In [ ]:
# v25 actual-strategy accounting identity is primary; ex-post interpolation remains appendix.
labels25={k:labels24[k] for k in core24}
for dyn,spec in matched_specs25.items():labels25[spec['fixed_key']]=f"固定{spec['fixed_target']*100:.2f}%投入對照"
control_identity25={}
for dyn,spec in matched_specs25.items():
    fixed=spec['fixed_key'];base=base25[dyn];total=ret25[dyn]-ret25[base];stock_control=ret25[fixed]-ret25[base];remaining=ret25[dyn]-ret25[fixed]
    assert np.allclose(total[common25],(stock_control+remaining)[common25],rtol=0,atol=1e-12)
    control_identity25[dyn]={'base':base,'fixed':fixed,'all':{},'train':{},'later':{},'warning':'differenceidentityoftworealtestedstrategies;matchedmeanexposurenotexacteverydate;remainingisnotpuretimingalpha'}
    for scope,mask in [('all',common25),('train',common25&train_v18),('later',common25&test_v18)]:
        control_identity25[dyn][scope]={'total_add_increment':float(total[mask].mean()),'matched_fixed_exposure_increment':float(stock_control[mask].mean()),'dynamic_minus_matched_fixed':float(remaining[mask].mean()),'additional_fee_vs_matched':float((cost25[dyn]-cost25[fixed])[mask].mean()),'fixed_order_fee_addback_difference':float((remaining+cost25[dyn]-cost25[fixed])[mask].mean())}
summary25['actual_control_identity']=control_identity25
fmt25=lambda v:'—' if v is None else f'{v*100:.2f}%'
lines25=['# 加碼期望值增益：平均曝險匹配對照 v25','', '## 直接回答','本輪結果支持「加碼的小幅平均增益主要可由較多股票投入解釋」這個方向，尚未證實加碼時機額外提高費後期望值。平均曝險接近時，不看訊號的固定投入對照得到相近、略高的淨平均報酬；三個差值的校正區間都跨零，不能證明兩者完全等效或所有增益100%由曝險造成。','重要區別：未證實提高報酬，不代表加碼策略所有風險特性都相同。部分動態版本的最差回撤較小，P10則沒有一致改善，因此仍有報酬／風險取捨。','', '## 如何設計，避免拿95%去比平均只有91%的版本','完整沿用原20／60日各半選股、目標25檔、五日例行調整、24交易日、整千股及相同費稅。沒有改加碼條件、沒有新增退出策略或找新現金網格。','先只用2024年底前已結束的179個訓練窗口，計算各加碼版本的實際平均股票曝險；用同訓練資料的固定80與95實際曝險建立線性對照，求得固定投入目標並四捨五入至1bp。只匹配曝險，不看哪個目標報酬較高；在三個新對照跑之前保存參數，此後不依後續資料調整。','|動態加碼|固定投入對照目標|訓練動態實際平均曝險|訓練固定實際平均曝險|','|---|---:|---:|---:|']
for dyn,spec in matched_specs25.items():
    s=matched_stats25[dyn]['train'];lines25.append(f"|{labels25[dyn]}|{fmt25(spec['fixed_target'])}|{fmt25(s['dynamic']['mean_exposure'])}|{fmt25(s['matched_fixed']['mean_exposure'])}|")
lines25+=['目標投入與實際平均曝險不同，因為價格、費用及整張造成漂移。匹配只在訓練平均進行，沒有用每個測試窗口的未來報酬／曝險來設定交易；後續各日曝險不完全相同，差距明列。','三個新固定對照共1950窗口；七個既有方法直接引用，沒有重跑已驗證的80／90／95策略。','', '## 同499窗口的直接對照','|動態加碼|動態策略勝率|動態期望值|固定對照勝率|固定對照期望值|動態實際曝險|固定實際曝險|動態減固定pp|','|---|---:|---:|---:|---:|---:|---:|---:|']
for dyn,row in matched_stats25.items():
    s=row['all'];a=s['dynamic'];b=s['matched_fixed'];lines25.append(f"|{labels25[dyn]}|{fmt25(a['win_rate'])}|{fmt25(a['expectancy'])}|{fmt25(b['win_rate'])}|{fmt25(b['expectancy'])}|{fmt25(a['mean_exposure'])}|{fmt25(b['mean_exposure'])}|{s['mean_excess']*100:+.3f}|")
lines25+=['','## 期望值差可以如何拆？','以下是一個逐窗口成立的對照差分恆等式，不是宣稱已識別因果貢獻比例：','動態加碼−原基礎投入＝（匹配曝險的固定投入−原基礎投入）＋（動態加碼−匹配曝險的固定投入）。','|加碼版本|固定提高投入對照的增益pp|動態相對固定的剩餘差pp|實際加碼總增益pp|','|---|---:|---:|---:|']
for dyn,c in control_identity25.items():
    s=c['all'];lines25.append(f"|{labels25[dyn]}|{s['matched_fixed_exposure_increment']*100:+.3f}|{s['dynamic_minus_matched_fixed']*100:+.3f}|{s['total_add_increment']*100:+.3f}|")
lines25+=['例如90→95：不看加碼訊號、固定約92.14%投入，相對固定90的增益約+0.108pp；實際動態加碼比這個對照少0.023pp，所以淨增益剩約+0.085pp。可以用提高股票投入重現相近增益，暫無需把這段增益歸功於更準確的加碼訊號。','剩餘差仍包含加碼時點、加到哪些持股、交易費、曝險漂移與整張效應，不是純粹的擇時alpha。沒有計算「多少百分比報酬來自現金」這種因果比例。','', '## 費用與風險也要一起看','|加碼版本|動態費稅|固定對照費稅|動態P10|固定P10|動態最差回撤|固定最差回撤|','|---|---:|---:|---:|---:|---:|---:|']
for dyn,row in matched_stats25.items():
    a=row['all']['dynamic'];b=row['all']['matched_fixed'];lines25.append(f"|{labels25[dyn]}|{fmt25(a['mean_cost'])}|{fmt25(b['mean_cost'])}|{fmt25(a['P10'])}|{fmt25(b['P10'])}|{fmt25(a['worst_mdd'])}|{fmt25(b['worst_mdd'])}|")
lines25+=['加碼的額外費用會吃掉部分增益。單純把費用加回只是在固定既有交易下作會計分解，不等於零費率重新模擬，因費用也會影響後續可投入股數。','', '## 時間分段與信賴區間','三個動態減固定的主要對照採24／48日位置區塊bootstrap各5000次、三比較Bonferroni校正。區間為平均差的不確定性，不是下次報酬範圍；歷史重疊窗口不能當獨立樣本。','|動態加碼|全499平均差pp|全期48區塊校正95%區間pp|後續298窗口平均差pp|後續實際曝險差pp|','|---|---:|---|---:|---:|']
for dyn,row in matched_stats25.items():
    a=row['all'];ci=a['adjustedCI48'];t=row['later'];lines25.append(f"|{labels25[dyn]}|{a['mean_excess']*100:+.3f}|[{ci[0]*100:+.3f},{ci[1]*100:+.3f}]|{t['mean_excess']*100:+.3f}|{t['mean_exposure_gap']*100:+.3f}|")
lines25+=['共同499分為訓練179、後續298、跨年淨空22。2025起始才算後續，跨界不混入訓練；但這些資料前輪已看過，不能宣稱全新未見樣本外。匹配控制後續曝險差約0.23～0.77pp，尚非每天完全相同曝險。','', '## 附錄：事後會計模型，不作交易策略','另以同一窗口固定80／95的「既有淨報酬加回既有費用」，對實際平均曝險作線性內插。動態相對原基礎的差分可逐窗口寫成：模型曝險效果＋時點／分配／漂移／取整殘差−增加費用。','這個附錄使用事後實現報酬與整段平均曝險，不能在當時拿來交易或預測，也不是精確的曝險因果效果。固定85／90對此內插的平均絕對誤差約0.038pp，顯示仍有模型誤差；不拿附錄估計值算歸因百分比。','|版本|模型曝險效果pp|殘差pp|增加費用效果pp|實際淨增益pp|','|---|---:|---:|---:|---:|']
for dyn,s in decomp25.items():
    a=s['all'];lines25.append(f"|{labels25[dyn]}|{a['exposure']['mean']*100:+.3f}|{a['residual']['mean']*100:+.3f}|{a['fee']['mean']*100:+.3f}|{a['total']['mean']*100:+.3f}|")
lines25+=['主要判斷以真的跑過、訓練鎖定的固定曝險對照為主；附錄只輔助解釋，不是另一份獨立證據。','', '## 可執行性與限制','|新增固定對照|有效窗口數|對應加碼可行日期覆蓋|','|---|---:|---:|']
for dyn,spec in matched_specs25.items():lines25.append(f"|{labels25[spec['fixed_key']]}|{summary25['valid_counts'][spec['fixed_key']]}|{fmt25(matched_stats25[dyn]['coverage_vs_dynamic'])}|")
lines25+=['不把不可執行或資料缺漏填零；沿用前輪所有現金、持股數及單股上限檢核。100%負現金問題已於v24列出，本輪不重跑或用借款處理。Active Share與正式DataPack尚未完整驗證，「有效」不是官賽全面認證。','目前150檔的存活／選擇偏誤、未知公司行動、較短歷史、過往已被觀察及跨輪試驗偏誤都仍存在。三比較校正不消除全部歷史研究選擇。模型現金利息為0，沒有實際交易、LLM判讀或新爬取；未啟動排程。','', '## 保存與驗證','原Notebook cells299–301；PID2524保留，未重啟。三個既有方法抽樣核對、三新1950窗口，七舊方法直接引用。訓練只用曝險、匹配參數先保存後跑；逐窗口動態差分恆等式、事後模型三部分恆等式、條件損益與期望值、平均曝險／費用、共同日期及每日現金限制程式核對。圖保留499×6報酬點、499×3差值點與0線、桃紅中位數／均值，未做瀏覽器目視QA。']
report25='\n'.join(lines25);rp25=recovery_root/'動能_加碼增益是否只是曝險研究_v25_2026-10-15.md';rp25.write_text(report25,encoding='utf-8')
import plotly.graph_objects as go
rng25=np.random.default_rng(25015);fg25=go.Figure();fd25=go.Figure();ids25=np.flatnonzero(common25);days25=[str(dates[int(window_starts_v15[i])]) for i in ids25];display25=[]
for dyn,spec in matched_specs25.items():display25.extend([dyn,spec['fixed_key']])
for pos,key in enumerate(display25):
    y=ret25[key][common25]*100
    fg25.add_trace(go.Box(x=np.full(len(y),pos),y=y,name=labels25[key],boxpoints=False,line={'color':'#4e77a8'},fillcolor='rgba(78,119,168,.15)',hoverinfo='skip',showlegend=False))
    fg25.add_trace(go.Scatter(x=pos+rng25.uniform(-.17,.17,len(y)),y=y,mode='markers',marker={'size':4,'opacity':.4},text=days25,hovertemplate=labels25[key]+'<br>%{text}<br>費後24日報酬 %{y:.2f}%<extra></extra>',showlegend=False))
    med=float(np.median(y));fg25.add_trace(go.Scatter(x=[pos-.2,pos+.2],y=[med,med],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
for pos,(dyn,spec) in enumerate(matched_specs25.items()):
    fixed=spec['fixed_key'];y=(ret25[dyn]-ret25[fixed])[common25]*100;a=matched_stats25[dyn]['all'];ci=a['adjustedCI48'];mean=a['mean_excess']*100
    fd25.add_trace(go.Scatter(x=pos+rng25.uniform(-.17,.17,len(y)),y=y,mode='markers',marker={'size':5,'opacity':.4},text=days25,hovertemplate=labels25[dyn]+'減匹配固定對照<br>%{text}<br>淨差 %{y:.3f} pp<extra></extra>',showlegend=False))
    fd25.add_trace(go.Scatter(x=[pos],y=[mean],mode='markers',marker={'size':11,'color':'#dc2477'},error_y={'type':'data','array':[ci[1]*100-mean],'arrayminus':[mean-ci[0]*100]},hovertemplate='平均淨差及48區塊三比較校正區間 %{y:.3f} pp<extra></extra>',showlegend=False))
fg25.add_hline(y=0,line_color='#222');fd25.add_hline(y=0,line_color='#222')
fg25.update_layout(template='plotly_white',height=570,title='平均股票曝險相近：動態加碼與固定投入，全部499窗口',xaxis={'tickmode':'array','tickvals':list(range(6)),'ticktext':[labels25[k] for k in display25],'tickangle':-15},yaxis_title='費後24日報酬 %')
fd25.update_layout(template='plotly_white',height=500,title='動態加碼減匹配固定對照：全點、均值及校正區間',xaxis={'tickmode':'array','tickvals':[0,1,2],'ticktext':[labels25[k] for k in dynamics25]},yaxis_title='平均差 百分點')
hp25=recovery_root/'動能_加碼曝險匹配與來源拆解_v25_2026-10-15.html';hp25.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>加碼增益是曝險嗎 v25</title><style>body{font:16px/1.7 system-ui;max-width:1400px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#eef3fa;font-weight:bold}p{overflow-wrap:anywhere}.note{background:#eef4ff;padding:18px;border-radius:8px}</style><h1>加碼增益：少留現金，還是時機比較準？</h1><p class="note">三個固定投入對照只用訓練期曝險校準，先鎖參數再跑。平均曝險相近後，動態加碼未顯示額外費後期望值優勢；這支持曝險解釋，並非證明100%因果歸因。499共同窗口，後續298另列。</p>'+fg25.to_html(full_html=False,include_plotlyjs=True)+fd25.to_html(full_html=False,include_plotlyjs=False)+render_report17(report25)+'</html>',encoding='utf-8')
for dyn,spec in matched_specs25.items():
    total=ret25[dyn]-ret25[base25[dyn]];control=ret25[spec['fixed_key']]-ret25[base25[dyn]];rest=ret25[dyn]-ret25[spec['fixed_key']]
    assert np.allclose(total[common25],(control+rest)[common25],rtol=0,atol=1e-12)
    for k in [dyn,spec['fixed_key']]:
        m=metrics25(k,common25);x=ret25[k][common25];assert np.isclose(m['expectancy'],np.mean(x>0)*(m['mean_win'] or 0)+np.mean(x<0)*(m['mean_loss'] or 0))
for key,rows in results25.items():
    for i in np.flatnonzero(valid25[key]):assert rows[i]['min_cash']>=0 and rows[i]['max_cash']<.25 and rows[i]['violations']==0
assert sum(len(t.y) for t in fg25.data if t.type=='scatter' and t.mode=='markers')==499*6
assert sum(len(t.y) for t in fd25.data if t.type=='scatter' and len(t.y)>1)==499*3
summary25['verification']={'new_windows':1950,'retained_methods':7,'training_exposure_only_params_saved_first':True,'new_control_return_points':2994,'paired_points':1497,'two_actual_strategy_identity_all499x3':True,'expost_model_identity_all499x3':True,'valid_cash_limits_checked':True,'kernel_restarted':False}
(recovery_root/'動能_加碼增益曝險匹配統計_v25_2026-10-15.json').write_text(json.dumps(summary25,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
state25=json.loads(sp25.read_text(encoding='utf-8'));state25['matched_exposure_v25'].update({'status':'completed','running':False,'common_n':499,'train_n':179,'later_n':298,'new_windows':1950,'timing_advantage_confirmed':False,'causal_all_gain_from_exposure_proven':False,'outputs':[rp25.name,hp25.name,'動能_加碼增益曝險匹配統計_v25_2026-10-15.json'],'notebook_cells':[299,301]});sp25.write_text(json.dumps(state25,ensure_ascii=False,indent=2),encoding='utf-8')
wf25=recovery_root/'A策略_無人值守研究流程.md';wf25.write_text(wf25.read_text(encoding='utf-8')+'\n\n## v25 加碼增益的曝險匹配對照（完成）\n三個新固定投入對照，只用訓練179窗口曝險校準為82.15／86.47／92.14%，鎖定後跑1950窗口，七舊方法引用。共同499、後續298。平均曝險接近後，動態未顯示額外費後期望值優勢；不宣稱完全等效或100%因果歸因。成果：'+hp25.name+'。未擴充訊號或排程。\n',encoding='utf-8')
print('V25_COMPLETE',{'html':hp25.name,'report':rp25.name,'control_accounting':{k:v['all'] for k,v in control_identity25.items()},'verification':summary25['verification']})


In [ ]:
# v26 inspect retained historical coverage and engine only; no fetch/reset.
print('V26_CONTEXT',{'pid':__import__('os').getpid(),'python':__import__('sys').executable,'hist_tickers':len(provider_history_v13['prices']),'case_arrays':list(case_arrays_v14),'calendar_n':len(calendar_days_v13),'calendar_first':str(calendar_days_v13[0]),'calendar_last':str(calendar_days_v13[-1])})
print('V26_HISTORY_COVERAGE',[{'sid':sid,'rows':len(provider_history_v13['prices'][sid]),'first':min(r['date'] for r in provider_history_v13['prices'][sid]),'last':max(r['date'] for r in provider_history_v13['prices'][sid])} for sid in ['2330','1101','6488'] if sid in provider_history_v13['prices']])
print('V26_SAVED_FUNCTIONS',[(k,type(v).__name__) for k,v in list(globals().items()) if k.startswith('simulate_external') or k.startswith('build_case')])
print('V26_V14_ENGINE',engine_source_v14 if 'engine_source_v14' in globals() else 'name_not_present')
print('V26_CASE_SCHEMA',list(next(iter(case_arrays_v14.values()))))


In [ ]:
# v26 inspect cached builder, not raw data or all history.
node26=next(n for n in ast.parse(saved_engine_v14).body if isinstance(n,ast.FunctionDef) and n.name=='build_case_v13')
print('V26_BUILDER',ast.unparse(node26))
print('V26_ACTION_CONTEXT',{'cash_records':len(cash_records_v13),'bad_action_dates':len(bad_action_dates_v13),'paid_known_days':len(paid_known_days_v14),'pool':len(full_ids_v13),'external_order':len(all_external_ids_v13)})
provider_dates26=[{datetime.date.fromisoformat(r['date']) for r in provider_history_v13['prices'][sid]} for sid in ['2330','1101','1216']]
print('V26_DATE_CHECK',{'provider_reference_counts':[len(s) for s in provider_dates26],'all_three_equal':provider_dates26[0]==provider_dates26[1]==provider_dates26[2],'provider_not_in_official_calendar':len(provider_dates26[0]-set(calendar_days_v13)),'official_in_range_not_provider':len({d for d in calendar_days_v13 if min(provider_dates26[0])<=d<=max(provider_dates26[0])}-provider_dates26[0])})


In [ ]:
# v26 long-history all-season24day exposure test, cached inputs only.
definition26={'purpose':'testdontleavecashacrosslonghistorynotonlyrecentbullsample','data':'existingFinMindprice2016-07-01..2025-12-31+dividends/globalactioncaches;noIOrequests','universe':'current150fixedhistorically,notPIT;188columnsbut150poolonly','selection':'price-only20/60half;prior60price/actionand20volumegates;noinstitutionaldataavailablehistorically,notfullA','methods':[.8,.9,.95,1.0],'fixed':'25target,equal,5dayadjust,24dayfreshcapitalwindows,1e9,lots1000,fee.001425tax.003','events':'reusev14cashonlyreceivable;unknownnoncash/paidactionsinvalidwhileheld;badpasteligibilityfixed;verified2404stopmarkonly2018-11-29..12-07rawquotemissing','primary':'allcalendar24dayrolling;80/90/95exactcommondata+constraintvalid;100ownvaliddiagnosticnotdropmainmask;annualOctcasescontrolverify','diagnostics':'startyeargroups,startmonthgroups,worstwindows,24nonoverlapstartphases;nofuturemarketlabelintrading','inference':'3primaryexcesspairsBonferroni3,24/48positionblockbootstrap5000;overlapnotiid;yearconsistencynotjustpooledmean','cash_limits':'nocashnegative,cash<25%,20..30held,caps2330 .25others .1five-sessiongrace;ActiveShareunverified','no_change_after_results':'noadditionalcashgrid,noLLM,noadaptivestop,no100resizingafterexecution','snapshot_sources':['外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz','完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz'],'limits':'currentpoolbias,corporateactionexclusions,provideraggregateVWAPnotcertifiedDataPack,short24daywindowsnottenyearcontinuouslyreinvestedaccount'}
hash26=hashlib.sha256(json.dumps(definition26,sort_keys=True).encode()).hexdigest();(recovery_root/'動能_2016至2025現金比例長期比較定義_v26_2026-10-15.json').write_text(json.dumps(definition26,ensure_ascii=False,indent=2),encoding='utf-8')
assert provider_dates26[0]==provider_dates26[1]==provider_dates26[2] and not(provider_dates26[0]-set(calendar_days_v13))
days26=sorted(provider_dates26[0]);master_case26={'key':'all_history_v26','year':0,'dates':days26,'start_index':61}
arrays26=build_case_v13(master_case26);arrays26['mark']=arrays26['close'].copy();arrays26['paid_known']=np.zeros_like(arrays26['bad']);arrays26['available']=np.isfinite(arrays26['execution'])&(arrays26['execution']>0)&np.isfinite(arrays26['volume'])&(arrays26['volume']>0)
for t,day in enumerate(days26):
    for j,sid in enumerate(all_external_ids_v13):arrays26['paid_known'][t,j]=(sid,day) in paid_known_days_v14
    if datetime.date(2018,11,29)<=day<=datetime.date(2018,12,7):
        j=ext_si_v13['2404'];assert not np.isfinite(arrays26['close'][t,j]);arrays26['mark'][t,j]=63.9;arrays26['available'][t,j]=False
case_arrays_v14['all_history_v26']=arrays26
# Precompute exactly the same existing price-only score, not another signal.
score26=np.full_like(arrays26['close'],np.nan);poolmask26=np.array([s in set(full_ids_v13) for s in all_external_ids_v13])
for q in range(60,len(days26)):
    rr=arrays26['returns'][q-59:q+1];vo=arrays26['volume'][q-19:q+1];eligible=poolmask26&np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(vo)&(vo>0),axis=0)&~np.any(arrays26['bad'][q-59:q+1],axis=0)&np.isfinite(arrays26['close'][q])&(arrays26['close'][q]>0)
    score26[q]=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,eligible)+.5*percentile(np.prod(1+rr,axis=0)-1,eligible)
tree26=ast.parse(engine_source_v14);fn26=next(n for n in tree26.body if isinstance(n,ast.FunctionDef))
fn26.name='simulate_longcash26';fn26.args.args.append(ast.arg(arg='invested'));fn26.args.defaults.append(ast.Constant(value=.9))
class InvestConstant26(ast.NodeTransformer):
    def visit_Constant(self,node):
        if isinstance(node.value,float) and node.value==.9:return ast.copy_location(ast.Name(id='invested',ctx=ast.Load()),node)
        return node
fn26.body=[InvestConstant26().visit(node) for node in fn26.body]
# Replace only eligible/score calculations with cached identical perdate results.
for node in ast.walk(fn26):
    if isinstance(node,ast.Assign) and len(node.targets)==1 and isinstance(node.targets[0],ast.Name):
        if node.targets[0].id=='eligible':node.value=ast.parse('np.isfinite(score26[t-1])',mode='eval').body
        elif node.targets[0].id=='score':node.value=ast.parse('score26[t-1]',mode='eval').body
engine26=ast.unparse(ast.fix_missing_locations(ast.Module(body=[fn26],type_ignores=[])))
# Preserve finite raw outcomes and execution constraints for invalid100diagnostics.
engine26=engine26.replace("'return': None if invalid else net", "'return': None if invalid else net, 'modeled_return':net")
engine26=engine26.replace('    traces = []','    traces = []\n    min_cash_ratio=1.; max_cash_ratio=0.; negative_cash_days=0')
engine26=engine26.replace('        held = int(np.count_nonzero(shares))','        min_cash_ratio=min(min_cash_ratio,cash/nav);max_cash_ratio=max(max_cash_ratio,cash/nav);negative_cash_days+=int(cash<0)\n        held = int(np.count_nonzero(shares))')
engine26=engine26.replace("'paid_policy': paid_policy", "'min_cash':min_cash_ratio,'max_cash':max_cash_ratio,'negative_cash_days':negative_cash_days, 'paid_policy': paid_policy")
exec(engine26,globals())
# Annual seasonal controls compare master-array engine with independently preserved v14 cases.
controls26=[]
for year,oldcase in long_cases_v12.items():
    idx=days26.index(datetime.date.fromisoformat(oldcase['start']));case={'key':'all_history_v26','year':year,'start_index':idx}
    now=simulate_longcash26(case,set(full_ids_v13),'equal',0,'unresolved',.9);old=simulate_repaired_v14(dict(oldcase,key=f'full150_{year}'),set(full_ids_v13),'equal',0,'unresolved')
    assert now['reasons']==old['reasons'] and now['violations']==old['violations']
    assert (now['return'] is None)==(old['return'] is None)
    if old['return'] is not None:assert all(np.isclose(now[k],old[k],rtol=0,atol=1e-12) for k in ['return','mdd','cost','turnover'])
    controls26.append({'year':year,'matched':True,'valid':old['return'] is not None})
starts26=np.arange(61,len(days26)-23);assert len(starts26)==2236
sp26=recovery_root/'A策略_無人值守研究狀態.json';state26=json.loads(sp26.read_text(encoding='utf-8'));state26['long_cash_v26']={'status':'defined_running','running':True,'hash':hash26,'years':[2016,2025],'windows_per_method':len(starts26),'methods':4,'annual_controls':controls26};sp26.write_text(json.dumps(state26,ensure_ascii=False,indent=2),encoding='utf-8')
print('V26_FROZEN',{'hash':hash26,'price_dates':len(days26),'windows':len(starts26),'first_start':str(days26[int(starts26[0])]),'last_start':str(days26[int(starts26[-1])]),'last_end':str(days26[-1]),'annual_controls':controls26,'unknown_events':len(arrays26['unknown_reference_events']),'no_requests':True},flush=True)


In [ ]:
# v26 full-history sliding24 windows, no new data or parameter search.
results26={};method_p26={'P80':.8,'P90':.9,'P95':.95,'P100':1.}
for key,p in method_p26.items():
    rows=[]
    for i,start in enumerate(starts26):
        case={'key':'all_history_v26','year':days26[int(start)].year,'start_index':int(start)};row=simulate_longcash26(case,set(full_ids_v13),'equal',0,'unresolved',p);row['start_index']=int(start);rows.append(row)
        if (i+1)%700==0:print('V26_PROGRESS',key,i+1,'/',len(starts26),flush=True)
    results26[key]=rows;print('V26_DONE',key,'valid',sum(r['return'] is not None for r in rows),'negcash',sum(r['negative_cash_days']>0 for r in rows),flush=True)
valid26={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results26.items()};ret26={k:np.array([np.nan if r['return'] is None else r['return'] for r in rows]) for k,rows in results26.items()};common26=valid26['P80']&valid26['P90']&valid26['P95'];allcommon26=common26&valid26['P100'];N26=len(starts26)
def metric26(k,mask):
    ix=np.flatnonzero(mask);x=ret26[k][mask];rows=results26[k]
    if not len(ix):return {'n':0}
    return {'n':len(ix),'win_rate':float(np.mean(x>0)),'expectancy':float(x.mean()),'mean_win':float(x[x>0].mean()) if np.any(x>0) else None,'mean_loss':float(x[x<0].mean()) if np.any(x<0) else None,'P10':float(np.quantile(x,.1)),'worst_return':float(x.min()),'worst_mdd':float(min(rows[i]['mdd'] for i in ix)),'mean_cost':float(np.mean([rows[i]['cost'] for i in ix])),'mean_turnover':float(np.mean([rows[i]['turnover'] for i in ix]))}
pairdefs26=[('P80','P90'),('P95','P90'),('P95','P80')];pairs26={};boot_means26={}
for a,b in pairdefs26:
    name=a+'_minus_'+b;d=ret26[a]-ret26[b];pairs26[name]={'n':int(common26.sum()),'mean_excess':float(np.mean(d[common26])),'beat_rate':float(np.mean(d[common26]>1e-12)),'candidate':metric26(a,common26),'base':metric26(b,common26)};boot_means26[name]={}
# Batch bootstrap original chronological start positions; never compress missing windows.
for block in [24,48]:
    rng=np.random.default_rng(26015+block);nb=int(np.ceil(N26/block));boots={name:[] for name in pairs26}
    for chunk in range(50):
        start=rng.integers(0,N26-block+1,size=(100,nb));inds=(start[:,:,None]+np.arange(block)).reshape(100,-1)[:,:N26]
        for a,b in pairdefs26:
            name=a+'_minus_'+b;d=np.where(common26,ret26[a]-ret26[b],np.nan);sample=d[inds];count=np.isfinite(sample).sum(axis=1);boots[name].extend((np.nansum(sample,axis=1)/count).tolist())
    for name,v in boots.items():pairs26[name][f'CI{block}']=np.quantile(v,[.025,.975]).tolist();pairs26[name][f'familyCI{block}']=np.quantile(v,[.05/6,1-.05/6]).tolist();boot_means26[name][block]=np.array(v)
year26={};month26={}
for year in range(2016,2026):
    mask=np.array([days26[int(s)].year==year for s in starts26]);year26[str(year)]={'planned_n':int(mask.sum()),'common_n':int((common26&mask).sum()),'methods':{k:metric26(k,common26&mask) for k in ['P80','P90','P95']},'valid_each':{k:int((v&mask).sum()) for k,v in valid26.items()},'negative_cash_each':{k:sum(bool(mask[i]) and row['negative_cash_days']>0 for i,row in enumerate(rows)) for k,rows in results26.items()}}
for month in range(1,13):
    mask=np.array([days26[int(s)].month==month for s in starts26]);month26[str(month)]={'planned_n':int(mask.sum()),'common_n':int((common26&mask).sum()),'methods':{k:metric26(k,common26&mask) for k in ['P80','P90','P95']}}
phase26={}
for phase in range(24):
    mask=common26&(np.arange(N26)%24==phase);phase26[str(phase)]={k:metric26(k,mask) for k in ['P80','P90','P95']}
coverage26={}
for k,rows in results26.items():coverage26[k]={'planned':N26,'valid_n':int(valid26[k].sum()),'invalid_data_action_execution':sum(r['invalid'] for r in rows),'violation_windows':sum(r['violations']>0 for r in rows),'negative_cash_windows':sum(r['negative_cash_days']>0 for r in rows),'cash25_windows':sum(r['max_cash']>=.25 for r in rows),'reason_counts':dict(Counter(reason for r in rows for reason in r['reasons']))}
worst26={}
for k in ['P80','P90','P95']:
    ix=sorted(np.flatnonzero(common26),key=lambda i:ret26[k][i])[:15];worst26[k]=[{'start':results26[k][i]['start'],'end':results26[k][i]['end'],'return':float(ret26[k][i]),'mdd':results26[k][i]['mdd']} for i in ix]
annual26={}
for year,base in long_cases_v12.items():
    i=int(np.where(starts26==days26.index(datetime.date.fromisoformat(base['start'])))[0][0]);annual26[str(year)]={k:{f:results26[k][i][f] for f in ['start','end','return','mdd','cost','violations','reasons','negative_cash_days']} for k in results26}
summary26={'definition':definition26,'hash':hash26,'price_first':str(days26[0]),'price_last':str(days26[-1]),'first_window':results26['P90'][0]['start'],'last_window_start':results26['P90'][-1]['start'],'planned_per_method':N26,'core_common_n':int(common26.sum()),'all4_common_n':int(allcommon26.sum()),'core_common_indices':np.flatnonzero(common26).tolist(),'metrics':{k:metric26(k,common26) for k in ['P80','P90','P95']},'P100_ownvalid_metric_not_comparable':metric26('P100',valid26['P100']),'pairs':pairs26,'years_startyear':year26,'months_startmonth':month26,'nonoverlap_phases':phase26,'coverage':coverage26,'worst_windows':worst26,'annual_october_points':annual26,'annual_original_controls':controls26,'no_independent_window_claim':True}
(recovery_root/'動能_2016至2025現金比例逐窗口_v26_2026-10-15.json').write_text(json.dumps({'definition':definition26,'engine_sha256':hashlib.sha256(engine26.encode()).hexdigest(),'results':results26},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
(recovery_root/'動能_2016至2025現金比例統計_v26_2026-10-15.json').write_text(json.dumps(summary26,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V26_STATS',json.dumps({'n':summary26['core_common_n'],'all4':summary26['all4_common_n'],'metrics':summary26['metrics'],'pairs':{k:{f:v[f] for f in ['mean_excess','familyCI24','familyCI48']} for k,v in pairs26.items()},'coverage':coverage26,'years':{y:{'n':s['common_n'],'metrics':{k:{f:v[f] for f in ['win_rate','expectancy','worst_mdd'] if f in v} for k,v in s['methods'].items()}} for y,s in year26.items()},'worst':{k:v[:2] for k,v in worst26.items()}},ensure_ascii=False,default=serial_family_v15),flush=True)


In [ ]:
# v26 diagnose exclusion concentration, not hide incomplete long-history results.
unknown_group26=Counter(r['sid'] for r in arrays26['unknown_reference_events']);unknown_year26=Counter(r['date'][:4] for r in arrays26['unknown_reference_events'])
quality26={'unknown_reference_event_n':len(arrays26['unknown_reference_events']),'unknown_reference_by_year':dict(unknown_year26),'most_flagged_tickers':unknown_group26.most_common(8),'unknown_examples':arrays26['unknown_reference_events'][:8],'core_coverage':float(common26.mean()),'monthly_coverage':{m:{'valid':r['common_n'],'planned':r['planned_n'],'fraction':r['common_n']/r['planned_n'] if r['planned_n'] else None} for m,r in month26.items()},'bad_event_policy':'unresolvedratherthaninferdividends/referencefromprice','new_data_queries':0}
(recovery_root/'動能_2016至2025資料覆蓋與缺口_v26_2026-10-15.json').write_text(json.dumps(quality26,ensure_ascii=False,indent=2,default=serial_family_v15),encoding='utf-8')
print('V26_QUALITY',json.dumps(quality26,ensure_ascii=False,default=serial_family_v15))


In [ ]:
# Public cached dividend/reference and listing-time audit; no new quote assumptions.
audit26={}
for sid,day in [('6505','2016-07-04'),('3260','2016-07-04'),('5876','2016-07-05')]:
    audit26[sid]={'price':price_lookup_v13[sid].get(day),'dividend_records':[r for r in provider_history_v13['dividends'].get(sid,[]) if str(r.get('CashExDividendTradingDate',''))[:10]==day],'cash_amount_model':cash_amount_v13.get((datetime.date.fromisoformat(day),sid))}
listing26={str(r[0]):str(r[1]) for r in raw_data['stocks'] if len(r)>1};listing_sample26={sid:listing26.get(sid) for sid in ['5876','6584','4772','7750','2330']}
print('V26_SOURCE_AUDIT',json.dumps({'examples':audit26,'listing':listing_sample26,'listed_dates_available':len(set(full_ids_v13)&set(listing26))},ensure_ascii=False,default=str))
print('V26_CASH_BUILD',cash_code_v14[:9000])


In [ ]:
# v27 source-quality amendment, not a return-optimized strategy change.
definition27={**definition26,'version':'v27_sourcequality_cashdiv_and_listing','amendment_reason':'v26auditshowsprelistingemergingquotesandknowncashdividenddaysspread0;notchoosehigherreturnparameters','eligibility_amendment':'onlycurrent150withDBlisteddate<=startof60sessionlookback;DBdatesnotallindividuallyofficialverified;conservativecurrentmarketlistinggate,nothistorical150reconstruction','return_amendment':'cash-adjustedwealthreturn=(close+knowncashdiv)/previousclose-1;do_notderivecashdividendfromspread0;notexchangeexrightreferenceformula','cash_flag_relaxation':'onlyexistinguniqueexplicitcashamount,spread0,notknownnoncasheventnorpaid/freeactionthatsamedate;unknownnonzero-referenceandallotherunknownactionsstillbad','no_raw_price_change':True,'previous_v26':'retainedlegacydiagnosticnotmainconclusion','primary_reference':'https://investoredu.twse.com.tw/pages/TWSE_InvestmentQA.aspx?ID=1'}
hash27=hashlib.sha256(json.dumps(definition27,sort_keys=True).encode()).hexdigest();(recovery_root/'動能_長歷史掛牌與現金股息修正定義_v27_2026-10-15.json').write_text(json.dumps(definition27,ensure_ascii=False,indent=2),encoding='utf-8')
arrays27={k:(v.copy() if isinstance(v,np.ndarray) else v) for k,v in arrays26.items()};days27=days26;starts27=starts26;N27=len(starts27);listed_array27=np.array([datetime.date.fromisoformat(listing26[s]) if s in listing26 else datetime.date.max for s in all_external_ids_v13],dtype=object)
blocked_cashdates27=set()
for sid,rows in provider_history_v13['dividends'].items():
    for row in rows:
        day=valid_event_day_v13(row.get('StockExDividendTradingDate'));amt=sum(number_v13(row.get(k,0)) if np.isfinite(number_v13(row.get(k,0))) else 0 for k in ['StockEarningsDistribution','StockStatutorySurplus','TotalNumberOfCashCapitalIncrease'])
        if day is not None and amt>0:blocked_cashdates27.add((day,sid))
cash_repaired27=[]
for t in range(1,len(days27)):
    for j in np.flatnonzero(arrays27['cashdiv'][t]>0):
        sid=all_external_ids_v13[j];day=days27[t];row=price_lookup_v13.get(sid,{}).get(str(day));pure=(day,sid) not in bad_action_dates_v13 and (day,sid) not in blocked_cashdates27
        if row and pure and number_v13(row.get('spread'))==0 and arrays27['bad'][t,j]:
            arrays27['bad'][t,j]=False;cash_repaired27.append({'stock':sid,'date':str(day),'cash':float(arrays27['cashdiv'][t,j]),'old_derived_reference':float(arrays26['close'][t,j]),'formula':'cashwealthreturn,notreferenceinference'})
    prev=arrays27['close'][t-1];now=arrays27['close'][t];valid=np.isfinite(prev)&np.isfinite(now)&(prev>0)&(now>0)
    arrays27['returns'][t]=np.nan;arrays27['returns'][t,valid]=(now[valid]+arrays27['cashdiv'][t,valid])/prev[valid]-1
case_arrays_v14['all_history_v27']=arrays27
score27=np.full_like(score26,np.nan);prelisting_rejected27=0
for q in range(60,len(days27)):
    rr=arrays27['returns'][q-59:q+1];vo=arrays27['volume'][q-19:q+1]
    source_ok=poolmask26&np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(vo)&(vo>0),axis=0)&~np.any(arrays27['bad'][q-59:q+1],axis=0)&np.isfinite(arrays27['close'][q])&(arrays27['close'][q]>0)
    listed=listed_array27<=days27[q-59];prelisting_rejected27+=int(np.sum(source_ok&~listed));ok=source_ok&listed
    score27[q]=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,ok)+.5*percentile(np.prod(1+rr,axis=0)-1,ok)
engine27=engine26.replace('simulate_longcash26','simulate_longcash27').replace('score26','score27');exec(engine27,globals())
controls27=[]
for year,base in long_cases_v12.items():
    s=days27.index(datetime.date.fromisoformat(base['start']));q=s-1;rr=arrays27['returns'][q-59:q+1];vo=arrays27['volume'][q-19:q+1];ok=poolmask26&(listed_array27<=days27[q-59])&np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(vo)&(vo>0),axis=0)&~np.any(arrays27['bad'][q-59:q+1],axis=0)&np.isfinite(arrays27['close'][q])&(arrays27['close'][q]>0)
    manual=.5*percentile(np.prod(1+rr[-20:],axis=0)-1,ok)+.5*percentile(np.prod(1+rr,axis=0)-1,ok);assert np.allclose(manual[ok],score27[q,ok])
    controls27.append({'year':year,'direct_score_matched':True,'eligible':int(ok.sum())})
for k in ['close','volume','execution','cashdiv','mark','available']:assert np.array_equal(arrays27[k],arrays26[k],equal_nan=True)
source_quality27={'known_cash_zero_spread_repaired':len(cash_repaired27),'repairs':cash_repaired27,'prelisting_eligible_stock_days_rejected':prelisting_rejected27,'listed_dates_source':'readonlyteamDBstockmaster150dates,currentmarketlistingnotfullPITuniverse','raw_price_valuation_cash_amount_unchanged':True,'remainingbad_n':int(arrays27['bad'].sum()),'oldbad_n':int(arrays26['bad'].sum()),'calendar_reference3stocks_match':True,'annual_direct_score_controls':controls27}
(recovery_root/'動能_長歷史來源修正稽核_v27_2026-10-15.json').write_text(json.dumps(source_quality27,ensure_ascii=False,indent=2,default=serial_family_v15),encoding='utf-8')
sp27=recovery_root/'A策略_無人值守研究狀態.json';state27=json.loads(sp27.read_text(encoding='utf-8'));state27['long_cash_v26'].update({'status':'completed_legacy_diagnostic_superseded_by_v27','running':False,'source_quality_warning':'prelistingandcashdivreference0audit','summary':'動能_2016至2025現金比例統計_v26_2026-10-15.json'});state27['long_cash_v27']={'status':'defined_running','running':True,'hash':hash27,'planned_per_method':2236,'methods':4,'source_quality_amendment':True};sp27.write_text(json.dumps(state27,ensure_ascii=False,indent=2),encoding='utf-8')
print('V27_QUALITY_READY',{'repairs':len(cash_repaired27),'prelisting_stockdays':prelisting_rejected27,'remaining_bad':int(arrays27['bad'].sum()),'controls':controls27,'rawquotesunchanged':True},flush=True)


In [ ]:
# v27 full-history sliding24 windows, no new data or parameter search.
results27={};method_p27={'P80':.8,'P90':.9,'P95':.95,'P100':1.}
for key,p in method_p27.items():
    rows=[]
    for i,start in enumerate(starts27):
        case={'key':'all_history_v27','year':days27[int(start)].year,'start_index':int(start)};row=simulate_longcash27(case,set(full_ids_v13),'equal',0,'unresolved',p);row['start_index']=int(start);rows.append(row)
        if (i+1)%700==0:print('V27_PROGRESS',key,i+1,'/',len(starts27),flush=True)
    results27[key]=rows;print('V27_DONE',key,'valid',sum(r['return'] is not None for r in rows),'negcash',sum(r['negative_cash_days']>0 for r in rows),flush=True)
valid27={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results27.items()};ret27={k:np.array([np.nan if r['return'] is None else r['return'] for r in rows]) for k,rows in results27.items()};common27=valid27['P80']&valid27['P90']&valid27['P95'];allcommon27=common27&valid27['P100'];N27=len(starts27)
def metric27(k,mask):
    ix=np.flatnonzero(mask);x=ret27[k][mask];rows=results27[k]
    if not len(ix):return {'n':0}
    return {'n':len(ix),'win_rate':float(np.mean(x>0)),'expectancy':float(x.mean()),'mean_win':float(x[x>0].mean()) if np.any(x>0) else None,'mean_loss':float(x[x<0].mean()) if np.any(x<0) else None,'P10':float(np.quantile(x,.1)),'worst_return':float(x.min()),'worst_mdd':float(min(rows[i]['mdd'] for i in ix)),'mean_cost':float(np.mean([rows[i]['cost'] for i in ix])),'mean_turnover':float(np.mean([rows[i]['turnover'] for i in ix]))}
pairdefs27=[('P80','P90'),('P95','P90'),('P95','P80')];pairs27={};boot_means27={}
for a,b in pairdefs27:
    name=a+'_minus_'+b;d=ret27[a]-ret27[b];pairs27[name]={'n':int(common27.sum()),'mean_excess':float(np.mean(d[common27])),'beat_rate':float(np.mean(d[common27]>1e-12)),'candidate':metric27(a,common27),'base':metric27(b,common27)};boot_means27[name]={}
# Batch bootstrap original chronological start positions; never compress missing windows.
for block in [24,48]:
    rng=np.random.default_rng(26015+block);nb=int(np.ceil(N27/block));boots={name:[] for name in pairs27}
    for chunk in range(50):
        start=rng.integers(0,N27-block+1,size=(100,nb));inds=(start[:,:,None]+np.arange(block)).reshape(100,-1)[:,:N27]
        for a,b in pairdefs27:
            name=a+'_minus_'+b;d=np.where(common27,ret27[a]-ret27[b],np.nan);sample=d[inds];count=np.isfinite(sample).sum(axis=1);boots[name].extend((np.nansum(sample,axis=1)/count).tolist())
    for name,v in boots.items():pairs27[name][f'CI{block}']=np.quantile(v,[.025,.975]).tolist();pairs27[name][f'familyCI{block}']=np.quantile(v,[.05/6,1-.05/6]).tolist();boot_means27[name][block]=np.array(v)
year27={};month27={}
for year in range(2016,2026):
    mask=np.array([days27[int(s)].year==year for s in starts27]);year27[str(year)]={'planned_n':int(mask.sum()),'common_n':int((common27&mask).sum()),'methods':{k:metric27(k,common27&mask) for k in ['P80','P90','P95']},'valid_each':{k:int((v&mask).sum()) for k,v in valid27.items()},'negative_cash_each':{k:sum(bool(mask[i]) and row['negative_cash_days']>0 for i,row in enumerate(rows)) for k,rows in results27.items()}}
for month in range(1,13):
    mask=np.array([days27[int(s)].month==month for s in starts27]);month27[str(month)]={'planned_n':int(mask.sum()),'common_n':int((common27&mask).sum()),'methods':{k:metric27(k,common27&mask) for k in ['P80','P90','P95']}}
phase27={}
for phase in range(24):
    mask=common27&(np.arange(N27)%24==phase);phase27[str(phase)]={k:metric27(k,mask) for k in ['P80','P90','P95']}
coverage27={}
for k,rows in results27.items():coverage27[k]={'planned':N27,'valid_n':int(valid27[k].sum()),'invalid_data_action_execution':sum(r['invalid'] for r in rows),'violation_windows':sum(r['violations']>0 for r in rows),'negative_cash_windows':sum(r['negative_cash_days']>0 for r in rows),'cash25_windows':sum(r['max_cash']>=.25 for r in rows),'reason_counts':dict(Counter(reason for r in rows for reason in r['reasons']))}
worst27={}
for k in ['P80','P90','P95']:
    ix=sorted(np.flatnonzero(common27),key=lambda i:ret27[k][i])[:15];worst27[k]=[{'start':results27[k][i]['start'],'end':results27[k][i]['end'],'return':float(ret27[k][i]),'mdd':results27[k][i]['mdd']} for i in ix]
annual27={}
for year,base in long_cases_v12.items():
    i=int(np.where(starts27==days27.index(datetime.date.fromisoformat(base['start'])))[0][0]);annual27[str(year)]={k:{f:results27[k][i][f] for f in ['start','end','return','mdd','cost','violations','reasons','negative_cash_days']} for k in results27}
summary27={'definition':definition27,'hash':hash27,'price_first':str(days27[0]),'price_last':str(days27[-1]),'first_window':results27['P90'][0]['start'],'last_window_start':results27['P90'][-1]['start'],'planned_per_method':N27,'core_common_n':int(common27.sum()),'all4_common_n':int(allcommon27.sum()),'core_common_indices':np.flatnonzero(common27).tolist(),'metrics':{k:metric27(k,common27) for k in ['P80','P90','P95']},'P100_ownvalid_metric_not_comparable':metric27('P100',valid27['P100']),'pairs':pairs27,'years_startyear':year27,'months_startmonth':month27,'nonoverlap_phases':phase27,'coverage':coverage27,'worst_windows':worst27,'annual_october_points':annual27,'annual_original_controls':controls27,'no_independent_window_claim':True}
(recovery_root/'動能_2016至2025現金比例逐窗口_v26_2027-10-15.json').write_text(json.dumps({'definition':definition27,'engine_sha256':hashlib.sha256(engine27.encode()).hexdigest(),'results':results27},ensure_ascii=False,default=serial_family_v15),encoding='utf-8')
(recovery_root/'動能_2016至2025現金比例統計_v26_2027-10-15.json').write_text(json.dumps(summary27,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
print('V27_STATS',json.dumps({'n':summary27['core_common_n'],'all4':summary27['all4_common_n'],'metrics':summary27['metrics'],'pairs':{k:{f:v[f] for f in ['mean_excess','familyCI24','familyCI48']} for k,v in pairs27.items()},'coverage':coverage27,'years':{y:{'n':s['common_n'],'metrics':{k:{f:v[f] for f in ['win_rate','expectancy','worst_mdd'] if f in v} for k,v in s['methods'].items()}} for y,s in year27.items()},'worst':{k:v[:2] for k,v in worst27.items()}},ensure_ascii=False,default=serial_family_v15),flush=True)


In [ ]:
# v27 long-history final report, all points, annual performance and missingness.
labels27={'P80':'股票80%／現金目標20%','P90':'股票90%／現金目標10%','P95':'股票95%／現金目標5%','P100':'名目股票100%／0%'}
fmt27=lambda x:'不可判定' if x is None else f'{x*100:.2f}%'
coverage27_pct=float(common27.mean());annual_summary27={}
for k in ['P80','P90','P95']:
    pos=sum(s['methods'][k].get('expectancy',-np.inf)>0 for s in year27.values());annual_summary27[k]={'positive_mean_startyears':pos,'negative_mean_startyears':[y for y,s in year27.items() if s['methods'][k].get('expectancy',0)<0]}
summary27['source_quality']=source_quality27;summary27['legacy_v26_comparison']={'n':summary26['core_common_n'],'cash90EV':summary26['metrics']['P90']['expectancy'],'status':'sourcequalitydefectsnotprimary'};summary27['annual_original_controls']=controls26;summary27['repaired_annual_score_controls']=controls27;summary27['year_consistency']=annual_summary27
lines27=['# 2016–2025長歷史：少留現金是否比較好？ v27','', '## 結論','將資料拉長後，在目前可判定的樣本中，提高股票投入比例提高平均24日費後報酬，但沒有顯著改變勝率，同時放大虧損與最差回撤。這支持較高曝險在這份長期正報酬樣本增加均值，不支持「零現金一定較好」或「投入更多就能避免虧損」。','95%減90%的平均增量約+0.149個百分點，24及48日區塊三比較校正區間都在0以上；但這仍限於目前股票池與可判定日期，不能排除存活／選擇偏誤或把它當成未來保證。2018與2022的起始窗口平均報酬為負，投入越高，平均損失越深。','', '## 時間與策略口徑','行情範圍2016-07-01至2025-12-31，先保留60日品質及訊號暖機，窗口起始2016-09-30至2025-11-27，每個起點觀察24交易日，每方法2236窗口、四方法共8944。不是只挑10月最後星期二，也不是十年資金連續持有的年化績效。','選股仍是20／60日動能排名各半、目標25檔、五日例行重選與等權調整，24日重新開始本金10億元，整千股、當日金額／股數均價成交、買賣手續費0.1425%、賣出另0.3%稅。這輪只改投入80／90／95／100%，沒有加碼、急停或找最佳比例。','早期沒有原A所需的法人資料品質欄位，所以這是價量版動能子研究，並非完全重現近年DB的原A。股票池是目前150檔的歷史可掛牌子集，不是各年度官方150檔，仍有存活與選擇偏誤；絕不能拿本表與v24／v25直接做差歸因。','', '## 先修正資料問題，再做主比較','舊v26診斷僅864個共同有效窗口，而且6～8月完全無有效起始日，不能當成全年結論。查核發現：','1. 快取含上市／上櫃前的興櫃價格，例如5876的DB掛牌日為2018-10-19，卻有2016行情。現在必須在60日回看開始前已達DB掛牌日期才可選；現有市場別的日期可能保守排除轉板前合法行情，未逐檔官方核實，也不是歷史官方股票池重建。','2. 部分金額明確的純現金除息日，價差欄為0；不能用收盤減0倒推參考價，再把與現金股利的差異全當未知公司行動。現改用（當日收盤＋已知現金股利）／前日收盤−1作現金調整日報酬，沒有從價格推造股息或改原始行情。',f"只放寬已知單一現金股利、spread=0、且同日沒有已知配股／現增／其他非標準事件的{len(cash_repaired27)}筆錯置旗標。未知配股、減資、認購權、其他參考差異仍保留不可判定。股息計入應收資產，不加入下單現金。2018漢唐已驗證停牌僅保留最後估值、禁止成交，不捏造價格。",'證交所說明：除息權利屬除息前持有人、純現金除息參考價以前收減現金股利計算；這與投資組合的現金財富報酬是不同概念。來源：https://investoredu.twse.com.tw/pages/TWSE_InvestmentQA.aspx?ID=1 。','v26原始診斷、數值與來源稽核全部保留；本輪v27才是修正後主表。來源修正理由與規則先保存，再重新計算，沒有因報酬高低挑修正方式。','', '## 全期共同可判定日期',f"三個保留現金緩衝的方法共同{summary27['core_common_n']}/{N27}窗口（{coverage27_pct*100:.2f}%）。其餘缺漏與違規不填0、不默默稱已避險。100%另列，不把主表縮成其僅209個成功子樣本。",'|投入比例|費後勝率|24日期望值|平均獲利|平均虧損|P10|最差24日窗口內回撤|平均費稅|','|---|---:|---:|---:|---:|---:|---:|---:|']
for k,m in summary27['metrics'].items():lines27.append('|'+labels27[k]+'|'+'|'.join(fmt27(m[f]) for f in ['win_rate','expectancy','mean_win','mean_loss','P10','worst_mdd','mean_cost'])+'|')
lines27+=['','## 同日期配對與統計不確定性','按全部2236個起始日位置做24／48日區塊bootstrap，各5000次，保留缺漏位置、不把有效窗口壓成無缺口的時間軸；三個配對Bonferroni。這是平均差的區間，不是下一場報酬區間，滑動窗口非獨立試驗。','|候選減對照|平均差pp|24區塊家族95%區間pp|48區塊家族95%區間pp|','|---|---:|---|---|']
for name,s in pairs27.items():
    a,b=name.split('_minus_');c=s['familyCI24'];d=s['familyCI48'];lines27.append(f"|{labels27[a]}減{labels27[b]}|{s['mean_excess']*100:+.3f}|[{c[0]*100:+.3f},{c[1]*100:+.3f}]|[{d[0]*100:+.3f},{d[1]*100:+.3f}]|")
lines27+=['','## 逐年：不是每一年都該加大投入','每列是該起始年度的24日窗口平均，不是該年連續持有的年度報酬；跨年窗口歸起始年。2016只含9月底後，2025起點只到11月底。','|起始年|共同有效／計畫窗口|80%勝率／期望值|90%勝率／期望值|95%勝率／期望值|','|---|---:|---|---|---|']
for year,s in year27.items():lines27.append('|'+year+'|'+str(s['common_n'])+'/'+str(s['planned_n'])+'|'+'|'.join(fmt27(s['methods'][k]['win_rate'])+'／'+fmt27(s['methods'][k]['expectancy']) for k in ['P80','P90','P95'])+'|')
lines27+=['2018與2022的負期望值窗口年份中，保留較多現金降低平均損失；其他正期望值年份通常是高投入提高均值。這只是事後年度分組，不是交易時知道下一年漲跌的切換規則。','', '## 最差窗口不是假設：2020-02-14→2020-03-19','|投入比例|該24日費後結果|該窗口內最大回撤|','|---|---:|---:|']
for k in ['P80','P90','P95']:
    r=worst27[k][0];lines27.append('|'+labels27[k]+'|'+fmt27(r['return'])+'|'+fmt27(r['mdd'])+'|')
lines27+=['這是同一真實歷史日期的比較，少留現金不會消除股市下跌。全期最高均值不能取代你願意承受多少損失的決定；這輪沒有替使用者自訂可接受回撤。','', '## 名目100%：不能只看成功窗口','|方法|計畫窗口|資料／公司行動不可判定|限制違反窗口|曾負現金窗口|最終有效窗口|','|---|---:|---:|---:|---:|---:|']
for k,c in coverage27.items():lines27.append(f"|{labels27[k]}|{c['planned']}|{c['invalid_data_action_execution']}|{c['violation_windows']}|{c['negative_cash_windows']}|{c['valid_n']}|")
lines27+=['各類問題可重疊，不能把排除數相加。100%先按前日淨值全部換股、再付費用，且實際均價不同於前收盤，因此容易負現金；本輪沒有事後用已知成交價縮股數讓它過關。整張仍可能留小額餘額，所以名目100%也不是每日現金精確0。','100%只有209個有效窗口，屬被執行條件篩選的子樣本，不能把它單獨的勝率或均值與1601窗口表比較。這不證明所有接近滿倉的預算方法都不可行，但足以否定本次「直接名目100%下單」可全面執行；必要費用與價差緩衝不能省略。80%也有14窗口曾被動現金>=25%，95%有1窗口曾負現金，尚非全面官方合規保證。','', '## 全年資料覆蓋與不重疊相位','圖另列每年每月有效起點覆蓋；未解事件547個持倉窗口、缺收盤140與成交不可用等問題仍存在，可能集中在特定月份，非隨機缺漏。覆蓋比原864窗口改善，仍不是完整無缺漏的十年。','依起點模24分為24個不重疊持有區間相位，JSON保存各相位勝率、期望值與回撤，沒有只挑最好相位。相位內仍可能有市場時序相關，不能稱獨立樣本；2016～2025的目前股票池偏誤也不會因區塊bootstrap消失。','', '## 目前可以採用的研究判斷','提高投入在這份較長歷史樣本增加平均費後收益，但主要是放大同一選股組合的曝險；勝率幾乎不變，負報酬年份與最差回撤更差。不能把「不要留現金」當普遍結論，也不能把95%相對90%的歷史正均值差當成新的選股能力。','後續若要接近滿倉，應另外定義事前費用／價格緩衝的股數預算，不能直接把100%當可用買股本金；這輪先完成歷史對照，沒有自動新增98%／99%網格或提交交易。','本研究固定現況150股票池且上市日期只作保守可用門檻，不是歷史市場全體；早期沒有法人品質欄位、不完整公司行動與部分資料缺漏、來源均價近似、官方Active Share及DataPack未驗證仍限制外推。沒有新的即時投資訊號、LLM判讀或外部付費資料。','', '## 保存與驗證','原Notebook cells302–310保存初次診斷、來源查核、預先修正定義、重算及成果。沿用PID2524未重啟。原v14十年年度控制在舊口徑全數重現；修正後十年起點分數直接計算與快取一致。原始價格／成交／股息金額／估值矩陣未改，純現金旗標修正及上市前資格排除有逐筆來源紀錄。2236×4新口徑窗口保存，與舊v26品質診斷分開；這不是8944個新獨立市場樣本。勝率／期望值恆等式、資產路徑終值、同日期差值、年份計數相加、24個相位與合法日末現金限制程式核對。圖1601×3全部點、桃紅中位數與0線，未做瀏覽器目視QA，也未重啟Run All。']
report27='\n'.join(lines27);rp27=recovery_root/'動能_2016至2025全年現金比例研究_v27_2026-10-15.md';rp27.write_text(report27,encoding='utf-8')
import plotly.graph_objects as go
rng27=np.random.default_rng(27015);fb27=go.Figure();fy27=go.Figure();ids27=np.flatnonzero(common27)
for pos,k in enumerate(['P80','P90','P95']):
    y=ret27[k][common27]*100
    fb27.add_trace(go.Box(x=np.full(len(y),pos),y=y,name=labels27[k],boxpoints=False,line={'color':'#4e77a8'},fillcolor='rgba(78,119,168,.15)',hoverinfo='skip',showlegend=False))
    fb27.add_trace(go.Scatter(x=pos+rng27.uniform(-.18,.18,len(y)),y=y,mode='markers',marker={'size':4,'opacity':.35},text=[results27[k][i]['start']+'→'+results27[k][i]['end'] for i in ids27],hovertemplate=labels27[k]+'<br>%{text}<br>24日淨報酬 %{y:.2f}%<extra></extra>',showlegend=False))
    med=float(np.median(y));fb27.add_trace(go.Scatter(x=[pos-.2,pos+.2],y=[med,med],mode='lines',line={'color':'#dc2477','width':4},hoverinfo='skip',showlegend=False))
    fy27.add_trace(go.Bar(x=list(year27),y=[r['methods'][k]['expectancy']*100 for r in year27.values()],name=labels27[k],customdata=[r['common_n'] for r in year27.values()],hovertemplate='%{x}年起點<br>24日平均 %{y:.2f}%<br>共同窗口 %{customdata}<extra>%{fullData.name}</extra>'))
fb27.add_hline(y=0,line_color='#222');fy27.add_hline(y=0,line_color='#222')
fb27.update_layout(template='plotly_white',height=540,title='2016–2025全部1601共同有效窗口：每個點、箱形圖及桃紅中位數',xaxis={'tickmode':'array','tickvals':[0,1,2],'ticktext':[labels27[k] for k in ['P80','P90','P95']]},yaxis_title='24日費後報酬 %')
fy27.update_layout(template='plotly_white',height=460,barmode='group',title='逐起始年度：24日平均報酬（不是年度連續持有報酬）',yaxis_title='24日期望值 %')
z27=[];custom27=[]
for year in range(2016,2026):
    zz=[];cc=[]
    for month in range(1,13):
        mask=np.array([days27[int(s)].year==year and days27[int(s)].month==month for s in starts27]);n=int(mask.sum());v=int((common27&mask).sum());zz.append(v/n*100 if n else None);cc.append([v,n])
    z27.append(zz);custom27.append(cc)
fc27=go.Figure(go.Heatmap(x=list(range(1,13)),y=list(range(2016,2026)),z=z27,customdata=custom27,zmin=0,zmax=100,colorscale='Blues',colorbar={'title':'可判定 %'},hovertemplate='%{y}年 %{x}月起點<br>有效 %{customdata[0]}／計畫 %{customdata[1]}<br>覆蓋 %{z:.1f}%<extra></extra>'))
fc27.update_layout(template='plotly_white',height=430,title='來源品質覆蓋：沒有把缺漏當成0報酬',xaxis_title='窗口起始月份',yaxis={'dtick':1})
hp27=recovery_root/'動能_十年全年投入比例與風險比較_v27_2026-10-15.html';hp27.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>2016–2025全年現金比例 v27</title><style>body{font:16px/1.7 system-ui;max-width:1400px;margin:28px auto;padding:0 20px;color:#172539}table{width:100%;border-collapse:collapse;font-size:14px}td{padding:9px;border-bottom:1px solid #ddd}tr:first-child{background:#eef3fa;font-weight:bold}p{overflow-wrap:anywhere}.note{background:#fff4db;padding:18px;border-radius:8px}</style><h1>2016–2025全年：少留現金是否比較好？</h1><p class="note">已跑四方法各2236個24日窗口，不只近期、不只10月。修正上市前行情與純現金股息口徑後，主表共同1601可判定窗口；股票池仍是目前150檔的歷史子集，有偏誤及缺漏。95%投入平均較高，負年份及最差回撤也更深；名目100%有1909窗口曾負現金，不得只挑209個成功子樣本。</p>'+fb27.to_html(full_html=False,include_plotlyjs=True)+fy27.to_html(full_html=False,include_plotlyjs=False)+fc27.to_html(full_html=False,include_plotlyjs=False)+render_report17(report27)+'<p><a href="https://investoredu.twse.com.tw/pages/TWSE_InvestmentQA.aspx?ID=1">證交所除權息權利與純現金除息說明</a></p></html>',encoding='utf-8')
for k,m in summary27['metrics'].items():
    x=ret27[k][common27];assert np.isclose(m['expectancy'],np.mean(x>0)*(m['mean_win'] or 0)+np.mean(x<0)*(m['mean_loss'] or 0))
    for i in np.flatnonzero(valid27[k]):
        row=results27[k][i];assert np.isclose(row['return'],row['path'][-1]/INITIAL-1) and row['min_cash']>=0 and row['max_cash']<.25 and row['violations']==0
for name,s in pairs27.items():assert np.isclose(s['mean_excess'],s['candidate']['expectancy']-s['base']['expectancy'])
assert sum(s['common_n'] for s in year27.values())==1601
assert sum(phase27[str(i)]['P90']['n'] for i in range(24))==1601
for q in range(60,len(days27)):assert not np.any(np.isfinite(score27[q])&(listed_array27>days27[q-59]))
assert sum(len(t.y) for t in fb27.data if t.type=='scatter' and t.mode=='markers')==1601*3
summary27['verification']={'annual_legacy_controls':10,'annual_directscore_controls':10,'master_sessions':2320,'windows_per_method':2236,'rawquotes_unchanged':True,'listed_gate_all_dates_verified':True,'cashflag_repairs':1400,'return_plot_points':4803,'year_month_coverage_cells':120,'payoff_identity':True,'path_end_identity':True,'year_and_phase_n_sum':True,'kernel_restarted':False}
(recovery_root/'動能_2016至2025現金比例統計_v27_2026-10-15.json').write_text(json.dumps(summary27,ensure_ascii=False,indent=2,default=serial_family_v15,allow_nan=False),encoding='utf-8')
state27=json.loads(sp27.read_text(encoding='utf-8'));state27['long_cash_v27'].update({'status':'completed','running':False,'core_common_n':1601,'coverage_fraction':float(common27.mean()),'literal100_negative_windows':1909,'price_only_not_full_A':True,'historical_official150_not_reconstructed':True,'outputs':[rp27.name,hp27.name,'動能_2016至2025現金比例統計_v27_2026-10-15.json'],'notebook_cells':[302,310],'verification':summary27['verification']});state27['historical_source_quality_v27']={'affected_legacy':'externalpricehistoryv12-v14andv26;rawsourceprelisting&cashdivspread0','main_replacement':hp27.name,'near_term_DB_methods_not_automatically_recomputed':True};sp27.write_text(json.dumps(state27,ensure_ascii=False,indent=2),encoding='utf-8')
wf27=recovery_root/'A策略_無人值守研究流程.md';wf27.write_text(wf27.read_text(encoding='utf-8')+'\n\n## v27 2016–2025全年投入比較（完成）\n四方法各2236個24日窗口，來源品質修正後三組共同1601。排除掛牌前行情、純現金股息以明確金額調整，不從spread0倒造參考價；未改原始價量／股息。95%平均較90%高0.149pp，但2018／2022負年份及最差回撤更差。名目100%有1909窗口負現金。價量子研究非完整A，現況股票池偏誤與28.4%未判定仍明列。原v12-v14／v26的歷史源有相關缺口，研究主結論改引用本輪；近期DB研究未自動重算。成果：'+hp27.name+'。\n',encoding='utf-8')
legacy_report27=recovery_root/'公司行動與成交修正_比較報告_v14_2026-10-15.md';legacy_text27=legacy_report27.read_text(encoding='utf-8')
if '## v27後續來源品質追蹤' not in legacy_text27:legacy_report27.write_text(legacy_text27+'\n\n## v27後續來源品質追蹤\n後續全年研究查到本系列快取含部分掛牌前興櫃行情，以及已知純現金除息日spread=0造成的參考價錯置旗標。v27新增保守掛牌日期門檻與現金財富報酬口徑，並保留所有原始價量。本文數字是舊定義的歷史紀錄，不應再當作已修正上述問題的主結論；最新來源稽核與結果見「'+rp27.name+'」。此追蹤不表示近年另一份DB資料已自動重算。\n',encoding='utf-8')
print('V27_COMPLETE',{'report':rp27.name,'html':hp27.name,'core_n':1601,'coverage':float(common27.mean()),'verification':summary27['verification'],'year_consistency':annual_summary27})


In [ ]:
# v29 inspect retained same-project kernel and saved result schema only.
print('V29_CONTEXT', {'pid':os.getpid(),'python':sys.executable,'dates':(str(dates[0]),str(dates[-1])),'shape':signal_returns_v15.shape,'score_keys':list(scores_v15),'factor_keys':list(v28_factor),'old_keys':list(v28_old)})
print('V29_SCHEMAS', {'factor':{k:(type(v).__name__,list(v)[:12] if isinstance(v,dict) else len(v)) for k,v in v28_factor.items()},'old':{k:(type(v).__name__,list(v)[:10] if isinstance(v,dict) else len(v)) for k,v in v28_old.items()}})
print('V29_ENGINE_HEAD',v28_replay_source[:1000])


In [ ]:
# v29 predeclare observational regime definitions; preserve existing trading rules.
v29_root=v28_root
v29_definition={'version':'v29_environment_observation','date':'2026-10-16','type':'observational_not_new_trading_strategy','reuse':'v16 MOM_E0 and FLOW_E0; no strategy rerun','lookback':20,'information_cutoff':'start-1 close','features':{'trend':'20-session equalweight150 proxy cumulative return positive vs nonpositive','breadth':'fraction positive20 returns >=0.5 vs below; minimum120 complete names','volatility':'20-session proxy daily std above training median vs at_or_below'},'volatility_threshold_training':'only starts with end<=2024-12-31; feature only, not future outcomes','proxy':'current150 daily mean of finite known-action-adjusted returns; >=120 names; changing constituents, not official index or executable portfolio','benchmark':'90% buyhold of gross proxy daily returns minus initial fee0.001425*0.9; no sale at end; no costs for implicit proxy membership changes; indicative nontradable reference, not proven alpha','sample':'both strategies valid; proxy outcome complete; features complete; no future data used to classify','intervals':'24 and48 original start-position circular blocks,3000 repetitions; three interaction contrasts Bonferroni; exploratory within-group95 not corrected','interaction':'difference of (MOM minus FLOW) in high state versus low state, not standalone strategy superiority','min_group_for_interpretation':50,'years':'grouped by start year; not independent unseen test','no_parameter_search':True,'new_trades':False,'limitations':['current150 survivorship','missingness nonrandom','repeated historical inspection','few independent regimes','price-derived regimes correlated with momentum','macro first-release data incomplete']}
(v29_root/'市場環境分層_實驗定義_v29_2026-10-16.json').write_text(json.dumps(v29_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v29_rows={k:v28_factor['results'][key] for k,key in [('MOM','MOM_E0'),('FLOW','FLOW_E0')]}
v29_starts=np.array([int(r['start']) for r in v29_rows['MOM']]);assert len(v29_starts)==650
assert all(int(a['start'])==int(b['start']) and int(a['end'])==int(b['end']) for a,b in zip(v29_rows['MOM'],v29_rows['FLOW']))
v29_good=np.isfinite(signal_returns_v15)&~bad_action_v15
v29_count=v29_good.sum(axis=1)
v29_proxy=np.divide(np.where(v29_good,signal_returns_v15,0.).sum(axis=1),v29_count,out=np.full(len(dates),np.nan),where=v29_count>=120)
v29_feature=[]
for start29 in v29_starts:
    q29=int(start29)-1;look29=signal_returns_v15[q29-19:q29+1];good29=v29_good[q29-19:q29+1].all(axis=0)
    valid29=bool(good29.sum()>=120 and np.isfinite(v29_proxy[q29-19:q29+1]).all())
    v29_feature.append({'start':int(start29),'cutoff':str(dates[q29]),'feature_valid':valid29,'names':int(good29.sum()),'trend':float(np.prod(1+v29_proxy[q29-19:q29+1])-1) if valid29 else None,'breadth':float(np.mean((np.prod(1+look29[:,good29],axis=0)-1)>0)) if valid29 else None,'vol':float(np.std(v29_proxy[q29-19:q29+1],ddof=1)) if valid29 else None})
v29_train=np.array([dates[int(s)+23]<=datetime.date(2024,12,31) for s in v29_starts])
v29_vol_threshold=float(np.median([f['vol'] for i,f in enumerate(v29_feature) if v29_train[i] and f['feature_valid']]))
v29_definition['locked_vol_threshold']=v29_vol_threshold
(v29_root/'市場環境分層_實驗定義_v29_2026-10-16.json').write_text(json.dumps(v29_definition,ensure_ascii=False,indent=2),encoding='utf-8')
print('V29_LOCKED',{'vol_threshold':v29_vol_threshold,'feature_valid':sum(f['feature_valid'] for f in v29_feature),'train_features':sum(v29_train[i] and f['feature_valid'] for i,f in enumerate(v29_feature))})


In [ ]:
# v29 paired observational analysis; existing results are referenced, not rerun.
v29_valid={k:np.array([not r['invalid'] and r['violations']==0 and r['return'] is not None for r in rows29]) for k,rows29 in v29_rows.items()}
v29_ret={k:np.array([r['return'] if ok else np.nan for r,ok in zip(v29_rows[k],v29_valid[k])]) for k in v29_rows}
v29_bench=np.array([.9*(np.prod(1+v29_proxy[int(s):int(s)+24])-1)-.9*.001425 if np.isfinite(v29_proxy[int(s):int(s)+24]).all() else np.nan for s in v29_starts])
v29_common=v29_valid['MOM']&v29_valid['FLOW']&np.array([f['feature_valid'] for f in v29_feature])&np.isfinite(v29_bench)
v29_state={
 'trend':np.array([f['trend']>0 if f['feature_valid'] else False for f in v29_feature]),
 'breadth':np.array([f['breadth']>=.5 if f['feature_valid'] else False for f in v29_feature]),
 'volatility':np.array([f['vol']>v29_vol_threshold if f['feature_valid'] else False for f in v29_feature])}
v29_labels={'trend':['市場20日非正','市場20日上漲'],'breadth':['上漲股票未過半','上漲股票至少一半'],'volatility':['波動較低','波動較高']}
v29_rng=np.random.default_rng(20261016);v29_boot={}
for b29 in [24,48]:
    seeds29=v29_rng.integers(0,650,size=(3000,int(np.ceil(650/b29))))
    v29_boot[b29]=((seeds29[:,:,None]+np.arange(b29))%650).reshape(3000,-1)[:,:650]
def v29_ci(values29,mask29,alpha=.05):
    out29={}
    for b29,ix29 in v29_boot.items():
        weights29=mask29[ix29];den29=weights29.sum(axis=1);num29=np.where(weights29,values29[ix29],0.).sum(axis=1)
        means29=np.divide(num29,den29,out=np.full(len(ix29),np.nan),where=den29>0)
        out29[str(b29)]=[float(x) for x in np.nanquantile(means29,[alpha/2,1-alpha/2])]
    return out29
def v29_metrics(k29,mask29):
    ix29=np.flatnonzero(mask29);x29=v29_ret[k29][ix29];r29=[v29_rows[k29][i] for i in ix29]
    if not len(x29):return {'n':0}
    return {'n':len(x29),'win':float(np.mean(x29>0)),'ev':float(x29.mean()),'mean_loss':float(x29[x29<0].mean()) if np.any(x29<0) else None,'p10':float(np.quantile(x29,.1)),'worst_mdd':float(min(r['mdd'] for r in r29)),'fee':float(np.mean([r['cost'] for r in r29])),'proxy_relative_ev':float(np.mean(x29-v29_bench[ix29])),'beat_proxy':float(np.mean(x29>v29_bench[ix29])),'mean_ci_exploratory':v29_ci(v29_ret[k29],mask29)}
v29_groups={};v29_interactions={};v29_difference=v29_ret['MOM']-v29_ret['FLOW']
for axis29,state29 in v29_state.items():
    v29_groups[axis29]={}
    for high29 in [False,True]:
        mask29=v29_common&(state29==high29);name29=v29_labels[axis29][int(high29)]
        events29=sum(mask29[i] and (i==0 or not mask29[i-1]) for i in range(650))
        v29_groups[axis29][name29]={'n':int(mask29.sum()),'contiguous_start_runs':int(events29),'caution':'less_than50' if mask29.sum()<50 else 'exploratory','methods':{k:v29_metrics(k,mask29) for k in v29_rows},'proxy_ev':float(v29_bench[mask29].mean()) if mask29.any() else None,'mom_minus_flow':float(v29_difference[mask29].mean()) if mask29.any() else None,'years':{str(yr29):{k:v29_metrics(k,mask29&np.array([dates[int(s)].year==yr29 for s in v29_starts])) for k in v29_rows} for yr29 in [2024,2025,2026]}}
    hi29=v29_common&state29;lo29=v29_common&~state29;cis29={}
    for b29,ix29 in v29_boot.items():
        diffs29=[]
        for mask29 in [hi29,lo29]:
            den29=mask29[ix29].sum(axis=1);num29=np.where(mask29[ix29],v29_difference[ix29],0).sum(axis=1);diffs29.append(np.divide(num29,den29,out=np.full(3000,np.nan),where=den29>0))
        cis29[str(b29)]=[float(x) for x in np.nanquantile(diffs29[0]-diffs29[1],[.05/6,1-.05/6])]
    v29_interactions[axis29]={'estimate':float(v29_difference[hi29].mean()-v29_difference[lo29].mean()),'family_ci':cis29,'interpretation':'MOM_FLOW_gap_high_minus_low_not_causal'}
v29_summary={'definition':v29_definition,'common_n':int(v29_common.sum()),'planned_n':650,'old_pair_n':int((v29_valid['MOM']&v29_valid['FLOW']).sum()),'overall':{k:v29_metrics(k,v29_common) for k in v29_rows},'groups':v29_groups,'interactions':v29_interactions,'macro_status':{'first_release_dataset':'not_yet_reconstructed','ndc_2024_revision':'new system enabled2024-03-04 for2024-01 release; no backfill with latest vintages','PMI':'database page403; no full monthly release archive yet','sources':['https://data.gov.tw/dataset/6099','https://www.ndc.gov.tw/News_Content.aspx?n=CFE1B745364D2499&s=9B95A3EDC8B036A7&sms=814775938D6333B4','https://index.ndc.gov.tw/n/zh_tw/data/PMI']}}
v29_points=[]
for i29 in range(650):
    v29_points.append({'start':str(dates[int(v29_starts[i29])]),'end':str(dates[int(v29_starts[i29])+23]),'common':bool(v29_common[i29]),'features':v29_feature[i29],'states':{k:bool(x[i29]) for k,x in v29_state.items()},'methods':{k:float(v29_ret[k][i29]) if v29_valid[k][i29] else None for k in v29_rows},'proxy':float(v29_bench[i29]) if np.isfinite(v29_bench[i29]) else None})
assert v29_summary['old_pair_n']==511
for axis29,group29 in v29_groups.items():
    assert sum(g['n'] for g in group29.values())==v29_summary['common_n']
for k29,m29 in v29_summary['overall'].items():
    x29=v29_ret[k29][v29_common];assert np.isclose(m29['ev'],x29.mean())
    for i29 in np.flatnonzero(v29_common):assert v29_feature[i29]['cutoff']<str(dates[int(v29_starts[i29])])
(v29_root/'市場環境分層_統計_v29_2026-10-16.json').write_text(json.dumps(v29_summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
(v29_root/'市場環境分層_全部窗口_v29_2026-10-16.json').write_text(json.dumps(v29_points,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('V29_RESULTS',{'n':v29_summary['common_n'],'groups':{a:{n:{'n':g['n'],'MOM':{x:g['methods']['MOM'][x] for x in ['win','ev','fee','worst_mdd']},'FLOW':{x:g['methods']['FLOW'][x] for x in ['win','ev','fee','worst_mdd']}} for n,g in gs.items()} for a,gs in v29_groups.items()},'interactions':v29_interactions})


In [ ]:
# v29 official macro source availability only; latest history is not first-release data.
import urllib.request,zipfile,io,csv
v29_macro_sources={
'ndc_history':'https://ws.ndc.gov.tw/Download.ashx?icon=.zip&n=5pmv5rCj5oyH5qiZ5Y%2BK54eI6JmfLnppcA%3D%3D&u=LzAwMS9hZG1pbmlzdHJhdG9yLzEwL3JlbGZpbGUvNTc4MS82MzkyL2VhMjM1YmQ5LWQwNTItNGE2OS1hYmZjLWQ1Yzc4NWQzZDBlMi56aXA%3D',
'pmi_history':'https://ws.ndc.gov.tw/Download.ashx?icon=.csv&n=6Ie654Gj5o6h6LO857aT55CG5Lq65oyH5pW4KHBtaeWPim5taSkuY3N2&u=LzAwMS9hZG1pbmlzdHJhdG9yLzEwL3JlbGZpbGUvNTc4MS82MzkxL2JmOGE0ZWI3LTEwZmUtNGZhMC1iNjQ2LTMwZTg5MGQwMjE4YS5jc3Y%3D',
'ndc_release_latest':'https://ws.ndc.gov.tw/Download.ashx?icon=.csv&n=5pyA5paw5pmv5rCj5qaC5rOB5paw6IGe56i%2FLmNzdg%3D%3D&u=LzAwMS9hZG1pbmlzdHJhdG9yLzEwL3JlbGZpbGUvNTc4MS8yNTEzOS85MThmYjdmMi00MTY1LTQ0ZjUtYjIzOC01NDc4NWQ0ODBlZTEuY3N2',
'pmi_release_latest':'https://ws.ndc.gov.tw/Download.ashx?icon=.csv&n=5pyA5paw6Ie654Gj5o6h6LO857aT55CG5Lq65oyH5pW45paw6IGe56i%2FLmNzdg%3D%3D&u=LzAwMS9hZG1pbmlzdHJhdG9yLzEwL3JlbGZpbGUvNTc4MS8yNTE0MC84Y2RiM2I4Ny0yNWIyLTRjYTUtOGFjOS0yYTRlMTQyNzE2OWEuY3N2'}
v29_macro_audit={};v29_macro_bytes={}
for name29,url29 in v29_macro_sources.items():
    try:
        req29=urllib.request.Request(url29,headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req29,timeout=18) as res29:data29=res29.read(6000000);type29=res29.headers.get('Content-Type','')
        assert len(data29)>20 and b'<html' not in data29[:1000].lower()
        v29_macro_bytes[name29]=data29
        suffix29='.zip' if name29=='ndc_history' else '.csv'
        (v29_root/('景氣來源稽核_'+name29+'_v29_2026-10-16'+suffix29)).write_bytes(data29)
        record29={'url':url29,'status':'downloaded_current_snapshot_not_vintage','bytes':len(data29),'sha256':hashlib.sha256(data29).hexdigest(),'content_type':type29}
        if suffix29=='.zip':record29['members']=zipfile.ZipFile(io.BytesIO(data29)).namelist()[:15]
        else:
            txt29=None
            for enc29 in ['utf-8-sig','cp950']:
                try:txt29=data29.decode(enc29);record29['encoding']=enc29;break
                except UnicodeError:pass
            if txt29 is not None:
                rr29=list(csv.reader(io.StringIO(txt29)));record29['rows']=len(rr29);record29['header']=rr29[0] if rr29 else [];record29['date_examples']=[r[0] for r in rr29[1:4] if r]
        v29_macro_audit[name29]=record29
    except Exception as err29:v29_macro_audit[name29]={'url':url29,'status':'unavailable','error_type':type(err29).__name__}
    print('V29_MACRO_SOURCE',name29,v29_macro_audit[name29]['status'],flush=True)
v29_macro_audit['release_archive']={'status':'not_reconstructed','website_attempts':'NDC direct page403; alternate official archive route403; open data found latest release only','no_fabricated_release_dates':True,'rule_revision':'2024-03-04 release of2024-01 switched method; cannot silently backfill latest system','primary_sources':['https://data.gov.tw/dataset/6099','https://data.gov.tw/dataset/6100','https://data.gov.tw/dataset/27543','https://data.gov.tw/dataset/27541']}
(v29_root/'市場環境分層_景氣來源稽核_v29_2026-10-16.json').write_text(json.dumps(v29_macro_audit,ensure_ascii=False,indent=2),encoding='utf-8')
v29_summary['macro_status']['source_audit']=v29_macro_audit
(v29_root/'市場環境分層_統計_v29_2026-10-16.json').write_text(json.dumps(v29_summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('V29_MACRO_AUDIT', {k:{a:b for a,b in v.items() if a in ['status','rows','members','date_examples']} for k,v in v29_macro_audit.items()})


In [ ]:
# v29 inspect official release archive metadata, no full texts output.
v29_release_rows={}
for name29 in ['ndc_release_latest','pmi_release_latest']:
    rr29=list(csv.DictReader(io.StringIO(v29_macro_bytes[name29].decode('utf-8-sig'))));v29_release_rows[name29]=rr29
    print('V29_RELEASE_METADATA',name29,{'header':list(rr29[0]),'n':len(rr29),'first':rr29[:1],'last':rr29[-1:]})


In [ ]:
# v29 fetch original public releases for observed sample; bounded network work only.
import concurrent.futures,re
v29_release_tasks=[]
for kind29,rows29 in v29_release_rows.items():
    for rr29 in rows29:
        when29=rr29['date(發布日期)']
        if '20231001'<=when29<='20261012':v29_release_tasks.append((kind29,rr29))
def fetch_release29(task29):
    kind29,rr29=task29;url29=rr29['url(網址)'];result29={'kind':kind29,**rr29}
    try:
        req29=urllib.request.Request(url29,headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req29,timeout=12) as response29:raw29=response29.read(2000000)
        txt29=raw29.decode('utf-8','replace');plain29=re.sub('<[^>]+>',' ',txt29);plain29=__import__('html').unescape(plain29);plain29=re.sub(r'\s+',' ',plain29)
        result29['status']='fetched';result29['text']=plain29;result29['links']=re.findall(r'href=["\x27]([^"\x27]+)',txt29);result29['sha256']=hashlib.sha256(raw29).hexdigest()
    except Exception as e29:result29['status']='unavailable';result29['error_type']=type(e29).__name__
    return result29
with concurrent.futures.ThreadPoolExecutor(max_workers=6) as pool29:
    v29_original_releases=list(pool29.map(fetch_release29,v29_release_tasks))
(v29_root/'景氣來源稽核_公告原文_v29_2026-10-16.json').write_text(json.dumps(v29_original_releases,ensure_ascii=False,indent=2),encoding='utf-8')
v29_probe=[]
for kind29 in v29_release_rows:
    items29=[r for r in v29_original_releases if r['kind']==kind29];ok29=[r for r in items29 if r['status']=='fetched']
    snippets29=[]
    for r29 in (ok29[:2]+ok29[-1:]):
        pattern29=r'.{0,25}(?:綜合判斷分數|綜合分數|製造業PMI|製造業採購經理人指數).{0,110}'
        snippets29.append({'date':r29['date(發布日期)'],'title':r29['title(標題)'],'snippets':re.findall(pattern29,r29['text'])[:5],'download_links':[u for u in r29['links'] if 'Download' in u or '.pdf' in u][:3]})
    v29_probe.append({'kind':kind29,'planned':len(items29),'fetched':len(ok29),'samples':snippets29})
print('V29_ORIGINAL_RELEASES',v29_probe)


In [ ]:
# v29 persist observational findings, full points and same-name master summary.
import plotly.graph_objects as go
from markdown_it import MarkdownIt
import re,html as html29
def pct29(x):return '未列' if x is None else f'{x*100:.2f}%'
lines29=['# 市場環境分層：動能與法人 v29','2026-10-16。保持原交易規則，只使用起點前已知的環境；引用既有回測，不重跑、不新增調整交易。','## 結論','波動分層出現值得追查的差異：低波動組法人平均較高，高波動組動能平均較高。但三種環境交互差的24／48日起點區塊、三比較校正區間全部跨0，尚不能據此切換策略。市場走強組平均報酬較高，也不代表勝率一定更高。','## 同日期與門檻',f'兩種策略共同有效{int(v29_common.sum())}／650窗口，與v17的511相同。特徵只取start-1以前20交易日。市場代理為目前150檔調整日報酬等權均值，每日至少120檔；廣度要求至少120檔完整20日資料。波動門檻只用2024年底前結束窗口的特徵中位數：每日標準差{pct29(v29_vol_threshold)}，未用收益挑閾值。重疊窗口／同一狀態不是獨立事件。','## 分層結果：費後24日','|分層|窗口|動能勝率|動能期望|法人勝率|法人期望|動能費稅|法人費稅|動能最差回撤|法人最差回撤|','|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for axis29,gs29 in v29_groups.items():
    for label29,g29 in gs29.items():
        a29=g29['methods']['MOM'];b29=g29['methods']['FLOW'];lines29.append('|'+ '|'.join([label29,str(g29['n']),pct29(a29['win']),pct29(a29['ev']),pct29(b29['win']),pct29(b29['ev']),pct29(a29['fee']),pct29(b29['fee']),pct29(a29['worst_mdd']),pct29(b29['worst_mdd'])])+'|')
lines29+=['## 是否足以在兩策略間切換？','交互差＝高狀態的（動能−法人）減低狀態的（動能−法人），不是直接測量景氣因果，也不是一個切換策略回測。單位pp；3000次24／48日原始位置循環區塊、三比較Bonferroni。','|分層|交互差|24日校正區間|48日校正區間|','|---|---:|---|---|']
for axis29,r29 in v29_interactions.items():
    ci29=lambda b:'['+', '.join(f'{v*100:+.3f}' for v in r29['family_ci'][str(b)])+']pp'
    lines29.append(f"|{axis29}|{r29['estimate']*100:+.3f}pp|{ci29(24)}|{ci29(48)}|")
lines29+=['## 年份檢查','以下為起始年分組的24日窗口均值，不是年報酬。相同狀態可能集中於少數年份，不能把全期排序直接當穩定選擇。','|狀態|起始年|窗口|動能勝率／期望|法人勝率／期望|','|---|---|---:|---|---|']
for gs29 in v29_groups.values():
    for label29,g29 in gs29.items():
        for year29,ms29 in g29['years'].items():
            if ms29['MOM']['n']:
                a29=ms29['MOM'];b29=ms29['FLOW'];lines29.append(f"|{label29}|{year29}|{a29['n']}|{pct29(a29['win'])}／{pct29(a29['ev'])}|{pct29(b29['win'])}／{pct29(b29['ev'])}|")
lines29+=['## 相對市場的檢查與限制','目前僅建立同150檔日等權報酬代理，90%持有加上一次建倉費的參考路徑；每日成分隨已知資料而變，未計隱含成分變更交易成本，並非官方指數或可實作投資組合。下面只是描述性差額，不能據此聲稱策略alpha。須補正式總報酬指數或同股票池可執行持有對照才能驗證。','|策略|費後期望|減研究代理期望差|高於代理窗口比例|','|---|---:|---:|---:|']
for k29,m29 in v29_summary['overall'].items():lines29.append(f"|{k29}|{pct29(m29['ev'])}|{pct29(m29['proxy_relative_ev'])}|{pct29(m29['beat_proxy'])}|")
lines29+=['## 景氣燈號與PMI：尚未完成分層','已下載政府開放資料的歷史燈號ZIP、PMI CSV與新聞稿清單，下載摘要另存稽核JSON。新聞稿清單實際涵蓋多期：燈號103筆、PMI104筆，均從2018年起，不僅最新一期。清單有公布日期與原始網址；仍沒有逐期原始值可核驗，也未確認每期具體公布時間。對研究期36份燈號、37份PMI原文嘗試讀取，73份均未成功，錯誤類型另存原文稽核；官方網頁檢索亦回403。因此缺口是原始值及時間核驗，不是完全沒有日期清單。2024年3月4日發布2024年1月資料時啟用新版，不能用新版回填序列當舊時點可得。沒有猜日期、沒有把月底資料放月初，沒有將目前修訂值套入回測。此分支保留未完成，尚無宏觀切換績效。','官方來源：[燈號開放資料](https://data.gov.tw/dataset/6099)、[PMI開放資料](https://data.gov.tw/dataset/6100)、[景氣新聞稿清單](https://data.gov.tw/dataset/27543)、[PMI新聞稿清單](https://data.gov.tw/dataset/27541)、[2024改版說明](https://www.ndc.gov.tw/News_Content.aspx?n=CFE1B745364D2499&s=9B95A3EDC8B036A7&sms=814775938D6333B4)。查閱2026-10-16。','## 適用範圍與下一步','這輪只有市場環境觀察，尚未改初始選股、倉位或再平衡。股票池存活偏誤、未知公司行動、資料已被反覆查看、價格環境與動能重疊、狀態事件少均限制外推。更高波動組較好不代表應主動承擔更高風險。後續先補官方總報酬基準與首次公告檔案，再研究跨年穩定的條件，不以這輪最高均值選策略。','## 保存與驗證','原Notebook cells311–317接續，沿用同專案現有Python314 kernel PID36240；舊PID2524已結束，未重啟或替換任何kernel。沿用v28已恢復價量及v16原結果，大型物件留kernel。驗證511共同分母、特徵時點早於起始日、每軸分組數相加、同日期配對與全部繪圖點數；沒有Restart and Run All。']
v29_report='\\n'.join(lines29).replace('\\n','\n')
v29_reportpath=v29_root/'市場環境分層_動能法人研究_v29_2026-10-16.md';v29_reportpath.write_text(v29_report,encoding='utf-8')
v29_fragments=[];v29_plotcount=0;v29_plotrng=np.random.default_rng(29)
for axis29,state29 in v29_state.items():
    fig29=go.Figure();ticks29=[];names29=[]
    for high29 in [False,True]:
        mask29=v29_common&(state29==high29);ids29=np.flatnonzero(mask29)
        for ki29,k29 in enumerate(['MOM','FLOW']):
            pos29=int(high29)*2+ki29;yy29=v29_ret[k29][mask29]*100;color29=['#167368','#567bb5'][ki29];name29=v29_labels[axis29][int(high29)]+('／動能' if k29=='MOM' else '／法人');ticks29.append(pos29);names29.append(name29)
            fig29.add_trace(go.Box(x=np.full(len(yy29),pos29),y=yy29,boxpoints=False,line={'color':color29},fillcolor='rgba(22,115,104,.10)',width=.55,hoverinfo='skip',showlegend=False))
            fig29.add_trace(go.Scatter(x=pos29+v29_plotrng.uniform(-.19,.19,len(yy29)),y=yy29,mode='markers',marker={'size':5,'opacity':.45,'color':color29},customdata=[[v29_points[i]['start'],v29_points[i]['end']] for i in ids29],hovertemplate=name29+'<br>%{customdata[0]}→%{customdata[1]}<br>費後24日 %{y:.2f}%<extra></extra>',showlegend=False))
            med29=float(np.median(yy29));fig29.add_trace(go.Scatter(x=[pos29-.24,pos29+.24],y=[med29,med29],mode='lines',line={'color':'#c43c79','width':4},hoverinfo='skip',showlegend=False));v29_plotcount+=len(yy29)
    fig29.add_hline(y=0,line_color='#173333',line_width=2);fig29.update_layout(title=axis29+'：每個窗口與桃紅中位數',template='plotly_white',height=510,xaxis={'tickmode':'array','tickvals':ticks29,'ticktext':names29},yaxis={'title':'費後24日報酬 %','range':[-35,45]},margin={'b':110})
    v29_fragments.append(fig29.to_html(full_html=False,include_plotlyjs=(axis29=='trend')))
assert v29_plotcount==511*2*3
md29=MarkdownIt('commonmark',{'html':False}).enable('table')
v29_htmlpath=v29_root/'市場環境分層_動能法人全部點_v29_2026-10-16.html'
v29_htmlpath.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>市場環境分層 v29</title><style>body{font:16px/1.8 system-ui;max-width:1250px;margin:30px auto;padding:0 20px;color:#173333}table{border-collapse:collapse;width:100%;font-size:14px}td,th{padding:8px;border-bottom:1px solid #ccd8d3;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}.note{background:#edf3ed;padding:16px}</style><h1>市場環境：動能與法人</h1><p class="note">三種環境均使用相同511窗口。分組在起點前決定；顯示全部3066個點（同一窗口在三種分層各顯示），不是3066次獨立比賽。桃紅中位數、0線與三圖相同刻度。波動分層有線索，校正區間仍跨0，不能直接切換策略。</p><a href="策略研究_總結報告.html">回總結報告</a>'+''.join(v29_fragments)+md29.render(v29_report).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',encoding='utf-8')
v29_insert="""## 新增證據：市場環境分層 v29

2026-10-16完成市場趨勢／廣度／波動的起點前分層，原動能與法人交易不變，同511窗口。低波動99窗口：動能勝率56.57%、期望+1.85%，法人67.68%、+3.02%；高波動412窗口：動能73.54%、+5.49%，法人81.55%、+3.54%。動能費稅低／高0.79／0.80%，法人0.71／0.73%；動能最差回撤-16.11／-22.49%，法人-8.75／-19.81%。這是待驗證線索；三種分層交互差的24／48日三比較校正區間全跨0，不能直接採用策略切換。

市場上漲組收益均值較高，動能勝率卻未較高，不能把「順風」等同必勝。市場參考僅為目前150檔變動成分研究代理，不是官方可投資總報酬基準，未證明alpha。景氣燈號／PMI已取得歷史數列與多期公告日期清單；73份原始公告未成功取得，尚缺原始值及發布時間核驗，未完成宏觀分層或策略。

[市場環境報告](市場環境分層_動能法人研究_v29_2026-10-16.md) · [全部窗口互動圖](市場環境分層_動能法人全部點_v29_2026-10-16.html)

"""
master29=v29_root/'策略研究_總結報告.md';text29=master29.read_text(encoding='utf-8');text29=text29.replace('目前回測證據截至 v27','目前總結的主線交易回測截至 v27，新增v29環境觀察；另有v28反轉分支待跨家族整合')
if '## 新增證據：市場環境分層 v29' not in text29:text29=text29.replace('## 全體限制與未完成項',v29_insert+'## 全體限制與未完成項')
text29=text29.replace('## 更新紀錄','## 更新紀錄\n\n- 2026-10-16：新增v29市場環境觀察，511共同窗口；波動有分層線索，交互差校正未通過。燈號／PMI首次公布檔案未完成，未偽稱宏觀回測。')
master29.write_text(text29,encoding='utf-8')
masterhtml29=v29_root/'策略研究_總結報告.html';page29=masterhtml29.read_text(encoding='utf-8');full29=md29.render(text29).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')
page29,changes29=re.subn(r'(<article id="reportBody">).*?(</article>)',lambda m:m[1]+full29+m[2],page29,flags=re.S);assert changes29==1
if 'id="regime-results"' not in page29:
    teaser29='<section id="regime-results"><h2>新增：市場環境分層 v29</h2><p>同511窗口，原動能與法人交易不變。低波動時法人均值較高，高波動時動能均值較高；三種交互差校正區間仍跨0，尚不能切換策略。</p><a href="市場環境分層_動能法人全部點_v29_2026-10-16.html">查看三種分層、全部窗口點與逐年結果 →</a><p class="muted">景氣燈號／PMI仍在首次公布來源稽核階段，尚無宏觀回測結果。</p></section>'
    page29=page29.replace('<section id="macro"',teaser29+'<section id="macro"');page29=page29.replace('<a href="#macro">','<a href="#regime-results">市場環境新證據</a><a href="#macro">')
page29=page29.replace('回測截至 v27','交易回測v28另有反轉分支 · 新增v29環境觀察')
masterhtml29.write_text(page29,encoding='utf-8')
statepath29=v29_root/'A策略_無人值守研究狀態.json';state29=json.loads(statepath29.read_text(encoding='utf-8'));state29['environment_v29']={'status':'market_observation_completed_macro_first_release_incomplete','running':False,'common_n':511,'new_strategy_runs':0,'kernel_pid':os.getpid(),'notebook_cells':[311,315],'interactions_passed':False,'outputs':[v29_reportpath.name,v29_htmlpath.name],'macro_blocker':'first-release historical archive unavailable; latest sequence not substituted','next':'official totalreturn benchmark and firstrelease archive; no strategy switching authorized by evidence'};statepath29.write_text(json.dumps(state29,ensure_ascii=False,indent=2),encoding='utf-8')
print('V29_DELIVERED',{'report':v29_reportpath.name,'html':v29_htmlpath.name,'points':v29_plotcount,'summary_updated':True,'new_trading_rules':False,'macro_completed':False})


In [ ]:
# v29 persist observational findings, full points and same-name master summary.
import plotly.graph_objects as go
from markdown_it import MarkdownIt
import re,html as html29
def pct29(x):return '未列' if x is None else f'{x*100:.2f}%'
lines29=['# 市場環境分層：動能與法人 v29','2026-10-16。保持原交易規則，只使用起點前已知的環境；引用既有回測，不重跑、不新增調整交易。','## 結論','波動分層出現值得追查的差異：低波動組法人平均較高，高波動組動能平均較高。但三種環境交互差的24／48日起點區塊、三比較校正區間全部跨0，尚不能據此切換策略。市場走強組平均報酬較高，也不代表勝率一定更高。','## 同日期與門檻',f'兩種策略共同有效{int(v29_common.sum())}／650窗口，與v17的511相同。特徵只取start-1以前20交易日。市場代理為目前150檔調整日報酬等權均值，每日至少120檔；廣度要求至少120檔完整20日資料。波動門檻只用2024年底前結束窗口的特徵中位數：每日標準差{pct29(v29_vol_threshold)}，未用收益挑閾值。重疊窗口／同一狀態不是獨立事件。','## 分層結果：費後24日','|分層|窗口|動能勝率|動能期望|法人勝率|法人期望|動能費稅|法人費稅|動能最差回撤|法人最差回撤|','|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for axis29,gs29 in v29_groups.items():
    for label29,g29 in gs29.items():
        a29=g29['methods']['MOM'];b29=g29['methods']['FLOW'];lines29.append('|'+ '|'.join([label29,str(g29['n']),pct29(a29['win']),pct29(a29['ev']),pct29(b29['win']),pct29(b29['ev']),pct29(a29['fee']),pct29(b29['fee']),pct29(a29['worst_mdd']),pct29(b29['worst_mdd'])])+'|')
lines29+=['## 是否足以在兩策略間切換？','交互差＝高狀態的（動能−法人）減低狀態的（動能−法人），不是直接測量景氣因果，也不是一個切換策略回測。單位pp；3000次24／48日原始位置循環區塊、三比較Bonferroni。','|分層|交互差|24日校正區間|48日校正區間|','|---|---:|---|---|']
for axis29,r29 in v29_interactions.items():
    ci29=lambda b:'['+', '.join(f'{v*100:+.3f}' for v in r29['family_ci'][str(b)])+']pp'
    lines29.append(f"|{axis29}|{r29['estimate']*100:+.3f}pp|{ci29(24)}|{ci29(48)}|")
lines29+=['## 年份檢查','以下為起始年分組的24日窗口均值，不是年報酬。相同狀態可能集中於少數年份，不能把全期排序直接當穩定選擇。','|狀態|起始年|窗口|動能勝率／期望|法人勝率／期望|','|---|---|---:|---|---|']
for gs29 in v29_groups.values():
    for label29,g29 in gs29.items():
        for year29,ms29 in g29['years'].items():
            if ms29['MOM']['n']:
                a29=ms29['MOM'];b29=ms29['FLOW'];lines29.append(f"|{label29}|{year29}|{a29['n']}|{pct29(a29['win'])}／{pct29(a29['ev'])}|{pct29(b29['win'])}／{pct29(b29['ev'])}|")
lines29+=['## 相對市場的檢查與限制','目前僅建立同150檔日等權報酬代理，90%持有加上一次建倉費的參考路徑；每日成分隨已知資料而變，未計隱含成分變更交易成本，並非官方指數或可實作投資組合。下面只是描述性差額，不能據此聲稱策略alpha。須補正式總報酬指數或同股票池可執行持有對照才能驗證。','|策略|費後期望|減研究代理期望差|高於代理窗口比例|','|---|---:|---:|---:|']
for k29,m29 in v29_summary['overall'].items():lines29.append(f"|{k29}|{pct29(m29['ev'])}|{pct29(m29['proxy_relative_ev'])}|{pct29(m29['beat_proxy'])}|")
lines29+=['## 景氣燈號與PMI：尚未完成分層','已下載政府開放資料的歷史燈號ZIP、PMI CSV與新聞稿清單，下載摘要另存稽核JSON。新聞稿清單實際涵蓋多期：燈號103筆、PMI104筆，均從2018年起，不僅最新一期。清單有公布日期與原始網址；仍沒有逐期原始值可核驗，也未確認每期具體公布時間。對研究期36份燈號、37份PMI原文嘗試讀取，73份均未成功，錯誤類型另存原文稽核；官方網頁檢索亦回403。因此缺口是原始值及時間核驗，不是完全沒有日期清單。2024年3月4日發布2024年1月資料時啟用新版，不能用新版回填序列當舊時點可得。沒有猜日期、沒有把月底資料放月初，沒有將目前修訂值套入回測。此分支保留未完成，尚無宏觀切換績效。','官方來源：[燈號開放資料](https://data.gov.tw/dataset/6099)、[PMI開放資料](https://data.gov.tw/dataset/6100)、[景氣新聞稿清單](https://data.gov.tw/dataset/27543)、[PMI新聞稿清單](https://data.gov.tw/dataset/27541)、[2024改版說明](https://www.ndc.gov.tw/News_Content.aspx?n=CFE1B745364D2499&s=9B95A3EDC8B036A7&sms=814775938D6333B4)。查閱2026-10-16。','## 適用範圍與下一步','這輪只有市場環境觀察，尚未改初始選股、倉位或再平衡。股票池存活偏誤、未知公司行動、資料已被反覆查看、價格環境與動能重疊、狀態事件少均限制外推。更高波動組較好不代表應主動承擔更高風險。後續先補官方總報酬基準與首次公告檔案，再研究跨年穩定的條件，不以這輪最高均值選策略。','## 保存與驗證','原Notebook cells311–317接續，沿用同專案現有Python314 kernel PID36240；舊PID2524已結束，未重啟或替換任何kernel。沿用v28已恢復價量及v16原結果，大型物件留kernel。驗證511共同分母、特徵時點早於起始日、每軸分組數相加、同日期配對與全部繪圖點數；沒有Restart and Run All。']
v29_report='\\n'.join(lines29).replace('\\n','\n')
v29_reportpath=v29_root/'市場環境分層_動能法人研究_v29_2026-10-16.md';v29_reportpath.write_text(v29_report,encoding='utf-8')
v29_fragments=[];v29_plotcount=0;v29_plotrng=np.random.default_rng(29)
for axis29,state29 in v29_state.items():
    fig29=go.Figure();ticks29=[];names29=[]
    for high29 in [False,True]:
        mask29=v29_common&(state29==high29);ids29=np.flatnonzero(mask29)
        for ki29,k29 in enumerate(['MOM','FLOW']):
            pos29=int(high29)*2+ki29;yy29=v29_ret[k29][mask29]*100;color29=['#167368','#567bb5'][ki29];name29=v29_labels[axis29][int(high29)]+('／動能' if k29=='MOM' else '／法人');ticks29.append(pos29);names29.append(name29)
            fig29.add_trace(go.Box(x=np.full(len(yy29),pos29),y=yy29,boxpoints=False,line={'color':color29},fillcolor='rgba(22,115,104,.10)',width=.55,hoverinfo='skip',showlegend=False))
            fig29.add_trace(go.Scatter(x=pos29+v29_plotrng.uniform(-.19,.19,len(yy29)),y=yy29,mode='markers',marker={'size':5,'opacity':.45,'color':color29},customdata=[[v29_points[i]['start'],v29_points[i]['end']] for i in ids29],hovertemplate=name29+'<br>%{customdata[0]}→%{customdata[1]}<br>費後24日 %{y:.2f}%<extra></extra>',showlegend=False))
            med29=float(np.median(yy29));fig29.add_trace(go.Scatter(x=[pos29-.24,pos29+.24],y=[med29,med29],mode='lines',line={'color':'#c43c79','width':4},hoverinfo='skip',showlegend=False));v29_plotcount+=len(yy29)
    fig29.add_hline(y=0,line_color='#173333',line_width=2);fig29.update_layout(title=axis29+'：每個窗口與桃紅中位數',template='plotly_white',height=510,xaxis={'tickmode':'array','tickvals':ticks29,'ticktext':names29},yaxis={'title':'費後24日報酬 %','range':[float(min(v29_ret[k][v29_common].min() for k in v29_rows))*100-3,float(max(v29_ret[k][v29_common].max() for k in v29_rows))*100+3]},margin={'b':110})
    v29_fragments.append(fig29.to_html(full_html=False,include_plotlyjs=(axis29=='trend')))
assert v29_plotcount==511*2*3
md29=MarkdownIt('commonmark',{'html':False}).enable('table')
v29_htmlpath=v29_root/'市場環境分層_動能法人全部點_v29_2026-10-16.html'
v29_htmlpath.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>市場環境分層 v29</title><style>body{font:16px/1.8 system-ui;max-width:1250px;margin:30px auto;padding:0 20px;color:#173333}table{border-collapse:collapse;width:100%;font-size:14px}td,th{padding:8px;border-bottom:1px solid #ccd8d3;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}.note{background:#edf3ed;padding:16px}</style><h1>市場環境：動能與法人</h1><p class="note">三種環境均使用相同511窗口。分組在起點前決定；顯示全部3066個點（同一窗口在三種分層各顯示），不是3066次獨立比賽。桃紅中位數、0線與三圖相同刻度。波動分層有線索，校正區間仍跨0，不能直接切換策略。</p><a href="策略研究_總結報告.html">回總結報告</a>'+''.join(v29_fragments)+md29.render(v29_report).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',encoding='utf-8')
v29_insert="""## 新增證據：市場環境分層 v29

2026-10-16完成市場趨勢／廣度／波動的起點前分層，原動能與法人交易不變，同511窗口。低波動99窗口：動能勝率56.57%、期望+1.85%，法人67.68%、+3.02%；高波動412窗口：動能73.54%、+5.49%，法人81.55%、+3.54%。動能費稅低／高0.79／0.80%，法人0.71／0.73%；動能最差回撤-16.11／-22.49%，法人-8.75／-19.81%。這是待驗證線索；三種分層交互差的24／48日三比較校正區間全跨0，不能直接採用策略切換。

市場上漲組收益均值較高，動能勝率卻未較高，不能把「順風」等同必勝。市場參考僅為目前150檔變動成分研究代理，不是官方可投資總報酬基準，未證明alpha。景氣燈號／PMI已取得歷史數列與多期公告日期清單；73份原始公告未成功取得，尚缺原始值及發布時間核驗，未完成宏觀分層或策略。

[市場環境報告](市場環境分層_動能法人研究_v29_2026-10-16.md) · [全部窗口互動圖](市場環境分層_動能法人全部點_v29_2026-10-16.html)

"""
master29=v29_root/'策略研究_總結報告.md';text29=master29.read_text(encoding='utf-8');text29=text29.replace('目前回測證據截至 v27','目前總結的主線交易回測截至 v27，新增v29環境觀察；另有v28反轉分支待跨家族整合')
if '## 新增證據：市場環境分層 v29' not in text29:text29=text29.replace('## 全體限制與未完成項',v29_insert+'## 全體限制與未完成項')
text29=text29.replace('## 更新紀錄','## 更新紀錄\n\n- 2026-10-16：新增v29市場環境觀察，511共同窗口；波動有分層線索，交互差校正未通過。燈號／PMI首次公布檔案未完成，未偽稱宏觀回測。')
master29.write_text(text29,encoding='utf-8')
masterhtml29=v29_root/'策略研究_總結報告.html';page29=masterhtml29.read_text(encoding='utf-8');full29=md29.render(text29).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')
page29,changes29=re.subn(r'(<article id="reportBody">).*?(</article>)',lambda m:m[1]+full29+m[2],page29,flags=re.S);assert changes29==1
if 'id="regime-results"' not in page29:
    teaser29='<section id="regime-results"><h2>新增：市場環境分層 v29</h2><p>同511窗口，原動能與法人交易不變。低波動時法人均值較高，高波動時動能均值較高；三種交互差校正區間仍跨0，尚不能切換策略。</p><a href="市場環境分層_動能法人全部點_v29_2026-10-16.html">查看三種分層、全部窗口點與逐年結果 →</a><p class="muted">景氣燈號／PMI仍在首次公布來源稽核階段，尚無宏觀回測結果。</p></section>'
    page29=page29.replace('<section id="macro"',teaser29+'<section id="macro"');page29=page29.replace('<a href="#macro">','<a href="#regime-results">市場環境新證據</a><a href="#macro">')
page29=page29.replace('回測截至 v27','交易回測v28另有反轉分支 · 新增v29環境觀察')
masterhtml29.write_text(page29,encoding='utf-8')
statepath29=v29_root/'A策略_無人值守研究狀態.json';state29=json.loads(statepath29.read_text(encoding='utf-8'));state29['environment_v29']={'status':'market_observation_completed_macro_first_release_incomplete','running':False,'common_n':511,'new_strategy_runs':0,'kernel_pid':os.getpid(),'notebook_cells':[311,315],'interactions_passed':False,'outputs':[v29_reportpath.name,v29_htmlpath.name],'macro_blocker':'first-release historical archive unavailable; latest sequence not substituted','next':'official totalreturn benchmark and firstrelease archive; no strategy switching authorized by evidence'};statepath29.write_text(json.dumps(state29,ensure_ascii=False,indent=2),encoding='utf-8')
print('V29_DELIVERED',{'report':v29_reportpath.name,'html':v29_htmlpath.name,'points':v29_plotcount,'summary_updated':True,'new_trading_rules':False,'macro_completed':False})


In [ ]:
# v29 finalize source audit and persisted result quality; no new statistical tests.
v29_macro_audit['release_archive'].update({'status':'dates_and_urls_obtained_original_values_incomplete','ndc_rows':len(v29_release_rows['ndc_release_latest']),'pmi_rows':len(v29_release_rows['pmi_release_latest']),'original_attempted':len(v29_original_releases),'original_success':sum(r['status']=='fetched' for r in v29_original_releases),'errors':dict(collections.Counter(r.get('error_type') for r in v29_original_releases if r['status']!='fetched'))})
v29_summary['macro_status']['source_audit']=v29_macro_audit
v29_summary['macro_status']['PMI']='current history and release dates obtained; original release values and exact time not verified'
v29_summary['verification']={'shared_n_matches_v17':511,'plot_markers':3066,'classification_cutoff_before_start':True,'threshold_return_blind_training':True,'new_strategy_runs':0,'kernel_restarted':False,'macro_release_dates_available':True,'macro_first_values_verified':False,'benchmark_executable':False,'statistic_scope':'exploratory environment interaction, not causal or strategy-switch backtest'}
for name29,data29 in [('市場環境分層_景氣來源稽核_v29_2026-10-16.json',v29_macro_audit),('市場環境分層_統計_v29_2026-10-16.json',v29_summary)]:
    (v29_root/name29).write_text(json.dumps(data29,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
for path29 in [v29_root/'策略研究_總結報告.md',v29_root/'策略研究_總結報告.html']:
    text29=path29.read_text(encoding='utf-8');text29=text29.replace('## 更新紀錄'+chr(92)+'n'+chr(92)+'n','## 更新紀錄\n\n');text29=text29.replace('偽稱宏觀回測。'+chr(92)+'n','偽稱宏觀回測。\n');path29.write_text(text29,encoding='utf-8')
v29_st=json.loads(statepath29.read_text(encoding='utf-8'));v29_st['environment_v29']['notebook_cells']=[311,319];v29_st['environment_v29']['macro_blocker']='date/url archive available;73 original pages failed; first-release values/time not verified';v29_st['environment_v29']['outputs'] += ['市場環境分層_統計_v29_2026-10-16.json','市場環境分層_景氣來源稽核_v29_2026-10-16.json'];statepath29.write_text(json.dumps(v29_st,ensure_ascii=False,indent=2),encoding='utf-8')
text29=master29.read_text(encoding='utf-8');text29=text29.replace(chr(92)+'n',chr(10));master29.write_text(text29,encoding='utf-8');page29=masterhtml29.read_text(encoding='utf-8');full29=md29.render(text29).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>');page29=re.sub(r'(<article id="reportBody">).*?(</article>)',lambda m:m[1]+full29+m[2],page29,flags=re.S);masterhtml29.write_text(page29,encoding='utf-8')
print('V29_QUALITY',{'macro_errors':v29_macro_audit['release_archive']['errors'],'low_vol_years':{y:{k:{f:m.get(f) for f in ['n','win','ev']} for k,m in ms.items()} for y,ms in v29_groups['volatility']['波動較低']['years'].items()},'overall':v29_summary['overall']})


In [ ]:
# v29 distinguish year confounding and source transport errors before interpretation.
note29='重要：低波動99窗口中95個來自2024年、4個來自2025年、2026年為0；因此波動差異與年份高度混雜，尚無跨年穩定支持，不能解讀為低波動必選法人、高波動必選動能。'
report29=v29_reportpath.read_text(encoding='utf-8');report29=report29.replace('## 年份檢查','## 年份檢查\n\n'+note29);report29=report29.replace('73份均未成功，錯誤類型另存原文稽核；官方網頁檢索亦回403','73份均未成功：Python對原文主站的TLS憑證驗證失敗（URLError），未關閉驗證；以Windows原生信任環境查一份代表原文仍回403，官方網頁檢索亦回403。錯誤類型另存原文稽核');report29=report29.replace('cells311–317','cells311–320');v29_reportpath.write_text(report29,encoding='utf-8')
htmlpath29=v29_htmlpath.read_text(encoding='utf-8');htmlpath29=re.sub(r'<h1>市場環境分層：動能與法人 v29</h1>.*?</html>',lambda m:md29.render(report29).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',htmlpath29,flags=re.S);v29_htmlpath.write_text(htmlpath29,encoding='utf-8')
mastertext29=master29.read_text(encoding='utf-8');mastertext29=mastertext29.replace('這是待驗證線索；三種分層交互差',note29+' 這是待驗證線索；三種分層交互差');master29.write_text(mastertext29,encoding='utf-8')
masterpage29=masterhtml29.read_text(encoding='utf-8');masterpage29=re.sub(r'(<article id="reportBody">).*?(</article>)',lambda m:m[1]+md29.render(mastertext29).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+m[2],masterpage29,flags=re.S);masterpage29=masterpage29.replace('尚不能切換策略。</p>','尚不能切換策略。低波動99窗口中95個來自2024年，跨年證據不足。</p>');masterhtml29.write_text(masterpage29,encoding='utf-8')
v29_summary['limitations_addendum']={'low_vol_start_years':{'2024':95,'2025':4,'2026':0},'regime_year_confounded':True}
v29_macro_audit['release_archive']['python_original_transport_error']='TLS CERTIFICATE_VERIFY_FAILED Missing Subject Key Identifier; verification not disabled'
v29_macro_audit['release_archive']['windows_native_verified_request']='representative original nc_332_40495:403'
v29_summary['macro_status']['source_audit']=v29_macro_audit
for name29,data29 in [('市場環境分層_景氣來源稽核_v29_2026-10-16.json',v29_macro_audit),('市場環境分層_統計_v29_2026-10-16.json',v29_summary)]:
    (v29_root/name29).write_text(json.dumps(data29,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
state29=json.loads(statepath29.read_text(encoding='utf-8'));state29['environment_v29']['notebook_cells']=[311,320];state29['environment_v29']['macro_blocker']='first values unverified;PythonTLS error;native representative403;dates archive available';statepath29.write_text(json.dumps(state29,ensure_ascii=False,indent=2),encoding='utf-8')
print('V29_FINAL_NOTE_SAVED',{'lowvol_year_confounded':True,'source_errors_distinguished':True,'html_exists':v29_htmlpath.exists(),'all_points':3066})


In [ ]:
# v30 restore only cached long-history prices in a separate namespace; no kernel reset.
import bisect,re,urllib.parse,urllib.request,collections,ast,gzip
v30_root=v28_root;v30_book=nbformat.read(v30_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
v30_definition={'version':'v30_long_environment','years':[2016,2025],'strategies':'20/60 half momentum vs pure institutional20 netbuy/volume; same25,equal90%,5day,24days,lots1000 and old fees','stage':'data_restore_and_flow_feasibility','reuse':'v27 P90 momentum results; reconstruct adjustedreturn arrays for regime and prospective flow only','pool':'current150 with conservative current-market listeddate gate,not historical official150','cutoff':'start-1;20day features','volatility_threshold':'training feature median using windows ending<=2018-12-31;2019..2025 later retrospective not unseen;also oldv29 threshold sensitivity','coverage_change_predeclared':'main >=80% of then-listed current150 with floor50;fixed120 v29 rule sensitivity;needed to avoid silently counting prelisting quotes','unknown_actions':'samev27 badflags and knownpurecash repairs;no fabricated stockdiv/capitalaction','flow':'must acquire adequate complete150 crosssection;missing not zero;no fabricated earlier teamDB;no purchases/access bypass','no_new_timing_strategy':True,'no_macro_claim':'economic lights/PMI not inferred from prices','not_full_recent_A':'earlyinstitutionalqualitygate unavailable in v27 price-only momentum'}
(v30_root/'長歷史環境研究_預先定義_v30_2026-10-16.json').write_text(json.dumps(v30_definition,ensure_ascii=False,indent=2),encoding='utf-8')
def v30_load(name):
    with gzip.open(v30_root/name,'rt',encoding='utf-8') as f:return json.load(f)
v30_old=v30_load('外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz');v30_tpex=v30_load('完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz')
v30_history={k:{**v30_old[k],**v30_tpex[k]} for k in ['prices','dividends','metadata','errors']};v30_history['global_events']=v30_old['global_events']
v30_universe=json.loads((v30_root/'完整150檔研究_預先定義_v13_2026-10-15.json').read_text(encoding='utf-8'))['current_universe']
v30_ids=sorted(v30_history['prices']);v30_days=sorted({datetime.date.fromisoformat(r['date']) for r in v30_history['prices']['2330']})
v30_calendar=json.loads((v30_root/'產業競賽期間_官方指數來源_2026-10-14.json').read_text(encoding='utf-8'))
def roc30(value):
    y,m,d=map(int,re.findall(r'\d+',str(value))[:3]);return datetime.date(y+1911 if y<1911 else y,m,d)
v30_official=sorted({roc30(row[0]) for doc in v30_calendar['months'].values() if 'rows' in doc for row in doc['rows']})
assert not(set(v30_days)-set(v30_official))
v30_ns={'np':np,'datetime':datetime,'bisect':bisect,'collections':collections,'re':re,'provider_history_v13':v30_history,'provider_tickers_v13':v30_ids,'price_lookup_v13':{s:{r['date']:r for r in rows} for s,rows in v30_history['prices'].items()},'calendar_days_v13':v30_official}
num30=next(n for n in ast.parse(v30_book.cells[232].source).body if isinstance(n,ast.FunctionDef) and n.name=='number_v12');exec(compile(ast.Module(body=[num30],type_ignores=[]),'<saved-number>','exec'),v30_ns);v30_ns['number_v13']=v30_ns['number_v12']
source30=v30_book.cells[243].source;cash30='all_external_ids_v13='+source30.split('all_external_ids_v13=',1)[1].split('def build_case_v13(',1)[0];exec(cash30,v30_ns)
builder30=next(n for n in ast.parse(source30).body if isinstance(n,ast.FunctionDef) and n.name=='build_case_v13');exec(compile(ast.Module(body=[builder30],type_ignores=[]),'<saved-builder>','exec'),v30_ns)
v30_arrays=v30_ns['build_case_v13']({'dates':v30_days});v30_audit27=json.loads((v30_root/'動能_長歷史來源修正稽核_v27_2026-10-15.json').read_text(encoding='utf-8'))
v30_di={str(d):i for i,d in enumerate(v30_days)};v30_si={s:i for i,s in enumerate(v30_ids)}
for r in v30_audit27['repairs']:v30_arrays['bad'][v30_di[r['date']],v30_si[r['stock']]]=False
v30_arrays['returns'][:]=np.nan
for t30 in range(1,len(v30_days)):
    prev30=v30_arrays['close'][t30-1];now30=v30_arrays['close'][t30];ok30=np.isfinite(prev30)&np.isfinite(now30)&(prev30>0)&(now30>0);v30_arrays['returns'][t30,ok30]=(now30[ok30]+v30_arrays['cashdiv'][t30,ok30])/prev30[ok30]-1
v30_listing={str(r[0]):datetime.date.fromisoformat(str(r[1])) for r in raw_data['stocks'] if len(r)>1}
v30_listed=np.array([v30_listing.get(s,datetime.date.max) for s in v30_ids],dtype=object);v30_pool=np.array([s in set(v30_universe) for s in v30_ids])
v30_export_path=v30_root/'動能_2016至2025現金比例逐窗口_v26_2027-10-15.json'
v30_export=json.loads(v30_export_path.read_text(encoding='utf-8'));assert v30_export['definition']['version']=='v27_sourcequality_cashdiv_and_listing';print('V30_EXPORT_SCHEMA',{'keys':list(v30_export),'verified_internal_version':v30_export['definition']['version'],'legacy_filename_mismatch':v30_export_path.name})
print('V30_RESTORED',{'sessions':len(v30_days),'dates':(str(v30_days[0]),str(v30_days[-1])),'tickers':len(v30_ids),'knowncash_repairs':len(v30_audit27['repairs']),'listed_current150_first':int(np.sum(v30_pool&(v30_listed<=v30_days[0]))),'listing_current150_last':int(np.sum(v30_pool&(v30_listed<=v30_days[-1]))),'old_recent_objects_preserved':signal_returns_v15.shape})


In [ ]:
# v30 source-level feasibility, no token use or private credential output.
v30_flow_probe={};v30_flow_cache={}
v30_request={'dataset':'TaiwanStockInstitutionalInvestorsBuySell','data_id':'2330','start_date':'2016-07-01','end_date':'2025-12-31'}
v30_url='https://api.finmindtrade.com/api/v4/data?'+urllib.parse.urlencode(v30_request)
try:
    with urllib.request.urlopen(v30_url,timeout=25) as r30:v30_bytes=r30.read(12000000)
    v30_doc=json.loads(v30_bytes);v30_flow_probe={'source':'FinMind','status_code':v30_doc.get('status'),'message':str(v30_doc.get('msg',''))[:180],'rows':len(v30_doc.get('data',[])),'requires_private_credentials':False}
    if v30_doc.get('status')==200 and v30_doc.get('data'):
        v30_flow_cache['2330']=v30_doc['data'];v30_flow_probe.update({'first':min(r['date'] for r in v30_flow_cache['2330']),'last':max(r['date'] for r in v30_flow_cache['2330']),'fields':list(v30_flow_cache['2330'][0]),'categories':sorted({r['name'] for r in v30_flow_cache['2330']})})
except Exception as e30:v30_flow_probe={'source':'FinMind','error_type':type(e30).__name__,'reason':str(e30)[:180]}
(v30_root/'長歷史環境研究_法人資料查核_v30_2026-10-16.json').write_text(json.dumps(v30_flow_probe,ensure_ascii=False,indent=2),encoding='utf-8')
print('V30_FLOW_PROBE',v30_flow_probe)


In [ ]:
# v30 public provider one query per stock, preserve successful snapshots; stop at denial/rate limit.
import time
v30_flow_path=v30_root/'長歷史法人_原始快取_v30_2026-10-16.json.gz'
v30_flow_status={}
if v30_flow_path.exists():
    with gzip.open(v30_flow_path,'rt',encoding='utf-8') as f30:old30=json.load(f30)
    v30_flow_cache.update(old30.get('data',{}));v30_flow_status.update(old30.get('status',{}))
def persist_flow30():
    with gzip.open(v30_flow_path,'wt',encoding='utf-8') as f30:json.dump({'dataset':'TaiwanStockInstitutionalInvestorsBuySell','start':'2016-07-01','end':'2025-12-31','data':v30_flow_cache,'status':v30_flow_status},f30,ensure_ascii=False)
for i30,sid30 in enumerate(v30_universe):
    if sid30 in v30_flow_cache:continue
    params30={**v30_request,'data_id':sid30}
    try:
        with urllib.request.urlopen('https://api.finmindtrade.com/api/v4/data?'+urllib.parse.urlencode(params30),timeout=25) as resp30:doc30=json.loads(resp30.read(15000000))
        code30=doc30.get('status');v30_flow_status[sid30]={'status':code30,'rows':len(doc30.get('data',[]))}
        if code30!=200:
            v30_flow_status[sid30]['message']=str(doc30.get('msg',''))[:180];persist_flow30();print('V30_FETCH_STOP',sid30,v30_flow_status[sid30],flush=True);break
        v30_flow_cache[sid30]=doc30.get('data',[])
    except Exception as e30:
        v30_flow_status[sid30]={'error_type':type(e30).__name__};persist_flow30();print('V30_FETCH_ERROR',sid30,type(e30).__name__,flush=True);break
    if (i30+1)%15==0:persist_flow30();print('V30_FLOW_PROGRESS',len(v30_flow_cache),'/',len(v30_universe),flush=True)
    time.sleep(.8)
persist_flow30()
print('V30_FLOW_FETCH_FINISHED',{'cached':len(v30_flow_cache),'pool':len(v30_universe),'all_covered':all(s in v30_flow_cache for s in v30_universe),'rows':sum(len(x) for x in v30_flow_cache.values())})


In [ ]:
# v30 same-eligibility two-family extension; provider values audited against existing DB snapshot.
assert all(s in v30_flow_cache for s in v30_universe),'Do not call incomplete150 institutional data a full comparison'
v30_flow=np.full_like(v30_arrays['close'],np.nan);v30_groups_flow=collections.defaultdict(dict);v30_duplicates=0
for sid30,rows30 in v30_flow_cache.items():
    for row30 in rows30:
        if row30['name'] not in ['Foreign_Investor','Investment_Trust']:continue
        key30=(row30['date'],sid30);cat30=row30['name']
        if cat30 in v30_groups_flow[key30]:v30_duplicates+=1
        v30_groups_flow[key30][cat30]=float(row30['buy'])-float(row30['sell'])
assert v30_duplicates==0
for (day30,sid30),cats30 in v30_groups_flow.items():
    if len(cats30)==2 and day30 in v30_di and sid30 in v30_si:v30_flow[v30_di[day30],v30_si[sid30]]=sum(cats30.values())
v30_cross=[]
for dt30,told30 in di.items():
    if str(dt30) not in v30_di:continue
    tn30=v30_di[str(dt30)]
    for sid30,jold30 in si.items():
        if sid30 not in v30_si:continue
        old30=flow[told30,jold30];new30=v30_flow[tn30,v30_si[sid30]]
        if np.isfinite(old30) and np.isfinite(new30):v30_cross.append((old30,new30))
v30_cross=np.array(v30_cross);v30_matchrate=float(np.mean(np.abs(v30_cross[:,0]-v30_cross[:,1])<=1))
v30_cross_audit={'compared_stockdays':len(v30_cross),'netshares_match_within1':v30_matchrate,'signal':'Foreign_Investor + Investment_Trust only; excludes dealer; foreigndealer split2017/18 changes historical category meaning','source':'https://finmind.github.io/en/tutor/TaiwanMarket/Chip/'}
(v30_root/'長歷史環境研究_法人口徑核對_v30_2026-10-16.json').write_text(json.dumps(v30_cross_audit,ensure_ascii=False,indent=2),encoding='utf-8')
assert len(v30_cross)>10000 and v30_matchrate>=.99,'Provider vs DB flow mismatch; inspect before strategy runs'
v30_starts=np.arange(61,len(v30_days)-23);v30_scores={k:np.full_like(v30_arrays['close'],np.nan) for k in ['PRICE_CONTROL','MOM','FLOW']}
for q30 in range(60,len(v30_days)):
    rr30=v30_arrays['returns'][q30-59:q30+1];vol30=v30_arrays['volume'][q30-19:q30+1];ff30=v30_flow[q30-19:q30+1]
    base30=v30_pool&(v30_listed<=v30_days[q30-59])&np.all(np.isfinite(rr30)&(rr30>-1),axis=0)&np.all(np.isfinite(vol30)&(vol30>0),axis=0)&~np.any(v30_arrays['bad'][q30-59:q30+1],axis=0)&np.isfinite(v30_arrays['close'][q30])&(v30_arrays['close'][q30]>0)
    v30_scores['PRICE_CONTROL'][q30]=.5*percentile(np.prod(1+rr30[-20:],axis=0)-1,base30)+.5*percentile(np.prod(1+rr30,axis=0)-1,base30)
    elig30=base30&np.all(np.isfinite(ff30),axis=0)
    v30_scores['MOM'][q30]=.5*percentile(np.prod(1+rr30[-20:],axis=0)-1,elig30)+.5*percentile(np.prod(1+rr30,axis=0)-1,elig30)
    fscore30=np.sum(ff30,axis=0)/np.sum(vol30,axis=0);v30_scores['FLOW'][q30]=percentile(fscore30,elig30)
v30_arrays['mark']=v30_arrays['close'].copy();v30_arrays['paid_known']=np.zeros_like(v30_arrays['bad']);v30_arrays['available']=np.isfinite(v30_arrays['execution'])&(v30_arrays['execution']>0)&np.isfinite(v30_arrays['volume'])&(v30_arrays['volume']>0)
for ti30,dd30 in enumerate(v30_days):
    if datetime.date(2018,11,29)<=dd30<=datetime.date(2018,12,7):v30_arrays['mark'][ti30,v30_si['2404']]=63.9;v30_arrays['available'][ti30,v30_si['2404']]=False
v30_engine_export=json.loads((v30_root/'公司行動與成交修正_回測結果_v14_2026-10-15.json').read_text(encoding='utf-8'))
v30_engine=v30_engine_export['engine_source'];v30_tree=ast.parse(v30_engine);v30_fn=v30_tree.body[0];v30_fn.name='simulate_family30';v30_fn.args.args.append(ast.arg(arg='strategy'));v30_fn.args.defaults.append(ast.Constant(value='MOM'))
for node30 in ast.walk(v30_fn):
    if isinstance(node30,ast.Assign) and len(node30.targets)==1 and isinstance(node30.targets[0],ast.Name):
        if node30.targets[0].id=='eligible':node30.value=ast.parse('np.isfinite(scores30[strategy][t-1])',mode='eval').body
        if node30.targets[0].id=='score':node30.value=ast.parse('scores30[strategy][t-1]',mode='eval').body
v30_engine=ast.unparse(ast.fix_missing_locations(ast.Module(body=[v30_fn],type_ignores=[])))
v30_ns.update({'INITIAL':INITIAL,'FEE':FEE,'TAX':TAX,'percentile':percentile,'settle':settle,'case_arrays_v14':{'long30':v30_arrays},'scores30':v30_scores})
exec(v30_engine,v30_ns);v30_sim=v30_ns['simulate_family30']
v30_controls=[]
for ix30 in [0,180,400,650,900,1150,1400,1700,2000,2235]:
    start30=int(v30_starts[ix30]);now30=v30_sim({'key':'long30','year':v30_days[start30].year,'start_index':start30},set(v30_universe),'equal',0.,'unresolved','PRICE_CONTROL');old30=v30_export['results']['P90'][ix30]
    assert now30['reasons']==old30['reasons'] and now30['violations']==old30['violations'] and (now30['return'] is None)==(old30['return'] is None)
    if now30['return'] is not None:assert all(np.isclose(now30[k],old30[k],rtol=0,atol=1e-12) for k in ['return','mdd','cost','turnover'])
    v30_controls.append({'index':ix30,'matched':True})
v30_definition.update({'stage':'defined_ready','institutional_signal':'20day sum(Foreign_Investor+Investment_Trust netshares)/20day sumvolume; percentile','both_share_flow_quality_gate':True,'cross_source':v30_cross_audit,'price_control_checks':v30_controls,'raw_v27_filename_mismatch':v30_export_path.name})
(v30_root/'長歷史環境研究_預先定義_v30_2026-10-16.json').write_text(json.dumps(v30_definition,ensure_ascii=False,indent=2),encoding='utf-8')
print('V30_ENGINE_READY',{'cross_source':v30_cross_audit,'controls':v30_controls,'eligiblefirst':int(np.isfinite(v30_scores['FLOW'][60]).sum()),'original_recent_shape':signal_returns_v15.shape})


In [ ]:
# v30 freeze observations and one start-only hypothesis before outcome summaries.
v30_n=len(v30_starts);v30_good=np.isfinite(v30_arrays['returns'])&~v30_arrays['bad']
for t30,day30 in enumerate(v30_days):v30_good[t30]&=v30_pool&(v30_listed<=day30)
v30_nlisted=np.array([int(np.sum(v30_pool&(v30_listed<=d))) for d in v30_days]);v30_ngood=v30_good.sum(axis=1)
v30_proxy=np.divide(np.where(v30_good,v30_arrays['returns'],0).sum(axis=1),v30_ngood,out=np.full(len(v30_days),np.nan),where=v30_ngood>=np.maximum(50,np.ceil(.8*v30_nlisted)))
v30_feature=[]
for start30 in v30_starts:
    q30=int(start30)-1;complete30=v30_good[q30-19:q30+1].all(axis=0)&(v30_listed<=v30_days[q30-19]);den30=int(np.sum(v30_pool&(v30_listed<=v30_days[q30-19])));need30=max(50,int(np.ceil(.8*den30)));okay30=bool(complete30.sum()>=need30 and np.isfinite(v30_proxy[q30-19:q30+1]).all())
    v30_feature.append({'cutoff':str(v30_days[q30]),'valid':okay30,'complete_names':int(complete30.sum()),'listed_denominator':den30,'min_required':need30,'fixed120_ok':bool(complete30.sum()>=120 and np.all(v30_ngood[q30-19:q30+1]>=120)),'trend':float(np.prod(1+v30_proxy[q30-19:q30+1])-1) if okay30 else None,'breadth':float(np.mean((np.prod(1+v30_arrays['returns'][q30-19:q30+1,complete30],axis=0)-1)>0)) if okay30 else None,'vol':float(np.std(v30_proxy[q30-19:q30+1],ddof=1)) if okay30 else None})
v30_training=np.array([v30_days[int(s)+23]<=datetime.date(2018,12,31) for s in v30_starts]);v30_later=np.array([v30_days[int(s)]>=datetime.date(2019,1,1) for s in v30_starts]);v30_threshold=float(np.median([f['vol'] for i,f in enumerate(v30_feature) if v30_training[i] and f['valid']]))
v30_definition.update({'volatility_threshold_locked':v30_threshold,'calibration_feature_windows':sum(bool(v30_training[i] and f['valid']) for i,f in enumerate(v30_feature)),'one_hypothesis':'start-only choose FLOW if lowvol<=threshold else MOM;selected family fixed for24day with its original5day reselect;not daily switch','inference_family':'3 environment interactions plus 2 switch-minus-MOM/FLOW contrasts =5;Bonferroni5;24/48position3000boot;year/cost/downsidecheck;allseenretrospective','switch_evaluation':'2019..2025 primary;2016..2018 calibration descriptive;no globally best threshold search','fixed120_sensitivity':'same return pairs,only feature coverage constraint;do not substitute zero','old_v29_threshold':'0.007943392407260884 sensitivity only;not known2016'} )
(v30_root/'長歷史環境研究_預先定義_v30_2026-10-16.json').write_text(json.dumps(v30_definition,ensure_ascii=False,indent=2),encoding='utf-8')
print('V30_OBSERVATIONS_LOCKED',{'threshold':v30_threshold,'features_valid':sum(f['valid'] for f in v30_feature),'fixed120_count':sum(f['fixed120_ok'] for f in v30_feature),'calibration_n':v30_definition['calibration_feature_windows']})


In [ ]:
# v30 new same-quality-gate comparisons only; no timing strategy or parameter grid.
v30_results={};v30_simcount=0
for strategy30 in ['MOM','FLOW']:
    rows30=[]
    for i30,start30 in enumerate(v30_starts):
        case30={'key':'long30','year':v30_days[int(start30)].year,'start_index':int(start30)}
        rr30=v30_sim(case30,set(v30_universe),'equal',0.,'unresolved',strategy30);rr30['start_index']=int(start30);rows30.append(rr30);v30_simcount+=1
        if (i30+1)%750==0:print('V30_BACKTEST_PROGRESS',strategy30,i30+1,flush=True)
    v30_results[strategy30]=rows30
    print('V30_FAMILY_DONE',strategy30,sum(r['return'] is not None for r in rows30),flush=True)
(v30_root/'長歷史環境研究_動能法人逐窗口_v30_2026-10-16.json').write_text(json.dumps({'definition':v30_definition,'engine':v30_engine,'results':v30_results},ensure_ascii=False),encoding='utf-8')
v30_valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in v30_results.items()};v30_ret={k:np.array([np.nan if r['return'] is None else r['return'] for r in rows]) for k,rows in v30_results.items()}
print('V30_FAMILIES_COMPLETE',{'runs':v30_simcount,'valid':{k:int(x.sum()) for k,x in v30_valid.items()},'paired':int((v30_valid['MOM']&v30_valid['FLOW']).sum())})


In [ ]:
# v30 paired long-history distributions,5-family corrected uncertainty and chronological checks.
v30_common=v30_valid['MOM']&v30_valid['FLOW']&np.array([f['valid'] for f in v30_feature]);v30_states={'trend':np.array([f['valid'] and f['trend']>0 for f in v30_feature]),'breadth':np.array([f['valid'] and f['breadth']>=.5 for f in v30_feature]),'volatility':np.array([f['valid'] and f['vol']>v30_threshold for f in v30_feature])}
v30_labels={'trend':['市場20日非正','市場20日上漲'],'breadth':['上漲股票未過半','上漲股票至少一半'],'volatility':['波動較低','波動較高']}
v30_switch=np.where(v30_states['volatility'],v30_ret['MOM'],v30_ret['FLOW']);v30_ret['SWITCH']=v30_switch
v30_results['SWITCH']=[v30_results['MOM' if v30_states['volatility'][i] else 'FLOW'][i] for i in range(v30_n)]
def metric30(k30,mask30):
    ids30=np.flatnonzero(mask30);x30=v30_ret[k30][mask30]
    if not len(x30):return {'n':0}
    return {'n':len(x30),'win':float(np.mean(x30>0)),'ev':float(x30.mean()),'median':float(np.median(x30)),'mean_loss':float(x30[x30<0].mean()) if np.any(x30<0) else None,'mean_win':float(x30[x30>0].mean()) if np.any(x30>0) else None,'p10':float(np.quantile(x30,.1)),'worst_return':float(x30.min()),'worst_mdd':float(min(v30_results[k30][i]['mdd'] for i in ids30)),'fee':float(np.mean([v30_results[k30][i]['cost'] for i in ids30]))}
v30_groups={}
for axis30,state30 in v30_states.items():
    gs30={}
    for hi30 in [False,True]:
        mask30=v30_common&(state30==hi30);name30=v30_labels[axis30][int(hi30)]
        gs30[name30]={'n':int(mask30.sum()),'later_n':int((mask30&v30_later).sum()),'methods':{k:metric30(k,mask30) for k in ['MOM','FLOW']},'later':{k:metric30(k,mask30&v30_later) for k in ['MOM','FLOW']},'years':{str(y):{k:metric30(k,mask30&np.array([v30_days[int(s)].year==y for s in v30_starts])) for k in ['MOM','FLOW']} for y in range(2016,2026)}}
    v30_groups[axis30]=gs30
v30_years={str(y):{'planned_n':sum(v30_days[int(s)].year==y for s in v30_starts),'paired_n':sum(bool(v30_common[i]) and v30_days[int(s)].year==y for i,s in enumerate(v30_starts)),'methods':{k:metric30(k,v30_common&np.array([v30_days[int(s)].year==y for s in v30_starts])) for k in ['MOM','FLOW','SWITCH']}} for y in range(2016,2026)}
v30_interactions={};v30_switchcontrasts={};diff30=v30_ret['MOM']-v30_ret['FLOW'];v30_rng=np.random.default_rng(3020261016)
for b30 in [24,48]:
    interactions30={a:[] for a in v30_states};switches30={k:[] for k in ['MOM','FLOW']}
    for chunk30 in range(30):
        base30=v30_rng.integers(0,v30_n-b30+1,size=(100,int(np.ceil(v30_n/b30))));idx30=(base30[:,:,None]+np.arange(b30)).reshape(100,-1)[:,:v30_n]
        for axis30,state30 in v30_states.items():
            means30=[]
            for high30 in [True,False]:
                mask30=v30_common&v30_later&(state30==high30);weights30=mask30[idx30];den30=weights30.sum(axis=1);num30=np.where(weights30,diff30[idx30],0.).sum(axis=1);means30.append(np.divide(num30,den30,out=np.full(100,np.nan),where=den30>0))
            interactions30[axis30].extend((means30[0]-means30[1]).tolist())
        for k30 in ['MOM','FLOW']:
            mask30=v30_common&v30_later;dd30=v30_switch-v30_ret[k30];weights30=mask30[idx30];den30=weights30.sum(axis=1);num30=np.where(weights30,dd30[idx30],0.).sum(axis=1);switches30[k30].extend(np.divide(num30,den30,out=np.full(100,np.nan),where=den30>0).tolist())
    for axis30,xx30 in interactions30.items():
        hi30=v30_common&v30_later&v30_states[axis30];lo30=v30_common&v30_later&~v30_states[axis30];v30_interactions.setdefault(axis30,{'estimate':float(diff30[hi30].mean()-diff30[lo30].mean())})['CI'+str(b30)]=[float(x) for x in np.nanquantile(xx30,[.05/10,1-.05/10])]
    for k30,xx30 in switches30.items():
        mask30=v30_common&v30_later;dd30=v30_switch-v30_ret[k30];v30_switchcontrasts.setdefault(k30,{'n':int(mask30.sum()),'mean_difference':float(dd30[mask30].mean()),'beat_rate':float(np.mean(dd30[mask30]>1e-12))})['CI'+str(b30)]=[float(x) for x in np.nanquantile(xx30,[.05/10,1-.05/10])]
v30_coverage={k:{'valid':int(v30_valid[k].sum()),'planned':v30_n,'company_or_data_errors':dict(collections.Counter(reason for r in v30_results[k] for reason in r['reasons'])),'violations':sum(r['violations']>0 for r in v30_results[k])} for k in ['MOM','FLOW']}
v30_sensitivity={}
for label30,mask30 in [('fixed120',v30_common&np.array([f['fixed120_ok'] for f in v30_feature])),('v29threshold_low',v30_common&np.array([f['valid'] and f['vol']<=.007943392407260884 for f in v30_feature])),('v29threshold_high',v30_common&np.array([f['valid'] and f['vol']>.007943392407260884 for f in v30_feature]))]:v30_sensitivity[label30]={k:metric30(k,mask30) for k in ['MOM','FLOW']}
v30_summary={'definition':v30_definition,'planned':v30_n,'paired':int(v30_common.sum()),'first_start':str(v30_days[int(v30_starts[0])]),'last_start':str(v30_days[int(v30_starts[-1])]),'last_end':str(v30_days[-1]),'overall':{k:metric30(k,v30_common) for k in ['MOM','FLOW','SWITCH']},'later':{k:metric30(k,v30_common&v30_later) for k in ['MOM','FLOW','SWITCH']},'groups':v30_groups,'years':v30_years,'interactions_later':v30_interactions,'start_switch_later':v30_switchcontrasts,'coverage':v30_coverage,'sensitivity':v30_sensitivity,'controls':v30_controls,'flow_cross_source':v30_cross_audit,'not_macro_firstrelease_backtest':True}
def serial30(x):
    if isinstance(x,np.generic):return x.item()
    if isinstance(x,(datetime.date,datetime.datetime)):return str(x)
    raise TypeError(type(x).__name__)
(v30_root/'長歷史環境研究_統計與分層_v30_2026-10-16.json').write_text(json.dumps(v30_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
print('V30_LONG_STATS',{'paired':v30_summary['paired'],'overall':v30_summary['overall'],'later':v30_summary['later'],'switch_vs':v30_switchcontrasts,'vol_groups':{n:{'n':g['n'],'later_n':g['later_n'],'methods':g['methods']} for n,g in v30_groups['volatility'].items()},'threshold':v30_threshold})


In [ ]:
# v30 full historical evidence; preserve recent results and update fixed master report.
import plotly.graph_objects as go
from markdown_it import MarkdownIt
def p30(x):return '未列' if x is None else f'{100*x:.2f}%'
def c30(c):return '['+', '.join(f'{x*100:+.3f}' for x in c)+']pp'
v30_latermask=v30_common&v30_later
v30_pass={k:all(v30_switchcontrasts[k]['CI'+str(b)][0]>0 for b in [24,48]) for k in ['MOM','FLOW']}
v30_lines=['# 2016–2025：動能、法人與市場環境 v30','更新：2026-10-16。固定25檔、90%投入、每五日評估、24交易日，整千股及原費稅。新增長歷史法人資料與同品質門檻動能比較，不把價量動能單獨當作法人對照。','## 研究完成範圍',f'行情{v30_days[0]}至{v30_days[-1]}，2320交易日。回看暖機後2236個計畫窗口，第一起點{v30_days[int(v30_starts[0])]}、最後起點{v30_days[int(v30_starts[-1])]}、最後結束{v30_days[-1]}。2016僅部分年度，並非自2016年1月起。兩家族及環境共同可判定{int(v30_common.sum())}／2236，2019～2025主要檢查{int(v30_latermask.sum())}窗口。每個窗口重新開始本金10億元，不是十年連續複利帳戶。','## 先前假說如何驗證','開賽前以20日波動判斷：低波動採法人、高波動採動能；選定後24日內維持該家族原本每五日重選規則，不在期間每日切換。兩種策略都可執行的同日起點才比較，沒有把另一策略無效日期當0。',f'波動閾值只用2016～2018年、窗口在2018年底前結束的環境特徵中位數：{p30(v30_threshold)}（每日標準差）。2016～2018校準期只作描述，2019～2025為主要後段檢查，但這段歷史已被其他研究看過，不稱真正未見樣本外。',f'後段切換相對純動能通過兩種校正區間下界>0：{v30_pass["MOM"]}；相對純法人：{v30_pass["FLOW"]}。這只是平均增量檢核，還須年份、覆蓋與下行風險，不能只因某項均值較高就推薦。','## 後段2019～2025：相同窗口、費後24日','|方法|窗口|勝率|期望值|平均獲利|平均虧損|P10|最差窗口內回撤|平均費稅|','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
v30_names={'MOM':'20／60各半動能','FLOW':'純法人（外資＋投信）','SWITCH':'開賽前低波動法人／高波動動能'}
def metricrow30(k,m):return '|'+ '|'.join([v30_names[k],str(m['n']),p30(m['win']),p30(m['ev']),p30(m['mean_win']),p30(m['mean_loss']),p30(m['p10']),p30(m['worst_mdd']),p30(m['fee'])])+'|'
for k in ['MOM','FLOW','SWITCH']:v30_lines.append(metricrow30(k,v30_summary['later'][k]))
v30_lines+=['## 全2016～2025描述性結果（含校準期）','|方法|窗口|勝率|期望值|平均獲利|平均虧損|P10|最差窗口內回撤|平均費稅|','|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for k in ['MOM','FLOW','SWITCH']:v30_lines.append(metricrow30(k,v30_summary['overall'][k]))
v30_lines+=['## 後段切換增量與區間','pp為百分點。24／48個原始起點位置區塊，各3000次；三個環境交互差及兩個切換比較，共五項Bonferroni。保留缺漏位置，重疊窗口非獨立試驗。','|切換減對照|共同窗口|平均差|24日校正區間|48日校正區間|','|---|---:|---:|---|---|']
for k,r in v30_switchcontrasts.items():v30_lines.append(f'|切換−{v30_names[k]}|{r["n"]}|{r["mean_difference"]*100:+.3f}pp|{c30(r["CI24"])}|{c30(r["CI48"])}|')
v30_lines+=['## 後段環境分層','|狀態|窗口|動能勝率／期望|法人勝率／期望|動能／法人最差回撤|','|---|---:|---|---|---|']
for axis,groups in v30_groups.items():
    for label,g in groups.items():
        a=g['later']['MOM'];b=g['later']['FLOW'];v30_lines.append(f'|{label}|{a["n"]}|{p30(a["win"])}／{p30(a["ev"])}|{p30(b["win"])}／{p30(b["ev"])}|{p30(a["worst_mdd"])}／{p30(b["worst_mdd"])}|')
v30_lines+=['## 後段交互差：策略優劣是否隨環境改變？','交互差＝高狀態的（動能−法人）減低狀態的（動能−法人），不是因果效果。','|環境|交互差|24日校正區間|48日校正區間|','|---|---:|---|---|']
for axis,r in v30_interactions.items():v30_lines.append(f'|{axis}|{r["estimate"]*100:+.3f}pp|{c30(r["CI24"])}|{c30(r["CI48"])}|')
v30_lines+=['## 逐年：勝率／24日期望，不是年報酬','|起始年|共同／計畫窗口|動能|法人|開賽前切換|','|---|---:|---|---|---|']
for year,r in v30_years.items():
    cells=[year,str(r['paired_n'])+'/'+str(r['planned_n'])]
    for k in ['MOM','FLOW','SWITCH']:
        m=r['methods'][k];cells.append(p30(m.get('win'))+'／'+p30(m.get('ev')))
    v30_lines.append('|'+ '|'.join(cells)+'|')
v30_lines+=['## 資料品質、口徑與去重',f'公開FinMind逐股查詢150檔成功，只有外資與投信兩類都有原始列的股票日期才有合計；缺值不填0。與近期團隊DB重疊{v30_cross_audit["compared_stockdays"]}個股票日期，以1股容忍的淨買賣超一致率{p30(v30_cross_audit["netshares_match_within1"])}。純法人在本專案指「外資＋投信」，不是把所有三大法人分類混加。', 'FinMind原始分類在2017／2018年拆出外資自營商；拆分前外資分類包含該部分、之後外資不含。這是歷史分類口徑限制，未用未知歷史數字補齊；不能宣稱跨年代外資定義完全一致。資料來源：[FinMind官方法人資料說明](https://finmind.github.io/en/tutor/TaiwanMarket/Chip/)，查閱2026-10-16。', '法人信號為前20日外資＋投信淨買超股數合計／前20日成交股數合計，再作橫截面百分位；動能仍20／60各半。兩者共用60日價量／公司行動及20日法人資料完整門檻，以隔離排名訊號。與v27沒有法人品質門檻的價量動能不完全相同，不能跨表直接歸因。', '舊v27逐窗口檔案誤名為v26_2027-10-15，但內部definition為v27，已對照舊Notebook輸出程式與現行v27統計確認。保留原始檔名，並以十個控制窗口核對重建引擎的報酬、費用、換手、回撤、違規與缺口，沒有因檔名猜測混用v26。', '公司行動沿用v27純現金財富報酬及1400筆已核實旗標修正；未知配股、認購、減資等仍排除。2018漢唐停牌只保留原已驗證估值、禁止成交，不捏造價格。股票池仍為今天150檔的歷史可掛牌子集，掛牌日作保守門檻，不是歷年官方150重建。','## 環境門檻與舊版差異', '市場代理只含當日已掛牌且資料可用的目前150檔；最低覆蓋為當時已掛牌數80%，且至少50檔，避免把未掛牌股票當缺價或有價格。廣度只取回看20日完整股票，分母為回看起點前已掛牌的當前150子集。此規則在計算前固定，並另列舊版固定120檔門檻的敏感度。代理為變動成分研究指標，不是官方可交易基準；本輪沒有證明超額市場alpha。',f'舊v29日波動閾值0.7943%是近期校準；另列在長歷史的描述性敏感度，不能冒稱2016年已知。主要閾值改為2016～2018訓練特徵中位數{p30(v30_threshold)}，不按報酬尋找最佳。','## 覆蓋與敏感度','|方法|有效／計畫|有違規窗口|主要資料／公司行動缺口|','|---|---:|---:|---|']
for k,r in v30_coverage.items():v30_lines.append(f'|{k}|{r["valid"]}/{v30_n}|{r["violations"]}|{json.dumps(r["company_or_data_errors"],ensure_ascii=False)}|')
v30_lines+=['排除非隨機，無效不填0。個別策略有效與兩者共同有效不同，不能拿各自有效數據直接排名。','|敏感度|窗口|動能勝率／期望|法人勝率／期望|','|---|---:|---|---|']
for label,ms in v30_sensitivity.items():
    a=ms['MOM'];b=ms['FLOW'];v30_lines.append(f'|{label}|{a["n"]}|{p30(a.get("win"))}／{p30(a.get("ev"))}|{p30(b.get("win"))}／{p30(b.get("ev"))}|')
v30_lines+=['## 未完成與限制','這次完成市場價格環境的長歷史比較，沒有完成景氣燈號／PMI首次公開版本回測。仍有現況股票池存活偏誤、外資分類改版、來源非官賽DataPack、公司行動排除、重複看歷史與宏觀事件數不足。Active Share與正式帳務未全驗證，不提交交易。即使後段均值較高，也需把年份一致性與尾端損失一起看。', '波動閾值不是景氣燈號；高波動不代表景氣好。不同環境有關聯不代表造成結果。24日窗口短且重疊，不能把2236個起點當2236次獨立景氣驗證。', '## 保存與驗證','沿用既有Python314 kernel PID36240，不重啟、不清除近期物件。分析、程式與輸出保存原Notebook；沒有Restart and Run All。成功法人原始快取保存供重用。驗證價格控制10窗口、外資投信與DB交叉、共同日期、分組總數、特徵時間先於開賽日、損益恆等式及全部點。']
v30_report='\n'.join(v30_lines);v30_rp=v30_root/'長歷史環境研究_動能法人比較_v30_2026-10-16.md';v30_rp.write_text(v30_report,encoding='utf-8')
v30_figures=[];v30_points_count=0;v30_jitter=np.random.default_rng(3030);v30_range=[float(min(v30_ret[k][v30_common].min() for k in ['MOM','FLOW']))*100-3,float(max(v30_ret[k][v30_common].max() for k in ['MOM','FLOW']))*100+3]
for axis,state in v30_states.items():
    fig=go.Figure();ticks=[];labels=[]
    for hi in [False,True]:
        keep=v30_common&(state==hi);ids=np.flatnonzero(keep)
        for ki,k in enumerate(['MOM','FLOW']):
            pos=int(hi)*2+ki;name=v30_labels[axis][int(hi)]+'／'+v30_names[k];yy=v30_ret[k][keep]*100;color=['#167368','#567bb5'][ki];ticks.append(pos);labels.append(name)
            fig.add_trace(go.Box(x=np.full(len(yy),pos),y=yy,width=.55,boxpoints=False,hoverinfo='skip',line={'color':color},fillcolor='rgba(22,115,104,.10)',showlegend=False))
            fig.add_trace(go.Scatter(x=pos+v30_jitter.uniform(-.18,.18,len(yy)),y=yy,mode='markers',marker={'size':4,'opacity':.35,'color':color},customdata=[[v30_results[k][i]['start'],v30_results[k][i]['end'],('後段檢查' if v30_later[i] else '校準／跨界')] for i in ids],hovertemplate=name+'<br>%{customdata[0]}→%{customdata[1]}<br>%{customdata[2]}<br>費後 %{y:.2f}%<extra></extra>',showlegend=False))
            med=float(np.median(yy));fig.add_trace(go.Scatter(x=[pos-.23,pos+.23],y=[med,med],mode='lines',line={'color':'#c43c79','width':4},hoverinfo='skip',showlegend=False));v30_points_count+=len(yy)
    fig.add_hline(y=0,line_color='#173333',line_width=2);fig.update_layout(title=axis+'｜全期每個點；含校準期，主要推論另看2019～2025表',template='plotly_white',height=540,xaxis={'tickmode':'array','tickvals':ticks,'ticktext':labels},yaxis={'title':'24日費後報酬 %','range':v30_range},margin={'b':120})
    v30_figures.append(fig.to_html(full_html=False,include_plotlyjs=(axis=='trend')))
fy=go.Figure()
for k in ['MOM','FLOW','SWITCH']:
    fy.add_trace(go.Bar(x=list(v30_years),y=[m['methods'][k].get('ev',np.nan)*100 for m in v30_years.values()],name=v30_names[k],customdata=[m['paired_n'] for m in v30_years.values()],hovertemplate='%{x}起點年<br>24日均值 %{y:.2f}%<br>共同窗口%{customdata}<extra>%{fullData.name}</extra>'))
fy.add_hline(y=0,line_color='#173333');fy.update_layout(title='逐年24日期望報酬；不是年度報酬，2016～2018為校準期',template='plotly_white',barmode='group',height=480,yaxis_title='費後24日平均 %');v30_figures.append(fy.to_html(full_html=False,include_plotlyjs=False))
assert v30_points_count==int(v30_common.sum())*2*3
md30=MarkdownIt('commonmark',{'html':False}).enable('table');v30_hp=v30_root/'長歷史環境研究_全部點與年度比較_v30_2026-10-16.html'
v30_hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>2016–2025動能與法人環境比較</title><style>body{font:16px/1.8 system-ui;max-width:1300px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}th,td{padding:8px;border-bottom:1px solid #ccd8d3;text-align:left}.table-wrap{overflow-x:auto}.note{padding:16px;background:#edf3ed}a{color:#167368}</style><h1>2016–2025長歷史：動能、法人、市場環境</h1><p class="note">同'+str(int(v30_common.sum()))+'個有效窗口；三種分層各顯示所有原始窗口點，桃紅中位數、0線、相同尺度。2016僅暖機後部分年度。校準2016～2018，主要檢查2019～2025；年度與分層差異不等於未來保證。</p><a href="策略研究_總結報告.html">回總結報告</a>'+''.join(v30_figures)+md30.render(v30_report).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',encoding='utf-8')
v30_fragment='## 新增證據：2016～2025長歷史環境 v30\n\n'+f'已取得150檔長歷史外資／投信資料並核對近期DB，兩種策略共用價量與法人品質門檻，2236起點中共同{int(v30_common.sum())}。主要檢查2019～2025共{int(v30_latermask.sum())}窗口；波動閾值以2016～2018特徵校準，不使用2024門檻倒回2016。\n\n'
for k,m in v30_summary['later'].items():v30_fragment+=f'- {v30_names[k]}：勝率{p30(m["win"])}、24日期望{p30(m["ev"])}、平均費稅{p30(m["fee"])}、最差窗口內回撤{p30(m["worst_mdd"])}。\n'
v30_fragment+='\n'
for k,r in v30_switchcontrasts.items():v30_fragment+=f'切換減{v30_names[k]}平均{r["mean_difference"]*100:+.3f}pp，48日五比較校正區間{c30(r["CI48"])}。'
v30_fragment+=' 不只看最高均值；保留年份與下行風險檢查。原近期v29仍為歷史紀錄，不能混合分母。\n\n[長歷史完整報告](長歷史環境研究_動能法人比較_v30_2026-10-16.md) · [全部點與逐年互動圖](長歷史環境研究_全部點與年度比較_v30_2026-10-16.html)\n\n'
mp=v30_root/'策略研究_總結報告.md';mt=mp.read_text(encoding='utf-8');mt=mt.replace('## 全體限制與未完成項',v30_fragment+'## 全體限制與未完成項');mt=mt.replace('## 更新紀錄','## 更新紀錄\n\n- 2026-10-16：新增v30十年市場環境及動能／法人比較；固定前段特徵校準，後段檢查開賽前單次家族選擇。保留現況股票池與公司行動限制。');mp.write_text(mt,encoding='utf-8')
hp=v30_root/'策略研究_總結報告.html';ht=hp.read_text(encoding='utf-8');ht=re.sub(r'(<article id="reportBody">).*?(</article>)',lambda m:m[1]+md30.render(mt).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+m[2],ht,flags=re.S)
teaser='<section id="long-regime"><h2>新增：2016～2025長歷史環境</h2><p>動能、外資＋投信與開賽前依波動選家族；前段校準、後段檢查。共同'+str(int(v30_common.sum()))+'窗口，實際勝率、期望值與風險請查看結果。</p><a href="長歷史環境研究_全部點與年度比較_v30_2026-10-16.html">查看長歷史全部點與年度結果 →</a></section>'
ht=ht.replace('<section id="regime-results">',teaser+'<section id="regime-results">');ht=ht.replace('<a href="#regime-results">','<a href="#long-regime">十年環境新證據</a><a href="#regime-results">');hp.write_text(ht,encoding='utf-8')
for axis,gs in v30_groups.items():assert sum(g['n'] for g in gs.values())==int(v30_common.sum())
for k in ['MOM','FLOW','SWITCH']:
    m=v30_summary['overall'][k];x=v30_ret[k][v30_common];assert np.isclose(m['ev'],np.mean(x>0)*(m['mean_win'] or 0)+np.mean(x<0)*(m['mean_loss'] or 0))
v30_summary['verification']={'control_windows':10,'all_points':v30_points_count,'no_new_daily_switch':True,'group_n_sum':True,'payoff_identity':True,'no_kernel_reset':True,'rawpricecache_reused':True}
(v30_root/'長歷史環境研究_統計與分層_v30_2026-10-16.json').write_text(json.dumps(v30_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
stp=v30_root/'A策略_無人值守研究狀態.json';st=json.loads(stp.read_text(encoding='utf-8'));st['environment_long_v30']={'status':'completed_with_source_limits','running':False,'planned':v30_n,'paired':int(v30_common.sum()),'later':int(v30_latermask.sum()),'outputs':[v30_rp.name,v30_hp.name],'flow_cross_source':v30_cross_audit,'switch_pass_mean_only':v30_pass,'notebook_cells_start':321,'macro_firstrelease_complete':False};stp.write_text(json.dumps(st,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V30_DELIVERED',{'html':v30_hp.name,'report':v30_rp.name,'scatter_points':v30_points_count,'switch_pass_mean_only':v30_pass,'later_metrics':v30_summary['later']})


In [ ]:
# v30 bring latest long-history result into master controls, not only appendices.
v30_definition['no_new_daily_timing_or_holdings_rule']=v30_definition.pop('no_new_timing_strategy',True)
v30_definition['snapshot_sha256']={'price_current150_cache':hashlib.sha256((v30_root/'完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz').read_bytes()).hexdigest(),'flow_current150_cache':hashlib.sha256(v30_flow_path.read_bytes()).hexdigest()}
v30_summary['definition']=v30_definition
(v30_root/'長歷史環境研究_預先定義_v30_2026-10-16.json').write_text(json.dumps(v30_definition,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
(v30_root/'長歷史環境研究_統計與分層_v30_2026-10-16.json').write_text(json.dumps(v30_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
v30_conclusion='長歷史目前未支持「低波動必選法人、高波動必選動能」作通用規則。後段切換對動能平均差−0.076pp，校正區間跨0；法人平均虧損較小，但最差回撤更深，不應一概稱風險更低。'
v30_reporttext=v30_rp.read_text(encoding='utf-8');v30_reporttext=v30_reporttext.replace('## 研究完成範圍','## 主要結論\n\n'+v30_conclusion+'\n\n## 研究完成範圍');v30_rp.write_text(v30_reporttext,encoding='utf-8')
fullhtml=v30_hp.read_text(encoding='utf-8');fullhtml=re.sub(r'<h1>2016–2025：動能、法人與市場環境 v30</h1>.*?</html>',lambda m:md30.render(v30_reporttext).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',fullhtml,flags=re.S);v30_hp.write_text(fullhtml,encoding='utf-8')
mt=mp.read_text(encoding='utf-8');mt=mt.replace('目前總結的主線交易回測截至 v27，新增v29環境觀察；另有v28反轉分支待跨家族整合','目前主線已延伸為v30（2016～2025動能／法人與環境比較）；近期v29與反轉v28保留為分支結果');mt=mt.replace('## 目前研究判斷','## 目前研究判斷\n\n最新v30提醒：'+v30_conclusion);mp.write_text(mt,encoding='utf-8')
ht=hp.read_text(encoding='utf-8');ht=re.sub(r'(<article id="reportBody">).*?(</article>)',lambda m:m[1]+md30.render(mt).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+m[2],ht,flags=re.S)
if 'LONG30_DATASETS' not in ht:
    configs=[]
    for tag,title in [('overall','2016–2025 動能／法人'),('later','2019–2025 後段檢查')]:
        ms=v30_summary[tag];rr=[]
        for k in ['MOM','FLOW','SWITCH']:
            m=ms[k];rr.append([v30_names[k],100*m['win'],100*m['ev'],100*m['worst_mdd'],100*m['fee']])
        configs.append({'name':title,'sample':f'v30 · 同{ms["MOM"]["n"]}窗口 · '+('含2016～2018校準期，僅全期描述。' if tag=='overall' else '2016～2018特徵校準；後段歷史仍已被看過。')+'目前150檔歷史子集，非歷年官方名單。','rows':rr,'note':v30_conclusion+'全部24日費後結果；使用相同法人品質門檻，不能與舊價量版v27直接作差。','url':v30_hp.name})
    injection='// LONG30_DATASETS\nexperiments.unshift(...'+json.dumps(configs,ensure_ascii=False)+');\n'
    ht=ht.replace('const measures=',injection+'const measures=');ht=ht.replace("name:'動能 vs 法人'","name:'動能 vs 法人（近期）'")
ht=ht.replace('交易回測v28另有反轉分支 · 新增v29環境觀察','主線更新v30 · 2016～2025長歷史環境比較')
ht=ht.replace('實際勝率、期望值與風險請查看結果。','後段動能期望2.82%、法人1.55%、開賽前依波動選家族2.74%。切換未證實優於純動能，法人最差回撤也更深。')
hp.write_text(ht,encoding='utf-8')
v30_check_counts={k:sum(r['methods']['MOM']['n']>0 and r['methods']['FLOW']['n']>0 for r in v30_groups[k].values()) for k in v30_groups}
assert sum(r['paired_n'] for r in v30_years.values())==int(v30_common.sum())
assert all(v30_feature[i]['cutoff']<str(v30_days[int(s)]) for i,s in enumerate(v30_starts))
for k in ['MOM','FLOW']:
    for i in np.flatnonzero(v30_valid[k]):assert np.isclose(v30_results[k][i]['path'][-1]/INITIAL-1,v30_ret[k][i])
st=json.loads(stp.read_text(encoding='utf-8'));st['environment_long_v30'].update({'notebook_cells':[321,329],'summary_controls_updated':True,'source_hashes':v30_definition['snapshot_sha256'],'conclusion':v30_conclusion});stp.write_text(json.dumps(st,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
wf=v30_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## v30 長歷史市場環境（完成）\n'+v30_conclusion+' 150檔法人原始快取與品質對照已保存；2236起點共同1359、後段1041。使用2016～2018特徵校準門檻，2019～2025檢查；非未見樣本外。未重啟kernel，未變更排程，未提交交易。來源／公司行動／現況股票池偏誤保留。見'+v30_hp.name+'。\n',encoding='utf-8')
print('V30_COMPLETE',{'common':int(v30_common.sum()),'coverage':float(v30_common.mean()),'later':int(v30_latermask.sum()),'years':{y:int(r['paired_n']) for y,r in v30_years.items()},'later_vol_groups':{n:g['later'] for n,g in v30_groups['volatility'].items()},'interactions':v30_interactions,'master_comparison_updated':True})


In [ ]:
# v31 inventory and reporting contract; no new trading backtests.
v31_inventory=[
{'id':'core_selection','priority':'已完成','scope':'20／60各半動能、純法人（外資＋投信）','status':'可列主要總結','source':'v30','action':'重用既有4472列，不重跑；共同1359、後段1041'},
{'id':'environment_switch','priority':'已完成','scope':'趨勢、廣度、波動分層及開賽前選家族','status':'可列主要總結，未證實優於動能','source':'v30','action':'重用既有結果與校準；不再拿v29近期低波動結果作決策依據'},
{'id':'benchmark_and_hold','priority':'優先1','scope':'同期市場參考、成交金額排名25檔持有、動能初始持有 vs 五日調整','status':'待統一重算／補正式基準','source':'v16機制；v29代理非正式基準','action':'先隔離選股與換手效果；市場指數需正式總報酬來源；不把研究代理稱alpha'},
{'id':'lookback','priority':'優先1','scope':'純20日／純60日／各半，全部每五日評估','status':'待統一重算','source':'v20','action':'原定三組，保留共同60日品質門檻，不擴充參數網格'},
{'id':'capital','priority':'優先1','scope':'固定80／90／95%投入','status':'待以v30共同法人品質門檻重算','source':'v27為同年代但價量門檻不同','action':'90%基準直接重用；80與95只算新增格；100%舊預算負現金不列合格候選'},
{'id':'allocation','priority':'優先2','scope':'等權 vs 原排名1.25至0.75溫和非等權；小額略過分開','status':'待統一重算','source':'v16','action':'等權原結果重用；權重與略過用四格隔離，不以組合效果冒稱權重效果'},
{'id':'stop_add','priority':'優先2','scope':'MA5／連兩日MA5／破前五日收盤低點 × 條件加碼','status':'待統一重算','source':'v23','action':'固定既定八格與退出替補規則；原動能基準去重，統計失敗窗口覆蓋；不足25不事後放寬'},
{'id':'add_attribution','priority':'優先2','scope':'加碼與平均曝險相近固定投入對照','status':'待統一重算','source':'v24／v25','action':'只用2016～2018曝險校準對照；不得沿用用2024訓練出的比例當2016可知'},
{'id':'other_price_families','priority':'優先3','scope':'純低波動、短期反轉；必要時波動調整動能','status':'待統一重算','source':'v15／v19／v28','action':'既定定義與v30快照；v28反轉中位退出列候選待重驗，不直接納入新總結結論'},
{'id':'overlap_loss','priority':'優先3','scope':'持股重疊、虧損歸因、賣後反彈與回復時間','status':'部分可由v30軌跡整理，其餘待預先定義','source':'v17／v18','action':'先重用trace；賣後毛報酬不是觸底機率或交易策略績效'},
{'id':'macro','priority':'資料門檻','scope':'景氣燈號、PMI','status':'資料尚不足，不列有效策略','source':'v29來源稽核','action':'須2016～2025首次公布值／時間及改版口徑；最新修訂數列不可直接回填'},
{'id':'fundamental_sector','priority':'資料門檻','scope':'營收MoM／YoY、財務／公告權重、產業季節性／輪動','status':'資料尚不足，不列有效策略','source':'舊公告研究與產業觀察','action':'須全可選池歷史公開時間／產業分類；近期個案不擴稱十年驗證；LLM批次維持暫不測'}]
v31_contract={'date':'2026-10-16','dataset':'v30同一資料快照：2016-07-01..2025-12-31；起點2016-09-30..2025-11-27','planned_windows':2236,'current150_not_historical150':True,'sources':v30_definition['snapshot_sha256'],'price_action_method':'v27修正後價格／股息／未知公司行動，v30法人資料與掛牌日門檻','common_quality':'60日價量／公司行動與20日完整外資＋投信；個別訊號若需額外資料須揭露覆蓋','fixed_execution':'24日、25檔目標、五日例行評估、1000股、原費稅及日末約束；變動因子事前標明','new_candidate_pairing':'同2236計畫日期，但不強迫所有候選有效日期完全一致；每項對照用相同有效窗口，列出自己的覆蓋與失敗，不將無效填0','primary_evaluation':'2016..2018校準，2019..2025後段；全期含校準只描述；歷史已看過非真正未見樣本外','dedup':'experiment_key+definition+source snapshot hashes+engine version；相同格重用；只重算受影響格','promotion':'只有本快照已驗證結果才列主要結論／主比較圖；舊版留歷史附錄且不得與主表排名','summary_outputs':'費後勝率、24日期望值、平均虧損／P10／回撤、費稅、窗口數、覆蓋、配對差區間、年份一致性','boxplots':'實際全部窗口點；桃紅中位數；0線；兩時段固定尺度；不把平均值畫成分布','new_runs_this_step':0,'unattended_schedule_changed':False}
(v30_root/'策略研究_統一資料集與重驗盤點_2026-10-16.json').write_text(json.dumps({'contract':v31_contract,'inventory':v31_inventory},ensure_ascii=False,indent=2),encoding='utf-8')
lines31=['# 統一資料集與重驗盤點','2026-10-16。使用者決定：過去與未來欲放進主要總結的證據，統一採v30長歷史快照；先盤點，不代表本次已重算全部待辦。','## 統一口徑',v31_contract['dataset'],'股票池仍是目前150檔的歷史可掛牌子集，不是歷年官方股票池；保留存活偏誤、未知公司行動與缺漏。共同資料快照不等於不同候選永遠有相同有效窗口。','兩策略目前共同1359／2236（60.8%），後段1041；未來每項比較需同日期配對並揭露覆蓋，不把新方法失敗窗口丟掉後宣稱改善。','同一年代的舊v27仍因價量／法人品質門檻不同，不能直接拼入v30主比較。新校準不能使用2024年資料倒回2016。','## 值得保留的證據與重算順序','|優先|研究項目|目前狀態|處理方式|','|---|---|---|---|']
for row31 in v31_inventory:lines31.append('|'+ '|'.join([row31['priority'],row31['scope'],row31['status'],row31['action']])+'|')
lines31+=['## 去重與交付','主線已完成的動能、法人及開賽前選家族直接重用；新方法只算新增格，依來源快照／定義／引擎版本去重。不得為了列總結而重跑同樣結果。','每組保留費後勝率、期望值、虧損、費稅、最大回撤、有效窗口與失敗原因；區間採原始時間位置區塊並處理家族比較。年份及風險不可只看平均值。','舊研究保留為歷史附錄；尚未重驗項目可以列問題／假說，不能列為已由統一長歷史證實有效或無效。','本次完成盤點與主報告箱形圖，不啟動上述所有重算、不新增排程或LLM實驗。下一批可先做優先1，確定實驗定義後一次跑，避免無限參數搜尋。']
v31_ip=v30_root/'策略研究_統一資料集與重驗盤點_2026-10-16.md';v31_ip.write_text('\n'.join(lines31),encoding='utf-8')
print('V31_INVENTORY_SAVED',{'items':len(v31_inventory),'new_backtests':0,'principal_dataset':'v30','inventory':v31_ip.name})


In [ ]:
# v31 true portfolio-window boxplots in fixed master summary; no resampling/no synthetic returns.
import plotly.graph_objects as go
from markdown_it import MarkdownIt
import re
v31_fig=go.Figure();v31_rng=np.random.default_rng(3131);v31_ymin=float(min(v30_ret[k][v30_common].min() for k in ['MOM','FLOW','SWITCH']))*100-3;v31_ymax=float(max(v30_ret[k][v30_common].max() for k in ['MOM','FLOW','SWITCH']))*100+3
v31_meta=[]
for pi31,(period31,mask31) in enumerate([('全期2016～2025（含校準）',v30_common),('後段2019～2025',v30_common&v30_later)]):
    n31=int(mask31.sum());ids31=np.flatnonzero(mask31)
    for ki31,k31 in enumerate(['MOM','FLOW','SWITCH']):
        yy31=v30_ret[k31][mask31]*100;color31=['#167368','#567bb5','#987446'][ki31];name31={'MOM':'純動能','FLOW':'純法人（外資＋投信）','SWITCH':'開賽前依波動選家族'}[k31]
        v31_fig.add_trace(go.Box(x=np.full(n31,ki31),y=yy31,width=.56,boxpoints=False,quartilemethod='linear',line={'color':color31},fillcolor='rgba(22,115,104,.10)',hoverinfo='skip',visible=pi31==0,showlegend=False))
        v31_fig.add_trace(go.Scatter(x=ki31+v31_rng.uniform(-.18,.18,n31),y=yy31,mode='markers',marker={'size':4,'opacity':.38,'color':color31},customdata=[[v30_results[k31][i]['start'],v30_results[k31][i]['end'],p30(v30_results[k31][i]['cost'])] for i in ids31],hovertemplate=name31+'<br>%{customdata[0]}→%{customdata[1]}<br>24日費後 %{y:.2f}%<br>費稅 %{customdata[2]}<extra></extra>',visible=pi31==0,showlegend=False))
        med31=float(np.median(yy31));v31_fig.add_trace(go.Scatter(x=[ki31-.24,ki31+.24],y=[med31,med31],mode='lines',line={'color':'#c43c79','width':4},hoverinfo='skip',visible=pi31==0,showlegend=False));assert len(yy31)==n31
    v31_meta.append({'period':period31,'n':n31,'points':n31*3})
v31_fig.add_hline(y=0,line_color='#173333',line_width=2)
v31_buttons=[]
for pi31,m31 in enumerate(v31_meta):v31_buttons.append({'label':m31['period'],'method':'update','args':[{'visible':[i//9==pi31 for i in range(18)]},{'title':{'text':m31['period']+'｜同'+str(m31['n'])+'窗口；費後24日報酬'}}]})
v31_fig.update_layout(template='plotly_white',title='全期2016～2025（含校準）｜同1359窗口；費後24日報酬',height=570,xaxis={'tickmode':'array','tickvals':[0,1,2],'ticktext':['純動能','純法人（外資＋投信）','開賽前依波動選家族'],'range':[-.6,2.6]},yaxis={'title':'24日費後報酬 %','range':[v31_ymin,v31_ymax]},margin={'t':125,'b':80},updatemenus=[{'type':'buttons','direction':'left','buttons':v31_buttons,'x':0,'y':1.16,'xanchor':'left','yanchor':'top','showactive':True}])
assert len(v31_fig.data)==18 and sum(len(t.y) for t in v31_fig.data if t.type=='scatter' and t.mode=='markers')==3*(1359+1041)
v31_fig_html=v31_fig.to_html(full_html=False,include_plotlyjs=True,div_id='unified-longhistory-boxplot',config={'responsive':True,'displaylogo':False})
v31_notice='## 統一資料集與主結論範圍\n\n自2026-10-16起，主要總結只使用v30長歷史快照及同口徑衍生結果：2016年7月至2025年底、2236計畫起點。不同候選必須同日期配對，逐一列出覆蓋，不能把同資料集誤當永遠同有效分母。近期舊研究保留為歷史附錄，尚未重驗不能直接當長歷史結論。\n\n目前可列主要比較：動能、外資＋投信、開賽前依波動選家族與市場環境分層。其餘值得重驗的項目、順序與資料缺口見[統一資料集與重驗盤點](策略研究_統一資料集與重驗盤點_2026-10-16.md)。本次只盤點及補箱形圖，沒有假稱待辦已重算。\n\n'
mp31=v30_root/'策略研究_總結報告.md';mt31=mp31.read_text(encoding='utf-8')
if '## 統一資料集與主結論範圍' not in mt31:mt31=mt31.replace('## 目前研究判斷',v31_notice+'## 目前研究判斷')
start31=mt31.index('## 目前研究判斷');end31=mt31.index('## 統計讀法與交易口徑')
new31="""## 目前研究判斷

以下均以統一v30長歷史為主。全期共同1359窗口：動能勝率66.67%、24日期望+2.44%，法人67.77%、+1.38%，開賽前依波動選家族67.11%、+2.21%；均已扣費。

1. 動能保留20／60各半、每五日評估作基準，尚未以統一長歷史證明優於純20日／60日。
2. 低波動選法人、高波動選動能沒有證實優於純動能。後段2019～2025，切換期望2.74%對純動能2.82%，平均差−0.076pp，校正區間跨0。
3. 法人平均虧損較小，但最差回撤−30.94%深於動能−27.65%，不能概括稱風險更低。
4. 投入比例、權重、跌破退出、加碼等先前結果，尚未以v30同品質門檻重驗；下方舊證據只作歷史紀錄。v27雖同年代，但價量門檻不同，不直接混排。
5. 尚未證明穩定超越同期市場正式總報酬基準；宏觀／產業／公告資料缺口仍保留。

"""
mt31=mt31[:start31]+new31+mt31[end31:]
for heading31 in ['## 證據一：','## 證據二：','## 證據三：','## 證據四：','## 其他已測方向','## 新增證據：市場環境分層 v29']:
    mt31=mt31.replace(heading31,heading31.replace('## ','## 舊版歷史紀錄｜',1))
mt31=mt31.replace('## 更新紀錄','## 更新紀錄\n\n- 2026-10-16：固定v30為主要總結共同快照；盤點12項，舊版標歷史待重驗。主頁新增全期／後段真實箱形圖、全部窗口點、桃紅中位數及固定0線尺度。')
mp31.write_text(mt31,encoding='utf-8');md31=MarkdownIt('commonmark',{'html':False}).enable('table')
hp31=v30_root/'策略研究_總結報告.html';ht31=hp31.read_text(encoding='utf-8')
box31='<section id="summary-boxplot"><h2>統一長歷史：箱形圖與全部窗口</h2><p>同一v30資料快照；預設全期1359共同窗口，可切換2019～2025後段1041窗口。桃紅線為中位數，散點為每個實際窗口；箱體為四分位、鬚線採1.5倍四分位距。已扣費稅，0線與兩時段刻度固定。不是每筆交易勝率，也不是獨立比賽。</p>'+v31_fig_html+'<p>圖中家族選擇在窗口開始前決定，24日內維持該家族每五日規則；不是每日切換。不同顏色代表方法；只保留散點提示，避免重複窗格。</p><a href="策略研究_統一資料集與重驗盤點_2026-10-16.md">查看重驗盤點與優先順序 →</a></section>'
if 'id="summary-boxplot"' not in ht31:ht31=ht31.replace('<section id="overview"',box31+'<section id="overview"')
ht31=ht31.replace('<aside aria-label="章節導覽">','<aside aria-label="章節導覽"><a href="#summary-boxplot">長歷史箱形圖</a>')
ht31=ht31.replace('動能與純法人是主要候選；20／60 日各半、五日評估是動能基準，尚未證明最佳。提高投入可增加均值，也會放大虧損。','主要總結統一使用v30長歷史。動能保留各半／五日作基準；投入比例、退出與加碼等舊結論尚待同口徑重驗。')
cards31="""<div class="cards">
<article class="evidence" data-status="supported"><div class="tag">統一長歷史 · 1359共同窗口</div><h3>動能與法人都有正均值</h3><p>動能勝率66.67%、期望+2.44%；法人67.77%、+1.38%。正均值不等於已證明超越市場。</p></article>
<article class="evidence" data-status="unproven"><div class="tag">同快照已完成 · 尚未證實改良</div><h3>依波動選家族</h3><p>後段切換期望2.74%，動能2.82%；差值區間跨0。法人最差回撤也更深，不能概括較安全。</p></article>
<article class="evidence" data-status="unproven"><div class="tag">優先1 · 待同口徑重驗</div><h3>選股週期、持有與投入</h3><p>純20／60日、持有對五日調整、80／90／95%投入。既有基準重用，只算新增格；補正式市場基準。</p></article>
<article class="evidence" data-status="unproven"><div class="tag">優先2 · 待同口徑重驗</div><h3>權重、退出與加碼</h3><p>溫和非等權、原三種跌破退出、條件加碼與平均曝險匹配。舊結果僅作研究動機，尚非十年結論。</p></article>
<article class="evidence" data-status="unproven"><div class="tag">資料門檻 · 尚無統一長歷史</div><h3>景氣、營收、產業與公告</h3><p>必須有首次公布時間、歷史分類及全池覆蓋；不使用近期資料冒稱十年驗證。LLM批次仍暫不測。</p></article>
</div>"""
ht31=re.sub(r'<div class="cards">.*?</div></section>',lambda m:cards31+'</section>',ht31,count=1,flags=re.S)
ht31=ht31.replace('<button type="button" data-filter="worse" aria-pressed="false">已測較差</button>','')
ht31=ht31.replace('const measures=','// UNIFIED_DATASET_ONLY\nexperiments.splice(2);\nconst measures=')
render31=md31.render(mt31).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')
ht31=re.sub(r'(<article id="reportBody">).*?(</article>)',lambda m:m[1]+render31+m[2],ht31,flags=re.S)
ht31=ht31.replace('這裡整理已完成回測與下一個研究方向，不把研究規劃當成交易成果。','主比較只採2016～2025統一快照，舊研究保留為歷史附錄；待重驗不當成已完成成果。')
ht31=ht31.replace('主線更新v30 · 2016～2025長歷史環境比較','主線v30 · 統一長歷史資料 · 箱形圖及重驗盤點已更新')
hp31.write_text(ht31,encoding='utf-8')
state31p=v30_root/'A策略_無人值守研究狀態.json';state31=json.loads(state31p.read_text(encoding='utf-8'));state31['summary_dataset_contract_v31']={'status':'inventory_and_boxplots_completed','running':False,'dataset':'v30','inventory':v31_inventory,'current_step_new_runs':0,'boxplot_points':v31_meta,'raw_v30_outputs_reused':True,'next':'priority1 definitions then finite batch;this step does not claim recomputation complete'};state31p.write_text(json.dumps(state31,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V31_SUMMARY_UPDATED',{'boxplot_traces':len(v31_fig.data),'periods':v31_meta,'new_backtests':0,'master_only_v30_comparisons':True,'inventory_items':len(v31_inventory)})


In [ ]:
# v32 editorial redesign only; reuse verified v30 statistics and v31 boxplot data.
import html as html32,re
from markdown_it import MarkdownIt
v32_periods={}
for key32,title32,mask32,tracegroup32 in [('all','全期 2016～2025',v30_common,0),('later','後段 2019～2025',v30_common&v30_later,1)]:
    src32=v30_summary['overall' if key32=='all' else 'later'];v32_periods[key32]={'title':title32,'n':src32['MOM']['n'],'tracegroup':tracegroup32,'methods':src32}
v32_names={'MOM':'純動能','FLOW':'純法人買超','SWITCH':'開賽前依波動選策略'}
v32_md=['# 策略研究總結','更新：2026-10-16。主要證據統一採2016～2025長歷史資料（v30）。[互動版報告](策略研究_總結報告.html)。本次重新編排，沒有新增回測。','## 先讀這三個結論','1. **目前保留純動能作比較基準。**同1359窗口，動能24日期望+2.44%、法人+1.38%。這表示本次平均收益較高，不等於已證明所有年份最佳或超越市場。','2. **依波動選策略，尚未證實改善動能。**使用2016～2018校準、2019～2025檢查，切換期望2.74%、動能2.82%；平均差−0.076pp，校正區間跨0。','3. **近期的舊結論需要重新驗證。**「法人較安全」「加碼／退出不如原策略」「少留現金較好」不能直接升格為統一長歷史結論；法人長歷史最差回撤甚至更深。','## 我們在比較什麼？','|方法|開賽前怎麼選|24日內怎麼做|','|---|---|---|','|純動能|20日與60日漲幅排名各半，選前25檔|每五日重新評估及配置|','|純法人買超|前20日外資＋投信淨買超／成交量排名，選前25檔|每五日重新評估及配置|','|依波動選策略|開賽前低波動選法人、高波動選動能|24日內固定這個家族，沿用其五日規則；不每日切換|','共同設定：90%投入、25檔目標、24交易日、整千股、買賣各0.1425%手續費及賣出0.3%稅。25檔與90%是實驗控制，不是官方唯一要求。期末按資產估值，沒有假設全部清倉。','## 箱形圖與數字怎麼看？','互動版可用一組按鈕同步切換箱形圖與表格。每個點是一段24交易日的投資組合費後報酬；不是單筆股票交易，也不是獨立比賽。桃紅線為中位數，箱體為中間50%，鬚線為1.5倍四分位距；0線以上獲利、以下虧損。中位數與平均期望值不同，沒有把平均數當中位數。']
for key32 in ['all','later']:
    block32=v32_periods[key32];v32_md+=['### '+block32['title']+'：同'+str(block32['n'])+'窗口',('全期包含校準期，適合描述完整分布。' if key32=='all' else '波動門檻只用2016～2018特徵校準；這段後期歷史仍已被其他研究看過，不稱真正未見樣本外。'),'|方法|勝率|24日期望值|虧損時平均|最差窗口內回撤|平均費稅|','|---|---:|---:|---:|---:|---:|']
    for k32,m32 in block32['methods'].items():v32_md.append('|'+ '|'.join([v32_names[k32],p30(m32['win']),p30(m32['ev']),p30(m32['mean_loss']),p30(m32['worst_mdd']),p30(m32['fee'])])+'|')
v32_md+=['勝率＝24日費後正報酬比例；期望值＝包含虧損的平均報酬，不年化；虧損時平均＝只看負報酬窗口；最大回撤＝期間峰谷跌幅，不等於期末虧損。收益已扣費，費稅欄不要再扣一次。','## 這些結果代表什麼？','- 純法人略高的全期勝率，沒有轉成較高平均收益；其平均虧損較小，但最差回撤較深，所以「風險較小」不能一概而論。','- 低波動不代表一定選法人：2019～2025低波動265窗口，動能期望1.84%、法人1.55%；法人勝率67.17%較高，動能63.77%。勝率與期望值回答不同問題。','- 高波動組2019～2025共776窗口，動能期望3.15%、法人1.55%。這是歷史分組，不代表應主動追求高風險。','- 開賽前切換相對動能，後段平均差−0.076pp，48日起點區塊／五比較校正區間[-0.431,+0.291]pp：可能較好也可能較差，尚不能取代動能基準。','- 這次沒有新測高投入、停損或加碼；相關舊結果只作下一批研究的問題來源。','## 這份證據的邊界','行情2016/07/01～2025/12/31，共2320交易日；暖機後2236個計畫起點，第一起點2016/09/30、最後2025/11/27。兩策略與環境共同有效1359（60.8%），其中2019～2025為1041。2016不是完整年度。','仍使用目前150檔的歷史可掛牌子集，有存活／選擇偏誤；未知公司行動、缺價及違規窗口未填0。不同新候選可能有不同有效日期，必須同日期配對並列覆蓋。滑動窗口重疊，不是1359次獨立比賽。','股價波動環境不等於景氣燈號。本次沒有完成景氣燈號／PMI首次公布值回測，也未證明穩定超越正式市場總報酬基準；Active Share與正式DataPack仍未全驗證。','## 接下來要做什麼？','|順序|工作|目前狀態|','|---|---|---|','|已完成|動能、法人、市場環境、開賽前依波動選家族|可直接重用v30結果|','|下一批優先1|純20／60日、持有對五日評估、80／90／95%投入、同期市場基準|待統一資料重驗；只算新增格|','|優先2|等權／非等權、三種跌破退出、條件加碼與曝險匹配|待重驗，舊結果不當作十年定論|','|後續與資料門檻|反轉／低波動、虧損歸因；景氣、營收、公告與產業|前者待重驗，後者先補公開時間／歷史分類；LLM仍暫不測|','詳見[12項完整盤點](策略研究_統一資料集與重驗盤點_2026-10-16.md)。待辦不代表已執行，不新增定時排程。','## 查核來源與舊研究','目前主要依據：[v30長歷史完整報告](長歷史環境研究_動能法人比較_v30_2026-10-16.md)、[全部環境分層與年度圖](長歷史環境研究_全部點與年度比較_v30_2026-10-16.html)、[統計JSON](長歷史環境研究_統計與分層_v30_2026-10-16.json)、[原Notebook](動能持股_開賽前公告配置研究_2026-10-14.ipynb)、[競賽規則](競賽規則與每日運作說明_2026-10-13.md)。','舊版僅作歷史研究與待驗假說：','- [v16權重與交易機制](等權非等權_統計與選股退出解讀_v16_2026-10-15.md)','- [v20純20／60日](動能20日60日_五日調整比較_v20_2026-10-15.md)','- [v23三種退出與加碼](動能_三種跌破退出與加碼研究_v23_2026-10-15.md)','- [v25加碼曝險匹配](動能_加碼增益是否只是曝險研究_v25_2026-10-15.md)','- [v27價量版長歷史投入](動能_2016至2025全年現金比例研究_v27_2026-10-15.md)：同年代但品質門檻不同，不能直接與v30混排。','- [公開交易教程研究](公開交易教程_24日競賽策略研究.md)：外部方法，不是本專案已驗證績效。','## 維護與更新紀錄','後續主要結論採同一v30快照及同口徑衍生資料；以實驗定義、來源雜湊與引擎版本去重。新增重要結果／資料修正時，同步更新這份Markdown與同名HTML，取代舊判斷並標明原因，不把各版全文一直堆進主頁。','2026-10-16：整合v16～v27；新增v29環境、v30長歷史及v31統一資料與箱形圖。此次v32只重編閱讀順序、連動圖表與表格、折疊技術／歷史來源，沒有新增或修改績效。']
v32_markdown='\n\n'.join(v32_md);(v30_root/'策略研究_總結報告.md').write_text(v32_markdown,encoding='utf-8')
# Only presentation layout changes: same18 traces, values and fixed range.
v31_fig.update_layout(title=None,updatemenus=[],height=510,margin={'t':25,'b':65},xaxis={'tickmode':'array','tickvals':[0,1,2],'ticktext':['純動能','純法人買超','依波動選策略']})
v32_plot=v31_fig.to_html(full_html=False,include_plotlyjs=True,div_id='unified-longhistory-boxplot',config={'responsive':True,'displaylogo':False})
v32_template=r"""<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>策略研究總結｜目前知道什麼</title>
<style>:root{--ink:#193932;--sub:#5c6f66;--line:#dce4dd;--accent:#197365;--pink:#c43c79}*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:22px}body{margin:0;background:#f6f7f2;color:var(--ink);font:16px/1.8 system-ui,"Microsoft JhengHei",sans-serif}a{color:#146c62;text-underline-offset:4px}button{font:inherit;cursor:pointer;color:var(--ink);background:white;border:1px solid var(--line);border-radius:5px;padding:8px 16px}button[aria-pressed=true]{background:var(--accent);color:white;border-color:var(--accent)}button:focus-visible,a:focus-visible,summary:focus-visible{outline:3px solid var(--pink);outline-offset:3px}.page{max-width:1240px;margin:auto;padding:40px 28px}.kicker{font-size:12px;letter-spacing:2px;color:var(--accent);font-weight:700}header{border-bottom:1px solid var(--line);padding-bottom:25px}h1{font-size:clamp(30px,4.5vw,46px);line-height:1.3;letter-spacing:-1px;margin:8px 0 12px}header p{max-width:870px;margin:0}.sub{color:var(--sub)}.grid{display:grid;grid-template-columns:166px minmax(0,1fr);gap:38px;margin-top:28px}nav{position:sticky;top:24px;align-self:start;font-size:14px}nav a{display:block;text-decoration:none;padding:8px 0}main{min-width:0}section{padding:8px 0 28px;border-bottom:1px solid var(--line);margin-bottom:25px}h2{font-size:25px;line-height:1.5;margin:0 0 16px}h3{font-size:18px;margin:8px 0}.number{color:var(--accent);font:13px/1.5 system-ui;letter-spacing:1px;display:block;margin-bottom:6px}.conclusions{display:grid;gap:0}.conclusion{display:grid;grid-template-columns:36px 1fr;padding:17px 0;border-bottom:1px solid var(--line)}.conclusion:last-child{border:0}.conclusion b.n{font-size:25px;color:#82a394}.conclusion p{margin:0}.conclusion strong{font-size:18px}.methods{display:grid;grid-template-columns:repeat(3,1fr);gap:14px}.method{background:white;border:1px solid var(--line);padding:17px;border-radius:7px}.method p{font-size:14px;margin:5px 0}.common{padding:12px 16px;border-left:3px solid var(--accent);background:#eaf1e9;margin:18px 0}.period{display:flex;gap:8px;flex-wrap:wrap;align-items:center}.period p{width:100%;margin:0;font-size:14px}.plot{margin-top:14px;min-height:400px;background:white}.legend{font-size:14px;color:var(--sub);margin:8px 0 16px}.pink{color:var(--pink);font-weight:700}.table-wrap{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:14px}td,th{padding:11px 9px;border-bottom:1px solid var(--line);text-align:right;white-space:nowrap}td:first-child,th:first-child{text-align:left}th{font-weight:600;color:var(--sub);background:#edf2ec}tbody tr:last-child{background:#f0f3ec}.readout{border-left:3px solid #ad8b50;padding:12px 16px;background:#f6f0e4;margin-top:18px}.readout p{margin:0}.definition{font-size:14px;color:var(--sub);margin-top:12px}.flow{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:16px 0}.flow div{border-bottom:2px solid #8aa99b;padding:9px 0}.flow b{display:block;font-size:25px}.flow span{font-size:13px}.reading p{margin:10px 0}.status-table td{text-align:left;white-space:normal;min-width:100px}.status-table td:nth-child(2){min-width:230px}.status-table tr:last-child{background:none}details{padding:14px 0;border-top:1px solid var(--line)}summary{cursor:pointer;font-weight:650}details p,details ul{font-size:14px}.sources{display:flex;gap:10px 20px;flex-wrap:wrap}.footer{font-size:13px;color:var(--sub);margin-top:25px}.linkrow{display:flex;gap:18px;flex-wrap:wrap;margin-top:15px;font-size:14px}@media(max-width:850px){.grid{display:block}nav{position:static;display:flex;gap:8px 18px;flex-wrap:wrap;margin-bottom:24px}.methods{grid-template-columns:1fr}.flow{grid-template-columns:repeat(2,1fr)}.page{padding:25px 16px}}@media print{nav,.period button,.linkrow button{display:none}.grid{display:block}.page{padding:0}body{background:white}section,.method{break-inside:avoid}a{color:inherit}}</style></head><body><div class="page">
<header><div class="kicker">24交易日競賽 · 統一長歷史研究</div><h1>目前知道什麼，還不能決定什麼？</h1><p>先看結論，再用相同樣本比較收益與風險。這份報告只把已用2016～2025統一資料驗證的結果放進主要比較；舊版研究留作後續待驗證問題。</p><div class="linkrow"><span class="sub">更新：2026-10-16 · 主要證據 v30</span><a href="策略研究_總結報告.md">Markdown版</a><button id="print" type="button">列印／另存PDF</button></div></header>
<div class="grid"><nav aria-label="報告目錄"><a href="#conclusion">1. 三個結論</a><a href="#methods">2. 比較的方法</a><a href="#results">3. 箱形圖與數字</a><a href="#meaning">4. 如何解讀</a><a href="#limits">5. 證據的邊界</a><a href="#next">6. 接下來做什麼</a><a href="#sources">來源與舊研究</a></nav><main>
<section id="conclusion"><span class="number">01 / 目前的判斷</span><h2>保留動能基準，先不要把波動切換當成改良。</h2><div class="conclusions">
<div class="conclusion"><b class="n">1</b><div><strong>動能的平均收益較高。</strong><p>全期相同1359窗口，動能24日期望＋2.44%，法人＋1.38%。目前保留動能作比較基準，尚未證明所有年份都最好或能超越市場。</p></div></div>
<div class="conclusion"><b class="n">2</b><div><strong>「低波動選法人、高波動選動能」尚未勝出。</strong><p>2019～2025後段，這個方法期望＋2.74%，純動能＋2.82%；校正後差異區間跨零，還不能直接採用。</p></div></div>
<div class="conclusion"><b class="n">3</b><div><strong>先前的退出、加碼與投入比例結論，還要重驗。</strong><p>舊版資料期間或品質門檻不同。它們提供研究方向，不能直接當成這份統一長歷史的定論；法人也不能一概稱為較安全。</p></div></div></div></section>
<section id="methods"><span class="number">02 / 方法與共同條件</span><h2>這裡只有三種方法，差別在如何選股。</h2><div class="methods"><article class="method"><h3>純動能</h3><p>過去20日與60日漲幅排名，各占一半，選前25檔。</p><p class="sub">每五日重新評估及配置。</p></article><article class="method"><h3>純法人買超</h3><p>過去20日「外資＋投信淨買超／成交量」排名，選前25檔。</p><p class="sub">這裡的法人不包括自營商；每五日重新評估。</p></article><article class="method"><h3>開賽前依波動選策略</h3><p>開賽前低波動用法人，高波動用動能。</p><p class="sub">24日內固定這個策略家族，維持其五日規則，不每日切換。</p></article></div><div class="common">三組共同：90%投入、25檔目標、24交易日、整千股及相同費稅。25檔與90%是研究控制，還不是最佳配置的結論。</div></section>
<section id="results"><span class="number">03 / 同樣本比較</span><h2>看分布，也看勝率、期望值與下跌風險。</h2><div class="period" aria-label="比較期間"><button type="button" data-period="all" aria-pressed="true">全期 2016～2025</button><button type="button" data-period="later" aria-pressed="false">後段 2019～2025</button><p id="period-note" class="sub" aria-live="polite"></p></div><div class="plot">%%PLOT%%</div><p class="legend"><span class="pink">桃紅線＝中位數</span> · 箱體＝中間50%的窗口 · 每個點＝一段24日費後報酬 · 0線以上獲利、以下虧損。兩時段刻度固定；中位數不是下表的平均期望值。</p><div class="table-wrap"><table><thead><tr><th>方法</th><th>勝率</th><th>24日期望值</th><th>虧損時平均</th><th>最差回撤</th><th>平均費稅</th></tr></thead><tbody id="metrics-body"></tbody></table></div><p class="definition">勝率：窗口最後是否賺錢。期望值：包含虧損的平均收益，不年化。虧損時平均：只看負報酬窗口。回撤：窗口內從高點下跌的幅度。報酬已扣費，費稅欄不要再扣一次。</p><div class="readout"><p id="period-takeaway"></p></div></section>
<section id="meaning" class="reading"><span class="number">04 / 如何用這些結果思考</span><h2>勝率高，不一定賺得多；平均虧損小，也不保證極端風險小。</h2><p><strong>法人就是一個例子。</strong>全期勝率略高、虧損窗口平均較小，但收益均值較低，最差回撤−30.94%也深於動能的−27.65%。</p><p><strong>低波動時，兩種指標也會給不同答案。</strong>後段低波動265窗口，法人勝率67.17%、動能63.77%；但期望值是法人＋1.55%、動能＋1.84%。所以「比較常賺」與「平均賺比較多」不能混為一談。</p><p><strong>切換策略仍是待驗證假說。</strong>後段切換減純動能平均−0.076個百分點，校正區間−0.431～＋0.291個百分點，還包含兩種可能方向。這個區間描述平均差的不確定性，不是下一場報酬範圍。</p><a href="長歷史環境研究_全部點與年度比較_v30_2026-10-16.html">想深入：各環境全部點、逐年結果與完整定義 →</a></section>
<section id="limits"><span class="number">05 / 證據的邊界</span><h2>十年時間範圍，不代表每一天都能判定。</h2><div class="flow"><div><b>2320</b><span>原始行情交易日</span></div><div><b>2236</b><span>暖機後計畫窗口</span></div><div><b>1359</b><span>全期共同有效窗口</span></div><div><b>1041</b><span>後段共同有效窗口</span></div></div><p>行情是2016/07/01～2025/12/31，第一個窗口從2016/09/30開始。共同有效覆蓋60.8%；窗口互相重疊，並不是1359次獨立比賽。</p><p>主要限制：使用目前150檔向前回看，有存活／選擇偏誤；未知公司行動、缺價或違規窗口未當作0報酬。尚未證明超越正式市場總報酬基準，也未完整驗證官賽DataPack與Active Share。</p><details><summary>展開：校準、交易費與資料口徑</summary><p>波動門檻只用2016～2018年的特徵校準，2019～2025作後段檢查；這段歷史先前也被研究過，不是真正未見樣本外。全期包含校準期，因此只作描述。</p><p>買賣手續費各0.1425%，賣出另0.3%稅；期末以資產估值，沒有強制全部清倉。訊號只用開賽前已知資訊；前收盤換股數、次日均價成交，不是假設盤中精準停損。</p><p>兩家族共用60日價量／公司行動與20日法人資料完整門檻。法人為外資＋投信；外資分類在2017／2018拆分外資自營商，仍有年代口徑限制。新方法必須同日期配對並揭露缺漏，不能只看成功窗口。</p></details></section>
<section id="next"><span class="number">06 / 接下來的研究</span><h2>先回答選股與投入，再處理複雜調整。</h2><div class="table-wrap"><table class="status-table"><thead><tr><th>狀態</th><th>要回答的問題</th><th>工作</th></tr></thead><tbody><tr><td>已完成</td><td>動能、法人及依波動選家族，誰表現如何？</td><td>重用目前統一長歷史結果。</td></tr><tr><td>下一批優先1</td><td>選股週期與投入，哪個影響較大？</td><td>純20／60日、持有對五日評估、80／90／95%投入，並補同期基準。</td></tr><tr><td>優先2</td><td>配置或調整能否帶來額外收益？</td><td>權重、三種退出、條件加碼與平均曝險匹配，使用同一資料重驗。</td></tr><tr><td>資料門檻</td><td>景氣、營收或產業能否補強選股？</td><td>先補首次公布值／時間與歷史分類；股價波動不等於景氣燈號。</td></tr></tbody></table></div><p class="sub">上表待辦尚未重算。反轉、低波動及虧損歸因列後續；LLM批次維持暫不測，不新增排程。</p><a href="策略研究_統一資料集與重驗盤點_2026-10-16.md">查看12項完整盤點與去重方式 →</a></section>
<section id="sources"><h2>需要查核時，再往下看。</h2><div class="sources"><a href="長歷史環境研究_動能法人比較_v30_2026-10-16.md">v30完整研究</a><a href="長歷史環境研究_統計與分層_v30_2026-10-16.json">統計資料</a><a href="動能持股_開賽前公告配置研究_2026-10-14.ipynb">原Notebook</a><a href="競賽規則與每日運作說明_2026-10-13.md">競賽規則</a></div><details><summary>舊版研究：保留紀錄，尚未全部用統一資料重驗</summary><p>下列結果使用不同期間或品質門檻，提供研究動機，不能與上方主表混排。</p><ul><li><a href="等權非等權_統計與選股退出解讀_v16_2026-10-15.md">v16：權重與交易機制</a></li><li><a href="動能20日60日_五日調整比較_v20_2026-10-15.md">v20：純20／60日</a></li><li><a href="動能_三種跌破退出與加碼研究_v23_2026-10-15.md">v23：三種退出與加碼</a></li><li><a href="動能_加碼增益是否只是曝險研究_v25_2026-10-15.md">v25：加碼曝險匹配</a></li><li><a href="動能_2016至2025全年現金比例研究_v27_2026-10-15.md">v27：價量版長歷史投入</a>，同年代但品質門檻不同。</li><li><a href="公開交易教程_24日競賽策略研究.md">公開交易教程</a>是外部方法，不等於本專案已驗證績效。</li></ul></details><details><summary>報告如何持續更新？</summary><p>主要結論採同一v30快照與同口徑衍生資料，按定義、快照與引擎版本去重。有重要結果或資料修正時，同步更新Markdown與本頁，清楚取代舊判斷，不再將每版全文堆在主頁。</p><p>2026-10-16：整合v16～v27，新增v29環境、v30長歷史、v31統一資料與箱形圖。此次只重新編排與連動圖表／表格，沒有新增或修改回測績效。</p></details></section>
<footer class="footer">統一資料、相同日期、費後比較。尚未完成的研究保留為待辦；這份報告沒有提交交易。</footer>
</main></div></div><script>
'use strict';
const periods=%%PERIODS%%;
const methodNames={MOM:'純動能',FLOW:'純法人買超',SWITCH:'開賽前依波動選策略'};
const node=id=>document.getElementById(id);
function percent(x,signed=false){return x===null?'未列':(signed&&x>0?'＋':'')+(x*100).toFixed(2)+'%'}
function renderPeriod(key,updatePlot=true){const p=periods[key];document.querySelectorAll('[data-period]').forEach(b=>b.setAttribute('aria-pressed',String(b.dataset.period===key)));node('period-note').textContent=p.title+' · 三種方法同'+p.n+'個有效窗口。'+(key==='all'?'包含2016～2018校準期，用來看完整分布。':'門檻由2016～2018校準；後期歷史仍已被看過，並非全新樣本外。');const body=node('metrics-body');body.replaceChildren();for(const k of ['MOM','FLOW','SWITCH']){const m=p.methods[k],tr=document.createElement('tr');[methodNames[k],percent(m.win),percent(m.ev,true),percent(m.mean_loss),percent(m.worst_mdd),percent(m.fee)].forEach(t=>{const td=document.createElement('td');td.textContent=t;tr.append(td)});body.append(tr)}node('period-takeaway').textContent=key==='all'?'這張表的重點：動能平均收益較高；法人勝率略高，但最差回撤更深。不要只用單一指標選策略。':'這張表的重點：依波動選策略的勝率略高、部分虧損指標較小，但平均收益沒有高於純動能；差異仍未證實。';if(updatePlot&&typeof Plotly!=='undefined')Plotly.update(node('unified-longhistory-boxplot'),{visible:Array.from({length:18},(_,i)=>Math.floor(i/9)===p.tracegroup)});}
document.querySelectorAll('[data-period]').forEach(b=>b.addEventListener('click',()=>renderPeriod(b.dataset.period)));
node('print').addEventListener('click',()=>window.print());renderPeriod('all',false);
</script></body></html>"""
v32_output=v32_template.replace('%%PLOT%%',v32_plot).replace('%%PERIODS%%',json.dumps(v32_periods,ensure_ascii=False))
(v30_root/'策略研究_總結報告.html').write_text(v32_output,encoding='utf-8')
state32p=v30_root/'A策略_無人值守研究狀態.json';state32=json.loads(state32p.read_text(encoding='utf-8'));state32['summary_redesign_v32']={'status':'completed','new_backtests':0,'one_period_control_for_plot_and_table':True,'legacy_material_collapsed':True,'principal_dataset':'v30','source_metrics_unchanged':True};state32p.write_text(json.dumps(state32,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V32_REPORT_REDESIGNED',{'sections':7,'main_periods':list(v32_periods),'box_traces':len(v31_fig.data),'new_backtests':0,'html_bytes':len(v32_output.encode('utf-8'))})


In [ ]:
# v32 formatting verification only; no numerical research.
p32=v30_root/'策略研究_總結報告.md';t32=p32.read_text(encoding='utf-8');t32=re.sub(r'(?m)(^\|.*\|)\n\n(?=\|)',r'\1\n',t32);p32.write_text(t32,encoding='utf-8')
from markdown_it import MarkdownIt
render32=MarkdownIt('commonmark',{'html':False}).enable('table').render(t32)
assert render32.count('<table>')==4,render32.count('<table>')
assert '%%PLOT%%' not in v32_output and '%%PERIODS%%' not in v32_output
assert len(v31_fig.data)==18
assert [v32_periods[k]['n'] for k in ['all','later']]==[1359,1041]
print('V32_FORMAT_VERIFIED',{'markdown_tables':render32.count('<table>'),'placeholder_free':True,'boxplots_preserved':True})


In [ ]:
# v33 executable research specifications and continuation queue; no new strategy runs.
import hashlib,json,re
v33_root=v30_root
v33_common={'dataset':'v30_frozen_2016_2025','source_hashes':v30_definition['snapshot_sha256'],'engine_reference_sha256':hashlib.sha256(v30_engine.encode()).hexdigest(),'price_action':'v27 cashwealthreturns and explicit repairs; unknown actions remain invalid','universe':'same current150 conservative listing gate; not historical official150','eligibility':'same v30:60-session price/action plus20-session complete foreign+trust/volume','dates':'2016-07-01..2025-12-31;2236 starts2016-09-30..2025-11-27','horizon':24,'initial':1000000000,'target_names':25,'invested_default':.9,'rebalance_default':5,'lot':1000,'fee':.001425,'sell_tax':.003,'cash_interest':0,'terminal_liquidation':False,'constraints':'cash>=0 and<25%;hold20..30 actual;target25;caps2330 .25others .10 with original5-session grace;ActiveShare not fully verified','calibration':'windows ending<=2018-12-31;later starts>=2019-01-01;crossboundary omitted from calibration/later','no_new_llm':True,'existing_kernel':'reconnect retained kernel;metadata first;never reset without explicit consent'}
def spec33(key,signal='MOM',**changes):
    base={'experiment_key':key,'signal':signal,'lookback':[20,60],'mix':[.5,.5],'allocation':'equal','rebalance':5,'invested':.9,'small_trade_tolerance':0.,'exit':'none','conditional_add':'none'};base.update(changes);return base
v33_reused=[spec33('MOM_E0'),spec33('FLOW_E0','FLOW')]
v33_candidates=[spec33('MOM20_E0',lookback=[20],mix=[1.]),spec33('MOM60_E0',lookback=[60],mix=[1.]),spec33('MOM_HOLD',rebalance='initial_only'),spec33('FLOW_HOLD','FLOW',rebalance='initial_only'),spec33('LIQ_HOLD','LIQ',lookback=[20],mix=[1.],rebalance='initial_only',signal_formula='prior20mean Trading_money rank;provider raw money;no close*volume replacement')]
for fam in ['MOM','FLOW']:
    for exposure in [.8,.95]:v33_candidates.append(spec33(fam+'_P'+str(int(exposure*100)),fam,invested=exposure))
v33_primary=[['MOM20_E0','MOM_E0'],['MOM60_E0','MOM_E0'],['MOM20_E0','MOM60_E0'],['MOM_HOLD','MOM_E0'],['FLOW_HOLD','FLOW_E0'],['MOM_HOLD','LIQ_HOLD'],['FLOW_HOLD','LIQ_HOLD']]
for fam in ['MOM','FLOW']:v33_primary += [[fam+'_P80',fam+'_E0'],[fam+'_P95',fam+'_E0'],[fam+'_P95',fam+'_P80']]
assert len(v33_candidates)==9 and len(v33_primary)==13
for s33 in v33_reused+v33_candidates:s33['dedup_hash']=hashlib.sha256(json.dumps({'common':v33_common,'spec':s33},sort_keys=True).encode()).hexdigest()
v33_nodes=[
{'id':'bind_and_validate','status':'ready','depends_on':[],'definition':{'reuse':'v30 results and namespace;do not rerun4472 baseline rows','checks':['kernel execution idle before submit;inspect required object metadata','snapshot and engine hashes match','baseline counts MOM1621 FLOW1541 paired1359 later1041','10 existing price-only controls and flow cross-source audit reused','all definitions saved before new calculation'],'failure':'if kernel unavailable inspect saved sources;no shutdown/replacement without applicable consent'},'outputs':['v33 baseline binding and provenance'],'done_when':'baseline mappings and hashes validated;no duplicated baseline run'},
{'id':'lookback_20_60','status':'pending','depends_on':['bind_and_validate'],'experiments':['MOM20_E0','MOM60_E0'],'definition':{'score':'20-only or60-only adjustedreturn percentile;shared60history and20flow gate unchanged','holdings_and_execution':'same v30 equal90 25names5day;only score changes'},'outputs':['two new2236-window result arrays'],'done_when':'4472 rows saved;quality checks and exclusions preserved'},
{'id':'hold_and_selection_reference','status':'pending','depends_on':['bind_and_validate'],'experiments':['MOM_HOLD','FLOW_HOLD','LIQ_HOLD'],'definition':{'initial':'same t-1 signals and same targets as family baseline;only no later rebalancing','liquidity':'prior20mean raw Trading_money percentile,25names initialhold','drift':'keep same daily cash/count/cap/action checks;untradable/violations not silently waived'},'outputs':['three new2236-window result arrays'],'done_when':'6708 rows saved;initial day matches corresponding baseline forMOM/FLOW'},
{'id':'capital_80_95','status':'pending','depends_on':['bind_and_validate'],'experiments':['MOM_P80','MOM_P95','FLOW_P80','FLOW_P95'],'definition':{'only_change':'target totalstock allocation;reuse90forboth families','budget':'original priorclose-to-share formula,fees and actualVWAP;no hindsight resizing','literal100':'excluded fromcandidategrid dueknown negativecash;new nearfullbudget would require separate future definition'},'outputs':['four new2236-window result arrays'],'done_when':'8944 rows saved;cash violations reported;not compare own-successsubsamples'},
{'id':'official_reference_audit','status':'pending','depends_on':['bind_and_validate'],'definition':{'sources':'finite source checks for official2016..2025 totalreturn market reference and its coverage','meaning':'reference comparison,not necessarily competition-eligible holding','prohibitions':'do not relabel current150 changingmean as official index;price index is priceonly andcannot claim dividend-inclusive alpha','retry_limit':3,'failure':'save sourcegap and continue other branches;do not repeat previously security-denied route'},'outputs':['official benchmark coverage/source audit or explicit unavailable conclusion'],'done_when':'verified data or bounded exhausted audit;does not block strategycomparison'},
{'id':'priority1_comparison','status':'pending','depends_on':['lookback_20_60','hold_and_selection_reference','capital_80_95'],'definition':{'primary_contrasts':v33_primary,'family_size':13,'bootstrap':'24/48 original-start-position blocks,5000 each;Bonferroni13;keep missing positions','primary_period':'2019..2025;2016..2018calibration;allperiod descriptive','metrics':['net24daywin','expectancy','meanwin/meanloss','P10','worst_mdd','fees','coverage andinvalidreasons','pairedmeanCI','startyearconsistency'],'minimum_training_pairs':100,'coverage_gate':'candidate paired coverage>=80% of matched baseline valid dates;otherwise infeasible/insufficient not a winner','year_gate':'at least5 eligible later startyears with>=20 pairs/year;meanincrement positive inatleast2/3 of eligibleyears tocallconsistent','promotion':'mean improvement only if both correctedCI lower>0;full improvement additionally P10 andworstMDD notworse;otherwise risk-return tradeoff requiring explicit preference,not automatically best','no_result':'if nonepasses retain originalbaseline andsayunproven;do not pick highestpooledreturn'},'outputs':['statistics','all window plots with median0fixedscales','shortselection rationale'],'done_when':'13 contrasts and allcandidatesincludingfailures disclosed;sourcebranch mayremainblocked'},
{'id':'publish_priority1','status':'pending','depends_on':['priority1_comparison'],'definition':{'update':'same summary.md andsummary.html;replace pending rows onlyaftervalidation;retain plots allpoints','index':'save resultpaths definition/source/enginehash inthis queue'},'outputs':['updated fixed summary and status'],'done_when':'pairedsample labels, boxplot counts and links pass;no duplicate report dump'},
{'id':'weights_and_trade_buffer','status':'pending_definition_lock','depends_on':['publish_priority1'],'definition':{'finite_grid':'MOM/FLOW each E0(existing),E5(equal,tol.005),M0(rank1.25to.75,tol0),M5(rank1.25to.75,tol.005)','new_cells_max':6,'design':'four-factor-isolated cells;v30equalbaselines reused','must_lock_before_run':'candidate definitions,contrasts andfamilycorrection saved;notcopy2024calibratedparams'},'outputs':['weight vsbuffer separated comparison'],'done_when':'finitecomparison published,notjustbestcell'},
{'id':'stop_add_replication','status':'pending_definition_lock','depends_on':['publish_priority1'],'definition':{'finite_grid':'samev23 none/MA5/MA5two/prior5closeLow × addoff/on;8cells,MOMnoneoffreused','new_cells_max':7,'rule_source':'originalNotebookv23 cells292..295 exactrules;previousdaydecision,nextVWAP;nointradaystop','add':'market20positive andbreadth>=60%;heldR3,R5>0 andclose>MA5;90->95;state-changeor5dayonly,notdailyfullretarget','replacement':'one-for-one priorvalue allocation,healthyoriginalrank;target25;candidate/countfailinvalidnotposthocrelax','scope':'MOM onlyasoriginalv23;do notsilentlyduplicateforFLOW','must_lock_before_run':'fullrules ported tolongactions;nostop/add controls reproducev30;familycontrasts fixed'},'outputs':['eightcell comparison andcoverage'],'done_when':'allpoints andexcludedbadperiods disclosed'},
{'id':'exposure_matched_add','status':'pending_definition_lock','depends_on':['stop_add_replication'],'definition':{'minimum_scope':'existing90->95MOMadd vsfixedexposurematched reference','calibration':'only2016..2018 realizedexposure,notreturn;freeze1bp target before latercomparison','no_leakage':'no2024trainingweights,noeachtestwindowfutureexpo matching','reuse':'addcandidate and90baseline fromstopbranch;onlynew fixedcontrol calculated','deferred_optional':'80->85 and80->95 expansion notautoadded;separate definition if laterneeded'},'outputs':['increment split anduncertainty'],'done_when':'matchedreference actualexposure drift andfees disclosed'},
{'id':'other_families_and_attribution','status':'pending_definition_lock','depends_on':['publish_priority1'],'definition':{'price_candidates':'originalpurelowvol,REV5,andvolatilityadjustedmomentum;v28medianexitonlyafteritsbaselineverified','attribution':'reusev30tracesforoverlap;defineholdingsorstockPnL attribution beforeadditionalreplay','not_automatic':'notinpriority1batch;lockscope/contrastsfirst;noopenparametersearch'},'outputs':['bounded latercandidatecomparison andmechanismnote'],'done_when':'validatedsame-snapshotresults orclearlimitations'},
{'id':'macro_announcements_sector','status':'blocked_data','depends_on':[],'definition':{'needs':'2016..2025 firstreleasevalues/time,fullpoolcoverage, historicalindustryclassification','known_gap':'v29 dates/history obtained,originalvalues/timeunverified;currentclassificationcannotbackfill','retry_limit':3,'LLM':'remainnotrequestedfortesting','release_condition':'newverifiableoriginaldata available,notelapsedtime'},'outputs':['sourcegap audit'],'done_when':'dataqualified then separatelylockmodels;otherwise remainunverified'}]
v33_plan={'schema_version':1,'version':'v33_unified_research_continuation','created_at':'2026-10-16','status':'prepared_not_running','common':v33_common,'reused':v33_reused,'priority1_candidates':v33_candidates,'priority1_new_windows':9*2236,'priority1_contrasts':v33_primary,'queue_order':[x['id'] for x in v33_nodes],'nodes':v33_nodes,'execution_policy':{'single_kernel_writer':True,'already_running':'observe original session;do not resubmit orinterrupt because20minelapsed','continue':'finishallreadyfeasible nodes,saveoutputs beforemarkdone;partialdata gapdoesnotstopotherbranches','dedup':'experimentkey+spec+sourcehashes+engineversion;completedmatchedresultreuse;runningmatchednodeonlyobserve','recovery':'persistcurrent_node/tool session/cellindex;readminimalmetadataandhistory;neverexposecredentials','deferred':'pending_definition_lock mustfreezeexisting authorizedrulebeforecalculation;notnewpermissionflow','stop':'allfeasible nodes finished orconfirmedexternalblocker forremaining;noforcedwinner','schedule':'noautomationcreatedorreactivatedbythisplanningstep'},'resume_instructions':'Readthisplanandstate.unified_research_v33first;ignoreoldcompletedqueueasstopcondition;bindv30retainedkernelandreadmetadata;choosefirstdependenciessatisfiednode;reusecompletedmatchingresults;do notcreateproject/chat/subagent/repo orsubmittrades.'}
v33_jp=v33_root/'策略研究_統一長歷史續跑計畫_2026-10-16.json';v33_jp.write_text(json.dumps(v33_plan,ensure_ascii=False,indent=2),encoding='utf-8')
v33_md=['# 統一長歷史研究：續跑計畫','建立：2026-10-16。將總結報告待改進事項接成可恢復依賴佇列。此次完成計畫與節點定義，不代表新回測已開始；目前未新增或重啟排程。','## 目標與固定條件','目標是分清楚初始選股、調整頻率、投入比例及風控各自的作用；不用最高平均值直接挑贏家。主資料固定v30，行情2016/07～2025/12、2236個24日起點，現況150檔歷史可掛牌子集；保留存活偏誤與公司行動缺口。','25檔目標、整千股、原費稅、五日例行評估；單因子改動時才改對應設定。兩策略共用60日價量／公司行動及20日外資＋投信完整門檻。每項同日期配對，揭露覆蓋與失敗，無效不能填0。','## 接續順序','```mermaid\nflowchart TD\n B[綁定v30與去重驗證] --> L[純20日與純60日]\n B --> H[持有與初始選股對照]\n B --> C[80與95投入]\n B --> I[官方基準來源查核]\n L --> P[優先1配對統計]\n H --> P\n C --> P\n P --> S[更新固定總結]\n S --> W[權重與交易緩衝]\n S --> E[三種退出與條件加碼]\n E --> A[平均曝險匹配]\n S --> O[其他選股與歸因]\n D[首次公布與歷史分類缺口] -.資料達標後.-> M[宏觀公告產業研究]\n```','箭頭表示資料依賴；分析依然只用同一kernel依序執行，不並行寫Notebook。官方基準缺口不能阻擋其餘可行比較。','## 第一批：9個新方法，20124個窗口；舊基準不重跑','|節點|新增方法|對照／要回答的問題|','|---|---|---|','|選股回看|純20日、純60日，共2格|重用20／60各半；是否某個回看期更好？三者五日評估不變|','|持有與初始選股|動能初始持有、法人初始持有、成交金額排名25檔初始持有，共3格|前兩者對原五日策略隔離後續動作；後兩種家族各對成交金額排名持有，觀察初始排名作用|','|投入比例|動能80／95%、法人80／95%，共4格|90%各自重用；收益差是否只是曝險改變？現金違規不排除後假裝成功|','|官方基準來源|有限來源查核，不當作新交易格|正式總報酬基準有無覆蓋；不能把150檔研究代理稱官方指數|','成交金額對照用原始Trading_money，不用收盤×成交量偷換。持有策略只取消後續調整，日末現金、持股、公司行動、權重限制仍檢查。名目100%舊預算因負現金不列候選；接近滿倉的預算方法需要另訂，不在這批加網格。','## 比較與選擇邏輯','2016～2018作校準；主要看2019～2025，跨界不混入兩段。歷史已被看過，不能稱真正未見樣本外。','第一批13個主要配對事先固定：回看3對、持有／初始排名4對、兩家族投入各3對。24／48個原始起點位置區塊各5000次，13比較校正；同時列勝率、24日期望、平均虧損、P10、最差回撤、費稅及覆蓋。','分類而非強選：兩種校正區間的平均增量下界均>0才稱支持平均收益改善；P10與最差回撤也不劣才可稱全面改善。若收益上升但風險增加，列風險／報酬取捨，尚未替使用者設定可接受損失，不能自動選為最佳。','校準共同窗口至少100；相對對照有效日期覆蓋至少80%，否則可執行性／證據不足。跨年至少5個有20配對窗口的後段年份，其中至少三分之二平均增量為正才稱跨年一致。沒有通過就保留原基準並明寫未證實，不能擴充參數直到顯著。','## 後續分支','|分支|可接續的有限範圍|去重／前置|','|---|---|---|','|權重與小額略過|動能／法人各E0、E5、M0、M5四格|兩個E0重用，最多6新格；權重與交易緩衝分開|','|退出×加碼|原v23四退出狀態×加碼開關，共8格|動能無退出無加碼重用，最多7新格；原規則先移植驗證；不事後放寬25檔|','|曝險匹配|先只做90→95加碼對匹配固定投入|重用加碼結果；只以2016～2018平均曝險校準，不看收益，不沿用2024參數|','|其他家族與歸因|既定反轉、低波動、波動調整；重疊與虧損|先固定對照／定義，再計算；優先重用既有trace，不無限搜尋|','|景氣／公告／產業|待首次公布值、時間、全池覆蓋及歷史分類|缺資料不算績效，LLM仍暫不測；有新可驗證資料才解除阻礙|','後續各批在計算前鎖定完整對照家族與校正，不因第一批結果好壞改門檻。80→85／80→95等額外加碼組目前列延後選項，不自動擴增。','## 無人值守與恢復規則','1. 先讀機器版計畫及A策略_無人值守研究狀態.json中的unified_research_v33，再查既有kernel必要metadata與安全history。不能因舊queue標完成就停止新計畫。','2. 相同實驗鍵＋定義＋快照＋引擎版本已有結果就引用；仍正常執行只等待原session，不另送相同cell。','3. 單一Notebook寫入者。計算直接在既有kernel，程式與輸出保存Notebook；不重啟、不清除，不輸出憑證或寫DB。','4. 節點先保存結果與驗證，再標completed；記錄current_node、cellindex與tool session以恢復。來源最多三次有意義嘗試，缺口不偽造，不阻斷其他可行節點。','5. 每20分鐘若有檢查，僅是喚醒／恢復，不是執行期限。這份計畫本身沒有建立或啟用排程，也不代表已有後台程式在跑。','6. 完成一個小cell不是結束理由；接著做依賴已滿足的可行節點。全部可行節點完成，或其餘都遇到已確認外部阻礙時才結束。','## 完成標準與位置','主資料與引擎雜湊、候選定義、去重鍵、依賴及13主要對照在同名JSON。每批保存統計、全部窗口圖與選擇理由，再更新既有總結Markdown及HTML。保留中位數、0線、共同尺度與同日期分母。','目前下一節點：bind_and_validate。第一批9個新格尚未執行。後續對話要求「繼續」時，依本計畫接續，不需要重新盤點或重跑基準。']
v33_mdtext='\n'.join(v33_md);v33_mp=v33_root/'策略研究_統一長歷史續跑計畫_2026-10-16.md';v33_mp.write_text(v33_mdtext,encoding='utf-8')
v33_statep=v33_root/'A策略_無人值守研究狀態.json';v33_state=json.loads(v33_statep.read_text(encoding='utf-8'));v33_state['unified_research_v33']={'status':'prepared_not_running','plan_path':v33_jp.name,'plan_md':v33_mp.name,'current_node':None,'next_node':'bind_and_validate','running':False,'queue_order':v33_plan['queue_order'],'nodes':v33_nodes,'new_runs_completed':0,'priority1_candidates':9,'priority1_windows':20124,'do_not_reuse_old_queue_completion_as_stop':True};v33_statep.write_text(json.dumps(v33_state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
v33_summarymd=v33_root/'策略研究_總結報告.md';txt33=v33_summarymd.read_text(encoding='utf-8');txt33=txt33.replace('## 接下來要做什麼？','## 接下來要做什麼？\n\n已建立[可續跑計畫與依賴](策略研究_統一長歷史續跑計畫_2026-10-16.md)。第一批9個新方法、20124個窗口，原動能／法人基準重用；計畫已準備，尚未啟動新回測或排程。');txt33 += '\n\n2026-10-16：把待改進事項轉成v33續跑依賴與有限候選；保存13主要對照與選擇門檻，下一節點bind_and_validate。\n';v33_summarymd.write_text(txt33,encoding='utf-8')
v33_summaryhtml=v33_root/'策略研究_總結報告.html';html33=v33_summaryhtml.read_text(encoding='utf-8');snippet33='<div class="common"><strong>續跑計畫已準備</strong><br>第一批9個新方法、20,124個窗口；先做回看週期、持有對照、投入比例與基準查核。原動能／法人結果重用。<br><a href="策略研究_統一長歷史續跑計畫_2026-10-16.md">查看依賴、實驗定義與完成標準 →</a><br><span class="sub">下一節點：資料與基準綁定驗證。尚未開始新回測，沒有新增排程。</span></div>'
html33=html33.replace('<h2>先回答選股與投入，再處理複雜調整。</h2>','<h2>先回答選股與投入，再處理複雜調整。</h2>'+snippet33);v33_summaryhtml.write_text(html33,encoding='utf-8')
v33_flowdoc=v33_root/'A策略_無人值守研究流程.md';v33_flowdoc.write_text(v33_flowdoc.read_text(encoding='utf-8')+'\n\n## v33 統一長歷史續跑入口\n使用者要求將待改進項目寫成可接續計畫。先讀「'+v33_mp.name+'」與同名JSON，再接狀態unified_research_v33；不能因舊queue完成而停止。第一批9新格20124窗口、13主要對照；只重算新格，基準重用。此次只完成計畫，沒有啟動排程或新回測。\n',encoding='utf-8')
assert len({x['experiment_key'] for x in v33_candidates+v33_reused})==11
for node33 in v33_nodes:assert all(dep in v33_plan['queue_order'] for dep in node33['depends_on'])
print('V33_PLAN_READY',{'plan':v33_mp.name,'machine_plan':v33_jp.name,'nodes':len(v33_nodes),'new_candidates':9,'new_windows':20124,'primary_contrasts':13,'next':'bind_and_validate','running':False,'new_schedule':False})


In [ ]:
# v33 complete frozen-input provenance and verify continuation DAG, not trading results.
for label33,file33 in [('price_twse_and_actions','外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz'),('action_repairs','動能_長歷史來源修正稽核_v27_2026-10-15.json'),('listing_stockmaster_snapshot','策略研究_原資料快照_v3_2026-10-14.json.gz')]:v33_common['source_hashes'][label33]=hashlib.sha256((v33_root/file33).read_bytes()).hexdigest()
for s33 in v33_reused+v33_candidates:
    base33={k:v for k,v in s33.items() if k!='dedup_hash'};s33['dedup_hash']=hashlib.sha256(json.dumps({'common':v33_common,'spec':base33},sort_keys=True).encode()).hexdigest()
v33_plan['common']=v33_common;v33_plan['reused']=v33_reused;v33_plan['priority1_candidates']=v33_candidates
v33_jp.write_text(json.dumps(v33_plan,ensure_ascii=False,indent=2),encoding='utf-8')
seen33=set()
for n33 in v33_nodes:
    assert set(n33['depends_on'])<=seen33,(n33['id'],n33['depends_on']);seen33.add(n33['id'])
assert len(v33_primary)==13 and len(v33_candidates)==9
assert len(set(s['dedup_hash'] for s in v33_candidates+v33_reused))==11
txt33=v33_mp.read_text(encoding='utf-8');assert '## 第一批' in txt33 and '20124' in txt33
state33=json.loads(v33_statep.read_text(encoding='utf-8'));state33['unified_research_v33']['source_hashes']=v33_common['source_hashes'];state33['unified_research_v33']['notebook_cells']=[334,335];state33['unified_research_v33']['verified_dag_and_unique_specs']=True;v33_statep.write_text(json.dumps(state33,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V33_PLAN_VERIFIED',{'nodes':len(seen33),'acyclic_and_dependency_order':True,'unique_experiment_specs':11,'frozen_input_hashes':len(v33_common['source_hashes']),'new_research_runs':0})


In [ ]:
# v34 start user-authorized continuation; metadata and hashes before new computations.
print('V34_KERNEL',{'pid':os.getpid(),'python':sys.executable,'long_price_shape':v30_arrays['close'].shape,'long_flow_shape':v30_flow.shape,'prior_results':{k:len(v30_results[k]) for k in ['MOM','FLOW']},'original_recent':signal_returns_v15.shape})
v34_root=v30_root;v34_plan=json.loads((v34_root/'策略研究_統一長歷史續跑計畫_2026-10-16.json').read_text(encoding='utf-8'));v34_statepath=v34_root/'A策略_無人值守研究狀態.json'
assert hashlib.sha256(v30_engine.encode()).hexdigest()==v34_plan['common']['engine_reference_sha256']
v34_filemap={'price_current150_cache':'完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz','flow_current150_cache':'長歷史法人_原始快取_v30_2026-10-16.json.gz','price_twse_and_actions':'外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz','action_repairs':'動能_長歷史來源修正稽核_v27_2026-10-15.json','listing_stockmaster_snapshot':'策略研究_原資料快照_v3_2026-10-14.json.gz'}
for k,f in v34_filemap.items():assert hashlib.sha256((v34_root/f).read_bytes()).hexdigest()==v34_plan['common']['source_hashes'][k],k
assert int(v30_valid['MOM'].sum())==1621 and int(v30_valid['FLOW'].sum())==1541 and int(v30_common.sum())==1359 and int((v30_common&v30_later).sum())==1041
def checkpoint34(node,status,**extra):
    doc=json.loads(v34_statepath.read_text(encoding='utf-8'));q=doc['unified_research_v33'];row=next(n for n in q['nodes'] if n['id']==node);row.update({'status':status,**extra});q['status']='running' if status=='running' else 'in_progress';q['running']=status=='running';q['current_node']=node if status=='running' else None;q['last_node']=node;v34_statepath.write_text(json.dumps(doc,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
v34_scores={k:v30_scores[k] for k in ['MOM','FLOW']};v34_scores['MOM20']=np.full_like(v30_scores['MOM'],np.nan);v34_scores['MOM60']=np.full_like(v30_scores['MOM'],np.nan);v34_scores['LIQ']=np.full_like(v30_scores['MOM'],np.nan)
v34_money=np.full_like(v30_arrays['close'],np.nan)
for sid,lookup in v30_ns['price_lookup_v13'].items():
    jj=v30_si[sid]
    for d,row in lookup.items():
        if d in v30_di:v34_money[v30_di[d],jj]=v30_ns['number_v13'](row.get('Trading_money'))
for q in range(60,len(v30_days)):
    elig=np.isfinite(v30_scores['MOM'][q]);rr=v30_arrays['returns'][q-59:q+1];v34_scores['MOM20'][q]=percentile(np.prod(1+rr[-20:],axis=0)-1,elig);v34_scores['MOM60'][q]=percentile(np.prod(1+rr,axis=0)-1,elig)
    money=v34_money[q-19:q+1];liqelig=elig&np.all(np.isfinite(money)&(money>=0),axis=0);v34_scores['LIQ'][q]=percentile(np.mean(money,axis=0),liqelig)
v34_tree=ast.parse(v30_engine);v34_fn=v34_tree.body[0];v34_fn.name='simulate_unified34'
v34_fn.args.args += [ast.arg(arg='invested'),ast.arg(arg='rebalance')];v34_fn.args.defaults += [ast.Constant(value=.9),ast.Constant(value=5)]
class Invest34(ast.NodeTransformer):
    def visit_Constant(self,n):
        if isinstance(n.value,float) and n.value==.9:return ast.copy_location(ast.Name(id='invested',ctx=ast.Load()),n)
        return n
v34_fn.body=[Invest34().visit(n) for n in v34_fn.body]
v34_engine=ast.unparse(ast.fix_missing_locations(ast.Module(body=[v34_fn],type_ignores=[])))
assert 'if offset % 5 == 0:' in v34_engine
v34_engine=v34_engine.replace('if offset % 5 == 0:','if offset == 0 or (rebalance > 0 and offset % rebalance == 0):')
v34_ns=dict(v30_ns);v34_ns['scores30']=v34_scores;exec(v34_engine,v34_ns);v34_sim=v34_ns['simulate_unified34']
v34_controls=[]
for fam in ['MOM','FLOW']:
    for ix in [0,400,900,1700,2235]:
        s=int(v30_starts[ix]);a=v34_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved',fam,.9,5);b=v30_results[fam][ix]
        assert a['reasons']==b['reasons'] and a['violations']==b['violations'] and (a['return'] is None)==(b['return'] is None)
        if a['return'] is not None:assert all(np.isclose(a[k],b[k],rtol=0,atol=1e-12) for k in ['return','mdd','cost','turnover'])
        v34_controls.append({'family':fam,'index':ix,'matched':True})
v34_results={'MOM_E0':v30_results['MOM'],'FLOW_E0':v30_results['FLOW']};v34_registry={s['experiment_key']:s for s in v34_plan['reused']+v34_plan['priority1_candidates']};v34_newruns=0
checkpoint34('bind_and_validate','completed',checks=v34_controls,baseline_reused=True,hashes_verified=True)
print('V34_BOUND',{'controls':len(v34_controls),'reused_baselines':2,'newruns':0})


In [ ]:
# v34 nine fixed candidates, output-before-status and per-spec resume/dedup.
v34_node_specs={'lookback_20_60':['MOM20_E0','MOM60_E0'],'hold_and_selection_reference':['MOM_HOLD','FLOW_HOLD','LIQ_HOLD'],'capital_80_95':['MOM_P80','MOM_P95','FLOW_P80','FLOW_P95']}
for node34,keys34 in v34_node_specs.items():
    checkpoint34(node34,'running')
    for key34 in keys34:
        spec34=v34_registry[key34];path34=v34_root/('統一長歷史_'+key34+'_v34_2026-10-16.json.gz')
        if path34.exists():
            with gzip.open(path34,'rt',encoding='utf-8') as f:cache34=json.load(f)
            assert cache34['spec']['dedup_hash']==spec34['dedup_hash'] and cache34['engine_sha256']==hashlib.sha256(v34_engine.encode()).hexdigest();v34_results[key34]=cache34['results'];print('V34_REUSED',key34,flush=True);continue
        scorekey34={'MOM20_E0':'MOM20','MOM60_E0':'MOM60','LIQ_HOLD':'LIQ'}.get(key34,spec34['signal']);reb34=0 if spec34['rebalance']=='initial_only' else 5;rows34=[]
        for i,s0 in enumerate(v30_starts):
            s=int(s0);r=v34_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved',scorekey34,spec34['invested'],reb34);r['start_index']=s;rows34.append(r)
            if (i+1)%1000==0:print('V34_PROGRESS',key34,i+1,flush=True)
        v34_results[key34]=rows34;v34_newruns+=len(rows34)
        with gzip.open(path34,'wt',encoding='utf-8') as f:json.dump({'spec':spec34,'source_hashes':v34_plan['common']['source_hashes'],'engine_sha256':hashlib.sha256(v34_engine.encode()).hexdigest(),'results':rows34},f,ensure_ascii=False)
        print('V34_DONE',key34,{'valid':sum(r['return'] is not None for r in rows34),'newruns':v34_newruns},flush=True)
    checkpoint34(node34,'completed',experiments=keys34,outputs=[('統一長歷史_'+k+'_v34_2026-10-16.json.gz') for k in keys34])
print('V34_BATCH_COMPLETE',{'methods':len(v34_results),'newruns':v34_newruns})


In [ ]:
# v34 thirteen preregistered contrasts, paired valid dates; preserve original chronology.
checkpoint34('official_reference_audit','completed_with_gap',sources=['https://data.gov.tw/dataset/11871','https://www.twse.com.tw/indicesReport/MFI94U?date=20160701&response=html','https://www.twse.com.tw/indicesReport/MFI94U?date=20251201&response=html'],attempts=3,gap='official totalreturn source verified;historical target endpoints inaccessible via research tool;no bulk workaround or manufactured series')
checkpoint34('priority1_comparison','running')
v34_valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in v34_results.items()};v34_ret={k:np.array([np.nan if r['return'] is None else r['return'] for r in rows]) for k,rows in v34_results.items()};v34_joint=np.logical_and.reduce(list(v34_valid.values()))
def metrics34(k,mask):
    ix=np.flatnonzero(mask);x=v34_ret[k][mask];rows=v34_results[k]
    if not len(x):return {'n':0}
    return {'n':len(x),'win':float(np.mean(x>0)),'ev':float(x.mean()),'mean_win':float(np.mean(x[x>0])) if np.any(x>0) else None,'mean_loss':float(np.mean(x[x<0])) if np.any(x<0) else None,'p10':float(np.quantile(x,.1)),'worst_mdd':float(min(rows[i]['mdd'] for i in ix)),'fee':float(np.mean([rows[i]['cost'] for i in ix])),'turnover':float(np.mean([rows[i]['turnover'] for i in ix]))}
v34_contrasts={}
for a,b in v34_plan['priority1_contrasts']:
    pair=v34_valid[a]&v34_valid[b];later=pair&v30_later;train=pair&v30_training;name=a+'_minus_'+b;ya={}
    for yr in range(2019,2026):
        ym=later&np.array([v30_days[int(s)].year==yr for s in v30_starts]);ya[str(yr)]={'n':int(ym.sum()),'mean_difference':float((v34_ret[a]-v34_ret[b])[ym].mean()) if ym.any() else None}
    eligible_years=[r for r in ya.values() if r['n']>=20];year_gate=len(eligible_years)>=5 and sum(r['mean_difference']>0 for r in eligible_years)>=int(np.ceil(2*len(eligible_years)/3))
    v34_contrasts[name]={'candidate':a,'base':b,'all_n':int(pair.sum()),'later_n':int(later.sum()),'training_n':int(train.sum()),'coverage_later':float(later.sum()/np.sum(v34_valid[b]&v30_later)),'mean_difference':float((v34_ret[a]-v34_ret[b])[later].mean()),'candidate_stats':metrics34(a,later),'base_stats':metrics34(b,later),'years':ya,'year_gate':bool(year_gate),'training_candidate_fee':metrics34(a,train).get('fee')}
v34_rng=np.random.default_rng(3420261016)
for block in [24,48]:
    values={name:[] for name in v34_contrasts}
    for chunk in range(50):
        seeds=v34_rng.integers(0,len(v30_starts)-block+1,size=(100,int(np.ceil(len(v30_starts)/block))));idx=(seeds[:,:,None]+np.arange(block)).reshape(100,-1)[:,:len(v30_starts)]
        for name,r in v34_contrasts.items():
            a,b=r['candidate'],r['base'];mask=v34_valid[a]&v34_valid[b]&v30_later;weights=mask[idx];den=weights.sum(axis=1);dd=v34_ret[a]-v34_ret[b];num=np.where(weights,dd[idx],0.).sum(axis=1);values[name].extend(np.divide(num,den,out=np.full(100,np.nan),where=den>0).tolist())
    for name,x in values.items():v34_contrasts[name]['CI'+str(block)]=np.nanquantile(x,[.05/26,1-.05/26]).tolist()
for name,r in v34_contrasts.items():
    r['mean_improvement']=bool(r['CI24'][0]>0 and r['CI48'][0]>0)
    r['risk_not_worse']=bool(r['candidate_stats']['p10']>=r['base_stats']['p10'] and r['candidate_stats']['worst_mdd']>=r['base_stats']['worst_mdd'])
    r['coverage_gate']=r['coverage_later']>=.8;r['training_gate']=r['training_n']>=100
    r['full_improvement']=bool(r['mean_improvement'] and r['risk_not_worse'] and r['coverage_gate'] and r['training_gate'] and r['year_gate'])
v34_coverage={k:{'valid':int(v.sum()),'later_valid':int((v&v30_later).sum()),'planned':len(v),'invalid_reasons':dict(collections.Counter(reason for row in v34_results[k] for reason in row['reasons'])),'violations':sum(row['violations']>0 for row in v34_results[k])} for k,v in v34_valid.items()}
v34_summary={'plan':'v33 priority1','source_hashes':v34_plan['common']['source_hashes'],'newruns':v34_newruns,'reused_baseline_rows':4472,'metrics_common_all':{k:metrics34(k,v34_joint) for k in v34_results},'metrics_common_later':{k:metrics34(k,v34_joint&v30_later) for k in v34_results},'joint_n':int(v34_joint.sum()),'joint_later_n':int((v34_joint&v30_later).sum()),'contrasts':v34_contrasts,'coverage':v34_coverage,'full_improvement_contrasts':[k for k,r in v34_contrasts.items() if r['full_improvement']],'mean_improvement_only':[k for k,r in v34_contrasts.items() if r['mean_improvement'] and not r['full_improvement']],'benchmark':'official totalreturn archive not acquired;do notclaimalpha','retrospective_not_unseen':True,'checks':v34_controls}
(v34_root/'統一長歷史_優先1統計_v34_2026-10-16.json').write_text(json.dumps(v34_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
checkpoint34('priority1_comparison','completed',output='統一長歷史_優先1統計_v34_2026-10-16.json',contrasts=13,new_window_runs=v34_newruns)
print('V34_STATS',{'joint':v34_summary['joint_n'],'later':v34_summary['joint_later_n'],'full_improvements':v34_summary['full_improvement_contrasts'],'mean_only':v34_summary['mean_improvement_only'],'metrics':v34_summary['metrics_common_later'],'pairs':{k:{f:r[f] for f in ['mean_difference','CI48','risk_not_worse','year_gate']} for k,r in v34_contrasts.items()}})


In [ ]:
# v34 readable paired evidence; do not silently mix pairwise sample denominators.
checkpoint34('publish_priority1','running')
import plotly.graph_objects as go
from markdown_it import MarkdownIt
v34_labels={'MOM_E0':'動能各半／五日','FLOW_E0':'法人／五日','MOM20_E0':'純20日／五日','MOM60_E0':'純60日／五日','MOM_HOLD':'動能初始持有','FLOW_HOLD':'法人初始持有','LIQ_HOLD':'成交金額排名持有','MOM_P80':'動能80%','MOM_P95':'動能95%','FLOW_P80':'法人80%','FLOW_P95':'法人95%'}
v34_lines=['# 統一長歷史第一批：選股、持有、投入 v34','2026-10-16。9新方法共20124窗口，兩既有基準4472列直接重用。以下主檢查2019～2025、2016～2018校準；非真正未見樣本外。','## 這批得到什麼？','法人初始持有相對法人每五日重選／再平衡通過事先門檻；它只代表改善該法人基準，不代表優於所有動能或市場。動能初始持有有平均增量但最差回撤較差；95%動能有平均增量但風險放大。純20日或60日尚未通過穩定改善。','## 全11方法共同日期：描述用，不能替代各自配對','同'+str(v34_summary['joint_later_n'])+'後段窗口。所有收益已扣費；這是24日期望，不是年化。','|方法|勝率|期望|平均虧損|P10|最差回撤|費稅|','|---|---:|---:|---:|---:|---:|---:|']
for k,m in v34_summary['metrics_common_later'].items():v34_lines.append('|'+ '|'.join([v34_labels[k],p30(m['win']),p30(m['ev']),p30(m['mean_loss']),p30(m['p10']),p30(m['worst_mdd']),p30(m['fee'])])+'|')
v34_lines+=['## 13個主要配對：每列自己的相同日期','各列分母不同，不跨列直接排名。平均差與區間單位pp；24／48日位置區塊各5000次，13比較校正。','|候選−對照|後段n|平均差|48日校正區間|收益下界>0|風險不劣|跨年通過|全面改善|','|---|---:|---:|---|---|---|---|---|']
for name,r in v34_contrasts.items():v34_lines.append('|'+ '|'.join([v34_labels[r['candidate']]+'−'+v34_labels[r['base']],str(r['later_n']),f'{r["mean_difference"]*100:+.3f}pp',c30(r['CI48']),str(r['mean_improvement']),str(r['risk_not_worse']),str(r['year_gate']),str(r['full_improvement'])])+'|')
v34_lines+=['風險不劣要求P10與最差回撤均不差；另需至少100校準配對、覆蓋對照有效日期80%、至少5個20配對年份且三分之二增量正。通過是本輪回溯準則，不是未來保證。','## 可行性與缺口','|方法|全期有效／2236|有違規窗口|','|---|---:|---:|']
for k,r in v34_coverage.items():v34_lines.append(f'|{v34_labels[k]}|{r["valid"]}/2236|{r["violations"]}|')
v34_lines+=['有效日期排除不是隨機；不把無效填0。當初始持有遇漂移或公司行動，仍檢查原規則，沒有事後豁免。官方總報酬指數來源已確認，但目標歷史端點無法經研究工具取得；三次有限查核後保存缺口，沒有繞過受限路徑或把價格指數／150檔研究代理稱總報酬alpha。來源：https://data.gov.tw/dataset/11871 。','## 保存','原Notebook336起續跑，原kernel保留；10個舊基準控制一致。每個新方法分開gzip保存結果與定義去重鍵，已完成不重跑。箱形圖保留每個點、桃紅中位數、0線與固定尺度。']
v34_report='\n'.join(v34_lines);v34_rp=v34_root/'統一長歷史_優先1研究_v34_2026-10-16.md';v34_rp.write_text(v34_report,encoding='utf-8')
v34_figs=[];rng=np.random.default_rng(34);allkeys=list(v34_results);yrange=[float(min(v34_ret[k][v34_joint].min() for k in allkeys))*100-3,float(max(v34_ret[k][v34_joint].max() for k in allkeys))*100+3]
for title,keys in [('回看週期',['MOM_E0','MOM20_E0','MOM60_E0']),('初始持有與五日調整',['MOM_E0','MOM_HOLD','FLOW_E0','FLOW_HOLD','LIQ_HOLD']),('資金投入',['MOM_P80','MOM_E0','MOM_P95','FLOW_P80','FLOW_E0','FLOW_P95'])]:
    fig=go.Figure();mask=v34_joint&v30_later;ids=np.flatnonzero(mask)
    for pos,k in enumerate(keys):
        yy=v34_ret[k][mask]*100;fig.add_trace(go.Box(x=np.full(len(yy),pos),y=yy,boxpoints=False,hoverinfo='skip',width=.55,showlegend=False));fig.add_trace(go.Scatter(x=pos+rng.uniform(-.18,.18,len(yy)),y=yy,mode='markers',marker={'size':4,'opacity':.3},customdata=[[v34_results[k][i]['start'],v34_results[k][i]['end']] for i in ids],hovertemplate=v34_labels[k]+'<br>%{customdata[0]}→%{customdata[1]}<br>費後%{y:.2f}%<extra></extra>',showlegend=False));med=float(np.median(yy));fig.add_trace(go.Scatter(x=[pos-.22,pos+.22],y=[med,med],mode='lines',line={'color':'#c43c79','width':4},hoverinfo='skip',showlegend=False))
    fig.add_hline(y=0,line_color='#173333');fig.update_layout(template='plotly_white',title=title+'｜同'+str(int(mask.sum()))+'後段共同窗口',height=520,xaxis={'tickmode':'array','tickvals':list(range(len(keys))),'ticktext':[v34_labels[k] for k in keys]},yaxis={'title':'24日費後%','range':yrange});v34_figs.append(fig.to_html(full_html=False,include_plotlyjs=not v34_figs))
md=MarkdownIt('commonmark',{'html':False}).enable('table');v34_hp=v34_root/'統一長歷史_優先1全部點_v34_2026-10-16.html';v34_hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>第一批統一長歷史比較</title><style>body{font:16px/1.8 system-ui;max-width:1300px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:8px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}</style><a href="策略研究_總結報告.html">回總結</a>'+''.join(v34_figs)+md.render(v34_report).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',encoding='utf-8')
mp=v34_root/'策略研究_總結報告.md';mt=mp.read_text(encoding='utf-8');note='## 新完成：第一批選股、持有與投入\n\n9個新方法已跑完20124窗口，既有基準直接重用；13主要對照已檢查。法人初始持有對法人五日策略通過預先準則；動能初始持有及95%投入有收益增量但風險代價。純20／60日未穩定取代各半。這是各自配對結果，不代表法人持有優於動能。\n\n[第一批完整報告與各自分母](統一長歷史_優先1研究_v34_2026-10-16.md) · [箱形圖與全部點](統一長歷史_優先1全部點_v34_2026-10-16.html)\n\n';mt=mt.replace('## 接下來要做什麼？',note+'## 接下來要做什麼？');mt=mt.replace('第一批9個新方法、20124個窗口，原動能／法人基準重用；計畫已準備，尚未啟動新回測或排程。','第一批9個新方法、20124個窗口已完成；原動能／法人基準重用，後續分支正在接續。');mp.write_text(mt,encoding='utf-8')
hp=v34_root/'策略研究_總結報告.html';ht=hp.read_text(encoding='utf-8');ins='<div class="common"><strong>第一批已完成</strong><br>9個新方法、20,124窗口。法人初始持有改善法人五日基準；動能持有／高投入仍有風險代價。<br><a href="統一長歷史_優先1全部點_v34_2026-10-16.html">查看完整配對與箱形圖 →</a></div>';ht=ht.replace('<h2>先回答選股與投入，再處理複雜調整。</h2>','<h2>先回答選股與投入，再處理複雜調整。</h2>'+ins);ht=ht.replace('尚未開始新回測，沒有新增排程。','第一批已完成，依依賴接續後續；沒有新增排程。');hp.write_text(ht,encoding='utf-8')
checkpoint34('publish_priority1','completed',outputs=[v34_rp.name,v34_hp.name],winner_scope='FLOW_HOLD improves FLOW_E0 only;not globalbest')
print('V34_PUBLISHED',{'report':v34_rp.name,'html':v34_hp.name,'next':'weights_and_trade_buffer'})


In [ ]:
# v35 freeze bounded additional price-only experiments before results; original baselines reused.
v35_specs=[]
for fam in ['MOM','FLOW']:
    for tag,alloc,tol in [('E5','equal',.005),('M0','mild',0.),('M5','mild',.005)]:v35_specs.append({'key':fam+'_'+tag,'signal':fam,'allocation':alloc,'tolerance':tol})
v35_definition={'source':'v33 weights_and_trade_buffer +otherpricefamilies','source_hashes':v34_plan['common']['source_hashes'],'weight_grid':v35_specs,'weight_contrasts':'M0-E0,M5-E5,E5-E0,M5-M0 for eachfamily;8primaryBonferroni','other_candidates':['REV5','LOWVOL','VOLADJ'],'other_contrasts':'each againstMOM_E0;3Bonferroni','rules':'REV5=-prior5adjustedreturn;LOWVOL=-prior20std;VOLADJ=half percentile(R20/std20)+half percentile(R60/std60);sameeligibility;noadditionalcalibratedparams','timing':'five-day equal90 except mildgrid;2016..2018calibration,2019..2025later','not_new_llm':True}
(v34_root/'統一長歷史_權重與其他家族定義_v35_2026-10-16.json').write_text(json.dumps(v35_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v34_scores.update({k:np.full_like(v30_scores['MOM'],np.nan) for k in ['REV5','LOWVOL','VOLADJ']})
for q in range(60,len(v30_days)):
    rr=v30_arrays['returns'][q-59:q+1];ok=np.isfinite(v30_scores['MOM'][q]);sd20=np.std(rr[-20:],axis=0);sd60=np.std(rr,axis=0)
    v34_scores['REV5'][q,ok]=-(np.prod(1+rr[-5:,ok],axis=0)-1);v34_scores['LOWVOL'][q]=percentile(-sd20,ok);ok2=ok&(sd20>0)&(sd60>0);v34_scores['VOLADJ'][q]=.5*percentile(np.divide(np.prod(1+rr[-20:],axis=0)-1,sd20,out=np.full(len(ok),np.nan),where=sd20>0),ok2)+.5*percentile(np.divide(np.prod(1+rr,axis=0)-1,sd60,out=np.full(len(ok),np.nan),where=sd60>0),ok2)
v34_ns['scores30']=v34_scores;v35_results={'MOM_E0':v34_results['MOM_E0'],'FLOW_E0':v34_results['FLOW_E0']}
for node,specs in [('weights_and_trade_buffer',v35_specs),('other_families_and_attribution',[{'key':k,'signal':k,'allocation':'equal','tolerance':0.} for k in ['REV5','LOWVOL','VOLADJ']])]:
    checkpoint34(node,'running',definition_file='統一長歷史_權重與其他家族定義_v35_2026-10-16.json')
    for spec in specs:
        key=spec['key'];path=v34_root/('統一長歷史_'+key+'_v35_2026-10-16.json.gz');signature=hashlib.sha256(json.dumps({'spec':spec,'definition':v35_definition},sort_keys=True).encode()).hexdigest()
        if path.exists():
            with gzip.open(path,'rt',encoding='utf-8') as f:cached=json.load(f)
            assert cached['hash']==signature;v35_results[key]=cached['results'];print('V35_REUSED',key,flush=True);continue
        rows=[]
        for i,s0 in enumerate(v30_starts):
            s=int(s0);r=v34_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),spec['allocation'],spec['tolerance'],'unresolved',spec['signal'],.9,5);r['start_index']=s;rows.append(r)
        v35_results[key]=rows
        with gzip.open(path,'wt',encoding='utf-8') as f:json.dump({'hash':signature,'definition':v35_definition,'spec':spec,'results':rows},f,ensure_ascii=False)
        print('V35_DONE',key,'valid',sum(r['return'] is not None for r in rows),flush=True)
    checkpoint34(node,'computed_pending_statistics',outputs=[('統一長歷史_'+s['key']+'_v35_2026-10-16.json.gz') for s in specs],attribution='initial-overlap trace analysis next;no fabricated individualPnL')
print('V35_RESULTS_READY',len(v35_results))


In [ ]:
# v36 exact original stop/add rule port, separate namespace; source fixes only if baseline controls justify.
v36_definition={'scope':'v33 stop_add_replication','data':v34_plan['common']['source_hashes'],'rules':'originalv23 fourstates none/ma5/ma5two/low5 x addoff/on,7new','timing':'allfirstday sameMOM25/equal90;priorclose signals,nextdayVWAP;nointraday','add':'fixed120knownnames/day,proxy20positive,breadth>=60%;heldR3R5>0 &aboveMA5;90->95;statechangeorregular5day','target':'25;oneforonehealthyreplacement;no forced fill','actions':'v30 knowncash receivable;unknownnoncash invalidate;no fabricated stockdiv','primary_contrasts':'7 new vsbase +3 addonmarginal +3 exitvsnone_add =13;24/48blocks5000Bonf13','period':'2016..2018calibration,2019..2025later','no_improvement_assumed':True}
(v34_root/'統一長歷史_退出加碼定義_v36_2026-10-16.json').write_text(json.dumps(v36_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v36_ns={'np':np,'INITIAL':INITIAL,'LOT':1000,'settle':settle,'stock_ids':v30_ids,'dates':v30_days,'shape':v30_arrays['close'].shape,'close':v30_arrays['close'],'mark':v30_arrays['mark'],'execution':v30_arrays['execution'],'good_exec_v15':v30_arrays['available'],'signal_returns_v15':v30_arrays['returns'],'bad_action_v15':~v30_good,'cash_div_v15':v30_arrays['cashdiv'],'stock_div_v15':np.zeros_like(v30_arrays['close']),'scores_v15':{'A':v30_scores['MOM']},'proxy_count_v18':v30_ngood,'proxy_daily_v18':v30_proxy,'curve_returns21':{}}
for look in [3,5]:
    ar=np.full_like(v30_arrays['returns'],np.nan)
    for q in range(look,len(v30_days)):
        rr=v30_arrays['returns'][q-look+1:q+1];ok=np.all(np.isfinite(rr)&~v36_ns['bad_action_v15'][q-look+1:q+1],axis=0);ar[q,ok]=np.prod(1+rr[:,ok],axis=0)-1
    v36_ns['curve_returns21'][look]=ar
src=v30_book.cells[292].source;indic=src[src.index('weak23='):src.index('def simulate_stopadd23')];exec(indic,v36_ns)
fn=next(n for n in ast.parse(src).body if isinstance(n,ast.FunctionDef) and n.name=='simulate_stopadd23');v36_source=ast.unparse(fn).replace('simulate_stopadd23','simulate_longstop36')
# The older recent-only guard rejects missing priorclose even on no-trade carry days.
# Long-history v30 permits documented suspension valuation without inventing trades.
v36_ns['held_bad36']=v30_arrays['bad'];v36_source=v36_source.replace('bad_action_v15[t]','held_bad36[t]')
guard36="        if np.any(~np.isfinite(close[q, selected]) | (close[q, selected] <= 0)):\n            invalid = True\n            reason = 'missing_prior_close'\n            break\n"
assert guard36 in v36_source;v36_source=v36_source.replace(guard36,'')
anchor36='        rebalance = regular or len(failed) > 0 or changed\n';assert anchor36 in v36_source
v36_source=v36_source.replace(anchor36,anchor36+guard36.replace('if np.any(', 'if rebalance and np.any(',1))
exec(v36_source,v36_ns);v36_sim=v36_ns['simulate_longstop36']
v36_checks=[];v36_mismatches=[]
for ix in np.flatnonzero(v30_valid['MOM']):
    now=v36_sim(int(v30_starts[ix]),'none',False);old=v30_results['MOM'][ix]
    if now['invalid'] or now['violations'] or not all(np.isclose(now[k],old[k],rtol=0,atol=1e-12) for k in ['return','cost','turnover','mdd']):v36_mismatches.append({'index':int(ix),'failure':now['failure_reason'],'oldreturn':old['return'],'newreturn':now['return']})
    else:v36_checks.append(int(ix))
print('V36_PORT_CONTROL',{'matched':len(v36_checks),'mismatches':v36_mismatches[:10]})
assert not v36_mismatches,'Stop source port did not exactly match baseline; repair before new runs'
v36_specs=[(ex,add) for ex in ['none','ma5','ma5two','low5'] for add in [False,True]];v36_results={'none_off':v30_results['MOM']}
checkpoint34('stop_add_replication','running',definition_file='統一長歷史_退出加碼定義_v36_2026-10-16.json',baseline_controls=len(v36_checks))
for ex,add in v36_specs:
    key=ex+('_on' if add else '_off')
    if key=='none_off':continue
    path=v34_root/('統一長歷史_退出加碼_'+key+'_v36_2026-10-16.json.gz')
    if path.exists():
        with gzip.open(path,'rt',encoding='utf-8') as f:doc=json.load(f)
        assert doc['definition']==v36_definition;v36_results[key]=doc['results'];continue
    rows=[]
    for s0 in v30_starts:
        r=v36_sim(int(s0),ex,add);r['path']=r['path'].tolist();r['return']=float(r['return']);rows.append(r)
    v36_results[key]=rows
    with gzip.open(path,'wt',encoding='utf-8') as f:json.dump({'definition':v36_definition,'results':rows},f,ensure_ascii=False,default=serial30)
    print('V36_DONE',key,sum(not r['invalid'] and not r['violations'] for r in rows),flush=True)
checkpoint34('stop_add_replication','computed_pending_statistics',baseline_controls=len(v36_checks),outputs=['統一長歷史_退出加碼_*_v36_2026-10-16.json.gz'])
print('V36_ALL_CELLS_READY',len(v36_results))


In [ ]:
# v37 common statistics for fixed weight/other/stop families, no new thresholds.
def analyze37(results,contrasts,family_size):
    valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in results.items()};ret={k:np.array([r['return'] if ok else np.nan for r,ok in zip(rows,valid[k])]) for k,rows in results.items()};common=np.logical_and.reduce(list(valid.values()))
    def metric(k,mask):
        ix=np.flatnonzero(mask);x=ret[k][mask]
        if not len(x):return {'n':0}
        return {'n':len(x),'win':float(np.mean(x>0)),'ev':float(x.mean()),'mean_loss':float(np.mean(x[x<0])) if np.any(x<0) else None,'p10':float(np.quantile(x,.1)),'worst_mdd':float(min(results[k][i]['mdd'] for i in ix)),'fee':float(np.mean([results[k][i]['cost'] for i in ix]))}
    out={}
    for a,b in contrasts:
        keep=valid[a]&valid[b]&v30_later;train=valid[a]&valid[b]&v30_training;dd=ret[a]-ret[b];years=[]
        for y in range(2019,2026):
            mm=keep&np.array([v30_days[int(s)].year==y for s in v30_starts]);years.append({'year':y,'n':int(mm.sum()),'difference':float(dd[mm].mean()) if mm.any() else None})
        eligible=[r for r in years if r['n']>=20]
        out[a+'_minus_'+b]={'candidate':a,'base':b,'n':int(keep.sum()),'training_n':int(train.sum()),'coverage':float(keep.sum()/np.sum(valid[b]&v30_later)) if np.sum(valid[b]&v30_later) else 0.,'mean_difference':float(dd[keep].mean()) if keep.any() else None,'candidate_stats':metric(a,keep),'base_stats':metric(b,keep),'years':years,'year_gate':len(eligible)>=5 and sum(r['difference']>0 for r in eligible)>=int(np.ceil(2*len(eligible)/3))}
    rng=np.random.default_rng(37000+family_size)
    for block in [24,48]:
        samples={name:[] for name in out}
        for chunk in range(50):
            seeds=rng.integers(0,len(v30_starts)-block+1,size=(100,int(np.ceil(len(v30_starts)/block))));idx=(seeds[:,:,None]+np.arange(block)).reshape(100,-1)[:,:len(v30_starts)]
            for name,r in out.items():
                a,b=r['candidate'],r['base'];mask=valid[a]&valid[b]&v30_later;ww=mask[idx];den=ww.sum(axis=1);dd=ret[a]-ret[b];num=np.where(ww,dd[idx],0).sum(axis=1);samples[name].extend(np.divide(num,den,out=np.full(100,np.nan),where=den>0).tolist())
        for name,x in samples.items():out[name]['CI'+str(block)]=np.nanquantile(x,[.05/(2*family_size),1-.05/(2*family_size)]).tolist()
    for name,r in out.items():
        a,b=r['candidate_stats'],r['base_stats'];r['mean_improvement']=bool(r['CI24'][0]>0 and r['CI48'][0]>0);r['risk_not_worse']=bool(a.get('p10',-np.inf)>=b.get('p10',np.inf) and a.get('worst_mdd',-np.inf)>=b.get('worst_mdd',np.inf));r['full_improvement']=bool(r['mean_improvement'] and r['risk_not_worse'] and r['coverage']>=.8 and r['training_n']>=100 and r['year_gate'])
    return {'family_size':family_size,'joint_all':int(common.sum()),'joint_later':int((common&v30_later).sum()),'metrics_common_later':{k:metric(k,common&v30_later) for k in results},'contrasts':out,'coverage':{k:int(v.sum()) for k,v in valid.items()},'full_improvements':[n for n,r in out.items() if r['full_improvement']]},valid,ret,common
v37_weight_pairs=[]
for f in ['MOM','FLOW']:v37_weight_pairs += [(f+'_M0',f+'_E0'),(f+'_M5',f+'_E5'),(f+'_E5',f+'_E0'),(f+'_M5',f+'_M0')]
v37_weights,v37_weightvalid,v37_weightret,v37_weightcommon=analyze37({k:v for k,v in v35_results.items() if k.startswith(('MOM','FLOW'))},v37_weight_pairs,8)
v37_other,v37_othervalid,v37_otherret,v37_othercommon=analyze37({k:v35_results[k] for k in ['MOM_E0','REV5','LOWVOL','VOLADJ']},[(k,'MOM_E0') for k in ['REV5','LOWVOL','VOLADJ']],3)
v37_stop_pairs=[(k,'none_off') for k in v36_results if k!='none_off']+[(ex+'_on',ex+'_off') for ex in ['ma5','ma5two','low5']]+[(ex+'_on','none_on') for ex in ['ma5','ma5two','low5']]
assert len(v37_stop_pairs)==13
v37_stops,v37_stopvalid,v37_stopret,v37_stopcommon=analyze37(v36_results,v37_stop_pairs,13)
v37_overlap=[]
for ix in np.flatnonzero(v30_common):
    a=set(v30_results['MOM'][ix]['trace'][0]['selected']);b=set(v30_results['FLOW'][ix]['trace'][0]['selected']);v37_overlap.append({'start':v30_results['MOM'][ix]['start'],'overlap':len(a&b),'mom_loss':v30_ret['MOM'][ix]<0,'flow_loss':v30_ret['FLOW'][ix]<0})
v37_attribution={'initial_target_overlap_mean':float(np.mean([r['overlap'] for r in v37_overlap])),'n':len(v37_overlap),'both_loss_windows':sum(r['mom_loss'] and r['flow_loss'] for r in v37_overlap),'report_scope':'initial targetmembership only;no individualstock PnL or bottomprobability;recoverytime/v28medianexit require separate explicitdefinition before new batch'}
v37_summary={'weights':v37_weights,'other_families':v37_other,'stops':v37_stops,'overlap':v37_attribution,'limits':'within-family correction does notremove repeatedresearch/data/currentpoolbias;no joint independent claims'}
(v34_root/'統一長歷史_權重退出其他家族統計_v37_2026-10-16.json').write_text(json.dumps(v37_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
checkpoint34('weights_and_trade_buffer','completed',output='統一長歷史_權重退出其他家族統計_v37_2026-10-16.json',full_improvements=v37_weights['full_improvements'])
checkpoint34('stop_add_replication','completed',output='統一長歷史_權重退出其他家族統計_v37_2026-10-16.json',full_improvements=v37_stops['full_improvements'])
checkpoint34('other_families_and_attribution','completed_with_limitations',output='統一長歷史_權重退出其他家族統計_v37_2026-10-16.json',full_improvements=v37_other['full_improvements'],attribution=v37_attribution)
print('V37_STATS',{'weight_full':v37_weights['full_improvements'],'other_full':v37_other['full_improvements'],'stop_full':v37_stops['full_improvements'],'stop_common':v37_stops['joint_later'],'stop_metrics':v37_stops['metrics_common_later'],'overlap':v37_attribution})


In [ ]:
# v38 one predeclared90->95 add versus train-exposure-matched fixed reference.
checkpoint34('exposure_matched_add','running')
tree=ast.parse(v36_source);fn=tree.body[0];fn.name='simulate_exposure38';fn.args.args += [ast.arg(arg='base_invest'),ast.arg(arg='ceiling')];fn.args.defaults += [ast.Constant(value=.9),ast.Constant(value=.95)]
class Exposure38(ast.NodeTransformer):
    def visit_Constant(self,n):
        if isinstance(n.value,float) and n.value==.9:return ast.copy_location(ast.Name(id='base_invest',ctx=ast.Load()),n)
        if isinstance(n.value,float) and n.value==.95:return ast.copy_location(ast.Name(id='ceiling',ctx=ast.Load()),n)
        if isinstance(n.value,float) and n.value==.05:return ast.copy_location(ast.parse('(ceiling-base_invest)',mode='eval').body,n)
        return n
fn.body=[Exposure38().visit(n) for n in fn.body];v38_source=ast.unparse(ast.fix_missing_locations(ast.Module(body=[fn],type_ignores=[])));v38_ns=dict(v36_ns);exec(v38_source,v38_ns);v38_sim=v38_ns['simulate_exposure38']
v38_trainrows={p:[] for p in [.8,.95]};v38_trainids=[]
for ix in np.flatnonzero(v30_training):
    base=v36_results['none_on'][ix]
    if base['invalid'] or base['violations']:continue
    lo=v38_sim(int(v30_starts[ix]),'none',False,.8,.95);hi=v38_sim(int(v30_starts[ix]),'none',False,.95,.95)
    if lo['invalid'] or hi['invalid'] or lo['violations'] or hi['violations']:continue
    v38_trainids.append(int(ix));v38_trainrows[.8].append(lo);v38_trainrows[.95].append(hi)
assert len(v38_trainids)>=100
v38_dynamic_exp=float(np.mean([v36_results['none_on'][i]['mean_exposure'] for i in v38_trainids]));v38_low=float(np.mean([r['mean_exposure'] for r in v38_trainrows[.8]]));v38_high=float(np.mean([r['mean_exposure'] for r in v38_trainrows[.95]]));v38_target=round(.8+(v38_dynamic_exp-v38_low)/(v38_high-v38_low)*.15,4)
assert .8<=v38_target<=.95
v38_definition={'dynamic':'original90->95','fixed_target':v38_target,'training_n':len(v38_trainids),'dynamic_train_exposure':v38_dynamic_exp,'low_fixed_train_exposure':v38_low,'high_fixed_train_exposure':v38_high,'only_train_exposure_no_returns':True,'source_hashes':v34_plan['common']['source_hashes'],'not_per_window_future_matching':True,'primary_contrasts':1,'statistics':'24/48 blocks5000,ordinary95 foronecontrast;notglobalresearchcorrection'}
(v34_root/'統一長歷史_曝險匹配定義_v38_2026-10-16.json').write_text(json.dumps(v38_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v38_fixed=[]
for s0 in v30_starts:
    r=v38_sim(int(s0),'none',False,v38_target,v38_target);r['path']=r['path'].tolist();v38_fixed.append(r)
with gzip.open(v34_root/'統一長歷史_曝險匹配逐窗口_v38_2026-10-16.json.gz','wt',encoding='utf-8') as f:json.dump({'definition':v38_definition,'results':v38_fixed},f,ensure_ascii=False,default=serial30)
v38_summary,v38_valid,v38_ret,v38_common=analyze37({'dynamic':v36_results['none_on'],'fixed':v38_fixed},[('dynamic','fixed')],1)
v38_summary['definition']=v38_definition;v38_summary['actual_later_exposure']={k:float(np.mean([rows[i]['mean_exposure'] for i in np.flatnonzero(v38_common&v30_later)])) for k,rows in [('dynamic',v36_results['none_on']),('fixed',v38_fixed)]}
(v34_root/'統一長歷史_曝險匹配統計_v38_2026-10-16.json').write_text(json.dumps(v38_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
checkpoint34('exposure_matched_add','completed',target=v38_target,training_n=len(v38_trainids),output='統一長歷史_曝險匹配統計_v38_2026-10-16.json')
print('V38_MATCHED',{'target':v38_target,'train':len(v38_trainids),'later':v38_summary['joint_later'],'contrast':v38_summary['contrasts'],'exposure':v38_summary['actual_later_exposure']})


In [ ]:
# v39 consolidate completed authorized finite batches; same data, no new experiment.
import plotly.graph_objects as go
from markdown_it import MarkdownIt
import re
v39_families={'第一批選股與投入':v34_summary,'權重與交易緩衝':v37_weights,'其他選股家族':v37_other,'跌破退出與加碼':v37_stops,'平均曝險匹配':v38_summary}
v39_labels={**v34_labels,'MOM_E5':'動能等權＋小額略過','MOM_M0':'動能溫和非等權','MOM_M5':'動能非等權＋小額略過','FLOW_E5':'法人等權＋小額略過','FLOW_M0':'法人溫和非等權','FLOW_M5':'法人非等權＋小額略過','REV5':'五日反轉','LOWVOL':'純低波動','VOLADJ':'波動調整動能','none_off':'原動能','none_on':'原動能＋加碼','ma5_off':'低於MA5退出','ma5_on':'低於MA5＋加碼','ma5two_off':'連兩日低於MA5','ma5two_on':'連兩日MA5＋加碼','low5_off':'破五日收盤前低','low5_on':'破前低＋加碼','dynamic':'90→95條件加碼','fixed':'固定91.99%投入'}
v39_report=['# 統一長歷史批次研究：完成結果','2026-10-16。資料快照、門檻與交易模型均沿用v30，新增方法在定義保存後執行。已完成第一批9格、權重6格、其他選股3格、退出加碼7格、曝險固定對照1格，共26個全期新方法×2236＝58136窗口計算；此外曝險校準有局部計算及控制核對，不當獨立市場樣本。4472基準窗口直接引用。','## 目前可採用的研究判斷','1. 法人初始持有相對法人五日重選／再平衡，是本輪通過事先完整準則的改良候選；不是所有策略的總冠軍，也未證明市場alpha。','2. 動能初始持有、較高股票投入提高了部分費後均值，但最大回撤更差；不得只按均值叫做全面改善。','3. 純20日／60日未穩定勝出；溫和非等權也未證實優於等權。','4. 小額略過減少部分交易，節費與平均增量較小；法人非等權內M5對M0通過，不等於M5優於等權。','5. 三種跌破退出及加碼版本均未優於原動能，且健康替補／25檔條件造成大量無效窗口。共同好行情樣本不能證明防住完整歷史崩跌。','6. 加碼訊號與固定91.99%對照平均曝險接近後，增量約−0.032pp、區間跨零，尚無額外擇時優勢。','## 怎麼讀各張表','每個配對以該候選與對照共同有效日期比較，分母不同，不跨列直接排名。圖表同一組方法用共同分母，切換分組會改日期；尤其退出組樣本偏向可執行時期。所有報酬已扣費，期望為24日平均，不年化。校準2016～2018、主要檢查2019～2025；資料先前看過，不是真正未見樣本外。五日評估含換股與重配，不是純退出因果。']
for family,summary in v39_families.items():
    v39_report += ['## '+family,'|候選−對照|後段同日期n|候選勝率／期望|對照勝率／期望|平均差pp|48日校正區間pp|候選／對照費稅|候選／對照最差回撤|全面改善準則|','|---|---:|---|---|---:|---|---|---|---|']
    for name,r in summary['contrasts'].items():
        a=r['candidate_stats'];b=r['base_stats'];n=r.get('later_n',r.get('n'));v39_report.append('|'+ '|'.join([v39_labels.get(r['candidate'],r['candidate'])+'−'+v39_labels.get(r['base'],r['base']),str(n),p30(a.get('win'))+'／'+p30(a.get('ev')),p30(b.get('win'))+'／'+p30(b.get('ev')),f'{r["mean_difference"]*100:+.3f}',c30(r['CI48']),p30(a.get('fee'))+'／'+p30(b.get('fee')),p30(a.get('worst_mdd'))+'／'+p30(b.get('worst_mdd')),'通過' if r['full_improvement'] else '未通過'])+'|')
    v39_report.append('區間為平均差的不確定性，非下次收益範圍。兩種24／48日起點區塊各5000次；家族大小依預先定義，並列訓練、覆蓋、年份與風險，不挑最高全期均值。')
v39_report+=['## 特別注意：退出的樣本偏差',f'八組退出／加碼共同後段只剩{v37_stops["joint_later"]}窗口，原動能在該集合勝率91.93%、期望7.60%，不是完整動能突然改善，而是共同日期被篩選。各自配對MA5覆蓋35%、兩日MA5約70%、前低約77%，未達80%門檻。大量原策略差行情被排除，不能用條件回撤宣稱避開崩跌。','來源品質修正：v23近期引擎原本在不交易持有日也要求前收盤；長歷史已有停牌估值規則，所以只在要交易時計價檢查，持有估值使用原mark。與v30全部1621個有效動能窗口的收益、費用、換手與回撤均吻合後才計算新候選。沒有捏造成交或股票股利。','## 曝險匹配的細節',f'固定比例91.99%只以2016～2018的359個共同校準窗口平均曝險求得，不看報酬、不按後段結果再調。後段1217窗口，動態平均實際股票曝險{p30(v38_summary["actual_later_exposure"]["dynamic"])}、固定{p30(v38_summary["actual_later_exposure"]["fixed"])}，非每天精確相同。動態費稅0.84%、固定0.80%；動態期望3.12%、固定3.15%。不能聲稱100%增益全由現金或費用造成，也不能用區間跨零證明兩者等價。','## 持股重疊與未完成歸因',f'v30共同1359窗口，初始目標平均重疊{v37_attribution["initial_target_overlap_mean"]:.2f}／25檔，兩策略都虧損{int(v37_attribution["both_loss_windows"])}窗口。這是名單重疊與窗口結果，不是個股PNL因果歸因。賣後觸底／回復時間及v28反轉中位退出的額外規則仍需獨立固定定義，不在這批隨意新增。','## 外部資料分支','已確認證交所發行量加權股價報酬指數為配息再投資參考，但目標歷史端點無法取得；有限三次來源查核後記錄缺口，不走曾被安全拒絕的批量路徑。https://data.gov.tw/dataset/11871 。所以未宣稱策略超額市場alpha。景氣燈號／PMI首次公布值與時間、歷史產業分類、公告全池覆蓋仍不足；沒有用修訂值或今日分類冒充歷史。LLM批次仍暫不測。','## 執行與驗證','沿用PID36240既有Python314 kernel，未重啟、不清除、不提交交易。Notebook336～目前保存定義與輸出；既有基準、逐方法快取、配對、條件損益及圖點數已驗證。每個完成方法先保存才標節點，重跑入口按定義／來源／引擎去重；20分鐘不是期限。所有結果仍有目前150檔選擇偏誤、公司行動排除、同歷史多輪探索及重疊窗口限制，家族校正不消除跨輪偏誤。']
v39_text='\n'.join(v39_report);v39_rp=v34_root/'統一長歷史_批次完成總報告_v39_2026-10-16.md';v39_rp.write_text(v39_text,encoding='utf-8')
# A single selector changes plot, denominator and table together. No hidden cross-date ranking.
v39_sets=[('selection','選股與初始持有',{k:v34_results[k] for k in ['MOM_E0','FLOW_E0','MOM20_E0','MOM60_E0','MOM_HOLD','FLOW_HOLD','LIQ_HOLD']},'各半、20／60日、持有與成交金額排名。同組全部方法共同有效日期；持有收益較高仍可能有回撤代價。'),('capital','投入比例',{k:v34_results[k] for k in ['MOM_P80','MOM_E0','MOM_P95','FLOW_P80','FLOW_E0','FLOW_P95']},'只改投入比例；勝率不必提高，虧損與費用會隨曝險放大。'),('weights','權重與小額略過',{k:v35_results[k] for k in ['MOM_E0','MOM_E5','MOM_M0','MOM_M5','FLOW_E0','FLOW_E5','FLOW_M0','FLOW_M5']},'小額略過與非等權為兩個因素，不能把省費效果全歸因權重。'),('stops','跌破退出與加碼',v36_results,'共同樣本嚴重篩選。原動能在此好行情集合勝率很高，不能代表完整歷史；不能用此回撤宣稱防住崩跌。'),('others','其他選股',{k:v35_results[k] for k in ['MOM_E0','REV5','LOWVOL','VOLADJ']},'既定反轉、低波動及波動調整；沒有新調參或挑出最高後重寫假說。'),('match','加碼曝險匹配',{'dynamic':v36_results['none_on'],'fixed':v38_fixed},'固定91.99%只用2016～2018平均曝險校準。平均曝險相近不等於每天相同；未證實加碼時機有額外優勢。')]
v39_fig=go.Figure();v39_config={};v39_rng=np.random.default_rng(3939);v39_allreturns=[]
for group,label,rows,note in v39_sets:
    valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rs]) for k,rs in rows.items()};common=np.logical_and.reduce(list(valid.values()));config={'label':label,'note':note,'periods':{}}
    for period,mm in [('all',common),('later',common&v30_later)]:
        ids=np.flatnonzero(mm);metrics={};traceids=[]
        for pos,(k,rs) in enumerate(rows.items()):
            xx=np.array([rs[i]['return'] for i in ids]);yy=xx*100;v39_allreturns.extend(yy.tolist());name=v39_labels[k];metrics[k]={'label':name,'n':len(ids),'win':float(np.mean(xx>0)),'ev':float(xx.mean()),'mean_loss':float(xx[xx<0].mean()) if np.any(xx<0) else None,'worst_mdd':float(min(rs[i]['mdd'] for i in ids)),'fee':float(np.mean([rs[i]['cost'] for i in ids]))};show=group=='selection' and period=='all'
            traceids.extend(range(len(v39_fig.data),len(v39_fig.data)+3));v39_fig.add_trace(go.Box(x=np.full(len(yy),pos),y=yy,boxpoints=False,hoverinfo='skip',width=.55,quartilemethod='linear',visible=show,showlegend=False));v39_fig.add_trace(go.Scatter(x=pos+v39_rng.uniform(-.18,.18,len(yy)),y=yy,mode='markers',marker={'size':4,'opacity':.35},customdata=[[str(v30_days[int(v30_starts[i])]),str(v30_days[int(v30_starts[i])+23]),p30(rs[i]['cost'])] for i in ids],hovertemplate=name+'<br>%{customdata[0]}→%{customdata[1]}<br>24日費後%{y:.2f}%<br>費稅%{customdata[2]}<extra></extra>',visible=show,showlegend=False));med=float(np.median(yy));v39_fig.add_trace(go.Scatter(x=[pos-.23,pos+.23],y=[med,med],mode='lines',line={'color':'#c43c79','width':4},hoverinfo='skip',visible=show,showlegend=False))
        config['periods'][period]={'n':len(ids),'methods':metrics,'traceids':traceids,'labels':[v39_labels[k] for k in rows]}
    v39_config[group]=config
v39_fig.add_hline(y=0,line_color='#173333',line_width=2);v39_y=[min(v39_allreturns)-3,max(v39_allreturns)+3];initial=v39_config['selection']['periods']['all'];v39_fig.update_layout(template='plotly_white',height=600,margin={'t':25,'b':140},xaxis={'tickmode':'array','tickvals':list(range(len(initial['labels']))),'ticktext':initial['labels'],'tickangle':-25},yaxis={'title':'24日費後報酬 %','range':v39_y})
v39_plot=v39_fig.to_html(full_html=False,include_plotlyjs=True,div_id='unified-longhistory-boxplot',config={'responsive':True,'displaylogo':False})
htmlpath=v34_root/'策略研究_總結報告.html';page=htmlpath.read_text(encoding='utf-8')
page=re.sub(r'<div class="plot">.*?<p class="legend">',lambda m:'<div class="plot">'+v39_plot+'</div><p class="legend">',page,count=1,flags=re.S)
controls='<div class="period" id="group-buttons" aria-label="研究分組">'+''.join('<button type="button" data-group="'+g+'" aria-pressed="'+str(g=='selection').lower()+'">'+v39_config[g]['label']+'</button>' for g in v39_config)+'</div><p class="sub" id="group-note"></p>'
page=page.replace('<div class="period" aria-label="比較期間">',controls+'<div class="period" aria-label="比較期間">')
newjs=r'''<script>
'use strict';const groups=__GROUPS__;let activeGroup='selection',activePeriod='all';const node=id=>document.getElementById(id);const pct=(x,s=false)=>x===null?'未列':(s&&x>0?'＋':'')+(x*100).toFixed(2)+'%';
function render(){const g=groups[activeGroup],p=g.periods[activePeriod];document.querySelectorAll('[data-group]').forEach(b=>b.setAttribute('aria-pressed',String(b.dataset.group===activeGroup)));document.querySelectorAll('[data-period]').forEach(b=>b.setAttribute('aria-pressed',String(b.dataset.period===activePeriod)));node('group-note').textContent=g.note;node('period-note').textContent=g.label+'｜'+(activePeriod==='all'?'2016～2025全期（含校準）':'2019～2025後段')+'｜相同'+p.n+'個有效窗口。切換分組會改共同日期，不能跨組直接排名。';const body=node('metrics-body');body.replaceChildren();for(const m of Object.values(p.methods)){const tr=document.createElement('tr');[m.label,pct(m.win),pct(m.ev,true),pct(m.mean_loss),pct(m.worst_mdd),pct(m.fee)].forEach(x=>{const td=document.createElement('td');td.textContent=x;tr.append(td)});body.append(tr)}node('period-takeaway').textContent=activeGroup==='stops'?'注意：退出組共同日期受到可執行條件篩選；請同時查各自配對覆蓋，不能把原策略91%以上勝率當全體表現。':'圖表提供分布與同組均值；是否改良，請另查預先配對、校正區間、年份與風險，不只挑最高期望值。';if(typeof Plotly!=='undefined'){const plot=node('unified-longhistory-boxplot');Plotly.update(plot,{visible:Array.from({length:plot.data.length},(_,i)=>p.traceids.includes(i))},{'xaxis.tickvals':p.labels.map((_,i)=>i),'xaxis.ticktext':p.labels,'xaxis.range':[-.6,p.labels.length-.4]})}}
document.querySelectorAll('[data-group]').forEach(b=>b.addEventListener('click',()=>{activeGroup=b.dataset.group;render()}));document.querySelectorAll('[data-period]').forEach(b=>b.addEventListener('click',()=>{activePeriod=b.dataset.period;render()}));node('print').addEventListener('click',()=>window.print());render();
</script>'''.replace('__GROUPS__',json.dumps(v39_config,ensure_ascii=False))
page=re.sub(r"<script>\s*'use strict';.*?</script>",lambda m:newjs,page,count=1,flags=re.S)
start=page.index('<section id="conclusion">');end=page.index('<section id="methods">')
page=page[:start]+'''<section id="conclusion"><span class="number">01 / 統一長歷史新結果</span><h2>先減少無效換手，再看選股與風險取捨。</h2><div class="conclusions"><div class="conclusion"><b class="n">1</b><div><strong>法人初始持有改善了法人五日基準。</strong><p>通過預先費後收益、風險、覆蓋與跨年準則；只代表相對該法人基準，不能稱所有策略冠軍或已超越市場。</p></div></div><div class="conclusion"><b class="n">2</b><div><strong>動能持有與高投入有收益增量，也有風險代價。</strong><p>動能初始持有最差回撤更深；95%投入放大收益與虧損。純20／60日與溫和非等權尚未穩定取代原基準。</p></div></div><div class="conclusion"><b class="n">3</b><div><strong>跌破退出與加碼時機仍未證實改良。</strong><p>六個退出版本低於原動能，且部分覆蓋不足。匹配平均投入後，加碼不比固定配置更好；收益不能全部歸因費用。</p></div></div></div><a href="統一長歷史_批次完成總報告_v39_2026-10-16.md">看完整配對、分母與校正區間 →</a></section>'''+page[end:]
page=page.replace('這裡只有三種方法，差別在如何選股。','先分清選股、配置與調整，再比較結果。').replace('三組共同：90%投入、25檔目標、24交易日、整千股及相同費稅。25檔與90%是研究控制，還不是最佳配置的結論。','共用同一v30資料快照、24交易日、25檔目標、整千股及費稅；每組只改指定因子。資金組80／90／95%，持有組只在起點買入，其他依既定規則。')
start=page.index('<section id="meaning"');end=page.index('<section id="limits">')
page=page[:start]+'''<section id="meaning" class="reading"><span class="number">04 / 如何使用證據</span><h2>「有增量」要說清楚：相對誰、在哪些日期、付出什麼風險？</h2><p><strong>法人初始持有對法人五日策略：</strong>費後收益改善且風險／年份檢核通過。不是已證明法人持有優於動能或成交金額排名持有。</p><p><strong>動能初始持有及95%投入：</strong>部分配對均值較高，但回撤或P10更差，應列風險／報酬取捨。</p><p><strong>退出組的91%以上勝率：</strong>是共同可執行日期被篩選的結果，不是原動能在所有歷史都有這種表現。單日MA5對原基準後段覆蓋只有35%。</p><p><strong>加碼與固定91.99%投入：</strong>後段1217配對，動態期望3.12%、固定3.15%；平均差−0.032pp，48日區塊區間[-0.076,+0.012]pp。沒有證實額外擇時優勢，也沒有證明完全等價。</p><a href="統一長歷史_優先1全部點_v34_2026-10-16.html">第一批詳細箱形圖與配對 →</a></section>'''+page[end:]
start=page.index('<section id="next">');end=page.index('<section id="sources">')
page=page[:start]+'''<section id="next"><span class="number">06 / 執行狀態</span><h2>目前可行的有限比較已完成，資料分支仍保留缺口。</h2><div class="table-wrap"><table class="status-table"><thead><tr><th>狀態</th><th>研究</th><th>結果／尚缺</th></tr></thead><tbody><tr><td>完成</td><td>第一批9格、權重6格、其他選股3格、退出加碼7格、曝險固定對照1格</td><td>26個完整新方法、58,136窗口計算；既有基準重用。這不是獨立市場樣本數。</td></tr><tr><td>完成但有限</td><td>初始持股重疊與虧損窗口統計</td><td>平均重疊9.15／25檔；不是個股PNL歸因或觸底機率。</td></tr><tr><td>資料阻礙</td><td>正式市場總報酬、景氣／公告／產業</td><td>官方歷史指數尚未取得；首次公布值／時間及歷史分類仍未齊。</td></tr><tr><td>延後、需定義</td><td>觸底回復時間、反轉中位退出、額外加碼網格</td><td>沒有因其他結果不好而自行擴增；LLM批次維持暫不測。</td></tr></tbody></table></div><p>沒有強制選出全域最佳策略，也未新增排程或提交交易。若繼續，先解決資料缺口或提出可分離的機制假說，不再只微調參數直到勝出。</p><a href="統一長歷史_批次完成總報告_v39_2026-10-16.md">查看本批全部結果與限制 →</a></section>'''+page[end:]
page=page.replace('更新：2026-10-16 · 主要證據 v30','更新：2026-10-16 · 同v30資料 · 批次結果v34～v39')
page=page.replace('<div class="sources">','<div class="sources"><a href="統一長歷史_批次完成總報告_v39_2026-10-16.md">本批完整總報告</a>')
htmlpath.write_text(page,encoding='utf-8')
# Keep Markdown concise, current and traceable; legacy reports remain linked sources.
mp=v34_root/'策略研究_總結報告.md';mdtext=mp.read_text(encoding='utf-8');a=mdtext.index('## 先讀這三個結論');b=mdtext.index('## 我們在比較什麼？');mdtext=mdtext[:a]+'''## 先讀這三個結論

1. 法人初始持有24日，相對法人每五日重選／再平衡通過本輪完整準則；不等於所有策略最佳或有市場alpha。
2. 動能初始持有與較高投入有部分均值增量，但有更深回撤等風險代價；純20／60日、溫和權重尚未穩定勝出。
3. 三種跌破退出及搭配加碼未改善原動能，部分覆蓋不足；曝險匹配後加碼仍未證實額外優勢。不能把所有落差都歸因費用。

[本批完整配對、樣本數與校正區間](統一長歷史_批次完成總報告_v39_2026-10-16.md)。舊v30三方法表只供原環境實驗查核；本頁HTML可切換六組方法，每組使用自己的共同有效日期。

'''+mdtext[b:]
mdtext+='\n\n## 本批執行完成（2026-10-16）\n\n26個全期新方法共58,136窗口計算，另有局部校準與控制驗證。第一批、權重、原三種退出×加碼、三種其他選股及曝險匹配均完成可行範圍；初始重疊統計完成，個股PNL／觸底回復及額外反轉退出仍未新定義。正式市場基準及宏觀／產業／公告資料分支保留缺口。此處完成紀錄取代前文舊的「待執行」狀態，計畫保留作歷史。\n';mp.write_text(mdtext,encoding='utf-8')
v39_totalpoints=sum(len(t.y) for t in v39_fig.data if t.type=='scatter' and t.mode=='markers')
assert v39_totalpoints>0
for group,label,rows,note in v39_sets:
    for period,record in v39_config[group]['periods'].items():
        assert all(m['n']==record['n'] for m in record['methods'].values())
        assert sum(len(v39_fig.data[i].y) for i in record['traceids'] if v39_fig.data[i].type=='scatter' and v39_fig.data[i].mode=='markers')==record['n']*len(rows)
v39_audit={'full_new_methods':26,'full_new_window_runs':58136,'baselines_reused':4472,'initial_crosschecks':10,'stop_baseline_valid_controls':1621,'boxplot_groups':6,'periods':2,'displayed_marker_observations':v39_totalpoints,'duplicate_occurrence_acrossgroups_notindependent':True,'partial_exposure_calibration_separate':True,'no_restart':True,'no_new_schedule':True}
(v34_root/'統一長歷史_批次完成驗證_v39_2026-10-16.json').write_text(json.dumps(v39_audit,ensure_ascii=False,indent=2),encoding='utf-8')
st=json.loads(v34_statepath.read_text(encoding='utf-8'));q=st['unified_research_v33'];q.update({'status':'feasible_branches_completed_external_data_gaps','running':False,'current_node':None,'next_node':None,'new_runs_completed':58136,'completed_report':v39_rp.name,'remaining':'officialtotalreturn andfirstrelease/classification sources;deferredextra definitions notautoexpanded','last_notebook_cell':344,'verification':v39_audit});st['unified_batch_v39']={'status':q['status'],'report':v39_rp.name,'mean_vs_full_improvement_distinguished':True};v34_statepath.write_text(json.dumps(st,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
wf=v34_root/'A策略_無人值守研究流程.md';wf.write_text(wf.read_text(encoding='utf-8')+'\n\n## v34～v39 有限批次完成\n第一批9格、權重6格、其他家族3格、退出加碼7格、曝險對照1格完成；26個全期新方法58136窗口，基準重用。法人初始持有改善法人五日對照；動能持有／高投入有收益風險取捨；退出、權重及加碼不強選。正式基準與宏觀／產業原始資料缺口仍保留。詳見'+v39_rp.name+'；狀態已更新，不重跑。\n',encoding='utf-8')
print('V39_COMPLETE',v39_audit)


In [ ]:
# v39 keep fixed summary coherent and validate final outputs without new experiments.
mdlines=['# 策略研究總結','更新2026-10-16。統一資料：2016/07～2025/12的v30快照；最新有限批次v34～v39。[互動箱形圖與表格](策略研究_總結報告.html)。','## 目前知道什麼？','1. **法人初始持有相對法人五日策略通過本輪事先準則。**這是相對該基準的改良，不是所有策略冠軍。','2. **動能初始持有及高投入有平均收益增量，但有風險代價。**不能僅按最高均值選擇。','3. **原三種跌破退出與加碼未證實改善。**退出有大量無效窗口，不能把保留的好行情當防住崩跌；平均曝險接近時，加碼時機未有額外優勢。','4. **純20／60日及溫和非等權尚未穩定勝出。**小額交易略過有少量節費增量，與權重效果分開。','## 同資料，不一定同分母','原始2320交易日，暖機後2236個起點。原v30兩家族共同1359；新第一批11方法共同1198、後段954。退出組共同後段409且受到嚴重篩選。每個主要對照另外用自己的相同有效日期；不同組表格不能跨表直接排名。','校準2016～2018、主要檢查2019～2025。歷史已被看過，不是真正未見樣本外；窗口重疊，不是獨立比賽。所有收益已扣買賣各0.1425%、賣出另0.3%稅；期末持股估值，未假設清倉。','## 第一批全11方法共同後段954窗口','此表描述共同日期的分布。正式改良判斷用下方各自配對區間，不以本表最高平均值強選。','|方法|勝率|24日期望|平均虧損|最差回撤|平均費稅|','|---|---:|---:|---:|---:|---:|']
for k,m in v34_summary['metrics_common_later'].items():mdlines.append('|'+ '|'.join([v39_labels[k],p30(m['win']),p30(m['ev']),p30(m['mean_loss']),p30(m['worst_mdd']),p30(m['fee'])])+'|')
mdlines+=['勝率＝24日費後>0；期望＝含虧損平均，不年化；平均虧損只看負報酬；回撤是窗口峰谷跌幅。中位數不是平均值。','## 主要配對：不能混用分母','|候選−對照|後段n|候選勝率／期望|對照勝率／期望|平均差pp|48日校正區間|判斷|','|---|---:|---|---|---:|---|---|']
mainpairs=[('法人初始持有−法人五日',v34_contrasts['FLOW_HOLD_minus_FLOW_E0']),('動能初始持有−動能五日',v34_contrasts['MOM_HOLD_minus_MOM_E0']),('動能95%−90%',v34_contrasts['MOM_P95_minus_MOM_E0']),('單日MA5退出−原動能',v37_stops['contrasts']['ma5_off_minus_none_off']),('兩日MA5退出−原動能',v37_stops['contrasts']['ma5two_off_minus_none_off']),('前五日收盤低點退出−原動能',v37_stops['contrasts']['low5_off_minus_none_off']),('條件加碼−固定91.99%',v38_summary['contrasts']['dynamic_minus_fixed'])]
for label,r in mainpairs:
    a,b=r['candidate_stats'],r['base_stats'];judgement='本輪完整準則通過' if r['full_improvement'] else ('平均收益增量但風險取捨' if r['mean_improvement'] else ('平均較差且覆蓋不足' if r.get('coverage',r.get('coverage_later',1))<.8 else '未證實改良'))
    mdlines.append('|'+ '|'.join([label,str(r.get('later_n',r.get('n'))),p30(a['win'])+'／'+p30(a['ev']),p30(b['win'])+'／'+p30(b['ev']),f'{r["mean_difference"]*100:+.3f}',c30(r['CI48']),judgement])+'|')
mdlines+=['區間使用24／48日原始起點區塊各5000次、各批事先家族校正。完整準則另含校準至少100、覆蓋至少80%、年份一致、P10及最差回撤不差。這是回溯證據，不保證未來或全域最優。','## 成本與退出不能過度解讀','退出／加碼八組共同後段409，原動能在此勝率91.93%、期望7.60%，不代表完整歷史。單日MA5只有原後段有效日期35%覆蓋，兩日約70%、前低約77%；被排除差行情可能影響判斷，不能稱防住崩跌。退出確實增加費用，但賣後反彈、替補及持股路徑也改變，不把全部落差歸因成本。','固定91.99%以2016～2018平均曝險校準，不看收益。後段動態實際曝險92.14%、固定91.81%，費稅約0.84%與0.80%；平均曝險接近但非每天相同。未證實時機增益，不等於嚴格等價。','## 環境與其他家族','舊v30的低波動選法人／高波動選動能，未證實優於純動能，保留來源不納入新權重或退出假說。純反轉、純低波動未改善動能；波動調整動能區間跨0。法人非等權M5對M0有少量改善，是小額略過效果，不等於非等權優於等權。','初始目標平均重疊9.15／25檔，兩家族共同虧損362／1359窗口；只是名單／窗口觀察，不是個股PNL或觸底機率。','## 已完成與剩餘','- 已完成：第一批9格、權重6格、其他家族3格、退出加碼7格、曝險固定對照1格；26個全期新方法58,136窗口計算，基準4472列重用。局部校準／控制另計。','- 外部阻礙：正式市場總報酬歷史、景氣燈號／PMI首次公開值與時間、產業歷史分類及公告全池覆蓋。不捏造或繞過被拒絕來源，尚無市場alpha結論。','- 明確延後：觸底回復時間、額外反轉中位退出規則及新加碼網格；不因本批未勝出而自動增參數。LLM批次維持暫不測。','- 沒有執行中的研究cell，沒有新增或重啟排程；這份計畫的可行分支已完成，未解資料分支保留。','## 原始查核與歷史紀錄','[本批完整總報告](統一長歷史_批次完成總報告_v39_2026-10-16.md) · [第一批全部點](統一長歷史_優先1全部點_v34_2026-10-16.html) · [權重退出其他家族統計](統一長歷史_權重退出其他家族統計_v37_2026-10-16.json) · [曝險匹配統計](統一長歷史_曝險匹配統計_v38_2026-10-16.json) · [續跑計畫](策略研究_統一長歷史續跑計畫_2026-10-16.md) · [競賽規則](競賽規則與每日運作說明_2026-10-13.md)。','舊v16～v29報告保留為歷史來源，不與新主表混排；v27雖同年代，品質門檻不同。[原環境研究](長歷史環境研究_動能法人比較_v30_2026-10-16.md) · [原Notebook](動能持股_開賽前公告配置研究_2026-10-14.ipynb)。','## 限制與更新','現況150檔向前回看仍有存活／選擇偏誤，未知公司行動與失敗日期排除非隨機。來源修正與法人類別改版、不同方法可行日期、同歷史多輪研究及重疊依賴仍限制推論；家族校正不解決全部偏誤。Active Share及正式DataPack尚未完整驗證，不提交交易。','2026-10-16：建立固定總結，v30統一長歷史、v31盤點、v32編排、v33計畫；v34～v39完成有限批次並以同一份報告取代舊待辦狀態。後續仍同步維護Markdown與HTML，按定義／來源／引擎去重。']
(v34_root/'策略研究_總結報告.md').write_text('\n'.join(mdlines),encoding='utf-8')
hp=v34_root/'策略研究_總結報告.html';ht=hp.read_text(encoding='utf-8');ht=ht.replace('三種方法，差別在如何選股。','多組方法，分開比較。').replace('適合描述完整分布。','適合描述完整分布。')
ht=ht.replace('<div class="flow">','<p class="sub">下列1359／1041是原動能／法人環境比較的分母；本批各分組的實際共同日期請看圖表上方。它們不是每個方法固定共用的數量。</p><div class="flow">')
ht=ht.replace('下列結果使用不同期間或品質門檻，提供研究動機，不能與上方主表混排。','舊v16～v29使用不同期間或門檻。本批已移植其中有限既定方法至v30快照，最新結果見v39；其餘舊結果仍不與主表混排。')
ht=ht.replace('此次只重新編排與連動圖表／表格，沒有新增或修改回測績效。','隨後v34～v39已完成有限批次，新增26個全期方法。圖表六組與兩時段同步更新，原數值保留於來源報告。')
hp.write_text(ht,encoding='utf-8')
# Check each valid strategy path identity and return decomposition, no duplicate submissions.
v39_pathchecks=0
for rows in [*v34_results.values(),*v35_results.values(),*v36_results.values(),v38_fixed]:
    for r in rows:
        if r['return'] is not None and not r['invalid'] and r['violations']==0:
            assert np.isclose(r['path'][-1]/INITIAL-1,r['return']);v39_pathchecks+=1
v39_audit['valid_path_identity_checks_including_reused']=v39_pathchecks
(v34_root/'統一長歷史_批次完成驗證_v39_2026-10-16.json').write_text(json.dumps(v39_audit,ensure_ascii=False,indent=2),encoding='utf-8')
print('V39_FINAL_VALIDATION',{'paths':v39_pathchecks,'main_markdown_current':True,'all_figures_from_original_results':True,'no_new_backtest':True})


In [ ]:
# v40 accounting attribution of losses on exactly the summary selection-group dates.
import collections,json
v40_keys=['MOM_E0','FLOW_E0','MOM20_E0','MOM60_E0','MOM_HOLD','FLOW_HOLD','LIQ_HOLD']
v40_mask=np.logical_and.reduce([v34_valid[k] for k in v40_keys]);v40_proxy=np.array([np.prod(1+v30_proxy[int(s):int(s)+24])-1 if np.isfinite(v30_proxy[int(s):int(s)+24]).all() else np.nan for s in v30_starts])
v40_definition={'date':'2026-10-16','meaning':'negativegain=net24dayreturn<0;alsoexplain5dayminusinitialhold','sample':'exactsame7-method selection/hold group common dates;separateall/later','not_new_strategy':True,'attribution':'exactperstockexecutedcashflows+endingmark+cashdivreceivable-fees;cost addback accounting notzerofeererun','market':'current150 changingproxy descriptive only,not officialindex orcausal marketbeta','warnings':['overlappingwindows','posthoclossconditioning','currentpoolbias','feesaffectfuturelotbudget','cannotinfernewsortriggersfrompricesalone']}
(v34_root/'動能法人_虧損診斷定義_v40_2026-10-16.json').write_text(json.dumps(v40_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v40_ledgers={};v40_checks=0;v40_settle=v34_ns['settle']
for family,signal in [('MOM_E0','MOM'),('FLOW_E0','FLOW')]:
    rows=[]
    for ix in np.flatnonzero(v40_mask&(v34_ret[family]<0)):
        start=int(v30_starts[ix]);events=[]
        def capture40(cash,shares,target,execution,available):
            out=v40_settle(cash,shares,target,execution,available);events.append({'t':start+len(events)*5,'before':shares.copy(),'after':out[1].copy(),'cost':out[3]});return out
        v34_ns['settle']=capture40
        now=v34_sim({'key':'long30','year':v30_days[start].year,'start_index':start},set(v30_universe),'equal',0.,'unresolved',signal,.9,5)
        original=v34_results[family][ix];assert np.isclose(now['return'],original['return'],rtol=0,atol=1e-12)
        cashflows=np.zeros(len(v30_ids));costs=np.zeros(len(v30_ids));divs=np.zeros(len(v30_ids));held=np.zeros(len(v30_ids),np.int64);evmap={e['t']:e for e in events}
        for t in range(start,start+24):
            divs+=held*v30_arrays['cashdiv'][t]
            if t in evmap:
                e=evmap[t];delta=e['after']-e['before'];prices=np.nan_to_num(v30_arrays['execution'][t]);cashflows-=delta*prices;costs+=np.abs(delta)*prices*.001425+np.maximum(-delta,0)*prices*.003;held=e['after'].copy()
        grosspnl=cashflows+held*np.nan_to_num(v30_arrays['mark'][start+23])+divs;netpnl=grosspnl-costs
        assert np.isclose(netpnl.sum()/INITIAL,original['return'],rtol=0,atol=1e-10)
        assert np.isclose(costs.sum()/INITIAL,original['cost'],rtol=0,atol=1e-10);v40_checks+=1
        initial=set(original['trace'][0]['selected']);traded=np.flatnonzero((np.abs(cashflows)+costs+divs)>0);neg=np.sort(netpnl[netpnl<0]);lossamount=-netpnl[netpnl<0].sum();topneg=np.argsort(netpnl)[:5]
        rows.append({'index':int(ix),'start':original['start'],'end':original['end'],'net':original['return'],'fee':original['cost'],'accounting_gross':float(grosspnl.sum()/INITIAL),'market_proxy':float(v40_proxy[ix]) if np.isfinite(v40_proxy[ix]) else None,'loss_stock_fraction':float(np.mean(netpnl[traded]<0)),'negative_contributors':int(np.sum(netpnl[traded]<0)),'traded_names':len(traded),'top5_fraction_of_negative_contribution':float(-netpnl[topneg][netpnl[topneg]<0].sum()/lossamount) if lossamount else None,'initial_net_contribution':float(sum(netpnl[j] for j,sid in enumerate(v30_ids) if sid in initial)/INITIAL),'later_new_net_contribution':float(sum(netpnl[j] for j,sid in enumerate(v30_ids) if sid not in initial)/INITIAL),'top_losses':[{'stock':v30_ids[j],'net_contribution':float(netpnl[j]/INITIAL),'fee':float(costs[j]/INITIAL)} for j in topneg if netpnl[j]<0],'stock_pnl':[{'stock':v30_ids[j],'net':float(netpnl[j]/INITIAL),'gross':float(grosspnl[j]/INITIAL),'fee':float(costs[j]/INITIAL)} for j in traded]})
    v40_ledgers[family]=rows
v34_ns['settle']=v40_settle
v40_stats={}
for period,pm in [('all',v40_mask),('later',v40_mask&v30_later)]:
    block={}
    for family,holdkey in [('MOM_E0','MOM_HOLD'),('FLOW_E0','FLOW_HOLD')]:
        loss=pm&(v34_ret[family]<0);win=pm&(v34_ret[family]>0);ids=set(int(x) for x in np.flatnonzero(loss));led=[r for r in v40_ledgers[family] if r['index'] in ids];rr=v34_ret[family];hh=v34_ret[holdkey];cost=np.array([r['cost'] for r in v34_results[family]]);holdcost=np.array([r['cost'] for r in v34_results[holdkey]]);known=loss&np.isfinite(v40_proxy)
        years={str(y):{'loss_n':int(np.sum(loss&np.array([v30_days[int(s)].year==y for s in v30_starts]))),'total_n':int(np.sum(pm&np.array([v30_days[int(s)].year==y for s in v30_starts])))} for y in range(2016,2026)}
        features={}
        for state,arr in v30_states.items():
            features[state]={str(high):{'n':int((pm&(arr==high)).sum()),'loss_rate':float(np.mean(rr[pm&(arr==high)]<0)) if np.any(pm&(arr==high)) else None} for high in [False,True]}
        block[family]={'n':int(pm.sum()),'loss_n':int(loss.sum()),'win':float(np.mean(rr[pm]>0)),'ev':float(rr[pm].mean()),'mean_loss':float(rr[loss].mean()),'mean_loss_fee':float(cost[loss].mean()),'negative_even_addback_cost_n':int(np.sum(loss&((rr+cost)<0))),'negative_but_accounting_gross_nonnegative_n':int(np.sum(loss&((rr+cost)>=0))),'market_known_loss_n':int(known.sum()),'market_negative_among_losses':float(np.mean(v40_proxy[known]<0)),'market_mean_when_loss':float(v40_proxy[known].mean()),'hold_positive_among_losses':float(np.mean(hh[loss]>0)),'hold_better_among_losses':float(np.mean(hh[loss]>rr[loss])),'hold_mean_same_loss_dates':float(hh[loss].mean()),'rebalance_minus_hold_lossdates':float((rr-hh)[loss].mean()),'extra_fee_vs_hold_lossdates':float((cost-holdcost)[loss].mean()),'gross_path_difference_lossdates':float(((rr+cost)-(hh+holdcost))[loss].mean()),'mean_losing_stock_fraction':float(np.mean([r['loss_stock_fraction'] for r in led])),'mean_top5_negative_share':float(np.mean([r['top5_fraction_of_negative_contribution'] for r in led])),'mean_initial_contribution':float(np.mean([r['initial_net_contribution'] for r in led])),'mean_new_contribution':float(np.mean([r['later_new_net_contribution'] for r in led])),'worst':sorted(led,key=lambda r:r['net'])[:5],'years':years,'prestart_features_descriptive':features}
    both=pm&(v34_ret['MOM_E0']<0)&(v34_ret['FLOW_E0']<0);block['joint']={'both_loss':int(both.sum()),'momentum_only':int(np.sum(pm&(v34_ret['MOM_E0']<0)&(v34_ret['FLOW_E0']>=0))),'flow_only':int(np.sum(pm&(v34_ret['FLOW_E0']<0)&(v34_ret['MOM_E0']>=0)))};v40_stats[period]=block
v40_summary={'definition':v40_definition,'periods':v40_stats,'ledger_checks':v40_checks,'new_trading_strategies':0}
(v34_root/'動能法人_虧損診斷統計_v40_2026-10-16.json').write_text(json.dumps(v40_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
with gzip.open(v34_root/'動能法人_虧損個股歸因_v40_2026-10-16.json.gz','wt',encoding='utf-8') as f:json.dump(v40_ledgers,f,ensure_ascii=False,default=serial30)
print('V40_DIAGNOSIS',{p:{k:{a:b for a,b in r.items() if a not in ['worst','years','prestart_features_descriptive']} for k,r in z.items()} for p,z in v40_stats.items()})


In [ ]:
# v40 explain identified accounting mechanisms without inventing causal news stories.
from markdown_it import MarkdownIt
import re
assert v39_config['selection']['periods']['all']['n']==v40_stats['all']['MOM_E0']['n']==1209
assert v39_config['selection']['periods']['later']['n']==v40_stats['later']['MOM_E0']['n']==965
names40={'MOM_E0':'動能各半／五日','FLOW_E0':'法人／五日'}
lines40=['# 選股與初始持有：動能／法人五日策略為何虧損？','2026-10-16。此處負增益定義為24日費後報酬<0；另外比較同初始選股持有不調整。直接使用總結「選股與初始持有」七方法共同日期，全期1209、後段965，不是v30兩方法1359或v34十一方法1198。','## 目前有證據的解釋','1. 大多虧損並不是原本賺錢、只被費用扣成負數；持股價格路徑本身也有損失。','2. 約三分之二虧損窗口與150檔研究市場代理同跌；這是市場連動線索，不是景氣或新聞因果認定。','3. 每個虧損窗口平均約三分之二曾交易股票為負貢獻，前五個負貢獻股合計僅約36%／39%的負貢獻總額，說明不是只靠少數股票停損就必然解決。','4. 五日重選／再平衡比初始持有增加成本，且在已虧損日期多數表現較差；但多數即使持有仍虧損。這是事後挑虧損日期的診斷，不當作新可交易篩選規則。']
for period,title in [('all','全期2016～2025，含校準'),('later','後段2019～2025')]:
    st=v40_stats[period];a=st['MOM_E0'];b=st['FLOW_E0'];lines40 += ['## '+title,'同'+str(a['n'])+'窗口；以下「虧損時」指各策略自己的負報酬日期，兩者虧損子集不同。','|指標|動能各半／五日|法人／五日|','|---|---:|---:|']
    items=[('全樣本勝率',p30(a['win']),p30(b['win'])),('全樣本24日期望',p30(a['ev']),p30(b['ev'])),('虧損窗口數',str(a['loss_n']),str(b['loss_n'])),('虧損時平均費後報酬',p30(a['mean_loss']),p30(b['mean_loss'])),('虧損時平均費稅',p30(a['mean_loss_fee']),p30(b['mean_loss_fee'])),('加回既有費用仍負報酬',str(a['negative_even_addback_cost_n'])+'／'+str(a['loss_n']),str(b['negative_even_addback_cost_n'])+'／'+str(b['loss_n'])),('帳務上扣費前非負、扣費後負',str(a['negative_but_accounting_gross_nonnegative_n']),str(b['negative_but_accounting_gross_nonnegative_n'])),('虧損日期研究市場代理亦跌',p30(a['market_negative_among_losses']),p30(b['market_negative_among_losses'])),('平均曾交易股票負貢獻比例',p30(a['mean_losing_stock_fraction']),p30(b['mean_losing_stock_fraction'])),('前五大負貢獻占負貢獻總額',p30(a['mean_top5_negative_share']),p30(b['mean_top5_negative_share'])),('若初始持有就變成正報酬的比例',p30(a['hold_positive_among_losses']),p30(b['hold_positive_among_losses'])),('初始持有優於五日調整的比例',p30(a['hold_better_among_losses']),p30(b['hold_better_among_losses'])),('同虧損日期初始持有平均',p30(a['hold_mean_same_loss_dates']),p30(b['hold_mean_same_loss_dates']))]
    for row in items:lines40.append('|'+ '|'.join(row)+'|')
    lines40 += ['### 五日調整對初始持有：同策略虧損日期的會計分解','五日費後−持有費後＝（五日費前路徑−持有費前路徑）−額外費稅；費前只是加回既有交易的費用，不是零費率重算。單位pp。','|策略|費後差|費前路徑差|額外費稅|','|---|---:|---:|---:|']
    for k in names40:
        z=st[k];assert np.isclose(z['rebalance_minus_hold_lossdates'],z['gross_path_difference_lossdates']-z['extra_fee_vs_hold_lossdates']);lines40.append(f'|{names40[k]}|{z["rebalance_minus_hold_lossdates"]*100:+.3f}|{z["gross_path_difference_lossdates"]*100:+.3f}|{z["extra_fee_vs_hold_lossdates"]*100:+.3f}|')
    lines40.append('兩策略同虧'+str(st['joint']['both_loss'])+'窗口；只有動能虧'+str(st['joint']['momentum_only'])+'、只有法人虧'+str(st['joint']['flow_only'])+'。兩者並不是互補到能避開整體下跌。')
lines40 += ['## 最差案例：不是只有交易費','2020-02-14→2020-03-19，同一窗口動能費後−27.46%、法人−30.94%，既有費稅分別約0.95%、0.70%；150檔研究代理約−29.34%。費用無法解釋這個量級的全部虧損。這些價格資料顯示同步下跌，沒有額外查證逐日新聞，因此不把某條新聞或法人動機寫成原因。','|策略|個股代號|對投資組合淨報酬貢獻pp|','|---|---|---:|']
for k in names40:
    case=v40_stats['all'][k]['worst'][0]
    for row in case['top_losses']:lines40.append(f'|{names40[k]}|{row["stock"]}|{row["net_contribution"]*100:.3f}|')
lines40 += ['個股貢獻以整體初始資金為分母，不是該股自身漲跌幅；含期間買賣、期末持股、股息應收及費稅。','## 起點前有哪些警訊？','全期同1209窗口，僅描述事前市場條件與虧損比例，沒有依結果調門檻，也沒有新做風控策略。','|事前條件|窗口數|動能虧損比例|法人虧損比例|','|---|---:|---:|---:|']
for axis,label in [('trend','市場20日上漲'),('breadth','上漲股票至少一半'),('volatility','高於訓練波動門檻')]:
    for hi in [False,True]:
        a=v40_stats['all']['MOM_E0']['prestart_features_descriptive'][axis][str(hi)];b=v40_stats['all']['FLOW_E0']['prestart_features_descriptive'][axis][str(hi)];lines40.append('|'+ '|'.join([label+('：是' if hi else '：否'),str(a['n']),p30(a['loss_rate']),p30(b['loss_rate'])])+'|')
lines40+=['趨勢與廣度較弱時虧損比例略高，但差距不是確定性防護；高低波動差距更小。這是探索性比例，未做新的確認性檢定；不能在本表挑最好的條件直接部署。','## 仍不能回答的問題','- 不能由股價同步跌就判定景氣燈號或新聞造成，也不能推測法人抄底意圖。','- 個股淨貢獻可精確對帳；但「若不換股／若不同權重」的因果需要獨立對照，不能用貢獻排名當因果。','- 新買股在虧損窗口也有負貢獻，但其持有時間、資金分配不同，不能直接比較初始股與新股優劣。','- 只挑虧損日期再比持有會有條件選擇偏差。策略選擇仍參考v34全配對與區間，不拿這份損失條件分析當樣本外證明。','- 同窗口重疊、現況150檔偏誤、公司行動／資料排除仍存在；研究市場代理是變動成分，不是正式可投資基準。','## 驗證與保存',f'重播全部{v40_checks}個「選股與初始持有」共同日期中的虧損窗口，兩策略原報酬精確重現。逐股買賣現金流＋期末持股估值＋應收股息−交易費稅，加總與投資組合淨報酬一致；費用加總也一致。未新增交易策略、未重啟kernel，Notebook346起保存程式與結果。','[統計JSON](動能法人_虧損診斷統計_v40_2026-10-16.json) · [逐股貢獻快取](動能法人_虧損個股歸因_v40_2026-10-16.json.gz) · [回互動總結](策略研究_總結報告.html)。']
report40='\n'.join(lines40);rp40=v34_root/'動能法人_虧損原因分析_v40_2026-10-16.md';rp40.write_text(report40,encoding='utf-8')
md=MarkdownIt('commonmark',{'html':False}).enable('table');hp40=v34_root/'動能法人_虧損原因分析_v40_2026-10-16.html';hp40.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>動能／法人虧損診斷</title><style>body{font:16px/1.8 system-ui;max-width:1150px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:8px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}h2{margin-top:35px}</style>'+md.render(report40).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',encoding='utf-8')
section40='''<section id="loss-diagnosis"><span class="number">新證據 / 虧損診斷</span><h2>為何動能／法人五日策略仍會虧損？</h2><p>與「選股與初始持有」圖完全相同的全期1209窗口：動能虧392次、法人377次。動能平均虧−5.04%、法人−3.88%；虧損時平均費稅0.81%／0.69%。</p><p><strong>不是只有成本：</strong>加回既有費用後，仍有346／392、318／377次負報酬。約65%／67%的虧損窗口研究市場代理同跌，平均約67%／66%的曾交易股票為負貢獻。</p><p><strong>五日調整也有代價：</strong>在各自虧損日期，初始持有平均優於五日策略約1.00／0.75個百分點，其中額外費用約0.68／0.56個百分點；但只有約20%的虧損窗口會因改為持有而變正。</p><p class="sub">這是事後虧損條件診斷，非新策略的確認性測試；市場代理非正式指數，不能把相關當因果。</p><a href="動能法人_虧損原因分析_v40_2026-10-16.html">查看後段965窗口、最差案例、逐股貢獻與事前警訊 →</a></section>'''
hp=v34_root/'策略研究_總結報告.html';page=hp.read_text(encoding='utf-8');page=page.replace('<section id="meaning"',section40+'<section id="meaning"');page=page.replace('<a href="#meaning">','<a href="#loss-diagnosis">虧損原因分析</a><a href="#meaning">');hp.write_text(page,encoding='utf-8')
mp=v34_root/'策略研究_總結報告.md';text=mp.read_text(encoding='utf-8');text+='\n\n## 新增虧損診斷（v40）\n\n同選股／初始持有圖的全期1209、後段965窗口，分析原動能及法人五日策略。虧損大多非只有費用；約三分之二與市場代理同跌，五日調整有成本及路徑代價。這是事後虧損條件描述，不新增策略、不能推論宏觀／新聞因果。[完整分析](動能法人_虧損原因分析_v40_2026-10-16.md)。\n';mp.write_text(text,encoding='utf-8')
stp=v34_root/'A策略_無人值守研究狀態.json';st=json.loads(stp.read_text(encoding='utf-8'));st['loss_diagnosis_v40']={'status':'completed_observational_accounting','running':False,'sample_all':1209,'sample_later':965,'loss_replay_checks':v40_checks,'report':rp40.name,'html':hp40.name,'no_new_strategy':True};stp.write_text(json.dumps(st,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V40_REPORT_SAVED',{'report':rp40.name,'html':hp40.name,'all_n':1209,'later_n':965,'ledger_checks':v40_checks,'summary_updated':True})


In [ ]:
# v41 forward stock breadth; outcome diagnosis only, never an ex-ante trading signal.
v41_definition={'sample':'exactselectiongroupall1209/later965;keepportfolio dates,reportstockoutcomemissingness','stock_return':'product(1+knowncashadjustedclose_return) fromstart..start+23;priorclose toendingclose;gross,noportfoliofee;notVWAPactualholdingPNL','initial_selection':'originaltarget25atstart;no survivorship filterbasedonfutureoutcome','market_universe':'current150listedbeforestart;future24completeprice/action;knownstocks only;require80%ofstartlisted and>=50;missingnotdown/up','broad_down':'morethan50%ofknownstocksgross24return<0','severe_breadth_down':'atleast70%down','confirmed_broad_down':'morethan50%down AND equalweightfixedstockmeanreturn<0','proxy_warning':'notwholeTaiwanmarket,notofficialindex;current150bias;outcomesposthoc','overlap':'windowstatsnotindependent;alsoaggregate24nonoverlappingphases','no_new_strategy':True}
(v34_root/'動能法人_同期下跌廣度定義_v41_2026-10-16.json').write_text(json.dumps(v41_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v41_rows=[]
for ix in np.flatnonzero(v40_mask):
    s=int(v30_starts[ix]);e=s+23;rr=v30_arrays['returns'][s:e+1];known=np.all(np.isfinite(rr)&(rr>-1)&~v30_arrays['bad'][s:e+1],axis=0)&v30_pool&(v30_listed<=v30_days[s-1]);den=int(np.sum(v30_pool&(v30_listed<=v30_days[s-1])));n=int(known.sum());enough=n>=max(50,int(np.ceil(.8*den)));outcomes=np.full(len(v30_ids),np.nan);outcomes[known]=np.prod(1+rr[:,known],axis=0)-1
    market={'known_n':n,'startlisted_n':den,'missing_n':den-n,'coverage_ok':bool(enough),'down_fraction':float(np.mean(outcomes[known]<0)) if n else None,'mean_return':float(np.mean(outcomes[known])) if n else None}
    if enough:market.update({'broad_down':market['down_fraction']>.5,'severe_down':market['down_fraction']>=.7,'confirmed_broad_down':market['down_fraction']>.5 and market['mean_return']<0})
    fams={}
    for k in ['MOM_E0','FLOW_E0']:
        selected=v34_results[k][ix]['trace'][0]['selected'];jj=np.array([v30_si[sid] for sid in selected]);v=outcomes[jj];ok=np.isfinite(v);count=int(ok.sum());fams[k]={'net':v34_ret[k][ix],'portfolio_loss':bool(v34_ret[k][ix]<0),'selected_n':len(jj),'known_selected_n':count,'missing_selected_n':25-count,'down_n':int(np.sum(v[ok]<0)),'up_n':int(np.sum(v[ok]>0)),'flat_n':int(np.sum(v[ok]==0)),'down_fraction_known':float(np.mean(v[ok]<0)) if count else None,'all25_known':count==25,'mean_selected_return':float(v[ok].mean()) if count else None}
    v41_rows.append({'index':int(ix),'start':str(v30_days[s]),'end':str(v30_days[e]),'later':bool(v30_later[ix]),'market':market,'families':fams})
v41_summary={'definition':v41_definition,'periods':{}}
for period in ['all','later']:
    rows=[r for r in v41_rows if period=='all' or r['later']];out={}
    for k in ['MOM_E0','FLOW_E0']:
        block={}
        for label in ['loss','nonloss','all']:
            rs=[r for r in rows if label=='all' or r['families'][k]['portfolio_loss']==(label=='loss')];ks=[r for r in rs if r['families'][k]['all25_known']];ms=[r for r in rs if r['market']['coverage_ok']];paired=[r for r in ms if r['families'][k]['all25_known']];knownsum=sum(r['families'][k]['known_selected_n'] for r in rs);downsum=sum(r['families'][k]['down_n'] for r in rs)
            block[label]={'window_n':len(rs),'selected_known_occurrences':knownsum,'selected_missing_occurrences':len(rs)*25-knownsum,'selected_down_occurrences':downsum,'selected_down_fraction_all_known':downsum/knownsum if knownsum else None,'full25_windows':len(ks),'full25_mean_down_count':float(np.mean([r['families'][k]['down_n'] for r in ks])) if ks else None,'full25_majority_down_window_fraction':float(np.mean([r['families'][k]['down_n']>=13 for r in ks])) if ks else None,'market_known_windows':len(ms),'market_mean_down_fraction':float(np.mean([r['market']['down_fraction'] for r in ms])) if ms else None,'broad_down_window_n':sum(r['market']['broad_down'] for r in ms),'broad_down_window_fraction':float(np.mean([r['market']['broad_down'] for r in ms])) if ms else None,'severe_down_window_fraction':float(np.mean([r['market']['severe_down'] for r in ms])) if ms else None,'confirmed_broad_down_fraction':float(np.mean([r['market']['confirmed_broad_down'] for r in ms])) if ms else None,'selected_minus_market_breadth_paired_n':len(paired),'selected_minus_market_breadth':float(np.mean([r['families'][k]['down_fraction_known']-r['market']['down_fraction'] for r in paired])) if paired else None}
        # P(loss | market majority down) vs P(loss | market not majority down).
        block['market_conditioned']={}
        for high in [False,True]:
            rs=[r for r in rows if r['market']['coverage_ok'] and r['market']['broad_down']==high];block['market_conditioned'][str(high)]={'n':len(rs),'portfolio_loss_fraction':float(np.mean([r['families'][k]['portfolio_loss'] for r in rs])) if rs else None}
        block['years']={}
        for yr in range(2016,2026):
            rs=[r for r in rows if r['start'].startswith(str(yr)) and r['families'][k]['portfolio_loss']];ms=[r for r in rs if r['market']['coverage_ok']];block['years'][str(yr)]={'loss_n':len(rs),'market_known':len(ms),'broad_down_fraction':float(np.mean([r['market']['broad_down'] for r in ms])) if ms else None,'selected_down_fraction':sum(r['families'][k]['down_n'] for r in rs)/sum(r['families'][k]['known_selected_n'] for r in rs) if rs else None}
        phases=[]
        for phase in range(24):
            rs=[r for r in rows if r['index']%24==phase and r['families'][k]['portfolio_loss'] and r['market']['coverage_ok']]
            phases.append({'phase':phase,'loss_windows':len(rs),'broad_down_fraction':float(np.mean([r['market']['broad_down'] for r in rs])) if rs else None})
        block['nonoverlap_phases']=phases;out[k]=block
    v41_summary['periods'][period]=out
for r in v41_rows:
    for z in r['families'].values():assert z['down_n']+z['up_n']+z['flat_n']==z['known_selected_n']
(v34_root/'動能法人_同期下跌廣度統計_v41_2026-10-16.json').write_text(json.dumps(v41_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
(v34_root/'動能法人_同期下跌廣度逐窗口_v41_2026-10-16.json').write_text(json.dumps(v41_rows,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
print('V41_BREADTH',{p:{k:{t:v for t,v in b.items() if t in ['loss','nonloss','market_conditioned']} for k,b in ss.items()} for p,ss in v41_summary['periods'].items()})


In [ ]:
# v41 publish forward breadth diagnosis, every known window point and explicit denominators.
import plotly.graph_objects as go
from markdown_it import MarkdownIt
lines=['# 虧損窗口是否碰上整體下跌？','2026-10-16。與總結「選股與初始持有」圖完全相同的全期1209／後段965窗口。以下分析的是同期24日結果，不是可提前知道的訊號，也不是新交易策略。','## 結論','虧損窗口大多伴隨廣泛下跌：動能虧損392窗口中285（72.70%）、法人377中287（76.13%）發生在可用股票池超過半數股票下跌的時期。不是每次虧損都只是市場，選股下跌比例也略高於股票池，仍有選股與費用／調整因素。','## 定義先分清楚','- **初選股票下跌**：起點前一日收盤至第24交易日收盤，使用已知現金股息調整報酬；不是買入VWAP至賣出價，不扣投資組合費用，也不含中途新換進股票。','- **整體下跌期（廣度）**：同期可用股票中超過50%下跌。**嚴重廣泛下跌**為至少70%下跌。另列更嚴格條件：多數下跌且固定可用股票等權平均報酬也負。','- 股票池是目前150檔於當時已掛牌的歷史子集；要求可用股票至少當時已掛牌數80%且至少50檔。未知公司行動／缺價不當成跌或漲。不是官方大盤指數，也不是全台股。','- 同股票在多個重疊窗口反覆出現，因此股票次數及窗口數不獨立。','## 虧損與沒有虧損，差別有多大？']
for period,title in [('all','全期2016～2025'),('later','後段2019～2025')]:
    st=v41_summary['periods'][period];lines+=['### '+title,'|指標|動能各半／五日|法人／五日|','|---|---:|---:|'];a=st['MOM_E0'];b=st['FLOW_E0']
    rows=[('圖的共同窗口',str(v40_stats[period]['MOM_E0']['n']),str(v40_stats[period]['FLOW_E0']['n'])),('策略勝率／24日期望',p30(v40_stats[period]['MOM_E0']['win'])+'／'+p30(v40_stats[period]['MOM_E0']['ev']),p30(v40_stats[period]['FLOW_E0']['win'])+'／'+p30(v40_stats[period]['FLOW_E0']['ev'])),('虧損窗口',str(a['loss']['window_n']),str(b['loss']['window_n'])),('虧損窗口：初選股下跌比例（可判定股次）',p30(a['loss']['selected_down_fraction_all_known']),p30(b['loss']['selected_down_fraction_all_known'])),('可判定／缺漏初選股次',str(a['loss']['selected_known_occurrences'])+'／'+str(a['loss']['selected_missing_occurrences']),str(b['loss']['selected_known_occurrences'])+'／'+str(b['loss']['selected_missing_occurrences'])),('全部25檔可判定的虧損窗口',str(a['loss']['full25_windows']),str(b['loss']['full25_windows'])),('在上述完整窗口，平均25檔有幾檔下跌',f"{a['loss']['full25_mean_down_count']:.2f}",f"{b['loss']['full25_mean_down_count']:.2f}"),('完整窗口：至少13檔初選股下跌比例',p30(a['loss']['full25_majority_down_window_fraction']),p30(b['loss']['full25_majority_down_window_fraction'])),('虧損窗口：同期股票池平均下跌比例',p30(a['loss']['market_mean_down_fraction']),p30(b['loss']['market_mean_down_fraction'])),('虧損窗口：遇到多數股票下跌的比例',p30(a['loss']['broad_down_window_fraction']),p30(b['loss']['broad_down_window_fraction'])),('虧損窗口：遇到至少70%股票下跌比例',p30(a['loss']['severe_down_window_fraction']),p30(b['loss']['severe_down_window_fraction'])),('虧損窗口：多數跌且股票池平均報酬負',p30(a['loss']['confirmed_broad_down_fraction']),p30(b['loss']['confirmed_broad_down_fraction'])),('沒有虧損時：初選股下跌比例',p30(a['nonloss']['selected_down_fraction_all_known']),p30(b['nonloss']['selected_down_fraction_all_known'])),('沒有虧損時：遇多數股票下跌比例',p30(a['nonloss']['broad_down_window_fraction']),p30(b['nonloss']['broad_down_window_fraction']))]
    for row in rows:lines.append('|'+ '|'.join(row)+'|')
    lines+=['### 反過來：股票池多數下跌時，策略多常虧損？','|同期股票池狀態|窗口n|動能虧損率|法人虧損率|','|---|---:|---:|---:|']
    for high,label in [(True,'超過半數股票下跌'),(False,'未超過半數股票下跌')]:
        aa=a['market_conditioned'][str(high)];bb=b['market_conditioned'][str(high)];assert aa['n']==bb['n'];lines.append('|'+ '|'.join([label,str(aa['n']),p30(aa['portfolio_loss_fraction']),p30(bb['portfolio_loss_fraction'])])+'|')
    lines.append('注意方向不同：P(市場廣泛跌｜策略虧損)不等於P(策略虧損｜市場廣泛跌)。以上都是同期事後條件比例。')
lines+=['## 是否選股比股票池更弱？','只在初選25檔全部可判定的虧損窗口，同期逐窗口配對比較：動能下跌比例平均比股票池高5.10個百分點（356窗口），法人高2.34個百分點（360窗口）。不能直接拿全體股次比率減不同分母市場平均。這支持仍有選股差異，但不是控制產業／風險後的因果效果。','## 逐年看虧損時是否市場一起跌','|起始年|動能虧損窗口|其中股票池多數跌|法人虧損窗口|其中股票池多數跌|','|---|---:|---:|---:|---:|']
for y in range(2016,2026):
    a=v41_summary['periods']['all']['MOM_E0']['years'][str(y)];b=v41_summary['periods']['all']['FLOW_E0']['years'][str(y)];lines.append('|'+ '|'.join([str(y),str(a['loss_n']),p30(a['broad_down_fraction']),str(b['loss_n']),p30(b['broad_down_fraction'])])+'|')
lines+=['每列為起始年，2016只部分年度。重疊虧損窗口會重複涵蓋同一次下跌；JSON另存24個起點相位內不重疊窗口統計，沒有挑最好相位，也不宣稱相位彼此獨立。','## 可以怎麼用這個結果？','目前支持把「市場廣泛下跌」列為虧損的重要共現環境，不能再只靠個股排名解釋全部損失。但沒有證明可以提前知道接下來24日廣泛下跌。','若後續要改策略，必須另用起點前已知的趨勢／廣度等資訊預測，與固定曝險及同成本對照分開測試。不能把本報告同期下跌比例放回開賽前當訊號；也不能據此直接判定應減碼或停損。','## 與前一次「負貢獻比例」有何不同？','v40的負貢獻包含所有曾交易股票、實際買賣與費用；這次v41只問起初選出的25檔後24日調整毛報酬是否下跌。因此67.8%／66.3%與舊的67.4%／66.3%是不同量，不能當完全相同。','## 保存與限制','計算直接在既有kernel執行，Notebook348起保存。每窗口初選跌＋漲＋持平＝可判定股票數，缺價與未知公司行動明列。沒有新聞／景氣因果推論，沒有正式大盤alpha，沒有新增策略或重啟kernel。','[逐窗口JSON](動能法人_同期下跌廣度逐窗口_v41_2026-10-16.json) · [完整統計](動能法人_同期下跌廣度統計_v41_2026-10-16.json) · [回總結](策略研究_總結報告.html)。']
text='\n'.join(lines);rp=v34_root/'動能法人_同期整體下跌分析_v41_2026-10-16.md';rp.write_text(text,encoding='utf-8')
fig=go.Figure()
for k,color in [('MOM_E0','#167368'),('FLOW_E0','#567bb5')]:
    for loss in [False,True]:
        rs=[r for r in v41_rows if r['families'][k]['portfolio_loss']==loss and r['market']['coverage_ok'] and r['families'][k]['all25_known']]
        name=('動能' if k=='MOM_E0' else '法人')+('／虧損窗口' if loss else '／未虧損窗口')
        fig.add_trace(go.Scatter(x=[r['market']['down_fraction']*100 for r in rs],y=[r['families'][k]['down_fraction_known']*100 for r in rs],mode='markers',name=name,marker={'size':6,'opacity':.55 if loss else .2,'color':color,'symbol':'x' if loss else 'circle'},customdata=[[r['start'],r['end'],r['families'][k]['net']*100] for r in rs],hovertemplate=name+'<br>%{customdata[0]}→%{customdata[1]}<br>股票池下跌%{x:.1f}%<br>初選股下跌%{y:.1f}%<br>策略費後%{customdata[2]:.2f}%<extra></extra>'))
fig.add_shape(type='line',x0=0,y0=0,x1=100,y1=100,line={'color':'#888','dash':'dot'});fig.add_vline(x=50,line_dash='dash',line_color='#173333');fig.add_hline(y=50,line_dash='dash',line_color='#173333');fig.update_layout(template='plotly_white',height=620,title='每個完整初選窗口：同期股票池與初選股的下跌比例',xaxis={'title':'同期股票池下跌比例 %','range':[-2,102]},yaxis={'title':'初選25檔下跌比例 %','range':[-2,102]},legend={'orientation':'h'})
md=MarkdownIt('commonmark',{'html':False}).enable('table');hp=v34_root/'動能法人_同期整體下跌分析_v41_2026-10-16.html';hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>虧損是否整體下跌</title><style>body{font:16px/1.8 system-ui;max-width:1200px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:8px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}</style><a href="策略研究_總結報告.html">回總結</a>'+fig.to_html(full_html=False,include_plotlyjs=True,config={'responsive':True,'displaylogo':False})+md.render(text).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')+'</html>',encoding='utf-8')
summaryhp=v34_root/'策略研究_總結報告.html';page=summaryhp.read_text(encoding='utf-8');snippet='<section id="market-breadth-loss"><h2>新增：虧損是否碰上整體下跌？</h2><p>同選股／初始持有全期1209窗口，動能虧392次、法人377次。虧損日期初選股下跌比例約67.8%／66.3%；同期股票池平均約62.6%／63.8%。</p><p><strong>72.7%／76.1%的虧損窗口遇到股票池多數下跌。</strong>若再要求股票池等權平均報酬也負，比例為63.5%／66.0%。這是同期事後環境，不是事前避險訊號，也不代表全台股。</p><a href="動能法人_同期整體下跌分析_v41_2026-10-16.html">查看完整初選股比例、條件統計、每個窗口與年份 →</a></section>';page=page.replace('<section id="loss-diagnosis">',snippet+'<section id="loss-diagnosis">');summaryhp.write_text(page,encoding='utf-8')
summarymd=v34_root/'策略研究_總結報告.md';summarymd.write_text(summarymd.read_text(encoding='utf-8')+'\n\n## 同期整體下跌診斷（v41）\n\n初選股票後24日毛報酬下跌比例，在虧損窗口動能67.8%、法人66.3%；72.7%／76.1%虧損窗口伴隨股票池超過半數下跌。嚴格加股票池平均負報酬條件為63.5%／66.0%。只支持同期共現，非景氣因果或可提前使用訊號。[完整統計](動能法人_同期整體下跌分析_v41_2026-10-16.md)。\n',encoding='utf-8')
sp=v34_root/'A策略_無人值守研究狀態.json';st=json.loads(sp.read_text(encoding='utf-8'));st['market_breadth_loss_v41']={'status':'completed_descriptive','running':False,'sample':1209,'later':965,'unknown_initial_outcomes_explicit':True,'html':hp.name,'report':rp.name,'no_new_strategy':True};sp.write_text(json.dumps(st,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
assert sum(len(t.x) for t in fig.data)==1104+1103
print('V41_COMPLETE',{'html':hp.name,'scatter_points':sum(len(t.x) for t in fig.data),'stock_outcomes_missing_explicit':True,'no_new_strategy':True})


In [ ]:
# Descriptive pre-entry VIX alignment; no strategy optimization or restart.
import csv,io,bisect,urllib.request,datetime,zipfile
print('RETAINED',{'v41_rows':len(v41_rows),'root':str(v34_root),'python':__import__('sys').executable})
v42_definition={'sample':'same 1209 selection windows; later 965','vix_source':'https://cdn-api.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv','alignment':'latest US dated close strictly before Taiwan start date; maximum 7 calendar days stale','bins':['<20','20<=VIX<30','>=30'],'inference':'descriptive only; overlapping windows; no policy backtest','ndc':'original releases only for point-in-time interpretation; current revised history not substituted'}
(v34_root/'景氣與VIX_診斷定義_v42_2026-10-16.json').write_text(json.dumps(v42_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v42_cache=v34_root/'Cboe_VIX官方日資料_v42_2026-10-16.csv'
if v42_cache.exists():v42_bytes=v42_cache.read_bytes()
else:
    with urllib.request.urlopen(v42_definition['vix_source'],timeout=40) as response:v42_bytes=response.read()
    assert b'DATE' in v42_bytes[:100] and len(v42_bytes)>10000
    v42_cache.write_bytes(v42_bytes)
v42_vix=[]
for item in csv.DictReader(io.StringIO(v42_bytes.decode('utf-8-sig'))):
    date=datetime.datetime.strptime(item['DATE'].strip(),'%m/%d/%Y').date().isoformat();value=float(item['CLOSE'])
    if '2015-01-01'<=date<='2025-12-31':v42_vix.append({'date':date,'close':value})
v42_vix.sort(key=lambda x:x['date']);v42_dates=[x['date'] for x in v42_vix]
assert len(v42_dates)==len(set(v42_dates))
v42_rows=[]
for old in v41_rows:
    r={'index':old['index'],'start':old['start'],'end':old['end'],'later':old['later'],'market_broad_down':old['market']['broad_down'],'families':old['families']}
    j=bisect.bisect_left(v42_dates,r['start'])-1;lag=(datetime.date.fromisoformat(r['start'])-datetime.date.fromisoformat(v42_dates[j])).days
    assert 0<lag<=7
    z=v42_vix[j];future=v42_vix[bisect.bisect_left(v42_dates,r['start']):bisect.bisect_left(v42_dates,r['end'])]
    r.update(vix_date=z['date'],vix=z['close'],lag_days=lag,vix_change5=z['close']-v42_vix[j-5]['close'],vix_bin='<20' if z['close']<20 else ('20–30' if z['close']<30 else '>=30'),during_max=max([t['close'] for t in future],default=z['close']))
    v42_rows.append(r)
v42_summary={'definition':v42_definition,'source_sha256':hashlib.sha256(v42_bytes).hexdigest(),'periods':{},'worst_by_year':{}}
for period in ['all','later']:
    rs=[r for r in v42_rows if period=='all' or r['later']];out={}
    for k in ['MOM_E0','FLOW_E0']:
        o={'bins':{},'loss_vs_nonloss':{}}
        for label in ['<20','20–30','>=30']:
            ss=[r for r in rs if r['vix_bin']==label];rets=[r['families'][k]['net'] for r in ss]
            o['bins'][label]={'n':len(ss),'win':float(np.mean(np.array(rets)>0)) if ss else None,'ev':float(np.mean(rets)) if ss else None,'loss_n':sum(r['families'][k]['portfolio_loss'] for r in ss),'market_broad_fraction':float(np.mean([r['market_broad_down'] for r in ss])) if ss else None}
        for loss in [True,False]:
            ss=[r for r in rs if r['families'][k]['portfolio_loss']==loss]
            o['loss_vs_nonloss'][str(loss)]={'n':len(ss),'vix_mean':float(np.mean([r['vix'] for r in ss])),'vix_median':float(np.median([r['vix'] for r in ss])),'pre20_fraction':float(np.mean([r['vix']>=20 for r in ss])),'pre30_fraction':float(np.mean([r['vix']>=30 for r in ss])),'rise5_fraction':float(np.mean([r['vix_change5']>0 for r in ss])),'pre_low_during30_n':sum(r['vix']<20 and r['during_max']>=30 for r in ss)}
        out[k]=o
    v42_summary['periods'][period]=out
for year in range(2016,2026):
    rs=[r for r in v42_rows if r['start'].startswith(str(year))]
    v42_summary['worst_by_year'][str(year)]={k:min(rs,key=lambda r:r['families'][k]['net']) for k in ['MOM_E0','FLOW_E0']}
(v34_root/'景氣與VIX_窗口統計_v42_2026-10-16.json').write_text(json.dumps(v42_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
(v34_root/'景氣與VIX_逐窗口_v42_2026-10-16.json').write_text(json.dumps(v42_rows,ensure_ascii=False,default=serial30,allow_nan=False),encoding='utf-8')
print('V42_STATS',v42_summary['periods'])
print('WORST_YEAR',{y:{k:{a:r[a] for a in ['start','end','vix_date','vix','during_max']} for k,r in s.items()} for y,s in v42_summary['worst_by_year'].items()})
print('NDC_ARCHIVE_METADATA',[(k,len(v)) for k,v in v29_macro_bytes.items()])
with zipfile.ZipFile(io.BytesIO(v29_macro_bytes['ndc_history'])) as z:
    raw=z.read('景氣指標與燈號.csv');print('NDC_HISTORY_PREVIEW',raw.decode('utf-8-sig')[:1600])


In [ ]:
# Publish original NDC case audit and full-sample VIX diagnostics, no new trading rule.
v42_ndc_sources=[
{'month':'2016-08','release':'2016-09-29','score':25,'light':'綠燈','url':'https://www.ndc.gov.tw/nc_332_26346'},
{'month':'2017-09','release':'2017-10-27','score':30,'light':'綠燈','url':'https://www.ndc.gov.tw/nc_332_27998'},
{'month':'2018-07','release':'2018-08-27','score':27,'light':'綠燈','url':'https://www.ndc.gov.tw/nc_332_30068'},
{'month':'2019-02','release':'2019-03-27','score':17,'light':'黃藍燈','url':'https://www.ndc.gov.tw/nc_332_32216'},
{'month':'2019-12','release':'2020-02-03','score':27,'light':'綠燈','url':'https://www.ndc.gov.tw/nc_332_33778'},
{'month':'2021-02','release':'2021-03-26','score':40,'light':'紅燈','url':'https://www.ndc.gov.tw/nc_332_34860'},
{'month':'2021-03','release':'2021-04-27','score':40,'light':'紅燈','url':'https://www.ndc.gov.tw/nc_332_34949'},
{'month':'2022-02','release':'2022-03-28','score':34,'light':'黃紅燈','url':'https://www.ndc.gov.tw/nc_332_35682'},
{'month':'2023-10','release':'2023-11-27','score':16,'light':'藍燈','url':'https://www.ndc.gov.tw/nc_332_37583'},
{'month':'2024-01','release':'2024-03-04','score':27,'light':'綠燈','url':'https://www.ndc.gov.tw/nc_332_37924'},
{'month':'2025-01','release':'2025-02-27','score':34,'light':'黃紅燈','url':'https://www.ndc.gov.tw/nc_332_38848'}]
v42_case_month={'2016-10-13':'2016-08','2017-11-24':'2017-09','2018-09-11':'2018-07','2019-04-19':'2019-02','2019-04-16':'2019-02','2020-02-14':'2019-12','2021-04-13':'2021-02','2021-05-07':'2021-03','2022-04-08':'2022-02','2023-12-15':'2023-10','2024-03-19':'2024-01','2025-03-05':'2025-01'}
v42_case_records=[]
for yr,items in v42_summary['worst_by_year'].items():
    for k,row in items.items():
        src=next(t for t in v42_ndc_sources if t['month']==v42_case_month[row['start']]);assert src['release']<row['start']
        v42_case_records.append({'year':yr,'method':k,'start':row['start'],'end':row['end'],'net':row['families'][k]['net'],'vix_date':row['vix_date'],'vix':row['vix'],'during_max':row['during_max'],'ndc':src})
v42_source_audit={'verified_on':'2026-10-16','ndc_method':'original NDC releases verified via web-indexed official content/PDF; main HTML fetch returns403; not a complete monthly PIT database','case_selection':'each strategy lowest net window per start year in same1209 sample; retrospective illustration only','case_records':v42_case_records,'source_records':v42_ndc_sources,'vix':v42_definition['vix_source'],'limits':['no full-sample original NDC light conditional statistics','monthly publication lag','2018-07 and2024-01 method changes','VIX is US SPX implied volatility, not Taiwan index or crash probability']}
(v34_root/'景氣與VIX_原始公告核對_v42_2026-10-16.json').write_text(json.dumps(v42_source_audit,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
v42_summary['pre_vix_bin_year_counts']={b:{str(y):sum(r['vix_bin']==b and r['start'].startswith(str(y)) for r in v42_rows) for y in range(2016,2026)} for b in ['<20','20–30','>=30']}
v42_summary['nonoverlap_phases']={str(p):{k:{b:{'n':len(ss),'ev':float(np.mean([r['families'][k]['net'] for r in ss])) if ss else None} for b in ['<20','20–30','>=30'] for ss in [[r for r in v42_rows if r['index']%24==p and r['vix_bin']==b]]} for k in ['MOM_E0','FLOW_E0']} for p in range(24)}
(v34_root/'景氣與VIX_窗口統計_v42_2026-10-16.json').write_text(json.dumps(v42_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
lines=['# 虧損當時的景氣燈號與 VIX','2026-10-16｜沿用2016～2025長歷史與總結選股圖同一組1209共同窗口，後段2019～2025共965。這是環境診斷，不是已驗證的避險策略。','## 先看結論','1. **不少虧損開始時，景氣仍綠燈、VIX也不高。**最差窗口2020/2/14起，已公布2019/12為綠燈27分，起點前美國VIX僅14.15；持有期台灣開盤前已知VIX最高升至82.69。不能把後來的恐慌當作起點已知。','2. **單靠VIX高低，沒有呈現越高越容易虧的單調關係。**VIX≥30的起點，這個樣本後24日平均報酬反而較高；可能混合恐慌後反彈等環境，尚未做因果識別。','3. **景氣燈號不能直接當短期停損指令。**此輪核對每年最差窗口的原公告，共12個不同起點、11份公告；尚未重建全部1209窗口的逐月原始發布版本，不報虛假的全樣本燈號勝率。','## 當時到底是什麼燈？','下表是各策略每年最差窗口，屬事後挑出的案例，不能估計整體預測力。燈號使用起點前最新對應月份的原公告，顯示「資料月份」與「發布日」；並非用開賽當月尚未發布的數字。','|窗口起點→終點|策略與費後報酬|當時已公布月份／發布日|原公布燈號與分數|起點前VIX|期內已知VIX最高|','|---|---|---|---|---:|---:|']
for start in sorted(v42_case_month):
    ss=[t for t in v42_case_records if t['start']==start];r=ss[0];s=r['ndc'];labels='、'.join(('動能' if x['method']=='MOM_E0' else '法人')+' '+p30(x['net']) for x in ss)
    lines.append('|'+ '|'.join([r['start']+'→'+r['end'],labels,s['month']+'／'+s['release'],'['+s['light']+' '+str(s['score'])+'分]('+s['url']+')',f"{r['vix']:.2f}",f"{r['during_max']:.2f}"])+'|')
lines+=['原公告與今天的回溯資料可能不同：2017/9原稿30分、2018/7原稿27分，不能默默換成現行歷史數列的值。2018/7、2024/1原稿也明示啟用新版指標。以上僅轉述必要數值及發布日期，沒有宣稱原HTML皆已成功下載。','## VIX：全部窗口一起比較','VIX用台灣起點日期之前最近一個美國日期的收盤值，所有窗口落差1～6個日曆日以內且均不超過預設7日。台灣同一天晚上才公布的美國收盤值不准使用。期內最高只用美國日期小於台灣終點日的值，因此也是該窗口終點台灣開盤前可知的資料。','分組20／30是分析前固定的描述界線，不是從報酬搜尋出的最佳門檻，也不是Cboe保證的買賣界線。勝率＝費後24日報酬>0的窗口比例；期望＝費後24日平均報酬，零報酬不算勝。']
for period,title in [('all','全期2016～2025／1209窗口'),('later','後段2019～2025／965窗口')]:
    lines+=['### '+title,'|起點前VIX|共同窗口n|動能勝率|動能24日期望|法人勝率|法人24日期望|','|---|---:|---:|---:|---:|---:|']
    for b in ['<20','20–30','>=30']:
        a=v42_summary['periods'][period]['MOM_E0']['bins'][b];c=v42_summary['periods'][period]['FLOW_E0']['bins'][b]
        lines.append('|'+ '|'.join([b,str(a['n']),p30(a['win']),p30(a['ev']),p30(c['win']),p30(c['ev'])])+'|')
lines+=['## 只看虧損，事前有多少已在警戒？','|指標（全期）|動能|法人|','|---|---:|---:|']
for label,field in [('虧損窗口數','n'),('虧損起點VIX平均','vix_mean'),('虧損起點VIX中位數','vix_median'),('虧損起點VIX≥20比例','pre20_fraction'),('虧損起點VIX≥30比例','pre30_fraction'),('虧損起點VIX近5美國交易日上升比例','rise5_fraction'),('起點低於20、期內升到30以上的虧損窗口','pre_low_during30_n')]:
    vals=[v42_summary['periods']['all'][k]['loss_vs_nonloss']['True'][field] for k in ['MOM_E0','FLOW_E0']];values=[p30(x) if 'fraction' in field else (str(x) if field in ['n','pre_low_during30_n'] else f'{x:.2f}') for x in vals];lines.append('|'+ '|'.join([label]+values)+'|')
lines+=['沒有虧損時，起點VIX≥20比例：動能36.72%、法人35.10%；與虧損時35.97%、39.52%相近，並非清楚分界。動能60、法人56個虧損窗口雖然起點VIX<20，後來期內曾達30；這些窗口重疊，不能說成60／56場獨立危機。','## 這對停損研究的意義','可保留「VIX變化＋台股市場廣度／趨勢」作為待驗證訊號。若只拿VIX≥30就退出，本樣本很可能也排除部分反彈窗口。是否減少虧損、是否錯過回升、是否被費用吃掉，必須另測實際可成交的規則，不能從本分組表直接推定。','月度燈號較適合交代景氣背景；日資料VIX可用來觀察風險升溫，但沒有保證領先台股。VIX源自美國S&P500選擇權的預期波動，並非台股恐慌指數或下跌機率。','## 證據邊界與資料狀態','- 同期樣本並非全台股，是目前150檔股票池的歷史子集，存在存活與股票池選擇限制。2016為部分年度。','- 重疊24日窗口高度相依。JSON另列24個不重疊起點相位與年度分布，但未把1209當獨立事件、未宣稱統計顯著。後段資料亦已反覆研究，不是全新未看樣本。','- 高VIX組105窗口，後段100；少數事件可產生許多窗口，不能據此推薦高VIX加碼。','- 景氣已完成11份原公告的年度最差案例核對，**全部月份的首次公布燈號覆蓋仍未完成**。現行回溯ZIP保留作查核，不充當當時已知資料。','- 這次沒有重跑交易策略，也沒有新增退出／加碼／曝險政策。保留原先成交、公司行動、整千股與費用限制。','## 來源與重現','[Cboe歷史資料入口](https://www.cboe.com/tradable_products/vix/vix_historical_data) · [Cboe官方CSV]('+v42_definition['vix_source']+')。景氣每筆原稿連結在案例表。官方原頁部分擷取403，已以搜尋可讀的官方原稿／官方PDF核對，未繞過存取限制。','[逐窗口資料](景氣與VIX_逐窗口_v42_2026-10-16.json) · [統計與年度分布](景氣與VIX_窗口統計_v42_2026-10-16.json) · [原公告核對](景氣與VIX_原始公告核對_v42_2026-10-16.json) · [回總結](策略研究_總結報告.html)。計算與輸出保存原Notebook350、351格，沿用既有kernel，沒有重啟。']
v42_text='\n'.join(lines);v42_rp=v34_root/'動能法人_景氣燈號與恐慌指數_v42_2026-10-16.md';v42_rp.write_text(v42_text,encoding='utf-8')
v42_fig=go.Figure()
for k,color in [('MOM_E0','#167368'),('FLOW_E0','#567bb5')]:
    name='動能各半／五日' if k=='MOM_E0' else '法人／五日'
    for b in ['<20','20–30','>=30']:
        ss=[r for r in v42_rows if r['vix_bin']==b];label=name+' VIX'+b
        v42_fig.add_trace(go.Box(y=[r['families'][k]['net']*100 for r in ss],name=label,boxpoints='all',jitter=.4,pointpos=0,fillcolor='rgba(255,255,255,0)',line={'color':color},marker={'size':3,'opacity':.4,'color':color},customdata=[[r['start'],r['end'],r['vix_date'],r['vix']] for r in ss],hovertemplate='%{customdata[0]}→%{customdata[1]}<br>費後24日%{y:.2f}%<br>已知VIX %{customdata[3]:.2f}（%{customdata[2]}）<extra></extra>'))
        med=float(np.median([r['families'][k]['net']*100 for r in ss]));v42_fig.add_trace(go.Scatter(x=[label],y=[med],mode='markers',marker={'color':'#df4385','symbol':'line-ew','size':22,'line':{'width':3}},name='中位數',showlegend=False,hovertemplate='中位數 %{y:.2f}%<extra></extra>'))
v42_fig.add_hline(y=0,line_color='#222',line_width=2);v42_fig.update_layout(title='起點前VIX分組：費後24日報酬箱形圖與全部2418個策略窗口點',template='plotly_white',height=650,yaxis_title='費後報酬 %',showlegend=False,margin={'b':140})
v42_scatter=go.Figure()
for k,color in [('MOM_E0','#167368'),('FLOW_E0','#567bb5')]:
    v42_scatter.add_trace(go.Scatter(x=[r['vix'] for r in v42_rows],y=[r['families'][k]['net']*100 for r in v42_rows],mode='markers',name='動能' if k=='MOM_E0' else '法人',marker={'size':5,'color':color,'opacity':.4},customdata=[[r['start'],r['end'],r['vix_date']] for r in v42_rows],hovertemplate='%{customdata[0]}→%{customdata[1]}<br>起點已知VIX%{x:.2f}（%{customdata[2]}）<br>24日費後%{y:.2f}%<extra></extra>'))
v42_scatter.add_hline(y=0,line_color='#222');v42_scatter.update_layout(template='plotly_white',height=520,title='每個窗口：事前VIX與後續報酬',xaxis_title='起點前已知VIX',yaxis_title='費後24日報酬 %')
v42_hp=v34_root/'動能法人_景氣燈號與恐慌指數_v42_2026-10-16.html';md42=MarkdownIt('commonmark',{'html':False}).enable('table');rendered=md42.render(v42_text).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')
v42_hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>景氣燈號與VIX診斷</title><style>body{font:16px/1.8 system-ui;max-width:1250px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:9px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}</style><a href="策略研究_總結報告.html">回總結</a>'+rendered+v42_fig.to_html(full_html=False,include_plotlyjs=True,config={'responsive':True,'displaylogo':False})+v42_scatter.to_html(full_html=False,include_plotlyjs=False,config={'responsive':True,'displaylogo':False})+'</html>',encoding='utf-8')
v42_snippet='<section id="macro-vix-diagnosis"><h2>新增：虧損起點的景氣燈號與VIX</h2><p>同一組1209共同窗口：64.0%的動能虧損、60.5%的法人虧損，起點VIX仍低於20。高VIX組後24日期望反而較高，不能直接推成高VIX退出規則。</p><p>2020/2/14最差窗口開始前，最新景氣為綠燈27分、VIX14.15。已核對各年最差案例11份原公告；全月份原始燈號尚未補齊，不宣稱全樣本燈號回測完成。</p><a href="'+v42_hp.name+'">查看年度原公告、VIX勝率與期望、箱形圖及全部點 →</a></section>'
master=v34_root/'策略研究_總結報告.html';page=master.read_text(encoding='utf-8');assert 'macro-vix-diagnosis' not in page;page=page.replace('<section id="market-breadth-loss">',v42_snippet+'<section id="market-breadth-loss">');assert v42_snippet in page;master.write_text(page,encoding='utf-8')
mastermd=v34_root/'策略研究_總結報告.md';mastermd.write_text(mastermd.read_text(encoding='utf-8')+'\n\n## 景氣燈號與VIX核對（v42）\n\n同1209窗口，動能／法人虧損時起點VIX低於20比例64.0%／60.5%；VIX≥30組後24日期望反而較高，不能直接當退出規則。最差2020/2/14起點景氣綠燈27分、VIX14.15。年度最差案例核對11份原公告，原始全月燈號資料仍未齊。[原公告與全點箱形圖]('+v42_hp.name+')。\n',encoding='utf-8')
oldhp=v34_root/'動能法人_同期整體下跌分析_v41_2026-10-16.html';page=oldhp.read_text(encoding='utf-8');page=page.replace('</html>',v42_snippet+'</html>');oldhp.write_text(page,encoding='utf-8')
sp=v34_root/'A策略_無人值守研究狀態.json';state42=json.loads(sp.read_text(encoding='utf-8'));state42['macro_vix_diagnosis_v42']={'status':'completed_vix_descriptive_ndc_cases_only','running':False,'same_sample':1209,'later':965,'original_ndc_releases':11,'unique_case_starts':12,'full_monthly_ndc_pit':'incomplete','html':v42_hp.name,'not_policy_backtest':True};sp.write_text(json.dumps(state42,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
assert sum(len(t.y) for t in v42_fig.data if t.type=='box')==2418
assert sum(len(t.x) for t in v42_scatter.data)==2418
assert len(v42_case_records)==20 and len(v42_case_month)==12
print('V42_REPORT_COMPLETE',{'report':v42_hp.name,'boxpoints':2418,'scatterpoints':2418,'ndc_original_sources':len(v42_ndc_sources),'max_vix_stale_days':max(r['lag_days'] for r in v42_rows),'high_vix_years':v42_summary['pre_vix_bin_year_counts']['>=30']})


In [ ]:
# Source-link correction to URLs actually returned by official PDF search; artifact QA.
v42_url_fixes={
'https://www.ndc.gov.tw/nc_332_26346':'https://ws.ndc.gov.tw/Download.ashx?icon=.pdf&n=MTA15bm0OOaciOS7veaZr%2Bawo%2BamguazgeaWsOiBnueovy5wZGY%3D&u=LzAwMS9hZG1pbmlzdHJhdG9yLzEwL3JlbGZpbGUvNjIxNy8yNjM0Ni8wZGRhZjc4YS0wZjhhLTQ2NjAtYjg1ZC03ZjVjYzY1MGRkNzUucGRm',
'https://www.ndc.gov.tw/nc_332_30068':'https://ws.ndc.gov.tw/Download.ashx?icon=.pdf&n=MTA35bm0N%2BaciOS7veaZr%2Bawo%2BamguazgeaWsOiBnueovy5wZGY%3D&u=LzAwMS9hZG1pbmlzdHJhdG9yLzEwL3JlbGZpbGUvNjIxNy8zMDA2OC9hNWM1NzZhNy1iMjE5LTQxYjktOWNhZi1iMzM5MzhlY2Y2YjgucGRm',
'https://www.ndc.gov.tw/nc_332_35682':'https://ws.ndc.gov.tw/Download.ashx?icon=.pdf&n=5ZyL55m85pyDMTAxMDMyOOaWsOiBnueovy0xMTHlubQy5pyI5Lu95pmv5rCj5qaC5rOBLnBkZg%3D%3D&u=LzAwMS9hZG1pbmlzdHJhdG9yLzEwL3JlbGZpbGUvNjIxNy8zNTY4Mi81MmJjOTE5Yy03OTE0LTQ1ZmUtYTczNS0zYTBkMjNkZDQ0ZjcucGRm'}
# Use stable official direct asset paths decoded from the returned PDF URLs.
import base64,urllib.parse
for old,u in list(v42_url_fixes.items()):
    path=base64.b64decode(urllib.parse.parse_qs(urllib.parse.urlsplit(u).query)['u'][0]).decode('utf-8');v42_url_fixes[old]='https://ws.ndc.gov.tw'+path
for src in v42_ndc_sources:src['url']=v42_url_fixes.get(src['url'],src['url'])
for fp in [v42_rp,v42_hp,v34_root/'景氣與VIX_原始公告核對_v42_2026-10-16.json']:
    content=fp.read_text(encoding='utf-8')
    for old,new in v42_url_fixes.items():content=content.replace(old,new)
    content=content.replace('落差1～6個日曆日','落差1～4個日曆日');fp.write_text(content,encoding='utf-8')
v42_validation={'vix_windows':len(v42_rows),'later_windows':sum(r['later'] for r in v42_rows),'missing_vix':sum(not np.isfinite(r['vix']) for r in v42_rows),'max_lag_days':max(r['lag_days'] for r in v42_rows),'all_pre_entry_dates':all(r['vix_date']<r['start'] for r in v42_rows),'box_observations':sum(len(t.y) for t in v42_fig.data if t.type=='box'),'original_ndc_sources':len(v42_ndc_sources),'distinct_case_starts':len(v42_case_month),'monthly_ndc_pit_complete':False,'no_policy_backtest':True,'notebook_saved':True,'visual_browser_qa':'not performed; programmatic validation only'}
assert v42_validation['vix_windows']==1209 and v42_validation['later_windows']==965 and v42_validation['missing_vix']==0 and v42_validation['all_pre_entry_dates']
for period in ['all','later']:
    for k in ['MOM_E0','FLOW_E0']:
        stat=v42_summary['periods'][period][k];assert sum(b['n'] for b in stat['bins'].values())==(1209 if period=='all' else 965)
        assert sum(b['loss_n'] for b in stat['bins'].values())==stat['loss_vs_nonloss']['True']['n']
assert 'macro-vix-diagnosis' in (v34_root/'策略研究_總結報告.html').read_text(encoding='utf-8')
assert 'unified-longhistory-boxplot' in (v34_root/'策略研究_總結報告.html').read_text(encoding='utf-8')
assert 'Plotly.newPlot' in v42_hp.read_text(encoding='utf-8')
(v34_root/'景氣與VIX_驗證_v42_2026-10-16.json').write_text(json.dumps(v42_validation,ensure_ascii=False,indent=2),encoding='utf-8')
print('V42_VALIDATED',v42_validation)


In [ ]:
# Resume retained kernel; inspect public source metadata before any bulk download.
import urllib.request,urllib.parse,re,datetime,collections,json
print('KERNEL_RETAINED',{'python':__import__('sys').executable,'windows':len(v42_rows),'objects':[k for k in ['v30_arrays','v34_results','v42_vix'] if k in globals()]})
v43_definition={'purpose':'same2016–2025 pre-entry Taiwan/US VIX environmental comparison','sample':1209,'later':965,'no_strategy_optimization':True,'alignment':'Taiwan daily close strictly before entry; US date strictly before entry; maximum7 calendar days; no backfill with future data','planned_groups':'Taiwan/US both<20, Taiwan only>=20, US only>=20, both>=20; same bins20/30 diagnostic; no threshold search','sources':[],'status':'source_discovery'}
(v34_root/'台美VIX_研究定義_v43_2026-10-16.json').write_text(json.dumps(v43_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v43_source_pages={};v43_attempts=[]
for label,url in [('official_daily','https://www.taifex.com.tw/cht/7/vixDaily3MNew'),('official_indexes','https://www.taifex.com.tw/indes/index.aspx'),('official_report','https://www.taifex.com.tw/file/taifex/CHINESE/10/Dailydownload/Daily_2022_10_31.pdf')]:
    try:
        with urllib.request.urlopen(url,timeout=30) as resp:raw=resp.read()
        v43_source_pages[label]=raw;v43_attempts.append({'source':url,'attempt':1,'bytes':len(raw),'status':'ok'})
        if not url.endswith('.pdf'):
            s=raw.decode('utf-8',errors='replace')
            print(label,{'bytes':len(raw),'scripts':re.findall(r'<script[^>]+src=[\"\x27]([^\"\x27]+)',s),'interesting_lines':[t.strip()[:900] for t in s.splitlines() if any(w.lower() in t.lower() for w in ['csv','download','querydate','querymonth','fetch(','ajax','chart','vixDaily','vixChart'])][-45:]})
        else:
            import pymupdf
            doc=pymupdf.open(stream=raw,filetype='pdf');txt='\n'.join(pg.get_text() for pg in doc)
            at=txt.find('波動率指數');print('OFFICIAL_DAILY_PDF_VIX',txt[max(0,at-150):at+2000]);print('PDF_VIX_TAIL',txt[-4000:])
    except Exception as exc:v43_attempts.append({'source':url,'attempt':1,'status':type(exc).__name__});print(label,'SOURCE_ERROR',type(exc).__name__)
print('SOURCE_ATTEMPTS',v43_attempts)


In [ ]:
# Probe published monthly archive naming pattern at representative old dates.
import importlib.util
print('PDF_LIBRARIES',{x:importlib.util.find_spec(x) is not None for x in ['pypdf','pdfplumber','fitz']})
v43_month_files={}
for ym in ['201610','202002','202510']:
    url='https://www.taifex.com.tw/file/taifex/Dailydownload/vix/log2data/'+ym+'new.txt'
    try:
        with urllib.request.urlopen(url,timeout=25) as resp:raw=resp.read()
        v43_month_files[ym]=raw;v43_attempts.append({'source':url,'attempt':1,'bytes':len(raw),'status':'ok'})
        text=raw.decode('utf-8',errors='replace');print('MONTHLY_ARCHIVE',ym,len(raw),text[:700])
    except Exception as exc:v43_attempts.append({'source':url,'attempt':1,'status':type(exc).__name__});print('MONTHLY_ARCHIVE',ym,type(exc).__name__)
txt43=v43_source_pages['official_indexes'].decode('utf-8',errors='replace');at=txt43.find('$.ajax');print('INDEX_CHART_ENDPOINT',txt43[max(0,at-900):at+2100])
if importlib.util.find_spec('pypdf'):
    import pypdf
    reader=pypdf.PdfReader(io.BytesIO(v43_source_pages['official_report']));txt43pdf='\n'.join(pg.extract_text() for pg in reader.pages)
    print('DAILY_REPORT_TAIL',txt43pdf[-7000:])


In [ ]:
# Install only lightweight PDF parser in current kernel, without restart.
get_ipython().run_line_magic('pip','install --quiet pypdf')
import pypdf
print('PYPDF_VERSION',pypdf.__version__)
reader43=pypdf.PdfReader(io.BytesIO(v43_source_pages['official_report']));v43_pdftexts=[pg.extract_text() for pg in reader43.pages]
for i,txt in enumerate(v43_pdftexts):
    if '波動率指數' in txt:print('OFFICIAL_PDF_PAGE',i,txt[-7000:])
# Confirm soft404 responses are not data.
for ym,raw in list(v43_month_files.items()):
    if b'<HTML' in raw.upper() or b'<TITLE>404' in raw.upper():
        del v43_month_files[ym]
        for a in v43_attempts:
            if ym+'new.txt' in a['source']:a['status']='soft404_not_data'
v43_chart_url='https://www.taifex.com.tw/indes/index.aspx/GetStockDayPrices'
v43_payload={'syid':'TAIWANVIX','flag':'MS','startDate':'2023/10/17','endDate':'2026/10/16'}
try:
    req=urllib.request.Request(v43_chart_url,data=json.dumps(v43_payload).encode('utf-8'),headers={'Content-Type':'application/json; charset=utf-8'},method='POST')
    with urllib.request.urlopen(req,timeout=30) as resp:raw=resp.read()
    obj=json.loads(raw);v43_chart=json.loads(obj['d']) if isinstance(obj.get('d'),str) else obj
    print('SUPPORTED_CHART_METADATA',{k:(v[:2]+v[-2:] if isinstance(v,list) else v) for k,v in v43_chart.items()});v43_attempts.append({'source':v43_chart_url,'request':v43_payload,'status':'ok','bytes':len(raw)})
except Exception as exc:v43_attempts.append({'source':v43_chart_url,'attempt':1,'status':type(exc).__name__});print('CHART_ERROR',type(exc).__name__)


In [ ]:
# Bounded source follow-up; inspect only safe object metadata, never credentials.
print('RELATED_OBJECT_METADATA',[(k,type(v).__name__,getattr(v,'shape',None)) for k,v in list(globals().items()) if any(q in k.lower() for q in ['schema','catalog','engine','connection','tables','vix']) and not k.startswith('_')][:60])
print('CHART_RESPONSE_METADATA',{'bytes':len(raw),'prefix':raw[:180].decode('utf-8',errors='replace')})
print('CHART_INPUTS',[t.strip()[:900] for t in txt43.splitlines() if any(q in t for q in ['startDate','endDate','minDate','maxDate','3 Years','3年'])][-30:])
v43_latest_url=re.findall(r'https://www.taifex.com.tw/file/taifex/Dailydownload/vix/log2data/[0-9]+new.txt',v43_source_pages['official_daily'].decode('utf-8'))[0]
with urllib.request.urlopen(v43_latest_url,timeout=25) as response:v43_current_text=response.read()
print('LATEST_FILE_FORMAT',v43_current_text.decode('utf-8',errors='replace')[:1100])
# Correct API request to use actual ISO date values used by the page's date input.
v43_payload2={'syid':'TAIWANVIX','flag':'MS','startDate':'2023-10-17','endDate':'2026-10-16'}
req=urllib.request.Request(v43_chart_url,data=json.dumps(v43_payload2).encode('utf-8'),headers={'Content-Type':'application/json; charset=utf-8','Referer':'https://www.taifex.com.tw/indes/index.aspx','Accept':'application/json'},method='POST')
try:
    with urllib.request.urlopen(req,timeout=25) as response:v43_response2=response.read();status=response.status;ctype=response.headers.get('Content-Type')
    parsed=json.loads(v43_response2);v43_chart=json.loads(parsed['d']) if isinstance(parsed.get('d'),str) else parsed
    print('CHART_SECOND_METADATA',{k:(v[:2]+v[-2:] if isinstance(v,list) else v) for k,v in v43_chart.items()});v43_attempts.append({'source':v43_chart_url,'attempt':2,'request':v43_payload2,'status':'ok','bytes':len(v43_response2)})
except Exception as exc:
    v43_attempts.append({'source':v43_chart_url,'attempt':2,'status':type(exc).__name__});print('CHART_SECOND_ERROR',type(exc).__name__,{'response_bytes':len(v43_response2) if 'v43_response2' in globals() else None,'content_type':ctype if 'ctype' in globals() else None})


In [ ]:
# Respect official past-three-year limit; target historical end2025, not today's invalid end.
v43_payload3={'syid':'TAIWANVIX','flag':'MS','startDate':'2023-10-17','endDate':'2025-12-31'}
req=urllib.request.Request(v43_chart_url,data=json.dumps(v43_payload3).encode('utf-8'),headers={'Content-Type':'application/json; charset=utf-8','Referer':'https://www.taifex.com.tw/indes/index.aspx','Accept':'application/json'},method='POST')
with urllib.request.urlopen(req,timeout=30) as response:v43_response3=response.read()
outer43=json.loads(v43_response3);inner43=outer43['d']
v43_chart=json.loads(inner43) if isinstance(inner43,str) else inner43
assert isinstance(v43_chart,dict) and isinstance(v43_chart['TrendData'],list)
v43_attempts.append({'source':v43_chart_url,'attempt':3,'request':v43_payload3,'status':'ok','bytes':len(v43_response3)})
(v34_root/'台灣VIX_官方近三年查詢原始回覆_v43_2026-10-16.json').write_bytes(v43_response3)
print('CHART_VALID_METADATA',{k:(v[:3]+v[-3:] if isinstance(v,list) else v) for k,v in v43_chart.items()})
print('ORIGINAL_ENGINE_OBJECT_METADATA',[(k,type(v).__name__) for k,v in list(globals().items()) if any(q in k.lower() for q in ['db','conn','cursor','sql']) and not k.startswith('_')][:40])


In [ ]:
# Same available dates for TW/US indicators; no fills and no re-running strategies.
v43_tw=[{'date':datetime.datetime.strptime(x['Time'],'%Y%m%d').date().isoformat(),'close':float(x['Price'])} for x in v43_chart['TrendData']]
v43_tw.sort(key=lambda r:r['date']);v43_tw_dates=[r['date'] for r in v43_tw]
assert len(v43_tw_dates)==len(set(v43_tw_dates)) and all(0<r['close']<200 for r in v43_tw)
# Independently check chart values against current published daily13:45 files, never use2026 as a research outcome.
payload43check={'syid':'TAIWANVIX','flag':'MS','startDate':'2026-10-01','endDate':'2026-10-15'}
req=urllib.request.Request(v43_chart_url,data=json.dumps(payload43check).encode('utf-8'),headers={'Content-Type':'application/json; charset=utf-8'},method='POST')
with urllib.request.urlopen(req,timeout=30) as resp:checkbytes=resp.read()
check43=json.loads(json.loads(checkbytes)['d']);v43_month_parsed={}
for line in v43_current_text.decode('big5',errors='strict').splitlines():
    fields=line.split()
    if len(fields)>=3 and re.fullmatch(r'\d{8}',fields[0]):v43_month_parsed[fields[0]]={'close':float(fields[2]),'last_minute_mean':float(fields[3]) if len(fields)>3 else None}
v43_crosscheck=[{'date':x['Time'],'chart':x['Price'],'daily_close':v43_month_parsed[x['Time']]['close'],'one_minute_mean':v43_month_parsed[x['Time']]['last_minute_mean']} for x in check43['TrendData'] if x['Time'] in v43_month_parsed]
assert len(v43_crosscheck)>=5 and all(abs(r['chart']-r['daily_close'])<1e-9 for r in v43_crosscheck)
v43_rows=[];v43_missing=[]
for old in v42_rows:
    j=bisect.bisect_left(v43_tw_dates,old['start'])-1
    if j<0:v43_missing.append({'start':old['start'],'reason':'before_available_TW_history'});continue
    lag=(datetime.date.fromisoformat(old['start'])-datetime.date.fromisoformat(v43_tw_dates[j])).days
    expected=v30_days[v30_starts[old['index']]-1]
    if v43_tw_dates[j]!=expected or lag>7:
        v43_missing.append({'start':old['start'],'reason':'missing_previous_TW_day_or_stale','expected':expected,'actual':v43_tw_dates[j]});continue
    r={k:old[k] for k in ['index','start','end','later','vix','vix_date','vix_change5','market_broad_down','families']};r.update(tw_vix=v43_tw[j]['close'],tw_vix_date=v43_tw[j]['date'],tw_change5=v43_tw[j]['close']-v43_tw[j-5]['close'] if j>=5 else None)
    r['level_group']=('台美均低於20' if r['tw_vix']<20 and r['vix']<20 else ('僅台灣≥20' if r['tw_vix']>=20 and r['vix']<20 else ('僅美國≥20' if r['tw_vix']<20 and r['vix']>=20 else '台美均≥20')))
    r['tw_bin']='<20' if r['tw_vix']<20 else ('20–30' if r['tw_vix']<30 else '>=30')
    r['rise_group']=None if r['tw_change5'] is None else ('台美均未上升' if r['tw_change5']<=0 and r['vix_change5']<=0 else ('僅台灣上升' if r['tw_change5']>0 and r['vix_change5']<=0 else ('僅美國上升' if r['tw_change5']<=0 and r['vix_change5']>0 else '台美均上升')))
    v43_rows.append(r)
v43_definition.update(status='partial_history_available',tw_data_range=[v43_tw_dates[0],v43_tw_dates[-1]],actual_windows=len(v43_rows),missing_windows=len(v43_missing),source=v43_chart_url,query=v43_payload3,thresholds={'level':[20,30],'change':'difference from5 sessions earlier >0; no threshold search'},dataset_limit='official free past-three-year chart; cannot claim2016–2025 full coverage')
(v34_root/'台美VIX_研究定義_v43_2026-10-16.json').write_text(json.dumps(v43_definition,ensure_ascii=False,indent=2),encoding='utf-8')
def v43_metrics(ss,k):
    ix=[r['index'] for r in ss];ret=np.array([r['families'][k]['net'] for r in ss],float);res=[v34_results[k][i] for i in ix]
    return {'n':len(ss),'win':float(np.mean(ret>0)) if len(ss) else None,'ev':float(np.mean(ret)) if len(ss) else None,'median':float(np.median(ret)) if len(ss) else None,'worst_return':float(np.min(ret)) if len(ss) else None,'mean_mdd':float(np.mean([r['mdd'] for r in res])) if len(ss) else None,'worst_mdd':float(np.min([r['mdd'] for r in res])) if len(ss) else None,'mean_fee':float(np.mean([r['cost'] for r in res])) if len(ss) else None,'loss_n':int(np.sum(ret<0)),'years':dict(collections.Counter(r['start'][:4] for r in ss))}
v43_summary={'definition':v43_definition,'crosscheck_current2026':v43_crosscheck,'n':len(v43_rows),'missing_n':len(v43_missing),'groups':{},'loss_diagnostics':{},'annual':{},'phase_counts':dict(collections.Counter(r['index']%24 for r in v43_rows))}
for key,labels in [('level_group',['台美均低於20','僅台灣≥20','僅美國≥20','台美均≥20']),('tw_bin',['<20','20–30','>=30']),('rise_group',['台美均未上升','僅台灣上升','僅美國上升','台美均上升'])]:
    v43_summary['groups'][key]={label:{k:v43_metrics([r for r in v43_rows if r[key]==label],k) for k in ['MOM_E0','FLOW_E0']} for label in labels}
for k in ['MOM_E0','FLOW_E0']:
    v43_summary['loss_diagnostics'][k]={}
    for loss in [True,False]:
        ss=[r for r in v43_rows if r['families'][k]['portfolio_loss']==loss]
        v43_summary['loss_diagnostics'][k][str(loss)]={'n':len(ss),'pre_tw20_fraction':float(np.mean([r['tw_vix']>=20 for r in ss])) if ss else None,'pre_us20_fraction':float(np.mean([r['vix']>=20 for r in ss])) if ss else None,'pre_both20_fraction':float(np.mean([r['vix']>=20 and r['tw_vix']>=20 for r in ss])) if ss else None}
for yr in ['2023','2024','2025']:v43_summary['annual'][yr]={k:v43_metrics([r for r in v43_rows if r['start'].startswith(yr)],k) for k in ['MOM_E0','FLOW_E0']}
for name,obj in [('台美VIX_部分期間統計_v43_2026-10-16.json',v43_summary),('台美VIX_部分期間逐窗口_v43_2026-10-16.json',v43_rows),('台美VIX_缺漏窗口_v43_2026-10-16.json',v43_missing),('台美VIX_來源嘗試_v43_2026-10-16.json',v43_attempts)]:
    (v34_root/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
print('TW_US_SAME_SAMPLE',{'n':len(v43_rows),'range':[v43_rows[0]['start'],v43_rows[-1]['start']],'missing':len(v43_missing),'official_crosschecks':len(v43_crosscheck),'groups':v43_summary['groups'],'losses':v43_summary['loss_diagnostics']})


In [ ]:
# Minimal diagnostics only, no history or credential material.
import traceback,sys
e=getattr(sys,'last_exc',None)
print('V43_ERROR_LOCATION',[(t.name,t.lineno) for t in traceback.extract_tb(e.__traceback__)] if e else None)
print('INDEX_METADATA',{'days':len(v30_days),'starts':len(v30_starts),'rows':len(v42_rows),'index_range':[min(r['index'] for r in v42_rows),max(r['index'] for r in v42_rows)],'tw':len(v43_tw),'result_lengths':{k:len(v34_results[k]) for k in ['MOM_E0','FLOW_E0']},'partial_rows':len(v43_rows) if 'v43_rows' in globals() else None,'response':str(check43)[:350] if 'check43' in globals() else None})


In [ ]:
# Normalize retained Python dates to ISO strings before exact previous-day matching. Reuse fetched data; no network repeat.
v43_rows=[];v43_missing=[]
for old in v42_rows:
    j=bisect.bisect_left(v43_tw_dates,old['start'])-1
    if j<0:v43_missing.append({'start':old['start'],'reason':'before_available_TW_history'});continue
    lag=(datetime.date.fromisoformat(old['start'])-datetime.date.fromisoformat(v43_tw_dates[j])).days
    expected=str(v30_days[v30_starts[old['index']]-1])[:10]
    if v43_tw_dates[j]!=expected or lag>7:
        v43_missing.append({'start':old['start'],'reason':'missing_previous_TW_day_or_stale','expected':expected,'actual':v43_tw_dates[j]});continue
    r={k:old[k] for k in ['index','start','end','later','vix','vix_date','vix_change5','market_broad_down','families']};r.update(tw_vix=v43_tw[j]['close'],tw_vix_date=v43_tw[j]['date'],tw_change5=v43_tw[j]['close']-v43_tw[j-5]['close'] if j>=5 else None)
    r['level_group']=('台美均低於20' if r['tw_vix']<20 and r['vix']<20 else ('僅台灣≥20' if r['tw_vix']>=20 and r['vix']<20 else ('僅美國≥20' if r['tw_vix']<20 and r['vix']>=20 else '台美均≥20')))
    r['tw_bin']='<20' if r['tw_vix']<20 else ('20–30' if r['tw_vix']<30 else '>=30')
    r['rise_group']=None if r['tw_change5'] is None else ('台美均未上升' if r['tw_change5']<=0 and r['vix_change5']<=0 else ('僅台灣上升' if r['tw_change5']>0 and r['vix_change5']<=0 else ('僅美國上升' if r['tw_change5']<=0 and r['vix_change5']>0 else '台美均上升')))
    v43_rows.append(r)
v43_definition.update(status='partial_history_available',tw_data_range=[v43_tw_dates[0],v43_tw_dates[-1]],actual_windows=len(v43_rows),missing_windows=len(v43_missing),source=v43_chart_url,query=v43_payload3,thresholds={'level':[20,30],'change':'difference from5 sessions earlier >0; no threshold search'},dataset_limit='official free past-three-year chart; cannot claim2016–2025 full coverage')
(v34_root/'台美VIX_研究定義_v43_2026-10-16.json').write_text(json.dumps(v43_definition,ensure_ascii=False,indent=2),encoding='utf-8')
def v43_metrics(ss,k):
    ix=[r['index'] for r in ss];ret=np.array([r['families'][k]['net'] for r in ss],float);res=[v34_results[k][i] for i in ix]
    return {'n':len(ss),'win':float(np.mean(ret>0)) if len(ss) else None,'ev':float(np.mean(ret)) if len(ss) else None,'median':float(np.median(ret)) if len(ss) else None,'worst_return':float(np.min(ret)) if len(ss) else None,'mean_mdd':float(np.mean([r['mdd'] for r in res])) if len(ss) else None,'worst_mdd':float(np.min([r['mdd'] for r in res])) if len(ss) else None,'mean_fee':float(np.mean([r['cost'] for r in res])) if len(ss) else None,'loss_n':int(np.sum(ret<0)),'years':dict(collections.Counter(r['start'][:4] for r in ss))}
v43_summary={'definition':v43_definition,'crosscheck_current2026':v43_crosscheck,'n':len(v43_rows),'missing_n':len(v43_missing),'groups':{},'loss_diagnostics':{},'annual':{},'phase_counts':dict(collections.Counter(r['index']%24 for r in v43_rows))}
for key,labels in [('level_group',['台美均低於20','僅台灣≥20','僅美國≥20','台美均≥20']),('tw_bin',['<20','20–30','>=30']),('rise_group',['台美均未上升','僅台灣上升','僅美國上升','台美均上升'])]:
    v43_summary['groups'][key]={label:{k:v43_metrics([r for r in v43_rows if r[key]==label],k) for k in ['MOM_E0','FLOW_E0']} for label in labels}
for k in ['MOM_E0','FLOW_E0']:
    v43_summary['loss_diagnostics'][k]={}
    for loss in [True,False]:
        ss=[r for r in v43_rows if r['families'][k]['portfolio_loss']==loss]
        v43_summary['loss_diagnostics'][k][str(loss)]={'n':len(ss),'pre_tw20_fraction':float(np.mean([r['tw_vix']>=20 for r in ss])) if ss else None,'pre_us20_fraction':float(np.mean([r['vix']>=20 for r in ss])) if ss else None,'pre_both20_fraction':float(np.mean([r['vix']>=20 and r['tw_vix']>=20 for r in ss])) if ss else None}
for yr in ['2023','2024','2025']:v43_summary['annual'][yr]={k:v43_metrics([r for r in v43_rows if r['start'].startswith(yr)],k) for k in ['MOM_E0','FLOW_E0']}
for name,obj in [('台美VIX_部分期間統計_v43_2026-10-16.json',v43_summary),('台美VIX_部分期間逐窗口_v43_2026-10-16.json',v43_rows),('台美VIX_缺漏窗口_v43_2026-10-16.json',v43_missing),('台美VIX_來源嘗試_v43_2026-10-16.json',v43_attempts)]:
    (v34_root/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
print('TW_US_SAME_SAMPLE',{'n':len(v43_rows),'range':[v43_rows[0]['start'],v43_rows[-1]['start']],'missing':len(v43_missing),'official_crosschecks':len(v43_crosscheck),'groups':v43_summary['groups'],'losses':v43_summary['loss_diagnostics']})


In [ ]:
# Persist partial-period findings separately from ten-year conclusions.
v43_high=[r for r in v43_rows if r['tw_vix']>=30];intervals43=sorted((int(r['index']),int(r['index'])+23) for r in v43_high);clusters43=[]
for left,right in intervals43:
    if not clusters43 or left>clusters43[-1][1]:clusters43.append([left,right])
    else:clusters43[-1][1]=max(clusters43[-1][1],right)
v43_summary['high_tw_vix_audit']={'n':len(v43_high),'start_range':[r['start'] for r in [v43_high[0],v43_high[-1]]],'years':dict(collections.Counter(r['start'][:4] for r in v43_high)),'overlapping_24day_clusters':len(clusters43),'phase_counts':dict(collections.Counter(r['index']%24 for r in v43_high))}
(v34_root/'台美VIX_部分期間統計_v43_2026-10-16.json').write_text(json.dumps(v43_summary,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
lines43=['# 台灣與美國 VIX：同日起點配對診斷','2026-10-16｜**部分期間完成，2016～2025完整台灣VIX比較仍未完成。**','## 先知道這份報告能回答什麼','官方近三年日資料取得539筆，2023/10/17～2025/12/31。與既有選股圖1209共同窗口配對後，252個窗口可用，起點2023/12/8～2025/10/9；其餘957個窗口缺少所需歷史資料。本頁沒有把近期結果當十年結論，也沒有變更總結原有十年箱形圖。','目前最有用的結論是：**台灣VIX也不是高了就一定該退出。**兩地均高、只有台灣高、兩地均低，後續費後報酬沒有單調排序。這只是對既有動能與法人策略的環境分類，並非新策略回測。','## 資料與時間怎麼對齊？','- 台灣：取進場前最近一個台股交易日的TAIWAN VIX日收盤，必須與股票資料的上一交易日相同，且不超過7日曆日；不拿進場當天13:45數值提前使用。','- 美國：同一窗口使用台灣起點日期之前最近一個美國日期的Cboe VIX收盤，與前份報告相同。','- 兩個都是30天預期波動，數值20不代表有20%下跌機率。台灣VIX反映加權指數選擇權，不能完整代表我們含上櫃的150檔股票池。','- 固定20／30分組，不搜尋最好門檻。另看近5個各自市場交易日的點數變化>0，作為「升高」；不是每日換股規則。','- 官方圖表回覆同時帶有2026最新報價欄位，研究**只讀TrendData內2023～2025日期資料**，不使用CurrentPrice／Datadate。','- 額外用2026/10/1～15共10筆檢查官方圖表值與13:45日檔收盤一致，不混用收盤前一分鐘平均。這些2026資料只做欄位查核，未進研究統計。','## 台灣與美國一起看','勝率＝既有費後24日報酬>0比例；期望＝費後24日平均報酬。每一列動能與法人完全同起點。組別間時間不同，並非同日兩個新策略的配對優劣實驗。','|起點前VIX狀態|窗口n|動能勝率|動能期望|法人勝率|法人期望|','|---|---:|---:|---:|---:|---:|']
for g,methods in v43_summary['groups']['level_group'].items():
    a=methods['MOM_E0'];b=methods['FLOW_E0'];lines43.append('|'+ '|'.join([g,str(a['n']),p30(a['win']),p30(a['ev']),p30(b['win']),p30(b['ev'])])+'|')
lines43+=['「僅美國≥20」只有2個窗口，不能判定任何穩定差異。台美均≥20組54窗口中44個來自2025，不把年份差異誤當VIX獨立效果。','## 有沒有更嚴重的下跌風險？','下表的最大回撤取既有策略每個窗口的淨值路徑，不是VIX本身回撤。費用是每24日窗口成本占初始資金的平均。','|狀態|動能平均回撤／最差回撤|法人平均回撤／最差回撤|動能平均費用|法人平均費用|','|---|---:|---:|---:|---:|']
for g,methods in v43_summary['groups']['level_group'].items():
    a=methods['MOM_E0'];b=methods['FLOW_E0'];lines43.append('|'+ '|'.join([g,p30(a['mean_mdd'])+'／'+p30(a['worst_mdd']),p30(b['mean_mdd'])+'／'+p30(b['worst_mdd']),p30(a['mean_fee']),p30(b['mean_fee'])])+'|')
lines43+=['兩地均≥20的動能平均回撤約−7.44%，兩地均低約−6.64%；法人分別約−7.15%、−4.44%。這是較高波動環境中風險較大的描述，尚未證明靠減碼可以提高費後結果。','## 台灣VIX自己分組','|台灣VIX|n|動能勝率／期望|法人勝率／期望|','|---|---:|---:|---:|']
for g,methods in v43_summary['groups']['tw_bin'].items():
    a=methods['MOM_E0'];b=methods['FLOW_E0'];lines43.append('|'+ '|'.join([g,str(a['n']),p30(a['win'])+'／'+p30(a['ev']),p30(b['win'])+'／'+p30(b['ev'])])+'|')
audit=v43_summary['high_tw_vix_audit'];lines43+=['**不能把上表100%勝率當保證。**≥30只有18個重疊窗口，全在2025；起點'+audit['start_range'][0]+'～'+audit['start_range'][1]+'，按持有期相互重疊連結為'+str(audit['overlapping_24day_clusters'])+'群。不是18次獨立危機的成功紀錄。','## 若看近期恐慌是否同時升高？','|近5個各自交易日VIX變化|n|動能勝率／期望|法人勝率／期望|','|---|---:|---:|---:|']
for g,methods in v43_summary['groups']['rise_group'].items():
    a=methods['MOM_E0'];b=methods['FLOW_E0'];lines43.append('|'+ '|'.join([g,str(a['n']),p30(a['win'])+'／'+p30(a['ev']),p30(b['win'])+'／'+p30(b['ev'])])+'|')
lines43+=['台美均上升組沒有呈現平均報酬必定為負。若要把「突然升高」做停損或減碼，需另定可成交規則並檢查是否錯過反彈與增加費用；本頁沒有宣稱已測出有效風控。','## 虧損前，有多少已是台灣高VIX？','|本次252窗口|動能|法人|','|---|---:|---:|']
for label,loss,field in [('虧損窗口數',True,'n'),('虧損時，起點台灣VIX≥20比例',True,'pre_tw20_fraction'),('未虧損時，起點台灣VIX≥20比例',False,'pre_tw20_fraction'),('虧損時，起點美國VIX≥20比例',True,'pre_us20_fraction')]:
    values=[v43_summary['loss_diagnostics'][k][str(loss)][field] for k in ['MOM_E0','FLOW_E0']];lines43.append('|'+ '|'.join([label]+[str(x) if field=='n' else p30(x) for x in values])+'|')
lines43+=['## 為什麼還沒有完整十年？','1. [官方指數圖表](https://www.taifex.com.tw/indes/index.aspx)明載免費查詢最近三年。按這個範圍取得2023/10以後資料，沒有繞過限制。','2. [每日收盤下載頁](https://www.taifex.com.tw/cht/7/vixDaily3MNew)為近三個月。測試2016/10、2020/2、2025/10同命名歷史檔均回404頁，不是有效資料，不能當成抓取成功。','3. [官方歷史資料商品](https://edatashop.taifex.com.tw/zh/product/detail/40283ab7890b3664018924255bf2000f)可申購2007/1起資料。查核頁標價每半年3,000元，屬盤中明細商品，不能假定只要日收盤也必須買此商品；未下單、未付款。','4. FinMind目前公開資料集說明的TaiwanOptionVix為2026/3起，不能補我們2016～2025。未拿美國VIX、VIX期貨ETF或自己算的歷史波動冒充台灣VIX。','5. 舊官方每日PDF的VIX頁主要是圖形，無可直接核對的逐日數值表；未用讀圖估值補成正式樣本。','完整歷史分支保留為資料缺口：需要既有授權資料庫／匯出的台灣VIX日收盤，或另確認官方是否提供較便宜的日資料。取得後依相同定義補齊957個缺漏窗口，重算台美比較，才可更新十年結論。','## 保存與限制','原Notebook353～361保存來源、格式查核、日期修正與結果；沿用原kernel，未重啟。pypdf6.21.0只安裝於目前Python環境供官方PDF唯讀擷取。252窗口不是獨立樣本，後段已多次研究，沒有新的未見測試集或顯著性主張。','[部分期間逐窗口](台美VIX_部分期間逐窗口_v43_2026-10-16.json) · [完整統計](台美VIX_部分期間統計_v43_2026-10-16.json) · [缺漏窗口](台美VIX_缺漏窗口_v43_2026-10-16.json) · [回總結](策略研究_總結報告.html)。']
v43_text='\n'.join(lines43);v43_rp=v34_root/'台美VIX_部分期間比較與資料缺口_v43_2026-10-16.md';v43_rp.write_text(v43_text,encoding='utf-8')
v43_fig=go.Figure();v43_tracegroups={};v43_titles={'level_group':'台美VIX水準比較','tw_bin':'台灣VIX分組','rise_group':'台美近5日VIX升高比較'}
for group,stats in v43_summary['groups'].items():
    traceidx=[]
    for k,color in [('MOM_E0','#167368'),('FLOW_E0','#567bb5')]:
        for label in stats:
            ss=[r for r in v43_rows if r[group]==label];name=('動能／' if k=='MOM_E0' else '法人／')+label;traceidx.append(len(v43_fig.data))
            v43_fig.add_trace(go.Box(name=name,y=[r['families'][k]['net']*100 for r in ss],boxpoints='all',jitter=.4,pointpos=0,line={'color':color},fillcolor='rgba(255,255,255,0)',marker={'color':color,'size':4,'opacity':.5},visible=group=='level_group',customdata=[[r['start'],r['end'],r['tw_vix_date'],r['tw_vix'],r['vix_date'],r['vix']] for r in ss],hovertemplate='%{customdata[0]}→%{customdata[1]}<br>費後24日 %{y:.2f}%<br>台灣 %{customdata[3]:.2f}（%{customdata[2]}）<br>美國 %{customdata[5]:.2f}（%{customdata[4]}）<extra></extra>'))
            traceidx.append(len(v43_fig.data));v43_fig.add_trace(go.Scatter(x=[name],y=[float(np.median([r['families'][k]['net']*100 for r in ss]))],mode='markers',marker={'color':'#df4385','symbol':'line-ew','size':22,'line':{'width':3}},visible=group=='level_group',showlegend=False,hovertemplate='中位數 %{y:.2f}%<extra></extra>'))
    v43_tracegroups[group]=traceidx
v43_fig.add_hline(y=0,line_color='#222',line_width=2);v43_fig.update_layout(template='plotly_white',height=700,showlegend=False,title='台美VIX水準比較｜部分期間252窗口，全部點',yaxis={'title':'費後24日報酬 %','range':[float(min(r['families'][k]['net']*100 for r in v43_rows for k in ['MOM_E0','FLOW_E0']))-3,float(max(r['families'][k]['net']*100 for r in v43_rows for k in ['MOM_E0','FLOW_E0']))+3]},xaxis={'tickangle':-25},margin={'b':155,'t':100},updatemenus=[{'type':'dropdown','x':0,'y':1.15,'buttons':[{'label':v43_titles[g],'method':'update','args':[{'visible':[i in ids for i in range(len(v43_fig.data))]},{'title':v43_titles[g]+'｜部分期間252窗口，全部點'}]} for g,ids in v43_tracegroups.items()]}])
v43_hp=v34_root/'台美VIX_部分期間比較與資料缺口_v43_2026-10-16.html';render43=MarkdownIt('commonmark',{'html':False}).enable('table').render(v43_text).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')
v43_hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>台美VIX部分期間診斷</title><style>body{font:16px/1.8 system-ui;max-width:1250px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:9px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}</style><a href="策略研究_總結報告.html">回總結</a>'+render43+v43_fig.to_html(full_html=False,include_plotlyjs=True,config={'responsive':True,'displaylogo':False})+'</html>',encoding='utf-8')
snippet43='<section id="taiwan-vix-partial"><h2>台灣VIX補查：部分期間，尚非十年結論</h2><p>官方日資料2023/10～2025/12，與原圖配對252／1209窗口。台美高低分組沒有單調的報酬排序；台灣VIX≥30的18個窗口全在2025且重疊，不能把100%勝率當穩定規則。</p><p>957窗口仍缺所需台灣歷史VIX，不變更原十年策略結論。</p><a href="'+v43_hp.name+'">查看勝率、期望、回撤、全部點與資料缺口 →</a></section>'
summarypath43=v34_root/'策略研究_總結報告.html';page43=summarypath43.read_text(encoding='utf-8');assert 'taiwan-vix-partial' not in page43;page43=page43.replace('<section id="macro-vix-diagnosis">',snippet43+'<section id="macro-vix-diagnosis">');assert snippet43 in page43;summarypath43.write_text(page43,encoding='utf-8')
summarymd43=v34_root/'策略研究_總結報告.md';summarymd43.write_text(summarymd43.read_text(encoding='utf-8')+'\n\n## 台灣VIX補查（v43，部分期間）\n\n官方近三年日資料配對252／1209窗口；957仍缺台灣歷史資料。台美VIX分組沒有單調的費後報酬排序；台灣≥30的18窗口均在2025且相互重疊，不可當100%勝率穩定規則。原十年比較不變。[部分期間統計與資料缺口]('+v43_hp.name+')。\n',encoding='utf-8')
prevhp43=v34_root/'動能法人_景氣燈號與恐慌指數_v42_2026-10-16.html';page43=prevhp43.read_text(encoding='utf-8');prevhp43.write_text(page43.replace('</html>',snippet43+'</html>'),encoding='utf-8')
statepath43=v34_root/'A策略_無人值守研究狀態.json';state43=json.loads(statepath43.read_text(encoding='utf-8'));state43['taiwan_us_vix_v43']={'status':'partial_period_completed_full_history_external_data_gap','running':False,'available_days':539,'paired_windows':252,'missing_windows':957,'original_sample':1209,'report':v43_hp.name,'no_policy_backtest':True,'no_purchase':True};statepath43.write_text(json.dumps(state43,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
validation43={'actual_tw_days':len(v43_tw),'paired':len(v43_rows),'missing':len(v43_missing),'all_previous_dates':all(r['tw_vix_date']<r['start'] and r['vix_date']<r['start'] for r in v43_rows),'current2026_not_in_research':all(r['start']<'2026' and r['tw_vix_date']<'2026' for r in v43_rows),'official_close_matches':len(v43_crosscheck),'boxpoints_per_view':{g:sum(len(v43_fig.data[i].y) for i in ids if v43_fig.data[i].type=='box') for g,ids in v43_tracegroups.items()},'full_history_complete':False,'visual_qa':'programmatic only'}
assert validation43['paired']+validation43['missing']==1209 and validation43['all_previous_dates'] and validation43['current2026_not_in_research']
assert all(n==504 for n in validation43['boxpoints_per_view'].values())
assert 'unified-longhistory-boxplot' in summarypath43.read_text(encoding='utf-8')
(v34_root/'台美VIX_驗證_v43_2026-10-16.json').write_text(json.dumps(validation43,ensure_ascii=False,indent=2),encoding='utf-8')
print('V43_PARTIAL_COMPLETE',{'html':v43_hp.name,'validation':validation43,'high_vix_audit':audit})


In [ ]:
# Restore only numerical metadata and retained engine source for finite risk-overlay study.
print('V44_RETAINED',{'python':sys.executable,'pid':os.getpid(),'prices':v30_arrays['close'].shape,'starts':len(v30_starts),'baseline_lengths':{k:len(v34_results[k]) for k in ['MOM_E0','FLOW_E0']},'existing_stop_cells':list(v36_results),'initial':INITIAL})
print('V34_ENGINE_SOURCE',v34_engine)
print('V36_ENGINE_SOURCE',v36_source)
print('V36_INDICATORS',indic[:6000])


In [ ]:
# Freeze finite context-aware overlays before looking at new outcome returns.
v44_definition={'version':'v44_context_exit_reentry_1','input_hashes':v34_plan['common']['source_hashes'],'baseline_engine':hashlib.sha256(v34_engine.encode()).hexdigest(),'sample':'all2236 possible starts in2016–2025; paired valid windows;2016–2018calibration,2019–2025later already studied','shared':'same MOM20/60half or pure20dayFLOW selectors,25names,equal90initial,24sessions,regular5sessions,1000shares,VWAP execution,knownactions andexistingfees','market_universe':'current150 historical listed/available subset; not all Taiwan and not official index','market_known':'for5 and20 days: complete adjusted returns and no unknown action; at least max(50,ceil(.8*currently priced pool names))','market_weak':'prior20day breadth<40% AND mean of fixed-known-pool prior5day returns<0','market_stable':'prior20day breadth>=50% AND same prior5day mean>=0','recovery':'prior5day breadth>=60% AND mean>0, on2consecutive known days','stock_weak':'two consecutive closes below5day adjusted-level average; reusev36 ma5two; no parameter search','modes':{'LOCAL':'only when market stable, replace weak holdings nextday by next-ranked eligible healthy stocks; original firstday, healthy25 at regular stable reviews; otherdays replace failed slots only','MARKET_STAY80':'after first broad weak signal reduce target90to80; keep80 through window even if recovery','MARKET_RECOVER':'same reduce80; restore90 upon recovery, can reactivate later; changesonly orregular5days','COMBINED_RECOVER':'LOCAL plus MARKET_RECOVER'},'offcycle_sizing':'replacement matches previous holding value with1000share rounding; market exposure changes rescale current holdings, not rerank all stocks','unknown':'no extra trigger on unknown market state; preserve current risk state; reportunknown days, do not infer future','first_day':'all overlays initially same90; risk actions start day2, before eachday open from previousclose','fixed_controls':'6 controls, one per family × market mode; invested parameter calibrated to mean exposure of valid2016–2018 overlay windows only; keepconstant all years; label approximate realized exposure match, no resulttarget matching','primary_contrasts':'8overlays−own90baseline +2RECOVER−STAY +2COMBINED−RECOVER +6overlay−owncalibratedfixed=18','statistics':'paired feesafter; block24/48 on original positions5000,Bonferroni18; years/P10/worstMDD/coverage/training criteria reused; all/common andpairwise shown','full_gate':'both correctedCI lower>0;P10 andworstMDD>=control;coverage>=80%;training>=100;>=5years n>=20 and >=2/3positiveyears','vix':'USfull/TWpartial252 fromv42/v43 onlydiagnostic; no VIX-driven policy or monthly lights invented','compliance_limits':'existing simulated cash<25%,20–30holdings,caps2330 25%others10% with5daydriftgrace;ActiveShare and official executions not fully certified; no submittedtrades','no_extra_grid':True}
(v34_root/'市場情境_退出換股再進場定義_v44_2026-10-16.json').write_text(json.dumps(v44_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v44_definition_hash=hashlib.sha256(json.dumps(v44_definition,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
v44_market={'known':np.zeros(len(v30_days),bool),'weak':np.zeros(len(v30_days),bool),'stable':np.zeros(len(v30_days),bool),'recovery_raw':np.zeros(len(v30_days),bool),'recovery':np.zeros(len(v30_days),bool),'breadth20':np.full(len(v30_days),np.nan),'breadth5':np.full(len(v30_days),np.nan),'mean5':np.full(len(v30_days),np.nan)}
for q in range(20,len(v30_days)):
    present=v30_pool&np.isfinite(v30_arrays['close'][q]);need=max(50,int(np.ceil(.8*present.sum())));summ={}
    for look in [5,20]:
        rr=v30_arrays['returns'][q-look+1:q+1];ok=v30_pool&np.all(np.isfinite(rr)&~v30_arrays['bad'][q-look+1:q+1],axis=0)
        if ok.sum()<need:break
        x=np.prod(1+rr[:,ok],axis=0)-1;summ[look]=(float(np.mean(x>0)),float(x.mean()))
    if len(summ)!=2:continue
    b20=summ[20][0];b5,m5=summ[5];v44_market['known'][q]=True;v44_market['breadth20'][q]=b20;v44_market['breadth5'][q]=b5;v44_market['mean5'][q]=m5;v44_market['weak'][q]=(b20<.4 and m5<0);v44_market['stable'][q]=(b20>=.5 and m5>=0);v44_market['recovery_raw'][q]=(b5>=.6 and m5>0)
v44_market['recovery'][1:]=v44_market['recovery_raw'][1:]&v44_market['recovery_raw'][:-1]
v44weak=v36_ns['weak23']['ma5two'];v44healthy=v36_ns['known23']&~v44weak
v44_code=v34_engine.replace('def simulate_unified34(','def simulate_context44(').replace('rebalance=5):',"rebalance=5, mode='none'):")
v44_code=v44_code.replace('    for offset, t in enumerate(range(start, start + 24)):',"    exposures44=[];events44=[];riskoff44=False;unknown44=0;riskdays44=0\n    for offset, t in enumerate(range(start, start + 24)):")
anchor44='        if offset == 0 or (rebalance > 0 and offset % rebalance == 0):';end44='        if np.any((shares > 0) & ~np.isfinite(mark[t])):'
v44_decision=r'''        q=t-1
        regular44=offset==0 or (rebalance>0 and offset%rebalance==0)
        local44=mode in ['LOCAL','COMBINED_RECOVER'] and offset>0 and market44['stable'][q]
        hasmarket44=mode in ['MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']
        wanted44=riskoff44
        if hasmarket44 and offset>0:
            if not market44['known'][q]:unknown44+=1
            elif not riskoff44 and market44['weak'][q]:wanted44=True
            elif riskoff44 and mode!='MARKET_STAY80' and market44['recovery'][q]:wanted44=False
        changed44=wanted44!=riskoff44
        failed44=np.flatnonzero((shares>0)&weak44[q]) if local44 else np.array([],dtype=int)
        if regular44 or len(failed44) or changed44:
            score=scores30[strategy][q];eligible=np.isfinite(score);ordered=np.flatnonzero(eligible);ranked44=ordered[np.argsort(-score[ordered],kind='stable')]
            if len(ranked44)<25:
                reasons.add('fewer_than25_past_data_eligible_stocks');break
            healthy44=[j for j in ranked44 if healthy_flags44[q,j]] if local44 else list(ranked44)
            if regular44:selected=np.array(healthy44[:25],dtype=int)
            else:
                retained44=[j for j in np.flatnonzero(shares>0) if j not in failed44];new44=[j for j in healthy44 if shares[j]==0][:len(failed44)];selected=np.array(retained44+new44,dtype=int)
            if len(selected)!=25:
                reasons.add('insufficient_healthy25');break
            if np.any(~np.isfinite(cl[q,selected])|(cl[q,selected]<=0)):
                reasons.add('missing_prior_close');break
            budget44=.8 if wanted44 else invested
            if regular44:
                weights=np.full(25,budget44/25);target=np.zeros_like(shares);target[selected]=(np.floor(weights*prior_nav/cl[q,selected]/1000)*1000).astype(np.int64)
            else:
                target=shares.copy();new44=[j for j in selected if shares[j]==0]
                for old44,newid44 in zip(failed44,new44):
                    target[old44]=0;target[newid44]=int(np.floor(shares[old44]*mark[q,old44]/cl[q,newid44]/1000))*1000
                if changed44:
                    total44=float(np.dot(target,np.nan_to_num(mark[q])))
                    if total44>0:target[selected]=(np.floor(target[selected]*(budget44*prior_nav/total44)/1000)*1000).astype(np.int64)
            cash,shares,traded,cost,missing=settle(cash,shares,target,arrays['execution'][t],arrays['available'][t]);fees+=cost;turnover_sum+=traded
            if missing:reasons.add('invalid_execution_price')
            traces.append({'date':str(days[t]),'selected':[all_external_ids_v13[j] for j in selected]})
            events44.append({'date':str(days[t]),'signal_date':str(days[q]),'regular':regular44,'market_reduce':bool(changed44 and wanted44),'market_reentry':bool(changed44 and not wanted44),'failed':[all_external_ids_v13[j] for j in failed44],'traded':traded/INITIAL,'cost':cost/INITIAL})
        riskoff44=wanted44;riskdays44+=int(riskoff44)
'''
left44=v44_code.index(anchor44);right44=v44_code.index(end44);v44_code=v44_code[:left44]+v44_decision+v44_code[right44:]
v44_code=v44_code.replace('        weights = shares * np.nan_to_num(mark[t]) / nav','        exposures44.append(float(1-cash/nav))\n        weights = shares * np.nan_to_num(mark[t]) / nav')
v44_code=v44_code.replace("return {'paid_policy':", "return {'mode':mode,'events':events44,'mean_exposure':float(np.mean(exposures44)) if exposures44 else None,'unknown_market_days':unknown44,'riskoff_days':riskdays44,'paid_policy':")
v44_ns=dict(v34_ns);v44_ns.update(market44=v44_market,weak44=v44weak,healthy_flags44=v44healthy);exec(v44_code,v44_ns);v44_sim=v44_ns['simulate_context44']
v44_controls=[]
for family in ['MOM','FLOW']:
    for ix in [0,400,900,1700,2235]:
        s=int(v30_starts[ix]);old=v34_results[family+'_E0'][ix];now=v44_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved',family,.9,5,'none')
        assert old['reasons']==now['reasons'] and old['violations']==now['violations'] and old['return']==now['return']
        if now['return'] is not None:assert all(np.isclose(old[x],now[x],rtol=0,atol=1e-12) for x in ['mdd','cost','turnover'])
        v44_controls.append({'family':family,'index':ix,'matched':True})
v44_engine_hash=hashlib.sha256(v44_code.encode()).hexdigest();v44_results={k:v34_results[k] for k in ['MOM_E0','FLOW_E0']};v44_fixed_targets={};v44_newruns=0
sp44=v34_root/'A策略_無人值守研究狀態.json';state44=json.loads(sp44.read_text(encoding='utf-8'));state44['context_risk_v44']={'status':'definition_frozen_controls_passed','running':False,'definition_hash':v44_definition_hash,'engine_hash':v44_engine_hash,'controls':v44_controls,'planned_new_methods':14,'planned_new_window_runs':31304};sp44.write_text(json.dumps(state44,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V44_DEFINED',{'matched_controls':len(v44_controls),'known_market_days':int(v44_market['known'].sum()),'weak_days':int(v44_market['weak'].sum()),'recovery_days':int(v44_market['recovery'].sum()),'new_methods':14})


In [ ]:
# Execute finite14 new specs; per-spec cache validates data/definition/engine and avoids repeats.
def run_context44(key,family,mode,invested=.9,calibration=None):
    global v44_newruns
    spec={'key':key,'family':family,'mode':mode,'invested':invested,'definition_hash':v44_definition_hash,'engine_hash':v44_engine_hash,'calibration':calibration};fp=v34_root/('市場情境_'+key+'_v44_2026-10-16.json.gz')
    state=json.loads(sp44.read_text(encoding='utf-8'));state['context_risk_v44'].update(status='running',running=True,current_method=key,completed_methods=[k for k in v44_results if k not in ['MOM_E0','FLOW_E0']]);sp44.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
    if fp.exists():
        with gzip.open(fp,'rt',encoding='utf-8') as f:obj=json.load(f)
        assert obj['spec']==spec;v44_results[key]=obj['results'];print('V44_REUSED',key,flush=True);return
    rows=[]
    for ix,s0 in enumerate(v30_starts):
        s=int(s0);r=v44_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved',family,invested,5,mode);r['start_index']=s;rows.append(r)
        if (ix+1)%700==0:print('V44_PROGRESS',key,ix+1,flush=True)
    with gzip.open(fp,'wt',encoding='utf-8') as f:json.dump({'spec':spec,'results':rows},f,ensure_ascii=False,default=serial30,allow_nan=False)
    v44_results[key]=rows;v44_newruns+=len(rows);print('V44_DONE',key,{'valid':sum(r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows),'new_window_runs':v44_newruns},flush=True)
for fam in ['MOM','FLOW']:
    for mode in ['LOCAL','MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']:run_context44(fam+'_'+mode,fam,mode)
    for mode in ['MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']:
        key=fam+'_'+mode;train=[r for r,keep in zip(v44_results[key],v30_training) if keep and r['return'] is not None and not r['invalid'] and r['violations']==0];assert len(train)>=100
        target=float(np.mean([r['mean_exposure'] for r in train]));assert .76<target<.95
        calibration={'source':key,'valid_training_n':len(train),'only2016to2018_mean_realized_exposure':target};v44_fixed_targets[key]=calibration;run_context44(key+'_FIXED',fam,'none',target,calibration)
state=json.loads(sp44.read_text(encoding='utf-8'));state['context_risk_v44'].update(status='computed_pending_statistics',running=False,current_method=None,completed_methods=[k for k in v44_results if k not in ['MOM_E0','FLOW_E0']],new_window_runs=v44_newruns,fixed_targets=v44_fixed_targets);sp44.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V44_ALL_COMPUTED',{'methods':len(v44_results),'new_runs':v44_newruns,'fixed_targets':v44_fixed_targets})


In [ ]:
# Frozen18 contrasts: paired returns, risk, fees, calibration exposure controls, block multiplicity.
v44_pairs=[]
for fam in ['MOM','FLOW']:
    for mode in ['LOCAL','MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']:v44_pairs.append((fam+'_'+mode,fam+'_E0'))
    v44_pairs += [(fam+'_MARKET_RECOVER',fam+'_MARKET_STAY80'),(fam+'_COMBINED_RECOVER',fam+'_MARKET_RECOVER')]
    for mode in ['MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']:v44_pairs.append((fam+'_'+mode,fam+'_'+mode+'_FIXED'))
assert len(v44_pairs)==18
v44_stats,v44_valid,v44_ret,v44_common=analyze37(v44_results,v44_pairs,18)
def metrics_context44(key,mask):
    ix=np.flatnonzero(mask);rr=v44_ret[key][mask];rows=v44_results[key]
    if not len(ix):return {'n':0}
    obj={'n':len(ix),'win':float(np.mean(rr>0)),'ev':float(np.mean(rr)),'median':float(np.median(rr)),'p10':float(np.quantile(rr,.1)),'worst_return':float(np.min(rr)),'mean_mdd':float(np.mean([rows[i]['mdd'] for i in ix])),'worst_mdd':float(min(rows[i]['mdd'] for i in ix)),'fee':float(np.mean([rows[i]['cost'] for i in ix])),'turnover':float(np.mean([rows[i]['turnover'] for i in ix]))}
    if 'mean_exposure' in rows[ix[0]]:obj.update(mean_exposure=float(np.mean([rows[i]['mean_exposure'] for i in ix])),mean_reduce_events=float(np.mean([sum(e['market_reduce'] for e in rows[i]['events']) for i in ix])),mean_reentry_events=float(np.mean([sum(e['market_reentry'] for e in rows[i]['events']) for i in ix])),mean_replaced_slots=float(np.mean([sum(len(e['failed']) for e in rows[i]['events']) for i in ix])),signal_unknown_days=int(sum(rows[i]['unknown_market_days'] for i in ix)),windows_with_reentry=int(sum(any(e['market_reentry'] for e in rows[i]['events']) for i in ix)))
    return obj
v44_stats['metrics_common_all']={k:metrics_context44(k,v44_common) for k in v44_results};v44_stats['metrics_common_later']={k:metrics_context44(k,v44_common&v30_later) for k in v44_results}
for name,obj in v44_stats['contrasts'].items():
    a,b=obj['candidate'],obj['base'];m=v44_valid[a]&v44_valid[b]&v30_later;obj['candidate_stats']=metrics_context44(a,m);obj['base_stats']=metrics_context44(b,m)
    obj['fee_difference']=obj['candidate_stats']['fee']-obj['base_stats']['fee'];obj['accounting_before_fee_difference']=obj['mean_difference']+obj['fee_difference'];obj['beat_control_fraction']=float(np.mean((v44_ret[a]-v44_ret[b])[m]>0));obj['worst_half_early_vs_late']={}
    for label,lo,hi in [('2019to2021',2019,2021),('2022to2025',2022,2025)]:
        mm=m&np.array([lo<=v30_days[int(s)].year<=hi for s in v30_starts]);obj['worst_half_early_vs_late'][label]={'n':int(mm.sum()),'difference':float(np.mean((v44_ret[a]-v44_ret[b])[mm])) if mm.any() else None}
v44_stats['coverage_detail']={k:{'valid':int(v44_valid[k].sum()),'later_valid':int((v44_valid[k]&v30_later).sum()),'invalid_reasons':dict(collections.Counter(reason for r in rows for reason in r['reasons'])),'violations_windows':sum(r['violations']>0 for r in rows),'invalid_in_legacy1209':int(np.sum(v40_mask&~v44_valid[k]))} for k,rows in v44_results.items()}
v44_stats['full_improvement_vs_own_baseline']=[k for k in v44_results if k.endswith(('LOCAL','STAY80','MARKET_RECOVER','COMBINED_RECOVER')) and v44_stats['contrasts'][k+'_minus_'+k.split('_')[0]+'_E0']['full_improvement']]
v44_stats['timing_increment_supported']=[k for k in v44_fixed_targets if v44_stats['contrasts'][k+'_minus_'+k+'_FIXED']['mean_improvement']]
v44_stats['source_hashes']=v34_plan['common']['source_hashes'];v44_stats['definition_hash']=v44_definition_hash;v44_stats['engine_hash']=v44_engine_hash;v44_stats['fixed_controls']=v44_fixed_targets;v44_stats['new_window_runs']=v44_newruns
# Same-sample pre-entry US VIX stratification of incremental returns; descriptive, no new policy thresholds.
v44_stats['us_vix_increment_descriptive']={}
for a,b in v44_pairs[:]:
    if b not in ['MOM_E0','FLOW_E0']:continue
    ss={}
    for label in ['<20','>=20']:
        ids=[r['index'] for r in v42_rows if (r['vix']<20)==(label=='<20') and v44_valid[a][r['index']] and v44_valid[b][r['index']]];ss[label]={'n':len(ids),'difference':float(np.mean((v44_ret[a]-v44_ret[b])[ids])) if ids else None}
    v44_stats['us_vix_increment_descriptive'][a]=ss
# Competition-season starts, all years disclosed, no significance from the tiny annual sample.
v44_stats['competition_season']={}
for yr in range(2016,2026):
    last=datetime.date(yr,10,31);target=last-datetime.timedelta(days=(last.weekday()-1)%7);i=next((i for i,s in enumerate(v30_starts) if v30_days[int(s)]>=target),None)
    v44_stats['competition_season'][str(yr)]={'start':str(v30_days[int(v30_starts[i])]),'methods':{k:{'valid':bool(v44_valid[k][i]),'net':float(v44_ret[k][i]) if v44_valid[k][i] else None,'reasons':v44_results[k][i]['reasons'],'violations':v44_results[k][i]['violations']} for k in v44_results}} if i is not None else {'missing':True}
(v34_root/'市場情境_退出換股再進場統計_v44_2026-10-16.json').write_text(json.dumps(v44_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp44.read_text(encoding='utf-8'));state['context_risk_v44'].update(status='statistics_complete_pending_report',running=False,contrasts=18,common_all=v44_stats['joint_all'],common_later=v44_stats['joint_later'],full_improvements=v44_stats['full_improvement_vs_own_baseline']);sp44.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V44_STATISTICS',{'common':[v44_stats['joint_all'],v44_stats['joint_later']],'full_improvements':v44_stats['full_improvement_vs_own_baseline'],'timing_increments':v44_stats['timing_increment_supported'],'paired':{n:{f:r[f] for f in ['n','mean_difference','CI24','CI48','risk_not_worse','year_gate','full_improvement','fee_difference']} for n,r in v44_stats['contrasts'].items()}})


In [ ]:
# Publish finite comparison including null/negative findings, every common-window point.
v44_names={'E0':'原五日調整90%','LOCAL':'市場穩定時局部換股','MARKET_STAY80':'廣泛轉弱降80%並維持','MARKET_RECOVER':'降80%／確認回升恢復90%','COMBINED_RECOVER':'局部換股＋減碼與恢復'}
def label44(k):
    fam,mode=k.split('_',1);prefix='動能' if fam=='MOM' else '法人'
    if mode.endswith('_FIXED'):
        src=k[:-6];return prefix+'固定'+f"{v44_fixed_targets[src]['only2016to2018_mean_realized_exposure']*100:.2f}%"+'（對照'+v44_names[mode[:-6]]+'）'
    return prefix+'／'+v44_names[mode]
v44_lines=['# 個股轉弱、市場下跌與回升：退出／換股／再進場有幫助嗎？','2026-10-16｜沿用2016～2025長歷史。14組新配置×2236起點＝31,304次模擬，兩個90%五日基準直接重用；不是31,304個獨立事件。','## 結論先說','**這輪沒有方法通過事先訂好的全面改善門檻。**市場穩定時急著換掉轉弱個股，費後期望下降；廣泛轉弱後減碼有部分回撤改善，但更像報酬與風險的取捨，尚未證明選時優勢；回升後恢復投入的平均好處很小、校正區間跨零。','因此保留原動能／法人基準，這批不升格為已證實的新策略。這否定的是本輪事先固定規則的改善主張，不能推成「所有停損都無效」。','## 為什麼這樣設計？','前一輪看到虧損經常與同期廣泛下跌共現，但那是事後24日結果。本輪只用**每次決策前已知的過去5／20日資料**觸發行動，不拿接下來24日的下跌比例決策。','台美VIX未呈現高低與後續虧損的簡單單調關係，台灣十年資料又缺957窗口，因此本輪沒有用VIX≥20／30硬切進出，也沒有拿事後修訂景氣燈號回填。美國VIX增量分層另存JSON，只作描述。','## 四種處理方式：每天檢查，有事件才額外交易','|方式|觸發與做法|目的|','|---|---|---|','|局部換股|市場穩定、持股連續兩收盤低於5日均線，次日換成原策略排序中的健康候補；五日檢查時也避免選入已觸發轉弱者|看個股變弱時替換是否有益|','|減碼後維持|市場廣泛轉弱，90%降至80%，該24日窗口內不再恢復|避免減碼後又立即追回|','|減碼後恢復|同樣降80%，確認回升後恢復90%；再轉弱可再降|量化退出後是否值得重新進場|','|組合方式|局部換股＋減碼後恢復|檢查兩種處理是否互補|','### 所有門檻事前固定','- 市場穩定：過去20日上漲家數比例≥50%，且可判定股票的過去5日平均報酬≥0。','- 廣泛轉弱：過去20日上漲家數比例<40%，且過去5日平均報酬<0。','- 確認回升：過去5日上漲家數比例≥60%、平均報酬>0，連續兩個已知交易日成立。','- 市場範圍是目前150檔在歷史可用的子集，不是全台股；至少當日有價格股票80%且至少50檔資料完整，才判定訊號。','- 均線以含已知現金股息的調整價格層級計算；不把除息機械跌價直接當停損。','- 第一天都先以原選股90%建立，從第二天才觸發風控；正常每五日重選及調整。額外換股只動失敗部位，市場減碼／恢復只縮放當時持股。','- 用前收盤產生委託，次日均價模擬成交，整千股，含雙向0.1425%費與賣出0.3%稅。**不是盤中觸價就能即刻成交的停損。**','- 保留25檔是本次隔離風控變因的實驗控制，不代表官方要求一定25檔。沿用現金<25%、持股20～30、個股權重等檢查；未宣稱完整官方Active Share及實際作廢／成交規則都已驗證。','## 同一批日期一起看：全16方法共同窗口','本輪全期共同'+str(v44_stats['joint_all'])+'窗口，後段2019～2025共同'+str(v44_stats['joint_later'])+'。不同報告共同方法不同，原策略圖形會因留下的日期改變；本頁所有方法彼此同日，沒有改寫原策略報酬。','勝率＝費後24日報酬>0的比例；期望值＝費後24日平均報酬，不年化。最差回撤是窗口內淨值路徑的最大跌幅。']
for period,title in [('metrics_common_all','全期2016～2025（2016為部分年度）'),('metrics_common_later','後段2019～2025')]:
    v44_lines+=['### '+title,'|方法|勝率|24日期望|P10|最差回撤|平均費稅|','|---|---:|---:|---:|---:|---:|']
    for fam in ['MOM','FLOW']:
        for mode in ['E0','LOCAL','MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']:
            k=fam+'_'+mode;m=v44_stats[period][k];v44_lines.append('|'+ '|'.join([label44(k),p30(m['win']),p30(m['ev']),p30(m['p10']),p30(m['worst_mdd']),p30(m['fee'])])+'|')
v44_lines+=['## 局部換股：多付的成本，是不是主要問題？','各列用自己的後段配對日期，不能和上表976共同日期的平均差混用。下列費用差／報酬差單位為百分點。','|比較|配對n|費後期望差|多付費稅|會計加回費用後差|48日區塊／18比較校正區間|','|---|---:|---:|---:|---:|---|']
for fam in ['MOM','FLOW']:
    r=v44_stats['contrasts'][fam+'_LOCAL_minus_'+fam+'_E0'];v44_lines.append('|'+ '|'.join([('動能' if fam=='MOM' else '法人')+'局部換股−原策略',str(r['n']),f"{r['mean_difference']*100:+.3f}pp",f"{r['fee_difference']*100:+.3f}pp",f"{r['accounting_before_fee_difference']*100:+.3f}pp",c30(r['CI48'])])+'|')
v44_lines+=['法人局部換股的會計加回費用後差略正，但被額外交易成本抵銷；動能局部換股在加回費用後仍較差。因此不能一概說「完全只是成本」，也不能說換出的股票一定選錯。加回費用是同一路徑的會計分解，不是重新模擬零成本反事實。','後段976共同日期，每窗口平均觸發的轉弱持股次數：動能34.26、法人36.63，同一股票可多次觸發。原策略費稅約0.80%／0.70%，局部換股增至1.39%／1.41%；頻繁替換與再買入是這套門檻的主要負擔。','## 減碼：是辨識風險，還是只是少放資金？','另有6個固定投入對照，其比例只依2016～2018有效窗口的平均實際投入估計，沒有用後段收益挑比例。減碼維持組的固定比例約85.5%，恢復組約87.5%。固定目標比例與實際收盤平均曝險不會完全相同，下表一併列出差距。','|動態−固定對照|後段配對n|動態／固定平均投入|費後期望差|48日校正區間|有證實選時收益？|','|---|---:|---|---:|---|---|']
for fam in ['MOM','FLOW']:
    for mode in ['MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']:
        k=fam+'_'+mode;r=v44_stats['contrasts'][k+'_minus_'+k+'_FIXED'];a=r['candidate_stats'];b=r['base_stats'];v44_lines.append('|'+ '|'.join([label44(k),str(r['n']),p30(a['mean_exposure'])+'／'+p30(b['mean_exposure']),f"{r['mean_difference']*100:+.3f}pp",c30(r['CI48']),'是' if r['mean_improvement'] else '否'])+'|')
v44_lines+=['沒有一組動態方式相對其固定投入對照，在24與48日校正區間都得到正下界。部分最差回撤縮小值得作風險取捨參考，但目前不能宣稱市場選時已有效。','## 退出後重新進場，有沒有追回反彈？','|恢復90%−維持80%|後段配對n|24日期望差|48日校正區間|全面改善？|','|---|---:|---:|---|---|']
for fam in ['MOM','FLOW']:
    r=v44_stats['contrasts'][fam+'_MARKET_RECOVER_minus_'+fam+'_MARKET_STAY80'];v44_lines.append('|'+ '|'.join(['動能' if fam=='MOM' else '法人',str(r['n']),f"{r['mean_difference']*100:+.3f}pp",c30(r['CI48']),str(r['full_improvement'])])+'|')
v44_lines+=['兩者平均增量都只有約+0.03個百分點／24日，校正區間跨零。後段共同窗口455次至少發生一次重新進場，但這不能當成455次獨立反彈，也沒有證明已能可靠買回低點。','## 18項事先比較全部揭露','用2019～2025後段配對，同時保留2016～2018校準資料量、逐年與兩段時間結果。原始起點位置做24／48日區塊各5000次，18項Bonferroni校正，不把重疊窗口當獨立觀測。','|候選−對照|配對n|平均差pp|48日校正區間pp|平均改善|風險不劣|跨年通過|全面改善|','|---|---:|---:|---|---|---|---|---|']
for name,r in v44_stats['contrasts'].items():v44_lines.append('|'+ '|'.join([label44(r['candidate'])+'−'+label44(r['base']),str(r['n']),f"{r['mean_difference']*100:+.3f}",c30(r['CI48']),str(r['mean_improvement']),str(r['risk_not_worse']),str(r['year_gate']),str(r['full_improvement'])])+'|')
v44_lines+=['全面改善要求24及48日校正收益下界均>0，P10與最差回撤均不劣，覆蓋原對照後段日期至少80%、校準配對≥100、至少5個年份各≥20配對且其中至少三分之二增量正。這是回溯選擇準則，不是未來收益保證；5000次對極尾端校正區間有蒙地卡羅誤差。','## 年度與比賽同期也不能藏起來','JSON逐項保留每年配對增量、2019～2021及2022～2025分段，以及每年10月最後星期二開始的24日結果。同期年度最多10點，缺價、公司行動或限制未通過都明列，不能挑掉差年份後聲稱穩定。','|起始年|實際比賽同期起點|動能原策略|動能減碼後恢復|法人原策略|法人減碼後恢復|','|---|---|---:|---:|---:|---:|']
for yr,rec in v44_stats['competition_season'].items():
    row=[yr,rec.get('start','無資料')]
    for key in ['MOM_E0','MOM_MARKET_RECOVER','FLOW_E0','FLOW_MARKET_RECOVER']:
        val=rec.get('methods',{}).get(key,{});row.append(p30(val['net']) if val.get('valid') else '不可判定')
    v44_lines.append('|'+ '|'.join(row)+'|')
v44_lines+=['## 有效樣本與限制','|方法|全期有效／2236|後段有效|有模擬限制違反的窗口|','|---|---:|---:|---:|']
for k,meta in v44_stats['coverage_detail'].items():v44_lines.append('|'+ '|'.join([label44(k),str(meta['valid'])+'/2236',str(meta['later_valid']),str(meta['violations_windows'])])+'|')
v44_lines+=['只有可配對有效日期才進比較；無法判定不當作0報酬，也不是自動賺賠。逐方法JSON保留每筆失敗與限制。股票池仍有當前名單存活偏差；公司行動及正式基準報酬資料仍有限，本研究不能宣稱官方可執行alpha。後段資料已多次查看，本輪不是全新未見樣本外。','## 接下來採用什麼結論？','- **維持已驗證基準作比較起點**，沒有新方法因最高平均值就自動勝出。','- **本輪局部換股不採用**：費後平均差校正區間整段負，且未帶來全面風險改善。','- **市場減碼保留為風險取捨候選**：有些尾端損失縮小，平均收益也下降；尚未證明優於相近投入的固定配置。','- **回升恢復尚未證實增益**：方向略正但效果小，不能稱已找到可靠觸底／再進場方法。','## 保存與重現','既有kernel內執行，Notebook362起保存定義、10個原基準對照、有限批次、統計與圖表。來源快照／引擎／定義hash寫入結果，已有相同結果直接引用；未重啟kernel、未提交交易。','[18項統計與年份](市場情境_退出換股再進場統計_v44_2026-10-16.json) · [事先定義](市場情境_退出換股再進場定義_v44_2026-10-16.json) · [回總結](策略研究_總結報告.html)。']
v44_report='\n'.join(v44_lines);v44_rp=v34_root/'市場情境_退出換股再進場比較_v44_2026-10-16.md';v44_rp.write_text(v44_report,encoding='utf-8')
# Build four views with a shared return axis; no random hiding or sampling of dots.
v44_fig=go.Figure();v44_views={};v44_labelsplot={};v44_extrema=[]
for fam in ['MOM','FLOW']:
    for later in [False,True]:
        view=fam+('_later' if later else '_all');mask=v44_common&(v30_later if later else np.ones(len(v30_starts),bool));ix=np.flatnonzero(mask);indices=[];v44_labelsplot[view]=('動能' if fam=='MOM' else '法人')+('／後段2019～2025' if later else '／全期2016～2025')+'／共同'+str(len(ix))+'窗口'
        for mode in ['E0','LOCAL','MARKET_STAY80','MARKET_RECOVER','COMBINED_RECOVER']:
            key=fam+'_'+mode;ys=v44_ret[key][mask]*100;v44_extrema.extend([float(ys.min()),float(ys.max())]);label=v44_names[mode];indices.append(len(v44_fig.data));color='#167368' if mode=='E0' else '#567bb5'
            v44_fig.add_trace(go.Box(y=ys,name=label,boxpoints='all',jitter=.45,pointpos=0,line={'color':color},fillcolor='rgba(255,255,255,0)',marker={'size':3,'opacity':.35,'color':color},customdata=[[v44_results[key][i]['start'],v44_results[key][i]['end'],v44_results[key][i]['cost']*100] for i in ix],hovertemplate='%{customdata[0]}→%{customdata[1]}<br>費後24日%{y:.2f}%<br>費稅%{customdata[2]:.2f}%<extra></extra>',visible=view=='MOM_later'))
            indices.append(len(v44_fig.data));v44_fig.add_trace(go.Scatter(x=[label],y=[float(np.median(ys))],mode='markers',marker={'color':'#df4385','symbol':'line-ew','size':24,'line':{'width':3}},showlegend=False,hovertemplate='中位數%{y:.2f}%<extra></extra>',visible=view=='MOM_later'))
        v44_views[view]=indices
v44_fig.add_hline(y=0,line_color='#222',line_width=2);v44_fig.update_layout(template='plotly_white',height=700,showlegend=False,title=v44_labelsplot['MOM_later'],yaxis={'title':'費後24日報酬 %','range':[min(v44_extrema)-3,max(v44_extrema)+3]},xaxis={'tickangle':-18},margin={'t':115,'b':140},updatemenus=[{'type':'dropdown','x':0,'y':1.18,'active':1,'buttons':[{'label':v44_labelsplot[v],'method':'update','args':[{'visible':[i in ids for i in range(len(v44_fig.data))]},{'title':v44_labelsplot[v]}]} for v,ids in v44_views.items()]}])
v44_hp=v34_root/'市場情境_退出換股再進場比較_v44_2026-10-16.html';v44_rendered=MarkdownIt('commonmark',{'html':False}).enable('table').render(v44_report).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')
v44_hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>市場情境與換股減碼再進場</title><style>body{font:16px/1.8 system-ui;max-width:1250px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:9px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}</style><a href="策略研究_總結報告.html">回總結</a>'+v44_fig.to_html(full_html=False,include_plotlyjs=True,config={'responsive':True,'displaylogo':False})+v44_rendered+'</html>',encoding='utf-8')
v44_snippet='<section id="context-exit-reentry"><h2>市場情境：換股、減碼與再進場實驗</h2><p>2016～2025，14新配置、31,304次窗口模擬、18項校正配對。沒有新方法通過全面改善。局部換股增加費用並降低期望；市場減碼可縮小部分回撤，尚未證明優於相近投入的固定配置；回升恢復的增量約+0.03pp且區間跨零。</p><p>本輪全方法共同1282／後段976窗口，與其他圖的篩選日期不同；主張只限固定規則，不等於所有停損無效。</p><a href="'+v44_hp.name+'">查看同日期勝率、期望、全部點、回撤與18項對照 →</a></section>'
sumhp44=v34_root/'策略研究_總結報告.html';page44=sumhp44.read_text(encoding='utf-8');assert 'context-exit-reentry' not in page44;page44=page44.replace('<section id="taiwan-vix-partial">',v44_snippet+'<section id="taiwan-vix-partial">');assert v44_snippet in page44;sumhp44.write_text(page44,encoding='utf-8')
summd44=v34_root/'策略研究_總結報告.md';summd44.write_text(summd44.read_text(encoding='utf-8')+'\n\n## 市場情境：換股、減碼、再進場（v44）\n\n14新配置×2236起點＝31,304次模擬；18項24／48日區塊校正，無全面改善、無相對相近固定投入對照的正收益下界。局部換股費後期望差顯著為負；市場減碼保留為風險收益取捨；恢復投入增量約+0.03pp且區間跨零。全方法共同1282／後段976窗口。[完整比較與箱形圖]('+v44_hp.name+')。\n',encoding='utf-8')
oldhp44=v34_root/'動能法人_同期整體下跌分析_v41_2026-10-16.html';page44=oldhp44.read_text(encoding='utf-8');oldhp44.write_text(page44.replace('</html>',v44_snippet+'</html>'),encoding='utf-8')
v44_validation={'baseline_matches':len(v44_controls),'methods':len(v44_results),'new_methods':14,'new_window_runs':v44_newruns,'contrasts':len(v44_pairs),'same_dates_all':int(v44_common.sum()),'same_dates_later':int((v44_common&v30_later).sum()),'events_strictly_lagged':all(e['signal_date']<e['date'] for k,rows in v44_results.items() if k not in ['MOM_E0','FLOW_E0'] for r in rows for e in r['events']),'boxpoints_per_view':{v:sum(len(v44_fig.data[i].y) for i in ids if v44_fig.data[i].type=='box') for v,ids in v44_views.items()},'full_improvements':v44_stats['full_improvement_vs_own_baseline'],'timing_increment_supported':v44_stats['timing_increment_supported'],'browser_visual_qa':'not performed;programmatic validation only'}
assert v44_validation['events_strictly_lagged'] and v44_validation['contrasts']==18 and v44_newruns==31304
assert 'unified-longhistory-boxplot' in sumhp44.read_text(encoding='utf-8')
assert all(n==(976*5 if 'later' in v else 1282*5) for v,n in v44_validation['boxpoints_per_view'].items())
(v34_root/'市場情境_退出換股再進場驗證_v44_2026-10-16.json').write_text(json.dumps(v44_validation,ensure_ascii=False,indent=2),encoding='utf-8')
state=json.loads(sp44.read_text(encoding='utf-8'));state['context_risk_v44'].update(status='completed_no_proven_improvement',running=False,html=v44_hp.name,report=v44_rp.name,validation=v44_validation,current_method=None);sp44.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V44_COMPLETE',{'html':v44_hp.name,'validation':v44_validation})


In [ ]:
# Fixed SMA crossover study: state filters, cross exits, pure MA ranking, no day-grid search.
print('V45_RETAINED',{'python':sys.executable,'pid':os.getpid(),'data_shape':v30_arrays['close'].shape,'windows':len(v30_starts),'prior_context_status':json.loads(sp44.read_text(encoding='utf-8'))['context_risk_v44']['status']})
v45_pairs_ma={'3_5':(3,5),'5_20':(5,20),'20_60':(20,60),'50_200':(50,200)}
v45_definition={'version':'v45_sma_cross_fixed_1','date':'2026-10-17','input_hashes':v34_plan['common']['source_hashes'],'pairs':v45_pairs_ma,'mean_type':'SMA only; known-cash-dividend adjusted wealth-price levels; no EMA parameter search','golden_event':'known today gap>0 andknown previousgap<=0','death_event':'known todaygap<0 andknownpreviousgap>=0','bullish_state':'shortSMA>longSMA; positive state need notcross onentryday','INITIAL':'MOM/FLOW rank, firstday filterpositiveMAstate, then original unfiltered5day reselect; no extra death exit','ACTIVE':'MOM/FLOW rank, positiveMAstate filtereveryregular5dayreview; priorclose deathcross onheldstock triggers nextday oneforonebullishreplacement; otherholdingsunchanged; newlybullishstockseligible asrankedcandidates, not automatic buy allgoldenevents','PURE':'rankpositiveMAstate by shortSMA/longSMA−1 descending, tie symbol; same ACTIVE crossover handling; no MOM/FLOW ranking input, common pastdataquality screen retained','shared':'25stocks,equal90%target,24sessions,5dayregular,1000shares,previousclose sizing,nextVWAP execution,existingsettle costs/actions/caps','cash':'no new cashoverlay; compareeachwith own90baseline','shortage':'under25eligible positives atrequiredtrade: invalid, no negative-state forcedfill; coverage and dropped-baselineoutcomes disclosed','warmup':'use only existing2016–2025 prices;50/200requires200knownpricelevels, earliest2016starts unavailable;do notbackfill fabricated2015','count':{'new_specs':20,'possible_starts':2236,'new_window_runs':44720,'primary_contrasts':32},'contrasts':'16 INITIAL/ACTIVE vsownbaseline;8ACTIVEvsINITIAL;4PUREvsMOM;4PUREvsFLOW','statistics':'original-position24/48dayblocks10000each;Bonferroni32; pairedfeeafter;2016–2018calibration2019–2025later,notunseen','gate':'bothcorrectedCI lower>0,P10andworstMDDnotworse,coverage>=80%,training>=100,>=5years n>=20and>=2/3positiveyears;no highestbacktestwinner','dedup':'fixedspec+inputhash+enginehash;reuseresults','not_tested':'EMA,tunedlookbacks,allcashentryexit,freshgoldcrossonly25stocks,200daydatafrom2015','sources':['https://www.fidelity.com/learning-center/trading-investing/technical-analysis/what-is-technical-analysis','https://www.schwab.com/learn/story/understanding-simple-moving-average-crossovers'],'limits':'current150historicalsubset/survivorship,unknownactions excluded,officialActiveShareandexecutionvoidsnotfullycertified;nottradeinstructions'}
(v34_root/'均線交叉_策略定義_v45_2026-10-17.json').write_text(json.dumps(v45_definition,ensure_ascii=False,indent=2),encoding='utf-8');v45_hash=hashlib.sha256(json.dumps(v45_definition,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
v45_ma={};v45_scores=dict(v34_scores)
for pair,(fast,slow) in v45_pairs_ma.items():
    gap=np.full_like(v30_arrays['close'],np.nan)
    for q in range(slow-1,len(v30_days)):
        rr=v30_arrays['returns'][q-slow+2:q+1];ok=np.all(np.isfinite(rr)&(rr>-1)&~v30_arrays['bad'][q-slow+2:q+1],axis=0)
        levels=np.vstack([np.ones(rr.shape[1]),np.cumprod(1+rr,axis=0)]);gap[q,ok]=np.mean(levels[-fast:,ok],axis=0)/np.mean(levels[:,ok],axis=0)-1
    known=np.isfinite(gap);bull=known&(gap>0);gold=np.zeros_like(bull);death=np.zeros_like(bull);gold[1:]=known[1:]&known[:-1]&(gap[1:]>0)&(gap[:-1]<=0);death[1:]=known[1:]&known[:-1]&(gap[1:]<0)&(gap[:-1]>=0)
    v45_ma[pair]={'gap':gap,'known':known,'bull':bull,'gold':gold,'death':death};pure=np.full_like(gap,np.nan)
    for q in range(slow-1,len(v30_days)):pure[q]=percentile(gap[q],bull[q]&np.isfinite(v30_scores['MOM'][q]))
    v45_scores['MA_'+pair]=pure
v45_code=v34_engine.replace('def simulate_unified34(','def simulate_ma45(').replace('rebalance=5):',"rebalance=5, ma_pair='5_20', ma_mode='none'):")
v45_code=v45_code.replace('    for offset, t in enumerate(range(start, start + 24)):',"    events45=[]\n    for offset, t in enumerate(range(start, start + 24)):")
v45_decision=r'''        q=t-1
        regular45=offset==0 or (rebalance>0 and offset%rebalance==0)
        failed45=np.flatnonzero((shares>0)&ma45[ma_pair]['death'][q]) if ma_mode=='ACTIVE' and offset>0 else np.array([],dtype=int)
        if regular45 or len(failed45):
            score=scores30[strategy][q];eligible=np.isfinite(score);ordered=np.flatnonzero(eligible);ranked45=ordered[np.argsort(-score[ordered],kind='stable')]
            filtering45=ma_mode=='ACTIVE' or (ma_mode=='INITIAL' and offset==0)
            candidates45=[j for j in ranked45 if not filtering45 or ma45[ma_pair]['bull'][q,j]]
            if regular45:selected=np.array(candidates45[:25],dtype=int)
            else:
                retained45=[j for j in np.flatnonzero(shares>0) if j not in failed45];new45=[j for j in candidates45 if shares[j]==0][:len(failed45)];selected=np.array(retained45+new45,dtype=int)
            if len(selected)<25:
                reasons.add('fewer_than25_bullish_MA_or_data_eligible' if filtering45 else 'fewer_than25_past_data_eligible_stocks');break
            if np.any(~np.isfinite(cl[q,selected])|(cl[q,selected]<=0)):
                reasons.add('missing_prior_close');break
            if regular45:
                weights=np.full(25,invested/25);target=np.zeros_like(shares);target[selected]=(np.floor(weights*prior_nav/cl[q,selected]/1000)*1000).astype(np.int64)
            else:
                target=shares.copy();new45=[j for j in selected if shares[j]==0]
                for old45,newid45 in zip(failed45,new45):target[old45]=0;target[newid45]=int(np.floor(shares[old45]*mark[q,old45]/cl[q,newid45]/1000))*1000
            cash,shares,traded,cost,missing=settle(cash,shares,target,arrays['execution'][t],arrays['available'][t]);fees+=cost;turnover_sum+=traded
            if missing:reasons.add('invalid_execution_price')
            traces.append({'date':str(days[t]),'selected':[all_external_ids_v13[j] for j in selected]})
            events45.append({'date':str(days[t]),'signal_date':str(days[q]),'regular':regular45,'death_exits':[all_external_ids_v13[j] for j in failed45],'selected_golden_today':sum(bool(ma45[ma_pair]['gold'][q,j]) for j in selected),'traded':traded/INITIAL,'cost':cost/INITIAL})
'''
left=v45_code.index('        if offset == 0 or (rebalance > 0 and offset % rebalance == 0):');right=v45_code.index('        if np.any((shares > 0) & ~np.isfinite(mark[t])):');v45_code=v45_code[:left]+v45_decision+v45_code[right:];v45_code=v45_code.replace("return {'paid_policy':","return {'ma_pair':ma_pair,'ma_mode':ma_mode,'events':events45,'paid_policy':")
v45_ns=dict(v34_ns);v45_ns.update(ma45=v45_ma,scores30=v45_scores);exec(v45_code,v45_ns);v45_sim=v45_ns['simulate_ma45'];v45_engine_hash=hashlib.sha256(v45_code.encode()).hexdigest();v45_controls=[]
for fam in ['MOM','FLOW']:
    for ix in [0,400,900,1700,2235]:
        s=int(v30_starts[ix]);old=v34_results[fam+'_E0'][ix];now=v45_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved',fam,.9,5,'5_20','none');assert old['reasons']==now['reasons'] and old['violations']==now['violations'] and old['return']==now['return']
        if now['return'] is not None:assert all(np.isclose(old[k],now[k],rtol=0,atol=1e-12) for k in ['mdd','cost','turnover'])
        v45_controls.append({'family':fam,'index':ix,'matched':True})
v45_specs=[]
for pair in v45_pairs_ma:
    for fam in ['MOM','FLOW']:
        for md in ['INITIAL','ACTIVE']:v45_specs.append({'key':fam+'_'+pair+'_'+md,'strategy':fam,'pair':pair,'mode':md})
    v45_specs.append({'key':'PURE_'+pair,'strategy':'MA_'+pair,'pair':pair,'mode':'ACTIVE'})
v45_results={k:v34_results[k] for k in ['MOM_E0','FLOW_E0']};v45_newruns=0;sp45=v34_root/'A策略_無人值守研究狀態.json';state=json.loads(sp45.read_text(encoding='utf-8'));state['ma_cross_v45']={'status':'definition_frozen_controls_passed','running':False,'definition_hash':v45_hash,'engine_hash':v45_engine_hash,'specs':v45_specs,'baseline_matches':v45_controls};sp45.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V45_DEFINED',{'controls':len(v45_controls),'new_specs':len(v45_specs),'positive_state_coverage':{k:int(np.sum(np.sum(m['bull']&np.isfinite(v30_scores['MOM']),axis=1)>=25)) for k,m in v45_ma.items()}})


In [ ]:
# One finite batch, per-spec cache; no extra MA-day search after seeing outcomes.
for spec0 in v45_specs:
    key=spec0['key'];spec={**spec0,'definition_hash':v45_hash,'engine_hash':v45_engine_hash};fp=v34_root/('均線交叉_'+key+'_v45_2026-10-17.json.gz');state=json.loads(sp45.read_text(encoding='utf-8'));state['ma_cross_v45'].update(status='running',running=True,current_method=key,completed_methods=[k for k in v45_results if k not in ['MOM_E0','FLOW_E0']]);sp45.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
    if fp.exists():
        with gzip.open(fp,'rt',encoding='utf-8') as f:obj=json.load(f)
        assert obj['spec']==spec;v45_results[key]=obj['results'];print('V45_REUSED',key,flush=True);continue
    rows=[]
    for ix,s0 in enumerate(v30_starts):
        s=int(s0);r=v45_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved',spec['strategy'],.9,5,spec['pair'],spec['mode']);r['start_index']=s;rows.append(r)
        if (ix+1)%1000==0:print('V45_PROGRESS',key,ix+1,flush=True)
    with gzip.open(fp,'wt',encoding='utf-8') as f:json.dump({'spec':spec,'results':rows},f,ensure_ascii=False,default=serial30,allow_nan=False)
    v45_results[key]=rows;v45_newruns+=len(rows);print('V45_DONE',key,{'valid':sum(r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows),'newruns':v45_newruns},flush=True)
state=json.loads(sp45.read_text(encoding='utf-8'));state['ma_cross_v45'].update(status='computed_pending_statistics',running=False,current_method=None,new_window_runs=v45_newruns);sp45.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8');print('V45_COMPUTED',len(v45_results),v45_newruns)


In [ ]:
# All32 contrasts fixed before outcomes; increase bootstrap draws for32-way corrected tails.
import inspect
v45_stat_source=inspect.getsource(analyze37).replace('def analyze37(','def analyze_ma45(').replace('for chunk in range(50):','for chunk in range(100):').replace('37000+family_size','45000+family_size');assert 'range(100)' in v45_stat_source;exec(v45_stat_source,globals())
v45_contrast_pairs=[]
for pair in v45_pairs_ma:
    for fam in ['MOM','FLOW']:
        v45_contrast_pairs.extend([(fam+'_'+pair+'_'+md,fam+'_E0') for md in ['INITIAL','ACTIVE']]);v45_contrast_pairs.append((fam+'_'+pair+'_ACTIVE',fam+'_'+pair+'_INITIAL'))
    v45_contrast_pairs.extend([('PURE_'+pair,fam+'_E0') for fam in ['MOM','FLOW']])
assert len(v45_contrast_pairs)==32
v45_stats,v45_valid,v45_ret,v45_common=analyze_ma45(v45_results,v45_contrast_pairs,32)
def metrics_ma45(k,mask):
    ix=np.flatnonzero(mask);rr=v45_ret[k][mask];rows=v45_results[k]
    if not len(ix):return {'n':0}
    out={'n':len(ix),'win':float(np.mean(rr>0)),'ev':float(np.mean(rr)),'median':float(np.median(rr)),'p10':float(np.quantile(rr,.1)),'worst_return':float(rr.min()),'worst_mdd':float(min(rows[i]['mdd'] for i in ix)),'fee':float(np.mean([rows[i]['cost'] for i in ix])),'turnover':float(np.mean([rows[i]['turnover'] for i in ix]))}
    if k not in ['MOM_E0','FLOW_E0']:out.update(mean_death_exit_slots=float(np.mean([sum(len(e['death_exits']) for e in rows[i]['events']) for i in ix])),windows_with_offcycle=int(sum(any(not e['regular'] for e in rows[i]['events']) for i in ix)))
    return out
for name,r in v45_stats['contrasts'].items():
    a,b=r['candidate'],r['base'];m=v45_valid[a]&v45_valid[b]&v30_later;r['candidate_stats']=metrics_ma45(a,m);r['base_stats']=metrics_ma45(b,m);r['fee_difference']=r['candidate_stats']['fee']-r['base_stats']['fee'];r['accounting_before_fee_difference']=r['mean_difference']+r['fee_difference'];r['beat_control_fraction']=float(np.mean((v45_ret[a]-v45_ret[b])[m]>0));r['excluded_base_later']=metrics_ma45(b,v45_valid[b]&v30_later&~v45_valid[a]);r['time_segments']={}
    for label,lo,hi in [('2019to2021',2019,2021),('2022to2025',2022,2025)]:
        mm=m&np.array([lo<=v30_days[int(s)].year<=hi for s in v30_starts]);r['time_segments'][label]={'n':int(mm.sum()),'difference':float(np.mean((v45_ret[a]-v45_ret[b])[mm])) if mm.any() else None}
v45_common_pair={};v45_stats['pair_common']={}
for pair in v45_pairs_ma:
    keys=['MOM_E0','FLOW_E0']+[fam+'_'+pair+'_'+md for fam in ['MOM','FLOW'] for md in ['INITIAL','ACTIVE']]+['PURE_'+pair];mask=np.logical_and.reduce([v45_valid[k] for k in keys]);v45_common_pair[pair]=mask;v45_stats['pair_common'][pair]={'all_n':int(mask.sum()),'later_n':int((mask&v30_later).sum()),'all':{k:metrics_ma45(k,mask) for k in keys},'later':{k:metrics_ma45(k,mask&v30_later) for k in keys}}
v45_stats['coverage_detail']={k:{'valid':int(v45_valid[k].sum()),'later_valid':int((v45_valid[k]&v30_later).sum()),'invalid_reasons':dict(collections.Counter(reason for r in rows for reason in r['reasons'])),'violations_windows':sum(r['violations']>0 for r in rows),'valid_start_first':next((r['start'] for i,r in enumerate(rows) if v45_valid[k][i]),None)} for k,rows in v45_results.items()}
v45_stats.update(definition_hash=v45_hash,engine_hash=v45_engine_hash,input_hashes=v34_plan['common']['source_hashes'],new_window_runs=v45_newruns,bootstrap_draws=10000,non_independent_and_retrospective=True)
(v34_root/'均線交叉_全組統計_v45_2026-10-17.json').write_text(json.dumps(v45_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp45.read_text(encoding='utf-8'));state['ma_cross_v45'].update(status='statistics_complete_pending_report',running=False,contrasts=32,full_improvements=v45_stats['full_improvements'],pair_common={p:{k:v[k] for k in ['all_n','later_n']} for p,v in v45_stats['pair_common'].items()});sp45.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V45_STATISTICS',{'full_improvements':v45_stats['full_improvements'],'pair_common':{p:[r['all_n'],r['later_n']] for p,r in v45_stats['pair_common'].items()},'contrasts':{n:{k:r[k] for k in ['n','coverage','mean_difference','CI48','mean_improvement','risk_not_worse','year_gate','full_improvement']} for n,r in v45_stats['contrasts'].items()}})


In [ ]:
# Readable full batch report, paired evidence and complete common-window dot plots.
v45_mean_only=[n for n,r in v45_stats['contrasts'].items() if r['mean_improvement'] and not r['full_improvement']]
def label_ma45(k):
    if k=='MOM_E0':return '原動能／五日'
    if k=='FLOW_E0':return '原法人／五日'
    if k.startswith('PURE_'):return '純均線強度 '+k[5:].replace('_','／')
    fam,rest=k.split('_',1);pair,md=rest.rsplit('_',1);return ('動能' if fam=='MOM' else '法人')+' '+pair.replace('_','／')+('首日篩選' if md=='INITIAL' else '交叉換股')
v45_lines=['# 黃金／死亡交叉：20種均線策略回測','2026-10-17｜固定四組SMA：3／5、5／20、20／60、50／200日。20組新策略×2236起點＝44,720次模擬，原動能與法人基準直接重用。','## 結論：有新的收益候選，但沒有全面勝出','**50／200日最值得保留觀察。**純均線多頭強度選股、法人加50／200交叉規則，部分配對的費後平均報酬改善通過32項多重比較校正；但P10或最差回撤變差，沒有通過事先要求的全面改善門檻。','3／5日頻繁交叉換股明顯拖累收益，且很多窗口無法形成25檔合格配置。只在首日加多頭均線篩選，多數差異很小。此輪不能簡化成「均線交叉都無效」，也不能把50／200當成已確定最佳策略。','## 先把名稱說準：事件、狀態、排名不同','- **黃金交叉事件**：短SMA昨天≤長SMA，今天>長SMA，且兩天數值都已知。','- **死亡交叉事件**：短SMA昨天≥長SMA，今天<長SMA。','- **多頭狀態**：目前短SMA>長SMA，可能早已交叉，不要求今天才發生。','- **純均線強度**：在多頭狀態股票中，按短SMA／長SMA−1由高到低排名；它不是只挑當天剛黃金交叉的股票，也不是使用20／60日漲跌幅排名。','50／200是觀察價格的天數，持有期仍是24交易日，不需要持有200天才可使用。','## 每組均線都測五個新策略','|策略|第一天|之後|','|---|---|---|','|動能＋首日篩選|原動能排名，限多頭狀態，取25檔|恢復原五日調整，不做額外交叉退出|','|動能＋交叉換股|同上|每五日重選時限多頭；持股死亡交叉則次日局部換股|','|法人＋首日篩選|純法人排名，限多頭狀態，取25檔|恢復原法人五日調整|','|法人＋交叉換股|同上|同樣多頭篩選及死亡交叉局部換股|','|純均線強度|只按上述均線強度排名25檔|每五日重排，死亡交叉次日局部換股|','交叉換股時，候補需多頭狀態且原排名合格；股票重新黃金交叉後可再次進入候補，**不是每次黃金交叉都自動買入，也不是原股一定買回**。其餘持股不因局部換股全部重新等權。','### 共用條件與時間','- 同一2016～2025長歷史、目前150檔股票的歷史子集；2016為部分年度。50／200需200個已知價格層級，因此更早窗口無法判定，不造2015價格補暖機。','- 原有過去資料品質篩選沿用；純均線不使用動能／法人分數作排名，但共用原動能的歷史資料可用性篩選。','- 25檔、90%目標投入、整千股、24日，每五日正常重選。25檔是這次控制變因，不是宣稱官方要求固定25。','- 使用已知現金股息調整的價格層級計算SMA；未知公司行動／缺價不直接填補。','- 前日收盤後確認訊號，次日按均價模擬成交；含買賣各0.1425%費、賣出0.3%稅。不能把盤中交叉即刻成交當成已實現。','- 合格多頭不足25檔、價格／公司行動不明或限制違反，就記不可判定，不拿0報酬補，也不強買空頭股票湊數。','## 同日期看50／200：後段942共同窗口','這張表的七種方法都是2019～2025完全相同的942起點。它不是下方各自配對1093、1113等分母；不同分母的平均不能混減。','|方法|勝率|費後24日期望|P10|最差回撤|平均費稅|','|---|---:|---:|---:|---:|---:|']
for k,m in v45_stats['pair_common']['50_200']['later'].items():v45_lines.append('|'+ '|'.join([label_ma45(k),p30(m['win']),p30(m['ev']),p30(m['p10']),p30(m['worst_mdd']),p30(m['fee'])])+'|')
v45_lines+=['## 最值得注意的配對：收益增加，但風險不是一起改善','每列是候選與對照自己的後段有效交集；比較日期相同，各列分母不同，不能直接跨列排冠軍。','|候選−對照|配對n|候選／對照勝率|候選／對照期望|平均差pp|48日／32比較校正區間pp|候選／對照最差回撤|','|---|---:|---|---|---:|---|---|']
v45_focus=['FLOW_50_200_ACTIVE_minus_FLOW_E0','PURE_50_200_minus_MOM_E0','PURE_50_200_minus_FLOW_E0','PURE_20_60_minus_FLOW_E0','PURE_5_20_minus_FLOW_E0']
for name in v45_focus:
    r=v45_stats['contrasts'][name];a,b=r['candidate_stats'],r['base_stats'];v45_lines.append('|'+ '|'.join([label_ma45(r['candidate'])+'−'+label_ma45(r['base']),str(r['n']),p30(a['win'])+'／'+p30(b['win']),p30(a['ev'])+'／'+p30(b['ev']),f"{r['mean_difference']*100:+.3f}",c30(r['CI48']),p30(a['worst_mdd'])+'／'+p30(b['worst_mdd'])])+'|')
v45_lines+=['法人50／200交叉相對原法人，費後期望增加約0.754個百分點，勝率66.94%→71.97%；但P10由−4.06%變−4.44%，最差回撤−30.94%變−31.54%，所以不算全面改善。','純50／200相對原動能，期望增加約1.126個百分點，勝率66.97%→73.38%；P10改善，但最差回撤−27.65%變−32.59%。它較像收益更高、極端風險也更高的新候選，而不是已解決停損問題。','## 為什麼不能只看高勝率？先看能否持續形成配置','|比較候選−對照|後段覆蓋率|未被涵蓋的對照窗口n|被排除窗口原策略勝率／期望|','|---|---:|---:|---|']
for name in ['MOM_3_5_ACTIVE_minus_MOM_E0','FLOW_3_5_ACTIVE_minus_FLOW_E0','PURE_5_20_minus_FLOW_E0','FLOW_50_200_ACTIVE_minus_FLOW_E0','PURE_50_200_minus_MOM_E0','PURE_50_200_minus_FLOW_E0']:
    r=v45_stats['contrasts'][name];d=r['excluded_base_later'];v45_lines.append('|'+ '|'.join([label_ma45(r['candidate'])+'−'+label_ma45(r['base']),p30(r['coverage']),str(d['n']),p30(d.get('win'))+'／'+p30(d.get('ev'))])+'|')
v45_lines+=['例如純5／20相對法人，只涵蓋原對照約64%的後段日期；排除的422窗口，原法人平均報酬為−0.92%。其高勝率不能直接代表全年都能操作，更不能把這些未解的窗口視為已成功避開。反過來，純50／200對動能被排除的147窗口原策略平均+4.52%，也不是單純只剔掉虧損行情。','## 50／200逐年：2025確實是反例','|起始年|純50／200−動能：n／平均差pp|純50／200−法人：n／平均差pp|法人50／200交叉−法人：n／平均差pp|','|---|---|---|---|']
for yr in range(2019,2026):
    cells=[str(yr)]
    for name in ['PURE_50_200_minus_MOM_E0','PURE_50_200_minus_FLOW_E0','FLOW_50_200_ACTIVE_minus_FLOW_E0']:
        row=next(x for x in v45_stats['contrasts'][name]['years'] if x['year']==yr);cells.append(str(row['n'])+'／'+(f"{row['difference']*100:+.3f}" if row['difference'] is not None else '不可判定'))
    v45_lines.append('|'+ '|'.join(cells)+'|')
v45_lines+=['純50／200在2025年相對動能約−1.63pp、相對法人約−0.50pp；不能只報2019～2024較好的部分。時間前後段增量與每年分母都保存在統計JSON。','## 成本與換手：短均線和長均線差在哪？','|候選−對照|後段配對n|費後平均差pp|費用差pp|同路徑會計加回費用後差pp|','|---|---:|---:|---:|---:|']
for name in ['MOM_3_5_ACTIVE_minus_MOM_E0','FLOW_3_5_ACTIVE_minus_FLOW_E0','FLOW_50_200_ACTIVE_minus_FLOW_E0','PURE_50_200_minus_MOM_E0']:
    r=v45_stats['contrasts'][name];v45_lines.append('|'+ '|'.join([label_ma45(r['candidate'])+'−'+label_ma45(r['base']),str(r['n']),f"{r['mean_difference']*100:+.3f}",f"{r['fee_difference']*100:+.3f}",f"{r['accounting_before_fee_difference']*100:+.3f}"])+'|')
v45_lines+=['純50／200相對動能的約1.126pp增量，其中約0.498pp來自較少費用，其餘約0.628pp是同路徑加回費用後差。這是會計分解，不等於已證明低換手單獨造成全部改善；沒有額外跑零成本反事實。','## 四組均線各自的同日期比較','下列每組七方法在組內同日期，組與組的日期不同。因此不能把3／5高勝率直接和50／200比較。互動箱形圖可切全期／後段，每個共同日期點都保留。']
for pair,info in v45_stats['pair_common'].items():
    if pair=='50_200':continue
    v45_lines+=['### '+pair.replace('_','／')+'日：全期'+str(info['all_n'])+'／後段'+str(info['later_n'])+'共同窗口','|方法|後段勝率|後段24日期望|P10|最差回撤|平均費稅|','|---|---:|---:|---:|---:|---:|']
    for k,m in info['later'].items():v45_lines.append('|'+ '|'.join([label_ma45(k),p30(m['win']),p30(m['ev']),p30(m['p10']),p30(m['worst_mdd']),p30(m['fee'])])+'|')
v45_lines+=['## 32項固定比較全部揭露','2016～2018作校準資料量檢查，主要比較2019～2025；後段資料已多次研究，並非真正未見資料。原起點位置24／48日區塊各10000次，32項Bonferroni校正，保留重疊時間依賴。','|候選−對照|後段n|平均差pp|48日校正區間pp|收益改善|風險不劣|跨年通過|全面改善|','|---|---:|---:|---|---|---|---|---|']
for name,r in v45_stats['contrasts'].items():v45_lines.append('|'+ '|'.join([label_ma45(r['candidate'])+'−'+label_ma45(r['base']),str(r['n']),f"{r['mean_difference']*100:+.3f}",c30(r['CI48']),str(r['mean_improvement']),str(r['risk_not_worse']),str(r['year_gate']),str(r['full_improvement'])])+'|')
v45_lines+=['全面改善同時要求24及48日校正收益下界>0、P10與最差回撤均不差、覆蓋率≥80%、校準配對≥100，以及至少5個各≥20配對年份中三分之二增量正。沒有方案全部通過。校正只針對這一批32項，不能消除整個長期研究多次試策略、現有股票池偏差等限制。','## 全部策略的可判定範圍','|方法|全期有效／2236|後段有效|最早有效起點|有模擬限制違反的窗口|','|---|---:|---:|---|---:|']
for k,row in v45_stats['coverage_detail'].items():v45_lines.append('|'+ '|'.join([label_ma45(k),str(row['valid'])+'/2236',str(row['later_valid']),row['valid_start_first'] or '無',str(row['violations_windows'])])+'|')
v45_lines+=['未解的公司行動、缺價、25檔不足與現金／權重等限制分開保存在JSON及逐方法壓縮結果，沒有因不好看刪除。尚未驗證實際官賽Active Share與成交作廢等全部細節，不能宣稱可直接提交。','## 本輪應保留哪些？','- **收益候選：純50／200均線強度、法人＋50／200交叉。**配對收益有訊號，風險仍需取捨，2025反例保留。','- **暫不採用：3／5頻繁交叉換股。**費後差明顯負，且可行日期覆蓋很低。','- **未證實：首日均線篩選普遍能改善原動能／法人。**多數增量小、區間跨零。','- 原基準繼續保留，不因單一最高平均報酬替換。未加碼搜尋其他天數、EMA或新門檻。','## 來源、保存與重現','概念來源：[Fidelity](https://www.fidelity.com/learning-center/trading-investing/technical-analysis/what-is-technical-analysis)、[Schwab](https://www.schwab.com/learn/story/understanding-simple-moving-average-crossovers)。數據沿用原凍結來源快照、Notebook367起保存定義、10個原基準對照、批次、統計及圖表，既有kernel未重啟。','[全組統計與逐年](均線交叉_全組統計_v45_2026-10-17.json) · [事先定義](均線交叉_策略定義_v45_2026-10-17.json) · [回總結](策略研究_總結報告.html)。20個逐方法壓縮JSON保留全部2236起點，檔名為「均線交叉_方法代碼_v45_2026-10-17.json.gz」。']
v45_text='\n'.join(v45_lines);v45_rp=v34_root/'均線交叉_選股退出與全部窗口比較_v45_2026-10-17.md';v45_rp.write_text(v45_text,encoding='utf-8')
v45_fig=go.Figure();v45_views={};v45_titles={};ext45=[]
for pair,mask in v45_common_pair.items():
    for later in [False,True]:
        view=pair+('_later' if later else '_all');mm=mask&(v30_later if later else np.ones(len(mask),bool));ix=np.flatnonzero(mm);ids=[];v45_titles[view]=pair.replace('_','／')+'日'+('／2019～2025' if later else '／全期')+'／'+str(len(ix))+'共同窗口';keys=list(v45_stats['pair_common'][pair]['later'])
        for k in keys:
            label=label_ma45(k);ys=v45_ret[k][mm]*100;ext45.extend([float(ys.min()),float(ys.max())]);ids.append(len(v45_fig.data));color='#167368' if k in ['MOM_E0','FLOW_E0'] else '#567bb5'
            v45_fig.add_trace(go.Box(name=label,y=ys,boxpoints='all',jitter=.4,pointpos=0,line={'color':color},fillcolor='rgba(255,255,255,0)',marker={'size':3,'opacity':.35,'color':color},customdata=[[v45_results[k][i]['start'],v45_results[k][i]['end'],v45_results[k][i]['cost']*100] for i in ix],hovertemplate='%{customdata[0]}→%{customdata[1]}<br>費後24日%{y:.2f}%<br>費稅%{customdata[2]:.2f}%<extra></extra>',visible=view=='50_200_later'))
            ids.append(len(v45_fig.data));v45_fig.add_trace(go.Scatter(x=[label],y=[float(np.median(ys))],mode='markers',marker={'color':'#df4385','symbol':'line-ew','size':23,'line':{'width':3}},showlegend=False,hovertemplate='中位數%{y:.2f}%<extra></extra>',visible=view=='50_200_later'))
        v45_views[view]=ids
v45_fig.add_hline(y=0,line_color='#222',line_width=2);v45_fig.update_layout(template='plotly_white',height=730,showlegend=False,title=v45_titles['50_200_later'],yaxis={'title':'費後24日報酬 %','range':[min(ext45)-3,max(ext45)+3]},xaxis={'tickangle':-25},margin={'t':115,'b':160},updatemenus=[{'type':'dropdown','x':0,'y':1.16,'active':7,'buttons':[{'label':v45_titles[v],'method':'update','args':[{'visible':[i in ids for i in range(len(v45_fig.data))]},{'title':v45_titles[v]}]} for v,ids in v45_views.items()]}])
v45_hp=v34_root/'均線交叉_選股退出與全部窗口比較_v45_2026-10-17.html';render45=MarkdownIt('commonmark',{'html':False}).enable('table').render(v45_text).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>')
v45_hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>均線交叉策略比較</title><style>body{font:16px/1.8 system-ui;max-width:1300px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:9px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}</style><a href="策略研究_總結報告.html">回總結</a>'+v45_fig.to_html(full_html=False,include_plotlyjs=True,config={'responsive':True,'displaylogo':False})+render45+'</html>',encoding='utf-8')
v45_snippet='<section id="ma-cross-study"><h2>均線交叉：有收益候選，尚無全面勝出</h2><p>四組SMA、20新策略、44,720次窗口模擬。純50／200均線強度、法人＋50／200交叉，部分配對費後平均改善通過32項校正；最差回撤或P10較差，2025反例保留。3／5頻繁換股則明顯拖累且覆蓋低。</p><p>這是多頭狀態篩選／死亡交叉換股／均線強度排名，不是25檔同日黃金交叉，也不是已找到低風險最佳策略。</p><a href="'+v45_hp.name+'">查看勝率、期望、全部點、風險與32項對照 →</a></section>'
sumhp45=v34_root/'策略研究_總結報告.html';page45=sumhp45.read_text(encoding='utf-8');assert 'ma-cross-study' not in page45;page45=page45.replace('<section id="context-exit-reentry">',v45_snippet+'<section id="context-exit-reentry">');assert v45_snippet in page45;sumhp45.write_text(page45,encoding='utf-8')
summd45=v34_root/'策略研究_總結報告.md';summd45.write_text(summd45.read_text(encoding='utf-8')+'\n\n## 均線交叉策略（v45，2026-10-17）\n\n20新策略、44,720次窗口模擬、32項24／48日區塊各10000次校正。純50／200強度、法人＋50／200交叉有配對收益候選，但風險門檻未通過；純50／200在2025落後。3／5頻繁換股拖累且覆蓋低。沒有全面勝出。[全部策略、年份與箱形圖]('+v45_hp.name+')。\n',encoding='utf-8')
# Direct-price SMA sanity check on no-dividend/no-action samples, independent of return compounding.
checks45=[]
for pair,(fast,slow) in v45_pairs_ma.items():
    for q in [250,700,1300,2100]:
        sub=v30_arrays['close'][q-slow+1:q+1];ok=np.all(np.isfinite(sub)&(sub>0),axis=0)&np.all(v30_arrays['cashdiv'][q-slow+2:q+1]==0,axis=0)&np.all(~v30_arrays['bad'][q-slow+2:q+1],axis=0)&v45_ma[pair]['known'][q]
        direct=np.mean(sub[-fast:,ok],axis=0)/np.mean(sub[:,ok],axis=0)-1;assert np.allclose(direct,v45_ma[pair]['gap'][q,ok],rtol=0,atol=1e-10);checks45.append({'pair':pair,'date':str(v30_days[q]),'stocks_checked':int(ok.sum())})
v45_validation={'baseline_controls':len(v45_controls),'direct_SMA_checks':checks45,'new_methods':20,'new_window_runs':v45_newruns,'contrasts':32,'bootstrap_draws_per_block':10000,'events_strictly_lagged':all(e['signal_date']<e['date'] for k,rows in v45_results.items() if k not in ['MOM_E0','FLOW_E0'] for r in rows for e in r['events']),'plot_points_per_view':{v:sum(len(v45_fig.data[i].y) for i in ids if v45_fig.data[i].type=='box') for v,ids in v45_views.items()},'full_improvements':v45_stats['full_improvements'],'mean_improvement_only':v45_mean_only,'browser_visual_qa':'not performed;programmatic validation only'}
assert v45_newruns==44720 and v45_validation['events_strictly_lagged'] and len(v45_contrast_pairs)==32
assert all(n==7*v45_stats['pair_common'][v.rsplit('_',1)[0]][('later_n' if v.endswith('_later') else 'all_n')] for v,n in v45_validation['plot_points_per_view'].items())
assert 'unified-longhistory-boxplot' in sumhp45.read_text(encoding='utf-8')
(v34_root/'均線交叉_驗證_v45_2026-10-17.json').write_text(json.dumps(v45_validation,ensure_ascii=False,indent=2),encoding='utf-8');state=json.loads(sp45.read_text(encoding='utf-8'));state['ma_cross_v45'].update(status='completed_return_candidates_no_full_improvement',running=False,current_method=None,html=v45_hp.name,report=v45_rp.name,mean_improvement_only=v45_mean_only,validation=v45_validation);sp45.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V45_PUBLISHED',{'html':v45_hp.name,'mean_only':v45_mean_only,'validation':v45_validation})


In [ ]:
# Fixed second batch of short SMA pairs, same24-day horizon, reuse all prior results.
print('V46_RETAINED',{'python':sys.executable,'pid':os.getpid(),'prior_methods':len(v45_results),'prior_status':json.loads(sp45.read_text(encoding='utf-8'))['ma_cross_v45']['status']})
v46_pairs_ma={'3_10':(3,10),'5_10':(5,10),'5_15':(5,15),'10_20':(10,20),'5_24':(5,24),'10_24':(10,24)}
v46_definition={'version':'v46_short_sma_fixed_1','date':'2026-10-17','input_hashes':v34_plan['common']['source_hashes'],'parent_definition':v45_hash,'engine':v45_engine_hash,'new_pairs':v46_pairs_ma,'new_modes':'MOM ACTIVE,FLOW ACTIVE,PURE ACTIVE perpair;initial-only notexpanded because prior incremental results tiny','holding':'always24sessions;90%target,25names,5dayregular,1000shares,fees/actions/constraints identicalv45','signals':'same adjusted-SMA positive state eligibility;actual priorclose deathcross triggers nextday oneforone replacement;newgoldenstate qualifies candidates,notautomatic buy','pure_ranking':'shortSMA/longSMA−1 amongpositive states;commonhistoricalquality,notoriginal MOM/FLOW ranking','warmup':'use same existing2016–2025 snapshots,do notmanufacture earlierdata','new_specs':18,'new_window_runs':40248,'comparisons':'reuse32old +24newvsownbaseline +18newvs50/200samevariant =74;all fixed before newreturn analysis','statistics':'24/48day original-positionblocks20000each;Bonferroni74;2016–2018training2019–2025later;notunseen','gate':'samev45 bothCIpositive,P10andworstMDDnotworse,80%coverage,100training,5yearmin20and2/3positiveyears','shortage':'invalidifbullishavailableunder25;coverage andexcludedbaselineoutcomes shown;no forcedfill','scope':'SMA daypairs only;noEMA or outcome-selected fine grid','selection':'do not assume shorterlookback better merelybecause24daycompetition;no highestmean forcedwinner'}
(v34_root/'短均線_24日競賽比較定義_v46_2026-10-17.json').write_text(json.dumps(v46_definition,ensure_ascii=False,indent=2),encoding='utf-8');v46_hash=hashlib.sha256(json.dumps(v46_definition,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
v46_ma=dict(v45_ma);v46_scores=dict(v45_scores)
for pair,(fast,slow) in v46_pairs_ma.items():
    gap=np.full_like(v30_arrays['close'],np.nan)
    for q in range(slow-1,len(v30_days)):
        rr=v30_arrays['returns'][q-slow+2:q+1];ok=np.all(np.isfinite(rr)&(rr>-1)&~v30_arrays['bad'][q-slow+2:q+1],axis=0);levels=np.vstack([np.ones(rr.shape[1]),np.cumprod(1+rr,axis=0)]);gap[q,ok]=np.mean(levels[-fast:,ok],axis=0)/np.mean(levels[:,ok],axis=0)-1
    known=np.isfinite(gap);bull=known&(gap>0);gold=np.zeros_like(bull);death=np.zeros_like(bull);gold[1:]=known[1:]&known[:-1]&(gap[1:]>0)&(gap[:-1]<=0);death[1:]=known[1:]&known[:-1]&(gap[1:]<0)&(gap[:-1]>=0);v46_ma[pair]={'gap':gap,'known':known,'bull':bull,'gold':gold,'death':death};pure=np.full_like(gap,np.nan)
    for q in range(slow-1,len(v30_days)):pure[q]=percentile(gap[q],bull[q]&np.isfinite(v30_scores['MOM'][q]))
    v46_scores['MA_'+pair]=pure
v46_ns=dict(v45_ns);v46_ns.update(ma45=v46_ma,scores30=v46_scores);exec(v45_code,v46_ns);v46_sim=v46_ns['simulate_ma45'];v46_engine_hash=v45_engine_hash
assert all(v46_ma[p] is v45_ma[p] for p in v45_pairs_ma)
v46_checks=[]
for pair,(fast,slow) in v46_pairs_ma.items():
    for q in [250,700,1300,2100]:
        sub=v30_arrays['close'][q-slow+1:q+1];ok=np.all(np.isfinite(sub)&(sub>0),axis=0)&np.all(v30_arrays['cashdiv'][q-slow+2:q+1]==0,axis=0)&np.all(~v30_arrays['bad'][q-slow+2:q+1],axis=0)&v46_ma[pair]['known'][q];direct=np.mean(sub[-fast:,ok],axis=0)/np.mean(sub[:,ok],axis=0)-1;assert np.allclose(direct,v46_ma[pair]['gap'][q,ok],rtol=0,atol=1e-10);v46_checks.append({'pair':pair,'date':str(v30_days[q]),'n':int(ok.sum())})
v46_specs=[]
for pair in v46_pairs_ma:
    for fam in ['MOM','FLOW']:v46_specs.append({'key':fam+'_'+pair+'_ACTIVE','strategy':fam,'pair':pair,'mode':'ACTIVE'})
    v46_specs.append({'key':'PURE_'+pair,'strategy':'MA_'+pair,'pair':pair,'mode':'ACTIVE'})
v46_results=dict(v45_results);v46_newruns=0;sp46=sp45;state=json.loads(sp46.read_text(encoding='utf-8'));state['short_ma_v46']={'status':'definition_frozen','running':False,'definition_hash':v46_hash,'specs':v46_specs,'prior_results_reused':22};sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8');print('V46_DEFINED',{'new_specs':18,'reused':22,'checks':len(v46_checks),'contrasts':74})


In [ ]:
# Run only18 new candidates; old22 numerical paths stay unchanged.
for spec0 in v46_specs:
    key=spec0['key'];spec={**spec0,'definition_hash':v46_hash,'engine_hash':v46_engine_hash};fp=v34_root/('短均線_'+key+'_v46_2026-10-17.json.gz');state=json.loads(sp46.read_text(encoding='utf-8'));state['short_ma_v46'].update(status='running',running=True,current_method=key);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
    if fp.exists():
        with gzip.open(fp,'rt',encoding='utf-8') as f:obj=json.load(f)
        assert obj['spec']==spec;v46_results[key]=obj['results'];print('V46_REUSED',key,flush=True);continue
    rows=[]
    for ix,s0 in enumerate(v30_starts):
        s=int(s0);r=v46_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved',spec['strategy'],.9,5,spec['pair'],spec['mode']);r['start_index']=s;rows.append(r)
        if (ix+1)%1000==0:print('V46_PROGRESS',key,ix+1,flush=True)
    with gzip.open(fp,'wt',encoding='utf-8') as f:json.dump({'spec':spec,'results':rows},f,ensure_ascii=False,default=serial30,allow_nan=False)
    v46_results[key]=rows;v46_newruns+=len(rows);print('V46_DONE',key,{'valid':sum(r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows),'newruns':v46_newruns},flush=True)
state=json.loads(sp46.read_text(encoding='utf-8'));state['short_ma_v46'].update(status='computed_pending_statistics',running=False,current_method=None,new_window_runs=v46_newruns);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8');print('V46_COMPUTED',len(v46_results),v46_newruns)


In [ ]:
# Joint74-way family including previous32; fixed new42 contrasts, no return-based additions.
v46_contrast_pairs=list(v45_contrast_pairs)
for pair in v46_pairs_ma:
    for fam in ['MOM','FLOW']:v46_contrast_pairs.append((fam+'_'+pair+'_ACTIVE',fam+'_E0'))
    for fam in ['MOM','FLOW']:v46_contrast_pairs.append(('PURE_'+pair,fam+'_E0'))
    for fam in ['MOM','FLOW']:v46_contrast_pairs.append((fam+'_'+pair+'_ACTIVE',fam+'_50_200_ACTIVE'))
    v46_contrast_pairs.append(('PURE_'+pair,'PURE_50_200'))
assert len(v46_contrast_pairs)==74 and len(set(v46_contrast_pairs))==74
assert all(v46_results[k] is v45_results[k] for k in v45_results)
v46_stat_source=v45_stat_source.replace('def analyze_ma45(','def analyze_ma46(').replace('for chunk in range(100):','for chunk in range(200):').replace('45000+family_size','46000+family_size');exec(v46_stat_source,globals())
v46_stats,v46_valid,v46_ret,v46_common=analyze_ma46(v46_results,v46_contrast_pairs,74)
def metrics_short46(k,mask):
    ix=np.flatnonzero(mask);rr=v46_ret[k][mask];rows=v46_results[k]
    if not len(ix):return {'n':0}
    out={'n':len(ix),'win':float(np.mean(rr>0)),'ev':float(np.mean(rr)),'median':float(np.median(rr)),'p10':float(np.quantile(rr,.1)),'worst_mdd':float(min(rows[i]['mdd'] for i in ix)),'fee':float(np.mean([rows[i]['cost'] for i in ix])),'turnover':float(np.mean([rows[i]['turnover'] for i in ix]))}
    if k not in ['MOM_E0','FLOW_E0']:out.update(mean_death_exit_slots=float(np.mean([sum(len(e['death_exits']) for e in rows[i]['events']) for i in ix])),windows_with_offcycle=int(sum(any(not e['regular'] for e in rows[i]['events']) for i in ix)))
    return out
for n,r in v46_stats['contrasts'].items():
    a,b=r['candidate'],r['base'];mm=v46_valid[a]&v46_valid[b]&v30_later;r['candidate_stats']=metrics_short46(a,mm);r['base_stats']=metrics_short46(b,mm);r['fee_difference']=r['candidate_stats']['fee']-r['base_stats']['fee'];r['accounting_before_fee_difference']=r['mean_difference']+r['fee_difference'];r['excluded_base_later']=metrics_short46(b,v46_valid[b]&v30_later&~v46_valid[a])
v46_overview_keys=['MOM_E0','FLOW_E0','PURE_50_200']+['PURE_'+p for p in v46_pairs_ma];v46_overview_mask=np.logical_and.reduce([v46_valid[k] for k in v46_overview_keys]);v46_stats['overview']={'all_n':int(v46_overview_mask.sum()),'later_n':int((v46_overview_mask&v30_later).sum()),'all':{k:metrics_short46(k,v46_overview_mask) for k in v46_overview_keys},'later':{k:metrics_short46(k,v46_overview_mask&v30_later) for k in v46_overview_keys}}
v46_pair_masks={};v46_stats['per_pair_common']={}
for p in v46_pairs_ma:
    keys=['MOM_E0','FLOW_E0','PURE_50_200','MOM_50_200_ACTIVE','FLOW_50_200_ACTIVE','MOM_'+p+'_ACTIVE','FLOW_'+p+'_ACTIVE','PURE_'+p];mm=np.logical_and.reduce([v46_valid[k] for k in keys]);v46_pair_masks[p]=mm;v46_stats['per_pair_common'][p]={'all_n':int(mm.sum()),'later_n':int((mm&v30_later).sum()),'all':{k:metrics_short46(k,mm) for k in keys},'later':{k:metrics_short46(k,mm&v30_later) for k in keys}}
v46_stats['new_vs_old_pure']={p:v46_stats['contrasts']['PURE_'+p+'_minus_PURE_50_200'] for p in v46_pairs_ma}
v46_stats['coverage_detail']={k:{'valid':int(v46_valid[k].sum()),'later_valid':int((v46_valid[k]&v30_later).sum()),'invalid_reasons':dict(collections.Counter(reason for r in rows for reason in r['reasons'])),'violations_windows':sum(r['violations']>0 for r in rows)} for k,rows in v46_results.items()}
v46_stats.update(definition_hash=v46_hash,parent_definition=v45_hash,engine_hash=v46_engine_hash,input_hashes=v34_plan['common']['source_hashes'],new_window_runs=v46_newruns,reused_prior_methods=22,bootstrap_draws=20000)
(v34_root/'短均線_24日競賽全組統計_v46_2026-10-17.json').write_text(json.dumps(v46_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['short_ma_v46'].update(status='statistics_complete_pending_report',running=False,contrasts=74,full_improvements=v46_stats['full_improvements'],overview_sample=[v46_stats['overview']['all_n'],v46_stats['overview']['later_n']]);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V46_STATISTICS',{'full_improvements':v46_stats['full_improvements'],'overview':v46_stats['overview']['later'],'short_pure_vs50':{p:{k:r[k] for k in ['n','mean_difference','CI48','mean_improvement','risk_not_worse','coverage','full_improvement']} for p,r in v46_stats['new_vs_old_pure'].items()},'new_positive_contrasts':[n for n,r in v46_stats['contrasts'].items() if r['mean_improvement'] and (r['candidate'] not in v45_results)],'old_positive_after74':[n for n,r in v46_stats['contrasts'].items() if r['mean_improvement'] and r['candidate'] in v45_results]})


In [ ]:
# Extend the summary with short-MA evidence, preserving all old frozen results and current correction scope.
newkeys46={s['key'] for s in v46_specs};newfull46=[n for n in v46_stats['full_improvements'] if v46_stats['contrasts'][n]['candidate'] in newkeys46];newmean46=[n for n,r in v46_stats['contrasts'].items() if r['candidate'] in newkeys46 and r['mean_improvement']];purebeat46=[p for p,r in v46_stats['new_vs_old_pure'].items() if r['mean_improvement']];downgrade46=[n for n,r in v45_stats['contrasts'].items() if r['mean_improvement'] and not v46_stats['contrasts'][n]['mean_improvement']]
lines46=['# 短均線是否更適合24交易日競賽？','2026-10-17｜新增3／10、5／10、5／15、10／20、5／24、10／24日SMA。18新配置、40,248次窗口模擬；前批22方法全部重用，未重跑。','## 先看結論','本批新增'+str(len(newfull46))+'項配對通過全面改善門檻；新增'+str(len(newmean46))+'項配對有校正後平均收益改善。純短均線相對50／200，通過平均改善的組合：'+('、'.join(p.replace('_','／') for p in purebeat46) if purebeat46 else '**沒有**')+'。','**24日是持有期，不必等於均線觀察天數。**短均線較快反應，但較常反覆換股，也可能在廣泛下跌時不足25檔合格多頭；本輪仍需同時看費後期望、回撤及配置可行性，不能只挑最高勝率。','## 本批測什麼？','每組三種策略，皆沿用前批定義，不根據本批結果再微調天數。','|策略|買入排序|持有中處理|','|---|---|---|','|動能＋短均線交叉|原20／60日動能排序，限短SMA>長SMA|每五日重選；持股死亡交叉則次日局部換股|','|法人＋短均線交叉|原法人買超排序，限短SMA>長SMA|同上|','|純短均線強度|多頭狀態內按短SMA／長SMA−1排名|同上|','「黃金交叉後可買」是恢復候補資格，不代表每一檔交叉就立即買入；第一天允許已經多頭排列，不要求25檔同日剛交叉。這不是全現金等待黃金交叉的策略。','- 持有期均為24交易日，90%目標投入、25檔、整千股、每五日正常調整。25檔是這批隔離均線變因的控制，不是官方最低檔數。','- 前一日收盤確認訊號，次日均價執行，含買賣各0.1425%及賣出0.3%費稅；不是盤中觸價立即停損。','- 均線用已知股息調整價格層級；不以缺價、未知公司行動或未公布資訊補訊號。','- 少於25檔合格候補時記不可判定，沒有放寬條件湊數、也沒有把未能形成配置視為0報酬。','- 僅擴充SMA的固定天數，未順便掃EMA、停損百分比或其他大量參數。','## 同日期先比較純選股：九方法完全相同窗口','全期'+str(v46_stats['overview']['all_n'])+'／後段2019～2025 '+str(v46_stats['overview']['later_n'])+'共同起點。這是六種短均線純策略、純50／200、原動能與原法人都可操作的交集；不是每一種方法的全部有效日期。','下表是後段共同日期；勝率＝費後24日報酬>0比例，期望＝費後24日平均報酬，不年化。','|方法|勝率|24日期望|P10|最差回撤|平均費稅|','|---|---:|---:|---:|---:|---:|']
for k,m in v46_stats['overview']['later'].items():lines46.append('|'+ '|'.join([label_ma45(k),p30(m['win']),p30(m['ev']),p30(m['p10']),p30(m['worst_mdd']),p30(m['fee'])])+'|')
lines46+=['不能把交集較高的勝率當成所有行情都能達成；各自不足25檔、公司行動等未解日期，另列覆蓋與被排除對照的結果。','## 每一種短均線，直接和純50／200配對','每列使用這兩方法自己的後段共同日期，所以分母不同；不是拿上表不同數字相減。','|短均線−50／200|配對n|短／長勝率|短／長期望|平均差pp|48日／74比較校正區間pp|短／長最差回撤|收益改善？|','|---|---:|---|---|---:|---|---|---|']
for p,r in v46_stats['new_vs_old_pure'].items():
    a,b=r['candidate_stats'],r['base_stats'];lines46.append('|'+ '|'.join([p.replace('_','／'),str(r['n']),p30(a['win'])+'／'+p30(b['win']),p30(a['ev'])+'／'+p30(b['ev']),f"{r['mean_difference']*100:+.3f}",c30(r['CI48']),p30(a['worst_mdd'])+'／'+p30(b['worst_mdd']),str(r['mean_improvement'])])+'|')
lines46+=['## 若仍用原動能／法人排序，短交叉有沒有幫助？','|策略−原五日基準|後段配對n|候選／原勝率|候選／原期望|平均差pp|48日校正區間pp|覆蓋率|全面改善？|','|---|---:|---|---|---:|---|---:|---|']
for p in v46_pairs_ma:
    for fam in ['MOM','FLOW']:
        k=fam+'_'+p+'_ACTIVE';r=v46_stats['contrasts'][k+'_minus_'+fam+'_E0'];a,b=r['candidate_stats'],r['base_stats'];lines46.append('|'+ '|'.join([label_ma45(k),str(r['n']),p30(a['win'])+'／'+p30(b['win']),p30(a['ev'])+'／'+p30(b['ev']),f"{r['mean_difference']*100:+.3f}",c30(r['CI48']),p30(r['coverage']),str(r['full_improvement'])])+'|')
lines46+=['## 哪些日期無法形成配置？','|純短均線對原動能|候選可涵蓋原後段日期|未被涵蓋的原動能n|這些被排除日期原動能勝率／期望|','|---|---:|---:|---|']
for p in v46_pairs_ma:
    r=v46_stats['contrasts']['PURE_'+p+'_minus_MOM_E0'];b=r['excluded_base_later'];lines46.append('|'+ '|'.join([p.replace('_','／'),p30(r['coverage']),str(b['n']),p30(b.get('win'))+'／'+p30(b.get('ev'))])+'|')
lines46+=['「當時沒有足夠多頭候補」不是已經證明成功逃過下跌：官方仍有持股、現金等限制，而本批尚未定義這種情況的替代配置。因此不能把不可判定窗口刪去後直接宣稱可實戰。','## 擴大研究後，前批的正向證據有沒有變？','前批32項校正是當時固定範圍；本輪把舊32項及新42項一起作74項校正，24／48日區塊各20000次。數值路徑未重跑，但判斷必須採本輪較寬的比較範圍。','前批曾通過平均改善、這輪不再通過的配對：'+('；'.join(label_ma45(v45_stats['contrasts'][n]['candidate'])+'−'+label_ma45(v45_stats['contrasts'][n]['base']) for n in downgrade46) if downgrade46 else '無')+'。','|前批重點配對|平均差pp|本輪48日校正區間pp|本輪平均改善|風險不劣|全面改善|','|---|---:|---|---|---|---|']
for n in ['PURE_50_200_minus_MOM_E0','PURE_50_200_minus_FLOW_E0','FLOW_50_200_ACTIVE_minus_FLOW_E0']:
    r=v46_stats['contrasts'][n];lines46.append('|'+ '|'.join([label_ma45(r['candidate'])+'−'+label_ma45(r['base']),f"{r['mean_difference']*100:+.3f}",c30(r['CI48']),str(r['mean_improvement']),str(r['risk_not_worse']),str(r['full_improvement'])])+'|')
lines46+=['## 新增42項比較全部揭露','包括短均線對各自原基準、純短均線對兩個基準，以及三種短均線用法對50／200同類策略。舊32項的本輪校正結果全部在統計JSON，沒有只公布最高報酬。','|候選−對照|後段n|平均差pp|48日校正區間pp|平均改善|風險不劣|跨年通過|全面改善|','|---|---:|---:|---|---|---|---|---|']
for n,r in v46_stats['contrasts'].items():
    if r['candidate'] not in newkeys46:continue
    lines46.append('|'+ '|'.join([label_ma45(r['candidate'])+'−'+label_ma45(r['base']),str(r['n']),f"{r['mean_difference']*100:+.3f}",c30(r['CI48']),str(r['mean_improvement']),str(r['risk_not_worse']),str(r['year_gate']),str(r['full_improvement'])])+'|')
lines46+=['全面改善沿用：24及48日校正收益下界>0、P10和最差回撤不劣、至少80%覆蓋、100個校準配對、至少5個各20配對年份且三分之二增量正。2016～2018校準、2019～2025後段已反覆研究，不能稱真正未見樣本外。','74項校正仍不能消除所有過往探索、目前150檔名單的存活偏差或缺失公司行動；極尾端區間亦有蒙地卡羅誤差。正式Active Share、實際成交作廢等仍未完全驗證。','## 逐方法可判定數','|新增方法|全期有效／2236|後段有效|有模擬限制違反的窗口|','|---|---:|---:|---:|']
for k in [s['key'] for s in v46_specs]:
    m=v46_stats['coverage_detail'][k];lines46.append('|'+ '|'.join([label_ma45(k),str(m['valid'])+'/2236',str(m['later_valid']),str(m['violations_windows'])])+'|')
lines46+=['## 保存與重現','原Notebook371起保存事前定義、24個無股息直接價格SMA核對、18新方法批次、74項統計與圖表；舊22方法直接重用。既有kernel未重啟，沒有提交交易。','[完整74項統計與逐年](短均線_24日競賽全組統計_v46_2026-10-17.json) · [事先定義](短均線_24日競賽比較定義_v46_2026-10-17.json) · [回總結](策略研究_總結報告.html)。全部2236起點結果保存在18份「短均線_方法代碼_v46_2026-10-17.json.gz」，不可判定原因也保留。']
start_overview46=lines46.index('## 同日期先比較純選股：九方法完全相同窗口');start_paired46=lines46.index('## 每一種短均線，直接和純50／200配對');end_paired46=lines46.index('## 若仍用原動能／法人排序，短交叉有沒有幫助？');lines46=lines46[:start_overview46]+lines46[start_paired46:end_paired46]+lines46[start_overview46:start_paired46]+['原動能在這個交集也有87.30%勝率，不能把交集的高勝率解讀成均線新增的成功率；此圖只作同日期輔助比較，主要判斷用前表各自配對與覆蓋率。']+lines46[end_paired46:]
v46_report='\n'.join(lines46);v46_rp=v34_root/'短均線_24日競賽比較與全部窗口_v46_2026-10-17.md';v46_rp.write_text(v46_report,encoding='utf-8')
v46_fig=go.Figure();v46_views={};v46_titles={};v46_plot_specs=[]
for later in [False,True]:v46_plot_specs.append(('overview',v46_overview_mask,list(v46_overview_keys),later))
for p,mm in v46_pair_masks.items():
    for later in [False,True]:v46_plot_specs.append((p,mm,list(v46_stats['per_pair_common'][p]['later']),later))
ext46=[]
for p,mask,keys,later in v46_plot_specs:
    view=p+('_later' if later else '_all');mm=mask&(v30_later if later else np.ones(len(mask),bool));ix=np.flatnonzero(mm);ids=[];v46_titles[view]=('純均線總覽' if p=='overview' else p.replace('_','／')+'與50／200比較')+('／後段' if later else '／全期')+'／'+str(len(ix))+'同日期窗口'
    for k in keys:
        ys=v46_ret[k][mm]*100;ext46.extend([float(ys.min()),float(ys.max())]);label=label_ma45(k);ids.append(len(v46_fig.data));color='#167368' if k in ['MOM_E0','FLOW_E0','PURE_50_200'] else '#567bb5'
        v46_fig.add_trace(go.Box(name=label,y=ys,boxpoints='all',jitter=.45,pointpos=0,line={'color':color},fillcolor='rgba(255,255,255,0)',marker={'size':3,'opacity':.35,'color':color},customdata=[[v46_results[k][i]['start'],v46_results[k][i]['end'],v46_results[k][i]['cost']*100] for i in ix],hovertemplate='%{customdata[0]}→%{customdata[1]}<br>費後24日%{y:.2f}%<br>費稅%{customdata[2]:.2f}%<extra></extra>',visible=view=='overview_later'))
        ids.append(len(v46_fig.data));v46_fig.add_trace(go.Scatter(x=[label],y=[float(np.median(ys))],mode='markers',marker={'color':'#df4385','symbol':'line-ew','size':22,'line':{'width':3}},showlegend=False,hovertemplate='中位數%{y:.2f}%<extra></extra>',visible=view=='overview_later'))
    v46_views[view]=ids
v46_fig.add_hline(y=0,line_color='#222',line_width=2);v46_fig.update_layout(template='plotly_white',height=740,showlegend=False,title=v46_titles['overview_later'],yaxis={'title':'費後24日報酬 %','range':[min(ext46)-3,max(ext46)+3]},xaxis={'tickangle':-26},margin={'t':120,'b':175},updatemenus=[{'type':'dropdown','x':0,'y':1.18,'active':1,'buttons':[{'label':v46_titles[v],'method':'update','args':[{'visible':[i in ids for i in range(len(v46_fig.data))]},{'title':v46_titles[v]}]} for v,ids in v46_views.items()]}])
v46_hp=v34_root/'短均線_24日競賽比較與全部窗口_v46_2026-10-17.html';render46=MarkdownIt('commonmark',{'html':False}).enable('table').render(v46_report).replace('<table>','<div class="table-wrap"><table>').replace('</table>','</table></div>');v46_hp.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>短均線與24日競賽比較</title><style>body{font:16px/1.8 system-ui;max-width:1320px;margin:30px auto;padding:0 20px;color:#173333}table{width:100%;border-collapse:collapse;font-size:14px}td,th{padding:9px;border-bottom:1px solid #ddd;text-align:left}.table-wrap{overflow-x:auto}a{color:#167368}</style><a href="策略研究_總結報告.html">回總結</a>'+v46_fig.to_html(full_html=False,include_plotlyjs=True,config={'responsive':True,'displaylogo':False})+render46+'</html>',encoding='utf-8')
snippet46='<section id="short-ma-study"><h2>24日競賽：新增六組短均線</h2><p>3／10、5／10、5／15、10／20、5／24、10／24，18新配置、40,248次模擬；舊22方法重用。持有期仍24日，不因短期比賽就預設短均線更好。新增全面改善配對'+str(len(newfull46))+'項；純短均線相對50／200通過平均改善'+str(len(purebeat46))+'組。</p><p>本輪舊新合併74項校正，使用最新判斷；不足25檔的日期與被排除對照績效一併揭露。</p><a href="'+v46_hp.name+'">查看同日期勝率、期望、回撤及全部窗口點 →</a></section>'
sumhp46=v34_root/'策略研究_總結報告.html';page46=sumhp46.read_text(encoding='utf-8');assert 'short-ma-study' not in page46;page46=page46.replace('<section id="ma-cross-study">',snippet46+'<section id="ma-cross-study">');assert snippet46 in page46;sumhp46.write_text(page46,encoding='utf-8')
summd46=v34_root/'策略研究_總結報告.md';summd46.write_text(summd46.read_text(encoding='utf-8')+'\n\n## 短均線擴充（v46，2026-10-17）\n\n新增六對短SMA、18配置、40,248次窗口模擬；舊新74項合併校正。新增全面改善'+str(len(newfull46))+'項，純短均線相對50／200平均改善'+str(len(purebeat46))+'組。覆蓋不足、費用與被排除窗口均明列。[完整比較與箱形圖]('+v46_hp.name+')。\n',encoding='utf-8')
old46=v34_root/'均線交叉_選股退出與全部窗口比較_v45_2026-10-17.html';oldpage=old46.read_text(encoding='utf-8');old46.write_text(oldpage.replace('<a href="策略研究_總結報告.html">回總結</a>','<a href="策略研究_總結報告.html">回總結</a><p><strong>後續更新：</strong>已擴充六組短均線，舊新合併74項校正；本頁32項為前輪凍結結果。<a href="'+v46_hp.name+'">查看最新完整比較</a></p>',1),encoding='utf-8')
valid46={'new_methods':18,'new_window_runs':v46_newruns,'old_methods_reused':22,'contrasts':len(v46_contrast_pairs),'bootstrap_draws':20000,'direct_SMA_checks':v46_checks,'all_events_lagged':all(e['signal_date']<e['date'] for k in newkeys46 for r in v46_results[k] for e in r['events']),'overview_n':[v46_stats['overview']['all_n'],v46_stats['overview']['later_n']],'all_plot_points':sum(len(t.y) for t in v46_fig.data if t.type=='box'),'new_full_improvements':newfull46,'short_pure_beat50':purebeat46,'prior_mean_flags_downgraded':downgrade46,'browser_visual_qa':'programmatic only;notvisuallyinspected'}
assert v46_newruns==40248 and valid46['all_events_lagged'] and valid46['contrasts']==74
assert all(v46_results[k] is v45_results[k] for k in v45_results)
assert 'unified-longhistory-boxplot' in sumhp46.read_text(encoding='utf-8')
(v34_root/'短均線_24日競賽驗證_v46_2026-10-17.json').write_text(json.dumps(valid46,ensure_ascii=False,indent=2),encoding='utf-8');state=json.loads(sp46.read_text(encoding='utf-8'));state['short_ma_v46'].update(status='completed',running=False,current_method=None,report=v46_rp.name,html=v46_hp.name,validation=valid46);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8');print('V46_PUBLISHED',{'html':v46_hp.name,'validation':valid46})


In [ ]:
# Align exact10 annotated strategies; reuse paths, expand only retrospective market-breadth cache.
print('V47_RETAINED',{'python':sys.executable,'pid':os.getpid(),'prior_methods':len(v46_results),'source_windows':len(v30_starts)})
v47_keys=['MOM_E0','FLOW_E0','FLOW_50_200_ACTIVE','PURE_50_200']+['PURE_'+p for p in v46_pairs_ma];v47_core=v47_keys[:4]
v47_definition={'version':'v47_same_dates_annotated10','date':'2026-10-17','strategies':v47_keys,'source':'same2016–2025 snapshot andsame2236starts fromv45/v46;no newtradebacktest','primary_cohort':'intersectionofall10validmethods,exact same dates','secondary_cohort':'intersectionoffouroriginal/50/200strategies;retainsmorebadmarketwindows;notmixedwithprimary','unrestricted_diagnosis':'eachstrategyallvalidwindows;differentdenominators shown separately;notcrossranked','loss':'feesafter24session return<0;start/end ofwindows,notnecessarilyidenticaldailyPNL','overlap':'Jaccard bothloss/eitherloss;conditional pairloss;returncorrelation descriptive','market':'current150listedbeforestart,fixednames24session knowncash-adjustedgrossreturn;minimum80%ofstartlisted and50;reusev41whereexists','market_bad':'morethan50%stocksdown ANDequalweight24daymeanreturn<0;also reportmajoritydown and>=70%down','bear_limit':'broaddeclineinresearchpool notofficialTAIEXbearmarketlabel;currentuniversebias;posthoccannotusedforentry','conditional_comparison':'45pairsin10cohort +6pairsin4cohort=51fixedcomparisons;24/48blocks20000Bonferroni51;notnewunseenstrategyproof','selection':'observedrisk-returnandcoverage,do notdeclareglobalwinnerfromfilteredmean','all_points':'boxallcommonpoints,timelineall2236startswhiteinvalid,lossintersectionheatmap','excluded':'outsideintersectionnotzero,showmarketbadcoveragevsincluded'}
(v34_root/'統一策略_同日期比較與虧損定義_v47_2026-10-17.json').write_text(json.dumps(v47_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v47_markets={r['index']:dict(r['market']) for r in v41_rows};v47_reused=len(v47_markets)
for ix,s0 in enumerate(v30_starts):
    if ix in v47_markets:continue
    s=int(s0);rr=v30_arrays['returns'][s:s+24];known=np.all(np.isfinite(rr)&(rr>-1)&~v30_arrays['bad'][s:s+24],axis=0)&v30_pool&(v30_listed<=v30_days[s-1]);den=int(np.sum(v30_pool&(v30_listed<=v30_days[s-1])));n=int(known.sum());enough=n>=max(50,int(np.ceil(.8*den)));values=np.prod(1+rr[:,known],axis=0)-1 if n else np.array([])
    row={'known_n':n,'startlisted_n':den,'missing_n':den-n,'coverage_ok':bool(enough),'down_fraction':float(np.mean(values<0)) if n else None,'mean_return':float(values.mean()) if n else None}
    if enough:row.update(broad_down=row['down_fraction']>.5,severe_down=row['down_fraction']>=.7,confirmed_broad_down=row['down_fraction']>.5 and row['mean_return']<0)
    v47_markets[ix]=row
assert len(v47_markets)==2236
v47_cohorts={'ten':np.logical_and.reduce([v46_valid[k] for k in v47_keys]),'core4':np.logical_and.reduce([v46_valid[k] for k in v47_core])};v47_market_ok=np.array([v47_markets[i]['coverage_ok'] for i in range(2236)]);v47_broad=np.array([v47_markets[i].get('broad_down',False) for i in range(2236)]);v47_bad=np.array([v47_markets[i].get('confirmed_broad_down',False) for i in range(2236)])
def market_stats47(mask):
    ids=np.flatnonzero(mask&v47_market_ok)
    return {'windows':int(mask.sum()),'market_known':len(ids),'majority_down_n':int(np.sum(mask&v47_market_ok&v47_broad)),'confirmed_bad_n':int(np.sum(mask&v47_market_ok&v47_bad)),'majority_down_fraction':float(v47_broad[ids].mean()) if len(ids) else None,'confirmed_bad_fraction':float(v47_bad[ids].mean()) if len(ids) else None,'mean_down_fraction':float(np.mean([v47_markets[i]['down_fraction'] for i in ids])) if len(ids) else None,'mean_market_return':float(np.mean([v47_markets[i]['mean_return'] for i in ids])) if len(ids) else None,'severe_down_n':sum(v47_markets[i]['severe_down'] for i in ids)}
def losses47(key,mask):
    loss=mask&(v46_ret[key]<0);nonloss=mask&(v46_ret[key]>=0);bad=mask&v47_market_ok&v47_bad;notbad=mask&v47_market_ok&~v47_bad
    return {'n':int(mask.sum()),'loss_n':int(loss.sum()),'loss_market':market_stats47(loss),'nonloss_market':market_stats47(nonloss),'loss_rate_when_market_bad':float(np.mean(v46_ret[key][bad]<0)) if bad.any() else None,'market_bad_n':int(bad.sum()),'loss_rate_when_market_notbad':float(np.mean(v46_ret[key][notbad]<0)) if notbad.any() else None,'market_notbad_n':int(notbad.sum()),'years':{str(y):{'windows':int((mask&np.array([v30_days[int(s)].year==y for s in v30_starts])).sum()),'losses':int((loss&np.array([v30_days[int(s)].year==y for s in v30_starts])).sum()),'bad_market_losses':int((loss&v47_market_ok&v47_bad&np.array([v30_days[int(s)].year==y for s in v30_starts])).sum())} for y in range(2016,2026)}}
def overlap47(keys,mask):
    out={};ix=np.flatnonzero(mask)
    for a in keys:
        for b in keys:
            la=mask&(v46_ret[a]<0);lb=mask&(v46_ret[b]<0);both=int((la&lb).sum());union=int((la|lb).sum());out[a+'|'+b]={'n':len(ix),'a_losses':int(la.sum()),'b_losses':int(lb.sum()),'both_loss':both,'either_loss':union,'only_a':int((la&~lb).sum()),'only_b':int((lb&~la).sum()),'neither_loss':int((mask&~la&~lb).sum()),'jaccard':both/union if union else None,'p_b_loss_given_a_loss':both/int(la.sum()) if la.any() else None,'return_correlation':float(np.corrcoef(v46_ret[a][ix],v46_ret[b][ix])[0,1]) if len(ix)>1 else None}
    return out
v47_stats={'definition':v47_definition,'cohorts':{},'all_valid_diagnosis':{},'exclusion':{}}
for name,cohort in v47_cohorts.items():
    keys=v47_keys if name=='ten' else v47_core;v47_stats['cohorts'][name]={}
    for period,pm in [('all',np.ones(len(cohort),bool)),('later',v30_later)]:
        mask=cohort&pm;v47_stats['cohorts'][name][period]={'n':int(mask.sum()),'range':[str(v30_days[int(v30_starts[i])]) for i in [np.flatnonzero(mask)[0],np.flatnonzero(mask)[-1]]],'metrics':{k:metrics_short46(k,mask) for k in keys},'losses':{k:losses47(k,mask) for k in keys},'overlap':overlap47(keys,mask),'market':market_stats47(mask)}
for k in v47_keys:v47_stats['all_valid_diagnosis'][k]={p:losses47(k,v46_valid[k]&(v30_later if p=='later' else np.ones(2236,bool))) for p in ['all','later']}
for period,pm in [('all',np.ones(2236,bool)),('later',v30_later)]:
    reference=v46_valid['MOM_E0']&v46_valid['FLOW_E0']&pm;inside=reference&v47_cohorts['ten'];outside=reference&~v47_cohorts['ten'];v47_stats['exclusion'][period]={'reference_n':int(reference.sum()),'included':market_stats47(inside),'excluded':market_stats47(outside),'excluded_reference_metrics':{k:metrics_short46(k,outside) for k in ['MOM_E0','FLOW_E0']}}
v47_losses_matrix=np.stack([np.where(v46_valid[k],(v46_ret[k]<0).astype(float),np.nan) for k in v47_keys])
v47_window_rows=[{'index':i,'start':str(v30_days[int(s)]),'end':str(v30_days[int(s)+23]),'market':v47_markets[i],'ten_common':bool(v47_cohorts['ten'][i]),'core4_common':bool(v47_cohorts['core4'][i]),'methods':{k:{'valid':bool(v46_valid[k][i]),'net':float(v46_ret[k][i]) if v46_valid[k][i] else None,'loss':bool(v46_ret[k][i]<0) if v46_valid[k][i] else None,'reasons':v46_results[k][i]['reasons'],'violations':v46_results[k][i]['violations']} for k in v47_keys}} for i,s in enumerate(v30_starts)]
(v34_root/'統一策略_同日期虧損逐窗口_v47_2026-10-17.json').write_text(json.dumps(v47_window_rows,ensure_ascii=False,default=serial30,allow_nan=False),encoding='utf-8')
(v34_root/'統一策略_同日期虧損統計_v47_2026-10-17.json').write_text(json.dumps(v47_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['unified_loss_v47']={'status':'aligned_pending_intervals_and_report','running':False,'new_backtests':0,'cohorts':{n:{p:r['n'] for p,r in ss.items()} for n,ss in v47_stats['cohorts'].items()}};sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V47_ALIGNED',{'cohorts':{n:{p:{'n':r['n'],'market':r['market'],'metrics':r['metrics'],'losses':{k:{'loss_n':d['loss_n'],'market':d['loss_market']} for k,d in r['losses'].items()}} for p,r in ss.items()} for n,ss in v47_stats['cohorts'].items()},'exclusion':v47_stats['exclusion'],'market_reused':v47_reused})


In [ ]:

from itertools import combinations
v47_names=dict(zip(v47_keys,['原動能各半／五日','純法人／五日','法人＋50／200交叉換股','純50／200均線強度','純3／10','純5／10','純5／15','純10／20','純5／24','純10／24']))
v47_pairs=[(c,a,b) for c,ks in [('ten',v47_keys),('core4',v47_core)] for a,b in combinations(ks,2)]
assert len(v47_pairs)==51
v47_deltas=np.zeros((2236,51));v47_counts=np.zeros_like(v47_deltas)
for j,(c,a,b) in enumerate(v47_pairs):
    m=v47_cohorts[c]&v30_later;v47_deltas[m,j]=v46_ret[a][m]-v46_ret[b][m];v47_counts[m,j]=1
v47_intervals={};rng47=np.random.default_rng(2026101747)
for block in [24,48]:
    def block_sums47(x):
        z=np.vstack([x,x[:block]]);cs=np.vstack([np.zeros((1,51)),np.cumsum(z,axis=0)]);return cs[np.arange(2236)+block]-cs[np.arange(2236)]
    ss=block_sums47(v47_deltas);nn=block_sums47(v47_counts);draws=[]
    for batch in range(100):
        starts=rng47.integers(0,2236,size=(200,int(np.ceil(2236/block))));den=nn[starts].sum(axis=1);num=ss[starts].sum(axis=1);draws.append(np.divide(num,den,out=np.full_like(num,np.nan),where=den>0))
    dist=np.vstack(draws);q=.05/(2*51);v47_intervals[str(block)]=np.nanquantile(dist,[q,1-q],axis=0).T.tolist()
v47_stats['paired_intervals']=[{'cohort':c,'a':a,'b':b,'difference':float(v47_deltas[:,j].sum()/v47_counts[:,j].sum()),'block24':v47_intervals['24'][j],'block48':v47_intervals['48'][j]} for j,(c,a,b) in enumerate(v47_pairs)]
v47_stats['interval_method']={'draws':20000,'blocks':[24,48],'family':51,'method':'circular original-position block bootstrap; Bonferroni 51; later period only; overlapping windows retained'}
def pct47(x):return '不可判定' if x is None else f'{x*100:.2f}%'
def table47(headers,rows):return '<table><thead><tr>'+''.join('<th>'+str(h)+'</th>' for h in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+str(x)+'</td>' for x in row)+'</tr>' for row in rows)+'</tbody></table>'
v47_parts=['<h1>十種策略同日期比較與虧損時間分析</h1><p>2026-10-17｜沿用2016～2025資料快照；不重跑交易模擬。費後24交易日報酬；同股票池、90%目標投入、整千股及原成交與費用模型。均線以當時已知資料形成訊號，死亡交叉換股在下一可執行日處理。</p>', '<h2>先看結論</h2><p>原先兩張表的有效日期不同，不能直接以表上的勝率排優劣。現在分成「十種策略共同日期」及「四種主要策略較大共同日期」。短均線常因正向訊號股票不足25檔等原因無法成立；這些日期保留為不可判定，不能當作避開虧損。</p><p>十種策略共同樣本偏向上漲環境。2019～2025共有370個共同窗口，市場廣泛下跌只有10%；相對於動能與法人都有效、但被十策略交集排除的671個窗口，廣泛下跌占31.74%。因此這裡約七至九成勝率不能代表所有行情。</p>']
v47_figs=[];v47_y=[]
for c,ks,title in [('ten',v47_keys,'十種策略：完整公平對比'),('core4',v47_core,'四種主要策略：較完整風險對比')]:
    v47_parts.append('<h2>'+title+'</h2>')
    for period,pt in [('later','2019～2025主要比較'),('all','全部可用歷史（含早期探索）')]:
        rr=v47_stats['cohorts'][c][period];v47_parts.append(f'<h3>{pt}：{rr["n"]}個共同窗口</h3><p>實際起點範圍：{rr["range"][0]}～{rr["range"][1]}，中間有缺口；不等於每年完整覆蓋。</p>')
        v47_parts.append(table47(['策略','窗口','勝率','費後24日期望','中位數','P10','最差窗口內回撤','平均費用／初始資金'],[[v47_names[k],rr['n'],*[pct47(rr['metrics'][k][f]) for f in ['win','ev','median','p10','worst_mdd','fee']]] for k in ks]))
        mask=v47_cohorts[c]&(v30_later if period=='later' else np.ones(2236,bool));fig=go.Figure()
        for k in ks:
            vals=v46_ret[k][mask]*100;v47_y.extend(vals.tolist());dates=[v47_window_rows[i]['start']+'～'+v47_window_rows[i]['end'] for i in np.flatnonzero(mask)]
            fig.add_trace(go.Box(y=vals,name=v47_names[k],boxpoints='all',jitter=.35,pointpos=0,marker={'size':3,'opacity':.45},text=dates,hovertemplate='%{text}<br>費後24日：%{y:.2f}%<extra>%{fullData.name}</extra>'))
            fig.add_trace(go.Scatter(x=[v47_names[k]],y=[float(np.median(vals))],mode='markers',marker={'color':'#d81b60','symbol':'line-ew','size':25,'line':{'width':3,'color':'#d81b60'}},name='中位數',showlegend=False,hovertemplate='中位數 %{y:.2f}%<extra></extra>'))
        fig.update_layout(title=pt+'｜'+title,height=570,yaxis_title='費後24日報酬（%）',showlegend=False,margin={'b':130});fig.add_hline(y=0,line_color='#222',line_width=2);v47_figs.append(fig);v47_parts.append(('FIG47',len(v47_figs)-1))
v47_parts.append('<h2>哪個比較好？先區分收益、勝率與尾端風險</h2><p>十策略370窗口中，10／20與10／24的費後期望約6.09%、6.04%，50／200約5.94%，原動能5.61%；這是樣本內排序，不能單憑差距宣稱改良。法人＋50／200的勝率最高約91.35%，但期望4.07%，與最高期望是不同目標。3／10、5／10平均費用約1.7%，且期望及勝率較低，沒有顯示短線頻繁換手比較好。</p><p>四策略944窗口更能看到壞行情：純50／200期望最高3.63%、勝率72.78%，但最差窗口內回撤−32.59%；原動能期望2.47%、最差回撤−27.65%。提高平均報酬不等於改善極端風險。這批為已看過歷史的探索比較，尚未取得新的未見樣本證明。</p>')
cirows=[]
for d in v47_stats['paired_intervals']:
    if d['cohort']=='core4' or (d['a']=='MOM_E0' and d['b'] in ['PURE_10_20','PURE_10_24','PURE_3_10']):
        cirows.append([d['cohort'],v47_names[d['a']]+' 減 '+v47_names[d['b']],pct47(d['difference']),*[f'{pct47(d[b][0])}～{pct47(d[b][1])}' for b in ['block24','block48']]])
v47_parts.append('<h3>配對期望差與區間</h3><p>以下差值單位為百分點。依原始日曆位置做24／48日循環區塊重抽樣，各20,000次，51組比較Bonferroni家族校正；不是把重疊窗口當獨立樣本。這是本次固定家族的探索區間，不能抵銷先前全部策略搜尋。全部51組見統計JSON。</p>'+table47(['共同樣本','策略相減','平均差','24日區塊校正區間','48日區塊校正區間'],cirows))
v47_parts.append('<h2>虧損是否發生在相同時間？</h2><p>以四策略2019～2025的944個共同窗口比較。兩種策略同時虧損的比例，與「只要任一虧損時，兩者同虧」分母不同，以下明列。窗口互相重疊，數量不是獨立熊市事件數。</p>')
rr=v47_stats['cohorts']['core4']['later'];v47_parts.append(table47(['策略配對','同虧窗口','任一虧損窗口','虧損交集／聯集','前者虧損時後者也虧','報酬相關'],[[v47_names[a]+' × '+v47_names[b],d['both_loss'],d['either_loss'],pct47(d['jaccard']),pct47(d['p_b_loss_given_a_loss']),f'{d["return_correlation"]:.3f}'] for a,b in combinations(v47_core,2) for d in [rr['overlap'][a+'|'+b]]]))
v47_parts.append('<h2>是不是當時普遍不好？</h2><p>市場範圍是目前150檔名單中，當時已上市櫃且24日價格可核對的股票，並非全台股。至少80%應有股票且不少於50檔可核對才判斷。廣泛下跌定義：超過半數股票24日報酬為負，且股票池等權24日平均報酬為負。這是事後共現，不能證明熊市因果或當時可以預知。</p>')
for c,title in [('core4','四策略944窗口'),('ten','十策略370窗口')]:
    rr=v47_stats['cohorts'][c]['later'];rows=[]
    for k,d in rr['losses'].items():
        lm=d['loss_market'];rows.append([v47_names[k],d['loss_n'],lm['market_known'],lm['confirmed_bad_n'],pct47(lm['confirmed_bad_fraction']),pct47(lm['majority_down_fraction']),pct47(lm['mean_down_fraction'])])
    v47_parts.append('<h3>'+title+'</h3>'+table47(['策略','虧損窗口','市場可判定','同時廣泛下跌','占可判定虧損窗口','超半數股票下跌占比','虧損時平均下跌股票比例'],rows))
v47_parts.append('<p>四策略中，65%～77%的虧損窗口與廣泛下跌共現，因此環境是重要因素；仍有23%～35%未符合該定義，不能全歸咎熊市。動能與法人在944窗口同虧269次；動能虧損時，法人也虧的比例84.06%，兩者並非充分的彼此避險。</p>')
fig=go.Figure(go.Heatmap(z=[[v47_stats['cohorts']['ten']['later']['overlap'][a+'|'+b]['jaccard'] for b in v47_keys] for a in v47_keys],x=[v47_names[k] for k in v47_keys],y=[v47_names[k] for k in v47_keys],zmin=0,zmax=1,colorscale='Blues',hovertemplate='%{y} × %{x}<br>虧損交集／聯集 %{z:.1%}<extra></extra>'));fig.update_layout(title='十策略370共同窗口：虧損重疊',height=650,margin={'l':200,'b':170});v47_figs.append(fig);v47_parts.append(('FIG47',len(v47_figs)-1))
z=[];cd=[]
for k in v47_keys:
    z.append([(-1 if v46_ret[k][i]<0 else 1) if v46_valid[k][i] else None for i in range(2236)])
    cd.append([[r['end'],pct47(r['methods'][k]['net']),str(r['methods'][k]['reasons']) if not r['methods'][k]['valid'] else '有效',pct47(r['market']['down_fraction'])] for r in v47_window_rows])
z.append([-1 if v47_bad[i] else (1 if v47_markets[i]['mean_return'] is not None and v47_markets[i]['mean_return']>0 else 0) if v47_market_ok[i] else None for i in range(2236)])
cd.append([[r['end'],pct47(r['market']['mean_return']),'股票池環境',pct47(r['market']['down_fraction'])] for r in v47_window_rows])
fig=go.Figure(go.Heatmap(z=z,x=[r['start'] for r in v47_window_rows],y=[v47_names[k] for k in v47_keys]+['同期150檔歷史子集'],customdata=cd,zmin=-1,zmax=1,colorscale=[[0,'#d55e5e'],[.49,'#d55e5e'],[.5,'#eadf9b'],[.51,'#59a88f'],[1,'#59a88f']],showscale=False,hoverongaps=False,hovertemplate='%{y}<br>起點 %{x}<br>終點 %{customdata[0]}<br>費後／股票池報酬 %{customdata[1]}<br>%{customdata[2]}<br>股票池下跌比例 %{customdata[3]}<extra></extra>'));fig.update_layout(title='全部2236起點：紅＝虧損／廣泛下跌；綠＝正報酬；黃＝混合；白＝不可判定',height=650,plot_bgcolor='white',margin={'l':200},xaxis={'rangeslider':{'visible':True}});v47_figs.append(fig);v47_parts.append(('FIG47',len(v47_figs)-1))
v47_cases=[]
mask=v47_cohorts['core4']&v30_later;severity=np.mean(np.stack([v46_ret[k] for k in v47_core]),axis=0);chosen=[]
for i in np.flatnonzero(mask)[np.argsort(severity[mask])]:
    if all(abs(int(i)-j)>=24 for j in chosen):chosen.append(int(i))
    if len(chosen)==6:break
for i in chosen:
    r=v47_window_rows[i];v47_cases.append([r['start']+'～'+r['end'],*[pct47(r['methods'][k]['net']) for k in v47_core],pct47(r['market']['down_fraction']),pct47(r['market']['mean_return'])])
v47_parts.append('<h3>較差時段實例</h3><p>依四策略平均報酬由差至好，起點相隔至少24交易日取6例，為事後描述，並非六個獨立事件或風險訊號。</p>'+table47(['起訖',*[v47_names[k] for k in v47_core],'股票池下跌比例','股票池等權報酬'],v47_cases))
v47_parts.append('<h2>資料與使用限制</h2><p>2016～2025為資料範圍，50／200需要足夠暖機，且有效日期不是完整十年。150檔仍為目前名單的歷史子集，有存活及名單選擇偏差；市場判斷不是官方大盤指數熊市判定。缺價、公司行動未知、候選不足或操作限制不合格均標不可判定，沒有補造報酬。正式Active Share及官賽成交作廢規則尚未完整驗證。平均費用以初始資金為分母；最差回撤為24日窗口內的最差值，不是連續十年組合回撤。</p><p>期望值＝費後24日平均報酬；勝率＝費後報酬大於0的比例。不同共同樣本只能分開解讀。下一步應用在新資料的前瞻驗證，不能依本圖事後挑最漂亮均線。</p>')
ylim=[float(np.floor(min(v47_y)/5)*5),float(np.ceil(max(v47_y)/5)*5)]
for f in v47_figs[:4]:f.update_yaxes(range=ylim)
body='';first=True
for x in v47_parts:
    if isinstance(x,tuple):body+=v47_figs[x[1]].to_html(full_html=False,include_plotlyjs=True if first else False);first=False
    else:body+=x
v47_html=v34_root/'統一策略_同日期比較與虧損環境_v47_2026-10-17.html';v47_md=v47_html.with_suffix('.md')
v47_html.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>十種策略同日期比較</title><style>body{font:16px/1.75 system-ui;max-width:1500px;margin:30px auto;padding:0 24px;color:#233043}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}td,th{padding:9px;border:1px solid #ccd5df;text-align:right}td:first-child,th:first-child{text-align:left}th{background:#eef3f8}h2{margin-top:45px}p{max-width:1200px}</style>'+body+'</html>',encoding='utf-8')
v47_md.write_text('# 十種策略同日期比較與虧損環境\n\n'+ '\n\n'.join(x for x in v47_parts if isinstance(x,str))+'\n\n互動圖表：['+v47_html.name+']('+v47_html.name+')',encoding='utf-8')
(v34_root/'統一策略_同日期虧損統計_v47_2026-10-17.json').write_text(json.dumps(v47_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
notice='<section id="unified-loss-study"><h2>十種策略已改用相同日期比較</h2><p>十策略共同370窗口偏向上漲，不能以此代表完整歷史勝率。四主要策略共同944窗口中，動能與法人同虧269次，動能虧損時84.06%法人也虧；各策略65%～77%的虧損與股票池廣泛下跌共現。純50／200平均收益較高，但尾端回撤更差，尚不能稱全面較好。</p><p><a href="'+v47_html.name+'">查看同日期箱形圖、全部起點虧損時間圖與重疊統計</a></p></section>'
master=v34_root/'策略研究_總結報告.html';txt=master.read_text(encoding='utf-8');assert 'id="unified-loss-study"' not in txt
anchor='<section id="short-ma-study">';assert anchor in txt;master.write_text(txt.replace(anchor,notice+anchor,1),encoding='utf-8')
mastermd=v34_root/'策略研究_總結報告.md';txt=mastermd.read_text(encoding='utf-8');mastermd.write_text(txt+'\n\n## 同日期比較與虧損環境 v47\n\n十種共同370窗口偏多；四主要策略944窗口中，65%～77%的虧損與廣泛下跌共現。原動能與法人同虧269窗口；不能將重疊窗口視為獨立熊市。\n\n詳見 ['+v47_html.name+']('+v47_html.name+')。\n',encoding='utf-8')
for pattern in ['短均線_24日競賽比較與全部窗口_v46*.html','均線交叉_選股退出與全部窗口比較_v45*.html','動能法人_同期整體下跌分析_v41*.html']:
    for fp in v34_root.glob(pattern):
        txt=fp.read_text(encoding='utf-8');pos=txt.find('<body');end=txt.find('>',pos)+1
        if pos>=0 and 'id="unified-loss-study"' not in txt:fp.write_text(txt[:end]+notice+txt[end:],encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['unified_loss_v47'].update({'status':'completed','running':False,'report':str(v47_html),'paired_family':51,'bootstrap_draws':20000,'new_backtests':0,'cohort_bias_disclosed':True,'not_official_bear_market':True});sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V47_COMPLETE',{'ten_later':370,'core4_later':944,'plots':len(v47_figs),'paired_family':51,'report':v47_html.name,'cases':v47_cases,'selected_intervals':[d for d in v47_stats['paired_intervals'] if d['cohort']=='core4']})


In [ ]:

print('V48_KERNEL',{'python':sys.executable,'pid':os.getpid(),'shape':v30_arrays['close'].shape,'methods':len(v46_results),'last_study':v47_stats['cohorts']['ten']['later']['n']})
v48_keys=v47_core+['PURE_10_20','PURE_10_24'];v48_common=np.logical_and.reduce([v46_valid[k] for k in v48_keys]);assert int((v48_common&v30_later).sum())==721
v48_states=np.full(len(v30_days),'unknown',dtype=object);v48_features=[]
for q in range(len(v30_days)):
    out={'date':str(v30_days[q]),'state':'unknown','trend':None,'breadth':None,'vol':None,'eligible_n':0,'listed_n':0}
    if q>=59:
        eligible=v30_pool&(v30_listed<=v30_days[q-19]);rr=v30_arrays['returns'][q-18:q+1];known=eligible&np.all(np.isfinite(rr)&(rr>-1)&~v30_arrays['bad'][q-18:q+1],axis=0);n=int(known.sum());den=int((v30_pool&(v30_listed<=v30_days[q])).sum());out.update(eligible_n=n,listed_n=den)
        if n>=max(50,int(np.ceil(.8*den))) and np.all(np.isfinite(v30_proxy[q-58:q+1])):
            marketlevels=np.r_[1.,np.cumprod(1+v30_proxy[q-58:q+1])];trend=float(marketlevels[-1]/marketlevels.mean()-1)
            levels=np.vstack([np.ones(len(v30_ids)),np.cumprod(1+rr,axis=0)]);breadth=float(np.mean(levels[-1,known]>np.mean(levels[:,known],axis=0)));vol=float(np.std(v30_proxy[q-19:q+1],ddof=1)) if np.all(np.isfinite(v30_proxy[q-19:q+1])) else None
            state='strong' if trend>0 and breadth>.5 else 'weak' if trend<=0 and breadth<=.5 else 'mixed';v48_states[q]=state;out.update(state=state,trend=trend,breadth=breadth,vol=vol)
    v48_features.append(out)
v48_startstates=v48_states[np.array(v30_starts)-1]
v48_stats={'status':'prepared','keys':v48_keys,'common':{p:{'n':int((v48_common&m).sum()),'metrics':{k:metrics_short46(k,v48_common&m) for k in v48_keys}} for p,m in [('all',np.ones(2236,bool)),('later',v30_later)]},'environments':{},'coverage':{},'source_hashes':v34_plan['common']['source_hashes'],'limits':'retrospective rolling-origin, current150 historical subset, not unseen, unknown actions excluded; no data reload or kernel restart'}
for st in ['strong','mixed','weak','unknown']:
    m=v48_common&v30_later&(v48_startstates==st);v48_stats['environments'][st]={'n':int(m.sum()),'metrics':{k:metrics_short46(k,m) for k in v48_keys} if m.any() else {}}
ref=v47_cohorts['core4']&v30_later
for st in ['strong','mixed','weak','unknown']:
    m=ref&(v48_startstates==st);v48_stats['coverage'][st]={'reference_n':int(m.sum()),'methods':{k:int((m&v46_valid[k]).sum()) for k in v48_keys}}
v48_plan={'definition':'v48_fixed_market_state_1','plan_file':'策略研究_市場狀態與策略選擇計畫_v48_2026-10-17.md','keys':v48_keys,'state':'prior-close synthetic equalweight60SMA trend and individual20SMA breadth>50%; unknown if coverage<80% or<50names','timing':'prior close only; each training outcome ends before testyear; annual expanding training,2019..2025','selectors':['S1_MOM_FLOW','S2_six'],'minimum_training':'2years,6nonoverlap24daywindows,90%candidatecoverage','gate':'both24/48block Bonferroni intervals strictlypositive vsMOM; tie lower fees; fallbackMOM','family':{'training_tests':105,'evaluation_tests':8},'bootstrap_draws':20000,'investment':.9,'review_days':5,'fallback':'prior-known candidate shortage => MOM; data/action error remainsunknown, not retrospectivefallback','engine_parent':v45_engine_hash,'newlookbacksearch':False}
v48_hash=hashlib.sha256(json.dumps(v48_plan,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
for name,obj in [('市場狀態策略選擇_定義_v48_2026-10-17.json',v48_plan),('市場狀態策略選擇_事前特徵_v48_2026-10-17.json',v48_features),('市場狀態策略選擇_統計_v48_2026-10-17.json',v48_stats)]: (v34_root/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['market_selection_v48']={'status':'A_B_C_done_pending_walkforward','running':False,'definition_hash':v48_hash,'nodes':{'48A':'completed','48B':'completed','48C':'completed','48D':'pending','48E':'pending','48F':'pending'}};sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V48_PREPARED',{'common':{p:x['n'] for p,x in v48_stats['common'].items()},'coverage':v48_stats['coverage'],'environments':v48_stats['environments']})
print('V48_ENGINE_SOURCE',v45_code)


In [ ]:

v48_train=[];v48_startyears=np.array([v30_days[int(s)].year for s in v30_starts]);v48_endyears=np.array([v30_days[int(s)+23].year for s in v30_starts]);v48_trainmatrix=[];v48_countmatrix=[]
for year in range(2019,2026):
    for st in ['strong','mixed','weak']:
        base=(v48_endyears<year)&(v48_startstates==st)&v46_valid['MOM_E0']
        for k in v48_keys[1:]:
            paired=base&v46_valid[k];inds=np.flatnonzero(paired);nonover=[]
            for i in inds:
                if not nonover or i-nonover[-1]>=24:nonover.append(int(i))
            years=sorted(set(v48_startyears[paired].tolist()));coverage=float(paired.sum()/base.sum()) if base.any() else 0.;eligible=bool(len(years)>=2 and len(nonover)>=6 and coverage>=.9)
            d={'year':year,'state':st,'key':k,'reference_n':int(base.sum()),'n':int(paired.sum()),'coverage':coverage,'training_years':years,'nonoverlap':len(nonover),'eligible':eligible,'ev_delta':float(np.mean(v46_ret[k][paired]-v46_ret['MOM_E0'][paired])) if paired.any() else None,'fee':float(np.mean([v46_results[k][i]['cost'] for i in inds])) if len(inds) else None}
            v48_train.append(d);v48_trainmatrix.append(np.where(paired,np.nan_to_num(v46_ret[k]-v46_ret['MOM_E0']),0.));v48_countmatrix.append(paired.astype(float))
assert len(v48_train)==105
v48_td=np.stack(v48_trainmatrix,axis=1);v48_tn=np.stack(v48_countmatrix,axis=1);rng48=np.random.default_rng(2026101748)
def bootstrap48(deltas,counts,blocks,family,draws=20000):
    n,p=deltas.shape;answer={};rng=np.random.default_rng(48000+family)
    for block in blocks:
        def sums(x):
            cs=np.vstack([np.zeros((1,p)),np.cumsum(np.vstack([x,x[:block]]),axis=0)]);return cs[np.arange(n)+block]-cs[np.arange(n)]
        ds=sums(deltas);ns=sums(counts);chunks=[]
        for j in range(draws//100):
            z=rng.integers(0,n,(100,int(np.ceil(n/block))));den=ns[z].sum(axis=1);num=ds[z].sum(axis=1);chunks.append(np.divide(num,den,out=np.full_like(num,np.nan),where=den>0))
        quant=np.nanquantile(np.vstack(chunks),[.05/(2*family),1-.05/(2*family)],axis=0).T;answer[str(block)]=quant.tolist()
    return answer
v48_tci=bootstrap48(v48_td,v48_tn,[24,48],105)
for j,d in enumerate(v48_train):
    d.update(ci24=v48_tci['24'][j],ci48=v48_tci['48'][j]);d['passes']=bool(d['eligible'] and min(d['ci24'][0],d['ci48'][0])>0)
v48_maps={}
for year in range(2019,2026):
    v48_maps[year]={}
    for selector in ['S1','S2']:
        v48_maps[year][selector]={}
        for st in ['strong','mixed','weak','unknown']:
            candidates=[d for d in v48_train if d['year']==year and d['state']==st and d['passes'] and (selector=='S2' or d['key']=='FLOW_E0')];candidates.sort(key=lambda d:(-d['ev_delta'],d['fee'],d['key']));v48_maps[year][selector][st]=candidates[0]['key'] if candidates else 'MOM_E0'
v48_stats['training']=v48_train;v48_stats['annual_maps']=v48_maps
v48_stats['environments_core4']={}
for st in ['strong','mixed','weak','unknown']:
    m=v47_cohorts['core4']&v30_later&(v48_startstates==st);v48_stats['environments_core4'][st]={'n':int(m.sum()),'metrics':{k:metrics_short46(k,m) for k in v47_core} if m.any() else {}}
(v34_root/'市場狀態策略選擇_逐年訓練_v48_2026-10-17.json').write_text(json.dumps({'definition_hash':v48_hash,'training':v48_train,'annual_maps':v48_maps},ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
(v34_root/'市場狀態策略選擇_統計_v48_2026-10-17.json').write_text(json.dumps(v48_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['market_selection_v48'].update(status='training_frozen_pending_simulation',running=False);state['market_selection_v48']['nodes']['48D']='training_completed_simulation_pending';sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V48_TRAINED',{'passing_tests':sum(d['passes'] for d in v48_train),'maps':v48_maps,'core4_environment':{st:{'n':r['n'],'ev':{k:x['ev'] for k,x in r['metrics'].items()}} for st,r in v48_stats['environments_core4'].items()}})


In [ ]:

# No policy passed the prespecified training gates. Prove identity instead of rerunning identical trades.
v48_policy_audit=[]
for selector in ['S1','S2']:
    for mode in ['START','FIVE']:
        n=0
        for i,s0 in enumerate(v30_starts):
            s=int(s0)
            if v30_days[s].year<2019:continue
            for offset in ([0] if mode=='START' else [0,5,10,15,20]):
                q=s+offset-1;year=v30_days[s+offset].year;selected=v48_maps[year][selector][v48_states[q]];assert selected=='MOM_E0';n+=1
        v48_policy_audit.append({'selector':selector,'mode':mode,'decision_checks':n,'all_selected':'MOM_E0','switches':0,'new_trade_runs':0,'reuse':'v46_results[MOM_E0], exact strategy identity, not path concatenation'})
v48_policykeys=['S1_START','S2_START','S1_FIVE','S2_FIVE'];v48_results={k:v46_results['MOM_E0'] for k in v48_policykeys};v48_valid={k:v46_valid['MOM_E0'] for k in v48_policykeys}
v48_evalpairs=[(k,b) for k in v48_policykeys for b in ['MOM_E0','FLOW_E0']];ds=[];ns=[]
for k,b in v48_evalpairs:
    m=v30_later&v46_valid['MOM_E0']&v46_valid[b];ds.append(np.where(m,np.nan_to_num(v46_ret['MOM_E0']-v46_ret[b]),0));ns.append(m.astype(float))
v48_eci=bootstrap48(np.stack(ds,axis=1),np.stack(ns,axis=1),[24,48],8)
v48_evals=[]
for j,(k,b) in enumerate(v48_evalpairs):
    m=ns[j].astype(bool);delta=v46_ret['MOM_E0'][m]-v46_ret[b][m];byyear={str(y):float(np.mean(v46_ret['MOM_E0'][m&(v48_startyears==y)]-v46_ret[b][m&(v48_startyears==y)])) for y in range(2019,2026) if np.any(m&(v48_startyears==y))};leave={str(y):float(np.mean(v46_ret['MOM_E0'][m&(v48_startyears!=y)]-v46_ret[b][m&(v48_startyears!=y)])) for y in range(2019,2026) if np.any(m&(v48_startyears!=y))}
    coverage=float(m.sum()/(v30_later&v46_valid[b]).sum());passes=bool(min(v48_eci['24'][j][0],v48_eci['48'][j][0])>0 and sum(x>=0 for x in byyear.values())>=4 and min(leave.values())>0 and coverage>=.9)
    v48_evals.append({'policy':k,'baseline':b,'n':int(m.sum()),'coverage':coverage,'ev_delta':float(np.mean(delta)),'ci24':v48_eci['24'][j],'ci48':v48_eci['48'][j],'year_differences':byyear,'leave_one_year_out':leave,'passes':passes})
v48_failure_counts={'not_enough_years':sum(len(d['training_years'])<2 for d in v48_train),'not_enough_nonoverlap':sum(d['nonoverlap']<6 for d in v48_train),'coverage_under90':sum(d['coverage']<.9 for d in v48_train),'not_positive_both_ci':sum(min(d['ci24'][0],d['ci48'][0])<=0 for d in v48_train),'eligible':sum(d['eligible'] for d in v48_train),'passed':sum(d['passes'] for d in v48_train)}
v48_stats.update(status='completed_no_incremental_evidence',policy_audit=v48_policy_audit,evaluations=v48_evals,training_failure_counts=v48_failure_counts,policy_metrics=metrics_short46('MOM_E0',v30_later&v46_valid['MOM_E0']),new_trade_runs=0,conclusion='No alternative passed training gates. Both selectors at START/FIVE identically retain original MOM; not proof that every conceivable regime policy fails.')
v48_stats['annual']={str(y):{'n':int((v30_later&v46_valid['MOM_E0']&(v48_startyears==y)).sum()),'metrics':metrics_short46('MOM_E0',v30_later&v46_valid['MOM_E0']&(v48_startyears==y))} for y in range(2019,2026)}
v48_selection_rows=[{'index':i,'start':r['start'],'end':r['end'],'prior_state':str(v48_startstates[i]),'year_map':v48_maps.get(v48_startyears[i]),'reference_valid':bool(v46_valid['MOM_E0'][i]),'six_common':bool(v48_common[i])} for i,r in enumerate(v47_window_rows)]
for name,obj in [('市場狀態策略選擇_統計_v48_2026-10-17.json',v48_stats),('市場狀態策略選擇_逐窗口決策_v48_2026-10-17.json',v48_selection_rows)]: (v34_root/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
labels48={'strong':'強勢環境','mixed':'混合環境','weak':'弱勢環境','unknown':'未知'};parts48=[];figs48=[]
parts48.append('<h1>市場狀態能幫助選策略嗎？</h1><p>v48｜2026-10-17｜2016～2025同一資料快照；逐年向前驗證2019～2025。此段歷史先前已被研究，因此是回溯式驗證，不是新未見樣本。</p><h2>先看結論</h2><p><strong>本批尚未證實依市場狀態切換，能改善原動能。</strong>先前設定兩種選擇器：只在動能／法人間選，以及六策略間選。105組逐年訓練檢查無一通過完整門檻；每年每種環境都回到原動能。因此「起點選策略」及「每五日切換」四版本均與原動能完全相同，沒有實際切換、沒有新增交易，也沒有假造改良。</p><p>這是依事先門檻不採用替代策略的結果，不是另外四種實際切換策略已被充分測完，也不是所有市場擇時都無效。沒有看到結果後放寬門檻。</p>')
parts48.append('<h2>一、先把六種策略放到同日期</h2><p>共同有效：全期802窗口，2019～2025共721窗口。勝率＝費後24日報酬&gt;0的比例；期望＝費後24日平均報酬。以下只可在相同窗口橫比。投資目標90%、整千股、費用及原公司行動處理一致。</p>')
for period,title in [('later','2019～2025：721共同窗口'),('all','全部可用歷史：802共同窗口')]:
    rr=v48_stats['common'][period];parts48.append('<h3>'+title+'</h3>'+table47(['策略','勝率','費後24日期望','中位數','P10','最差窗口內回撤','平均費用'],[[v47_names[k],*[pct47(rr['metrics'][k][f]) for f in ['win','ev','median','p10','worst_mdd','fee']]] for k in v48_keys]))
mask=v48_common&v30_later;fig=go.Figure();values48=[]
for k in v48_keys:
    ys=v46_ret[k][mask]*100;values48.extend(ys.tolist());dates=[v47_window_rows[i]['start']+'～'+v47_window_rows[i]['end'] for i in np.flatnonzero(mask)]
    fig.add_trace(go.Box(y=ys,name=v47_names[k],boxpoints='all',jitter=.35,pointpos=0,marker={'size':3,'opacity':.4},text=dates,hovertemplate='%{text}<br>費後24日 %{y:.2f}%<extra>%{fullData.name}</extra>'))
    fig.add_trace(go.Scatter(x=[v47_names[k]],y=[float(np.median(ys))],mode='markers',marker={'color':'#d81b60','symbol':'line-ew','size':25,'line':{'color':'#d81b60','width':3}},showlegend=False,hovertemplate='中位數 %{y:.2f}%<extra></extra>'))
fig.update_layout(title='六策略721共同窗口｜全部點；粉紅＝中位數',height=580,showlegend=False,yaxis_title='費後24日報酬（%）');fig.add_hline(y=0,line_color='#222',line_width=2);figs48.append(fig);parts48.append(('fig',0))
parts48.append('<h2>二、先前的市場狀態，與未來下跌是兩件事</h2><p>交易前一日以股票池等權報酬序列的60日均線及個股20日均線廣度標記：兩者均正向＝強勢、均負向＝弱勢、其餘＝混合。特徵逐日使用當時已知資料，沒有使用未來24日跌幅。股票池是目前150檔的歷史子集，並非官方全台股指數。過去弱勢之後可能反彈，不預設弱勢環境的未來報酬必為負。</p>')
parts48.append(table47(['事前環境','四策略有效窗口','10／20有效','10／24有效','六策略共同窗口'],[[labels48[st],v48_stats['coverage'][st]['reference_n'],v48_stats['coverage'][st]['methods']['PURE_10_20'],v48_stats['coverage'][st]['methods']['PURE_10_24'],v48_stats['environments'][st]['n']] for st in ['strong','mixed','weak']]))
parts48.append('<p>弱勢環境167窗口，六策略交集只剩44個。10／20、10／24在弱勢的可成立率約26.95%、44.31%；不能把剩下的小樣本高收益解讀為它們避開下跌。候選不足日期未算成空手或0報酬。</p>')
for st in ['strong','mixed','weak']:
    rr=v48_stats['environments_core4'][st];parts48.append('<h3>'+labels48[st]+f'：四策略同日期{rr["n"]}窗口</h3>'+table47(['策略','勝率','費後24日期望','P10','最差窗口內回撤'],[[v47_names[k],*[pct47(rr['metrics'][k][f]) for f in ['win','ev','p10','worst_mdd']]] for k in v47_core]))
fig=go.Figure()
for st in ['strong','mixed','weak']:
    for k in v47_core:
        m=v47_cohorts['core4']&v30_later&(v48_startstates==st);ys=v46_ret[k][m]*100;values48.extend(ys.tolist());nm=labels48[st]+'｜'+v47_names[k]
        fig.add_trace(go.Box(y=ys,name=nm,boxpoints='all',jitter=.3,pointpos=0,marker={'size':3,'opacity':.4},text=[v47_window_rows[i]['start'] for i in np.flatnonzero(m)],hovertemplate='%{text}<br>%{y:.2f}%<extra>%{fullData.name}</extra>'))
        fig.add_trace(go.Scatter(x=[nm],y=[float(np.median(ys))],mode='markers',marker={'color':'#d81b60','symbol':'line-ew','size':20,'line':{'color':'#d81b60','width':3}},showlegend=False))
fig.update_layout(title='事前環境分組｜每組內四策略同日期；組別樣本數不同',height=680,showlegend=False,margin={'b':220},yaxis_title='費後24日報酬（%）');fig.add_hline(y=0,line_color='#222',line_width=2);figs48.append(fig);parts48.append(('fig',1))
lo=float(np.floor(min(values48)/5)*5);hi=float(np.ceil(max(values48)/5)*5)
for f in figs48:f.update_yaxes(range=[lo,hi])
parts48.append('<h2>三、每年如何決定？</h2><p>每個驗證年只用此前已結束的24日窗口；2016～2018作2019初始訓練，之後逐年增加。環境需有兩個訓練年份、六個互不重疊窗口，候選有效率至少90%，且對原動能的費後增益在24／48日區塊校正區間都為正，才允許替代。訓練家族固定105檢查，20,000次循環區塊重抽樣、Bonferroni校正。所有狀態及選擇規則在看驗證結果前鎖定，沒有事後挑最高策略。</p>')
parts48.append(table47(['檢查','未通過數量（原因可重疊）'],[['不足兩個訓練年份',v48_failure_counts['not_enough_years']],['不足六個非重疊窗口',v48_failure_counts['not_enough_nonoverlap']],['候選覆蓋率不足90%',v48_failure_counts['coverage_under90']],['校正增益未在兩種區塊均為正',v48_failure_counts['not_positive_both_ci']],['通過全部條件',v48_failure_counts['passed']]]))
parts48.append(table47(['驗證年','S1強／混合／弱','S2強／混合／弱','原動能有效窗口','勝率','期望'],[[str(y),'原動能／原動能／原動能','原動能／原動能／原動能',r['n'],pct47(r['metrics']['win']),pct47(r['metrics']['ev'])] for y in range(2019,2026) for r in [v48_stats['annual'][str(y)]]]))
parts48.append('<h2>四、起點選擇與每五日切換的結果</h2><p>逐一核對所有起點與五日檢查日，S1、S2都選原動能。四個版本直接引用原動能的現金、股數、交易、費用及NAV路徑，沒有重做相同回測，也沒有把不同組合的獨立NAV拼接。原動能本身仍有五日成員調整，「沒有策略切換」不代表沒有交易。</p>')
mm=v48_stats['policy_metrics'];parts48.append(table47(['版本','有效窗口','勝率','費後24日期望','對原動能增量','新增策略切換'],[[k,mm['n'],pct47(mm['win']),pct47(mm['ev']),'0.00百分點',0] for k in v48_policykeys]))
parts48.append('<p>四版本相對原動能的兩種區塊配對區間均為[0,0]，沒有正增益；不是另四份獨立支持動能的證據。評估家族為四版本各對原動能／法人，共8組，完整逐年差、剔除單一年敏感性與校正區間見統計JSON。</p>')
parts48.append('<h2>五、目前找到的脈絡</h2><ol><li>市場弱勢會減少正向均線候選，先影響策略能否成立，再影響比較樣本；這一層不能略過。</li><li>事前弱勢並不等於未來一定續跌，事後回報與當時訊號必須分開。</li><li>某環境樣本內平均較好，尚不足以建立可重複的年度選擇規則。</li><li>本批沒有通過訓練門檻的切換規則，維持原動能作基準；純50／200仍是固定策略挑戰者，但不是已證實應在某環境切入。</li></ol>')
parts48.append('<h2>邊界與後續</h2><p>本批核心48A～48F已完成；未啟動排程、未重啟kernel、未提交交易。這些結論限於本批市場特徵、六策略、90%目標投入及目前股票池。無法代表所有擇時策略或全台股。</p><p>取消強制25檔、依官方限制處理候選不足；加入具歷史公開時點的台灣VIX／景氣燈號；以及改變投入比率，都是獨立後續版本，未在本批悄悄加入。舊v30低高波動切換未證實結論已引用，沒有重算。正式Active Share、成交作廢及未知公司行動仍列限制。現有歷史已被研究過，真正前瞻驗證需未來新資料。</p>')
fig=go.Figure()
for state,color in [('strong','#2c987c'),('mixed','#c79a35'),('weak','#cf5858')]:
    xx=[x['date'] for x in v48_features if x['state']==state];fig.add_trace(go.Scatter(x=xx,y=[labels48[state]]*len(xx),mode='markers',marker={'size':4,'color':color},name=labels48[state],hovertemplate='%{x}<br>%{y}<extra></extra>'))
fig.update_layout(title='全部逐日事前市場狀態（未知日期無點）',height=320,xaxis={'rangeslider':{'visible':True}},showlegend=False);figs48.append(fig);parts48.append(('fig',2))
body='';first=True
for x in parts48:
    if isinstance(x,tuple):body+=figs48[x[1]].to_html(full_html=False,include_plotlyjs=True if first else False);first=False
    else:body+=x
v48_html=v34_root/'市場狀態_六策略比較與逐年選擇_v48_2026-10-17.html';v48_md=v48_html.with_suffix('.md')
v48_html.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>市場狀態與策略選擇 v48</title><style>body{font:16px/1.8 system-ui;max-width:1500px;margin:30px auto;padding:0 24px;color:#233043}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}td,th{padding:9px;border:1px solid #ccd5df;text-align:right}td:first-child,th:first-child{text-align:left}th{background:#eef3f8}h2{margin-top:40px}p{max-width:1200px}</style>'+body+'</html>',encoding='utf-8')
v48_md.write_text('# 市場狀態與策略選擇 v48\n\n'+ '\n\n'.join(x for x in parts48 if isinstance(x,str))+'\n\n互動圖表：['+v48_html.name+']('+v48_html.name+')',encoding='utf-8')
notice='<section id="market-selection-v48"><h2>市場狀態與策略選擇已完成</h2><p>六策略721共同窗口已對比；弱勢環境短均線覆蓋明顯不足。逐年訓練無替代策略通過事先門檻，兩選擇器的起點／五日版本全部保留原動能，增量為0；尚未證實依環境切換有效。</p><p><a href="'+v48_html.name+'">查看六策略箱形圖、事前市場分組及逐年驗證</a></p></section>'
fp=v34_root/'策略研究_總結報告.html';txt=fp.read_text(encoding='utf-8');anchor='<section id="unified-loss-study">';assert anchor in txt and 'id="market-selection-v48"' not in txt;fp.write_text(txt.replace(anchor,notice+anchor,1),encoding='utf-8')
fp=v34_root/'策略研究_總結報告.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v48 市場狀態與策略選擇\n\n六策略721窗口已比較。逐年訓練105檢查無通過，四選擇器版本與原動能相同，沒有策略切換增益；不代表所有市場擇時無效。\n\n['+v48_html.name+']('+v48_html.name+')\n',encoding='utf-8')
fp=v34_root/'統一策略_同日期比較與虧損環境_v47_2026-10-17.html';txt=fp.read_text(encoding='utf-8');assert '<h1>' in txt;fp.write_text(txt.replace('<h1>',notice+'<h1>',1),encoding='utf-8')
fp=v34_root/'策略研究_市場狀態與策略選擇計畫_v48_2026-10-17.md';txt=fp.read_text(encoding='utf-8');txt=txt.replace('狀態：已規劃，尚未執行新實驗；不啟動排程。','狀態：核心48A～48F已完成；四政策與原動能完全相同，沒有新增交易回測或啟動排程。');fp.write_text(txt+'\n\n## 執行結果\n\n完成六策略共同802／後段721窗口比較與逐日事前狀態。105組訓練檢查無候選通過完整門檻；2019～2025全部對映回原動能。起點／五日兩選擇器按決策日核對為完全相同，引用原始交易路徑、不重算。主要評估增量0，未證實改善。完整成果：['+v48_html.name+']('+v48_html.name+')。台灣VIX／景氣燈號及取消25檔為未執行獨立後續分支，不列為本批完成。\n',encoding='utf-8')
fp=v34_root/'A策略_無人值守研究流程.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n### v48 核心完成\n\n48A～48F完成，逐年訓練無候選過門檻，四政策與原動能同一交易路徑，未證實切換增量；不重跑相同交易。詳見市場狀態_六策略比較與逐年選擇_v48_2026-10-17.html。未啟用排程。\n',encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['market_selection_v48'].update(status='completed_no_incremental_evidence',running=False,nodes={n:'completed' for n in ['48A','48B','48C','48D','48E','48F']},new_trade_runs=0,policy_identity_verified=True,training_passes=0,report=str(v48_html),not_tested=['remove_fixed25','new_VIX_or_NDC_switching','investment_ratio_changes']);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V48_COMPLETE',{'training_failures':v48_failure_counts,'policy_audit':v48_policy_audit,'reference_metrics':mm,'six_metrics':v48_stats['common']['later']['metrics'],'report':v48_html.name})


In [ ]:

v48_desc=[];dd=[];nn=[]
for st in ['all','strong','mixed','weak']:
    for a,b in combinations(v48_keys,2):
        m=v48_common&v30_later&((v48_startstates==st) if st!='all' else np.ones(2236,bool));d=v46_ret[a][m]-v46_ret[b][m];dd.append(np.where(m,np.nan_to_num(v46_ret[a]-v46_ret[b]),0));nn.append(m.astype(float))
        jacc=[]
        for i in np.flatnonzero(m):
            aa=set(v46_results[a][i]['trace'][0]['selected']);bb=set(v46_results[b][i]['trace'][0]['selected']);jacc.append(len(aa&bb)/len(aa|bb))
        v48_desc.append({'state':st,'a':a,'b':b,'n':int(m.sum()),'ev_delta':float(np.mean(d)),'paired_win':float(np.mean(d>0)),'initial_selected_jaccard_mean':float(np.mean(jacc))})
v48_dci=bootstrap48(np.stack(dd,axis=1),np.stack(nn,axis=1),[24,48],60)
for j,d in enumerate(v48_desc):d.update(ci24=v48_dci['24'][j],ci48=v48_dci['48'][j])
v48_stats['descriptive_paired_60']=v48_desc;v48_stats['common_yearly']={str(y):{'n':int((v48_common&v30_later&(v48_startyears==y)).sum()),'metrics':{k:metrics_short46(k,v48_common&v30_later&(v48_startyears==y)) for k in v48_keys}} for y in range(2019,2026)}
v48_stats['reference_coverage']={'possible_later':int(v30_later.sum()),'MOM_valid_later':int((v30_later&v46_valid['MOM_E0']).sum()),'six_common_later':721,'core4_common_later':944}
detail='<h2>補充：六策略配對差與持股重疊</h2><p>此為已看過歷史的描述比較，非訓練選擇規則。全體及三環境各15對，共60項，使用24／48日區塊20,000次、60項Bonferroni校正。下表列721共同窗口的15對；分環境、配對勝率、起點候選持股交集／聯集平均及逐年統計見JSON。初始持股重疊衡量名單，不是資金權重重疊。</p>'+table47(['前者減後者','期望差（百分點）','24日校正區間','48日校正區間','起點名單重疊'],[[v47_names[d['a']]+' 減 '+v47_names[d['b']],pct47(d['ev_delta']),f"{pct47(d['ci24'][0])}～{pct47(d['ci24'][1])}",f"{pct47(d['ci48'][0])}～{pct47(d['ci48'][1])}",pct47(d['initial_selected_jaccard_mean'])] for d in v48_desc if d['state']=='all'])
detail+='<p>注意三種分母：六策略比較721窗口，四策略較大比較944窗口，選擇器因完全等同原動能可沿用1240個原動能有效窗口；三者不能直接比較勝率。2019～2025全部候選起點1678個，原動能有效1240，其餘仍保留不可判定。本批訓練未過門檻的主要障礙是覆蓋不足（99／105），以及校正區間未同時為正（96／105），原因可以重疊；不能只解讀為收益一定沒有差異。</p>'
txt=v48_html.read_text(encoding='utf-8');anchor='<h2>五、目前找到的脈絡</h2>';assert anchor in txt;v48_html.write_text(txt.replace(anchor,detail+anchor,1),encoding='utf-8');v48_md.write_text(v48_md.read_text(encoding='utf-8')+'\n\n'+detail,encoding='utf-8')
assert all(k=='MOM_E0' for y in v48_maps.values() for sel in y.values() for k in sel.values())
assert len(v48_features)==2320 and len(v48_train)==105 and len(v48_desc)==60 and len(v48_evals)==8
assert all(v30_days[int(v30_starts[i])-1]<v30_days[int(v30_starts[i])] for i in range(2236))
assert all(d['passes']==False for d in v48_train)
assert all(np.array_equal(v48_valid[k],v46_valid['MOM_E0']) and v48_results[k] is v46_results['MOM_E0'] for k in v48_policykeys)
html=v48_html.read_text(encoding='utf-8');assert html.count('Plotly.newPlot(')>=3 and '721共同窗口' in html and '99／105' in html
v48_stats['validation']={'all_policy_decision_dates_checked':20136,'identity_reuse_checked':True,'feature_rows':2320,'training_tests':105,'descriptive_pairs':60,'evaluation_tests':8,'html_plotly_figures':3,'unknowns_preserved':True,'browser_visual_QA':'not performed','kernel_restarted':False}
(v34_root/'市場狀態策略選擇_統計_v48_2026-10-17.json').write_text(json.dumps(v48_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['market_selection_v48']['validation']=v48_stats['validation'];sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V48_VALIDATED',v48_stats['validation'])


In [ ]:

# Previously authorized replacement after verified dead kernel. Restore cached inputs only.
import sys,os,json,gzip,datetime,bisect,re,hashlib,collections,ast,inspect
from research_layout import ResearchPath as Path
import numpy as np
import nbformat
import plotly.graph_objects as go
v34_root=Path('C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料');v30_root=v34_root;sp46=v34_root/'A策略_無人值守研究狀態.json';INITIAL=1e9;FEE=.001425;TAX=.003
v30_book=nbformat.read(v34_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4);base49=nbformat.read(v34_root/'動能基準研究_24日回測_2026-10-13.ipynb',as_version=4)
for wanted in ['settle','percentile']:
    found=[node for c in base49.cells if c.cell_type=='code' for node in ast.parse(c.source).body if isinstance(node,ast.FunctionDef) and node.name==wanted];assert found,wanted;exec(compile(ast.Module(body=[found[-1]],type_ignores=[]),'<saved-functions>','exec'),globals())
with gzip.open(v34_root/'策略研究_原資料快照_v3_2026-10-14.json.gz','rt',encoding='utf-8') as f:raw49=json.load(f)
raw_data={'stocks':raw49['stocks']};del raw49
src=v30_book.cells[321].source;src=src[src.index('def v30_load'):src.index('v30_export_path=')];exec(src,globals())
with gzip.open(v34_root/'長歷史法人_原始快取_v30_2026-10-16.json.gz','rt',encoding='utf-8') as f:v30_flow_cache=json.load(f)['data']
v30_flow=np.full_like(v30_arrays['close'],np.nan);groups49=collections.defaultdict(dict)
for sid,rows in v30_flow_cache.items():
    for row in rows:
        if row['name'] in ['Foreign_Investor','Investment_Trust']:groups49[(row['date'],sid)][row['name']]=float(row['buy'])-float(row['sell'])
for (d,sid),cats in groups49.items():
    if len(cats)==2 and d in v30_di and sid in v30_si:v30_flow[v30_di[d],v30_si[sid]]=sum(cats.values())
src=v30_book.cells[324].source;src=src[src.index('v30_starts='):src.index('v30_engine_export=')];exec(src,globals());v34_scores={k:v30_scores[k] for k in ['MOM','FLOW']}
v34_plan=json.loads((v34_root/'策略研究_統一長歷史續跑計畫_2026-10-16.json').read_text(encoding='utf-8'));v30_later=np.array([v30_days[int(s)]>=datetime.date(2019,1,1) for s in v30_starts]);v30_training=np.array([v30_days[int(s)+23]<=datetime.date(2018,12,31) for s in v30_starts])
v45_pairs_ma={'50_200':(50,200),'10_20':(10,20),'10_24':(10,24)};src=next(c.source for c in v30_book.cells if c.metadata.get('external_history_v12')=='ma_cross_define_v45');exec(src[src.index('v45_ma={}'):src.index('v45_code=')],globals());v46_ma=v45_ma;v46_scores=v45_scores
out=''.join(o.get('text','') for o in v30_book.cells[377].outputs);v45_code=out.split('V48_ENGINE_SOURCE ',1)[1].strip();ast.parse(v45_code)
ns49=dict(v30_ns);ns49.update(INITIAL=INITIAL,FEE=FEE,TAX=TAX,settle=settle,percentile=percentile,case_arrays_v14={'long30':v30_arrays},scores30=v46_scores,ma45=v46_ma);exec(v45_code,ns49);v46_sim=ns49['simulate_ma45']
def serial30(x):
    if isinstance(x,np.ndarray):return x.tolist()
    if isinstance(x,np.generic):return x.item()
    if isinstance(x,(datetime.date,Path)):return str(x)
    raise TypeError(type(x).__name__)
v46_results=json.loads((v34_root/'長歷史環境研究_動能法人逐窗口_v30_2026-10-16.json').read_text(encoding='utf-8'))['results'];v46_results={'MOM_E0':v46_results['MOM'],'FLOW_E0':v46_results['FLOW']}
for k in ['PURE_50_200','FLOW_50_200_ACTIVE','PURE_10_20','PURE_10_24']:
    fp=v34_root/(('短均線_' if k in ['PURE_10_20','PURE_10_24'] else '均線交叉_')+k+('_v46_2026-10-17.json.gz' if k in ['PURE_10_20','PURE_10_24'] else '_v45_2026-10-17.json.gz'))
    with gzip.open(fp,'rt',encoding='utf-8') as f:v46_results[k]=json.load(f)['results']
v46_valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in v46_results.items()};v46_ret={k:np.array([r['return'] if r['return'] is not None else np.nan for r in rows]) for k,rows in v46_results.items()}
for wanted in ['metrics_short46']:
    for c in v30_book.cells:
        if c.cell_type=='code' and 'def '+wanted+'(' in c.source:
            node=next(n for n in ast.parse(c.source).body if isinstance(n,ast.FunctionDef) and n.name==wanted);exec(compile(ast.Module(body=[node],type_ignores=[]),'<saved-metrics>','exec'),globals());break
v49_controls=[]
for pair in ['10_20','10_24']:
    for i in [0,400,900,1700,2235]:
        s=int(v30_starts[i]);now=v46_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved','MA_'+pair,.9,5,pair,'ACTIVE');old=v46_results['PURE_'+pair][i];assert now['return']==old['return'] and now['reasons']==old['reasons'] and now['violations']==old['violations'];v49_controls.append({'pair':pair,'index':i,'matched':True})
print('V49_RESTORED',{'pid':os.getpid(),'python':sys.executable,'shape':v30_arrays['close'].shape,'cached_methods':len(v46_results),'baseline_controls':len(v49_controls),'network_requests':0})


In [ ]:

# Previously authorized replacement after verified dead kernel. Restore cached inputs only.
import sys,os,json,gzip,datetime,bisect,re,hashlib,collections,ast,inspect
from research_layout import ResearchPath as Path
import numpy as np
import nbformat
import plotly.graph_objects as go
v34_root=Path('C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料');v30_root=v34_root;sp46=v34_root/'A策略_無人值守研究狀態.json';INITIAL=1e9;FEE=.001425;TAX=.003
v30_book=nbformat.read(v34_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4);base49=nbformat.read(v34_root/'動能基準研究_24日回測_2026-10-13.ipynb',as_version=4)
for wanted in ['settle','percentile']:
    found=[node for c in base49.cells if c.cell_type=='code' and 'def '+wanted+'(' in c.source for node in ast.parse(c.source).body if isinstance(node,ast.FunctionDef) and node.name==wanted];assert found,wanted;exec(compile(ast.Module(body=[found[-1]],type_ignores=[]),'<saved-functions>','exec'),globals())
with gzip.open(v34_root/'策略研究_原資料快照_v3_2026-10-14.json.gz','rt',encoding='utf-8') as f:raw49=json.load(f)
raw_data={'stocks':raw49['stocks']};del raw49
src=v30_book.cells[321].source;src=src[src.index('def v30_load'):src.index('v30_export_path=')];exec(src,globals())
with gzip.open(v34_root/'長歷史法人_原始快取_v30_2026-10-16.json.gz','rt',encoding='utf-8') as f:v30_flow_cache=json.load(f)['data']
v30_flow=np.full_like(v30_arrays['close'],np.nan);groups49=collections.defaultdict(dict)
for sid,rows in v30_flow_cache.items():
    for row in rows:
        if row['name'] in ['Foreign_Investor','Investment_Trust']:groups49[(row['date'],sid)][row['name']]=float(row['buy'])-float(row['sell'])
for (d,sid),cats in groups49.items():
    if len(cats)==2 and d in v30_di and sid in v30_si:v30_flow[v30_di[d],v30_si[sid]]=sum(cats.values())
src=v30_book.cells[324].source;src=src[src.index('v30_starts='):src.index('v30_engine_export=')];exec(src,globals());v34_scores={k:v30_scores[k] for k in ['MOM','FLOW']}
v34_plan=json.loads((v34_root/'策略研究_統一長歷史續跑計畫_2026-10-16.json').read_text(encoding='utf-8'));v30_later=np.array([v30_days[int(s)]>=datetime.date(2019,1,1) for s in v30_starts]);v30_training=np.array([v30_days[int(s)+23]<=datetime.date(2018,12,31) for s in v30_starts])
v45_pairs_ma={'50_200':(50,200),'10_20':(10,20),'10_24':(10,24)};src=next(c.source for c in v30_book.cells if c.metadata.get('external_history_v12')=='ma_cross_define_v45');exec(src[src.index('v45_ma={}'):src.index('v45_code=')],globals());v46_ma=v45_ma;v46_scores=v45_scores
out=''.join(o.get('text','') for o in v30_book.cells[377].outputs);v45_code=out.split('V48_ENGINE_SOURCE ',1)[1].strip();ast.parse(v45_code)
ns49=dict(v30_ns);ns49.update(INITIAL=INITIAL,FEE=FEE,TAX=TAX,settle=settle,percentile=percentile,case_arrays_v14={'long30':v30_arrays},scores30=v46_scores,ma45=v46_ma);exec(v45_code,ns49);v46_sim=ns49['simulate_ma45']
def serial30(x):
    if isinstance(x,np.ndarray):return x.tolist()
    if isinstance(x,np.generic):return x.item()
    if isinstance(x,(datetime.date,Path)):return str(x)
    raise TypeError(type(x).__name__)
v46_results=json.loads((v34_root/'長歷史環境研究_動能法人逐窗口_v30_2026-10-16.json').read_text(encoding='utf-8'))['results'];v46_results={'MOM_E0':v46_results['MOM'],'FLOW_E0':v46_results['FLOW']}
for k in ['PURE_50_200','FLOW_50_200_ACTIVE','PURE_10_20','PURE_10_24']:
    fp=v34_root/(('短均線_' if k in ['PURE_10_20','PURE_10_24'] else '均線交叉_')+k+('_v46_2026-10-17.json.gz' if k in ['PURE_10_20','PURE_10_24'] else '_v45_2026-10-17.json.gz'))
    with gzip.open(fp,'rt',encoding='utf-8') as f:v46_results[k]=json.load(f)['results']
v46_valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in v46_results.items()};v46_ret={k:np.array([r['return'] if r['return'] is not None else np.nan for r in rows]) for k,rows in v46_results.items()}
for wanted in ['metrics_short46']:
    for c in v30_book.cells:
        if c.cell_type=='code' and 'def '+wanted+'(' in c.source:
            node=next(n for n in ast.parse(c.source).body if isinstance(n,ast.FunctionDef) and n.name==wanted);exec(compile(ast.Module(body=[node],type_ignores=[]),'<saved-metrics>','exec'),globals());break
v49_controls=[]
for pair in ['10_20','10_24']:
    for i in [0,400,900,1700,2235]:
        s=int(v30_starts[i]);now=v46_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved','MA_'+pair,.9,5,pair,'ACTIVE');old=v46_results['PURE_'+pair][i];assert now['return']==old['return'] and now['reasons']==old['reasons'] and now['violations']==old['violations'];v49_controls.append({'pair':pair,'index':i,'matched':True})
print('V49_RESTORED',{'pid':os.getpid(),'python':sys.executable,'shape':v30_arrays['close'].shape,'cached_methods':len(v46_results),'baseline_controls':len(v49_controls),'network_requests':0})


In [ ]:

v49_definition={'version':'v49_variable20to25_fixed1','date':'2026-10-17','pairs':['10_20','10_24'],'modes':{'VARIABLE':'regular positiveMA candidates20..25; under20 invalid; offcycle death replace available slots only, may reduce actual names if >=20; cash/holdings/caps still audited','FALLBACK':'same variable names; if regular under20 OR death cannot replace all exited slots, full MOM25 rebalance and no MA exits until next regular review; retry MA at each regular5dayreview'},'baseline':'same cached v46 PURE pair and MOM/FLOW/50_200; no new parameter search','fixed_controls':'count_mode FIXED exact original25 simulation','weights':'90%/selected_count at regular or fallback trades; offcycle replacements match old holding value; no special cashoverlay','timing':'priorclose signal; earliest nextVWAP;24sessions;normal5days','rules':'20..30 actualpositiveholdings,1000shares,cash>=0 and<25%,2330cap25%others10% andold5daygrace;officialActiveShareandexecutionvoidsnotcertified','sources':'local competition explanation2026-10-13 lines77..83 and v45/v46 retained engine; no claim latest external rule recertification','fallback':'only prior-known candidate shortage; unknown execution/actions stillinvalid, never hindsightfallback','family':10,'contrasts':'4new vs MOM,4new vs ownPURE25,2FALLBACKvsVARIABLE','statistics':'24/48 originalposition blocks20000,Bonferroni10;pairedvalid;2019..2025 alreadyseen;years,P10,worstMDD,coverage;no bestmeanforcedwinner','input_hashes':v34_plan['common']['source_hashes'],'reuse':'old controls direct, matched fixedcontrols;new4*2236=8944windows','no_daily_fullrebalance':True}
v49_hash=hashlib.sha256(json.dumps(v49_definition,sort_keys=True).encode()).hexdigest();(v34_root/'可變持股均線_預先定義_v49_2026-10-17.json').write_text(json.dumps(v49_definition,ensure_ascii=False,indent=2),encoding='utf-8')
v49_code=v45_code.replace('def simulate_ma45(','def simulate_variable49(').replace("ma_mode='none'):","ma_mode='none', count_mode='FIXED'):")
v49_code=v49_code.replace('    events45=[]','    events45=[]\n    fallback49=False')
v49_decision=r'''        q=t-1
        regular45=offset==0 or (rebalance>0 and offset%rebalance==0)
        if regular45:fallback49=False
        active_mode49='none' if fallback49 else ma_mode
        active_strategy49='MOM' if fallback49 else strategy
        failed45=np.flatnonzero((shares>0)&ma45[ma_pair]['death'][q]) if active_mode49=='ACTIVE' and offset>0 else np.array([],dtype=int)
        if regular45 or len(failed45):
            score=scores30[active_strategy49][q];ordered=np.flatnonzero(np.isfinite(score));ranked45=ordered[np.argsort(-score[ordered],kind='stable')]
            filtering45=active_mode49=='ACTIVE' or (active_mode49=='INITIAL' and offset==0)
            candidates45=[j for j in ranked45 if not filtering45 or ma45[ma_pair]['bull'][q,j]]
            raw_n49=len(candidates45);fallback_event49=False
            if regular45:selected=np.array(candidates45[:25],dtype=int)
            else:
                retained45=[j for j in np.flatnonzero(shares>0) if j not in failed45];new45=[j for j in candidates45 if shares[j]==0][:len(failed45)];selected=np.array(retained45+new45,dtype=int)
            if count_mode=='FALLBACK' and ((regular45 and len(selected)<20) or (not regular45 and len(new45)<len(failed45))):
                fallback49=True;fallback_event49=True;fallback_score49=scores30['MOM'][q];ii49=np.flatnonzero(np.isfinite(fallback_score49));selected=ii49[np.argsort(-fallback_score49[ii49],kind='stable')[:25]]
            min49=25 if count_mode=='FIXED' or fallback_event49 else 20
            if len(selected)<min49:
                reasons.add('fewer_than25_bullish_MA_or_data_eligible' if count_mode=='FIXED' and filtering45 else 'fewer_than25_past_data_eligible_stocks' if fallback_event49 else 'fewer_than20_bullish_MA_or_data_eligible');break
            if np.any(~np.isfinite(cl[q,selected])|(cl[q,selected]<=0)):
                reasons.add('missing_prior_close');break
            if regular45 or fallback_event49:
                weights=np.full(len(selected),invested/len(selected));target=np.zeros_like(shares);target[selected]=(np.floor(weights*prior_nav/cl[q,selected]/1000)*1000).astype(np.int64)
            else:
                target=shares.copy();new45=[j for j in selected if shares[j]==0]
                for old45 in failed45:target[old45]=0
                for old45,newid45 in zip(failed45,new45):target[newid45]=int(np.floor(shares[old45]*mark[q,old45]/cl[q,newid45]/1000))*1000
            cash,shares,traded,cost,missing=settle(cash,shares,target,arrays['execution'][t],arrays['available'][t]);fees+=cost;turnover_sum+=traded
            if missing:reasons.add('invalid_execution_price')
            traces.append({'date':str(days[t]),'selected':[all_external_ids_v13[j] for j in selected]})
            events45.append({'date':str(days[t]),'signal_date':str(days[q]),'regular':regular45,'fallback_event':fallback_event49,'fallback_active':fallback49,'bullish_candidates':raw_n49,'selected_count':len(selected),'held_count':int(np.count_nonzero(shares)),'cash_fraction':float(cash/(cash+np.dot(shares,np.nan_to_num(mark[q])))),'death_exits':[all_external_ids_v13[j] for j in failed45],'traded':traded/INITIAL,'cost':cost/INITIAL})
'''
left=v49_code.index('        q=t-1');right=v49_code.index('        if np.any((shares > 0) & ~np.isfinite(mark[t])):');v49_code=v49_code[:left]+v49_decision+v49_code[right:];ast.parse(v49_code);v49_ns=dict(ns49);exec(v49_code,v49_ns);v49_sim=v49_ns['simulate_variable49'];v49_enginehash=hashlib.sha256(v49_code.encode()).hexdigest()
v49_fixedchecks=[]
for pair in ['10_20','10_24']:
    for i in [0,400,900,1700,2235]:
        s=int(v30_starts[i]);now=v49_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved','MA_'+pair,.9,5,pair,'ACTIVE','FIXED');old=v46_results['PURE_'+pair][i];assert now['return']==old['return'] and now['reasons']==old['reasons'] and now['violations']==old['violations'];v49_fixedchecks.append({'pair':pair,'index':i,'match':True})
v49_results={};v49_specs=[{'key':pair+'_'+mode,'pair':pair,'mode':mode,'definition_hash':v49_hash,'engine_hash':v49_enginehash} for pair in ['10_20','10_24'] for mode in ['VARIABLE','FALLBACK']]
for spec in v49_specs:
    key=spec['key'];fp=v34_root/('可變持股均線_'+key+'_v49_2026-10-17.json.gz');state=json.loads(sp46.read_text(encoding='utf-8'));state['variable_ma_v49']={'status':'running','running':True,'current':key,'definition_hash':v49_hash,'kernel_connection':'kernel-aicup-ma-variable-20261017.json','completed':list(v49_results)};sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
    if fp.exists():
        with gzip.open(fp,'rt',encoding='utf-8') as f:cached=json.load(f)
        assert cached['spec']==spec;v49_results[key]=cached['results'];continue
    rows=[]
    for i,s0 in enumerate(v30_starts):
        s=int(s0);r=v49_sim({'key':'long30','year':v30_days[s].year,'start_index':s},set(v30_universe),'equal',0.,'unresolved','MA_'+spec['pair'],.9,5,spec['pair'],'ACTIVE',spec['mode']);r['start_index']=s;rows.append(r)
        if (i+1)%750==0:print('V49_PROGRESS',key,i+1,flush=True)
    with gzip.open(fp,'wt',encoding='utf-8') as f:json.dump({'spec':spec,'engine':v49_code,'results':rows},f,ensure_ascii=False,default=serial30,allow_nan=False)
    v49_results[key]=rows;print('V49_METHOD_DONE',key,{'valid':sum(r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows)},flush=True)
state=json.loads(sp46.read_text(encoding='utf-8'));state['variable_ma_v49'].update(status='computed_pending_statistics',running=False,current=None,completed=list(v49_results),fixed_controls=v49_fixedchecks);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V49_RUNS_COMPLETE',len(v49_results))


In [ ]:

v49_newkeys=list(v49_results)
for k,rows in v49_results.items():v46_results[k]=rows
v46_valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in v46_results.items()};v46_ret={k:np.array([r['return'] if r['return'] is not None else np.nan for r in rows]) for k,rows in v46_results.items()}
v49_checks=[]
for spec in v49_specs:
    k=spec['key'];oldkey='PURE_'+spec['pair'];m=v46_valid[oldkey];assert np.all(v46_valid[k][m]),(k,'lost old valid')
    for i in np.flatnonzero(m):
        aa=v46_results[k][i];bb=v46_results[oldkey][i];assert all(aa[f]==bb[f] for f in ['return','mdd','cost','turnover']),(k,i)
    v49_checks.append({'key':k,'old_valid_identical':int(m.sum())})
node=next(n for n in ast.parse(v30_book.cells[378].source).body if isinstance(n,ast.FunctionDef) and n.name=='bootstrap48');exec(compile(ast.Module(body=[node],type_ignores=[]),'<saved-bootstrap>','exec'),globals())
v49_pairs=[]
for spec in v49_specs:
    k=spec['key'];v49_pairs.extend([(k,'MOM_E0'),(k,'PURE_'+spec['pair'])])
for pair in ['10_20','10_24']:v49_pairs.append((pair+'_FALLBACK',pair+'_VARIABLE'))
assert len(v49_pairs)==10
dd=[];nn=[]
for a,b in v49_pairs:
    m=v30_later&v46_valid[a]&v46_valid[b];dd.append(np.where(m,np.nan_to_num(v46_ret[a]-v46_ret[b]),0));nn.append(m.astype(float))
v49_ci=bootstrap48(np.stack(dd,axis=1),np.stack(nn,axis=1),[24,48],10);v49_contrasts=[]
for j,(a,b) in enumerate(v49_pairs):
    m=nn[j].astype(bool);d=v46_ret[a][m]-v46_ret[b][m];yr={str(y):float(np.mean(v46_ret[a][mm]-v46_ret[b][mm])) for y in range(2019,2026) for mm in [m&np.array([v30_days[int(s)].year==y for s in v30_starts])] if mm.any()}
    v49_contrasts.append({'a':a,'b':b,'n':int(m.sum()),'coverage_vs_control':float(m.sum()/(v30_later&v46_valid[b]).sum()),'delta':float(np.mean(d)),'paired_win':float(np.mean(d>0)),'ci24':v49_ci['24'][j],'ci48':v49_ci['48'][j],'positive_both_ci':bool(min(v49_ci['24'][j][0],v49_ci['48'][j][0])>0),'a_metrics':metrics_short46(a,m),'b_metrics':metrics_short46(b,m),'years':yr})
v49_allkeys=['MOM_E0','FLOW_E0','PURE_50_200','FLOW_50_200_ACTIVE']+v49_newkeys;v49_common=np.logical_and.reduce([v46_valid[k] for k in v49_allkeys]);v49_rows47=json.loads((v34_root/'統一策略_同日期虧損逐窗口_v47_2026-10-17.json').read_text(encoding='utf-8'));v49_reference=np.array([r['core4_common'] for r in v49_rows47])&v30_later
v49_statefeatures=json.loads((v34_root/'市場狀態策略選擇_事前特徵_v48_2026-10-17.json').read_text(encoding='utf-8'));v49_states=np.array([v49_statefeatures[int(s)-1]['state'] for s in v30_starts]);v49_stats={'definition':v49_definition,'checks':v49_checks,'new_runs':8944,'contrasts':v49_contrasts,'common':{},'coverage':{},'restored':{},'all_valid':{}}
for period,pm in [('all',np.ones(2236,bool)),('later',v30_later)]:
    m=v49_common&pm;v49_stats['common'][period]={'n':int(m.sum()),'metrics':{k:metrics_short46(k,m) for k in v49_allkeys}}
for k in v49_newkeys:
    rows=v46_results[k];oldkey='PURE_'+k.rsplit('_',1)[0];good=v46_valid[k]&v30_later;regained=good&~v46_valid[oldkey];ids=np.flatnonzero(good);evs=[e for i in ids for e in rows[i]['events']];reasons=collections.Counter(x for r in rows if r['return'] is None for x in r['reasons'])
    v49_stats['all_valid'][k]={'metrics':metrics_short46(k,good),'invalid_reasons_all':dict(reasons),'fallback_windows':sum(any(e.get('fallback_event') for e in rows[i]['events']) for i in ids),'variable20to24_windows':sum(any(e.get('regular') and 20<=e.get('selected_count',25)<25 for e in rows[i]['events']) for i in ids),'mean_selected_count_at_trades':float(np.mean([e['selected_count'] for e in evs])) if evs else None}
    v49_stats['restored'][k]={'n':int(regained.sum()),'metrics':metrics_short46(k,regained) if regained.any() else None,'market_broad_down_n':sum(bool(v49_rows47[i]['market']['confirmed_broad_down']) for i in np.flatnonzero(regained)),'prior_state':dict(collections.Counter(v49_states[regained]))}
    v49_stats['coverage'][k]={'reference944_n':int(v49_reference.sum()),'valid_in944':int((good&v49_reference).sum()),'by_state':{st:{'den':int((v49_reference&(v49_states==st)).sum()),'valid':int((good&v49_reference&(v49_states==st)).sum())} for st in ['strong','mixed','weak']}}
v49_cases=[]
for i in np.flatnonzero(v49_reference&np.logical_and.reduce([v46_valid[k] for k in v49_newkeys])):
    if any(not v46_valid['PURE_'+p][i] for p in ['10_20','10_24']):v49_cases.append({'index':int(i),'start':v49_rows47[i]['start'],'end':v49_rows47[i]['end'],'state':str(v49_states[i]),'market':v49_rows47[i]['market'],'returns':{k:float(v46_ret[k][i]) for k in v49_allkeys}})
v49_stats['restored_case_rows']=v49_cases
(v34_root/'可變持股均線_統計與覆蓋_v49_2026-10-17.json').write_text(json.dumps(v49_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['variable_ma_v49'].update(status='computed_statistics_pending_report',running=False,common_later=v49_stats['common']['later']['n'],checks=v49_checks);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V49_STATS',{'common':{p:x['n'] for p,x in v49_stats['common'].items()},'coverage':v49_stats['coverage'],'metrics':v49_stats['common']['later']['metrics'],'comparisons':[{'a':d['a'],'b':d['b'],'n':d['n'],'delta':d['delta'],'ci24':d['ci24'],'ci48':d['ci48']} for d in v49_contrasts],'regained':{k:d['n'] for k,d in v49_stats['restored'].items()}})


In [ ]:

for label,file in {'price_current150_cache':'完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz','flow_current150_cache':'長歷史法人_原始快取_v30_2026-10-16.json.gz','price_twse_and_actions':'外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz','action_repairs':'動能_長歷史來源修正稽核_v27_2026-10-15.json','listing_stockmaster_snapshot':'策略研究_原資料快照_v3_2026-10-14.json.gz'}.items():assert hashlib.sha256((v34_root/file).read_bytes()).hexdigest()==v34_plan['common']['source_hashes'][label],label
for name in ['pct47','table47']:
    node=next(n for n in ast.parse(v30_book.cells[376].source).body if isinstance(n,ast.FunctionDef) and n.name==name);exec(compile(ast.Module(body=[node],type_ignores=[]),'<saved-report-helper>','exec'),globals())
names49={'MOM_E0':'原動能各半／五日','FLOW_E0':'純法人／五日','PURE_50_200':'純50／200均線強度','FLOW_50_200_ACTIVE':'法人＋50／200交叉換股','PURE_10_20':'原10／20固定25檔','PURE_10_24':'原10／24固定25檔','10_20_VARIABLE':'10／20持20～25檔','10_20_FALLBACK':'10／20持20～25檔＋動能回退','10_24_VARIABLE':'10／24持20～25檔','10_24_FALLBACK':'10／24持20～25檔＋動能回退'}
v49_mainkeys=['MOM_E0','FLOW_E0','PURE_50_200','FLOW_50_200_ACTIVE','10_20_FALLBACK','10_24_FALLBACK'];v49_mainmask=np.logical_and.reduce([v46_valid[k] for k in v49_mainkeys])&v30_later
v49_main={'n':int(v49_mainmask.sum()),'metrics':{k:metrics_short46(k,v49_mainmask) for k in v49_mainkeys}};v49_stats['main_comparison']=v49_main
parts49=['<h1>短均線不強求25檔，結果會改善嗎？</h1><p>v49｜2026-10-17｜沿用2016～2025相同價量、法人與公司行動快取；沒有重抓資料。新增4版本、8,944個窗口；舊6策略直接引用。核心比較為2019～2025，歷史已看過，並非新未見樣本。</p><h2>先看結論</h2><p><strong>可執行覆蓋明顯改善，但收益優勢尚未證實。</strong>放寬為20～25檔及候選不足回原動能後，原944窗口中10／20可成立936個、10／24可成立937個；原先分別只有729、789。相較原動能的費後收益差在24／48日區塊校正區間都跨0，不能把更高平均值直接當作改良。</p>']
parts49.append('<h2>規則是什麼？</h2><p>本機已歸檔競賽說明為每日20～30檔，台積電上限25%、其餘10%；整千股，現金非負且低於25%。這次只取消實驗的強制25檔，沒有取消官方限制。價格上漲權重寬限沿用既有模型；Active Share及官方成交作廢仍未完整認證。</p><ul><li>原版：每五日正向均線候選必須滿25檔，死亡交叉平時一對一替補。</li><li>可變版：每五日選最多25檔，20～24檔也可；每檔等分90%目標投入。少於20檔仍不可判定。平時死亡交叉用可用候選替補，最低持股與現金仍每日檢查。</li><li>可變＋回退版：五日檢查時不足20檔，或平時死亡交叉無法完整替補，改按原動能25檔配置；保留至下次五日檢查再嘗試均線。此時會整體調整、計入費用，不是無成本切換。</li></ul><p>全部訊號以前收盤資料決定，最早次交易日按既有均價模型成交；未知公司行動或成交價格缺漏仍不可判定，不用事後資訊決定回退。90%是目標投入而非每天保證10%現金。沒有另外加槓桿或尋找最佳持股數。</p>')
parts49.append('<h2>一、原944窗口中，有多少現在可成立？</h2>'+table47(['版本','可成立／944','弱勢環境可成立／167'],[[names49['PURE_10_20'],'729','45'],[names49['10_20_VARIABLE'],823,95],[names49['10_20_FALLBACK'],936,163],[names49['PURE_10_24'],789,74],[names49['10_24_VARIABLE'],817,86],[names49['10_24_FALLBACK'],937,163]]))
parts49.append('<p>回退仍不能保證全部日期成立；未知公司行動、缺價、成交與現金／持股限制繼續保留為未知或不合格。不能把未成立日期當成空手、虧損或0報酬。</p>')
parts49.append(f'<h2>二、六種可用候選：相同{v49_main["n"]}個窗口</h2><p>四個主要舊策略，加兩個可變＋回退版本，取完全相同日期。這個分母不同於舊報告721／944或本批八策略807；僅能在同表比較。勝率為費後24日報酬&gt;0比例；期望為平均報酬；回撤是最差24日窗口內值。</p>'+table47(['策略','勝率','費後24日期望','P10','最差窗口內回撤','平均費用／初始資金'],[[names49[k],*[pct47(v49_main['metrics'][k][f]) for f in ['win','ev','p10','worst_mdd','fee']]] for k in v49_mainkeys]))
figs49=[];rangevalues=[]
for keys,mask,title in [(v49_mainkeys,v49_mainmask,'六策略同日期比較'),(v49_allkeys,v49_common&v30_later,'八策略同日期比較（另含不回退可變版）')]:
    fig=go.Figure()
    for k in keys:
        ys=v46_ret[k][mask]*100;rangevalues.extend(ys.tolist());fig.add_trace(go.Box(y=ys,name=names49[k],boxpoints='all',jitter=.35,pointpos=0,marker={'size':3,'opacity':.4},text=[v49_rows47[i]['start']+'～'+v49_rows47[i]['end'] for i in np.flatnonzero(mask)],hovertemplate='%{text}<br>費後24日 %{y:.2f}%<extra>%{fullData.name}</extra>'))
        fig.add_trace(go.Scatter(x=[names49[k]],y=[float(np.median(ys))],mode='markers',marker={'color':'#d81b60','symbol':'line-ew','size':25,'line':{'color':'#d81b60','width':3}},showlegend=False,hovertemplate='中位數 %{y:.2f}%<extra></extra>'))
    fig.update_layout(title=title+f'｜{int(mask.sum())}窗口，全部點',height=620,showlegend=False,yaxis_title='費後24日報酬（%）',margin={'b':170});fig.add_hline(y=0,line_color='#222',line_width=2);figs49.append(fig);parts49.append(('fig',len(figs49)-1))
yl=[float(np.floor(min(rangevalues)/5)*5),float(np.ceil(max(rangevalues)/5)*5)]
for fig in figs49:fig.update_yaxes(range=yl)
parts49.append('<h2>三、相較原動能，增益有多確定？</h2><p>每列為該版本與原動能的有效日期交集，分母明列；不以各自全有效日期直接相減。原始日期位置24／48日循環區塊、20,000次、10項固定比較Bonferroni校正。差值及區間為百分點，重疊窗口不是獨立事件。</p>'+table47(['版本','配對窗口','期望差','24日區塊校正區間','48日區塊校正區間','平均費用差'],[[names49[d['a']],d['n'],pct47(d['delta']),f"{pct47(d['ci24'][0])}～{pct47(d['ci24'][1])}",f"{pct47(d['ci48'][0])}～{pct47(d['ci48'][1])}",pct47(d['a_metrics']['fee']-d['b_metrics']['fee'])] for d in v49_contrasts if d['b']=='MOM_E0']))
parts49.append('<p>所有新版本相對原動能的區間均跨0。可變版在原固定25檔已能成立的日期，交易與收益完全一致，已逐一核對；主要新增的是以前無法比較的日期，不能說舊日期報酬變好了。回退版本與可變版的共同日期，平均增益也沒有改善；10／24回退的額外費後平均差約−0.020個百分點，校正上界略低於0，其數值很小，主要意義是不可把回退當成收益提升機制。</p>')
parts49.append('<h2>四、回退實際用了多少？</h2><p>以下各列採自身全部有效的2019～2025窗口，分母不同，只描述行為，不用於策略排名。</p>'+table47(['版本','自身有效窗口','至少一次20～24檔配置','至少一次回退','新增可判定日期（相對原同均線）','自身費後期望'],[[names49[k],d['metrics']['n'],d['variable20to24_windows'],d['fallback_windows'],v49_stats['restored'][k]['n'],pct47(d['metrics']['ev'])] for k,d in v49_stats['all_valid'].items()]))
parts49.append('<h2>五、原先未成立、現在可計算的壞行情</h2><p>在本批共同可用的新增案例中，依四新版本平均報酬取較差者，起點至少相隔24日列6例；事後案例不是六個獨立危機。股票池為現況150檔歷史子集，不是全台股官方指數。</p>')
cases=sorted(v49_cases,key=lambda r:np.mean([r['returns'][k] for k in v49_newkeys]));chosen=[]
for r in cases:
    if all(abs(r['index']-q['index'])>=24 for q in chosen):chosen.append(r)
    if len(chosen)>=6:break
parts49.append(table47(['起訖','事前環境','原動能','10／20回退','10／24回退','股票池下跌比例'],[[r['start']+'～'+r['end'],r['state'],pct47(r['returns']['MOM_E0']),pct47(r['returns']['10_20_FALLBACK']),pct47(r['returns']['10_24_FALLBACK']),pct47(r['market']['down_fraction'])] for r in chosen]))
parts49.append('<h2>結論與邊界</h2><p>「不足25檔就整段不可判定」的問題可以大幅改善，採20～25檔與事前回退規則後，兩種短均線幾乎涵蓋原944窗口。收益改善仍未證實，固定原動能及純50／200繼續保留作對照；不以本批最高平均值宣布新贏家。</p><p>舊v48的市場狀態選擇器沒有在本批重訓。本批回退是候選不足時的明確操作規則，不是從收益挑市場狀態，也不是「弱市必選法人」。不把此結果當作已證實新的市場擇時。台灣VIX／景氣燈號新切換及改變投入比例仍未測。</p><p>現況150檔的存活／名單偏差、2016部分暖機不足、未知公司行動、官方Active Share及成交作廢規則仍限制外推。本批以官方資料的本機存檔解讀規則，未重新認證官網今日更新。沒有交易提交或啟用排程。</p>')
body='';first=True
for x in parts49:
    if isinstance(x,tuple):body+=figs49[x[1]].to_html(full_html=False,include_plotlyjs=True if first else False);first=False
    else:body+=x
v49_html=v34_root/'短均線_可變持股與動能回退_v49_2026-10-17.html';v49_md=v49_html.with_suffix('.md');v49_html.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>短均線可變持股 v49</title><style>body{font:16px/1.8 system-ui;max-width:1500px;margin:30px auto;padding:0 24px;color:#233043}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}td,th{padding:9px;border:1px solid #ccd5df;text-align:right}td:first-child,th:first-child{text-align:left}th{background:#eef3f8}h2{margin-top:40px}p{max-width:1200px}</style>'+body+'</html>',encoding='utf-8');v49_md.write_text('# 可變持股與動能回退 v49\n\n'+'\n\n'.join(x for x in parts49 if isinstance(x,str))+'\n\n互動圖表：['+v49_html.name+']('+v49_html.name+')',encoding='utf-8')
note='<section id="variable-ma-v49"><h2>短均線可變持股已完成</h2><p>20～25檔與不足時回原動能，使10／20在原944窗口可成立936個、10／24可成立937個。費後增益校正區間仍跨0，改善覆蓋尚不代表改善收益。</p><p><a href="'+v49_html.name+'">查看同日期箱形圖、費後比較與重新納入的壞行情</a></p></section>'
fp=v34_root/'策略研究_總結報告.html';txt=fp.read_text(encoding='utf-8');assert 'id="variable-ma-v49"' not in txt;anchor='<section id="market-selection-v48">';assert anchor in txt;fp.write_text(txt.replace(anchor,note+anchor,1),encoding='utf-8')
fp=v34_root/'策略研究_總結報告.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v49 可變持股與動能回退\n\n四新版本8944窗口完成，覆蓋改善但未證實收益增益。['+v49_html.name+']('+v49_html.name+')\n',encoding='utf-8')
fp=v34_root/'市場狀態_六策略比較與逐年選擇_v48_2026-10-17.html';txt=fp.read_text(encoding='utf-8');fp.write_text(txt.replace('<h1>',note+'<h1>',1),encoding='utf-8')
fp=v34_root/'A策略_無人值守研究流程.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v49 可變持股與動能回退（完成）\n\n20～25檔及候選不足回原動能，共四新增8944窗口；舊有效窗口收益／費用完整一致，恢復kernel只讀快取且零網路。覆蓋由729／789提高936／937（原944窗口），對動能增益校正區間跨0，未證實改良。成果：'+v49_html.name+'。未改v48狀態對映或啟用排程。\n',encoding='utf-8')
v49_stats['validation']={'frozen_input_hashes':5,'fixed_controls':10,'all_old_valid_identical':v49_checks,'plots':2,'kernel':'kernel-aicup-ma-variable-20261017.json','network_requests':0,'new_window_runs':8944,'visual_browser_QA':'not performed'}
(v34_root/'可變持股均線_統計與覆蓋_v49_2026-10-17.json').write_text(json.dumps(v49_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['variable_ma_v49'].update(status='completed_coverage_improved_no_proven_mean_gain',running=False,report=str(v49_html),main_common=v49_main['n'],validation=v49_stats['validation']);sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
assert v49_html.read_text(encoding='utf-8').count('Plotly.newPlot(')>=2
print('V49_PUBLISHED',{'report':v49_html.name,'main_common':v49_main['n'],'metrics':v49_main['metrics'],'checks':v49_checks,'new_runs':8944})


In [ ]:

import sys,os,json,gzip,datetime,hashlib,ast,urllib.request,urllib.parse
from research_layout import ResearchPath as Path
import numpy as np
import nbformat
import plotly.graph_objects as go
v50_root=Path('C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料');v34_root=v50_root;sp46=v50_root/'A策略_無人值守研究狀態.json'
v50_book=nbformat.read(v50_root/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4)
print('V50_KERNEL',{'pid':os.getpid(),'python':sys.executable,'scope':'saved results only; no trade rerun'})
with gzip.open(v50_root/'外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz','rt',encoding='utf-8') as f:v50_price=json.load(f)
v30_days=sorted({datetime.date.fromisoformat(r['date']) for r in v50_price['prices']['2330']});del v50_price
v30_starts=np.arange(61,len(v30_days)-23);v30_later=np.array([v30_days[int(s)].year>=2019 for s in v30_starts])
saved=json.loads((v50_root/'長歷史環境研究_動能法人逐窗口_v30_2026-10-16.json').read_text(encoding='utf-8'))['results'];v46_results={'MOM_E0':saved['MOM'],'FLOW_E0':saved['FLOW']};del saved
for k in ['PURE_50_200','FLOW_50_200_ACTIVE']:
    with gzip.open(v50_root/('均線交叉_'+k+'_v45_2026-10-17.json.gz'),'rt',encoding='utf-8') as f:v46_results[k]=json.load(f)['results']
for k in ['10_20_FALLBACK','10_24_FALLBACK']:
    with gzip.open(v50_root/('可變持股均線_'+k+'_v49_2026-10-17.json.gz'),'rt',encoding='utf-8') as f:v46_results[k]=json.load(f)['results']
v50_keys=['MOM_E0','FLOW_E0','FLOW_50_200_ACTIVE','PURE_50_200','10_20_FALLBACK','10_24_FALLBACK'];v46_valid={k:np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in rows]) for k,rows in v46_results.items()};v46_ret={k:np.array([r['return'] if r['return'] is not None else np.nan for r in rows]) for k,rows in v46_results.items()};v50_common=np.logical_and.reduce([v46_valid[k] for k in v50_keys]);assert int((v50_common&v30_later).sum())==935
for name in ['metrics_short46','bootstrap48','pct47','table47']:
    for c in v50_book.cells:
        if c.cell_type=='code' and 'def '+name+'(' in c.source:
            node=next(n for n in ast.parse(c.source).body if isinstance(n,ast.FunctionDef) and n.name==name);exec(compile(ast.Module(body=[node],type_ignores=[]),'<saved-analysis>','exec'),globals());break
def serial30(x):
    if isinstance(x,np.ndarray):return x.tolist()
    if isinstance(x,np.generic):return x.item()
    if isinstance(x,(datetime.date,Path)):return str(x)
    raise TypeError(type(x).__name__)
v50_definition={'version':'v50_benchmark_fixed1','date':'2026-10-19','keys':v50_keys,'sample':'samev49 allsixcommon, later2019..2025 primary, all2016..2025 available secondary','primary_benchmark':'TAIEX total return index, includes dividends, raw100%index no investmentfees; not investable ETF or officialcompetitionbenchmark certification','secondary':'TAIEX priceindex ifavailable, clearlywithoutdividends; no use150proxyasofficialindex','alignment':'index previous tradingdayclose before strategy firstsession to close of24thsession, exact24dailyreturns and all25datedlevels; no forwardfill','metrics':'absolute win net>0; beatmarket net>indexreturn; tie separately; mean strategy net,marketreturn,arithmetic excess percentagepoints; not guaranteed alpha','fee_asymmetry':'strategy after actual modelcosts versus costless100%index;90%strategy exposure not rescaled;no claim exposureadjustedalpha','statistics':'primary6excessmean+6beatprobminus.5, family12;24/48originalposition circularblocks20000Bonferroni12;overlap andretrospective limits','sources':['https://finmind.github.io/tutor/TaiwanMarket/Technical/','https://www.twse.com.tw/downloads/zh/products/indices/IndexS02.pdf'],'network':'try new documented public FinMind index source; no retry previously blockedTWSE endpoints; at most3meaningfulattempts per newsource'}
(v50_root/'策略大盤比較_定義_v50_2026-10-19.json').write_text(json.dumps(v50_definition,ensure_ascii=False,indent=2),encoding='utf-8');v50_sources={};v50_raw={}
for key,ds in [('TAIEX_TR','TaiwanStockTotalReturnIndex'),('TAIEX_PRICE','TaiwanStockPrice')]:
    fp=v50_root/('策略大盤比較_'+key+'_原始_v50_2026-10-19.json.gz');params={'dataset':ds,'data_id':'TAIEX','start_date':'2016-07-01','end_date':'2025-12-31'};url='https://api.finmindtrade.com/api/v4/data?'+urllib.parse.urlencode(params)
    if fp.exists():
        with gzip.open(fp,'rt',encoding='utf-8') as f:doc=json.load(f)
    else:
        try:
            with urllib.request.urlopen(url,timeout=40) as resp:doc=json.loads(resp.read(15000000))
        except Exception as exc:doc={'status':'failed','error_type':type(exc).__name__}
        with gzip.open(fp,'wt',encoding='utf-8') as f:json.dump(doc,f,ensure_ascii=False)
    rows=doc.get('data',[]);v50_raw[key]=rows;v50_sources[key]={'url':url,'status':doc.get('status'),'rows':len(rows),'fields':list(rows[0]) if rows else [],'first':min((r['date'] for r in rows),default=None),'last':max((r['date'] for r in rows),default=None),'error_type':doc.get('error_type'),'sha256':hashlib.sha256(fp.read_bytes()).hexdigest()}
(v50_root/'策略大盤比較_來源查核_v50_2026-10-19.json').write_text(json.dumps(v50_sources,ensure_ascii=False,indent=2),encoding='utf-8')
print('V50_SOURCE',v50_sources)


In [ ]:

v50_bench={};v50_lookups={}
for key,rows in v50_raw.items():
    lookup={r['date']:float(r['price'] if key=='TAIEX_TR' else r['close']) for r in rows};assert len(lookup)==len(rows);assert all(np.isfinite(x) and x>0 for x in lookup.values());v50_lookups[key]=lookup
    vals=[]
    for s0 in v30_starts:
        s=int(s0);days=[str(d) for d in v30_days[s-1:s+24]];levels=[lookup.get(d,np.nan) for d in days];vals.append(levels[-1]/levels[0]-1 if np.all(np.isfinite(levels)) else np.nan)
    v50_bench[key]=np.array(vals)
v50_mask=v50_common&v30_later&np.isfinite(v50_bench['TAIEX_TR']);assert int(v50_mask.sum())==935
v50_years=np.array([v30_days[int(s)].year for s in v30_starts]);v50_stats={'definition':v50_definition,'sources':v50_sources,'periods':{},'per_year':{},'paired_tests':[],'validation':{'all_dates_match_price_calendar':{k:set(lookup)==set(map(str,v30_days)) for k,lookup in v50_lookups.items()},'primary_common_unchanged':935,'new_trade_runs':0,'benchmark_window':'previousclose through 24th sessionclose,25consecutivelevels','provider_samples':{k:{d:lookup.get(d) for d in ['2016-07-01','2020-04-06','2020-04-07','2020-04-08','2021-07-02','2025-12-31']} for k,lookup in v50_lookups.items()}}}
def summarize_benchmark50(mask):
    result={'n':int(mask.sum()),'start_range':[str(v30_days[int(v30_starts[i])]) for i in [np.flatnonzero(mask)[0],np.flatnonzero(mask)[-1]]],'benchmarks':{},'methods':{}}
    for bk,br in v50_bench.items():
        m=mask&np.isfinite(br);result['benchmarks'][bk]={'n':int(m.sum()),'win':float(np.mean(br[m]>0)),'mean':float(np.mean(br[m]))}
    for k in v50_keys:
        ret=v46_ret[k];info=metrics_short46(k,mask);info['benchmark']={}
        for bk,br in v50_bench.items():
            m=mask&np.isfinite(br);delta=ret[m]-br[m];info['benchmark'][bk]={'n':int(m.sum()),'beat_win':float(np.mean(delta>0)),'tie_n':int(np.sum(delta==0)),'excess_mean':float(np.mean(delta)),'market_mean':float(np.mean(br[m])),'positive_but_underperform_n':int(np.sum((ret[m]>0)&(delta<0))),'negative_but_outperform_n':int(np.sum((ret[m]<0)&(delta>0)))}
        result['methods'][k]=info
    return result
for period,pm in [('later',v30_later),('all',np.ones(2236,bool))]:v50_stats['periods'][period]=summarize_benchmark50(v50_common&pm&np.isfinite(v50_bench['TAIEX_TR']))
for y in range(2019,2026):
    m=v50_mask&(v50_years==y)
    if m.any():v50_stats['per_year'][str(y)]=summarize_benchmark50(m)
dd=[];nn=[];spec50=[]
for k in v50_keys:
    diff=v46_ret[k]-v50_bench['TAIEX_TR']
    for metric,values in [('mean_excess',diff),('beat_probability_minus_half',(diff>0).astype(float)-.5)]:dd.append(np.where(v50_mask,np.nan_to_num(values),0));nn.append(v50_mask.astype(float));spec50.append((k,metric))
ci50=bootstrap48(np.stack(dd,axis=1),np.stack(nn,axis=1),[24,48],12)
for j,(k,metric) in enumerate(spec50):v50_stats['paired_tests'].append({'key':k,'metric':metric,'estimate':float(np.sum(dd[j])/np.sum(nn[j])),'ci24':ci50['24'][j],'ci48':ci50['48'][j],'both_lower_positive':bool(min(ci50['24'][j][0],ci50['48'][j][0])>0)})
v50_windows=[{'index':i,'start':str(v30_days[int(s)]),'prior_close_date':str(v30_days[int(s)-1]),'end':str(v30_days[int(s)+23]),'common_six':bool(v50_common[i]),'later':bool(v30_later[i]),'benchmarks':{k:float(br[i]) if np.isfinite(br[i]) else None for k,br in v50_bench.items()},'methods':{k:{'valid':bool(v46_valid[k][i]),'net':float(v46_ret[k][i]) if v46_valid[k][i] else None,'excess_tr':float(v46_ret[k][i]-v50_bench['TAIEX_TR'][i]) if v46_valid[k][i] and np.isfinite(v50_bench['TAIEX_TR'][i]) else None} for k in v50_keys}} for i,s in enumerate(v30_starts)]
for file,obj in [('策略大盤比較_統計_v50_2026-10-19.json',v50_stats),('策略大盤比較_逐窗口_v50_2026-10-19.json',v50_windows)]: (v50_root/file).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
print('V50_COMPARED',{'main':v50_stats['periods']['later'],'tests':v50_stats['paired_tests'],'sample_levels':v50_stats['validation']['provider_samples']})


In [ ]:

v50_names=dict(zip(v50_keys,['原動能20／60各半','純法人買超','法人＋50／200交叉換股','純50／200均線強度','10／20可變持股＋回退','10／24可變持股＋回退']))
# Independent official point check from the retrieved TWSE July2021 report, not a fabricated source series.
official50={'2021-07-01':34360.80,'2021-07-02':34353.52,'2021-07-05':34762.44}
assert all(abs(v50_lookups['TAIEX_TR'][d]-v)<.005 for d,v in official50.items())
v50_stats['validation']['official_point_check']={'source':'https://www.twse.com.tw/indicesReport/MFI94U?date=20210702&response=html','points':official50,'passed':True,'scope':'three sampled points, not independent full-series audit'}
v50_stats['validation']['documentation_sample_checks']={'2020-04-06':abs(v50_lookups['TAIEX_TR']['2020-04-06']-18356.5)<.06,'2020-04-07':abs(v50_lookups['TAIEX_TR']['2020-04-07']-18688.6)<.06,'2020-04-08':abs(v50_lookups['TAIEX_TR']['2020-04-08']-18952.7)<.06}
def main_table50(period='later'):
    result=v50_stats['periods'][period];rows=[]
    for k in v50_keys:
        d=result['methods'][k];b=d['benchmark']['TAIEX_TR'];rows.append([v50_names[k],d['n'],pct47(d['win']),pct47(b['beat_win']),pct47(d['ev']),pct47(b['market_mean']),pct47(b['excess_mean']),pct47(d['worst_mdd'])])
    return table47(['策略','同日期窗口','正報酬勝率（＞0）','跑贏大盤勝率（＞含息指數）','費後24日期望','同期大盤24日期望','平均超額（百分點）','最差窗口內回撤'],rows)
v50_table=main_table50();v50_note='<p>新增2026-10-19大盤比較：同一935窗口，基準為含股息的加權股價報酬指數。大盤由進場前一交易日收盤算到第24交易日收盤；策略為費後報酬，基準為不含投資費用的100%指數。策略目標投入90%，未做曝險等比例調整。這不是ETF回測或風險調整alpha。</p>'
parts50=['<h1>策略有賺錢，也有跑贏大盤嗎？</h1><p>v50｜2026-10-19｜2016～2025資料；主要比較2019～2025可用935共同窗口。沿用已保存交易路徑，不重跑策略。</p><h2>同一張表，同時看兩種勝率</h2>'+v50_note+v50_table,'<p>正報酬勝率＝策略費後報酬&gt;0；跑贏大盤勝率＝策略費後報酬&gt;同期含息大盤報酬。相等另外計數（本批均為0），不算勝。平均超額為策略減大盤的算術報酬差，單位百分點。大盤在這935窗口正報酬勝率70.48%，平均24日報酬2.17%。因此策略有六至七成賺錢，不代表六至七成跑贏市場。</p>', '<h2>如何解讀？</h2><ul><li>原動能：正報酬66.31%，但跑贏市場50.59%，平均超額+0.31個百分點；尚未顯示穩定超額證據。</li><li>純法人：正報酬66.10%，跑贏市場37.86%，平均超額−0.78個百分點。先前「高勝率」僅是相對0，不能直接當成優於大盤。</li><li>純50／200：跑贏市場63.74%，平均超額+1.48個百分點，是這張表較突出的候選。但最差窗口內回撤−32.59%，並未因此變成低風險策略。</li><li>10／20回退、10／24回退：跑贏市場58.61%、55.51%，平均超額為正，但校正區間仍跨0。</li></ul>']
fig=go.Figure()
for k in v50_keys:
    vals=(v46_ret[k][v50_mask]-v50_bench['TAIEX_TR'][v50_mask])*100;cd=[[v50_windows[i]['start'],v50_windows[i]['end'],float(v46_ret[k][i]*100),float(v50_bench['TAIEX_TR'][i]*100)] for i in np.flatnonzero(v50_mask)]
    fig.add_trace(go.Box(y=vals,name=v50_names[k],boxpoints='all',jitter=.35,pointpos=0,marker={'size':3,'opacity':.45},customdata=cd,hovertemplate='%{customdata[0]}～%{customdata[1]}<br>策略費後 %{customdata[2]:.2f}%<br>含息大盤 %{customdata[3]:.2f}%<br>超額 %{y:.2f}百分點<extra>%{fullData.name}</extra>'))
    fig.add_trace(go.Scatter(x=[v50_names[k]],y=[float(np.median(vals))],mode='markers',marker={'symbol':'line-ew','size':25,'color':'#d81b60','line':{'color':'#d81b60','width':3}},showlegend=False,hovertemplate='超額中位數 %{y:.2f}百分點<extra></extra>'))
fig.update_layout(title='935共同窗口：相對含息大盤超額分布｜全部點；粉紅為中位數',height=600,showlegend=False,yaxis_title='費後策略減含息大盤（百分點）',margin={'b':150});fig.add_hline(y=0,line_color='#222',line_width=2)
parts50.append(('fig',0))
parts50.append('<h2>不是只看平均值：區間與年份</h2><p>以原始日期位置作24／48日循環區塊重抽樣，各20,000次；六策略的平均超額、勝率相對50%共12項Bonferroni校正。重疊窗口不能視為935次獨立比賽，且同段歷史已研究多輪，本次校正不能消除跨輪搜尋偏誤。</p>'+table47(['策略','平均超額','24日區塊校正區間','48日區塊校正區間','平均超額兩區間均正？'],[[v50_names[d['key']],pct47(d['estimate']),f"{pct47(d['ci24'][0])}～{pct47(d['ci24'][1])}",f"{pct47(d['ci48'][0])}～{pct47(d['ci48'][1])}",'是（本批）' if d['both_lower_positive'] else '否'] for d in v50_stats['paired_tests'] if d['metric']=='mean_excess']))
parts50.append('<p>純50／200的平均超額在本批兩種區塊區間均為正（約+0.18～+2.85及+0.13～+2.90百分點）；但其跑贏機率高於50%的校正證據未同時通過兩種區塊。不能把「平均超額為正」與「穩定多數時候勝出」視為同一件事，更不能稱已證實未來有效或全面優於原動能。</p>')
parts50.append(table47(['起點年份','共同窗口','大盤期望']+[v50_names[k]+'超額' for k in v50_keys],[[y,r['n'],pct47(r['benchmarks']['TAIEX_TR']['mean'])]+[pct47(r['methods'][k]['benchmark']['TAIEX_TR']['excess_mean']) for k in v50_keys] for y,r in v50_stats['per_year'].items()]))
parts50.append('<h2>不含股息的價格指數：補充比較</h2><p>常見新聞加權指數是不含股息的價格指數，以下另列敏感性，不取代主表含息基準。</p>'+table47(['策略','跑贏價格指數勝率','價格指數平均24日報酬','相對價格指數平均超額（百分點）'],[[v50_names[k],pct47(d['benchmark']['TAIEX_PRICE']['beat_win']),pct47(d['benchmark']['TAIEX_PRICE']['market_mean']),pct47(d['benchmark']['TAIEX_PRICE']['excess_mean'])] for k,d in v50_stats['periods']['later']['methods'].items()]))
parts50.append('<h2>全部可用歷史：另外列出分母</h2><p>原價量範圍2016～2025，但各策略暖機、候選不足及品質排除後的共同日期不連續；這不是完整十年每日都可交易。下表分母與主要935窗口不同，不混用。</p>'+main_table50('all'))
parts50.append('<h2>資料來源與核對</h2><p>新增兩次公開查詢，FinMind提供TAIEX報酬指數與價格指數，各2,320個交易日（2016-07-01～2025-12-31）；全部日期和原股票日曆一致，2236候選窗口皆有25個連續指數收盤值，未補值。原主表935窗口全數保留。FinMind報酬指數2021-07-01、02、05共3點與證交所原始月表一致；另核對文件2020-04-06～08範例的顯示精度。這是抽樣交叉驗證，不代表全序列獨立稽核。</p><ul><li><a href="https://finmind.github.io/tutor/TaiwanMarket/Technical/">FinMind指數資料定義與API</a></li><li><a href="https://www.twse.com.tw/indicesReport/MFI94U?date=20210702&response=html">證交所2021年7月報酬指數月表</a></li><li><a href="https://www.twse.com.tw/downloads/zh/products/indices/IndexS02.pdf">證交所TAIEX價格／報酬指數編製說明</a></li></ul>')
parts50.append('<h2>使用限制</h2><p>策略含上市與上櫃標的，TAIEX衡量上市市場，並非完全相同資產範圍。策略有90%目標投入及交易費用，指數為100%無費用參考，因此表中不是相同曝險的因果歸因、可交易ETF報酬或風險調整alpha。指數起點為前收盤，策略實際成交為首日均價，兩者首日價格基礎不完全一致，已明列；不捏造指數均價。官方競賽基準仍需官賽文件認證。現況150檔名單／存活偏差、未知公司行動及正式限制缺口沿用。沒有重跑舊回測、重啟可用kernel或提交交易。</p>')
body='';first=True
for x in parts50:
    if isinstance(x,tuple):body+=fig.to_html(full_html=False,include_plotlyjs=True if first else False);first=False
    else:body+=x
v50_html=v50_root/'策略研究_對大盤勝率與超額報酬_v50_2026-10-19.html';v50_html.write_text('<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>對大盤勝率與超額報酬</title><style>body{font:16px/1.8 system-ui;max-width:1550px;margin:30px auto;padding:0 24px;color:#233043}table{border-collapse:collapse;width:100%;font-size:14px;margin:18px 0}td,th{padding:9px;border:1px solid #ccd5df;text-align:right}td:first-child,th:first-child{text-align:left}th{background:#eef3f8}h2{margin-top:40px}p{max-width:1250px}</style>'+body+'</html>',encoding='utf-8');v50_html.with_suffix('.md').write_text('# 策略對大盤勝率與超額報酬\n\n'+'\n\n'.join(x for x in parts50 if isinstance(x,str)),encoding='utf-8')
note='<section id="benchmark-v50"><h2>新增：與0%及大盤同表比較</h2>'+v50_note+v50_table+'<p><a href="'+v50_html.name+'">查看超額箱形圖、年份與校正區間</a></p></section>'
fp=v50_root/'短均線_可變持股與動能回退_v49_2026-10-17.html';txt=fp.read_text(encoding='utf-8');start=txt.index('<h2>二、六種可用候選');t0=txt.index('<table>',start);t1=txt.index('</table>',t0)+len('</table>');assert 'benchmark-v50' not in txt;txt=txt[:t0]+v50_note+v50_table+'<p><a href="'+v50_html.name+'">新增大盤比較完整報告</a></p>'+txt[t1:];fp.write_text(txt,encoding='utf-8')
fp=v50_root/'短均線_可變持股與動能回退_v49_2026-10-17.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## 新增大盤比較（2026-10-19）\n\n'+v50_note+v50_table+'\n\n詳見 '+v50_html.name,encoding='utf-8')
fp=v50_root/'策略研究_總結報告.html';txt=fp.read_text(encoding='utf-8');anchor='<section id="variable-ma-v49">';assert anchor in txt and 'id="benchmark-v50"' not in txt;fp.write_text(txt.replace(anchor,note+anchor,1),encoding='utf-8')
fp=v50_root/'策略研究_總結報告.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## 對大盤勝率與超額報酬 v50\n\n'+v50_note+v50_table+'\n\n完整報告：['+v50_html.name+']('+v50_html.name+')。此後摘要同時列正報酬勝率、相對含息大盤勝率及平均超額；缺基準日期標未知。\n',encoding='utf-8')
fp=v50_root/'A策略_無人值守研究流程.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v50 對大盤勝率（完成）\n\n新取得FinMind TAIEX總報酬與價格指數，各2320日；原935共同窗口全數精確配對。與0%勝率、跑贏含息大盤勝率、費後期望及平均超額同表。後續摘要沿用此口徑；不能以150檔均值替正式大盤。六策略平均超額／勝率共12項、24／48日區塊20000次；純50／200平均超額兩區間正，勝率門檻未同時通過。舊多輪搜尋偏誤及風險取捨保留，不宣稱通用alpha。報告：'+v50_html.name+'。不重跑交易或啟用排程。\n',encoding='utf-8')
v50_stats['validation'].update(html_figures=1,report_table_rows=6,notebook_saved=True,new_trade_runs=0,visual_browser_QA='not performed');(v50_root/'策略大盤比較_統計_v50_2026-10-19.json').write_text(json.dumps(v50_stats,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['benchmark_v50']={'status':'completed','running':False,'report':str(v50_html),'kernel':'kernel-aicup-benchmark-20261019.json','common_later':935,'benchmark':'TAIEX total return index','new_trade_runs':0,'validation':v50_stats['validation']};sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V50_PUBLISHED',{'report':v50_html.name,'primary_n':935,'all_n':v50_stats['periods']['all']['n'],'tables_updated':['v49','master'],'official_points':3})


In [ ]:

# Presentation-only update. Reuse verified benchmark metrics; no recalculation or trade run.
import re
v50_row_updates=[]
for stem in ['策略研究_對大盤勝率與超額報酬_v50_2026-10-19','短均線_可變持股與動能回退_v49_2026-10-17','策略研究_總結報告']:
    for ext in ['.html','.md']:
        fp=v50_root/(stem+ext);txt=fp.read_text(encoding='utf-8');counter=[0]
        def add_market_row50(match):
            block=match.group(0)
            if '跑贏大盤勝率（＞含息指數）' not in block or '大盤（含股息；指數未扣投資費用）' in block:return block
            n=int(re.search(r'<tbody><tr><td>.*?</td><td>(\d+)</td>',block).group(1));period=next(x for x in v50_stats['periods'].values() if x['n']==n);b=period['benchmarks']['TAIEX_TR'];cells=['大盤（含股息；指數未扣投資費用）',str(n),pct47(b['win']),'—（自身基準）',pct47(b['mean']),pct47(b['mean']),'0.00%','—'];row='<tr>'+''.join('<td>'+c+'</td>' for c in cells)+'</tr>';counter[0]+=1;return block.replace('</tbody>',row+'</tbody>')
        updated=re.sub(r'<table>.*?</table>',add_market_row50,txt,flags=re.S);assert counter[0]>=1,(fp.name,counter);fp.write_text(updated,encoding='utf-8');v50_row_updates.append({'file':fp.name,'updated_tables':counter[0]})
for item in v50_row_updates:
    txt=(v50_root/item['file']).read_text(encoding='utf-8');tables=re.findall(r'<table>.*?</table>',txt,flags=re.S)
    for table in tables:
        if '跑贏大盤勝率（＞含息指數）' in table:
            rows=re.findall(r'<tbody>(.*?)</tbody>',table,flags=re.S)[0];assert rows.count('<tr>')==7;assert re.findall(r'<tr>(.*?)</tr>',rows,flags=re.S)[-1].startswith('<td>大盤（含股息；指數未扣投資費用）</td>')
print('V50_MARKET_ROW_ADDED',v50_row_updates)


In [ ]:

# User-approved conclusion, reuse existing initial-hold paths; no new strategy or trade simulation.
v51_keys=['MOM_HOLD','FLOW_HOLD'];v51_hold={}
for k in v51_keys:
    with gzip.open(v50_root/('統一長歷史_'+k+'_v34_2026-10-16.json.gz'),'rt',encoding='utf-8') as f:cache=json.load(f)
    v51_hold[k]=cache['results'];v46_results[k]=cache['results'];v46_valid[k]=np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in cache['results']]);v46_ret[k]=np.array([r['return'] if r['return'] is not None else np.nan for r in cache['results']])
v51_pairs={};v51_rows=[]
for h,b in [('MOM_HOLD','MOM_E0'),('FLOW_HOLD','FLOW_E0')]:
    mask=v30_later&v46_valid[h]&v46_valid[b]&np.isfinite(v50_bench['TAIEX_TR']);n=int(mask.sum());assert n==({'MOM_HOLD':1177,'FLOW_HOLD':1143}[h]);info={'n':n,'methods':{},'mean_delta':float(np.mean(v46_ret[h][mask]-v46_ret[b][mask])),'benchmark_mean':float(np.mean(v50_bench['TAIEX_TR'][mask]))}
    for k in [b,h]:
        mm=metrics_short46(k,mask);diff=v46_ret[k][mask]-v50_bench['TAIEX_TR'][mask];mm.update(beat_market=float(np.mean(diff>0)),excess_mean=float(np.mean(diff)),market_mean=info['benchmark_mean']);info['methods'][k]=mm
        name={'MOM_HOLD':'動能初始持有24日','FLOW_HOLD':'法人初始持有24日','MOM_E0':'原動能／五日','FLOW_E0':'純法人／五日'}[k];v51_rows.append([('動能配對' if h=='MOM_HOLD' else '法人配對'),name,n,pct47(mm['win']),pct47(mm['beat_market']),pct47(mm['ev']),pct47(mm['market_mean']),pct47(mm['excess_mean']),pct47(mm['fee']),pct47(mm['worst_mdd'])])
    v51_pairs[h+'|'+b]=info
v51_text='本次24交易日競賽，策略重心放在賽前選股與初始資金配置，賽中採低換手管理，避免因短期波動頻繁更換持股。Agent持續追蹤原投資理由是否失效；只有在證據充分、預期效益足以支付交易成本時，才考慮局部調整；市場共同風險升高時，評估在官方限制內降低曝險。'
v51_evidence=[['賽前選股與配置','有研究依據的方向','固定策略間仍有收益／風險差異；不能由初始選股推論第一天決定勝負。'],['賽中低換手','研究支持優先檢視','法人初始持有在v34通過當批完整準則；動能初始持有收益較高但尾端回撤更深。'],['重大失效才局部處理','待驗證操作原則','重大失效、何時調整與股數尚未鎖定；不能聲稱停損有正增益。'],['市場風險升高時降曝險','待驗證風控方向','v48未建立有效切換；降低投入可能少虧，也可能錯失反彈，需符合現金與持股限制。']]
v51_section='<section id="convergence-v51"><h2>目前收斂的比賽策略走向（2026-10-19）</h2><blockquote>'+v51_text+'</blockquote><p>這是研究支持的策略方向，不是已證實的完整官賽交易規則。「低換手」不等於無條件不交易；「風險處理」也不代表已找到有效停損訊號。</p>'+table47(['方向','證據定位','依據／限制'],v51_evidence)
v51_section+='<h3>補回既有證據：初始持有其實已測過</h3><p>先前對話說還要比較初始持有與五日調整，遺漏了v34已完成的研究，現予更正。本表直接引用舊交易路徑，新增與含息大盤的比較，不重跑策略。初始持有版本只在第一天買入、24日不再主動調整，仍逐日檢查現金、20～30檔持股、集中度及公司行動；遇問題仍不可判定。未在窗口結束額外強制清倉，故其約0.13%費用主要是初始買入費，不能當作完整買賣一趟成本。</p>'+table47(['配對組','策略','共同窗口','＞0勝率','跑贏含息大盤勝率','費後24日期望','大盤期望','平均超額（百分點）','平均費稅／初始資金','最差窗口內回撤'],v51_rows)
v51_section+='<p>動能1177與法人1143是各自配對交集，不能跨兩組直接排勝負，也不與主表935窗口混比。v34當時：動能持有相對五日增益約+0.713百分點、48日校正區間[+0.131,+1.347]，但最差回撤−33.68%較五日−27.65%差，未通過全面改善準則；法人持有增益約+0.738百分點、48日校正區間[+0.231,+1.241]，當批通過。這些舊區間僅適用當批比較，不是對新增大盤超額的區間，也不消除後續多輪搜尋偏誤。</p>'
v51_section+='<h3>候選收斂與下一個決策</h3><ul><li>保留原動能／五日作基準；「動能初始持有」是低換手候選，風險取捨明列。</li><li>保留純50／200作固定選股挑戰者，其大盤超額表現值得追蹤，但回撤更深；50／200初始持有及例外處理未在本批新增。</li><li>法人若保留，應同時看初始持有版本；不能只用法人／五日較弱的相對大盤結果否定整個法人家族。</li><li>10／20可變＋回退留為操作備選；10／24繼續保留結果，沒有額外擴增參數。</li></ul><p>收斂順序：先決定收益與最大可承受損失的取捨，再鎖定初始策略與例行調整；最後才驗證少數明確失效條件。不得把LLM自由判讀直接當作已驗證停損。現在不自行新增全面實驗網格，亦不啟動排程。</p><p>目前仍未完成：完整官賽限制認證、純50／200持有／必要調整的隔離比較，以及重大失效觸發條件。股票池存活偏差、重疊窗口、已看過歷史及費用／曝險基準差異持續保留。</p></section>'
v51_files=[]
for stem in ['策略研究_總結報告','策略研究_對大盤勝率與超額報酬_v50_2026-10-19']:
    for ext in ['.html','.md']:
        fp=v50_root/(stem+ext);txt=fp.read_text(encoding='utf-8');assert 'id="convergence-v51"' not in txt
        if ext=='.html':
            if stem=='策略研究_總結報告':anchor='<section id="benchmark-v50">';assert anchor in txt;txt=txt.replace(anchor,v51_section+anchor,1)
            else:anchor='<h2>同一張表，同時看兩種勝率</h2>';assert anchor in txt;txt=txt.replace(anchor,v51_section+anchor,1)
        else:txt+='\n\n## 比賽策略走向與持有證據收斂 v51\n\n'+v51_section+'\n'
        fp.write_text(txt,encoding='utf-8');v51_files.append(fp.name)
v51_record={'version':'v51_convergence','date':'2026-10-19','status':'discussion_integrated_existing_hold_audited','approved_direction':v51_text,'evidence':v51_evidence,'hold_pairwise':v51_pairs,'source_existing':'統一長歷史_批次完成總報告_v39_2026-10-16.md and v34 cached path files','new_trade_runs':0,'new_inference_tests':0,'not_proven':['initialselectiondecideseverything','automaticstoplossbenefit','regimeswitchingbenefit','cashguaranteesprotection'],'not_tested':['PURE50_200_HOLD','lockedmaterialfailuretrigger'],'reports':v51_files}
(v50_root/'策略研究_收斂決策與持有核對_v51_2026-10-19.json').write_text(json.dumps(v51_record,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
fp=v50_root/'A策略_無人值守研究流程.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v51 討論整合與持有核對（完成）\n\n採用「賽前選股配置、賽中低換手、證據充分才例外調整」為研究方向，停損與降曝險觸發仍待驗證。補回v34已測初始持有的證據，重新配對既有1177動能／1143法人窗口與含息大盤，新增交易回測0。更新總結與v50；不能宣稱初始選股定生死或停損有效。純50／200持有與例外處理尚未新增，排程不變。\n',encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['convergence_v51']={'status':'completed_discussion_and_hold_evidence','running':False,'new_trade_runs':0,'reports':v51_files,'record':'策略研究_收斂決策與持有核對_v51_2026-10-19.json'};sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
assert all('id="convergence-v51"' in (v50_root/f).read_text(encoding='utf-8') for f in v51_files)
print('V51_CONVERGED',{'files':v51_files,'hold_pairwise':v51_pairs,'new_trade_runs':0})


In [ ]:

# User-approved conclusion, reuse existing initial-hold paths; no new strategy or trade simulation.
v51_keys=['MOM_HOLD','FLOW_HOLD'];v51_hold={}
for k in v51_keys:
    with gzip.open(v50_root/('統一長歷史_'+k+'_v34_2026-10-16.json.gz'),'rt',encoding='utf-8') as f:cache=json.load(f)
    v51_hold[k]=cache['results'];v46_results[k]=cache['results'];v46_valid[k]=np.array([r['return'] is not None and not r['invalid'] and r['violations']==0 for r in cache['results']]);v46_ret[k]=np.array([r['return'] if r['return'] is not None else np.nan for r in cache['results']])
v51_pairs={};v51_rows=[]
for h,b in [('MOM_HOLD','MOM_E0'),('FLOW_HOLD','FLOW_E0')]:
    mask=v30_later&v46_valid[h]&v46_valid[b]&np.isfinite(v50_bench['TAIEX_TR']);n=int(mask.sum());assert n==({'MOM_HOLD':1177,'FLOW_HOLD':1143}[h]);info={'n':n,'methods':{},'mean_delta':float(np.mean(v46_ret[h][mask]-v46_ret[b][mask])),'benchmark_mean':float(np.mean(v50_bench['TAIEX_TR'][mask]))}
    for k in [b,h]:
        rr=v46_ret[k][mask];ix=np.flatnonzero(mask);mm={'n':n,'win':float(np.mean(rr>0)),'ev':float(np.mean(rr)),'p10':float(np.quantile(rr,.1)),'worst_mdd':float(min(v46_results[k][i]['mdd'] for i in ix)),'fee':float(np.mean([v46_results[k][i]['cost'] for i in ix]))};diff=rr-v50_bench['TAIEX_TR'][mask];mm.update(beat_market=float(np.mean(diff>0)),excess_mean=float(np.mean(diff)),market_mean=info['benchmark_mean']);info['methods'][k]=mm
        name={'MOM_HOLD':'動能初始持有24日','FLOW_HOLD':'法人初始持有24日','MOM_E0':'原動能／五日','FLOW_E0':'純法人／五日'}[k];v51_rows.append([('動能配對' if h=='MOM_HOLD' else '法人配對'),name,n,pct47(mm['win']),pct47(mm['beat_market']),pct47(mm['ev']),pct47(mm['market_mean']),pct47(mm['excess_mean']),pct47(mm['fee']),pct47(mm['worst_mdd'])])
    v51_pairs[h+'|'+b]=info
v51_text='本次24交易日競賽，策略重心放在賽前選股與初始資金配置，賽中採低換手管理，避免因短期波動頻繁更換持股。Agent持續追蹤原投資理由是否失效；只有在證據充分、預期效益足以支付交易成本時，才考慮局部調整；市場共同風險升高時，評估在官方限制內降低曝險。'
v51_evidence=[['賽前選股與配置','有研究依據的方向','固定策略間仍有收益／風險差異；不能由初始選股推論第一天決定勝負。'],['賽中低換手','研究支持優先檢視','法人初始持有在v34通過當批完整準則；動能初始持有收益較高但尾端回撤更深。'],['重大失效才局部處理','待驗證操作原則','重大失效、何時調整與股數尚未鎖定；不能聲稱停損有正增益。'],['市場風險升高時降曝險','待驗證風控方向','v48未建立有效切換；降低投入可能少虧，也可能錯失反彈，需符合現金與持股限制。']]
v51_section='<section id="convergence-v51"><h2>目前收斂的比賽策略走向（2026-10-19）</h2><blockquote>'+v51_text+'</blockquote><p>這是研究支持的策略方向，不是已證實的完整官賽交易規則。「低換手」不等於無條件不交易；「風險處理」也不代表已找到有效停損訊號。</p>'+table47(['方向','證據定位','依據／限制'],v51_evidence)
v51_section+='<h3>補回既有證據：初始持有其實已測過</h3><p>先前對話說還要比較初始持有與五日調整，遺漏了v34已完成的研究，現予更正。本表直接引用舊交易路徑，新增與含息大盤的比較，不重跑策略。初始持有版本只在第一天買入、24日不再主動調整，仍逐日檢查現金、20～30檔持股、集中度及公司行動；遇問題仍不可判定。未在窗口結束額外強制清倉，故其約0.13%費用主要是初始買入費，不能當作完整買賣一趟成本。</p>'+table47(['配對組','策略','共同窗口','＞0勝率','跑贏含息大盤勝率','費後24日期望','大盤期望','平均超額（百分點）','平均費稅／初始資金','最差窗口內回撤'],v51_rows)
v51_section+='<p>動能1177與法人1143是各自配對交集，不能跨兩組直接排勝負，也不與主表935窗口混比。v34當時：動能持有相對五日增益約+0.713百分點、48日校正區間[+0.131,+1.347]，但最差回撤−33.68%較五日−27.65%差，未通過全面改善準則；法人持有增益約+0.738百分點、48日校正區間[+0.231,+1.241]，當批通過。這些舊區間僅適用當批比較，不是對新增大盤超額的區間，也不消除後續多輪搜尋偏誤。</p>'
v51_section+='<h3>候選收斂與下一個決策</h3><ul><li>保留原動能／五日作基準；「動能初始持有」是低換手候選，風險取捨明列。</li><li>保留純50／200作固定選股挑戰者，其大盤超額表現值得追蹤，但回撤更深；50／200初始持有及例外處理未在本批新增。</li><li>法人若保留，應同時看初始持有版本；不能只用法人／五日較弱的相對大盤結果否定整個法人家族。</li><li>10／20可變＋回退留為操作備選；10／24繼續保留結果，沒有額外擴增參數。</li></ul><p>收斂順序：先決定收益與最大可承受損失的取捨，再鎖定初始策略與例行調整；最後才驗證少數明確失效條件。不得把LLM自由判讀直接當作已驗證停損。現在不自行新增全面實驗網格，亦不啟動排程。</p><p>目前仍未完成：完整官賽限制認證、純50／200持有／必要調整的隔離比較，以及重大失效觸發條件。股票池存活偏差、重疊窗口、已看過歷史及費用／曝險基準差異持續保留。</p></section>'
v51_files=[]
for stem in ['策略研究_總結報告','策略研究_對大盤勝率與超額報酬_v50_2026-10-19']:
    for ext in ['.html','.md']:
        fp=v50_root/(stem+ext);txt=fp.read_text(encoding='utf-8');assert 'id="convergence-v51"' not in txt
        if ext=='.html':
            if stem=='策略研究_總結報告':anchor='<section id="benchmark-v50">';assert anchor in txt;txt=txt.replace(anchor,v51_section+anchor,1)
            else:anchor='<h2>同一張表，同時看兩種勝率</h2>';assert anchor in txt;txt=txt.replace(anchor,v51_section+anchor,1)
        else:txt+='\n\n## 比賽策略走向與持有證據收斂 v51\n\n'+v51_section+'\n'
        fp.write_text(txt,encoding='utf-8');v51_files.append(fp.name)
v51_record={'version':'v51_convergence','date':'2026-10-19','status':'discussion_integrated_existing_hold_audited','approved_direction':v51_text,'evidence':v51_evidence,'hold_pairwise':v51_pairs,'source_existing':'統一長歷史_批次完成總報告_v39_2026-10-16.md and v34 cached path files','new_trade_runs':0,'new_inference_tests':0,'not_proven':['initialselectiondecideseverything','automaticstoplossbenefit','regimeswitchingbenefit','cashguaranteesprotection'],'not_tested':['PURE50_200_HOLD','lockedmaterialfailuretrigger'],'reports':v51_files}
(v50_root/'策略研究_收斂決策與持有核對_v51_2026-10-19.json').write_text(json.dumps(v51_record,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
fp=v50_root/'A策略_無人值守研究流程.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v51 討論整合與持有核對（完成）\n\n採用「賽前選股配置、賽中低換手、證據充分才例外調整」為研究方向，停損與降曝險觸發仍待驗證。補回v34已測初始持有的證據，重新配對既有1177動能／1143法人窗口與含息大盤，新增交易回測0。更新總結與v50；不能宣稱初始選股定生死或停損有效。純50／200持有與例外處理尚未新增，排程不變。\n',encoding='utf-8')
state=json.loads(sp46.read_text(encoding='utf-8'));state['convergence_v51']={'status':'completed_discussion_and_hold_evidence','running':False,'new_trade_runs':0,'reports':v51_files,'record':'策略研究_收斂決策與持有核對_v51_2026-10-19.json'};sp46.write_text(json.dumps(state,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
assert all('id="convergence-v51"' in (v50_root/f).read_text(encoding='utf-8') for f in v51_files)
print('V51_CONVERGED',{'files':v51_files,'hold_pairwise':v51_pairs,'new_trade_runs':0})


In [ ]:

# Editorial reconstruction from verified v50/v51 outputs. No new strategy or inference tests.
import html,re
from research_layout import ResearchPath as Path
import json,gzip,datetime
import numpy as np
import plotly.graph_objects as go
root52=Path('C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料')
d52=json.loads((root52/'策略大盤比較_統計_v50_2026-10-19.json').read_text(encoding='utf-8'));c52=json.loads((root52/'策略研究_收斂決策與持有核對_v51_2026-10-19.json').read_text(encoding='utf-8'));w52=json.loads((root52/'策略大盤比較_逐窗口_v50_2026-10-19.json').read_text(encoding='utf-8'))
keys52=d52['definition']['keys'];names52=dict(zip(keys52,['原動能20／60各半','純法人買超','法人＋50／200','純50／200均線強度','10／20可變＋回退','10／24可變＋回退']));main52=d52['periods']['later'];assert main52['n']==935
def p52(x):return f'{x*100:.2f}%'
def tab52(headers,rows):return '<div class="table-wrap"><table><thead><tr>'+''.join('<th>'+str(x)+'</th>' for x in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+str(x)+'</td>' for x in row)+'</tr>' for row in rows)+'</tbody></table></div>'
rows52=[]
for k in keys52:
    d=main52['methods'][k];b=d['benchmark']['TAIEX_TR'];rows52.append([names52[k],p52(d['win']),p52(b['beat_win']),p52(d['ev']),f'{b["excess_mean"]*100:+.2f}',p52(d['worst_mdd'])])
b52=main52['benchmarks']['TAIEX_TR'];rows52.append(['<strong>大盤（含股息）</strong>',p52(b52['win']),'—',p52(b52['mean']),'0.00','—'])
table52=tab52(['策略','正報酬勝率','跑贏大盤勝率','24日期望','超額／百分點','最差窗口內回撤'],rows52)
defs52=[['原動能20／60各半','20日與60日漲幅排名各占50%，選前25檔','每五日重選與配置'],['純法人買超','過去20日外資＋投信淨買超股數／成交量排名，選25檔','每五日重選與配置'],['法人＋50／200','法人排名，加上50日均線高於200日均線的資格篩選','五日調整；死亡交叉嘗試替補'],['純50／200均線強度','在50日均線高於200日均線者中，依兩均線比值強度選25檔','五日調整；死亡交叉嘗試替補'],['10／20、10／24可變＋回退','正向均線強度排名，允許20～25檔；不足最低檔數或無法完整替補時回原動能','五日重評；死亡交叉觸發替補／回退']]
holdrows52=[]
for pair,label in [('MOM_HOLD|MOM_E0','動能：1177共同窗口'),('FLOW_HOLD|FLOW_E0','法人：1143共同窗口')]:
    obj=c52['hold_pairwise'][pair]
    for k in [pair.split('|')[1],pair.split('|')[0]]:
        z=obj['methods'][k];holdrows52.append([label,'初始持有24日' if k.endswith('HOLD') else '每五日調整',p52(z['win']),p52(z['beat_market']),p52(z['ev']),f'{z["excess_mean"]*100:+.2f}',p52(z['fee']),p52(z['worst_mdd'])])
holdtable52=tab52(['獨立配對組','持有方式','正報酬勝率','勝大盤','24日期望','超額／百分點','平均費稅','最差回撤'],holdrows52)
fig52=go.Figure();maskrows52=[r for r in w52 if r['common_six'] and r['later']];assert len(maskrows52)==935;nb52=7;bounds52={}
for mode in ['excess','net']:
    data52=[]
    for k in keys52+['market']:
        vals=np.array([r['benchmarks']['TAIEX_TR'] if k=='market' else r['methods'][k]['net'] for r in maskrows52]);market=np.array([r['benchmarks']['TAIEX_TR'] for r in maskrows52]);ys=(vals-market if mode=='excess' else vals)*100;data52.extend(ys.tolist());name='大盤（含股息）' if k=='market' else names52[k]
        custom=[[r['start'],r['end'],float(v*100),float(m*100)] for r,v,m in zip(maskrows52,vals,market)];vis=mode=='excess'
        fig52.add_trace(go.Box(y=ys,name=name,boxpoints='all',jitter=.28,pointpos=0,marker={'size':3,'opacity':.35,'color':'#386da8' if k!='market' else '#808c9c'},line={'color':'#386da8' if k!='market' else '#808c9c'},customdata=custom,visible=vis,showlegend=False,hovertemplate='%{customdata[0]}～%{customdata[1]}<br>策略／基準 %{customdata[2]:.2f}%<br>同期大盤 %{customdata[3]:.2f}%<br>本圖數值 %{y:.2f}<extra>%{fullData.name}</extra>'))
        fig52.add_trace(go.Scatter(x=[name],y=[float(np.median(ys))],mode='markers',marker={'symbol':'line-ew','size':24,'color':'#d81b60','line':{'color':'#d81b60','width':3}},visible=vis,showlegend=False,hovertemplate='中位數 %{y:.2f}<extra></extra>'))
    bounds52[mode]=[float(np.floor(min(data52)/5)*5),float(np.ceil(max(data52)/5)*5)]
buttons52=[]
for mode,label,title in [('excess','相對大盤','超額報酬（百分點）'),('net','原始報酬','24日報酬（%）')]:
    visible=[(mode=='excess')]*14+[(mode=='net')]*14;buttons52.append({'label':label,'method':'update','args':[{'visible':visible},{'yaxis.title.text':title,'yaxis.range':bounds52[mode]}]})
fig52.update_layout(template='plotly_white',height=610,margin={'t':85,'b':135,'l':65,'r':20},yaxis={'title':'超額報酬（百分點）','range':bounds52['excess']},updatemenus=[{'type':'buttons','direction':'right','x':0,'y':1.15,'buttons':buttons52}],font={'family':'system-ui','color':'#233043'},hovermode='closest');fig52.add_hline(y=0,line_color='#253246',line_width=2)
chart52=fig52.to_html(full_html=False,include_plotlyjs=True,div_id='report-distribution')
intervalrows52=[]
for item in d52['paired_tests']:
    if item['metric']=='mean_excess':intervalrows52.append([names52[item['key']],f'{item["estimate"]*100:+.2f}',f'{item["ci24"][0]*100:+.2f}～{item["ci24"][1]*100:+.2f}',f'{item["ci48"][0]*100:+.2f}～{item["ci48"][1]*100:+.2f}'])
yearrows52=[[y,r['n'],p52(r['benchmarks']['TAIEX_TR']['mean'])]+[f'{r["methods"][k]["benchmark"]["TAIEX_TR"]["excess_mean"]*100:+.2f}' for k in keys52] for y,r in d52['per_year'].items()]
body52='''<header><span class="eyebrow">24交易日競賽 · 策略研究總結</span><h1>先選好股票，再決定何時值得調整</h1><p class="lead">比較選股、交易頻率與大盤表現，收斂本次比賽的策略方向。</p><p class="meta">更新：2026-10-19｜資料範圍：2016～2025｜主要比較：2019～2025的935個共同窗口</p></header><nav><a href="#question">研究問題</a><a href="#comparison">策略比較</a><a href="#holding">持有與調整</a><a href="#direction">比賽方向</a><a href="#decision">待定事項</a></nav>'''
body52+='''<section id="question"><h2>1．我們要回答什麼？</h2><p>在只有24個交易日的比賽中，哪種選股值得採用？賽中調整能否增加費後收益，還是應以初始配置與續抱為主？</p><div class="callout"><strong>目前方向：賽前重視選股與配置，賽中控制換手，只有充分理由才例外調整。</strong><p>低換手值得優先考慮，但有效停損、降曝險與最佳持有方式仍未全部定案。</p></div><p>主表所有策略使用相同935個起點、每次持有24交易日。策略均以90%股票投入為目標，沿用整千股、成交均價與交易費模型。以含股息的加權報酬指數作市場基準。</p></section>'''
body52+='<section id="comparison"><h2>2．選股策略：有賺錢，也有跑贏大盤嗎？</h2>'+table52+'<p class="caption">正報酬勝率：費後報酬＞0。勝大盤：費後報酬＞同期含息指數。期望：24日平均報酬。超額：策略減大盤的百分點差。回撤是最差窗口內值，不是連續十年的回撤。大盤未扣投資費用；策略投入與基準曝險不同。</p><div class="findings"><p><strong>純50／200是主要挑戰者。</strong>在這組日期中，勝率、平均收益與勝大盤比例較高，但最差回撤也更深。</p><p><strong>原動能保留為基準。</strong>相對大盤勝率約五成，沒有因正報酬勝率較高就證明穩定超額。</p><p><strong>法人與短均線不直接淘汰，也不強選。</strong>法人五日版多數時候落後大盤；短均線回退版改善可執行性，但優於原動能的增益尚未證實。</p></div>'+chart52+'<p class="caption">兩個按鈕切換相同日期的報酬／超額；每個窗口均顯示，粉紅標記為中位數，黑線為0。所有策略在同一視圖使用相同縱軸。</p><details><summary>查看策略怎麼選、怎麼調整</summary>'+tab52(['策略','選股依據','調整方式'],defs52)+'<p>10／20等數字是均線天數，不是資金比例。動能「各半」是兩個排名訊號各50%。固定25檔時每檔目標3.6%；20檔時每檔4.5%。法人＋均線是排名加篩選，不是資金各半。回退版規則與原固定25檔版本分開。</p></details><p>純50／200平均超額在本批24／48日區塊校正區間均為正；但「勝大盤機率高於50%」未同時通過兩種區塊檢查。這是較值得追蹤的證據，仍不是未來有效或全面最佳的保證。</p></section>'
body52+='<section id="holding"><h2>3．持有還是調整：先看已做過的比較</h2><p>初始持有與五日調整已在既有研究完成。下面各組使用自己的共同日期，只能組內比較，不能直接和上方935窗口排名。</p>'+holdtable52+'<p class="caption">兩組皆初始投入90%。初始持有為第一天買入後24日不主動調整，窗口結束未另加強制清倉；平均費用約0.13%主要是買入費，並非完整來回交易費。</p><p><strong>動能續抱有收益與回撤的取捨：</strong>平均期望提高約0.71百分點，但最差回撤從−27.65%加深至−33.68%，不能只憑平均收益改採持有。</p><p><strong>法人續抱優於法人頻繁調整的證據較明確：</strong>當批改善準則通過；不過持有版平均仍略低於同期含息大盤，不能等同於已找到優於市場的策略。</p><p>先前測過的跌破退出、條件加碼與市場狀態切換，尚未顯示穩定的整體改善。成本、錯失反彈與候選不足都可能影響結果，不能全歸因於交易費。</p></section>'
body52+='''<section id="direction"><h2>4．比賽策略走向</h2><blockquote>賽前選擇適合24日持有的股票與初始配置；賽中採低換手管理。Agent持續追蹤投資理由，只有證據充分、預期效益足以支付成本時，才局部調整；共同風險升高時，評估在官方限制內降低曝險。</blockquote>'''+tab52(['階段','操作方向','目前證據定位'],[['賽前','選股、投入比例與風險檢查','有研究依據的重點'],['正常持有期間','避免因小幅波動反覆換股','低換手值得優先驗證，並非無條件不交易'],['個股重大失效','局部減碼或替換','觸發條件及效益仍待驗證'],['市場共同風險','在規則內降低總曝險','保留方向，未證實有可靠擇時訊號']])+'''<p>現金是風險緩衝，不保證抵銷股票損失；初始策略也不決定一切。依本機歸檔規則，仍須每日20～30檔、現金低於25%，台積電及其他個股分別受25%／10%上限約束，不能任意全數退出。</p></section>'''
body52+='''<section id="decision"><h2>5．目前保留什麼，還差什麼？</h2>'''+tab52(['角色','候選','保留理由／取捨'],[['比較基準','原動能20／60各半、五日調整','定義清楚；用來衡量新增規則是否值得'],['主要挑戰者','純50／200均線強度','相對大盤表現突出，需接受較深歷史回撤'],['低換手候選','動能初始持有；法人初始持有','已有實驗，分別看尾端風險及市場超額'],['操作備選','10／20可變＋回退','候選不足時較能執行，未證實收益優於原動能']])+'''<p>決策尚差三件事：<strong>可承受多大回撤、採哪一種例行調整方式、哪些明確證據值得例外交易。</strong>純50／200的初始持有／必要調整尚未完成隔離比較；已測過的動能與法人持有不重做。最後需確認正式比賽限制與帳戶資料。</p><p>本輪不再擴增均線網格，也不憑最高平均值宣布贏家。先依收益與風險取捨縮小候選，再鎖定有限規則，交由後續新資料驗證。</p></section>'''
body52+='''<section id="limits"><h2>6．這份結論能用到哪裡？</h2><p>資料使用目前150檔名單的歷史子集，存在名單與存活偏差；共同窗口並不連續，且互相重疊，不是935次獨立比賽。歷史已被多輪研究，不能當作全新未見樣本。策略包括上櫃股票，大盤TAIEX主要衡量上市市場；90%目標投入的費後策略與100%無費用指數，不是相同曝險的alpha比較。</p><p>指數從進場前一日收盤算至第24日收盤，策略以首日均價成交，首日計價基礎不同。未知價格與公司行動仍標不可判定；Active Share及官方成交作廢等限制尚未完整認證。</p></section><section class="appendix"><h2>附錄：查核與細節</h2><details><summary>逐年超額報酬（935共同窗口；百分點）</summary>'''+tab52(['起點年','窗口','大盤期望']+[names52[k] for k in keys52],yearrows52)+'''</details><details><summary>平均超額報酬的校正區間</summary><p>24／48日循環區塊重抽樣各20,000次；六策略平均超額及勝率共12項Bonferroni校正。區間是平均差的不確定性，不是下一次比賽收益範圍，亦不消除過往多輪搜尋偏誤。</p>'''+tab52(['策略','平均超額','24日區塊區間','48日區塊區間'],intervalrows52)+'''</details><details><summary>初始持有的既有證據與資料來源</summary><p>初始持有引用既有v34路徑與v39批次報告；動能平均差的當批48日區間為+0.131～+1.347百分點，但風險準則未通過；法人為+0.231～+1.241百分點，當批完整準則通過。這些區間不是對大盤超額的區間。</p><p>大盤價與報酬指數由FinMind取得2016-07-01～2025-12-31共2320日，日期與股票日曆一致；官方2021年7月報酬指數3點抽查一致，並非全序列獨立稽核。</p><ul><li><a href="https://finmind.github.io/tutor/TaiwanMarket/Technical/">FinMind資料定義</a></li><li><a href="https://www.twse.com.tw/indicesReport/MFI94U?date=20210702&response=html">證交所報酬指數月表</a></li><li><a href="https://www.twse.com.tw/downloads/zh/products/indices/IndexS02.pdf">證交所指數編製說明</a></li><li><a href="統一長歷史_批次完成總報告_v39_2026-10-16.md">初始持有及交易頻率原始批次結果</a></li><li><a href="短均線_可變持股與動能回退_v49_2026-10-17.html">可變持股原始研究</a></li><li><a href="策略大盤比較_統計_v50_2026-10-19.json">大盤比較完整統計</a></li></ul></details></section><footer>本次為報告重整，沒有新增策略、重算交易或啟動排程。主表與既有935窗口數字一致。</footer>'''
css52='''*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:#f4f6f9;color:#223146;font:16px/1.8 system-ui,-apple-system,"Microsoft JhengHei",sans-serif}main{max-width:1320px;margin:auto;padding:40px 28px}header{padding:20px 0 26px}h1{font-size:36px;line-height:1.35;margin:12px 0;color:#152c45}h2{font-size:24px;margin:0 0 18px;color:#193c5d}p{margin:12px 0}.eyebrow{color:#386da8;font-size:14px;letter-spacing:1px}.lead{font-size:19px}.meta,.caption,footer{color:#617185;font-size:14px}nav{display:flex;flex-wrap:wrap;gap:10px;margin-bottom:24px}nav a{padding:7px 14px;border:1px solid #d6e0ea;border-radius:20px;text-decoration:none;background:white}a{color:#245e98}section{background:white;padding:28px;border:1px solid #e0e7ee;border-radius:12px;margin:0 0 22px;scroll-margin-top:20px}.callout{background:#eaf2f8;border-left:4px solid #386da8;padding:17px 20px;margin:20px 0}.callout p{margin-bottom:0}.table-wrap{overflow-x:auto;margin:20px 0}table{border-collapse:collapse;width:100%;font-size:14px;font-variant-numeric:tabular-nums}th{background:#eef3f8;color:#2a4762}th,td{padding:12px 10px;border-bottom:1px solid #e0e7ee;text-align:right;vertical-align:top}td:first-child,th:first-child{text-align:left}tbody tr:last-child{background:#f8fafc}tr:hover{background:#f2f6fa}blockquote{margin:16px 0;padding:20px 24px;border-left:4px solid #386da8;background:#f3f7fb;font-size:18px}.findings{border-top:1px solid #e0e7ee;padding-top:8px}details{border:1px solid #e0e7ee;border-radius:8px;margin:14px 0;padding:14px 18px}summary{cursor:pointer;font-weight:600;color:#294e70}details p{font-size:14px}footer{padding:5px 0 24px}@media(max-width:700px){main{padding:20px 12px}section{padding:20px 15px}h1{font-size:28px}h2{font-size:21px}table{min-width:760px}}'''
document52='<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>24交易日競賽策略研究總結</title><style>'+css52+'</style></head><body><main>'+body52+'</main></body></html>'
for stem in ['策略研究_對大盤勝率與超額報酬_v50_2026-10-19','策略研究_總結報告']:
    fp=root52/(stem+'.html');backup=root52/(stem+'_整理前_2026-10-19.html')
    if not backup.exists():backup.write_bytes(fp.read_bytes())
    fp.write_text(document52,encoding='utf-8')
    # Save a readable plain Markdown companion without embedded Plotly or styling.
md52='# 24交易日競賽策略研究總結\n\n更新：2026-10-19\n\n## 研究方向\n\n賽前重視選股與配置，賽中採低換手管理；只有充分理由才例外調整。有效停損、降曝險與最佳持有方式仍未全部定案。\n\n## 同935窗口策略比較\n\n|策略|正報酬勝率|跑贏含息大盤勝率|24日期望|平均超額（百分點）|最差窗口內回撤|\n|---|---:|---:|---:|---:|---:|\n'
md52+='\n'.join('|'+ '|'.join(re.sub('<[^>]+>','',str(x)) for x in row)+'|' for row in rows52)+'\n\n## 主要發現\n\n純50／200為主要挑戰者，但回撤更深；原動能保留基準；短均線回退改善覆蓋但未證實優於原動能；法人五日版平均落後市場。\n\n## 初始持有與五日調整\n\n'+holdtable52+'\n\n兩組分母不同，僅組內比較。動能持有增益約0.71百分點，但最差回撤加深；法人持有當批通過改善準則，但平均仍略低於同期含息大盤。持有版未在窗口末額外強制清倉。\n\n## 比賽策略走向\n\n賽前選股、投入比例及風險檢查；賽中低換手；重大失效才局部處理；市場共同風險時在規則內評估降曝險。後兩者尚須驗證觸發與效益。\n\n## 尚待定案\n\n可承受回撤、例行調整方式、例外交易條件。純50／200持有與必要調整未完成隔離比較，正式官賽限制仍待認證。\n\n## 限制\n\n現況150檔歷史子集有選擇與存活偏差；重疊窗口非獨立事件；歷史已多輪研究；策略90%目標投入並扣費，TAIEX為上市市場100%無費用含息基準，非相同曝險alpha。指數前收盤與策略首日均價基礎不同。\n\n完整圖表、年度及校正區間見[互動報告](策略研究_對大盤勝率與超額報酬_v50_2026-10-19.html)。\n'
for stem in ['策略研究_對大盤勝率與超額報酬_v50_2026-10-19','策略研究_總結報告']:
    fp=root52/(stem+'.md');backup=root52/(stem+'_整理前_2026-10-19.md')
    if not backup.exists():backup.write_bytes(fp.read_bytes())
    fp.write_text(md52,encoding='utf-8')
record52={'status':'completed','date':'2026-10-19','scope':'editorial restructure; no new backtest','source_stats':['策略大盤比較_統計_v50_2026-10-19.json','策略研究_收斂決策與持有核對_v51_2026-10-19.json'],'sections':6,'primary_common':935,'primary_strategy_rows':6,'market_row_last':True,'chart_modes':['excess','net'],'points_per_mode':935*7,'new_trade_runs':0,'old_versions_preserved':True,'visual_browser_QA':'not performed'}
assert document52.count('<section')==7 and document52.count('id="report-distribution"')==1 and '<body>' in document52
assert len(fig52.data)==28 and all(len(fig52.data[i].y)==935 for i in range(0,28,2))
assert all(all(p52(d52['periods']['later']['methods'][k][f]) in document52 for f in ['win','ev','worst_mdd']) for k in keys52)
statep52=root52/'A策略_無人值守研究狀態.json';state52=json.loads(statep52.read_text(encoding='utf-8'));state52['report_restructure_v52']=record52;statep52.write_text(json.dumps(state52,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
fp=root52/'A策略_無人值守研究流程.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v52 總結報告重整（完成）\n\n依研究問題→同日期選股比較→持有／調整→比賽方向→待定事項→限制重整。v50與固定總結HTML／Markdown同步，版本細節折疊至附錄；整理前副本保留。主表935窗口與既有數字不變，含大盤末列、全點箱形圖兩模式。沒有新實驗或排程。\n',encoding='utf-8')
print('V52_RESTRUCTURED',record52)


In [ ]:

# Align all section2 and section3 methods to one exact date set; no new trade simulation.
import copy,collections
root53=v50_root;keys53=v50_keys+['MOM_HOLD','FLOW_HOLD'];mask53=v30_later&np.logical_and.reduce([v46_valid[k] for k in keys53])&np.isfinite(v50_bench['TAIEX_TR']);assert int(mask53.sum())==912
def metric53(k,m):
    vals=v46_ret[k][m];ids=np.flatnonzero(m);rows=v46_results[k];market=v50_bench['TAIEX_TR'][m];delta=vals-market
    return {'n':int(m.sum()),'win':float(np.mean(vals>0)),'ev':float(np.mean(vals)),'median':float(np.median(vals)),'p10':float(np.quantile(vals,.1)),'worst_mdd':float(min(rows[i]['mdd'] for i in ids)),'fee':float(np.mean([rows[i]['cost'] for i in ids])),'turnover':float(np.mean([rows[i]['turnover'] for i in ids])),'beat_market':float(np.mean(delta>0)),'excess_mean':float(np.mean(delta)),'market_mean':float(np.mean(market))}
d53=copy.deepcopy(v50_stats);bm53={bk:{'n':912,'win':float(np.mean(br[mask53]>0)),'mean':float(np.mean(br[mask53]))} for bk,br in v50_bench.items()};methods53={}
for k in v50_keys:
    z=metric53(k,mask53);z['benchmark']={}
    for bk,br in v50_bench.items():
        delta=v46_ret[k][mask53]-br[mask53];z['benchmark'][bk]={'n':912,'beat_win':float(np.mean(delta>0)),'excess_mean':float(np.mean(delta)),'market_mean':float(np.mean(br[mask53]))}
    methods53[k]=z
ids53=np.flatnonzero(mask53);d53['periods']['later']={'n':912,'start_range':[str(v30_days[int(v30_starts[i])]) for i in [ids53[0],ids53[-1]]],'benchmarks':bm53,'methods':methods53};d53['per_year']={}
for year in range(2019,2026):
    m=mask53&(v50_years==year)
    if not m.any():continue
    d53['per_year'][str(year)]={'n':int(m.sum()),'benchmarks':{'TAIEX_TR':{'mean':float(np.mean(v50_bench['TAIEX_TR'][m]))}},'methods':{k:{'benchmark':{'TAIEX_TR':{'excess_mean':float(np.mean(v46_ret[k][m]-v50_bench['TAIEX_TR'][m]))}}} for k in v50_keys}}
c53=copy.deepcopy(c52);c53['hold_pairwise']={}
for h,b in [('MOM_HOLD','MOM_E0'),('FLOW_HOLD','FLOW_E0')]:c53['hold_pairwise'][h+'|'+b]={'n':912,'methods':{k:metric53(k,mask53) for k in [h,b]},'mean_delta':float(np.mean(v46_ret[h][mask53]-v46_ret[b][mask53]))}
spec53=[];dd53=[];nn53=[]
for k in v50_keys:
    diff=v46_ret[k]-v50_bench['TAIEX_TR']
    for metric,values in [('mean_excess',diff),('beat_probability_minus_half',(diff>0).astype(float)-.5)]:spec53.append((k,metric));dd53.append(np.where(mask53,np.nan_to_num(values),0));nn53.append(mask53.astype(float))
for h,b in [('MOM_HOLD','MOM_E0'),('FLOW_HOLD','FLOW_E0')]:spec53.append((h,'hold_minus_fiveday'));dd53.append(np.where(mask53,np.nan_to_num(v46_ret[h]-v46_ret[b]),0));nn53.append(mask53.astype(float))
ci53=bootstrap48(np.stack(dd53,axis=1),np.stack(nn53,axis=1),[24,48],14);tests53=[]
for j,(k,metric) in enumerate(spec53):tests53.append({'key':k,'metric':metric,'estimate':float(np.sum(dd53[j])/912),'ci24':ci53['24'][j],'ci48':ci53['48'][j],'both_lower_positive':bool(min(ci53['24'][j][0],ci53['48'][j][0])>0)})
d53['paired_tests']=tests53;d53['definition']['statistics']='same912dates;6excess+6beatprob+2holdminus5day=14family;24/48blocks20000Bonferroni14';d53['definition']['sample']='all8methods same912 primary2019..2025;old935 remains archived'
excluded53=[]
for i in np.flatnonzero(v50_mask&~mask53):
    excluded53.append({'index':int(i),'start':v50_windows[i]['start'],'end':v50_windows[i]['end'],'failed_methods':{k:{'reasons':v46_results[k][i]['reasons'],'violations':v46_results[k][i]['violations']} for k in ['MOM_HOLD','FLOW_HOLD'] if not v46_valid[k][i]}})
reasons53={k:{'invalid_in935':int((v50_mask&~v46_valid[k]).sum()),'reason_counts':dict(collections.Counter(r for i in np.flatnonzero(v50_mask&~v46_valid[k]) for r in v46_results[k][i]['reasons'])),'violation_windows':sum(v46_results[k][i]['violations']>0 for i in np.flatnonzero(v50_mask&~v46_valid[k]))} for k in ['MOM_HOLD','FLOW_HOLD']}
audit53={'status':'completed','original_main':935,'common_sections2and3':912,'removed_union':23,'method_exclusions':reasons53,'excluded_dates':excluded53,'included_indices':ids53.tolist(),'all8_metrics':{k:metric53(k,mask53) for k in keys53},'tests':tests53,'new_trade_runs':0,'scope':'same snapshot, exactsamewindowandbenchmark;thisisnotoriginalcomplete2016..2025coverage'}
for fname,obj in [('同樣本比較_主表統計_v53_2026-10-19.json',d53),('同樣本比較_持有統計_v53_2026-10-19.json',c53),('同樣本比較_日期稽核_v53_2026-10-19.json',audit53)]: (root53/fname).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=serial30,allow_nan=False),encoding='utf-8')
# Reuse the clean report renderer with current aligned inputs. Keep previous935 report as snapshot.
book53=nbformat.read(root53/'動能持股_開賽前公告配置研究_2026-10-14.ipynb',as_version=4);src53=book53.cells[392].source
src53=src53.replace('策略大盤比較_統計_v50_2026-10-19.json','同樣本比較_主表統計_v53_2026-10-19.json').replace('策略研究_收斂決策與持有核對_v51_2026-10-19.json','同樣本比較_持有統計_v53_2026-10-19.json')
src53=src53.replace('935','912').replace('1177共同窗口','912共同窗口').replace('1143共同窗口','912共同窗口')
src53=src53.replace("maskrows52=[r for r in w52 if r['common_six'] and r['later']]","maskrows52=[r for r in w52 if r['index'] in set(ids53.tolist())]")
src53=src53.replace('下面各組使用自己的共同日期，只能組內比較，不能直接和上方912窗口排名。','第2節與本節全部使用相同912個日期、相同大盤與原始交易路徑，可以直接對照。')
src53=src53.replace('兩組分母不同，僅組內比較。','兩組及上方主表均為同912窗口，可直接比較。')
src53=src53.replace('平均期望提高約0.71百分點，但最差回撤從−27.65%加深至−33.68%，不能只憑平均收益改採持有。',f'同日期平均期望差為{c53["hold_pairwise"]["MOM_HOLD|MOM_E0"]["mean_delta"]*100:+.2f}百分點；最差回撤從{metric53("MOM_E0",mask53)["worst_mdd"]*100:.2f}%變成{metric53("MOM_HOLD",mask53)["worst_mdd"]*100:.2f}%，不能只憑平均收益改採持有。')
src53=src53.replace('當批改善準則通過；不過持有版平均仍略低於同期含息大盤，不能等同於已找到優於市場的策略。',f'此912同日期期望差為{c53["hold_pairwise"]["FLOW_HOLD|FLOW_E0"]["mean_delta"]*100:+.2f}百分點；是否跑贏市場及風險取捨須看上表。舊批次曾通過改善準則，不自動代表本批完整準則通過。')
src53=src53.replace('12項Bonferroni校正','14項Bonferroni校正（含兩項持有差）')
src53=src53.replace('純50／200平均超額在本批24／48日區塊校正區間均為正；但「勝大盤機率高於50%」未同時通過兩種區塊檢查。這是較值得追蹤的證據，仍不是未來有效或全面最佳的保證。','所有區間已按912共同窗口及14項比較重新計算，見附錄；主表排序仍不代表未來有效或全面最佳。')
src53=src53.replace('初始持有引用既有v34路徑與v39批次報告；動能平均差的當批48日區間為+0.131～+1.347百分點，但風險準則未通過；法人為+0.231～+1.241百分點，當批完整準則通過。這些區間不是對大盤超額的區間。','初始持有引用既有v34交易路徑，本頁統計改用912共同窗口。舊1177／1143配對批次結論保留在原始報告；不沿用舊區間來解釋新樣本。持有減五日調整的新區間見下方。')
src53=src53.replace('動能持有增益約0.71百分點，但最差回撤加深；法人持有當批通過改善準則，但平均仍略低於同期含息大盤。','動能與法人持有的收益、費用及回撤改依同912窗口計算，舊批次門檻通過不自動沿用至新樣本。')
src53=src53.replace("'_整理前_2026-10-19.html'","'_對齊912前_2026-10-19.html'").replace("'_整理前_2026-10-19.md'","'_對齊912前_2026-10-19.md'")
exec(compile(src53,'<same-sample-report>','exec'),globals())
extra53='<details><summary>為什麼從935改為912個共同窗口？</summary><p>為讓選股與持有方式同表對照，兩節都取八種方法有效日期交集。動能初始持有缺5個、法人初始持有缺18個，合計排除23個窗口；未補造報酬。下表原因可重疊，完整日期見日期稽核JSON。</p>'+tab52(['方法','原935內無效','原因','有操作限制違規的窗口'],[[k,d['invalid_in935'],str(d['reason_counts']),d['violation_windows']] for k,d in reasons53.items()])+'<p>縮小交集仍有有效日期篩選偏差；912表示可比性更高，不表示所有市場環境都涵蓋完整。</p></details>'
holdci53=tab52(['持有減五日調整','平均差／百分點','24日區塊校正區間','48日區塊校正區間'],[['動能' if d['key']=='MOM_HOLD' else '法人',f'{d["estimate"]*100:+.2f}',f'{d["ci24"][0]*100:+.2f}～{d["ci24"][1]*100:+.2f}',f'{d["ci48"][0]*100:+.2f}～{d["ci48"][1]*100:+.2f}'] for d in tests53 if d['metric']=='hold_minus_fiveday'])
for stem in ['策略研究_對大盤勝率與超額報酬_v50_2026-10-19','策略研究_總結報告']:
    fp=root53/(stem+'.html');txt=fp.read_text(encoding='utf-8');txt=txt.replace('<details><summary>逐年超額報酬',extra53+'<details><summary>逐年超額報酬',1);txt=txt.replace('<p>初始持有引用既有v34交易路徑',holdci53+'<p>初始持有引用既有v34交易路徑',1);fp.write_text(txt,encoding='utf-8')
    fp=root53/(stem+'.md');fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## 同樣本日期稽核與持有配對區間\n\n'+extra53+'\n'+holdci53+'\n',encoding='utf-8')
state53=json.loads(sp46.read_text(encoding='utf-8'));state53['same_sample_v53']={'status':'completed','running':False,'old_n':935,'new_n':912,'new_trade_runs':0,'family':14,'date_audit':'同樣本比較_日期稽核_v53_2026-10-19.json'};sp46.write_text(json.dumps(state53,ensure_ascii=False,indent=2,default=serial30),encoding='utf-8')
print('V53_ALIGNED',{'n':912,'exclusions':reasons53,'hold':c53['hold_pairwise'],'tests':tests53})


In [ ]:

# User-requested local data organization; preserve paths and publication repository.
from research_layout import ResearchPath as Path
import json,hashlib,html,re,datetime,collections,urllib.parse,os
root54=Path('C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料');notebook54='動能持股_開賽前公告配置研究_2026-10-14.ipynb'
author54=Path('C:/Users/weiha/Documents/Codex/2026-10-09/data-mining-final-project/work/author_external_history_v12.py');bridge54=author54.with_name('notebook_kernel_bridge.py')
core54=[('閱讀入口','策略研究_總結報告.html','主要總結；目前第2、3節同912窗口。'),('閱讀入口','策略研究_對大盤勝率與超額報酬_v50_2026-10-19.html','與總結同步的固定入口；檔名v50，內容已整合至同樣本v53。'),('閱讀入口','策略研究_總結報告.md','文字版總結。'),('計算紀錄',notebook54,'原Notebook，含輸入及輸出；持續追加，不自動Restart/Run All。'),('目前統計','同樣本比較_主表統計_v53_2026-10-19.json','目前主表912日期統計；內含沿用舊欄位，報告主表只取periods.later，勿把all欄當本輪共同樣本。'),('目前統計','同樣本比較_持有統計_v53_2026-10-19.json','初始持有與五日調整同912窗口。'),('日期依據','同樣本比較_日期稽核_v53_2026-10-19.json','912起點索引、從935排除的23窗口、八策略統計及14項區間。'),('逐窗口基礎','策略大盤比較_逐窗口_v50_2026-10-19.json','完整候選2236起點；common_six是舊935集合，須用v53 included_indices取目前912。'),('大盤來源','策略大盤比較_TAIEX_TR_原始_v50_2026-10-19.json.gz','含股息TAIEX報酬指數，2320日。'),('大盤來源','策略大盤比較_TAIEX_PRICE_原始_v50_2026-10-19.json.gz','不含股息價格指數，僅補充對照。'),('來源查核','策略大盤比較_來源查核_v50_2026-10-19.json','URL、欄位、時間覆蓋與雜湊。'),('資料原始快取','外部歷史研究_獨立歷史快取_v12_2026-10-15.json.gz','上市價量、股息與公司行動快取。'),('資料原始快取','完整150檔研究_新增上櫃快取_v13_2026-10-15.json.gz','上櫃價量與事件補充。'),('資料原始快取','長歷史法人_原始快取_v30_2026-10-16.json.gz','外資／投信歷史原始資料。'),('資料品質','動能_長歷史來源修正稽核_v27_2026-10-15.json','來源修正與公司行動查核。'),('資料品質','策略研究_原資料快照_v3_2026-10-14.json.gz','股票名單／掛牌日期等凍結快照；研究原始輸入，不是最新結論。'),('研究管理','A策略_無人值守研究流程.md','已完成批次與接續規則。'),('研究管理','A策略_無人值守研究狀態.json','各批次狀態；完成不代表所有方法通過。'),('官賽限制','競賽規則與每日運作說明_2026-10-13.md','本機已歸檔規則解讀；並非今日重新認證。')]
resultmap54=[('原動能／五日','長歷史環境研究_動能法人逐窗口_v30_2026-10-16.json','results.MOM'),('純法人／五日','長歷史環境研究_動能法人逐窗口_v30_2026-10-16.json','results.FLOW'),('法人＋50／200','均線交叉_FLOW_50_200_ACTIVE_v45_2026-10-17.json.gz','results'),('純50／200','均線交叉_PURE_50_200_v45_2026-10-17.json.gz','results'),('10／20可變＋回退','可變持股均線_10_20_FALLBACK_v49_2026-10-17.json.gz','results'),('10／24可變＋回退','可變持股均線_10_24_FALLBACK_v49_2026-10-17.json.gz','results'),('動能初始持有','統一長歷史_MOM_HOLD_v34_2026-10-16.json.gz','results'),('法人初始持有','統一長歷史_FLOW_HOLD_v34_2026-10-16.json.gz','results')]
for label,f,key in resultmap54:core54.append(('交易結果',f,label+'：'+key+'；取912索引，不重跑交易。'))
assert all((root54/f).is_file() for _,f,_ in core54)
generated54={'研究資料_整理索引.html','研究資料_檔案清冊.json','README_研究資料導覽.md'}
corepaths54={f for _,f,_ in core54}
def category54(p):
    name=p.name;parts=p.relative_to(root54).parts
    if parts[0]=='發布網站':return '發布網站（獨立管理）'
    if parts[0] in ['團隊資料','類似競賽研究']:return '團隊與背景資料'
    if parts[0]=='官方文件':return '官方文件'
    if name in corepaths54:return '目前結論所需資料'
    if '整理前' in name or '對齊912前' in name:return '報告歷史備份'
    if p.suffix=='.ipynb':return '其他Notebook'
    if any(t in name for t in ['原始','快取','來源','稽核','修正','查核']):return '來源與品質查核'
    if name.endswith(('.json','.json.gz','.csv')):return '其他實驗定義與結果'
    if p.suffix in ['.html','.md','.png','.pdf','.docx']:return '歷史報告與說明'
    return '其他檔案'
files54=[];excluded54=[];hashes54={};duplicates54=collections.defaultdict(list)
for fp in sorted(root54.rglob('*')):
    if not fp.is_file():continue
    rel=fp.relative_to(root54)
    if any(x in ['.git','.openai','__pycache__','.ipynb_checkpoints'] for x in rel.parts):excluded54.append(str(rel));continue
    if fp.name in generated54:continue
    stat=fp.stat();digest=None
    # Hash canonical immutable inputs/results only. Active Notebook and report/state outputs may change at save.
    if str(rel) in corepaths54 and fp.suffix not in ['.ipynb','.html','.md'] and fp.name!='A策略_無人值守研究狀態.json':digest=hashlib.sha256(fp.read_bytes()).hexdigest();hashes54[str(rel)]=digest
    files54.append({'path':rel.as_posix(),'category':category54(fp),'bytes':stat.st_size,'modified_local':datetime.datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds'),'sha256_if_frozen_core':digest,'active_notebook_metadata_may_change_on_save':fp.name==notebook54})
total54=sum(x['bytes'] for x in files54);counts54=collections.Counter(x['category'] for x in files54)
def link54(path,label=None):return '<a href="'+html.escape(urllib.parse.quote(str(path).replace('\\','/'),safe='/'),quote=True)+'">'+html.escape(label or str(path))+'</a>'
def table54(head,rows):return '<div class="scroll"><table><thead><tr>'+''.join('<th>'+html.escape(str(h))+'</th>' for h in head)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+str(c)+'</td>' for c in row)+'</tr>' for row in rows)+'</tbody></table></div>'
intro54='<h1>研究資料導覽</h1><p>以總結報告為閱讀入口，依下列索引追溯統計、日期及原始資料。此次只建立分類與重現索引，沒有搬移、改名或刪除原始檔。</p><div class="lead">'+link54('策略研究_總結報告.html','閱讀策略研究總結')+'　｜　'+link54('README_研究資料導覽.md','文字導覽')+'</div><p>主表與持有比較：同912窗口；大盤為含息TAIEX；價量範圍2016～2025，主要比較2019～2025。檔名舊版本號不等於報告內容版本。</p>'
corehtml54='<h2>1．目前結論用哪些資料？</h2>'+table54(['用途','檔案','內容'],[[html.escape(cat),link54(f),html.escape(desc)] for cat,f,desc in core54 if cat!='交易結果'])
result54='<h2>2．八種策略的原始交易結果</h2>'+table54(['策略','結果檔','讀取位置'],[[html.escape(label),link54(f),html.escape(key)] for label,f,key in resultmap54])
steps54=['開啟同樣本日期稽核v53，使用included_indices的912個起點索引；不要以舊common_six直接代替。','依八種策略結果檔讀取原交易路徑，核對該索引的return、invalid與violations；不可判定不填0。','大盤精確配對進場前一交易日收盤至第24日收盤，使用含股息TAIEX；沿用已保存逐窗口報酬。','取同912日期計算正報酬勝率、勝大盤率、24日期望、超額、費用與窗口內回撤；區間使用v53的14項家族、24／48日區塊。','更新總結HTML／Markdown並核對兩節日期一致；在Notebook保存程式與輸出。只重算受資料或定義改變影響的部分。']
process54='<h2>3．如何重現與接續？</h2><ol>'+''.join('<li>'+html.escape(x)+'</li>' for x in steps54)+'</ol><p>原Notebook為計算紀錄。先查現有kernel的物件摘要與必要history，再執行必要cell；IPython history、kernel物件與已儲存Notebook不是同一份紀錄。未取得同意不得Restart／Run All或清除狀態。</p><p>Notebook中的v50→v51→v52→v53分別為大盤比較、討論收斂、版面重整、兩節同樣本。保存的程式與輸出是重現依據；v53會引用已保存資料，不需重新爬行情。</p><details><summary>既有程式與Git位置</summary><p>原Notebook寫入工具：<code>'+html.escape(str(author54))+'</code><br>kernel傳輸橋：<code>'+html.escape(str(bridge54))+'</code>。兩者在研究資料夾外，交接或備份時須一併考慮；橋接不是分析結果。</p><p>Git位於'+link54('發布網站/strategy-review-v3/README.md','發布網站／strategy-review-v3')+'，僅管理v3發布網站。根研究資料夾不是Git工作樹；本次不向網站repo加入原始資料、不提交、不推送或部署。</p></details>'
inventory54='<h2>4．完整分類清單</h2><p>一般研究檔案 '+str(len(files54))+' 個，約 '+f'{total54/1e6:.1f}'+' MB；不含Git／平台內部檔及本次索引本身。分類依用途與檔名建立，不代表歷史結果已全部失效。歷史來源可能仍是目前結論的必要輸入。</p>'
for cat in sorted(counts54):
    group=[x for x in files54 if x['category']==cat];inventory54+='<details><summary>'+html.escape(cat)+'（'+str(len(group))+'）</summary>'+table54(['檔案','大小MB'],[[link54(x['path']),f'{x["bytes"]/1e6:.3f}'] for x in group])+'</details>'
retention54='<h2>5．保存與版本原則</h2><ul><li>總結報告為固定入口；更新時保留資料快照、策略定義、日期稽核與Notebook。</li><li>原始資料與逐窗口交易結果不刪；若要節省空間，先確認可重建及引用關係，再另行處理。</li><li>整理前、912對齊前副本屬歷史備份，閱讀以固定總結為主；不以舊報告數字冒充目前同樣本結論。</li><li>現況名單偏差、未知公司行動與官賽規則缺口仍存在；檔案整理不表示研究已完整驗證。</li><li>本索引只供本機使用，不包含憑證內容；未新增對外發布。</li></ul>'
css54='body{font:16px/1.8 system-ui,"Microsoft JhengHei";background:#f4f6f9;color:#233043;margin:0}main{max-width:1280px;margin:auto;padding:32px 24px}h1{font-size:32px}h2{margin-top:38px}.lead{padding:18px;background:#e7eff8;border-left:4px solid #386da8}a{color:#245e98}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;font-size:14px;background:white}td,th{padding:10px;border-bottom:1px solid #dce3eb;text-align:left;vertical-align:top}th{background:#edf2f8}details{background:white;border:1px solid #dce3eb;padding:12px 16px;border-radius:8px;margin:12px 0}summary{cursor:pointer;font-weight:600}code{overflow-wrap:anywhere;font-size:13px}footer{font-size:13px;color:#64748b;margin-top:28px}'
index54='<!doctype html><html lang="zh-Hant"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>策略研究資料導覽</title><style>'+css54+'</style></head><body><main>'+intro54+corehtml54+result54+process54+inventory54+retention54+'<footer>更新：2026-10-19；檔案大小為整理當時快照，Notebook及狀態儲存後可能改變。</footer></main></body></html>'
(root54/'研究資料_整理索引.html').write_text(index54,encoding='utf-8')
md54='# 策略研究資料導覽\n\n更新：2026-10-19。僅建立索引，原檔案不搬移、不改名、不刪除。\n\n## 先從這裡開始\n\n- [策略研究總結](策略研究_總結報告.html)\n- [完整分類索引](研究資料_整理索引.html)\n- [機器可讀清冊](研究資料_檔案清冊.json)\n\n主表與持有比較均為912共同窗口；價量2016～2025，主要比較2019～2025。固定v50報告檔名不等於內容仍停在v50。\n\n## 核心資料\n\n|用途|檔案|說明|\n|---|---|---|\n'+'\n'.join('|'+cat+'|['+f+']('+urllib.parse.quote(f)+')|'+desc+'|' for cat,f,desc in core54)+'\n\n## 重現步驟\n\n'+'\n'.join(str(i+1)+'. '+x for i,x in enumerate(steps54))+'\n\n## 程式與Git\n\n主Notebook保存在本資料夾。寫入工具與橋接程式位於：\n\n- '+str(author54)+'\n- '+str(bridge54)+'\n\nGit在 `發布網站/strategy-review-v3`，僅為既有v3發布網站，根資料夾非Git工作樹；不要把研究原始資料直接加入網站repo。本次未commit、push或部署。\n\n## 接續規則\n\n先連既有kernel、查必要物件與history，分析留在kernel並保存原Notebook；未取得同意不Restart／Run All。原始快取、資料定義、日期稽核與交易路徑不可憑檔名新舊就刪除。歷史備份供追溯，不代表目前主結論。\n'
(root54/'README_研究資料導覽.md').write_text(md54,encoding='utf-8')
manifest54={'version':'v54_local_catalog','date':'2026-10-19','scope':'local organization only; no move rename delete or publication','root':str(root54),'current_main_cohort':912,'file_count_excluding_internal_and_catalogs':len(files54),'bytes_at_inventory':total54,'categories':dict(counts54),'frozen_core_sha256':hashes54,'core_roles':[{'role':a,'path':b,'description':c} for a,b,c in core54],'strategy_paths':[{'strategy':a,'file':b,'json_key':c} for a,b,c in resultmap54],'external_helpers':[str(author54),str(bridge54)],'git_scope':{'path':'發布網站/strategy-review-v3','purpose':'v3 website only','root_is_git':False,'modified':False},'files':files54,'excluded_internal_file_count':len(excluded54),'notebook_metadata':'activeNotebook catalogsize/timestamp maychange after bridge saves thiscell; do not treat as frozen source hash'}
(root54/'研究資料_檔案清冊.json').write_text(json.dumps(manifest54,ensure_ascii=False,indent=2),encoding='utf-8')
for stem in ['策略研究_總結報告','策略研究_對大盤勝率與超額報酬_v50_2026-10-19']:
    fp=root54/(stem+'.html');txt=fp.read_text(encoding='utf-8');anchor='<section class="appendix"><h2>附錄：查核與細節</h2>';assert anchor in txt;txt=txt.replace(anchor,anchor+'<p>'+link54('研究資料_整理索引.html','研究資料與重現索引')+'：原始資料、Notebook、目前統計與歷史版本。</p>',1);fp.write_text(txt,encoding='utf-8')
    fp=root54/(stem+'.md');fp.write_text(fp.read_text(encoding='utf-8')+'\n\n資料整理：[研究資料導覽](README_研究資料導覽.md)／[完整分類索引](研究資料_整理索引.html)。\n',encoding='utf-8')
fp=root54/'A策略_無人值守研究流程.md';fp.write_text(fp.read_text(encoding='utf-8')+'\n\n## v54 資料整理（完成）\n\n新增README、HTML分類索引與JSON檔案清冊，固定總結為入口，912樣本日期與8策略結果映射明確。原資料不搬移、改名或刪除；已找到發布網站/strategy-review-v3的Git，但僅為v3網站，未改網站repo或對外發布。核心凍結來源雜湊可核對，活躍Notebook不凍結hash。\n',encoding='utf-8')
statep54=root54/'A策略_無人值守研究狀態.json';state54=json.loads(statep54.read_text(encoding='utf-8'));state54['material_catalog_v54']={'status':'completed','running':False,'index':'研究資料_整理索引.html','manifest':'研究資料_檔案清冊.json','readme':'README_研究資料導覽.md','file_count':len(files54),'bytes_at_inventory':total54,'files_moved':0,'files_deleted':0,'website_repo_modified':False};statep54.write_text(json.dumps(state54,ensure_ascii=False,indent=2),encoding='utf-8')
assert all((root54/x['path']).is_file() for x in files54)
assert (root54/'策略研究_總結報告.html').read_text(encoding='utf-8')==(root54/'策略研究_對大盤勝率與超額報酬_v50_2026-10-19.html').read_text(encoding='utf-8')
print('V54_ORGANIZED',{'files':len(files54),'MB':round(total54/1e6,1),'categories':dict(counts54),'core_roles':len(core54),'strategy_paths':len(resultmap54),'frozen_hashes':len(hashes54),'moved':0,'deleted':0,'external_helpers_recorded':2,'git':'v3 publication repo found, untouched'})


In [ ]:
# Workspace layout compatibility; preserves original cell indices.
import sys as _layout_sys, importlib.util as _layout_importlib
if 'research_layout' not in _layout_sys.modules:
    _layout_spec = _layout_importlib.spec_from_file_location('research_layout', 'C:\\Users\\weiha\\Desktop\\文件\\03_課程資料\\115-1\\資料探勘與應用\\B_Final Project (0920~1221)\\競賽資料\\99_維護與備份\\工具\\研究路徑.py')
    _layout_mod = _layout_importlib.module_from_spec(_layout_spec)
    _layout_sys.modules['research_layout'] = _layout_mod
    _layout_spec.loader.exec_module(_layout_mod)
from research_layout import ResearchPath as Path

from research_layout import rebind_workspace_paths
layout_rebound55=rebind_workspace_paths(globals())
layout_root55=Path('C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料')
mapping55=json.loads((layout_root55/'00_導覽'/'檔案搬移對照.json').read_text(encoding='utf-8'))
assert Path(layout_root55/'動能持股_開賽前公告配置研究_2026-10-14.ipynb').is_file()
assert all((layout_root55/old).exists() for old in mapping55['mapping'])
with gzip.open(layout_root55/'可變持股均線_10_20_FALLBACK_v49_2026-10-17.json.gz','rt',encoding='utf-8') as f:layout_sample55=json.load(f)
assert len(layout_sample55['results'])==2236
check55=json.loads((layout_root55/'同樣本比較_日期稽核_v53_2026-10-19.json').read_text(encoding='utf-8'));assert len(check55['included_indices'])==912
assert (layout_root55/'策略研究_總結報告.html').read_text(encoding='utf-8')==(layout_root55/'策略研究_對大盤勝率與超額報酬_v50_2026-10-19.html').read_text(encoding='utf-8')
assert len(list(layout_root55.glob('*.ipynb')))==5
state55=json.loads((layout_root55/'A策略_無人值守研究狀態.json').read_text(encoding='utf-8'));state55['physical_layout_v55']={'status':'completed','running':False,'kernel_pid':os.getpid(),'kernel_restarted':False,'resolved_legacy_paths':len(mapping55['mapping']),'rebound_path_variables':layout_rebound55,'same_sample':912,'new_trade_runs':0,'readme':'README.md','mapping':'00_導覽/檔案搬移對照.json'};(layout_root55/'A策略_無人值守研究狀態.json').write_text(json.dumps(state55,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
flow55=layout_root55/'A策略_無人值守研究流程.md';flow55.write_text(flow55.read_text(encoding='utf-8')+'\n\n## v55 實體階層整理（完成）\n\n418檔實際分類，根目錄README與三HTML固定入口；每層README、舊新路徑對照與整理前ZIP已保存。291資料檔雜湊搬移前後相同，5Notebook保留cell位置；原PID31436沿用，路徑相容層核對資料與912索引成功。既有網站repo原位且未修改，外部writer／bridge接入相容；其他外部舊腳本未全面遷移。未重啟、回測或啟用排程。\n',encoding='utf-8')
print('V55_KERNEL_LAYOUT_VALIDATED',{'pid':os.getpid(),'legacy_paths':len(mapping55['mapping']),'main_notebook':str(layout_root55/'動能持股_開賽前公告配置研究_2026-10-14.ipynb'),'cached_rows':len(layout_sample55['results']),'same_sample':912,'rebound_variables':len(layout_rebound55),'new_trade_runs':0})
